In [ ]:
# US complete pipeline — paste this ENTIRE file into ONE Google Colab code cell.
# This includes every runtime module; it does not download your project code.
# Run the cell to publish data and all scores. Google credentials AND a FRED key
# are required. Secrets may also be configured in Colab Secrets using these names:
# GOOGLE_APPLICATION_CREDENTIALS (path), FRED_API_KEY,
# TELEGRAM_CHAT_ID and TELEGRAM_BOT_TOKEN (optional notification credentials).
# The credential JSON must be uploaded in each fresh runtime.
# Existing workbook IDs/structures and authorised imports are required.
# Edit STAGE to run one part independently; "all" runs everything in order.
DATASET = "all"  # all, coincident, leading; each has separate workbooks
STAGE = "all"  # importer, analysis, relationships, spread, correlation,
               # momentum, spread-momentum, correlation-momentum

import os
import sys
from pathlib import Path

SOURCES = {}

# Embedded source: run_us_pipeline.py
SOURCES['run_us_pipeline.py'] = (
    '#!/usr/bin/env python3\n'
    '"""Run all US data and scores, or select one independently runnable stage."""\n'
    '\n'
    'from __future__ import annotations\n'
    'import argparse\n'
    'from collections import deque\n'
    'from datetime import datetime, timezone\n'
    'import getpass\n'
    'import importlib.util\n'
    'import json\n'
    'import os\n'
    'from pathlib import Path\n'
    'import subprocess\n'
    'import sys\n'
    'import threading\n'
    'import time\n'
    'from uuid import uuid4\n'
    'from run_lock import acquire_lock\n'
    'from notifications.lifecycle import (\n'
    '    run_notified,\n'
    '    record_failure,\n'
    '    STAGE_NAMES,\n'
    '    set_run_details,\n'
    '    stage_event,\n'
    ')\n'
    '\n'
    'ROOT = Path(__file__).resolve().parent\n'
    'STOP_REQUEST = None\n'
    '\n'
    '\n'
    'class PipelineStopped(KeyboardInterrupt):\n'
    '    """An authorised stop request, handled by the pipeline\'s own process."""\n'
    '\n'
    '\n'
    'def check_stop():\n'
    '    if STOP_REQUEST is not None and STOP_REQUEST.exists():\n'
    '        raise PipelineStopped(\n'
    '            "Stopped from Telegram; partial writes are not rolled back"\n'
    '        )\n'
    '\n'
    '\n'
    'STAGES = (\n'
    '    "importer",\n'
    '    "analysis",\n'
    '    "relationships",\n'
    '    "spread",\n'
    '    "correlation",\n'
    '    "momentum",\n'
    '    "spread-momentum",\n'
    '    "correlation-momentum",\n'
    ')\n'
    'PACKAGES = {\n'
    '    "pandas": "pandas>=2,<4",\n'
    '    "numpy": "numpy>=1.26,<3",\n'
    '    "requests": "requests>=2.31,<3",\n'
    '    "bs4": "beautifulsoup4>=4.12,<5",\n'
    '    "xlrd": "xlrd>=2,<3",\n'
    '    "openpyxl": "openpyxl>=3.1,<4",\n'
    '    "google.auth": "google-auth>=2.30,<3",\n'
    '    "googleapiclient": "google-api-python-client>=2.140,<3",\n'
    '    "gspread": "gspread>=6,<7",\n'
    '}\n'
    '\n'
    '\n'
    'def dependencies():\n'
    '    missing = []\n'
    '    for module, requirement in PACKAGES.items():\n'
    '        try:\n'
    '            found = importlib.util.find_spec(module)\n'
    '        except ModuleNotFoundError:\n'
    '            found = None\n'
    '        if found is None:\n'
    '            missing.append(requirement)\n'
    '    if missing:\n'
    '        print("SETUP | Installing missing dependencies", flush=True)\n'
    '        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)\n'
    '\n'
    '\n'
    'def parse_args(argv=None):\n'
    '    p = argparse.ArgumentParser(description=__doc__)\n'
    '    p.add_argument("--stage", choices=("all", *STAGES), default="all")\n'
    '    p.add_argument(\n'
    '        "--dataset",\n'
    '        choices=("all", "coincident", "leading"),\n'
    '        default="all",\n'
    '        help="Relationship dataset; macro import/analysis still refresh the shared upstream workbook",\n'
    '    )\n'
    '    p.add_argument(\n'
    '        "--credentials", type=Path, default=os.getenv("GOOGLE_APPLICATION_CREDENTIALS")\n'
    '    )\n'
    '    p.add_argument("--api-key", default=os.getenv("FRED_API_KEY"))\n'
    '    p.add_argument("--months", type=int)\n'
    '    p.add_argument(\n'
    '        "--spreadsheet-id",\n'
    '        default=os.getenv(\n'
    '            "GOOGLE_SPREADSHEET_ID", "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c"\n'
    '        ),\n'
    '    )\n'
    '    p.add_argument(\n'
    '        "--wide-spreadsheet-id",\n'
    '        default=os.getenv(\n'
    '            "GOOGLE_WIDE_SPREADSHEET_ID", "1qIyTo-8esI23kaobPgnQT65Q6ykbjCXtPyNXC9Xo3KM"\n'
    '        ),\n'
    '    )\n'
    '    p.add_argument(\n'
    '        "--wide-sheet-name", default=os.getenv("GOOGLE_WIDE_SHEET_NAME", "All data")\n'
    '    )\n'
    '    p.add_argument(\n'
    '        "--self-test",\n'
    '        action="store_true",\n'
    '        help="Run offline regression tests; no source downloads or spreadsheet writes",\n'
    '    )\n'
    '    return p.parse_args(argv)\n'
    '\n'
    '\n'
    'def build_commands(args):\n'
    '    python = sys.executable\n'
    '    importer = [\n'
    '        python,\n'
    '        str(ROOT / "monthly_indicators.py"),\n'
    '        "--spreadsheet-id",\n'
    '        args.spreadsheet_id,\n'
    '        "--wide-spreadsheet-id",\n'
    '        args.wide_spreadsheet_id,\n'
    '        "--wide-sheet-name",\n'
    '        args.wide_sheet_name,\n'
    '        "--recent-only",\n'
    '        "--months",\n'
    '        str(args.months if args.months is not None else 12),\n'
    '    ]\n'
    '    analysis = [\n'
    '        python,\n'
    '        str(ROOT / "us_indicator_analysis.py"),\n'
    '        "--all",\n'
    '        "--spreadsheet-id",\n'
    '        args.spreadsheet_id,\n'
    '        "--verify",\n'
    '    ]\n'
    '    relationships = [\n'
    '        python,\n'
    '        str(ROOT / "run_relationships.py"),\n'
    '        "--dataset",\n'
    '        args.dataset,\n'
    '    ]\n'
    '    if args.stage not in ("all", "importer", "analysis", "relationships"):\n'
    '        relationships += ["--" + args.stage]\n'
    '    if args.stage == "all":\n'
    '        return [\n'
    '            ("Macro importer", importer),\n'
    '            ("Macro analysis", analysis),\n'
    '            (f"Spread, correlation and momentum ({args.dataset})", relationships),\n'
    '        ]\n'
    '    return [\n'
    '        (\n'
    '            args.stage,\n'
    '            (\n'
    '                importer\n'
    '                if args.stage == "importer"\n'
    '                else analysis if args.stage == "analysis" else relationships\n'
    '            ),\n'
    '        )\n'
    '    ]\n'
    '\n'
    '\n'
    'def run_stage(index, total, name, command, env, log):\n'
    '    check_stop()\n'
    '    started = time.monotonic()\n'
    '    stop = threading.Event()\n'
    '    print(f"\\n[{index}/{total}] {name}\\n" + "â”€" * 58, flush=True)\n'
    '\n'
    '    child = None\n'
    '    cancelled = threading.Event()\n'
    '\n'
    '    def heartbeat():\n'
    '        next_progress = time.monotonic() + 30\n'
    '        while not stop.wait(1):\n'
    '            if STOP_REQUEST is not None and STOP_REQUEST.exists() and child is not None:\n'
    '                cancelled.set()\n'
    '                if child.poll() is None:\n'
    '                    if os.name == "nt":\n'
    '                        subprocess.run(\n'
    '                            ["taskkill.exe", "/PID", str(child.pid), "/T", "/F"],\n'
    '                            stdout=subprocess.DEVNULL,\n'
    '                            stderr=subprocess.DEVNULL,\n'
    '                            timeout=15,\n'
    '                        )\n'
    '                    else:\n'
    '                        child.terminate()\n'
    '                return\n'
    '            if time.monotonic() >= next_progress:\n'
    '                print(\n'
    '                    f"  RUNNING | {name} | {time.monotonic()-started:.0f}s elapsed",\n'
    '                    flush=True,\n'
    '                )\n'
    '                next_progress = time.monotonic() + 30\n'
    '\n'
    '    thread = threading.Thread(target=heartbeat, daemon=True)\n'
    '    thread.start()\n'
    '    tail = deque(maxlen=8)\n'
    '    try:\n'
    '        child = subprocess.Popen(\n'
    '            command,\n'
    '            cwd=ROOT,\n'
    '            env=env,\n'
    '            stdout=subprocess.PIPE,\n'
    '            stderr=subprocess.STDOUT,\n'
    '            text=True,\n'
    '            bufsize=1,\n'
    '        )\n'
    '        for line in child.stdout:\n'
    '            tail.append(line.rstrip()[-1000:])\n'
    '            print("  " + line.rstrip(), flush=True)\n'
    '            log.write(line)\n'
    '            log.flush()\n'
    '        code = child.wait()\n'
    '        if cancelled.is_set():\n'
    '            raise PipelineStopped(\n'
    '                "Stopped from Telegram; partial writes are not rolled back"\n'
    '            )\n'
    '        check_stop()\n'
    '        if code:\n'
    '            raise RuntimeError(\n'
    '                f"{name} exited with status {code}; later stages were not run\\n"\n'
    '                + "\\n".join(tail)[-750:]\n'
    '            )\n'
    '    except BaseException:\n'
    '        if child is not None and child.poll() is None:\n'
    '            child.terminate()\n'
    '            try:\n'
    '                child.wait(timeout=10)\n'
    '            except subprocess.TimeoutExpired:\n'
    '                child.kill()\n'
    '                child.wait()\n'
    '        raise\n'
    '    finally:\n'
    '        stop.set()\n'
    '        thread.join()\n'
    '    elapsed = time.monotonic() - started\n'
    '    print(f"  DONE | {name} | {elapsed:.1f}s", flush=True)\n'
    '    return {"stage": name, "status": "completed", "seconds": round(elapsed, 2)}\n'
    '\n'
    '\n'
    'def main(argv=None):\n'
    '    global STOP_REQUEST\n'
    '    args = parse_args(argv)\n'
    '    if args.self_test:\n'
    '        return _run(args)\n'
    '    (ROOT / "work").mkdir(parents=True, exist_ok=True)\n'
    '    with (ROOT / "work" / "complete_pipeline.lock").open("a+") as lock:\n'
    '        try:\n'
    '            acquire_lock(lock)\n'
    '        except OSError:\n'
    '            from notifications.telegram import send_telegram\n'
    '\n'
    '            send_telegram(\n'
    '                "Another pipeline run is active. This request was skipped.",\n'
    '                "Pipeline already running",\n'
    '            )\n'
    '            return 0\n'
    '        run_id = uuid4().hex\n'
    '        STOP_REQUEST = ROOT / "work" / f"stop-{run_id}.request"\n'
    '        active = ROOT / "work" / "active-pipeline.json"\n'
    '        active.write_text(\n'
    '            json.dumps({"pid": os.getpid(), "run_id": run_id, "stage": args.stage}),\n'
    '            encoding="utf-8",\n'
    '        )\n'
    '        try:\n'
    '            return run_notified(\n'
    '                STAGE_NAMES[args.stage],\n'
    '                lambda: _run(args),\n'
    '                argv=["--dataset", args.dataset, "--api-key", args.api_key or ""],\n'
    '            )\n'
    '        finally:\n'
    '            active.unlink(missing_ok=True)\n'
    '            STOP_REQUEST.unlink(missing_ok=True)\n'
    '            STOP_REQUEST = None\n'
    '\n'
    '\n'
    'def _run(args):\n'
    '    if args.months is not None and args.months < 1:\n'
    '        raise ValueError("--months must be positive")\n'
    '    if args.spreadsheet_id == args.wide_spreadsheet_id:\n'
    '        raise ValueError("Macro and wide workbook IDs must differ")\n'
    '    if sys.version_info < (3, 10):\n'
    '        raise ValueError("Python 3.10 or later is required")\n'
    '    check_stop()\n'
    '    dependencies()\n'
    '    check_stop()\n'
    '    if args.self_test:\n'
    '        for script in ("monthly_indicators.py", "us_indicator_analysis.py"):\n'
    '            subprocess.run(\n'
    '                [sys.executable, str(ROOT / script), "--self-test"],\n'
    '                cwd=ROOT,\n'
    '                check=True,\n'
    '            )\n'
    '        subprocess.run(\n'
    '            [sys.executable, "-m", "pip", "install", "pytest>=8,<10"], check=True\n'
    '        )\n'
    '        return subprocess.run(\n'
    '            [sys.executable, "-m", "pytest", "-q", "tests"], cwd=ROOT\n'
    '        ).returncode\n'
    '    from config import credentials_path\n'
    '\n'
    '    try:\n'
    '        credential = credentials_path(\n'
    '            str(args.credentials) if args.credentials else None\n'
    '        )\n'
    '    except ValueError:\n'
    '        if not sys.stdin.isatty():\n'
    '            raise\n'
    '        credential = credentials_path(\n'
    '            input("Google service-account JSON path: ").strip()\n'
    '        )\n'
    '    if args.stage in ("all", "importer") and not args.api_key:\n'
    '        if not sys.stdin.isatty():\n'
    '            raise ValueError("Set FRED_API_KEY or pass --api-key")\n'
    '        args.api_key = getpass.getpass(\n'
    '            "FRED API key (paste, then press Enter; input is hidden): "\n'
    '        )\n'
    '        if not args.api_key:\n'
    '            raise ValueError("FRED API key is required")\n'
    '    env = {\n'
    '        **os.environ,\n'
    '        "GOOGLE_APPLICATION_CREDENTIALS": str(credential.resolve()),\n'
    '        "GOOGLE_SPREADSHEET_ID": args.spreadsheet_id,\n'
    '        "PYTHONUNBUFFERED": "1",\n'
    '        "US_PIPELINE_COMPACT_OUTPUT": "1",\n'
    '        "US_PIPELINE_FORCE_DIRECT_GOOGLE_API": "1",\n'
    '    }\n'
    '    if args.api_key:\n'
    '        env["FRED_API_KEY"] = args.api_key\n'
    '    env["US_PIPELINE_NOTIFICATIONS_MANAGED"] = "1"\n'
    '    run_dir = ROOT / "work" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")\n'
    '    run_dir.mkdir(parents=True)\n'
    '    env["US_PIPELINE_CHANGE_REPORT"] = str(run_dir / "changes.jsonl")\n'
    '    report = {"status": "running", "dataset": args.dataset, "stages": []}\n'
    '    started = time.monotonic()\n'
    '\n'
    '    def save_report():\n'
    '        from notifications.changes import events\n'
    '\n'
    '        report["changes"] = events(run_dir / "changes.jsonl")\n'
    '        report["seconds"] = round(time.monotonic() - started, 2)\n'
    '        temporary = run_dir / "summary.tmp"\n'
    '        temporary.write_text(json.dumps(report, indent=2), encoding="utf-8")\n'
    '        temporary.replace(run_dir / "summary.json")\n'
    '\n'
    '    try:\n'
    '        commands = build_commands(args)\n'
    '        set_run_details(\n'
    '            log_dir=run_dir,\n'
    '            stages=[name for name, _ in commands],\n'
    '            secrets=(args.api_key,),\n'
    '        )\n'
    '        print(\n'
    '            f"US PIPELINE | {len(commands)} stage(s) | live updates\\nLogs: {run_dir}",\n'
    '            flush=True,\n'
    '        )\n'
    '        with (run_dir / "pipeline.log").open("w") as log:\n'
    '            for index, (name, command) in enumerate(commands, 1):\n'
    '                report["current_stage"] = f"{index}/{len(commands)} {name}"\n'
    '                save_report()\n'
    '                stage_event(index, len(commands), name, "started")\n'
    '                stage_started = time.monotonic()\n'
    '                report["stages"].append(\n'
    '                    run_stage(index, len(commands), name, command, env, log)\n'
    '                )\n'
    '                save_report()\n'
    '                stage_event(\n'
    '                    index,\n'
    '                    len(commands),\n'
    '                    name,\n'
    '                    "finished",\n'
    '                    seconds=time.monotonic() - stage_started,\n'
    '                )\n'
    '        report["status"] = "completed"\n'
    '    except (Exception, KeyboardInterrupt) as exc:\n'
    '        report["status"] = (\n'
    '            "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"\n'
    '        )\n'
    '        report["error"] = str(exc)\n'
    '        if args.api_key:\n'
    '            report["error"] = report["error"].replace(args.api_key, "[redacted]")\n'
    '        print(\n'
    '            f\'\\nSTOPPED | {exc or "Interrupted"}\\nInspect logs: {run_dir}\',\n'
    '            file=sys.stderr,\n'
    '        )\n'
    '        record_failure(\n'
    '            f\'{type(exc).__name__}: {exc or "Interrupted"}\', secrets=(args.api_key,)\n'
    '        )\n'
    '        return 130 if isinstance(exc, KeyboardInterrupt) else 1\n'
    '    finally:\n'
    '        save_report()\n'
    '    print(\n'
    '        f\'\\nCOMPLETE | All selected stages verified | {report["seconds"]:.1f}s\',\n'
    '        flush=True,\n'
    '    )\n'
    '    return 0\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    try:\n'
    '        raise SystemExit(main())\n'
    '    except Exception as exc:\n'
    '        print(f"ERROR | {exc}", file=sys.stderr)\n'
    '        raise SystemExit(1)\n'
)

# Embedded source: monthly_indicators.py
SOURCES['monthly_indicators.py'] = (
    '#!/usr/bin/env python3\n'
    '"""\n'
    'Export monthly economic indicators to Google Sheets.\n'
    '\n'
    'Example:\n'
    '    python3 monthly_indicators.py --api-key YOUR_FRED_KEY\n'
    '\n'
    'You must pass the FRED API key in the run command with --api-key.\n'
    'ISM-only exports do not require a FRED API key.\n'
    '"""\n'
    '\n'
    'from __future__ import annotations\n'
    '\n'
    'import argparse\n'
    'import base64\n'
    'import hashlib\n'
    'import importlib.util\n'
    'import json\n'
    'import logging\n'
    'import os\n'
    'import re\n'
    'import subprocess\n'
    'import sys\n'
    'import tempfile\n'
    'import time\n'
    'from contextlib import redirect_stdout\n'
    'from dataclasses import dataclass\n'
    'from datetime import date, datetime, timedelta, timezone\n'
    'from decimal import ROUND_HALF_UP, Decimal, InvalidOperation\n'
    'from io import BytesIO, StringIO\n'
    'from pathlib import Path\n'
    'from time import sleep\n'
    'from typing import Iterable\n'
    'from urllib.error import HTTPError, URLError\n'
    'from urllib.parse import parse_qsl, quote, urlencode, urljoin, urlparse\n'
    'from urllib.request import urlopen\n'
    '\n'
    'BOOTSTRAP_PACKAGES = {\n'
    '    "pandas": "pandas>=2.0",\n'
    '    "requests": "requests>=2.31",\n'
    '    "bs4": "beautifulsoup4>=4.12",\n'
    '    "urllib3": "urllib3>=2.0",\n'
    '    "xlrd": "xlrd>=2.0.1",\n'
    '    "openpyxl": "openpyxl>=3.1",\n'
    '    "googleapiclient": "google-api-python-client>=2.140",\n'
    '}\n'
    '\n'
    '\n'
    'def ensure_dependencies() -> None:\n'
    '    """Install missing runtime packages so the script can be run with one command."""\n'
    '    missing = [\n'
    '        package\n'
    '        for module, package in BOOTSTRAP_PACKAGES.items()\n'
    '        if importlib.util.find_spec(module) is None\n'
    '    ]\n'
    '    if not missing:\n'
    '        return\n'
    '\n'
    '    print("Installing missing Python packages: " + ", ".join(missing))\n'
    '    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])\n'
    '\n'
    '\n'
    'ensure_dependencies()\n'
    '\n'
    'import pandas as pd\n'
    'import requests\n'
    'from bs4 import BeautifulSoup\n'
    'from requests.adapters import HTTPAdapter\n'
    'from urllib3.util.retry import Retry\n'
    '\n'
    'FRED_API_BASE = "https://api.stlouisfed.org/fred"\n'
    'IMF_DATAMAPPER_API_BASE = "https://www.imf.org/external/datamapper/api/v2"\n'
    'DEFAULT_IMF_COUNTRY = "USA"\n'
    'GOOGLE_TOKEN_URL = "https://oauth2.googleapis.com/token"\n'
    'GOOGLE_SHEETS_API_BASE = "https://sheets.googleapis.com/v4/spreadsheets"\n'
    'GOOGLE_REQUEST_CONNECT_TIMEOUT_SECONDS = 15\n'
    'GOOGLE_REQUEST_READ_TIMEOUT_SECONDS = 90\n'
    'GOOGLE_REQUEST_MAX_ATTEMPTS = 5\n'
    'DEFAULT_GOOGLE_SPREADSHEET_ID = "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c"\n'
    'DEFAULT_WIDE_SPREADSHEET_ID = "1qIyTo-8esI23kaobPgnQT65Q6ykbjCXtPyNXC9Xo3KM"\n'
    'COMPACT_TERMINAL = os.getenv("US_PIPELINE_COMPACT_OUTPUT") == "1"\n'
    'ANALYSIS_DECIMAL_PLACES = 5\n'
    'ANALYSIS_VALUE_QUANTUM = Decimal("0.00001")\n'
    '\n'
    '\n'
    'def find_default_google_credentials() -> str:\n'
    '    configured = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")\n'
    '    if configured:\n'
    '        return configured\n'
    '    script_path = Path(__file__).resolve()\n'
    '    for parent in (script_path.parent, *script_path.parents):\n'
    '        candidate = parent / "google_credentials.json"\n'
    '        if candidate.is_file():\n'
    '            return str(candidate)\n'
    '    return str(script_path.parent / "google_credentials.json")\n'
    '\n'
    '\n'
    'DEFAULT_GOOGLE_CREDENTIALS = find_default_google_credentials()\n'
    'DEFAULT_GOOGLE_SHEET_NAME = "Monthly Indicators"\n'
    'DEFAULT_WIDE_SHEET_NAME = "All data"\n'
    'DEFAULT_GOOGLE_STATUS_SHEET_NAME = "Source Status"\n'
    'DEFAULT_GOOGLE_METADATA_SHEET_NAME = "Indicator Metadata"\n'
    'DEFAULT_GOOGLE_VALIDATION_SHEET_NAME = "Validation Report"\n'
    'DEFAULT_GOOGLE_DIAGNOSTIC_SHEET_NAME = "Selection Diagnostics"\n'
    'LEGACY_SHEET_NAMES = {"FRED Indicators", "ISM Indicators"}\n'
    'CENTRAL_BANK_LIQUIDITY_SHEET = "Central bank liquidity growth"\n'
    '(Path(__file__).parent / "work").mkdir(exist_ok=True)\n'
    'ERROR_LOG_PATH = Path(__file__).parent / "work" / "monthly_indicators_errors.log"\n'
    'SOURCE_ARCHIVE_ROOT = Path(__file__).parent / "work" / "source_archive"\n'
    'OUTPUT_ROOT = Path(__file__).parent / "work" / "importer_outputs"\n'
    'GOOGLE_SHEET_AUDIT_CSV = OUTPUT_ROOT / "google_sheet_audit.csv"\n'
    'ISM_REPORT_INDEX_URL = (\n'
    '    "https://www.ismworld.org/supply-management-news-and-reports/"\n'
    '    "reports/ism-pmi-reports/"\n'
    ')\n'
    'NYFED_SCE_DATA_URL = (\n'
    '    "https://www.newyorkfed.org/medialibrary/interactives/sce/sce/"\n'
    '    "downloads/data/frbny-sce-data.xlsx?sc_lang=en"\n'
    ')\n'
    'NYFED_SCE_PAGE_URL = "https://www.newyorkfed.org/microeconomics/sce"\n'
    'MICHIGAN_CONSUMER_EXPECTATIONS_URL = "https://www.sca.isr.umich.edu/files/chicer.xls"\n'
    'MICHIGAN_CONSUMER_SENTIMENT_URL = "https://www.sca.isr.umich.edu/files/chicsr.xls"\n'
    'MICHIGAN_CHARTS_PAGE_URL = "https://www.sca.isr.umich.edu/charts.html"\n'
    'OFFICIAL_SOURCE_HOSTS = {\n'
    '    "www.ismworld.org",\n'
    '    "ismworld.org",\n'
    '    "www.newyorkfed.org",\n'
    '    "newyorkfed.org",\n'
    '    "www.sca.isr.umich.edu",\n'
    '    "sca.isr.umich.edu",\n'
    '    "www.imf.org",\n'
    '    "imf.org",\n'
    '}\n'
    'MIN_HTML_BYTES = 1_000\n'
    'MIN_WORKBOOK_BYTES = 5_000\n'
    '\n'
    '\n'
    'class OfficialSourceRedirectError(RuntimeError):\n'
    '    """Raised when an official source redirects outside its approved data hosts."""\n'
    '\n'
    '    def __init__(self, source_name: str, url: str) -> None:\n'
    '        super().__init__(f"{source_name} returned a non-official URL: {url}")\n'
    '        self.url = url\n'
    '\n'
    '\n'
    '@dataclass\n'
    'class SourceStatus:\n'
    '    source: str\n'
    '    status: str\n'
    '    latest_data_month: str\n'
    '    retrieved_at: str\n'
    '    source_url: str\n'
    '    sha256: str\n'
    '    byte_size: int\n'
    '    content_type: str\n'
    '    schema_fingerprint: str\n'
    '    archive_file: str\n'
    '    revision: str = ""\n'
    '\n'
    '\n'
    'MONTHS = [\n'
    '    "january",\n'
    '    "february",\n'
    '    "march",\n'
    '    "april",\n'
    '    "may",\n'
    '    "june",\n'
    '    "july",\n'
    '    "august",\n'
    '    "september",\n'
    '    "october",\n'
    '    "november",\n'
    '    "december",\n'
    ']\n'
    'MONTH_NUMBERS = {name: idx + 1 for idx, name in enumerate(MONTHS)}\n'
    'MONTH_ALIASES = {\n'
    '    "jan": "january",\n'
    '    "feb": "february",\n'
    '    "mar": "march",\n'
    '    "apr": "april",\n'
    '    "jun": "june",\n'
    '    "jul": "july",\n'
    '    "aug": "august",\n'
    '    "sep": "september",\n'
    '    "sept": "september",\n'
    '    "oct": "october",\n'
    '    "nov": "november",\n'
    '    "dec": "december",\n'
    '}\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class Indicator:\n'
    '    name: str\n'
    '    series_id: str | None = None\n'
    '    units: str = "lin"\n'
    '    numerator_series_id: str | None = None\n'
    '    denominator_series_id: str | None = None\n'
    '    calculation: str | None = None\n'
    '    source: str = "fred"\n'
    '    source_column: str | None = None\n'
    '    release_lag_months: int = 0\n'
    '    monthly_aggregation: str = "latest"\n'
    '    monthly_method: str = (\n'
    '        "Most recent source observation assigned to the row month; otherwise blank"\n'
    '    )\n'
    '    unit_label: str = ""\n'
    '    release_rule: str = ""\n'
    '\n'
    '\n'
    '@dataclass\n'
    'class IsmReportData:\n'
    '    report_date: date\n'
    '    source_url: str\n'
    '    values: dict[str, float | None]\n'
    '    publication_date: date | None = None\n'
    '\n'
    '\n'
    'SOURCE_STATUSES: dict[str, SourceStatus] = {}\n'
    'VALIDATION_MESSAGES: list[dict[str, str]] = []\n'
    'LAST_SELECTIONS: dict[tuple[str, date], dict | None] = {}\n'
    'DEFAULT_OUTPUT_MONTHS = 12\n'
    'CURRENT_OUTPUT_MONTHS = DEFAULT_OUTPUT_MONTHS\n'
    'FULL_HISTORY_MODE = False\n'
    '\n'
    '\n'
    'def set_source_status(status: SourceStatus) -> None:\n'
    '    """Keep the newest period when one source archives several historical files."""\n'
    '    existing = SOURCE_STATUSES.get(status.source)\n'
    '    if existing is None or status.latest_data_month >= existing.latest_data_month:\n'
    '        SOURCE_STATUSES[status.source] = status\n'
    '\n'
    '\n'
    'def is_central_bank_component(indicator_name: str) -> bool:\n'
    '    """Return true for source series that live in the shared liquidity tab."""\n'
    '    return indicator_name in CENTRAL_BANK_LIQUIDITY_COMPONENTS\n'
    '\n'
    '\n'
    'def is_imf_support_indicator(indicator_name: str) -> bool:\n'
    '    """Return true for IMF inputs that feed the published coverage-ratio tab."""\n'
    '    return indicator_name in IMF_SUPPORT_INDICATORS\n'
    '\n'
    '\n'
    'def is_imf_coverage_indicator(indicator_name: str) -> bool:\n'
    '    """Return true for source/derived indicators belonging to the shared IMF tab."""\n'
    '    return (\n'
    '        indicator_name in IMF_SUPPORT_INDICATORS or indicator_name == IMF_COVERAGE_SHEET\n'
    '    )\n'
    '\n'
    '\n'
    'def has_dedicated_indicator_tab(indicator_name: str) -> bool:\n'
    '    """Return true when an indicator should have its own Google Sheet tab."""\n'
    '    return not is_central_bank_component(\n'
    '        indicator_name\n'
    '    ) and not is_imf_coverage_indicator(indicator_name)\n'
    '\n'
    '\n'
    'INDICATORS = [\n'
    '    Indicator(\n'
    '        "Manufacturing PMI",\n'
    '        source="ism",\n'
    '        source_column="manufacturing_pmi",\n'
    '        monthly_method="Most recent official report observation assigned to the row month; otherwise blank",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Manufacturing new orders",\n'
    '        source="ism",\n'
    '        source_column="manufacturing_new_orders",\n'
    '        monthly_method="Most recent official report observation assigned to the row month; otherwise blank",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Services PMI",\n'
    '        source="ism",\n'
    '        source_column="services_pmi",\n'
    '        monthly_method="Most recent official report observation assigned to the row month; otherwise blank",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Services new orders",\n'
    '        source="ism",\n'
    '        source_column="services_new_orders",\n'
    '        monthly_method="Most recent official report observation assigned to the row month; otherwise blank",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Services business activity",\n'
    '        source="ism",\n'
    '        source_column="services_business_activity",\n'
    '        monthly_method="Most recent official report observation assigned to the row month; otherwise blank",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Industrial production growth",\n'
    '        "INDPRO",\n'
    '        units="lin",\n'
    '        unit_label="Index 2017=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Real consumer spending growth",\n'
    '        "PCEC96",\n'
    '        units="lin",\n'
    '        unit_label="Billions of chained 2017 dollars",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Consumer sentiment",\n'
    '        source="michigan_sentiment",\n'
    '        source_column="ICS_M",\n'
    '        unit_label="Index",\n'
    '        release_rule="Michigan final monthly release; preliminary only if final is unavailable",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Consumer expectations",\n'
    '        source="michigan_sca",\n'
    '        source_column="ICE_M",\n'
    '        unit_label="Index",\n'
    '        release_rule="Michigan final monthly release; preliminary only if final is unavailable",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Initial unemployment claims",\n'
    '        "ICSA",\n'
    '        monthly_aggregation="latest",\n'
    '        monthly_method="Latest weekly observation in the row month",\n'
    '        unit_label="Number",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Employment growth",\n'
    '        "PAYEMS",\n'
    '        units="lin",\n'
    '        unit_label="Thousands of persons",\n'
    '    ),\n'
    '    Indicator("Average hours worked", "AWHAETP"),\n'
    '    Indicator("Building permits", "PERMIT"),\n'
    '    Indicator(\n'
    '        "Core consumption inflation",\n'
    '        "PCEPILFE",\n'
    '        units="lin",\n'
    '        unit_label="Index 2017=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Core CPI inflation",\n'
    '        "CPILFESL",\n'
    '        units="lin",\n'
    '        unit_label="Index 1982-1984=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Headline CPI inflation",\n'
    '        "CPIAUCSL",\n'
    '        units="lin",\n'
    '        unit_label="Index 1982-1984=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Core producer price inflation",\n'
    '        "PPIFES",\n'
    '        units="lin",\n'
    '        unit_label="Index Apr 2010=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Headline producer price inflation",\n'
    '        "PPIFIS",\n'
    '        units="lin",\n'
    '        unit_label="Index Nov 2009=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Wage growth",\n'
    '        "ECIWAG",\n'
    '        units="lin",\n'
    '        unit_label="Index Dec 2005=100",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Consumer inflation expectations",\n'
    '        source="nyfed_sce",\n'
    '        source_column="Median one-year ahead expected inflation rate",\n'
    '        release_lag_months=1,\n'
    '        unit_label="Percent",\n'
    '        release_rule="NY Fed SCE publication date; conservative 20th-of-next-month fallback",\n'
    '        monthly_method="Most recent NY Fed SCE survey observation assigned to the row month; otherwise blank",\n'
    '    ),\n'
    '    Indicator("10Y real yield", "DFII10"),\n'
    '    Indicator("2Y government bond yield", "DGS2"),\n'
    '    Indicator("10Y government bond yield", "DGS10"),\n'
    '    Indicator("10Y-3M yield curve spread", "T10Y3M"),\n'
    '    Indicator("10Y term premium", "THREEFYTP10"),\n'
    '    Indicator(\n'
    '        "Overnight interbank rate",\n'
    '        "IRSTCI01USM156N",\n'
    '        unit_label="Percent",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level",\n'
    '        "WALCL",\n'
    '        units="lin",\n'
    '        unit_label="Millions of U.S. dollars",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level",\n'
    '        "WDTGAL",\n'
    '        units="lin",\n'
    '        unit_label="Millions of U.S. dollars",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations",\n'
    '        "RRPONTSYD",\n'
    '        units="lin",\n'
    '        unit_label="Billions of U.S. dollars",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Banking system reserves",\n'
    '        "WRESBAL",\n'
    '        unit_label="Millions of U.S. dollars",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Broad money growth",\n'
    '        "M2SL",\n'
    '        units="lin",\n'
    '        unit_label="Billions of dollars",\n'
    '    ),\n'
    '    Indicator("Fiscal balance % GDP", "FYFSGDA188S"),\n'
    '    Indicator(\n'
    '        "Real government spending growth",\n'
    '        "GCEC1",\n'
    '        units="lin",\n'
    '        unit_label="Billions of chained 2017 dollars, SAAR",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Government revenue growth",\n'
    '        "MTSR133FMS",\n'
    '        units="lin",\n'
    '        unit_label="Millions of dollars, not seasonally adjusted",\n'
    '    ),\n'
    '    Indicator("Public debt % GDP", "GFDEGDQ188S"),\n'
    '    Indicator(\n'
    '        "Government revenue % GDP",\n'
    '        "rev",\n'
    '        source="imf_datamapper",\n'
    '        source_column=DEFAULT_IMF_COUNTRY,\n'
    '        unit_label="Percent of GDP",\n'
    '        monthly_method="Annual IMF FPP observation matched to its December year-end month",\n'
    '        release_rule="Latest available IMF DataMapper FPP release metadata",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Interest paid on public debt % GDP",\n'
    '        "ie",\n'
    '        source="imf_datamapper",\n'
    '        source_column=DEFAULT_IMF_COUNTRY,\n'
    '        unit_label="Percent of GDP",\n'
    '        monthly_method="Annual IMF FPP observation matched to its December year-end month",\n'
    '        release_rule="Latest available IMF DataMapper FPP release metadata",\n'
    '    ),\n'
    '    Indicator(\n'
    '        "Government interest coverage ratio",\n'
    '        "rev_over_ie",\n'
    '        source="imf_datamapper",\n'
    '        source_column=DEFAULT_IMF_COUNTRY,\n'
    '        calculation="rev / ie",\n'
    '        unit_label="Ratio (x)",\n'
    '        monthly_method="Annual IMF FPP revenue divided by interest paid, matched to December year-end month",\n'
    '        release_rule="Latest available IMF DataMapper FPP release metadata",\n'
    '    ),\n'
    '    Indicator("Financial conditions index", "NFCI"),\n'
    '    Indicator(\n'
    '        "Bank credit growth",\n'
    '        "TOTLL",\n'
    '        units="lin",\n'
    '        unit_label="Billions of U.S. dollars",\n'
    '    ),\n'
    '    Indicator("Corporate credit spread", "BAA10Y"),\n'
    ']\n'
    'EXPECTED_HEADERS = [\n'
    '    "Manufacturing PMI",\n'
    '    "Manufacturing new orders",\n'
    '    "Services PMI",\n'
    '    "Services new orders",\n'
    '    "Services business activity",\n'
    '    "Industrial production growth",\n'
    '    "Real consumer spending growth",\n'
    '    "Consumer sentiment",\n'
    '    "Consumer expectations",\n'
    '    "Initial unemployment claims",\n'
    '    "Employment growth",\n'
    '    "Average hours worked",\n'
    '    "Building permits",\n'
    '    "Core consumption inflation",\n'
    '    "Core CPI inflation",\n'
    '    "Headline CPI inflation",\n'
    '    "Core producer price inflation",\n'
    '    "Headline producer price inflation",\n'
    '    "Wage growth",\n'
    '    "Consumer inflation expectations",\n'
    '    "10Y real yield",\n'
    '    "2Y government bond yield",\n'
    '    "10Y government bond yield",\n'
    '    "10Y-3M yield curve spread",\n'
    '    "10Y term premium",\n'
    '    "Overnight interbank rate",\n'
    '    "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level",\n'
    '    "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level",\n'
    '    "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations",\n'
    '    "Banking system reserves",\n'
    '    "Broad money growth",\n'
    '    "Fiscal balance % GDP",\n'
    '    "Real government spending growth",\n'
    '    "Government revenue growth",\n'
    '    "Public debt % GDP",\n'
    '    "Government revenue % GDP",\n'
    '    "Interest paid on public debt % GDP",\n'
    '    "Government interest coverage ratio",\n'
    '    "Financial conditions index",\n'
    '    "Bank credit growth",\n'
    '    "Corporate credit spread",\n'
    ']\n'
    '\n'
    'CENTRAL_BANK_LIQUIDITY_COMPONENTS = {\n'
    '    "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level": {\n'
    '        "column": 1,\n'
    '        "header": "Central Bank Total Assets",\n'
    '    },\n'
    '    "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level": {\n'
    '        "column": 2,\n'
    '        "header": "Cental Bank General Account",\n'
    '    },\n'
    '    "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations": {\n'
    '        "column": 3,\n'
    '        "header": "Central Bank RRP",\n'
    '    },\n'
    '}\n'
    'IMF_SUPPORT_INDICATORS = {\n'
    '    "Government revenue % GDP",\n'
    '    "Interest paid on public debt % GDP",\n'
    '}\n'
    'IMF_COVERAGE_SHEET = "Government interest coverage ratio"\n'
    'IMF_COVERAGE_INPUT_COLUMNS = {\n'
    '    "Government revenue % GDP": 1,\n'
    '    "Interest paid on public debt % GDP": 2,\n'
    '}\n'
    'ANALYSIS_CONTROL_PANEL_SHEET = "Control Panel"\n'
    'EXPECTED_ANALYSIS_SHEET_ORDER = tuple(\n'
    '    dict.fromkeys(\n'
    '        (\n'
    '            CENTRAL_BANK_LIQUIDITY_SHEET\n'
    '            if name in CENTRAL_BANK_LIQUIDITY_COMPONENTS\n'
    '            else (\n'
    '                IMF_COVERAGE_SHEET\n'
    '                if name in IMF_SUPPORT_INDICATORS or name == IMF_COVERAGE_SHEET\n'
    '                else name\n'
    '            )\n'
    '        )\n'
    '        for name in EXPECTED_HEADERS\n'
    '    )\n'
    ')\n'
    'ANALYSIS_CONTROL_PANEL_RANGE = (\n'
    '    f"\'{ANALYSIS_CONTROL_PANEL_SHEET}\'!" f"A3:A{len(EXPECTED_ANALYSIS_SHEET_ORDER) + 2}"\n'
    ')\n'
    'QUARTER_END_MONTHS = {3, 6, 9, 12}\n'
    'ANNUAL_OUTPUT_MONTH = 12\n'
    'SPARSE_FRED_FREQUENCY_PREFIXES = ("Quarterly", "Annual")\n'
    'GOOGLE_SHEETS_WRITE_SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]\n'
    'GOOGLE_SHEETS_READONLY_SCOPES = [\n'
    '    "https://www.googleapis.com/auth/spreadsheets.readonly"\n'
    ']\n'
    '\n'
    '_ISM_CACHE: dict[int, pd.DataFrame] = {}\n'
    '_EXPLICIT_BLANK_OUTPUT_MONTHS: dict[str, set[date]] = {}\n'
    '\n'
    '# Preserve existing values when a current source no longer exposes older\n'
    '# history. Entries belong here only after the individual workbook cells have\n'
    '# been independently verified as erroneous, never merely because a response\n'
    '# omits their dates.\n'
    '_VERIFIED_INVALID_EXISTING_MONTHS: dict[str, set[date]] = {\n'
    '    "Core CPI inflation": {date(2025, 10, 1)},\n'
    '    "Headline CPI inflation": {date(2025, 10, 1)},\n'
    '    "Consumer inflation expectations": {date(2013, 6, 1)},\n'
    '}\n'
    '_NYFED_SCE_CACHE: pd.DataFrame | None = None\n'
    '_MICHIGAN_CONSUMER_EXPECTATIONS_CACHE: pd.DataFrame | None = None\n'
    '_MICHIGAN_CONSUMER_SENTIMENT_CACHE: pd.DataFrame | None = None\n'
    '_FRED_METADATA_CACHE: dict[str, dict] = {}\n'
    '_IMF_FPP_CACHE: dict[str, dict] = {}\n'
    'LAST_FRED_REQUEST_AT = 0.0\n'
    'FRED_MIN_REQUEST_INTERVAL_SECONDS = 0.65\n'
    'FRED_EXPECTED_METADATA = {\n'
    '    "CPIAUCSL": {\n'
    '        "title_contains": "Consumer Price Index for All Urban Consumers: All Items",\n'
    '        "units_contains": "Index 1982-1984=100",\n'
    '        "frequency": "Monthly",\n'
    '    },\n'
    '    "DFII10": {\n'
    '        "title_contains": "10-Year Constant Maturity",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '    "DGS2": {\n'
    '        "title_contains": "2-Year Constant Maturity",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '    "DGS10": {\n'
    '        "title_contains": "10-Year Constant Maturity",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '    "T10Y3M": {\n'
    '        "title_contains": "10-Year Treasury Constant Maturity Minus 3-Month",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '    "THREEFYTP10": {\n'
    '        "title_contains": "Term Premium on a 10 Year Zero Coupon Bond",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '    "IRSTCI01USM156N": {\n'
    '        "title_contains": "Call Money/Interbank Rate: Total for United States",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Monthly",\n'
    '    },\n'
    '    "INDPRO": {\n'
    '        "title_contains": "Industrial Production",\n'
    '        "units_contains": "Index 2017=100",\n'
    '        "frequency": "Monthly",\n'
    '    },\n'
    '    "PCEC96": {\n'
    '        "title_contains": "Real Personal Consumption Expenditures",\n'
    '        "units_contains": "Chained 2017 Dollars",\n'
    '        "frequency": "Monthly",\n'
    '    },\n'
    '    "ECIWAG": {\n'
    '        "title_contains": "Employment Cost Index: Wages and Salaries",\n'
    '        "units_contains": "Index Dec 2005=100",\n'
    '        "frequency": "Quarterly",\n'
    '    },\n'
    '    "MTSR133FMS": {\n'
    '        "title_contains": "Total Federal Receipts",\n'
    '        "units_contains": "Millions of Dollars",\n'
    '        "frequency": "Monthly",\n'
    '    },\n'
    '    "WALCL": {\n'
    '        "title_contains": "Assets",\n'
    '        "units_contains": "Millions of U.S. Dollars",\n'
    '        "frequency": "Weekly, As of Wednesday",\n'
    '    },\n'
    '    "WDTGAL": {\n'
    '        "title_contains": "U.S. Treasury, General Account",\n'
    '        "units_contains": "Millions of U.S. Dollars",\n'
    '        "frequency": "Weekly, As of Wednesday",\n'
    '    },\n'
    '    "RRPONTSYD": {\n'
    '        "title_contains": "Overnight Reverse Repurchase Agreements",\n'
    '        "units_contains": "Billions of U.S. Dollars",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '    "WRESBAL": {\n'
    '        "title_contains": "Reserve Balances with Federal Reserve Banks",\n'
    '        "units_contains": "Millions of U.S. Dollars",\n'
    '        "frequency": "Weekly, Ending Wednesday",\n'
    '    },\n'
    '    "NFCI": {\n'
    '        "title_contains": "Chicago Fed National Financial Conditions Index",\n'
    '        "units_contains": "Index",\n'
    '        "frequency": "Weekly, Ending Friday",\n'
    '    },\n'
    '    "TOTLL": {\n'
    '        "title_contains": "Loans and Leases in Bank Credit",\n'
    '        "units_contains": "Billions of U.S. Dollars",\n'
    '        "frequency": "Weekly, Ending Wednesday",\n'
    '    },\n'
    '    "BAA10Y": {\n'
    '        "title_contains": "Baa Corporate Bond Yield Relative",\n'
    '        "units_contains": "Percent",\n'
    '        "frequency": "Daily",\n'
    '    },\n'
    '}\n'
    'ISM_VALUE_COLUMNS = [\n'
    '    "manufacturing_pmi",\n'
    '    "manufacturing_new_orders",\n'
    '    "services_pmi",\n'
    '    "services_new_orders",\n'
    '    "services_business_activity",\n'
    ']\n'
    'ISM_VALUE_LABELS = {\n'
    '    "manufacturing_pmi": "Manufacturing PMI",\n'
    '    "manufacturing_new_orders": "Manufacturing new orders",\n'
    '    "services_pmi": "Services PMI",\n'
    '    "services_new_orders": "Services new orders",\n'
    '    "services_business_activity": "Services business activity",\n'
    '}\n'
    '\n'
    '\n'
    'def subtract_months(input_date: date, months: int) -> date:\n'
    '    month_index = input_date.month - 1 - months\n'
    '    year = input_date.year + month_index // 12\n'
    '    month = month_index % 12 + 1\n'
    '    days_in_month = [\n'
    '        31,\n'
    '        29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,\n'
    '        31,\n'
    '        30,\n'
    '        31,\n'
    '        30,\n'
    '        31,\n'
    '        31,\n'
    '        30,\n'
    '        31,\n'
    '        30,\n'
    '        31,\n'
    '    ]\n'
    '    day = min(input_date.day, days_in_month[month - 1])\n'
    '    return date(year, month, day)\n'
    '\n'
    '\n'
    'def month_start(input_date: date) -> date:\n'
    '    return date(input_date.year, input_date.month, 1)\n'
    '\n'
    '\n'
    'def add_months(input_date: date, months: int) -> date:\n'
    '    month_index = input_date.month - 1 + months\n'
    '    year = input_date.year + month_index // 12\n'
    '    month = month_index % 12 + 1\n'
    '    return date(year, month, 1)\n'
    '\n'
    '\n'
    'def month_end(input_date: date) -> date:\n'
    '    return add_months(input_date, 1) - date.resolution\n'
    '\n'
    '\n'
    'def nth_weekday_of_month(input_month: date, occurrence: int) -> date:\n'
    '    """Return the nth Monday-Friday date in a month."""\n'
    '    current = month_start(input_month)\n'
    '    found = 0\n'
    '    while True:\n'
    '        if current.weekday() < 5:\n'
    '            found += 1\n'
    '            if found == occurrence:\n'
    '                return current\n'
    '        current += date.resolution\n'
    '\n'
    '\n'
    'def last_weekday_of_month(input_month: date, weekday: int) -> date:\n'
    '    current = month_end(input_month)\n'
    '    while current.weekday() != weekday:\n'
    '        current -= date.resolution\n'
    '    return current\n'
    '\n'
    '\n'
    'def recent_month_starts(months: int) -> list[date]:\n'
    '    current_month = month_start(date.today())\n'
    '    first_month = add_months(current_month, -(months - 1))\n'
    '    return [add_months(first_month, offset) for offset in range(months)]\n'
    '\n'
    '\n'
    'def ism_release_month(report_month: date) -> date:\n'
    '    """Place ISM data in the month when ISM publishes it."""\n'
    '    return add_months(report_month, 1)\n'
    '\n'
    '\n'
    'def make_requests_session() -> requests.Session:\n'
    '    """Create a browser-like HTTP session with retries for official source pages."""\n'
    '    session = requests.Session()\n'
    '    session.headers.update(\n'
    '        {\n'
    '            "User-Agent": (\n'
    '                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "\n'
    '                "AppleWebKit/537.36 (KHTML, like Gecko) "\n'
    '                "Chrome/125.0 Safari/537.36"\n'
    '            ),\n'
    '            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",\n'
    '            "Accept-Language": "en-US,en;q=0.9",\n'
    '        }\n'
    '    )\n'
    '    retries = Retry(\n'
    '        total=3,\n'
    '        connect=3,\n'
    '        read=3,\n'
    '        backoff_factor=0.7,\n'
    '        status_forcelist=(429, 500, 502, 503, 504),\n'
    '        allowed_methods=("GET", "HEAD"),\n'
    '    )\n'
    '    adapter = HTTPAdapter(max_retries=retries)\n'
    '    session.mount("https://", adapter)\n'
    '    session.mount("http://", adapter)\n'
    '    return session\n'
    '\n'
    '\n'
    'ISM_PRESS_RELEASES = {\n'
    '    (\n'
    '        "pmi",\n'
    '        "2026-06-01",\n'
    '    ): "https://www.prnewswire.com/news-releases/manufacturing-pmi-at-53-3-june-2026-ism-manufacturing-pmi-report-302814991.html",\n'
    '    (\n'
    '        "services",\n'
    '        "2026-06-01",\n'
    '    ): "https://www.prnewswire.com/news-releases/services-pmi-at-54-june-2026-ism-services-pmi-report-302817275.html",\n'
    '}\n'
    '\n'
    '\n'
    'def validate_official_url(url: str, source_name: str) -> None:\n'
    '    """Reject redirects or discovered links outside the official source domains."""\n'
    '    parsed = urlparse(url)\n'
    '    if url in ISM_PRESS_RELEASES.values():\n'
    '        return  # Exact ISM-issued releases, not general third-party news pages.\n'
    '    if parsed.scheme != "https" or parsed.hostname not in OFFICIAL_SOURCE_HOSTS:\n'
    '        raise OfficialSourceRedirectError(source_name, url)\n'
    '\n'
    '\n'
    'def is_ism_login_redirect(exc: BaseException) -> bool:\n'
    '    """Recognize ISM\'s current SSO redirect without trusting it as report data."""\n'
    '    if not isinstance(exc, OfficialSourceRedirectError):\n'
    '        return False\n'
    '    parsed = urlparse(exc.url)\n'
    '    return parsed.hostname == "ecommerce.ismworld.org" and parsed.path.lower().endswith(\n'
    '        "/sso/login.aspx"\n'
    '    )\n'
    '\n'
    '\n'
    'def fetch_url(\n'
    '    session: requests.Session,\n'
    '    url: str,\n'
    '    pause: float = 0.3,\n'
    '    minimum_bytes: int = 1,\n'
    '    request_headers: dict[str, str] | None = None,\n'
    ') -> requests.Response:\n'
    '    """Fetch one official URL and raise a clear error for failed responses."""\n'
    '    validate_official_url(url, "Official source")\n'
    '    response = session.get(url, timeout=30, headers=request_headers)\n'
    '    response.raise_for_status()\n'
    '    validate_official_url(response.url, "Official source redirect")\n'
    '    if len(response.content) < minimum_bytes:\n'
    '        raise RuntimeError(\n'
    '            f"Official source response was unexpectedly small "\n'
    '            f"({len(response.content)} bytes): {response.url}"\n'
    '        )\n'
    '    if pause:\n'
    '        sleep(pause)\n'
    '    return response\n'
    '\n'
    '\n'
    'def fresh_ism_url(url: str) -> str:\n'
    '    """Bypass stale ISM CDN pages while preserving the official URL path."""\n'
    '    parsed = urlparse(url)\n'
    '    query = dict(parse_qsl(parsed.query, keep_blank_values=True))\n'
    '    query["_ism_refresh"] = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f")\n'
    '    return parsed._replace(query=urlencode(query)).geturl()\n'
    '\n'
    '\n'
    'ISM_FRESH_HEADERS = {\n'
    '    "User-Agent": (\n'
    '        "Mozilla/5.0 (compatible; EconomicDataBot/1.0; " "+https://www.ismworld.org/)"\n'
    '    ),\n'
    '    "Cache-Control": "no-cache, no-store, max-age=0",\n'
    '    "Pragma": "no-cache",\n'
    '}\n'
    '\n'
    '\n'
    'def discover_download_url(\n'
    '    session: requests.Session,\n'
    '    page_url: str,\n'
    '    href_pattern: str,\n'
    '    link_text_pattern: str | None = None,\n'
    ') -> str | None:\n'
    '    """Find a download URL from an official source page."""\n'
    '    response = fetch_url(session, page_url, pause=0, minimum_bytes=MIN_HTML_BYTES)\n'
    '    return discover_download_url_from_html(\n'
    '        response.text,\n'
    '        page_url,\n'
    '        href_pattern,\n'
    '        link_text_pattern,\n'
    '    )\n'
    '\n'
    '\n'
    'def discover_download_url_from_html(\n'
    '    html: str,\n'
    '    page_url: str,\n'
    '    href_pattern: str,\n'
    '    link_text_pattern: str | None = None,\n'
    ') -> str | None:\n'
    '    """Find and validate a download URL in already-fetched official HTML."""\n'
    '    soup = BeautifulSoup(html, "html.parser")\n'
    '    for link in soup.find_all("a", href=True):\n'
    '        href = link["href"]\n'
    '        text = " ".join(link.get_text(" ", strip=True).split())\n'
    '        href_matches = re.search(href_pattern, href, flags=re.IGNORECASE)\n'
    '        text_matches = link_text_pattern is None or re.search(\n'
    '            link_text_pattern, text, flags=re.IGNORECASE\n'
    '        )\n'
    '        if href_matches and text_matches:\n'
    '            discovered = urljoin(page_url, href)\n'
    '            validate_official_url(discovered, "Discovered download")\n'
    '            return discovered\n'
    '    return None\n'
    '\n'
    '\n'
    'def discover_current_ism_urls(\n'
    '    session: requests.Session,\n'
    ') -> dict[str, str]:\n'
    '    """Discover current report links so minor ISM URL changes do not break the scraper."""\n'
    '    response = fetch_url(\n'
    '        session,\n'
    '        fresh_ism_url(ISM_REPORT_INDEX_URL),\n'
    '        pause=0,\n'
    '        minimum_bytes=MIN_HTML_BYTES,\n'
    '        request_headers=ISM_FRESH_HEADERS,\n'
    '    )\n'
    '    soup = BeautifulSoup(response.text, "html.parser")\n'
    '    discovered: dict[str, str] = {}\n'
    '    for link in soup.find_all("a", href=True):\n'
    '        href = urljoin(ISM_REPORT_INDEX_URL, link["href"])\n'
    '        text = " ".join(link.get_text(" ", strip=True).split()).lower()\n'
    '        if "past month" in text:\n'
    '            continue\n'
    '        heading = link.find_previous(["h2", "h3", "h4"])\n'
    '        heading_text = heading.get_text(" ", strip=True).lower() if heading else ""\n'
    '        if "manufacturing" in heading_text or re.search(\n'
    '            r"/ism-pmi-reports/pmi/", href, flags=re.IGNORECASE\n'
    '        ):\n'
    '            discovered.setdefault("pmi", href)\n'
    '        elif "services" in heading_text or re.search(\n'
    '            r"/ism-pmi-reports/services/", href, flags=re.IGNORECASE\n'
    '        ):\n'
    '            discovered.setdefault("services", href)\n'
    '    return discovered\n'
    '\n'
    '\n'
    'def fetch_first_available(\n'
    '    session: requests.Session,\n'
    '    urls: Iterable[str],\n'
    '    description: str,\n'
    ') -> requests.Response:\n'
    '    """Try multiple official URLs before reporting a source failure."""\n'
    '    errors = []\n'
    '    for url in urls:\n'
    '        try:\n'
    '            return fetch_url(\n'
    '                session,\n'
    '                url,\n'
    '                pause=0,\n'
    '                minimum_bytes=MIN_WORKBOOK_BYTES,\n'
    '            )\n'
    '        except OfficialSourceRedirectError:\n'
    '            raise\n'
    '        except Exception as exc:\n'
    '            errors.append(f"{url}: {exc}")\n'
    '    raise RuntimeError(f"Could not download {description}. Tried: {\' | \'.join(errors)}")\n'
    '\n'
    '\n'
    'def safe_filename(value: str) -> str:\n'
    '    """Convert a source or period label into a stable filesystem name."""\n'
    '    return re.sub(r"[^a-z0-9._-]+", "-", value.lower()).strip("-")\n'
    '\n'
    '\n'
    'def atomic_write_bytes(path: Path, content: bytes) -> None:\n'
    '    """Write an archive file atomically so interrupted runs cannot corrupt it."""\n'
    '    path.parent.mkdir(parents=True, exist_ok=True)\n'
    '    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")\n'
    '    temporary.write_bytes(content)\n'
    '    temporary.replace(path)\n'
    '\n'
    '\n'
    'def atomic_write_json(path: Path, payload: dict) -> None:\n'
    '    atomic_write_bytes(\n'
    '        path,\n'
    '        json.dumps(payload, indent=2, sort_keys=True).encode("utf-8"),\n'
    '    )\n'
    '\n'
    '\n'
    'def normalize_archive_values(values: dict[str, object]) -> dict[str, object]:\n'
    '    """Convert parsed values to deterministic JSON-compatible metadata."""\n'
    '    normalized = {}\n'
    '    for key, value in sorted(values.items()):\n'
    '        if isinstance(value, (date, datetime)):\n'
    '            normalized[key] = value.isoformat()\n'
    '        elif pd.isna(value):\n'
    '            normalized[key] = None\n'
    '        elif isinstance(value, (int, float, str, bool)) or value is None:\n'
    '            normalized[key] = value\n'
    '        else:\n'
    '            normalized[key] = str(value)\n'
    '    return normalized\n'
    '\n'
    '\n'
    'def archive_validated_source(\n'
    '    source: str,\n'
    '    period: str,\n'
    '    response: requests.Response,\n'
    '    extension: str,\n'
    '    schema: Iterable[str],\n'
    '    parsed_values: dict[str, object],\n'
    ') -> SourceStatus:\n'
    '    """Archive one validated official response and record immutable metadata."""\n'
    '    retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")\n'
    '    content = response.content\n'
    '    digest = hashlib.sha256(content).hexdigest()\n'
    '    schema_items = sorted(str(item) for item in schema)\n'
    '    schema_fingerprint = hashlib.sha256(\n'
    '        "\\n".join(schema_items).encode("utf-8")\n'
    '    ).hexdigest()\n'
    '    source_dir = SOURCE_ARCHIVE_ROOT / safe_filename(source)\n'
    '    period_name = safe_filename(period)\n'
    '    raw_path = source_dir / f"{period_name}-{digest[:16]}.{extension.lstrip(\'.\')}"\n'
    '    metadata_path = raw_path.with_name(\n'
    '        f"{raw_path.name}.{schema_fingerprint[:12]}.json"\n'
    '    )\n'
    '    period_pointer = source_dir / f"{period_name}-latest.json"\n'
    '    source_pointer = source_dir / "latest.json"\n'
    '    normalized_values = normalize_archive_values(parsed_values)\n'
    '\n'
    '    revision_fields: set[str] = set()\n'
    '    if period_pointer.exists():\n'
    '        try:\n'
    '            previous = json.loads(period_pointer.read_text(encoding="utf-8"))\n'
    '            previous_values = previous.get("parsed_values", {})\n'
    '            if previous_values != normalized_values:\n'
    '                revision_fields.update(\n'
    '                    key\n'
    '                    for key in set(previous_values) | set(normalized_values)\n'
    '                    if previous_values.get(key) != normalized_values.get(key)\n'
    '                )\n'
    '        except (OSError, ValueError, TypeError):\n'
    '            revision_fields.add("unreadable previous period metadata")\n'
    '\n'
    '    previous_latest = None\n'
    '    if source_pointer.exists():\n'
    '        try:\n'
    '            previous_latest = json.loads(source_pointer.read_text(encoding="utf-8"))\n'
    '            previous_values = previous_latest.get("parsed_values", {})\n'
    '            common_keys = {\n'
    '                key\n'
    '                for key in set(previous_values) & set(normalized_values)\n'
    '                if re.match(r"^\\d{4}-\\d{2}", key)\n'
    '            }\n'
    '            revision_fields.update(\n'
    '                key\n'
    '                for key in common_keys\n'
    '                if previous_values.get(key) != normalized_values.get(key)\n'
    '            )\n'
    '        except (OSError, ValueError, TypeError):\n'
    '            revision_fields.add("unreadable previous source metadata")\n'
    '\n'
    '    revision = (\n'
    '        "Changed fields: " + ", ".join(sorted(revision_fields))\n'
    '        if revision_fields\n'
    '        else ""\n'
    '    )\n'
    '    if revision:\n'
    '        logging.warning(\n'
    '            "%s revised previously archived data for %s: %s",\n'
    '            source,\n'
    '            period,\n'
    '            ", ".join(sorted(revision_fields)),\n'
    '        )\n'
    '\n'
    '    metadata = {\n'
    '        "source": source,\n'
    '        "period": period,\n'
    '        "retrieved_at": retrieved_at,\n'
    '        "requested_url": response.request.url if response.request else response.url,\n'
    '        "final_url": response.url,\n'
    '        "content_type": response.headers.get("Content-Type", ""),\n'
    '        "byte_size": len(content),\n'
    '        "sha256": digest,\n'
    '        "schema": schema_items,\n'
    '        "schema_fingerprint": schema_fingerprint,\n'
    '        "parsed_values": normalized_values,\n'
    '        "archive_file": str(raw_path),\n'
    '        "revision": revision,\n'
    '    }\n'
    '    if not raw_path.exists():\n'
    '        atomic_write_bytes(raw_path, content)\n'
    '    if not metadata_path.exists():\n'
    '        atomic_write_json(metadata_path, metadata)\n'
    '    atomic_write_json(period_pointer, metadata)\n'
    '    previous_latest_period = (\n'
    '        str(previous_latest.get("period", ""))\n'
    '        if isinstance(previous_latest, dict)\n'
    '        else ""\n'
    '    )\n'
    '    if not previous_latest_period or period >= previous_latest_period:\n'
    '        atomic_write_json(source_pointer, metadata)\n'
    '\n'
    '    status = SourceStatus(\n'
    '        source=source,\n'
    '        status="OK",\n'
    '        latest_data_month=period,\n'
    '        retrieved_at=retrieved_at,\n'
    '        source_url=response.url,\n'
    '        sha256=digest,\n'
    '        byte_size=len(content),\n'
    '        content_type=response.headers.get("Content-Type", ""),\n'
    '        schema_fingerprint=schema_fingerprint,\n'
    '        archive_file=str(raw_path),\n'
    '        revision=revision,\n'
    '    )\n'
    '    set_source_status(status)\n'
    '    return status\n'
    '\n'
    '\n'
    'def load_archived_source(source: str, period: str) -> tuple[bytes, dict] | None:\n'
    '    """Load the latest previously validated official response for one period."""\n'
    '    pointer = (\n'
    '        SOURCE_ARCHIVE_ROOT\n'
    '        / safe_filename(source)\n'
    '        / f"{safe_filename(period)}-latest.json"\n'
    '    )\n'
    '    if not pointer.exists():\n'
    '        return None\n'
    '    try:\n'
    '        metadata = json.loads(pointer.read_text(encoding="utf-8"))\n'
    '        raw_path = Path(metadata["archive_file"])\n'
    '        if not raw_path.is_file():\n'
    '            # Archive metadata can be copied with the project. Resolve the\n'
    '            # immutable payload beside its pointer when the original absolute\n'
    '            # workstation path no longer exists.\n'
    '            raw_path = pointer.parent / raw_path.name\n'
    '        content = raw_path.read_bytes()\n'
    '    except (OSError, KeyError, ValueError, TypeError):\n'
    '        return None\n'
    '    if hashlib.sha256(content).hexdigest() != metadata.get("sha256"):\n'
    '        raise RuntimeError(\n'
    '            f"Archived {source} data for {period} failed checksum validation."\n'
    '        )\n'
    '    return content, metadata\n'
    '\n'
    '\n'
    'def load_latest_archived_source(source: str) -> tuple[bytes, dict] | None:\n'
    '    """Load the latest checksum-verified archive for one official source."""\n'
    '    pointer = SOURCE_ARCHIVE_ROOT / safe_filename(source) / "latest.json"\n'
    '    if not pointer.exists():\n'
    '        return None\n'
    '    try:\n'
    '        metadata = json.loads(pointer.read_text(encoding="utf-8"))\n'
    '        period = metadata["period"]\n'
    '    except (OSError, KeyError, ValueError, TypeError):\n'
    '        return None\n'
    '    return load_archived_source(source, period)\n'
    '\n'
    '\n'
    'def response_from_archive(content: bytes, metadata: dict) -> requests.Response:\n'
    '    """Recreate a minimal response object from a checksum-verified archive."""\n'
    '    response = requests.Response()\n'
    '    response.status_code = 200\n'
    '    response._content = content\n'
    '    response.url = metadata.get("final_url", "")\n'
    '    response.headers["Content-Type"] = metadata.get("content_type", "")\n'
    '    prepared = requests.Request(\n'
    '        "GET",\n'
    '        metadata.get("requested_url") or response.url,\n'
    '    ).prepare()\n'
    '    response.request = prepared\n'
    '    return response\n'
    '\n'
    '\n'
    'def source_status_values() -> list[list[object]]:\n'
    '    """Build the Google Sheets source-status table."""\n'
    '    headers = [\n'
    '        "Source",\n'
    '        "Status",\n'
    '        "Latest data month",\n'
    '        "Retrieved at UTC",\n'
    '        "Source URL",\n'
    '        "SHA-256",\n'
    '        "Bytes",\n'
    '        "Schema fingerprint",\n'
    '        "Archive file",\n'
    '        "Revision",\n'
    '    ]\n'
    '    rows = [\n'
    '        [\n'
    '            status.source,\n'
    '            status.status,\n'
    '            status.latest_data_month,\n'
    '            status.retrieved_at,\n'
    '            status.source_url,\n'
    '            status.sha256,\n'
    '            status.byte_size,\n'
    '            status.schema_fingerprint,\n'
    '            status.archive_file,\n'
    '            status.revision,\n'
    '        ]\n'
    '        for status in sorted(SOURCE_STATUSES.values(), key=lambda item: item.source)\n'
    '    ]\n'
    '    return [headers, *rows]\n'
    '\n'
    '\n'
    'def indicator_metadata_values(\n'
    '    api_key: str,\n'
    '    indicators: list[Indicator],\n'
    ') -> list[list[object]]:\n'
    '    """Explain the source, dates, transformation, and monthly method for every column."""\n'
    '    headers = [\n'
    '        "Indicator",\n'
    '        "Source",\n'
    '        "Official source URL",\n'
    '        "Series / file",\n'
    '        "Native frequency",\n'
    '        "Native units",\n'
    '        "Output units",\n'
    '        "Transformation",\n'
    '        "Observation date meaning",\n'
    '        "Release date rule",\n'
    '        "Monthly selection",\n'
    '        "Final / preliminary policy",\n'
    '        "Point-in-time selections (row | observation | release)",\n'
    '    ]\n'
    '    rows = []\n'
    '    for indicator in indicators:\n'
    '        if indicator.source == "fred":\n'
    '            series_ids = [\n'
    '                item\n'
    '                for item in [\n'
    '                    indicator.series_id,\n'
    '                    indicator.numerator_series_id,\n'
    '                    indicator.denominator_series_id,\n'
    '                ]\n'
    '                if item\n'
    '            ]\n'
    '            metadata = (\n'
    '                get_fred_series_metadata(api_key, series_ids[0]) if series_ids else {}\n'
    '            )\n'
    '            source = "FRED / ALFRED real-time API"\n'
    '            source_url = (\n'
    '                f"https://fred.stlouisfed.org/series/{series_ids[0]}"\n'
    '                if series_ids\n'
    '                else "https://fred.stlouisfed.org/"\n'
    '            )\n'
    '            code = " / ".join(series_ids)\n'
    '            frequency = metadata.get("frequency", "")\n'
    '            native_units = metadata.get("units", "")\n'
    '            output_units = indicator.unit_label or (\n'
    '                "Percent change from year ago"\n'
    '                if indicator.units == "pc1"\n'
    '                else native_units\n'
    '            )\n'
    '            transformation = (\n'
    '                "Percent change from year ago, calculated from current FRED levels"\n'
    '                if indicator.units == "pc1"\n'
    '                else indicator.calculation or "None"\n'
    '            )\n'
    '            observation_meaning = "FRED observation date"\n'
    '            release_rule = "ALFRED realtime_start (initial public vintage date)"\n'
    '            final_policy = (\n'
    '                "Current FRED value matched to its observation date; "\n'
    '                "initial ALFRED publication date retained as metadata"\n'
    '            )\n'
    '        elif indicator.source == "ism":\n'
    '            source = "Institute for Supply Management"\n'
    '            source_url = ISM_REPORT_INDEX_URL\n'
    '            code = indicator.source_column or ""\n'
    '            frequency = "Monthly"\n'
    '            native_units = "Diffusion index"\n'
    '            output_units = "Index"\n'
    '            transformation = "None"\n'
    '            observation_meaning = "Survey reference month"\n'
    '            release_rule = (\n'
    '                "Official page publication metadata; conservative 7th/10th "\n'
    '                "of following month fallback"\n'
    '            )\n'
    '            final_policy = "Official monthly report"\n'
    '        elif indicator.source == "nyfed_sce":\n'
    '            source = "Federal Reserve Bank of New York SCE"\n'
    '            source_url = NYFED_SCE_PAGE_URL\n'
    '            code = "frbny-sce-data.xlsx / one-year median"\n'
    '            frequency = "Monthly"\n'
    '            native_units = "Percent"\n'
    '            output_units = "Percent"\n'
    '            transformation = "None; median one-year-ahead expectation"\n'
    '            observation_meaning = "Survey month"\n'
    '            release_rule = indicator.release_rule\n'
    '            final_policy = "Official monthly SCE workbook"\n'
    '        elif indicator.source == "imf_datamapper":\n'
    '            source = "IMF DataMapper"\n'
    '            source_url = IMF_DATAMAPPER_API_BASE\n'
    '            code = f"{indicator.series_id or \'\'} / {indicator.source_column or DEFAULT_IMF_COUNTRY}"\n'
    '            frequency = "Annual"\n'
    '            native_units = "% of GDP"\n'
    '            output_units = indicator.unit_label or "% of GDP"\n'
    '            transformation = "None; raw IMF FPP annual value"\n'
    '            observation_meaning = "Annual fiscal year-end observation"\n'
    '            release_rule = indicator.release_rule\n'
    '            final_policy = "Latest available IMF DataMapper FPP observations"\n'
    '        else:\n'
    '            source = "University of Michigan Surveys of Consumers"\n'
    '            source_url = MICHIGAN_CHARTS_PAGE_URL\n'
    '            code = (\n'
    '                "chicsr.xls / ICS_M"\n'
    '                if indicator.source == "michigan_sentiment"\n'
    '                else "chicer.xls / ICE_M"\n'
    '            )\n'
    '            frequency = "Monthly"\n'
    '            native_units = "Index"\n'
    '            output_units = "Index"\n'
    '            transformation = "None"\n'
    '            observation_meaning = "Survey month"\n'
    '            release_rule = indicator.release_rule\n'
    '            final_policy = indicator.release_rule\n'
    '\n'
    '        rows.append(\n'
    '            [\n'
    '                indicator.name,\n'
    '                source,\n'
    '                source_url,\n'
    '                code,\n'
    '                frequency,\n'
    '                native_units,\n'
    '                output_units,\n'
    '                transformation,\n'
    '                observation_meaning,\n'
    '                release_rule,\n'
    '                indicator.monthly_method,\n'
    '                final_policy,\n'
    '                "; ".join(\n'
    '                    (\n'
    '                        (\n'
    '                            f"{row_date.isoformat()} | "\n'
    '                            f"{selection[\'observation_date\'].isoformat()} | "\n'
    '                            f"{selection[\'release_date\'].isoformat()}"\n'
    '                        )\n'
    '                        if selection\n'
    '                        else f"{row_date.isoformat()} | missing"\n'
    '                    )\n'
    '                    for row_date in [\n'
    '                        month_end(item)\n'
    '                        for item in recent_month_starts(CURRENT_OUTPUT_MONTHS)\n'
    '                    ]\n'
    '                    for selection in [LAST_SELECTIONS.get((indicator.name, row_date))]\n'
    '                ),\n'
    '            ]\n'
    '        )\n'
    '    return [headers, *rows]\n'
    '\n'
    '\n'
    'def decimal_places(value: float) -> int:\n'
    '    token = format(float(value), ".12f").rstrip("0").rstrip(".")\n'
    '    return len(token.split(".", 1)[1]) if "." in token else 0\n'
    '\n'
    '\n'
    'def build_validation_report(\n'
    '    indicators: list[Indicator],\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    months: int,\n'
    ') -> list[list[object]]:\n'
    '    """Validate source records and observation-period month-end selections."""\n'
    '    messages = list(VALIDATION_MESSAGES)\n'
    '    data_by_name = dict(workbook_data)\n'
    '    row_dates = [month_end(item) for item in recent_month_starts(months)]\n'
    '    ranges = {\n'
    '        "Manufacturing PMI": (0, 100),\n'
    '        "Manufacturing new orders": (0, 100),\n'
    '        "Services PMI": (0, 100),\n'
    '        "Services new orders": (0, 100),\n'
    '        "Services business activity": (0, 100),\n'
    '        "Consumer sentiment": (0, 200),\n'
    '        "Consumer expectations": (0, 200),\n'
    '        "Consumer inflation expectations": (-5, 25),\n'
    '        "Industrial production growth": (50, 150),\n'
    '        "Real consumer spending growth": (10_000, 30_000),\n'
    '        "Employment growth": (100_000, 200_000),\n'
    '        "Core consumption inflation": (50, 250),\n'
    '        "Core CPI inflation": (50, 500),\n'
    '        "Headline CPI inflation": (50, 500),\n'
    '        "Core producer price inflation": (50, 300),\n'
    '        "Headline producer price inflation": (50, 300),\n'
    '        "Wage growth": (50, 300),\n'
    '        "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level": (\n'
    '            1_000_000,\n'
    '            15_000_000,\n'
    '        ),\n'
    '        "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level": (\n'
    '            0,\n'
    '            3_000_000,\n'
    '        ),\n'
    '        "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations": (\n'
    '            0,\n'
    '            5_000,\n'
    '        ),\n'
    '        "Banking system reserves": (1_000_000, 6_000_000),\n'
    '        "Broad money growth": (10_000, 30_000),\n'
    '        "Real government spending growth": (1_000, 10_000),\n'
    '        "Government revenue growth": (0, 2_000_000),\n'
    '        "Government revenue % GDP": (0, 100),\n'
    '        "Interest paid on public debt % GDP": (0, 50),\n'
    '        "Government interest coverage ratio": (0, 100),\n'
    '        "Bank credit growth": (10_000, 30_000),\n'
    '        "Corporate credit spread": (0, 20),\n'
    '    }\n'
    '    intentional_level_columns = {\n'
    '        "Industrial production growth",\n'
    '        "Real consumer spending growth",\n'
    '        "Employment growth",\n'
    '        "Wage growth",\n'
    '        "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level",\n'
    '        "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level",\n'
    '        "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations",\n'
    '        "Broad money growth",\n'
    '        "Real government spending growth",\n'
    '        "Government revenue growth",\n'
    '        "Bank credit growth",\n'
    '    }\n'
    '    intentional_index_columns = {\n'
    '        "Core consumption inflation",\n'
    '        "Core CPI inflation",\n'
    '        "Headline CPI inflation",\n'
    '        "Core producer price inflation",\n'
    '        "Headline producer price inflation",\n'
    '    }\n'
    '    expected_sources = {\n'
    '        "fred": "FRED / ALFRED",\n'
    '        "ism": "Institute for Supply Management",\n'
    '        "nyfed_sce": "Federal Reserve Bank of New York SCE",\n'
    '        "michigan_sca": "University of Michigan Surveys of Consumers",\n'
    '        "michigan_sentiment": "University of Michigan Surveys of Consumers",\n'
    '        "imf_datamapper": "IMF DataMapper FPP",\n'
    '    }\n'
    '\n'
    '    for indicator in indicators:\n'
    '        records = data_by_name.get(indicator.name, [])\n'
    '        seen = set()\n'
    '        for record in records:\n'
    '            if not isinstance(record.get("observation_date"), date):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": "Candidate record is missing a valid observation date.",\n'
    '                    }\n'
    '                )\n'
    '            if not isinstance(record.get("release_date"), date):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": "Candidate record is missing a valid release date.",\n'
    '                    }\n'
    '                )\n'
    '            if record.get("source") != expected_sources.get(indicator.source):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Unexpected source identity {record.get(\'source\')!r}; "\n'
    '                            f"expected {expected_sources.get(indicator.source)!r}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            if indicator.source == "fred" and not isinstance(\n'
    '                record.get("realtime_start"),\n'
    '                date,\n'
    '            ):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": "FRED record is missing ALFRED realtime_start.",\n'
    '                    }\n'
    '                )\n'
    '            elif indicator.source == "fred" and record.get(\n'
    '                "realtime_start"\n'
    '            ) != record.get("release_date"):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            "FRED release_date does not match its initial "\n'
    '                            "ALFRED realtime_start."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            if indicator.source == "fred" and not str(\n'
    '                record.get("value_vintage", "")\n'
    '            ).startswith("Current FRED"):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            "FRED value is not marked as the current API value "\n'
    '                            "matched to its observation date."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            key = (\n'
    '                record.get("observation_date"),\n'
    '                record.get("release_date"),\n'
    '                record.get("selection_date"),\n'
    '            )\n'
    '            if key in seen:\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": f"Duplicate observation/release record: {key}",\n'
    '                    }\n'
    '                )\n'
    '            seen.add(key)\n'
    '\n'
    '        if (\n'
    '            "growth" in indicator.name.lower()\n'
    '            and indicator.units != "pc1"\n'
    '            and indicator.name not in intentional_level_columns\n'
    '        ):\n'
    '            messages.append(\n'
    '                {\n'
    '                    "severity": "ERROR",\n'
    '                    "indicator": indicator.name,\n'
    '                    "message": "Column is labelled growth but transformation is not pc1.",\n'
    '                }\n'
    '            )\n'
    '        if indicator.name in intentional_level_columns:\n'
    '            messages.append(\n'
    '                {\n'
    '                    "severity": "WARNING",\n'
    '                    "indicator": indicator.name,\n'
    '                    "message": (\n'
    '                        "Legacy column name is intentionally preserved, but values "\n'
    '                        "are raw FRED levels rather than growth rates."\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '        if indicator.name in intentional_index_columns:\n'
    '            messages.append(\n'
    '                {\n'
    '                    "severity": "WARNING",\n'
    '                    "indicator": indicator.name,\n'
    '                    "message": (\n'
    '                        "Legacy inflation column name is intentionally preserved, "\n'
    '                        "but the value is the raw FRED price index."\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '\n'
    '        for row_date in row_dates:\n'
    '            selected = latest_available_record(records, row_date)\n'
    '            if selected is None:\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "WARNING",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"No source observation belongs to {row_date:%B %Y}; "\n'
    '                            "output is blank."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '                continue\n'
    '            if selected["observation_date"] > row_date:\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Observation date {selected[\'observation_date\']} is after "\n'
    '                            f"row date {row_date}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            if (\n'
    '                selected["release_date"] > row_date\n'
    '                and selected.get("period_aligned") is not True\n'
    '            ):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Release date {selected[\'release_date\']} is after "\n'
    '                            f"row date {row_date}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            if (\n'
    '                indicator.source == "fred"\n'
    '                and selected["realtime_start"] > row_date\n'
    '                and selected.get("period_aligned") is not True\n'
    '            ):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"ALFRED realtime_start {selected[\'realtime_start\']} is "\n'
    '                            f"after row date {row_date}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            elif selected["release_date"] > row_date:\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "WARNING",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Observation-period dating places this value at "\n'
    '                            f"{row_date}; it was published on "\n'
    '                            f"{selected[\'release_date\']}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            lower_upper = ranges.get(indicator.name)\n'
    '            if (\n'
    '                lower_upper\n'
    '                and not lower_upper[0] <= selected["value"] <= lower_upper[1]\n'
    '            ):\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "ERROR",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Impossible value {selected[\'value\']} at {row_date}; "\n'
    '                            f"expected {lower_upper}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '            if decimal_places(selected["value"]) > 6:\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "WARNING",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Suspicious precision ({decimal_places(selected[\'value\'])} "\n'
    '                            f"decimals) at {row_date}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '\n'
    '        latest_row_date = row_dates[-1]\n'
    '        latest_selected = latest_available_record(records, latest_row_date)\n'
    '        if latest_selected is not None:\n'
    '            metadata = _FRED_METADATA_CACHE.get(indicator.series_id or "", {})\n'
    '            frequency = metadata.get("frequency", "Monthly")\n'
    '            stale_days = (\n'
    '                45\n'
    '                if frequency.startswith(("Daily", "Weekly"))\n'
    '                else 100 if frequency.startswith(("Monthly", "Quarterly")) else 400\n'
    '            )\n'
    '            age_days = (latest_row_date - latest_selected["release_date"]).days\n'
    '            if age_days > stale_days:\n'
    '                messages.append(\n'
    '                    {\n'
    '                        "severity": "WARNING",\n'
    '                        "indicator": indicator.name,\n'
    '                        "message": (\n'
    '                            f"Selected source data is stale by {age_days} days "\n'
    '                            f"for frequency {frequency}."\n'
    '                        ),\n'
    '                    }\n'
    '                )\n'
    '\n'
    '    headers = ["Severity", "Indicator", "Message"]\n'
    '    if not messages:\n'
    '        return [headers, ["OK", "All indicators", "No validation warnings or errors."]]\n'
    '    rows = [[item["severity"], item["indicator"], item["message"]] for item in messages]\n'
    '    return [headers, *rows]\n'
    '\n'
    '\n'
    'def selection_diagnostic_values(\n'
    '    indicators: list[Indicator],\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    months: int,\n'
    ') -> list[list[object]]:\n'
    '    """Explain every observation-period selection for the latest three row dates."""\n'
    '    headers = [\n'
    '        "Row date",\n'
    '        "Indicator",\n'
    '        "Observation date",\n'
    '        "Release date",\n'
    '        "Realtime start",\n'
    '        "Selected value",\n'
    '        "Source",\n'
    '        "Source series / field",\n'
    '        "Reason selected",\n'
    '    ]\n'
    '    data_by_name = dict(workbook_data)\n'
    '    row_dates = [month_end(item) for item in recent_month_starts(months)][-3:]\n'
    '    rows = []\n'
    '    for row_date in row_dates:\n'
    '        for indicator in indicators:\n'
    '            selected = latest_available_record(\n'
    '                data_by_name.get(indicator.name, []),\n'
    '                row_date,\n'
    '            )\n'
    '            if selected is None:\n'
    '                rows.append(\n'
    '                    [\n'
    '                        format_output_month(row_date),\n'
    '                        indicator.name,\n'
    '                        "",\n'
    '                        "",\n'
    '                        "",\n'
    '                        "",\n'
    '                        "",\n'
    '                        "",\n'
    '                        "Blank: no observation belongs to the row month.",\n'
    '                    ]\n'
    '                )\n'
    '                continue\n'
    '            rows.append(\n'
    '                [\n'
    '                    format_output_month(row_date),\n'
    '                    indicator.name,\n'
    '                    selected["observation_date"].isoformat(),\n'
    '                    selected["release_date"].isoformat(),\n'
    '                    (\n'
    '                        selected["realtime_start"].isoformat()\n'
    '                        if isinstance(selected.get("realtime_start"), date)\n'
    '                        else ""\n'
    '                    ),\n'
    '                    selected["value"],\n'
    '                    selected.get("source", ""),\n'
    '                    selected.get("source_series_id", ""),\n'
    '                    (\n'
    '                        f"Selected as the average of {selected[\'monthly_average_count\']} "\n'
    '                        "source observations assigned to the row month."\n'
    '                        if selected.get("monthly_average_count")\n'
    '                        else "Selected as the most recent source observation "\n'
    '                        "assigned to the row month."\n'
    '                    ),\n'
    '                ]\n'
    '            )\n'
    '    return [headers, *rows]\n'
    '\n'
    '\n'
    'def save_clean_csv_outputs(\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    metadata_values: list[list[object]],\n'
    '    validation_values: list[list[object]],\n'
    '    diagnostic_values: list[list[object]],\n'
    ') -> None:\n'
    '    """Write the same three clean tables locally for inspection and downstream use."""\n'
    '    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n'
    '    table_values = sheet_values_for_google(workbook_data)\n'
    '    pd.DataFrame(table_values[1:], columns=table_values[0]).to_csv(\n'
    '        OUTPUT_ROOT / "monthly_indicators.csv",\n'
    '        index=False,\n'
    '    )\n'
    '    pd.DataFrame(metadata_values[1:], columns=metadata_values[0]).to_csv(\n'
    '        OUTPUT_ROOT / "indicator_metadata.csv",\n'
    '        index=False,\n'
    '    )\n'
    '    pd.DataFrame(validation_values[1:], columns=validation_values[0]).to_csv(\n'
    '        OUTPUT_ROOT / "validation_report.csv",\n'
    '        index=False,\n'
    '    )\n'
    '    pd.DataFrame(diagnostic_values[1:], columns=diagnostic_values[0]).to_csv(\n'
    '        OUTPUT_ROOT / "selection_diagnostics.csv",\n'
    '        index=False,\n'
    '    )\n'
    '\n'
    '\n'
    'def print_changed_cells(\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    ') -> None:\n'
    '    """Print every cell that differs from the preceding local export."""\n'
    '    output_path = OUTPUT_ROOT / "monthly_indicators.csv"\n'
    '    if not output_path.exists():\n'
    '        return\n'
    '    try:\n'
    '        previous = pd.read_csv(output_path)\n'
    '    except Exception:\n'
    '        logging.warning("Could not read the preceding CSV for before/after comparison.")\n'
    '        return\n'
    '\n'
    '    if "Date" not in previous.columns:\n'
    '        return\n'
    '\n'
    '    current_values = sheet_values_for_google(workbook_data)\n'
    '    current = pd.DataFrame(current_values[1:], columns=current_values[0])\n'
    '    common_columns = [\n'
    '        column\n'
    '        for column in current.columns\n'
    '        if column != "Date" and column in previous.columns\n'
    '    ]\n'
    '    comparison = previous[["Date", *common_columns]].merge(\n'
    '        current[["Date", *common_columns]],\n'
    '        on="Date",\n'
    '        how="inner",\n'
    '        suffixes=(" before", " after"),\n'
    '    )\n'
    '    if comparison.empty:\n'
    '        return\n'
    '    changes = []\n'
    '    for _, row in comparison.iterrows():\n'
    '        for column in common_columns:\n'
    '            before = row[f"{column} before"]\n'
    '            after = row[f"{column} after"]\n'
    '            both_missing = pd.isna(before) and pd.isna(after)\n'
    '            one_missing = pd.isna(before) != pd.isna(after)\n'
    '            if both_missing or (\n'
    '                not one_missing and google_values_equivalent(before, after)\n'
    '            ):\n'
    '                continue\n'
    '            changes.append(\n'
    '                {\n'
    '                    "Date": row["Date"],\n'
    '                    "Indicator": column,\n'
    '                    "Before": "" if pd.isna(before) else before,\n'
    '                    "After": "" if pd.isna(after) else after,\n'
    '                }\n'
    '            )\n'
    '    if COMPACT_TERMINAL:\n'
    '        print(f"Changes detected: {len(changes)} cells.")\n'
    '    elif changes:\n'
    '        print_console_table(\n'
    '            "Changed cells:",\n'
    '            ["Date", "Indicator", "Before", "After"],\n'
    '            (\n'
    '                [item["Date"], item["Indicator"], item["Before"], item["After"]]\n'
    '                for item in changes\n'
    '            ),\n'
    '        )\n'
    '    else:\n'
    '        print("\\nChanged cells: none.")\n'
    '\n'
    '\n'
    'def console_cell(value: object) -> str:\n'
    '    """Return a consistent display value for command-line tables."""\n'
    '    return "" if value is None else str(value)\n'
    '\n'
    '\n'
    'def print_console_table(\n'
    '    title: str | None,\n'
    '    headers: list[str],\n'
    '    rows: Iterable[Iterable[object]],\n'
    ') -> None:\n'
    '    """Print an aligned text table without changing the underlying CSV output."""\n'
    '    display_rows = [\n'
    '        [console_cell(value) for value in row][: len(headers)] for row in rows\n'
    '    ]\n'
    '    display_rows = [row + [""] * (len(headers) - len(row)) for row in display_rows]\n'
    '    widths = [\n'
    '        (\n'
    '            max(len(header), *(len(row[index]) for row in display_rows))\n'
    '            if display_rows\n'
    '            else len(header)\n'
    '        )\n'
    '        for index, header in enumerate(headers)\n'
    '    ]\n'
    '    if title:\n'
    '        print(f"\\n{title}")\n'
    '    print(\n'
    '        " | ".join(header.ljust(widths[index]) for index, header in enumerate(headers))\n'
    '    )\n'
    '    print("-+-".join("-" * width for width in widths))\n'
    '    for row in display_rows:\n'
    '        print(\n'
    '            " | ".join(row[index].ljust(widths[index]) for index in range(len(headers)))\n'
    '        )\n'
    '\n'
    '\n'
    'def validate_excel_signature(\n'
    '    content: bytes, source_name: str, legacy_xls: bool = False\n'
    ') -> None:\n'
    '    """Use file metadata bytes to reject HTML error pages disguised as spreadsheets."""\n'
    '    expected = b"\\xd0\\xcf\\x11\\xe0" if legacy_xls else b"PK"\n'
    '    if not content.startswith(expected):\n'
    '        file_type = "XLS" if legacy_xls else "XLSX"\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} download is no longer a valid {file_type} workbook; "\n'
    '            "the source URL or website format may have changed."\n'
    '        )\n'
    '\n'
    '\n'
    'def validate_response_content_type(\n'
    '    response: requests.Response,\n'
    '    source_name: str,\n'
    '    allowed_fragments: Iterable[str],\n'
    ') -> None:\n'
    '    """Reject explicit content types that do not match the expected source format."""\n'
    '    content_type = response.headers.get("Content-Type", "").lower()\n'
    '    if content_type and not any(\n'
    '        fragment in content_type for fragment in allowed_fragments\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} returned unexpected Content-Type \'{content_type}\'."\n'
    '        )\n'
    '\n'
    '\n'
    'def text_from_html(html: str) -> str:\n'
    '    """Convert HTML into normalized text while preserving report line breaks."""\n'
    '    soup = BeautifulSoup(html, "html.parser")\n'
    '    for tag in soup(["script", "style", "noscript"]):\n'
    '        tag.decompose()\n'
    '    text = soup.get_text("\\n", strip=True)\n'
    '    text = text.replace("\\xa0", " ")\n'
    '    text = re.sub(r"[ \\t]+", " ", text)\n'
    '    text = re.sub(r"\\n{2,}", "\\n", text)\n'
    '    return text\n'
    '\n'
    '\n'
    'def parse_html_publication_date(html: str) -> date | None:\n'
    '    """Read common HTML metadata and JSON-LD publication-date fields."""\n'
    '    soup = BeautifulSoup(html, "html.parser")\n'
    '    candidates = []\n'
    '    for tag in soup.find_all("meta"):\n'
    '        key = (\n'
    '            tag.get("property") or tag.get("name") or tag.get("itemprop") or ""\n'
    '        ).lower()\n'
    '        if key in {\n'
    '            "article:published_time",\n'
    '            "date",\n'
    '            "datepublished",\n'
    '            "publish_date",\n'
    '            "publication_date",\n'
    '        }:\n'
    '            candidates.append(tag.get("content"))\n'
    '    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):\n'
    '        try:\n'
    '            payload = json.loads(script.string or "")\n'
    '        except (TypeError, json.JSONDecodeError):\n'
    '            continue\n'
    '        stack = payload if isinstance(payload, list) else [payload]\n'
    '        while stack:\n'
    '            item = stack.pop()\n'
    '            if isinstance(item, dict):\n'
    '                candidates.extend(\n'
    '                    item.get(key)\n'
    '                    for key in ("datePublished", "dateCreated")\n'
    '                    if item.get(key)\n'
    '                )\n'
    '                stack.extend(item.values())\n'
    '            elif isinstance(item, list):\n'
    '                stack.extend(item)\n'
    '    for candidate in candidates:\n'
    '        parsed = pd.to_datetime(candidate, errors="coerce", utc=True)\n'
    '        if not pd.isna(parsed):\n'
    '            return parsed.date()\n'
    '    return None\n'
    '\n'
    '\n'
    'def canonical_month(raw: str) -> str:\n'
    '    """Normalize full or abbreviated month names to lowercase full names."""\n'
    '    token = raw.strip().lower().rstrip(".")\n'
    '    return MONTH_ALIASES.get(token[:4], MONTH_ALIASES.get(token[:3], token))\n'
    '\n'
    '\n'
    'def parse_ism_report_date(text: str, expected_kind: str) -> date | None:\n'
    '    """Extract the month and year from an ISM report body."""\n'
    '    month_pattern = (\n'
    '        r"January|February|March|April|May|June|July|August|September|"\n'
    '        r"October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec"\n'
    '    )\n'
    '    patterns = [\n'
    '        rf"\\b({month_pattern})\\s+(20\\d{{2}})\\s+ISM\\S*\\s+{expected_kind}\\s+PMI",\n'
    '        rf"\\b({month_pattern})\\s+(20\\d{{2}})\\s+{expected_kind}\\s+Index",\n'
    '        rf"\\b{expected_kind}\\s+PMI\\S*\\s+Report.*?\\b({month_pattern})\\s+(20\\d{{2}})\\b",\n'
    '    ]\n'
    '    for pattern in patterns:\n'
    '        match = re.search(pattern, text, flags=re.IGNORECASE)\n'
    '        if not match:\n'
    '            continue\n'
    '        month_name = canonical_month(match.group(1))\n'
    '        if month_name in MONTH_NUMBERS:\n'
    '            return date(int(match.group(2)), MONTH_NUMBERS[month_name], 1)\n'
    '    return None\n'
    '\n'
    '\n'
    'def parse_ism_report_date_from_html(html: str, expected_kind: str) -> date | None:\n'
    '    """Prefer the visible report heading over stale metadata or comparison text."""\n'
    '    soup = BeautifulSoup(html, "html.parser")\n'
    '    heading_text = "\\n".join(\n'
    '        heading.get_text(" ", strip=True)\n'
    '        for heading in soup.find_all(["h1", "h2", "h3"])\n'
    '    )\n'
    '    return parse_ism_report_date(heading_text, expected_kind) or parse_ism_report_date(\n'
    '        text_from_html(html),\n'
    '        expected_kind,\n'
    '    )\n'
    '\n'
    '\n'
    'def first_number(patterns: Iterable[str], text: str) -> float | None:\n'
    '    """Return the first numeric value matched by any regex pattern."""\n'
    '    for pattern in patterns:\n'
    '        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE)\n'
    '        if match:\n'
    '            return float(match.group(1))\n'
    '    return None\n'
    '\n'
    '\n'
    'def checked_value(\n'
    '    source_name: str,\n'
    '    field_name: str,\n'
    '    primary: float | None,\n'
    '    corroborating: float | None,\n'
    '    tolerance: float = 0.11,\n'
    ') -> float | None:\n'
    '    """Use one value while rejecting disagreements between official page sections."""\n'
    '    if (\n'
    '        primary is not None\n'
    '        and corroborating is not None\n'
    '        and abs(primary - corroborating) > tolerance\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} contains conflicting {field_name} values: "\n'
    '            f"{primary} versus {corroborating}."\n'
    '        )\n'
    '    return primary if primary is not None else corroborating\n'
    '\n'
    '\n'
    'def first_table_value(labels: Iterable[str], text: str) -> float | None:\n'
    '    """Read the first numeric value after an ISM table label."""\n'
    '    for label in labels:\n'
    '        escaped = re.escape(label)\n'
    '        value = first_number(\n'
    '            [\n'
    '                rf"^{escaped}\\s*(?:®)?\\s+(\\d{{1,3}}(?:\\.\\d+)?)\\b",\n'
    '                rf"\\n{escaped}\\s*(?:®)?\\s+(\\d{{1,3}}(?:\\.\\d+)?)\\b",\n'
    '            ],\n'
    '            text,\n'
    '        )\n'
    '        if value is not None:\n'
    '            return value\n'
    '    return None\n'
    '\n'
    '\n'
    'def html_table_row_numbers(html: str, labels: Iterable[str]) -> list[float] | None:\n'
    '    """Read numeric cells from a real HTML table row identified by its first cell."""\n'
    '    soup = BeautifulSoup(html, "html.parser")\n'
    '    normalized_labels = {re.sub(r"\\s+", " ", label).strip().lower() for label in labels}\n'
    '    for row in soup.find_all("tr"):\n'
    '        cells = [\n'
    '            re.sub(r"\\s+", " ", cell.get_text(" ", strip=True)).strip()\n'
    '            for cell in row.find_all(["th", "td"])\n'
    '        ]\n'
    '        if not cells:\n'
    '            continue\n'
    '        row_label = cells[0].replace("®", "").strip().lower()\n'
    '        if not any(\n'
    '            row_label == label or row_label.startswith(label)\n'
    '            for label in normalized_labels\n'
    '        ):\n'
    '            continue\n'
    '        numbers = []\n'
    '        for cell in cells[1:]:\n'
    '            match = re.fullmatch(r"\\s*([+-]?\\d{1,3}(?:\\.\\d+)?)\\s*%?\\s*", cell)\n'
    '            if match:\n'
    '                numbers.append(float(match.group(1)))\n'
    '        if numbers:\n'
    '            return numbers\n'
    '    return None\n'
    '\n'
    '\n'
    'def current_html_table_value(html: str, labels: Iterable[str]) -> float | None:\n'
    '    """Return the first numeric value from a matching HTML table row."""\n'
    '    numbers = html_table_row_numbers(html, labels)\n'
    '    return numbers[0] if numbers else None\n'
    '\n'
    '\n'
    'def manufacturing_value_from_services_comparison(\n'
    '    label: str,\n'
    '    text: str,\n'
    '    html: str | None = None,\n'
    ') -> float | None:\n'
    '    """Read Manufacturing\'s current value from the Services comparison table."""\n'
    '    if html:\n'
    '        numbers = html_table_row_numbers(html, [label])\n'
    '        # Services comparison rows end with Manufacturing current, prior, change.\n'
    '        if numbers and len(numbers) >= 6:\n'
    '            return numbers[-3]\n'
    '\n'
    '    escaped = re.escape(label)\n'
    '    match = re.search(\n'
    '        rf"^{escaped}\\s*(?:®)?\\s+([^\\n]+)$",\n'
    '        text,\n'
    '        flags=re.IGNORECASE | re.MULTILINE,\n'
    '    )\n'
    '    if not match:\n'
    '        return None\n'
    '    numbers = re.findall(r"[+-]?\\d+(?:\\.\\d+)?", match.group(1))\n'
    '    # The final three values are Manufacturing current, previous, and change.\n'
    '    if len(numbers) < 3:\n'
    '        return None\n'
    '    return float(numbers[-3])\n'
    '\n'
    '\n'
    'def parse_ism_manufacturing_report(html: str, source_url: str) -> IsmReportData | None:\n'
    '    """Parse Manufacturing PMI and New Orders from an official ISM page."""\n'
    '    text = text_from_html(html)\n'
    '    if "content you are looking for is no longer available" in text.lower():\n'
    '        return None\n'
    '\n'
    '    report_date = parse_ism_report_date_from_html(html, "Manufacturing")\n'
    '    if not report_date:\n'
    '        raise RuntimeError("ISM Manufacturing report date could not be identified.")\n'
    '\n'
    '    manufacturing_pmi = checked_value(\n'
    '        "ISM Manufacturing report",\n'
    '        "Manufacturing PMI",\n'
    '        current_html_table_value(html, ["Manufacturing PMI"])\n'
    '        or first_table_value(["Manufacturing PMI"], text),\n'
    '        first_number(\n'
    '            [\n'
    '                r"Manufacturing PMI\\S*\\s+at\\s+(\\d{1,3}(?:\\.\\d+)?)\\s*%",\n'
    '                r"Manufacturing PMI\\S*\\s+registered\\s+(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '                r"Manufacturing PMI\\S*[^.\\n]{0,160}?\\breading\\s+(?:of\\s+)?"\n'
    '                r"\\(?(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '            ],\n'
    '            text,\n'
    '        ),\n'
    '    )\n'
    '\n'
    '    manufacturing_new_orders = checked_value(\n'
    '        "ISM Manufacturing report",\n'
    '        "Manufacturing New Orders",\n'
    '        current_html_table_value(html, ["New Orders"])\n'
    '        or first_table_value(["New Orders"], text),\n'
    '        first_number(\n'
    '            [\n'
    '                r"New Orders Index[^.\\n]{0,160}?\\bregister(?:ed|ing)\\s+"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '                r"New Orders Index[^.\\n]{0,160}?\\breading\\s+(?:of\\s+)?"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '            ],\n'
    '            text,\n'
    '        ),\n'
    '    )\n'
    '\n'
    '    return IsmReportData(\n'
    '        report_date,\n'
    '        source_url,\n'
    '        {\n'
    '            "manufacturing_pmi": manufacturing_pmi,\n'
    '            "manufacturing_new_orders": manufacturing_new_orders,\n'
    '        },\n'
    '        parse_html_publication_date(html),\n'
    '    )\n'
    '\n'
    '\n'
    'def parse_ism_services_report(html: str, source_url: str) -> IsmReportData | None:\n'
    '    """Parse Services PMI, New Orders, and Business Activity from ISM."""\n'
    '    text = text_from_html(html)\n'
    '    if "content you are looking for is no longer available" in text.lower():\n'
    '        return None\n'
    '\n'
    '    report_date = parse_ism_report_date_from_html(html, "Services")\n'
    '    if not report_date:\n'
    '        raise RuntimeError("ISM Services report date could not be identified.")\n'
    '\n'
    '    services_pmi = checked_value(\n'
    '        "ISM Services report",\n'
    '        "Services PMI",\n'
    '        current_html_table_value(html, ["Services PMI"])\n'
    '        or first_table_value(["Services PMI"], text),\n'
    '        first_number(\n'
    '            [\n'
    '                r"Services PMI\\S*\\s+at\\s+(\\d{1,3}(?:\\.\\d+)?)\\s*%",\n'
    '                r"Services PMI\\S*\\s+registered\\s+(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '                r"Services PMI\\S*[^.\\n]{0,160}?\\breading\\s+(?:of\\s+)?"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '            ],\n'
    '            text,\n'
    '        ),\n'
    '    )\n'
    '\n'
    '    services_new_orders = checked_value(\n'
    '        "ISM Services report",\n'
    '        "Services New Orders",\n'
    '        current_html_table_value(html, ["New Orders"])\n'
    '        or first_table_value(["New Orders"], text),\n'
    '        first_number(\n'
    '            [\n'
    '                r"New Orders Index\\s+at\\s+(\\d{1,3}(?:\\.\\d+)?)\\s*%",\n'
    '                r"New Orders Index[^.\\n]{0,160}?\\bregister(?:ed|ing)\\s+"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '                r"New Orders Index[^.\\n]{0,160}?\\breading\\s+(?:of\\s+)?"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '            ],\n'
    '            text,\n'
    '        ),\n'
    '    )\n'
    '\n'
    '    services_business_activity = checked_value(\n'
    '        "ISM Services report",\n'
    '        "Services Business Activity",\n'
    '        current_html_table_value(\n'
    '            html,\n'
    '            ["Business Activity/Production", "Business Activity"],\n'
    '        )\n'
    '        or first_table_value(["Business Activity/Production"], text),\n'
    '        first_number(\n'
    '            [\n'
    '                r"Business Activity Index\\s+at\\s+(\\d{1,3}(?:\\.\\d+)?)\\s*%",\n'
    '                r"Business Activity Index[^.\\n]{0,200}?\\bto\\s+"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '                r"Business Activity Index[^.\\n]{0,160}?\\bregistered\\s+"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '                r"Business Activity Index[^.\\n]{0,160}?\\breading\\s+(?:of\\s+)?"\n'
    '                r"(\\d{1,3}(?:\\.\\d+)?)\\s*percent",\n'
    '            ],\n'
    '            text,\n'
    '        ),\n'
    '    )\n'
    '\n'
    '    manufacturing_pmi = manufacturing_value_from_services_comparison(\n'
    '        "Services PMI",\n'
    '        text,\n'
    '        html,\n'
    '    )\n'
    '    manufacturing_new_orders = manufacturing_value_from_services_comparison(\n'
    '        "New Orders",\n'
    '        text,\n'
    '        html,\n'
    '    )\n'
    '\n'
    '    return IsmReportData(\n'
    '        report_date,\n'
    '        source_url,\n'
    '        {\n'
    '            "services_pmi": services_pmi,\n'
    '            "services_new_orders": services_new_orders,\n'
    '            "services_business_activity": services_business_activity,\n'
    '            "manufacturing_pmi": manufacturing_pmi,\n'
    '            "manufacturing_new_orders": manufacturing_new_orders,\n'
    '        },\n'
    '        parse_html_publication_date(html),\n'
    '    )\n'
    '\n'
    '\n'
    'def ism_report_url(kind: str, month_slug: str) -> str:\n'
    '    return urljoin(ISM_REPORT_INDEX_URL, f"{kind}/{month_slug}/")\n'
    '\n'
    '\n'
    'def month_sequence_ending_at(end_month: date, count: int) -> list[date]:\n'
    '    months = []\n'
    '    year = end_month.year\n'
    '    month = end_month.month\n'
    '    for _ in range(count):\n'
    '        months.append(date(year, month, 1))\n'
    '        month -= 1\n'
    '        if month == 0:\n'
    '            month = 12\n'
    '            year -= 1\n'
    '    return months\n'
    '\n'
    '\n'
    'def latest_contiguous_ism_rows(frame: pd.DataFrame, months: int) -> pd.DataFrame:\n'
    '    """Return the latest contiguous block of ISM release months."""\n'
    '    if frame.empty:\n'
    '        return frame\n'
    '    sorted_frame = frame.sort_values("date").reset_index(drop=True)\n'
    '    release_months = [\n'
    '        date(item.year, item.month, 1)\n'
    '        for item in pd.to_datetime(sorted_frame["date"], errors="coerce")\n'
    '    ]\n'
    '    for end_index in range(len(sorted_frame) - 1, months - 2, -1):\n'
    '        start_index = end_index - months + 1\n'
    '        block_months = release_months[start_index : end_index + 1]\n'
    '        expected = list(reversed(month_sequence_ending_at(block_months[-1], months)))\n'
    '        if block_months == expected:\n'
    '            return sorted_frame.iloc[start_index : end_index + 1].reset_index(drop=True)\n'
    '    return sorted_frame.tail(months).reset_index(drop=True)\n'
    '\n'
    '\n'
    'def candidate_months_from_today(lookback: int = 18) -> list[date]:\n'
    '    return month_sequence_ending_at(date.today().replace(day=1), lookback)\n'
    '\n'
    '\n'
    'def load_ism_backup_response(session, kind, target_month, parser):\n'
    '    """Use a verified local archive or an explicitly registered ISM release."""\n'
    '    source_name = "ISM Manufacturing" if kind == "pmi" else "ISM Services"\n'
    '    period = target_month.isoformat()\n'
    '    archived = load_archived_source(source_name, period)\n'
    '    if archived is not None:\n'
    '        content, metadata = archived\n'
    '        return response_from_archive(content, metadata), True\n'
    '    url = ISM_PRESS_RELEASES.get((kind, period))\n'
    '    if not url:\n'
    '        return None, False\n'
    '    print(\n'
    '        f"  ISM {kind}: checking original press release for {target_month:%B %Y}...",\n'
    '        flush=True,\n'
    '    )\n'
    '    try:\n'
    '        response = fetch_url(session, url, minimum_bytes=MIN_HTML_BYTES)\n'
    '    except requests.RequestException:\n'
    '        logging.warning(\n'
    '            "ISM original press release unavailable for %s %s; preserving existing cells.",\n'
    '            kind,\n'
    '            period,\n'
    '        )\n'
    '        return None, False\n'
    '    validate_response_content_type(\n'
    '        response, source_name, ("text/html", "application/xhtml")\n'
    '    )\n'
    '    parsed = parser(response.text, response.url)\n'
    '    if not parsed or parsed.report_date != target_month:\n'
    '        raise RuntimeError(f"ISM press release does not match {kind} {period}.")\n'
    '    if not parsed.publication_date or parsed.publication_date.replace(\n'
    '        day=1\n'
    '    ) != ism_release_month(target_month):\n'
    '        raise RuntimeError(\n'
    '            f"ISM press release has an unexpected publication date for {period}."\n'
    '        )\n'
    '    return response, False\n'
    '\n'
    '\n'
    'def scrape_ism_month(\n'
    '    session: requests.Session,\n'
    '    target_month: date,\n'
    '    scraped_at: str,\n'
    '    logger: logging.Logger,\n'
    '    url_overrides: dict[str, str] | None = None,\n'
    ') -> dict[str, object] | None:\n'
    '    """Scrape one month from ISM, tolerating missing Manufacturing or Services pages."""\n'
    '    month_slug = MONTHS[target_month.month - 1]\n'
    '    row: dict[str, object] = {\n'
    '        "date": ism_release_month(target_month).isoformat(),\n'
    '        "report_month": target_month.isoformat(),\n'
    '        "scraped_at": scraped_at,\n'
    '    }\n'
    '    for kind, parser in [\n'
    '        ("pmi", parse_ism_manufacturing_report),\n'
    '        ("services", parse_ism_services_report),\n'
    '    ]:\n'
    '        source_name = "ISM Manufacturing" if kind == "pmi" else "ISM Services"\n'
    '        period = target_month.isoformat()\n'
    '        url = fresh_ism_url(\n'
    '            (url_overrides or {}).get(kind) or ism_report_url(kind, month_slug)\n'
    '        )\n'
    '        response = None\n'
    '        used_archive = False\n'
    '        try:\n'
    '            response = fetch_url(\n'
    '                session,\n'
    '                url,\n'
    '                minimum_bytes=MIN_HTML_BYTES,\n'
    '                request_headers=ISM_FRESH_HEADERS,\n'
    '            )\n'
    '        except OfficialSourceRedirectError as exc:\n'
    '            if not is_ism_login_redirect(exc):\n'
    '                raise\n'
    '            logger.warning(\n'
    '                "ISM %s report for %s redirected to the login page; treating it as unavailable.",\n'
    '                kind,\n'
    '                target_month,\n'
    '            )\n'
    '        except requests.HTTPError as exc:\n'
    '            status = exc.response.status_code if exc.response is not None else None\n'
    '            if status not in {404, 410}:\n'
    '                raise\n'
    '            logger.warning(\n'
    '                "Could not fetch ISM %s report for %s: %s", kind, target_month, exc\n'
    '            )\n'
    '        except requests.RequestException as exc:\n'
    '            logger.warning(\n'
    '                "Could not fetch ISM %s report for %s: %s", kind, target_month, exc\n'
    '            )\n'
    '\n'
    '        if response is None:\n'
    '            response, used_archive = load_ism_backup_response(\n'
    '                session, kind, target_month, parser\n'
    '            )\n'
    '            if response is None:\n'
    '                continue\n'
    '\n'
    '        validate_response_content_type(\n'
    '            response,\n'
    '            f"ISM {kind} report",\n'
    '            ("text/html", "application/xhtml"),\n'
    '        )\n'
    '        parsed = parser(response.text, response.url)\n'
    '\n'
    '        if not parsed or parsed.report_date != target_month:\n'
    "            # ISM's CDN can briefly serve the prior month's body at the new\n"
    "            # month's canonical URL. Retry with a new cache token before\n"
    '            # concluding that the requested report is unavailable.\n'
    '            if not used_archive:\n'
    '                try:\n'
    '                    refreshed_response = fetch_url(\n'
    '                        session,\n'
    '                        fresh_ism_url(url),\n'
    '                        minimum_bytes=MIN_HTML_BYTES,\n'
    '                        request_headers=ISM_FRESH_HEADERS,\n'
    '                    )\n'
    '                    refreshed_parsed = parser(\n'
    '                        refreshed_response.text,\n'
    '                        refreshed_response.url,\n'
    '                    )\n'
    '                    if (\n'
    '                        refreshed_parsed\n'
    '                        and refreshed_parsed.report_date == target_month\n'
    '                    ):\n'
    '                        response = refreshed_response\n'
    '                        parsed = refreshed_parsed\n'
    '                except (OfficialSourceRedirectError, requests.RequestException):\n'
    '                    pass\n'
    '\n'
    '        if not parsed or parsed.report_date != target_month:\n'
    '            # ISM sometimes returns a valid generic/current report page with\n'
    '            # HTTP 200 for an older month slug. That is not the requested\n'
    '            # release, so use the checksum-verified period archive if present.\n'
    '            if used_archive:\n'
    '                continue\n'
    '            response, used_archive = load_ism_backup_response(\n'
    '                session, kind, target_month, parser\n'
    '            )\n'
    '            if response is None:\n'
    '                continue\n'
    '            parsed = parser(response.text, response.url)\n'
    '            if not parsed or parsed.report_date != target_month:\n'
    '                continue\n'
    '        if not used_archive:\n'
    '            archive_validated_source(\n'
    '                source_name,\n'
    '                period,\n'
    '                response,\n'
    '                "html",\n'
    '                parsed.values.keys(),\n'
    '                parsed.values,\n'
    '            )\n'
    '        else:\n'
    '            archived_metadata = load_archived_source(source_name, period)\n'
    '            if archived_metadata is not None:\n'
    '                _, metadata = archived_metadata\n'
    '                set_source_status(\n'
    '                    SourceStatus(\n'
    '                        source=source_name,\n'
    '                        status="OK (archive)",\n'
    '                        latest_data_month=period,\n'
    '                        retrieved_at=metadata.get("retrieved_at", ""),\n'
    '                        source_url=metadata.get("final_url", ""),\n'
    '                        sha256=metadata.get("sha256", ""),\n'
    '                        byte_size=int(metadata.get("byte_size", 0)),\n'
    '                        content_type=metadata.get("content_type", ""),\n'
    '                        schema_fingerprint=metadata.get("schema_fingerprint", ""),\n'
    '                        archive_file=metadata.get("archive_file", ""),\n'
    '                        revision=metadata.get("revision", ""),\n'
    '                    )\n'
    '                )\n'
    '        row[f"{kind}_source_url"] = parsed.source_url\n'
    '        row[f"{kind}_publication_date"] = (\n'
    '            parsed.publication_date.isoformat() if parsed.publication_date else None\n'
    '        )\n'
    '        for column, value in parsed.values.items():\n'
    '            if value is None:\n'
    '                continue\n'
    '            existing = row.get(column)\n'
    '            if existing is not None and abs(float(existing) - value) > 0.11:\n'
    '                raise RuntimeError(\n'
    '                    f"ISM reports disagree for {target_month:%B %Y} {column}: "\n'
    '                    f"{existing} versus {value}."\n'
    '                )\n'
    '            if existing is None:\n'
    '                row[column] = value\n'
    '\n'
    '    has_value = any(row.get(column) is not None for column in ISM_VALUE_COLUMNS)\n'
    '    return row if has_value else None\n'
    '\n'
    '\n'
    'def find_latest_ism_month(session: requests.Session, logger: logging.Logger) -> date:\n'
    '    """Find the newest recent month with at least one official ISM report."""\n'
    '    scraped_at = datetime.now(timezone.utc).isoformat(timespec="seconds")\n'
    '    try:\n'
    '        current_urls = discover_current_ism_urls(session)\n'
    '    except Exception as exc:\n'
    '        if is_ism_login_redirect(exc):\n'
    '            logger.info(\n'
    '                "ISM report index redirected to its login page; "\n'
    '                "trying the known public monthly report URLs."\n'
    '            )\n'
    '        else:\n'
    '            logger.warning(\n'
    '                "ISM report index is unreachable; trying known URLs and "\n'
    '                "checksum-verified archives: %s",\n'
    '                exc,\n'
    '            )\n'
    '        current_urls = {}\n'
    '\n'
    '    if current_urls:\n'
    '        for candidate in candidate_months_from_today():\n'
    '            if scrape_ism_month(\n'
    '                session,\n'
    '                candidate,\n'
    '                scraped_at,\n'
    '                logger,\n'
    '                url_overrides=current_urls,\n'
    '            ):\n'
    '                return candidate\n'
    '\n'
    '    logger.info(\n'
    '        "ISM current-report links were not recognized; trying known monthly URLs."\n'
    '    )\n'
    '    for candidate in candidate_months_from_today():\n'
    '        if scrape_ism_month(session, candidate, scraped_at, logger):\n'
    '            return candidate\n'
    '    raise RuntimeError("No parseable recent ISM report pages were found.")\n'
    '\n'
    '\n'
    'def scrape_ism_pmi(months: int) -> pd.DataFrame:\n'
    '    """Scrape official ISM PMI values into a DataFrame."""\n'
    '    logger = logging.getLogger("economic_indicator_export")\n'
    '    session = make_requests_session()\n'
    '    latest_month = find_latest_ism_month(session, logger)\n'
    '    scraped_at = datetime.now(timezone.utc).isoformat(timespec="seconds")\n'
    '    try:\n'
    '        current_urls = discover_current_ism_urls(session)\n'
    '    except Exception:\n'
    '        current_urls = {}\n'
    '    rows = []\n'
    '    for target_month in month_sequence_ending_at(latest_month, months):\n'
    '        row = scrape_ism_month(\n'
    '            session,\n'
    '            target_month,\n'
    '            scraped_at,\n'
    '            logger,\n'
    '            url_overrides=current_urls if target_month == latest_month else None,\n'
    '        )\n'
    '        if row:\n'
    '            rows.append(row)\n'
    '    if not rows:\n'
    '        raise RuntimeError("ISM returned no parseable PMI rows.")\n'
    '    frame = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)\n'
    '    validate_ism_frame(frame, expected_months=months)\n'
    '    return frame\n'
    '\n'
    '\n'
    'def validate_numeric_column(\n'
    '    frame: pd.DataFrame,\n'
    '    column: str,\n'
    '    source_name: str,\n'
    '    minimum: float,\n'
    '    maximum: float,\n'
    ') -> None:\n'
    '    """Reject missing, nonnumeric, or implausible official-source values."""\n'
    '    if column not in frame.columns:\n'
    '        raise RuntimeError(f"{source_name} format changed; missing \'{column}\'.")\n'
    '    numeric = pd.to_numeric(frame[column], errors="coerce")\n'
    '    if numeric.dropna().empty:\n'
    '        raise RuntimeError(f"{source_name} has no numeric values for \'{column}\'.")\n'
    '    invalid = numeric.dropna()[~numeric.dropna().between(minimum, maximum)]\n'
    '    if not invalid.empty:\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} returned an implausible \'{column}\' value: {invalid.iloc[0]}."\n'
    '        )\n'
    '\n'
    '\n'
    'def validate_release_freshness(\n'
    '    latest_release_month: date,\n'
    '    source_name: str,\n'
    '    current_month_deadline_day: int,\n'
    ') -> None:\n'
    '    """Require the current release after its conservative monthly deadline."""\n'
    '    current_month = month_start(date.today())\n'
    '    oldest_allowed = (\n'
    '        current_month\n'
    '        if date.today().day >= current_month_deadline_day\n'
    '        else add_months(current_month, -1)\n'
    '    )\n'
    '    if latest_release_month < oldest_allowed:\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} appears stale. Latest release month is "\n'
    '            f"{latest_release_month:%B %Y}; expected {oldest_allowed:%B %Y} or later."\n'
    '        )\n'
    '    if latest_release_month > current_month:\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} returned a future release month: "\n'
    '            f"{latest_release_month:%B %Y}."\n'
    '        )\n'
    '\n'
    '\n'
    'def validate_monthly_dates(\n'
    '    frame: pd.DataFrame,\n'
    '    source_name: str,\n'
    '    date_column: str = "date",\n'
    '    continuity_months: int = 24,\n'
    ') -> None:\n'
    '    """Reject invalid, duplicate, future, or recently gapped monthly dates."""\n'
    '    if date_column not in frame.columns:\n'
    '        raise RuntimeError(f"{source_name} format changed; missing \'{date_column}\'.")\n'
    '    parsed = pd.to_datetime(frame[date_column], errors="coerce")\n'
    '    if parsed.isna().any():\n'
    '        raise RuntimeError(f"{source_name} contains invalid monthly dates.")\n'
    '    normalized = [date(item.year, item.month, 1) for item in parsed]\n'
    '    if len(normalized) != len(set(normalized)):\n'
    '        raise RuntimeError(f"{source_name} contains duplicate monthly dates.")\n'
    '    if normalized != sorted(normalized):\n'
    '        raise RuntimeError(f"{source_name} monthly dates are not sorted.")\n'
    '    if normalized[-1] > month_start(date.today()):\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} contains a future month: {normalized[-1]:%B %Y}."\n'
    '        )\n'
    '    tail = normalized[-min(continuity_months, len(normalized)) :]\n'
    '    expected = list(reversed(month_sequence_ending_at(tail[-1], len(tail))))\n'
    '    if tail != expected:\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} contains a gap in its recent monthly history."\n'
    '        )\n'
    '\n'
    '\n'
    'def validate_observation_tail(\n'
    '    observations: list[dict],\n'
    '    source_name: str,\n'
    '    expected_months: int,\n'
    ') -> None:\n'
    '    """Require every requested recent month exactly once with a finite value."""\n'
    '    if len(observations) != expected_months:\n'
    '        raise RuntimeError(\n'
    '            f"{source_name} returned {len(observations)} monthly values; "\n'
    '            f"{expected_months} were requested."\n'
    '        )\n'
    '    dates = [item["date"] for item in observations]\n'
    '    if len(dates) != len(set(dates)):\n'
    '        raise RuntimeError(f"{source_name} returned duplicate output months.")\n'
    '    expected_dates = list(reversed(month_sequence_ending_at(dates[-1], len(dates))))\n'
    '    if dates != expected_dates:\n'
    '        raise RuntimeError(f"{source_name} output months contain a gap.")\n'
    '    for item in observations:\n'
    '        value = item.get("value")\n'
    '        if value is None or not isinstance(value, (int, float)) or not pd.notna(value):\n'
    '            raise RuntimeError(f"{source_name} returned a missing or nonnumeric value.")\n'
    '\n'
    '\n'
    'def validate_ism_frame(frame: pd.DataFrame, expected_months: int | None = None) -> None:\n'
    '    """Validate known ISM values; missing releases never clear existing history."""\n'
    '    frame = frame.sort_values("date").copy()\n'
    '    parsed_dates = pd.to_datetime(frame["date"], errors="coerce")\n'
    '    if parsed_dates.isna().any():\n'
    '        raise RuntimeError("ISM returned an invalid release date.")\n'
    '    if parsed_dates.duplicated().any():\n'
    '        raise RuntimeError("ISM returned duplicate release months.")\n'
    '    release_dates = [date(item.year, item.month, 1) for item in parsed_dates]\n'
    '    if not release_dates:\n'
    '        raise RuntimeError("ISM returned no verified release months.")\n'
    '    latest_release = release_dates[-1]\n'
    '    expected_dates = list(\n'
    '        reversed(\n'
    '            month_sequence_ending_at(\n'
    '                latest_release, expected_months or len(release_dates)\n'
    '            )\n'
    '        )\n'
    '    )\n'
    '    missing_dates = sorted(set(expected_dates) - set(release_dates))\n'
    '    if missing_dates:\n'
    '        add_validation_warning(\n'
    '            "ISM",\n'
    '            "Unavailable release month(s): "\n'
    '            + ", ".join(item.isoformat() for item in missing_dates)\n'
    '            + "; existing spreadsheet cells will be preserved.",\n'
    '        )\n'
    '\n'
    '    if "report_month" not in frame.columns:\n'
    '        raise RuntimeError("ISM format changed; missing \'report_month\'.")\n'
    '    parsed_report_months = pd.to_datetime(frame["report_month"], errors="coerce")\n'
    '    if parsed_report_months.isna().any():\n'
    '        raise RuntimeError("ISM returned an invalid report month.")\n'
    '\n'
    '    latest = frame.iloc[-1]\n'
    '    for column in ISM_VALUE_COLUMNS:\n'
    '        values = (\n'
    '            frame[column]\n'
    '            if column in frame\n'
    '            else pd.Series(float("nan"), index=frame.index)\n'
    '        )\n'
    '        numeric = pd.to_numeric(values, errors="coerce")\n'
    '        if (values.notna() & numeric.isna()).any():\n'
    '            raise RuntimeError(f"ISM returned a nonnumeric \'{column}\' value.")\n'
    '        if not numeric.dropna().between(0, 100).all():\n'
    '            raise RuntimeError(f"ISM returned an implausible \'{column}\' value.")\n'
    '        missing = frame.loc[numeric.isna(), "report_month"].astype(str).str[:7].tolist()\n'
    '        if missing:\n'
    '            add_validation_warning(\n'
    '                ISM_VALUE_LABELS[column],\n'
    '                f"Official ISM {ISM_VALUE_LABELS[column]} unavailable for "\n'
    '                + ", ".join(missing)\n'
    '                + "; existing spreadsheet cells will be preserved.",\n'
    '            )\n'
    '\n'
    '    if all(pd.isna(latest.get(column)) for column in ISM_VALUE_COLUMNS):\n'
    '        raise RuntimeError(\n'
    '            "ISM newest report month contains no official indicator values."\n'
    '        )\n'
    '\n'
    '    for report_timestamp, release_timestamp in zip(parsed_report_months, parsed_dates):\n'
    '        report_month = report_timestamp.date()\n'
    '        release_month = release_timestamp.date()\n'
    '        if ism_release_month(report_month) != release_month:\n'
    '            raise RuntimeError("ISM report month and release month are inconsistent.")\n'
    '    validate_release_freshness(\n'
    '        latest_release,\n'
    '        "ISM",\n'
    '        current_month_deadline_day=10,\n'
    '    )\n'
    '    logging.info(\n'
    '        "Validated ISM: %s available release months through %s; updating verified values only.",\n'
    '        len(frame),\n'
    '        latest_release,\n'
    '    )\n'
    '\n'
    '\n'
    'def fred_get(endpoint: str, params: dict[str, str]) -> dict:\n'
    '    global LAST_FRED_REQUEST_AT\n'
    '    query = urlencode({**params, "file_type": "json"})\n'
    '    url = f"{FRED_API_BASE}/{endpoint}?{query}"\n'
    '\n'
    '    last_error = None\n'
    '    for attempt in range(1, 4):\n'
    '        retry_seconds = 2 * attempt\n'
    '        elapsed = time.monotonic() - LAST_FRED_REQUEST_AT\n'
    '        if elapsed < FRED_MIN_REQUEST_INTERVAL_SECONDS:\n'
    '            sleep(FRED_MIN_REQUEST_INTERVAL_SECONDS - elapsed)\n'
    '        try:\n'
    '            LAST_FRED_REQUEST_AT = time.monotonic()\n'
    '            with urlopen(url, timeout=30) as response:\n'
    '                return json.loads(response.read().decode("utf-8"))\n'
    '        except HTTPError as exc:\n'
    '            details = exc.read().decode("utf-8", errors="replace")\n'
    '            last_error = RuntimeError(f"FRED returned an error ({exc.code}): {details}")\n'
    '            if exc.code == 429:\n'
    '                retry_seconds = 20 * attempt\n'
    '            if exc.code not in {429, 500, 502, 503, 504} or attempt == 3:\n'
    '                raise last_error from exc\n'
    '        except (URLError, TimeoutError) as exc:\n'
    '            last_error = RuntimeError(\n'
    '                f"Could not reach FRED: {getattr(exc, \'reason\', exc)}"\n'
    '            )\n'
    '            if attempt == 3:\n'
    '                raise last_error from exc\n'
    '\n'
    '        sleep(retry_seconds)\n'
    '\n'
    '    raise last_error or RuntimeError("FRED request failed.")\n'
    '\n'
    '\n'
    'def imf_datamapper_get(path: str, params: dict[str, str] | None = None) -> dict:\n'
    '    """Fetch JSON from the official IMF DataMapper API with simple retries."""\n'
    '    query = f"?{urlencode(params)}" if params else ""\n'
    '    url = f"{IMF_DATAMAPPER_API_BASE}/{path.strip(\'/\')}{query}"\n'
    '\n'
    '    last_error = None\n'
    '    for attempt in range(1, 4):\n'
    '        try:\n'
    '            with urlopen(url, timeout=30) as response:\n'
    '                return json.loads(response.read().decode("utf-8"))\n'
    '        except HTTPError as exc:\n'
    '            details = exc.read().decode("utf-8", errors="replace")\n'
    '            last_error = RuntimeError(\n'
    '                f"IMF DataMapper returned an error ({exc.code}): {details}"\n'
    '            )\n'
    '            if exc.code not in {429, 500, 502, 503, 504} or attempt == 3:\n'
    '                raise last_error from exc\n'
    '        except URLError as exc:\n'
    '            last_error = RuntimeError(f"Could not reach IMF DataMapper: {exc.reason}")\n'
    '            if attempt == 3:\n'
    '                raise last_error from exc\n'
    '\n'
    '        sleep(2 * attempt)\n'
    '\n'
    '    raise last_error or RuntimeError("IMF DataMapper request failed.")\n'
    '\n'
    '\n'
    'def parse_optional_datetime(value: object) -> datetime | None:\n'
    '    """Parse an optional source timestamp without assuming one exact format."""\n'
    '    if not value:\n'
    '        return None\n'
    '    parsed = pd.to_datetime(str(value), errors="coerce", utc=True)\n'
    '    if pd.isna(parsed):\n'
    '        return None\n'
    '    return parsed.to_pydatetime()\n'
    '\n'
    '\n'
    'def parse_imf_year_values(data: dict, series_id: str, country: str) -> dict[int, float]:\n'
    '    """Extract {year: value} from IMF DataMapper\'s nested values payload."""\n'
    '    values = data.get("values", {})\n'
    '    candidates = [\n'
    '        values.get(series_id, {}).get(country),\n'
    '        values.get(country, {}).get(series_id),\n'
    '        values.get(series_id.upper(), {}).get(country),\n'
    '    ]\n'
    '    raw_series = next((item for item in candidates if isinstance(item, dict)), None)\n'
    '    if raw_series is None:\n'
    '        raise RuntimeError(\n'
    '            f"IMF DataMapper returned no {series_id} values for {country}."\n'
    '        )\n'
    '\n'
    '    parsed: dict[int, float] = {}\n'
    '    for raw_year, raw_value in raw_series.items():\n'
    '        try:\n'
    '            year = int(raw_year)\n'
    '            value = float(Decimal(str(raw_value)))\n'
    '        except (TypeError, ValueError, InvalidOperation):\n'
    '            continue\n'
    '        parsed[year] = value\n'
    '    if not parsed:\n'
    '        raise RuntimeError(\n'
    '            f"IMF DataMapper returned no numeric {series_id} values for {country}."\n'
    '        )\n'
    '    return parsed\n'
    '\n'
    '\n'
    'def get_imf_indicator_metadata(series_ids: Iterable[str]) -> dict[str, dict]:\n'
    '    """Fetch metadata for the IMF indicators used by this calculation."""\n'
    '    data = imf_datamapper_get("indicators")\n'
    '    indicators = data.get("indicators", {})\n'
    '    metadata = {}\n'
    '    for series_id in series_ids:\n'
    '        item = indicators.get(series_id)\n'
    '        if not isinstance(item, dict):\n'
    '            raise RuntimeError(\n'
    '                f"IMF DataMapper indicators metadata is missing {series_id}."\n'
    '            )\n'
    '        metadata[series_id] = item\n'
    '    return metadata\n'
    '\n'
    '\n'
    'def get_imf_fpp_country_data(country: str) -> dict:\n'
    '    """Fetch and cache the IMF FPP revenue and interest series for one country."""\n'
    '    cached = _IMF_FPP_CACHE.get(country)\n'
    '    if cached is not None:\n'
    '        return cached\n'
    '\n'
    '    metadata = get_imf_indicator_metadata(["rev", "ie"])\n'
    '    revenue_data = imf_datamapper_get(f"rev/{country}")\n'
    '    interest_data = imf_datamapper_get(f"ie/{country}")\n'
    '    revenue_by_year = parse_imf_year_values(revenue_data, "rev", country)\n'
    '    interest_by_year = parse_imf_year_values(interest_data, "ie", country)\n'
    '\n'
    '    latest_modified = max(\n'
    '        (\n'
    '            parse_optional_datetime(metadata_item.get("last-modified"))\n'
    '            for metadata_item in metadata.values()\n'
    '        ),\n'
    '        default=None,\n'
    '    )\n'
    '    retrieved_at = datetime.now(timezone.utc)\n'
    '    release_date = latest_modified.date() if latest_modified else retrieved_at.date()\n'
    '    latest_year = max(set(revenue_by_year) | set(interest_by_year))\n'
    '    source_url = f"{IMF_DATAMAPPER_API_BASE}/rev/{country} | {IMF_DATAMAPPER_API_BASE}/ie/{country}"\n'
    '    payload_hash = hashlib.sha256(\n'
    '        json.dumps(\n'
    '            {\n'
    '                "rev": revenue_by_year,\n'
    '                "ie": interest_by_year,\n'
    '                "metadata": metadata,\n'
    '            },\n'
    '            sort_keys=True,\n'
    '        ).encode("utf-8")\n'
    '    ).hexdigest()\n'
    '    set_source_status(\n'
    '        SourceStatus(\n'
    '            source="IMF DataMapper FPP",\n'
    '            status="OK",\n'
    '            latest_data_month=f"{latest_year}-12-01",\n'
    '            retrieved_at=retrieved_at.isoformat(),\n'
    '            source_url=source_url,\n'
    '            sha256=payload_hash,\n'
    '            byte_size=len(json.dumps({"rev": revenue_data, "ie": interest_data})),\n'
    '            content_type="application/json",\n'
    '            schema_fingerprint="values.rev.country.year + values.ie.country.year",\n'
    '            archive_file="",\n'
    '            revision=(\n'
    '                f"rev last-modified={metadata[\'rev\'].get(\'last-modified\', \'\')}; "\n'
    '                f"ie last-modified={metadata[\'ie\'].get(\'last-modified\', \'\')}"\n'
    '            ),\n'
    '        )\n'
    '    )\n'
    '    cached = {\n'
    '        "metadata": metadata,\n'
    '        "release_date": release_date,\n'
    '        "rev": revenue_by_year,\n'
    '        "ie": interest_by_year,\n'
    '    }\n'
    '    _IMF_FPP_CACHE[country] = cached\n'
    '    return cached\n'
    '\n'
    '\n'
    'def get_imf_public_finance_observations(\n'
    '    indicator: Indicator, months: int\n'
    ') -> list[dict]:\n'
    '    """Return annual IMF FPP records matched to their year-end month."""\n'
    '    country = indicator.source_column or DEFAULT_IMF_COUNTRY\n'
    '    series_id = indicator.series_id\n'
    '    if series_id not in {"rev", "ie", "rev_over_ie"}:\n'
    '        raise RuntimeError(\n'
    '            f"Unsupported IMF FPP series for {indicator.name}: {series_id}"\n'
    '        )\n'
    '\n'
    '    data = get_imf_fpp_country_data(country)\n'
    '    if series_id == "rev_over_ie":\n'
    '        series_by_year = {}\n'
    '        for year in sorted(set(data["rev"]) | set(data["ie"])):\n'
    '            revenue = data["rev"].get(year)\n'
    '            interest = data["ie"].get(year)\n'
    '            if revenue is None or interest is None:\n'
    '                add_validation_warning(\n'
    '                    indicator.name,\n'
    '                    f"IMF rev or ie is missing for {country} {year}; coverage ratio left blank.",\n'
    '                )\n'
    '                continue\n'
    '            if interest == 0:\n'
    '                add_validation_warning(\n'
    '                    indicator.name,\n'
    '                    f"IMF ie is zero for {country} {year}; coverage ratio left blank.",\n'
    '                )\n'
    '                continue\n'
    '            series_by_year[year] = revenue / interest\n'
    '    else:\n'
    '        series_by_year = data[series_id]\n'
    '    release_date = data["release_date"]\n'
    '\n'
    '    observations = []\n'
    '    for year, value in sorted(series_by_year.items()):\n'
    '        observation_date = date(year, 12, 31)\n'
    '        observations.append(\n'
    '            {\n'
    '                "country": country,\n'
    '                "year": year,\n'
    '                "observation_date": observation_date,\n'
    '                "release_date": release_date,\n'
    '                "realtime_start": None,\n'
    '                "source": "IMF DataMapper FPP",\n'
    '                "source_series_id": (\n'
    '                    "rev/ie" if series_id == "rev_over_ie" else series_id\n'
    '                ),\n'
    '                "selection_date": date(year, 12, 1),\n'
    '                "period_aligned": True,\n'
    '                "history_complete": FULL_HISTORY_MODE,\n'
    '                "value_vintage": "Latest IMF DataMapper FPP values",\n'
    '                "value": value,\n'
    '            }\n'
    '        )\n'
    '\n'
    '    validate_release_records(observations, indicator.name)\n'
    '    return observations\n'
    '\n'
    '\n'
    'def print_imf_public_finance_audit(\n'
    '    country: str = DEFAULT_IMF_COUNTRY,\n'
    '    limit: int = 10,\n'
    ') -> None:\n'
    '    """Print raw IMF revenue and interest inputs for manual verification."""\n'
    '    data = get_imf_fpp_country_data(country)\n'
    '    rows = []\n'
    '    for year in sorted(set(data["rev"]) & set(data["ie"]))[-limit:]:\n'
    '        revenue = data["rev"][year]\n'
    '        interest = data["ie"][year]\n'
    '        coverage_ratio = "" if interest == 0 else revenue / interest\n'
    '        rows.append([country, year, revenue, interest, coverage_ratio])\n'
    '    if COMPACT_TERMINAL:\n'
    '        print(\n'
    '            f"IMF public-finance history: {len(rows)} audit years "\n'
    '            f"through {rows[-1][1] if rows else \'n/a\'}."\n'
    '        )\n'
    '        return\n'
    '    print_console_table(\n'
    '        "IMF Public Finance audit:",\n'
    '        [\n'
    '            "Country",\n'
    '            "Year",\n'
    '            "Government revenue (% GDP)",\n'
    '            "Interest paid on public debt (% GDP)",\n'
    '            "Coverage ratio",\n'
    '        ],\n'
    '        rows,\n'
    '    )\n'
    '\n'
    '\n'
    'def normalize_fred_metadata_text(value: str) -> str:\n'
    '    """Normalize small FRED wording changes without hiding real series changes."""\n'
    '    return re.sub(r"[^a-z0-9]+", " ", value.lower().replace("u.s.", "us")).strip()\n'
    '\n'
    '\n'
    'def add_validation_warning(indicator: str, message: str) -> None:\n'
    '    """Record a non-blocking validation issue for the report sheets."""\n'
    '    VALIDATION_MESSAGES.append(\n'
    '        {\n'
    '            "severity": "WARNING",\n'
    '            "indicator": indicator,\n'
    '            "message": message,\n'
    '        }\n'
    '    )\n'
    '\n'
    '\n'
    'def get_fred_series_metadata(api_key: str, series_id: str) -> dict:\n'
    '    """Fetch and validate FRED\'s series identity metadata."""\n'
    '    if series_id in _FRED_METADATA_CACHE:\n'
    '        return _FRED_METADATA_CACHE[series_id]\n'
    '\n'
    '    data = fred_get(\n'
    '        "series",\n'
    '        {\n'
    '            "api_key": api_key,\n'
    '            "series_id": series_id,\n'
    '        },\n'
    '    )\n'
    '    series = data.get("seriess", [])\n'
    '    if not series:\n'
    '        raise RuntimeError(f"FRED did not return a series for {series_id}.")\n'
    '    metadata = series[0]\n'
    '    if metadata.get("id") != series_id:\n'
    '        raise RuntimeError(\n'
    '            f"FRED metadata identity mismatch: requested {series_id}, "\n'
    '            f"received {metadata.get(\'id\')!r}."\n'
    '        )\n'
    '    required_fields = [\n'
    '        "title",\n'
    '        "units",\n'
    '        "frequency",\n'
    '        "observation_start",\n'
    '        "last_updated",\n'
    '    ]\n'
    '    missing = [field for field in required_fields if not metadata.get(field)]\n'
    '    if missing:\n'
    '        raise RuntimeError(\n'
    '            f"FRED metadata changed for {series_id}; missing: {\', \'.join(missing)}."\n'
    '        )\n'
    '\n'
    '    expected = FRED_EXPECTED_METADATA.get(series_id, {})\n'
    '    title_contains = expected.get("title_contains")\n'
    '    units_contains = expected.get("units_contains")\n'
    '    expected_frequency = expected.get("frequency")\n'
    '    if title_contains and normalize_fred_metadata_text(\n'
    '        title_contains\n'
    '    ) not in normalize_fred_metadata_text(metadata["title"]):\n'
    '        add_validation_warning(\n'
    '            f"FRED {series_id}",\n'
    '            "FRED title wording differs from the expected metadata guard: "\n'
    '            f"{metadata[\'title\']}.",\n'
    '        )\n'
    '    if units_contains and normalize_fred_metadata_text(\n'
    '        units_contains\n'
    '    ) not in normalize_fred_metadata_text(metadata["units"]):\n'
    '        add_validation_warning(\n'
    '            f"FRED {series_id}",\n'
    '            "FRED units wording differs from the expected metadata guard: "\n'
    '            f"{metadata[\'units\']}.",\n'
    '        )\n'
    '    if expected_frequency and metadata["frequency"] != expected_frequency:\n'
    '        add_validation_warning(\n'
    '            f"FRED {series_id}",\n'
    '            "FRED frequency wording differs from the expected metadata guard: "\n'
    '            f"{metadata[\'frequency\']}.",\n'
    '        )\n'
    '\n'
    '    _FRED_METADATA_CACHE[series_id] = metadata\n'
    '    return metadata\n'
    '\n'
    '\n'
    'def get_series_title(api_key: str, series_id: str) -> str:\n'
    '    return get_fred_series_metadata(api_key, series_id)["title"]\n'
    '\n'
    '\n'
    'def get_fred_observations(\n'
    '    api_key: str,\n'
    '    series_id: str,\n'
    '    start_date: date,\n'
    '    end_date: date,\n'
    '    units: str = "lin",\n'
    '    frequency: str = "",\n'
    ') -> list[dict]:\n'
    '    # Fetch the current published FRED values for the exact observation dates.\n'
    "    # A second ALFRED request supplies each observation's initial publication\n"
    '    # date without replacing the current value with its unrevised first print.\n'
    '    current_data = fred_get(\n'
    '        "series/observations",\n'
    '        {\n'
    '            "api_key": api_key,\n'
    '            "series_id": series_id,\n'
    '            "observation_start": start_date.isoformat(),\n'
    '            "observation_end": end_date.isoformat(),\n'
    '            "sort_order": "asc",\n'
    '            "units": "lin",\n'
    '        },\n'
    '    )\n'
    '    current_items = current_data.get("observations", [])\n'
    '    try:\n'
    '        reported_count = int(current_data.get("count", len(current_items)))\n'
    '    except (TypeError, ValueError) as exc:\n'
    '        raise RuntimeError(\n'
    '            f"FRED {series_id} returned an invalid observation count."\n'
    '        ) from exc\n'
    '    if len(current_items) < reported_count:\n'
    '        raise RuntimeError(\n'
    '            f"FRED {series_id} returned only {len(current_items)} of "\n'
    '            f"{reported_count} observations; refusing a truncated history."\n'
    '        )\n'
    '    # Current FRED values above are always fetched for the complete history.\n'
    '    # Exact ALFRED release-date metadata is limited to the latest four years;\n'
    '    # older observations use the documented metadata-only fallback below.\n'
    '    # This avoids hundreds of vintage calls without reducing value history.\n'
    '    initial_release_observations = []\n'
    '    window_months = 60\n'
    '    window_start = max(start_date, subtract_months(end_date, 48))\n'
    '    alfred_history_start = date(1991, 1, 1)\n'
    '    while window_start <= end_date:\n'
    '        window_end = min(\n'
    '            end_date,\n'
    '            add_months(window_start, window_months) - timedelta(days=1),\n'
    '        )\n'
    '        if window_end < alfred_history_start:\n'
    '            window_start = window_end + timedelta(days=1)\n'
    '            continue\n'
    '        realtime_start = max(window_start, alfred_history_start)\n'
    '        try:\n'
    '            initial_release_data = fred_get(\n'
    '                "series/observations",\n'
    '                {\n'
    '                    "api_key": api_key,\n'
    '                    "series_id": series_id,\n'
    '                    "observation_start": window_start.isoformat(),\n'
    '                    "observation_end": window_end.isoformat(),\n'
    '                    "realtime_start": realtime_start.isoformat(),\n'
    '                    "realtime_end": min(\n'
    "                        # FRED's real-time calendar can still be on the prior\n"
    '                        # US date during an evening run in Europe. This field\n'
    '                        # is release-date metadata only; current values come\n'
    '                        # from the separate non-vintage request above.\n'
    '                        end_date - timedelta(days=1),\n'
    '                        add_months(window_end, 3),\n'
    '                    ).isoformat(),\n'
    '                    "sort_order": "asc",\n'
    '                    "units": "lin",\n'
    '                    "output_type": "4",\n'
    '                },\n'
    '            )\n'
    '        except RuntimeError as exc:\n'
    '            if "does not exist in ALFRED" not in str(exc):\n'
    '                raise\n'
    '            initial_release_data = {"observations": []}\n'
    '        initial_release_observations.extend(\n'
    '            initial_release_data.get("observations", [])\n'
    '        )\n'
    '        window_start = window_end + timedelta(days=1)\n'
    '    initial_release_dates: dict[str, date] = {}\n'
    '    for item in initial_release_observations:\n'
    '        if not item.get("realtime_start"):\n'
    '            continue\n'
    '        released = datetime.strptime(item["realtime_start"], "%Y-%m-%d").date()\n'
    '        observed_key = item["date"]\n'
    '        previous = initial_release_dates.get(observed_key)\n'
    '        if previous is None or released < previous:\n'
    '            initial_release_dates[observed_key] = released\n'
    '\n'
    '    observations = []\n'
    '    missing_initial_release_dates = 0\n'
    '    for item in current_items:\n'
    '        raw_value = item.get("value")\n'
    '        if raw_value in (None, "."):\n'
    '            continue\n'
    '\n'
    '        try:\n'
    '            value = float(Decimal(raw_value))\n'
    '        except (InvalidOperation, ValueError):\n'
    '            continue\n'
    '\n'
    '        observation_date = datetime.strptime(item["date"], "%Y-%m-%d").date()\n'
    '        if frequency.startswith("Quarterly"):\n'
    '            selection_date = add_months(\n'
    '                month_start(observation_date),\n'
    '                2,\n'
    '            )\n'
    '        elif frequency.startswith("Annual"):\n'
    '            selection_date = date(observation_date.year, 12, 1)\n'
    '        else:\n'
    '            selection_date = month_start(observation_date)\n'
    '        release_date = initial_release_dates.get(item["date"])\n'
    '        if release_date is None:\n'
    '            # ALFRED real-time vintages do not cover every observation in\n'
    '            # century-long histories. The value and output month remain exact;\n'
    '            # this fallback is metadata-only because FRED records are explicitly\n'
    '            # period-aligned below.\n'
    '            release_date = observation_date\n'
    '            missing_initial_release_dates += 1\n'
    '\n'
    '        observations.append(\n'
    '            {\n'
    '                "observation_date": observation_date,\n'
    '                "release_date": release_date,\n'
    '                "realtime_start": release_date,\n'
    '                "source": "FRED / ALFRED",\n'
    '                "source_series_id": series_id,\n'
    '                "native_frequency": frequency,\n'
    '                "selection_date": selection_date,\n'
    '                "period_aligned": True,\n'
    '                "value_vintage": "Current FRED value",\n'
    '                "value": value,\n'
    '            }\n'
    '        )\n'
    '\n'
    '    if missing_initial_release_dates:\n'
    '        add_validation_warning(\n'
    '            f"FRED {series_id}",\n'
    '            f"ALFRED had no initial vintage date for {missing_initial_release_dates} "\n'
    '            "historical observations; their observation dates were retained as "\n'
    '            "metadata-only release-date fallbacks.",\n'
    '        )\n'
    '\n'
    '    if units == "lin":\n'
    '        return observations\n'
    '    if units != "pc1":\n'
    '        raise RuntimeError(\n'
    '            f"Unsupported point-in-time FRED transformation for {series_id}: {units}"\n'
    '        )\n'
    '    return transform_percent_change_from_year_ago(\n'
    '        observations,\n'
    '        frequency,\n'
    '        series_id,\n'
    '    )\n'
    '\n'
    '\n'
    'def prior_year_target(input_date: date) -> date:\n'
    '    """Return the same calendar date one year earlier, including leap-day safety."""\n'
    '    try:\n'
    '        return input_date.replace(year=input_date.year - 1)\n'
    '    except ValueError:\n'
    '        return input_date.replace(year=input_date.year - 1, day=28)\n'
    '\n'
    '\n'
    'def transform_percent_change_from_year_ago(\n'
    '    records: list[dict],\n'
    '    frequency: str,\n'
    '    series_id: str,\n'
    ') -> list[dict]:\n'
    '    """Reproduce FRED pc1 from current levels without losing release dates."""\n'
    '    transformed = []\n'
    '    ordered = sorted(records, key=lambda item: item["observation_date"])\n'
    '    for record in ordered:\n'
    '        target = prior_year_target(record["observation_date"])\n'
    '        if frequency.startswith("Weekly"):\n'
    '            candidates = [\n'
    '                item\n'
    '                for item in ordered\n'
    '                if item["observation_date"] <= target\n'
    '                and (target - item["observation_date"]).days <= 8\n'
    '            ]\n'
    '            prior = max(\n'
    '                candidates,\n'
    '                key=lambda item: item["observation_date"],\n'
    '                default=None,\n'
    '            )\n'
    '        else:\n'
    '            prior = next(\n'
    '                (item for item in ordered if item["observation_date"] == target),\n'
    '                None,\n'
    '            )\n'
    '        if prior is None or prior["value"] == 0:\n'
    '            continue\n'
    '        transformed.append(\n'
    '            {\n'
    '                "observation_date": record["observation_date"],\n'
    '                "release_date": max(\n'
    '                    record["release_date"],\n'
    '                    prior["release_date"],\n'
    '                ),\n'
    '                "realtime_start": max(\n'
    '                    record["realtime_start"],\n'
    '                    prior["realtime_start"],\n'
    '                ),\n'
    '                "source": "FRED / ALFRED",\n'
    '                "source_series_id": series_id,\n'
    '                "native_frequency": frequency,\n'
    '                "selection_date": record["selection_date"],\n'
    '                "period_aligned": True,\n'
    '                "value_vintage": "Current FRED value",\n'
    '                "value": ((record["value"] / prior["value"]) - 1.0) * 100.0,\n'
    '            }\n'
    '        )\n'
    '    if not transformed:\n'
    '        raise RuntimeError(\n'
    '            f"Could not calculate year-over-year growth for FRED series {series_id}."\n'
    '        )\n'
    '    return transformed\n'
    '\n'
    '\n'
    'def validate_release_records(records: list[dict], series_name: str) -> None:\n'
    '    """Validate normalized observation/release/value records."""\n'
    '    seen = set()\n'
    '    for record in records:\n'
    '        observation_date = record.get("observation_date")\n'
    '        release_date = record.get("release_date")\n'
    '        value = record.get("value")\n'
    '        if not isinstance(observation_date, date) or not isinstance(release_date, date):\n'
    '            raise RuntimeError(\n'
    '                f"{series_name} lost observation or release date metadata."\n'
    '            )\n'
    '        if release_date < observation_date and "Daily" not in series_name:\n'
    '            VALIDATION_MESSAGES.append(\n'
    '                {\n'
    '                    "severity": "WARNING",\n'
    '                    "indicator": series_name,\n'
    '                    "message": (\n'
    '                        f"Release date {release_date} precedes observation date "\n'
    '                        f"{observation_date}; verify source dating convention."\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '        if value is None or not pd.notna(value):\n'
    '            raise RuntimeError(f"{series_name} contains a missing value record.")\n'
    '        key = (\n'
    '            observation_date,\n'
    '            release_date,\n'
    '            record.get("selection_date"),\n'
    '        )\n'
    '        if key in seen:\n'
    '            raise RuntimeError(\n'
    '                f"{series_name} contains duplicate observation/release dates."\n'
    '            )\n'
    '        seen.add(key)\n'
    '\n'
    '\n'
    'def latest_available_record(records: list[dict], row_date: date) -> dict | None:\n'
    '    """Return the requested month-end aggregation; never carry values forward."""\n'
    '    row_month = month_start(row_date)\n'
    '    eligible = [\n'
    '        record\n'
    '        for record in records\n'
    '        if isinstance(record.get("release_date"), date)\n'
    '        and isinstance(record.get("observation_date"), date)\n'
    '        and record["observation_date"] <= row_date\n'
    '        and month_start(record.get("selection_date", record["release_date"]))\n'
    '        == row_month\n'
    '        and (record.get("period_aligned") is True or record["release_date"] <= row_date)\n'
    '    ]\n'
    '    if not eligible:\n'
    '        return None\n'
    '    latest = max(\n'
    '        eligible,\n'
    '        key=lambda record: (record["observation_date"], record["release_date"]),\n'
    '    )\n'
    '    numeric_values = [\n'
    '        record["value"] for record in eligible if pd.notna(record.get("value"))\n'
    '    ]\n'
    '    if len(numeric_values) != len(eligible):\n'
    '        raise RuntimeError("Monthly aggregation received a missing source value.")\n'
    '    aggregation_methods = {\n'
    '        str(record.get("monthly_aggregation", "latest")) for record in eligible\n'
    '    }\n'
    '    if len(aggregation_methods) != 1:\n'
    '        raise RuntimeError("Monthly aggregation received conflicting methods.")\n'
    '    aggregation_method = next(iter(aggregation_methods))\n'
    '    if aggregation_method == "latest" or len(eligible) == 1:\n'
    '        return latest\n'
    '    if aggregation_method != "mean":\n'
    '        raise RuntimeError(f"Unsupported monthly aggregation: {aggregation_method}")\n'
    '\n'
    '    averaged = dict(latest)\n'
    '    averaged["value"] = sum(numeric_values) / len(numeric_values)\n'
    '    averaged["observation_date"] = max(\n'
    '        record["observation_date"] for record in eligible\n'
    '    )\n'
    '    averaged["release_date"] = max(record["release_date"] for record in eligible)\n'
    '    realtime_dates = [\n'
    '        record.get("realtime_start")\n'
    '        for record in eligible\n'
    '        if isinstance(record.get("realtime_start"), date)\n'
    '    ]\n'
    '    averaged["realtime_start"] = max(realtime_dates) if realtime_dates else None\n'
    '    averaged["monthly_average_count"] = len(eligible)\n'
    '    averaged["value_vintage"] = (\n'
    '        f"Monthly average of {len(eligible)} source observations"\n'
    '    )\n'
    '    return averaged\n'
    '\n'
    '\n'
    'def shift_monthly_observations(\n'
    '    raw_observations: list[dict],\n'
    '    months: int,\n'
    '    release_lag_months: int,\n'
    ') -> list[dict]:\n'
    '    """Shift monthly observations into their publication month without duplicating values."""\n'
    '    wanted_months = set(recent_month_starts(months))\n'
    '    shifted = []\n'
    '    for observation in raw_observations:\n'
    '        release_month = add_months(month_start(observation["date"]), release_lag_months)\n'
    '        if release_month in wanted_months:\n'
    '            shifted.append({"date": release_month, "value": observation["value"]})\n'
    '    return shifted\n'
    '\n'
    '\n'
    'def get_monthly_observations(\n'
    '    api_key: str,\n'
    '    series_id: str,\n'
    '    months: int,\n'
    '    units: str = "lin",\n'
    ') -> list[dict]:\n'
    '    metadata = get_fred_series_metadata(api_key, series_id)\n'
    '    today = date.today()\n'
    '    if FULL_HISTORY_MODE:\n'
    '        lookback_start = datetime.strptime(\n'
    '            metadata["observation_start"], "%Y-%m-%d"\n'
    '        ).date()\n'
    '    else:\n'
    '        # Four years covers sparse annual/quarterly releases and pc1 base\n'
    '        # values. Full-history mode uses bounded ALFRED windows instead.\n'
    '        lookback_start = subtract_months(today, max(months + 25, 48))\n'
    '    records = get_fred_observations(\n'
    '        api_key,\n'
    '        series_id,\n'
    '        lookback_start,\n'
    '        today,\n'
    '        units,\n'
    '        metadata["frequency"],\n'
    '    )\n'
    '    validate_release_records(records, series_id)\n'
    '    # This evidence flag, rather than the provider name, authorises removal of\n'
    '    # stale workbook rows during a full authoritative reconciliation.\n'
    '    for record in records:\n'
    '        record["history_complete"] = FULL_HISTORY_MODE\n'
    '    return records\n'
    '\n'
    '\n'
    'def get_indicator_observations(\n'
    '    api_key: str, indicator: Indicator, months: int\n'
    ') -> list[dict]:\n'
    '    if indicator.source == "ism":\n'
    '        return get_ism_indicator_observations(indicator, months)\n'
    '    if indicator.source == "nyfed_sce":\n'
    '        return get_nyfed_sce_indicator_observations(indicator, months)\n'
    '    if indicator.source == "michigan_sca":\n'
    '        return get_michigan_consumer_expectations_observations(indicator, months)\n'
    '    if indicator.source == "michigan_sentiment":\n'
    '        return get_michigan_consumer_sentiment_observations(indicator, months)\n'
    '    if indicator.source == "imf_datamapper":\n'
    '        return get_imf_public_finance_observations(indicator, months)\n'
    '    if indicator.series_id:\n'
    '        return get_monthly_observations(\n'
    '            api_key,\n'
    '            indicator.series_id,\n'
    '            months,\n'
    '            indicator.units,\n'
    '        )\n'
    '    raise RuntimeError(\n'
    '        f"Unsupported indicator source for {indicator.name}: {indicator.source}"\n'
    '    )\n'
    '\n'
    '\n'
    'def get_indicator_observations_safely(\n'
    '    api_key: str,\n'
    '    indicator: Indicator,\n'
    '    months: int,\n'
    ') -> list[dict]:\n'
    '    """Fetch one indicator and reject missing data before it reaches Google Sheets."""\n'
    '    try:\n'
    '        observations = get_indicator_observations(api_key, indicator, months)\n'
    '        for observation in observations:\n'
    '            observation["monthly_aggregation"] = indicator.monthly_aggregation\n'
    '        if not observations and indicator.source not in {"fred", "ism"}:\n'
    '            raise RuntimeError("the source returned no requested monthly values")\n'
    '        if not observations and indicator.source == "fred":\n'
    '            if not indicator.name.startswith("Internal "):\n'
    '                logging.info(\n'
    '                    "%s has no actual FRED observation in the requested months; "\n'
    '                    "leaving those cells blank.",\n'
    '                    indicator.name,\n'
    '                )\n'
    '        return observations\n'
    '    except Exception as exc:\n'
    '        raise RuntimeError(f"{indicator.name} failed validation: {exc}") from exc\n'
    '\n'
    '\n'
    'def get_ism_indicator_observations(indicator: Indicator, months: int) -> list[dict]:\n'
    '    if not indicator.source_column:\n'
    '        raise RuntimeError(f"{indicator.name} needs an ISM column name.")\n'
    '\n'
    '    if months not in _ISM_CACHE:\n'
    '        _ISM_CACHE[months] = scrape_ism_pmi(months=months)\n'
    '\n'
    '    frame = _ISM_CACHE[months]\n'
    '    if frame.empty or indicator.source_column not in frame.columns:\n'
    '        return []\n'
    '\n'
    '    observations = []\n'
    '    for _, row in frame.iterrows():\n'
    '        value = row[indicator.source_column]\n'
    '        if value is None or value != value:\n'
    '            continue\n'
    '        observation_date = datetime.strptime(\n'
    '            str(row["report_month"])[:10],\n'
    '            "%Y-%m-%d",\n'
    '        ).date()\n'
    '        is_services = indicator.source_column.startswith("services_")\n'
    '        publication_column = (\n'
    '            "services_publication_date" if is_services else "pmi_publication_date"\n'
    '        )\n'
    '        publication_value = row.get(publication_column)\n'
    '        if publication_value is not None and publication_value == publication_value:\n'
    '            release_date = datetime.strptime(\n'
    '                str(publication_value)[:10],\n'
    '                "%Y-%m-%d",\n'
    '            ).date()\n'
    '        else:\n'
    '            publication_month = add_months(observation_date, 1)\n'
    '            # Calendar fallback is intentionally conservative to avoid early use.\n'
    '            release_date = publication_month.replace(day=10 if is_services else 7)\n'
    '            VALIDATION_MESSAGES.append(\n'
    '                {\n'
    '                    "severity": "WARNING",\n'
    '                    "indicator": indicator.name,\n'
    '                    "message": (\n'
    '                        f"ISM publication metadata was unavailable for "\n'
    '                        f"{observation_date:%B %Y}; used conservative "\n'
    '                        f"{release_date} fallback."\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '        observations.append(\n'
    '            {\n'
    '                "observation_date": observation_date,\n'
    '                "release_date": release_date,\n'
    '                "realtime_start": None,\n'
    '                "source": "Institute for Supply Management",\n'
    '                "source_series_id": indicator.source_column,\n'
    '                "selection_date": month_start(release_date),\n'
    '                "period_aligned": True,\n'
    '                "value": float(value),\n'
    '            }\n'
    '        )\n'
    '    _EXPLICIT_BLANK_OUTPUT_MONTHS.pop(indicator.name, None)\n'
    '    validate_release_records(observations, indicator.name)\n'
    '    return observations\n'
    '\n'
    '\n'
    'def parse_yyyymm(value: object) -> date | None:\n'
    '    """Parse dates stored as YYYYMM numbers in official chart workbooks."""\n'
    '    if value is None or value != value:\n'
    '        return None\n'
    '    token = str(int(value)) if isinstance(value, (int, float)) else str(value).strip()\n'
    '    if not re.fullmatch(r"\\d{6}", token):\n'
    '        return None\n'
    '    try:\n'
    '        parsed = date(int(token[:4]), int(token[4:]), 1)\n'
    '    except ValueError:\n'
    '        return None\n'
    '    if parsed.year < 2000 or parsed > month_start(date.today()):\n'
    '        return None\n'
    '    return parsed\n'
    '\n'
    '\n'
    'def parse_nyfed_page_latest(html: str) -> tuple[date, float] | None:\n'
    '    """Optionally corroborate the workbook against the NY Fed\'s current summary."""\n'
    '    text = text_from_html(html)\n'
    '    month_match = re.search(\n'
    '        r"\\b(" + "|".join(name.title() for name in MONTHS) + r")\\s+Survey\\b",\n'
    '        text,\n'
    '        flags=re.IGNORECASE,\n'
    '    )\n'
    '    value_match = re.search(\n'
    '        r"inflation expectations.{0,300}?"\n'
    '        r"(?:to|at)\\s+(-?\\d{1,2}(?:\\.\\d+)?)\\s*percent"\n'
    '        r".{0,100}?one-year-ahead",\n'
    '        text,\n'
    '        flags=re.IGNORECASE | re.DOTALL,\n'
    '    )\n'
    '    if not month_match or not value_match:\n'
    '        return None\n'
    '    month_name = canonical_month(month_match.group(1))\n'
    '    month_number = MONTH_NUMBERS[month_name]\n'
    '    current_year = date.today().year\n'
    '    # A December survey can be the latest page during January.\n'
    '    year = (\n'
    '        current_year - 1\n'
    '        if date.today().month == 1 and month_number == 12\n'
    '        else current_year\n'
    '    )\n'
    '    return date(year, month_number, 1), float(value_match.group(1))\n'
    '\n'
    '\n'
    'def get_nyfed_sce_frame() -> pd.DataFrame:\n'
    '    """Download the official NY Fed SCE chart data workbook."""\n'
    '    global _NYFED_SCE_CACHE\n'
    '    if _NYFED_SCE_CACHE is not None:\n'
    '        return _NYFED_SCE_CACHE\n'
    '\n'
    '    session = make_requests_session()\n'
    '    page_from_archive = False\n'
    '    try:\n'
    '        page_response = fetch_url(\n'
    '            session,\n'
    '            NYFED_SCE_PAGE_URL,\n'
    '            pause=0,\n'
    '            minimum_bytes=MIN_HTML_BYTES,\n'
    '        )\n'
    '    except requests.RequestException:\n'
    '        archived_page = load_latest_archived_source("NY Fed SCE Page")\n'
    '        if archived_page is None:\n'
    '            raise\n'
    '        page_response = response_from_archive(*archived_page)\n'
    '        page_from_archive = True\n'
    '        logging.info("Using checksum-verified archive for the NY Fed SCE page.")\n'
    '    validate_response_content_type(\n'
    '        page_response,\n'
    '        "NY Fed SCE page",\n'
    '        ("text/html", "application/xhtml"),\n'
    '    )\n'
    '    discovered_url = discover_download_url_from_html(\n'
    '        page_response.text,\n'
    '        NYFED_SCE_PAGE_URL,\n'
    '        href_pattern=r"frbny-sce-data\\.xlsx",\n'
    '        link_text_pattern=r"Chart Data",\n'
    '    )\n'
    '    workbook_from_archive = False\n'
    '    try:\n'
    '        response = fetch_first_available(\n'
    '            session,\n'
    '            [url for url in [discovered_url, NYFED_SCE_DATA_URL] if url],\n'
    '            "NY Fed SCE chart data workbook",\n'
    '        )\n'
    '    except RuntimeError:\n'
    '        archived_workbook = load_latest_archived_source("NY Fed SCE Workbook")\n'
    '        if archived_workbook is None:\n'
    '            raise\n'
    '        response = response_from_archive(*archived_workbook)\n'
    '        workbook_from_archive = True\n'
    '        logging.info("Using checksum-verified archive for the NY Fed SCE workbook.")\n'
    '    validate_response_content_type(\n'
    '        response,\n'
    '        "NY Fed SCE workbook",\n'
    '        (\n'
    '            "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",\n'
    '            "application/vnd.ms-excel",\n'
    '            "application/octet-stream",\n'
    '            "application/zip",\n'
    '            "application/x-zip",\n'
    '        ),\n'
    '    )\n'
    '    validate_excel_signature(response.content, "NY Fed SCE")\n'
    '    try:\n'
    '        workbook = pd.ExcelFile(BytesIO(response.content))\n'
    '        if "Inflation expectations" not in workbook.sheet_names:\n'
    '            raise RuntimeError(\n'
    '                "NY Fed SCE workbook format changed; "\n'
    '                "missing \'Inflation expectations\' worksheet."\n'
    '            )\n'
    '        frame = workbook.parse("Inflation expectations", header=3)\n'
    '    except Exception as exc:\n'
    '        if isinstance(exc, RuntimeError):\n'
    '            raise\n'
    '        raise RuntimeError(\n'
    '            "Could not read the NY Fed SCE inflation expectations workbook."\n'
    '        ) from exc\n'
    '\n'
    '    if frame.shape[1] < 3:\n'
    '        raise RuntimeError("NY Fed SCE workbook has too few columns.")\n'
    '    frame = frame.rename(columns={frame.columns[0]: "date"})\n'
    '    frame["date"] = frame["date"].apply(parse_yyyymm)\n'
    '    frame = frame.dropna(subset=["date"])\n'
    '    required_columns = {"Median one-year ahead expected inflation rate"}\n'
    '    missing_columns = required_columns - set(frame.columns)\n'
    '    if missing_columns:\n'
    '        raise RuntimeError(\n'
    '            "NY Fed SCE workbook format changed; missing columns: "\n'
    '            + ", ".join(sorted(missing_columns))\n'
    '        )\n'
    '    if frame.empty:\n'
    '        raise RuntimeError(\n'
    '            "NY Fed SCE workbook did not contain parseable monthly rows."\n'
    '        )\n'
    '    frame = frame.sort_values("date").reset_index(drop=True)\n'
    '    validate_monthly_dates(frame, "NY Fed SCE")\n'
    '    for column in required_columns:\n'
    '        validate_numeric_column(frame, column, "NY Fed SCE", -20, 50)\n'
    '\n'
    '    page_latest = parse_nyfed_page_latest(page_response.text)\n'
    '    if page_latest is not None:\n'
    '        page_month, page_value = page_latest\n'
    '        workbook_latest = frame.iloc[-1]\n'
    '        workbook_month = workbook_latest["date"]\n'
    '        workbook_value = float(\n'
    '            workbook_latest["Median one-year ahead expected inflation rate"]\n'
    '        )\n'
    '        if page_month != workbook_month or abs(page_value - workbook_value) > 0.051:\n'
    '            raise RuntimeError(\n'
    '                "NY Fed SCE webpage and workbook disagree on the latest "\n'
    '                f"one-year inflation expectation: page={page_month} {page_value}, "\n'
    '                f"workbook={workbook_month} {workbook_value}."\n'
    '            )\n'
    '\n'
    '    latest_period = frame.iloc[-1]["date"].isoformat()\n'
    '    latest_values = {\n'
    '        column: float(frame.iloc[-1][column]) for column in required_columns\n'
    '    }\n'
    '    revision_values = {\n'
    '        f"{row[\'date\'].isoformat()}:{column}": float(row[column])\n'
    '        for _, row in frame.tail(24).iterrows()\n'
    '        for column in required_columns\n'
    '    }\n'
    '    if not page_from_archive:\n'
    '        archive_validated_source(\n'
    '            "NY Fed SCE Page",\n'
    '            latest_period,\n'
    '            page_response,\n'
    '            "html",\n'
    '            ("survey_month", "one_year_inflation_expectation"),\n'
    '            {\n'
    '                "survey_month": page_latest[0] if page_latest else latest_period,\n'
    '                "one_year_inflation_expectation": (\n'
    '                    page_latest[1]\n'
    '                    if page_latest\n'
    '                    else latest_values["Median one-year ahead expected inflation rate"]\n'
    '                ),\n'
    '            },\n'
    '        )\n'
    '    if not workbook_from_archive:\n'
    '        archive_validated_source(\n'
    '            "NY Fed SCE Workbook",\n'
    '            latest_period,\n'
    '            response,\n'
    '            "xlsx",\n'
    '            frame.columns,\n'
    '            revision_values,\n'
    '        )\n'
    '    else:\n'
    '        archived_workbook = load_latest_archived_source("NY Fed SCE Workbook")\n'
    '        if archived_workbook is not None:\n'
    '            _, metadata = archived_workbook\n'
    '            set_source_status(\n'
    '                SourceStatus(\n'
    '                    source="NY Fed SCE Workbook",\n'
    '                    status="OK (archive)",\n'
    '                    latest_data_month=metadata.get("period", ""),\n'
    '                    retrieved_at=metadata.get("retrieved_at", ""),\n'
    '                    source_url=metadata.get("final_url", ""),\n'
    '                    sha256=metadata.get("sha256", ""),\n'
    '                    byte_size=int(metadata.get("byte_size", 0)),\n'
    '                    content_type=metadata.get("content_type", ""),\n'
    '                    schema_fingerprint=metadata.get("schema_fingerprint", ""),\n'
    '                    archive_file=metadata.get("archive_file", ""),\n'
    '                    revision=metadata.get("revision", ""),\n'
    '                )\n'
    '            )\n'
    '\n'
    '    logging.info(\n'
    '        "Validated NY Fed SCE workbook through %s from %s.",\n'
    '        frame.iloc[-1]["date"],\n'
    '        response.url,\n'
    '    )\n'
    '    frame.attrs["latest_publication_date"] = parse_html_publication_date(\n'
    '        page_response.text\n'
    '    )\n'
    '    _NYFED_SCE_CACHE = frame\n'
    '    return frame\n'
    '\n'
    '\n'
    'def get_nyfed_sce_indicator_observations(\n'
    '    indicator: Indicator, months: int\n'
    ') -> list[dict]:\n'
    '    """Return monthly NY Fed SCE inflation expectation observations."""\n'
    '    if not indicator.source_column:\n'
    '        raise RuntimeError(f"{indicator.name} needs a NY Fed SCE column name.")\n'
    '\n'
    '    frame = get_nyfed_sce_frame()\n'
    '    if indicator.source_column not in frame.columns:\n'
    '        return []\n'
    '\n'
    '    ordered = frame.sort_values("date")\n'
    '    recent = (\n'
    '        ordered\n'
    '        if FULL_HISTORY_MODE\n'
    '        else ordered.tail(months + indicator.release_lag_months)\n'
    '    )\n'
    '    observations = []\n'
    '    latest_observation_date = frame.iloc[-1]["date"]\n'
    '    latest_publication_date = frame.attrs.get("latest_publication_date")\n'
    '    for _, row in recent.iterrows():\n'
    '        value = row[indicator.source_column]\n'
    '        if value is None or value != value:\n'
    '            continue\n'
    '        release_date = add_months(row["date"], 1).replace(day=20)\n'
    '        if (\n'
    '            row["date"] == latest_observation_date\n'
    '            and isinstance(latest_publication_date, date)\n'
    '            and latest_publication_date >= row["date"]\n'
    '        ):\n'
    '            release_date = latest_publication_date\n'
    '        observations.append(\n'
    '            {\n'
    '                "observation_date": row["date"],\n'
    '                "release_date": release_date,\n'
    '                "realtime_start": None,\n'
    '                "source": "Federal Reserve Bank of New York SCE",\n'
    '                "source_series_id": indicator.source_column,\n'
    '                "selection_date": add_months(\n'
    '                    month_start(row["date"]),\n'
    '                    indicator.release_lag_months,\n'
    '                ),\n'
    '                "period_aligned": True,\n'
    '                "history_complete": FULL_HISTORY_MODE,\n'
    '                "value": float(value),\n'
    '            }\n'
    '        )\n'
    '    validate_release_records(observations, "NY Fed SCE")\n'
    '    validate_release_freshness(\n'
    '        month_start(observations[-1]["release_date"]),\n'
    '        "NY Fed SCE",\n'
    '        current_month_deadline_day=20,\n'
    '    )\n'
    '    return observations\n'
    '\n'
    '\n'
    'def get_michigan_consumer_expectations_frame() -> pd.DataFrame:\n'
    '    """Download the official Michigan SCA Index of Consumer Expectations workbook."""\n'
    '    global _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE\n'
    '    if _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE is not None:\n'
    '        return _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE\n'
    '\n'
    '    session = make_requests_session()\n'
    '    page_from_archive = False\n'
    '    try:\n'
    '        page_response = fetch_url(\n'
    '            session,\n'
    '            MICHIGAN_CHARTS_PAGE_URL,\n'
    '            pause=0,\n'
    '            minimum_bytes=MIN_HTML_BYTES,\n'
    '        )\n'
    '    except requests.RequestException:\n'
    '        archived_page = load_latest_archived_source("Michigan SCA Page")\n'
    '        if archived_page is None:\n'
    '            raise\n'
    '        page_response = response_from_archive(*archived_page)\n'
    '        page_from_archive = True\n'
    '        logging.info("Using checksum-verified archive for the Michigan SCA page.")\n'
    '    validate_response_content_type(\n'
    '        page_response,\n'
    '        "University of Michigan SCA page",\n'
    '        ("text/html", "application/xhtml"),\n'
    '    )\n'
    '    discovered_url = discover_download_url_from_html(\n'
    '        page_response.text,\n'
    '        MICHIGAN_CHARTS_PAGE_URL,\n'
    '        href_pattern=r"chicer\\.xls",\n'
    '    )\n'
    '    workbook_from_archive = False\n'
    '    try:\n'
    '        response = fetch_first_available(\n'
    '            session,\n'
    '            [\n'
    '                url\n'
    '                for url in [discovered_url, MICHIGAN_CONSUMER_EXPECTATIONS_URL]\n'
    '                if url\n'
    '            ],\n'
    '            "University of Michigan consumer expectations workbook",\n'
    '        )\n'
    '    except RuntimeError:\n'
    '        archived_workbook = load_latest_archived_source("Michigan SCA Workbook")\n'
    '        if archived_workbook is None:\n'
    '            raise\n'
    '        response = response_from_archive(*archived_workbook)\n'
    '        workbook_from_archive = True\n'
    '        logging.info("Using checksum-verified archive for the Michigan SCA workbook.")\n'
    '    validate_response_content_type(\n'
    '        response,\n'
    '        "University of Michigan SCA workbook",\n'
    '        (\n'
    '            "application/vnd.ms-excel",\n'
    '            "application/octet-stream",\n'
    '            "application/xls",\n'
    '            "application/x-msexcel",\n'
    '            "application/x-ole-storage",\n'
    '            "application/vnd.ms-office",\n'
    '        ),\n'
    '    )\n'
    '    validate_excel_signature(\n'
    '        response.content,\n'
    '        "University of Michigan SCA",\n'
    '        legacy_xls=True,\n'
    '    )\n'
    '    try:\n'
    '        workbook = pd.ExcelFile(BytesIO(response.content), engine="xlrd")\n'
    '        if "Data" not in workbook.sheet_names:\n'
    '            raise RuntimeError(\n'
    '                "Michigan workbook format changed; missing \'Data\' worksheet."\n'
    '            )\n'
    '        raw = workbook.parse("Data", header=None)\n'
    '    except ImportError as exc:\n'
    '        raise RuntimeError(\n'
    '            "Missing Michigan Excel dependency. Re-run the script so it can install xlrd automatically."\n'
    '        ) from exc\n'
    '    except Exception as exc:\n'
    '        if isinstance(exc, RuntimeError):\n'
    '            raise\n'
    '        raise RuntimeError(\n'
    '            "Could not read the Michigan consumer expectations workbook."\n'
    '        ) from exc\n'
    '\n'
    '    if raw.shape[1] < 2:\n'
    '        raise RuntimeError("Michigan workbook has too few columns.")\n'
    '    rows = []\n'
    '    for _, row in raw.iterrows():\n'
    '        item_date = pd.to_datetime(row.iloc[0], errors="coerce")\n'
    '        value = pd.to_numeric(row.iloc[1], errors="coerce")\n'
    '        if pd.isna(item_date) or pd.isna(value):\n'
    '            continue\n'
    '        if item_date.year < 1950 or item_date.year > date.today().year:\n'
    '            continue\n'
    '        rows.append(\n'
    '            {\n'
    '                "date": date(item_date.year, item_date.month, 1),\n'
    '                "ICE_M": float(value),\n'
    '            }\n'
    '        )\n'
    '\n'
    '    if not rows:\n'
    '        raise RuntimeError(\n'
    '            "Michigan workbook did not contain parseable consumer expectations rows."\n'
    '        )\n'
    '\n'
    '    _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE = (\n'
    '        pd.DataFrame(rows).sort_values("date").reset_index(drop=True)\n'
    '    )\n'
    '    validate_monthly_dates(\n'
    '        _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE,\n'
    '        "University of Michigan SCA",\n'
    '    )\n'
    '    validate_numeric_column(\n'
    '        _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE,\n'
    '        "ICE_M",\n'
    '        "University of Michigan SCA",\n'
    '        0,\n'
    '        200,\n'
    '    )\n'
    '    latest_period = _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE.iloc[-1]["date"].isoformat()\n'
    '    revision_values = {\n'
    '        row["date"].isoformat(): float(row["ICE_M"])\n'
    '        for _, row in _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE.tail(24).iterrows()\n'
    '    }\n'
    '    if not page_from_archive:\n'
    '        archive_validated_source(\n'
    '            "Michigan SCA Page",\n'
    '            latest_period,\n'
    '            page_response,\n'
    '            "html",\n'
    '            ("Index of Consumer Expectations",),\n'
    '            {"latest_data_month": latest_period},\n'
    '        )\n'
    '    if not workbook_from_archive:\n'
    '        archive_validated_source(\n'
    '            "Michigan SCA Workbook",\n'
    '            latest_period,\n'
    '            response,\n'
    '            "xls",\n'
    '            ("date", "ICE_M"),\n'
    '            revision_values,\n'
    '        )\n'
    '    else:\n'
    '        archived_workbook = load_latest_archived_source("Michigan SCA Workbook")\n'
    '        if archived_workbook is not None:\n'
    '            _, metadata = archived_workbook\n'
    '            set_source_status(\n'
    '                SourceStatus(\n'
    '                    source="Michigan SCA Workbook",\n'
    '                    status="OK (archive)",\n'
    '                    latest_data_month=metadata.get("period", ""),\n'
    '                    retrieved_at=metadata.get("retrieved_at", ""),\n'
    '                    source_url=metadata.get("final_url", ""),\n'
    '                    sha256=metadata.get("sha256", ""),\n'
    '                    byte_size=int(metadata.get("byte_size", 0)),\n'
    '                    content_type=metadata.get("content_type", ""),\n'
    '                    schema_fingerprint=metadata.get("schema_fingerprint", ""),\n'
    '                    archive_file=metadata.get("archive_file", ""),\n'
    '                    revision=metadata.get("revision", ""),\n'
    '                )\n'
    '            )\n'
    '    logging.info(\n'
    '        "Validated University of Michigan SCA workbook through %s from %s.",\n'
    '        _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE.iloc[-1]["date"],\n'
    '        response.url,\n'
    '    )\n'
    '    return _MICHIGAN_CONSUMER_EXPECTATIONS_CACHE\n'
    '\n'
    '\n'
    'def get_michigan_consumer_expectations_observations(\n'
    '    indicator: Indicator,\n'
    '    months: int,\n'
    ') -> list[dict]:\n'
    '    """Return monthly University of Michigan consumer expectations values."""\n'
    '    column = indicator.source_column or "ICE_M"\n'
    '    frame = get_michigan_consumer_expectations_frame()\n'
    '    if frame.empty or column not in frame.columns:\n'
    '        return []\n'
    '\n'
    '    ordered = frame.sort_values("date")\n'
    '    recent = ordered if FULL_HISTORY_MODE else ordered.tail(months)\n'
    '    observations = []\n'
    '    for _, row in recent.iterrows():\n'
    '        value = row[column]\n'
    '        if value is None or value != value:\n'
    '            continue\n'
    '        observation_date = row["date"]\n'
    '        final_release = last_weekday_of_month(observation_date, 4)\n'
    '        release_date = (\n'
    '            min(date.today(), final_release)\n'
    '            if month_start(observation_date) == month_start(date.today())\n'
    '            else final_release\n'
    '        )\n'
    '        observations.append(\n'
    '            {\n'
    '                "observation_date": observation_date,\n'
    '                "release_date": release_date,\n'
    '                "realtime_start": None,\n'
    '                "source": "University of Michigan Surveys of Consumers",\n'
    '                "source_series_id": column,\n'
    '                "value": float(value),\n'
    '            }\n'
    '        )\n'
    '    validate_release_records(observations, "University of Michigan SCA")\n'
    '    validate_release_freshness(\n'
    '        month_start(observations[-1]["release_date"]),\n'
    '        "University of Michigan SCA",\n'
    '        current_month_deadline_day=20,\n'
    '    )\n'
    '    return observations\n'
    '\n'
    '\n'
    'def get_michigan_consumer_sentiment_frame() -> pd.DataFrame:\n'
    '    """Download the official Michigan Consumer Sentiment Index workbook."""\n'
    '    global _MICHIGAN_CONSUMER_SENTIMENT_CACHE\n'
    '    if _MICHIGAN_CONSUMER_SENTIMENT_CACHE is not None:\n'
    '        return _MICHIGAN_CONSUMER_SENTIMENT_CACHE\n'
    '\n'
    '    session = make_requests_session()\n'
    '    page_from_archive = False\n'
    '    try:\n'
    '        page_response = fetch_url(\n'
    '            session,\n'
    '            MICHIGAN_CHARTS_PAGE_URL,\n'
    '            pause=0,\n'
    '            minimum_bytes=MIN_HTML_BYTES,\n'
    '        )\n'
    '    except requests.RequestException:\n'
    '        archived_page = load_latest_archived_source("Michigan SCA Page")\n'
    '        if archived_page is None:\n'
    '            raise\n'
    '        page_response = response_from_archive(*archived_page)\n'
    '        page_from_archive = True\n'
    '        logging.info("Using checksum-verified archive for the Michigan SCA page.")\n'
    '    validate_response_content_type(\n'
    '        page_response,\n'
    '        "University of Michigan SCA page",\n'
    '        ("text/html", "application/xhtml"),\n'
    '    )\n'
    '    discovered_url = discover_download_url_from_html(\n'
    '        page_response.text,\n'
    '        MICHIGAN_CHARTS_PAGE_URL,\n'
    '        href_pattern=r"chicsr\\.xls",\n'
    '    )\n'
    '    workbook_from_archive = False\n'
    '    try:\n'
    '        response = fetch_first_available(\n'
    '            session,\n'
    '            [url for url in [discovered_url, MICHIGAN_CONSUMER_SENTIMENT_URL] if url],\n'
    '            "University of Michigan consumer sentiment workbook",\n'
    '        )\n'
    '    except RuntimeError:\n'
    '        archived_workbook = load_latest_archived_source(\n'
    '            "Michigan Consumer Sentiment Workbook"\n'
    '        )\n'
    '        if archived_workbook is None:\n'
    '            raise\n'
    '        response = response_from_archive(*archived_workbook)\n'
    '        workbook_from_archive = True\n'
    '        logging.info(\n'
    '            "Using checksum-verified archive for the Michigan sentiment workbook."\n'
    '        )\n'
    '    validate_response_content_type(\n'
    '        response,\n'
    '        "University of Michigan sentiment workbook",\n'
    '        (\n'
    '            "application/vnd.ms-excel",\n'
    '            "application/octet-stream",\n'
    '            "application/xls",\n'
    '            "application/x-msexcel",\n'
    '            "application/x-ole-storage",\n'
    '            "application/vnd.ms-office",\n'
    '        ),\n'
    '    )\n'
    '    validate_excel_signature(\n'
    '        response.content,\n'
    '        "University of Michigan Consumer Sentiment",\n'
    '        legacy_xls=True,\n'
    '    )\n'
    '    try:\n'
    '        workbook = pd.ExcelFile(BytesIO(response.content), engine="xlrd")\n'
    '        if "Data" not in workbook.sheet_names:\n'
    '            raise RuntimeError("Michigan sentiment workbook is missing \'Data\'.")\n'
    '        raw = workbook.parse("Data", header=None)\n'
    '    except Exception as exc:\n'
    '        if isinstance(exc, RuntimeError):\n'
    '            raise\n'
    '        raise RuntimeError("Could not read the Michigan sentiment workbook.") from exc\n'
    '\n'
    '    rows = []\n'
    '    for _, row in raw.iterrows():\n'
    '        item_date = pd.to_datetime(row.iloc[0], errors="coerce")\n'
    '        value = pd.to_numeric(row.iloc[1], errors="coerce")\n'
    '        if pd.isna(item_date) or pd.isna(value):\n'
    '            continue\n'
    '        rows.append(\n'
    '            {\n'
    '                "date": date(item_date.year, item_date.month, 1),\n'
    '                "ICS_M": float(value),\n'
    '            }\n'
    '        )\n'
    '    frame = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)\n'
    '    if frame.empty:\n'
    '        raise RuntimeError("Michigan sentiment workbook contained no monthly rows.")\n'
    '    validate_monthly_dates(frame, "University of Michigan Consumer Sentiment")\n'
    '    validate_numeric_column(frame, "ICS_M", "Michigan Consumer Sentiment", 0, 200)\n'
    '    latest_period = frame.iloc[-1]["date"].isoformat()\n'
    '    if not page_from_archive:\n'
    '        archive_validated_source(\n'
    '            "Michigan SCA Page",\n'
    '            latest_period,\n'
    '            page_response,\n'
    '            "html",\n'
    '            ("Consumer Sentiment Index",),\n'
    '            {"latest_sentiment_data_month": latest_period},\n'
    '        )\n'
    '    if not workbook_from_archive:\n'
    '        archive_validated_source(\n'
    '            "Michigan Consumer Sentiment Workbook",\n'
    '            latest_period,\n'
    '            response,\n'
    '            "xls",\n'
    '            ("date", "ICS_M"),\n'
    '            {\n'
    '                row["date"].isoformat(): float(row["ICS_M"])\n'
    '                for _, row in frame.tail(24).iterrows()\n'
    '            },\n'
    '        )\n'
    '    else:\n'
    '        archived_workbook = load_latest_archived_source(\n'
    '            "Michigan Consumer Sentiment Workbook"\n'
    '        )\n'
    '        if archived_workbook is not None:\n'
    '            _, metadata = archived_workbook\n'
    '            set_source_status(\n'
    '                SourceStatus(\n'
    '                    source="Michigan Consumer Sentiment Workbook",\n'
    '                    status="OK (archive)",\n'
    '                    latest_data_month=metadata.get("period", ""),\n'
    '                    retrieved_at=metadata.get("retrieved_at", ""),\n'
    '                    source_url=metadata.get("final_url", ""),\n'
    '                    sha256=metadata.get("sha256", ""),\n'
    '                    byte_size=int(metadata.get("byte_size", 0)),\n'
    '                    content_type=metadata.get("content_type", ""),\n'
    '                    schema_fingerprint=metadata.get("schema_fingerprint", ""),\n'
    '                    archive_file=metadata.get("archive_file", ""),\n'
    '                    revision=metadata.get("revision", ""),\n'
    '                )\n'
    '            )\n'
    '    _MICHIGAN_CONSUMER_SENTIMENT_CACHE = frame\n'
    '    return frame\n'
    '\n'
    '\n'
    'def get_michigan_consumer_sentiment_observations(\n'
    '    indicator: Indicator,\n'
    '    months: int,\n'
    ') -> list[dict]:\n'
    '    frame = get_michigan_consumer_sentiment_frame()\n'
    '    recent = frame if FULL_HISTORY_MODE else frame.tail(months)\n'
    '    records = []\n'
    '    for _, row in recent.iterrows():\n'
    '        observation_date = row["date"]\n'
    '        final_release = last_weekday_of_month(observation_date, 4)\n'
    '        release_date = (\n'
    '            min(date.today(), final_release)\n'
    '            if month_start(observation_date) == month_start(date.today())\n'
    '            else final_release\n'
    '        )\n'
    '        records.append(\n'
    '            {\n'
    '                "observation_date": observation_date,\n'
    '                "release_date": release_date,\n'
    '                "realtime_start": None,\n'
    '                "source": "University of Michigan Surveys of Consumers",\n'
    '                "source_series_id": indicator.source_column or "ICS_M",\n'
    '                "value": float(row[indicator.source_column or "ICS_M"]),\n'
    '            }\n'
    '        )\n'
    '    validate_release_records(records, "Michigan Consumer Sentiment")\n'
    '    return records\n'
    '\n'
    '\n'
    'def check_non_api_sources(months: int) -> None:\n'
    '    """Smoke test the official scraped/downloaded sources."""\n'
    '\n'
    '    def load_and_cache_ism() -> pd.DataFrame:\n'
    '        warning_start = len(VALIDATION_MESSAGES)\n'
    '        frame = scrape_ism_pmi(months)\n'
    '        _ISM_CACHE[months] = frame\n'
    '        for warning in VALIDATION_MESSAGES[warning_start:]:\n'
    '            print(f"  ISM: {warning[\'message\']}", flush=True)\n'
    '        return frame\n'
    '\n'
    '    nyfed_indicator = next(item for item in INDICATORS if item.source == "nyfed_sce")\n'
    '    michigan_indicator = next(\n'
    '        item for item in INDICATORS if item.source == "michigan_sca"\n'
    '    )\n'
    '    michigan_sentiment_indicator = next(\n'
    '        item for item in INDICATORS if item.source == "michigan_sentiment"\n'
    '    )\n'
    '    checks = [\n'
    '        ("ISM PMI", load_and_cache_ism),\n'
    '        (\n'
    '            "NY Fed SCE",\n'
    '            lambda: pd.DataFrame(\n'
    '                get_nyfed_sce_indicator_observations(nyfed_indicator, months)\n'
    '            ),\n'
    '        ),\n'
    '        (\n'
    '            "Michigan SCA",\n'
    '            lambda: pd.DataFrame(\n'
    '                get_michigan_consumer_expectations_observations(\n'
    '                    michigan_indicator,\n'
    '                    months,\n'
    '                )\n'
    '            ),\n'
    '        ),\n'
    '        (\n'
    '            "Michigan Sentiment",\n'
    '            lambda: pd.DataFrame(\n'
    '                get_michigan_consumer_sentiment_observations(\n'
    '                    michigan_sentiment_indicator,\n'
    '                    months,\n'
    '                )\n'
    '            ),\n'
    '        ),\n'
    '    ]\n'
    '    failures = []\n'
    '    for name, loader in checks:\n'
    '        try:\n'
    '            frame = loader()\n'
    '            if getattr(frame, "empty", False):\n'
    '                raise RuntimeError("no rows parsed")\n'
    '            print(f"{name}: OK ({len(frame)} rows)")\n'
    '        except Exception as exc:\n'
    '            print(f"{name}: FAILED ({exc})")\n'
    '            failures.append(f"{name}: {exc}")\n'
    '    if failures:\n'
    '        raise RuntimeError("Non-API source check failed: " + " | ".join(failures))\n'
    '\n'
    '\n'
    'def build_table(\n'
    '    indicators: list[tuple[str, list[dict]]],\n'
    ') -> tuple[list[str], list[list[object]]]:\n'
    '    month_dates = [\n'
    '        month_end(item) for item in recent_month_starts(CURRENT_OUTPUT_MONTHS)\n'
    '    ]\n'
    '    headers = ["Date"] + [title for title, _ in indicators]\n'
    '    rows = []\n'
    '    LAST_SELECTIONS.clear()\n'
    '\n'
    '    for month_date in month_dates:\n'
    '        row = [month_date]\n'
    '        for title, observations in indicators:\n'
    '            selected = latest_available_record(observations, month_date)\n'
    '            LAST_SELECTIONS[(title, month_date)] = selected\n'
    '            row.append(analysis_value(selected["value"]) if selected else None)\n'
    '        rows.append(row)\n'
    '\n'
    '    return headers, rows\n'
    '\n'
    '\n'
    'def format_output_month(value: date) -> str:\n'
    '    """Display release months as calendar month-end dates in DD/MM/YYYY format."""\n'
    '    return month_end(value).strftime("%d/%m/%Y")\n'
    '\n'
    '\n'
    'def sheet_values_for_google(\n'
    '    indicators: list[tuple[str, list[dict]]],\n'
    ') -> list[list[object]]:\n'
    '    headers, rows = build_table(indicators)\n'
    '    values = [headers]\n'
    '    for row in rows:\n'
    '        values.append(\n'
    '            [\n'
    '                format_output_month(value) if isinstance(value, date) else value\n'
    '                for value in row\n'
    '            ]\n'
    '        )\n'
    '    return values\n'
    '\n'
    '\n'
    'def validate_non_fred_output_alignment(\n'
    '    indicators: list[Indicator],\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    months: int,\n'
    ') -> None:\n'
    '    """Validate present non-FRED values while allowing months with no release."""\n'
    '    observations_by_name = dict(workbook_data)\n'
    '    expected_dates = [month_end(item) for item in recent_month_starts(months)]\n'
    '    for indicator in indicators:\n'
    '        if indicator.source == "fred":\n'
    '            continue\n'
    '        observations = observations_by_name.get(indicator.name, [])\n'
    '        for row_date in expected_dates:\n'
    '            selected = latest_available_record(observations, row_date)\n'
    '            if selected is None:\n'
    '                continue\n'
    '            if (\n'
    '                selected["release_date"] > row_date\n'
    '                and selected.get("period_aligned") is not True\n'
    '            ):\n'
    '                raise RuntimeError(\n'
    '                    f"{indicator.name} selected a release after {row_date}."\n'
    '                )\n'
    '\n'
    '\n'
    'def mutable_output_months() -> set[date]:\n'
    '    """Months whose latest daily/weekly point legitimately changes during a rerun."""\n'
    '    return {month_start(date.today())}\n'
    '\n'
    '\n'
    'def verify_google_sheet_values(\n'
    '    expected: list[list[object]],\n'
    '    actual: list[list[object]],\n'
    ') -> None:\n'
    '    """Verify every nonblank value after Google Sheets reports a successful write."""\n'
    '    if not actual or actual[0] != expected[0]:\n'
    '        raise RuntimeError("Google Sheets read-back header verification failed.")\n'
    '    if len(actual) < len(expected):\n'
    '        raise RuntimeError(\n'
    '            f"Google Sheets read-back returned {len(actual)} rows; "\n'
    '            f"{len(expected)} were written."\n'
    '        )\n'
    '    for row_index, expected_row in enumerate(expected):\n'
    '        actual_row = actual[row_index] if row_index < len(actual) else []\n'
    '        for column_index, expected_value in enumerate(expected_row):\n'
    '            if expected_value is None:\n'
    '                continue\n'
    '            if column_index >= len(actual_row):\n'
    '                if expected_value == "":\n'
    '                    continue\n'
    '                raise RuntimeError(\n'
    '                    f"Google Sheets read-back is missing row {row_index + 1}, "\n'
    '                    f"column {column_index + 1}."\n'
    '                )\n'
    '            actual_value = actual_row[column_index]\n'
    '            if google_values_equivalent(expected_value, actual_value):\n'
    '                continue\n'
    '            if column_index == 0:\n'
    '                expected_date = normalise_observation_date(expected_value)\n'
    '                actual_date = normalise_observation_date(actual_value)\n'
    '                if expected_date is not None and expected_date == actual_date:\n'
    '                    continue\n'
    '            header = (\n'
    '                expected[0][column_index]\n'
    '                if column_index < len(expected[0])\n'
    '                else "unknown"\n'
    '            )\n'
    '            raise RuntimeError(\n'
    '                "Google Sheets read-back verification failed at "\n'
    '                f"row {row_index + 1}, column \'{header}\': "\n'
    '                f"wrote {expected_value!r}, read {actual_value!r}."\n'
    '            )\n'
    '\n'
    '\n'
    'def google_sheet_table_is_current(\n'
    '    expected: list[list[object]],\n'
    '    actual: list[list[object]],\n'
    ') -> bool:\n'
    '    """Compare complete value tables while tolerating Google date serialization."""\n'
    '    if not expected or not actual or len(expected) != len(actual):\n'
    '        return False\n'
    '    if [normalise_header_name(item) for item in expected[0]] != [\n'
    '        normalise_header_name(item) for item in actual[0]\n'
    '    ]:\n'
    '        return False\n'
    '    for row_index, expected_row in enumerate(expected[1:], start=1):\n'
    '        actual_row = actual[row_index]\n'
    '        for column_index, expected_value in enumerate(expected_row):\n'
    '            actual_value = (\n'
    '                actual_row[column_index] if column_index < len(actual_row) else ""\n'
    '            )\n'
    '            if column_index == 0:\n'
    '                if normalise_observation_date(\n'
    '                    expected_value\n'
    '                ) != normalise_observation_date(actual_value):\n'
    '                    return False\n'
    '                continue\n'
    '            if expected_value in (None, ""):\n'
    '                if actual_value not in (None, ""):\n'
    '                    return False\n'
    '                continue\n'
    '            if not google_values_equivalent(expected_value, actual_value):\n'
    '                return False\n'
    '        if any(value not in (None, "") for value in actual_row[len(expected_row) :]):\n'
    '            return False\n'
    '    return True\n'
    '\n'
    '\n'
    'def _indexed_sheet_rows(\n'
    '    values: list[list[object]],\n'
    '    context: str,\n'
    '    allow_undated_data_rows: bool = False,\n'
    ') -> dict[date, list[object]]:\n'
    '    """Index a date-led table while rejecting malformed or duplicate data rows."""\n'
    '    indexed: dict[date, list[object]] = {}\n'
    '    for row_number, row in enumerate(values[1:], start=2):\n'
    '        raw_date = row[0] if row else None\n'
    '        observed = normalise_observation_date(raw_date)\n'
    '        if observed is None:\n'
    '            if any(item not in (None, "") for item in row[1:]):\n'
    '                if allow_undated_data_rows:\n'
    '                    continue\n'
    '                raise RuntimeError(\n'
    '                    f"{context}: row {row_number} contains data but has no valid date ({raw_date!r})."\n'
    '                )\n'
    '            continue\n'
    '        if observed in indexed:\n'
    '            raise RuntimeError(f"{context}: duplicate month {observed.isoformat()}.")\n'
    '        indexed[observed] = row\n'
    '    return indexed\n'
    '\n'
    '\n'
    'def indicator_input_table_is_current(\n'
    '    existing: list[list[object]],\n'
    '    incoming: list[list[object]],\n'
    ') -> bool:\n'
    '    """Compare an indicator A:B table without relying on Google date rendering."""\n'
    '    if len(existing) != len(incoming) or not existing or not incoming:\n'
    '        return False\n'
    '    if [normalise_header_name(item) for item in existing[0]] != [\n'
    '        normalise_header_name(item) for item in incoming[0]\n'
    '    ]:\n'
    '        return False\n'
    '    for old_row, new_row in zip(existing[1:], incoming[1:]):\n'
    '        old_date = normalise_observation_date(old_row[0] if old_row else None)\n'
    '        new_date = normalise_observation_date(new_row[0] if new_row else None)\n'
    '        if old_date != new_date:\n'
    '            return False\n'
    '        old_value = normalise_value(old_row[1] if len(old_row) > 1 else "")\n'
    '        new_value = normalise_value(new_row[1] if len(new_row) > 1 else "")\n'
    '        if old_value is None or new_value is None:\n'
    '            if old_value is not None or new_value is not None:\n'
    '                return False\n'
    '        elif not google_values_equivalent(old_value, new_value):\n'
    '            return False\n'
    '    return True\n'
    '\n'
    '\n'
    'def validate_historical_table_replacement(\n'
    '    existing: list[list[object]],\n'
    '    incoming: list[list[object]],\n'
    '    context: str,\n'
    '    *,\n'
    '    allow_absent_existing_dates: bool = False,\n'
    '    allow_undated_existing_rows: bool = False,\n'
    '    allow_blank_replacements: bool = False,\n'
    '    allow_value_revisions: bool = False,\n'
    '    mutable_months: set[date] | None = None,\n'
    ') -> None:\n'
    '    """Protect history while allowing explicit cleanup and authoritative revisions."""\n'
    '    date_headers = {\n'
    '        "date",\n'
    '        "observation date",\n'
    '        "month",\n'
    '        "month-end date",\n'
    '        "month end date",\n'
    '        "period",\n'
    '    }\n'
    '    if (\n'
    '        not incoming\n'
    '        or not incoming[0]\n'
    '        or normalise_header_name(incoming[0][0]) not in date_headers\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            f"{context}: generated replacement has no valid Date header."\n'
    '        )\n'
    '    if not existing:\n'
    '        return\n'
    '    if [normalise_header_name(item) for item in existing[0]] != [\n'
    '        normalise_header_name(item) for item in incoming[0]\n'
    '    ]:\n'
    '        raise RuntimeError(\n'
    '            f"{context}: refusing overwrite because headers differ; "\n'
    '            f"existing={existing[0]!r}, incoming={incoming[0]!r}."\n'
    '        )\n'
    '    old_rows = _indexed_sheet_rows(\n'
    '        existing,\n'
    '        f"{context} existing table",\n'
    '        allow_undated_data_rows=allow_undated_existing_rows,\n'
    '    )\n'
    '    new_rows = _indexed_sheet_rows(incoming, f"{context} replacement table")\n'
    '    mutable_months = mutable_months or set()\n'
    '    mismatches: list[str] = []\n'
    '    for column_index, header in enumerate(existing[0][1:], start=1):\n'
    '        for observed, old_row in sorted(old_rows.items()):\n'
    '            old_raw = old_row[column_index] if len(old_row) > column_index else ""\n'
    '            old_value = normalise_value(old_raw)\n'
    '            if old_value is None:\n'
    '                continue\n'
    '            if observed in mutable_months:\n'
    '                continue\n'
    '            if observed not in new_rows and allow_absent_existing_dates:\n'
    '                continue\n'
    '            new_row = new_rows.get(observed, [])\n'
    '            new_raw = new_row[column_index] if len(new_row) > column_index else ""\n'
    '            new_value = normalise_value(new_raw)\n'
    '            if new_value is None and allow_blank_replacements:\n'
    '                continue\n'
    '            if new_value is not None and allow_value_revisions:\n'
    '                continue\n'
    '            if new_value is None or not google_values_equivalent(old_value, new_value):\n'
    '                mismatches.append(\n'
    '                    f"{header} {observed.isoformat()}: existing={old_raw!r}, incoming={new_raw!r}"\n'
    '                )\n'
    '    if mismatches:\n'
    '        preview = "; ".join(mismatches[:5])\n'
    '        suffix = f"; plus {len(mismatches) - 5} more" if len(mismatches) > 5 else ""\n'
    '        raise RuntimeError(\n'
    '            f"{context}: historical overlap does not match ({preview}{suffix}). "\n'
    '            "Only genuinely new or previously blank cells may be added."\n'
    '        )\n'
    '\n'
    '\n'
    'def validate_overlapping_indicator_values(\n'
    '    existing: dict[date, dict],\n'
    '    incoming: dict[date, float | None],\n'
    '    context: str,\n'
    '    *,\n'
    '    allow_value_revisions: bool = False,\n'
    '    mutable_months: set[date] | None = None,\n'
    ') -> None:\n'
    '    """Reject unexplained revisions to populated closed months before any write."""\n'
    '    mutable_months = mutable_months or set()\n'
    '    mismatches = []\n'
    '    for observed, new_value in sorted(incoming.items()):\n'
    '        old = existing.get(observed)\n'
    '        old_value = old.get("value") if old else None\n'
    '        if old_value is None:\n'
    '            continue\n'
    '        if not FULL_HISTORY_MODE:\n'
    '            # Incremental imports never revise or clear a populated cell. The\n'
    "            # source's changed/blank value is intentionally ignored below.\n"
    '            continue\n'
    '        if observed in mutable_months:\n'
    '            continue\n'
    '        if new_value is not None and allow_value_revisions:\n'
    '            continue\n'
    '        if new_value is None or not google_values_equivalent(old_value, new_value):\n'
    '            mismatches.append(\n'
    '                f"{observed.isoformat()}: existing={old.get(\'raw_value\')!r}, incoming={new_value!r}"\n'
    '            )\n'
    '    if mismatches:\n'
    '        preview = "; ".join(mismatches[:5])\n'
    '        suffix = f"; plus {len(mismatches) - 5} more" if len(mismatches) > 5 else ""\n'
    '        raise RuntimeError(\n'
    '            f"{context}: historical overlap does not match ({preview}{suffix}). "\n'
    '            "Only new dates or blank historical cells may be written."\n'
    '        )\n'
    '\n'
    '\n'
    'def google_values_equivalent(expected: object, actual: object) -> bool:\n'
    '    """Treat equivalent Google numeric representations as equal."""\n'
    '    if str(expected) == str(actual):\n'
    '        return True\n'
    '    if isinstance(expected, bool) or isinstance(actual, bool):\n'
    '        return expected is actual\n'
    '    try:\n'
    '        expected_number = Decimal(str(expected).strip())\n'
    '        actual_number = Decimal(str(actual).strip())\n'
    '    except (InvalidOperation, ValueError, AttributeError):\n'
    '        return False\n'
    '    difference = abs(expected_number - actual_number)\n'
    '    if difference <= Decimal("0.000000001"):\n'
    '        return True\n'
    '    scale = max(abs(expected_number), abs(actual_number), Decimal("1"))\n'
    '    return difference <= max(Decimal("0.0001"), scale * Decimal("0.00000001"))\n'
    '\n'
    '\n'
    'def column_letter(column_index: int) -> str:\n'
    '    """Convert a zero-based column index to a Google Sheets column letter."""\n'
    '    column_number = column_index + 1\n'
    '    letters = ""\n'
    '    while column_number:\n'
    '        column_number, remainder = divmod(column_number - 1, 26)\n'
    '        letters = chr(65 + remainder) + letters\n'
    '    return letters\n'
    '\n'
    '\n'
    'def quote_sheet_name(sheet_name: str) -> str:\n'
    '    return "\'" + sheet_name.replace("\'", "\'\'") + "\'"\n'
    '\n'
    '\n'
    'def plausible_sheet_year(year: int) -> bool:\n'
    '    """Keep spreadsheet date detection from accepting malformed ancient/future years."""\n'
    '    return 1900 <= year <= date.today().year + 2\n'
    '\n'
    '\n'
    'def normalise_observation_date(value: object) -> date | None:\n'
    '    """Normalise sheet/source dates to the first day of their observation month."""\n'
    '    if value in (None, ""):\n'
    '        return None\n'
    '    if isinstance(value, datetime):\n'
    '        if plausible_sheet_year(value.year):\n'
    '            return date(value.year, value.month, 1)\n'
    '        return None\n'
    '    if isinstance(value, date):\n'
    '        if plausible_sheet_year(value.year):\n'
    '            return date(value.year, value.month, 1)\n'
    '        return None\n'
    '    if isinstance(value, (int, float)) and not isinstance(value, bool):\n'
    '        try:\n'
    '            serial = float(value)\n'
    '            if not 1 <= serial <= 80000:\n'
    '                return None\n'
    '            parsed = date(1899, 12, 30) + timedelta(days=serial)\n'
    '            if plausible_sheet_year(parsed.year):\n'
    '                return date(parsed.year, parsed.month, 1)\n'
    '            return None\n'
    '        except (OverflowError, ValueError, TypeError):\n'
    '            return None\n'
    '\n'
    '    text = str(value).strip()\n'
    '    if not text or text.lower() in {"n/a", "na", "null", "none", "-"}:\n'
    '        return None\n'
    '    iso_date = re.fullmatch(r"(\\d{4})[-/](\\d{1,2})[-/](\\d{1,2})", text)\n'
    '    if iso_date:\n'
    '        year, month, _day = map(int, iso_date.groups())\n'
    '        if plausible_sheet_year(year) and 1 <= month <= 12:\n'
    '            return date(year, month, 1)\n'
    '        return None\n'
    '    month_only = re.fullmatch(r"(\\d{4})[-/](\\d{1,2})", text)\n'
    '    if month_only:\n'
    '        year, month = map(int, month_only.groups())\n'
    '        if plausible_sheet_year(year) and 1 <= month <= 12:\n'
    '            return date(year, month, 1)\n'
    '        return None\n'
    '    try:\n'
    '        parsed = pd.to_datetime(text, errors="coerce", dayfirst=True)\n'
    '        if pd.isna(parsed):\n'
    '            parsed = pd.to_datetime(text, errors="coerce", dayfirst=False)\n'
    '        if pd.isna(parsed) or not plausible_sheet_year(parsed.year):\n'
    '            return None\n'
    '        return date(parsed.year, parsed.month, 1)\n'
    '    except (OverflowError, ValueError, TypeError):\n'
    '        return None\n'
    '\n'
    '\n'
    'def normalise_value(value: object) -> float | None:\n'
    '    """Normalise spreadsheet/source numeric values for comparison."""\n'
    '    if value in (None, ""):\n'
    '        return None\n'
    '    if isinstance(value, bool):\n'
    '        return None\n'
    '    if isinstance(value, (int, float)):\n'
    '        return float(value) if pd.notna(value) else None\n'
    '    text = str(value).strip()\n'
    '    if not text or text.lower() in {"n/a", "na", "null", "none", "-"}:\n'
    '        return None\n'
    '    text = text.replace(",", "").replace("%", "")\n'
    '    try:\n'
    '        return float(Decimal(text))\n'
    '    except (InvalidOperation, ValueError):\n'
    '        return None\n'
    '\n'
    '\n'
    'def analysis_value(value: object) -> float | None:\n'
    '    """Return an analysis-tab value rounded deterministically to five decimals."""\n'
    '    normalised = normalise_value(value)\n'
    '    if normalised is None:\n'
    '        return None\n'
    '    return float(\n'
    '        Decimal(str(normalised)).quantize(\n'
    '            ANALYSIS_VALUE_QUANTUM,\n'
    '            rounding=ROUND_HALF_UP,\n'
    '        )\n'
    '    )\n'
    '\n'
    '\n'
    'def normalise_header_name(value: object) -> str:\n'
    '    return re.sub(r"\\s+", " ", str(value).strip().lower())\n'
    '\n'
    '\n'
    'def get_sheet_layout(\n'
    '    values: list[list[object]],\n'
    '    indicator_name: str | None = None,\n'
    ') -> dict:\n'
    '    """Validate the fixed indicator-sheet layout: column A date, column B value."""\n'
    '    if not values:\n'
    '        return {"date_col": 0, "value_col": 1, "header_row": 0}\n'
    '\n'
    '    first_row = [normalise_header_name(item) for item in values[0]]\n'
    '    if first_row and first_row[0] not in {\n'
    '        "date",\n'
    '        "observation date",\n'
    '        "month",\n'
    '        "month-end date",\n'
    '        "month end date",\n'
    '        "period",\n'
    '    }:\n'
    '        raise RuntimeError("Column A header is not a recognised date header.")\n'
    '    if len(first_row) < 2 or not first_row[1]:\n'
    '        raise RuntimeError("Column B header is missing.")\n'
    '    return {"date_col": 0, "value_col": 1, "header_row": 0}\n'
    '\n'
    '\n'
    'def read_existing_indicator_data(\n'
    '    values: list[list[object]], layout: dict\n'
    ') -> dict[date, dict]:\n'
    '    """Read existing dates from column A and values from column B only."""\n'
    '    existing: dict[date, dict] = {}\n'
    '    for row_index, row in enumerate(values[1:], start=2):\n'
    '        raw_date = row[0] if row else None\n'
    '        observed = normalise_observation_date(raw_date)\n'
    '        if observed is None:\n'
    '            continue\n'
    '        raw_value = row[1] if len(row) > 1 else None\n'
    '        existing[observed] = {\n'
    '            "row": row_index,\n'
    '            "value": normalise_value(raw_value),\n'
    '            "raw_value": raw_value,\n'
    '        }\n'
    '    return existing\n'
    '\n'
    '\n'
    'def monthly_records_for_indicator(records: list[dict]) -> dict[date, float]:\n'
    '    """Aggregate incoming records by source month using their declared method."""\n'
    '    latest_by_day: dict[date, dict] = {}\n'
    '    for record in records:\n'
    '        observed = record.get("observation_date")\n'
    '        value = normalise_value(record.get("value"))\n'
    '        if not isinstance(observed, date) or value is None:\n'
    '            continue\n'
    '        day_key = observed\n'
    '        current = latest_by_day.get(day_key)\n'
    '        if current is None or (\n'
    '            record.get("release_date", date.min),\n'
    '            record.get("realtime_start", date.min) or date.min,\n'
    '        ) >= (\n'
    '            current.get("release_date", date.min),\n'
    '            current.get("realtime_start", date.min) or date.min,\n'
    '        ):\n'
    '            latest_by_day[day_key] = {**record, "value": value}\n'
    '\n'
    '    grouped: dict[date, list[dict]] = {}\n'
    '    for record in latest_by_day.values():\n'
    '        month_key = normalise_observation_date(record["observation_date"])\n'
    '        if month_key is not None:\n'
    '            grouped.setdefault(month_key, []).append(record)\n'
    '\n'
    '    aggregated: dict[date, float] = {}\n'
    '    for month_key, month_records in grouped.items():\n'
    '        methods = {\n'
    '            str(record.get("monthly_aggregation", "latest")) for record in month_records\n'
    '        }\n'
    '        if len(methods) != 1:\n'
    '            raise RuntimeError("Monthly aggregation received conflicting methods.")\n'
    '        method = next(iter(methods))\n'
    '        if method == "mean":\n'
    '            aggregated[month_key] = sum(\n'
    '                float(record["value"]) for record in month_records\n'
    '            ) / len(month_records)\n'
    '        elif method == "latest":\n'
    '            latest = max(\n'
    '                month_records,\n'
    '                key=lambda record: (\n'
    '                    record["observation_date"],\n'
    '                    record.get("release_date", date.min),\n'
    '                ),\n'
    '            )\n'
    '            aggregated[month_key] = float(latest["value"])\n'
    '        else:\n'
    '            raise RuntimeError(f"Unsupported monthly aggregation: {method}")\n'
    '    return aggregated\n'
    '\n'
    '\n'
    'def uses_sparse_source_dates(records: list[dict]) -> bool:\n'
    '    """Return true for sources whose valid output dates are not a rolling monthly window."""\n'
    '    if not records:\n'
    '        return False\n'
    '    first = records[0]\n'
    '    if first.get("source") == "IMF DataMapper FPP":\n'
    '        return True\n'
    '    if first.get("source") != "FRED / ALFRED":\n'
    '        return False\n'
    '    frequency = str(first.get("native_frequency", ""))\n'
    '    return frequency.startswith(SPARSE_FRED_FREQUENCY_PREFIXES)\n'
    '\n'
    '\n'
    'def has_complete_source_history(records: list[dict]) -> bool:\n'
    '    """Return true only when the fetch path proved it supplied full history."""\n'
    '    if not records:\n'
    '        return False\n'
    '    # A provider can return a syntactically valid but partial response. Never\n'
    '    # infer destructively safe coverage from its name alone. Limited-history\n'
    '    # sources such as ISM and Michigan are never authoritative for deletion.\n'
    '    complete_sources = {\n'
    '        "FRED / ALFRED",\n'
    '        "IMF DataMapper FPP",\n'
    '        "Federal Reserve Bank of New York SCE",\n'
    '    }\n'
    '    return records[0].get("source") in complete_sources and all(\n'
    '        record.get("history_complete") is True for record in records\n'
    '    )\n'
    '\n'
    '\n'
    'def output_month_records_for_indicator(records: list[dict]) -> dict[date, float]:\n'
    '    """Use the same month-end point-in-time values as the trusted combined output."""\n'
    '    if uses_sparse_source_dates(records):\n'
    '        rows: dict[date, float] = {}\n'
    '        for record in records:\n'
    '            selection_date = record.get("selection_date")\n'
    '            value = analysis_value(record.get("value"))\n'
    '            if isinstance(selection_date, date) and value is not None:\n'
    '                rows[month_start(selection_date)] = value\n'
    '        return rows\n'
    '\n'
    '    rows: dict[date, float] = {}\n'
    '    if FULL_HISTORY_MODE:\n'
    '        row_months = sorted(\n'
    '            {\n'
    '                month_start(record.get("selection_date", record["observation_date"]))\n'
    '                for record in records\n'
    '                if isinstance(record.get("observation_date"), date)\n'
    '                and isinstance(\n'
    '                    record.get("selection_date", record.get("release_date")),\n'
    '                    date,\n'
    '                )\n'
    '            }\n'
    '        )\n'
    '    else:\n'
    '        row_months = recent_month_starts(CURRENT_OUTPUT_MONTHS)\n'
    '    for row_month in row_months:\n'
    '        row_date = month_end(row_month)\n'
    '        selected = latest_available_record(records, row_date)\n'
    '        if selected is None:\n'
    '            continue\n'
    '        value = analysis_value(selected.get("value"))\n'
    '        if value is not None:\n'
    '            rows[month_start(row_date)] = value\n'
    '    return rows\n'
    '\n'
    '\n'
    'def sheet_rows_for_indicator(\n'
    '    indicator_name: str,\n'
    '    records: list[dict],\n'
    ') -> dict[date, float | None]:\n'
    '    """Return source rows plus explicit blanks for unavailable official releases."""\n'
    '    rows: dict[date, float | None] = dict(output_month_records_for_indicator(records))\n'
    '    for output_month in _EXPLICIT_BLANK_OUTPUT_MONTHS.get(indicator_name, set()):\n'
    '        rows[output_month] = None\n'
    '    return rows\n'
    '\n'
    '\n'
    'def analysis_source_columns() -> dict[str, tuple[str, int]]:\n'
    '    """Map every standalone-table indicator to its raw analysis-tab column."""\n'
    '    columns = {\n'
    '        indicator.name: (indicator.name, 1)\n'
    '        for indicator in INDICATORS\n'
    '        if has_dedicated_indicator_tab(indicator.name)\n'
    '    }\n'
    '    for indicator_name, target in CENTRAL_BANK_LIQUIDITY_COMPONENTS.items():\n'
    '        columns[indicator_name] = (CENTRAL_BANK_LIQUIDITY_SHEET, target["column"])\n'
    '    coverage_sheet = IMF_COVERAGE_SHEET\n'
    '    columns["Government revenue % GDP"] = (coverage_sheet, 1)\n'
    '    columns["Interest paid on public debt % GDP"] = (coverage_sheet, 2)\n'
    '    columns[coverage_sheet] = (coverage_sheet, 3)\n'
    '    return columns\n'
    '\n'
    '\n'
    'def wide_indicator_header(indicator_name: str) -> str:\n'
    '    """Use readable standalone headers without changing analysis-tab names."""\n'
    '    if indicator_name in CENTRAL_BANK_LIQUIDITY_COMPONENTS:\n'
    '        header = CENTRAL_BANK_LIQUIDITY_COMPONENTS[indicator_name]["header"]\n'
    '        return header.replace("Cental Bank", "Central Bank")\n'
    '    return indicator_name\n'
    '\n'
    '\n'
    'def analysis_input_ranges() -> list[str]:\n'
    '    """Return the smallest analysis ranges needed to rebuild the wide table."""\n'
    '    widest_by_sheet: dict[str, int] = {}\n'
    '    for sheet_name, column_index in analysis_source_columns().values():\n'
    '        widest_by_sheet[sheet_name] = max(\n'
    '            widest_by_sheet.get(sheet_name, 0), column_index\n'
    '        )\n'
    '    return [\n'
    '        f"{quote_sheet_name(sheet_name)}!A:{column_letter(column_index)}"\n'
    '        for sheet_name, column_index in widest_by_sheet.items()\n'
    '    ]\n'
    '\n'
    '\n'
    'def analysis_number_format_requests(metadata: dict) -> list[dict]:\n'
    '    """Format every numeric analysis input column to exactly five decimals."""\n'
    '    sheet_ids = {\n'
    '        sheet["properties"]["title"]: sheet["properties"]["sheetId"]\n'
    '        for sheet in metadata.get("sheets", [])\n'
    '    }\n'
    '    requests = []\n'
    '    for sheet_name in EXPECTED_ANALYSIS_SHEET_ORDER:\n'
    '        sheet_id = sheet_ids.get(sheet_name)\n'
    '        if sheet_id is None:\n'
    '            continue\n'
    '        end_column_index = (\n'
    '            4 if sheet_name in {CENTRAL_BANK_LIQUIDITY_SHEET, IMF_COVERAGE_SHEET} else 2\n'
    '        )\n'
    '        requests.append(\n'
    '            {\n'
    '                "repeatCell": {\n'
    '                    "range": {\n'
    '                        "sheetId": sheet_id,\n'
    '                        "startRowIndex": 1,\n'
    '                        "startColumnIndex": 1,\n'
    '                        "endColumnIndex": end_column_index,\n'
    '                    },\n'
    '                    "cell": {\n'
    '                        "userEnteredFormat": {\n'
    '                            "numberFormat": {\n'
    '                                "type": "NUMBER",\n'
    '                                "pattern": "0.00000",\n'
    '                            }\n'
    '                        }\n'
    '                    },\n'
    '                    "fields": "userEnteredFormat.numberFormat",\n'
    '                }\n'
    '            }\n'
    '        )\n'
    '    return requests\n'
    '\n'
    '\n'
    'def merge_analysis_values_into_wide_table(\n'
    '    range_values: dict[str, list[list[object]]],\n'
    '    fresh_data: list[tuple[str, list[dict]]] | None = None,\n'
    ') -> list[list[object]]:\n'
    '    """Build one continuous month-end table from analysis inputs plus fresh data."""\n'
    '    by_indicator: dict[str, dict[date, float]] = {\n'
    '        indicator.name: {} for indicator in INDICATORS\n'
    '    }\n'
    '    for indicator_name, (sheet_name, column_index) in analysis_source_columns().items():\n'
    '        rows = range_values.get(sheet_name, [])\n'
    '        for row in rows[1:]:\n'
    '            observed = normalise_observation_date(row[0] if row else None)\n'
    '            raw_value = row[column_index] if len(row) > column_index else None\n'
    '            value = normalise_value(raw_value)\n'
    '            if observed is not None and value is not None:\n'
    '                by_indicator[indicator_name][observed] = value\n'
    '\n'
    '    for indicator_name, records in fresh_data or []:\n'
    '        if indicator_name in by_indicator:\n'
    '            # Source omission is not evidence that an older workbook value is\n'
    '            # wrong. Preserve inaccessible history and remove only cells that\n'
    '            # were separately verified as invalid.\n'
    '            for observed in _VERIFIED_INVALID_EXISTING_MONTHS.get(\n'
    '                indicator_name, set()\n'
    '            ):\n'
    '                by_indicator[indicator_name].pop(observed, None)\n'
    '        sparse_rule = sparse_date_rule_for_records(records)\n'
    '        if sparse_rule is not None and indicator_name in by_indicator:\n'
    '            by_indicator[indicator_name] = {\n'
    '                observed: value\n'
    '                for observed, value in by_indicator[indicator_name].items()\n'
    '                if sparse_date_is_valid(observed, sparse_rule)\n'
    '            }\n'
    '        for observed, value in sheet_rows_for_indicator(\n'
    '            indicator_name, records\n'
    '        ).items():\n'
    '            if indicator_name in by_indicator and value is None:\n'
    '                by_indicator[indicator_name].pop(observed, None)\n'
    '            elif indicator_name in by_indicator:\n'
    '                by_indicator[indicator_name][observed] = value\n'
    '\n'
    '    populated_dates = sorted(\n'
    '        set().union(*(values.keys() for values in by_indicator.values()))\n'
    '    )\n'
    '    headers = [\n'
    '        "Date",\n'
    '        *[wide_indicator_header(indicator.name) for indicator in INDICATORS],\n'
    '    ]\n'
    '    if not populated_dates:\n'
    '        return [headers]\n'
    '\n'
    '    current = min(populated_dates)\n'
    '    final = max(populated_dates)\n'
    '    month_keys = []\n'
    '    while current <= final:\n'
    '        month_keys.append(current)\n'
    '        current = add_months(current, 1)\n'
    '    rows = [\n'
    '        [\n'
    '            (month_end(month_key) - date(1899, 12, 30)).days,\n'
    '            *[\n'
    '                by_indicator[indicator.name].get(month_key, "")\n'
    '                for indicator in INDICATORS\n'
    '            ],\n'
    '        ]\n'
    '        for month_key in month_keys\n'
    '    ]\n'
    '    return [headers, *rows]\n'
    '\n'
    '\n'
    'def preserve_inaccessible_wide_history(\n'
    '    table_values: list[list[object]],\n'
    '    existing_values: list[list[object]],\n'
    '    fresh_data: list[tuple[str, list[dict]]] | None,\n'
    ') -> list[list[object]]:\n'
    '    """Carry forward old wide cells unless a specific cleanup rule rejects them."""\n'
    '    if not table_values or not existing_values:\n'
    '        return table_values\n'
    '    if [normalise_header_name(item) for item in table_values[0]] != [\n'
    '        normalise_header_name(item) for item in existing_values[0]\n'
    '    ]:\n'
    '        return table_values\n'
    '\n'
    '    fresh_by_indicator = dict(fresh_data or [])\n'
    '    indicator_by_column = {\n'
    '        wide_indicator_header(indicator.name): indicator.name\n'
    '        for indicator in INDICATORS\n'
    '    }\n'
    '    rows_by_month: dict[date, list[object]] = {}\n'
    '    for row in table_values[1:]:\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed is not None:\n'
    '            rows_by_month[observed] = list(row) + [""] * (\n'
    '                len(table_values[0]) - len(row)\n'
    '            )\n'
    '\n'
    '    for old_row in existing_values[1:]:\n'
    '        observed = normalise_observation_date(old_row[0] if old_row else None)\n'
    '        if observed is None:\n'
    '            continue\n'
    '        row = rows_by_month.setdefault(\n'
    '            observed,\n'
    '            [\n'
    '                (month_end(observed) - date(1899, 12, 30)).days,\n'
    '                *([""] * (len(table_values[0]) - 1)),\n'
    '            ],\n'
    '        )\n'
    '        for column_index, header in enumerate(table_values[0][1:], start=1):\n'
    '            if column_index >= len(old_row):\n'
    '                continue\n'
    '            old_value = normalise_value(old_row[column_index])\n'
    '            new_value = normalise_value(row[column_index])\n'
    '            if old_value is None or new_value is not None:\n'
    '                continue\n'
    '            indicator_name = indicator_by_column.get(str(header))\n'
    '            if indicator_name is None:\n'
    '                continue\n'
    '            if observed in _VERIFIED_INVALID_EXISTING_MONTHS.get(indicator_name, set()):\n'
    '                continue\n'
    '            if observed in _EXPLICIT_BLANK_OUTPUT_MONTHS.get(indicator_name, set()):\n'
    '                continue\n'
    '            records = fresh_by_indicator.get(indicator_name, [])\n'
    '            sparse_rule = sparse_date_rule_for_records(records)\n'
    '            if sparse_rule is not None and not sparse_date_is_valid(\n'
    '                observed, sparse_rule\n'
    '            ):\n'
    '                continue\n'
    '            row[column_index] = old_row[column_index]\n'
    '\n'
    '    return [\n'
    '        table_values[0],\n'
    '        *[row for _observed, row in sorted(rows_by_month.items())],\n'
    '    ]\n'
    '\n'
    '\n'
    'def fetch_analysis_input_values_direct(\n'
    '    access_token: str,\n'
    '    analysis_spreadsheet_id: str,\n'
    ') -> dict[str, list[list[object]]]:\n'
    '    """Read complete raw-input columns from the existing analysis workbook."""\n'
    '    base_url = f"{GOOGLE_SHEETS_API_BASE}/{analysis_spreadsheet_id}"\n'
    '    ranges = analysis_input_ranges()\n'
    '    query = urlencode(\n'
    '        [("ranges", item) for item in ranges]\n'
    '        + [("valueRenderOption", "UNFORMATTED_VALUE")]\n'
    '    )\n'
    '    response = google_sheets_request(\n'
    '        "GET",\n'
    '        f"{base_url}/values:batchGet?{query}",\n'
    '        access_token,\n'
    '    )\n'
    '    result: dict[str, list[list[object]]] = {}\n'
    '    for requested_range, value_range in zip(ranges, response.get("valueRanges", [])):\n'
    '        sheet_name = requested_range.split("!", 1)[0][1:-1].replace("\'\'", "\'")\n'
    '        result[sheet_name] = value_range.get("values", [])\n'
    '    return result\n'
    '\n'
    '\n'
    'def ensure_wide_sheet_direct(\n'
    '    base_url: str,\n'
    '    access_token: str,\n'
    '    sheet_name: str,\n'
    ') -> dict:\n'
    '    """Resolve the wide tab, reusing a blank default Sheet1 when available."""\n'
    '    metadata = google_sheets_request("GET", base_url, access_token)\n'
    '    sheets = metadata.get("sheets", [])\n'
    '    existing = next(\n'
    '        (sheet for sheet in sheets if sheet["properties"]["title"] == sheet_name),\n'
    '        None,\n'
    '    )\n'
    '    if existing is not None:\n'
    '        return existing\n'
    '    default_sheet = next(\n'
    '        (sheet for sheet in sheets if sheet["properties"]["title"] == "Sheet1"),\n'
    '        None,\n'
    '    )\n'
    '    if len(sheets) == 1 and default_sheet is not None:\n'
    '        values = google_sheets_request(\n'
    '            "GET",\n'
    '            google_values_url(base_url, "\'Sheet1\'!A1:Z10"),\n'
    '            access_token,\n'
    '        ).get("values", [])\n'
    '        if not values:\n'
    '            google_sheets_request(\n'
    '                "POST",\n'
    '                f"{base_url}:batchUpdate",\n'
    '                access_token,\n'
    '                {\n'
    '                    "requests": [\n'
    '                        {\n'
    '                            "updateSheetProperties": {\n'
    '                                "properties": {\n'
    '                                    "sheetId": default_sheet["properties"]["sheetId"],\n'
    '                                    "title": sheet_name,\n'
    '                                },\n'
    '                                "fields": "title",\n'
    '                            }\n'
    '                        }\n'
    '                    ]\n'
    '                },\n'
    '            )\n'
    '            default_sheet["properties"]["title"] = sheet_name\n'
    '            return default_sheet\n'
    '    google_sheets_request(\n'
    '        "POST",\n'
    '        f"{base_url}:batchUpdate",\n'
    '        access_token,\n'
    '        {"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},\n'
    '    )\n'
    '    metadata = google_sheets_request("GET", base_url, access_token)\n'
    '    return next(\n'
    '        sheet\n'
    '        for sheet in metadata.get("sheets", [])\n'
    '        if sheet["properties"]["title"] == sheet_name\n'
    '    )\n'
    '\n'
    '\n'
    'def sync_wide_indicator_workbook(\n'
    '    analysis_spreadsheet_id: str,\n'
    '    wide_spreadsheet_id: str,\n'
    '    credentials_path: Path,\n'
    '    sheet_name: str,\n'
    '    fresh_data: list[tuple[str, list[dict]]] | None = None,\n'
    '    validate_only: bool = False,\n'
    ') -> dict[str, object]:\n'
    '    """Rebuild the standalone wide table without changing analysis-tab structure."""\n'
    '    if analysis_spreadsheet_id == wide_spreadsheet_id:\n'
    '        raise RuntimeError("Wide and analysis spreadsheet IDs must be different.")\n'
    '    access_token = get_service_account_access_token(credentials_path)\n'
    '    range_values = fetch_analysis_input_values_direct(\n'
    '        access_token,\n'
    '        analysis_spreadsheet_id,\n'
    '    )\n'
    '    table_values = merge_analysis_values_into_wide_table(range_values, fresh_data)\n'
    '    base_url = f"{GOOGLE_SHEETS_API_BASE}/{wide_spreadsheet_id}"\n'
    '    metadata = google_sheets_request("GET", base_url, access_token)\n'
    '    existing_sheet = next(\n'
    '        (\n'
    '            sheet\n'
    '            for sheet in metadata.get("sheets", [])\n'
    '            if sheet["properties"]["title"] == sheet_name\n'
    '        ),\n'
    '        None,\n'
    '    )\n'
    '    old_values: list[list[object]] = []\n'
    '    if existing_sheet is not None:\n'
    '        old_values = google_sheets_request(\n'
    '            "GET",\n'
    '            google_values_url(base_url, quote_sheet_name(sheet_name)),\n'
    '            access_token,\n'
    '        ).get("values", [])\n'
    '        table_values = preserve_inaccessible_wide_history(\n'
    '            table_values,\n'
    '            old_values,\n'
    '            fresh_data,\n'
    '        )\n'
    '    summary = {\n'
    '        "spreadsheet_id": wide_spreadsheet_id,\n'
    '        "sheet_name": sheet_name,\n'
    '        "rows": max(0, len(table_values) - 1),\n'
    '        "columns": len(table_values[0]),\n'
    '        "first_date": "",\n'
    '        "last_date": "",\n'
    '    }\n'
    '    if len(table_values) > 1:\n'
    '        summary["first_date"] = format_output_month(\n'
    '            date(1899, 12, 30) + timedelta(days=int(table_values[1][0]))\n'
    '        )\n'
    '        summary["last_date"] = format_output_month(\n'
    '            date(1899, 12, 30) + timedelta(days=int(table_values[-1][0]))\n'
    '        )\n'
    '\n'
    '    if existing_sheet is not None and not FULL_HISTORY_MODE:\n'
    '        fills, appends, merged_values = prepare_incremental_table_changes(\n'
    '            old_values,\n'
    '            table_values,\n'
    '            f"Standalone sheet {sheet_name!r}",\n'
    '        )\n'
    '        summary["rows"] = max(0, len(merged_values) - 1)\n'
    '        if len(merged_values) > 1:\n'
    '            summary["first_date"] = format_output_month(\n'
    '                date(1899, 12, 30)\n'
    '                + timedelta(days=int(merged_values[1][0]))\n'
    '            )\n'
    '            summary["last_date"] = format_output_month(\n'
    '                date(1899, 12, 30)\n'
    '                + timedelta(days=int(merged_values[-1][0]))\n'
    '            )\n'
    '        if not fills and not appends:\n'
    '            return summary\n'
    '        if validate_only:\n'
    '            return summary\n'
    '\n'
    '        properties = existing_sheet["properties"]\n'
    '        sheet_id = properties["sheetId"]\n'
    '        grid = properties.get("gridProperties", {})\n'
    '        required_rows = len(merged_values)\n'
    '        required_columns = len(merged_values[0])\n'
    '        if (\n'
    '            required_rows > int(grid.get("rowCount", 0))\n'
    '            or required_columns > int(grid.get("columnCount", 0))\n'
    '        ):\n'
    '            google_sheets_request(\n'
    '                "POST",\n'
    '                f"{base_url}:batchUpdate",\n'
    '                access_token,\n'
    '                {\n'
    '                    "requests": [\n'
    '                        {\n'
    '                            "updateSheetProperties": {\n'
    '                                "properties": {\n'
    '                                    "sheetId": sheet_id,\n'
    '                                    "gridProperties": {\n'
    '                                        "rowCount": max(\n'
    '                                            required_rows,\n'
    '                                            int(grid.get("rowCount", 0)),\n'
    '                                        ),\n'
    '                                        "columnCount": max(\n'
    '                                            required_columns,\n'
    '                                            int(grid.get("columnCount", 0)),\n'
    '                                        ),\n'
    '                                    },\n'
    '                                },\n'
    '                                "fields": "gridProperties(rowCount,columnCount)",\n'
    '                            }\n'
    '                        }\n'
    '                    ]\n'
    '                },\n'
    '            )\n'
    '\n'
    '        read_url = google_values_url(base_url, quote_sheet_name(sheet_name))\n'
    '        current_values = google_sheets_request("GET", read_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        if current_values != old_values:\n'
    '            raise RuntimeError(\n'
    '                f"Standalone sheet {sheet_name!r} changed after preflight; refusing stale incremental write."\n'
    '            )\n'
    '\n'
    '        if old_values:\n'
    '            data_ranges = incremental_table_value_ranges(\n'
    '                sheet_name,\n'
    '                fills,\n'
    '                appends,\n'
    '                len(old_values),\n'
    '            )\n'
    '            google_sheets_request(\n'
    '                "POST",\n'
    '                f"{base_url}/values:batchUpdate?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"valueInputOption": "RAW", "data": data_ranges},\n'
    '            )\n'
    '        else:\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/{quote(f\'{quote_sheet_name(sheet_name)}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"values": merged_values},\n'
    '            )\n'
    '\n'
    '        if appends or not old_values:\n'
    '            format_start = len(old_values) if old_values else 1\n'
    '            format_end = len(merged_values)\n'
    '            format_requests = [\n'
    '                {\n'
    '                    "repeatCell": {\n'
    '                        "range": {\n'
    '                            "sheetId": sheet_id,\n'
    '                            "startRowIndex": format_start,\n'
    '                            "endRowIndex": format_end,\n'
    '                            "startColumnIndex": 0,\n'
    '                            "endColumnIndex": 1,\n'
    '                        },\n'
    '                        "cell": {\n'
    '                            "userEnteredFormat": {\n'
    '                                "numberFormat": {\n'
    '                                    "type": "DATE",\n'
    '                                    "pattern": "dd/mm/yyyy",\n'
    '                                }\n'
    '                            }\n'
    '                        },\n'
    '                        "fields": "userEnteredFormat.numberFormat",\n'
    '                    }\n'
    '                },\n'
    '                {\n'
    '                    "repeatCell": {\n'
    '                        "range": {\n'
    '                            "sheetId": sheet_id,\n'
    '                            "startRowIndex": format_start,\n'
    '                            "endRowIndex": format_end,\n'
    '                            "startColumnIndex": 1,\n'
    '                            "endColumnIndex": len(merged_values[0]),\n'
    '                        },\n'
    '                        "cell": {\n'
    '                            "userEnteredFormat": {\n'
    '                                "numberFormat": {\n'
    '                                    "type": "NUMBER",\n'
    '                                    "pattern": "0.00000",\n'
    '                                }\n'
    '                            }\n'
    '                        },\n'
    '                        "fields": "userEnteredFormat.numberFormat",\n'
    '                    }\n'
    '                },\n'
    '            ]\n'
    '            if not old_values:\n'
    '                format_requests.append(\n'
    '                    {\n'
    '                        "repeatCell": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "startRowIndex": 0,\n'
    '                                "endRowIndex": 1,\n'
    '                                "startColumnIndex": 0,\n'
    '                                "endColumnIndex": len(merged_values[0]),\n'
    '                            },\n'
    '                            "cell": {\n'
    '                                "userEnteredFormat": {\n'
    '                                    "textFormat": {"bold": True},\n'
    '                                    "wrapStrategy": "WRAP",\n'
    '                                    "verticalAlignment": "BOTTOM",\n'
    '                                }\n'
    '                            },\n'
    '                            "fields": "userEnteredFormat(textFormat,wrapStrategy,verticalAlignment)",\n'
    '                        }\n'
    '                    }\n'
    '                )\n'
    '            google_sheets_request(\n'
    '                "POST",\n'
    '                f"{base_url}:batchUpdate",\n'
    '                access_token,\n'
    '                {"requests": format_requests},\n'
    '            )\n'
    '\n'
    '        written = google_sheets_request("GET", read_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        verify_google_sheet_values(merged_values, written)\n'
    '        return summary\n'
    '\n'
    '    if existing_sheet is not None:\n'
    '        validate_historical_table_replacement(\n'
    '            old_values,\n'
    '            table_values,\n'
    '            f"Standalone sheet {sheet_name!r}",\n'
    '            allow_absent_existing_dates=fresh_data is not None,\n'
    '            allow_undated_existing_rows=fresh_data is not None,\n'
    '            allow_blank_replacements=fresh_data is not None,\n'
    '            allow_value_revisions=fresh_data is not None and FULL_HISTORY_MODE,\n'
    '            mutable_months=mutable_output_months(),\n'
    '        )\n'
    '        if google_sheet_table_is_current(table_values, old_values):\n'
    '            return summary\n'
    '    if validate_only:\n'
    '        return summary\n'
    '\n'
    '    target_sheet = ensure_wide_sheet_direct(base_url, access_token, sheet_name)\n'
    '    properties = target_sheet["properties"]\n'
    '    sheet_id = properties["sheetId"]\n'
    '    grid = properties.get("gridProperties", {})\n'
    '    row_count = max(int(grid.get("rowCount", 0)), len(table_values))\n'
    '    column_count = max(int(grid.get("columnCount", 0)), len(table_values[0]))\n'
    '    if row_count != grid.get("rowCount") or column_count != grid.get("columnCount"):\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}:batchUpdate",\n'
    '            access_token,\n'
    '            {\n'
    '                "requests": [\n'
    '                    {\n'
    '                        "updateSheetProperties": {\n'
    '                            "properties": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "gridProperties": {\n'
    '                                    "rowCount": row_count,\n'
    '                                    "columnCount": column_count,\n'
    '                                },\n'
    '                            },\n'
    '                            "fields": "gridProperties(rowCount,columnCount)",\n'
    '                        }\n'
    '                    }\n'
    '                ]\n'
    '            },\n'
    '        )\n'
    '\n'
    '    quoted_sheet = quote_sheet_name(sheet_name)\n'
    '    range_url = f"{base_url}/values/{quote(quoted_sheet, safe=\'\')}"\n'
    '    read_url = google_values_url(base_url, quoted_sheet)\n'
    '    if not old_values:\n'
    '        old_values = google_sheets_request("GET", read_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '    validate_historical_table_replacement(\n'
    '        old_values,\n'
    '        table_values,\n'
    '        f"Standalone sheet {sheet_name!r}",\n'
    '        allow_absent_existing_dates=fresh_data is not None,\n'
    '        allow_undated_existing_rows=fresh_data is not None,\n'
    '        allow_blank_replacements=fresh_data is not None,\n'
    '        allow_value_revisions=fresh_data is not None and FULL_HISTORY_MODE,\n'
    '        mutable_months=mutable_output_months(),\n'
    '    )\n'
    '    try:\n'
    '        current_values = google_sheets_request("GET", read_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        if current_values != old_values:\n'
    '            raise RuntimeError(\n'
    '                f"Standalone sheet {sheet_name!r} changed after preflight; refusing stale overwrite."\n'
    '            )\n'
    '        google_sheets_request("POST", f"{range_url}:clear", access_token, {})\n'
    '        google_sheets_request(\n'
    '            "PUT",\n'
    '            f"{base_url}/values/{quote(f\'{quoted_sheet}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '            access_token,\n'
    '            {"values": table_values},\n'
    '        )\n'
    '        written = google_sheets_request("GET", read_url, access_token).get("values", [])\n'
    '        verify_google_sheet_values(table_values, written)\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}:batchUpdate",\n'
    '            access_token,\n'
    '            {\n'
    '                "requests": [\n'
    '                    {\n'
    '                        "repeatCell": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "startRowIndex": 0,\n'
    '                                "endRowIndex": 1,\n'
    '                                "startColumnIndex": 0,\n'
    '                                "endColumnIndex": len(table_values[0]),\n'
    '                            },\n'
    '                            "cell": {\n'
    '                                "userEnteredFormat": {\n'
    '                                    "textFormat": {"bold": True},\n'
    '                                    "wrapStrategy": "WRAP",\n'
    '                                    "verticalAlignment": "BOTTOM",\n'
    '                                }\n'
    '                            },\n'
    '                            "fields": "userEnteredFormat(textFormat,wrapStrategy,verticalAlignment)",\n'
    '                        }\n'
    '                    },\n'
    '                    {\n'
    '                        "repeatCell": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "startRowIndex": 1,\n'
    '                                "endRowIndex": len(table_values),\n'
    '                                "startColumnIndex": 0,\n'
    '                                "endColumnIndex": 1,\n'
    '                            },\n'
    '                            "cell": {\n'
    '                                "userEnteredFormat": {\n'
    '                                    "numberFormat": {\n'
    '                                        "type": "DATE",\n'
    '                                        "pattern": "dd/mm/yyyy",\n'
    '                                    }\n'
    '                                }\n'
    '                            },\n'
    '                            "fields": "userEnteredFormat.numberFormat",\n'
    '                        }\n'
    '                    },\n'
    '                    {\n'
    '                        "repeatCell": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "startRowIndex": 1,\n'
    '                                "endRowIndex": len(table_values),\n'
    '                                "startColumnIndex": 1,\n'
    '                                "endColumnIndex": len(table_values[0]),\n'
    '                            },\n'
    '                            "cell": {\n'
    '                                "userEnteredFormat": {\n'
    '                                    "numberFormat": {\n'
    '                                        "type": "NUMBER",\n'
    '                                        "pattern": "0.00000",\n'
    '                                    }\n'
    '                                }\n'
    '                            },\n'
    '                            "fields": "userEnteredFormat.numberFormat",\n'
    '                        }\n'
    '                    },\n'
    '                    {\n'
    '                        "updateSheetProperties": {\n'
    '                            "properties": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "gridProperties": {\n'
    '                                    "frozenRowCount": 1,\n'
    '                                    "frozenColumnCount": 1,\n'
    '                                },\n'
    '                            },\n'
    '                            "fields": "gridProperties(frozenRowCount,frozenColumnCount)",\n'
    '                        }\n'
    '                    },\n'
    '                    {\n'
    '                        "updateDimensionProperties": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "dimension": "ROWS",\n'
    '                                "startIndex": 0,\n'
    '                                "endIndex": 1,\n'
    '                            },\n'
    '                            "properties": {"pixelSize": 48},\n'
    '                            "fields": "pixelSize",\n'
    '                        }\n'
    '                    },\n'
    '                    {\n'
    '                        "updateDimensionProperties": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "dimension": "COLUMNS",\n'
    '                                "startIndex": 0,\n'
    '                                "endIndex": 1,\n'
    '                            },\n'
    '                            "properties": {"pixelSize": 105},\n'
    '                            "fields": "pixelSize",\n'
    '                        }\n'
    '                    },\n'
    '                    {\n'
    '                        "updateDimensionProperties": {\n'
    '                            "range": {\n'
    '                                "sheetId": sheet_id,\n'
    '                                "dimension": "COLUMNS",\n'
    '                                "startIndex": 1,\n'
    '                                "endIndex": len(table_values[0]),\n'
    '                            },\n'
    '                            "properties": {"pixelSize": 135},\n'
    '                            "fields": "pixelSize",\n'
    '                        }\n'
    '                    },\n'
    '                ]\n'
    '            },\n'
    '        )\n'
    '    except Exception:\n'
    '        logging.exception(\n'
    '            "Wide Google Sheet update failed; restoring its previous contents."\n'
    '        )\n'
    '        google_sheets_request("POST", f"{range_url}:clear", access_token, {})\n'
    '        if old_values:\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/{quote(f\'{quoted_sheet}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"values": old_values},\n'
    '            )\n'
    '        raise\n'
    '    return summary\n'
    '\n'
    '\n'
    'def prepare_incremental_table_changes(\n'
    '    existing_values: list[list[object]],\n'
    '    incoming_values: list[list[object]],\n'
    '    context: str,\n'
    ') -> tuple[\n'
    '    list[tuple[int, int, object]], list[list[object]], list[list[object]]\n'
    ']:\n'
    '    """Plan blank-cell fills and newer-row appends without revising stored data."""\n'
    '    if not incoming_values or not incoming_values[0]:\n'
    '        raise RuntimeError(f"{context}: incoming table has no header row.")\n'
    '\n'
    '    headers = list(incoming_values[0])\n'
    '    width = len(headers)\n'
    '    if existing_values and [normalise_header_name(value) for value in existing_values[0]] != [\n'
    '        normalise_header_name(value) for value in headers\n'
    '    ]:\n'
    '        raise RuntimeError(f"{context}: existing headers do not match the source table.")\n'
    '\n'
    '    def rectangular(row: list[object]) -> list[object]:\n'
    '        return (list(row[:width]) + [""] * width)[:width]\n'
    '\n'
    '    existing_rows = [rectangular(row) for row in existing_values[1:]]\n'
    '    incoming_rows = [rectangular(row) for row in incoming_values[1:]]\n'
    '\n'
    '    existing_by_date: dict[date, tuple[int, list[object]]] = {}\n'
    '    previous_date: date | None = None\n'
    '    for row_number, row in enumerate(existing_rows, start=2):\n'
    '        observed = normalise_observation_date(row[0])\n'
    '        if observed is None:\n'
    '            raise RuntimeError(f"{context}: existing row {row_number} has no valid date.")\n'
    '        if observed in existing_by_date:\n'
    '            raise RuntimeError(f"{context}: duplicate existing date {observed.isoformat()}.")\n'
    '        if previous_date is not None and observed <= previous_date:\n'
    '            raise RuntimeError(f"{context}: existing dates are not strictly increasing.")\n'
    '        existing_by_date[observed] = (row_number, row)\n'
    '        previous_date = observed\n'
    '\n'
    '    incoming_by_date: dict[date, list[object]] = {}\n'
    '    previous_date = None\n'
    '    for row_number, row in enumerate(incoming_rows, start=2):\n'
    '        observed = normalise_observation_date(row[0])\n'
    '        if observed is None:\n'
    '            raise RuntimeError(f"{context}: incoming row {row_number} has no valid date.")\n'
    '        if observed in incoming_by_date:\n'
    '            raise RuntimeError(f"{context}: duplicate incoming date {observed.isoformat()}.")\n'
    '        if previous_date is not None and observed <= previous_date:\n'
    '            raise RuntimeError(f"{context}: incoming dates are not strictly increasing.")\n'
    '        incoming_by_date[observed] = row\n'
    '        previous_date = observed\n'
    '\n'
    '    latest_existing = max(existing_by_date, default=None)\n'
    '    fills: list[tuple[int, int, object]] = []\n'
    '    appends: list[list[object]] = []\n'
    '    merged_rows = [list(row) for row in existing_rows]\n'
    '\n'
    '    for observed, incoming_row in incoming_by_date.items():\n'
    '        current = existing_by_date.get(observed)\n'
    '        if current is None:\n'
    '            if latest_existing is not None and observed <= latest_existing:\n'
    '                raise RuntimeError(\n'
    '                    f"{context}: missing historical date {observed.isoformat()}; "\n'
    '                    "incremental sync will not insert or reorder history."\n'
    '                )\n'
    '            appends.append(incoming_row)\n'
    '            continue\n'
    '\n'
    '        row_number, existing_row = current\n'
    '        merged_row = merged_rows[row_number - 2]\n'
    '        for column_index in range(1, width):\n'
    '            new_value = incoming_row[column_index]\n'
    '            if normalise_value(existing_row[column_index]) is not None:\n'
    '                continue\n'
    '            if normalise_value(new_value) is None:\n'
    '                continue\n'
    '            fills.append((row_number, column_index, new_value))\n'
    '            merged_row[column_index] = new_value\n'
    '\n'
    '    merged = [headers, *merged_rows, *appends]\n'
    '    return fills, appends, merged\n'
    '\n'
    '\n'
    'def incremental_table_value_ranges(\n'
    '    sheet_name: str,\n'
    '    fills: list[tuple[int, int, object]],\n'
    '    appends: list[list[object]],\n'
    '    existing_row_count: int,\n'
    ') -> list[dict[str, object]]:\n'
    '    """Turn incremental cell changes into bounded Google Sheets value ranges."""\n'
    '    quoted_sheet = quote_sheet_name(sheet_name)\n'
    '    ranges: list[dict[str, object]] = []\n'
    '    by_row: dict[int, dict[int, object]] = {}\n'
    '    for row_number, column_index, value in fills:\n'
    '        by_row.setdefault(row_number, {})[column_index] = value\n'
    '\n'
    '    for row_number, columns in sorted(by_row.items()):\n'
    '        ordered = sorted(columns.items())\n'
    '        start = 0\n'
    '        while start < len(ordered):\n'
    '            end = start + 1\n'
    '            while end < len(ordered) and ordered[end][0] == ordered[end - 1][0] + 1:\n'
    '                end += 1\n'
    '            first_column = ordered[start][0]\n'
    '            last_column = ordered[end - 1][0]\n'
    '            ranges.append(\n'
    '                {\n'
    '                    "range": (\n'
    '                        f"{quoted_sheet}!{column_letter(first_column)}{row_number}:"\n'
    '                        f"{column_letter(last_column)}{row_number}"\n'
    '                    ),\n'
    '                    "values": [[value for _column, value in ordered[start:end]]],\n'
    '                }\n'
    '            )\n'
    '            start = end\n'
    '\n'
    '    if appends:\n'
    '        width = max(len(row) for row in appends)\n'
    '        padded = [list(row) + [""] * (width - len(row)) for row in appends]\n'
    '        start_row = existing_row_count + 1\n'
    '        end_row = start_row + len(padded) - 1\n'
    '        ranges.append(\n'
    '            {\n'
    '                "range": (\n'
    '                    f"{quoted_sheet}!A{start_row}:"\n'
    '                    f"{column_letter(width - 1)}{end_row}"\n'
    '                ),\n'
    '                "values": padded,\n'
    '            }\n'
    '        )\n'
    '    return ranges\n'
    '\n'
    '\n'
    'def prepare_sheet_changes(\n'
    '    existing: dict[date, dict],\n'
    '    incoming: dict[date, float | None],\n'
    ') -> tuple[list[tuple[int, date, float | None]], list[tuple[date, float]]]:\n'
    '    """Fill blank cells and append newer dates; full-history mode may revise values."""\n'
    '    updates: list[tuple[int, date, float | None]] = []\n'
    '    appends: list[tuple[date, float]] = []\n'
    '    latest_existing_date = max(existing, default=None)\n'
    '    for observed, value in sorted(incoming.items()):\n'
    '        existing_row = existing.get(observed)\n'
    '        if value is None:\n'
    '            if (\n'
    '                FULL_HISTORY_MODE\n'
    '                and existing_row is not None\n'
    '                and existing_row.get("value") is not None\n'
    '            ):\n'
    '                updates.append((existing_row["row"], observed, value))\n'
    '            continue\n'
    '        if existing_row is not None:\n'
    '            existing_value = existing_row.get("value")\n'
    '            if existing_value is None or (\n'
    '                FULL_HISTORY_MODE\n'
    '                and not google_values_equivalent(value, existing_value)\n'
    '            ):\n'
    '                updates.append((existing_row["row"], observed, value))\n'
    '            continue\n'
    '        if latest_existing_date is None or observed > latest_existing_date:\n'
    '            appends.append((observed, value))\n'
    '    return updates, appends\n'
    '\n'
    '\n'
    'def indicator_sheet_name(indicator_name: str, sheet_names: set[str]) -> str | None:\n'
    '    """Match only the exact indicator tab name."""\n'
    '    return indicator_name if indicator_name in sheet_names else None\n'
    '\n'
    '\n'
    'def sync_result_row(\n'
    '    indicator: str,\n'
    '    sheet_name: str,\n'
    '    latest_before: date | None,\n'
    '    revised: int,\n'
    '    appended: int,\n'
    '    latest_after: date | None,\n'
    '    warning: str = "",\n'
    '    *,\n'
    '    source_change_counts: dict[str, int] | None = None,\n'
    ') -> dict:\n'
    '    result = {\n'
    '        "indicator": indicator,\n'
    '        "sheet_name": sheet_name,\n'
    '        "latest_before": latest_before.isoformat() if latest_before else "",\n'
    '        "revised": revised,\n'
    '        "appended": appended,\n'
    '        "latest_after": latest_after.isoformat() if latest_after else "",\n'
    '        "no_new_value": "yes" if appended == 0 else "no",\n'
    '        "warning": warning,\n'
    '    }\n'
    '    if source_change_counts is not None:\n'
    '        result["source_change_counts"] = source_change_counts\n'
    '    return result\n'
    '\n'
    '\n'
    'def validate_analysis_workbook_topology(\n'
    '    sheet_names: set[str],\n'
    '    control_values: list[list[object]],\n'
    ') -> None:\n'
    '    """Require the exact tabs and Control Panel order used by the analysis engine."""\n'
    '    required_tabs = {\n'
    '        ANALYSIS_CONTROL_PANEL_SHEET,\n'
    '        *EXPECTED_ANALYSIS_SHEET_ORDER,\n'
    '    }\n'
    '    missing_tabs = sorted(required_tabs - sheet_names)\n'
    '    if missing_tabs:\n'
    '        raise RuntimeError(\n'
    '            "Analysis workbook is missing required tabs: " + ", ".join(missing_tabs)\n'
    '        )\n'
    '\n'
    '    actual = [\n'
    '        str(row[0]).strip() if row and row[0] not in (None, "") else ""\n'
    '        for row in control_values\n'
    '    ]\n'
    '    actual.extend([""] * (len(EXPECTED_ANALYSIS_SHEET_ORDER) - len(actual)))\n'
    '    actual = actual[: len(EXPECTED_ANALYSIS_SHEET_ORDER)]\n'
    '    expected = list(EXPECTED_ANALYSIS_SHEET_ORDER)\n'
    '    if actual == expected:\n'
    '        return\n'
    '\n'
    '    mismatches = [\n'
    '        f"A{row_number}: expected={expected_name!r}, actual={actual_name!r}"\n'
    '        for row_number, (expected_name, actual_name) in enumerate(\n'
    '            zip(expected, actual), start=3\n'
    '        )\n'
    '        if expected_name != actual_name\n'
    '    ]\n'
    '    preview = "; ".join(mismatches[:5])\n'
    '    suffix = f"; plus {len(mismatches) - 5} more" if len(mismatches) > 5 else ""\n'
    '    raise RuntimeError(\n'
    '        "Analysis Control Panel does not match the required 37-indicator order "\n'
    '        f"({preview}{suffix})."\n'
    '    )\n'
    '\n'
    '\n'
    'def ensure_google_sheet_client(\n'
    '    sheets,\n'
    '    spreadsheet_id: str,\n'
    '    sheet_name: str,\n'
    '    allow_legacy_rename: bool = False,\n'
    ') -> None:\n'
    '    metadata = sheets.get(spreadsheetId=spreadsheet_id).execute()\n'
    '    existing = metadata.get("sheets", [])\n'
    '    if any(sheet["properties"]["title"] == sheet_name for sheet in existing):\n'
    '        return\n'
    '    if allow_legacy_rename:\n'
    '        legacy_sheet = next(\n'
    '            (\n'
    '                sheet\n'
    '                for sheet in existing\n'
    '                if sheet["properties"]["title"] in LEGACY_SHEET_NAMES\n'
    '            ),\n'
    '            None,\n'
    '        )\n'
    '        if legacy_sheet is not None:\n'
    '            sheets.batchUpdate(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                body={\n'
    '                    "requests": [\n'
    '                        {\n'
    '                            "updateSheetProperties": {\n'
    '                                "properties": {\n'
    '                                    "sheetId": legacy_sheet["properties"]["sheetId"],\n'
    '                                    "title": sheet_name,\n'
    '                                },\n'
    '                                "fields": "title",\n'
    '                            }\n'
    '                        }\n'
    '                    ]\n'
    '                },\n'
    '            ).execute()\n'
    '            return\n'
    '    sheets.batchUpdate(\n'
    '        spreadsheetId=spreadsheet_id,\n'
    '        body={"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},\n'
    '    ).execute()\n'
    '\n'
    '\n'
    'def update_google_sheet(\n'
    '    spreadsheet_id: str,\n'
    '    credentials_path: Path,\n'
    '    indicators: list[tuple[str, list[dict]]],\n'
    '    sheet_name: str,\n'
    '    metadata_values: list[list[object]],\n'
    '    validation_values: list[list[object]],\n'
    '    diagnostic_values: list[list[object]],\n'
    ') -> None:\n'
    '    logging.getLogger("google.oauth2._client").setLevel(logging.ERROR)\n'
    '\n'
    '    try:\n'
    '        if os.getenv("US_PIPELINE_FORCE_DIRECT_GOOGLE_API") == "1":\n'
    '            raise ImportError("Direct Google Sheets API selected by pipeline runner.")\n'
    '        from google.oauth2.service_account import Credentials\n'
    '        from googleapiclient.discovery import build\n'
    '    except ImportError as exc:\n'
    '        logging.info(\n'
    '            "Google client libraries are unavailable; using direct Sheets API fallback."\n'
    '        )\n'
    '        update_google_sheet_direct(\n'
    '            spreadsheet_id,\n'
    '            credentials_path,\n'
    '            indicators,\n'
    '            sheet_name,\n'
    '            metadata_values,\n'
    '            validation_values,\n'
    '            diagnostic_values,\n'
    '        )\n'
    '        return\n'
    '\n'
    '    if not credentials_path.exists():\n'
    '        raise SystemExit(f"Google credentials file not found: {credentials_path}")\n'
    '\n'
    '    credentials = Credentials.from_service_account_file(\n'
    '        credentials_path,\n'
    '        scopes=GOOGLE_SHEETS_WRITE_SCOPES,\n'
    '    )\n'
    '    service = build("sheets", "v4", credentials=credentials)\n'
    '    sheets = service.spreadsheets()\n'
    '    tables = {\n'
    '        sheet_name: sheet_values_for_google(indicators),\n'
    '        DEFAULT_GOOGLE_STATUS_SHEET_NAME: source_status_values(),\n'
    '        DEFAULT_GOOGLE_METADATA_SHEET_NAME: metadata_values,\n'
    '        DEFAULT_GOOGLE_VALIDATION_SHEET_NAME: validation_values,\n'
    '        DEFAULT_GOOGLE_DIAGNOSTIC_SHEET_NAME: diagnostic_values,\n'
    '    }\n'
    '    for table_name in tables:\n'
    '        ensure_google_sheet_client(\n'
    '            sheets,\n'
    '            spreadsheet_id,\n'
    '            table_name,\n'
    '            allow_legacy_rename=table_name == sheet_name,\n'
    '        )\n'
    '\n'
    '    old_tables = {\n'
    '        table_name: (\n'
    '            sheets.values()\n'
    '            .get(spreadsheetId=spreadsheet_id, range=f"\'{table_name}\'")\n'
    '            .execute()\n'
    '            .get("values", [])\n'
    '        )\n'
    '        for table_name in tables\n'
    '    }\n'
    '    try:\n'
    '        for table_name, table_values in tables.items():\n'
    '            sheets.values().clear(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"\'{table_name}\'",\n'
    '                body={},\n'
    '            ).execute()\n'
    '            sheets.values().update(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"\'{table_name}\'!A1",\n'
    '                valueInputOption="RAW",\n'
    '                body={"values": table_values},\n'
    '            ).execute()\n'
    '            written = (\n'
    '                sheets.values()\n'
    '                .get(spreadsheetId=spreadsheet_id, range=f"\'{table_name}\'")\n'
    '                .execute()\n'
    '                .get("values", [])\n'
    '            )\n'
    '            verify_google_sheet_values(table_values, written)\n'
    '    except Exception:\n'
    '        logging.exception(\n'
    '            "Google Sheets update failed; restoring the previous sheet contents."\n'
    '        )\n'
    '        for table_name, old_values in old_tables.items():\n'
    '            sheets.values().clear(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"\'{table_name}\'",\n'
    '                body={},\n'
    '            ).execute()\n'
    '            if old_values:\n'
    '                sheets.values().update(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=f"\'{table_name}\'!A1",\n'
    '                    valueInputOption="RAW",\n'
    '                    body={"values": old_values},\n'
    '                ).execute()\n'
    '        raise\n'
    '\n'
    '\n'
    'def base64url_encode(raw: bytes) -> str:\n'
    '    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode("ascii")\n'
    '\n'
    '\n'
    'def sign_with_openssl(private_key: str, signing_input: str) -> str:\n'
    '    from cryptography.hazmat.primitives import hashes, serialization\n'
    '    from cryptography.hazmat.primitives.asymmetric import padding\n'
    '\n'
    '    key = serialization.load_pem_private_key(private_key.encode("utf-8"), password=None)\n'
    '    signature = key.sign(\n'
    '        signing_input.encode("utf-8"), padding.PKCS1v15(), hashes.SHA256()\n'
    '    )\n'
    '    return base64.urlsafe_b64encode(signature).rstrip(b"=").decode("ascii")\n'
    '\n'
    '\n'
    'def get_service_account_access_token(credentials_path: Path) -> str:\n'
    '    if not credentials_path.exists():\n'
    '        raise SystemExit(f"Google credentials file not found: {credentials_path}")\n'
    '\n'
    '    credentials = json.loads(credentials_path.read_text())\n'
    '    now = int(time.time())\n'
    '    header = {"alg": "RS256", "typ": "JWT"}\n'
    '    payload = {\n'
    '        "iss": credentials["client_email"],\n'
    '        "scope": "https://www.googleapis.com/auth/spreadsheets",\n'
    '        "aud": GOOGLE_TOKEN_URL,\n'
    '        "iat": now,\n'
    '        "exp": now + 3600,\n'
    '    }\n'
    '    signing_input = ".".join(\n'
    '        [\n'
    '            base64url_encode(json.dumps(header, separators=(",", ":")).encode("utf-8")),\n'
    '            base64url_encode(\n'
    '                json.dumps(payload, separators=(",", ":")).encode("utf-8")\n'
    '            ),\n'
    '        ]\n'
    '    )\n'
    '    assertion = f"{signing_input}.{sign_with_openssl(credentials[\'private_key\'], signing_input)}"\n'
    '    response = urlopen(\n'
    '        GOOGLE_TOKEN_URL,\n'
    '        data=urlencode(\n'
    '            {\n'
    '                "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",\n'
    '                "assertion": assertion,\n'
    '            }\n'
    '        ).encode("utf-8"),\n'
    '        timeout=30,\n'
    '    )\n'
    '    token_response = json.loads(response.read().decode("utf-8"))\n'
    '    return token_response["access_token"]\n'
    '\n'
    '\n'
    'def google_sheets_request(\n'
    '    method: str,\n'
    '    url: str,\n'
    '    access_token: str,\n'
    '    body: dict | None = None,\n'
    ') -> dict:\n'
    '    import requests\n'
    '\n'
    '    response = None\n'
    '    for attempt in range(1, GOOGLE_REQUEST_MAX_ATTEMPTS + 1):\n'
    '        try:\n'
    '            response = requests.request(\n'
    '                method,\n'
    '                url,\n'
    '                headers={\n'
    '                    "Authorization": f"Bearer {access_token}",\n'
    '                    "Content-Type": "application/json",\n'
    '                },\n'
    '                json=body,\n'
    '                timeout=(\n'
    '                    GOOGLE_REQUEST_CONNECT_TIMEOUT_SECONDS,\n'
    '                    GOOGLE_REQUEST_READ_TIMEOUT_SECONDS,\n'
    '                ),\n'
    '            )\n'
    '        except (requests.Timeout, requests.ConnectionError):\n'
    '            if attempt >= GOOGLE_REQUEST_MAX_ATTEMPTS:\n'
    '                raise\n'
    '            logging.warning(\n'
    '                "Google Sheets request failed transiently; retrying (%s/%s).",\n'
    '                attempt,\n'
    '                GOOGLE_REQUEST_MAX_ATTEMPTS,\n'
    '            )\n'
    '            sleep(min(30, 2**attempt))\n'
    '            continue\n'
    '        if response.status_code < 400:\n'
    '            return response.json() if response.text else {}\n'
    '        retryable = response.status_code in {429, 500, 502, 503, 504} or (\n'
    '            response.status_code == 404 and method == "GET"\n'
    '        )\n'
    '        if not retryable or attempt == GOOGLE_REQUEST_MAX_ATTEMPTS:\n'
    '            break\n'
    '        retry_after = response.headers.get("Retry-After")\n'
    '        if retry_after:\n'
    '            delay = float(retry_after)\n'
    '        elif response.status_code == 429:\n'
    '            # Per-user Sheets read quotas reset on a minute boundary. Short\n'
    '            # exponential retries only exhaust all attempts inside the same\n'
    '            # quota window.\n'
    '            delay = 60.0\n'
    '        else:\n'
    '            delay = min(30, 2**attempt)\n'
    '        sleep(delay)\n'
    '    assert response is not None\n'
    '    raise RuntimeError(\n'
    '        f"Google Sheets returned {response.status_code}: {response.text}"\n'
    '    )\n'
    '\n'
    '\n'
    'def google_values_url(base_url: str, range_name: str) -> str:\n'
    '    """Build a Sheets values URL that reads actual stored values, not display text."""\n'
    '    return (\n'
    '        f"{base_url}/values/{quote(range_name, safe=\'\')}"\n'
    '        "?valueRenderOption=UNFORMATTED_VALUE"\n'
    '    )\n'
    '\n'
    '\n'
    'def ensure_google_sheet_direct(\n'
    '    base_url: str,\n'
    '    access_token: str,\n'
    '    sheet_name: str,\n'
    '    allow_legacy_rename: bool = False,\n'
    ') -> None:\n'
    '    metadata = google_sheets_request("GET", base_url, access_token)\n'
    '    sheets = metadata.get("sheets", [])\n'
    '    if any(sheet["properties"]["title"] == sheet_name for sheet in sheets):\n'
    '        return\n'
    '    if allow_legacy_rename:\n'
    '        legacy_sheet = next(\n'
    '            (\n'
    '                sheet\n'
    '                for sheet in sheets\n'
    '                if sheet["properties"]["title"] in LEGACY_SHEET_NAMES\n'
    '            ),\n'
    '            None,\n'
    '        )\n'
    '        if legacy_sheet is not None:\n'
    '            google_sheets_request(\n'
    '                "POST",\n'
    '                f"{base_url}:batchUpdate",\n'
    '                access_token,\n'
    '                {\n'
    '                    "requests": [\n'
    '                        {\n'
    '                            "updateSheetProperties": {\n'
    '                                "properties": {\n'
    '                                    "sheetId": legacy_sheet["properties"]["sheetId"],\n'
    '                                    "title": sheet_name,\n'
    '                                },\n'
    '                                "fields": "title",\n'
    '                            }\n'
    '                        }\n'
    '                    ]\n'
    '                },\n'
    '            )\n'
    '            return\n'
    '    google_sheets_request(\n'
    '        "POST",\n'
    '        f"{base_url}:batchUpdate",\n'
    '        access_token,\n'
    '        {"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},\n'
    '    )\n'
    '\n'
    '\n'
    'def update_google_sheet_direct(\n'
    '    spreadsheet_id: str,\n'
    '    credentials_path: Path,\n'
    '    indicators: list[tuple[str, list[dict]]],\n'
    '    sheet_name: str,\n'
    '    metadata_values: list[list[object]],\n'
    '    validation_values: list[list[object]],\n'
    '    diagnostic_values: list[list[object]],\n'
    ') -> None:\n'
    '    access_token = get_service_account_access_token(credentials_path)\n'
    '    base_url = f"{GOOGLE_SHEETS_API_BASE}/{spreadsheet_id}"\n'
    '    tables = {\n'
    '        sheet_name: sheet_values_for_google(indicators),\n'
    '        DEFAULT_GOOGLE_STATUS_SHEET_NAME: source_status_values(),\n'
    '        DEFAULT_GOOGLE_METADATA_SHEET_NAME: metadata_values,\n'
    '        DEFAULT_GOOGLE_VALIDATION_SHEET_NAME: validation_values,\n'
    '        DEFAULT_GOOGLE_DIAGNOSTIC_SHEET_NAME: diagnostic_values,\n'
    '    }\n'
    '    for table_name in tables:\n'
    '        ensure_google_sheet_direct(\n'
    '            base_url,\n'
    '            access_token,\n'
    '            table_name,\n'
    '            allow_legacy_rename=table_name == sheet_name,\n'
    '        )\n'
    '    range_urls = {\n'
    '        table_name: f"{base_url}/values/{quote(repr(table_name), safe=\'\')}"\n'
    '        for table_name in tables\n'
    '    }\n'
    '    old_tables = {\n'
    '        table_name: google_sheets_request(\n'
    '            "GET",\n'
    '            range_urls[table_name],\n'
    '            access_token,\n'
    '        ).get("values", [])\n'
    '        for table_name in tables\n'
    '    }\n'
    '    try:\n'
    '        for table_name, table_values in tables.items():\n'
    '            range_url = range_urls[table_name]\n'
    '            google_sheets_request("POST", f"{range_url}:clear", access_token, {})\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/"\n'
    '                f"{quote(f\'{table_name!r}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"values": table_values},\n'
    '            )\n'
    '            written = google_sheets_request(\n'
    '                "GET",\n'
    '                range_url,\n'
    '                access_token,\n'
    '            ).get("values", [])\n'
    '            verify_google_sheet_values(table_values, written)\n'
    '    except Exception:\n'
    '        logging.exception(\n'
    '            "Google Sheets update failed; restoring the previous sheet contents."\n'
    '        )\n'
    '        for table_name, old_values in old_tables.items():\n'
    '            range_url = range_urls[table_name]\n'
    '            google_sheets_request("POST", f"{range_url}:clear", access_token, {})\n'
    '            if old_values:\n'
    '                google_sheets_request(\n'
    '                    "PUT",\n'
    '                    f"{base_url}/values/"\n'
    '                    f"{quote(f\'{table_name!r}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '                    access_token,\n'
    '                    {"values": old_values},\n'
    '                )\n'
    '        raise\n'
    '\n'
    '\n'
    'def sync_indicator_sheet_with_service(\n'
    '    sheets,\n'
    '    spreadsheet_id: str,\n'
    '    sheet_name: str,\n'
    '    indicator_name: str,\n'
    '    records: list[dict],\n'
    '    validate_only: bool = False,\n'
    ') -> dict:\n'
    '    """Append newer observations to one indicator\'s A:B tab without rewriting history."""\n'
    '    values = (\n'
    '        sheets.values()\n'
    '        .get(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=f"{quote_sheet_name(sheet_name)}!A:B",\n'
    '            valueRenderOption="UNFORMATTED_VALUE",\n'
    '        )\n'
    '        .execute()\n'
    '        .get("values", [])\n'
    '    )\n'
    '    layout = get_sheet_layout(values, indicator_name)\n'
    '    existing = read_existing_indicator_data(values, layout)\n'
    '    incoming = sheet_rows_for_indicator(indicator_name, records)\n'
    '    if not incoming:\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            max(existing, default=None),\n'
    '            0,\n'
    '            0,\n'
    '            max(existing, default=None),\n'
    '            "No valid source observation available; left unchanged.",\n'
    '        )\n'
    '    validate_overlapping_indicator_values(\n'
    '        existing,\n'
    '        incoming,\n'
    '        f"Indicator sheet {sheet_name!r}",\n'
    '        allow_value_revisions=FULL_HISTORY_MODE,\n'
    '        mutable_months=mutable_output_months(),\n'
    '    )\n'
    '\n'
    '    if FULL_HISTORY_MODE:\n'
    '        verified_invalid = _VERIFIED_INVALID_EXISTING_MONTHS.get(indicator_name, set())\n'
    '        rebuild_rows = {\n'
    '            observed: item["value"]\n'
    '            for observed, item in existing.items()\n'
    '            if item.get("value") is not None and observed not in verified_invalid\n'
    '        }\n'
    '        rebuild_rows.update(incoming)\n'
    '        header = [values[0][0], values[0][1]]\n'
    '        table_values = [\n'
    '            header,\n'
    '            *[\n'
    '                [format_output_month(observed), value]\n'
    '                for observed, value in sorted(rebuild_rows.items())\n'
    '            ],\n'
    '        ]\n'
    '        validate_historical_table_replacement(\n'
    '            values,\n'
    '            table_values,\n'
    '            f"Indicator sheet {sheet_name!r}",\n'
    '            allow_absent_existing_dates=bool(verified_invalid),\n'
    '            allow_value_revisions=True,\n'
    '            mutable_months=mutable_output_months(),\n'
    '        )\n'
    '        if indicator_input_table_is_current(values, table_values):\n'
    '            return sync_result_row(\n'
    '                indicator_name,\n'
    '                sheet_name,\n'
    '                max(existing, default=None),\n'
    '                0,\n'
    '                0,\n'
    '                max(rebuild_rows, default=None),\n'
    '            )\n'
    '        if validate_only:\n'
    '            return sync_result_row(\n'
    '                indicator_name,\n'
    '                sheet_name,\n'
    '                max(existing, default=None),\n'
    '                len(rebuild_rows),\n'
    '                0,\n'
    '                max(rebuild_rows, default=None),\n'
    '                "DRY RUN: would rebuild the reconciled A:B history.",\n'
    '            )\n'
    '        range_name = f"{quote_sheet_name(sheet_name)}!A:B"\n'
    '        old_values = values\n'
    '        try:\n'
    '            current_values = (\n'
    '                sheets.values()\n'
    '                .get(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=range_name,\n'
    '                    valueRenderOption="UNFORMATTED_VALUE",\n'
    '                )\n'
    '                .execute()\n'
    '                .get("values", [])\n'
    '            )\n'
    '            if current_values != old_values:\n'
    '                raise RuntimeError(\n'
    '                    f"Indicator sheet {sheet_name!r} changed after preflight; refusing stale overwrite."\n'
    '                )\n'
    '            sheets.values().clear(\n'
    '                spreadsheetId=spreadsheet_id, range=range_name, body={}\n'
    '            ).execute()\n'
    '            sheets.values().update(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"{quote_sheet_name(sheet_name)}!A1",\n'
    '                valueInputOption="USER_ENTERED",\n'
    '                body={"values": table_values},\n'
    '            ).execute()\n'
    '            written = (\n'
    '                sheets.values()\n'
    '                .get(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=range_name,\n'
    '                    valueRenderOption="UNFORMATTED_VALUE",\n'
    '                )\n'
    '                .execute()\n'
    '                .get("values", [])\n'
    '            )\n'
    '            verify_google_sheet_values(table_values, written)\n'
    '        except Exception:\n'
    '            sheets.values().clear(\n'
    '                spreadsheetId=spreadsheet_id, range=range_name, body={}\n'
    '            ).execute()\n'
    '            if old_values:\n'
    '                sheets.values().update(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=f"{quote_sheet_name(sheet_name)}!A1",\n'
    '                    valueInputOption="USER_ENTERED",\n'
    '                    body={"values": old_values},\n'
    '                ).execute()\n'
    '            raise\n'
    '        from notifications.changes import record, compare\n'
    '\n'
    '        dates = sorted(set(existing) | set(rebuild_rows))\n'
    '        before = [[existing.get(d, {}).get("value")] for d in dates]\n'
    '        after = [[rebuild_rows.get(d)] for d in dates]\n'
    '        record("Source data", indicator_name, **compare(before, after))\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            max(existing, default=None),\n'
    '            len(rebuild_rows),\n'
    '            0,\n'
    '            max(rebuild_rows, default=None),\n'
    '            "Rebuilt reconciled A:B history.",\n'
    '        )\n'
    '\n'
    '    fills, appends = prepare_sheet_changes(existing, incoming)\n'
    '    latest_before = max(existing, default=None)\n'
    '    latest_after = max(set(existing) | set(incoming), default=None)\n'
    '    if not fills and not appends:\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            0,\n'
    '            0,\n'
    '            latest_before,\n'
    '            "No newer source observation available; left unchanged.",\n'
    '        )\n'
    '\n'
    '    if validate_only:\n'
    '        preview = ", ".join(\n'
    '            f"{format_output_month(observed)}={value}" for observed, value in appends\n'
    '        )\n'
    '        fill_preview = ", ".join(\n'
    '            f"row {row} {format_output_month(observed)}={value}"\n'
    '            for row, observed, value in fills\n'
    '        )\n'
    '        pieces = []\n'
    '        if fill_preview:\n'
    '            pieces.append(f"update cells {fill_preview}")\n'
    '        if preview:\n'
    '            pieces.append(f"append {preview}")\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            len(fills),\n'
    '            len(appends),\n'
    '            latest_after,\n'
    '            "DRY RUN: would " + "; ".join(pieces) + ".",\n'
    '        )\n'
    '\n'
    '    if fills:\n'
    '        value_col_letter = column_letter(1)\n'
    '        sheets.values().batchUpdate(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            body={\n'
    '                "valueInputOption": "USER_ENTERED",\n'
    '                "data": [\n'
    '                    {\n'
    '                        "range": f"{quote_sheet_name(sheet_name)}!{value_col_letter}{row_number}",\n'
    '                        "values": [[value]],\n'
    '                    }\n'
    '                    for row_number, _observed, value in fills\n'
    '                ],\n'
    '            },\n'
    '        ).execute()\n'
    '\n'
    '    rows = [[format_output_month(observed), value] for observed, value in appends]\n'
    '    if rows:\n'
    '        sheets.values().append(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=f"{quote_sheet_name(sheet_name)}!A:B",\n'
    '            valueInputOption="USER_ENTERED",\n'
    '            insertDataOption="INSERT_ROWS",\n'
    '            body={"values": rows},\n'
    '        ).execute()\n'
    '\n'
    '    return sync_result_row(\n'
    '        indicator_name,\n'
    '        sheet_name,\n'
    '        latest_before,\n'
    '        len(fills),\n'
    '        len(appends),\n'
    '        latest_after,\n'
    '    )\n'
    '\n'
    '\n'
    'def sync_indicator_sheet_direct(\n'
    '    base_url: str,\n'
    '    access_token: str,\n'
    '    sheet_name: str,\n'
    '    indicator_name: str,\n'
    '    records: list[dict],\n'
    '    validate_only: bool = False,\n'
    ') -> dict:\n'
    '    range_name = quote_sheet_name(sheet_name)\n'
    '    values = google_sheets_request(\n'
    '        "GET",\n'
    '        google_values_url(base_url, f"{range_name}!A:B"),\n'
    '        access_token,\n'
    '    ).get("values", [])\n'
    '    layout = get_sheet_layout(values, indicator_name)\n'
    '    existing = read_existing_indicator_data(values, layout)\n'
    '    incoming = sheet_rows_for_indicator(indicator_name, records)\n'
    '    if not incoming:\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            max(existing, default=None),\n'
    '            0,\n'
    '            0,\n'
    '            max(existing, default=None),\n'
    '            "No valid source observation available; left unchanged.",\n'
    '        )\n'
    '    validate_overlapping_indicator_values(\n'
    '        existing,\n'
    '        incoming,\n'
    '        f"Indicator sheet {sheet_name!r}",\n'
    '        allow_value_revisions=FULL_HISTORY_MODE,\n'
    '        mutable_months=mutable_output_months(),\n'
    '    )\n'
    '\n'
    '    if FULL_HISTORY_MODE:\n'
    '        verified_invalid = _VERIFIED_INVALID_EXISTING_MONTHS.get(indicator_name, set())\n'
    '        rebuild_rows = {\n'
    '            observed: item["value"]\n'
    '            for observed, item in existing.items()\n'
    '            if item.get("value") is not None and observed not in verified_invalid\n'
    '        }\n'
    '        rebuild_rows.update(incoming)\n'
    '        header = [values[0][0], values[0][1]]\n'
    '        table_values = [\n'
    '            header,\n'
    '            *[\n'
    '                [format_output_month(observed), value]\n'
    '                for observed, value in sorted(rebuild_rows.items())\n'
    '            ],\n'
    '        ]\n'
    '        validate_historical_table_replacement(\n'
    '            values,\n'
    '            table_values,\n'
    '            f"Indicator sheet {sheet_name!r}",\n'
    '            allow_absent_existing_dates=bool(verified_invalid),\n'
    '            allow_value_revisions=True,\n'
    '            mutable_months=mutable_output_months(),\n'
    '        )\n'
    '        if indicator_input_table_is_current(values, table_values):\n'
    '            return sync_result_row(\n'
    '                indicator_name,\n'
    '                sheet_name,\n'
    '                max(existing, default=None),\n'
    '                0,\n'
    '                0,\n'
    '                max(rebuild_rows, default=None),\n'
    '            )\n'
    '        if validate_only:\n'
    '            return sync_result_row(\n'
    '                indicator_name,\n'
    '                sheet_name,\n'
    '                max(existing, default=None),\n'
    '                len(rebuild_rows),\n'
    '                0,\n'
    '                max(rebuild_rows, default=None),\n'
    '                "DRY RUN: would rebuild the reconciled A:B history.",\n'
    '            )\n'
    '        clear_url = f"{base_url}/values/" f"{quote(f\'{range_name}!A:B\', safe=\'\')}:clear"\n'
    '        old_values = values\n'
    '        try:\n'
    '            current_values = google_sheets_request(\n'
    '                "GET", google_values_url(base_url, f"{range_name}!A:B"), access_token\n'
    '            ).get("values", [])\n'
    '            if current_values != old_values:\n'
    '                raise RuntimeError(\n'
    '                    f"Indicator sheet {sheet_name!r} changed after preflight; refusing stale overwrite."\n'
    '                )\n'
    '            google_sheets_request("POST", clear_url, access_token, {})\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/{quote(f\'{range_name}!A1\', safe=\'\')}?valueInputOption=USER_ENTERED",\n'
    '                access_token,\n'
    '                {"values": table_values},\n'
    '            )\n'
    '            written = google_sheets_request(\n'
    '                "GET", google_values_url(base_url, f"{range_name}!A:B"), access_token\n'
    '            ).get("values", [])\n'
    '            verify_google_sheet_values(table_values, written)\n'
    '        except Exception:\n'
    '            google_sheets_request("POST", clear_url, access_token, {})\n'
    '            if old_values:\n'
    '                google_sheets_request(\n'
    '                    "PUT",\n'
    '                    f"{base_url}/values/{quote(f\'{range_name}!A1\', safe=\'\')}?valueInputOption=USER_ENTERED",\n'
    '                    access_token,\n'
    '                    {"values": old_values},\n'
    '                )\n'
    '            raise\n'
    '        from notifications.changes import record, compare\n'
    '\n'
    '        dates = sorted(set(existing) | set(rebuild_rows))\n'
    '        before = [[existing.get(d, {}).get("value")] for d in dates]\n'
    '        after = [[rebuild_rows.get(d)] for d in dates]\n'
    '        record("Source data", indicator_name, **compare(before, after))\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            max(existing, default=None),\n'
    '            len(rebuild_rows),\n'
    '            0,\n'
    '            max(rebuild_rows, default=None),\n'
    '            "Rebuilt reconciled A:B history.",\n'
    '        )\n'
    '\n'
    '    fills, appends = prepare_sheet_changes(existing, incoming)\n'
    '    latest_before = max(existing, default=None)\n'
    '    latest_after = max(set(existing) | set(incoming), default=None)\n'
    '    if not fills and not appends:\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            0,\n'
    '            0,\n'
    '            latest_before,\n'
    '            "No newer source observation available; left unchanged.",\n'
    '        )\n'
    '\n'
    '    if validate_only:\n'
    '        preview = ", ".join(\n'
    '            f"{format_output_month(observed)}={value}" for observed, value in appends\n'
    '        )\n'
    '        fill_preview = ", ".join(\n'
    '            f"row {row} {format_output_month(observed)}={value}"\n'
    '            for row, observed, value in fills\n'
    '        )\n'
    '        pieces = []\n'
    '        if fill_preview:\n'
    '            pieces.append(f"update cells {fill_preview}")\n'
    '        if preview:\n'
    '            pieces.append(f"append {preview}")\n'
    '        return sync_result_row(\n'
    '            indicator_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            len(fills),\n'
    '            len(appends),\n'
    '            latest_after,\n'
    '            "DRY RUN: would " + "; ".join(pieces) + ".",\n'
    '        )\n'
    '\n'
    '    if fills:\n'
    '        value_col_letter = column_letter(1)\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}/values:batchUpdate",\n'
    '            access_token,\n'
    '            {\n'
    '                "valueInputOption": "USER_ENTERED",\n'
    '                "data": [\n'
    '                    {\n'
    '                        "range": f"{range_name}!{value_col_letter}{row_number}",\n'
    '                        "values": [[value]],\n'
    '                    }\n'
    '                    for row_number, _observed, value in fills\n'
    '                ],\n'
    '            },\n'
    '        )\n'
    '\n'
    '    rows = [[format_output_month(observed), value] for observed, value in appends]\n'
    '    if rows:\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}/values/{quote(f\'{range_name}!A:B\', safe=\'\')}:append"\n'
    '            "?valueInputOption=USER_ENTERED&insertDataOption=INSERT_ROWS",\n'
    '            access_token,\n'
    '            {"values": rows},\n'
    '        )\n'
    '\n'
    '    return sync_result_row(\n'
    '        indicator_name,\n'
    '        sheet_name,\n'
    '        latest_before,\n'
    '        len(fills),\n'
    '        len(appends),\n'
    '        latest_after,\n'
    '    )\n'
    '\n'
    '\n'
    'def central_bank_append_rows(\n'
    '    values: list[list[object]],\n'
    '    component_data: dict[str, list[dict]],\n'
    ') -> tuple[\n'
    '    date | None, list[tuple[int, int, date, float]], list[list[object]], date | None\n'
    ']:\n'
    '    """Prepare blank fills and append-only rows for the shared central-bank tab."""\n'
    '    if values:\n'
    '        layout = get_sheet_layout(values, CENTRAL_BANK_LIQUIDITY_SHEET)\n'
    '    else:\n'
    '        layout = {"date_col": 0, "value_col": 1, "header_row": 0}\n'
    '    existing = read_existing_indicator_data(values, layout)\n'
    '    latest_before = max(existing, default=None)\n'
    '\n'
    '    existing_rows: dict[date, dict] = {}\n'
    '    for row_index, row in enumerate(values[1:], start=2):\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed is not None:\n'
    '            existing_rows[observed] = {"row": row_index, "values": row}\n'
    '\n'
    '    fills: list[tuple[int, int, date, float]] = []\n'
    '    by_month: dict[date, list[object]] = {}\n'
    '    for indicator_name, records in component_data.items():\n'
    '        target = CENTRAL_BANK_LIQUIDITY_COMPONENTS[indicator_name]\n'
    '        for month_key, value in output_month_records_for_indicator(records).items():\n'
    '            existing_row = existing_rows.get(month_key)\n'
    '            if existing_row is not None:\n'
    '                row_values = existing_row["values"]\n'
    '                raw_value = (\n'
    '                    row_values[target["column"]]\n'
    '                    if len(row_values) > target["column"]\n'
    '                    else None\n'
    '                )\n'
    '                existing_value = normalise_value(raw_value)\n'
    '                if existing_value is not None and not google_values_equivalent(\n'
    '                    value, existing_value\n'
    '                ):\n'
    '                    if FULL_HISTORY_MODE:\n'
    '                        raise RuntimeError(\n'
    '                            f"{CENTRAL_BANK_LIQUIDITY_SHEET}: historical overlap does not match for "\n'
    '                            f"{indicator_name} {month_key.isoformat()}: existing={raw_value!r}, incoming={value!r}."\n'
    '                        )\n'
    '                    continue\n'
    '                if existing_value is None:\n'
    '                    fills.append(\n'
    '                        (existing_row["row"], target["column"], month_key, value)\n'
    '                    )\n'
    '                continue\n'
    '            if latest_before is not None and month_key <= latest_before:\n'
    '                continue\n'
    '            by_month.setdefault(month_key, [format_output_month(month_key), "", "", ""])\n'
    '            by_month[month_key][target["column"]] = value\n'
    '\n'
    '    rows = [row for _month, row in sorted(by_month.items()) if any(row[1:])]\n'
    '    latest_after = max(\n'
    '        [latest_before] + sorted(by_month) if latest_before else sorted(by_month),\n'
    '        default=latest_before,\n'
    '    )\n'
    '    return latest_before, fills, rows, latest_after\n'
    '\n'
    '\n'
    'def central_bank_source_change_counts(\n'
    '    fills: list[tuple[int, int, date, float]], rows: list[list[object]]\n'
    ') -> dict[str, int]:\n'
    '    """Report blank backfills as additions, never as source-value revisions."""\n'
    '    return {\n'
    '        "added": len(fills)\n'
    '        + sum(value not in (None, "") for row in rows for value in row[1:]),\n'
    '        "updated": 0,\n'
    '        "removed": 0,\n'
    '    }\n'
    '\n'
    '\n'
    'def central_bank_full_table_values(\n'
    '    values: list[list[object]],\n'
    '    component_data: dict[str, list[dict]],\n'
    ') -> list[list[object]]:\n'
    '    """Build the complete shared central-bank A:D table from source histories."""\n'
    '    header = (\n'
    '        list(values[0][:4])\n'
    '        if values\n'
    '        else [\n'
    '            "Month-End Date",\n'
    '            *[\n'
    '                target["header"]\n'
    '                for target in CENTRAL_BANK_LIQUIDITY_COMPONENTS.values()\n'
    '            ],\n'
    '        ]\n'
    '    )\n'
    '    header.extend([""] * (4 - len(header)))\n'
    '    by_month: dict[date, list[object]] = {}\n'
    '    for row in values[1:]:\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed is None:\n'
    '            continue\n'
    '        preserved = [format_output_month(observed), "", "", ""]\n'
    '        for column_index in range(1, 4):\n'
    '            if len(row) > column_index:\n'
    '                preserved[column_index] = row[column_index]\n'
    '        by_month[observed] = preserved\n'
    '    for indicator_name, records in component_data.items():\n'
    '        target = CENTRAL_BANK_LIQUIDITY_COMPONENTS[indicator_name]\n'
    '        for month_key, value in output_month_records_for_indicator(records).items():\n'
    '            row = by_month.setdefault(\n'
    '                month_key,\n'
    '                [format_output_month(month_key), "", "", ""],\n'
    '            )\n'
    '            row[target["column"]] = value\n'
    '    return [header, *[row for _month, row in sorted(by_month.items())]]\n'
    '\n'
    '\n'
    'def sync_central_bank_liquidity_with_service(\n'
    '    sheets,\n'
    '    spreadsheet_id: str,\n'
    '    component_data: dict[str, list[dict]],\n'
    '    sheet_names: set[str],\n'
    '    validate_only: bool,\n'
    ') -> dict:\n'
    '    sheet_name = CENTRAL_BANK_LIQUIDITY_SHEET\n'
    '    if sheet_name not in sheet_names:\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            "",\n'
    '            None,\n'
    '            0,\n'
    '            0,\n'
    '            None,\n'
    '            "Missing matching central-bank liquidity tab; skipped.",\n'
    '        )\n'
    '    values = (\n'
    '        sheets.values()\n'
    '        .get(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=f"{quote_sheet_name(sheet_name)}!A:D",\n'
    '            valueRenderOption="UNFORMATTED_VALUE",\n'
    '        )\n'
    '        .execute()\n'
    '        .get("values", [])\n'
    '    )\n'
    '    if FULL_HISTORY_MODE:\n'
    '        complete_history = bool(component_data) and all(\n'
    '            has_complete_source_history(records) for records in component_data.values()\n'
    '        )\n'
    '        table_values = central_bank_full_table_values(values, component_data)\n'
    '        validate_historical_table_replacement(\n'
    '            values,\n'
    '            table_values,\n'
    '            f"Shared sheet {sheet_name!r}",\n'
    '            allow_absent_existing_dates=False,\n'
    '            allow_undated_existing_rows=complete_history,\n'
    '            allow_value_revisions=complete_history,\n'
    '            mutable_months=mutable_output_months(),\n'
    '        )\n'
    '        latest_before = max(\n'
    '            read_existing_indicator_data(values, get_sheet_layout(values, sheet_name)),\n'
    '            default=None,\n'
    '        )\n'
    '        latest_after = max(\n'
    '            (normalise_observation_date(row[0]) for row in table_values[1:]),\n'
    '            default=None,\n'
    '        )\n'
    '        if google_sheet_table_is_current(table_values, values):\n'
    '            return sync_result_row(\n'
    '                sheet_name, sheet_name, latest_before, 0, 0, latest_after\n'
    '            )\n'
    '        if validate_only:\n'
    '            return sync_result_row(\n'
    '                sheet_name,\n'
    '                sheet_name,\n'
    '                latest_before,\n'
    '                len(table_values) - 1,\n'
    '                0,\n'
    '                latest_after,\n'
    '                "DRY RUN: would rebuild the complete shared A:D history.",\n'
    '            )\n'
    '        range_name = f"{quote_sheet_name(sheet_name)}!A:D"\n'
    '        try:\n'
    '            current_values = (\n'
    '                sheets.values()\n'
    '                .get(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=range_name,\n'
    '                    valueRenderOption="UNFORMATTED_VALUE",\n'
    '                )\n'
    '                .execute()\n'
    '                .get("values", [])\n'
    '            )\n'
    '            if current_values != values:\n'
    '                raise RuntimeError(\n'
    '                    f"Shared sheet {sheet_name!r} changed after preflight; "\n'
    '                    "refusing stale overwrite."\n'
    '                )\n'
    '            sheets.values().clear(\n'
    '                spreadsheetId=spreadsheet_id, range=range_name, body={}\n'
    '            ).execute()\n'
    '            sheets.values().update(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"{quote_sheet_name(sheet_name)}!A1",\n'
    '                valueInputOption="USER_ENTERED",\n'
    '                body={"values": table_values},\n'
    '            ).execute()\n'
    '            written = (\n'
    '                sheets.values()\n'
    '                .get(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=range_name,\n'
    '                    valueRenderOption="UNFORMATTED_VALUE",\n'
    '                )\n'
    '                .execute()\n'
    '                .get("values", [])\n'
    '            )\n'
    '            verify_google_sheet_values(table_values, written)\n'
    '        except Exception:\n'
    '            sheets.values().clear(\n'
    '                spreadsheetId=spreadsheet_id, range=range_name, body={}\n'
    '            ).execute()\n'
    '            if values:\n'
    '                sheets.values().update(\n'
    '                    spreadsheetId=spreadsheet_id,\n'
    '                    range=f"{quote_sheet_name(sheet_name)}!A1",\n'
    '                    valueInputOption="USER_ENTERED",\n'
    '                    body={"values": values},\n'
    '                ).execute()\n'
    '            raise\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            len(table_values) - 1,\n'
    '            0,\n'
    '            latest_after,\n'
    '            "Rebuilt complete shared A:D history.",\n'
    '        )\n'
    '    latest_before, fills, rows, latest_after = central_bank_append_rows(\n'
    '        values, component_data\n'
    '    )\n'
    '    if not fills and not rows:\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            0,\n'
    '            0,\n'
    '            latest_before,\n'
    '            "No newer source observation available; left unchanged.",\n'
    '        )\n'
    '    if validate_only:\n'
    '        preview = "; ".join(str(row) for row in rows)\n'
    '        fill_preview = "; ".join(\n'
    '            f"row {row} {column_letter(col)} {format_output_month(month)}={value}"\n'
    '            for row, col, month, value in fills\n'
    '        )\n'
    '        pieces = []\n'
    '        if fill_preview:\n'
    '            pieces.append(f"update cells {fill_preview}")\n'
    '        if preview:\n'
    '            pieces.append(f"append {preview}")\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            len(fills),\n'
    '            len(rows),\n'
    '            latest_after,\n'
    '            "DRY RUN: would " + "; ".join(pieces) + ".",\n'
    '            source_change_counts=central_bank_source_change_counts(fills, rows),\n'
    '        )\n'
    '    if fills:\n'
    '        sheets.values().batchUpdate(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            body={\n'
    '                "valueInputOption": "USER_ENTERED",\n'
    '                "data": [\n'
    '                    {\n'
    '                        "range": f"{quote_sheet_name(sheet_name)}!{column_letter(col)}{row}",\n'
    '                        "values": [[value]],\n'
    '                    }\n'
    '                    for row, col, _month, value in fills\n'
    '                ],\n'
    '            },\n'
    '        ).execute()\n'
    '    if rows:\n'
    '        sheets.values().append(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=f"{quote_sheet_name(sheet_name)}!A:D",\n'
    '            valueInputOption="USER_ENTERED",\n'
    '            insertDataOption="INSERT_ROWS",\n'
    '            body={"values": rows},\n'
    '        ).execute()\n'
    '    return sync_result_row(\n'
    '        sheet_name,\n'
    '        sheet_name,\n'
    '        latest_before,\n'
    '        len(fills),\n'
    '        len(rows),\n'
    '        latest_after,\n'
    '        source_change_counts=central_bank_source_change_counts(fills, rows),\n'
    '    )\n'
    '\n'
    '\n'
    'def sync_central_bank_liquidity_direct(\n'
    '    base_url: str,\n'
    '    access_token: str,\n'
    '    component_data: dict[str, list[dict]],\n'
    '    sheet_names: set[str],\n'
    '    validate_only: bool,\n'
    ') -> dict:\n'
    '    sheet_name = CENTRAL_BANK_LIQUIDITY_SHEET\n'
    '    if sheet_name not in sheet_names:\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            "",\n'
    '            None,\n'
    '            0,\n'
    '            0,\n'
    '            None,\n'
    '            "Missing matching central-bank liquidity tab; skipped.",\n'
    '        )\n'
    '    range_name = quote_sheet_name(sheet_name)\n'
    '    values = google_sheets_request(\n'
    '        "GET",\n'
    '        google_values_url(base_url, f"{range_name}!A:D"),\n'
    '        access_token,\n'
    '    ).get("values", [])\n'
    '    if FULL_HISTORY_MODE:\n'
    '        complete_history = bool(component_data) and all(\n'
    '            has_complete_source_history(records) for records in component_data.values()\n'
    '        )\n'
    '        table_values = central_bank_full_table_values(values, component_data)\n'
    '        validate_historical_table_replacement(\n'
    '            values,\n'
    '            table_values,\n'
    '            f"Shared sheet {sheet_name!r}",\n'
    '            allow_absent_existing_dates=False,\n'
    '            allow_undated_existing_rows=complete_history,\n'
    '            allow_value_revisions=complete_history,\n'
    '            mutable_months=mutable_output_months(),\n'
    '        )\n'
    '        latest_before = max(\n'
    '            read_existing_indicator_data(values, get_sheet_layout(values, sheet_name)),\n'
    '            default=None,\n'
    '        )\n'
    '        latest_after = max(\n'
    '            (normalise_observation_date(row[0]) for row in table_values[1:]),\n'
    '            default=None,\n'
    '        )\n'
    '        if google_sheet_table_is_current(table_values, values):\n'
    '            return sync_result_row(\n'
    '                sheet_name, sheet_name, latest_before, 0, 0, latest_after\n'
    '            )\n'
    '        if validate_only:\n'
    '            return sync_result_row(\n'
    '                sheet_name,\n'
    '                sheet_name,\n'
    '                latest_before,\n'
    '                len(table_values) - 1,\n'
    '                0,\n'
    '                latest_after,\n'
    '                "DRY RUN: would rebuild the complete shared A:D history.",\n'
    '            )\n'
    '        full_range = f"{range_name}!A:D"\n'
    '        clear_url = f"{base_url}/values/{quote(full_range, safe=\'\')}:clear"\n'
    '        try:\n'
    '            current_values = google_sheets_request(\n'
    '                "GET", google_values_url(base_url, full_range), access_token\n'
    '            ).get("values", [])\n'
    '            if current_values != values:\n'
    '                raise RuntimeError(\n'
    '                    f"Shared sheet {sheet_name!r} changed after preflight; "\n'
    '                    "refusing stale overwrite."\n'
    '                )\n'
    '            google_sheets_request("POST", clear_url, access_token, {})\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/{quote(f\'{range_name}!A1\', safe=\'\')}?valueInputOption=USER_ENTERED",\n'
    '                access_token,\n'
    '                {"values": table_values},\n'
    '            )\n'
    '            written = google_sheets_request(\n'
    '                "GET", google_values_url(base_url, full_range), access_token\n'
    '            ).get("values", [])\n'
    '            verify_google_sheet_values(table_values, written)\n'
    '        except Exception:\n'
    '            google_sheets_request("POST", clear_url, access_token, {})\n'
    '            if values:\n'
    '                google_sheets_request(\n'
    '                    "PUT",\n'
    '                    f"{base_url}/values/{quote(f\'{range_name}!A1\', safe=\'\')}?valueInputOption=USER_ENTERED",\n'
    '                    access_token,\n'
    '                    {"values": values},\n'
    '                )\n'
    '            raise\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            len(table_values) - 1,\n'
    '            0,\n'
    '            latest_after,\n'
    '            "Rebuilt complete shared A:D history.",\n'
    '        )\n'
    '    latest_before, fills, rows, latest_after = central_bank_append_rows(\n'
    '        values, component_data\n'
    '    )\n'
    '    if not fills and not rows:\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            0,\n'
    '            0,\n'
    '            latest_before,\n'
    '            "No newer source observation available; left unchanged.",\n'
    '        )\n'
    '    if validate_only:\n'
    '        preview = "; ".join(str(row) for row in rows)\n'
    '        fill_preview = "; ".join(\n'
    '            f"row {row} {column_letter(col)} {format_output_month(month)}={value}"\n'
    '            for row, col, month, value in fills\n'
    '        )\n'
    '        pieces = []\n'
    '        if fill_preview:\n'
    '            pieces.append(f"update cells {fill_preview}")\n'
    '        if preview:\n'
    '            pieces.append(f"append {preview}")\n'
    '        return sync_result_row(\n'
    '            sheet_name,\n'
    '            sheet_name,\n'
    '            latest_before,\n'
    '            len(fills),\n'
    '            len(rows),\n'
    '            latest_after,\n'
    '            "DRY RUN: would " + "; ".join(pieces) + ".",\n'
    '            source_change_counts=central_bank_source_change_counts(fills, rows),\n'
    '        )\n'
    '    if fills:\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}/values:batchUpdate",\n'
    '            access_token,\n'
    '            {\n'
    '                "valueInputOption": "USER_ENTERED",\n'
    '                "data": [\n'
    '                    {\n'
    '                        "range": f"{range_name}!{column_letter(col)}{row}",\n'
    '                        "values": [[value]],\n'
    '                    }\n'
    '                    for row, col, _month, value in fills\n'
    '                ],\n'
    '            },\n'
    '        )\n'
    '    if rows:\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}/values/{quote(f\'{range_name}!A:D\', safe=\'\')}:append"\n'
    '            "?valueInputOption=USER_ENTERED&insertDataOption=INSERT_ROWS",\n'
    '            access_token,\n'
    '            {"values": rows},\n'
    '        )\n'
    '    return sync_result_row(\n'
    '        sheet_name,\n'
    '        sheet_name,\n'
    '        latest_before,\n'
    '        len(fills),\n'
    '        len(rows),\n'
    '        latest_after,\n'
    '        source_change_counts=central_bank_source_change_counts(fills, rows),\n'
    '    )\n'
    '\n'
    '\n'
    'def print_sync_summary(results: list[dict], *, validate_only: bool = False) -> None:\n'
    '    """Print a concise per-indicator sync summary."""\n'
    '    if not validate_only:\n'
    '        from notifications.changes import record\n'
    '\n'
    '        for item in results:\n'
    '            warning = str(item.get("warning", ""))\n'
    '            if warning.startswith("Rebuilt reconciled A:B history"):\n'
    '                continue  # Exact before/after counts were recorded at the write.\n'
    '            counts = item.get("source_change_counts", {})\n'
    '            unknown = bool(\n'
    '                warning\n'
    '                and not warning.startswith(("No newer source", "No valid source"))\n'
    '            )\n'
    '            record(\n'
    '                "Source data",\n'
    '                item["indicator"],\n'
    '                added=counts.get("added", item.get("appended", 0)),\n'
    '                updated=0\n'
    '                if unknown\n'
    '                else counts.get("updated", item.get("revised", 0)),\n'
    '                removed=counts.get("removed", 0),\n'
    '                unknown=unknown,\n'
    '            )\n'
    '\n'
    '    if not results:\n'
    '        print("No indicator sheets were synced.")\n'
    '        return\n'
    '    if COMPACT_TERMINAL:\n'
    '        safe_warnings = (\n'
    '            "DRY RUN:",\n'
    '            "No newer source observation",\n'
    '            "No valid source observation",\n'
    '            "Rebuilt ",\n'
    '        )\n'
    '        problems = [\n'
    '            item\n'
    '            for item in results\n'
    '            if item.get("warning")\n'
    '            and not str(item["warning"]).startswith(safe_warnings)\n'
    '        ]\n'
    '        action = "checked" if validate_only else "synced"\n'
    '        print(\n'
    '            f"Indicator sheets: {len(results)} {action}, " f"{len(problems)} problems."\n'
    '        )\n'
    '        for item in problems:\n'
    '            print(f"  WARNING {item[\'indicator\']}: {item[\'warning\']}")\n'
    '        return\n'
    '    print_console_table(\n'
    '        "Indicator sheet sync summary:",\n'
    '        [\n'
    '            "Indicator",\n'
    '            "Sheet",\n'
    '            "Latest before",\n'
    '            "Changed cells",\n'
    '            "Appended",\n'
    '            "Latest after",\n'
    '            "No new value",\n'
    '            "Warning",\n'
    '        ],\n'
    '        [\n'
    '            [\n'
    '                item["indicator"],\n'
    '                item["sheet_name"],\n'
    '                item["latest_before"],\n'
    '                item["revised"],\n'
    '                item["appended"],\n'
    '                item["latest_after"],\n'
    '                item["no_new_value"],\n'
    '                item["warning"],\n'
    '            ]\n'
    '            for item in results\n'
    '        ],\n'
    '    )\n'
    '\n'
    '\n'
    'def assert_sync_preflight_passed(results: list[dict]) -> None:\n'
    '    """Turn per-tab safety failures into a pipeline-wide stop before any writes."""\n'
    '    safe_warnings = (\n'
    '        "DRY RUN:",\n'
    '        "No newer source observation",\n'
    '        "No valid source observation",\n'
    '        "Rebuilt ",\n'
    '    )\n'
    '    failures = [\n'
    '        f"{item[\'indicator\']}: {item[\'warning\']}"\n'
    '        for item in results\n'
    '        if item.get("warning") and not str(item["warning"]).startswith(safe_warnings)\n'
    '    ]\n'
    '    if failures:\n'
    '        raise RuntimeError(\n'
    '            "Indicator-sheet safety preflight failed; no writes were attempted: "\n'
    '            + " | ".join(failures)\n'
    '        )\n'
    '\n'
    '\n'
    'def sparse_date_rule_for_records(records: list[dict]) -> str | None:\n'
    '    """Return the expected date rule for sparse source records."""\n'
    '    if not records:\n'
    '        return None\n'
    '    first = records[0]\n'
    '    if first.get("source") == "IMF DataMapper FPP":\n'
    '        return "annual"\n'
    '    if first.get("source") != "FRED / ALFRED":\n'
    '        return None\n'
    '    frequency = str(first.get("native_frequency", ""))\n'
    '    if frequency.startswith("Quarterly"):\n'
    '        return "quarterly"\n'
    '    if frequency.startswith("Annual"):\n'
    '        return "annual"\n'
    '    return None\n'
    '\n'
    '\n'
    'def sparse_date_is_valid(observed: date, rule: str) -> bool:\n'
    '    if rule == "quarterly":\n'
    '        return observed.month in QUARTER_END_MONTHS\n'
    '    if rule == "annual":\n'
    '        return observed.month == ANNUAL_OUTPUT_MONTH\n'
    '    return True\n'
    '\n'
    '\n'
    'def sparse_replacement_date(observed: date, rule: str) -> date:\n'
    '    """Return the canonical row month for an old sparse-source row."""\n'
    '    if rule == "annual":\n'
    '        return date(observed.year, ANNUAL_OUTPUT_MONTH, 1)\n'
    '    if rule == "quarterly":\n'
    '        quarter_end_month = ((observed.month - 1) // 3 + 1) * 3\n'
    '        return date(observed.year, quarter_end_month, 1)\n'
    '    return month_start(observed)\n'
    '\n'
    '\n'
    'def sparse_cleanup_plan(\n'
    '    values: list[list[object]],\n'
    '    records: list[dict],\n'
    ') -> list[int]:\n'
    '    """Return sheet row numbers for old sparse rows that use the wrong month."""\n'
    '    rule = sparse_date_rule_for_records(records)\n'
    '    if rule is None:\n'
    '        return []\n'
    '    incoming_dates = set(output_month_records_for_indicator(records))\n'
    '    if not incoming_dates:\n'
    '        return []\n'
    '    rows_to_delete: list[int] = []\n'
    '    for row_index, row in enumerate(values[1:], start=2):\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed is None or sparse_date_is_valid(observed, rule):\n'
    '            continue\n'
    '        same_period_replacement = sparse_replacement_date(observed, rule)\n'
    '        if same_period_replacement in incoming_dates:\n'
    '            rows_to_delete.append(row_index)\n'
    '    return rows_to_delete\n'
    '\n'
    '\n'
    'def cleanup_sparse_date_rows_with_service(\n'
    '    sheets,\n'
    '    spreadsheet_id: str,\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    validate_only: bool,\n'
    ') -> list[dict]:\n'
    '    """Delete old off-quarter/off-year-end rows created by earlier sparse-date logic."""\n'
    '    metadata = sheets.get(spreadsheetId=spreadsheet_id).execute()\n'
    '    sheet_by_name = {\n'
    '        sheet["properties"]["title"]: sheet["properties"]["sheetId"]\n'
    '        for sheet in metadata.get("sheets", [])\n'
    '    }\n'
    '    results: list[dict] = []\n'
    '    requests: list[dict] = []\n'
    '    for indicator_name, records in workbook_data:\n'
    '        if is_central_bank_component(indicator_name) or is_imf_coverage_indicator(\n'
    '            indicator_name\n'
    '        ):\n'
    '            continue\n'
    '        if sparse_date_rule_for_records(records) is None:\n'
    '            continue\n'
    '        sheet_name = indicator_sheet_name(indicator_name, set(sheet_by_name))\n'
    '        if sheet_name is None:\n'
    '            continue\n'
    '        values = (\n'
    '            sheets.values()\n'
    '            .get(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"{quote_sheet_name(sheet_name)}!A:B",\n'
    '            )\n'
    '            .execute()\n'
    '            .get("values", [])\n'
    '        )\n'
    '        rows_to_delete = sparse_cleanup_plan(values, records)\n'
    '        if not rows_to_delete:\n'
    '            continue\n'
    '        for row_number in sorted(rows_to_delete, reverse=True):\n'
    '            requests.append(\n'
    '                {\n'
    '                    "deleteDimension": {\n'
    '                        "range": {\n'
    '                            "sheetId": sheet_by_name[sheet_name],\n'
    '                            "dimension": "ROWS",\n'
    '                            "startIndex": row_number - 1,\n'
    '                            "endIndex": row_number,\n'
    '                        }\n'
    '                    }\n'
    '                }\n'
    '            )\n'
    '        warning = (\n'
    '            "DRY RUN: would delete sparse misdated rows "\n'
    '            if validate_only\n'
    '            else "Deleted sparse misdated rows "\n'
    '        ) + ", ".join(str(row) for row in rows_to_delete)\n'
    '        results.append(\n'
    '            sync_result_row(\n'
    '                indicator_name,\n'
    '                sheet_name,\n'
    '                None,\n'
    '                len(rows_to_delete),\n'
    '                0,\n'
    '                None,\n'
    '                warning,\n'
    '            )\n'
    '        )\n'
    '    if requests and not validate_only:\n'
    '        sheets.batchUpdate(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            body={"requests": requests},\n'
    '        ).execute()\n'
    '    return results\n'
    '\n'
    '\n'
    'def cleanup_sparse_date_rows_direct(\n'
    '    base_url: str,\n'
    '    access_token: str,\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    validate_only: bool,\n'
    ') -> list[dict]:\n'
    '    """Direct Sheets API fallback for sparse-date cleanup."""\n'
    '    metadata = google_sheets_request("GET", base_url, access_token)\n'
    '    sheet_by_name = {\n'
    '        sheet["properties"]["title"]: sheet["properties"]["sheetId"]\n'
    '        for sheet in metadata.get("sheets", [])\n'
    '    }\n'
    '    results: list[dict] = []\n'
    '    requests: list[dict] = []\n'
    '    for indicator_name, records in workbook_data:\n'
    '        if is_central_bank_component(indicator_name) or is_imf_coverage_indicator(\n'
    '            indicator_name\n'
    '        ):\n'
    '            continue\n'
    '        if sparse_date_rule_for_records(records) is None:\n'
    '            continue\n'
    '        sheet_name = indicator_sheet_name(indicator_name, set(sheet_by_name))\n'
    '        if sheet_name is None:\n'
    '            continue\n'
    '        range_name = quote_sheet_name(sheet_name)\n'
    '        values = google_sheets_request(\n'
    '            "GET",\n'
    '            google_values_url(base_url, f"{range_name}!A:B"),\n'
    '            access_token,\n'
    '        ).get("values", [])\n'
    '        rows_to_delete = sparse_cleanup_plan(values, records)\n'
    '        if not rows_to_delete:\n'
    '            continue\n'
    '        for row_number in sorted(rows_to_delete, reverse=True):\n'
    '            requests.append(\n'
    '                {\n'
    '                    "deleteDimension": {\n'
    '                        "range": {\n'
    '                            "sheetId": sheet_by_name[sheet_name],\n'
    '                            "dimension": "ROWS",\n'
    '                            "startIndex": row_number - 1,\n'
    '                            "endIndex": row_number,\n'
    '                        }\n'
    '                    }\n'
    '                }\n'
    '            )\n'
    '        warning = (\n'
    '            "DRY RUN: would delete sparse misdated rows "\n'
    '            if validate_only\n'
    '            else "Deleted sparse misdated rows "\n'
    '        ) + ", ".join(str(row) for row in rows_to_delete)\n'
    '        results.append(\n'
    '            sync_result_row(\n'
    '                indicator_name,\n'
    '                sheet_name,\n'
    '                None,\n'
    '                len(rows_to_delete),\n'
    '                0,\n'
    '                None,\n'
    '                warning,\n'
    '            )\n'
    '        )\n'
    '    if requests and not validate_only:\n'
    '        google_sheets_request(\n'
    '            "POST",\n'
    '            f"{base_url}:batchUpdate",\n'
    '            access_token,\n'
    '            {"requests": requests},\n'
    '        )\n'
    '    return results\n'
    '\n'
    '\n'
    'def cleanup_sparse_date_rows(\n'
    '    spreadsheet_id: str,\n'
    '    credentials_path: Path,\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    validate_only: bool,\n'
    ') -> list[dict]:\n'
    '    """Clean old sparse-date rows after the correct quarter/year-end rows exist."""\n'
    '    if not credentials_path.exists():\n'
    '        raise SystemExit(f"Google credentials file not found: {credentials_path}")\n'
    '    try:\n'
    '        if os.getenv("US_PIPELINE_FORCE_DIRECT_GOOGLE_API") == "1":\n'
    '            raise ImportError("Direct Google Sheets API selected by pipeline runner.")\n'
    '        from google.oauth2.service_account import Credentials\n'
    '        from googleapiclient.discovery import build\n'
    '    except ImportError:\n'
    '        access_token = get_service_account_access_token(credentials_path)\n'
    '        base_url = f"{GOOGLE_SHEETS_API_BASE}/{spreadsheet_id}"\n'
    '        results = cleanup_sparse_date_rows_direct(\n'
    '            base_url,\n'
    '            access_token,\n'
    '            workbook_data,\n'
    '            validate_only,\n'
    '        )\n'
    '    else:\n'
    '        credentials = Credentials.from_service_account_file(\n'
    '            credentials_path,\n'
    '            scopes=GOOGLE_SHEETS_WRITE_SCOPES,\n'
    '        )\n'
    '        service = build("sheets", "v4", credentials=credentials)\n'
    '        results = cleanup_sparse_date_rows_with_service(\n'
    '            service.spreadsheets(),\n'
    '            spreadsheet_id,\n'
    '            workbook_data,\n'
    '            validate_only,\n'
    '        )\n'
    '    if results:\n'
    '        print("\\nSparse-date cleanup summary:")\n'
    '        print_sync_summary(results, validate_only=validate_only)\n'
    '    else:\n'
    '        print("\\nSparse-date cleanup summary: no misdated sparse rows found.")\n'
    '    return results\n'
    '\n'
    '\n'
    'def central_bank_component_data(\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    ') -> dict[str, list[dict]]:\n'
    '    """Return the source records that belong in the shared liquidity tab."""\n'
    '    return {\n'
    '        indicator_name: records\n'
    '        for indicator_name, records in workbook_data\n'
    '        if is_central_bank_component(indicator_name)\n'
    '    }\n'
    '\n'
    '\n'
    'def dedicated_indicator_data(\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    ') -> list[tuple[str, list[dict]]]:\n'
    '    """Return source records that should sync to one indicator tab each."""\n'
    '    return [\n'
    '        (indicator_name, records)\n'
    '        for indicator_name, records in workbook_data\n'
    '        if has_dedicated_indicator_tab(indicator_name)\n'
    '    ]\n'
    '\n'
    '\n'
    'def imf_coverage_component_data(\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    ') -> dict[str, list[dict]]:\n'
    '    """Return the two official IMF inputs written to the shared coverage tab."""\n'
    '    return {\n'
    '        indicator_name: records\n'
    '        for indicator_name, records in workbook_data\n'
    '        if indicator_name in IMF_COVERAGE_INPUT_COLUMNS\n'
    '    }\n'
    '\n'
    '\n'
    'def imf_coverage_sheet_values(\n'
    '    component_data: dict[str, list[dict]],\n'
    ') -> list[list[object]]:\n'
    '    """Build the complete, sorted A:C raw-input table for the IMF analysis tab."""\n'
    '    by_indicator = {\n'
    '        indicator_name: output_month_records_for_indicator(records)\n'
    '        for indicator_name, records in component_data.items()\n'
    '    }\n'
    '    months = sorted(set().union(*(values.keys() for values in by_indicator.values())))\n'
    '    headers = ["Date", "Government Revenue % GDP", "Government Interest % GDP"]\n'
    '    return [\n'
    '        headers,\n'
    '        *[\n'
    '            [\n'
    '                (month_end(month_key) - date(1899, 12, 30)).days,\n'
    '                by_indicator.get("Government revenue % GDP", {}).get(month_key, ""),\n'
    '                by_indicator.get("Interest paid on public debt % GDP", {}).get(\n'
    '                    month_key, ""\n'
    '                ),\n'
    '            ]\n'
    '            for month_key in months\n'
    '        ],\n'
    '    ]\n'
    '\n'
    '\n'
    'def merge_imf_coverage_history(\n'
    '    existing: list[list[object]],\n'
    '    incoming: list[list[object]],\n'
    ') -> list[list[object]]:\n'
    '    """Preserve legacy annual IMF values while canonicalising dates to year-end."""\n'
    '    if not incoming:\n'
    '        raise RuntimeError("Generated IMF coverage table is empty.")\n'
    '    if existing and [normalise_header_name(item) for item in existing[0]] != [\n'
    '        normalise_header_name(item) for item in incoming[0]\n'
    '    ]:\n'
    '        raise RuntimeError(\n'
    '            f"Shared sheet {IMF_COVERAGE_SHEET!r}: refusing overwrite because headers differ; "\n'
    '            f"existing={existing[0]!r}, incoming={incoming[0]!r}."\n'
    '        )\n'
    '\n'
    '    def values_by_year(\n'
    '        table: list[list[object]], context: str\n'
    '    ) -> dict[int, list[object]]:\n'
    '        result: dict[int, list[object]] = {}\n'
    '        for row_number, row in enumerate(table[1:], start=2):\n'
    '            observed = normalise_observation_date(row[0] if row else None)\n'
    '            raw_components = [\n'
    '                row[column] if len(row) > column else "" for column in (1, 2)\n'
    '            ]\n'
    '            if observed is None:\n'
    '                if any(normalise_value(value) is not None for value in raw_components):\n'
    '                    raise RuntimeError(\n'
    '                        f"{context}: row {row_number} has IMF values but no valid date."\n'
    '                    )\n'
    '                continue\n'
    '            year_values = result.setdefault(observed.year, ["", ""])\n'
    '            for index, raw_value in enumerate(raw_components):\n'
    '                value = normalise_value(raw_value)\n'
    '                if value is None:\n'
    '                    continue\n'
    '                prior_value = normalise_value(year_values[index])\n'
    '                if prior_value is None:\n'
    '                    year_values[index] = raw_value\n'
    '                elif not google_values_equivalent(prior_value, value):\n'
    '                    raise RuntimeError(\n'
    '                        f"{context}: conflicting duplicate IMF values for "\n'
    '                        f"{observed.year}, column {index + 2}: "\n'
    '                        f"{year_values[index]!r} versus {raw_value!r}."\n'
    '                    )\n'
    '        return result\n'
    '\n'
    '    old_by_year = values_by_year(existing, f"Shared sheet {IMF_COVERAGE_SHEET!r}")\n'
    '    new_by_year = values_by_year(incoming, "Generated IMF coverage table")\n'
    '    rows: list[list[object]] = [list(incoming[0])]\n'
    '    for year in sorted(set(old_by_year) | set(new_by_year)):\n'
    '        old_values = old_by_year.get(year, ["", ""])\n'
    '        new_values = new_by_year.get(year, ["", ""])\n'
    '        merged_values = [\n'
    '            new_raw if normalise_value(new_raw) is not None else old_raw\n'
    '            for old_raw, new_raw in zip(old_values, new_values)\n'
    '        ]\n'
    '        rows.append(\n'
    '            [\n'
    '                (date(year, 12, 31) - date(1899, 12, 30)).days,\n'
    '                *merged_values,\n'
    '            ]\n'
    '        )\n'
    '    return rows\n'
    '\n'
    '\n'
    'def imf_coverage_sync_result(\n'
    '    table_values: list[list[object]], warning: str = ""\n'
    ') -> dict:\n'
    '    incoming_dates = [normalise_observation_date(row[0]) for row in table_values[1:]]\n'
    '    incoming_dates = [item for item in incoming_dates if item is not None]\n'
    '    return sync_result_row(\n'
    '        IMF_COVERAGE_SHEET,\n'
    '        IMF_COVERAGE_SHEET,\n'
    '        None,\n'
    '        max(0, len(table_values) - 1) * 2,\n'
    '        max(0, len(table_values) - 1),\n'
    '        max(incoming_dates, default=None),\n'
    '        warning,\n'
    '    )\n'
    '\n'
    '\n'
    'def sync_imf_coverage_with_service(\n'
    '    sheets,\n'
    '    spreadsheet_id: str,\n'
    '    component_data: dict[str, list[dict]],\n'
    '    sheet_names: set[str],\n'
    '    validate_only: bool,\n'
    ') -> dict:\n'
    '    """Rebuild A:C of the shared IMF tab while preserving analysis columns D onward."""\n'
    '    if IMF_COVERAGE_SHEET not in sheet_names:\n'
    '        return sync_result_row(\n'
    '            IMF_COVERAGE_SHEET,\n'
    '            "",\n'
    '            None,\n'
    '            0,\n'
    '            0,\n'
    '            None,\n'
    '            "Missing matching IMF coverage tab; skipped.",\n'
    '        )\n'
    '    table_values = imf_coverage_sheet_values(component_data)\n'
    '    range_name = f"{quote_sheet_name(IMF_COVERAGE_SHEET)}!A:C"\n'
    '    old_values = (\n'
    '        sheets.values()\n'
    '        .get(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=range_name,\n'
    '            valueRenderOption="UNFORMATTED_VALUE",\n'
    '        )\n'
    '        .execute()\n'
    '        .get("values", [])\n'
    '    )\n'
    '    if not FULL_HISTORY_MODE:\n'
    '        fills, appends, merged_values = prepare_incremental_table_changes(\n'
    '            old_values,\n'
    '            table_values,\n'
    '            f"Shared sheet {IMF_COVERAGE_SHEET!r}",\n'
    '        )\n'
    '        old_dates = [\n'
    '            normalise_observation_date(row[0] if row else None)\n'
    '            for row in old_values[1:]\n'
    '        ]\n'
    '        new_dates = [\n'
    '            normalise_observation_date(row[0] if row else None)\n'
    '            for row in merged_values[1:]\n'
    '        ]\n'
    '        latest_before = max((item for item in old_dates if item), default=None)\n'
    '        latest_after = max((item for item in new_dates if item), default=None)\n'
    '        result = sync_result_row(\n'
    '            IMF_COVERAGE_SHEET,\n'
    '            IMF_COVERAGE_SHEET,\n'
    '            latest_before,\n'
    '            len(fills),\n'
    '            len(appends),\n'
    '            latest_after,\n'
    '        )\n'
    '        if not fills and not appends:\n'
    '            return result\n'
    '        if validate_only:\n'
    '            result["warning"] = (\n'
    '                "DRY RUN: incremental preflight passed; populated historical values would be preserved."\n'
    '            )\n'
    '            return result\n'
    '\n'
    '        current_values = (\n'
    '            sheets.values()\n'
    '            .get(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=range_name,\n'
    '                valueRenderOption="UNFORMATTED_VALUE",\n'
    '            )\n'
    '            .execute()\n'
    '            .get("values", [])\n'
    '        )\n'
    '        if current_values != old_values:\n'
    '            raise RuntimeError(\n'
    '                f"Shared sheet {IMF_COVERAGE_SHEET!r} changed after preflight; refusing stale incremental write."\n'
    '            )\n'
    '        if old_values:\n'
    '            data_ranges = incremental_table_value_ranges(\n'
    '                IMF_COVERAGE_SHEET,\n'
    '                fills,\n'
    '                appends,\n'
    '                len(old_values),\n'
    '            )\n'
    '            sheets.values().batchUpdate(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                body={"valueInputOption": "RAW", "data": data_ranges},\n'
    '            ).execute()\n'
    '        else:\n'
    '            sheets.values().update(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"{quote_sheet_name(IMF_COVERAGE_SHEET)}!A1",\n'
    '                valueInputOption="RAW",\n'
    '                body={"values": merged_values},\n'
    '            ).execute()\n'
    '        written = (\n'
    '            sheets.values()\n'
    '            .get(spreadsheetId=spreadsheet_id, range=range_name)\n'
    '            .execute()\n'
    '            .get("values", [])\n'
    '        )\n'
    '        verify_google_sheet_values(merged_values, written)\n'
    '        return result\n'
    '\n'
    '    table_values = merge_imf_coverage_history(old_values, table_values)\n'
    '    validate_historical_table_replacement(\n'
    '        old_values,\n'
    '        table_values,\n'
    '        f"Shared sheet {IMF_COVERAGE_SHEET!r}",\n'
    '        allow_absent_existing_dates=True,\n'
    '        allow_value_revisions=True,\n'
    '    )\n'
    '    if google_sheet_table_is_current(table_values, old_values):\n'
    '        return imf_coverage_sync_result(table_values)\n'
    '    if validate_only:\n'
    '        return imf_coverage_sync_result(\n'
    '            table_values,\n'
    '            "DRY RUN: historical preflight passed; would rebuild the complete IMF A:C input table.",\n'
    '        )\n'
    '    try:\n'
    '        current_values = (\n'
    '            sheets.values()\n'
    '            .get(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=range_name,\n'
    '                valueRenderOption="UNFORMATTED_VALUE",\n'
    '            )\n'
    '            .execute()\n'
    '            .get("values", [])\n'
    '        )\n'
    '        if current_values != old_values:\n'
    '            raise RuntimeError(\n'
    '                f"Shared sheet {IMF_COVERAGE_SHEET!r} changed after preflight; refusing stale overwrite."\n'
    '            )\n'
    '        sheets.values().clear(\n'
    '            spreadsheetId=spreadsheet_id, range=range_name, body={}\n'
    '        ).execute()\n'
    '        sheets.values().update(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=f"{quote_sheet_name(IMF_COVERAGE_SHEET)}!A1",\n'
    '            valueInputOption="RAW",\n'
    '            body={"values": table_values},\n'
    '        ).execute()\n'
    '        written = (\n'
    '            sheets.values()\n'
    '            .get(spreadsheetId=spreadsheet_id, range=range_name)\n'
    '            .execute()\n'
    '            .get("values", [])\n'
    '        )\n'
    '        verify_google_sheet_values(table_values, written)\n'
    '    except Exception:\n'
    '        sheets.values().clear(\n'
    '            spreadsheetId=spreadsheet_id, range=range_name, body={}\n'
    '        ).execute()\n'
    '        if old_values:\n'
    '            sheets.values().update(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                range=f"{quote_sheet_name(IMF_COVERAGE_SHEET)}!A1",\n'
    '                valueInputOption="RAW",\n'
    '                body={"values": old_values},\n'
    '            ).execute()\n'
    '        raise\n'
    '    return imf_coverage_sync_result(table_values)\n'
    '\n'
    '\n'
    'def sync_imf_coverage_direct(\n'
    '    base_url: str,\n'
    '    access_token: str,\n'
    '    component_data: dict[str, list[dict]],\n'
    '    sheet_names: set[str],\n'
    '    validate_only: bool,\n'
    ') -> dict:\n'
    '    """Direct-Sheets-API version of the shared IMF input-table rebuild."""\n'
    '    if IMF_COVERAGE_SHEET not in sheet_names:\n'
    '        return sync_result_row(\n'
    '            IMF_COVERAGE_SHEET,\n'
    '            "",\n'
    '            None,\n'
    '            0,\n'
    '            0,\n'
    '            None,\n'
    '            "Missing matching IMF coverage tab; skipped.",\n'
    '        )\n'
    '    table_values = imf_coverage_sheet_values(component_data)\n'
    '    quoted_sheet = quote_sheet_name(IMF_COVERAGE_SHEET)\n'
    '    range_name = f"{quoted_sheet}!A:C"\n'
    '    range_url = google_values_url(base_url, range_name)\n'
    '    clear_url = f"{base_url}/values/{quote(range_name, safe=\'\')}:clear"\n'
    '    old_values = google_sheets_request("GET", range_url, access_token).get("values", [])\n'
    '    if not FULL_HISTORY_MODE:\n'
    '        fills, appends, merged_values = prepare_incremental_table_changes(\n'
    '            old_values,\n'
    '            table_values,\n'
    '            f"Shared sheet {IMF_COVERAGE_SHEET!r}",\n'
    '        )\n'
    '        old_dates = [\n'
    '            normalise_observation_date(row[0] if row else None)\n'
    '            for row in old_values[1:]\n'
    '        ]\n'
    '        new_dates = [\n'
    '            normalise_observation_date(row[0] if row else None)\n'
    '            for row in merged_values[1:]\n'
    '        ]\n'
    '        latest_before = max((item for item in old_dates if item), default=None)\n'
    '        latest_after = max((item for item in new_dates if item), default=None)\n'
    '        result = sync_result_row(\n'
    '            IMF_COVERAGE_SHEET,\n'
    '            IMF_COVERAGE_SHEET,\n'
    '            latest_before,\n'
    '            len(fills),\n'
    '            len(appends),\n'
    '            latest_after,\n'
    '        )\n'
    '        if not fills and not appends:\n'
    '            return result\n'
    '        if validate_only:\n'
    '            result["warning"] = (\n'
    '                "DRY RUN: incremental preflight passed; populated historical values would be preserved."\n'
    '            )\n'
    '            return result\n'
    '\n'
    '        current_values = google_sheets_request("GET", range_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        if current_values != old_values:\n'
    '            raise RuntimeError(\n'
    '                f"Shared sheet {IMF_COVERAGE_SHEET!r} changed after preflight; refusing stale incremental write."\n'
    '            )\n'
    '        if old_values:\n'
    '            data_ranges = incremental_table_value_ranges(\n'
    '                IMF_COVERAGE_SHEET,\n'
    '                fills,\n'
    '                appends,\n'
    '                len(old_values),\n'
    '            )\n'
    '            google_sheets_request(\n'
    '                "POST",\n'
    '                f"{base_url}/values:batchUpdate?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"valueInputOption": "RAW", "data": data_ranges},\n'
    '            )\n'
    '        else:\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/{quote(f\'{quote_sheet_name(IMF_COVERAGE_SHEET)}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"values": merged_values},\n'
    '            )\n'
    '        written = google_sheets_request("GET", range_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        verify_google_sheet_values(merged_values, written)\n'
    '        return result\n'
    '\n'
    '    table_values = merge_imf_coverage_history(old_values, table_values)\n'
    '    validate_historical_table_replacement(\n'
    '        old_values,\n'
    '        table_values,\n'
    '        f"Shared sheet {IMF_COVERAGE_SHEET!r}",\n'
    '        allow_absent_existing_dates=True,\n'
    '        allow_value_revisions=True,\n'
    '    )\n'
    '    if google_sheet_table_is_current(table_values, old_values):\n'
    '        return imf_coverage_sync_result(table_values)\n'
    '    if validate_only:\n'
    '        return imf_coverage_sync_result(\n'
    '            table_values,\n'
    '            "DRY RUN: historical preflight passed; would rebuild the complete IMF A:C input table.",\n'
    '        )\n'
    '    try:\n'
    '        current_values = google_sheets_request("GET", range_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        if current_values != old_values:\n'
    '            raise RuntimeError(\n'
    '                f"Shared sheet {IMF_COVERAGE_SHEET!r} changed after preflight; refusing stale overwrite."\n'
    '            )\n'
    '        google_sheets_request("POST", clear_url, access_token, {})\n'
    '        google_sheets_request(\n'
    '            "PUT",\n'
    '            f"{base_url}/values/{quote(f\'{quoted_sheet}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '            access_token,\n'
    '            {"values": table_values},\n'
    '        )\n'
    '        written = google_sheets_request("GET", range_url, access_token).get(\n'
    '            "values", []\n'
    '        )\n'
    '        verify_google_sheet_values(table_values, written)\n'
    '    except Exception:\n'
    '        google_sheets_request("POST", clear_url, access_token, {})\n'
    '        if old_values:\n'
    '            google_sheets_request(\n'
    '                "PUT",\n'
    '                f"{base_url}/values/{quote(f\'{quoted_sheet}!A1\', safe=\'\')}?valueInputOption=RAW",\n'
    '                access_token,\n'
    '                {"values": old_values},\n'
    '            )\n'
    '        raise\n'
    '    return imf_coverage_sync_result(table_values)\n'
    '\n'
    '\n'
    'def expected_indicator_tabs() -> list[str]:\n'
    '    """Return the indicator tabs that should exist in the dedicated workbook."""\n'
    '    return [\n'
    '        indicator.name\n'
    '        for indicator in INDICATORS\n'
    '        if has_dedicated_indicator_tab(indicator.name)\n'
    '    ] + [CENTRAL_BANK_LIQUIDITY_SHEET, IMF_COVERAGE_SHEET]\n'
    '\n'
    '\n'
    'def sheet_date_stats(values: list[list[object]]) -> dict:\n'
    '    """Inspect column A without mutating the sheet."""\n'
    '    dates: list[date] = []\n'
    '    duplicate_dates: set[date] = set()\n'
    '    seen_dates: set[date] = set()\n'
    '    blank_values = 0\n'
    '    non_numeric_values = 0\n'
    '    for row in values[1:]:\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed is None:\n'
    '            continue\n'
    '        if observed in seen_dates:\n'
    '            duplicate_dates.add(observed)\n'
    '        seen_dates.add(observed)\n'
    '        dates.append(observed)\n'
    '        raw_value = row[1] if len(row) > 1 else None\n'
    '        if raw_value in (None, ""):\n'
    '            blank_values += 1\n'
    '        elif normalise_value(raw_value) is None:\n'
    '            non_numeric_values += 1\n'
    '    latest = max(dates, default=None)\n'
    '    return {\n'
    '        "latest": latest,\n'
    '        "dates": dates,\n'
    '        "valid_date_rows": len(dates),\n'
    '        "duplicate_dates": sorted(duplicate_dates),\n'
    '        "blank_values": blank_values,\n'
    '        "non_numeric_values": non_numeric_values,\n'
    '    }\n'
    '\n'
    '\n'
    'def latest_sheet_value(\n'
    '    values: list[list[object]],\n'
    '    target_date: date,\n'
    '    value_col: int = 1,\n'
    ') -> float | None:\n'
    '    """Return a numeric sheet value for one normalized observation month."""\n'
    '    for row in values[1:]:\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed != target_date:\n'
    '            continue\n'
    '        raw_value = row[value_col] if len(row) > value_col else None\n'
    '        return normalise_value(raw_value)\n'
    '    return None\n'
    '\n'
    '\n'
    'def sheet_values_by_date(\n'
    '    values: list[list[object]],\n'
    '    value_col: int = 1,\n'
    ') -> dict[date, float | None]:\n'
    '    """Map normalized sheet dates to numeric values without changing the workbook."""\n'
    '    mapped: dict[date, float | None] = {}\n'
    '    for row in values[1:]:\n'
    '        observed = normalise_observation_date(row[0] if row else None)\n'
    '        if observed is None:\n'
    '            continue\n'
    '        raw_value = row[value_col] if len(row) > value_col else None\n'
    '        mapped[observed] = normalise_value(raw_value)\n'
    '    return mapped\n'
    '\n'
    '\n'
    'def source_sheet_mismatches(\n'
    '    values: list[list[object]],\n'
    '    source_records: list[dict],\n'
    '    value_col: int = 1,\n'
    ') -> list[tuple[date, float | None, float]]:\n'
    '    """Find dated rows where the source output and sheet value differ."""\n'
    '    incoming = output_month_records_for_indicator(source_records)\n'
    '    existing = sheet_values_by_date(values, value_col)\n'
    '    mismatches: list[tuple[date, float | None, float]] = []\n'
    '    for observed, source_value in sorted(incoming.items()):\n'
    '        if observed not in existing:\n'
    '            continue\n'
    '        sheet_value = existing[observed]\n'
    '        if sheet_value is None or not google_values_equivalent(\n'
    '            source_value, sheet_value\n'
    '        ):\n'
    '            mismatches.append((observed, sheet_value, source_value))\n'
    '    return mismatches\n'
    '\n'
    '\n'
    'def format_mismatch_warning(\n'
    '    mismatches: list[tuple[date, float | None, float]],\n'
    ') -> str:\n'
    '    """Summarize sheet/source value disagreements for audit output."""\n'
    '    return "Sheet/source mismatches found: " + "; ".join(\n'
    '        f"{observed.isoformat()} sheet={sheet_value} source={source_value}"\n'
    '        for observed, sheet_value, source_value in mismatches[-5:]\n'
    '    )\n'
    '\n'
    '\n'
    'def sheet_quality_warnings(\n'
    '    stats: dict,\n'
    '    indicator_source: str,\n'
    '    source_frequency: str,\n'
    ') -> list[str]:\n'
    '    """Return non-mutating sheet health warnings for one audit row."""\n'
    '    warnings: list[str] = []\n'
    '    if stats["duplicate_dates"]:\n'
    '        warnings.append("Duplicate dates in column A.")\n'
    '    if stats["blank_values"]:\n'
    '        warnings.append(f"{stats[\'blank_values\']} dated rows have blank values.")\n'
    '    if stats["non_numeric_values"]:\n'
    '        warnings.append(\n'
    '            f"{stats[\'non_numeric_values\']} dated rows have non-numeric values."\n'
    '        )\n'
    '    if indicator_source == "imf_datamapper":\n'
    '        non_december_dates = [\n'
    '            item for item in stats.get("dates", []) if item.month != ANNUAL_OUTPUT_MONTH\n'
    '        ]\n'
    '        if non_december_dates:\n'
    '            warnings.append(\n'
    '                "Annual IMF values should be dated to December year-end; "\n'
    '                "non-December rows found: "\n'
    '                + ", ".join(item.isoformat() for item in non_december_dates[-5:])\n'
    '            )\n'
    '    if source_frequency.startswith("Quarterly"):\n'
    '        off_quarter_dates = [\n'
    '            item\n'
    '            for item in stats.get("dates", [])\n'
    '            if item.month not in QUARTER_END_MONTHS\n'
    '        ]\n'
    '        if off_quarter_dates:\n'
    '            warnings.append(\n'
    '                "Quarterly FRED values should be dated to quarter-end month; "\n'
    '                "off-quarter rows found: "\n'
    '                + ", ".join(item.isoformat() for item in off_quarter_dates[-5:])\n'
    '            )\n'
    '    elif source_frequency.startswith("Annual"):\n'
    '        off_annual_dates = [\n'
    '            item for item in stats.get("dates", []) if item.month != ANNUAL_OUTPUT_MONTH\n'
    '        ]\n'
    '        if off_annual_dates:\n'
    '            warnings.append(\n'
    '                "Annual FRED values should be dated to December year-end; "\n'
    '                "off-year-end rows found: "\n'
    '                + ", ".join(item.isoformat() for item in off_annual_dates[-5:])\n'
    '            )\n'
    '    return warnings\n'
    '\n'
    '\n'
    'def source_output_for_records(records: list[dict]) -> tuple[date | None, float | None]:\n'
    '    """Return the newest month/value that the extraction layer would write."""\n'
    '    incoming = output_month_records_for_indicator(records)\n'
    '    if not incoming:\n'
    '        return None, None\n'
    '    latest = max(incoming)\n'
    '    return latest, incoming[latest]\n'
    '\n'
    '\n'
    'def source_descriptor(indicator: Indicator) -> tuple[str, str]:\n'
    '    """Summarize one indicator\'s official source for audit output."""\n'
    '    if indicator.source == "fred":\n'
    '        series_id = indicator.series_id or ""\n'
    '        return "FRED / ALFRED", f"https://fred.stlouisfed.org/series/{series_id}"\n'
    '    if indicator.source == "ism":\n'
    '        return "Institute for Supply Management", ISM_REPORT_INDEX_URL\n'
    '    if indicator.source == "nyfed_sce":\n'
    '        return "New York Fed SCE", NYFED_SCE_PAGE_URL\n'
    '    if indicator.source in {"michigan_sca", "michigan_sentiment"}:\n'
    '        return "University of Michigan Surveys of Consumers", MICHIGAN_CHARTS_PAGE_URL\n'
    '    if indicator.source == "imf_datamapper":\n'
    '        return "IMF DataMapper FPP", IMF_DATAMAPPER_API_BASE\n'
    '    return indicator.source, ""\n'
    '\n'
    '\n'
    'def audit_sheet_rows(\n'
    '    sheet_names: set[str],\n'
    '    read_values,\n'
    '    source_data: dict[str, list[dict]] | None = None,\n'
    '    source_errors: dict[str, str] | None = None,\n'
    ') -> list[list[object]]:\n'
    '    """Build a complete read-only workbook audit table."""\n'
    '    source_data = source_data or {}\n'
    '    source_errors = source_errors or {}\n'
    '    headers = [\n'
    '        "Indicator",\n'
    '        "Sheet",\n'
    '        "Source",\n'
    '        "Source URL",\n'
    '        "Latest sheet date",\n'
    '        "Latest source date",\n'
    '        "Latest sheet value",\n'
    '        "Latest source value",\n'
    '        "Status",\n'
    '        "Warnings",\n'
    '        "Valid date rows",\n'
    '        "Blank value rows",\n'
    '        "Non-numeric value rows",\n'
    '        "Duplicate dates",\n'
    '    ]\n'
    '    rows: list[list[object]] = [headers]\n'
    '\n'
    '    for indicator in INDICATORS:\n'
    '        if not has_dedicated_indicator_tab(indicator.name):\n'
    '            continue\n'
    '        source_name, source_url = source_descriptor(indicator)\n'
    '        sheet_name = indicator_sheet_name(indicator.name, sheet_names)\n'
    '        if sheet_name is None:\n'
    '            rows.append(\n'
    '                [\n'
    '                    indicator.name,\n'
    '                    "",\n'
    '                    source_name,\n'
    '                    source_url,\n'
    '                    "",\n'
    '                    "",\n'
    '                    "",\n'
    '                    "",\n'
    '                    "FAILED",\n'
    '                    "Missing matching indicator tab.",\n'
    '                    0,\n'
    '                    0,\n'
    '                    0,\n'
    '                    "",\n'
    '                ]\n'
    '            )\n'
    '            continue\n'
    '        try:\n'
    '            values = read_values(sheet_name, "B")\n'
    '            get_sheet_layout(values, indicator.name)\n'
    '            stats = sheet_date_stats(values)\n'
    '        except Exception as exc:\n'
    '            rows.append(\n'
    '                [\n'
    '                    indicator.name,\n'
    '                    sheet_name,\n'
    '                    source_name,\n'
    '                    source_url,\n'
    '                    "",\n'
    '                    "",\n'
    '                    "",\n'
    '                    "",\n'
    '                    "FAILED",\n'
    '                    str(exc),\n'
    '                    0,\n'
    '                    0,\n'
    '                    0,\n'
    '                    "",\n'
    '                ]\n'
    '            )\n'
    '            continue\n'
    '\n'
    '        source_records = source_data.get(indicator.name, [])\n'
    '        source_latest, source_value = source_output_for_records(source_records)\n'
    '        latest_sheet = stats["latest"]\n'
    '        sheet_value = latest_sheet_value(values, latest_sheet) if latest_sheet else None\n'
    '        source_frequency = (\n'
    '            str(source_records[0].get("native_frequency", "")) if source_records else ""\n'
    '        )\n'
    '        warnings = sheet_quality_warnings(\n'
    '            stats,\n'
    '            indicator.source,\n'
    '            source_frequency,\n'
    '        )\n'
    '        mismatches = source_sheet_mismatches(values, source_records)\n'
    '        if mismatches:\n'
    '            warnings.append(format_mismatch_warning(mismatches))\n'
    '        if indicator.name in source_errors:\n'
    '            warnings.append(\n'
    '                f"Source extraction failed: {source_errors[indicator.name]}"\n'
    '            )\n'
    '        if source_latest and latest_sheet and source_latest > latest_sheet:\n'
    '            status = "NEEDS UPDATE"\n'
    '        elif mismatches:\n'
    '            status = "VALUE MISMATCH"\n'
    '        elif (\n'
    '            source_latest\n'
    '            and latest_sheet == source_latest\n'
    '            and source_value is not None\n'
    '            and sheet_value is not None\n'
    '            and not google_values_equivalent(source_value, sheet_value)\n'
    '        ):\n'
    '            status = "VALUE MISMATCH"\n'
    '            warnings.append("Latest sheet value differs from extracted source value.")\n'
    '        elif source_latest and latest_sheet and latest_sheet > source_latest:\n'
    '            status = "REVIEW"\n'
    '            warnings.append("Sheet is ahead of currently extracted source output.")\n'
    '        elif source_latest is None and indicator.name in source_errors:\n'
    '            status = "FAILED"\n'
    '        else:\n'
    '            status = "OK"\n'
    '\n'
    '        rows.append(\n'
    '            [\n'
    '                indicator.name,\n'
    '                sheet_name,\n'
    '                source_name,\n'
    '                source_url,\n'
    '                latest_sheet.isoformat() if latest_sheet else "",\n'
    '                source_latest.isoformat() if source_latest else "",\n'
    '                sheet_value if sheet_value is not None else "",\n'
    '                source_value if source_value is not None else "",\n'
    '                status,\n'
    '                " ".join(warnings),\n'
    '                stats["valid_date_rows"],\n'
    '                stats["blank_values"],\n'
    '                stats["non_numeric_values"],\n'
    '                ", ".join(item.isoformat() for item in stats["duplicate_dates"]),\n'
    '            ]\n'
    '        )\n'
    '\n'
    '    central_sheet = CENTRAL_BANK_LIQUIDITY_SHEET\n'
    '    source_name = "FRED / ALFRED"\n'
    '    source_url = " / ".join(\n'
    '        f"https://fred.stlouisfed.org/series/{indicator.series_id}"\n'
    '        for indicator in INDICATORS\n'
    '        if is_central_bank_component(indicator.name)\n'
    '    )\n'
    '    if central_sheet not in sheet_names:\n'
    '        rows.append(\n'
    '            [\n'
    '                central_sheet,\n'
    '                "",\n'
    '                source_name,\n'
    '                source_url,\n'
    '                "",\n'
    '                "",\n'
    '                "",\n'
    '                "",\n'
    '                "FAILED",\n'
    '                "Missing matching central-bank liquidity tab.",\n'
    '                0,\n'
    '                0,\n'
    '                0,\n'
    '                "",\n'
    '            ]\n'
    '        )\n'
    '    else:\n'
    '        try:\n'
    '            values = read_values(central_sheet, "D")\n'
    '            get_sheet_layout(values, central_sheet)\n'
    '            stats = sheet_date_stats(values)\n'
    '        except Exception as exc:\n'
    '            values = []\n'
    '            stats = {\n'
    '                "latest": None,\n'
    '                "dates": [],\n'
    '                "valid_date_rows": 0,\n'
    '                "duplicate_dates": [],\n'
    '                "blank_values": 0,\n'
    '                "non_numeric_values": 0,\n'
    '            }\n'
    '            central_layout_error = str(exc)\n'
    '        else:\n'
    '            central_layout_error = ""\n'
    '\n'
    '        for component_name, target in CENTRAL_BANK_LIQUIDITY_COMPONENTS.items():\n'
    '            source_latest, source_value = source_output_for_records(\n'
    '                source_data.get(component_name, [])\n'
    '            )\n'
    '            latest_sheet = stats["latest"]\n'
    '            sheet_value = (\n'
    '                latest_sheet_value(values, latest_sheet, target["column"])\n'
    '                if latest_sheet and values\n'
    '                else None\n'
    '            )\n'
    '            source_records = source_data.get(component_name, [])\n'
    '            mismatches = source_sheet_mismatches(\n'
    '                values,\n'
    '                source_records,\n'
    '                target["column"],\n'
    '            )\n'
    '            warnings: list[str] = []\n'
    '            status = "OK"\n'
    '            if central_layout_error:\n'
    '                status = "FAILED"\n'
    '                warnings.append(central_layout_error)\n'
    '            if component_name in source_errors:\n'
    '                status = "FAILED"\n'
    '                warnings.append(\n'
    '                    f"Source extraction failed: {source_errors[component_name]}"\n'
    '                )\n'
    '            if source_latest and latest_sheet and source_latest > latest_sheet:\n'
    '                status = "NEEDS UPDATE"\n'
    '            elif mismatches:\n'
    '                status = "VALUE MISMATCH"\n'
    '                warnings.append(format_mismatch_warning(mismatches))\n'
    '            elif (\n'
    '                source_latest\n'
    '                and latest_sheet == source_latest\n'
    '                and source_value is not None\n'
    '                and sheet_value is not None\n'
    '                and not google_values_equivalent(source_value, sheet_value)\n'
    '            ):\n'
    '                status = "VALUE MISMATCH"\n'
    '                warnings.append(\n'
    '                    "Latest sheet value differs from extracted source value."\n'
    '                )\n'
    '            rows.append(\n'
    '                [\n'
    '                    target["header"],\n'
    '                    central_sheet,\n'
    '                    source_name,\n'
    '                    source_url,\n'
    '                    latest_sheet.isoformat() if latest_sheet else "",\n'
    '                    source_latest.isoformat() if source_latest else "",\n'
    '                    sheet_value if sheet_value is not None else "",\n'
    '                    source_value if source_value is not None else "",\n'
    '                    status,\n'
    '                    " ".join(warnings),\n'
    '                    stats["valid_date_rows"],\n'
    '                    "",\n'
    '                    "",\n'
    '                    ", ".join(item.isoformat() for item in stats["duplicate_dates"]),\n'
    '                ]\n'
    '            )\n'
    '\n'
    '    coverage_sheet = IMF_COVERAGE_SHEET\n'
    '    if coverage_sheet not in sheet_names:\n'
    '        rows.append(\n'
    '            [\n'
    '                coverage_sheet,\n'
    '                "",\n'
    '                "IMF DataMapper FPP",\n'
    '                IMF_DATAMAPPER_API_BASE,\n'
    '                "",\n'
    '                "",\n'
    '                "",\n'
    '                "",\n'
    '                "FAILED",\n'
    '                "Missing matching IMF coverage tab.",\n'
    '                0,\n'
    '                0,\n'
    '                0,\n'
    '                "",\n'
    '            ]\n'
    '        )\n'
    '    else:\n'
    '        try:\n'
    '            values = read_values(coverage_sheet, "D")\n'
    '            get_sheet_layout(values, coverage_sheet)\n'
    '            stats = sheet_date_stats(values)\n'
    '            latest_sheet = stats["latest"]\n'
    '        except Exception as exc:\n'
    '            values = []\n'
    '            latest_sheet = None\n'
    '            stats = {\n'
    '                "valid_date_rows": 0,\n'
    '                "duplicate_dates": [],\n'
    '                "blank_values": 0,\n'
    '                "non_numeric_values": 0,\n'
    '                "dates": [],\n'
    '            }\n'
    '            coverage_layout_error = str(exc)\n'
    '        else:\n'
    '            coverage_layout_error = ""\n'
    '\n'
    '        coverage_columns = {\n'
    '            "Government revenue % GDP": 1,\n'
    '            "Interest paid on public debt % GDP": 2,\n'
    '            IMF_COVERAGE_SHEET: 3,\n'
    '        }\n'
    '        for indicator_name, value_col in coverage_columns.items():\n'
    '            indicator = next(item for item in INDICATORS if item.name == indicator_name)\n'
    '            source_name, source_url = source_descriptor(indicator)\n'
    '            source_records = source_data.get(indicator_name, [])\n'
    '            source_latest, source_value = source_output_for_records(source_records)\n'
    '            sheet_value = (\n'
    '                latest_sheet_value(values, latest_sheet, value_col)\n'
    '                if latest_sheet and values\n'
    '                else None\n'
    '            )\n'
    '            warnings = sheet_quality_warnings(stats, "imf_datamapper", "")\n'
    '            status = "OK"\n'
    '            if coverage_layout_error:\n'
    '                status = "FAILED"\n'
    '                warnings.append(coverage_layout_error)\n'
    '            if latest_sheet and sheet_value is None:\n'
    '                status = "FAILED"\n'
    '                warnings.append("Latest shared IMF row is missing this required value.")\n'
    '            mismatches = source_sheet_mismatches(values, source_records, value_col)\n'
    '            if mismatches:\n'
    '                status = "VALUE MISMATCH"\n'
    '                warnings.append(format_mismatch_warning(mismatches))\n'
    '            if indicator_name in source_errors:\n'
    '                status = "FAILED"\n'
    '                warnings.append(\n'
    '                    f"Source extraction failed: {source_errors[indicator_name]}"\n'
    '                )\n'
    '            if source_latest and latest_sheet and source_latest > latest_sheet:\n'
    '                status = "NEEDS UPDATE"\n'
    '            rows.append(\n'
    '                [\n'
    '                    indicator_name,\n'
    '                    coverage_sheet,\n'
    '                    source_name,\n'
    '                    source_url,\n'
    '                    latest_sheet.isoformat() if latest_sheet else "",\n'
    '                    source_latest.isoformat() if source_latest else "",\n'
    '                    sheet_value if sheet_value is not None else "",\n'
    '                    source_value if source_value is not None else "",\n'
    '                    status,\n'
    '                    " ".join(warnings),\n'
    '                    stats["valid_date_rows"],\n'
    '                    "",\n'
    '                    "",\n'
    '                    ", ".join(item.isoformat() for item in stats["duplicate_dates"]),\n'
    '                ]\n'
    '            )\n'
    '\n'
    '    return rows\n'
    '\n'
    '\n'
    'def write_audit_csv(rows: list[list[object]]) -> None:\n'
    '    OUTPUT_ROOT.mkdir(exist_ok=True)\n'
    '    pd.DataFrame(rows[1:], columns=rows[0]).to_csv(GOOGLE_SHEET_AUDIT_CSV, index=False)\n'
    '\n'
    '\n'
    'def print_audit_summary(rows: list[list[object]]) -> None:\n'
    '    print_console_table(\n'
    '        "Google Sheet audit summary:",\n'
    '        [\n'
    '            "Indicator",\n'
    '            "Sheet",\n'
    '            "Latest sheet",\n'
    '            "Latest source",\n'
    '            "Sheet value",\n'
    '            "Source value",\n'
    '            "Status",\n'
    '            "Warnings",\n'
    '        ],\n'
    '        (\n'
    '            [row[0], row[1], row[4], row[5], row[6], row[7], row[8], row[9]]\n'
    '            for row in rows[1:]\n'
    '        ),\n'
    '    )\n'
    '    counts = pd.Series([row[8] for row in rows[1:]]).value_counts().to_dict()\n'
    '    count_summary = ", ".join(f"{key}={value}" for key, value in sorted(counts.items()))\n'
    '    print(f"\\nAudit status counts: {count_summary}")\n'
    '    print(f"Full audit CSV: {GOOGLE_SHEET_AUDIT_CSV}")\n'
    '\n'
    '\n'
    'def audit_google_workbook(\n'
    '    spreadsheet_id: str,\n'
    '    credentials_path: Path,\n'
    '    source_data: dict[str, list[dict]] | None = None,\n'
    '    source_errors: dict[str, str] | None = None,\n'
    ') -> None:\n'
    '    """Read the target workbook and print a non-mutating tab audit."""\n'
    '    if not credentials_path.exists():\n'
    '        raise SystemExit(f"Google credentials file not found: {credentials_path}")\n'
    '\n'
    '    access_token = None\n'
    '    sheets = None\n'
    '    try:\n'
    '        if os.getenv("US_PIPELINE_FORCE_DIRECT_GOOGLE_API") == "1":\n'
    '            raise ImportError("Direct Google Sheets API selected by pipeline runner.")\n'
    '        from google.oauth2.service_account import Credentials\n'
    '        from googleapiclient.discovery import build\n'
    '    except ImportError:\n'
    '        access_token = get_service_account_access_token(credentials_path)\n'
    '        base_url = f"{GOOGLE_SHEETS_API_BASE}/{spreadsheet_id}"\n'
    '        metadata = google_sheets_request("GET", base_url, access_token)\n'
    '        sheet_names = [\n'
    '            sheet["properties"]["title"] for sheet in metadata.get("sheets", [])\n'
    '        ]\n'
    '\n'
    '    else:\n'
    '        credentials = Credentials.from_service_account_file(\n'
    '            credentials_path,\n'
    '            scopes=GOOGLE_SHEETS_READONLY_SCOPES,\n'
    '        )\n'
    '        service = build("sheets", "v4", credentials=credentials)\n'
    '        sheets = service.spreadsheets()\n'
    '        metadata = sheets.get(spreadsheetId=spreadsheet_id).execute()\n'
    '        sheet_names = [\n'
    '            sheet["properties"]["title"] for sheet in metadata.get("sheets", [])\n'
    '        ]\n'
    '\n'
    '    sheet_name_set = set(sheet_names)\n'
    '    expected_tabs = expected_indicator_tabs()\n'
    '    missing_tabs = [name for name in expected_tabs if name not in sheet_name_set]\n'
    '\n'
    '    audit_ranges = [\n'
    '        f"{quote_sheet_name(sheet_name)}!A:"\n'
    '        f"{\'D\' if sheet_name in {CENTRAL_BANK_LIQUIDITY_SHEET, IMF_COVERAGE_SHEET} else \'B\'}"\n'
    '        for sheet_name in expected_tabs\n'
    '        if sheet_name in sheet_name_set\n'
    '    ]\n'
    '    if access_token is not None:\n'
    '        query = urlencode(\n'
    '            [("ranges", item) for item in audit_ranges]\n'
    '            + [("valueRenderOption", "UNFORMATTED_VALUE")]\n'
    '        )\n'
    '        batch_response = google_sheets_request(\n'
    '            "GET",\n'
    '            f"{base_url}/values:batchGet?{query}",\n'
    '            access_token,\n'
    '        )\n'
    '    else:\n'
    '        batch_response = (\n'
    '            sheets.values()\n'
    '            .batchGet(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                ranges=audit_ranges,\n'
    '                valueRenderOption="UNFORMATTED_VALUE",\n'
    '            )\n'
    '            .execute()\n'
    '        )\n'
    '    cached_values = {\n'
    '        requested_range: value_range.get("values", [])\n'
    '        for requested_range, value_range in zip(\n'
    '            audit_ranges,\n'
    '            batch_response.get("valueRanges", []),\n'
    '        )\n'
    '    }\n'
    '\n'
    '    def read_values(sheet_name: str, end_column: str) -> list[list[object]]:\n'
    '        range_name = f"{quote_sheet_name(sheet_name)}!A:{end_column}"\n'
    '        return cached_values.get(range_name, [])\n'
    '\n'
    '    print_console_table(\n'
    '        "Google Sheet workbook:",\n'
    '        ["Check", "Result"],\n'
    '        [\n'
    '            ["Workbook", f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}"],\n'
    '            ["Tabs found", len(sheet_names)],\n'
    '            [\n'
    '                "Missing expected tabs",\n'
    '                ", ".join(missing_tabs) if missing_tabs else "none",\n'
    '            ],\n'
    '        ],\n'
    '    )\n'
    '    audit_rows = audit_sheet_rows(\n'
    '        sheet_name_set, read_values, source_data, source_errors\n'
    '    )\n'
    '    write_audit_csv(audit_rows)\n'
    '    print_audit_summary(audit_rows)\n'
    '\n'
    '    check_tabs = [\n'
    '        "Real consumer spending growth",\n'
    '        "Consumer expectations",\n'
    '        "Employment growth",\n'
    '        "Core consumption inflation",\n'
    '        "Core producer price inflation",\n'
    '        "Headline producer price inflation",\n'
    '        "Consumer inflation expectations",\n'
    '        CENTRAL_BANK_LIQUIDITY_SHEET,\n'
    '        "Banking system reserves",\n'
    '        "Government revenue growth",\n'
    '        "Financial conditions index",\n'
    '    ]\n'
    '    for sheet_name in check_tabs:\n'
    '        if sheet_name not in sheet_name_set:\n'
    '            print(f"\\n{sheet_name}: MISSING")\n'
    '            continue\n'
    '        end_column = (\n'
    '            "D"\n'
    '            if sheet_name in {CENTRAL_BANK_LIQUIDITY_SHEET, IMF_COVERAGE_SHEET}\n'
    '            else "B"\n'
    '        )\n'
    '        values = read_values(sheet_name, end_column)\n'
    '        rows = [row for row in values[1:] if row and normalise_observation_date(row[0])]\n'
    '        blank_recent = [\n'
    '            row for row in rows[-12:] if len(row) < 2 or normalise_value(row[1]) is None\n'
    '        ]\n'
    '        latest_date = normalise_observation_date(rows[-1][0]) if rows else None\n'
    '        print_console_table(\n'
    '            sheet_name,\n'
    '            ["Check", "Result"],\n'
    '            [\n'
    '                ["Header", values[0] if values else []],\n'
    '                ["Latest valid date", latest_date.isoformat() if latest_date else ""],\n'
    '                ["Recent blank value rows", len(blank_recent)],\n'
    '            ],\n'
    '        )\n'
    '        if rows:\n'
    '            preview_headers = [console_cell(value) for value in values[0]]\n'
    '            print_console_table(\n'
    '                "Last 5 rows:",\n'
    '                preview_headers,\n'
    '                rows[-5:],\n'
    '            )\n'
    '\n'
    '\n'
    'def sync_all_indicators(\n'
    '    spreadsheet_id: str,\n'
    '    credentials_path: Path,\n'
    '    workbook_data: list[tuple[str, list[dict]]],\n'
    '    validate_only: bool = False,\n'
    '    _preflight: bool = False,\n'
    ') -> list[dict]:\n'
    '    """Sync every indicator to its dedicated Google Sheets tab."""\n'
    '    logging.getLogger("google.oauth2._client").setLevel(logging.ERROR)\n'
    '    if not credentials_path.exists():\n'
    '        raise SystemExit(f"Google credentials file not found: {credentials_path}")\n'
    '    if not validate_only:\n'
    '        preflight_results = sync_all_indicators(\n'
    '            spreadsheet_id,\n'
    '            credentials_path,\n'
    '            workbook_data,\n'
    '            validate_only=True,\n'
    '            _preflight=True,\n'
    '        )\n'
    '        assert_sync_preflight_passed(preflight_results)\n'
    '\n'
    '    try:\n'
    '        if os.getenv("US_PIPELINE_FORCE_DIRECT_GOOGLE_API") == "1":\n'
    '            raise ImportError("Direct Google Sheets API selected by pipeline runner.")\n'
    '        from google.oauth2.service_account import Credentials\n'
    '        from googleapiclient.discovery import build\n'
    '    except ImportError:\n'
    '        access_token = get_service_account_access_token(credentials_path)\n'
    '        base_url = f"{GOOGLE_SHEETS_API_BASE}/{spreadsheet_id}"\n'
    '        metadata = google_sheets_request("GET", base_url, access_token)\n'
    '        sheet_names = {\n'
    '            sheet["properties"]["title"] for sheet in metadata.get("sheets", [])\n'
    '        }\n'
    '        control_values = google_sheets_request(\n'
    '            "GET",\n'
    '            google_values_url(base_url, ANALYSIS_CONTROL_PANEL_RANGE),\n'
    '            access_token,\n'
    '        ).get("values", [])\n'
    '        validate_analysis_workbook_topology(sheet_names, control_values)\n'
    '        results = []\n'
    '        central_bank_data = central_bank_component_data(workbook_data)\n'
    '        if central_bank_data:\n'
    '            if COMPACT_TERMINAL and not _preflight:\n'
    '                print(f"Checking sheet: {CENTRAL_BANK_LIQUIDITY_SHEET}...", flush=True)\n'
    '            try:\n'
    '                results.append(\n'
    '                    sync_central_bank_liquidity_direct(\n'
    '                        base_url,\n'
    '                        access_token,\n'
    '                        central_bank_data,\n'
    '                        sheet_names,\n'
    '                        validate_only,\n'
    '                    )\n'
    '                )\n'
    '            except Exception as exc:\n'
    '                if isinstance(exc, requests.RequestException):\n'
    '                    raise\n'
    '                results.append(\n'
    '                    sync_result_row(\n'
    '                        CENTRAL_BANK_LIQUIDITY_SHEET,\n'
    '                        CENTRAL_BANK_LIQUIDITY_SHEET,\n'
    '                        None,\n'
    '                        0,\n'
    '                        0,\n'
    '                        None,\n'
    '                        str(exc),\n'
    '                    )\n'
    '                )\n'
    '        coverage_data = imf_coverage_component_data(workbook_data)\n'
    '        if coverage_data:\n'
    '            if COMPACT_TERMINAL and not _preflight:\n'
    '                print(f"Checking sheet: {IMF_COVERAGE_SHEET}...", flush=True)\n'
    '            try:\n'
    '                results.append(\n'
    '                    sync_imf_coverage_direct(\n'
    '                        base_url,\n'
    '                        access_token,\n'
    '                        coverage_data,\n'
    '                        sheet_names,\n'
    '                        validate_only,\n'
    '                    )\n'
    '                )\n'
    '            except Exception as exc:\n'
    '                if isinstance(exc, requests.RequestException):\n'
    '                    raise\n'
    '                results.append(\n'
    '                    sync_result_row(\n'
    '                        IMF_COVERAGE_SHEET,\n'
    '                        IMF_COVERAGE_SHEET,\n'
    '                        None,\n'
    '                        0,\n'
    '                        0,\n'
    '                        None,\n'
    '                        str(exc),\n'
    '                    )\n'
    '                )\n'
    '        for indicator_name, records in dedicated_indicator_data(workbook_data):\n'
    '            sheet_name = indicator_sheet_name(indicator_name, sheet_names)\n'
    '            if sheet_name is None:\n'
    '                results.append(\n'
    '                    sync_result_row(\n'
    '                        indicator_name,\n'
    '                        "",\n'
    '                        None,\n'
    '                        0,\n'
    '                        0,\n'
    '                        None,\n'
    '                        "Missing matching indicator tab; skipped.",\n'
    '                    )\n'
    '                )\n'
    '                continue\n'
    '            if COMPACT_TERMINAL and not _preflight:\n'
    '                print(f"Checking sheet: {sheet_name}...", flush=True)\n'
    '            try:\n'
    '                results.append(\n'
    '                    sync_indicator_sheet_direct(\n'
    '                        base_url,\n'
    '                        access_token,\n'
    '                        sheet_name,\n'
    '                        indicator_name,\n'
    '                        records,\n'
    '                        validate_only,\n'
    '                    )\n'
    '                )\n'
    '            except Exception as exc:\n'
    '                if isinstance(exc, requests.RequestException):\n'
    '                    raise\n'
    '                results.append(\n'
    '                    sync_result_row(\n'
    '                        indicator_name,\n'
    '                        sheet_name,\n'
    '                        None,\n'
    '                        0,\n'
    '                        0,\n'
    '                        None,\n'
    '                        str(exc),\n'
    '                    )\n'
    '                )\n'
    '        if not validate_only and not _preflight:\n'
    '            assert_sync_preflight_passed(results)\n'
    '            format_requests = analysis_number_format_requests(metadata)\n'
    '            if format_requests:\n'
    '                google_sheets_request(\n'
    '                    "POST",\n'
    '                    f"{base_url}:batchUpdate",\n'
    '                    access_token,\n'
    '                    {"requests": format_requests},\n'
    '                )\n'
    '        if not _preflight:\n'
    '            print_sync_summary(results, validate_only=validate_only)\n'
    '        return results\n'
    '\n'
    '    credentials = Credentials.from_service_account_file(\n'
    '        credentials_path,\n'
    '        scopes=GOOGLE_SHEETS_WRITE_SCOPES,\n'
    '    )\n'
    '    service = build("sheets", "v4", credentials=credentials)\n'
    '    sheets = service.spreadsheets()\n'
    '    metadata = sheets.get(spreadsheetId=spreadsheet_id).execute()\n'
    '    sheet_names = {sheet["properties"]["title"] for sheet in metadata.get("sheets", [])}\n'
    '    control_values = (\n'
    '        sheets.values()\n'
    '        .get(\n'
    '            spreadsheetId=spreadsheet_id,\n'
    '            range=ANALYSIS_CONTROL_PANEL_RANGE,\n'
    '            valueRenderOption="UNFORMATTED_VALUE",\n'
    '        )\n'
    '        .execute()\n'
    '        .get("values", [])\n'
    '    )\n'
    '    validate_analysis_workbook_topology(sheet_names, control_values)\n'
    '\n'
    '    results = []\n'
    '    central_bank_data = central_bank_component_data(workbook_data)\n'
    '    if central_bank_data:\n'
    '        try:\n'
    '            results.append(\n'
    '                sync_central_bank_liquidity_with_service(\n'
    '                    sheets,\n'
    '                    spreadsheet_id,\n'
    '                    central_bank_data,\n'
    '                    sheet_names,\n'
    '                    validate_only,\n'
    '                )\n'
    '            )\n'
    '        except Exception as exc:\n'
    '            results.append(\n'
    '                sync_result_row(\n'
    '                    CENTRAL_BANK_LIQUIDITY_SHEET,\n'
    '                    CENTRAL_BANK_LIQUIDITY_SHEET,\n'
    '                    None,\n'
    '                    0,\n'
    '                    0,\n'
    '                    None,\n'
    '                    str(exc),\n'
    '                )\n'
    '            )\n'
    '    coverage_data = imf_coverage_component_data(workbook_data)\n'
    '    if coverage_data:\n'
    '        try:\n'
    '            results.append(\n'
    '                sync_imf_coverage_with_service(\n'
    '                    sheets,\n'
    '                    spreadsheet_id,\n'
    '                    coverage_data,\n'
    '                    sheet_names,\n'
    '                    validate_only,\n'
    '                )\n'
    '            )\n'
    '        except Exception as exc:\n'
    '            results.append(\n'
    '                sync_result_row(\n'
    '                    IMF_COVERAGE_SHEET,\n'
    '                    IMF_COVERAGE_SHEET,\n'
    '                    None,\n'
    '                    0,\n'
    '                    0,\n'
    '                    None,\n'
    '                    str(exc),\n'
    '                )\n'
    '            )\n'
    '    for indicator_name, records in dedicated_indicator_data(workbook_data):\n'
    '        sheet_name = indicator_sheet_name(indicator_name, sheet_names)\n'
    '        if sheet_name is None:\n'
    '            results.append(\n'
    '                sync_result_row(\n'
    '                    indicator_name,\n'
    '                    "",\n'
    '                    None,\n'
    '                    0,\n'
    '                    0,\n'
    '                    None,\n'
    '                    "Missing matching indicator tab; skipped.",\n'
    '                )\n'
    '            )\n'
    '            continue\n'
    '        try:\n'
    '            results.append(\n'
    '                sync_indicator_sheet_with_service(\n'
    '                    sheets,\n'
    '                    spreadsheet_id,\n'
    '                    sheet_name,\n'
    '                    indicator_name,\n'
    '                    records,\n'
    '                    validate_only,\n'
    '                )\n'
    '            )\n'
    '        except Exception as exc:\n'
    '            results.append(\n'
    '                sync_result_row(\n'
    '                    indicator_name,\n'
    '                    sheet_name,\n'
    '                    None,\n'
    '                    0,\n'
    '                    0,\n'
    '                    None,\n'
    '                    str(exc),\n'
    '                )\n'
    '            )\n'
    '    if not validate_only and not _preflight:\n'
    '        assert_sync_preflight_passed(results)\n'
    '        format_requests = analysis_number_format_requests(metadata)\n'
    '        if format_requests:\n'
    '            sheets.batchUpdate(\n'
    '                spreadsheetId=spreadsheet_id,\n'
    '                body={"requests": format_requests},\n'
    '            ).execute()\n'
    '    if not _preflight:\n'
    '        print_sync_summary(results, validate_only=validate_only)\n'
    '    return results\n'
    '\n'
    '\n'
    'def collect_source_data_for_audit(\n'
    '    api_key: str, months: int\n'
    ') -> tuple[dict[str, list[dict]], dict[str, str]]:\n'
    '    """Collect source records for audit without letting one failed indicator stop the report."""\n'
    '    source_data: dict[str, list[dict]] = {}\n'
    '    source_errors: dict[str, str] = {}\n'
    '    for indicator in INDICATORS:\n'
    '        try:\n'
    '            source_data[indicator.name] = get_indicator_observations_safely(\n'
    '                api_key,\n'
    '                indicator,\n'
    '                months,\n'
    '            )\n'
    '        except Exception as exc:\n'
    '            source_errors[indicator.name] = str(exc)\n'
    '    return source_data, source_errors\n'
    '\n'
    '\n'
    'def run_internal_self_tests() -> None:\n'
    '    """Run fast offline checks before any external source or Google Sheet is used."""\n'
    '    global SOURCE_ARCHIVE_ROOT\n'
    '\n'
    '    def expect_failure(description: str, operation) -> None:\n'
    '        try:\n'
    '            operation()\n'
    '        except Exception:\n'
    '            return\n'
    '        raise RuntimeError(f"Internal test failed: {description} was not rejected.")\n'
    '\n'
    '    if [indicator.name for indicator in INDICATORS] != EXPECTED_HEADERS:\n'
    '        raise RuntimeError("Internal test failed: indicator headers or order changed.")\n'
    '    if len(EXPECTED_ANALYSIS_SHEET_ORDER) != 37:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: analysis sheet topology is not 37 tabs."\n'
    '        )\n'
    '    valid_analysis_tabs = {\n'
    '        ANALYSIS_CONTROL_PANEL_SHEET,\n'
    '        *EXPECTED_ANALYSIS_SHEET_ORDER,\n'
    '    }\n'
    '    valid_control_values = [[name] for name in EXPECTED_ANALYSIS_SHEET_ORDER]\n'
    '    validate_analysis_workbook_topology(\n'
    '        valid_analysis_tabs,\n'
    '        valid_control_values,\n'
    '    )\n'
    '    if analysis_value("1.234564") != 1.23456:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: five-decimal analysis rounding changed."\n'
    '        )\n'
    '    if analysis_value("1.234565") != 1.23457:\n'
    '        raise RuntimeError("Internal test failed: analysis half-up rounding changed.")\n'
    '    format_fixture = analysis_number_format_requests(\n'
    '        {\n'
    '            "sheets": [\n'
    '                {"properties": {"title": name, "sheetId": index}}\n'
    '                for index, name in enumerate(EXPECTED_ANALYSIS_SHEET_ORDER, start=1)\n'
    '            ]\n'
    '        }\n'
    '    )\n'
    '    if len(format_fixture) != len(EXPECTED_ANALYSIS_SHEET_ORDER):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: analysis number formatting lost a tab."\n'
    '        )\n'
    '    if any(\n'
    '        request["repeatCell"]["cell"]["userEnteredFormat"]["numberFormat"]["pattern"]\n'
    '        != "0.00000"\n'
    '        for request in format_fixture\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: analysis five-decimal format changed."\n'
    '        )\n'
    '    expect_failure(\n'
    '        "missing required analysis tab",\n'
    '        lambda: validate_analysis_workbook_topology(\n'
    '            valid_analysis_tabs - {EXPECTED_ANALYSIS_SHEET_ORDER[0]},\n'
    '            valid_control_values,\n'
    '        ),\n'
    '    )\n'
    '    blank_control_values = [list(row) for row in valid_control_values]\n'
    '    blank_control_values[10] = [""]\n'
    '    expect_failure(\n'
    '        "blank analysis Control Panel row",\n'
    '        lambda: validate_analysis_workbook_topology(\n'
    '            valid_analysis_tabs,\n'
    '            blank_control_values,\n'
    '        ),\n'
    '    )\n'
    '    if IMF_SUPPORT_INDICATORS != {\n'
    '        "Government revenue % GDP",\n'
    '        "Interest paid on public debt % GDP",\n'
    '    }:\n'
    '        raise RuntimeError("Internal test failed: IMF support indicator rules changed.")\n'
    '    if has_dedicated_indicator_tab("Government interest coverage ratio") is not False:\n'
    '        raise RuntimeError("Internal test failed: IMF coverage tab rule changed.")\n'
    '    if has_dedicated_indicator_tab("Government revenue % GDP") is not False:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: IMF revenue support tab rule changed."\n'
    '        )\n'
    '    if (\n'
    '        has_dedicated_indicator_tab(\n'
    '            "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level"\n'
    '        )\n'
    '        is not False\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: central-bank shared tab rule changed."\n'
    '        )\n'
    '    expected_tabs = expected_indicator_tabs()\n'
    '    if "Government interest coverage ratio" not in expected_tabs:\n'
    '        raise RuntimeError("Internal test failed: coverage ratio tab was dropped.")\n'
    '    if "Government revenue % GDP" in expected_tabs:\n'
    '        raise RuntimeError("Internal test failed: IMF support tab was added back.")\n'
    '    if CENTRAL_BANK_LIQUIDITY_SHEET not in expected_tabs:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: central-bank liquidity tab was dropped."\n'
    '        )\n'
    '    syncable_names = [\n'
    '        name\n'
    '        for name, _records in dedicated_indicator_data(\n'
    '            [(indicator.name, []) for indicator in INDICATORS]\n'
    '        )\n'
    '    ]\n'
    '    if "Government revenue % GDP" in syncable_names:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: IMF revenue support series was syncable."\n'
    '        )\n'
    '    if "Interest paid on public debt % GDP" in syncable_names:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: IMF interest support series was syncable."\n'
    '        )\n'
    '    if "Government interest coverage ratio" in syncable_names:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: derived IMF coverage ratio was syncable."\n'
    '        )\n'
    '    coverage_fixture = imf_coverage_sheet_values(\n'
    '        {\n'
    '            "Government revenue % GDP": [\n'
    '                {\n'
    '                    "observation_date": date(2024, 12, 31),\n'
    '                    "selection_date": date(2024, 12, 31),\n'
    '                    "value": 30.0,\n'
    '                    "source": "IMF DataMapper FPP",\n'
    '                }\n'
    '            ],\n'
    '            "Interest paid on public debt % GDP": [\n'
    '                {\n'
    '                    "observation_date": date(2024, 12, 31),\n'
    '                    "selection_date": date(2024, 12, 31),\n'
    '                    "value": 4.0,\n'
    '                    "source": "IMF DataMapper FPP",\n'
    '                }\n'
    '            ],\n'
    '        }\n'
    '    )\n'
    '    if coverage_fixture != [\n'
    '        ["Date", "Government Revenue % GDP", "Government Interest % GDP"],\n'
    '        [(date(2024, 12, 31) - date(1899, 12, 30)).days, 30.0, 4.0],\n'
    '    ]:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: IMF coverage A:C table mapping changed."\n'
    '        )\n'
    '    legacy_coverage = [\n'
    '        ["Date", "Government Revenue % GDP", "Government Interest % GDP"],\n'
    '        [(date(2022, 1, 1) - date(1899, 12, 30)).days, 28.0, 3.0],\n'
    '        [(date(2023, 1, 1) - date(1899, 12, 30)).days, 29.0, 3.5],\n'
    '    ]\n'
    '    official_coverage = [\n'
    '        ["Date", "Government Revenue % GDP", "Government Interest % GDP"],\n'
    '        [(date(2023, 12, 31) - date(1899, 12, 30)).days, 29.5, ""],\n'
    '        [(date(2024, 12, 31) - date(1899, 12, 30)).days, 30.0, 4.0],\n'
    '    ]\n'
    '    merged_coverage = merge_imf_coverage_history(\n'
    '        legacy_coverage,\n'
    '        official_coverage,\n'
    '    )\n'
    '    if merged_coverage != [\n'
    '        ["Date", "Government Revenue % GDP", "Government Interest % GDP"],\n'
    '        [(date(2022, 12, 31) - date(1899, 12, 30)).days, 28.0, 3.0],\n'
    '        [(date(2023, 12, 31) - date(1899, 12, 30)).days, 29.5, 3.5],\n'
    '        [(date(2024, 12, 31) - date(1899, 12, 30)).days, 30.0, 4.0],\n'
    '    ]:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: legacy IMF history was not preserved at year-end."\n'
    '        )\n'
    '    expect_failure(\n'
    '        "conflicting duplicate IMF year",\n'
    '        lambda: merge_imf_coverage_history(\n'
    '            [\n'
    '                legacy_coverage[0],\n'
    '                legacy_coverage[1],\n'
    '                [\n'
    '                    (date(2022, 12, 31) - date(1899, 12, 30)).days,\n'
    '                    99.0,\n'
    '                    3.0,\n'
    '                ],\n'
    '            ],\n'
    '            official_coverage,\n'
    '        ),\n'
    '    )\n'
    '    table_buffer = StringIO()\n'
    '    with redirect_stdout(table_buffer):\n'
    '        print_console_table("Internal table", ["A", "B"], [[1, "two"], ["three"]])\n'
    '    table_output = table_buffer.getvalue()\n'
    '    if "three |" not in table_output or "A     | B" not in table_output:\n'
    '        raise RuntimeError("Internal test failed: console table formatting changed.")\n'
    '    audit_warnings = sheet_quality_warnings(\n'
    '        {\n'
    '            "duplicate_dates": [date(2026, 1, 1)],\n'
    '            "blank_values": 1,\n'
    '            "non_numeric_values": 1,\n'
    '            "dates": [date(2026, 1, 1), date(2026, 3, 1)],\n'
    '        },\n'
    '        "fred",\n'
    '        "Quarterly",\n'
    '    )\n'
    '    if not any("Duplicate dates" in warning for warning in audit_warnings):\n'
    '        raise RuntimeError("Internal test failed: duplicate-date warning changed.")\n'
    '    if not any("off-quarter rows" in warning for warning in audit_warnings):\n'
    '        raise RuntimeError("Internal test failed: quarterly sparse warning changed.")\n'
    '    mismatch_warning = format_mismatch_warning([(date(2026, 5, 1), 1.0, 2.0)])\n'
    '    if "2026-05-01 sheet=1.0 source=2.0" not in mismatch_warning:\n'
    '        raise RuntimeError("Internal test failed: mismatch warning format changed.")\n'
    '\n'
    '    industrial = next(item for item in INDICATORS if item.series_id == "INDPRO")\n'
    '    spending = next(item for item in INDICATORS if item.series_id == "PCEC96")\n'
    '    inflation = next(\n'
    '        item for item in INDICATORS if item.name == "Consumer inflation expectations"\n'
    '    )\n'
    '    requested_fred_sources = {\n'
    '        "Headline CPI inflation": "CPIAUCSL",\n'
    '        "10Y real yield": "DFII10",\n'
    '        "2Y government bond yield": "DGS2",\n'
    '        "10Y government bond yield": "DGS10",\n'
    '        "10Y-3M yield curve spread": "T10Y3M",\n'
    '        "10Y term premium": "THREEFYTP10",\n'
    '        "Overnight interbank rate": "IRSTCI01USM156N",\n'
    '        "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level": "WALCL",\n'
    '        "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level": "WDTGAL",\n'
    '        "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations": "RRPONTSYD",\n'
    '        "Banking system reserves": "WRESBAL",\n'
    '        "Financial conditions index": "NFCI",\n'
    '        "Bank credit growth": "TOTLL",\n'
    '        "Corporate credit spread": "BAA10Y",\n'
    '    }\n'
    '    configured_fred_sources = {\n'
    '        indicator.name: indicator.series_id\n'
    '        for indicator in INDICATORS\n'
    '        if indicator.name in requested_fred_sources\n'
    '    }\n'
    '    if configured_fred_sources != requested_fred_sources:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: requested FRED source mapping changed."\n'
    '        )\n'
    '    if any(\n'
    '        indicator.monthly_aggregation != "latest"\n'
    '        for indicator in INDICATORS\n'
    '        if indicator.name in requested_fred_sources\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: requested month-end selection changed."\n'
    '        )\n'
    '    if (industrial.units, industrial.release_lag_months) != ("lin", 0):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: industrial production settings changed."\n'
    '        )\n'
    '    if (spending.units, spending.release_lag_months) != ("lin", 0):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: real consumer spending settings changed."\n'
    '        )\n'
    '    if inflation.release_lag_months != 1:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: inflation expectations release dating changed."\n'
    '        )\n'
    '\n'
    '    imf_revenue_indicator = next(\n'
    '        item for item in INDICATORS if item.name == "Government revenue % GDP"\n'
    '    )\n'
    '    imf_interest_indicator = next(\n'
    '        item for item in INDICATORS if item.name == "Interest paid on public debt % GDP"\n'
    '    )\n'
    '    imf_coverage_indicator = next(\n'
    '        item for item in INDICATORS if item.name == "Government interest coverage ratio"\n'
    '    )\n'
    '    original_imf_get = imf_datamapper_get\n'
    '    original_statuses = dict(SOURCE_STATUSES)\n'
    '    original_messages = list(VALIDATION_MESSAGES)\n'
    '    original_imf_cache = dict(_IMF_FPP_CACHE)\n'
    '    try:\n'
    '\n'
    '        def fake_imf_get(path: str, params: dict[str, str] | None = None) -> dict:\n'
    '            if path == "indicators":\n'
    '                return {\n'
    '                    "indicators": {\n'
    '                        "rev": {\n'
    '                            "label": "Government revenue, percent of GDP",\n'
    '                            "source": "Public Finances in Modern History Database",\n'
    '                            "unit": "% of GDP",\n'
    '                            "dataset": "FPP",\n'
    '                            "last-modified": "2026-01-15 00:00:00",\n'
    '                        },\n'
    '                        "ie": {\n'
    '                            "label": "Interest paid on public debt, percent of GDP",\n'
    '                            "source": "Public Finances in Modern History Database",\n'
    '                            "unit": "% of GDP",\n'
    '                            "dataset": "FPP",\n'
    '                            "last-modified": "2026-01-15 00:00:00",\n'
    '                        },\n'
    '                    }\n'
    '                }\n'
    '            if path == "rev/USA":\n'
    '                return {"values": {"rev": {"USA": {"2024": "29.5"}}}}\n'
    '            if path == "ie/USA":\n'
    '                return {"values": {"ie": {"USA": {"2024": "4.0"}}}}\n'
    '            raise RuntimeError(path)\n'
    '\n'
    '        globals()["imf_datamapper_get"] = fake_imf_get\n'
    '        _IMF_FPP_CACHE.clear()\n'
    '        SOURCE_STATUSES.clear()\n'
    '        VALIDATION_MESSAGES.clear()\n'
    '        imf_revenue_records = get_imf_public_finance_observations(\n'
    '            imf_revenue_indicator,\n'
    '            3,\n'
    '        )\n'
    '        imf_interest_records = get_imf_public_finance_observations(\n'
    '            imf_interest_indicator,\n'
    '            3,\n'
    '        )\n'
    '        imf_coverage_records = get_imf_public_finance_observations(\n'
    '            imf_coverage_indicator,\n'
    '            3,\n'
    '        )\n'
    '        if (\n'
    '            len(imf_revenue_records) != 1\n'
    '            or len(imf_interest_records) != 1\n'
    '            or len(imf_coverage_records) != 1\n'
    '        ):\n'
    '            raise RuntimeError("Internal test failed: IMF annual record count changed.")\n'
    '        if {record["value"] for record in imf_revenue_records} != {29.5}:\n'
    '            raise RuntimeError("Internal test failed: IMF revenue values changed.")\n'
    '        if {record["value"] for record in imf_interest_records} != {4.0}:\n'
    '            raise RuntimeError("Internal test failed: IMF interest values changed.")\n'
    '        if {record["value"] for record in imf_coverage_records} != {7.375}:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: IMF coverage ratio calculation changed."\n'
    '            )\n'
    '        if imf_revenue_records[0]["selection_date"] != date(2024, 12, 1):\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: IMF annual selection date changed."\n'
    '            )\n'
    '        if latest_available_record(imf_revenue_records, date(2026, 6, 30)) is not None:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: IMF annual value was carried forward."\n'
    '            )\n'
    '        selected_december = latest_available_record(\n'
    '            imf_revenue_records, date(2024, 12, 31)\n'
    '        )\n'
    '        if not selected_december or selected_december["value"] != 29.5:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: IMF annual value did not match December."\n'
    '            )\n'
    '    finally:\n'
    '        globals()["imf_datamapper_get"] = original_imf_get\n'
    '        _IMF_FPP_CACHE.clear()\n'
    '        _IMF_FPP_CACHE.update(original_imf_cache)\n'
    '        SOURCE_STATUSES.clear()\n'
    '        SOURCE_STATUSES.update(original_statuses)\n'
    '        VALIDATION_MESSAGES.clear()\n'
    '        VALIDATION_MESSAGES.extend(original_messages)\n'
    '\n'
    '    current_month = month_start(date.today())\n'
    '    original_fred_get = fred_get\n'
    '    try:\n'
    '\n'
    '        def fake_fred_get(endpoint: str, params: dict[str, str]) -> dict:\n'
    '            if params.get("output_type") == "4":\n'
    '                return {\n'
    '                    "observations": [\n'
    '                        {\n'
    '                            "date": "2026-04-01",\n'
    '                            "realtime_start": "2026-05-15",\n'
    '                            "value": "101.0",\n'
    '                        }\n'
    '                    ]\n'
    '                }\n'
    '            return {\n'
    '                "observations": [\n'
    '                    {\n'
    '                        "date": "2026-04-01",\n'
    '                        "realtime_start": "2026-06-22",\n'
    '                        "value": "102.5",\n'
    '                    }\n'
    '                ]\n'
    '            }\n'
    '\n'
    '        globals()["fred_get"] = fake_fred_get\n'
    '        current_value_records = get_fred_observations(\n'
    '            "test",\n'
    '            "INTERNAL",\n'
    '            date(2026, 4, 1),\n'
    '            date(2026, 4, 30),\n'
    '            frequency="Monthly",\n'
    '        )\n'
    '        if (\n'
    '            len(current_value_records) != 1\n'
    '            or current_value_records[0]["value"] != 102.5\n'
    '            or current_value_records[0]["release_date"] != date(2026, 5, 15)\n'
    '            or current_value_records[0]["observation_date"] != date(2026, 4, 1)\n'
    '        ):\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: current FRED value/date matching changed."\n'
    '            )\n'
    '    finally:\n'
    '        globals()["fred_get"] = original_fred_get\n'
    '\n'
    '    point_in_time_records = [\n'
    '        {\n'
    '            "observation_date": date(2026, 3, 1),\n'
    '            "release_date": date(2026, 4, 15),\n'
    '            "value": 10.0,\n'
    '        },\n'
    '        {\n'
    '            "observation_date": date(2026, 4, 1),\n'
    '            "release_date": date(2026, 5, 15),\n'
    '            "value": 11.0,\n'
    '        },\n'
    '    ]\n'
    '    validate_release_records(point_in_time_records, "Internal point-in-time")\n'
    '    april_selection = latest_available_record(\n'
    '        point_in_time_records,\n'
    '        date(2026, 4, 30),\n'
    '    )\n'
    '    may_selection = latest_available_record(\n'
    '        point_in_time_records,\n'
    '        date(2026, 5, 31),\n'
    '    )\n'
    '    if (\n'
    '        april_selection != point_in_time_records[0]\n'
    '        or may_selection != point_in_time_records[1]\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: release-date-aware selection changed."\n'
    '        )\n'
    '    if latest_available_record(point_in_time_records, date(2026, 3, 31)) is not None:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: unreleased data created look-ahead bias."\n'
    '        )\n'
    '    if latest_available_record(point_in_time_records, date(2026, 6, 30)) is not None:\n'
    '        raise RuntimeError("Internal test failed: an older value was carried forward.")\n'
    '    monthly_average_records = [\n'
    '        {\n'
    '            "observation_date": date(2026, 4, 1),\n'
    '            "release_date": date(2026, 4, 2),\n'
    '            "selection_date": date(2026, 4, 1),\n'
    '            "period_aligned": True,\n'
    '            "monthly_aggregation": "mean",\n'
    '            "value": 10.0,\n'
    '        },\n'
    '        {\n'
    '            "observation_date": date(2026, 4, 8),\n'
    '            "release_date": date(2026, 4, 9),\n'
    '            "selection_date": date(2026, 4, 1),\n'
    '            "period_aligned": True,\n'
    '            "monthly_aggregation": "mean",\n'
    '            "value": 14.0,\n'
    '        },\n'
    '    ]\n'
    '    averaged_selection = latest_available_record(\n'
    '        monthly_average_records,\n'
    '        date(2026, 4, 30),\n'
    '    )\n'
    '    if (\n'
    '        not averaged_selection\n'
    '        or averaged_selection["value"] != 12.0\n'
    '        or averaged_selection.get("monthly_average_count") != 2\n'
    '    ):\n'
    '        raise RuntimeError("Internal test failed: monthly averaging changed.")\n'
    '    latest_selection = latest_available_record(\n'
    '        [\n'
    '            {**record, "monthly_aggregation": "latest"}\n'
    '            for record in monthly_average_records\n'
    '        ],\n'
    '        date(2026, 4, 30),\n'
    '    )\n'
    '    if not latest_selection or latest_selection["value"] != 14.0:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: month-end latest-point selection changed."\n'
    '        )\n'
    '    observation_period_record = {\n'
    '        "observation_date": date(2026, 5, 1),\n'
    '        "release_date": date(2026, 6, 15),\n'
    '        "selection_date": date(2026, 5, 1),\n'
    '        "period_aligned": True,\n'
    '        "value": 12.0,\n'
    '    }\n'
    '    if (\n'
    '        latest_available_record(\n'
    '            [observation_period_record],\n'
    '            date(2026, 5, 31),\n'
    '        )\n'
    '        != observation_period_record\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: FRED observation-period dating changed."\n'
    '        )\n'
    '    if (\n'
    '        latest_available_record(\n'
    '            [observation_period_record],\n'
    '            date(2026, 6, 30),\n'
    '        )\n'
    '        is not None\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: observation-period value was carried forward."\n'
    '        )\n'
    '    original_output_months = CURRENT_OUTPUT_MONTHS\n'
    '    try:\n'
    '        globals()["CURRENT_OUTPUT_MONTHS"] = 3\n'
    '        pmi_like_records = [\n'
    '            {\n'
    '                "observation_date": date(2026, 5, 1),\n'
    '                "release_date": date(2026, 6, 3),\n'
    '                "selection_date": date(2026, 6, 1),\n'
    '                "period_aligned": True,\n'
    '                "value": 52.1,\n'
    '            }\n'
    '        ]\n'
    '        pmi_output_records = output_month_records_for_indicator(pmi_like_records)\n'
    '        if pmi_output_records != {date(2026, 6, 1): 52.1}:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: PMI-style release-month tab output changed."\n'
    '            )\n'
    '        quarterly_fred_records = [\n'
    '            {\n'
    '                "observation_date": date(2026, 1, 1),\n'
    '                "release_date": date(2026, 3, 27),\n'
    '                "realtime_start": date(2026, 3, 27),\n'
    '                "source": "FRED / ALFRED",\n'
    '                "source_series_id": "INTERNALQ",\n'
    '                "native_frequency": "Quarterly",\n'
    '                "selection_date": date(2026, 3, 1),\n'
    '                "period_aligned": True,\n'
    '                "value": 123.4,\n'
    '            }\n'
    '        ]\n'
    '        quarterly_output_records = output_month_records_for_indicator(\n'
    '            quarterly_fred_records\n'
    '        )\n'
    '        if quarterly_output_records != {date(2026, 3, 1): 123.4}:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: sparse quarterly FRED output was hidden."\n'
    '            )\n'
    '        cleanup_rows = sparse_cleanup_plan(\n'
    '            [\n'
    '                ["Date", "Value"],\n'
    '                ["31/01/2026", "123.4"],\n'
    '                ["31/03/2026", "123.4"],\n'
    '            ],\n'
    '            quarterly_fred_records,\n'
    '        )\n'
    '        if cleanup_rows != [2]:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: sparse-date cleanup target changed."\n'
    '            )\n'
    '    finally:\n'
    '        globals()["CURRENT_OUTPUT_MONTHS"] = original_output_months\n'
    '    transformed_growth = transform_percent_change_from_year_ago(\n'
    '        [\n'
    '            {\n'
    '                "observation_date": date(2025, 4, 1),\n'
    '                "release_date": date(2025, 5, 15),\n'
    '                "realtime_start": date(2025, 5, 15),\n'
    '                "selection_date": date(2025, 4, 1),\n'
    '                "value": 100.0,\n'
    '            },\n'
    '            {\n'
    '                "observation_date": date(2026, 4, 1),\n'
    '                "release_date": date(2026, 5, 15),\n'
    '                "realtime_start": date(2026, 5, 15),\n'
    '                "selection_date": date(2026, 4, 1),\n'
    '                "value": 103.0,\n'
    '            },\n'
    '        ],\n'
    '        "Monthly",\n'
    '        "Internal growth",\n'
    '    )\n'
    '    if (\n'
    '        len(transformed_growth) != 1\n'
    '        or transformed_growth[0]["observation_date"] != date(2026, 4, 1)\n'
    '        or transformed_growth[0]["release_date"] != date(2026, 5, 15)\n'
    '        or transformed_growth[0]["realtime_start"] != date(2026, 5, 15)\n'
    '        or transformed_growth[0]["source"] != "FRED / ALFRED"\n'
    '        or transformed_growth[0]["source_series_id"] != "Internal growth"\n'
    '        or transformed_growth[0]["selection_date"] != date(2026, 4, 1)\n'
    '        or transformed_growth[0]["period_aligned"] is not True\n'
    '        or transformed_growth[0]["value_vintage"] != "Current FRED value"\n'
    '        or abs(transformed_growth[0]["value"] - 3.0) > 1e-12\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: year-over-year transformation changed."\n'
    '        )\n'
    '    if format_output_month(date(2026, 5, 1)) != "31/05/2026":\n'
    '        raise RuntimeError("Internal test failed: output date format changed.")\n'
    '    if format_output_month(date(2026, 6, 1)) != "30/06/2026":\n'
    '        raise RuntimeError("Internal test failed: June month-end output date changed.")\n'
    '    if normalise_observation_date("-1490-01-01") is not None:\n'
    '        raise RuntimeError("Internal test failed: malformed ancient date was accepted.")\n'
    '    if normalise_observation_date(-1234567) is not None:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: impossible spreadsheet serial date was accepted."\n'
    '        )\n'
    '    if sparse_replacement_date(date(2026, 1, 1), "quarterly") != date(2026, 3, 1):\n'
    '        raise RuntimeError("Internal test failed: Q1 sparse replacement date changed.")\n'
    '    if sparse_replacement_date(date(2026, 10, 1), "annual") != date(2026, 12, 1):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: annual sparse replacement date changed."\n'
    '        )\n'
    '    annual_fred = Indicator("Internal annual FRED", "INTERNAL")\n'
    '    original_get_indicator = get_indicator_observations\n'
    '    try:\n'
    '        globals()["get_indicator_observations"] = lambda *_args, **_kwargs: []\n'
    '        if get_indicator_observations_safely("test", annual_fred, 3) != []:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: empty sparse FRED data was not preserved."\n'
    '            )\n'
    '        expect_failure(\n'
    '            "empty non-FRED source",\n'
    '            lambda: get_indicator_observations_safely(\n'
    '                "test",\n'
    '                Indicator("Internal scraped source", source="nyfed_sce"),\n'
    '                3,\n'
    '            ),\n'
    '        )\n'
    '    finally:\n'
    '        globals()["get_indicator_observations"] = original_get_indicator\n'
    '\n'
    '    manufacturing_html = """\n'
    '    <h1>May 2026 ISM Manufacturing PMI Report</h1>\n'
    '    <div>Manufacturing PMI 54.0 52.7 +1.3</div>\n'
    '    <div>New Orders 56.8 54.1 +2.7</div>\n'
    '    """\n'
    '    manufacturing = parse_ism_manufacturing_report(manufacturing_html, "internal-test")\n'
    '    if not manufacturing or manufacturing.values != {\n'
    '        "manufacturing_pmi": 54.0,\n'
    '        "manufacturing_new_orders": 56.8,\n'
    '    }:\n'
    '        raise RuntimeError("Internal test failed: Manufacturing ISM parser changed.")\n'
    '\n'
    '    manufacturing_prior_value_html = """\n'
    '    <h1>May 2026 ISM Manufacturing PMI Report</h1>\n'
    '    <div>Manufacturing PMI 54.0 53.4 +0.6</div>\n'
    '    <p>The Manufacturing PMI registered 54.0 percent in May, compared with\n'
    '    the 53.4-percent reading recorded in April.</p>\n'
    '    <div>New Orders 56.8 54.1 +2.7</div>\n'
    '    """\n'
    '    manufacturing_prior_value = parse_ism_manufacturing_report(\n'
    '        manufacturing_prior_value_html,\n'
    '        "internal-test",\n'
    '    )\n'
    '    if (\n'
    '        not manufacturing_prior_value\n'
    '        or manufacturing_prior_value.values["manufacturing_pmi"] != 54.0\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: prior-month ISM value was mistaken for current data."\n'
    '        )\n'
    '\n'
    '    services_html = """\n'
    '    <h1>March 2026 ISM Services PMI Report</h1>\n'
    '    <div>Services PMI 54.0 56.1 -2.1 Growing Slower 21 52.7 52.4 +0.3</div>\n'
    '    <div>Business Activity/Production 53.9 59.9 -6.0 Growing Slower 21 55.1 53.5 +1.6</div>\n'
    '    <div>New Orders 60.6 58.6 +2.0 Growing Faster 10 53.5 55.8 -2.3</div>\n'
    '    """\n'
    '    services = parse_ism_services_report(services_html, "internal-test")\n'
    '    expected_services = {\n'
    '        "services_pmi": 54.0,\n'
    '        "services_new_orders": 60.6,\n'
    '        "services_business_activity": 53.9,\n'
    '        "manufacturing_pmi": 52.7,\n'
    '        "manufacturing_new_orders": 53.5,\n'
    '    }\n'
    '    if not services or services.values != expected_services:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: Services ISM parser or fallback changed."\n'
    '        )\n'
    '\n'
    '    services_table_html = """\n'
    '    <h1>March 2026 ISM Services PMI Report</h1>\n'
    '    <table>\n'
    '      <tr><th>Index</th><th>Services Current</th><th>Services Prior</th>\n'
    '          <th>Change</th><th>Direction</th><th>Months</th>\n'
    '          <th>Manufacturing Current</th><th>Manufacturing Prior</th>\n'
    '          <th>Change</th></tr>\n'
    '      <tr><td>Services PMI®</td><td>54.0</td><td>56.1</td><td>-2.1</td>\n'
    '          <td>Growing Slower</td><td>21</td><td>52.7</td><td>52.4</td><td>+0.3</td></tr>\n'
    '      <tr><td>Business Activity/Production</td><td>53.9</td><td>59.9</td>\n'
    '          <td>-6.0</td><td>Growing Slower</td><td>21</td>\n'
    '          <td>55.1</td><td>53.5</td><td>+1.6</td></tr>\n'
    '      <tr><td>New Orders</td><td>60.6</td><td>58.6</td><td>+2.0</td>\n'
    '          <td>Growing Faster</td><td>10</td><td>53.5</td><td>55.8</td><td>-2.3</td></tr>\n'
    '    </table>\n'
    '    """\n'
    '    services_table = parse_ism_services_report(services_table_html, "internal-test")\n'
    '    if not services_table or services_table.values != expected_services:\n'
    '        raise RuntimeError("Internal test failed: ISM HTML table parser changed.")\n'
    '    services_stale_metadata_html = """\n'
    '    <div class="stale-metadata">June 2026 Services Index</div>\n'
    '    <h1>July 2026 ISM Services PMI Report</h1>\n'
    '    <div>Services PMI 54.1 54.0 +0.1</div>\n'
    '    <div>Business Activity/Production 59.1 55.4 +3.7</div>\n'
    '    <div>New Orders 57.2 55.1 +2.1</div>\n'
    '    """\n'
    '    services_stale_metadata = parse_ism_services_report(\n'
    '        services_stale_metadata_html,\n'
    '        "internal-test",\n'
    '    )\n'
    '    if (\n'
    '        not services_stale_metadata\n'
    '        or services_stale_metadata.report_date != date(2026, 7, 1)\n'
    '        or services_stale_metadata.values["services_pmi"] != 54.1\n'
    '        or services_stale_metadata.values["services_new_orders"] != 57.2\n'
    '        or services_stale_metadata.values["services_business_activity"] != 59.1\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: stale ISM metadata overrode the visible report heading."\n'
    '        )\n'
    '    ism_block = pd.DataFrame(\n'
    '        {\n'
    '            "date": [\n'
    '                "2026-01-01",\n'
    '                "2026-03-01",\n'
    '                "2026-04-01",\n'
    '                "2026-05-01",\n'
    '                "2026-06-01",\n'
    '            ],\n'
    '            **{column: [50.0, 51.0, 52.0, 53.0, 54.0] for column in ISM_VALUE_COLUMNS},\n'
    '        }\n'
    '    )\n'
    '    latest_ism_block = latest_contiguous_ism_rows(ism_block, 4)\n'
    '    if latest_ism_block["date"].tolist() != [\n'
    '        "2026-03-01",\n'
    '        "2026-04-01",\n'
    '        "2026-05-01",\n'
    '        "2026-06-01",\n'
    '    ]:\n'
    '        raise RuntimeError("Internal test failed: ISM contiguous backfill changed.")\n'
    '\n'
    '    previous_release_month = add_months(current_month, -1)\n'
    '    latest_report_month = add_months(current_month, -1)\n'
    '    previous_report_month = add_months(current_month, -2)\n'
    '    partial_ism_frame = pd.DataFrame(\n'
    '        {\n'
    '            "date": [previous_release_month.isoformat(), current_month.isoformat()],\n'
    '            "report_month": [\n'
    '                previous_report_month.isoformat(),\n'
    '                latest_report_month.isoformat(),\n'
    '            ],\n'
    '            "manufacturing_pmi": [52.0, 55.6],\n'
    '            "manufacturing_new_orders": [53.0, 56.7],\n'
    '            "services_pmi": [51.0, None],\n'
    '            "services_new_orders": [52.0, None],\n'
    '            "services_business_activity": [53.0, None],\n'
    '        }\n'
    '    )\n'
    '    original_message_count = len(VALIDATION_MESSAGES)\n'
    '    try:\n'
    '        validate_ism_frame(partial_ism_frame, expected_months=2)\n'
    '        partial_messages = VALIDATION_MESSAGES[original_message_count:]\n'
    '        if len(partial_messages) != 3 or any(\n'
    '            "existing spreadsheet cells will be preserved" not in message["message"]\n'
    '            for message in partial_messages\n'
    '        ):\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: partial latest ISM warnings changed."\n'
    '            )\n'
    '    finally:\n'
    '        del VALIDATION_MESSAGES[original_message_count:]\n'
    '\n'
    '    historical_gap_frame = partial_ism_frame.copy()\n'
    '    historical_gap_frame.loc[0, "services_pmi"] = None\n'
    '    validate_ism_frame(historical_gap_frame, expected_months=2)\n'
    '\n'
    '    manufacturing_rows = output_month_records_for_indicator(\n'
    '        [\n'
    '            {\n'
    '                "observation_date": latest_report_month,\n'
    '                "release_date": current_month.replace(day=7),\n'
    '                "selection_date": current_month,\n'
    '                "period_aligned": True,\n'
    '                "value": 55.6,\n'
    '            }\n'
    '        ]\n'
    '    )\n'
    '    services_rows = output_month_records_for_indicator(\n'
    '        [\n'
    '            {\n'
    '                "observation_date": previous_report_month,\n'
    '                "release_date": previous_release_month.replace(day=10),\n'
    '                "selection_date": previous_release_month,\n'
    '                "period_aligned": True,\n'
    '                "value": 51.0,\n'
    '            }\n'
    '        ]\n'
    '    )\n'
    '    if manufacturing_rows.get(current_month) != 55.6:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: available latest ISM value was not exported."\n'
    '        )\n'
    '    if current_month in services_rows:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: missing latest ISM value was carried forward."\n'
    '        )\n'
    '\n'
    '    original_explicit_blanks = {\n'
    '        name: set(months) for name, months in _EXPLICIT_BLANK_OUTPUT_MONTHS.items()\n'
    '    }\n'
    '    try:\n'
    '        _EXPLICIT_BLANK_OUTPUT_MONTHS["Services PMI"] = {current_month}\n'
    '        partial_service_sheet_rows = sheet_rows_for_indicator(\n'
    '            "Services PMI",\n'
    '            [\n'
    '                {\n'
    '                    "observation_date": previous_report_month,\n'
    '                    "release_date": previous_release_month.replace(day=10),\n'
    '                    "selection_date": previous_release_month,\n'
    '                    "period_aligned": True,\n'
    '                    "value": 51.0,\n'
    '                }\n'
    '            ],\n'
    '        )\n'
    '        if partial_service_sheet_rows.get(current_month, "missing") is not None:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: unavailable latest ISM cell was not blanked."\n'
    '            )\n'
    '\n'
    '        stale_service_existing = read_existing_indicator_data(\n'
    '            [["Date", "Value"], [format_output_month(current_month), 99.0]],\n'
    '            {"date_col": 0, "value_col": 1, "header_row": 0},\n'
    '        )\n'
    '        blank_fills, _blank_appends = prepare_sheet_changes(\n'
    '            stale_service_existing,\n'
    '            partial_service_sheet_rows,\n'
    '        )\n'
    '        if blank_fills != [(2, current_month, None)]:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: stale latest ISM value was not scheduled for clearing."\n'
    '            )\n'
    '\n'
    '        partial_wide = merge_analysis_values_into_wide_table(\n'
    '            {\n'
    '                "Manufacturing PMI": [\n'
    '                    ["Date", "Manufacturing PMI"],\n'
    '                    [format_output_month(current_month), 55.6],\n'
    '                ],\n'
    '                "Services PMI": [\n'
    '                    ["Date", "Services PMI"],\n'
    '                    [format_output_month(current_month), 99.0],\n'
    '                ],\n'
    '            },\n'
    '            [\n'
    '                (\n'
    '                    "Services PMI",\n'
    '                    [\n'
    '                        {\n'
    '                            "observation_date": previous_report_month,\n'
    '                            "release_date": previous_release_month.replace(day=10),\n'
    '                            "selection_date": previous_release_month,\n'
    '                            "period_aligned": True,\n'
    '                            "value": 51.0,\n'
    '                        }\n'
    '                    ],\n'
    '                )\n'
    '            ],\n'
    '        )\n'
    '        service_column = partial_wide[0].index("Services PMI")\n'
    '        if partial_wide[-1][service_column] != "":\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: standalone table retained a stale ISM value."\n'
    '            )\n'
    '\n'
    '        _EXPLICIT_BLANK_OUTPUT_MONTHS["Services PMI"] = set()\n'
    '        published_service_rows = sheet_rows_for_indicator(\n'
    '            "Services PMI",\n'
    '            [\n'
    '                {\n'
    '                    "observation_date": latest_report_month,\n'
    '                    "release_date": current_month.replace(day=10),\n'
    '                    "selection_date": current_month,\n'
    '                    "period_aligned": True,\n'
    '                    "value": 54.0,\n'
    '                }\n'
    '            ],\n'
    '        )\n'
    '        later_fills, later_appends = prepare_sheet_changes(\n'
    '            read_existing_indicator_data(\n'
    '                [["Date", "Value"], [format_output_month(current_month), ""]],\n'
    '                {"date_col": 0, "value_col": 1, "header_row": 0},\n'
    '            ),\n'
    '            published_service_rows,\n'
    '        )\n'
    '        if later_fills != [(2, current_month, 54.0)] or later_appends:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: later ISM publication did not fill the blank."\n'
    '            )\n'
    '    finally:\n'
    '        _EXPLICIT_BLANK_OUTPUT_MONTHS.clear()\n'
    '        _EXPLICIT_BLANK_OUTPUT_MONTHS.update(original_explicit_blanks)\n'
    '\n'
    '    expect_failure(\n'
    '        "conflicting official values",\n'
    '        lambda: checked_value("test", "PMI", 50.0, 51.0),\n'
    '    )\n'
    '    expect_failure(\n'
    '        "unidentified ISM report",\n'
    '        lambda: parse_ism_manufacturing_report(\n'
    '            "<html><body>unexpected page</body></html>",\n'
    '            "internal-test",\n'
    '        ),\n'
    '    )\n'
    '    login_redirect = OfficialSourceRedirectError(\n'
    '        "Internal ISM test",\n'
    '        "https://ecommerce.ismworld.org/SSO/Login.aspx?DPLF=Y",\n'
    '    )\n'
    '    if not is_ism_login_redirect(login_redirect):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: ISM login redirect was not recognized."\n'
    '        )\n'
    '    fresh_test_url = fresh_ism_url(\n'
    '        "https://www.ismworld.org/reports/services/july/?keep=yes"\n'
    '    )\n'
    '    fresh_test_query = dict(parse_qsl(urlparse(fresh_test_url).query))\n'
    '    if (\n'
    '        urlparse(fresh_test_url).path != "/reports/services/july/"\n'
    '        or fresh_test_query.get("keep") != "yes"\n'
    '        or not fresh_test_query.get("_ism_refresh")\n'
    '    ):\n'
    '        raise RuntimeError("Internal test failed: ISM cache-busting URL changed.")\n'
    '    expect_failure(\n'
    '        "HTML disguised as XLSX",\n'
    '        lambda: validate_excel_signature(b"<html>error</html>", "Internal XLSX test"),\n'
    '    )\n'
    '    validate_excel_signature(b"PK\\x03\\x04workbook", "Internal XLSX test")\n'
    '    validate_excel_signature(\n'
    '        b"\\xd0\\xcf\\x11\\xe0workbook",\n'
    '        "Internal XLS test",\n'
    '        legacy_xls=True,\n'
    '    )\n'
    '\n'
    '    duplicate_dates = pd.DataFrame(\n'
    '        {\n'
    '            "date": [add_months(current_month, -1), add_months(current_month, -1)],\n'
    '            "value": [1.0, 2.0],\n'
    '        }\n'
    '    )\n'
    '    expect_failure(\n'
    '        "duplicate source months",\n'
    '        lambda: validate_monthly_dates(duplicate_dates, "Internal monthly source"),\n'
    '    )\n'
    '    expect_failure(\n'
    '        "missing requested observations",\n'
    '        lambda: validate_observation_tail(\n'
    '            [{"date": current_month, "value": 1.0}],\n'
    '            "Internal monthly source",\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    expect_failure(\n'
    '        "Google Sheets read-back mismatch",\n'
    '        lambda: verify_google_sheet_values(\n'
    '            [["Date", "Value"], [current_month.isoformat(), 1.0]],\n'
    '            [["Date", "Value"], [current_month.isoformat(), 2.0]],\n'
    '        ),\n'
    '    )\n'
    '    verify_google_sheet_values(\n'
    '        [["Date", "Value"], [current_month.isoformat(), 54.0]],\n'
    '        [["Date", "Value"], [current_month.isoformat(), "54"]],\n'
    '    )\n'
    '    verify_google_sheet_values(\n'
    '        [["Date", "Value"], [current_month.isoformat(), 3.50]],\n'
    '        [["Date", "Value"], [current_month.isoformat(), 3.5]],\n'
    '    )\n'
    '    verify_google_sheet_values(\n'
    '        [["Date", "Value"], [format_output_month(current_month), 3.5]],\n'
    '        [\n'
    '            ["Date", "Value"],\n'
    '            [(month_end(current_month) - date(1899, 12, 30)).days, 3.5],\n'
    '        ],\n'
    '    )\n'
    '    current_table = [\n'
    '        ["Date", "Value", "Optional"],\n'
    '        [format_output_month(current_month), 3.5, ""],\n'
    '    ]\n'
    '    current_table_google = [\n'
    '        ["Date", "Value", "Optional"],\n'
    '        [(month_end(current_month) - date(1899, 12, 30)).days, "3.5"],\n'
    '    ]\n'
    '    if not google_sheet_table_is_current(current_table, current_table_google):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: unchanged Google table was not recognised."\n'
    '        )\n'
    '    if google_sheet_table_is_current(\n'
    '        current_table,\n'
    '        [["Date", "Value", "Optional"], [current_month.isoformat(), 3.6]],\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: revised Google table was treated as current."\n'
    '        )\n'
    '    matching_history = [\n'
    '        ["Date", "Value"],\n'
    '        [format_output_month(add_months(current_month, -1)), 1.0],\n'
    '    ]\n'
    '    if not indicator_input_table_is_current(\n'
    '        matching_history,\n'
    '        [["Date", "Value"], [format_output_month(add_months(current_month, -1)), "1"]],\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: current indicator table was not recognised."\n'
    '        )\n'
    '    if indicator_input_table_is_current(\n'
    '        matching_history,\n'
    '        [["Date", "Value"], [format_output_month(add_months(current_month, -1)), 1.25]],\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: revised indicator table was treated as current."\n'
    '        )\n'
    '    validate_historical_table_replacement(\n'
    '        matching_history,\n'
    '        [*matching_history, [format_output_month(current_month), 2.0]],\n'
    '        "Internal replacement test",\n'
    '    )\n'
    '    validate_historical_table_replacement(\n'
    '        [["Date", "Value"], [format_output_month(current_month), 1.0]],\n'
    '        [["Date", "Value"], [format_output_month(current_month), 2.0]],\n'
    '        "Internal live-month replacement test",\n'
    '        mutable_months={current_month},\n'
    '    )\n'
    '    validate_historical_table_replacement(\n'
    '        matching_history,\n'
    '        [["Date", "Value"], [format_output_month(add_months(current_month, -1)), 1.25]],\n'
    '        "Internal authoritative revision test",\n'
    '        allow_value_revisions=True,\n'
    '    )\n'
    '    authoritative_history = [\n'
    '        ["Date", "Value"],\n'
    '        [format_output_month(add_months(current_month, -2)), 1.0],\n'
    '        [format_output_month(add_months(current_month, -1)), 2.0],\n'
    '        ["", 999.0],\n'
    '    ]\n'
    '    validate_historical_table_replacement(\n'
    '        authoritative_history,\n'
    '        [\n'
    '            ["Date", "Value"],\n'
    '            [format_output_month(add_months(current_month, -1)), 2.0],\n'
    '        ],\n'
    '        "Internal authoritative cleanup test",\n'
    '        allow_absent_existing_dates=True,\n'
    '        allow_undated_existing_rows=True,\n'
    '    )\n'
    '    validate_historical_table_replacement(\n'
    '        matching_history,\n'
    '        [["Date", "Value"], [format_output_month(add_months(current_month, -1)), ""]],\n'
    '        "Internal authoritative blank test",\n'
    '        allow_blank_replacements=True,\n'
    '    )\n'
    '    expect_failure(\n'
    '        "historical table replacement mismatch",\n'
    '        lambda: validate_historical_table_replacement(\n'
    '            matching_history,\n'
    '            [\n'
    '                ["Date", "Value"],\n'
    '                [format_output_month(add_months(current_month, -1)), 99.0],\n'
    '                [format_output_month(current_month), 2.0],\n'
    '            ],\n'
    '            "Internal replacement test",\n'
    '        ),\n'
    '    )\n'
    '    historical_existing = read_existing_indicator_data(\n'
    '        matching_history,\n'
    '        {"date_col": 0, "value_col": 1, "header_row": 0},\n'
    '    )\n'
    '    validate_overlapping_indicator_values(\n'
    '        historical_existing,\n'
    '        {add_months(current_month, -1): 1.0, current_month: 2.0},\n'
    '        "Internal overlap test",\n'
    '    )\n'
    '    current_existing = read_existing_indicator_data(\n'
    '        [["Date", "Value"], [format_output_month(current_month), 1.0]],\n'
    '        {"date_col": 0, "value_col": 1, "header_row": 0},\n'
    '    )\n'
    '    validate_overlapping_indicator_values(\n'
    '        current_existing,\n'
    '        {current_month: 2.0},\n'
    '        "Internal live-month overlap test",\n'
    '        mutable_months={current_month},\n'
    '    )\n'
    '    validate_overlapping_indicator_values(\n'
    '        historical_existing,\n'
    '        {add_months(current_month, -1): 1.25},\n'
    '        "Internal authoritative overlap revision test",\n'
    '        allow_value_revisions=True,\n'
    '    )\n'
    '    expect_failure(\n'
    '        "historical indicator overlap mismatch",\n'
    '        lambda: validate_overlapping_indicator_values(\n'
    '            historical_existing,\n'
    '            {add_months(current_month, -1): 2.0},\n'
    '            "Internal overlap test",\n'
    '        ),\n'
    '    )\n'
    '    verify_google_sheet_values(\n'
    '        [["Date", "Value"], [current_month.isoformat(), 837340.12424853]],\n'
    '        [["Date", "Value"], [current_month.isoformat(), "837340.1242"]],\n'
    '    )\n'
    '    verify_google_sheet_values(\n'
    '        [["Source", "Status", "Revision"], ["ISM", "OK", ""]],\n'
    '        [["Source", "Status", "Revision"], ["ISM", "OK"]],\n'
    '    )\n'
    '    wide_fixture = merge_analysis_values_into_wide_table(\n'
    '        {\n'
    '            "Manufacturing PMI": [\n'
    '                ["Date", "Manufacturing PMI"],\n'
    '                ["31/01/2020", 50.0],\n'
    '                ["31/03/2020", 42.0],\n'
    '            ],\n'
    '            CENTRAL_BANK_LIQUIDITY_SHEET: [\n'
    '                ["Date", "Assets", "TGA", "RRP"],\n'
    '                ["31/01/2020", 100.0, 20.0, 3.0],\n'
    '            ],\n'
    '            "Government interest coverage ratio": [\n'
    '                ["Date", "Revenue", "Interest", "Coverage"],\n'
    '                ["31/01/2020", 10.0, 2.0, 5.0],\n'
    '            ],\n'
    '        }\n'
    '    )\n'
    '    if len(wide_fixture) != 4:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: wide table did not create contiguous months."\n'
    '        )\n'
    '    wide_headers = wide_fixture[0]\n'
    '    january_row, february_row, march_row = wide_fixture[1:]\n'
    '    if january_row[wide_headers.index("Manufacturing PMI")] != 50.0:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: wide-table ordinary indicator mapping changed."\n'
    '        )\n'
    '    if february_row[wide_headers.index("Manufacturing PMI")] != "":\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: wide-table missing months must remain blank."\n'
    '        )\n'
    '    if march_row[wide_headers.index("Manufacturing PMI")] != 42.0:\n'
    '        raise RuntimeError("Internal test failed: wide-table month ordering changed.")\n'
    '    first_liquidity_component = next(iter(CENTRAL_BANK_LIQUIDITY_COMPONENTS))\n'
    '    if (\n'
    '        january_row[\n'
    '            wide_headers.index(wide_indicator_header(first_liquidity_component))\n'
    '        ]\n'
    '        != 100.0\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: wide-table liquidity mapping changed."\n'
    '        )\n'
    '    if january_row[wide_headers.index("Government interest coverage ratio")] != 5.0:\n'
    '        raise RuntimeError("Internal test failed: wide-table coverage mapping changed.")\n'
    '    reserves_column = wide_headers.index("Banking system reserves")\n'
    '    old_wide_row = list(january_row)\n'
    '    old_wide_row[reserves_column] = 10452.0\n'
    '    preserved_wide_row = preserve_inaccessible_wide_history(\n'
    '        wide_fixture,\n'
    '        [wide_headers, old_wide_row],\n'
    '        None,\n'
    '    )[1]\n'
    '    if preserved_wide_row[reserves_column] != 10452.0:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: inaccessible standalone history was deleted."\n'
    '        )\n'
    '    preserved_history_wide_fixture = merge_analysis_values_into_wide_table(\n'
    '        {\n'
    '            "Manufacturing PMI": [\n'
    '                ["Date", "Manufacturing PMI"],\n'
    '                ["31/01/2020", 50.0],\n'
    '                ["29/02/2020", 51.0],\n'
    '            ],\n'
    '            "10Y real yield": [\n'
    '                ["Date", "10Y real yield"],\n'
    '                ["31/01/2020", 9.9],\n'
    '                ["29/02/2020", 2.0],\n'
    '            ],\n'
    '        },\n'
    '        [\n'
    '            (\n'
    '                "10Y real yield",\n'
    '                [\n'
    '                    {\n'
    '                        "observation_date": date(2020, 2, 1),\n'
    '                        "release_date": date(2020, 2, 29),\n'
    '                        "selection_date": date(2020, 2, 1),\n'
    '                        "period_aligned": True,\n'
    '                        "source": "FRED / ALFRED",\n'
    '                        "native_frequency": "Daily",\n'
    '                        "history_complete": True,\n'
    '                        "value": 2.0,\n'
    '                    }\n'
    '                ],\n'
    '            ),\n'
    '            (\n'
    '                "Manufacturing PMI",\n'
    '                [\n'
    '                    {\n'
    '                        "observation_date": date(2020, 2, 1),\n'
    '                        "release_date": date(2020, 3, 1),\n'
    '                        "selection_date": date(2020, 3, 1),\n'
    '                        "period_aligned": True,\n'
    '                        "source": "Institute for Supply Management",\n'
    '                        "value": 52.0,\n'
    '                    }\n'
    '                ],\n'
    '            ),\n'
    '        ],\n'
    '    )\n'
    '    preserved_headers = preserved_history_wide_fixture[0]\n'
    '    real_yield_column = preserved_headers.index("10Y real yield")\n'
    '    if preserved_history_wide_fixture[1][real_yield_column] != 9.9:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: inaccessible historical data was deleted."\n'
    '        )\n'
    '    if preserved_history_wide_fixture[2][real_yield_column] != 2.0:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: fresh source value was not merged into preserved history."\n'
    '        )\n'
    '    pmi_column = preserved_headers.index("Manufacturing PMI")\n'
    '    if [row[pmi_column] for row in preserved_history_wide_fixture[1:]] != [\n'
    '        50.0,\n'
    '        51.0,\n'
    '        52.0,\n'
    '    ]:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: shorter PMI history did not preserve legacy values."\n'
    '        )\n'
    '    if has_complete_source_history([{"source": "FRED / ALFRED"}]):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: provider name alone authorised destructive cleanup."\n'
    '        )\n'
    '    if not has_complete_source_history(\n'
    '        [{"source": "FRED / ALFRED", "history_complete": True}]\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: verified full FRED history was not recognised."\n'
    '        )\n'
    '    if has_complete_source_history(\n'
    '        [\n'
    '            {\n'
    '                "source": "Institute for Supply Management",\n'
    '                "history_complete": True,\n'
    '            }\n'
    '        ]\n'
    '    ):\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: limited PMI history was authorised for deletion."\n'
    '        )\n'
    '    expect_failure(\n'
    '        "missing populated trailing Google cell",\n'
    '        lambda: verify_google_sheet_values(\n'
    '            [["Source", "Status", "Revision"], ["ISM", "OK", "changed"]],\n'
    '            [["Source", "Status", "Revision"], ["ISM", "OK"]],\n'
    '        ),\n'
    '    )\n'
    '    incoming_records = [\n'
    '        {\n'
    '            "observation_date": date(2026, 3, 1),\n'
    '            "release_date": date(2026, 4, 1),\n'
    '            "value": 48.7,\n'
    '        },\n'
    '        {\n'
    '            "observation_date": date(2026, 4, 1),\n'
    '            "release_date": date(2026, 5, 1),\n'
    '            "value": 51.6,\n'
    '        },\n'
    '        {\n'
    '            "observation_date": date(2026, 5, 1),\n'
    '            "release_date": date(2026, 6, 1),\n'
    '            "value": 52.1,\n'
    '        },\n'
    '        {\n'
    '            "observation_date": date(2026, 5, 20),\n'
    '            "release_date": date(2026, 6, 2),\n'
    '            "value": 54.1,\n'
    '        },\n'
    '    ]\n'
    '    deduped_incoming = monthly_records_for_indicator(incoming_records)\n'
    '    if deduped_incoming[date(2026, 5, 1)] != 54.1:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: incoming month-end selection changed."\n'
    '        )\n'
    '\n'
    '    empty_layout = get_sheet_layout([])\n'
    '    empty_existing = read_existing_indicator_data([], empty_layout)\n'
    '    empty_fills, empty_appends = prepare_sheet_changes(empty_existing, deduped_incoming)\n'
    '    if empty_fills:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: empty sheet produced fill operations."\n'
    '        )\n'
    '    if len(empty_appends) != 3:\n'
    '        raise RuntimeError("Internal test failed: empty sheet append planning changed.")\n'
    '\n'
    '    existing_values = [\n'
    '        ["Date", "Value"],\n'
    '        ["2026-03-01", "48.7"],\n'
    '        ["2026-04", "51.6"],\n'
    '        ["May 2026", "54.1"],\n'
    '    ]\n'
    '    layout = get_sheet_layout(existing_values)\n'
    '    existing = read_existing_indicator_data(existing_values, layout)\n'
    '    up_to_date_fills, up_to_date_appends = prepare_sheet_changes(\n'
    '        existing, deduped_incoming\n'
    '    )\n'
    '    if up_to_date_fills or up_to_date_appends:\n'
    '        raise RuntimeError("Internal test failed: up-to-date sheet sync changed.")\n'
    '    malformed_tail_values = [\n'
    '        ["Date", "Value"],\n'
    '        ["2026-04-01", "51.6"],\n'
    '        ["not a date", "ignore me"],\n'
    '        ["", ""],\n'
    '    ]\n'
    '    malformed_existing = read_existing_indicator_data(\n'
    '        malformed_tail_values,\n'
    '        get_sheet_layout(malformed_tail_values),\n'
    '    )\n'
    '    malformed_fills, malformed_appends = prepare_sheet_changes(\n'
    '        malformed_existing,\n'
    '        {date(2026, 5, 1): 52.1},\n'
    '    )\n'
    '    if malformed_fills or malformed_appends != [(date(2026, 5, 1), 52.1)]:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: malformed tail date handling changed."\n'
    '        )\n'
    '\n'
    '    one_new_incoming = {\n'
    '        **deduped_incoming,\n'
    '        date(2026, 6, 1): 55.0,\n'
    '    }\n'
    '    one_new_fills, one_new_appends = prepare_sheet_changes(existing, one_new_incoming)\n'
    '    if one_new_fills or one_new_appends != [(date(2026, 6, 1), 55.0)]:\n'
    '        raise RuntimeError("Internal test failed: single append planning changed.")\n'
    '\n'
    '    rerun_existing_values = [\n'
    '        *existing_values,\n'
    '        ["2026-06-01", "55.0"],\n'
    '    ]\n'
    '    rerun_existing = read_existing_indicator_data(rerun_existing_values, layout)\n'
    '    rerun_fills, rerun_appends = prepare_sheet_changes(rerun_existing, one_new_incoming)\n'
    '    if rerun_fills or rerun_appends:\n'
    '        raise RuntimeError("Internal test failed: rerun duplicate prevention changed.")\n'
    '\n'
    '    revised_incoming = {\n'
    '        **deduped_incoming,\n'
    '        date(2026, 4, 1): 52.0,\n'
    '    }\n'
    '    revised_fills, revised_appends = prepare_sheet_changes(existing, revised_incoming)\n'
    '    if revised_fills != [(3, date(2026, 4, 1), 52.0)] or revised_appends:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: changed official values were not updated."\n'
    '        )\n'
    '\n'
    '    blank_existing_values = [\n'
    '        ["Date", "Value"],\n'
    '        ["2026-03-01", ""],\n'
    '    ]\n'
    '    blank_existing = read_existing_indicator_data(blank_existing_values, layout)\n'
    '    blank_fills, blank_appends = prepare_sheet_changes(\n'
    '        blank_existing,\n'
    '        {date(2026, 3, 1): 48.7},\n'
    '    )\n'
    '    if blank_fills != [(2, date(2026, 3, 1), 48.7)] or blank_appends:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: blank existing value was not filled safely."\n'
    '        )\n'
    '    old_history_fills, old_history_only = prepare_sheet_changes(\n'
    '        existing,\n'
    '        {date(2026, 2, 1): 47.0},\n'
    '    )\n'
    '    if old_history_fills or old_history_only:\n'
    '        raise RuntimeError("Internal test failed: old missing history was backfilled.")\n'
    '\n'
    '    expect_failure(\n'
    '        "malformed indicator sheet date column",\n'
    '        lambda: get_sheet_layout([["Notes", "Value"], ["bad", "1"]]),\n'
    '    )\n'
    '    if indicator_sheet_name("Manufacturing PMI", {"manufacturing pmi"}) is not None:\n'
    '        raise RuntimeError("Internal test failed: inexact sheet matching was allowed.")\n'
    '    if (\n'
    '        indicator_sheet_name("Manufacturing PMI", {"Manufacturing PMI"})\n'
    '        != "Manufacturing PMI"\n'
    '    ):\n'
    '        raise RuntimeError("Internal test failed: exact sheet matching changed.")\n'
    '    if indicator_sheet_name("Missing Indicator", {"Manufacturing PMI"}) is not None:\n'
    '        raise RuntimeError("Internal test failed: missing sheet detection changed.")\n'
    '    month_end_header_layout = get_sheet_layout(\n'
    '        [["Month-End Date", "Central Bank Total Assets"]],\n'
    '        CENTRAL_BANK_LIQUIDITY_SHEET,\n'
    '    )\n'
    '    if (\n'
    '        month_end_header_layout["date_col"] != 0\n'
    '        or month_end_header_layout["value_col"] != 1\n'
    '    ):\n'
    '        raise RuntimeError("Internal test failed: month-end date sheet layout changed.")\n'
    '    original_output_months = CURRENT_OUTPUT_MONTHS\n'
    '    try:\n'
    '        globals()["CURRENT_OUTPUT_MONTHS"] = 1\n'
    '        central_fills, central_rows = central_bank_append_rows(\n'
    '            [\n'
    '                [\n'
    '                    "Date",\n'
    '                    "Central Bank Total Assets",\n'
    '                    "Cental Bank General Account",\n'
    '                    "Central Bank RRP",\n'
    '                ]\n'
    '            ],\n'
    '            {\n'
    '                "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level": [\n'
    '                    {\n'
    '                        "observation_date": month_start(date.today()),\n'
    '                        "release_date": month_start(date.today()),\n'
    '                        "selection_date": month_start(date.today()),\n'
    '                        "period_aligned": True,\n'
    '                        "value": 1.0,\n'
    '                    }\n'
    '                ],\n'
    '                "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level": [\n'
    '                    {\n'
    '                        "observation_date": month_start(date.today()),\n'
    '                        "release_date": month_start(date.today()),\n'
    '                        "selection_date": month_start(date.today()),\n'
    '                        "period_aligned": True,\n'
    '                        "value": 2.0,\n'
    '                    }\n'
    '                ],\n'
    '                "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations": [\n'
    '                    {\n'
    '                        "observation_date": month_start(date.today()),\n'
    '                        "release_date": month_start(date.today()),\n'
    '                        "selection_date": month_start(date.today()),\n'
    '                        "period_aligned": True,\n'
    '                        "value": 3.0,\n'
    '                    }\n'
    '                ],\n'
    '            },\n'
    '        )[1:3]\n'
    '        if central_fills:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: empty central-bank tab produced fill operations."\n'
    '            )\n'
    '        if central_rows != [\n'
    '            [format_output_month(month_start(date.today())), 1.0, 2.0, 3.0]\n'
    '        ]:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: central-bank shared tab row mapping changed."\n'
    '            )\n'
    '        central_blank_fills, central_blank_rows = central_bank_append_rows(\n'
    '            [\n'
    '                [\n'
    '                    "Date",\n'
    '                    "Central Bank Total Assets",\n'
    '                    "Cental Bank General Account",\n'
    '                    "Central Bank RRP",\n'
    '                ],\n'
    '                [format_output_month(month_start(date.today())), "", "", ""],\n'
    '            ],\n'
    '            {\n'
    '                "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level": [\n'
    '                    {\n'
    '                        "observation_date": month_start(date.today()),\n'
    '                        "release_date": month_start(date.today()),\n'
    '                        "selection_date": month_start(date.today()),\n'
    '                        "period_aligned": True,\n'
    '                        "value": 1.0,\n'
    '                    }\n'
    '                ],\n'
    '                "Liabilities and Capital: Liabilities: Deposits with F.R. Banks, Other Than Reserve Balances: U.S. Treasury, General Account: Wednesday Level": [\n'
    '                    {\n'
    '                        "observation_date": month_start(date.today()),\n'
    '                        "release_date": month_start(date.today()),\n'
    '                        "selection_date": month_start(date.today()),\n'
    '                        "period_aligned": True,\n'
    '                        "value": 2.0,\n'
    '                    }\n'
    '                ],\n'
    '                "Overnight Reverse Repurchase Agreements: Treasury Securities Sold by the Federal Reserve in the Temporary Open Market Operations": [\n'
    '                    {\n'
    '                        "observation_date": month_start(date.today()),\n'
    '                        "release_date": month_start(date.today()),\n'
    '                        "selection_date": month_start(date.today()),\n'
    '                        "period_aligned": True,\n'
    '                        "value": 3.0,\n'
    '                    }\n'
    '                ],\n'
    '            },\n'
    '        )[1:3]\n'
    '        if (\n'
    '            central_blank_fills\n'
    '            != [\n'
    '                (2, 1, month_start(date.today()), 1.0),\n'
    '                (2, 2, month_start(date.today()), 2.0),\n'
    '                (2, 3, month_start(date.today()), 3.0),\n'
    '            ]\n'
    '            or central_blank_rows\n'
    '        ):\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: central-bank blank columns were not filled safely."\n'
    '            )\n'
    '    finally:\n'
    '        globals()["CURRENT_OUTPUT_MONTHS"] = original_output_months\n'
    '\n'
    '    summary_month = add_months(current_month, -1)\n'
    '    nyfed_summary = f"""\n'
    '    <h1>{summary_month:%B} Survey: Inflation Expectations Down</h1>\n'
    '    <p>Median inflation expectations decreased to 3.5 percent at the\n'
    '    one-year-ahead horizon.</p>\n'
    '    """\n'
    '    parsed_summary = parse_nyfed_page_latest(nyfed_summary)\n'
    '    if parsed_summary != (summary_month, 3.5):\n'
    '        raise RuntimeError("Internal test failed: NY Fed summary parser changed.")\n'
    '\n'
    '    synthetic_sce_records = []\n'
    '    for source_month, value in [\n'
    '        (date(2026, 3, 1), 3.4),\n'
    '        (date(2026, 4, 1), 3.6),\n'
    '        (date(2026, 5, 1), 3.5),\n'
    '    ]:\n'
    '        synthetic_sce_records.append(\n'
    '            {\n'
    '                "observation_date": source_month,\n'
    '                "release_date": add_months(source_month, 1).replace(day=20),\n'
    '                "source": "Federal Reserve Bank of New York SCE",\n'
    '                "source_series_id": "Median one-year ahead expected inflation rate",\n'
    '                "selection_date": add_months(source_month, 1),\n'
    '                "period_aligned": True,\n'
    '                "value": value,\n'
    '            }\n'
    '        )\n'
    '    expected_sce_values = {\n'
    '        date(2026, 4, 30): 3.4,\n'
    '        date(2026, 5, 31): 3.6,\n'
    '        date(2026, 6, 30): 3.5,\n'
    '    }\n'
    '    for row_date, expected_value in expected_sce_values.items():\n'
    '        selected = latest_available_record(synthetic_sce_records, row_date)\n'
    '        if not selected or selected["value"] != expected_value:\n'
    '            raise RuntimeError(\n'
    '                "Internal test failed: NY Fed SCE release-month dating changed."\n'
    '            )\n'
    '    if latest_available_record(synthetic_sce_records, date(2026, 7, 31)) is not None:\n'
    '        raise RuntimeError(\n'
    '            "Internal test failed: NY Fed SCE value was carried forward."\n'
    '        )\n'
    '\n'
    '    original_archive_root = SOURCE_ARCHIVE_ROOT\n'
    '    original_statuses = dict(SOURCE_STATUSES)\n'
    '    try:\n'
    '        with tempfile.TemporaryDirectory() as temporary_directory:\n'
    '            SOURCE_ARCHIVE_ROOT = Path(temporary_directory)\n'
    '            response = requests.Response()\n'
    '            response.status_code = 200\n'
    '            response._content = b"<html>validated official data</html>"\n'
    '            response.url = "https://www.ismworld.org/test"\n'
    '            response.headers["Content-Type"] = "text/html"\n'
    '            response.request = requests.Request(\n'
    '                "GET",\n'
    '                "https://www.ismworld.org/test",\n'
    '            ).prepare()\n'
    '            status = archive_validated_source(\n'
    '                "Internal Archive",\n'
    '                current_month.isoformat(),\n'
    '                response,\n'
    '                "html",\n'
    '                ("value",),\n'
    '                {"value": 54.0},\n'
    '            )\n'
    '            archived = load_archived_source(\n'
    '                "Internal Archive",\n'
    '                current_month.isoformat(),\n'
    '            )\n'
    '            if (\n'
    '                archived is None\n'
    '                or status.sha256 != hashlib.sha256(response.content).hexdigest()\n'
    '            ):\n'
    '                raise RuntimeError(\n'
    '                    "Internal test failed: source archive was not readable."\n'
    '                )\n'
    '            raw_content, metadata = archived\n'
    '            if (\n'
    '                raw_content != response.content\n'
    '                or metadata["parsed_values"]["value"] != 54.0\n'
    '            ):\n'
    '                raise RuntimeError(\n'
    '                    "Internal test failed: source archive metadata changed."\n'
    '                )\n'
    '            archive_validated_source(\n'
    '                "Internal Archive",\n'
    '                add_months(current_month, -1).isoformat(),\n'
    '                response,\n'
    '                "html",\n'
    '                ("value",),\n'
    '                {"value": 53.0},\n'
    '            )\n'
    '            latest_archive = load_latest_archived_source("Internal Archive")\n'
    '            if (\n'
    '                latest_archive is None\n'
    '                or latest_archive[1]["period"] != current_month.isoformat()\n'
    '            ):\n'
    '                raise RuntimeError(\n'
    '                    "Internal test failed: source archive latest pointer moved backward."\n'
    '                )\n'
    '    finally:\n'
    '        SOURCE_ARCHIVE_ROOT = original_archive_root\n'
    '        SOURCE_STATUSES.clear()\n'
    '        SOURCE_STATUSES.update(original_statuses)\n'
    '\n'
    '\n'
    'def parse_args() -> argparse.Namespace:\n'
    '    parser = argparse.ArgumentParser(\n'
    '        description="Sync monthly indicator observations into dedicated Google Sheets tabs."\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--series-id",\n'
    '        default=os.getenv("FRED_SERIES_ID"),\n'
    '        help="Optional single FRED series ID, for example UNRATE or CPIAUCSL. If omitted, all built-in indicators are synced.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--indicator-name",\n'
    '        action="append",\n'
    '        default=[],\n'
    '        help=(\n'
    '            "Sync only the named built-in indicator. Repeat for multiple indicators. "\n'
    '            "Names must exactly match --list-indicators output."\n'
    '        ),\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--api-key",\n'
    '        default=os.getenv("FRED_API_KEY"),\n'
    '        help="FRED API key. Required in the run command when exporting FRED data.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--months",\n'
    '        type=int,\n'
    '        default=DEFAULT_OUTPUT_MONTHS,\n'
    '        help=f"Number of monthly rows to include. Default: {DEFAULT_OUTPUT_MONTHS}.",\n'
    '    )\n'
    '    history_mode = parser.add_mutually_exclusive_group()\n'
    '    history_mode.add_argument(\n'
    '        "--recent-only",\n'
    '        action="store_true",\n'
    '        help="Compatibility flag; incremental sync is already the default.",\n'
    '    )\n'
    '    history_mode.add_argument(\n'
    '        "--full-history",\n'
    '        action="store_true",\n'
    '        help="Explicitly reconcile full source history and permit value revisions.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--spreadsheet-id",\n'
    '        default=os.getenv("GOOGLE_SPREADSHEET_ID", DEFAULT_GOOGLE_SPREADSHEET_ID),\n'
    '        help="Existing analysis workbook ID whose individual indicator tabs are updated.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--wide-spreadsheet-id",\n'
    '        default=os.getenv("GOOGLE_WIDE_SPREADSHEET_ID", DEFAULT_WIDE_SPREADSHEET_ID),\n'
    '        help="Standalone workbook ID for the consolidated date-by-indicator table.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--wide-sheet-name",\n'
    '        default=os.getenv("GOOGLE_WIDE_SHEET_NAME", DEFAULT_WIDE_SHEET_NAME),\n'
    '        help="Tab name for the standalone consolidated table.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--skip-wide-sync",\n'
    '        action="store_true",\n'
    '        help="Update only the existing individual analysis tabs.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--skip-local-outputs",\n'
    '        action="store_true",\n'
    '        help="Do not replace the complete local CSV artifacts during a scoped recovery sync.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--wide-only-from-analysis",\n'
    '        action="store_true",\n'
    '        help="Rebuild only the standalone table from existing analysis-tab inputs; no source API key is required.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--credentials",\n'
    '        default=os.getenv("GOOGLE_APPLICATION_CREDENTIALS", DEFAULT_GOOGLE_CREDENTIALS),\n'
    '        help="Path to the Google service account JSON file. The project default is already set.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--sheet-name",\n'
    '        default=os.getenv("GOOGLE_SHEET_NAME", DEFAULT_GOOGLE_SHEET_NAME),\n'
    '        help="Legacy combined-table sheet name; dedicated indicator-tab sync does not use this in the normal export.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--list-indicators",\n'
    '        action="store_true",\n'
    '        help="Print the built-in indicators and exit.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--exclude-ism",\n'
    '        action="store_true",\n'
    '        help="Do not include the official ISM PMI indicators in the built-in export.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--ism-only",\n'
    '        action="store_true",\n'
    '        help="Export only the official ISM PMI indicators. This does not require a FRED API key.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--check-non-api",\n'
    '        action="store_true",\n'
    '        help="Check official ISM, NY Fed, and Michigan non-API sources, then exit.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--audit-google-sheet",\n'
    '        action="store_true",\n'
    '        help="Read the target Google workbook and print a non-mutating tab audit, then exit.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--cleanup-sparse-dates",\n'
    '        action="store_true",\n'
    '        help="Delete old off-quarter/off-year-end rows for quarterly/annual indicators after syncing correct rows.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--cleanup-sparse-dates-only",\n'
    '        action="store_true",\n'
    '        help="Only delete old off-quarter/off-year-end sparse rows; do not run the full sheet sync.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--self-test",\n'
    '        action="store_true",\n'
    '        help="Run the built-in offline safety tests, then exit.",\n'
    '    )\n'
    '    return parser.parse_args()\n'
    '\n'
    '\n'
    'def run(args: argparse.Namespace) -> int:\n'
    '    global CURRENT_OUTPUT_MONTHS, FULL_HISTORY_MODE\n'
    '    configured_history_mode = FULL_HISTORY_MODE\n'
    '    FULL_HISTORY_MODE = True\n'
    '    try:\n'
    '        run_internal_self_tests()\n'
    '    finally:\n'
    '        FULL_HISTORY_MODE = configured_history_mode\n'
    '    if args.self_test:\n'
    '        print("Internal self-test passed.")\n'
    '        return 0\n'
    '    SOURCE_STATUSES.clear()\n'
    '    VALIDATION_MESSAGES.clear()\n'
    '    _EXPLICIT_BLANK_OUTPUT_MONTHS.clear()\n'
    '    CURRENT_OUTPUT_MONTHS = args.months\n'
    '    FULL_HISTORY_MODE = bool(args.full_history)\n'
    '\n'
    '    if args.list_indicators:\n'
    '        for indicator in INDICATORS:\n'
    '            if indicator.source != "fred":\n'
    '                source = f"{indicator.source}:{indicator.source_column}"\n'
    '            else:\n'
    '                source = (\n'
    '                    indicator.series_id\n'
    '                    or f"{indicator.numerator_series_id}/{indicator.denominator_series_id}"\n'
    '                )\n'
    '            print(f"{indicator.name}: {source}")\n'
    '        return 0\n'
    '\n'
    '    if args.wide_only_from_analysis:\n'
    '        summary = sync_wide_indicator_workbook(\n'
    '            args.spreadsheet_id,\n'
    '            args.wide_spreadsheet_id,\n'
    '            Path(args.credentials),\n'
    '            args.wide_sheet_name,\n'
    '            validate_only=False,\n'
    '        )\n'
    '        action = "Would update" if False else "Updated"\n'
    '        print(\n'
    '            f"{action} standalone table: "\n'
    '            f"https://docs.google.com/spreadsheets/d/{args.wide_spreadsheet_id} "\n'
    '            f"tab={args.wide_sheet_name!r} rows={summary[\'rows\']} "\n'
    '            f"columns={summary[\'columns\']} dates={summary[\'first_date\']}..{summary[\'last_date\']}"\n'
    '        )\n'
    '        return 0\n'
    '\n'
    '    if args.check_non_api:\n'
    '        check_non_api_sources(args.months)\n'
    '        return 0\n'
    '\n'
    '    if args.cleanup_sparse_dates_only:\n'
    '        if not args.api_key:\n'
    '            print(\n'
    '                "Please provide a FRED API key in the run command with --api-key YOUR_KEY.",\n'
    '                file=sys.stderr,\n'
    '            )\n'
    '            return 2\n'
    '        sparse_workbook_data = []\n'
    '        for indicator in INDICATORS:\n'
    '            if not has_dedicated_indicator_tab(indicator.name):\n'
    '                continue\n'
    '            if indicator.source == "fred":\n'
    '                metadata = get_fred_series_metadata(\n'
    '                    args.api_key, indicator.series_id or ""\n'
    '                )\n'
    '                if not str(metadata.get("frequency", "")).startswith(\n'
    '                    SPARSE_FRED_FREQUENCY_PREFIXES\n'
    '                ):\n'
    '                    continue\n'
    '            elif indicator.source != "imf_datamapper":\n'
    '                continue\n'
    '            records = get_indicator_observations_safely(\n'
    '                args.api_key, indicator, args.months\n'
    '            )\n'
    '            if sparse_date_rule_for_records(records) is not None:\n'
    '                sparse_workbook_data.append((indicator.name, records))\n'
    '        cleanup_sparse_date_rows(\n'
    '            args.spreadsheet_id,\n'
    '            Path(args.credentials),\n'
    '            sparse_workbook_data,\n'
    '            False,\n'
    '        )\n'
    '        action = "Checked" if False else "Updated"\n'
    '        print(\n'
    '            f"{action} Google Sheet: https://docs.google.com/spreadsheets/d/{args.spreadsheet_id}"\n'
    '        )\n'
    '        return 0\n'
    '\n'
    '    if args.audit_google_sheet:\n'
    '        source_data = None\n'
    '        source_errors = None\n'
    '        if args.api_key:\n'
    '            print("Collecting current source output for Google Sheet audit...")\n'
    '            if any(indicator.source != "fred" for indicator in INDICATORS):\n'
    '                check_non_api_sources(args.months)\n'
    '            source_data, source_errors = collect_source_data_for_audit(\n'
    '                args.api_key,\n'
    '                args.months,\n'
    '            )\n'
    '        else:\n'
    '            print(\n'
    '                "Running sheet-only audit. Add --api-key to compare against live source output."\n'
    '            )\n'
    '        audit_google_workbook(\n'
    '            args.spreadsheet_id,\n'
    '            Path(args.credentials),\n'
    '            source_data,\n'
    '            source_errors,\n'
    '        )\n'
    '        return 0\n'
    '\n'
    '    selected_indicators = INDICATORS\n'
    '    if args.indicator_name:\n'
    '        requested_names = set(args.indicator_name)\n'
    '        known_names = {indicator.name for indicator in INDICATORS}\n'
    '        unknown_names = sorted(requested_names - known_names)\n'
    '        if unknown_names:\n'
    '            print(\n'
    '                "Unknown --indicator-name value(s): " + ", ".join(unknown_names),\n'
    '                file=sys.stderr,\n'
    '            )\n'
    '            return 2\n'
    '        selected_indicators = [\n'
    '            indicator for indicator in INDICATORS if indicator.name in requested_names\n'
    '        ]\n'
    '    elif args.ism_only:\n'
    '        selected_indicators = [\n'
    '            indicator for indicator in INDICATORS if indicator.source == "ism"\n'
    '        ]\n'
    '    elif args.exclude_ism:\n'
    '        selected_indicators = [\n'
    '            indicator for indicator in INDICATORS if indicator.source != "ism"\n'
    '        ]\n'
    '\n'
    '    needs_fred = any(indicator.source == "fred" for indicator in selected_indicators)\n'
    '    if needs_fred and not args.api_key:\n'
    '        print(\n'
    '            "Please provide a FRED API key in the run command with --api-key YOUR_KEY.",\n'
    '            file=sys.stderr,\n'
    '        )\n'
    '        return 2\n'
    '    if args.months < 1:\n'
    '        print("--months must be at least 1.", file=sys.stderr)\n'
    '        return 2\n'
    '\n'
    '    if args.series_id:\n'
    '        title = get_series_title(args.api_key, args.series_id)\n'
    '        observations = get_monthly_observations(\n'
    '            args.api_key, args.series_id, args.months\n'
    '        )\n'
    '        workbook_data = [(title, observations)]\n'
    '        single_indicator = Indicator(title, args.series_id)\n'
    '        sheet_values_for_google(workbook_data)\n'
    '        metadata_values = indicator_metadata_values(args.api_key, [single_indicator])\n'
    '        validation_values = build_validation_report(\n'
    '            [single_indicator],\n'
    '            workbook_data,\n'
    '            args.months,\n'
    '        )\n'
    '        diagnostic_values = selection_diagnostic_values(\n'
    '            [single_indicator],\n'
    '            workbook_data,\n'
    '            args.months,\n'
    '        )\n'
    '        if not args.skip_local_outputs:\n'
    '            save_clean_csv_outputs(\n'
    '                workbook_data,\n'
    '                metadata_values,\n'
    '                validation_values,\n'
    '                diagnostic_values,\n'
    '            )\n'
    '        if not False:\n'
    '            assert_sync_preflight_passed(\n'
    '                sync_all_indicators(\n'
    '                    args.spreadsheet_id,\n'
    '                    Path(args.credentials),\n'
    '                    workbook_data,\n'
    '                    validate_only=True,\n'
    '                    _preflight=True,\n'
    '                )\n'
    '            )\n'
    '        if not args.skip_wide_sync:\n'
    '            sync_wide_indicator_workbook(\n'
    '                args.spreadsheet_id,\n'
    '                args.wide_spreadsheet_id,\n'
    '                Path(args.credentials),\n'
    '                args.wide_sheet_name,\n'
    '                workbook_data,\n'
    '                False,\n'
    '            )\n'
    '        sync_results = sync_all_indicators(\n'
    '            args.spreadsheet_id,\n'
    '            Path(args.credentials),\n'
    '            workbook_data,\n'
    '            False,\n'
    '        )\n'
    '        assert_sync_preflight_passed(sync_results)\n'
    '        action = "Checked" if False else "Updated"\n'
    '        print(\n'
    '            f"{action} Google Sheet: https://docs.google.com/spreadsheets/d/{args.spreadsheet_id}"\n'
    '        )\n'
    '        return 0\n'
    '\n'
    '    if any(indicator.source != "fred" for indicator in selected_indicators):\n'
    '        print("Checking official non-FRED sources before export...")\n'
    '        check_non_api_sources(args.months)\n'
    '        print("All non-FRED source checks passed.")\n'
    '\n'
    '    workbook_data = []\n'
    '    printed_imf_audit = False\n'
    '    for indicator in selected_indicators:\n'
    '        observations = get_indicator_observations_safely(\n'
    '            args.api_key, indicator, args.months\n'
    '        )\n'
    '        workbook_data.append((indicator.name, observations))\n'
    '        if COMPACT_TERMINAL:\n'
    '            print(f"Fetched {len(observations)} observations for {indicator.name}")\n'
    '        if indicator.source == "imf_datamapper" and not printed_imf_audit:\n'
    '            print_imf_public_finance_audit(\n'
    '                indicator.source_column or DEFAULT_IMF_COUNTRY\n'
    '            )\n'
    '            printed_imf_audit = True\n'
    '\n'
    '    validate_non_fred_output_alignment(\n'
    '        selected_indicators,\n'
    '        workbook_data,\n'
    '        args.months,\n'
    '    )\n'
    '    sheet_values_for_google(workbook_data)\n'
    '    metadata_values = indicator_metadata_values(args.api_key, selected_indicators)\n'
    '    validation_values = build_validation_report(\n'
    '        selected_indicators,\n'
    '        workbook_data,\n'
    '        args.months,\n'
    '    )\n'
    '    diagnostic_values = selection_diagnostic_values(\n'
    '        selected_indicators,\n'
    '        workbook_data,\n'
    '        args.months,\n'
    '    )\n'
    '    if any(row[0] == "ERROR" for row in validation_values[1:]):\n'
    '        errors = [row[2] for row in validation_values[1:] if row[0] == "ERROR"]\n'
    '        raise RuntimeError("Validation report contains errors: " + " | ".join(errors))\n'
    '    print_changed_cells(workbook_data)\n'
    '    if not args.skip_local_outputs:\n'
    '        save_clean_csv_outputs(\n'
    '            workbook_data,\n'
    '            metadata_values,\n'
    '            validation_values,\n'
    '            diagnostic_values,\n'
    '        )\n'
    '    if not False:\n'
    '        assert_sync_preflight_passed(\n'
    '            sync_all_indicators(\n'
    '                args.spreadsheet_id,\n'
    '                Path(args.credentials),\n'
    '                workbook_data,\n'
    '                validate_only=True,\n'
    '                _preflight=True,\n'
    '            )\n'
    '        )\n'
    '    if not args.skip_wide_sync:\n'
    '        summary = sync_wide_indicator_workbook(\n'
    '            args.spreadsheet_id,\n'
    '            args.wide_spreadsheet_id,\n'
    '            Path(args.credentials),\n'
    '            args.wide_sheet_name,\n'
    '            workbook_data,\n'
    '            False,\n'
    '        )\n'
    '        action = "Would update" if False else "Updated"\n'
    '        print(\n'
    '            f"{action} standalone table: "\n'
    '            f"https://docs.google.com/spreadsheets/d/{args.wide_spreadsheet_id} "\n'
    '            f"tab={args.wide_sheet_name!r} rows={summary[\'rows\']} "\n'
    '            f"columns={summary[\'columns\']}"\n'
    '        )\n'
    '    sync_results = sync_all_indicators(\n'
    '        args.spreadsheet_id,\n'
    '        Path(args.credentials),\n'
    '        workbook_data,\n'
    '        False,\n'
    '    )\n'
    '    assert_sync_preflight_passed(sync_results)\n'
    '    if args.cleanup_sparse_dates:\n'
    '        cleanup_sparse_date_rows(\n'
    '            args.spreadsheet_id,\n'
    '            Path(args.credentials),\n'
    '            workbook_data,\n'
    '            False,\n'
    '        )\n'
    '    action = "Checked" if False else "Updated"\n'
    '    print(\n'
    '        f"{action} Google Sheet: https://docs.google.com/spreadsheets/d/{args.spreadsheet_id}"\n'
    '    )\n'
    '\n'
    '    return 0\n'
    '\n'
    '\n'
    'from notifications.lifecycle import notify_entrypoint, record_failure\n'
    '\n'
    '\n'
    '@notify_entrypoint("Indicators")\n'
    'def main() -> int:\n'
    '    console_handler = logging.StreamHandler()\n'
    '    if COMPACT_TERMINAL:\n'
    '        # Detailed retries and tracebacks remain in the log file. The terminal\n'
    '        # shows the same concise progress-and-summary style as the UK pipeline.\n'
    '        console_handler.setLevel(logging.CRITICAL)\n'
    '    file_handler = logging.FileHandler(ERROR_LOG_PATH, encoding="utf-8")\n'
    '    logging.basicConfig(\n'
    '        level=logging.INFO,\n'
    '        format="%(asctime)s %(levelname)s %(message)s",\n'
    '        handlers=[\n'
    '            console_handler,\n'
    '            file_handler,\n'
    '        ],\n'
    '        force=True,\n'
    '    )\n'
    '    args = parse_args()\n'
    '    try:\n'
    '        return run(args)\n'
    '    except Exception as exc:\n'
    '        logging.exception("Monthly Indicators export failed.")\n'
    '        record_failure(f"{type(exc).__name__}: {exc}")\n'
    '        action = (\n'
    '            "audit failed"\n'
    '            if args.audit_google_sheet\n'
    '            else "Google Sheet was not updated"\n'
    '        )\n'
    '        print(\n'
    '            f"\\nERROR: The {action} because a data source or Google Sheets check "\n'
    '            f"failed.\\n{exc}\\nDetails: {ERROR_LOG_PATH}",\n'
    '            file=sys.stderr,\n'
    '        )\n'
    '        return 1\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    raise SystemExit(main())\n'
)

# Embedded source: us_indicator_analysis.py
SOURCES['us_indicator_analysis.py'] = (
    '#!/usr/bin/env python3\n'
    '"""Fast deterministic Google Sheets indicator calculation engine."""\n'
    '\n'
    'from __future__ import annotations\n'
    '\n'
    'import argparse\n'
    'import bisect\n'
    'import logging\n'
    'import os\n'
    'from datetime import date, datetime, timedelta\n'
    'import math\n'
    'from pathlib import Path\n'
    'import random\n'
    'import sys\n'
    'import time\n'
    'from dataclasses import dataclass\n'
    'from decimal import Decimal, ROUND_HALF_UP\n'
    'from typing import Any, Callable, Sequence\n'
    '\n'
    'SPREADSHEET_ID = "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c"\n'
    '\n'
    '\n'
    'def find_default_service_account_path() -> str:\n'
    '    """Find credentials locally while still allowing an explicit override."""\n'
    '    configured = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")\n'
    '    if configured:\n'
    '        return configured\n'
    '    script_path = Path(__file__).resolve()\n'
    '    for parent in (script_path.parent, *script_path.parents):\n'
    '        for filename in ("google_credentials.json", "service_account.json"):\n'
    '            candidate = parent / filename\n'
    '            if candidate.is_file():\n'
    '                return str(candidate)\n'
    '    return str(script_path.parent / "google_credentials.json")\n'
    '\n'
    '\n'
    'DEFAULT_SERVICE_ACCOUNT_PATH = find_default_service_account_path()\n'
    'CONTROL_PANEL_SHEET_NAME = "Control Panel"\n'
    'CONTROL_PANEL_FIRST_INDICATOR_CELL = "A3"\n'
    'CONTROL_PANEL_SECOND_INDICATOR_CELL = "A4"\n'
    'CONTROL_PANEL_THIRD_INDICATOR_CELL = "A5"\n'
    'CONTROL_PANEL_FOURTH_INDICATOR_CELL = "A6"\n'
    'CONTROL_PANEL_FIFTH_INDICATOR_CELL = "A7"\n'
    'CONTROL_PANEL_SIXTH_INDICATOR_CELL = "A8"\n'
    'CONTROL_PANEL_SEVENTH_INDICATOR_CELL = "A9"\n'
    'CONTROL_PANEL_EIGHTH_INDICATOR_CELL = "A10"\n'
    'CONTROL_PANEL_NINTH_INDICATOR_CELL = "A11"\n'
    'CONTROL_PANEL_TENTH_INDICATOR_CELL = "A12"\n'
    'CONTROL_PANEL_ELEVENTH_INDICATOR_CELL = "A13"\n'
    'CONTROL_PANEL_TWELFTH_INDICATOR_CELL = "A14"\n'
    'CONTROL_PANEL_THIRTEENTH_INDICATOR_CELL = "A15"\n'
    'CONTROL_PANEL_FOURTEENTH_INDICATOR_CELL = "A16"\n'
    'CONTROL_PANEL_FIFTEENTH_INDICATOR_CELL = "A17"\n'
    'CONTROL_PANEL_SIXTEENTH_INDICATOR_CELL = "A18"\n'
    'CONTROL_PANEL_SEVENTEENTH_INDICATOR_CELL = "A19"\n'
    'CONTROL_PANEL_EIGHTEENTH_INDICATOR_CELL = "A20"\n'
    'CONTROL_PANEL_NINETEENTH_INDICATOR_CELL = "A21"\n'
    'CONTROL_PANEL_TWENTIETH_INDICATOR_CELL = "A22"\n'
    'CONTROL_PANEL_TWENTY_FIRST_INDICATOR_CELL = "A23"\n'
    'CONTROL_PANEL_TWENTY_SECOND_INDICATOR_CELL = "A24"\n'
    'CONTROL_PANEL_TWENTY_THIRD_INDICATOR_CELL = "A25"\n'
    'CONTROL_PANEL_TWENTY_FOURTH_INDICATOR_CELL = "A26"\n'
    'CONTROL_PANEL_TWENTY_FIFTH_INDICATOR_CELL = "A27"\n'
    'CONTROL_PANEL_TWENTY_SIXTH_INDICATOR_CELL = "A28"\n'
    'CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL = "A29"\n'
    'CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL = "A30"\n'
    'CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL = "A31"\n'
    'CONTROL_PANEL_THIRTIETH_INDICATOR_CELL = "A32"\n'
    'CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL = "A33"\n'
    'CONTROL_PANEL_THIRTY_SECOND_INDICATOR_CELL = "A34"\n'
    'CONTROL_PANEL_THIRTY_THIRD_INDICATOR_CELL = "A35"\n'
    'CONTROL_PANEL_THIRTY_FOURTH_INDICATOR_CELL = "A36"\n'
    'CONTROL_PANEL_THIRTY_FIFTH_INDICATOR_CELL = "A37"\n'
    'CONTROL_PANEL_THIRTY_SIXTH_INDICATOR_CELL = "A38"\n'
    'CONTROL_PANEL_THIRTY_SEVENTH_INDICATOR_CELL = "A39"\n'
    'GOOGLE_SHEETS_DATE_EPOCH = datetime(1899, 12, 30)\n'
    'LOGGER = logging.getLogger("indicator_engine")\n'
    '\n'
    'OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "Distance From 50",\n'
    '    "Level Score Raw",\n'
    '    "1M Change",\n'
    '    "3M Change",\n'
    '    "6M Change",\n'
    '    "Level PercentRank",\n'
    '    "Level Score",\n'
    '    "3M Change PercentRank",\n'
    '    "3M Momentum Score",\n'
    '    "6M Change PercentRank",\n'
    '    "6M Momentum Score",\n'
    '    "Early Cycle / Recovery Score",\n'
    '    "Threshold Transition Score",\n'
    '    "Overheating / Slowdown Score",\n'
    '    "Warning Score Raw",\n'
    '    "Level Composite Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "Regime",\n'
    '    "Regime Momentum Label",\n'
    '    "Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Warning Risk Label",\n'
    '    "Manufacturing Signal Label",\n'
    ']\n'
    '\n'
    'MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS = [\n'
    '    "Sample Count",\n'
    '    "Distance from 50",\n'
    '    "Distance from 50 Score",\n'
    '    "1M Change",\n'
    '    "3M Change",\n'
    '    "6M Change",\n'
    '    "Level Percentile",\n'
    '    "Level Score",\n'
    '    "3M Momentum Percentile",\n'
    '    "3M Momentum Score",\n'
    '    "6M Momentum Percentile",\n'
    '    "6M Momentum Score",\n'
    '    "Turning Point Score",\n'
    '    "50-Cross Signal",\n'
    '    "Late-Cycle Warning",\n'
    '    "Peak/Rollover Signal",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Slowdown Risk Flag",\n'
    ']\n'
    '\n'
    'INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS = [\n'
    '    "3M MA",\n'
    '    "Sample Count",\n'
    '    "1M % Change",\n'
    '    "3M Annualised Growth",\n'
    '    "6M Annualised Growth",\n'
    '    "YoY Growth",\n'
    '    "3M Change in YoY Growth",\n'
    '    "6M Change in YoY Growth",\n'
    '    "YoY Growth Percentile",\n'
    '    "YoY Growth Score",\n'
    '    "3M Growth Percentile",\n'
    '    "3M Growth Score",\n'
    '    "6M Growth Percentile",\n'
    '    "6M Growth Score",\n'
    '    "Momentum Percentile",\n'
    '    "Momentum Score",\n'
    '    "Industrial Cycle Turning Point Score",\n'
    '    "Industrial Downside Warning Score",\n'
    '    "Industrial Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Slowdown Risk Flag",\n'
    '    "YoY Growth Z-Score",\n'
    '    "3M Growth Z-Score",\n'
    '    "6M Growth Z-Score",\n'
    '    "Momentum Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'CONSUMER_SPENDING_OUTPUT_COLUMNS = [\n'
    '    "3M Moving Average",\n'
    '    "Smoothed Observation Count",\n'
    '    "1M Smoothed Growth",\n'
    '    "3M Annualised Smoothed Growth",\n'
    '    "6M Annualised Smoothed Growth",\n'
    '    "12M Smoothed Growth",\n'
    '    "3M Change in 12M Growth",\n'
    '    "6M Change in 12M Growth",\n'
    '    "12M Growth PercentRank",\n'
    '    "12M Growth Score",\n'
    '    "3M Annualised Growth PercentRank",\n'
    '    "3M Annualised Growth Score",\n'
    '    "6M Annualised Growth PercentRank",\n'
    '    "6M Annualised Growth Score",\n'
    '    "3M Change in 12M Growth PercentRank",\n'
    '    "3M Change in 12M Growth Score",\n'
    '    "Consumer Spending Recovery / Momentum Score",\n'
    '    "Consumer Spending Warning Score Raw",\n'
    '    "Consumer Spending Outlier Score",\n'
    '    "Consumer Spending Leading Score",\n'
    '    "Consumer Spending Warning Composite Score",\n'
    '    "Consumer Spending Final Warning Score",\n'
    '    "Consumer Spending Regime",\n'
    '    "Consumer Spending Regime Momentum Label",\n'
    '    "Consumer Spending Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Consumer Spending Warning Risk Label",\n'
    '    "Consumer Spending Signal Label",\n'
    '    "12M Growth Z-Score",\n'
    '    "3M Annualised Growth Z-Score",\n'
    '    "6M Annualised Growth Z-Score",\n'
    '    "3M Change in 12M Growth Z-Score",\n'
    '    "Consumer Spending Outlier Label",\n'
    ']\n'
    '\n'
    'INITIAL_CLAIMS_OUTPUT_COLUMNS = [\n'
    '    "3M Moving Average",\n'
    '    "Smoothed Observation Count",\n'
    '    "1M Smoothed Growth",\n'
    '    "3M Annualised Smoothed Growth",\n'
    '    "6M Annualised Smoothed Growth",\n'
    '    "12M Smoothed Growth",\n'
    '    "3M Change in 12M Growth",\n'
    '    "6M Change in 12M Growth",\n'
    '    "12M Claims Growth PercentRank",\n'
    '    "12M Claims Growth Score",\n'
    '    "3M Annualised Claims Growth PercentRank",\n'
    '    "3M Annualised Claims Growth Score",\n'
    '    "6M Annualised Claims Growth PercentRank",\n'
    '    "6M Annualised Claims Growth Score",\n'
    '    "3M Change in 12M Claims Growth PercentRank",\n'
    '    "3M Change in 12M Claims Growth Score",\n'
    '    "Initial Claims Recovery / Momentum Score",\n'
    '    "Initial Claims Surge Warning Score Raw",\n'
    '    "Initial Claims Outlier Score",\n'
    '    "Initial Claims Leading Score",\n'
    '    "Initial Claims Warning Composite Score",\n'
    '    "Initial Claims Final Warning Score",\n'
    '    "Initial Claims Labour Market Regime",\n'
    '    "Initial Claims Momentum Label",\n'
    '    "Initial Claims Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Initial Claims Warning Risk Label",\n'
    '    "Initial Claims Signal Label",\n'
    '    "12M Claims Growth Inverted Z-Score",\n'
    '    "3M Annualised Claims Growth Inverted Z-Score",\n'
    '    "6M Annualised Claims Growth Inverted Z-Score",\n'
    '    "3M Change in 12M Claims Growth Inverted Z-Score",\n'
    '    "Initial Claims Outlier Label",\n'
    ']\n'
    '\n'
    'EMPLOYMENT_GROWTH_OUTPUT_COLUMNS = [\n'
    '    "3M Moving Average",\n'
    '    "Smoothed Observation Count",\n'
    '    "1M Smoothed Growth",\n'
    '    "3M Annualised Smoothed Growth",\n'
    '    "6M Annualised Smoothed Growth",\n'
    '    "12M Smoothed Growth",\n'
    '    "3M Change in 12M Growth",\n'
    '    "6M Change in 12M Growth",\n'
    '    "12M Employment Growth PercentRank",\n'
    '    "12M Employment Growth Score",\n'
    '    "3M Annualised Employment Growth PercentRank",\n'
    '    "3M Annualised Employment Growth Score",\n'
    '    "6M Annualised Employment Growth PercentRank",\n'
    '    "6M Annualised Employment Growth Score",\n'
    '    "3M Change in 12M Employment Growth PercentRank",\n'
    '    "3M Change in 12M Employment Growth Score",\n'
    '    "Employment Recovery / Momentum Score",\n'
    '    "Employment Short-Term Warning Score Raw",\n'
    '    "Employment Outlier Score",\n'
    '    "Employment Leading Score",\n'
    '    "Employment Warning Composite Score",\n'
    '    "Employment Final Warning Score",\n'
    '    "Employment Labour Market Regime",\n'
    '    "Employment Momentum Label",\n'
    '    "Employment Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Employment Warning Risk Label",\n'
    '    "Employment Signal Label",\n'
    '    "12M Employment Growth Z-Score",\n'
    '    "3M Annualised Employment Growth Z-Score",\n'
    '    "6M Annualised Employment Growth Z-Score",\n'
    '    "3M Change in 12M Employment Growth Z-Score",\n'
    '    "Employment Outlier Label",\n'
    ']\n'
    '\n'
    'AVERAGE_HOURS_OUTPUT_COLUMNS = [\n'
    '    "3M MA",\n'
    '    "Sample Count",\n'
    '    "1M % Change",\n'
    '    "3M Annualised Growth",\n'
    '    "6M Annualised Growth",\n'
    '    "YoY Growth",\n'
    '    "3M Change in YoY Growth",\n'
    '    "6M Change in YoY Growth",\n'
    '    "YoY Growth Percentile",\n'
    '    "YoY Growth Score",\n'
    '    "3M Growth Percentile",\n'
    '    "3M Growth Score",\n'
    '    "6M Growth Percentile",\n'
    '    "6M Growth Score",\n'
    '    "Momentum Percentile",\n'
    '    "Momentum Score",\n'
    '    "Hours Cycle Turning Point Score",\n'
    '    "Hours Shock Score",\n'
    '    "Hours Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Slowdown Risk Flag",\n'
    '    "YoY Growth Z-Score",\n'
    '    "3M Growth Z-Score",\n'
    '    "6M Growth Z-Score",\n'
    '    "Momentum Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'BUILDING_PERMITS_OUTPUT_COLUMNS = [\n'
    '    "3M Moving Average",\n'
    '    "Smoothed Observation Count",\n'
    '    "1M Smoothed Growth",\n'
    '    "3M Annualised Smoothed Growth",\n'
    '    "6M Annualised Smoothed Growth",\n'
    '    "12M Smoothed Growth",\n'
    '    "3M Change in 12M Growth",\n'
    '    "6M Change in 12M Growth",\n'
    '    "12M Building Permits Growth PercentRank",\n'
    '    "12M Building Permits Growth Score",\n'
    '    "3M Annualised Building Permits Growth PercentRank",\n'
    '    "3M Annualised Building Permits Growth Score",\n'
    '    "6M Annualised Building Permits Growth PercentRank",\n'
    '    "6M Annualised Building Permits Growth Score",\n'
    '    "3M Change in 12M Building Permits Growth PercentRank",\n'
    '    "3M Change in 12M Building Permits Growth Score",\n'
    '    "Smoothed Level PercentRank",\n'
    '    "Smoothed Level Score",\n'
    '    "Building Permits Recovery / Momentum Score",\n'
    '    "Building Permits Short-Term Warning Score Raw",\n'
    '    "Building Permits Outlier Score",\n'
    '    "Building Permits Leading Score",\n'
    '    "Building Permits Warning Composite Score",\n'
    '    "Building Permits Final Warning Score",\n'
    '    "Building Permits Housing Activity Regime",\n'
    '    "Building Permits Momentum Label",\n'
    '    "Building Permits Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Building Permits Warning Risk Label",\n'
    '    "Building Permits Signal Label",\n'
    '    "12M Building Permits Growth Z-Score",\n'
    '    "3M Annualised Building Permits Growth Z-Score",\n'
    '    "6M Annualised Building Permits Growth Z-Score",\n'
    '    "3M Change in 12M Building Permits Growth Z-Score",\n'
    '    "Building Permits Outlier Label",\n'
    ']\n'
    '\n'
    'CORE_INFLATION_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "1M Annualised Inflation",\n'
    '    "3M Annualised Inflation",\n'
    '    "6M Annualised Inflation",\n'
    '    "12M Inflation",\n'
    '    "3M Change in 12M Inflation",\n'
    '    "6M Change in 12M Inflation",\n'
    '    "12M Core Inflation PercentRank",\n'
    '    "12M Core Inflation Score",\n'
    '    "3M Annualised Core Inflation PercentRank",\n'
    '    "3M Annualised Core Inflation Score",\n'
    '    "6M Annualised Core Inflation PercentRank",\n'
    '    "6M Annualised Core Inflation Score",\n'
    '    "3M Change in 12M Core Inflation PercentRank",\n'
    '    "3M Change in 12M Core Inflation Score",\n'
    '    "Distance From 2 Percent Target",\n'
    '    "Core Inflation Target Status",\n'
    '    "Core Inflation Momentum Composite Score",\n'
    '    "Core Inflation Recovery / Cooling Score",\n'
    '    "Core Inflation Target Pressure Score",\n'
    '    "Core Inflation Outlier Score",\n'
    '    "Core Inflation Leading Score",\n'
    '    "Core Inflation Warning Composite Score",\n'
    '    "Core Inflation Final Warning Score",\n'
    '    "Core Inflation Pressure Regime",\n'
    '    "Core Inflation Momentum Label",\n'
    '    "Core Inflation Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Core Inflation Warning Risk Label",\n'
    '    "Core Inflation Policy Signal Label",\n'
    '    "12M Core Inflation Z-Score",\n'
    '    "3M Annualised Core Inflation Z-Score",\n'
    '    "6M Annualised Core Inflation Z-Score",\n'
    '    "3M Change in 12M Core Inflation Z-Score",\n'
    '    "Core Inflation Outlier Label",\n'
    ']\n'
    '\n'
    'CORE_CPI_INFLATION_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "1M Annualised Inflation",\n'
    '    "3M Annualised Inflation",\n'
    '    "6M Annualised Inflation",\n'
    '    "12M Inflation",\n'
    '    "3M Change in 12M Inflation",\n'
    '    "6M Change in 12M Inflation",\n'
    '    "12M Core CPI Inflation PercentRank",\n'
    '    "12M Core CPI Inflation Score",\n'
    '    "3M Annualised Core CPI Inflation PercentRank",\n'
    '    "3M Annualised Core CPI Inflation Score",\n'
    '    "6M Annualised Core CPI Inflation PercentRank",\n'
    '    "6M Annualised Core CPI Inflation Score",\n'
    '    "3M Change in 12M Core CPI Inflation PercentRank",\n'
    '    "3M Change in 12M Core CPI Inflation Score",\n'
    '    "Distance From 2 Percent Target",\n'
    '    "Core CPI Inflation Target Status",\n'
    '    "Core CPI Inflation Momentum Composite Score",\n'
    '    "Core CPI Inflation Recovery / Cooling Score",\n'
    '    "Core CPI Inflation Target Pressure Score",\n'
    '    "Core CPI Inflation Outlier Score",\n'
    '    "Core CPI Inflation Leading Score",\n'
    '    "Core CPI Inflation Warning Composite Score",\n'
    '    "Core CPI Inflation Final Warning Score",\n'
    '    "Core CPI Inflation Pressure Regime",\n'
    '    "Core CPI Inflation Momentum Label",\n'
    '    "Core CPI Inflation Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Core CPI Inflation Warning Risk Label",\n'
    '    "Core CPI Inflation Policy Signal Label",\n'
    '    "12M Core CPI Inflation Z-Score",\n'
    '    "3M Annualised Core CPI Inflation Z-Score",\n'
    '    "6M Annualised Core CPI Inflation Z-Score",\n'
    '    "3M Change in 12M Core CPI Inflation Z-Score",\n'
    '    "Core CPI Inflation Outlier Label",\n'
    ']\n'
    '\n'
    'HEADLINE_CPI_OUTPUT_COLUMNS = [\n'
    '    "Sample Count",\n'
    '    "1M Annualised Inflation",\n'
    '    "3M Annualised Inflation",\n'
    '    "6M Annualised Inflation",\n'
    '    "YoY Inflation",\n'
    '    "3M Change in YoY Inflation",\n'
    '    "6M Change in YoY Inflation",\n'
    '    "YoY Inflation Percentile",\n'
    '    "YoY Inflation Score",\n'
    '    "3M Inflation Percentile",\n'
    '    "3M Inflation Score",\n'
    '    "6M Inflation Percentile",\n'
    '    "6M Inflation Score",\n'
    '    "Momentum Percentile",\n'
    '    "Momentum Score",\n'
    '    "YoY Inflation Target Gap",\n'
    '    "Target Regime",\n'
    '    "Inflation Impulse Score",\n'
    '    "Inflation Cycle Turning Point Score",\n'
    '    "Policy Pressure Score",\n'
    '    "Inflation Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Policy flag",\n'
    '    "YoY Inflation Z-Score",\n'
    '    "3M Inflation Z-Score",\n'
    '    "6M Inflation Z-Score",\n'
    '    "Momentum Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    '# Retain the existing constant name used by engine routing while exposing the\n'
    '# A18 schema name used by the sheet specification.\n'
    'HEADLINE_CPI_INFLATION_OUTPUT_COLUMNS = HEADLINE_CPI_OUTPUT_COLUMNS\n'
    '\n'
    'CORE_PPI_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "1M Annualised Inflation",\n'
    '    "3M Annualised Inflation",\n'
    '    "6M Annualised Inflation",\n'
    '    "12M Inflation",\n'
    '    "3M Change in 12M Inflation",\n'
    '    "6M Change in 12M Inflation",\n'
    '    "12M Core PPI PercentRank",\n'
    '    "12M Core PPI Score",\n'
    '    "3M Annualised Core PPI PercentRank",\n'
    '    "3M Annualised Core PPI Score",\n'
    '    "6M Annualised Core PPI PercentRank",\n'
    '    "6M Annualised Core PPI Score",\n'
    '    "3M Change in 12M Core PPI PercentRank",\n'
    '    "3M Change in 12M Core PPI Score",\n'
    '    "Distance From 2 Percent Target",\n'
    '    "Core PPI Target Status",\n'
    '    "Core PPI Momentum Composite Score",\n'
    '    "Core PPI Recovery / Cooling Score",\n'
    '    "Core PPI Target Pressure Score",\n'
    '    "Core PPI Outlier Score",\n'
    '    "Core PPI Leading Score",\n'
    '    "Core PPI Warning Composite Score",\n'
    '    "Core PPI Final Warning Score",\n'
    '    "Core PPI Pressure Regime",\n'
    '    "Core PPI Momentum Label",\n'
    '    "Core PPI Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Core PPI Warning Risk Label",\n'
    '    "Core PPI Policy Signal Label",\n'
    '    "12M Core PPI Z-Score",\n'
    '    "3M Annualised Core PPI Z-Score",\n'
    '    "6M Annualised Core PPI Z-Score",\n'
    '    "3M Change in 12M Core PPI Z-Score",\n'
    '    "Core PPI Outlier Label",\n'
    ']\n'
    '\n'
    'HEADLINE_PPI_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "1M Annualised Inflation",\n'
    '    "3M Annualised Inflation",\n'
    '    "6M Annualised Inflation",\n'
    '    "12M Inflation",\n'
    '    "3M Change in 12M Inflation",\n'
    '    "6M Change in 12M Inflation",\n'
    '    "12M Headline PPI PercentRank",\n'
    '    "12M Headline PPI Score",\n'
    '    "3M Annualised Headline PPI PercentRank",\n'
    '    "3M Annualised Headline PPI Score",\n'
    '    "6M Annualised Headline PPI PercentRank",\n'
    '    "6M Annualised Headline PPI Score",\n'
    '    "3M Change in 12M Headline PPI PercentRank",\n'
    '    "3M Change in 12M Headline PPI Score",\n'
    '    "Distance From 2 Percent Target",\n'
    '    "Headline PPI Target Status",\n'
    '    "Headline PPI Momentum Composite Score",\n'
    '    "Headline PPI Recovery / Cooling Score",\n'
    '    "Headline PPI Target Pressure Score",\n'
    '    "Headline PPI Outlier Score",\n'
    '    "Headline PPI Leading Score",\n'
    '    "Headline PPI Warning Composite Score",\n'
    '    "Headline PPI Final Warning Score",\n'
    '    "Headline PPI Pressure Regime",\n'
    '    "Headline PPI Momentum Label",\n'
    '    "Headline PPI Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Headline PPI Warning Risk Label",\n'
    '    "Headline PPI Policy Signal Label",\n'
    '    "12M Headline PPI Z-Score",\n'
    '    "3M Annualised Headline PPI Z-Score",\n'
    '    "6M Annualised Headline PPI Z-Score",\n'
    '    "3M Change in 12M Headline PPI Z-Score",\n'
    '    "Headline PPI Outlier Label",\n'
    ']\n'
    '\n'
    'WAGE_GROWTH_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "1-Period Annualised Wage Growth",\n'
    '    "2-Period Annualised Wage Growth",\n'
    '    "4-Period Wage Growth",\n'
    '    "1-Period Change in 4-Period Wage Growth",\n'
    '    "2-Period Change in 4-Period Wage Growth",\n'
    '    "4-Period Wage Growth PercentRank",\n'
    '    "4-Period Wage Growth Score",\n'
    '    "1-Period Annualised Wage Growth PercentRank",\n'
    '    "1-Period Annualised Wage Growth Score",\n'
    '    "2-Period Annualised Wage Growth PercentRank",\n'
    '    "2-Period Annualised Wage Growth Score",\n'
    '    "1-Period Change in 4-Period Wage Growth PercentRank",\n'
    '    "1-Period Change in 4-Period Wage Growth Score",\n'
    '    "Wage Pressure Status",\n'
    '    "Wage Pressure Momentum Composite Score",\n'
    '    "Wage Pressure Recovery / Cooling Score",\n'
    '    "Wage Pressure Persistence Score",\n'
    '    "Wage Pressure Outlier Score",\n'
    '    "Wage Pressure Leading Score",\n'
    '    "Wage Pressure Warning Composite Score",\n'
    '    "Wage Pressure Final Warning Score",\n'
    '    "Wage Pressure Regime",\n'
    '    "Wage Pressure Momentum Label",\n'
    '    "Wage Pressure Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Wage Pressure Warning Risk Label",\n'
    '    "Wage Pressure Policy Signal Label",\n'
    '    "4-Period Wage Growth Z-Score",\n'
    '    "1-Period Annualised Wage Growth Z-Score",\n'
    '    "2-Period Annualised Wage Growth Z-Score",\n'
    '    "1-Period Change in 4-Period Wage Growth Z-Score",\n'
    '    "Wage Pressure Outlier Label",\n'
    ']\n'
    '\n'
    'SHORT_TERM_INFLATION_EXPECTATIONS_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "3-Period Moving Average",\n'
    '    "1-Period Change",\n'
    '    "3-Period Change in Moving Average",\n'
    '    "6-Period Change in Moving Average",\n'
    '    "12-Period Change in Moving Average",\n'
    '    "Short-Term Expectations Level PercentRank",\n'
    '    "Short-Term Expectations Level Score",\n'
    '    "3-Period Expectations Change PercentRank",\n'
    '    "3-Period Expectations Change Score",\n'
    '    "6-Period Expectations Change PercentRank",\n'
    '    "6-Period Expectations Change Score",\n'
    '    "12-Period Expectations Change PercentRank",\n'
    '    "12-Period Expectations Change Score",\n'
    '    "Short-Term Inflation Expectations Status",\n'
    '    "Inflation Expectations Momentum Composite Score",\n'
    '    "Inflation Expectations Recovery / Cooling Score",\n'
    '    "Inflation Expectations Pressure Score",\n'
    '    "Inflation Expectations Outlier Score",\n'
    '    "Inflation Expectations Leading Score",\n'
    '    "Inflation Expectations Warning Composite Score",\n'
    '    "Inflation Expectations Final Warning Score",\n'
    '    "Inflation Expectations Regime",\n'
    '    "Inflation Expectations Momentum Label",\n'
    '    "Inflation Expectations Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Inflation Expectations Warning Risk Label",\n'
    '    "Inflation Expectations Policy Signal Label",\n'
    '    "Short-Term Expectations Level Z-Score",\n'
    '    "3-Period Expectations Change Z-Score",\n'
    '    "6-Period Expectations Change Z-Score",\n'
    '    "12-Period Expectations Change Z-Score",\n'
    '    "Inflation Expectations Outlier Label",\n'
    ']\n'
    '\n'
    'REAL_YIELD_OUTPUT_COLUMNS = [\n'
    '    "3M MA Real Yield",\n'
    '    "Sample Count",\n'
    '    "1M Real Yield Change",\n'
    '    "3M Real Yield Change",\n'
    '    "6M Real Yield Change",\n'
    '    "12M Real Yield Change",\n'
    '    "Real Yield Level Percentile",\n'
    '    "Real Yield Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Score",\n'
    '    "Real Yield Regime",\n'
    '    "Real Yield Asset Bias",\n'
    '    "1M Real Yield Shock Flag",\n'
    '    "Real Yield Impulse Score",\n'
    '    "Real Yield Cycle Turning Point Score",\n'
    '    "Real Yield Shock Score",\n'
    '    "Policy Restriction Score",\n'
    '    "Real Yield Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'TWO_YEAR_YIELD_OUTPUT_COLUMNS = [\n'
    '    "3M MA Yield",\n'
    '    "Sample Count",\n'
    '    "1M Yield Change",\n'
    '    "3M Yield Change",\n'
    '    "6M Yield Change",\n'
    '    "12M Yield Change",\n'
    '    "Yield Level Percentile",\n'
    '    "Yield Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Score",\n'
    '    "2Y Yield Regime",\n'
    '    "2Y Yield Policy Bias",\n'
    '    "1M 2Y Yield Shock Flag",\n'
    '    "2Y Yield Impulse Score",\n'
    '    "2Y Yield Cycle Turning Point Score",\n'
    '    "2Y Yield Shock Score",\n'
    '    "Policy Rate Restriction Score",\n'
    '    "2Y Yield Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'TEN_YEAR_YIELD_OUTPUT_COLUMNS = [\n'
    '    "3M MA Yield",\n'
    '    "Sample Count",\n'
    '    "1M Yield Change",\n'
    '    "3M Yield Change",\n'
    '    "6M Yield Change",\n'
    '    "12M Yield Change",\n'
    '    "Yield Level Percentile",\n'
    '    "Yield Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Score",\n'
    '    "10Y Yield Regime",\n'
    '    "10Y Yield Asset Bias",\n'
    '    "1M 10Y Yield Shock Flag",\n'
    '    "10Y Yield Impulse Score",\n'
    '    "10Y Yield Cycle Turning Point Score",\n'
    '    "10Y Yield Shock Score",\n'
    '    "Long-Rate Restriction Score",\n'
    '    "10Y Yield Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'YIELD_CURVE_OUTPUT_COLUMNS = [\n'
    '    "3M MA Curve Spread",\n'
    '    "Sample Count",\n'
    '    "1M Curve Change",\n'
    '    "3M Curve Change",\n'
    '    "6M Curve Change",\n'
    '    "12M Curve Change",\n'
    '    "Curve Level Percentile",\n'
    '    "Curve Stress Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Stress Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Stress Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Stress Score",\n'
    '    "Curve Regime",\n'
    '    "Recession Risk Bias",\n'
    '    "Curve Shock Flag",\n'
    '    "Curve Impulse Score",\n'
    '    "Curve Cycle Turning Point Score",\n'
    '    "Curve Shock Score",\n'
    '    "Recession Risk Pressure Score",\n'
    '    "Curve Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'TERM_PREMIUM_OUTPUT_COLUMNS = [\n'
    '    "3-Period Moving Average",\n'
    '    "Smoothed Observation Count",\n'
    '    "1-Period Change",\n'
    '    "3-Period Change",\n'
    '    "6-Period Change",\n'
    '    "12-Period Change",\n'
    '    "Term Premium Level PercentRank",\n'
    '    "Term Premium Level Score",\n'
    '    "3-Period Term Premium Change PercentRank",\n'
    '    "3-Period Term Premium Change Score",\n'
    '    "6-Period Term Premium Change PercentRank",\n'
    '    "6-Period Term Premium Change Score",\n'
    '    "12-Period Term Premium Change PercentRank",\n'
    '    "12-Period Term Premium Change Score",\n'
    '    "Term Premium Level Regime",\n'
    '    "Term Premium Asset Bias Label",\n'
    '    "1-Period Term Premium Shock Label",\n'
    '    "Term Premium Momentum Composite Score",\n'
    '    "Term Premium Recovery / Tightening Score",\n'
    '    "Term Premium Shock Score",\n'
    '    "Term Premium Restriction Score",\n'
    '    "Term Premium Outlier Score",\n'
    '    "Term Premium Leading Score",\n'
    '    "Term Premium Warning Composite Score",\n'
    '    "Term Premium Final Warning Score",\n'
    '    "Term Premium Pressure Regime",\n'
    '    "Term Premium Momentum Label",\n'
    '    "Term Premium Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Term Premium Warning Risk Label",\n'
    '    "Term Premium Level Z-Score",\n'
    '    "3-Period Term Premium Change Z-Score",\n'
    '    "6-Period Term Premium Change Z-Score",\n'
    '    "12-Period Term Premium Change Z-Score",\n'
    '    "Term Premium Outlier Label",\n'
    ']\n'
    '\n'
    'INTERBANK_RATE_OUTPUT_COLUMNS = [\n'
    '    "3M MA Rate",\n'
    '    "Sample Count",\n'
    '    "1M Rate Change",\n'
    '    "3M Rate Change",\n'
    '    "6M Rate Change",\n'
    '    "12M Rate Change",\n'
    '    "Rate Level Percentile",\n'
    '    "Rate Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Score",\n'
    '    "Interbank Rate Regime",\n'
    '    "Interbank Rate Policy Bias",\n'
    '    "1M Interbank Rate Shock Flag",\n'
    '    "Interbank Rate Impulse Score",\n'
    '    "Interbank Rate Cycle Turning Point Score",\n'
    '    "Interbank Rate Shock Score",\n'
    '    "Interbank Rate Pressure Score",\n'
    '    "Interbank Rate Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    '\n'
    'GOVERNMENT_REVENUE_OUTPUT_COLUMNS = [\n'
    '    "12M rolling Government Revenue",\n'
    '    "Sample Count",\n'
    '    "1M Revenue Growth",\n'
    '    "3M Revenue Growth",\n'
    '    "6M Revenue Growth",\n'
    '    "12M YoY Revenue Growth",\n'
    '    "3M Annualised Revenue Growth",\n'
    '    "6M Annualised Revneue Growth",\n'
    '    "3M Growth Percentile",\n'
    '    "3M Growth Score",\n'
    '    "6M Growth Percentile",\n'
    '    "6M Growth Score",\n'
    '    "12M YoY Growth Percentile",\n'
    '    "12M YoY Growth Score",\n'
    '    "3M Annualised Growth Percentile",\n'
    '    "3M Annualised Growth Score",\n'
    '    "Revenue Growth Regime",\n'
    '    "Fiscal Revenue Bias",\n'
    '    "Government Revenue Shock Flag",\n'
    '    "Government Revenue Impulse Score",\n'
    '    "Government Revenue Acceleration Score",\n'
    '    "Government Revenue Cycle Turning Point Score",\n'
    '    "Government Revenue Shock Score",\n'
    '    "Revenue Weakness Risk Score",\n'
    '    "Revenue Overheating Score",\n'
    '    "Government Revenue Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "3M Growth Z-Score",\n'
    '    "6M Growth Z-Score",\n'
    '    "12M Growth Z-Score",\n'
    '    "3M Annualised Growth Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'PUBLIC_DEBT_OUTPUT_COLUMNS = [\n'
    '    "2-Period Moving Average Public Debt",\n'
    '    "Public Debt Observation Count",\n'
    '    "1-Period Public Debt Change",\n'
    '    "2-Period Public Debt Change",\n'
    '    "4-Period Public Debt Change",\n'
    '    "1-Period Annualised Public Debt Change",\n'
    '    "2-Period Annualised Public Debt Change",\n'
    '    "Public Debt Level PercentRank",\n'
    '    "Public Debt Level Score",\n'
    '    "1-Period Public Debt Change PercentRank",\n'
    '    "1-Period Public Debt Change Score",\n'
    '    "2-Period Public Debt Change PercentRank",\n'
    '    "2-Period Public Debt Change Score",\n'
    '    "4-Period Public Debt Change PercentRank",\n'
    '    "4-Period Public Debt Change Score",\n'
    '    "Public Debt Regime",\n'
    '    "Public Debt Fiscal-Sustainability Bias Label",\n'
    '    "Public Debt Shock Label",\n'
    '    "Public Debt Momentum Composite Score",\n'
    '    "Public Debt Momentum Turn Score",\n'
    '    "Public Debt Level / Vulnerability Score",\n'
    '    "Public Debt Recovery / Deterioration Score",\n'
    '    "Public Debt Shock Score",\n'
    '    "Public Debt Outlier Score",\n'
    '    "Public Debt Leading Score",\n'
    '    "Public Debt Warning Composite Score",\n'
    '    "Public Debt Final Warning Score",\n'
    '    "Public Debt Conditions Regime",\n'
    '    "Public Debt Momentum Label",\n'
    '    "Public Debt Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Public Debt Warning Risk Label",\n'
    '    "Public Debt Level Z-Score",\n'
    '    "1-Period Public Debt Change Z-Score",\n'
    '    "2-Period Public Debt Change Z-Score",\n'
    '    "4-Period Public Debt Change Z-Score",\n'
    '    "Public Debt Outlier Label",\n'
    ']\n'
    '\n'
    'INTEREST_COVERAGE_OUTPUT_COLUMNS = [\n'
    '    "Raw Interest Coverage Ratio",\n'
    '    "3-Period Moving Average Interest Coverage",\n'
    '    "Interest Coverage Observation Count",\n'
    '    "1-Period Interest Coverage Change",\n'
    '    "3-Period Interest Coverage Change",\n'
    '    "5-Period Interest Coverage Change",\n'
    '    "Interest Coverage Level PercentRank",\n'
    '    "Interest Coverage Level Score",\n'
    '    "1-Period Interest Coverage Change PercentRank",\n'
    '    "1-Period Interest Coverage Change Score",\n'
    '    "3-Period Interest Coverage Change PercentRank",\n'
    '    "3-Period Interest Coverage Change Score",\n'
    '    "5-Period Interest Coverage Change PercentRank",\n'
    '    "5-Period Interest Coverage Change Score",\n'
    '    "Interest Coverage Regime",\n'
    '    "Interest Coverage Fiscal-Stress Bias Label",\n'
    '    "Interest Coverage Shock Label",\n'
    '    "Interest Coverage Momentum Composite Score",\n'
    '    "Interest Coverage Momentum Turn Score",\n'
    '    "Interest Coverage Level / Vulnerability Score",\n'
    '    "Interest Coverage Recovery / Deterioration Score",\n'
    '    "Interest Coverage Shock Score",\n'
    '    "Interest Coverage Outlier Score",\n'
    '    "Interest Coverage Leading Score",\n'
    '    "Interest Coverage Warning Composite Score",\n'
    '    "Interest Coverage Final Warning Score",\n'
    '    "Interest Coverage Conditions Regime",\n'
    '    "Interest Coverage Momentum Label",\n'
    '    "Interest Coverage Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Interest Coverage Warning Risk Label",\n'
    '    "Interest Coverage Level Z-Score",\n'
    '    "1-Period Interest Coverage Change Z-Score",\n'
    '    "3-Period Interest Coverage Change Z-Score",\n'
    '    "5-Period Interest Coverage Change Z-Score",\n'
    '    "Interest Coverage Outlier Label",\n'
    ']\n'
    '\n'
    'FINANCIAL_CONDITIONS_OUTPUT_COLUMNS = [\n'
    '    "Monthly Average Financial Conditions",\n'
    '    "Sample Count",\n'
    '    "3M MA Financial Conditions",\n'
    '    "1M Financial Conditions Change",\n'
    '    "3M Financial Conditions Change",\n'
    '    "6M Financial Conditions Change",\n'
    '    "12M Financial Conditions Change",\n'
    '    "Financial Conditions Level Percentile",\n'
    '    "Financial Conditions Level Stress Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Stress Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Stress Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Stress Score",\n'
    '    "Financial Conditions Regime",\n'
    '    "Risk Asset Bias",\n'
    '    "Financial Conditions Shock Flag",\n'
    '    "Financial Conditions Impulse Score",\n'
    '    "Financial Stress Risk Score",\n'
    '    "Financial Conditions Shock Score",\n'
    '    "Financial Conditions Cycle Turning Point Score",\n'
    '    "Financial Conditions Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'BANK_CREDIT_OUTPUT_COLUMNS = [\n'
    '    "3M MA Bank Credit",\n'
    '    "Sample Count",\n'
    '    "1M Bank Credit Growth",\n'
    '    "3M Bank Credit Growth",\n'
    '    "6M Bank Credit Growth",\n'
    '    "12M Bank Credit Growth",\n'
    '    "3M Annualised Bank Credit Growth",\n'
    '    "6M Annualised Bank Credit Growth",\n'
    '    "Bank Credit Level Percentile",\n'
    '    "Bank Credit Level Score",\n'
    '    "3M Growth Percentile",\n'
    '    "3M Growth Score",\n'
    '    "6M Growth Percentile",\n'
    '    "6M Growth Score",\n'
    '    "12M Growth Percentile",\n'
    '    "12M Growth Score",\n'
    '    "Bank Credit Regime",\n'
    '    "Risk Asset Bias",\n'
    '    "Bank Credit Growth Shock Flag",\n'
    '    "Bank Credit Impulse Score",\n'
    '    "Bank Credit Acceleration Score",\n'
    '    "Bank Credit Cycle Turning Point Score",\n'
    '    "Bank Credit Shock Score",\n'
    '    "Credit Contraction Risk Score",\n'
    '    "Credit Overheating Risk Score",\n'
    '    "Bank Credit Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Growth Z-Score",\n'
    '    "6M Growth Z-Score",\n'
    '    "12M Growth Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS = [\n'
    '    "3-Period Moving Average Corporate Credit Spread",\n'
    '    "Corporate Credit Spread Observation Count",\n'
    '    "1-Period Corporate Credit Spread Change",\n'
    '    "3-Period Corporate Credit Spread Change",\n'
    '    "6-Period Corporate Credit Spread Change",\n'
    '    "12-Period Corporate Credit Spread Change",\n'
    '    "Corporate Credit Spread Level PercentRank",\n'
    '    "Corporate Credit Spread Level Score",\n'
    '    "3-Period Corporate Credit Spread Change PercentRank",\n'
    '    "3-Period Corporate Credit Spread Change Score",\n'
    '    "6-Period Corporate Credit Spread Change PercentRank",\n'
    '    "6-Period Corporate Credit Spread Change Score",\n'
    '    "12-Period Corporate Credit Spread Change PercentRank",\n'
    '    "12-Period Corporate Credit Spread Change Score",\n'
    '    "Corporate Credit Spread Regime",\n'
    '    "Corporate Credit Spread Asset Bias Label",\n'
    '    "Corporate Credit Spread Shock Label",\n'
    '    "Corporate Credit Spread Leading Composite Score",\n'
    '    "Corporate Credit Spread Momentum Composite Score",\n'
    '    "Corporate Credit Spread Recovery / Deterioration Score",\n'
    '    "Corporate Credit Spread Shock Score",\n'
    '    "Corporate Credit Spread Outlier Score",\n'
    '    "Corporate Credit Spread Leading Score",\n'
    '    "Corporate Credit Spread Warning Composite Score",\n'
    '    "Corporate Credit Spread Final Warning Score",\n'
    '    "Corporate Credit Spread Conditions Regime",\n'
    '    "Corporate Credit Spread Momentum Label",\n'
    '    "Corporate Credit Spread Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Corporate Credit Spread Warning Risk Label",\n'
    '    "Corporate Credit Spread Level Z-Score",\n'
    '    "3-Period Corporate Credit Spread Change Z-Score",\n'
    '    "6-Period Corporate Credit Spread Change Z-Score",\n'
    '    "12-Period Corporate Credit Spread Change Z-Score",\n'
    '    "Corporate Credit Spread Outlier Label",\n'
    ']\n'
    '\n'
    'REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS = [\n'
    '    "2-Period Moving Average Real Government Spending",\n'
    '    "Real Government Spending Observation Count",\n'
    '    "1-Period Real Government Spending Growth",\n'
    '    "2-Period Real Government Spending Growth",\n'
    '    "4-Period Real Government Spending Growth",\n'
    '    "1-Period Annualised Real Government Spending Growth",\n'
    '    "2-Period Annualised Real Government Spending Growth",\n'
    '    "1-Period Real Government Spending Growth PercentRank",\n'
    '    "1-Period Real Government Spending Growth Score",\n'
    '    "2-Period Real Government Spending Growth PercentRank",\n'
    '    "2-Period Real Government Spending Growth Score",\n'
    '    "4-Period Real Government Spending Growth PercentRank",\n'
    '    "4-Period Real Government Spending Growth Score",\n'
    '    "Annualised Real Government Spending Growth PercentRank",\n'
    '    "Annualised Real Government Spending Growth Score",\n'
    '    "Real Government Spending Regime",\n'
    '    "Real Government Spending Asset Bias Label",\n'
    '    "Real Government Spending Shock Label",\n'
    '    "Real Government Spending Momentum Composite Score",\n'
    '    "Real Government Spending Momentum Turn Score",\n'
    '    "Real Government Spending Recovery / Deterioration Score",\n'
    '    "Real Government Spending Shock Score",\n'
    '    "Real Government Spending Outlier Score",\n'
    '    "Real Government Spending Leading Score",\n'
    '    "Real Government Spending Warning Composite Score",\n'
    '    "Real Government Spending Final Warning Score",\n'
    '    "Real Government Spending Conditions Regime",\n'
    '    "Real Government Spending Momentum Label",\n'
    '    "Real Government Spending Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Real Government Spending Warning Risk Label",\n'
    '    "1-Period Real Government Spending Growth Z-Score",\n'
    '    "2-Period Real Government Spending Growth Z-Score",\n'
    '    "4-Period Real Government Spending Growth Z-Score",\n'
    '    "Annualised Real Government Spending Growth Z-Score",\n'
    '    "Real Government Spending Outlier Label",\n'
    ']\n'
    '\n'
    'FISCAL_BALANCE_OUTPUT_COLUMNS = [\n'
    '    "3-Period Moving Average Fiscal Balance",\n'
    '    "Fiscal Balance Observation Count",\n'
    '    "1-Period Fiscal Balance Change",\n'
    '    "3-Period Fiscal Balance Change",\n'
    '    "5-Period Fiscal Balance Change",\n'
    '    "Fiscal Balance Level PercentRank",\n'
    '    "Fiscal Balance Level Score",\n'
    '    "1-Period Fiscal Balance Change PercentRank",\n'
    '    "1-Period Fiscal Balance Change Score",\n'
    '    "3-Period Fiscal Balance Change PercentRank",\n'
    '    "3-Period Fiscal Balance Change Score",\n'
    '    "5-Period Fiscal Balance Change PercentRank",\n'
    '    "5-Period Fiscal Balance Change Score",\n'
    '    "Fiscal Balance Regime",\n'
    '    "Fiscal Expansion Impulse Composite Score",\n'
    '    "Fiscal Impulse Bias Label",\n'
    '    "Fiscal Impulse Momentum Score",\n'
    '    "Fiscal Balance Interaction Score",\n'
    '    "Fiscal Deficit Sustainability Risk Score",\n'
    '    "Fiscal Balance Outlier Score",\n'
    '    "Fiscal Impulse Leading Score",\n'
    '    "Fiscal Impulse Warning Composite Score",\n'
    '    "Fiscal Impulse Final Warning Score",\n'
    '    "Fiscal Impulse Regime",\n'
    '    "Fiscal Impulse Momentum Label",\n'
    '    "Fiscal Impulse Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Fiscal Warning Risk Label",\n'
    '    "Fiscal Balance Level Z-Score",\n'
    '    "1-Period Fiscal Balance Change Z-Score",\n'
    '    "3-Period Fiscal Balance Change Z-Score",\n'
    '    "5-Period Fiscal Balance Change Z-Score",\n'
    '    "Fiscal Balance Outlier Label",\n'
    ']\n'
    '\n'
    'BROAD_MONEY_OUTPUT_COLUMNS = [\n'
    '    "3M MA Broad Money",\n'
    '    "Sample Count",\n'
    '    "1M Broad Money Growth",\n'
    '    "3M Broad Money Growth",\n'
    '    "6M Broad Money Growth",\n'
    '    "12M Broad Money Growth",\n'
    '    "3M Annualised Broad Money Growth",\n'
    '    "6M Annualised Broad Money Growth",\n'
    '    "Broad Money Level Percentile",\n'
    '    "Broad Money Level Score",\n'
    '    "3M Growth Percentile",\n'
    '    "3M Growth Score",\n'
    '    "6M Growth Percentile",\n'
    '    "6M Growth Score",\n'
    '    "12M Growth Percentile",\n'
    '    "12M Growth Score",\n'
    '    "Broad Money Regime",\n'
    '    "Risk Asset Liquidity Bias",\n'
    '    "Money Growth Shock Flag",\n'
    '    "Broad Money Impulse Score",\n'
    '    "Broad Money Cycle Turning Point Score",\n'
    '    "Money Growth Shock Score",\n'
    '    "Monetary Overheating Risk Score",\n'
    '    "Monetary Contraction Risk Score",\n'
    '    "Broad Money Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Growth Z-Score",\n'
    '    "6M Growth Z-Score",\n'
    '    "12M Growth Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'BANKING_RESERVES_OUTPUT_COLUMNS = [\n'
    '    "Monthly Average Reserves",\n'
    '    "Sample Count",\n'
    '    "3M MA Reserves",\n'
    '    "1M Reserve Change",\n'
    '    "3M Reserve Change",\n'
    '    "6M Reserve Change",\n'
    '    "12M Reserve Change",\n'
    '    "3M Reserve Growth",\n'
    '    "6M Reserve Growth",\n'
    '    "12M Reserve Growth",\n'
    '    "Reserve Level Percentile",\n'
    '    "Reserve Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Score",\n'
    '    "Reserve Liquidity Regime",\n'
    '    "Risk Asset Liquidity Bias",\n'
    '    "Reserve Shock Flag",\n'
    '    "Reserve Impulse Score",\n'
    '    "Reserve Cycle Turning Point Score",\n'
    '    "Reserve Shock Score",\n'
    '    "Reserve Drain Risk Score",\n'
    '    "Reserve Scarcity Stress Score",\n'
    '    "Reserve Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS = [\n'
    '    "Central Bank Net Liquidity",\n'
    '    "Monthly Net Liquidity",\n'
    '    "Sample Count",\n'
    '    "3M MA Net Liquidity",\n'
    '    "1M Net Liquidity Change",\n'
    '    "3M Net Liquidity Change",\n'
    '    "6M Net Liquidity Change",\n'
    '    "12M Net Liquidity Change",\n'
    '    "3M Net Liquidity Growth",\n'
    '    "6M Net Liquidity Growth",\n'
    '    "12M Net Liquidity Growth",\n'
    '    "Liquidity Level Percentile",\n'
    '    "Liquidity Level Score",\n'
    '    "3M Change Percentile",\n'
    '    "3M Change Score",\n'
    '    "6M Change Percentile",\n'
    '    "6M Change Score",\n'
    '    "12M Change Percentile",\n'
    '    "12M Change Score",\n'
    '    "Liquidity Regime",\n'
    '    "Risk Asset Liquidity Bias",\n'
    '    "Liquidity Shock Flag",\n'
    '    "Liquidity Impulse Score",\n'
    '    "Liquidity Cycle Turning Point Score",\n'
    '    "Liquidity Shock Score",\n'
    '    "Liquidity Withdrawal Risk Score",\n'
    '    "Liquidity Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "12M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'CONSUMER_SENTIMENT_LEGACY_OUTPUT_COLUMNS = [\n'
    '    "Observation Count",\n'
    '    "Distance From Rolling Median",\n'
    '    "1M Change",\n'
    '    "3M Change",\n'
    '    "6M Change",\n'
    '    "Level PercentRank",\n'
    '    "Sentiment Level Score",\n'
    '    "3M Change PercentRank",\n'
    '    "3M Momentum Score",\n'
    '    "6M Change PercentRank",\n'
    '    "6M Momentum Score",\n'
    '    "Consumer Sentiment Recovery / Momentum Score",\n'
    '    "Consumer Sentiment Low/High Turn Score",\n'
    '    "Consumer Sentiment Warning Score Raw",\n'
    '    "Consumer Sentiment Outlier Score",\n'
    '    "Consumer Sentiment Level Composite Score",\n'
    '    "Consumer Sentiment Leading Score",\n'
    '    "Consumer Sentiment Final Warning Score",\n'
    '    "Consumer Sentiment Regime",\n'
    '    "Consumer Sentiment Momentum Label",\n'
    '    "Consumer Sentiment Signal Interpretation",\n'
    '    "Reliability",\n'
    '    "Data Status",\n'
    '    "Consumer Sentiment Warning Risk Label",\n'
    '    "Consumer Sentiment Signal Label",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "Consumer Sentiment Outlier Label",\n'
    ']\n'
    '\n'
    'CONSUMER_SENTIMENT_OUTPUT_COLUMNS = [\n'
    '    "Sample Count",\n'
    '    "Distance from historical median",\n'
    '    "1M Change",\n'
    '    "3M Change",\n'
    '    "6M Change",\n'
    '    "Level Percentile",\n'
    '    "Level Score",\n'
    '    "3M Momentum Percentile",\n'
    '    "3M Momentum Score",\n'
    '    "6M Momentum Percentile",\n'
    '    "6M Momentum Score",\n'
    '    "Sentiment Cycle Turning Point Score",\n'
    '    "Contrarian Recovery Score",\n'
    '    "Sentiment Downside Warning Score",\n'
    '    "Sentiment Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Slowdown Risk Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS = [\n'
    '    "Sample Count",\n'
    '    "Distance from historical median",\n'
    '    "1M Change",\n'
    '    "3M Change",\n'
    '    "6M Change",\n'
    '    "Level Percentile",\n'
    '    "Level Score",\n'
    '    "3M Momentum Percentile",\n'
    '    "3M Momentum Score",\n'
    '    "6M Momentum Percentile",\n'
    '    "6M Momentum Score",\n'
    '    "Expectations Cycle Turning Point Score",\n'
    '    "Expectations Contrarian Recovery Score",\n'
    '    "Expectations Downside Warning Score",\n'
    '    "Expectations Extreme Score",\n'
    '    "Coincident Score",\n'
    '    "Leading Score",\n'
    '    "Warning Score",\n'
    '    "State",\n'
    '    "Regime",\n'
    '    "Signal Comment",\n'
    '    "Reliability",\n'
    '    "Signal Validity",\n'
    '    "Warning Flag",\n'
    '    "Slowdown Risk Flag",\n'
    '    "Level Z-Score",\n'
    '    "3M Change Z-Score",\n'
    '    "6M Change Z-Score",\n'
    '    "Extreme Flag",\n'
    ']\n'
    '\n'
    'LEVEL_SCORE_COLUMNS = [\n'
    '    "Level Score",\n'
    '    "Yield Level Score",\n'
    '    "YoY Inflation Score",\n'
    '    "YoY Growth Score",\n'
    '    "12M Growth Score",\n'
    '    "12M Claims Growth Score",\n'
    '    "12M Employment Growth Score",\n'
    '    "12M Average Hours Growth Score",\n'
    '    "12M Building Permits Growth Score",\n'
    '    "12M Core Inflation Score",\n'
    '    "12M Core CPI Inflation Score",\n'
    '    "12M Headline CPI Inflation Score",\n'
    '    "12M Core PPI Score",\n'
    '    "12M Headline PPI Score",\n'
    '    "4-Period Wage Growth Score",\n'
    '    "Short-Term Expectations Level Score",\n'
    '    "Real Yield Level Score",\n'
    '    "2Y Yield Level Score",\n'
    '    "10Y Yield Level Score",\n'
    '    "Curve Stress Level Score",\n'
    '    "Yield Curve Stress Level Score",\n'
    '    "Term Premium Level Score",\n'
    '    "Rate Level Score",\n'
    '    "Interbank Rate Level Score",\n'
    '    "Liquidity Level Score",\n'
    '    "Central Bank Liquidity Level Score",\n'
    '    "Reserve Level Score",\n'
    '    "Banking Reserves Level Score",\n'
    '    "Broad Money Level Score",\n'
    '    "Fiscal Balance Level Score",\n'
    '    "Public Debt Level Score",\n'
    '    "Interest Coverage Level Score",\n'
    '    "Financial Conditions Level Stress Score",\n'
    '    "Bank Credit Level Score",\n'
    '    "Corporate Credit Spread Level Score",\n'
    '    "1-Period Real Government Spending Growth Score",\n'
    '    "Sentiment Level Score",\n'
    '    "Expectations Level Score",\n'
    '    "Real Activity Level Score",\n'
    '    "Growth Score",\n'
    ']\n'
    'LEADING_SCORE_COLUMNS = [\n'
    '    "Leading Score",\n'
    '    "Industrial Leading Score",\n'
    '    "Consumer Spending Leading Score",\n'
    '    "Initial Claims Leading Score",\n'
    '    "Employment Leading Score",\n'
    '    "Average Hours Leading Score",\n'
    '    "Building Permits Leading Score",\n'
    '    "Core Inflation Leading Score",\n'
    '    "Core CPI Inflation Leading Score",\n'
    '    "Headline CPI Inflation Leading Score",\n'
    '    "Core PPI Leading Score",\n'
    '    "Headline PPI Leading Score",\n'
    '    "Wage Pressure Leading Score",\n'
    '    "Inflation Expectations Leading Score",\n'
    '    "Real Yield Leading Score",\n'
    '    "2Y Yield Leading Score",\n'
    '    "10Y Yield Leading Score",\n'
    '    "Yield Curve Stress Leading Score",\n'
    '    "Term Premium Leading Score",\n'
    '    "Interbank Rate Leading Score",\n'
    '    "Central Bank Liquidity Leading Score",\n'
    '    "Banking Reserves Leading Score",\n'
    '    "Broad Money Leading Score",\n'
    '    "Fiscal Impulse Leading Score",\n'
    '    "Real Government Spending Leading Score",\n'
    '    "Government Revenue Leading Score",\n'
    '    "Public Debt Leading Score",\n'
    '    "Interest Coverage Leading Score",\n'
    '    "Financial Conditions Leading Score",\n'
    '    "Bank Credit Leading Score",\n'
    '    "Corporate Credit Spread Leading Score",\n'
    '    "Consumer Sentiment Leading Score",\n'
    '    "Consumer Expectations Leading Score",\n'
    ']\n'
    'WARNING_SCORE_COLUMNS = [\n'
    '    "Warning Score",\n'
    '    "Industrial Final Warning Score",\n'
    '    "Consumer Spending Final Warning Score",\n'
    '    "Initial Claims Final Warning Score",\n'
    '    "Employment Final Warning Score",\n'
    '    "Average Hours Final Warning Score",\n'
    '    "Building Permits Final Warning Score",\n'
    '    "Core Inflation Final Warning Score",\n'
    '    "Core CPI Inflation Final Warning Score",\n'
    '    "Headline CPI Inflation Final Warning Score",\n'
    '    "Core PPI Final Warning Score",\n'
    '    "Headline PPI Final Warning Score",\n'
    '    "Wage Pressure Final Warning Score",\n'
    '    "Inflation Expectations Final Warning Score",\n'
    '    "Real Yield Final Warning Score",\n'
    '    "2Y Yield Final Warning Score",\n'
    '    "10Y Yield Final Warning Score",\n'
    '    "Yield Curve Stress Final Warning Score",\n'
    '    "Term Premium Final Warning Score",\n'
    '    "Interbank Rate Final Warning Score",\n'
    '    "Central Bank Liquidity Final Warning Score",\n'
    '    "Banking Reserves Final Warning Score",\n'
    '    "Broad Money Final Warning Score",\n'
    '    "Fiscal Impulse Final Warning Score",\n'
    '    "Real Government Spending Final Warning Score",\n'
    '    "Government Revenue Final Warning Score",\n'
    '    "Public Debt Final Warning Score",\n'
    '    "Interest Coverage Final Warning Score",\n'
    '    "Financial Conditions Final Warning Score",\n'
    '    "Bank Credit Final Warning Score",\n'
    '    "Corporate Credit Spread Final Warning Score",\n'
    '    "Consumer Sentiment Final Warning Score",\n'
    '    "Consumer Expectations Final Warning Score",\n'
    ']\n'
    'STATUS_COLUMNS = ["Data Status", "Signal Validity"]\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class EngineConfig:\n'
    '    spreadsheet_id: str\n'
    '    service_account_path: str\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class IndicatorRunSummary:\n'
    '    indicator_sheet_name: str\n'
    '    rows_read: int\n'
    '    rows_calculated: int\n'
    '    output_range: str\n'
    '    first_date: str\n'
    '    last_date: str\n'
    '    nonblank_raw_values: int\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class BatchIndicatorSpec:\n'
    '    sheet_name: str\n'
    '    control_cell: str\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class BatchIndicatorResult:\n'
    '    sheet_name: str\n'
    '    ok: bool\n'
    '    output_range: str = ""\n'
    '    error: str = ""\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class IndicatorEngineSpec:\n'
    '    name: str\n'
    '    calculate: Callable[[Any], Any]\n'
    '    input_start_col: str = "A"\n'
    '    input_end_col: str = "B"\n'
    '    output_start_col: str = "C"\n'
    '    output_end_col: str = "AB"\n'
    '    output_columns: Sequence[str] = ()\n'
    '    summary_level_col: str | None = None\n'
    '    summary_leading_col: str | None = None\n'
    '    summary_warning_col: str | None = None\n'
    '    summary_status_col: str = "Data Status"\n'
    '    input_value_render_option: str | None = None\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class PmiLabelConfig:\n'
    '    regime_subject: str\n'
    '    interpretation_subject: str\n'
    '    warning_subject: str\n'
    '    signal_subject: str\n'
    '\n'
    '\n'
    'MANUFACTURING_PMI_LABELS = PmiLabelConfig(\n'
    '    regime_subject="",\n'
    '    interpretation_subject="",\n'
    '    warning_subject="",\n'
    '    signal_subject="manufacturing",\n'
    ')\n'
    '\n'
    'NEW_ORDERS_LABELS = PmiLabelConfig(\n'
    '    regime_subject="New orders",\n'
    '    interpretation_subject="new orders",\n'
    '    warning_subject="new orders",\n'
    '    signal_subject="new orders",\n'
    ')\n'
    '\n'
    'SERVICES_PMI_LABELS = PmiLabelConfig(\n'
    '    regime_subject="Services",\n'
    '    interpretation_subject="services",\n'
    '    warning_subject="services",\n'
    '    signal_subject="services",\n'
    ')\n'
    '\n'
    'SERVICES_NEW_ORDERS_LABELS = PmiLabelConfig(\n'
    '    regime_subject="Services new orders",\n'
    '    interpretation_subject="services new orders",\n'
    '    warning_subject="services new orders",\n'
    '    signal_subject="services new orders",\n'
    ')\n'
    '\n'
    'SERVICES_BUSINESS_ACTIVITY_LABELS = PmiLabelConfig(\n'
    '    regime_subject="Services business activity",\n'
    '    interpretation_subject="services business activity",\n'
    '    warning_subject="services business activity",\n'
    '    signal_subject="services business activity",\n'
    ')\n'
    '\n'
    '\n'
    'class SimpleDataFrame:\n'
    '    """Tiny fallback used only when pandas is unavailable in local self-tests."""\n'
    '\n'
    '    def __init__(\n'
    '        self, rows: list[dict[str, Any]], columns: Sequence[str] | None = None\n'
    '    ):\n'
    '        self._rows = rows\n'
    '        self.columns = list(columns or (rows[0].keys() if rows else []))\n'
    '\n'
    '    def __len__(self) -> int:\n'
    '        return len(self._rows)\n'
    '\n'
    '    def __getitem__(self, key: str) -> list[Any]:\n'
    '        return [r.get(key, "") for r in self._rows]\n'
    '\n'
    '    @property\n'
    '    def shape(self) -> tuple[int, int]:\n'
    '        return (len(self._rows), len(self.columns))\n'
    '\n'
    '    def to_rows(self) -> list[list[Any]]:\n'
    '        return [[r.get(c, "") for c in self.columns] for r in self._rows]\n'
    '\n'
    '    def row(self, idx: int) -> dict[str, Any]:\n'
    '        return self._rows[idx]\n'
    '\n'
    '\n'
    'try:\n'
    '    import pandas as pd  # type: ignore\n'
    'except Exception:  # pragma: no cover - exercised in this environment\n'
    '    pd = None  # type: ignore\n'
    '\n'
    '\n'
    'def make_dataframe(rows: list[dict[str, Any]], columns: Sequence[str]) -> Any:\n'
    '    return (\n'
    '        pd.DataFrame(rows, columns=columns)\n'
    '        if pd is not None\n'
    '        else SimpleDataFrame(rows, columns)\n'
    '    )\n'
    '\n'
    '\n'
    'def df_records(df: Any) -> list[dict[str, Any]]:\n'
    '    if hasattr(df, "to_dict"):\n'
    '        return list(df.to_dict("records"))\n'
    '    return list(df._rows)\n'
    '\n'
    '\n'
    'def is_blank(value: object) -> bool:\n'
    '    if value is None:\n'
    '        return True\n'
    '    if isinstance(value, str):\n'
    '        return value.strip() == ""\n'
    '    try:\n'
    '        return bool(math.isnan(value))  # type: ignore[arg-type]\n'
    '    except Exception:\n'
    '        return False\n'
    '\n'
    '\n'
    'def to_float_or_none(value: object) -> float | None:\n'
    '    if value is None or isinstance(value, bool):\n'
    '        return None\n'
    '    if isinstance(value, (int, float)):\n'
    '        try:\n'
    '            return (\n'
    '                None\n'
    '                if math.isnan(float(value)) or math.isinf(float(value))\n'
    '                else float(value)\n'
    '            )\n'
    '        except Exception:\n'
    '            return None\n'
    '    text = str(value).strip()\n'
    '    if text == "":\n'
    '        return None\n'
    '    if text.endswith("%"):\n'
    '        try:\n'
    '            return float(text[:-1].replace(",", "")) / 100.0\n'
    '        except ValueError:\n'
    '            return None\n'
    '    try:\n'
    '        val = float(text.replace(",", ""))\n'
    '    except ValueError:\n'
    '        return None\n'
    '    return None if math.isnan(val) or math.isinf(val) else val\n'
    '\n'
    '\n'
    'def sheet_blank(value: object = None) -> str:\n'
    '    return ""\n'
    '\n'
    '\n'
    'def to_date_or_none(value: object) -> date | None:\n'
    '    if is_blank(value) or isinstance(value, bool):\n'
    '        return None\n'
    '    if isinstance(value, datetime):\n'
    '        return value.date()\n'
    '    if isinstance(value, date):\n'
    '        return value\n'
    '    if isinstance(value, (int, float)):\n'
    '        try:\n'
    '            serial = float(value)\n'
    '            if math.isnan(serial) or math.isinf(serial):\n'
    '                return None\n'
    '            return (GOOGLE_SHEETS_DATE_EPOCH + timedelta(days=serial)).date()\n'
    '        except (OverflowError, ValueError):\n'
    '            return None\n'
    '    text = str(value).strip()\n'
    '    if not text:\n'
    '        return None\n'
    '    numeric = to_float_or_none(text)\n'
    '    if numeric is not None and text.replace(".", "", 1).replace("-", "", 1).isdigit():\n'
    '        return to_date_or_none(numeric)\n'
    '    for fmt in (\n'
    '        "%Y-%m-%d",\n'
    '        "%m/%d/%Y",\n'
    '        "%m/%d/%y",\n'
    '        "%Y/%m/%d",\n'
    '        "%d-%b-%Y",\n'
    '        "%b %d, %Y",\n'
    '        "%B %d, %Y",\n'
    '    ):\n'
    '        try:\n'
    '            return datetime.strptime(text, fmt).date()\n'
    '        except ValueError:\n'
    '            pass\n'
    '    try:\n'
    '        return datetime.fromisoformat(text).date()\n'
    '    except ValueError:\n'
    '        return None\n'
    '\n'
    '\n'
    'def clamp(value: float, low: float = -10.0, high: float = 10.0) -> float:\n'
    '    return max(low, min(high, value))\n'
    '\n'
    '\n'
    'def percentrank_inc(values: Sequence[float], x: float) -> float | None:\n'
    '    clean = sorted(v for v in (to_float_or_none(v) for v in values) if v is not None)\n'
    '    n = len(clean)\n'
    '    if n < 2 or x < clean[0] or x > clean[-1]:\n'
    '        return None\n'
    '    left = bisect.bisect_left(clean, x)\n'
    '    right = bisect.bisect_right(clean, x)\n'
    '    if left != right:\n'
    '        return 0.0 if n == 1 else left / (n - 1)\n'
    '    lo_i, hi_i = left - 1, left\n'
    '    lo, hi = clean[lo_i], clean[hi_i]\n'
    '    if hi == lo:\n'
    '        return lo_i / (n - 1)\n'
    '    return (lo_i + (x - lo) / (hi - lo)) / (n - 1)\n'
    '\n'
    '\n'
    'def _score_percent_rank(rank: Any) -> Any:\n'
    '    return "" if is_blank(rank) else round((float(rank) * 20) - 10, 2)\n'
    '\n'
    '\n'
    'def _early(b: float, g: float, h: float) -> int:\n'
    '    if b < 50 and g > 2 and h > 0:\n'
    '        return 8\n'
    '    if b < 50 and g > 0 and h > 0:\n'
    '        return 5\n'
    '    if b > 50 and g < -2 and h < 0:\n'
    '        return -8\n'
    '    if b > 50 and g < 0 and h < 0:\n'
    '        return -5\n'
    '    if b >= 50 and g > 0 and h > 0:\n'
    '        return 4\n'
    '    if b < 50 and g < 0 and h < 0:\n'
    '        return -6\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _threshold(b: float, prev_b: float, g: float, h: float) -> int:\n'
    '    if prev_b < 50 and b >= 50 and g > 0 and h > 0:\n'
    '        return 6\n'
    '    if prev_b >= 50 and b < 50 and g < 0 and h < 0:\n'
    '        return -6\n'
    '    if b >= 52 and g > 0 and h > 0:\n'
    '        return 3\n'
    '    if b <= 48 and g < 0 and h < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _overheat(b: float, g: float, h: float) -> int:\n'
    '    if b >= 55 and g < 0 and h < 0:\n'
    '        return -5\n'
    '    if b >= 55 and g < -3:\n'
    '        return -7\n'
    '    if b >= 52 and g < 0 and h < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _warning_raw(b: float, g: float, h: float) -> int:\n'
    '    if b >= 58 and g <= -3 and h <= -3:\n'
    '        return -8\n'
    '    if b >= 55 and g <= -2 and h <= -2:\n'
    '        return -6\n'
    '    if b >= 52 and g < 0 and h < 0:\n'
    '        return -3\n'
    '    if b < 50 and g > 3 and h > 0:\n'
    '        return 6\n'
    '    if b < 48 and g > 0 and h > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def calculate_pmi_style_outputs(raw_df: Any, label_config: PmiLabelConfig) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    sorted_b: list[float] = []\n'
    '    sorted_g: list[float] = []\n'
    '    sorted_h: list[float] = []\n'
    '    numeric_count = 0\n'
    '    for i, b in enumerate(b_values):\n'
    '        out: dict[str, Any] = {c: "" for c in OUTPUT_COLUMNS}\n'
    '        if b is not None:\n'
    '            numeric_count += 1\n'
    '            bisect.insort(sorted_b, b)\n'
    '            c = numeric_count\n'
    '            d = b - 50\n'
    '            e = round(clamp(d * 2), 2)\n'
    '            f = (\n'
    '                b - b_values[i - 1]\n'
    '                if c >= 2 and i >= 1 and b_values[i - 1] is not None\n'
    '                else ""\n'
    '            )\n'
    '            g = (\n'
    '                b - b_values[i - 3]\n'
    '                if c >= 4 and i >= 3 and b_values[i - 3] is not None\n'
    '                else ""\n'
    '            )\n'
    '            h = (\n'
    '                b - b_values[i - 6]\n'
    '                if c >= 7 and i >= 6 and b_values[i - 6] is not None\n'
    '                else ""\n'
    '            )\n'
    '            if g != "":\n'
    '                bisect.insort(sorted_g, float(g))\n'
    '            if h != "":\n'
    '                bisect.insort(sorted_h, float(h))\n'
    '            pr_b = percentrank_inc(sorted_b, b) if c >= 60 else None\n'
    '            pr_g = (\n'
    '                percentrank_inc(sorted_g, float(g))\n'
    '                if g != "" and len(sorted_g) >= 60\n'
    '                else None\n'
    '            )\n'
    '            pr_h = (\n'
    '                percentrank_inc(sorted_h, float(h))\n'
    '                if h != "" and len(sorted_h) >= 60\n'
    '                else None\n'
    '            )\n'
    '            j, l, n = (\n'
    '                _score_percent_rank(pr_b),\n'
    '                _score_percent_rank(pr_g),\n'
    '                _score_percent_rank(pr_h),\n'
    '            )\n'
    '            o = _early(b, float(g), float(h)) if g != "" and h != "" else ""\n'
    '            prev = b_values[i - 1] if i >= 1 else None\n'
    '            p = (\n'
    '                _threshold(b, prev, float(g), float(h))\n'
    '                if g != "" and h != "" and prev is not None\n'
    '                else ""\n'
    '            )\n'
    '            q = _overheat(b, float(g), float(h)) if g != "" and h != "" else ""\n'
    '            r = _warning_raw(b, float(g), float(h)) if g != "" and h != "" else ""\n'
    '            s = round(clamp(0.7 * j + 0.3 * e), 2) if j != "" else ""\n'
    '            t = (\n'
    '                round(clamp(0.25 * j + 0.45 * l + 0.3 * n), 2)\n'
    '                if j != "" and l != "" and n != ""\n'
    '                else ""\n'
    '            )\n'
    '            u = (\n'
    '                round(clamp(0.15 * o + 0.2 * p + 0.25 * q + 0.4 * r), 2)\n'
    '                if all(x != "" for x in (o, p, q, r))\n'
    '                else ""\n'
    '            )\n'
    '            out.update(\n'
    '                {\n'
    '                    "Observation Count": c,\n'
    '                    "Distance From 50": d,\n'
    '                    "Level Score Raw": e,\n'
    '                    "1M Change": f,\n'
    '                    "3M Change": g,\n'
    '                    "6M Change": h,\n'
    '                    "Level PercentRank": pr_b if pr_b is not None else "",\n'
    '                    "Level Score": j,\n'
    '                    "3M Change PercentRank": pr_g if pr_g is not None else "",\n'
    '                    "3M Momentum Score": l,\n'
    '                    "6M Change PercentRank": pr_h if pr_h is not None else "",\n'
    '                    "6M Momentum Score": n,\n'
    '                    "Early Cycle / Recovery Score": o,\n'
    '                    "Threshold Transition Score": p,\n'
    '                    "Overheating / Slowdown Score": q,\n'
    '                    "Warning Score Raw": r,\n'
    '                    "Level Composite Score": s,\n'
    '                    "Leading Score": t,\n'
    '                    "Warning Score": u,\n'
    '                    "Regime": "Expansion" if b >= 50 else "Contraction",\n'
    '                    "Reliability": (\n'
    '                        "Insufficient history"\n'
    '                        if c < 60\n'
    '                        else ("Medium reliability" if c < 120 else "High reliability")\n'
    '                    ),\n'
    '                    "Data Status": (\n'
    '                        "Low Sample"\n'
    '                        if c < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(x == "" for x in (s, t, u))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '            if l != "":\n'
    '                out["Regime Momentum Label"] = _regime_momentum_label(\n'
    '                    b, l, label_config\n'
    '                )\n'
    '            if t != "" and u != "":\n'
    '                out["Signal Interpretation"] = _signal(t, u, label_config)\n'
    '            if u != "":\n'
    '                out["Warning Risk Label"] = _warning_label(u, label_config)\n'
    '            if g != "" and h != "":\n'
    '                out["Manufacturing Signal Label"] = _manufacturing_signal_label(\n'
    '                    b, float(g), float(h), label_config\n'
    '                )\n'
    '        else:\n'
    '            out["Data Status"] = "Missing Data"\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def calculate_manufacturing_pmi_outputs(raw_df: Any) -> Any:\n'
    '    return calculate_pmi_style_outputs(raw_df, MANUFACTURING_PMI_LABELS)\n'
    '\n'
    '\n'
    'def _a4_percent_rank_rounded_3(history: Sequence[float], current: Any) -> Any:\n'
    '    current_float = to_float_or_none(current)\n'
    '    if current_float is None or len(history) < 60:\n'
    '        return ""\n'
    '    rank = percentrank_inc(history, current_float)\n'
    '    return "" if rank is None else round(rank, 3)\n'
    '\n'
    '\n'
    'def _a4_current_arithmetic_number(value: Any) -> float | None:\n'
    '    if is_blank(value):\n'
    '        return 0.0\n'
    '    return to_float_or_none(value)\n'
    '\n'
    '\n'
    'def _a4_lag_arithmetic_number(value: Any) -> float | None:\n'
    '    if is_blank(value):\n'
    '        return None\n'
    '    return to_float_or_none(value)\n'
    '\n'
    '\n'
    'def _a4_spreadsheet_subtract_current_minus_lag(\n'
    '    values: Sequence[Any], idx: int, lag: int\n'
    ') -> Any:\n'
    '    if idx - lag < 0:\n'
    '        return ""\n'
    '    current = _a4_current_arithmetic_number(values[idx])\n'
    '    lagged = _a4_lag_arithmetic_number(values[idx - lag])\n'
    '    if current is None or lagged is None:\n'
    '        return ""\n'
    '    return current - lagged\n'
    '\n'
    '\n'
    'def calculate_manufacturing_new_orders_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_source_values = [\n'
    '        r.get("Raw Value", r.get("B", r.get("Value", ""))) for r in records\n'
    '    ]\n'
    '    b_values = [to_float_or_none(value) for value in raw_source_values]\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    sorted_b: list[float] = []\n'
    '    sorted_g: list[float] = []\n'
    '    sorted_h: list[float] = []\n'
    '    numeric_count = 0\n'
    '    for i, b in enumerate(b_values):\n'
    '        out: dict[str, Any] = {c: "" for c in MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS}\n'
    '        if b is not None:\n'
    '            numeric_count += 1\n'
    '            bisect.insort(sorted_b, b)\n'
    '        f = _a4_spreadsheet_subtract_current_minus_lag(raw_source_values, i, 1)\n'
    '        g = _a4_spreadsheet_subtract_current_minus_lag(raw_source_values, i, 3)\n'
    '        h = _a4_spreadsheet_subtract_current_minus_lag(raw_source_values, i, 6)\n'
    '        if g != "":\n'
    '            bisect.insort(sorted_g, float(g))\n'
    '        if h != "":\n'
    '            bisect.insort(sorted_h, float(h))\n'
    '        k = _a4_percent_rank_rounded_3(sorted_g, g) if g != "" else ""\n'
    '        l = _score_percent_rank(k)\n'
    '        m = _a4_percent_rank_rounded_3(sorted_h, h) if h != "" else ""\n'
    '        n = _score_percent_rank(m)\n'
    '        if b is not None:\n'
    '            c = numeric_count\n'
    '            d = b - 50\n'
    '            e = round(clamp(d * 2), 2)\n'
    '            i_rank = _a4_percent_rank_rounded_3(sorted_b, b)\n'
    '            j = _score_percent_rank(i_rank)\n'
    '            o = _early(b, float(g), float(h)) if g != "" and h != "" else ""\n'
    '            prev = b_values[i - 1] if i >= 1 else None\n'
    '            p = (\n'
    '                _threshold(b, prev, float(g), float(h))\n'
    '                if g != "" and h != "" and prev is not None\n'
    '                else ""\n'
    '            )\n'
    '            q = _overheat(b, float(g), float(h)) if g != "" and h != "" else ""\n'
    '            r = _warning_raw(b, float(g), float(h)) if g != "" and h != "" else ""\n'
    '            s = round(clamp(0.7 * j + 0.3 * e), 2) if j != "" else ""\n'
    '            t = (\n'
    '                round(clamp(0.25 * j + 0.45 * l + 0.3 * n), 2)\n'
    '                if j != "" and l != "" and n != ""\n'
    '                else ""\n'
    '            )\n'
    '            u = (\n'
    '                round(clamp(0.15 * o + 0.2 * p + 0.25 * q + 0.4 * r), 2)\n'
    '                if all(x != "" for x in (o, p, q, r))\n'
    '                else ""\n'
    '            )\n'
    '            out.update(\n'
    '                {\n'
    '                    "Sample Count": c,\n'
    '                    "Distance from 50": d,\n'
    '                    "Distance from 50 Score": e,\n'
    '                    "1M Change": f,\n'
    '                    "3M Change": g,\n'
    '                    "6M Change": h,\n'
    '                    "Level Percentile": i_rank,\n'
    '                    "Level Score": j,\n'
    '                    "3M Momentum Percentile": k,\n'
    '                    "3M Momentum Score": l,\n'
    '                    "6M Momentum Percentile": m,\n'
    '                    "6M Momentum Score": n,\n'
    '                    "Turning Point Score": o,\n'
    '                    "50-Cross Signal": p,\n'
    '                    "Late-Cycle Warning": q,\n'
    '                    "Peak/Rollover Signal": r,\n'
    '                    "Coincident Score": s,\n'
    '                    "Leading Score": t,\n'
    '                    "Warning Score": u,\n'
    '                    "State": "Expansion" if b >= 50 else "Contraction",\n'
    '                    "Reliability": (\n'
    '                        "Insufficient history"\n'
    '                        if c < 60\n'
    '                        else ("Medium reliability" if c < 120 else "High reliability")\n'
    '                    ),\n'
    '                    "Signal Validity": (\n'
    '                        "Low Sample"\n'
    '                        if c < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(x == "" for x in (s, t, u))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '            if l != "":\n'
    '                out["Regime"] = _regime_momentum_label(b, l, NEW_ORDERS_LABELS)\n'
    '            if t != "" and u != "":\n'
    '                out["Signal Comment"] = _signal(t, u, NEW_ORDERS_LABELS)\n'
    '            if u != "":\n'
    '                out["Warning Flag"] = _warning_label(u, NEW_ORDERS_LABELS)\n'
    '            if g != "" and h != "":\n'
    '                out["Slowdown Risk Flag"] = _manufacturing_signal_label(\n'
    '                    b, float(g), float(h), NEW_ORDERS_LABELS\n'
    '                )\n'
    '        else:\n'
    '            out.update(\n'
    '                {\n'
    '                    "1M Change": f,\n'
    '                    "3M Change": g,\n'
    '                    "6M Change": h,\n'
    '                    "3M Momentum Percentile": k,\n'
    '                    "3M Momentum Score": l,\n'
    '                    "6M Momentum Percentile": m,\n'
    '                    "6M Momentum Score": n,\n'
    '                    "Signal Validity": "Missing Data",\n'
    '                }\n'
    '            )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def calculate_services_pmi_outputs(raw_df: Any) -> Any:\n'
    '    return calculate_pmi_style_outputs(raw_df, SERVICES_PMI_LABELS)\n'
    '\n'
    '\n'
    'def calculate_services_new_orders_outputs(raw_df: Any) -> Any:\n'
    '    return calculate_pmi_style_outputs(raw_df, SERVICES_NEW_ORDERS_LABELS)\n'
    '\n'
    '\n'
    'def calculate_services_business_activity_outputs(raw_df: Any) -> Any:\n'
    '    return calculate_pmi_style_outputs(raw_df, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '\n'
    '\n'
    'def calculate_pmi_indicator_outputs(raw_df: Any) -> Any:\n'
    '    return calculate_manufacturing_pmi_outputs(raw_df)\n'
    '\n'
    '\n'
    'def _subject_prefix(subject: str) -> str:\n'
    '    return f"{subject} " if subject else ""\n'
    '\n'
    '\n'
    'def _regime_momentum_label(b: float, l: float, config: PmiLabelConfig) -> str:\n'
    '    subject = _subject_prefix(config.regime_subject)\n'
    '    if b >= 50 and l >= 2:\n'
    '        return (\n'
    '            f"{subject}expansion accelerating".capitalize()\n'
    '            if not subject\n'
    '            else f"{subject}expansion accelerating"\n'
    '        )\n'
    '    if b >= 50 and l <= -2:\n'
    '        return (\n'
    '            f"{subject}expansion slowing".capitalize()\n'
    '            if not subject\n'
    '            else f"{subject}expansion slowing"\n'
    '        )\n'
    '    if b >= 50 and -2 < l < 2:\n'
    '        return (\n'
    '            f"{subject}expansion stable".capitalize()\n'
    '            if not subject\n'
    '            else f"{subject}expansion stable"\n'
    '        )\n'
    '    if b < 50 and l >= 2:\n'
    '        return (\n'
    '            f"{subject}contraction easing".capitalize()\n'
    '            if not subject\n'
    '            else f"{subject}contraction easing"\n'
    '        )\n'
    '    if b < 50 and l <= -2:\n'
    '        return (\n'
    '            f"{subject}contraction worsening".capitalize()\n'
    '            if not subject\n'
    '            else f"{subject}contraction worsening"\n'
    '        )\n'
    '    if b < 50 and -2 < l < 2:\n'
    '        return (\n'
    '            f"{subject}contraction stable".capitalize()\n'
    '            if not subject\n'
    '            else f"{subject}contraction stable"\n'
    '        )\n'
    '    return ""\n'
    '\n'
    '\n'
    'def _signal(\n'
    '    t: float, u: float, config: PmiLabelConfig = MANUFACTURING_PMI_LABELS\n'
    ') -> str:\n'
    '    subject = _subject_prefix(config.interpretation_subject)\n'
    '    mixed_subject = (\n'
    '        f" {config.interpretation_subject}" if config.interpretation_subject else ""\n'
    '    )\n'
    '    if t >= 6 and u >= 0:\n'
    '        return (\n'
    '            f"Strong positive {subject}leading signal with supportive warning profile"\n'
    '        )\n'
    '    if t >= 6 and u < 0:\n'
    '        return f"Strong positive {subject}leading signal, but warning risk is negative"\n'
    '    if t >= 2 and u >= 0:\n'
    '        return f"Moderately positive {subject}leading signal"\n'
    '    if t >= 2 and u < 0:\n'
    '        return f"Positive {subject}leading signal, but fragile"\n'
    '    if t > -2 and u > -3 and u < 3:\n'
    '        return f"Neutral/mixed{mixed_subject} signal"\n'
    '    if t <= -6 and u <= 0:\n'
    '        return f"Strong negative {subject}leading signal with downside warning confirmation"\n'
    '    if t <= -6 and u > 0:\n'
    '        return (\n'
    '            f"Strong negative {subject}leading signal, but warning profile is improving"\n'
    '        )\n'
    '    if t <= -2 and u <= 0:\n'
    '        return f"Moderately negative {subject}leading signal"\n'
    '    if t <= -2 and u > 0:\n'
    '        return f"Negative {subject}leading signal, but stabilisation risk is present"\n'
    '    return f"Mixed/transition{mixed_subject} signal"\n'
    '\n'
    '\n'
    'def _warning_label(u: float, config: PmiLabelConfig) -> str:\n'
    '    subject = _subject_prefix(config.warning_subject)\n'
    '    if u <= -6:\n'
    '        return f"High {subject}warning risk"\n'
    '    if u <= -3:\n'
    '        return f"Moderate {subject}warning risk"\n'
    '    if u < 3:\n'
    '        return f"Neutral {subject}warning profile"\n'
    '    if u < 6:\n'
    '        return f"Constructive {subject}warning profile"\n'
    '    return f"Strong positive {subject}turn / warning cleared"\n'
    '\n'
    '\n'
    'def _manufacturing_signal_label(\n'
    '    b: float, g: float, h: float, config: PmiLabelConfig\n'
    ') -> str:\n'
    '    if config is NEW_ORDERS_LABELS:\n'
    '        if b < 48 and g < 0 and h < 0:\n'
    '            return "High new orders contraction risk"\n'
    '        if b < 50 and g < 0:\n'
    '            return "New orders contraction worsening"\n'
    '        if b < 50 and g > 0:\n'
    '            return "New orders contraction easing"\n'
    '        if b > 50 and g < 0:\n'
    '            return "New orders expansion losing momentum"\n'
    '        if b > 50 and g > 0:\n'
    '            return "New orders expansion strengthening"\n'
    '        return "Mixed new orders signal"\n'
    '    if config is SERVICES_PMI_LABELS:\n'
    '        if b < 48 and g < 0 and h < 0:\n'
    '            return "High services contraction risk"\n'
    '        if b < 50 and g < 0:\n'
    '            return "Services contraction worsening"\n'
    '        if b < 50 and g > 0:\n'
    '            return "Services contraction easing"\n'
    '        if b > 50 and g < 0:\n'
    '            return "Services expansion losing momentum"\n'
    '        if b > 50 and g > 0:\n'
    '            return "Services expansion strengthening"\n'
    '        return "Mixed services signal"\n'
    '    if config is SERVICES_NEW_ORDERS_LABELS:\n'
    '        if b < 48 and g < 0 and h < 0:\n'
    '            return "High services new orders contraction risk"\n'
    '        if b < 50 and g < 0:\n'
    '            return "Services new orders contraction worsening"\n'
    '        if b < 50 and g > 0:\n'
    '            return "Services new orders contraction easing"\n'
    '        if b > 50 and g < 0:\n'
    '            return "Services new orders expansion losing momentum"\n'
    '        if b > 50 and g > 0:\n'
    '            return "Services new orders expansion strengthening"\n'
    '        return "Mixed services new orders signal"\n'
    '    if config is SERVICES_BUSINESS_ACTIVITY_LABELS:\n'
    '        if b < 48 and g < 0 and h < 0:\n'
    '            return "High services business activity contraction risk"\n'
    '        if b < 50 and g < 0:\n'
    '            return "Services business activity contraction worsening"\n'
    '        if b < 50 and g > 0:\n'
    '            return "Services business activity contraction easing"\n'
    '        if b > 50 and g < 0:\n'
    '            return "Services business activity expansion losing momentum"\n'
    '        if b > 50 and g > 0:\n'
    '            return "Services business activity expansion strengthening"\n'
    '        return "Mixed services business activity signal"\n'
    '    if b < 48 and g < 0 and h < 0:\n'
    '        return "High manufacturing recession risk"\n'
    '    if b < 50 and g < 0:\n'
    '        return "Manufacturing contraction worsening"\n'
    '    if b < 50 and g > 0:\n'
    '        return "Manufacturing contraction easing"\n'
    '    if b > 50 and g < 0:\n'
    '        return "Expansion losing momentum"\n'
    '    if b > 50 and g > 0:\n'
    '        return "Expansion strengthening"\n'
    '    return "Mixed manufacturing signal"\n'
    '\n'
    '\n'
    'def _manufacturing(b: float, g: float, h: float) -> str:\n'
    '    return _manufacturing_signal_label(b, g, h, MANUFACTURING_PMI_LABELS)\n'
    '\n'
    '\n'
    'def _safe_growth(current: Any, lagged: Any, exponent: float = 1.0) -> Any:\n'
    '    c = to_float_or_none(current)\n'
    '    l = to_float_or_none(lagged)\n'
    '    if c is None or l is None or l == 0:\n'
    '        return ""\n'
    '    ratio = c / l\n'
    '    if ratio <= 0:\n'
    '        return ""\n'
    '    return (ratio**exponent) - 1\n'
    '\n'
    '\n'
    'def _industrial_recovery_score(h: Any, i: Any, j: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (h, i, j)):\n'
    '        return ""\n'
    '    if h < 0 and i > 0 and j > 0:\n'
    '        return 6\n'
    '    if h < 0 and i < 0 and j < 0:\n'
    '        return -6\n'
    '    if h > 0 and i < 0 and j < 0:\n'
    '        return -5\n'
    '    if h > 0 and i > 0 and j > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _industrial_warning_raw(h: Any, i: Any, j: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (h, i, j)):\n'
    '        return ""\n'
    '    if h < 0 and i < 0 and j < 0:\n'
    '        return -8\n'
    '    if h < 0 and i < 0:\n'
    '        return -6\n'
    '    if h >= 0 and i < 0 and j < 0:\n'
    '        return -5\n'
    '    if h >= 0 and i < 0:\n'
    '        return -3\n'
    '    if h < 0 and i > 0 and j > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def expanding_zscore_sample(\n'
    '    values: list[object], gates: list[int] | None = None, min_gate: int = 120\n'
    ') -> list[object]:\n'
    '    out: list[object] = []\n'
    '    n = 0\n'
    '    mean = 0.0\n'
    '    m2 = 0.0\n'
    '    for idx, value in enumerate(values):\n'
    '        x = to_float_or_none(value)\n'
    '        if x is not None:\n'
    '            n += 1\n'
    '            delta = x - mean\n'
    '            mean += delta / n\n'
    '            m2 += delta * (x - mean)\n'
    '        gate_ok = gates is None or gates[idx] >= min_gate\n'
    '        if x is None or not gate_ok or n < 2:\n'
    '            out.append("")\n'
    '            continue\n'
    '        variance = m2 / (n - 1)\n'
    '        if variance <= 0:\n'
    '            out.append("")\n'
    '        else:\n'
    '            out.append((x - mean) / math.sqrt(variance))\n'
    '    return out\n'
    '\n'
    '\n'
    'def _industrial_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 3\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 2\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _industrial_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme positive outlier"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme negative outlier"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong positive outlier"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong negative outlier"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _industrial_regime_momentum_label(h: Any, r: Any) -> str:\n'
    '    if any(is_blank(x) for x in (h, r)):\n'
    '        return ""\n'
    '    if h >= 0 and r >= 2:\n'
    '        return "Industrial expansion accelerating"\n'
    '    if h >= 0 and r <= -2:\n'
    '        return "Industrial expansion slowing"\n'
    '    if h >= 0 and -2 < r < 2:\n'
    '        return "Industrial expansion stable"\n'
    '    if h < 0 and r >= 2:\n'
    '        return "Industrial contraction easing"\n'
    '    if h < 0 and r <= -2:\n'
    '        return "Industrial contraction worsening"\n'
    '    if h < 0 and -2 < r < 2:\n'
    '        return "Industrial contraction stable"\n'
    '    return ""\n'
    '\n'
    '\n'
    'def _industrial_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return "Strong positive industrial production leading signal with supportive warning profile"\n'
    '    if w >= 6 and x < 0:\n'
    '        return "Strong positive industrial production leading signal, but warning risk is negative"\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive industrial production leading signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Positive industrial production leading signal, but fragile"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed industrial production signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong negative industrial production leading signal with downside warning confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong negative industrial production leading signal, but warning profile is improving"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately negative industrial production leading signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return "Negative industrial production leading signal, but stabilisation risk is present"\n'
    '    return "Mixed/transition industrial production signal"\n'
    '\n'
    '\n'
    'def _industrial_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x <= -6:\n'
    '        return "High industrial production warning risk"\n'
    '    if x <= -3:\n'
    '        return "Moderate industrial production warning risk"\n'
    '    if x < 3:\n'
    '        return "Neutral industrial production warning profile"\n'
    '    if x < 6:\n'
    '        return "Constructive industrial production warning profile"\n'
    '    return "Strong positive industrial production turn / warning cleared"\n'
    '\n'
    '\n'
    'def _industrial_signal_label(h: Any, r: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (h, r, x)):\n'
    '        return ""\n'
    '    if h < 0 and x <= -6:\n'
    '        return "High industrial slowdown/recession risk"\n'
    '    if h < 0 and x <= -3:\n'
    '        return "Industrial contraction risk"\n'
    '    if h < 0 and r >= 2:\n'
    '        return "Industrial contraction easing / recovery impulse"\n'
    '    if h >= 0 and x <= -3:\n'
    '        return "Industrial expansion fragile / downside risk"\n'
    '    if h >= 0 and r <= -2:\n'
    '        return "Industrial expansion losing momentum"\n'
    '    if h >= 0 and r >= 2:\n'
    '        return "Industrial expansion strengthening"\n'
    '    return "Neutral industrial production signal"\n'
    '\n'
    '\n'
    'def _industrial_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    number = to_float_or_none(value)\n'
    '    if number is None:\n'
    '        return ""\n'
    '    return f"{number * 100:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _industrial_percentile_display(value: Any) -> Any:\n'
    '    return _industrial_percent_display(value, 2)\n'
    '\n'
    '\n'
    'def _industrial_percent_rank_rounded_3(\n'
    '    history: Sequence[float], current: Any, sample_count: int\n'
    ') -> Any:\n'
    '    current_float = to_float_or_none(current)\n'
    '    if sample_count < 120 or current_float is None:\n'
    '        return ""\n'
    '    rank = percentrank_inc(history, current_float)\n'
    '    return "" if rank is None else round(rank, 3)\n'
    '\n'
    '\n'
    'def _industrial_downside_warning_score(l: Any, n: Any, p: Any, r: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (l, n, p, r)):\n'
    '        return ""\n'
    '    return round(clamp(0.40 * l + 0.25 * n + 0.20 * p + 0.15 * r), 2)\n'
    '\n'
    '\n'
    'def calculate_industrial_production_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals: list[Any] = []\n'
    '    d_vals: list[int] = []\n'
    '    numeric_b_count = 0\n'
    '    numeric_c_count = 0\n'
    '    for i, val in enumerate(b):\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        window = [x for x in b[max(0, i - 2) : i + 1] if x is not None]\n'
    '        c = (sum(window) / len(window)) if numeric_b_count >= 3 and window else ""\n'
    '        c_vals.append(c)\n'
    '        if c != "":\n'
    '            numeric_c_count += 1\n'
    '        d_vals.append(numeric_c_count)\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    j_vals = []\n'
    '    for idx, c in enumerate(c_vals):\n'
    '        d = d_vals[idx]\n'
    '        e_vals.append(_safe_growth(c, c_vals[idx - 1]) if d >= 2 and idx >= 1 else "")\n'
    '        f_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 3], 4) if d >= 4 and idx >= 3 else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 6], 2) if d >= 7 and idx >= 6 else ""\n'
    '        )\n'
    '        h_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 12]) if d >= 13 and idx >= 12 else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 3]\n'
    '            if d >= 16 and idx >= 3 and h_vals[idx] != "" and h_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        j_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 6]\n'
    '            if d >= 19 and idx >= 6 and h_vals[idx] != "" and h_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    af_vals = expanding_zscore_sample(h_vals, d_vals, 120)\n'
    '    ag_vals = expanding_zscore_sample(f_vals, d_vals, 120)\n'
    '    ah_vals = expanding_zscore_sample(g_vals, d_vals, 120)\n'
    '    ai_vals = expanding_zscore_sample(i_vals, d_vals, 120)\n'
    '    sorted_h: list[float] = []\n'
    '    sorted_f: list[float] = []\n'
    '    sorted_g: list[float] = []\n'
    '    sorted_i: list[float] = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS}\n'
    '        h = h_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        g = g_vals[idx]\n'
    '        i_ch = i_vals[idx]\n'
    '        for val, arr in ((h, sorted_h), (f, sorted_f), (g, sorted_g), (i_ch, sorted_i)):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        d = d_vals[idx]\n'
    '        k = _industrial_percent_rank_rounded_3(sorted_h, h, d)\n'
    '        m = _industrial_percent_rank_rounded_3(sorted_f, f, d)\n'
    '        o = _industrial_percent_rank_rounded_3(sorted_g, g, d)\n'
    '        q = _industrial_percent_rank_rounded_3(sorted_i, i_ch, d)\n'
    '        l = _score_percent_rank(k)\n'
    '        n = _score_percent_rank(m)\n'
    '        p = _score_percent_rank(o)\n'
    '        r = _score_percent_rank(q)\n'
    '        s = _industrial_recovery_score(h, i_ch, j_vals[idx])\n'
    '        t = _industrial_downside_warning_score(l, n, p, r)\n'
    '        zs = [af_vals[idx], ag_vals[idx], ah_vals[idx], ai_vals[idx]]\n'
    '        u = _industrial_outlier_score(zs)\n'
    '        v = (\n'
    '            round(clamp(0.45 * l + 0.25 * n + 0.2 * p + 0.1 * r), 2)\n'
    '            if all(x != "" for x in (l, n, p, r))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            round(clamp(0.1 * l + 0.25 * n + 0.15 * p + 0.45 * r + 0.05 * s), 2)\n'
    '            if all(x != "" for x in (l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.3 * s + 0.5 * t + 0.2 * u), 2)\n'
    '            if all(z != "" for z in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        out.update(\n'
    '            {\n'
    '                "3M MA": c_vals[idx],\n'
    '                "Sample Count": d,\n'
    '                "1M % Change": e_vals[idx],\n'
    '                "3M Annualised Growth": f,\n'
    '                "6M Annualised Growth": g,\n'
    '                "YoY Growth": h,\n'
    '                "3M Change in YoY Growth": _industrial_percent_display(i_ch),\n'
    '                "6M Change in YoY Growth": _industrial_percent_display(j_vals[idx]),\n'
    '                "YoY Growth Percentile": _industrial_percentile_display(k),\n'
    '                "YoY Growth Score": l,\n'
    '                "3M Growth Percentile": _industrial_percentile_display(m),\n'
    '                "3M Growth Score": n,\n'
    '                "6M Growth Percentile": _industrial_percentile_display(o),\n'
    '                "6M Growth Score": p,\n'
    '                "Momentum Percentile": _industrial_percentile_display(q),\n'
    '                "Momentum Score": r,\n'
    '                "Industrial Cycle Turning Point Score": s,\n'
    '                "Industrial Downside Warning Score": t,\n'
    '                "Industrial Extreme Score": u,\n'
    '                "Coincident Score": v,\n'
    '                "Leading Score": w,\n'
    '                "Warning Score": x,\n'
    '                "State": "" if h == "" else ("Expansion" if h >= 0 else "Contraction"),\n'
    '                "Regime": _industrial_regime_momentum_label(h, r),\n'
    '                "Signal Comment": _industrial_signal_interpretation(w, x),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if d < 60\n'
    '                        else ("Medium reliability" if d < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Signal Validity": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if d < 120\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(z == "" for z in (v, w, x))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Warning Flag": _industrial_warning_label(x),\n'
    '                "Slowdown Risk Flag": _industrial_signal_label(h, r, x),\n'
    '                "YoY Growth Z-Score": af_vals[idx],\n'
    '                "3M Growth Z-Score": ag_vals[idx],\n'
    '                "6M Growth Z-Score": ah_vals[idx],\n'
    '                "Momentum Z-Score": ai_vals[idx],\n'
    '                "Extreme Flag": _industrial_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _metric_counts(values: Sequence[Any]) -> list[int]:\n'
    '    out = []\n'
    '    count = 0\n'
    '    for v in values:\n'
    '        if to_float_or_none(v) is not None:\n'
    '            count += 1\n'
    '        out.append(count)\n'
    '    return out\n'
    '\n'
    '\n'
    'def _consumer_regime_momentum_label(h: Any, r: Any) -> str:\n'
    '    if any(is_blank(x) for x in (h, r)):\n'
    '        return ""\n'
    '    if h >= 0 and r >= 2:\n'
    '        return "Consumer spending expansion accelerating"\n'
    '    if h >= 0 and r <= -2:\n'
    '        return "Consumer spending expansion slowing"\n'
    '    if h >= 0 and -2 < r < 2:\n'
    '        return "Consumer spending expansion stable"\n'
    '    if h < 0 and r >= 2:\n'
    '        return "Consumer spending contraction easing"\n'
    '    if h < 0 and r <= -2:\n'
    '        return "Consumer spending contraction worsening"\n'
    '    if h < 0 and -2 < r < 2:\n'
    '        return "Consumer spending contraction stable"\n'
    '    return ""\n'
    '\n'
    '\n'
    'def _consumer_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return "Strong positive consumer spending leading signal with supportive warning profile"\n'
    '    if w >= 6 and x < 0:\n'
    '        return "Strong positive consumer spending leading signal, but warning risk is negative"\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive consumer spending leading signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Positive consumer spending leading signal, but fragile"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed consumer spending signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong negative consumer spending leading signal with downside warning confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong negative consumer spending leading signal, but warning profile is improving"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately negative consumer spending leading signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return "Negative consumer spending leading signal, but stabilisation risk is present"\n'
    '    return "Mixed/transition consumer spending signal"\n'
    '\n'
    '\n'
    'def _consumer_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x <= -6:\n'
    '        return "High consumer spending warning risk"\n'
    '    if x <= -3:\n'
    '        return "Moderate consumer spending warning risk"\n'
    '    if x < 3:\n'
    '        return "Neutral consumer spending warning profile"\n'
    '    if x < 6:\n'
    '        return "Constructive consumer spending warning profile"\n'
    '    return "Strong positive consumer spending turn / warning cleared"\n'
    '\n'
    '\n'
    'def _consumer_signal_label(h: Any, r: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (h, r, x)):\n'
    '        return ""\n'
    '    if h < 0 and x <= -6:\n'
    '        return "High consumer slowdown/recession risk"\n'
    '    if h < 0 and x <= -3:\n'
    '        return "Consumer spending contraction risk"\n'
    '    if h < 0 and r >= 2:\n'
    '        return "Consumer spending contraction easing / recovery impulse"\n'
    '    if h >= 0 and x <= -3:\n'
    '        return "Consumer spending expansion fragile / downside risk"\n'
    '    if h >= 0 and r <= -2:\n'
    '        return "Consumer spending losing momentum"\n'
    '    if h >= 0 and r >= 2:\n'
    '        return "Consumer spending strengthening"\n'
    '    return "Neutral consumer spending signal"\n'
    '\n'
    '\n'
    'def calculate_consumer_spending_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals = []\n'
    '    d_vals = []\n'
    '    numeric_b_count = 0\n'
    '    numeric_c_count = 0\n'
    '    for idx, val in enumerate(b):\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        window = [x for x in b[max(0, idx - 2) : idx + 1] if x is not None]\n'
    '        c = (sum(window) / len(window)) if numeric_b_count >= 3 and window else ""\n'
    '        c_vals.append(c)\n'
    '        if c != "":\n'
    '            numeric_c_count += 1\n'
    '        d_vals.append(numeric_c_count)\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    j_vals = []\n'
    '    for idx, c in enumerate(c_vals):\n'
    '        d = d_vals[idx]\n'
    '        e_vals.append(_safe_growth(c, c_vals[idx - 1]) if d >= 2 and idx >= 1 else "")\n'
    '        f_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 3], 4) if d >= 4 and idx >= 3 else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 6], 2) if d >= 7 and idx >= 6 else ""\n'
    '        )\n'
    '        h_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 12]) if d >= 13 and idx >= 12 else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 3]\n'
    '            if d >= 16 and idx >= 3 and h_vals[idx] != "" and h_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        j_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 6]\n'
    '            if d >= 19 and idx >= 6 and h_vals[idx] != "" and h_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    h_counts = _metric_counts(h_vals)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    i_counts = _metric_counts(i_vals)\n'
    '    af_vals = expanding_zscore_sample(h_vals, h_counts, 60)\n'
    '    ag_vals = expanding_zscore_sample(f_vals, f_counts, 60)\n'
    '    ah_vals = expanding_zscore_sample(g_vals, g_counts, 60)\n'
    '    ai_vals = expanding_zscore_sample(i_vals, i_counts, 60)\n'
    '    sorted_h = []\n'
    '    sorted_f = []\n'
    '    sorted_g = []\n'
    '    sorted_i = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in CONSUMER_SPENDING_OUTPUT_COLUMNS}\n'
    '        h = h_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        g = g_vals[idx]\n'
    '        i_ch = i_vals[idx]\n'
    '        for val, arr in ((h, sorted_h), (f, sorted_f), (g, sorted_g), (i_ch, sorted_i)):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        k = (\n'
    '            percentrank_inc(sorted_h, float(h))\n'
    '            if h_counts[idx] >= 60 and h != ""\n'
    '            else None\n'
    '        )\n'
    '        m = (\n'
    '            percentrank_inc(sorted_f, float(f))\n'
    '            if f_counts[idx] >= 60 and f != ""\n'
    '            else None\n'
    '        )\n'
    '        o = (\n'
    '            percentrank_inc(sorted_g, float(g))\n'
    '            if g_counts[idx] >= 60 and g != ""\n'
    '            else None\n'
    '        )\n'
    '        q = (\n'
    '            percentrank_inc(sorted_i, float(i_ch))\n'
    '            if i_counts[idx] >= 60 and i_ch != ""\n'
    '            else None\n'
    '        )\n'
    '        l = _score_percent_rank(k)\n'
    '        n = _score_percent_rank(m)\n'
    '        p = _score_percent_rank(o)\n'
    '        r = _score_percent_rank(q)\n'
    '        s = _industrial_recovery_score(h, i_ch, j_vals[idx])\n'
    '        t = _industrial_warning_raw(h, i_ch, j_vals[idx])\n'
    '        zs = [af_vals[idx], ag_vals[idx], ah_vals[idx], ai_vals[idx]]\n'
    '        u = _industrial_outlier_score(zs)\n'
    '        v = (\n'
    '            round(clamp(0.45 * l + 0.25 * n + 0.2 * p + 0.1 * r), 2)\n'
    '            if all(x != "" for x in (l, n, p, r))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            round(clamp(0.1 * l + 0.25 * n + 0.15 * p + 0.45 * r + 0.05 * s), 2)\n'
    '            if all(x != "" for x in (l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.3 * s + 0.5 * t + 0.2 * u), 2)\n'
    '            if all(z != "" for z in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        d = d_vals[idx]\n'
    '        out.update(\n'
    '            {\n'
    '                "3M Moving Average": c_vals[idx],\n'
    '                "Smoothed Observation Count": d,\n'
    '                "1M Smoothed Growth": e_vals[idx],\n'
    '                "3M Annualised Smoothed Growth": f,\n'
    '                "6M Annualised Smoothed Growth": g,\n'
    '                "12M Smoothed Growth": h,\n'
    '                "3M Change in 12M Growth": i_ch,\n'
    '                "6M Change in 12M Growth": j_vals[idx],\n'
    '                "12M Growth PercentRank": k if k is not None else "",\n'
    '                "12M Growth Score": l,\n'
    '                "3M Annualised Growth PercentRank": m if m is not None else "",\n'
    '                "3M Annualised Growth Score": n,\n'
    '                "6M Annualised Growth PercentRank": o if o is not None else "",\n'
    '                "6M Annualised Growth Score": p,\n'
    '                "3M Change in 12M Growth PercentRank": q if q is not None else "",\n'
    '                "3M Change in 12M Growth Score": r,\n'
    '                "Consumer Spending Recovery / Momentum Score": s,\n'
    '                "Consumer Spending Warning Score Raw": t,\n'
    '                "Consumer Spending Outlier Score": u,\n'
    '                "Consumer Spending Leading Score": v,\n'
    '                "Consumer Spending Warning Composite Score": w,\n'
    '                "Consumer Spending Final Warning Score": x,\n'
    '                "Consumer Spending Regime": (\n'
    '                    "" if h == "" else ("Expansion" if h >= 0 else "Contraction")\n'
    '                ),\n'
    '                "Consumer Spending Regime Momentum Label": _consumer_regime_momentum_label(\n'
    '                    h, r\n'
    '                ),\n'
    '                "Consumer Spending Signal Interpretation": _consumer_signal_interpretation(\n'
    '                    w, x\n'
    '                ),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if d < 60\n'
    '                        else ("Medium reliability" if d < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Data Status": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if d < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(z == "" for z in (v, w, x))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Consumer Spending Warning Risk Label": _consumer_warning_label(x),\n'
    '                "Consumer Spending Signal Label": _consumer_signal_label(h, r, x),\n'
    '                "12M Growth Z-Score": af_vals[idx],\n'
    '                "3M Annualised Growth Z-Score": ag_vals[idx],\n'
    '                "6M Annualised Growth Z-Score": ah_vals[idx],\n'
    '                "3M Change in 12M Growth Z-Score": ai_vals[idx],\n'
    '                "Consumer Spending Outlier Label": _industrial_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, CONSUMER_SPENDING_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _median_from_sorted(values: Sequence[float]) -> float:\n'
    '    n = len(values)\n'
    '    mid = n // 2\n'
    '    return values[mid] if n % 2 else (values[mid - 1] + values[mid]) / 2\n'
    '\n'
    '\n'
    'def _sentiment_recovery_score(distance: Any, three_month: Any, six_month: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (distance, three_month, six_month)):\n'
    '        return ""\n'
    '    if distance < 0 and three_month > 0 and six_month > 0:\n'
    '        return 6\n'
    '    if distance < 0 and three_month < 0 and six_month < 0:\n'
    '        return -6\n'
    '    if distance > 0 and three_month < 0 and six_month < 0:\n'
    '        return -5\n'
    '    if distance > 0 and three_month > 0 and six_month > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _sentiment_turn_score(\n'
    '    b: Any, level_rank: Any, three_month: Any, six_month: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (b, level_rank, three_month, six_month)):\n'
    '        return ""\n'
    '    if level_rank <= 0.2 and three_month > 0 and six_month > 0:\n'
    '        return 5\n'
    '    if level_rank <= 0.1 and three_month > 0:\n'
    '        return 4\n'
    '    if level_rank >= 0.8 and three_month < 0 and six_month < 0:\n'
    '        return -4\n'
    '    if level_rank >= 0.9 and three_month < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _sentiment_warning_raw(\n'
    '    distance: Any,\n'
    '    three_month: Any,\n'
    '    six_month: Any,\n'
    '    level_score: Any,\n'
    '    momentum_score: Any,\n'
    ') -> Any:\n'
    '    if any(\n'
    '        is_blank(x)\n'
    '        for x in (distance, three_month, six_month, level_score, momentum_score)\n'
    '    ):\n'
    '        return ""\n'
    '    if level_score <= -7 and momentum_score <= -2:\n'
    '        return -8\n'
    '    if level_score <= -3 and momentum_score <= -2:\n'
    '        return -6\n'
    '    if -3 < level_score < 3 and momentum_score <= -5:\n'
    '        return -4\n'
    '    if level_score >= 3 and momentum_score <= -2:\n'
    '        return -3\n'
    '    if level_score <= -3 and momentum_score >= 2:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _sentiment_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 3\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 2\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _sentiment_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if not zs or is_blank(zs[0]):\n'
    '        return ""\n'
    '    numeric = [z for z in zs if not is_blank(z)]\n'
    '    if any(z >= 3 for z in numeric):\n'
    '        return "Extreme positive outlier"\n'
    '    if any(z <= -3 for z in numeric):\n'
    '        return "Extreme negative outlier"\n'
    '    if any(z >= 2 for z in numeric):\n'
    '        return "Strong positive outlier"\n'
    '    if any(z <= -2 for z in numeric):\n'
    '        return "Strong negative outlier"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _sentiment_regime(level_score: Any) -> str:\n'
    '    if is_blank(level_score):\n'
    '        return ""\n'
    '    if level_score >= 3:\n'
    '        return "Strong sentiment"\n'
    '    if level_score <= -3:\n'
    '        return "Weak sentiment"\n'
    '    return "Neutral sentiment"\n'
    '\n'
    '\n'
    'def _sentiment_momentum_label(level_score: Any, momentum_score: Any) -> str:\n'
    '    if any(is_blank(x) for x in (level_score, momentum_score)):\n'
    '        return ""\n'
    '    if level_score >= 3 and momentum_score >= 2:\n'
    '        return "Strong sentiment, improving"\n'
    '    if level_score >= 3 and momentum_score <= -2:\n'
    '        return "Strong sentiment, weakening"\n'
    '    if level_score >= 3 and -2 < momentum_score < 2:\n'
    '        return "Strong sentiment, stable"\n'
    '    if level_score <= -3 and momentum_score >= 2:\n'
    '        return "Weak sentiment, improving"\n'
    '    if level_score <= -3 and momentum_score <= -2:\n'
    '        return "Weak sentiment, deteriorating"\n'
    '    if level_score <= -3 and -2 < momentum_score < 2:\n'
    '        return "Weak sentiment, stable"\n'
    '    if -3 < level_score < 3 and momentum_score >= 2:\n'
    '        return "Neutral sentiment, improving"\n'
    '    if -3 < level_score < 3 and momentum_score <= -2:\n'
    '        return "Neutral sentiment, weakening"\n'
    '    return "Neutral sentiment, stable"\n'
    '\n'
    '\n'
    'def _sentiment_signal_interpretation(leading_score: Any, warning_score: Any) -> str:\n'
    '    if any(is_blank(x) for x in (leading_score, warning_score)):\n'
    '        return ""\n'
    '    if leading_score >= 6 and warning_score >= 0:\n'
    '        return "Strong positive consumer sentiment leading signal with supportive warning profile"\n'
    '    if leading_score >= 6 and warning_score < 0:\n'
    '        return "Strong positive consumer sentiment leading signal, but warning risk is negative"\n'
    '    if leading_score >= 2 and warning_score >= 0:\n'
    '        return "Moderately positive consumer sentiment leading signal"\n'
    '    if leading_score >= 2 and warning_score < 0:\n'
    '        return "Positive consumer sentiment leading signal, but fragile"\n'
    '    if leading_score > -2 and warning_score > -3 and warning_score < 3:\n'
    '        return "Neutral/mixed consumer sentiment signal"\n'
    '    if leading_score <= -6 and warning_score <= 0:\n'
    '        return "Strong negative consumer sentiment leading signal with downside warning confirmation"\n'
    '    if leading_score <= -6 and warning_score > 0:\n'
    '        return "Strong negative consumer sentiment leading signal, but warning profile is improving"\n'
    '    if leading_score <= -2 and warning_score <= 0:\n'
    '        return "Moderately negative consumer sentiment leading signal"\n'
    '    if leading_score <= -2 and warning_score > 0:\n'
    '        return "Negative consumer sentiment leading signal, but stabilisation risk is present"\n'
    '    return "Mixed/transition consumer sentiment signal"\n'
    '\n'
    '\n'
    'def _sentiment_warning_label(warning_score: Any) -> str:\n'
    '    if is_blank(warning_score):\n'
    '        return ""\n'
    '    if warning_score <= -6:\n'
    '        return "High consumer sentiment warning risk"\n'
    '    if warning_score <= -3:\n'
    '        return "Moderate consumer sentiment warning risk"\n'
    '    if warning_score < 3:\n'
    '        return "Neutral consumer sentiment warning profile"\n'
    '    if warning_score < 6:\n'
    '        return "Constructive consumer sentiment warning profile"\n'
    '    return "Strong positive consumer sentiment turn / warning cleared"\n'
    '\n'
    '\n'
    'def _sentiment_signal_label(level_score: Any, momentum_score: Any) -> str:\n'
    '    if any(is_blank(x) for x in (level_score, momentum_score)):\n'
    '        return ""\n'
    '    if level_score <= -7 and momentum_score <= -2:\n'
    '        return "Severe consumer confidence stress"\n'
    '    if level_score <= -3 and momentum_score <= -2:\n'
    '        return "Consumer confidence deteriorating"\n'
    '    if level_score <= -3 and momentum_score >= 2:\n'
    '        return "Weak sentiment but improving"\n'
    '    if -3 < level_score < 3 and momentum_score <= -5:\n'
    '        return "Neutral sentiment deteriorating sharply"\n'
    '    if -3 < level_score < 3 and momentum_score >= 5:\n'
    '        return "Neutral sentiment improving sharply"\n'
    '    if level_score >= 3 and momentum_score <= -2:\n'
    '        return "Strong sentiment losing momentum"\n'
    '    if level_score >= 3 and momentum_score >= 2:\n'
    '        return "Consumer confidence strengthening"\n'
    '    return "Neutral consumer sentiment signal"\n'
    '\n'
    '\n'
    'def calculate_consumer_sentiment_legacy_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals: list[Any] = []\n'
    '    d_vals: list[Any] = []\n'
    '    e_vals: list[Any] = []\n'
    '    f_vals: list[Any] = []\n'
    '    g_vals: list[Any] = []\n'
    '    sorted_b: list[float] = []\n'
    '    numeric_b_count = 0\n'
    '    for idx, b in enumerate(b_values):\n'
    '        if b is None:\n'
    '            c_vals.append("")\n'
    '            d_vals.append("")\n'
    '            e_vals.append("")\n'
    '            f_vals.append("")\n'
    '            g_vals.append("")\n'
    '            continue\n'
    '        numeric_b_count += 1\n'
    '        bisect.insort(sorted_b, b)\n'
    '        c = numeric_b_count\n'
    '        c_vals.append(c)\n'
    '        d_vals.append(b - _median_from_sorted(sorted_b) if c >= 60 else "")\n'
    '        e_vals.append(\n'
    '            b - b_values[idx - 1]\n'
    '            if c >= 2 and idx >= 1 and b_values[idx - 1] is not None\n'
    '            else ""\n'
    '        )\n'
    '        f_vals.append(\n'
    '            b - b_values[idx - 3]\n'
    '            if c >= 4 and idx >= 3 and b_values[idx - 3] is not None\n'
    '            else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            b - b_values[idx - 6]\n'
    '            if c >= 7 and idx >= 6 and b_values[idx - 6] is not None\n'
    '            else ""\n'
    '        )\n'
    '\n'
    '    ab_vals = expanding_zscore_sample(\n'
    '        b_values, [c if c != "" else 0 for c in c_vals], 60\n'
    '    )\n'
    '    ac_vals = expanding_zscore_sample(f_vals, [c if c != "" else 0 for c in c_vals], 60)\n'
    '    ad_vals = expanding_zscore_sample(g_vals, [c if c != "" else 0 for c in c_vals], 60)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    sorted_level: list[float] = []\n'
    '    sorted_f: list[float] = []\n'
    '    sorted_g: list[float] = []\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, b in enumerate(b_values):\n'
    '        out = {col: "" for col in CONSUMER_SENTIMENT_LEGACY_OUTPUT_COLUMNS}\n'
    '        if b is None:\n'
    '            out["Data Status"] = "Missing Data"\n'
    '            rows.append(out)\n'
    '            continue\n'
    '        bisect.insort(sorted_level, b)\n'
    '        if f_vals[idx] != "":\n'
    '            bisect.insort(sorted_f, float(f_vals[idx]))\n'
    '        if g_vals[idx] != "":\n'
    '            bisect.insort(sorted_g, float(g_vals[idx]))\n'
    '        c = c_vals[idx]\n'
    '        h = percentrank_inc(sorted_level, b) if c >= 60 else None\n'
    '        j = (\n'
    '            percentrank_inc(sorted_f, float(f_vals[idx]))\n'
    '            if f_counts[idx] >= 60 and f_vals[idx] != ""\n'
    '            else None\n'
    '        )\n'
    '        l = (\n'
    '            percentrank_inc(sorted_g, float(g_vals[idx]))\n'
    '            if g_counts[idx] >= 60 and g_vals[idx] != ""\n'
    '            else None\n'
    '        )\n'
    '        i_score = _sentiment_score_percent_rank(h)\n'
    '        k_score = _sentiment_score_percent_rank(j)\n'
    '        m_score = _sentiment_score_percent_rank(l)\n'
    '        n_score = _sentiment_recovery_score(d_vals[idx], f_vals[idx], g_vals[idx])\n'
    '        o_score = _sentiment_turn_score(\n'
    '            b, h if h is not None else "", f_vals[idx], g_vals[idx]\n'
    '        )\n'
    '        p_score = _sentiment_warning_raw(\n'
    '            d_vals[idx], f_vals[idx], g_vals[idx], i_score, k_score\n'
    '        )\n'
    '        zs = [ab_vals[idx], ac_vals[idx], ad_vals[idx]]\n'
    '        q_score = _sentiment_outlier_score(zs)\n'
    '        r_score = (\n'
    '            round(clamp(0.75 * i_score + 0.25 * clamp(d_vals[idx] / 5)), 2)\n'
    '            if all(x != "" for x in (i_score, d_vals[idx]))\n'
    '            else ""\n'
    '        )\n'
    '        s_score = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.2 * i_score\n'
    '                    + 0.4 * k_score\n'
    '                    + 0.25 * m_score\n'
    '                    + 0.05 * n_score\n'
    '                    + 0.1 * o_score\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(x != "" for x in (i_score, k_score, m_score, n_score, o_score))\n'
    '            else ""\n'
    '        )\n'
    '        t_score = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.25 * n_score + 0.25 * o_score + 0.35 * p_score + 0.15 * q_score\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(x != "" for x in (n_score, o_score, p_score, q_score))\n'
    '            else ""\n'
    '        )\n'
    '        out.update(\n'
    '            {\n'
    '                "Observation Count": c,\n'
    '                "Distance From Rolling Median": d_vals[idx],\n'
    '                "1M Change": e_vals[idx],\n'
    '                "3M Change": f_vals[idx],\n'
    '                "6M Change": g_vals[idx],\n'
    '                "Level PercentRank": h if h is not None else "",\n'
    '                "Sentiment Level Score": i_score,\n'
    '                "3M Change PercentRank": j if j is not None else "",\n'
    '                "3M Momentum Score": k_score,\n'
    '                "6M Change PercentRank": l if l is not None else "",\n'
    '                "6M Momentum Score": m_score,\n'
    '                "Consumer Sentiment Recovery / Momentum Score": n_score,\n'
    '                "Consumer Sentiment Low/High Turn Score": o_score,\n'
    '                "Consumer Sentiment Warning Score Raw": p_score,\n'
    '                "Consumer Sentiment Outlier Score": q_score,\n'
    '                "Consumer Sentiment Level Composite Score": r_score,\n'
    '                "Consumer Sentiment Leading Score": s_score,\n'
    '                "Consumer Sentiment Final Warning Score": t_score,\n'
    '                "Consumer Sentiment Regime": _sentiment_regime(i_score),\n'
    '                "Consumer Sentiment Momentum Label": _sentiment_momentum_label(\n'
    '                    i_score, k_score\n'
    '                ),\n'
    '                "Consumer Sentiment Signal Interpretation": _sentiment_signal_interpretation(\n'
    '                    s_score, t_score\n'
    '                ),\n'
    '                "Reliability": (\n'
    '                    "Insufficient history"\n'
    '                    if c < 60\n'
    '                    else ("Medium reliability" if c < 120 else "High reliability")\n'
    '                ),\n'
    '                "Data Status": (\n'
    '                    "Low Sample"\n'
    '                    if c < 60\n'
    '                    else (\n'
    '                        "Incomplete Score"\n'
    '                        if any(x == "" for x in (r_score, s_score, t_score))\n'
    '                        else "Valid"\n'
    '                    )\n'
    '                ),\n'
    '                "Consumer Sentiment Warning Risk Label": _sentiment_warning_label(\n'
    '                    t_score\n'
    '                ),\n'
    '                "Consumer Sentiment Signal Label": _sentiment_signal_label(\n'
    '                    i_score, k_score\n'
    '                ),\n'
    '                "Level Z-Score": ab_vals[idx],\n'
    '                "3M Change Z-Score": ac_vals[idx],\n'
    '                "6M Change Z-Score": ad_vals[idx],\n'
    '                "Consumer Sentiment Outlier Label": _sentiment_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, CONSUMER_SENTIMENT_LEGACY_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _sentiment_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    if is_blank(value):\n'
    '        return ""\n'
    '    number = to_float_or_none(value)\n'
    '    if number is None:\n'
    '        return ""\n'
    '    try:\n'
    '        quantized = Decimal(str(round(number, 12))).quantize(\n'
    '            Decimal("1").scaleb(-places), rounding=ROUND_HALF_UP\n'
    '        )\n'
    '    except Exception:\n'
    '        return ""\n'
    '    return float(quantized)\n'
    '\n'
    '\n'
    'def _sentiment_percent_rank_rounded_3(history: Sequence[float], current: Any) -> Any:\n'
    '    current_float = to_float_or_none(current)\n'
    '    if current_float is None or len(history) < 60:\n'
    '        return ""\n'
    '    rank = percentrank_inc(history, current_float)\n'
    '    return _sentiment_sheets_round(rank, 3)\n'
    '\n'
    '\n'
    'def _sentiment_score_percent_rank(rank: Any) -> Any:\n'
    '    if is_blank(rank):\n'
    '        return ""\n'
    '    return _sentiment_sheets_round(rank * 20 - 10, 2)\n'
    '\n'
    '\n'
    'def _sentiment_percent_display(value: Any) -> Any:\n'
    '    number = to_float_or_none(value)\n'
    '    if number is None:\n'
    '        return ""\n'
    '    return f"{number * 100:.2f}%"\n'
    '\n'
    '\n'
    'def calculate_consumer_sentiment_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals: list[Any] = []\n'
    '    d_vals: list[Any] = []\n'
    '    e_vals: list[Any] = []\n'
    '    f_vals: list[Any] = []\n'
    '    g_vals: list[Any] = []\n'
    '    sorted_b: list[float] = []\n'
    '    numeric_b_count = 0\n'
    '    for idx, b in enumerate(b_values):\n'
    '        if b is None:\n'
    '            c_vals.append("")\n'
    '            d_vals.append("")\n'
    '            e_vals.append("")\n'
    '            f_vals.append("")\n'
    '            g_vals.append("")\n'
    '            continue\n'
    '        numeric_b_count += 1\n'
    '        bisect.insort(sorted_b, b)\n'
    '        c = numeric_b_count\n'
    '        c_vals.append(c)\n'
    '        d_vals.append(b - _median_from_sorted(sorted_b) if c >= 60 else "")\n'
    '        e_vals.append(\n'
    '            b - b_values[idx - 1]\n'
    '            if c >= 2 and idx >= 1 and b_values[idx - 1] is not None\n'
    '            else ""\n'
    '        )\n'
    '        f_vals.append(\n'
    '            b - b_values[idx - 3]\n'
    '            if c >= 4 and idx >= 3 and b_values[idx - 3] is not None\n'
    '            else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            b - b_values[idx - 6]\n'
    '            if c >= 7 and idx >= 6 and b_values[idx - 6] is not None\n'
    '            else ""\n'
    '        )\n'
    '\n'
    '    ab_vals = expanding_zscore_sample(\n'
    '        b_values, [c if c != "" else 0 for c in c_vals], 60\n'
    '    )\n'
    '    ac_vals = expanding_zscore_sample(f_vals, [c if c != "" else 0 for c in c_vals], 60)\n'
    '    ad_vals = expanding_zscore_sample(g_vals, [c if c != "" else 0 for c in c_vals], 60)\n'
    '    sorted_level: list[float] = []\n'
    '    sorted_f: list[float] = []\n'
    '    sorted_g: list[float] = []\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, b in enumerate(b_values):\n'
    '        out = {col: "" for col in CONSUMER_SENTIMENT_OUTPUT_COLUMNS}\n'
    '        if b is None:\n'
    '            out["Signal Validity"] = "Missing Data"\n'
    '            rows.append(out)\n'
    '            continue\n'
    '        bisect.insort(sorted_level, b)\n'
    '        if f_vals[idx] != "":\n'
    '            bisect.insort(sorted_f, float(f_vals[idx]))\n'
    '        if g_vals[idx] != "":\n'
    '            bisect.insort(sorted_g, float(g_vals[idx]))\n'
    '        c = c_vals[idx]\n'
    '        h = _sentiment_percent_rank_rounded_3(sorted_level, b)\n'
    '        j = (\n'
    '            _sentiment_percent_rank_rounded_3(sorted_f, f_vals[idx])\n'
    '            if f_vals[idx] != ""\n'
    '            else ""\n'
    '        )\n'
    '        l = (\n'
    '            _sentiment_percent_rank_rounded_3(sorted_g, g_vals[idx])\n'
    '            if g_vals[idx] != ""\n'
    '            else ""\n'
    '        )\n'
    '        i_score = _sentiment_score_percent_rank(h)\n'
    '        k_score = _sentiment_score_percent_rank(j)\n'
    '        m_score = _sentiment_score_percent_rank(l)\n'
    '        n_score = _sentiment_recovery_score(d_vals[idx], f_vals[idx], g_vals[idx])\n'
    '        o_score = _sentiment_turn_score(b, h, f_vals[idx], g_vals[idx])\n'
    '        p_score = _sentiment_warning_raw(\n'
    '            d_vals[idx], f_vals[idx], g_vals[idx], i_score, k_score\n'
    '        )\n'
    '        zs = [ab_vals[idx], ac_vals[idx], ad_vals[idx]]\n'
    '        q_score = _sentiment_outlier_score(zs)\n'
    '        r_score = (\n'
    '            _sentiment_sheets_round(clamp(0.75 * i_score + 0.05 * d_vals[idx]), 2)\n'
    '            if all(x != "" for x in (i_score, d_vals[idx]))\n'
    '            else ""\n'
    '        )\n'
    '        s_score = (\n'
    '            _sentiment_sheets_round(\n'
    '                clamp(\n'
    '                    0.20 * i_score\n'
    '                    + 0.40 * k_score\n'
    '                    + 0.25 * m_score\n'
    '                    + 0.05 * n_score\n'
    '                    + 0.10 * o_score\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(x != "" for x in (i_score, k_score, m_score, n_score, o_score))\n'
    '            else ""\n'
    '        )\n'
    '        t_score = (\n'
    '            _sentiment_sheets_round(\n'
    '                clamp(\n'
    '                    0.25 * n_score + 0.25 * o_score + 0.35 * p_score + 0.15 * q_score\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(x != "" for x in (n_score, o_score, p_score, q_score))\n'
    '            else ""\n'
    '        )\n'
    '        out.update(\n'
    '            {\n'
    '                "Sample Count": c,\n'
    '                "Distance from historical median": d_vals[idx],\n'
    '                "1M Change": e_vals[idx],\n'
    '                "3M Change": f_vals[idx],\n'
    '                "6M Change": g_vals[idx],\n'
    '                "Level Percentile": h,\n'
    '                "Level Score": i_score,\n'
    '                "3M Momentum Percentile": j,\n'
    '                "3M Momentum Score": k_score,\n'
    '                "6M Momentum Percentile": l,\n'
    '                "6M Momentum Score": m_score,\n'
    '                "Sentiment Cycle Turning Point Score": n_score,\n'
    '                "Contrarian Recovery Score": o_score,\n'
    '                "Sentiment Downside Warning Score": p_score,\n'
    '                "Sentiment Extreme Score": q_score,\n'
    '                "Coincident Score": r_score,\n'
    '                "Leading Score": s_score,\n'
    '                "Warning Score": t_score,\n'
    '                "State": _sentiment_regime(i_score),\n'
    '                "Regime": _sentiment_momentum_label(i_score, k_score),\n'
    '                "Signal Comment": _sentiment_signal_interpretation(s_score, t_score),\n'
    '                "Reliability": (\n'
    '                    "Insufficient history"\n'
    '                    if c < 60\n'
    '                    else ("Medium reliability" if c < 120 else "High reliability")\n'
    '                ),\n'
    '                "Signal Validity": (\n'
    '                    "Low Sample"\n'
    '                    if c < 60\n'
    '                    else (\n'
    '                        "Incomplete Score"\n'
    '                        if any(x == "" for x in (r_score, s_score, t_score))\n'
    '                        else "Valid"\n'
    '                    )\n'
    '                ),\n'
    '                "Warning Flag": _sentiment_warning_label(t_score),\n'
    '                "Slowdown Risk Flag": _sentiment_signal_label(i_score, k_score),\n'
    '                "Level Z-Score": ab_vals[idx],\n'
    '                "3M Change Z-Score": ac_vals[idx],\n'
    '                "6M Change Z-Score": ad_vals[idx],\n'
    '                "Extreme Flag": _sentiment_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, CONSUMER_SENTIMENT_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _expectations_regime(level_score: Any) -> str:\n'
    '    if is_blank(level_score):\n'
    '        return ""\n'
    '    if level_score >= 3:\n'
    '        return "Strong expectations"\n'
    '    if level_score <= -3:\n'
    '        return "Weak expectations"\n'
    '    return "Neutral expectations"\n'
    '\n'
    '\n'
    'def _expectations_momentum_label(level_score: Any, momentum_score: Any) -> str:\n'
    '    if any(is_blank(x) for x in (level_score, momentum_score)):\n'
    '        return ""\n'
    '    if level_score >= 3 and momentum_score >= 2:\n'
    '        return "Strong expectations, improving"\n'
    '    if level_score >= 3 and momentum_score <= -2:\n'
    '        return "Strong expectations, weakening"\n'
    '    if level_score >= 3 and -2 < momentum_score < 2:\n'
    '        return "Strong expectations, stable"\n'
    '    if level_score <= -3 and momentum_score >= 2:\n'
    '        return "Weak expectations, improving"\n'
    '    if level_score <= -3 and momentum_score <= -2:\n'
    '        return "Weak expectations, deteriorating"\n'
    '    if level_score <= -3 and -2 < momentum_score < 2:\n'
    '        return "Weak expectations, stable"\n'
    '    if -3 < level_score < 3 and momentum_score >= 2:\n'
    '        return "Neutral expectations, improving"\n'
    '    if -3 < level_score < 3 and momentum_score <= -2:\n'
    '        return "Neutral expectations, weakening"\n'
    '    return "Neutral expectations, stable"\n'
    '\n'
    '\n'
    'def _expectations_signal_interpretation(leading_score: Any, warning_score: Any) -> str:\n'
    '    if any(is_blank(x) for x in (leading_score, warning_score)):\n'
    '        return ""\n'
    '    if leading_score >= 6 and warning_score >= 0:\n'
    '        return "Strong positive consumer expectations leading signal with supportive warning profile"\n'
    '    if leading_score >= 6 and warning_score < 0:\n'
    '        return "Strong positive consumer expectations leading signal, but warning risk is negative"\n'
    '    if leading_score >= 2 and warning_score >= 0:\n'
    '        return "Moderately positive consumer expectations leading signal"\n'
    '    if leading_score >= 2 and warning_score < 0:\n'
    '        return "Positive consumer expectations leading signal, but fragile"\n'
    '    if leading_score > -2 and warning_score > -3 and warning_score < 3:\n'
    '        return "Neutral/mixed consumer expectations signal"\n'
    '    if leading_score <= -6 and warning_score <= 0:\n'
    '        return "Strong negative consumer expectations leading signal with downside warning confirmation"\n'
    '    if leading_score <= -6 and warning_score > 0:\n'
    '        return "Strong negative consumer expectations leading signal, but warning profile is improving"\n'
    '    if leading_score <= -2 and warning_score <= 0:\n'
    '        return "Moderately negative consumer expectations leading signal"\n'
    '    if leading_score <= -2 and warning_score > 0:\n'
    '        return "Negative consumer expectations leading signal, but stabilisation risk is present"\n'
    '    return "Mixed/transition consumer expectations signal"\n'
    '\n'
    '\n'
    'def _expectations_warning_label(warning_score: Any) -> str:\n'
    '    if is_blank(warning_score):\n'
    '        return ""\n'
    '    if warning_score <= -6:\n'
    '        return "High consumer expectations warning risk"\n'
    '    if warning_score <= -3:\n'
    '        return "Moderate consumer expectations warning risk"\n'
    '    if warning_score < 3:\n'
    '        return "Neutral consumer expectations warning profile"\n'
    '    if warning_score < 6:\n'
    '        return "Constructive consumer expectations warning profile"\n'
    '    return "Strong positive consumer expectations turn / warning cleared"\n'
    '\n'
    '\n'
    'def _expectations_signal_label(level_score: Any, momentum_score: Any) -> str:\n'
    '    if any(is_blank(x) for x in (level_score, momentum_score)):\n'
    '        return ""\n'
    '    if level_score <= -7 and momentum_score <= -2:\n'
    '        return "Severe consumer expectations stress"\n'
    '    if level_score <= -3 and momentum_score <= -2:\n'
    '        return "Consumer expectations deteriorating"\n'
    '    if level_score <= -3 and momentum_score >= 2:\n'
    '        return "Weak expectations but improving"\n'
    '    if -3 < level_score < 3 and momentum_score <= -5:\n'
    '        return "Neutral expectations deteriorating sharply"\n'
    '    if -3 < level_score < 3 and momentum_score >= 5:\n'
    '        return "Neutral expectations improving sharply"\n'
    '    if level_score >= 3 and momentum_score <= -2:\n'
    '        return "Strong expectations losing momentum"\n'
    '    if level_score >= 3 and momentum_score >= 2:\n'
    '        return "Consumer expectations strengthening"\n'
    '    return "Neutral consumer expectations signal"\n'
    '\n'
    '\n'
    'def calculate_consumer_expectations_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals: list[Any] = []\n'
    '    d_vals: list[Any] = []\n'
    '    e_vals: list[Any] = []\n'
    '    f_vals: list[Any] = []\n'
    '    g_vals: list[Any] = []\n'
    '    sorted_b: list[float] = []\n'
    '    numeric_b_count = 0\n'
    '    for idx, b in enumerate(b_values):\n'
    '        if b is None:\n'
    '            c_vals.append("")\n'
    '            d_vals.append("")\n'
    '            e_vals.append("")\n'
    '            f_vals.append("")\n'
    '            g_vals.append("")\n'
    '            continue\n'
    '        numeric_b_count += 1\n'
    '        bisect.insort(sorted_b, b)\n'
    '        c = numeric_b_count\n'
    '        c_vals.append(c)\n'
    '        d_vals.append(b - _median_from_sorted(sorted_b) if c >= 60 else "")\n'
    '        e_vals.append(\n'
    '            b - b_values[idx - 1]\n'
    '            if c >= 2 and idx >= 1 and b_values[idx - 1] is not None\n'
    '            else ""\n'
    '        )\n'
    '        f_vals.append(\n'
    '            b - b_values[idx - 3]\n'
    '            if c >= 4 and idx >= 3 and b_values[idx - 3] is not None\n'
    '            else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            b - b_values[idx - 6]\n'
    '            if c >= 7 and idx >= 6 and b_values[idx - 6] is not None\n'
    '            else ""\n'
    '        )\n'
    '\n'
    '    ab_vals = expanding_zscore_sample(\n'
    '        b_values, [c if c != "" else 0 for c in c_vals], 60\n'
    '    )\n'
    '    ac_vals = expanding_zscore_sample(f_vals, [c if c != "" else 0 for c in c_vals], 60)\n'
    '    ad_vals = expanding_zscore_sample(g_vals, [c if c != "" else 0 for c in c_vals], 60)\n'
    '    sorted_level: list[float] = []\n'
    '    sorted_f: list[float] = []\n'
    '    sorted_g: list[float] = []\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, b in enumerate(b_values):\n'
    '        out = {col: "" for col in CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS}\n'
    '        if b is None:\n'
    '            out["Signal Validity"] = "Missing Data"\n'
    '            rows.append(out)\n'
    '            continue\n'
    '        bisect.insort(sorted_level, b)\n'
    '        if f_vals[idx] != "":\n'
    '            bisect.insort(sorted_f, float(f_vals[idx]))\n'
    '        if g_vals[idx] != "":\n'
    '            bisect.insort(sorted_g, float(g_vals[idx]))\n'
    '        c = c_vals[idx]\n'
    '        h = _sentiment_percent_rank_rounded_3(sorted_level, b)\n'
    '        j = (\n'
    '            _sentiment_percent_rank_rounded_3(sorted_f, f_vals[idx])\n'
    '            if f_vals[idx] != ""\n'
    '            else ""\n'
    '        )\n'
    '        l = (\n'
    '            _sentiment_percent_rank_rounded_3(sorted_g, g_vals[idx])\n'
    '            if g_vals[idx] != ""\n'
    '            else ""\n'
    '        )\n'
    '        i_score = _sentiment_score_percent_rank(h)\n'
    '        k_score = _sentiment_score_percent_rank(j)\n'
    '        m_score = _sentiment_score_percent_rank(l)\n'
    '        n_score = _sentiment_recovery_score(d_vals[idx], f_vals[idx], g_vals[idx])\n'
    '        o_score = _sentiment_turn_score(b, h, f_vals[idx], g_vals[idx])\n'
    '        p_score = _sentiment_warning_raw(\n'
    '            d_vals[idx], f_vals[idx], g_vals[idx], i_score, k_score\n'
    '        )\n'
    '        zs = [ab_vals[idx], ac_vals[idx], ad_vals[idx]]\n'
    '        q_score = _sentiment_outlier_score(zs)\n'
    '        r_score = (\n'
    '            _sentiment_sheets_round(clamp(0.75 * i_score + 0.05 * d_vals[idx]), 2)\n'
    '            if all(x != "" for x in (i_score, d_vals[idx]))\n'
    '            else ""\n'
    '        )\n'
    '        s_score = (\n'
    '            _sentiment_sheets_round(\n'
    '                clamp(\n'
    '                    0.20 * i_score\n'
    '                    + 0.40 * k_score\n'
    '                    + 0.25 * m_score\n'
    '                    + 0.05 * n_score\n'
    '                    + 0.10 * o_score\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(x != "" for x in (i_score, k_score, m_score, n_score, o_score))\n'
    '            else ""\n'
    '        )\n'
    '        t_score = (\n'
    '            _sentiment_sheets_round(\n'
    '                clamp(\n'
    '                    0.25 * n_score + 0.25 * o_score + 0.35 * p_score + 0.15 * q_score\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(x != "" for x in (n_score, o_score, p_score, q_score))\n'
    '            else ""\n'
    '        )\n'
    '        out.update(\n'
    '            {\n'
    '                "Sample Count": c,\n'
    '                "Distance from historical median": d_vals[idx],\n'
    '                "1M Change": e_vals[idx],\n'
    '                "3M Change": f_vals[idx],\n'
    '                "6M Change": g_vals[idx],\n'
    '                "Level Percentile": h,\n'
    '                "Level Score": i_score,\n'
    '                "3M Momentum Percentile": j,\n'
    '                "3M Momentum Score": k_score,\n'
    '                "6M Momentum Percentile": l,\n'
    '                "6M Momentum Score": m_score,\n'
    '                "Expectations Cycle Turning Point Score": n_score,\n'
    '                "Expectations Contrarian Recovery Score": o_score,\n'
    '                "Expectations Downside Warning Score": p_score,\n'
    '                "Expectations Extreme Score": q_score,\n'
    '                "Coincident Score": r_score,\n'
    '                "Leading Score": s_score,\n'
    '                "Warning Score": t_score,\n'
    '                "State": _expectations_regime(i_score),\n'
    '                "Regime": _expectations_momentum_label(i_score, k_score),\n'
    '                "Signal Comment": _expectations_signal_interpretation(s_score, t_score),\n'
    '                "Reliability": (\n'
    '                    "Insufficient history"\n'
    '                    if c < 60\n'
    '                    else ("Medium reliability" if c < 120 else "High reliability")\n'
    '                ),\n'
    '                "Signal Validity": (\n'
    '                    "Low Sample"\n'
    '                    if c < 60\n'
    '                    else (\n'
    '                        "Incomplete Score"\n'
    '                        if any(x == "" for x in (r_score, s_score, t_score))\n'
    '                        else "Valid"\n'
    '                    )\n'
    '                ),\n'
    '                "Warning Flag": _expectations_warning_label(t_score),\n'
    '                "Slowdown Risk Flag": _expectations_signal_label(i_score, k_score),\n'
    '                "Level Z-Score": ab_vals[idx],\n'
    '                "3M Change Z-Score": ac_vals[idx],\n'
    '                "6M Change Z-Score": ad_vals[idx],\n'
    '                "Extreme Flag": _sentiment_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _inverted_score_percent_rank(rank: Any) -> Any:\n'
    '    return "" if is_blank(rank) else round(10 - (float(rank) * 20), 2)\n'
    '\n'
    '\n'
    'def expanding_inverted_zscore_sample(\n'
    '    values: list[object], gates: list[int] | None = None, min_gate: int = 60\n'
    ') -> list[object]:\n'
    '    out: list[object] = []\n'
    '    n = 0\n'
    '    mean = 0.0\n'
    '    m2 = 0.0\n'
    '    for idx, value in enumerate(values):\n'
    '        x = to_float_or_none(value)\n'
    '        if x is not None:\n'
    '            n += 1\n'
    '            delta = x - mean\n'
    '            mean += delta / n\n'
    '            m2 += delta * (x - mean)\n'
    '        gate_ok = gates is None or gates[idx] >= min_gate\n'
    '        if x is None or not gate_ok or n < 2:\n'
    '            out.append("")\n'
    '            continue\n'
    '        variance = m2 / (n - 1)\n'
    '        out.append("" if variance <= 0 else round((mean - x) / math.sqrt(variance), 2))\n'
    '    return out\n'
    '\n'
    '\n'
    'def _claims_recovery_score(h: Any, i: Any, j: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (h, i, j)):\n'
    '        return ""\n'
    '    if h > 0 and i < 0 and j < 0:\n'
    '        return 6\n'
    '    if h > 0 and i > 0 and j > 0:\n'
    '        return -6\n'
    '    if h < 0 and i > 0 and j > 0:\n'
    '        return -5\n'
    '    if h < 0 and i < 0 and j < 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _claims_surge_warning(e: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (e, f, g)):\n'
    '        return ""\n'
    '    if e > 0.05 and f > 0.20:\n'
    '        return -6\n'
    '    if e > 0.03 and f > 0.10:\n'
    '        return -4\n'
    '    if e < -0.05 and f < 0 and g < 0:\n'
    '        return 6\n'
    '    if e < -0.03 and f < 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _claims_regime(v: Any) -> str:\n'
    '    if is_blank(v):\n'
    '        return ""\n'
    '    if v >= 2:\n'
    '        return "Labour market improving"\n'
    '    if v <= -2:\n'
    '        return "Labour market weakening"\n'
    '    return "Neutral labour market"\n'
    '\n'
    '\n'
    'def _claims_momentum_label(v: Any, w: Any) -> str:\n'
    '    if any(is_blank(x) for x in (v, w)):\n'
    '        return ""\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Claims falling / labour market strengthening"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Claims low but deterioration risk rising"\n'
    '    if v >= 2 and -2 < w < 2:\n'
    '        return "Claims low / labour market stable"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Claims elevated but improving"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Claims rising / labour market deteriorating"\n'
    '    if v <= -2 and -2 < w < 2:\n'
    '        return "Claims elevated / labour market weak"\n'
    '    if -2 < v < 2 and w >= 2:\n'
    '        return "Claims trend improving"\n'
    '    if -2 < v < 2 and w <= -2:\n'
    '        return "Claims trend weakening"\n'
    '    return "Claims broadly neutral"\n'
    '\n'
    '\n'
    'def _claims_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return "Strong positive initial claims leading signal with supportive warning profile"\n'
    '    if w >= 6 and x < 0:\n'
    '        return "Strong positive initial claims leading signal, but warning risk is negative"\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive initial claims leading signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Positive initial claims leading signal, but fragile"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed initial claims signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong negative initial claims leading signal with downside warning confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong negative initial claims leading signal, but warning profile is improving"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately negative initial claims leading signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return (\n'
    '            "Negative initial claims leading signal, but stabilisation risk is present"\n'
    '        )\n'
    '    return "Mixed/transition initial claims signal"\n'
    '\n'
    '\n'
    'def _claims_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x <= -6:\n'
    '        return "High initial claims warning risk"\n'
    '    if x <= -3:\n'
    '        return "Moderate initial claims warning risk"\n'
    '    if x < 3:\n'
    '        return "Neutral initial claims warning profile"\n'
    '    if x < 6:\n'
    '        return "Constructive initial claims warning profile"\n'
    '    return "Strong positive initial claims turn / warning cleared"\n'
    '\n'
    '\n'
    'def _claims_signal_label(v: Any, w: Any, x: Any) -> str:\n'
    '    if any(is_blank(z) for z in (v, w, x)):\n'
    '        return ""\n'
    '    if x <= -6 and w <= -2:\n'
    '        return "Severe labour market deterioration risk"\n'
    '    if x <= -3 and w <= -2:\n'
    '        return "Initial claims deterioration warning"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Claims elevated but improving"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Claims low but deterioration starting"\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Claims falling / labour market strengthening"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Claims rising / labour market weakening"\n'
    '    return "Neutral initial claims signal"\n'
    '\n'
    '\n'
    'def _claims_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if not zs or is_blank(zs[0]):\n'
    '        return ""\n'
    '    vals = [z for z in zs if not is_blank(z)]\n'
    '    if min(vals) <= -2.5:\n'
    '        return "Extreme labour deterioration outlier"\n'
    '    if min(vals) <= -2:\n'
    '        return "Strong labour deterioration outlier"\n'
    '    if max(vals) >= 2.5:\n'
    '        return "Extreme labour improvement outlier"\n'
    '    if max(vals) >= 2:\n'
    '        return "Strong labour improvement outlier"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_initial_claims_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals = []\n'
    '    d_vals = []\n'
    '    numeric_b_count = 0\n'
    '    numeric_c_count = 0\n'
    '    for idx, val in enumerate(b):\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        window = [x for x in b[max(0, idx - 2) : idx + 1] if x is not None]\n'
    '        c = (sum(window) / len(window)) if numeric_b_count >= 3 and window else ""\n'
    '        c_vals.append(c)\n'
    '        if c != "":\n'
    '            numeric_c_count += 1\n'
    '        d_vals.append(numeric_c_count)\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    j_vals = []\n'
    '    for idx, c in enumerate(c_vals):\n'
    '        d = d_vals[idx]\n'
    '        e_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 1]) if d >= 2 and idx >= 1 and c != "" else ""\n'
    '        )\n'
    '        f_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 3], 4)\n'
    '            if d >= 4 and idx >= 3 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 6], 2)\n'
    '            if d >= 7 and idx >= 6 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        h_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 12])\n'
    '            if d >= 13 and idx >= 12 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 3]\n'
    '            if d >= 16 and idx >= 3 and h_vals[idx] != "" and h_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        j_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 6]\n'
    '            if d >= 19 and idx >= 6 and h_vals[idx] != "" and h_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    h_counts = _metric_counts(h_vals)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    i_counts = _metric_counts(i_vals)\n'
    '    af_vals = expanding_inverted_zscore_sample(h_vals, h_counts, 60)\n'
    '    ag_vals = expanding_inverted_zscore_sample(f_vals, f_counts, 60)\n'
    '    ah_vals = expanding_inverted_zscore_sample(g_vals, g_counts, 60)\n'
    '    ai_vals = expanding_inverted_zscore_sample(i_vals, i_counts, 60)\n'
    '    sorted_h = []\n'
    '    sorted_f = []\n'
    '    sorted_g = []\n'
    '    sorted_i = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in INITIAL_CLAIMS_OUTPUT_COLUMNS}\n'
    '        h = h_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        g = g_vals[idx]\n'
    '        i_ch = i_vals[idx]\n'
    '        for val, arr in ((h, sorted_h), (f, sorted_f), (g, sorted_g), (i_ch, sorted_i)):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        k = (\n'
    '            percentrank_inc(sorted_h, float(h))\n'
    '            if h_counts[idx] >= 60 and h != ""\n'
    '            else None\n'
    '        )\n'
    '        m = (\n'
    '            percentrank_inc(sorted_f, float(f))\n'
    '            if f_counts[idx] >= 60 and f != ""\n'
    '            else None\n'
    '        )\n'
    '        o = (\n'
    '            percentrank_inc(sorted_g, float(g))\n'
    '            if g_counts[idx] >= 60 and g != ""\n'
    '            else None\n'
    '        )\n'
    '        q = (\n'
    '            percentrank_inc(sorted_i, float(i_ch))\n'
    '            if i_counts[idx] >= 60 and i_ch != ""\n'
    '            else None\n'
    '        )\n'
    '        l = _inverted_score_percent_rank(k)\n'
    '        n = _inverted_score_percent_rank(m)\n'
    '        p = _inverted_score_percent_rank(o)\n'
    '        r = _inverted_score_percent_rank(q)\n'
    '        s = _claims_recovery_score(h, i_ch, j_vals[idx])\n'
    '        t = _claims_surge_warning(e_vals[idx], f, g)\n'
    '        zs = [af_vals[idx], ag_vals[idx], ah_vals[idx], ai_vals[idx]]\n'
    '        u = _industrial_outlier_score(zs)\n'
    '        v = (\n'
    '            round(clamp(0.45 * l + 0.25 * n + 0.2 * p + 0.1 * r), 2)\n'
    '            if all(x != "" for x in (l, n, p, r))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            round(clamp(0.1 * l + 0.3 * n + 0.2 * p + 0.35 * r + 0.05 * s), 2)\n'
    '            if all(x != "" for x in (l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.35 * s + 0.45 * t + 0.2 * u), 2)\n'
    '            if all(z != "" for z in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        d = d_vals[idx]\n'
    '        out.update(\n'
    '            {\n'
    '                "3M Moving Average": c_vals[idx],\n'
    '                "Smoothed Observation Count": d,\n'
    '                "1M Smoothed Growth": e_vals[idx],\n'
    '                "3M Annualised Smoothed Growth": f,\n'
    '                "6M Annualised Smoothed Growth": g,\n'
    '                "12M Smoothed Growth": h,\n'
    '                "3M Change in 12M Growth": i_ch,\n'
    '                "6M Change in 12M Growth": j_vals[idx],\n'
    '                "12M Claims Growth PercentRank": k if k is not None else "",\n'
    '                "12M Claims Growth Score": l,\n'
    '                "3M Annualised Claims Growth PercentRank": m if m is not None else "",\n'
    '                "3M Annualised Claims Growth Score": n,\n'
    '                "6M Annualised Claims Growth PercentRank": o if o is not None else "",\n'
    '                "6M Annualised Claims Growth Score": p,\n'
    '                "3M Change in 12M Claims Growth PercentRank": (\n'
    '                    q if q is not None else ""\n'
    '                ),\n'
    '                "3M Change in 12M Claims Growth Score": r,\n'
    '                "Initial Claims Recovery / Momentum Score": s,\n'
    '                "Initial Claims Surge Warning Score Raw": t,\n'
    '                "Initial Claims Outlier Score": u,\n'
    '                "Initial Claims Leading Score": v,\n'
    '                "Initial Claims Warning Composite Score": w,\n'
    '                "Initial Claims Final Warning Score": x,\n'
    '                "Initial Claims Labour Market Regime": _claims_regime(v),\n'
    '                "Initial Claims Momentum Label": _claims_momentum_label(v, w),\n'
    '                "Initial Claims Signal Interpretation": _claims_signal_interpretation(\n'
    '                    w, x\n'
    '                ),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if d < 60\n'
    '                        else ("Medium reliability" if d < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Data Status": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if d < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(z == "" for z in (v, w, x))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Initial Claims Warning Risk Label": _claims_warning_label(x),\n'
    '                "Initial Claims Signal Label": _claims_signal_label(v, w, x),\n'
    '                "12M Claims Growth Inverted Z-Score": af_vals[idx],\n'
    '                "3M Annualised Claims Growth Inverted Z-Score": ag_vals[idx],\n'
    '                "6M Annualised Claims Growth Inverted Z-Score": ah_vals[idx],\n'
    '                "3M Change in 12M Claims Growth Inverted Z-Score": ai_vals[idx],\n'
    '                "Initial Claims Outlier Label": _claims_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, INITIAL_CLAIMS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _employment_short_term_warning(e: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (e, f, g)):\n'
    '        return ""\n'
    '    if e < -0.005 and f < -0.03:\n'
    '        return -6\n'
    '    if e < -0.003 and f < -0.02:\n'
    '        return -4\n'
    '    if e > 0.005 and f > 0.03 and g > 0.02:\n'
    '        return 6\n'
    '    if e > 0.003 and f > 0.02:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _employment_regime(v: Any) -> str:\n'
    '    if is_blank(v):\n'
    '        return ""\n'
    '    if v >= 2:\n'
    '        return "Labour market expanding"\n'
    '    if v <= -2:\n'
    '        return "Labour market contracting"\n'
    '    return "Neutral labour market"\n'
    '\n'
    '\n'
    'def _employment_momentum_label(v: Any, w: Any) -> str:\n'
    '    if any(is_blank(x) for x in (v, w)):\n'
    '        return ""\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Employment growth strengthening"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Employment growth positive but deterioration risk rising"\n'
    '    if v >= 2 and -2 < w < 2:\n'
    '        return "Employment growth positive / stable"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Employment weak but improving"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Employment growth deteriorating"\n'
    '    if v <= -2 and -2 < w < 2:\n'
    '        return "Employment growth weak / stable"\n'
    '    if -2 < v < 2 and w >= 2:\n'
    '        return "Employment trend improving"\n'
    '    if -2 < v < 2 and w <= -2:\n'
    '        return "Employment trend weakening"\n'
    '    return "Employment growth broadly neutral"\n'
    '\n'
    '\n'
    'def _employment_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return (\n'
    '            "Strong positive employment leading signal with supportive warning profile"\n'
    '        )\n'
    '    if w >= 6 and x < 0:\n'
    '        return "Strong positive employment leading signal, but warning risk is negative"\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive employment leading signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Positive employment leading signal, but fragile"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed employment signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong negative employment leading signal with downside warning confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong negative employment leading signal, but warning profile is improving"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately negative employment leading signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return "Negative employment leading signal, but stabilisation risk is present"\n'
    '    return "Mixed/transition employment signal"\n'
    '\n'
    '\n'
    'def _employment_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x <= -6:\n'
    '        return "High employment warning risk"\n'
    '    if x <= -3:\n'
    '        return "Moderate employment warning risk"\n'
    '    if x < 3:\n'
    '        return "Neutral employment warning profile"\n'
    '    if x < 6:\n'
    '        return "Constructive employment warning profile"\n'
    '    return "Strong positive employment turn / warning cleared"\n'
    '\n'
    '\n'
    'def _employment_signal_label(v: Any, w: Any, x: Any) -> str:\n'
    '    if any(is_blank(z) for z in (v, w, x)):\n'
    '        return ""\n'
    '    if x <= -6 and w <= -2:\n'
    '        return "Severe employment deterioration risk"\n'
    '    if x <= -3 and w <= -2:\n'
    '        return "Employment deterioration warning"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Employment weak but improving"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Employment growth positive but deterioration starting"\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Employment growth strengthening"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Employment growth weakening / labour market contracting"\n'
    '    return "Neutral employment signal"\n'
    '\n'
    '\n'
    'def _employment_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme employment deterioration signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong employment deterioration signal"\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme employment improvement signal"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong employment improvement signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_employment_growth_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals = []\n'
    '    d_vals = []\n'
    '    numeric_b_count = 0\n'
    '    numeric_c_count = 0\n'
    '    for idx, val in enumerate(b):\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        window = [x for x in b[max(0, idx - 2) : idx + 1] if x is not None]\n'
    '        c = (sum(window) / len(window)) if numeric_b_count >= 3 and window else ""\n'
    '        c_vals.append(c)\n'
    '        if c != "":\n'
    '            numeric_c_count += 1\n'
    '        d_vals.append(numeric_c_count)\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    j_vals = []\n'
    '    for idx, c in enumerate(c_vals):\n'
    '        d = d_vals[idx]\n'
    '        e_vals.append(_safe_growth(c, c_vals[idx - 1]) if d >= 2 and idx >= 1 else "")\n'
    '        f_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 3], 4) if d >= 4 and idx >= 3 else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 6], 2) if d >= 7 and idx >= 6 else ""\n'
    '        )\n'
    '        h_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 12]) if d >= 13 and idx >= 12 else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 3]\n'
    '            if d >= 16 and idx >= 3 and h_vals[idx] != "" and h_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        j_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 6]\n'
    '            if d >= 19 and idx >= 6 and h_vals[idx] != "" and h_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    h_counts = _metric_counts(h_vals)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    i_counts = _metric_counts(i_vals)\n'
    '    af_vals = expanding_zscore_sample(h_vals, h_counts, 60)\n'
    '    ag_vals = expanding_zscore_sample(f_vals, f_counts, 60)\n'
    '    ah_vals = expanding_zscore_sample(g_vals, g_counts, 60)\n'
    '    ai_vals = expanding_zscore_sample(i_vals, i_counts, 60)\n'
    '    sorted_h = []\n'
    '    sorted_f = []\n'
    '    sorted_g = []\n'
    '    sorted_i = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in EMPLOYMENT_GROWTH_OUTPUT_COLUMNS}\n'
    '        h = h_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        g = g_vals[idx]\n'
    '        i_ch = i_vals[idx]\n'
    '        for val, arr in ((h, sorted_h), (f, sorted_f), (g, sorted_g), (i_ch, sorted_i)):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        k = (\n'
    '            percentrank_inc(sorted_h, float(h))\n'
    '            if h_counts[idx] >= 60 and h != ""\n'
    '            else None\n'
    '        )\n'
    '        m = (\n'
    '            percentrank_inc(sorted_f, float(f))\n'
    '            if f_counts[idx] >= 60 and f != ""\n'
    '            else None\n'
    '        )\n'
    '        o = (\n'
    '            percentrank_inc(sorted_g, float(g))\n'
    '            if g_counts[idx] >= 60 and g != ""\n'
    '            else None\n'
    '        )\n'
    '        q = (\n'
    '            percentrank_inc(sorted_i, float(i_ch))\n'
    '            if i_counts[idx] >= 60 and i_ch != ""\n'
    '            else None\n'
    '        )\n'
    '        l = _score_percent_rank(k)\n'
    '        n = _score_percent_rank(m)\n'
    '        p = _score_percent_rank(o)\n'
    '        r = _score_percent_rank(q)\n'
    '        s = _industrial_recovery_score(h, i_ch, j_vals[idx])\n'
    '        t = _employment_short_term_warning(e_vals[idx], f, g)\n'
    '        zs = [af_vals[idx], ag_vals[idx], ah_vals[idx], ai_vals[idx]]\n'
    '        u = _industrial_outlier_score(zs)\n'
    '        v = (\n'
    '            round(clamp(0.45 * l + 0.25 * n + 0.2 * p + 0.1 * r), 2)\n'
    '            if all(x != "" for x in (l, n, p, r))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            round(clamp(0.1 * l + 0.3 * n + 0.2 * p + 0.35 * r + 0.05 * s), 2)\n'
    '            if all(x != "" for x in (l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.35 * s + 0.45 * t + 0.2 * u), 2)\n'
    '            if all(z != "" for z in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        d = d_vals[idx]\n'
    '        out.update(\n'
    '            {\n'
    '                "3M Moving Average": c_vals[idx],\n'
    '                "Smoothed Observation Count": d,\n'
    '                "1M Smoothed Growth": e_vals[idx],\n'
    '                "3M Annualised Smoothed Growth": f,\n'
    '                "6M Annualised Smoothed Growth": g,\n'
    '                "12M Smoothed Growth": h,\n'
    '                "3M Change in 12M Growth": i_ch,\n'
    '                "6M Change in 12M Growth": j_vals[idx],\n'
    '                "12M Employment Growth PercentRank": k if k is not None else "",\n'
    '                "12M Employment Growth Score": l,\n'
    '                "3M Annualised Employment Growth PercentRank": (\n'
    '                    m if m is not None else ""\n'
    '                ),\n'
    '                "3M Annualised Employment Growth Score": n,\n'
    '                "6M Annualised Employment Growth PercentRank": (\n'
    '                    o if o is not None else ""\n'
    '                ),\n'
    '                "6M Annualised Employment Growth Score": p,\n'
    '                "3M Change in 12M Employment Growth PercentRank": (\n'
    '                    q if q is not None else ""\n'
    '                ),\n'
    '                "3M Change in 12M Employment Growth Score": r,\n'
    '                "Employment Recovery / Momentum Score": s,\n'
    '                "Employment Short-Term Warning Score Raw": t,\n'
    '                "Employment Outlier Score": u,\n'
    '                "Employment Leading Score": v,\n'
    '                "Employment Warning Composite Score": w,\n'
    '                "Employment Final Warning Score": x,\n'
    '                "Employment Labour Market Regime": _employment_regime(v),\n'
    '                "Employment Momentum Label": _employment_momentum_label(v, w),\n'
    '                "Employment Signal Interpretation": _employment_signal_interpretation(\n'
    '                    w, x\n'
    '                ),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if d < 60\n'
    '                        else ("Medium reliability" if d < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Data Status": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if d < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(z == "" for z in (v, w, x))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Employment Warning Risk Label": _employment_warning_label(x),\n'
    '                "Employment Signal Label": _employment_signal_label(v, w, x),\n'
    '                "12M Employment Growth Z-Score": af_vals[idx],\n'
    '                "3M Annualised Employment Growth Z-Score": ag_vals[idx],\n'
    '                "6M Annualised Employment Growth Z-Score": ah_vals[idx],\n'
    '                "3M Change in 12M Employment Growth Z-Score": ai_vals[idx],\n'
    '                "Employment Outlier Label": _employment_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, EMPLOYMENT_GROWTH_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _average_hours_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    if value is None or value == "":\n'
    '        return ""\n'
    '    try:\n'
    '        number = Decimal(str(value))\n'
    '    except Exception:\n'
    '        return ""\n'
    '    quantum = Decimal("1").scaleb(-places)\n'
    '    if number >= 0:\n'
    '        return float(number.quantize(quantum, rounding=ROUND_HALF_UP))\n'
    '    return float(-((-number).quantize(quantum, rounding=ROUND_HALF_UP)))\n'
    '\n'
    '\n'
    'def _average_hours_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    clean = [\n'
    '        number\n'
    '        for number in (to_float_or_none(value) for value in values)\n'
    '        if number is not None\n'
    '    ]\n'
    '    if not clean:\n'
    '        return ""\n'
    "    # Keep A14 independent of Python 3.13's compensated sum(). Google Sheets\n"
    '    # evaluates this short AVERAGE left-to-right, and the final bits determine\n'
    '    # ordering among otherwise tied average-hours growth observations.\n'
    '    total = clean[0]\n'
    '    for number in clean[1:]:\n'
    '        total += number\n'
    '    return total / len(clean)\n'
    '\n'
    '\n'
    'def _average_hours_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    current_float = to_float_or_none(current)\n'
    '    clean = sorted(\n'
    '        number\n'
    '        for number in (to_float_or_none(value) for value in values)\n'
    '        if number is not None\n'
    '    )\n'
    '    n = len(clean)\n'
    '    if (\n'
    '        current_float is None\n'
    '        or n < 2\n'
    '        or current_float < clean[0]\n'
    '        or current_float > clean[-1]\n'
    '    ):\n'
    '        return ""\n'
    '    left = bisect.bisect_left(clean, current_float)\n'
    '    right = bisect.bisect_right(clean, current_float)\n'
    '    if left != right:\n'
    '        # PERCENTRANK.INC assigns an exact observation the zero-based count of\n'
    "        # observations below it. The A14-specific mean above preserves Sheets'\n"
    '        # exact comparisons for values that merely display as duplicate 0.00%.\n'
    '        return left / (n - 1)\n'
    '    lower_index = left - 1\n'
    '    upper_index = left\n'
    '    lower = clean[lower_index]\n'
    '    upper = clean[upper_index]\n'
    '    if upper == lower:\n'
    '        return lower_index / (n - 1)\n'
    '    interpolated_index = lower_index + (current_float - lower) / (upper - lower)\n'
    '    return interpolated_index / (n - 1)\n'
    '\n'
    '\n'
    'def _average_hours_percent_rank_rounded_3(\n'
    '    history: Sequence[float], current: Any, numeric_count: int\n'
    ') -> Any:\n'
    '    if numeric_count < 60 or to_float_or_none(current) is None:\n'
    '        return ""\n'
    '    rank = _average_hours_percentrank_inc_sheets(history, current)\n'
    '    return "" if rank == "" else _average_hours_sheets_round(rank, 3)\n'
    '\n'
    '\n'
    'def _average_hours_score_percent_rank(rank: Any) -> Any:\n'
    '    if is_blank(rank):\n'
    '        return ""\n'
    '    return _average_hours_sheets_round((float(rank) * 20) - 10, 2)\n'
    '\n'
    '\n'
    'def _average_hours_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    number = to_float_or_none(value)\n'
    '    if number is None:\n'
    '        return ""\n'
    '    displayed = _average_hours_sheets_round(number * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _average_hours_recovery_score(h: Any, i: Any, j: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (h, i, j)):\n'
    '        return ""\n'
    '    if h < 0 and i > 0 and j > 0:\n'
    '        return 7\n'
    '    if h < 0 and i < 0 and j < 0:\n'
    '        return -7\n'
    '    if h > 0 and i < 0 and j < 0:\n'
    '        return -6\n'
    '    if h > 0 and i > 0 and j > 0:\n'
    '        return 5\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _average_hours_short_term_warning(e: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (e, f, g)):\n'
    '        return ""\n'
    '    if e < -0.0025 and f < -0.02:\n'
    '        return -6\n'
    '    if e < -0.0015 and f < -0.01:\n'
    '        return -4\n'
    '    if e > 0.0025 and f > 0.02 and g > 0.01:\n'
    '        return 6\n'
    '    if e > 0.0015 and f > 0.01:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _average_hours_regime(v: Any) -> str:\n'
    '    if is_blank(v):\n'
    '        return ""\n'
    '    if v >= 2:\n'
    '        return "Labour demand strengthening"\n'
    '    if v <= -2:\n'
    '        return "Labour demand weakening"\n'
    '    return "Neutral labour demand"\n'
    '\n'
    '\n'
    'def _average_hours_momentum_label(v: Any, w: Any) -> str:\n'
    '    if any(is_blank(x) for x in (v, w)):\n'
    '        return ""\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Average hours strengthening"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Average hours positive but deterioration risk rising"\n'
    '    if v >= 2 and -2 < w < 2:\n'
    '        return "Average hours positive / stable"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Average hours weak but improving"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Average hours deteriorating"\n'
    '    if v <= -2 and -2 < w < 2:\n'
    '        return "Average hours weak / stable"\n'
    '    if -2 < v < 2 and w >= 2:\n'
    '        return "Average hours trend improving"\n'
    '    if -2 < v < 2 and w <= -2:\n'
    '        return "Average hours trend weakening"\n'
    '    return "Average hours broadly neutral"\n'
    '\n'
    '\n'
    'def _average_hours_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return "Strong positive average-hours leading signal with supportive warning profile"\n'
    '    if w >= 6 and x < 0:\n'
    '        return (\n'
    '            "Strong positive average-hours leading signal, but warning risk is negative"\n'
    '        )\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive average-hours leading signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Positive average-hours leading signal, but fragile"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed average-hours signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong negative average-hours leading signal with downside warning confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong negative average-hours leading signal, but warning profile is improving"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately negative average-hours leading signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return (\n'
    '            "Negative average-hours leading signal, but stabilisation risk is present"\n'
    '        )\n'
    '    return "Mixed/transition average-hours signal"\n'
    '\n'
    '\n'
    'def _average_hours_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x <= -6:\n'
    '        return "High average-hours warning risk"\n'
    '    if x <= -3:\n'
    '        return "Moderate average-hours warning risk"\n'
    '    if x < 3:\n'
    '        return "Neutral average-hours warning profile"\n'
    '    if x < 6:\n'
    '        return "Constructive average-hours warning profile"\n'
    '    return "Strong positive average-hours turn / warning cleared"\n'
    '\n'
    '\n'
    'def _average_hours_signal_label(v: Any, w: Any, x: Any) -> str:\n'
    '    if any(is_blank(z) for z in (v, w, x)):\n'
    '        return ""\n'
    '    if x <= -6 and w <= -2:\n'
    '        return "Severe average-hours deterioration risk"\n'
    '    if x <= -3 and w <= -2:\n'
    '        return "Average hours deterioration warning"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Average hours weak but improving"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Average hours positive but deterioration starting"\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Average hours strengthening"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Average hours weakening / labour demand deteriorating"\n'
    '    return "Neutral average-hours signal"\n'
    '\n'
    '\n'
    'def _average_hours_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme hours-worked deterioration signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong hours-worked deterioration signal"\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme hours-worked improvement signal"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong hours-worked improvement signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_average_hours_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals = []\n'
    '    d_vals = []\n'
    '    numeric_b_count = 0\n'
    '    numeric_c_count = 0\n'
    '    for idx, val in enumerate(b):\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        window = [x for x in b[max(0, idx - 2) : idx + 1] if x is not None]\n'
    '        c = (\n'
    '            _average_hours_mean_sheets(window)\n'
    '            if numeric_b_count >= 3 and window\n'
    '            else ""\n'
    '        )\n'
    '        c_vals.append(c)\n'
    '        if c != "":\n'
    '            numeric_c_count += 1\n'
    '        d_vals.append(numeric_c_count)\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    j_vals = []\n'
    '    for idx, c in enumerate(c_vals):\n'
    '        d = d_vals[idx]\n'
    '        e_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 1]) if d >= 2 and idx >= 1 and c != "" else ""\n'
    '        )\n'
    '        f_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 3], 4)\n'
    '            if d >= 4 and idx >= 3 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 6], 2)\n'
    '            if d >= 7 and idx >= 6 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        h_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 12])\n'
    '            if d >= 13 and idx >= 12 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 3]\n'
    '            if d >= 16 and idx >= 3 and h_vals[idx] != "" and h_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        j_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 6]\n'
    '            if d >= 19 and idx >= 6 and h_vals[idx] != "" and h_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    h_counts = _metric_counts(h_vals)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    i_counts = _metric_counts(i_vals)\n'
    '    af_vals = expanding_zscore_sample(h_vals, h_counts, 60)\n'
    '    ag_vals = expanding_zscore_sample(f_vals, f_counts, 60)\n'
    '    ah_vals = expanding_zscore_sample(g_vals, g_counts, 60)\n'
    '    ai_vals = expanding_zscore_sample(i_vals, i_counts, 60)\n'
    '    sorted_h = []\n'
    '    sorted_f = []\n'
    '    sorted_g = []\n'
    '    sorted_i = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in AVERAGE_HOURS_OUTPUT_COLUMNS}\n'
    '        h = h_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        g = g_vals[idx]\n'
    '        i_ch = i_vals[idx]\n'
    '        for val, arr in ((h, sorted_h), (f, sorted_f), (g, sorted_g), (i_ch, sorted_i)):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        k = _average_hours_percent_rank_rounded_3(sorted_h, h, h_counts[idx])\n'
    '        m = _average_hours_percent_rank_rounded_3(sorted_f, f, f_counts[idx])\n'
    '        o = _average_hours_percent_rank_rounded_3(sorted_g, g, g_counts[idx])\n'
    '        q = _average_hours_percent_rank_rounded_3(sorted_i, i_ch, i_counts[idx])\n'
    '        l = _average_hours_score_percent_rank(k)\n'
    '        n = _average_hours_score_percent_rank(m)\n'
    '        p = _average_hours_score_percent_rank(o)\n'
    '        r = _average_hours_score_percent_rank(q)\n'
    '        s = _average_hours_recovery_score(h, i_ch, j_vals[idx])\n'
    '        t = _average_hours_short_term_warning(e_vals[idx], f, g)\n'
    '        zs = [af_vals[idx], ag_vals[idx], ah_vals[idx], ai_vals[idx]]\n'
    '        u = _industrial_outlier_score(zs)\n'
    '        v = (\n'
    '            _average_hours_sheets_round(\n'
    '                clamp(0.40 * l + 0.25 * n + 0.20 * p + 0.15 * r), 2\n'
    '            )\n'
    '            if all(x != "" for x in (l, n, p, r))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            _average_hours_sheets_round(\n'
    '                clamp(0.10 * l + 0.35 * n + 0.20 * p + 0.30 * r + 0.05 * s), 2\n'
    '            )\n'
    '            if all(x != "" for x in (l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            _average_hours_sheets_round(clamp(0.45 * s + 0.35 * t + 0.20 * u), 2)\n'
    '            if all(z != "" for z in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        d = d_vals[idx]\n'
    '        out.update(\n'
    '            {\n'
    '                "3M MA": c_vals[idx],\n'
    '                "Sample Count": d,\n'
    '                "1M % Change": e_vals[idx],\n'
    '                "3M Annualised Growth": f,\n'
    '                "6M Annualised Growth": g,\n'
    '                "YoY Growth": h,\n'
    '                "3M Change in YoY Growth": _average_hours_percent_display(i_ch),\n'
    '                "6M Change in YoY Growth": _average_hours_percent_display(j_vals[idx]),\n'
    '                "YoY Growth Percentile": k,\n'
    '                "YoY Growth Score": l,\n'
    '                "3M Growth Percentile": m,\n'
    '                "3M Growth Score": n,\n'
    '                "6M Growth Percentile": o,\n'
    '                "6M Growth Score": p,\n'
    '                "Momentum Percentile": q,\n'
    '                "Momentum Score": r,\n'
    '                "Hours Cycle Turning Point Score": s,\n'
    '                "Hours Shock Score": t,\n'
    '                "Hours Extreme Score": u,\n'
    '                "Coincident Score": v,\n'
    '                "Leading Score": w,\n'
    '                "Warning Score": x,\n'
    '                "State": _average_hours_regime(v),\n'
    '                "Regime": _average_hours_momentum_label(v, w),\n'
    '                "Signal Comment": _average_hours_signal_interpretation(w, x),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if d < 60\n'
    '                        else ("Medium reliability" if d < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Signal Validity": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if d < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(z == "" for z in (v, w, x))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Warning Flag": _average_hours_warning_label(x),\n'
    '                "Slowdown Risk Flag": _average_hours_signal_label(v, w, x),\n'
    '                "YoY Growth Z-Score": af_vals[idx],\n'
    '                "3M Growth Z-Score": ag_vals[idx],\n'
    '                "6M Growth Z-Score": ah_vals[idx],\n'
    '                "Momentum Z-Score": ai_vals[idx],\n'
    '                "Extreme Flag": _average_hours_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, AVERAGE_HOURS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _building_permits_recovery_score(h: Any, i: Any, j: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (h, i, j)):\n'
    '        return ""\n'
    '    if h < 0 and i > 0 and j > 0:\n'
    '        return 7\n'
    '    if h < 0 and i < 0 and j < 0:\n'
    '        return -7\n'
    '    if h > 0 and i < 0 and j < 0:\n'
    '        return -6\n'
    '    if h > 0 and i > 0 and j > 0:\n'
    '        return 5\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _building_permits_short_term_warning(e: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (e, f, g)):\n'
    '        return ""\n'
    '    if e < -0.05 and f < -0.25:\n'
    '        return -6\n'
    '    if e < -0.03 and f < -0.15:\n'
    '        return -4\n'
    '    if e > 0.05 and f > 0.25 and g > 0.10:\n'
    '        return 6\n'
    '    if e > 0.03 and f > 0.15:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _building_permits_regime(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 2:\n'
    '        return "Housing activity strengthening"\n'
    '    if x <= -2:\n'
    '        return "Housing activity weakening"\n'
    '    return "Neutral housing activity"\n'
    '\n'
    '\n'
    'def _building_permits_momentum_label(x: Any, y: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y)):\n'
    '        return ""\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Building permits strengthening"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Building permits positive but deterioration risk rising"\n'
    '    if x >= 2 and -2 < y < 2:\n'
    '        return "Building permits positive / stable"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Building permits weak but improving"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Building permits deteriorating"\n'
    '    if x <= -2 and -2 < y < 2:\n'
    '        return "Building permits weak / stable"\n'
    '    if -2 < x < 2 and y >= 2:\n'
    '        return "Building permits trend improving"\n'
    '    if -2 < x < 2 and y <= -2:\n'
    '        return "Building permits trend weakening"\n'
    '    return "Building permits broadly neutral"\n'
    '\n'
    '\n'
    'def _building_permits_signal_interpretation(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 6 and z >= 0:\n'
    '        return "Strong positive building-permits leading signal with supportive warning profile"\n'
    '    if y >= 6 and z < 0:\n'
    '        return "Strong positive building-permits leading signal, but warning risk is negative"\n'
    '    if y >= 2 and z >= 0:\n'
    '        return "Moderately positive building-permits leading signal"\n'
    '    if y >= 2 and z < 0:\n'
    '        return "Positive building-permits leading signal, but fragile"\n'
    '    if y > -2 and z > -3 and z < 3:\n'
    '        return "Neutral/mixed building-permits signal"\n'
    '    if y <= -6 and z <= 0:\n'
    '        return "Strong negative building-permits leading signal with downside warning confirmation"\n'
    '    if y <= -6 and z > 0:\n'
    '        return "Strong negative building-permits leading signal, but warning profile is improving"\n'
    '    if y <= -2 and z <= 0:\n'
    '        return "Moderately negative building-permits leading signal"\n'
    '    if y <= -2 and z > 0:\n'
    '        return "Negative building-permits leading signal, but stabilisation risk is present"\n'
    '    return "Mixed/transition building-permits signal"\n'
    '\n'
    '\n'
    'def _building_permits_warning_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z <= -6:\n'
    '        return "High building-permits warning risk"\n'
    '    if z <= -3:\n'
    '        return "Moderate building-permits warning risk"\n'
    '    if z < 3:\n'
    '        return "Neutral building-permits warning profile"\n'
    '    if z < 6:\n'
    '        return "Constructive building-permits warning profile"\n'
    '    return "Strong positive building-permits turn / warning cleared"\n'
    '\n'
    '\n'
    'def _building_permits_signal_label(x: Any, y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y, z)):\n'
    '        return ""\n'
    '    if z <= -6 and y <= -2:\n'
    '        return "Severe building-permits deterioration risk"\n'
    '    if z <= -3 and y <= -2:\n'
    '        return "Building-permits deterioration warning"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Building permits weak but improving"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Building permits positive but deterioration starting"\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Building permits strengthening"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Building permits weakening / housing activity deteriorating"\n'
    '    return "Neutral building-permits signal"\n'
    '\n'
    '\n'
    'def _building_permits_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme building-permits deterioration signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong building-permits deterioration signal"\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme building-permits improvement signal"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong building-permits improvement signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_building_permits_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals = []\n'
    '    d_vals = []\n'
    '    numeric_b_count = 0\n'
    '    numeric_c_count = 0\n'
    '    for idx, val in enumerate(b):\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        window = [x for x in b[max(0, idx - 2) : idx + 1] if x is not None]\n'
    '        c = (sum(window) / len(window)) if numeric_b_count >= 3 and window else ""\n'
    '        c_vals.append(c)\n'
    '        if c != "":\n'
    '            numeric_c_count += 1\n'
    '        d_vals.append(numeric_c_count)\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    j_vals = []\n'
    '    for idx, c in enumerate(c_vals):\n'
    '        d = d_vals[idx]\n'
    '        e_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 1]) if d >= 2 and idx >= 1 and c != "" else ""\n'
    '        )\n'
    '        f_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 3], 4)\n'
    '            if d >= 4 and idx >= 3 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        g_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 6], 2)\n'
    '            if d >= 7 and idx >= 6 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        h_vals.append(\n'
    '            _safe_growth(c, c_vals[idx - 12])\n'
    '            if d >= 13 and idx >= 12 and c != ""\n'
    '            else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 3]\n'
    '            if d >= 16 and idx >= 3 and h_vals[idx] != "" and h_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        j_vals.append(\n'
    '            h_vals[idx] - h_vals[idx - 6]\n'
    '            if d >= 19 and idx >= 6 and h_vals[idx] != "" and h_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    h_counts = _metric_counts(h_vals)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    i_counts = _metric_counts(i_vals)\n'
    '    c_counts = _metric_counts(c_vals)\n'
    '    ah_vals = expanding_zscore_sample(h_vals, h_counts, 60)\n'
    '    ai_vals = expanding_zscore_sample(f_vals, f_counts, 60)\n'
    '    aj_vals = expanding_zscore_sample(g_vals, g_counts, 60)\n'
    '    ak_vals = expanding_zscore_sample(i_vals, i_counts, 60)\n'
    '    sorted_h = []\n'
    '    sorted_f = []\n'
    '    sorted_g = []\n'
    '    sorted_i = []\n'
    '    sorted_c = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in BUILDING_PERMITS_OUTPUT_COLUMNS}\n'
    '        c = c_vals[idx]\n'
    '        h = h_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        g = g_vals[idx]\n'
    '        i_ch = i_vals[idx]\n'
    '        for val, arr in (\n'
    '            (h, sorted_h),\n'
    '            (f, sorted_f),\n'
    '            (g, sorted_g),\n'
    '            (i_ch, sorted_i),\n'
    '            (c, sorted_c),\n'
    '        ):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        k = (\n'
    '            percentrank_inc(sorted_h, float(h))\n'
    '            if h_counts[idx] >= 60 and h != ""\n'
    '            else None\n'
    '        )\n'
    '        m = (\n'
    '            percentrank_inc(sorted_f, float(f))\n'
    '            if f_counts[idx] >= 60 and f != ""\n'
    '            else None\n'
    '        )\n'
    '        o = (\n'
    '            percentrank_inc(sorted_g, float(g))\n'
    '            if g_counts[idx] >= 60 and g != ""\n'
    '            else None\n'
    '        )\n'
    '        q = (\n'
    '            percentrank_inc(sorted_i, float(i_ch))\n'
    '            if i_counts[idx] >= 60 and i_ch != ""\n'
    '            else None\n'
    '        )\n'
    '        s_rank = (\n'
    '            percentrank_inc(sorted_c, float(c))\n'
    '            if c_counts[idx] >= 60 and c != ""\n'
    '            else None\n'
    '        )\n'
    '        l = _score_percent_rank(k)\n'
    '        n = _score_percent_rank(m)\n'
    '        p = _score_percent_rank(o)\n'
    '        r = _score_percent_rank(q)\n'
    '        t_score = _score_percent_rank(s_rank)\n'
    '        u = _building_permits_recovery_score(h, i_ch, j_vals[idx])\n'
    '        v = _building_permits_short_term_warning(e_vals[idx], f, g)\n'
    '        zs = [ah_vals[idx], ai_vals[idx], aj_vals[idx], ak_vals[idx]]\n'
    '        w = _industrial_outlier_score(zs)\n'
    '        x = (\n'
    '            round(clamp(0.3 * l + 0.2 * n + 0.15 * p + 0.15 * r + 0.2 * t_score), 2)\n'
    '            if all(z != "" for z in (l, n, p, r, t_score))\n'
    '            else ""\n'
    '        )\n'
    '        y = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.1 * l + 0.3 * n + 0.2 * p + 0.25 * r + 0.1 * t_score + 0.05 * u\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(z != "" for z in (l, n, p, r, t_score, u))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            round(clamp(0.45 * u + 0.35 * v + 0.2 * w), 2)\n'
    '            if all(val != "" for val in (u, v, w))\n'
    '            else ""\n'
    '        )\n'
    '        d = d_vals[idx]\n'
    '        out.update(\n'
    '            {\n'
    '                "3M Moving Average": c,\n'
    '                "Smoothed Observation Count": d,\n'
    '                "1M Smoothed Growth": e_vals[idx],\n'
    '                "3M Annualised Smoothed Growth": f,\n'
    '                "6M Annualised Smoothed Growth": g,\n'
    '                "12M Smoothed Growth": h,\n'
    '                "3M Change in 12M Growth": i_ch,\n'
    '                "6M Change in 12M Growth": j_vals[idx],\n'
    '                "12M Building Permits Growth PercentRank": k if k is not None else "",\n'
    '                "12M Building Permits Growth Score": l,\n'
    '                "3M Annualised Building Permits Growth PercentRank": (\n'
    '                    m if m is not None else ""\n'
    '                ),\n'
    '                "3M Annualised Building Permits Growth Score": n,\n'
    '                "6M Annualised Building Permits Growth PercentRank": (\n'
    '                    o if o is not None else ""\n'
    '                ),\n'
    '                "6M Annualised Building Permits Growth Score": p,\n'
    '                "3M Change in 12M Building Permits Growth PercentRank": (\n'
    '                    q if q is not None else ""\n'
    '                ),\n'
    '                "3M Change in 12M Building Permits Growth Score": r,\n'
    '                "Smoothed Level PercentRank": s_rank if s_rank is not None else "",\n'
    '                "Smoothed Level Score": t_score,\n'
    '                "Building Permits Recovery / Momentum Score": u,\n'
    '                "Building Permits Short-Term Warning Score Raw": v,\n'
    '                "Building Permits Outlier Score": w,\n'
    '                "Building Permits Leading Score": x,\n'
    '                "Building Permits Warning Composite Score": y,\n'
    '                "Building Permits Final Warning Score": z,\n'
    '                "Building Permits Housing Activity Regime": _building_permits_regime(x),\n'
    '                "Building Permits Momentum Label": _building_permits_momentum_label(\n'
    '                    x, y\n'
    '                ),\n'
    '                "Building Permits Signal Interpretation": _building_permits_signal_interpretation(\n'
    '                    y, z\n'
    '                ),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if d < 60\n'
    '                        else ("Medium reliability" if d < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Data Status": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if d < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(val == "" for val in (x, y, z))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Building Permits Warning Risk Label": _building_permits_warning_label(\n'
    '                    z\n'
    '                ),\n'
    '                "Building Permits Signal Label": _building_permits_signal_label(\n'
    '                    x, y, z\n'
    '                ),\n'
    '                "12M Building Permits Growth Z-Score": ah_vals[idx],\n'
    '                "3M Annualised Building Permits Growth Z-Score": ai_vals[idx],\n'
    '                "6M Annualised Building Permits Growth Z-Score": aj_vals[idx],\n'
    '                "3M Change in 12M Building Permits Growth Z-Score": ak_vals[idx],\n'
    '                "Building Permits Outlier Label": _building_permits_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, BUILDING_PERMITS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _core_inflation_target_status(g: Any, e: Any) -> str:\n'
    '    if is_blank(g):\n'
    '        return ""\n'
    '    if g >= 0.03 and not is_blank(e) and e >= 0.03:\n'
    '        return "Clearly above target and persistent"\n'
    '    if g >= 0.03 and not is_blank(e) and e < 0.025:\n'
    '        return "Above target but cooling"\n'
    '    if g < 0.03 and not is_blank(e) and e >= 0.03:\n'
    '        return "Near/below target but re-accelerating"\n'
    '    if g >= 0.023:\n'
    '        return "Moderately above target"\n'
    '    if g >= 0.017:\n'
    '        return "Near target"\n'
    '    if g >= 0.01:\n'
    '        return "Below target / soft inflation"\n'
    '    return "Disinflation / deflation risk"\n'
    '\n'
    '\n'
    'def _core_inflation_recovery_cooling_score(g: Any, h: Any, i: Any, r: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (g, h, i, r)):\n'
    '        return ""\n'
    '    if g < 0.02 and h > 0 and i > 0:\n'
    '        return 6\n'
    '    if g >= 0.02 and h > 0 and i > 0:\n'
    '        return 5\n'
    '    if g > 0.02 and h < 0 and i < 0:\n'
    '        return -5\n'
    '    if g < 0.02 and h < 0 and i < 0:\n'
    '        return -6\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _core_inflation_target_pressure_score(r: Any, g: Any, e: Any, f: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (r, g, e, f)):\n'
    '        return ""\n'
    '    if r >= 0.01 and e > 0.03 and f > 0.03:\n'
    '        return 6\n'
    '    if r >= 0.005 and e > 0.025 and f > 0.025:\n'
    '        return 4\n'
    '    if r <= -0.005 and e < 0.015 and f < 0.015:\n'
    '        return -4\n'
    '    if r <= -0.01 and e < 0.01 and f < 0.01:\n'
    '        return -6\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _core_inflation_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _core_inflation_regime(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 2:\n'
    '        return "Inflation pressure rising"\n'
    '    if x <= -2:\n'
    '        return "Inflation pressure easing"\n'
    '    return "Neutral inflation pressure"\n'
    '\n'
    '\n'
    'def _core_inflation_momentum_label(x: Any, y: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y)):\n'
    '        return ""\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Core inflation pressure rising"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Core inflation elevated but cooling"\n'
    '    if x >= 2 and -2 < y < 2:\n'
    '        return "Core inflation pressure elevated / stable"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Core inflation weak but re-accelerating"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Core inflation pressure easing"\n'
    '    if x <= -2 and -2 < y < 2:\n'
    '        return "Core inflation pressure weak / stable"\n'
    '    if -2 < x < 2 and y >= 2:\n'
    '        return "Core inflation trend re-accelerating"\n'
    '    if -2 < x < 2 and y <= -2:\n'
    '        return "Core inflation trend cooling"\n'
    '    return "Core inflation broadly neutral"\n'
    '\n'
    '\n'
    'def _core_inflation_signal_interpretation(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 6 and z >= 0:\n'
    '        return "Strong upside core inflation leading signal with supportive policy-pressure profile"\n'
    '    if y >= 6 and z < 0:\n'
    '        return "Strong upside core inflation leading signal, but warning profile is cooling"\n'
    '    if y >= 2 and z >= 0:\n'
    '        return "Moderately positive core inflation-pressure signal"\n'
    '    if y >= 2 and z < 0:\n'
    '        return "Core inflation re-acceleration signal, but policy-pressure profile is cooling"\n'
    '    if y > -2 and z > -3 and z < 3:\n'
    '        return "Neutral/mixed core inflation signal"\n'
    '    if y <= -6 and z <= 0:\n'
    '        return "Strong disinflationary core inflation leading signal with downside confirmation"\n'
    '    if y <= -6 and z > 0:\n'
    '        return "Strong disinflationary core inflation leading signal, but policy pressure remains elevated"\n'
    '    if y <= -2 and z <= 0:\n'
    '        return "Moderately disinflationary core inflation signal"\n'
    '    if y <= -2 and z > 0:\n'
    '        return "Core inflation cooling signal, but policy pressure remains elevated"\n'
    '    return "Mixed/transition core inflation signal"\n'
    '\n'
    '\n'
    'def _core_inflation_warning_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "High upside core inflation warning risk"\n'
    '    if z >= 3:\n'
    '        return "Moderate upside core inflation warning risk"\n'
    '    if z > -3:\n'
    '        return "Neutral core inflation warning profile"\n'
    '    if z > -6:\n'
    '        return "Moderate disinflation warning risk"\n'
    '    return "High disinflation / deflation warning risk"\n'
    '\n'
    '\n'
    'def _core_inflation_policy_signal_label(z: Any, g: Any, e: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, g, e)):\n'
    '        return ""\n'
    '    if g >= 0.03 and e >= 0.03 and z >= 6:\n'
    '        return "Persistent above-target core inflation pressure"\n'
    '    if g >= 0.03 and e < 0.025 and z <= 3:\n'
    '        return "Above-target core inflation but cooling"\n'
    '    if g < 0.03 and e >= 0.03 and z >= 3:\n'
    '        return "Near/below target but core inflation re-accelerating"\n'
    '    if g < 0.01 and z <= -5:\n'
    '        return "Core disinflation / deflation-risk pressure"\n'
    '    if g >= 0.023 and z >= 5:\n'
    '        return "Core inflation pressure becoming policy-relevant"\n'
    '    if z >= 6 and e >= 0.025:\n'
    '        return "Early upside core inflation-pressure warning"\n'
    '    return "Normal core inflation policy regime"\n'
    '\n'
    '\n'
    'def _core_inflation_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside core inflation pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside core inflation pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside inflation / disinflation signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside inflation / disinflation signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_core_inflation_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_vals = []\n'
    '    numeric_b_count = 0\n'
    '    for val in b:\n'
    '        if val is not None:\n'
    '            numeric_b_count += 1\n'
    '        c_vals.append(numeric_b_count)\n'
    '    d_vals = []\n'
    '    e_vals = []\n'
    '    f_vals = []\n'
    '    g_vals = []\n'
    '    h_vals = []\n'
    '    i_vals = []\n'
    '    for idx, cur in enumerate(b):\n'
    '        c = c_vals[idx]\n'
    '        d_vals.append(\n'
    '            _safe_growth(cur, b[idx - 1], 12)\n'
    '            if c >= 2 and idx >= 1 and cur is not None\n'
    '            else ""\n'
    '        )\n'
    '        e_vals.append(_safe_growth(cur, b[idx - 3], 4) if c >= 4 and idx >= 3 else "")\n'
    '        f_vals.append(_safe_growth(cur, b[idx - 6], 2) if c >= 7 and idx >= 6 else "")\n'
    '        g_vals.append(_safe_growth(cur, b[idx - 12]) if c >= 13 and idx >= 12 else "")\n'
    '        h_vals.append(\n'
    '            g_vals[idx] - g_vals[idx - 3]\n'
    '            if c >= 16 and idx >= 3 and g_vals[idx] != "" and g_vals[idx - 3] != ""\n'
    '            else ""\n'
    '        )\n'
    '        i_vals.append(\n'
    '            g_vals[idx] - g_vals[idx - 6]\n'
    '            if c >= 19 and idx >= 6 and g_vals[idx] != "" and g_vals[idx - 6] != ""\n'
    '            else ""\n'
    '        )\n'
    '    g_counts = _metric_counts(g_vals)\n'
    '    e_counts = _metric_counts(e_vals)\n'
    '    f_counts = _metric_counts(f_vals)\n'
    '    h_counts = _metric_counts(h_vals)\n'
    '    ah_vals = expanding_zscore_sample(g_vals, c_vals, 60)\n'
    '    ai_vals = expanding_zscore_sample(e_vals, c_vals, 60)\n'
    '    aj_vals = expanding_zscore_sample(f_vals, c_vals, 60)\n'
    '    ak_vals = expanding_zscore_sample(h_vals, c_vals, 60)\n'
    '    sorted_g = []\n'
    '    sorted_e = []\n'
    '    sorted_f = []\n'
    '    sorted_h = []\n'
    '    rows = []\n'
    '    for idx in range(len(b)):\n'
    '        out = {col: "" for col in CORE_INFLATION_OUTPUT_COLUMNS}\n'
    '        g = g_vals[idx]\n'
    '        e = e_vals[idx]\n'
    '        f = f_vals[idx]\n'
    '        h = h_vals[idx]\n'
    '        for val, arr in ((g, sorted_g), (e, sorted_e), (f, sorted_f), (h, sorted_h)):\n'
    '            if val != "":\n'
    '                bisect.insort(arr, float(val))\n'
    '        j_rank = (\n'
    '            percentrank_inc(sorted_g, float(g))\n'
    '            if g_counts[idx] >= 60 and g != ""\n'
    '            else None\n'
    '        )\n'
    '        l_rank = (\n'
    '            percentrank_inc(sorted_e, float(e))\n'
    '            if e_counts[idx] >= 60 and e != ""\n'
    '            else None\n'
    '        )\n'
    '        n_rank = (\n'
    '            percentrank_inc(sorted_f, float(f))\n'
    '            if f_counts[idx] >= 60 and f != ""\n'
    '            else None\n'
    '        )\n'
    '        p_rank = (\n'
    '            percentrank_inc(sorted_h, float(h))\n'
    '            if h_counts[idx] >= 60 and h != ""\n'
    '            else None\n'
    '        )\n'
    '        k = _score_percent_rank(j_rank)\n'
    '        m = _score_percent_rank(l_rank)\n'
    '        o = _score_percent_rank(n_rank)\n'
    '        q = _score_percent_rank(p_rank)\n'
    '        r = g - 0.02 if g != "" else ""\n'
    '        t = (\n'
    '            round(clamp(0.45 * m + 0.35 * o + 0.2 * q), 2)\n'
    '            if all(x != "" for x in (m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        u = _core_inflation_recovery_cooling_score(g, h, i_vals[idx], r)\n'
    '        v = _core_inflation_target_pressure_score(r, g, e, f)\n'
    '        zs = [ah_vals[idx], ai_vals[idx], aj_vals[idx], ak_vals[idx]]\n'
    '        w = _core_inflation_outlier_score(zs)\n'
    '        target_gap_component = clamp(r * 500, -10, 10) if r != "" else ""\n'
    '        x = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.35 * k + 0.25 * m + 0.2 * o + 0.1 * q + 0.1 * target_gap_component\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(z != "" for z in (k, m, o, q, target_gap_component))\n'
    '            else ""\n'
    '        )\n'
    '        y = (\n'
    '            round(clamp(0.15 * k + 0.35 * m + 0.25 * o + 0.2 * q + 0.05 * u), 2)\n'
    '            if all(z != "" for z in (k, m, o, q, u))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            round(clamp(0.35 * u + 0.45 * v + 0.2 * w), 2)\n'
    '            if all(val != "" for val in (u, v, w))\n'
    '            else ""\n'
    '        )\n'
    '        c = c_vals[idx]\n'
    '        out.update(\n'
    '            {\n'
    '                "Observation Count": c,\n'
    '                "1M Annualised Inflation": d_vals[idx],\n'
    '                "3M Annualised Inflation": e,\n'
    '                "6M Annualised Inflation": f,\n'
    '                "12M Inflation": g,\n'
    '                "3M Change in 12M Inflation": h,\n'
    '                "6M Change in 12M Inflation": i_vals[idx],\n'
    '                "12M Core Inflation PercentRank": j_rank if j_rank is not None else "",\n'
    '                "12M Core Inflation Score": k,\n'
    '                "3M Annualised Core Inflation PercentRank": (\n'
    '                    l_rank if l_rank is not None else ""\n'
    '                ),\n'
    '                "3M Annualised Core Inflation Score": m,\n'
    '                "6M Annualised Core Inflation PercentRank": (\n'
    '                    n_rank if n_rank is not None else ""\n'
    '                ),\n'
    '                "6M Annualised Core Inflation Score": o,\n'
    '                "3M Change in 12M Core Inflation PercentRank": (\n'
    '                    p_rank if p_rank is not None else ""\n'
    '                ),\n'
    '                "3M Change in 12M Core Inflation Score": q,\n'
    '                "Distance From 2 Percent Target": r,\n'
    '                "Core Inflation Target Status": _core_inflation_target_status(g, e),\n'
    '                "Core Inflation Momentum Composite Score": t,\n'
    '                "Core Inflation Recovery / Cooling Score": u,\n'
    '                "Core Inflation Target Pressure Score": v,\n'
    '                "Core Inflation Outlier Score": w,\n'
    '                "Core Inflation Leading Score": x,\n'
    '                "Core Inflation Warning Composite Score": y,\n'
    '                "Core Inflation Final Warning Score": z,\n'
    '                "Core Inflation Pressure Regime": _core_inflation_regime(x),\n'
    '                "Core Inflation Momentum Label": _core_inflation_momentum_label(x, y),\n'
    '                "Core Inflation Signal Interpretation": _core_inflation_signal_interpretation(\n'
    '                    y, z\n'
    '                ),\n'
    '                "Reliability": (\n'
    '                    ""\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Insufficient history"\n'
    '                        if c < 60\n'
    '                        else ("Medium reliability" if c < 120 else "High reliability")\n'
    '                    )\n'
    '                ),\n'
    '                "Data Status": (\n'
    '                    "Missing Data"\n'
    '                    if b[idx] is None\n'
    '                    else (\n'
    '                        "Low Sample"\n'
    '                        if c < 60\n'
    '                        else (\n'
    '                            "Incomplete Score"\n'
    '                            if any(val == "" for val in (x, y, z))\n'
    '                            else "Valid"\n'
    '                        )\n'
    '                    )\n'
    '                ),\n'
    '                "Core Inflation Warning Risk Label": _core_inflation_warning_label(z),\n'
    '                "Core Inflation Policy Signal Label": _core_inflation_policy_signal_label(\n'
    '                    z, g, e\n'
    '                ),\n'
    '                "12M Core Inflation Z-Score": ah_vals[idx],\n'
    '                "3M Annualised Core Inflation Z-Score": ai_vals[idx],\n'
    '                "6M Annualised Core Inflation Z-Score": aj_vals[idx],\n'
    '                "3M Change in 12M Core Inflation Z-Score": ak_vals[idx],\n'
    '                "Core Inflation Outlier Label": _core_inflation_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, CORE_INFLATION_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _core_cpi_inflation_regime(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 2:\n'
    '        return "Core CPI inflation pressure rising"\n'
    '    if x <= -2:\n'
    '        return "Core CPI inflation pressure easing"\n'
    '    return "Neutral Core CPI inflation pressure"\n'
    '\n'
    '\n'
    'def _core_cpi_inflation_momentum_label(x: Any, y: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y)):\n'
    '        return ""\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Core CPI inflation pressure rising"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Core CPI inflation elevated but cooling"\n'
    '    if x >= 2 and -2 < y < 2:\n'
    '        return "Core CPI inflation pressure elevated / stable"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Core CPI inflation weak but re-accelerating"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Core CPI inflation pressure easing"\n'
    '    if x <= -2 and -2 < y < 2:\n'
    '        return "Core CPI inflation pressure weak / stable"\n'
    '    if -2 < x < 2 and y >= 2:\n'
    '        return "Core CPI inflation trend re-accelerating"\n'
    '    if -2 < x < 2 and y <= -2:\n'
    '        return "Core CPI inflation trend cooling"\n'
    '    return "Core CPI inflation broadly neutral"\n'
    '\n'
    '\n'
    'def _core_cpi_inflation_signal_interpretation(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 6 and z >= 0:\n'
    '        return "Strong upside Core CPI inflation leading signal with supportive policy-pressure profile"\n'
    '    if y >= 6 and z < 0:\n'
    '        return "Strong upside Core CPI inflation leading signal, but warning profile is cooling"\n'
    '    if y >= 2 and z >= 0:\n'
    '        return "Moderately positive Core CPI inflation-pressure signal"\n'
    '    if y >= 2 and z < 0:\n'
    '        return "Core CPI inflation re-acceleration signal, but policy-pressure profile is cooling"\n'
    '    if y > -2 and z > -3 and z < 3:\n'
    '        return "Neutral/mixed Core CPI inflation signal"\n'
    '    if y <= -6 and z <= 0:\n'
    '        return "Strong disinflationary Core CPI inflation leading signal with downside confirmation"\n'
    '    if y <= -6 and z > 0:\n'
    '        return "Strong disinflationary Core CPI inflation leading signal, but policy pressure remains elevated"\n'
    '    if y <= -2 and z <= 0:\n'
    '        return "Moderately disinflationary Core CPI inflation signal"\n'
    '    if y <= -2 and z > 0:\n'
    '        return "Core CPI inflation cooling signal, but policy pressure remains elevated"\n'
    '    return "Mixed/transition Core CPI inflation signal"\n'
    '\n'
    '\n'
    'def _core_cpi_inflation_warning_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "High upside Core CPI inflation warning risk"\n'
    '    if z >= 3:\n'
    '        return "Moderate upside Core CPI inflation warning risk"\n'
    '    if z > -3:\n'
    '        return "Neutral Core CPI inflation warning profile"\n'
    '    if z > -6:\n'
    '        return "Moderate Core CPI disinflation warning risk"\n'
    '    return "High Core CPI disinflation / deflation warning risk"\n'
    '\n'
    '\n'
    'def _core_cpi_inflation_policy_signal_label(z: Any, g: Any, e: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, g, e)):\n'
    '        return ""\n'
    '    if g >= 0.03 and e >= 0.03 and z >= 6:\n'
    '        return "Persistent above-target Core CPI inflation pressure"\n'
    '    if g >= 0.03 and e < 0.025 and z <= 3:\n'
    '        return "Above-target Core CPI inflation but cooling"\n'
    '    if g < 0.03 and e >= 0.03 and z >= 3:\n'
    '        return "Near/below target but Core CPI inflation re-accelerating"\n'
    '    if g < 0.01 and z <= -5:\n'
    '        return "Core CPI disinflation / deflation-risk pressure"\n'
    '    if g >= 0.023 and z >= 5:\n'
    '        return "Core CPI inflation pressure becoming policy-relevant"\n'
    '    if z >= 6 and e >= 0.025:\n'
    '        return "Early upside Core CPI inflation-pressure warning"\n'
    '    return "Normal Core CPI inflation policy regime"\n'
    '\n'
    '\n'
    'def calculate_core_cpi_inflation_outputs(raw_df: Any) -> Any:\n'
    '    base_rows = df_records(calculate_core_inflation_outputs(raw_df))\n'
    '    rows = []\n'
    '    for base in base_rows:\n'
    '        x = base["Core Inflation Leading Score"]\n'
    '        y = base["Core Inflation Warning Composite Score"]\n'
    '        z = base["Core Inflation Final Warning Score"]\n'
    '        g = base["12M Inflation"]\n'
    '        e = base["3M Annualised Inflation"]\n'
    '        out = {\n'
    '            "Observation Count": base["Observation Count"],\n'
    '            "1M Annualised Inflation": base["1M Annualised Inflation"],\n'
    '            "3M Annualised Inflation": e,\n'
    '            "6M Annualised Inflation": base["6M Annualised Inflation"],\n'
    '            "12M Inflation": g,\n'
    '            "3M Change in 12M Inflation": base["3M Change in 12M Inflation"],\n'
    '            "6M Change in 12M Inflation": base["6M Change in 12M Inflation"],\n'
    '            "12M Core CPI Inflation PercentRank": base[\n'
    '                "12M Core Inflation PercentRank"\n'
    '            ],\n'
    '            "12M Core CPI Inflation Score": base["12M Core Inflation Score"],\n'
    '            "3M Annualised Core CPI Inflation PercentRank": base[\n'
    '                "3M Annualised Core Inflation PercentRank"\n'
    '            ],\n'
    '            "3M Annualised Core CPI Inflation Score": base[\n'
    '                "3M Annualised Core Inflation Score"\n'
    '            ],\n'
    '            "6M Annualised Core CPI Inflation PercentRank": base[\n'
    '                "6M Annualised Core Inflation PercentRank"\n'
    '            ],\n'
    '            "6M Annualised Core CPI Inflation Score": base[\n'
    '                "6M Annualised Core Inflation Score"\n'
    '            ],\n'
    '            "3M Change in 12M Core CPI Inflation PercentRank": base[\n'
    '                "3M Change in 12M Core Inflation PercentRank"\n'
    '            ],\n'
    '            "3M Change in 12M Core CPI Inflation Score": base[\n'
    '                "3M Change in 12M Core Inflation Score"\n'
    '            ],\n'
    '            "Distance From 2 Percent Target": base["Distance From 2 Percent Target"],\n'
    '            "Core CPI Inflation Target Status": base["Core Inflation Target Status"],\n'
    '            "Core CPI Inflation Momentum Composite Score": base[\n'
    '                "Core Inflation Momentum Composite Score"\n'
    '            ],\n'
    '            "Core CPI Inflation Recovery / Cooling Score": base[\n'
    '                "Core Inflation Recovery / Cooling Score"\n'
    '            ],\n'
    '            "Core CPI Inflation Target Pressure Score": base[\n'
    '                "Core Inflation Target Pressure Score"\n'
    '            ],\n'
    '            "Core CPI Inflation Outlier Score": base["Core Inflation Outlier Score"],\n'
    '            "Core CPI Inflation Leading Score": x,\n'
    '            "Core CPI Inflation Warning Composite Score": y,\n'
    '            "Core CPI Inflation Final Warning Score": z,\n'
    '            "Core CPI Inflation Pressure Regime": _core_cpi_inflation_regime(x),\n'
    '            "Core CPI Inflation Momentum Label": _core_cpi_inflation_momentum_label(\n'
    '                x, y\n'
    '            ),\n'
    '            "Core CPI Inflation Signal Interpretation": _core_cpi_inflation_signal_interpretation(\n'
    '                y, z\n'
    '            ),\n'
    '            "Reliability": base["Reliability"],\n'
    '            "Data Status": base["Data Status"],\n'
    '            "Core CPI Inflation Warning Risk Label": _core_cpi_inflation_warning_label(\n'
    '                z\n'
    '            ),\n'
    '            "Core CPI Inflation Policy Signal Label": _core_cpi_inflation_policy_signal_label(\n'
    '                z, g, e\n'
    '            ),\n'
    '            "12M Core CPI Inflation Z-Score": base["12M Core Inflation Z-Score"],\n'
    '            "3M Annualised Core CPI Inflation Z-Score": base[\n'
    '                "3M Annualised Core Inflation Z-Score"\n'
    '            ],\n'
    '            "6M Annualised Core CPI Inflation Z-Score": base[\n'
    '                "6M Annualised Core Inflation Z-Score"\n'
    '            ],\n'
    '            "3M Change in 12M Core CPI Inflation Z-Score": base[\n'
    '                "3M Change in 12M Core Inflation Z-Score"\n'
    '            ],\n'
    '            "Core CPI Inflation Outlier Label": base["Core Inflation Outlier Label"],\n'
    '        }\n'
    '        rows.append(out)\n'
    '    return make_dataframe(rows, CORE_CPI_INFLATION_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _headline_cpi_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A18-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _headline_cpi_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    """A18-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _headline_cpi_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _headline_cpi_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _headline_cpi_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _headline_cpi_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        "" if is_blank(rank) else _headline_cpi_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _headline_cpi_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _headline_cpi_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _headline_cpi_impulse_score(m: Any, o: Any, q: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (m, o, q)):\n'
    '        return ""\n'
    '    return _headline_cpi_sheets_round(clamp(0.50 * m + 0.30 * o + 0.20 * q, -10, 10), 2)\n'
    '\n'
    '\n'
    'def _headline_cpi_coincident_score(\n'
    '    k: Any, m: Any, o: Any, q: Any, r_internal: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(value) for value in (k, m, o, q, r_internal)):\n'
    '        return ""\n'
    '    target_gap_component = clamp(r_internal * 500, -10, 10)\n'
    '    return _headline_cpi_sheets_round(\n'
    '        clamp(\n'
    '            0.30 * k + 0.30 * m + 0.20 * o + 0.10 * q + 0.10 * target_gap_component,\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _headline_cpi_leading_score(k: Any, m: Any, o: Any, q: Any, t: Any, u: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (k, m, o, q, t, u)):\n'
    '        return ""\n'
    '    return _headline_cpi_sheets_round(\n'
    '        clamp(0.10 * k + 0.30 * m + 0.20 * o + 0.20 * q + 0.15 * t + 0.05 * u, -10, 10),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _headline_cpi_warning_score(u: Any, v: Any, w: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (u, v, w)):\n'
    '        return ""\n'
    '    return _headline_cpi_sheets_round(clamp(0.30 * u + 0.40 * v + 0.30 * w, -10, 10), 2)\n'
    '\n'
    '\n'
    'def _headline_cpi_inflation_regime(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 2:\n'
    '        return "Headline CPI inflation pressure rising"\n'
    '    if x <= -2:\n'
    '        return "Headline CPI inflation pressure easing"\n'
    '    return "Neutral headline CPI inflation pressure"\n'
    '\n'
    '\n'
    'def _headline_cpi_inflation_momentum_label(x: Any, y: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y)):\n'
    '        return ""\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Headline CPI inflation pressure rising"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Headline CPI elevated but cooling"\n'
    '    if x >= 2 and -2 < y < 2:\n'
    '        return "Headline CPI inflation pressure elevated / stable"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Headline CPI weak but re-accelerating"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Headline CPI inflation pressure easing"\n'
    '    if x <= -2 and -2 < y < 2:\n'
    '        return "Headline CPI inflation pressure weak / stable"\n'
    '    if -2 < x < 2 and y >= 2:\n'
    '        return "Headline CPI trend re-accelerating"\n'
    '    if -2 < x < 2 and y <= -2:\n'
    '        return "Headline CPI trend cooling"\n'
    '    return "Headline CPI broadly neutral"\n'
    '\n'
    '\n'
    'def _headline_cpi_inflation_signal_interpretation(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 6 and z >= 0:\n'
    '        return "Strong upside headline CPI leading signal with supportive inflation-risk profile"\n'
    '    if y >= 6 and z < 0:\n'
    '        return (\n'
    '            "Strong upside headline CPI leading signal, but warning profile is cooling"\n'
    '        )\n'
    '    if y >= 2 and z >= 0:\n'
    '        return "Moderately positive headline CPI inflation-pressure signal"\n'
    '    if y >= 2 and z < 0:\n'
    '        return (\n'
    '            "Headline CPI re-acceleration signal, but inflation-risk profile is cooling"\n'
    '        )\n'
    '    if y > -2 and z > -3 and z < 3:\n'
    '        return "Neutral/mixed headline CPI signal"\n'
    '    if y <= -6 and z <= 0:\n'
    '        return "Strong disinflationary headline CPI leading signal with downside confirmation"\n'
    '    if y <= -6 and z > 0:\n'
    '        return "Strong disinflationary headline CPI leading signal, but inflation pressure remains elevated"\n'
    '    if y <= -2 and z <= 0:\n'
    '        return "Moderately disinflationary headline CPI signal"\n'
    '    if y <= -2 and z > 0:\n'
    '        return "Headline CPI cooling signal, but inflation pressure remains elevated"\n'
    '    return "Mixed/transition headline CPI signal"\n'
    '\n'
    '\n'
    'def _headline_cpi_inflation_warning_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "High upside headline CPI warning risk"\n'
    '    if z >= 3:\n'
    '        return "Moderate upside headline CPI warning risk"\n'
    '    if z > -3:\n'
    '        return "Neutral headline CPI warning profile"\n'
    '    if z > -6:\n'
    '        return "Moderate headline CPI disinflation warning risk"\n'
    '    return "High headline CPI disinflation / deflation warning risk"\n'
    '\n'
    '\n'
    'def _headline_cpi_inflation_policy_signal_label(z: Any, g: Any, e: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, g, e)):\n'
    '        return ""\n'
    '    if g >= 0.03 and e >= 0.03 and z >= 6:\n'
    '        return "Persistent above-target headline CPI pressure"\n'
    '    if g >= 0.03 and e < 0.025 and z <= 3:\n'
    '        return "Above-target headline CPI but cooling"\n'
    '    if g < 0.03 and e >= 0.03 and z >= 3:\n'
    '        return "Near/below target but headline CPI re-accelerating"\n'
    '    if g < 0.01 and z <= -5:\n'
    '        return "Headline CPI disinflation / deflation-risk pressure"\n'
    '    if g >= 0.023 and z >= 5:\n'
    '        return "Headline CPI pressure becoming policy-relevant"\n'
    '    if z >= 6 and e >= 0.025:\n'
    '        return "Early upside headline CPI pressure warning"\n'
    '    return "Normal headline CPI inflation regime"\n'
    '\n'
    '\n'
    'def _headline_cpi_inflation_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside headline CPI inflation pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside headline CPI inflation pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside headline CPI disinflation signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside headline CPI disinflation signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_headline_cpi_inflation_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    sample_counts = _metric_counts(raw_values)\n'
    '    one_month: list[Any] = []\n'
    '    three_month: list[Any] = []\n'
    '    six_month: list[Any] = []\n'
    '    yoy: list[Any] = []\n'
    '    three_month_yoy_change: list[Any] = []\n'
    '    six_month_yoy_change: list[Any] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        count = sample_counts[idx]\n'
    '        one_month.append(\n'
    '            _safe_growth(current, raw_values[idx - 1], 12)\n'
    '            if count >= 2 and idx >= 1 and current is not None\n'
    '            else ""\n'
    '        )\n'
    '        three_month.append(\n'
    '            _safe_growth(current, raw_values[idx - 3], 4)\n'
    '            if count >= 4 and idx >= 3\n'
    '            else ""\n'
    '        )\n'
    '        six_month.append(\n'
    '            _safe_growth(current, raw_values[idx - 6], 2)\n'
    '            if count >= 7 and idx >= 6\n'
    '            else ""\n'
    '        )\n'
    '        yoy.append(\n'
    '            _safe_growth(current, raw_values[idx - 12])\n'
    '            if count >= 13 and idx >= 12\n'
    '            else ""\n'
    '        )\n'
    '        three_month_yoy_change.append(\n'
    '            yoy[idx] - yoy[idx - 3]\n'
    '            if count >= 16\n'
    '            and idx >= 3\n'
    '            and not is_blank(yoy[idx])\n'
    '            and not is_blank(yoy[idx - 3])\n'
    '            else ""\n'
    '        )\n'
    '        six_month_yoy_change.append(\n'
    '            yoy[idx] - yoy[idx - 6]\n'
    '            if count >= 19\n'
    '            and idx >= 6\n'
    '            and not is_blank(yoy[idx])\n'
    '            and not is_blank(yoy[idx - 6])\n'
    '            else ""\n'
    '        )\n'
    '\n'
    '    yoy_ranks = _headline_cpi_metric_ranks_sheets(yoy)\n'
    '    three_month_ranks = _headline_cpi_metric_ranks_sheets(three_month)\n'
    '    six_month_ranks = _headline_cpi_metric_ranks_sheets(six_month)\n'
    '    momentum_ranks = _headline_cpi_metric_ranks_sheets(three_month_yoy_change)\n'
    '    yoy_zscores = expanding_zscore_sample(yoy, sample_counts, 60)\n'
    '    three_month_zscores = expanding_zscore_sample(three_month, sample_counts, 60)\n'
    '    six_month_zscores = expanding_zscore_sample(six_month, sample_counts, 60)\n'
    '    momentum_zscores = expanding_zscore_sample(\n'
    '        three_month_yoy_change, sample_counts, 60\n'
    '    )\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = sample_counts[idx]\n'
    '        e = three_month[idx]\n'
    '        f = six_month[idx]\n'
    '        g = yoy[idx]\n'
    '        h_internal = three_month_yoy_change[idx]\n'
    '        i_internal = six_month_yoy_change[idx]\n'
    '        j = yoy_ranks[idx]\n'
    '        k = _headline_cpi_score_percentile(j)\n'
    '        l = three_month_ranks[idx]\n'
    '        m = _headline_cpi_score_percentile(l)\n'
    '        n = six_month_ranks[idx]\n'
    '        o = _headline_cpi_score_percentile(n)\n'
    '        p = momentum_ranks[idx]\n'
    '        q = _headline_cpi_score_percentile(p)\n'
    '        r_internal = g - 0.02 if not is_blank(g) else ""\n'
    '        t = _headline_cpi_impulse_score(m, o, q)\n'
    '        u = _core_inflation_recovery_cooling_score(\n'
    '            g, h_internal, i_internal, r_internal\n'
    '        )\n'
    '        v = _core_inflation_target_pressure_score(r_internal, g, e, f)\n'
    '        zs = [\n'
    '            yoy_zscores[idx],\n'
    '            three_month_zscores[idx],\n'
    '            six_month_zscores[idx],\n'
    '            momentum_zscores[idx],\n'
    '        ]\n'
    '        w = _core_inflation_outlier_score(zs)\n'
    '        x = _headline_cpi_coincident_score(k, m, o, q, r_internal)\n'
    '        y = _headline_cpi_leading_score(k, m, o, q, t, u)\n'
    '        z = _headline_cpi_warning_score(u, v, w)\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if c < 60\n'
    '                else ("Medium reliability" if c < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        signal_validity = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if c < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(value) for value in (x, y, z))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Sample Count": c,\n'
    '                "1M Annualised Inflation": one_month[idx],\n'
    '                "3M Annualised Inflation": e,\n'
    '                "6M Annualised Inflation": f,\n'
    '                "YoY Inflation": g,\n'
    '                "3M Change in YoY Inflation": _headline_cpi_percent_display(h_internal),\n'
    '                "6M Change in YoY Inflation": _headline_cpi_percent_display(i_internal),\n'
    '                "YoY Inflation Percentile": j,\n'
    '                "YoY Inflation Score": k,\n'
    '                "3M Inflation Percentile": l,\n'
    '                "3M Inflation Score": m,\n'
    '                "6M Inflation Percentile": n,\n'
    '                "6M Inflation Score": o,\n'
    '                "Momentum Percentile": p,\n'
    '                "Momentum Score": q,\n'
    '                "YoY Inflation Target Gap": _headline_cpi_percent_display(r_internal),\n'
    '                "Target Regime": _core_inflation_target_status(g, e),\n'
    '                "Inflation Impulse Score": t,\n'
    '                "Inflation Cycle Turning Point Score": u,\n'
    '                "Policy Pressure Score": v,\n'
    '                "Inflation Extreme Score": w,\n'
    '                "Coincident Score": x,\n'
    '                "Leading Score": y,\n'
    '                "Warning Score": z,\n'
    '                "State": _headline_cpi_inflation_regime(x),\n'
    '                "Regime": _headline_cpi_inflation_momentum_label(x, y),\n'
    '                "Signal Comment": _headline_cpi_inflation_signal_interpretation(y, z),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": signal_validity,\n'
    '                "Warning Flag": _headline_cpi_inflation_warning_label(z),\n'
    '                "Policy flag": _headline_cpi_inflation_policy_signal_label(z, g, e),\n'
    '                "YoY Inflation Z-Score": zs[0],\n'
    '                "3M Inflation Z-Score": zs[1],\n'
    '                "6M Inflation Z-Score": zs[2],\n'
    '                "Momentum Z-Score": zs[3],\n'
    '                "Extreme Flag": _headline_cpi_inflation_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, HEADLINE_CPI_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _core_ppi_regime(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 2:\n'
    '        return "Core PPI pressure rising"\n'
    '    if x <= -2:\n'
    '        return "Core PPI pressure easing"\n'
    '    return "Neutral Core PPI pressure"\n'
    '\n'
    '\n'
    'def _core_ppi_momentum_label(x: Any, y: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y)):\n'
    '        return ""\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Core PPI pressure rising"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Core PPI elevated but cooling"\n'
    '    if x >= 2 and -2 < y < 2:\n'
    '        return "Core PPI pressure elevated / stable"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Core PPI weak but re-accelerating"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Core PPI pressure easing"\n'
    '    if x <= -2 and -2 < y < 2:\n'
    '        return "Core PPI pressure weak / stable"\n'
    '    if -2 < x < 2 and y >= 2:\n'
    '        return "Core PPI trend re-accelerating"\n'
    '    if -2 < x < 2 and y <= -2:\n'
    '        return "Core PPI trend cooling"\n'
    '    return "Core PPI broadly neutral"\n'
    '\n'
    '\n'
    'def _core_ppi_signal_interpretation(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 6 and z >= 0:\n'
    '        return "Strong upside Core PPI leading signal with supportive pipeline-pressure profile"\n'
    '    if y >= 6 and z < 0:\n'
    '        return "Strong upside Core PPI leading signal, but warning profile is cooling"\n'
    '    if y >= 2 and z >= 0:\n'
    '        return "Moderately positive Core PPI pipeline-inflation signal"\n'
    '    if y >= 2 and z < 0:\n'
    '        return (\n'
    '            "Core PPI re-acceleration signal, but pipeline-pressure profile is cooling"\n'
    '        )\n'
    '    if y > -2 and z > -3 and z < 3:\n'
    '        return "Neutral/mixed Core PPI signal"\n'
    '    if y <= -6 and z <= 0:\n'
    '        return (\n'
    '            "Strong disinflationary Core PPI leading signal with downside confirmation"\n'
    '        )\n'
    '    if y <= -6 and z > 0:\n'
    '        return "Strong disinflationary Core PPI leading signal, but pipeline pressure remains elevated"\n'
    '    if y <= -2 and z <= 0:\n'
    '        return "Moderately disinflationary Core PPI signal"\n'
    '    if y <= -2 and z > 0:\n'
    '        return "Core PPI cooling signal, but pipeline pressure remains elevated"\n'
    '    return "Mixed/transition Core PPI signal"\n'
    '\n'
    '\n'
    'def _core_ppi_warning_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "High upside Core PPI warning risk"\n'
    '    if z >= 3:\n'
    '        return "Moderate upside Core PPI warning risk"\n'
    '    if z > -3:\n'
    '        return "Neutral Core PPI warning profile"\n'
    '    if z > -6:\n'
    '        return "Moderate Core PPI disinflation warning risk"\n'
    '    return "High Core PPI disinflation / deflation warning risk"\n'
    '\n'
    '\n'
    'def _core_ppi_policy_signal_label(z: Any, g: Any, e: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, g, e)):\n'
    '        return ""\n'
    '    if g >= 0.03 and e >= 0.03 and z >= 6:\n'
    '        return "Persistent upside Core PPI pipeline pressure"\n'
    '    if g >= 0.03 and e < 0.025 and z <= 3:\n'
    '        return "Elevated Core PPI but cooling"\n'
    '    if g < 0.03 and e >= 0.03 and z >= 3:\n'
    '        return "Moderate Core PPI but re-accelerating"\n'
    '    if g < 0.01 and z <= -5:\n'
    '        return "Core PPI disinflation / producer-price weakness"\n'
    '    if g >= 0.023 and z >= 5:\n'
    '        return "Core PPI pressure becoming inflation-relevant"\n'
    '    if z >= 6 and e >= 0.025:\n'
    '        return "Early upside Core PPI pipeline-pressure warning"\n'
    '    return "Normal Core PPI inflation regime"\n'
    '\n'
    '\n'
    'def _core_ppi_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside Core PPI pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside Core PPI pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside Core PPI disinflation signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside Core PPI disinflation signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_core_ppi_outputs(raw_df: Any) -> Any:\n'
    '    base_rows = df_records(calculate_core_inflation_outputs(raw_df))\n'
    '    rows = []\n'
    '    for base in base_rows:\n'
    '        k = base["12M Core Inflation Score"]\n'
    '        m = base["3M Annualised Core Inflation Score"]\n'
    '        o = base["6M Annualised Core Inflation Score"]\n'
    '        q = base["3M Change in 12M Core Inflation Score"]\n'
    '        r = base["Distance From 2 Percent Target"]\n'
    '        u = base["Core Inflation Recovery / Cooling Score"]\n'
    '        v = base["Core Inflation Target Pressure Score"]\n'
    '        w = base["Core Inflation Outlier Score"]\n'
    '        t = (\n'
    '            round(clamp(0.45 * m + 0.35 * o + 0.2 * q, -10, 10), 2)\n'
    '            if all(x != "" for x in (m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        target_gap_component = clamp(r * 500, -10, 10) if r != "" else ""\n'
    '        x = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.35 * k\n'
    '                    + 0.25 * m\n'
    '                    + 0.2 * o\n'
    '                    + 0.1 * q\n'
    '                    + 0.1 * target_gap_component,\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(z != "" for z in (k, m, o, q, target_gap_component))\n'
    '            else ""\n'
    '        )\n'
    '        y = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.1 * k + 0.35 * m + 0.25 * o + 0.2 * q + 0.05 * t + 0.05 * u,\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(z != "" for z in (k, m, o, q, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            round(clamp(0.35 * u + 0.45 * v + 0.2 * w, -10, 10), 2)\n'
    '            if all(val != "" for val in (u, v, w))\n'
    '            else ""\n'
    '        )\n'
    '        c = base["Observation Count"]\n'
    '        g = base["12M Inflation"]\n'
    '        e = base["3M Annualised Inflation"]\n'
    '        data_status = base["Data Status"]\n'
    '        if data_status != "Missing Data":\n'
    '            data_status = (\n'
    '                "Low Sample"\n'
    '                if c < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(val == "" for val in (x, y, z))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        zs = [\n'
    '            base["12M Core Inflation Z-Score"],\n'
    '            base["3M Annualised Core Inflation Z-Score"],\n'
    '            base["6M Annualised Core Inflation Z-Score"],\n'
    '            base["3M Change in 12M Core Inflation Z-Score"],\n'
    '        ]\n'
    '        rows.append(\n'
    '            {\n'
    '                "Observation Count": c,\n'
    '                "1M Annualised Inflation": base["1M Annualised Inflation"],\n'
    '                "3M Annualised Inflation": e,\n'
    '                "6M Annualised Inflation": base["6M Annualised Inflation"],\n'
    '                "12M Inflation": g,\n'
    '                "3M Change in 12M Inflation": base["3M Change in 12M Inflation"],\n'
    '                "6M Change in 12M Inflation": base["6M Change in 12M Inflation"],\n'
    '                "12M Core PPI PercentRank": base["12M Core Inflation PercentRank"],\n'
    '                "12M Core PPI Score": k,\n'
    '                "3M Annualised Core PPI PercentRank": base[\n'
    '                    "3M Annualised Core Inflation PercentRank"\n'
    '                ],\n'
    '                "3M Annualised Core PPI Score": m,\n'
    '                "6M Annualised Core PPI PercentRank": base[\n'
    '                    "6M Annualised Core Inflation PercentRank"\n'
    '                ],\n'
    '                "6M Annualised Core PPI Score": o,\n'
    '                "3M Change in 12M Core PPI PercentRank": base[\n'
    '                    "3M Change in 12M Core Inflation PercentRank"\n'
    '                ],\n'
    '                "3M Change in 12M Core PPI Score": q,\n'
    '                "Distance From 2 Percent Target": r,\n'
    '                "Core PPI Target Status": base["Core Inflation Target Status"],\n'
    '                "Core PPI Momentum Composite Score": t,\n'
    '                "Core PPI Recovery / Cooling Score": u,\n'
    '                "Core PPI Target Pressure Score": v,\n'
    '                "Core PPI Outlier Score": w,\n'
    '                "Core PPI Leading Score": x,\n'
    '                "Core PPI Warning Composite Score": y,\n'
    '                "Core PPI Final Warning Score": z,\n'
    '                "Core PPI Pressure Regime": _core_ppi_regime(x),\n'
    '                "Core PPI Momentum Label": _core_ppi_momentum_label(x, y),\n'
    '                "Core PPI Signal Interpretation": _core_ppi_signal_interpretation(y, z),\n'
    '                "Reliability": base["Reliability"],\n'
    '                "Data Status": data_status,\n'
    '                "Core PPI Warning Risk Label": _core_ppi_warning_label(z),\n'
    '                "Core PPI Policy Signal Label": _core_ppi_policy_signal_label(z, g, e),\n'
    '                "12M Core PPI Z-Score": zs[0],\n'
    '                "3M Annualised Core PPI Z-Score": zs[1],\n'
    '                "6M Annualised Core PPI Z-Score": zs[2],\n'
    '                "3M Change in 12M Core PPI Z-Score": zs[3],\n'
    '                "Core PPI Outlier Label": _core_ppi_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, CORE_PPI_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _headline_ppi_regime(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 2:\n'
    '        return "Headline PPI pressure rising"\n'
    '    if x <= -2:\n'
    '        return "Headline PPI pressure easing"\n'
    '    return "Neutral Headline PPI pressure"\n'
    '\n'
    '\n'
    'def _headline_ppi_momentum_label(x: Any, y: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y)):\n'
    '        return ""\n'
    '    if x >= 2 and y >= 2:\n'
    '        return "Headline PPI pressure rising"\n'
    '    if x >= 2 and y <= -2:\n'
    '        return "Headline PPI elevated but cooling"\n'
    '    if x >= 2 and -2 < y < 2:\n'
    '        return "Headline PPI pressure elevated / stable"\n'
    '    if x <= -2 and y >= 2:\n'
    '        return "Headline PPI weak but re-accelerating"\n'
    '    if x <= -2 and y <= -2:\n'
    '        return "Headline PPI pressure easing"\n'
    '    if x <= -2 and -2 < y < 2:\n'
    '        return "Headline PPI pressure weak / stable"\n'
    '    if -2 < x < 2 and y >= 2:\n'
    '        return "Headline PPI trend re-accelerating"\n'
    '    if -2 < x < 2 and y <= -2:\n'
    '        return "Headline PPI trend cooling"\n'
    '    return "Headline PPI broadly neutral"\n'
    '\n'
    '\n'
    'def _headline_ppi_signal_interpretation(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 6 and z >= 0:\n'
    '        return "Strong upside Headline PPI leading signal with supportive pipeline-pressure profile"\n'
    '    if y >= 6 and z < 0:\n'
    '        return (\n'
    '            "Strong upside Headline PPI leading signal, but warning profile is cooling"\n'
    '        )\n'
    '    if y >= 2 and z >= 0:\n'
    '        return "Moderately positive Headline PPI pipeline-inflation signal"\n'
    '    if y >= 2 and z < 0:\n'
    '        return "Headline PPI re-acceleration signal, but pipeline-pressure profile is cooling"\n'
    '    if y > -2 and z > -3 and z < 3:\n'
    '        return "Neutral/mixed Headline PPI signal"\n'
    '    if y <= -6 and z <= 0:\n'
    '        return "Strong disinflationary Headline PPI leading signal with downside confirmation"\n'
    '    if y <= -6 and z > 0:\n'
    '        return "Strong disinflationary Headline PPI leading signal, but pipeline pressure remains elevated"\n'
    '    if y <= -2 and z <= 0:\n'
    '        return "Moderately disinflationary Headline PPI signal"\n'
    '    if y <= -2 and z > 0:\n'
    '        return "Headline PPI cooling signal, but pipeline pressure remains elevated"\n'
    '    return "Mixed/transition Headline PPI signal"\n'
    '\n'
    '\n'
    'def _headline_ppi_warning_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "High upside Headline PPI warning risk"\n'
    '    if z >= 3:\n'
    '        return "Moderate upside Headline PPI warning risk"\n'
    '    if z > -3:\n'
    '        return "Neutral Headline PPI warning profile"\n'
    '    if z > -6:\n'
    '        return "Moderate Headline PPI disinflation warning risk"\n'
    '    return "High Headline PPI disinflation / deflation warning risk"\n'
    '\n'
    '\n'
    'def _headline_ppi_policy_signal_label(z: Any, g: Any, e: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, g, e)):\n'
    '        return ""\n'
    '    if g >= 0.03 and e >= 0.03 and z >= 6:\n'
    '        return "Persistent upside Headline PPI pipeline pressure"\n'
    '    if g >= 0.03 and e < 0.025 and z <= 3:\n'
    '        return "Elevated Headline PPI but cooling"\n'
    '    if g < 0.03 and e >= 0.03 and z >= 3:\n'
    '        return "Moderate Headline PPI but re-accelerating"\n'
    '    if g < 0.01 and z <= -5:\n'
    '        return "Headline PPI disinflation / producer-price weakness"\n'
    '    if g >= 0.023 and z >= 5:\n'
    '        return "Headline PPI pressure becoming inflation-relevant"\n'
    '    if z >= 6 and e >= 0.025:\n'
    '        return "Early upside Headline PPI pipeline-pressure warning"\n'
    '    return "Normal Headline PPI inflation regime"\n'
    '\n'
    '\n'
    'def _headline_ppi_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside Headline PPI pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside Headline PPI pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside Headline PPI disinflation signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside Headline PPI disinflation signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_headline_ppi_outputs(raw_df: Any) -> Any:\n'
    '    base_rows = df_records(calculate_core_inflation_outputs(raw_df))\n'
    '    rows = []\n'
    '    for base in base_rows:\n'
    '        k = base["12M Core Inflation Score"]\n'
    '        m = base["3M Annualised Core Inflation Score"]\n'
    '        o = base["6M Annualised Core Inflation Score"]\n'
    '        q = base["3M Change in 12M Core Inflation Score"]\n'
    '        r = base["Distance From 2 Percent Target"]\n'
    '        u = base["Core Inflation Recovery / Cooling Score"]\n'
    '        v = base["Core Inflation Target Pressure Score"]\n'
    '        w = base["Core Inflation Outlier Score"]\n'
    '        t = (\n'
    '            round(clamp(0.5 * m + 0.3 * o + 0.2 * q, -10, 10), 2)\n'
    '            if all(x != "" for x in (m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        target_gap_component = clamp(r * 500, -10, 10) if r != "" else ""\n'
    '        x = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.3 * k + 0.3 * m + 0.2 * o + 0.1 * q + 0.1 * target_gap_component,\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(z != "" for z in (k, m, o, q, target_gap_component))\n'
    '            else ""\n'
    '        )\n'
    '        y = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.1 * k + 0.3 * m + 0.2 * o + 0.2 * q + 0.15 * t + 0.05 * u, -10, 10\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(z != "" for z in (k, m, o, q, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            round(clamp(0.3 * u + 0.4 * v + 0.3 * w, -10, 10), 2)\n'
    '            if all(val != "" for val in (u, v, w))\n'
    '            else ""\n'
    '        )\n'
    '        c = base["Observation Count"]\n'
    '        g = base["12M Inflation"]\n'
    '        e = base["3M Annualised Inflation"]\n'
    '        data_status = base["Data Status"]\n'
    '        if data_status != "Missing Data":\n'
    '            data_status = (\n'
    '                "Low Sample"\n'
    '                if c < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(val == "" for val in (x, y, z))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        zs = [\n'
    '            base["12M Core Inflation Z-Score"],\n'
    '            base["3M Annualised Core Inflation Z-Score"],\n'
    '            base["6M Annualised Core Inflation Z-Score"],\n'
    '            base["3M Change in 12M Core Inflation Z-Score"],\n'
    '        ]\n'
    '        rows.append(\n'
    '            {\n'
    '                "Observation Count": c,\n'
    '                "1M Annualised Inflation": base["1M Annualised Inflation"],\n'
    '                "3M Annualised Inflation": e,\n'
    '                "6M Annualised Inflation": base["6M Annualised Inflation"],\n'
    '                "12M Inflation": g,\n'
    '                "3M Change in 12M Inflation": base["3M Change in 12M Inflation"],\n'
    '                "6M Change in 12M Inflation": base["6M Change in 12M Inflation"],\n'
    '                "12M Headline PPI PercentRank": base["12M Core Inflation PercentRank"],\n'
    '                "12M Headline PPI Score": k,\n'
    '                "3M Annualised Headline PPI PercentRank": base[\n'
    '                    "3M Annualised Core Inflation PercentRank"\n'
    '                ],\n'
    '                "3M Annualised Headline PPI Score": m,\n'
    '                "6M Annualised Headline PPI PercentRank": base[\n'
    '                    "6M Annualised Core Inflation PercentRank"\n'
    '                ],\n'
    '                "6M Annualised Headline PPI Score": o,\n'
    '                "3M Change in 12M Headline PPI PercentRank": base[\n'
    '                    "3M Change in 12M Core Inflation PercentRank"\n'
    '                ],\n'
    '                "3M Change in 12M Headline PPI Score": q,\n'
    '                "Distance From 2 Percent Target": r,\n'
    '                "Headline PPI Target Status": base["Core Inflation Target Status"],\n'
    '                "Headline PPI Momentum Composite Score": t,\n'
    '                "Headline PPI Recovery / Cooling Score": u,\n'
    '                "Headline PPI Target Pressure Score": v,\n'
    '                "Headline PPI Outlier Score": w,\n'
    '                "Headline PPI Leading Score": x,\n'
    '                "Headline PPI Warning Composite Score": y,\n'
    '                "Headline PPI Final Warning Score": z,\n'
    '                "Headline PPI Pressure Regime": _headline_ppi_regime(x),\n'
    '                "Headline PPI Momentum Label": _headline_ppi_momentum_label(x, y),\n'
    '                "Headline PPI Signal Interpretation": _headline_ppi_signal_interpretation(\n'
    '                    y, z\n'
    '                ),\n'
    '                "Reliability": base["Reliability"],\n'
    '                "Data Status": data_status,\n'
    '                "Headline PPI Warning Risk Label": _headline_ppi_warning_label(z),\n'
    '                "Headline PPI Policy Signal Label": _headline_ppi_policy_signal_label(\n'
    '                    z, g, e\n'
    '                ),\n'
    '                "12M Headline PPI Z-Score": zs[0],\n'
    '                "3M Annualised Headline PPI Z-Score": zs[1],\n'
    '                "6M Annualised Headline PPI Z-Score": zs[2],\n'
    '                "3M Change in 12M Headline PPI Z-Score": zs[3],\n'
    '                "Headline PPI Outlier Label": _headline_ppi_outlier_label(zs),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, HEADLINE_PPI_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _wage_pressure_status(f: Any, e: Any) -> str:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if not is_blank(e) and f >= 0.05 and e >= 0.05:\n'
    '        return "Elevated and persistent wage pressure"\n'
    '    if not is_blank(e) and f >= 0.05 and e < 0.04:\n'
    '        return "Elevated but cooling wage pressure"\n'
    '    if not is_blank(e) and f < 0.05 and e >= 0.05:\n'
    '        return "Wage pressure re-accelerating"\n'
    '    if f >= 0.04:\n'
    '        return "Policy-relevant wage pressure"\n'
    '    if f >= 0.035:\n'
    '        return "Moderately elevated wage pressure"\n'
    '    if f >= 0.025:\n'
    '        return "Neutral wage pressure"\n'
    '    if f >= 0.015:\n'
    '        return "Soft wage pressure"\n'
    '    return "Wage disinflation risk"\n'
    '\n'
    '\n'
    'def _wage_recovery_cooling_score(f: Any, g: Any, h: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (f, g, h)):\n'
    '        return ""\n'
    '    if f < 0.03 and g > 0 and h > 0:\n'
    '        return 6\n'
    '    if f >= 0.03 and g > 0 and h > 0:\n'
    '        return 5\n'
    '    if f > 0.03 and g < 0 and h < 0:\n'
    '        return -5\n'
    '    if f < 0.03 and g < 0 and h < 0:\n'
    '        return -6\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _wage_persistence_score(d: Any, e: Any, f: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (d, e, f)):\n'
    '        return ""\n'
    '    if f >= 0.04 and d > 0.04 and e > 0.04:\n'
    '        return 6\n'
    '    if f >= 0.035 and d > 0.035 and e > 0.035:\n'
    '        return 4\n'
    '    if f <= 0.02 and d < 0.02 and e < 0.02:\n'
    '        return -6\n'
    '    if f <= 0.025 and d < 0.025 and e < 0.025:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _wage_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _wage_pressure_regime(v: Any) -> str:\n'
    '    if is_blank(v):\n'
    '        return ""\n'
    '    if v >= 2:\n'
    '        return "Wage pressure rising"\n'
    '    if v <= -2:\n'
    '        return "Wage pressure easing"\n'
    '    return "Neutral wage pressure"\n'
    '\n'
    '\n'
    'def _wage_pressure_momentum_label(v: Any, w: Any) -> str:\n'
    '    if any(is_blank(x) for x in (v, w)):\n'
    '        return ""\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Wage pressure rising"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Wage pressure elevated but cooling"\n'
    '    if v >= 2 and -2 < w < 2:\n'
    '        return "Wage pressure elevated / stable"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Wage pressure weak but re-accelerating"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Wage pressure easing"\n'
    '    if v <= -2 and -2 < w < 2:\n'
    '        return "Wage pressure weak / stable"\n'
    '    if -2 < v < 2 and w >= 2:\n'
    '        return "Wage pressure trend re-accelerating"\n'
    '    if -2 < v < 2 and w <= -2:\n'
    '        return "Wage pressure trend cooling"\n'
    '    return "Wage pressure broadly neutral"\n'
    '\n'
    '\n'
    'def _wage_pressure_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return "Strong upside wage-pressure leading signal with supportive labour-cost pressure profile"\n'
    '    if w >= 6 and x < 0:\n'
    '        return (\n'
    '            "Strong upside wage-pressure leading signal, but warning profile is cooling"\n'
    '        )\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive wage-pressure signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Wage-pressure re-acceleration signal, but warning profile is cooling"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed wage-pressure signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong wage-disinflation leading signal with downside confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong wage-disinflation leading signal, but labour-cost pressure remains elevated"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately disinflationary wage-pressure signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return "Wage-pressure cooling signal, but labour-cost pressure remains elevated"\n'
    '    return "Mixed/transition wage-pressure signal"\n'
    '\n'
    '\n'
    'def _wage_pressure_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 6:\n'
    '        return "High upside wage-pressure warning risk"\n'
    '    if x >= 3:\n'
    '        return "Moderate upside wage-pressure warning risk"\n'
    '    if x > -3:\n'
    '        return "Neutral wage-pressure warning profile"\n'
    '    if x > -6:\n'
    '        return "Moderate wage-disinflation warning risk"\n'
    '    return "High wage-disinflation warning risk"\n'
    '\n'
    '\n'
    'def _wage_pressure_policy_signal_label(x: Any, f: Any, d: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, f, d)):\n'
    '        return ""\n'
    '    if f >= 0.05 and d >= 0.05 and x >= 6:\n'
    '        return "Persistent wage-driven inflation pressure"\n'
    '    if f >= 0.05 and d < 0.04 and x <= 3:\n'
    '        return "Wage pressure elevated but cooling"\n'
    '    if f < 0.05 and d >= 0.05 and x >= 3:\n'
    '        return "Wage pressure re-accelerating"\n'
    '    if f >= 0.04 and x >= 5:\n'
    '        return "Wage pressure becoming policy-relevant"\n'
    '    if x >= 6 and d >= 0.04:\n'
    '        return "Early wage inflation-pressure warning"\n'
    '    if f < 0.02 and x <= -5:\n'
    '        return "Wage disinflation risk"\n'
    '    return "Normal wage-pressure regime"\n'
    '\n'
    '\n'
    'def _wage_pressure_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside wage pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside wage pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside wage-disinflation signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside wage-disinflation signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _rounded_expanding_zscores(\n'
    '    values: list[object], counts: list[int], min_count: int\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, counts, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def calculate_wage_growth_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    observation_counts = _metric_counts(raw_values)\n'
    '    one_period_growth: list[Any] = []\n'
    '    two_period_growth: list[Any] = []\n'
    '    four_period_growth: list[Any] = []\n'
    '    one_period_change: list[Any] = []\n'
    '    two_period_change: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = observation_counts[idx]\n'
    '        one_period_growth.append(\n'
    '            ""\n'
    '            if c < 2 or current is None or idx < 1\n'
    '            else _safe_growth(current, raw_values[idx - 1], 4)\n'
    '        )\n'
    '        two_period_growth.append(\n'
    '            ""\n'
    '            if c < 3 or current is None or idx < 2\n'
    '            else _safe_growth(current, raw_values[idx - 2], 2)\n'
    '        )\n'
    '        four_period_growth.append(\n'
    '            ""\n'
    '            if c < 5 or current is None or idx < 4\n'
    '            else _safe_growth(current, raw_values[idx - 4])\n'
    '        )\n'
    '\n'
    '        f = four_period_growth[idx]\n'
    '        if c < 6 or is_blank(f) or idx < 1 or is_blank(four_period_growth[idx - 1]):\n'
    '            one_period_change.append("")\n'
    '        else:\n'
    '            one_period_change.append(f - four_period_growth[idx - 1])\n'
    '        if c < 7 or is_blank(f) or idx < 2 or is_blank(four_period_growth[idx - 2]):\n'
    '            two_period_change.append("")\n'
    '        else:\n'
    '            two_period_change.append(f - four_period_growth[idx - 2])\n'
    '\n'
    '    f_counts = _metric_counts(four_period_growth)\n'
    '    d_counts = _metric_counts(one_period_growth)\n'
    '    e_counts = _metric_counts(two_period_growth)\n'
    '    g_counts = _metric_counts(one_period_change)\n'
    '    f_history: list[float] = []\n'
    '    d_history: list[float] = []\n'
    '    e_history: list[float] = []\n'
    '    g_history: list[float] = []\n'
    '    f_ranks: list[Any] = []\n'
    '    d_ranks: list[Any] = []\n'
    '    e_ranks: list[Any] = []\n'
    '    g_ranks: list[Any] = []\n'
    '    for f, d, e, g in zip(\n'
    '        four_period_growth, one_period_growth, two_period_growth, one_period_change\n'
    '    ):\n'
    '        fv, dv, ev, gv = (\n'
    '            to_float_or_none(f),\n'
    '            to_float_or_none(d),\n'
    '            to_float_or_none(e),\n'
    '            to_float_or_none(g),\n'
    '        )\n'
    '        if fv is not None:\n'
    '            bisect.insort(f_history, fv)\n'
    '        if dv is not None:\n'
    '            bisect.insort(d_history, dv)\n'
    '        if ev is not None:\n'
    '            bisect.insort(e_history, ev)\n'
    '        if gv is not None:\n'
    '            bisect.insort(g_history, gv)\n'
    '        f_rank = (\n'
    '            percentrank_inc(f_history, fv)\n'
    '            if fv is not None and len(f_history) >= 20\n'
    '            else None\n'
    '        )\n'
    '        d_rank = (\n'
    '            percentrank_inc(d_history, dv)\n'
    '            if dv is not None and len(d_history) >= 20\n'
    '            else None\n'
    '        )\n'
    '        e_rank = (\n'
    '            percentrank_inc(e_history, ev)\n'
    '            if ev is not None and len(e_history) >= 20\n'
    '            else None\n'
    '        )\n'
    '        g_rank = (\n'
    '            percentrank_inc(g_history, gv)\n'
    '            if gv is not None and len(g_history) >= 20\n'
    '            else None\n'
    '        )\n'
    '        f_ranks.append("" if f_rank is None else f_rank)\n'
    '        d_ranks.append("" if d_rank is None else d_rank)\n'
    '        e_ranks.append("" if e_rank is None else e_rank)\n'
    '        g_ranks.append("" if g_rank is None else g_rank)\n'
    '\n'
    '    f_zscores = _rounded_expanding_zscores(four_period_growth, f_counts, 20)\n'
    '    d_zscores = _rounded_expanding_zscores(one_period_growth, d_counts, 20)\n'
    '    e_zscores = _rounded_expanding_zscores(two_period_growth, e_counts, 20)\n'
    '    g_zscores = _rounded_expanding_zscores(one_period_change, g_counts, 20)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = observation_counts[idx]\n'
    '        d = one_period_growth[idx]\n'
    '        e = two_period_growth[idx]\n'
    '        f = four_period_growth[idx]\n'
    '        g = one_period_change[idx]\n'
    '        h = two_period_change[idx]\n'
    '        i = f_ranks[idx]\n'
    '        j = _score_percent_rank(i)\n'
    '        k = d_ranks[idx]\n'
    '        l = _score_percent_rank(k)\n'
    '        m = e_ranks[idx]\n'
    '        n = _score_percent_rank(m)\n'
    '        o = g_ranks[idx]\n'
    '        p = _score_percent_rank(o)\n'
    '        q = _wage_pressure_status(f, e)\n'
    '        r = (\n'
    '            round(clamp(0.45 * l + 0.35 * n + 0.2 * p, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        s = _wage_recovery_cooling_score(f, g, h)\n'
    '        t = _wage_persistence_score(d, e, f)\n'
    '        af = f_zscores[idx]\n'
    '        ag = d_zscores[idx]\n'
    '        ah = e_zscores[idx]\n'
    '        ai = g_zscores[idx]\n'
    '        u = _wage_outlier_score([af, ag, ah, ai])\n'
    '        v = (\n'
    '            round(clamp(0.4 * j + 0.25 * l + 0.25 * n + 0.1 * p, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.15 * j + 0.3 * l + 0.2 * n + 0.2 * p + 0.1 * r + 0.05 * s, -10, 10\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (j, l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.35 * s + 0.45 * t + 0.2 * u, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if c < 20\n'
    '                else (\n'
    '                    "Medium reliability"\n'
    '                    if c < 40\n'
    '                    else ("High reliability" if c < 80 else "Very high reliability")\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if c < 20\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (v, w, x))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Observation Count": c,\n'
    '                "1-Period Annualised Wage Growth": d,\n'
    '                "2-Period Annualised Wage Growth": e,\n'
    '                "4-Period Wage Growth": f,\n'
    '                "1-Period Change in 4-Period Wage Growth": g,\n'
    '                "2-Period Change in 4-Period Wage Growth": h,\n'
    '                "4-Period Wage Growth PercentRank": i,\n'
    '                "4-Period Wage Growth Score": j,\n'
    '                "1-Period Annualised Wage Growth PercentRank": k,\n'
    '                "1-Period Annualised Wage Growth Score": l,\n'
    '                "2-Period Annualised Wage Growth PercentRank": m,\n'
    '                "2-Period Annualised Wage Growth Score": n,\n'
    '                "1-Period Change in 4-Period Wage Growth PercentRank": o,\n'
    '                "1-Period Change in 4-Period Wage Growth Score": p,\n'
    '                "Wage Pressure Status": q,\n'
    '                "Wage Pressure Momentum Composite Score": r,\n'
    '                "Wage Pressure Recovery / Cooling Score": s,\n'
    '                "Wage Pressure Persistence Score": t,\n'
    '                "Wage Pressure Outlier Score": u,\n'
    '                "Wage Pressure Leading Score": v,\n'
    '                "Wage Pressure Warning Composite Score": w,\n'
    '                "Wage Pressure Final Warning Score": x,\n'
    '                "Wage Pressure Regime": _wage_pressure_regime(v),\n'
    '                "Wage Pressure Momentum Label": _wage_pressure_momentum_label(v, w),\n'
    '                "Wage Pressure Signal Interpretation": _wage_pressure_signal_interpretation(\n'
    '                    w, x\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Wage Pressure Warning Risk Label": _wage_pressure_warning_label(x),\n'
    '                "Wage Pressure Policy Signal Label": _wage_pressure_policy_signal_label(\n'
    '                    x, f, d\n'
    '                ),\n'
    '                "4-Period Wage Growth Z-Score": af,\n'
    '                "1-Period Annualised Wage Growth Z-Score": ag,\n'
    '                "2-Period Annualised Wage Growth Z-Score": ah,\n'
    '                "1-Period Change in 4-Period Wage Growth Z-Score": ai,\n'
    '                "Wage Pressure Outlier Label": _wage_pressure_outlier_label(\n'
    '                    [af, ag, ah, ai]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, WAGE_GROWTH_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _short_term_expectations_status(d: Any, f: Any) -> str:\n'
    '    if is_blank(d):\n'
    '        return ""\n'
    '    if not is_blank(f) and d >= 4 and f >= 0.3:\n'
    '        return "Elevated and rising short-term expectations"\n'
    '    if not is_blank(f) and d >= 4 and f < 0:\n'
    '        return "Elevated but cooling short-term expectations"\n'
    '    if not is_blank(f) and d >= 3.25 and f >= 0.3:\n'
    '        return "Above-normal expectations and re-accelerating"\n'
    '    if d >= 3.25:\n'
    '        return "Above-normal short-term expectations"\n'
    '    if d >= 2.5:\n'
    '        return "Moderate short-term expectations"\n'
    '    if d >= 2:\n'
    '        return "Low short-term expectations"\n'
    '    return "Very low / disinflationary expectations"\n'
    '\n'
    '\n'
    'def _inflation_expectations_recovery_cooling_score(d: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (d, f, g)):\n'
    '        return ""\n'
    '    if d < 3 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if d >= 3 and f > 0 and g > 0:\n'
    '        return 5\n'
    '    if d >= 3 and f < 0 and g < 0:\n'
    '        return -5\n'
    '    if d < 3 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _inflation_expectations_pressure_score(d: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (d, f, g)):\n'
    '        return ""\n'
    '    if d >= 4.5 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if d >= 3.5 and f > 0 and g > 0:\n'
    '        return 4\n'
    '    if d <= 2.5 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if d <= 3 and f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _inflation_expectations_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _inflation_expectations_regime(v: Any) -> str:\n'
    '    if is_blank(v):\n'
    '        return ""\n'
    '    if v >= 2:\n'
    '        return "Inflation expectations rising"\n'
    '    if v <= -2:\n'
    '        return "Inflation expectations easing"\n'
    '    return "Neutral inflation-expectations pressure"\n'
    '\n'
    '\n'
    'def _inflation_expectations_momentum_label(v: Any, w: Any) -> str:\n'
    '    if any(is_blank(x) for x in (v, w)):\n'
    '        return ""\n'
    '    if v >= 2 and w >= 2:\n'
    '        return "Inflation expectations rising"\n'
    '    if v >= 2 and w <= -2:\n'
    '        return "Inflation expectations elevated but cooling"\n'
    '    if v >= 2 and -2 < w < 2:\n'
    '        return "Inflation expectations elevated / stable"\n'
    '    if v <= -2 and w >= 2:\n'
    '        return "Inflation expectations weak but re-accelerating"\n'
    '    if v <= -2 and w <= -2:\n'
    '        return "Inflation expectations easing"\n'
    '    if v <= -2 and -2 < w < 2:\n'
    '        return "Inflation expectations weak / stable"\n'
    '    if -2 < v < 2 and w >= 2:\n'
    '        return "Inflation expectations trend re-accelerating"\n'
    '    if -2 < v < 2 and w <= -2:\n'
    '        return "Inflation expectations trend cooling"\n'
    '    return "Inflation expectations broadly neutral"\n'
    '\n'
    '\n'
    'def _inflation_expectations_signal_interpretation(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 6 and x >= 0:\n'
    '        return "Strong upside inflation-expectations leading signal with supportive expectations-pressure profile"\n'
    '    if w >= 6 and x < 0:\n'
    '        return "Strong upside inflation-expectations leading signal, but warning profile is cooling"\n'
    '    if w >= 2 and x >= 0:\n'
    '        return "Moderately positive inflation-expectations pressure signal"\n'
    '    if w >= 2 and x < 0:\n'
    '        return "Inflation-expectations re-acceleration signal, but warning profile is cooling"\n'
    '    if w > -2 and x > -3 and x < 3:\n'
    '        return "Neutral/mixed inflation-expectations signal"\n'
    '    if w <= -6 and x <= 0:\n'
    '        return "Strong disinflationary expectations leading signal with downside confirmation"\n'
    '    if w <= -6 and x > 0:\n'
    '        return "Strong disinflationary expectations leading signal, but expectations pressure remains elevated"\n'
    '    if w <= -2 and x <= 0:\n'
    '        return "Moderately disinflationary inflation-expectations signal"\n'
    '    if w <= -2 and x > 0:\n'
    '        return "Inflation-expectations cooling signal, but expectations pressure remains elevated"\n'
    '    return "Mixed/transition inflation-expectations signal"\n'
    '\n'
    '\n'
    'def _inflation_expectations_warning_label(x: Any) -> str:\n'
    '    if is_blank(x):\n'
    '        return ""\n'
    '    if x >= 6:\n'
    '        return "High upside inflation-expectations warning risk"\n'
    '    if x >= 3:\n'
    '        return "Moderate upside inflation-expectations warning risk"\n'
    '    if x > -3:\n'
    '        return "Neutral inflation-expectations warning profile"\n'
    '    if x > -6:\n'
    '        return "Moderate disinflationary-expectations warning risk"\n'
    '    return "High disinflationary-expectations warning risk"\n'
    '\n'
    '\n'
    'def _inflation_expectations_policy_signal_label(x: Any, d: Any, f: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, d, f)):\n'
    '        return ""\n'
    '    if d >= 4 and f >= 0.3 and x >= 6:\n'
    '        return "Short-term expectations de-anchoring risk"\n'
    '    if d >= 4 and f < 0 and x <= 3:\n'
    '        return "High inflation expectations but cooling"\n'
    '    if d < 4 and f >= 0.5 and x >= 5:\n'
    '        return "Inflation expectations re-accelerating sharply"\n'
    '    if d >= 3.5 and x >= 5:\n'
    '        return "Inflation expectations becoming policy-relevant"\n'
    '    if x >= 6 and f >= 0.1:\n'
    '        return "Early upside expectations-pressure warning"\n'
    '    if d <= 2 and x <= -5:\n'
    '        return "Disinflationary expectations risk"\n'
    '    return "Normal short-term expectations regime"\n'
    '\n'
    '\n'
    'def _inflation_expectations_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside inflation-expectations pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside inflation-expectations pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside expectations / disinflation signal"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside expectations / disinflation signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _expectations_metric_ranks(values: list[Any], min_count: int = 60) -> list[Any]:\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        rank = (\n'
    '            percentrank_inc(history, numeric)\n'
    '            if numeric is not None and len(history) >= min_count\n'
    '            else None\n'
    '        )\n'
    '        ranks.append("" if rank is None else rank)\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def calculate_short_term_inflation_expectations_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    observation_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = observation_counts[idx]\n'
    '        if c < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append("" if not window else sum(window) / len(window))\n'
    '\n'
    '        if c < 2 or current is None or idx < 1 or raw_values[idx - 1] is None:\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(current - raw_values[idx - 1])\n'
    '\n'
    '        d = moving_averages[idx]\n'
    '        for lag, min_count, target in (\n'
    '            (3, 6, three_period_changes),\n'
    '            (6, 9, six_period_changes),\n'
    '            (12, 15, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                c < min_count\n'
    '                or is_blank(d)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(d - moving_averages[idx - lag])\n'
    '\n'
    '    d_counts = _metric_counts(moving_averages)\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    d_ranks = _expectations_metric_ranks(moving_averages)\n'
    '    f_ranks = _expectations_metric_ranks(three_period_changes)\n'
    '    g_ranks = _expectations_metric_ranks(six_period_changes)\n'
    '    h_ranks = _expectations_metric_ranks(twelve_period_changes)\n'
    '    d_zscores = _rounded_expanding_zscores(moving_averages, d_counts, 60)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 60)\n'
    '    g_zscores = _rounded_expanding_zscores(six_period_changes, g_counts, 60)\n'
    '    h_zscores = _rounded_expanding_zscores(twelve_period_changes, h_counts, 60)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = observation_counts[idx]\n'
    '        d = moving_averages[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = d_ranks[idx]\n'
    '        j = _score_percent_rank(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _score_percent_rank(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _score_percent_rank(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _score_percent_rank(o)\n'
    '        q = _short_term_expectations_status(d, f)\n'
    '        r = (\n'
    '            round(clamp(0.45 * l + 0.35 * n + 0.2 * p, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        s = _inflation_expectations_recovery_cooling_score(d, f, g)\n'
    '        t = _inflation_expectations_pressure_score(d, f, g)\n'
    '        af = d_zscores[idx]\n'
    '        ag = f_zscores[idx]\n'
    '        ah = g_zscores[idx]\n'
    '        ai = h_zscores[idx]\n'
    '        u = _inflation_expectations_outlier_score([af, ag, ah, ai])\n'
    '        v = (\n'
    '            round(clamp(0.5 * j + 0.25 * l + 0.15 * n + 0.1 * p, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        w = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.15 * j + 0.3 * l + 0.25 * n + 0.15 * p + 0.1 * r + 0.05 * s,\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (j, l, n, p, r, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.35 * s + 0.45 * t + 0.2 * u, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if c < 60\n'
    '                else ("Medium reliability" if c < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if c < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (v, w, x))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Observation Count": c,\n'
    '                "3-Period Moving Average": d,\n'
    '                "1-Period Change": e,\n'
    '                "3-Period Change in Moving Average": f,\n'
    '                "6-Period Change in Moving Average": g,\n'
    '                "12-Period Change in Moving Average": h,\n'
    '                "Short-Term Expectations Level PercentRank": i,\n'
    '                "Short-Term Expectations Level Score": j,\n'
    '                "3-Period Expectations Change PercentRank": k,\n'
    '                "3-Period Expectations Change Score": l,\n'
    '                "6-Period Expectations Change PercentRank": m,\n'
    '                "6-Period Expectations Change Score": n,\n'
    '                "12-Period Expectations Change PercentRank": o,\n'
    '                "12-Period Expectations Change Score": p,\n'
    '                "Short-Term Inflation Expectations Status": q,\n'
    '                "Inflation Expectations Momentum Composite Score": r,\n'
    '                "Inflation Expectations Recovery / Cooling Score": s,\n'
    '                "Inflation Expectations Pressure Score": t,\n'
    '                "Inflation Expectations Outlier Score": u,\n'
    '                "Inflation Expectations Leading Score": v,\n'
    '                "Inflation Expectations Warning Composite Score": w,\n'
    '                "Inflation Expectations Final Warning Score": x,\n'
    '                "Inflation Expectations Regime": _inflation_expectations_regime(v),\n'
    '                "Inflation Expectations Momentum Label": _inflation_expectations_momentum_label(\n'
    '                    v, w\n'
    '                ),\n'
    '                "Inflation Expectations Signal Interpretation": _inflation_expectations_signal_interpretation(\n'
    '                    w, x\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Inflation Expectations Warning Risk Label": _inflation_expectations_warning_label(\n'
    '                    x\n'
    '                ),\n'
    '                "Inflation Expectations Policy Signal Label": _inflation_expectations_policy_signal_label(\n'
    '                    x, d, f\n'
    '                ),\n'
    '                "Short-Term Expectations Level Z-Score": af,\n'
    '                "3-Period Expectations Change Z-Score": ag,\n'
    '                "6-Period Expectations Change Z-Score": ah,\n'
    '                "12-Period Expectations Change Z-Score": ai,\n'
    '                "Inflation Expectations Outlier Label": _inflation_expectations_outlier_label(\n'
    '                    [af, ag, ah, ai]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, SHORT_TERM_INFLATION_EXPECTATIONS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _real_yield_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    if value is None or value == "":\n'
    '        return ""\n'
    '    try:\n'
    '        number = Decimal(str(value))\n'
    '    except Exception:\n'
    '        return ""\n'
    '    quantum = Decimal("1").scaleb(-places)\n'
    '    if number >= 0:\n'
    '        return float(number.quantize(quantum, rounding=ROUND_HALF_UP))\n'
    '    return float(-((-number).quantize(quantum, rounding=ROUND_HALF_UP)))\n'
    '\n'
    '\n'
    'def _real_yield_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    clean = [\n'
    '        number\n'
    '        for number in (to_float_or_none(value) for value in values)\n'
    '        if number is not None\n'
    '    ]\n'
    '    if not clean:\n'
    '        return ""\n'
    "    # A23 must not inherit Python 3.13's compensated sum(): the last-bit\n"
    "    # differences change Sheets' ordering of otherwise tied yield changes.\n"
    '    total = clean[0]\n'
    '    for number in clean[1:]:\n'
    '        total += number\n'
    '    return total / len(clean)\n'
    '\n'
    '\n'
    'def _real_yield_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    current_float = to_float_or_none(current)\n'
    '    vals = sorted(\n'
    '        number\n'
    '        for number in (to_float_or_none(value) for value in values)\n'
    '        if number is not None\n'
    '    )\n'
    '    n = len(vals)\n'
    '    if n == 0 or current_float is None:\n'
    '        return ""\n'
    '    if n == 1:\n'
    '        return 0.0 if vals[0] == current_float else ""\n'
    '    if current_float < vals[0] or current_float > vals[-1]:\n'
    '        return ""\n'
    '    lo = bisect.bisect_left(vals, current_float)\n'
    '    hi = bisect.bisect_right(vals, current_float)\n'
    '    if lo < hi:\n'
    '        return lo / (n - 1)\n'
    '    lower_index = lo - 1\n'
    '    upper_index = lo\n'
    '    lower_value = vals[lower_index]\n'
    '    upper_value = vals[upper_index]\n'
    '    if upper_value == lower_value:\n'
    '        return lower_index / (n - 1)\n'
    '    fraction = (current_float - lower_value) / (upper_value - lower_value)\n'
    '    return (lower_index + fraction) / (n - 1)\n'
    '\n'
    '\n'
    'def _real_yield_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _real_yield_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _real_yield_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _real_yield_score_percent_rank(rank: Any) -> Any:\n'
    '    if is_blank(rank):\n'
    '        return ""\n'
    '    return _real_yield_sheets_round((float(rank) * 20) - 10, 2)\n'
    '\n'
    '\n'
    'def _real_yield_zero_if_close(value: Any, tolerance: float = 1e-12) -> Any:\n'
    '    number = to_float_or_none(value)\n'
    '    if number is None:\n'
    '        return ""\n'
    '    return 0.0 if abs(number) <= tolerance else number\n'
    '\n'
    '\n'
    'def _real_yield_level_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    f_for_regime = 0.0 if is_blank(f) else float(f)\n'
    '    if c >= 2 and f_for_regime >= 0:\n'
    '        return "High and rising real-yield regime"\n'
    '    if c >= 2 and f_for_regime < 0:\n'
    '        return "High but cooling real-yield regime"\n'
    '    if c >= 0 and f_for_regime >= 0:\n'
    '        return "Positive real yields and rising"\n'
    '    if c >= 0 and f_for_regime < 0:\n'
    '        return "Positive real yields but easing"\n'
    '    if c < 0 and f_for_regime >= 0:\n'
    '        return "Negative real yields but rising"\n'
    '    if c < 0 and f_for_regime < 0:\n'
    '        return "Negative and easing real-yield regime"\n'
    '    return "Mixed real-yield regime"\n'
    '\n'
    '\n'
    'def _real_yield_asset_bias_label(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 6:\n'
    '        return "Bearish duration / bearish gold / equity valuation headwind"\n'
    '    if y >= 2:\n'
    '        return "Mild bearish duration and gold bias"\n'
    '    if y <= -6:\n'
    '        return "Bullish duration / bullish gold / supportive for long-duration equities"\n'
    '    if y <= -2:\n'
    '        return "Mild bullish duration and gold bias"\n'
    '    return "Neutral real-rate asset bias"\n'
    '\n'
    '\n'
    'def _real_yield_shock_label(e: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if e >= 0.3:\n'
    '        return "1M real-yield upside shock"\n'
    '    if e <= -0.3:\n'
    '        return "1M real-yield downside shock"\n'
    '    return "No 1M real-yield shock"\n'
    '\n'
    '\n'
    'def _real_yield_recovery_tightening_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c < 0 and f > 0 and g > 0:\n'
    '        return 5\n'
    '    if c >= 1.5 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 1.5 and f < 0 and g <= 0:\n'
    '        return -5\n'
    '    if c < 0 and f < 0 and g <= 0:\n'
    '        return -6\n'
    '    if f > 0 and g > 0:\n'
    '        return 4\n'
    '    if f < 0 and g <= 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_yield_shock_score(e: Any) -> Any:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if e >= 0.3:\n'
    '        return 6\n'
    '    if e >= 0.2:\n'
    '        return 4\n'
    '    if e <= -0.3:\n'
    '        return -6\n'
    '    if e <= -0.2:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_yield_restriction_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c >= 2 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 1.5 and f > 0 and g > 0:\n'
    '        return 4\n'
    '    if c <= 0 and f < 0 and g <= 0:\n'
    '        return -6\n'
    '    if c <= 0.5 and f < 0 and g <= 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_yield_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_yield_pressure_regime(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 2:\n'
    '        return "Real-yield pressure rising"\n'
    '    if y <= -2:\n'
    '        return "Real-yield pressure easing"\n'
    '    return "Neutral real-yield pressure"\n'
    '\n'
    '\n'
    'def _real_yield_momentum_label(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 2 and z >= 2:\n'
    '        return "Real-yield pressure rising"\n'
    '    if y >= 2 and z <= -2:\n'
    '        return "Real yields elevated but easing"\n'
    '    if y >= 2 and -2 < z < 2:\n'
    '        return "Real-yield pressure elevated / stable"\n'
    '    if y <= -2 and z >= 2:\n'
    '        return "Real yields low but tightening"\n'
    '    if y <= -2 and z <= -2:\n'
    '        return "Real-yield pressure easing"\n'
    '    if y <= -2 and -2 < z < 2:\n'
    '        return "Real-yield pressure low / stable"\n'
    '    if -2 < y < 2 and z >= 2:\n'
    '        return "Real-yield trend tightening"\n'
    '    if -2 < y < 2 and z <= -2:\n'
    '        return "Real-yield trend easing"\n'
    '    return "Real yields broadly neutral"\n'
    '\n'
    '\n'
    'def _real_yield_signal_interpretation(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 6 and aa >= 0:\n'
    '        return "Strong upside real-yield leading signal with supportive restriction profile"\n'
    '    if z >= 6 and aa < 0:\n'
    '        return "Strong upside real-yield leading signal, but restriction warning profile is easing"\n'
    '    if z >= 2 and aa >= 0:\n'
    '        return "Moderately positive real-yield tightening signal"\n'
    '    if z >= 2 and aa < 0:\n'
    '        return "Real-yield tightening signal, but restriction profile is easing"\n'
    '    if z > -2 and aa > -3 and aa < 3:\n'
    '        return "Neutral/mixed real-yield signal"\n'
    '    if z <= -6 and aa <= 0:\n'
    '        return "Strong real-yield easing signal with downside confirmation"\n'
    '    if z <= -6 and aa > 0:\n'
    '        return (\n'
    '            "Strong real-yield easing signal, but restriction pressure remains elevated"\n'
    '        )\n'
    '    if z <= -2 and aa <= 0:\n'
    '        return "Moderately negative real-yield/easing signal"\n'
    '    if z <= -2 and aa > 0:\n'
    '        return "Real-yield easing signal, but restriction pressure remains elevated"\n'
    '    return "Mixed/transition real-yield signal"\n'
    '\n'
    '\n'
    'def _real_yield_warning_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa <= -6:\n'
    '        return "High real-yield easing warning risk"\n'
    '    if aa <= -3:\n'
    '        return "Moderate real-yield easing warning risk"\n'
    '    if aa >= 6:\n'
    '        return "High real-yield tightening warning risk"\n'
    '    if aa >= 3:\n'
    '        return "Moderate real-yield tightening warning risk"\n'
    '    return "Neutral real-yield warning profile"\n'
    '\n'
    '\n'
    'def _real_yield_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside real-yield pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside real-yield pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside real-yield pressure"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside real-yield pressure"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_real_yield_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    raw_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    smoothed_count = 0\n'
    '    smoothed_counts_so_far: list[int] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if raw_counts[idx] < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append(\n'
    '                "" if not window else _real_yield_mean_sheets(window)\n'
    '            )\n'
    '\n'
    '        c = moving_averages[idx]\n'
    '        if not is_blank(c):\n'
    '            smoothed_count += 1\n'
    '        smoothed_counts_so_far.append(smoothed_count)\n'
    '        if (\n'
    '            smoothed_count < 2\n'
    '            or is_blank(c)\n'
    '            or idx < 1\n'
    '            or is_blank(moving_averages[idx - 1])\n'
    '        ):\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(c - moving_averages[idx - 1])\n'
    '        for lag, min_count, target in (\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                smoothed_count < min_count\n'
    '                or is_blank(c)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(c - moving_averages[idx - lag])\n'
    '\n'
    '    smoothed_counts = smoothed_counts_so_far\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    c_ranks = _real_yield_metric_ranks_sheets(moving_averages)\n'
    '    f_ranks = _real_yield_metric_ranks_sheets(three_period_changes)\n'
    '    g_ranks = _real_yield_metric_ranks_sheets(six_period_changes)\n'
    '    h_ranks = _real_yield_metric_ranks_sheets(twelve_period_changes)\n'
    '    c_zscores = _rounded_expanding_zscores(moving_averages, smoothed_counts, 60)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 60)\n'
    '    g_zscores = _rounded_expanding_zscores(six_period_changes, g_counts, 60)\n'
    '    h_zscores = _rounded_expanding_zscores(twelve_period_changes, h_counts, 60)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = smoothed_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _real_yield_score_percent_rank(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _real_yield_score_percent_rank(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _real_yield_score_percent_rank(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _real_yield_score_percent_rank(o)\n'
    '        t = (\n'
    '            _real_yield_sheets_round(\n'
    '                clamp(0.05 * j + 0.45 * l + 0.35 * n + 0.15 * p, -10, 10), 2\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        u = _real_yield_recovery_tightening_score(c, f, g)\n'
    '        v = _real_yield_shock_score(e)\n'
    '        w = _real_yield_restriction_score(c, f, g)\n'
    '        ah = c_zscores[idx]\n'
    '        ai = f_zscores[idx]\n'
    '        aj = g_zscores[idx]\n'
    '        ak = h_zscores[idx]\n'
    '        x = _real_yield_outlier_score([ah, ai, aj, ak])\n'
    '        y = (\n'
    '            _real_yield_sheets_round(\n'
    '                clamp(0.45 * j + 0.30 * l + 0.15 * n + 0.10 * p, -10, 10), 2\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        # Preserve the existing A23 leading structure, but feed it only the\n'
    '        # final rounded visible scores (including the rounded impulse T).\n'
    '        z = (\n'
    '            _real_yield_sheets_round(\n'
    '                clamp(\n'
    '                    0.15 * j + 0.30 * l + 0.25 * n + 0.15 * p + 0.10 * t + 0.05 * u,\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        aa = (\n'
    '            _real_yield_sheets_round(\n'
    '                clamp(0.30 * u + 0.25 * v + 0.30 * w + 0.15 * x, -10, 10), 2\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Real Yield": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Real Yield Change": _real_yield_zero_if_close(e),\n'
    '                "3M Real Yield Change": _real_yield_zero_if_close(f),\n'
    '                "6M Real Yield Change": _real_yield_zero_if_close(g),\n'
    '                "12M Real Yield Change": _real_yield_zero_if_close(h),\n'
    '                "Real Yield Level Percentile": i,\n'
    '                "Real Yield Level Score": j,\n'
    '                "3M Change Percentile": k,\n'
    '                "3M Change Score": l,\n'
    '                "6M Change Percentile": m,\n'
    '                "6M Change Score": n,\n'
    '                "12M Change Percentile": o,\n'
    '                "12M Change Score": p,\n'
    '                "Real Yield Regime": _real_yield_level_regime(c, f),\n'
    '                "Real Yield Asset Bias": _real_yield_asset_bias_label(y),\n'
    '                "1M Real Yield Shock Flag": _real_yield_shock_label(e),\n'
    '                "Real Yield Impulse Score": t,\n'
    '                "Real Yield Cycle Turning Point Score": u,\n'
    '                "Real Yield Shock Score": v,\n'
    '                "Policy Restriction Score": w,\n'
    '                "Real Yield Extreme Score": x,\n'
    '                "Coincident Score": y,\n'
    '                "Leading Score": z,\n'
    '                "Warning Score": aa,\n'
    '                "State": _real_yield_pressure_regime(y),\n'
    '                "Regime": _real_yield_momentum_label(y, z),\n'
    '                "Signal Comment": _real_yield_signal_interpretation(z, aa),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _real_yield_warning_label(aa),\n'
    '                "Level Z-Score": ah,\n'
    '                "3M Change Z-Score": ai,\n'
    '                "6M Change Z-Score": aj,\n'
    '                "12M Change Z-Score": ak,\n'
    '                "Extreme Flag": _real_yield_outlier_label([ah, ai, aj, ak]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, REAL_YIELD_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _two_year_yield_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A24-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _two_year_yield_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A24-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _two_year_yield_metric_ranks_sheets(\n'
    '    values: Sequence[Any], sample_counts: Sequence[int], min_sample_count: int = 60\n'
    ') -> list[Any]:\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for idx, value in enumerate(values):\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or sample_counts[idx] < min_sample_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _two_year_yield_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _two_year_yield_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _two_year_yield_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        ""\n'
    '        if is_blank(rank)\n'
    '        else _two_year_yield_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _two_year_yield_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _two_year_yield_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _two_year_yield_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A24 formulas from their rounded visible decimal operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _two_year_yield_rounded_expanding_zscores(\n'
    '    values: list[object], gates: list[int], min_count: int\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else _two_year_yield_sheets_round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, gates, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def _two_year_yield_threshold_value(value: Any) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    # Neutralize binary floating-point noise around spreadsheet threshold results.\n'
    '    if math.isclose(numeric, 0.30, rel_tol=0.0, abs_tol=1e-12):\n'
    '        return 0.30\n'
    '    if math.isclose(numeric, -0.30, rel_tol=0.0, abs_tol=1e-12):\n'
    '        return -0.30\n'
    '    return numeric\n'
    '\n'
    '\n'
    'def _two_year_yield_level_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    f_for_regime = 0.0 if is_blank(f) else f\n'
    '    if c >= 5 and f_for_regime >= 0:\n'
    '        return "High and rising 2Y yield regime"\n'
    '    if c >= 5 and f_for_regime < 0:\n'
    '        return "High but cooling 2Y yield regime"\n'
    '    if c >= 4 and f_for_regime >= 0:\n'
    '        return "Elevated and rising 2Y yield regime"\n'
    '    if c >= 4 and f_for_regime < 0:\n'
    '        return "Elevated but cooling 2Y yield regime"\n'
    '    if c >= 3 and f_for_regime >= 0:\n'
    '        return "Moderate and rising 2Y yield regime"\n'
    '    if c >= 3 and f_for_regime < 0:\n'
    '        return "Moderate but cooling 2Y yield regime"\n'
    '    if c < 3 and f_for_regime >= 0:\n'
    '        return "Low but rising 2Y yield regime"\n'
    '    if c < 3 and f_for_regime < 0:\n'
    '        return "Low and easing 2Y yield regime"\n'
    '    return "Mixed 2Y yield regime"\n'
    '\n'
    '\n'
    'def _two_year_yield_asset_bias_label(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 6:\n'
    '        return "Strong bearish front-end duration / hawkish policy bias"\n'
    '    if y >= 2:\n'
    '        return "Mild bearish front-end duration / tighter policy bias"\n'
    '    if y <= -6:\n'
    '        return "Strong bullish front-end duration / dovish policy bias"\n'
    '    if y <= -2:\n'
    '        return "Mild bullish front-end duration / easier policy bias"\n'
    '    return "Neutral front-end policy-rate bias"\n'
    '\n'
    '\n'
    'def _two_year_yield_shock_label(e: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _two_year_yield_threshold_value(e)\n'
    '    if threshold_value > 0.3:\n'
    '        return "1M 2Y yield upside shock"\n'
    '    if threshold_value <= -0.3:\n'
    '        return "1M 2Y yield downside shock"\n'
    '    return "No 1M 2Y yield shock"\n'
    '\n'
    '\n'
    'def _two_year_yield_recovery_tightening_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c < 1 and f > 0 and g > 0:\n'
    '        return 5\n'
    '    if c >= 3 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 3 and f < 0 and g < 0:\n'
    '        return -5\n'
    '    if c < 1 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if f > 0 and g > 0:\n'
    '        return 4\n'
    '    if f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _two_year_yield_shock_score(e: Any) -> Any:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _two_year_yield_threshold_value(e)\n'
    '    if threshold_value > 0.3:\n'
    '        return 6\n'
    '    if threshold_value >= 0.2:\n'
    '        return 4\n'
    '    if threshold_value <= -0.3:\n'
    '        return -6\n'
    '    if threshold_value <= -0.2:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _two_year_yield_restriction_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c >= 5 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 3 and f > 0 and g > 0:\n'
    '        return 4\n'
    '    if c <= 1 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if c <= 2 and f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _two_year_yield_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _two_year_yield_pressure_regime(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 2:\n'
    '        return "2Y yield pressure rising"\n'
    '    if y <= -2:\n'
    '        return "2Y yield pressure easing"\n'
    '    return "Neutral 2Y yield pressure"\n'
    '\n'
    '\n'
    'def _two_year_yield_momentum_label(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 2 and z >= 2:\n'
    '        return "2Y yield pressure rising"\n'
    '    if y >= 2 and z <= -2:\n'
    '        return "2Y yields elevated but easing"\n'
    '    if y >= 2 and -2 < z < 2:\n'
    '        return "2Y yield pressure elevated / stable"\n'
    '    if y <= -2 and z >= 2:\n'
    '        return "2Y yields low but tightening"\n'
    '    if y <= -2 and z <= -2:\n'
    '        return "2Y yield pressure easing"\n'
    '    if y <= -2 and -2 < z < 2:\n'
    '        return "2Y yield pressure low / stable"\n'
    '    if -2 < y < 2 and z >= 2:\n'
    '        return "2Y yield trend tightening"\n'
    '    if -2 < y < 2 and z <= -2:\n'
    '        return "2Y yield trend easing"\n'
    '    return "2Y yields broadly neutral"\n'
    '\n'
    '\n'
    'def _two_year_yield_signal_interpretation(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 6 and aa >= 0:\n'
    '        return "Strong upside 2Y yield leading signal with supportive policy-restriction profile"\n'
    '    if z >= 6 and aa < 0:\n'
    '        return "Strong upside 2Y yield leading signal, but policy-restriction warning profile is easing"\n'
    '    if z >= 2 and aa >= 0:\n'
    '        return "Moderately positive front-end yield tightening signal"\n'
    '    if z >= 2 and aa < 0:\n'
    '        return "2Y yield tightening signal, but policy-restriction profile is easing"\n'
    '    if z > -2 and aa > -3 and aa < 3:\n'
    '        return "Neutral/mixed 2Y yield signal"\n'
    '    if z <= -6 and aa <= 0:\n'
    '        return "Strong 2Y yield easing signal with downside confirmation"\n'
    '    if z <= -6 and aa > 0:\n'
    '        return "Strong 2Y yield easing signal, but policy restriction remains elevated"\n'
    '    if z <= -2 and aa <= 0:\n'
    '        return "Moderately negative 2Y yield/easing signal"\n'
    '    if z <= -2 and aa > 0:\n'
    '        return "2Y yield easing signal, but policy restriction remains elevated"\n'
    '    return "Mixed/transition 2Y yield signal"\n'
    '\n'
    '\n'
    'def _two_year_yield_warning_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa <= -6:\n'
    '        return "High 2Y yield easing warning risk"\n'
    '    if aa <= -3:\n'
    '        return "Moderate 2Y yield easing warning risk"\n'
    '    if aa >= 6:\n'
    '        return "High upside 2Y yield restriction warning risk"\n'
    '    if aa >= 3:\n'
    '        return "Moderate upside 2Y yield restriction warning risk"\n'
    '    return "Neutral 2Y yield warning profile"\n'
    '\n'
    '\n'
    'def _two_year_yield_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside 2Y yield pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside 2Y yield pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside 2Y yield pressure"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside 2Y yield pressure"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_two_year_yield_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    raw_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    smoothed_count = 0\n'
    '    smoothed_counts_so_far: list[int] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if raw_counts[idx] < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append(\n'
    '                "" if not window else _real_yield_mean_sheets(window)\n'
    '            )\n'
    '\n'
    '        c = moving_averages[idx]\n'
    '        if not is_blank(c):\n'
    '            smoothed_count += 1\n'
    '        smoothed_counts_so_far.append(smoothed_count)\n'
    '        if (\n'
    '            smoothed_count < 2\n'
    '            or is_blank(c)\n'
    '            or idx < 1\n'
    '            or is_blank(moving_averages[idx - 1])\n'
    '        ):\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(c - moving_averages[idx - 1])\n'
    '        for lag, min_count, target in (\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                smoothed_count < min_count\n'
    '                or is_blank(c)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(c - moving_averages[idx - lag])\n'
    '\n'
    '    smoothed_counts = smoothed_counts_so_far\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    # A24 intentionally gates all four ranks from Sample Count D, not from each\n'
    "    # metric's own history count.\n"
    '    c_ranks = _two_year_yield_metric_ranks_sheets(moving_averages, smoothed_counts)\n'
    '    f_ranks = _two_year_yield_metric_ranks_sheets(three_period_changes, smoothed_counts)\n'
    '    g_ranks = _two_year_yield_metric_ranks_sheets(six_period_changes, smoothed_counts)\n'
    '    h_ranks = _two_year_yield_metric_ranks_sheets(\n'
    '        twelve_period_changes, smoothed_counts\n'
    '    )\n'
    '    c_zscores = _two_year_yield_rounded_expanding_zscores(\n'
    '        moving_averages, smoothed_counts, 60\n'
    '    )\n'
    '    f_zscores = _two_year_yield_rounded_expanding_zscores(\n'
    '        three_period_changes, f_counts, 60\n'
    '    )\n'
    '    g_zscores = _two_year_yield_rounded_expanding_zscores(\n'
    '        six_period_changes, g_counts, 60\n'
    '    )\n'
    '    h_zscores = _two_year_yield_rounded_expanding_zscores(\n'
    '        twelve_period_changes, h_counts, 60\n'
    '    )\n'
    "    # Column X has its own A24 gate: it uses F's internal expanding sample\n"
    '    # z-score as soon as D reaches 60, independently of displayed column AI.\n'
    '    f_internal_zscores = _two_year_yield_rounded_expanding_zscores(\n'
    '        three_period_changes, smoothed_counts, 60\n'
    '    )\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = smoothed_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _two_year_yield_score_percentile(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _two_year_yield_score_percentile(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _two_year_yield_score_percentile(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _two_year_yield_score_percentile(o)\n'
    '        t = (\n'
    '            _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.05, j), (0.45, l), (0.35, n), (0.15, p)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        u = _two_year_yield_recovery_tightening_score(c, f, g)\n'
    '        v = _two_year_yield_shock_score(e)\n'
    '        w = _two_year_yield_restriction_score(c, f, g)\n'
    '        ah = c_zscores[idx]\n'
    '        ai = f_zscores[idx]\n'
    '        aj = g_zscores[idx]\n'
    '        ak = h_zscores[idx]\n'
    '        x = f_internal_zscores[idx]\n'
    '        y = (\n'
    '            _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.25, j), (0.30, l), (0.20, n), (0.10, p), (0.15, t)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.10, j), (0.35, l), (0.25, n), (0.15, p), (0.10, t), (0.05, u)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        aa = (\n'
    '            _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.30, u), (0.25, v), (0.30, w), (0.15, x)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Yield": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Yield Change": e,\n'
    '                "3M Yield Change": f,\n'
    '                "6M Yield Change": g,\n'
    '                "12M Yield Change": h,\n'
    '                "Yield Level Percentile": _two_year_yield_percent_display(i),\n'
    '                "Yield Level Score": j,\n'
    '                "3M Change Percentile": _two_year_yield_percent_display(k),\n'
    '                "3M Change Score": l,\n'
    '                "6M Change Percentile": _two_year_yield_percent_display(m),\n'
    '                "6M Change Score": n,\n'
    '                "12M Change Percentile": _two_year_yield_percent_display(o),\n'
    '                "12M Change Score": p,\n'
    '                "2Y Yield Regime": _two_year_yield_level_regime(c, f),\n'
    '                "2Y Yield Policy Bias": _two_year_yield_asset_bias_label(y),\n'
    '                "1M 2Y Yield Shock Flag": _two_year_yield_shock_label(e),\n'
    '                "2Y Yield Impulse Score": t,\n'
    '                "2Y Yield Cycle Turning Point Score": u,\n'
    '                "2Y Yield Shock Score": v,\n'
    '                "Policy Rate Restriction Score": w,\n'
    '                "2Y Yield Extreme Score": x,\n'
    '                "Coincident Score": y,\n'
    '                "Leading Score": z,\n'
    '                "Warning Score": aa,\n'
    '                "State": _two_year_yield_pressure_regime(y),\n'
    '                "Regime": _two_year_yield_momentum_label(y, z),\n'
    '                "Signal Comment": _two_year_yield_signal_interpretation(z, aa),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _two_year_yield_warning_label(aa),\n'
    '                "Level Z-Score": ah,\n'
    '                "3M Change Z-Score": ai,\n'
    '                "6M Change Z-Score": aj,\n'
    '                "12M Change Z-Score": ak,\n'
    '                "Extreme Flag": _two_year_yield_outlier_label([ah, ai, aj, ak]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, TWO_YEAR_YIELD_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _ten_year_yield_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A25-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _ten_year_yield_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A25-only spreadsheet-order mean used by the 3M moving average."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _ten_year_yield_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A25-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _ten_year_yield_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    """Rank each A25 metric only after that metric has 60 numeric observations."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _ten_year_yield_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _ten_year_yield_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _ten_year_yield_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        ""\n'
    '        if is_blank(rank)\n'
    '        else _ten_year_yield_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _ten_year_yield_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _ten_year_yield_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _ten_year_yield_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A25 formulas from their rounded visible decimal operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _ten_year_yield_zero_if_close(value: Any, tolerance: float = 1e-12) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    return 0.0 if abs(numeric) <= tolerance else numeric\n'
    '\n'
    '\n'
    'def _ten_year_yield_threshold_value(value: Any) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    # Spreadsheet differences can land a few binary ulps either side of a\n'
    '    # visible +/-0.20 or +/-0.30 boundary.\n'
    '    for boundary in (0.30, 0.20, -0.20, -0.30):\n'
    '        if math.isclose(numeric, boundary, rel_tol=0.0, abs_tol=1e-12):\n'
    '            return boundary\n'
    '    return numeric\n'
    '\n'
    '\n'
    'def _ten_year_yield_level_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    f_for_regime = 0.0 if is_blank(f) else f\n'
    '    if c >= 5 and f_for_regime >= 0:\n'
    '        return "High and rising 10Y yield regime"\n'
    '    if c >= 5 and f_for_regime < 0:\n'
    '        return "High but cooling 10Y yield regime"\n'
    '    if c >= 3 and f_for_regime >= 0:\n'
    '        return "Elevated and rising 10Y yield regime"\n'
    '    if c >= 3 and f_for_regime < 0:\n'
    '        return "Elevated but easing 10Y yield regime"\n'
    '    if c >= 1.5 and f_for_regime >= 0:\n'
    '        return "Moderate 10Y yield and rising"\n'
    '    if c >= 1.5 and f_for_regime < 0:\n'
    '        return "Moderate 10Y yield but easing"\n'
    '    if c < 1.5 and f_for_regime >= 0:\n'
    '        return "Low 10Y yield but rising"\n'
    '    if c < 1.5 and f_for_regime < 0:\n'
    '        return "Low and easing 10Y yield regime"\n'
    '    return "Mixed 10Y yield regime"\n'
    '\n'
    '\n'
    'def _ten_year_yield_asset_bias_label(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 6:\n'
    '        return "Strong bearish duration / higher discount-rate pressure"\n'
    '    if y >= 2:\n'
    '        return "Mild bearish duration / higher long-rate pressure"\n'
    '    if y <= -6:\n'
    '        return "Strong bullish duration / lower discount-rate support"\n'
    '    if y <= -2:\n'
    '        return "Mild bullish duration / lower long-rate support"\n'
    '    return "Neutral 10Y yield asset bias"\n'
    '\n'
    '\n'
    'def _ten_year_yield_shock_label(e: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _ten_year_yield_threshold_value(e)\n'
    '    if threshold_value >= 0.3:\n'
    '        return "1M 10Y yield upside shock"\n'
    '    if threshold_value <= -0.3:\n'
    '        return "1M 10Y yield downside shock"\n'
    '    return "No 1M 10Y yield shock"\n'
    '\n'
    '\n'
    'def _ten_year_yield_recovery_tightening_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c < 1.5 and f > 0 and g > 0:\n'
    '        return 5\n'
    '    if c >= 3 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 3 and f < 0 and g < 0:\n'
    '        return -5\n'
    '    if c < 1.5 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if f > 0 and g > 0:\n'
    '        return 4\n'
    '    if f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _ten_year_yield_shock_score(e: Any) -> Any:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _ten_year_yield_threshold_value(e)\n'
    '    if threshold_value >= 0.3:\n'
    '        return 6\n'
    '    if threshold_value >= 0.2:\n'
    '        return 4\n'
    '    if threshold_value <= -0.3:\n'
    '        return -6\n'
    '    if threshold_value <= -0.2:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _ten_year_yield_restriction_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c >= 5 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 3 and f > 0 and g > 0:\n'
    '        return 4\n'
    '    if c <= 1.5 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if c <= 2.5 and f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _ten_year_yield_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _ten_year_yield_pressure_regime(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 2:\n'
    '        return "10Y yield pressure rising"\n'
    '    if y <= -2:\n'
    '        return "10Y yield pressure easing"\n'
    '    return "Neutral 10Y yield pressure"\n'
    '\n'
    '\n'
    'def _ten_year_yield_momentum_label(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 2 and z >= 2:\n'
    '        return "10Y yield pressure rising"\n'
    '    if y >= 2 and z <= -2:\n'
    '        return "10Y yields elevated but easing"\n'
    '    if y >= 2 and -2 < z < 2:\n'
    '        return "10Y yield pressure elevated / stable"\n'
    '    if y <= -2 and z >= 2:\n'
    '        return "10Y yields low but rising"\n'
    '    if y <= -2 and z <= -2:\n'
    '        return "10Y yield pressure easing"\n'
    '    if y <= -2 and -2 < z < 2:\n'
    '        return "10Y yield pressure low / stable"\n'
    '    if -2 < y < 2 and z >= 2:\n'
    '        return "10Y yield trend rising"\n'
    '    if -2 < y < 2 and z <= -2:\n'
    '        return "10Y yield trend easing"\n'
    '    return "10Y yields broadly neutral"\n'
    '\n'
    '\n'
    'def _ten_year_yield_signal_interpretation(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 6 and aa >= 0:\n'
    '        return "Strong upside 10Y yield leading signal with supportive long-rate restriction profile"\n'
    '    if z >= 6 and aa < 0:\n'
    '        return "Strong upside 10Y yield leading signal, but long-rate restriction profile is easing"\n'
    '    if z >= 2 and aa >= 0:\n'
    '        return "Moderately positive 10Y yield tightening signal"\n'
    '    if z >= 2 and aa < 0:\n'
    '        return (\n'
    '            "10Y yield tightening signal, but long-rate restriction profile is easing"\n'
    '        )\n'
    '    if z > -2 and aa > -3 and aa < 3:\n'
    '        return "Neutral/mixed 10Y yield signal"\n'
    '    if z <= -6 and aa <= 0:\n'
    '        return "Strong 10Y yield easing signal with downside confirmation"\n'
    '    if z <= -6 and aa > 0:\n'
    '        return (\n'
    '            "Strong 10Y yield easing signal, but long-rate restriction remains elevated"\n'
    '        )\n'
    '    if z <= -2 and aa <= 0:\n'
    '        return "Moderately negative 10Y yield/easing signal"\n'
    '    if z <= -2 and aa > 0:\n'
    '        return "10Y yield easing signal, but long-rate restriction remains elevated"\n'
    '    return "Mixed/transition 10Y yield signal"\n'
    '\n'
    '\n'
    'def _ten_year_yield_warning_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 6:\n'
    '        return "High upside 10Y yield restriction warning risk"\n'
    '    if aa >= 3:\n'
    '        return "Moderate upside 10Y yield restriction warning risk"\n'
    '    if aa > -3:\n'
    '        return "Neutral 10Y yield warning profile"\n'
    '    if aa > -6:\n'
    '        return "Moderate 10Y yield easing warning risk"\n'
    '    return "High 10Y yield easing warning risk"\n'
    '\n'
    '\n'
    'def _ten_year_yield_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside 10Y yield pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside 10Y yield pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside 10Y yield pressure"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside 10Y yield pressure"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_ten_year_yield_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    raw_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    smoothed_count = 0\n'
    '    smoothed_counts_so_far: list[int] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if raw_counts[idx] < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append(\n'
    '                "" if not window else _ten_year_yield_mean_sheets(window)\n'
    '            )\n'
    '\n'
    '        c = moving_averages[idx]\n'
    '        if not is_blank(c):\n'
    '            smoothed_count += 1\n'
    '        smoothed_counts_so_far.append(smoothed_count)\n'
    '        if (\n'
    '            smoothed_count < 2\n'
    '            or is_blank(c)\n'
    '            or idx < 1\n'
    '            or is_blank(moving_averages[idx - 1])\n'
    '        ):\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(\n'
    '                _ten_year_yield_zero_if_close(c - moving_averages[idx - 1])\n'
    '            )\n'
    '        for lag, min_count, target in (\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                smoothed_count < min_count\n'
    '                or is_blank(c)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(\n'
    '                    _ten_year_yield_zero_if_close(c - moving_averages[idx - lag])\n'
    '                )\n'
    '\n'
    '    smoothed_counts = smoothed_counts_so_far\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    c_ranks = _ten_year_yield_metric_ranks_sheets(moving_averages)\n'
    '    f_ranks = _ten_year_yield_metric_ranks_sheets(three_period_changes)\n'
    '    g_ranks = _ten_year_yield_metric_ranks_sheets(six_period_changes)\n'
    '    h_ranks = _ten_year_yield_metric_ranks_sheets(twelve_period_changes)\n'
    '    c_zscores = _rounded_expanding_zscores(moving_averages, smoothed_counts, 60)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 60)\n'
    '    g_zscores = _rounded_expanding_zscores(six_period_changes, g_counts, 60)\n'
    '    h_zscores = _rounded_expanding_zscores(twelve_period_changes, h_counts, 60)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = smoothed_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _ten_year_yield_score_percentile(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _ten_year_yield_score_percentile(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _ten_year_yield_score_percentile(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _ten_year_yield_score_percentile(o)\n'
    '        t = (\n'
    '            _ten_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _ten_year_yield_weighted_sum(\n'
    '                        (0.45, l), (0.35, n), (0.15, p), (0.05, j)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (l, n, p, j))\n'
    '            else ""\n'
    '        )\n'
    '        u = _ten_year_yield_recovery_tightening_score(c, f, g)\n'
    '        v = _ten_year_yield_shock_score(e)\n'
    '        w = _ten_year_yield_restriction_score(c, f, g)\n'
    '        ah = c_zscores[idx]\n'
    '        ai = f_zscores[idx]\n'
    '        aj = g_zscores[idx]\n'
    '        ak = h_zscores[idx]\n'
    '        x = _ten_year_yield_outlier_score([ah, ai, aj, ak])\n'
    '        y = (\n'
    '            _ten_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _ten_year_yield_weighted_sum(\n'
    '                        (0.35, j), (0.25, l), (0.20, n), (0.10, p), (0.10, t)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            _ten_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _ten_year_yield_weighted_sum(\n'
    '                        (0.10, j), (0.35, l), (0.25, n), (0.15, p), (0.10, t), (0.05, u)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        aa = (\n'
    '            _ten_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _ten_year_yield_weighted_sum(\n'
    '                        (0.30, u), (0.25, v), (0.30, w), (0.15, x)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Yield": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Yield Change": e,\n'
    '                "3M Yield Change": f,\n'
    '                "6M Yield Change": g,\n'
    '                "12M Yield Change": h,\n'
    '                "Yield Level Percentile": _ten_year_yield_percent_display(i),\n'
    '                "Yield Level Score": j,\n'
    '                "3M Change Percentile": _ten_year_yield_percent_display(k),\n'
    '                "3M Change Score": l,\n'
    '                "6M Change Percentile": _ten_year_yield_percent_display(m),\n'
    '                "6M Change Score": n,\n'
    '                "12M Change Percentile": _ten_year_yield_percent_display(o),\n'
    '                "12M Change Score": p,\n'
    '                "10Y Yield Regime": _ten_year_yield_level_regime(c, f),\n'
    '                "10Y Yield Asset Bias": _ten_year_yield_asset_bias_label(y),\n'
    '                "1M 10Y Yield Shock Flag": _ten_year_yield_shock_label(e),\n'
    '                "10Y Yield Impulse Score": t,\n'
    '                "10Y Yield Cycle Turning Point Score": u,\n'
    '                "10Y Yield Shock Score": v,\n'
    '                "Long-Rate Restriction Score": w,\n'
    '                "10Y Yield Extreme Score": x,\n'
    '                "Coincident Score": y,\n'
    '                "Leading Score": z,\n'
    '                "Warning Score": aa,\n'
    '                "State": _ten_year_yield_pressure_regime(y),\n'
    '                "Regime": _ten_year_yield_momentum_label(y, z),\n'
    '                "Signal Comment": _ten_year_yield_signal_interpretation(z, aa),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _ten_year_yield_warning_label(aa),\n'
    '                "Level Z-Score": ah,\n'
    '                "3M Change Z-Score": ai,\n'
    '                "6M Change Z-Score": aj,\n'
    '                "12M Change Z-Score": ak,\n'
    '                "Extreme Flag": _ten_year_yield_outlier_label([ah, ai, aj, ak]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, TEN_YEAR_YIELD_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _score_percent_rank_inverted(percent_rank: Any) -> Any:\n'
    '    if is_blank(percent_rank):\n'
    '        return ""\n'
    '    return round(10 - (percent_rank * 20), 2)\n'
    '\n'
    '\n'
    'def _yield_curve_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A26-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _yield_curve_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A26-only spreadsheet-order mean for the 3M curve spread."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _yield_curve_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    """A26-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _yield_curve_metric_ranks_sheets(\n'
    '    values: Sequence[Any], sample_counts: Sequence[int], min_sample_count: int = 60\n'
    ') -> list[Any]:\n'
    '    """Gate every A26 percentile from Sample Count D, not metric counts."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for idx, value in enumerate(values):\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or sample_counts[idx] < min_sample_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _yield_curve_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _yield_curve_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _yield_curve_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        "" if is_blank(rank) else _yield_curve_sheets_round(10 - (float(rank) * 20), 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _yield_curve_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _yield_curve_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _yield_curve_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A26 formulas from rounded visible decimal operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _yield_curve_threshold_value(value: Any) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    for boundary in (-0.30, -0.20, 0.20, 0.30):\n'
    '        if math.isclose(numeric, boundary, rel_tol=0.0, abs_tol=1e-12):\n'
    '            return boundary\n'
    '    return numeric\n'
    '\n'
    '\n'
    'def _yield_curve_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    f_for_regime = 0.0 if is_blank(f) else f\n'
    '    if c >= 1.5 and f_for_regime >= 0:\n'
    '        return "Steep curve / easing-recovery signal"\n'
    '    if c >= 1.5 and f_for_regime < 0:\n'
    '        return "Steep curve but flattening"\n'
    '    if c >= 0.5 and f_for_regime >= 0:\n'
    '        return "Moderately positive curve and steepening"\n'
    '    if c >= 0.5 and f_for_regime < 0:\n'
    '        return "Positive curve but flattening"\n'
    '    if c > 0 and f_for_regime >= 0:\n'
    '        return "Near inversion but improving"\n'
    '    if c > 0 and f_for_regime < 0:\n'
    '        return "Near inversion and flattening"\n'
    '    if c <= 0 and f_for_regime >= 0:\n'
    '        return "Inverted but steepening"\n'
    '    if c <= 0 and f_for_regime < 0:\n'
    '        return "Inverted and flattening further"\n'
    '    return "Mixed curve regime"\n'
    '\n'
    '\n'
    'def _yield_curve_recession_risk_label(c: Any, f: Any, aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if not is_blank(c) and c < 0 and aa >= 6:\n'
    '        return "High recession-risk curve signal"\n'
    '    if not is_blank(c) and c < 0 and aa >= 2:\n'
    '        return "Elevated recession-risk curve signal"\n'
    '    if not is_blank(c) and c >= 0 and c < 0.5 and aa >= 5:\n'
    '        return "High late-cycle / near-inversion warning"\n'
    '    if not is_blank(c) and c >= 0 and c < 0.5 and aa >= 2:\n'
    '        return "Elevated late-cycle / near-inversion warning"\n'
    '    if (\n'
    '        not is_blank(c)\n'
    '        and not is_blank(f)\n'
    '        and c >= 0.5\n'
    '        and c < 1.5\n'
    '        and f < 0\n'
    '        and aa >= 5\n'
    '    ):\n'
    '        return "Curve flattening stress warning"\n'
    '    if not is_blank(c) and c >= 1.5 and aa <= -2:\n'
    '        return "Low recession-risk / steep curve signal"\n'
    '    if aa <= -6:\n'
    '        return "Strong curve-easing / recovery signal"\n'
    '    return "Neutral curve recession-risk signal"\n'
    '\n'
    '\n'
    'def _yield_curve_shock_label(e: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _yield_curve_threshold_value(e)\n'
    '    if threshold_value <= -0.3:\n'
    '        return "1M curve-flattening shock"\n'
    '    if threshold_value >= 0.3:\n'
    '        return "1M curve-steepening shock"\n'
    '    return "No 1M curve shock"\n'
    '\n'
    '\n'
    'def _yield_curve_flattening_stress_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c > 1 and f < 0 and g < 0:\n'
    '        return 4\n'
    '    if c <= 1 and c > 0 and f < 0 and g < 0:\n'
    '        return 5\n'
    '    if c <= 0 and f < 0 and g < 0:\n'
    '        return 6\n'
    '    if c <= 0 and f > 0 and g > 0:\n'
    '        return -4\n'
    '    if c > 1 and f > 0 and g > 0:\n'
    '        return -5\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _yield_curve_shock_score(e: Any) -> Any:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _yield_curve_threshold_value(e)\n'
    '    if threshold_value <= -0.3:\n'
    '        return 6\n'
    '    if threshold_value <= -0.2:\n'
    '        return 4\n'
    '    if threshold_value >= 0.3:\n'
    '        return -6\n'
    '    if threshold_value >= 0.2:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _yield_curve_inversion_restriction_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c <= 0 and f < 0 and g < 0:\n'
    '        return 6\n'
    '    if c <= 0.5 and f < 0 and g < 0:\n'
    '        return 4\n'
    '    if c >= 2 and f > 0 and g > 0:\n'
    '        return -6\n'
    '    if c >= 1 and f > 0 and g > 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _yield_curve_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return 6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return 4\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return -6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _yield_curve_stress_regime(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 2:\n'
    '        return "Yield-curve stress rising"\n'
    '    if y <= -2:\n'
    '        return "Yield-curve stress easing"\n'
    '    return "Neutral yield-curve stress"\n'
    '\n'
    '\n'
    'def _yield_curve_stress_momentum_label(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 2 and z >= 2:\n'
    '        return "Yield-curve stress rising"\n'
    '    if y >= 2 and z <= -2:\n'
    '        return "Yield-curve stress elevated but easing"\n'
    '    if y >= 2 and -2 < z < 2:\n'
    '        return "Yield-curve stress elevated / stable"\n'
    '    if y <= -2 and z >= 2:\n'
    '        return "Yield curve steep but flattening"\n'
    '    if y <= -2 and z <= -2:\n'
    '        return "Yield-curve stress easing"\n'
    '    if y <= -2 and -2 < z < 2:\n'
    '        return "Yield-curve stress low / stable"\n'
    '    if -2 < y < 2 and z >= 2:\n'
    '        return "Yield-curve stress trend rising"\n'
    '    if -2 < y < 2 and z <= -2:\n'
    '        return "Yield-curve stress trend easing"\n'
    '    return "Yield curve broadly neutral"\n'
    '\n'
    '\n'
    'def _yield_curve_signal_interpretation(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 6 and aa >= 0:\n'
    '        return "Strong upside yield-curve stress signal with supportive recession-risk profile"\n'
    '    if z >= 6 and aa < 0:\n'
    '        return "Strong upside yield-curve stress signal, but recession-risk profile is easing"\n'
    '    if z >= 2 and aa >= 0:\n'
    '        return "Moderately positive yield-curve stress signal"\n'
    '    if z >= 2 and aa < 0:\n'
    '        return "Yield-curve flattening signal, but recession-risk profile is easing"\n'
    '    if z > -2 and aa > -3 and aa < 3:\n'
    '        return "Neutral/mixed yield-curve signal"\n'
    '    if z <= -6 and aa <= 0:\n'
    '        return "Strong yield-curve steepening/easing signal with downside confirmation"\n'
    '    if z <= -6 and aa > 0:\n'
    '        return "Strong yield-curve steepening/easing signal, but recession-risk pressure remains elevated"\n'
    '    if z <= -2 and aa <= 0:\n'
    '        return "Moderately negative yield-curve stress / steepening signal"\n'
    '    if z <= -2 and aa > 0:\n'
    '        return "Yield-curve steepening signal, but recession-risk pressure remains elevated"\n'
    '    return "Mixed/transition yield-curve signal"\n'
    '\n'
    '\n'
    'def _yield_curve_warning_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 6:\n'
    '        return "High yield-curve recession-risk warning"\n'
    '    if aa >= 3:\n'
    '        return "Moderate yield-curve recession-risk warning"\n'
    '    if aa > -3:\n'
    '        return "Neutral yield-curve warning profile"\n'
    '    if aa > -6:\n'
    '        return "Moderate curve-steepening / easing warning"\n'
    '    return "High curve-steepening / easing signal"\n'
    '\n'
    '\n'
    'def _yield_curve_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme yield-curve stress / inversion pressure"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong yield-curve stress / inversion pressure"\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme curve-steepening / easing signal"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong curve-steepening / easing signal"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_yield_curve_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    raw_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    smoothed_count = 0\n'
    '    smoothed_counts_so_far: list[int] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if raw_counts[idx] < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append(\n'
    '                "" if not window else _yield_curve_mean_sheets(window)\n'
    '            )\n'
    '\n'
    '        c = moving_averages[idx]\n'
    '        if not is_blank(c):\n'
    '            smoothed_count += 1\n'
    '        smoothed_counts_so_far.append(smoothed_count)\n'
    '        if (\n'
    '            smoothed_count < 2\n'
    '            or is_blank(c)\n'
    '            or idx < 1\n'
    '            or is_blank(moving_averages[idx - 1])\n'
    '        ):\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(c - moving_averages[idx - 1])\n'
    '        for lag, min_count, target in (\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                smoothed_count < min_count\n'
    '                or is_blank(c)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(c - moving_averages[idx - lag])\n'
    '\n'
    '    smoothed_counts = smoothed_counts_so_far\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    # A26 intentionally gates all four ranks from Sample Count D.\n'
    '    c_ranks = _yield_curve_metric_ranks_sheets(moving_averages, smoothed_counts)\n'
    '    f_ranks = _yield_curve_metric_ranks_sheets(three_period_changes, smoothed_counts)\n'
    '    g_ranks = _yield_curve_metric_ranks_sheets(six_period_changes, smoothed_counts)\n'
    '    h_ranks = _yield_curve_metric_ranks_sheets(twelve_period_changes, smoothed_counts)\n'
    '    c_zscores = _rounded_expanding_zscores(moving_averages, smoothed_counts, 60)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 60)\n'
    '    g_zscores = _rounded_expanding_zscores(six_period_changes, g_counts, 60)\n'
    '    h_zscores = _rounded_expanding_zscores(twelve_period_changes, h_counts, 60)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = smoothed_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _yield_curve_score_percentile(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _yield_curve_score_percentile(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _yield_curve_score_percentile(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _yield_curve_score_percentile(o)\n'
    '        t = (\n'
    '            _yield_curve_sheets_round(\n'
    '                clamp(\n'
    '                    _yield_curve_weighted_sum(\n'
    '                        (0.05, j), (0.45, l), (0.35, n), (0.15, p)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        u = _yield_curve_flattening_stress_score(c, f, g)\n'
    '        v = _yield_curve_shock_score(e)\n'
    '        w = _yield_curve_inversion_restriction_score(c, f, g)\n'
    '        ah = c_zscores[idx]\n'
    '        ai = f_zscores[idx]\n'
    '        aj = g_zscores[idx]\n'
    '        ak = h_zscores[idx]\n'
    '        x = _yield_curve_outlier_score([ah, ai, aj, ak])\n'
    '        y = (\n'
    '            _yield_curve_sheets_round(\n'
    '                clamp(\n'
    '                    _yield_curve_weighted_sum(\n'
    '                        (0.45, j), (0.30, l), (0.15, n), (0.10, p)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            _yield_curve_sheets_round(\n'
    '                clamp(\n'
    '                    _yield_curve_weighted_sum(\n'
    '                        (0.20, j), (0.30, l), (0.25, n), (0.10, p), (0.10, t), (0.05, u)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        aa = (\n'
    '            _yield_curve_sheets_round(\n'
    '                clamp(\n'
    '                    _yield_curve_weighted_sum(\n'
    '                        (0.30, u), (0.25, v), (0.30, w), (0.15, x)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            "Insufficient history"\n'
    '            if d < 60\n'
    '            else ("Medium reliability" if d < 120 else "High reliability")\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Curve Spread": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Curve Change": e,\n'
    '                "3M Curve Change": f,\n'
    '                "6M Curve Change": g,\n'
    '                "12M Curve Change": h,\n'
    '                "Curve Level Percentile": _yield_curve_percent_display(i),\n'
    '                "Curve Stress Level Score": j,\n'
    '                "3M Change Percentile": _yield_curve_percent_display(k),\n'
    '                "3M Change Stress Score": l,\n'
    '                "6M Change Percentile": _yield_curve_percent_display(m),\n'
    '                "6M Change Stress Score": n,\n'
    '                "12M Change Percentile": _yield_curve_percent_display(o),\n'
    '                "12M Change Stress Score": p,\n'
    '                "Curve Regime": _yield_curve_regime(c, f),\n'
    '                "Recession Risk Bias": _yield_curve_recession_risk_label(c, f, aa),\n'
    '                "Curve Shock Flag": _yield_curve_shock_label(e),\n'
    '                "Curve Impulse Score": t,\n'
    '                "Curve Cycle Turning Point Score": u,\n'
    '                "Curve Shock Score": v,\n'
    '                "Recession Risk Pressure Score": w,\n'
    '                "Curve Extreme Score": x,\n'
    '                "Coincident Score": y,\n'
    '                "Leading Score": z,\n'
    '                "Warning Score": aa,\n'
    '                "State": _yield_curve_stress_regime(y),\n'
    '                "Regime": _yield_curve_stress_momentum_label(y, z),\n'
    '                "Signal Comment": _yield_curve_signal_interpretation(z, aa),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _yield_curve_warning_label(aa),\n'
    '                "Level Z-Score": ah,\n'
    '                "3M Change Z-Score": ai,\n'
    '                "6M Change Z-Score": aj,\n'
    '                "12M Change Z-Score": ak,\n'
    '                "Extreme Flag": _yield_curve_outlier_label([ah, ai, aj, ak]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, YIELD_CURVE_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _term_premium_level_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if not is_blank(f) and c >= 2 and f > 0:\n'
    '        return "High and rising term-premium regime"\n'
    '    if not is_blank(f) and c >= 2 and f <= 0:\n'
    '        return "High but cooling term-premium regime"\n'
    '    if not is_blank(f) and c >= 1 and f > 0:\n'
    '        return "Elevated and rising term-premium regime"\n'
    '    if not is_blank(f) and c >= 1 and f <= 0:\n'
    '        return "Elevated but easing term-premium regime"\n'
    '    if not is_blank(f) and c >= 0 and f > 0:\n'
    '        return "Positive term premium and rising"\n'
    '    if not is_blank(f) and c >= 0 and f <= 0:\n'
    '        return "Positive term premium but easing"\n'
    '    if not is_blank(f) and c < 0 and f > 0:\n'
    '        return "Negative term premium but rising"\n'
    '    if not is_blank(f) and c < 0 and f <= 0:\n'
    '        return "Negative and easing term-premium regime"\n'
    '    return "Mixed term-premium regime"\n'
    '\n'
    '\n'
    'def _term_premium_asset_bias_label(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 6:\n'
    '        return "Strong bearish duration / rising term-premium risk"\n'
    '    if y >= 2:\n'
    '        return "Mild bearish duration / higher term-premium pressure"\n'
    '    if y <= -6:\n'
    '        return "Strong bullish duration / falling term-premium support"\n'
    '    if y <= -2:\n'
    '        return "Mild bullish duration / lower term-premium pressure"\n'
    '    return "Neutral term-premium asset bias"\n'
    '\n'
    '\n'
    'def _term_premium_shock_label(e: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if e >= 0.3:\n'
    '        return "1M term-premium upside shock"\n'
    '    if e <= -0.3:\n'
    '        return "1M term-premium downside shock"\n'
    '    return "No 1M term-premium shock"\n'
    '\n'
    '\n'
    'def _term_premium_recovery_tightening_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c < 0 and f > 0 and g > 0:\n'
    '        return 5\n'
    '    if c >= 1 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 1 and f < 0 and g < 0:\n'
    '        return -5\n'
    '    if c < 0 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if f > 0 and g > 0:\n'
    '        return 4\n'
    '    if f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _term_premium_shock_score(e: Any) -> Any:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if e >= 0.3:\n'
    '        return 6\n'
    '    if e >= 0.2:\n'
    '        return 4\n'
    '    if e <= -0.3:\n'
    '        return -6\n'
    '    if e <= -0.2:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _term_premium_restriction_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c >= 2 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 1 and f > 0 and g > 0:\n'
    '        return 4\n'
    '    if c <= 0 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if c <= 0.5 and f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _term_premium_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _term_premium_pressure_regime(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 2:\n'
    '        return "Term-premium pressure rising"\n'
    '    if y <= -2:\n'
    '        return "Term-premium pressure easing"\n'
    '    return "Neutral term-premium pressure"\n'
    '\n'
    '\n'
    'def _term_premium_momentum_label(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 2 and z >= 2:\n'
    '        return "Term-premium pressure rising"\n'
    '    if y >= 2 and z <= -2:\n'
    '        return "Term premium elevated but easing"\n'
    '    if y >= 2 and -2 < z < 2:\n'
    '        return "Term-premium pressure elevated / stable"\n'
    '    if y <= -2 and z >= 2:\n'
    '        return "Term premium low but rising"\n'
    '    if y <= -2 and z <= -2:\n'
    '        return "Term-premium pressure easing"\n'
    '    if y <= -2 and -2 < z < 2:\n'
    '        return "Term-premium pressure low / stable"\n'
    '    if -2 < y < 2 and z >= 2:\n'
    '        return "Term-premium trend rising"\n'
    '    if -2 < y < 2 and z <= -2:\n'
    '        return "Term-premium trend easing"\n'
    '    return "Term premium broadly neutral"\n'
    '\n'
    '\n'
    'def _term_premium_signal_interpretation(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 6 and aa >= 0:\n'
    '        return "Strong upside term-premium signal with supportive risk-premium pressure profile"\n'
    '    if z >= 6 and aa < 0:\n'
    '        return "Strong upside term-premium signal, but risk-premium warning profile is easing"\n'
    '    if z >= 2 and aa >= 0:\n'
    '        return "Moderately positive term-premium pressure signal"\n'
    '    if z >= 2 and aa < 0:\n'
    '        return "Term-premium rising signal, but risk-premium pressure is easing"\n'
    '    if z > -2 and aa > -3 and aa < 3:\n'
    '        return "Neutral/mixed term-premium signal"\n'
    '    if z <= -6 and aa <= 0:\n'
    '        return "Strong term-premium easing signal with downside confirmation"\n'
    '    if z <= -6 and aa > 0:\n'
    '        return "Strong term-premium easing signal, but risk-premium pressure remains elevated"\n'
    '    if z <= -2 and aa <= 0:\n'
    '        return "Moderately negative term-premium/easing signal"\n'
    '    if z <= -2 and aa > 0:\n'
    '        return "Term-premium easing signal, but risk-premium pressure remains elevated"\n'
    '    return "Mixed/transition term-premium signal"\n'
    '\n'
    '\n'
    'def _term_premium_warning_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 6:\n'
    '        return "High upside term-premium risk warning"\n'
    '    if aa >= 3:\n'
    '        return "Moderate upside term-premium risk warning"\n'
    '    if aa > -3:\n'
    '        return "Neutral term-premium warning profile"\n'
    '    if aa > -6:\n'
    '        return "Moderate term-premium easing warning"\n'
    '    return "High term-premium easing signal"\n'
    '\n'
    '\n'
    'def _term_premium_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside term-premium pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside term-premium pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside term-premium pressure"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside term-premium pressure"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_term_premium_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    raw_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    smoothed_count = 0\n'
    '    smoothed_counts_so_far: list[int] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if raw_counts[idx] < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append("" if not window else sum(window) / len(window))\n'
    '\n'
    '        c = moving_averages[idx]\n'
    '        if not is_blank(c):\n'
    '            smoothed_count += 1\n'
    '        smoothed_counts_so_far.append(smoothed_count)\n'
    '        if (\n'
    '            smoothed_count < 2\n'
    '            or is_blank(c)\n'
    '            or idx < 1\n'
    '            or is_blank(moving_averages[idx - 1])\n'
    '        ):\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(c - moving_averages[idx - 1])\n'
    '        for lag, min_count, target in (\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                smoothed_count < min_count\n'
    '                or is_blank(c)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(c - moving_averages[idx - lag])\n'
    '\n'
    '    smoothed_counts = smoothed_counts_so_far\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    c_ranks = _expectations_metric_ranks(moving_averages)\n'
    '    f_ranks = _expectations_metric_ranks(three_period_changes)\n'
    '    g_ranks = _expectations_metric_ranks(six_period_changes)\n'
    '    h_ranks = _expectations_metric_ranks(twelve_period_changes)\n'
    '    c_zscores = _rounded_expanding_zscores(moving_averages, smoothed_counts, 60)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 60)\n'
    '    g_zscores = _rounded_expanding_zscores(six_period_changes, g_counts, 60)\n'
    '    h_zscores = _rounded_expanding_zscores(twelve_period_changes, h_counts, 60)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = smoothed_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _score_percent_rank(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _score_percent_rank(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _score_percent_rank(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _score_percent_rank(o)\n'
    '        t = (\n'
    '            round(clamp(0.45 * l + 0.35 * n + 0.15 * p + 0.05 * j, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (l, n, p, j))\n'
    '            else ""\n'
    '        )\n'
    '        u = _term_premium_recovery_tightening_score(c, f, g)\n'
    '        v = _term_premium_shock_score(e)\n'
    '        w = _term_premium_restriction_score(c, f, g)\n'
    '        ah = c_zscores[idx]\n'
    '        ai = f_zscores[idx]\n'
    '        aj = g_zscores[idx]\n'
    '        ak = h_zscores[idx]\n'
    '        x = _term_premium_outlier_score([ah, ai, aj, ak])\n'
    '        y = (\n'
    '            round(clamp(0.45 * j + 0.3 * l + 0.15 * n + 0.1 * p, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            round(\n'
    '                clamp(\n'
    '                    0.15 * j + 0.3 * l + 0.25 * n + 0.15 * p + 0.1 * t + 0.05 * u,\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        aa = (\n'
    '            round(clamp(0.3 * u + 0.25 * v + 0.3 * w + 0.15 * x, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3-Period Moving Average": c,\n'
    '                "Smoothed Observation Count": d,\n'
    '                "1-Period Change": e,\n'
    '                "3-Period Change": f,\n'
    '                "6-Period Change": g,\n'
    '                "12-Period Change": h,\n'
    '                "Term Premium Level PercentRank": i,\n'
    '                "Term Premium Level Score": j,\n'
    '                "3-Period Term Premium Change PercentRank": k,\n'
    '                "3-Period Term Premium Change Score": l,\n'
    '                "6-Period Term Premium Change PercentRank": m,\n'
    '                "6-Period Term Premium Change Score": n,\n'
    '                "12-Period Term Premium Change PercentRank": o,\n'
    '                "12-Period Term Premium Change Score": p,\n'
    '                "Term Premium Level Regime": _term_premium_level_regime(c, f),\n'
    '                "Term Premium Asset Bias Label": _term_premium_asset_bias_label(y),\n'
    '                "1-Period Term Premium Shock Label": _term_premium_shock_label(e),\n'
    '                "Term Premium Momentum Composite Score": t,\n'
    '                "Term Premium Recovery / Tightening Score": u,\n'
    '                "Term Premium Shock Score": v,\n'
    '                "Term Premium Restriction Score": w,\n'
    '                "Term Premium Outlier Score": x,\n'
    '                "Term Premium Leading Score": y,\n'
    '                "Term Premium Warning Composite Score": z,\n'
    '                "Term Premium Final Warning Score": aa,\n'
    '                "Term Premium Pressure Regime": _term_premium_pressure_regime(y),\n'
    '                "Term Premium Momentum Label": _term_premium_momentum_label(y, z),\n'
    '                "Term Premium Signal Interpretation": _term_premium_signal_interpretation(\n'
    '                    z, aa\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Term Premium Warning Risk Label": _term_premium_warning_label(aa),\n'
    '                "Term Premium Level Z-Score": ah,\n'
    '                "3-Period Term Premium Change Z-Score": ai,\n'
    '                "6-Period Term Premium Change Z-Score": aj,\n'
    '                "12-Period Term Premium Change Z-Score": ak,\n'
    '                "Term Premium Outlier Label": _term_premium_outlier_label(\n'
    '                    [ah, ai, aj, ak]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, TERM_PREMIUM_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _interbank_rate_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A28-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _interbank_rate_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A28-only spreadsheet-order mean using full numeric source precision."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _interbank_rate_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A28-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _interbank_rate_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    """Use each A28 metric\'s own numeric 60-observation history gate."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _interbank_rate_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _interbank_rate_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _interbank_rate_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        ""\n'
    '        if is_blank(rank)\n'
    '        else _interbank_rate_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _interbank_rate_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _interbank_rate_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _interbank_rate_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A28 formulas from rounded visible decimal operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _interbank_rate_threshold_value(value: Any) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    for boundary in (0.30, 0.20, -0.20, -0.30):\n'
    '        if math.isclose(numeric, boundary, rel_tol=0.0, abs_tol=1e-12):\n'
    '            return boundary\n'
    '    return numeric\n'
    '\n'
    '\n'
    'def _interbank_rate_level_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    f_for_regime = 0.0 if is_blank(f) else f\n'
    '    if c >= 3 and f_for_regime >= 0:\n'
    '        return "Elevated and rising interbank-rate regime"\n'
    '    if c >= 3 and f_for_regime < 0:\n'
    '        return "Elevated but easing interbank-rate regime"\n'
    '    if c >= 1 and f_for_regime >= 0:\n'
    '        return "Moderate interbank rate and rising"\n'
    '    if c >= 1 and f_for_regime < 0:\n'
    '        return "Moderate interbank rate but easing"\n'
    '    if c < 1 and f_for_regime >= 0:\n'
    '        return "Low interbank rate but rising"\n'
    '    if c < 1 and f_for_regime < 0:\n'
    '        return "Low and easing interbank-rate regime"\n'
    '    return "Mixed interbank-rate regime"\n'
    '\n'
    '\n'
    'def _interbank_rate_policy_bias_label(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 6:\n'
    '        return "Strong hawkish policy-rate / money-market tightening bias"\n'
    '    if y >= 2:\n'
    '        return "Mild hawkish policy-rate / tighter money-market bias"\n'
    '    if y <= -6:\n'
    '        return "Strong dovish policy-rate / money-market easing bias"\n'
    '    if y <= -2:\n'
    '        return "Mild dovish policy-rate / easier money-market bias"\n'
    '    return "Neutral interbank-rate policy bias"\n'
    '\n'
    '\n'
    'def _interbank_rate_shock_label(e: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _interbank_rate_threshold_value(e)\n'
    '    if threshold_value >= 0.3:\n'
    '        return "1M interbank-rate upside shock"\n'
    '    if threshold_value <= -0.3:\n'
    '        return "1M interbank-rate downside shock"\n'
    '    return "No 1M interbank-rate shock"\n'
    '\n'
    '\n'
    'def _interbank_rate_recovery_tightening_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c < 1 and f > 0 and g > 0:\n'
    '        return 5\n'
    '    if c >= 3 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 3 and f < 0 and g < 0:\n'
    '        return -5\n'
    '    if c < 1 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if f > 0 and g > 0:\n'
    '        return 4\n'
    '    if f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interbank_rate_shock_score(e: Any) -> Any:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    threshold_value = _interbank_rate_threshold_value(e)\n'
    '    if threshold_value >= 0.3:\n'
    '        return 6\n'
    '    if threshold_value >= 0.2:\n'
    '        return 4\n'
    '    if threshold_value <= -0.3:\n'
    '        return -6\n'
    '    if threshold_value <= -0.2:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interbank_rate_restriction_score(c: Any, f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, f, g)):\n'
    '        return ""\n'
    '    if c >= 5 and f > 0 and g > 0:\n'
    '        return 6\n'
    '    if c >= 3 and f > 0 and g > 0:\n'
    '        return 4\n'
    '    if c <= 1 and f < 0 and g < 0:\n'
    '        return -6\n'
    '    if c <= 2 and f < 0 and g < 0:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interbank_rate_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interbank_rate_pressure_regime(y: Any) -> str:\n'
    '    if is_blank(y):\n'
    '        return ""\n'
    '    if y >= 2:\n'
    '        return "Interbank-rate pressure rising"\n'
    '    if y <= -2:\n'
    '        return "Interbank-rate pressure easing"\n'
    '    return "Neutral interbank-rate pressure"\n'
    '\n'
    '\n'
    'def _interbank_rate_momentum_label(y: Any, z: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, z)):\n'
    '        return ""\n'
    '    if y >= 2 and z >= 2:\n'
    '        return "Interbank-rate pressure rising"\n'
    '    if y >= 2 and z <= -2:\n'
    '        return "Interbank-rate pressure elevated but easing"\n'
    '    if y >= 2 and -2 < z < 2:\n'
    '        return "Interbank-rate pressure elevated / stable"\n'
    '    if y <= -2 and z >= 2:\n'
    '        return "Interbank rate low but rising"\n'
    '    if y <= -2 and z <= -2:\n'
    '        return "Interbank-rate pressure easing"\n'
    '    if y <= -2 and -2 < z < 2:\n'
    '        return "Interbank-rate pressure low / stable"\n'
    '    if -2 < y < 2 and z >= 2:\n'
    '        return "Interbank-rate trend rising"\n'
    '    if -2 < y < 2 and z <= -2:\n'
    '        return "Interbank-rate trend easing"\n'
    '    return "Interbank rate broadly neutral"\n'
    '\n'
    '\n'
    'def _interbank_rate_signal_interpretation(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(v) for v in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 6 and aa >= 0:\n'
    '        return "Strong upside interbank-rate signal with supportive monetary-restriction profile"\n'
    '    if z >= 6 and aa < 0:\n'
    '        return "Strong upside interbank-rate signal, but monetary-restriction profile is easing"\n'
    '    if z >= 2 and aa >= 0:\n'
    '        return "Moderately positive interbank-rate tightening signal"\n'
    '    if z >= 2 and aa < 0:\n'
    '        return (\n'
    '            "Interbank-rate rising signal, but monetary-restriction pressure is easing"\n'
    '        )\n'
    '    if z > -2 and aa > -3 and aa < 3:\n'
    '        return "Neutral/mixed interbank-rate signal"\n'
    '    if z <= -6 and aa <= 0:\n'
    '        return "Strong interbank-rate easing signal with downside confirmation"\n'
    '    if z <= -6 and aa > 0:\n'
    '        return "Strong interbank-rate easing signal, but restriction pressure remains elevated"\n'
    '    if z <= -2 and aa <= 0:\n'
    '        return "Moderately negative interbank-rate/easing signal"\n'
    '    if z <= -2 and aa > 0:\n'
    '        return "Interbank-rate easing signal, but restriction pressure remains elevated"\n'
    '    return "Mixed/transition interbank-rate signal"\n'
    '\n'
    '\n'
    'def _interbank_rate_warning_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 6:\n'
    '        return "High upside interbank-rate restriction warning"\n'
    '    if aa >= 3:\n'
    '        return "Moderate upside interbank-rate restriction warning"\n'
    '    if aa > -3:\n'
    '        return "Neutral interbank-rate warning profile"\n'
    '    if aa > -6:\n'
    '        return "Moderate interbank-rate easing warning"\n'
    '    return "High interbank-rate easing signal"\n'
    '\n'
    '\n'
    'def _interbank_rate_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme upside interbank-rate pressure"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong upside interbank-rate pressure"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme downside interbank-rate pressure"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong downside interbank-rate pressure"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A29-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A29-only spreadsheet-order average over numeric cells."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A29-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    """Rank each A29 metric against its own expanding numeric history."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _central_bank_liquidity_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append(\n'
    '            "" if rank == "" else _central_bank_liquidity_sheets_round(rank, 3)\n'
    '        )\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        ""\n'
    '        if is_blank(rank)\n'
    '        else _central_bank_liquidity_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _central_bank_liquidity_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A29 composites from their rounded visible operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_rounded_expanding_zscores(\n'
    '    values: list[object], counts: list[int], min_count: int = 60\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else _central_bank_liquidity_sheets_round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, counts, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_regime(h: Any, j: Any, k: Any) -> str:\n'
    '    if is_blank(h):\n'
    '        return ""\n'
    '    if not is_blank(j) and not is_blank(k) and h > 0 and j > 0 and k > 0:\n'
    '        return "Liquidity expansion regime"\n'
    '    if not is_blank(j) and not is_blank(k) and h > 0 and j < 0 and k < 0:\n'
    '        return "Positive liquidity but contracting"\n'
    '    if not is_blank(j) and not is_blank(k) and h <= 0 and j > 0 and k > 0:\n'
    '        return "Low liquidity but recovering"\n'
    '    if not is_blank(j) and not is_blank(k) and h <= 0 and j < 0 and k < 0:\n'
    '        return "Liquidity contraction regime"\n'
    '    if not is_blank(j) and not is_blank(k) and j > 0 and k < 0:\n'
    '        return "Early liquidity recovery / mixed impulse"\n'
    '    if not is_blank(j) and not is_blank(k) and j < 0 and k > 0:\n'
    '        return "Early liquidity deterioration / mixed impulse"\n'
    '    return "Neutral/mixed liquidity regime"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_asset_bias_label(af: Any) -> str:\n'
    '    if is_blank(af):\n'
    '        return ""\n'
    '    if af >= 6:\n'
    '        return "Strongly supportive for risk assets / liquidity-sensitive assets"\n'
    '    if af >= 2:\n'
    '        return "Mildly supportive for risk assets"\n'
    '    if af <= -6:\n'
    '        return "Strong liquidity headwind for risk assets"\n'
    '    if af <= -2:\n'
    '        return "Mild liquidity headwind for risk assets"\n'
    '    return "Neutral liquidity bias"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_shock_label(i: Any) -> str:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 250000:\n'
    '        return "1M major liquidity injection shock"\n'
    '    if i >= 100000:\n'
    '        return "1M moderate liquidity injection"\n'
    '    if i <= -250000:\n'
    '        return "1M major liquidity drain shock"\n'
    '    if i <= -100000:\n'
    '        return "1M moderate liquidity drain"\n'
    '    return "No major liquidity shock"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_recovery_score(j: Any, k: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (j, k)):\n'
    '        return ""\n'
    '    if j > 0 and k > 0:\n'
    '        return 5\n'
    '    if j < 0 and k < 0:\n'
    '        return -5\n'
    '    if j > 0 and k < 0:\n'
    '        return 3\n'
    '    if j < 0 and k > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_shock_score(i: Any) -> Any:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 250000:\n'
    '        return 6\n'
    '    if i >= 100000:\n'
    '        return 4\n'
    '    if i <= -250000:\n'
    '        return -6\n'
    '    if i <= -100000:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_withdrawal_risk_score(\n'
    '    ao: Any, ap: Any, aq: Any, j: Any, k: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (ao, ap, aq, j, k)):\n'
    '        return ""\n'
    '    if ao > 1.5 and ap < -0.5 and aq < -0.5 and j < 0 and k < 0:\n'
    '        return -6\n'
    '    if ao > 1 and ap < 0 and aq < 0 and j < 0:\n'
    '        return -4\n'
    '    if ao < -1 and ap > 0 and aq > 0 and j > 0 and k > 0:\n'
    '        return 6\n'
    '    if ap > 1 and aq > 1 and j > 0 and k > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_conditions_regime(af: Any) -> str:\n'
    '    if is_blank(af):\n'
    '        return ""\n'
    '    if af >= 2:\n'
    '        return "Central bank liquidity improving"\n'
    '    if af <= -2:\n'
    '        return "Central bank liquidity deteriorating"\n'
    '    return "Neutral liquidity conditions"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_momentum_label(af: Any, ag: Any) -> str:\n'
    '    if any(is_blank(x) for x in (af, ag)):\n'
    '        return ""\n'
    '    if af >= 2 and ag >= 2:\n'
    '        return "Central bank liquidity improving"\n'
    '    if af >= 2 and ag <= -2:\n'
    '        return "Liquidity supportive but deteriorating"\n'
    '    if af >= 2 and -2 < ag < 2:\n'
    '        return "Liquidity supportive / stable"\n'
    '    if af <= -2 and ag >= 2:\n'
    '        return "Liquidity weak but improving"\n'
    '    if af <= -2 and ag <= -2:\n'
    '        return "Central bank liquidity deteriorating"\n'
    '    if af <= -2 and -2 < ag < 2:\n'
    '        return "Liquidity weak / stable"\n'
    '    if -2 < af < 2 and ag >= 2:\n'
    '        return "Liquidity impulse improving"\n'
    '    if -2 < af < 2 and ag <= -2:\n'
    '        return "Liquidity impulse deteriorating"\n'
    '    return "Central bank liquidity broadly neutral"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_signal_interpretation(ag: Any, ah: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ag, ah)):\n'
    '        return ""\n'
    '    if ag >= 6 and ah >= 0:\n'
    '        return "Strong positive liquidity impulse with supportive risk-asset liquidity profile"\n'
    '    if ag >= 6 and ah < 0:\n'
    '        return "Strong positive liquidity impulse, but withdrawal-risk profile remains mixed"\n'
    '    if ag >= 2 and ah >= 0:\n'
    '        return "Moderately positive liquidity impulse signal"\n'
    '    if ag >= 2 and ah < 0:\n'
    '        return (\n'
    '            "Liquidity impulse improving, but withdrawal-risk profile remains fragile"\n'
    '        )\n'
    '    if ag > -2 and ah > -3 and ah < 3:\n'
    '        return "Neutral/mixed liquidity signal"\n'
    '    if ag <= -6 and ah <= 0:\n'
    '        return "Strong liquidity deterioration signal with downside confirmation"\n'
    '    if ag <= -6 and ah > 0:\n'
    '        return "Strong liquidity deterioration signal, but liquidity support remains partly intact"\n'
    '    if ag <= -2 and ah <= 0:\n'
    '        return "Moderately negative liquidity impulse signal"\n'
    '    if ag <= -2 and ah > 0:\n'
    '        return "Liquidity impulse deteriorating, but support profile remains mixed"\n'
    '    return "Mixed/transition liquidity signal"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_warning_label(ah: Any) -> str:\n'
    '    if is_blank(ah):\n'
    '        return ""\n'
    '    if ah >= 6:\n'
    '        return "High upside liquidity-support warning"\n'
    '    if ah >= 3:\n'
    '        return "Moderate upside liquidity-support warning"\n'
    '    if ah > -3:\n'
    '        return "Neutral liquidity warning profile"\n'
    '    if ah > -6:\n'
    '        return "Moderate liquidity-withdrawal warning"\n'
    '    return "High liquidity-withdrawal warning"\n'
    '\n'
    '\n'
    'def _central_bank_liquidity_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme liquidity support"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong liquidity support"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme liquidity tightening"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong liquidity tightening"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _numeric_or_zero_for_liquidity(value: Any) -> tuple[float, bool]:\n'
    '    if is_blank(value):\n'
    '        return 0.0, True\n'
    '    numeric = to_float_or_none(value)\n'
    '    return (numeric, True) if numeric is not None else (0.0, False)\n'
    '\n'
    '\n'
    'def _liquidity_growth(current: Any, lagged: Any, min_count_ok: bool) -> Any:\n'
    '    if not min_count_ok or is_blank(current) or is_blank(lagged) or lagged == 0:\n'
    '        return ""\n'
    '    return current / lagged - 1\n'
    '\n'
    '\n'
    'def _mtd_average_liquidity(dates: list[Any], net_values: list[Any]) -> list[Any]:\n'
    '    month_groups: dict[tuple[int, int], list[tuple[date, float]]] = {}\n'
    '    for dt, net in zip(dates, net_values):\n'
    '        numeric = to_float_or_none(net)\n'
    '        if dt is None or numeric is None:\n'
    '            continue\n'
    '        month_groups.setdefault((dt.year, dt.month), []).append((dt, numeric))\n'
    '    prepared: dict[tuple[int, int], tuple[list[date], list[float]]] = {}\n'
    '    for key, pairs in month_groups.items():\n'
    '        pairs.sort(key=lambda item: item[0])\n'
    '        group_dates: list[date] = []\n'
    '        prefix: list[float] = [0.0]\n'
    '        for dt, value in pairs:\n'
    '            group_dates.append(dt)\n'
    '            prefix.append(prefix[-1] + value)\n'
    '        prepared[key] = (group_dates, prefix)\n'
    '    out: list[Any] = []\n'
    '    for dt in dates:\n'
    '        if dt is None:\n'
    '            out.append("")\n'
    '            continue\n'
    '        group_dates, prefix = prepared.get((dt.year, dt.month), ([], [0.0]))\n'
    '        count = bisect.bisect_right(group_dates, dt)\n'
    '        out.append("" if count == 0 else prefix[count] / count)\n'
    '    return out\n'
    '\n'
    '\n'
    'def _banking_reserves_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A30-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _banking_reserves_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A30-only spreadsheet-order average over numeric cells."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _banking_reserves_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A30-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _banking_reserves_metric_ranks_sheets(\n'
    '    values: Sequence[Any],\n'
    '    min_count: int = 60,\n'
    '    gate_counts: Sequence[int] | None = None,\n'
    ') -> list[Any]:\n'
    '    """Rank an A30 metric against its history after its availability gate opens."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for idx, value in enumerate(values):\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        gate_count = len(history) if gate_counts is None else gate_counts[idx]\n'
    '        if numeric is None or gate_count < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _banking_reserves_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _banking_reserves_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _banking_reserves_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        ""\n'
    '        if is_blank(rank)\n'
    '        else _banking_reserves_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _banking_reserves_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _banking_reserves_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _banking_reserves_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A30 composites from their rounded visible operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _banking_reserves_rounded_expanding_zscores(\n'
    '    values: list[object], counts: list[int], min_count: int = 60\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else _banking_reserves_sheets_round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, counts, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def _banking_reserves_positive_or_blank(value: Any) -> bool:\n'
    '    return value == "" or value > 0\n'
    '\n'
    '\n'
    'def _banking_reserves_negative_or_blank(value: Any) -> bool:\n'
    '    return value == "" or value < 0\n'
    '\n'
    '\n'
    'def _banking_reserves_regime(e: Any, g: Any, h: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if (\n'
    '        e > 0\n'
    '        and _banking_reserves_positive_or_blank(g)\n'
    '        and _banking_reserves_positive_or_blank(h)\n'
    '    ):\n'
    '        return "Reserve expansion regime"\n'
    '    if not is_blank(g) and not is_blank(h) and e > 0 and g < 0 and h < 0:\n'
    '        return "Positive reserves but draining"\n'
    '    if not is_blank(g) and not is_blank(h) and e <= 0 and g > 0 and h > 0:\n'
    '        return "Low reserves but recovering"\n'
    '    if not is_blank(g) and not is_blank(h) and e <= 0 and g < 0 and h < 0:\n'
    '        return "Reserve contraction / scarcity regime"\n'
    '    if not is_blank(g) and not is_blank(h) and g > 0 and h < 0:\n'
    '        return "Early reserve recovery / mixed impulse"\n'
    '    if not is_blank(g) and not is_blank(h) and g < 0 and h > 0:\n'
    '        return "Early reserve deterioration / mixed impulse"\n'
    '    return "Neutral/mixed reserve regime"\n'
    '\n'
    '\n'
    'def _banking_reserves_asset_bias_label(ad: Any) -> str:\n'
    '    if is_blank(ad):\n'
    '        return ""\n'
    '    if ad >= 6:\n'
    '        return "Strongly supportive for risk assets / liquidity-sensitive assets"\n'
    '    if ad >= 2:\n'
    '        return "Mildly supportive for risk assets"\n'
    '    if ad <= -6:\n'
    '        return "Strong reserve-liquidity headwind for risk assets"\n'
    '    if ad <= -2:\n'
    '        return "Mild reserve-liquidity headwind for risk assets"\n'
    '    return "Neutral reserve-liquidity bias"\n'
    '\n'
    '\n'
    'def _banking_reserves_shock_label(f: Any) -> str:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if f >= 250000:\n'
    '        return "1M major reserve injection shock"\n'
    '    if f >= 50000:\n'
    '        return "1M moderate reserve injection"\n'
    '    if f <= -250000:\n'
    '        return "1M major reserve drain shock"\n'
    '    if f <= -50000:\n'
    '        return "1M moderate reserve drain"\n'
    '    return "No major reserve shock"\n'
    '\n'
    '\n'
    'def _banking_reserves_recovery_score(g: Any, h: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (g, h)):\n'
    '        return ""\n'
    '    if g > 0 and h > 0:\n'
    '        return 5\n'
    '    if g < 0 and h < 0:\n'
    '        return -5\n'
    '    if g > 0 and h < 0:\n'
    '        return 3\n'
    '    if g < 0 and h > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _banking_reserves_shock_score(f: Any) -> Any:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if f >= 250000:\n'
    '        return 6\n'
    '    if f >= 50000:\n'
    '        return 4\n'
    '    if f <= -250000:\n'
    '        return -6\n'
    '    if f <= -50000:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _banking_reserves_withdrawal_recovery_score(\n'
    '    am: Any, an: Any, ao: Any, g: Any, h: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (am, an, ao, g, h)):\n'
    '        return ""\n'
    '    if am > 1.5 and an < -0.5 and ao < -0.5 and g < 0 and h < 0:\n'
    '        return -6\n'
    '    if am > 1 and an < 0 and ao < 0 and g < 0:\n'
    '        return -4\n'
    '    if am < -1 and an > 0 and ao > 0 and g > 0 and h > 0:\n'
    '        return 6\n'
    '    if an > 1 and ao > 1 and g > 0 and h > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _banking_reserves_scarcity_stress_score(\n'
    '    am: Any, an: Any, ao: Any, g: Any, h: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (am, an, ao, g, h)):\n'
    '        return ""\n'
    '    if am < -1.5 and an < 0 and ao < 0 and g < 0 and h < 0:\n'
    '        return -6\n'
    '    if am < -1 and an < 0 and g < 0:\n'
    '        return -4\n'
    '    if am < -1 and an > 0 and ao > 0 and g > 0 and h > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _banking_reserves_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _banking_reserves_conditions_regime(ad: Any) -> str:\n'
    '    if is_blank(ad):\n'
    '        return ""\n'
    '    if ad >= 2:\n'
    '        return "Banking reserves improving"\n'
    '    if ad <= -2:\n'
    '        return "Banking reserves deteriorating"\n'
    '    return "Neutral reserve conditions"\n'
    '\n'
    '\n'
    'def _banking_reserves_momentum_label(ad: Any, ae: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ad, ae)):\n'
    '        return ""\n'
    '    if ad >= 2 and ae >= 2:\n'
    '        return "Banking reserves improving"\n'
    '    if ad >= 2 and ae <= -2:\n'
    '        return "Reserves abundant but deteriorating"\n'
    '    if ad >= 2 and -2 < ae < 2:\n'
    '        return "Reserve conditions supportive / stable"\n'
    '    if ad <= -2 and ae >= 2:\n'
    '        return "Reserves weak but improving"\n'
    '    if ad <= -2 and ae <= -2:\n'
    '        return "Banking reserves deteriorating"\n'
    '    if ad <= -2 and -2 < ae < 2:\n'
    '        return "Reserve conditions weak / stable"\n'
    '    if -2 < ad < 2 and ae >= 2:\n'
    '        return "Reserve impulse improving"\n'
    '    if -2 < ad < 2 and ae <= -2:\n'
    '        return "Reserve impulse deteriorating"\n'
    '    return "Banking reserves broadly neutral"\n'
    '\n'
    '\n'
    'def _banking_reserves_signal_interpretation(ae: Any, af: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ae, af)):\n'
    '        return ""\n'
    '    if ae >= 6 and af >= 0:\n'
    '        return "Strong positive reserve-liquidity impulse with supportive banking-liquidity profile"\n'
    '    if ae >= 6 and af < 0:\n'
    '        return "Strong positive reserve-liquidity impulse, but drain/scarcity profile remains mixed"\n'
    '    if ae >= 2 and af >= 0:\n'
    '        return "Moderately positive reserve-liquidity impulse signal"\n'
    '    if ae >= 2 and af < 0:\n'
    '        return "Reserve impulse improving, but drain/scarcity risk remains fragile"\n'
    '    if ae > -2 and af > -3 and af < 3:\n'
    '        return "Neutral/mixed reserve-liquidity signal"\n'
    '    if ae <= -6 and af <= 0:\n'
    '        return (\n'
    '            "Strong reserve-liquidity deterioration signal with downside confirmation"\n'
    '        )\n'
    '    if ae <= -6 and af > 0:\n'
    '        return "Strong reserve-liquidity deterioration signal, but reserve support remains partly intact"\n'
    '    if ae <= -2 and af <= 0:\n'
    '        return "Moderately negative reserve-liquidity impulse signal"\n'
    '    if ae <= -2 and af > 0:\n'
    '        return (\n'
    '            "Reserve impulse deteriorating, but banking-liquidity support remains mixed"\n'
    '        )\n'
    '    return "Mixed/transition reserve-liquidity signal"\n'
    '\n'
    '\n'
    'def _banking_reserves_warning_label(af: Any) -> str:\n'
    '    if is_blank(af):\n'
    '        return ""\n'
    '    if af >= 6:\n'
    '        return "High upside reserve-liquidity support warning"\n'
    '    if af >= 3:\n'
    '        return "Moderate upside reserve-liquidity support warning"\n'
    '    if af > -3:\n'
    '        return "Neutral reserve-liquidity warning profile"\n'
    '    if af > -6:\n'
    '        return "Moderate reserve drain/scarcity warning"\n'
    '    return "High reserve drain/scarcity warning"\n'
    '\n'
    '\n'
    'def _banking_reserves_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme reserve-liquidity support"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong reserve-liquidity support"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme reserve-liquidity tightening / scarcity stress"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong reserve-liquidity tightening / scarcity stress"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _broad_money_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A31-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _broad_money_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A31-only spreadsheet-order average without rounding the result."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _broad_money_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    """A31-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _broad_money_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    """Rank each A31 metric after its own numeric history reaches min_count."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _broad_money_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _broad_money_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _broad_money_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        "" if is_blank(rank) else _broad_money_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _broad_money_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _broad_money_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _broad_money_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A31 composites from their rounded visible score operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _broad_money_rounded_expanding_zscores(\n'
    '    values: list[object], counts: list[int], min_count: int = 60\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else _broad_money_sheets_round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, counts, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def _broad_money_positive_or_blank(value: Any) -> bool:\n'
    '    return value == "" or value > 0\n'
    '\n'
    '\n'
    'def _broad_money_growth(current: Any, lagged: Any, min_count_ok: bool) -> Any:\n'
    '    if not min_count_ok or is_blank(current) or is_blank(lagged) or lagged == 0:\n'
    '        return ""\n'
    '    return current / lagged - 1\n'
    '\n'
    '\n'
    'def _broad_money_annualized_growth(value: Any, power: int) -> Any:\n'
    '    if is_blank(value):\n'
    '        return ""\n'
    '    try:\n'
    '        return (1 + value) ** power - 1\n'
    '    except Exception:\n'
    '        return ""\n'
    '\n'
    '\n'
    'def _broad_money_regime(c: Any, f: Any, g: Any, h: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if (\n'
    '        c > 0\n'
    '        and _broad_money_positive_or_blank(f)\n'
    '        and _broad_money_positive_or_blank(g)\n'
    '    ):\n'
    '        return "Broad-money expansion regime"\n'
    '    if not any(is_blank(x) for x in (f, g)) and c > 0 and f < 0 and g < 0:\n'
    '        return "Broad-money deterioration from high base"\n'
    '    if not any(is_blank(x) for x in (f, g)) and c <= 0 and f > 0 and g > 0:\n'
    '        return "Early broad-money recovery"\n'
    '    if not any(is_blank(x) for x in (f, g)) and c <= 0 and f < 0 and g < 0:\n'
    '        return "Broad-money contraction regime"\n'
    '    if not any(is_blank(x) for x in (f, g)) and f > 0 and g < 0:\n'
    '        return "Short-term broad-money recovery / mixed impulse"\n'
    '    if not any(is_blank(x) for x in (f, g)) and f < 0 and g > 0:\n'
    '        return "Short-term broad-money deterioration / mixed impulse"\n'
    '    return "Neutral/mixed broad-money regime"\n'
    '\n'
    '\n'
    'def _broad_money_asset_bias_label(ab: Any) -> str:\n'
    '    if is_blank(ab):\n'
    '        return ""\n'
    '    if ab >= 6:\n'
    '        return "Strongly supportive for liquidity-sensitive risk assets"\n'
    '    if ab >= 2:\n'
    '        return "Mildly supportive for liquidity-sensitive risk assets"\n'
    '    if ab <= -6:\n'
    '        return "Strong broad-money liquidity headwind for risk assets"\n'
    '    if ab <= -2:\n'
    '        return "Mild broad-money liquidity headwind for risk assets"\n'
    '    return "Neutral broad-money liquidity bias"\n'
    '\n'
    '\n'
    'def _broad_money_growth_shock_label(i: Any) -> str:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 0.15:\n'
    '        return "Extreme 3M annualised money-growth surge"\n'
    '    if i >= 0.10:\n'
    '        return "Strong 3M annualised money-growth surge"\n'
    '    if i <= -0.05:\n'
    '        return "Sharp 3M annualised money contraction"\n'
    '    if i <= 0:\n'
    '        return "Negative 3M annualised money growth"\n'
    '    if i <= 0.03:\n'
    '        return "Very weak 3M annualised money growth"\n'
    '    return "No major money-growth shock"\n'
    '\n'
    '\n'
    'def _broad_money_recovery_score(f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (f, g)):\n'
    '        return ""\n'
    '    if f > 0 and g > 0:\n'
    '        return 5\n'
    '    if f < 0 and g < 0:\n'
    '        return -5\n'
    '    if f > 0 and g < 0:\n'
    '        return 3\n'
    '    if f < 0 and g > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _broad_money_shock_score(i: Any) -> Any:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 0.15:\n'
    '        return 6\n'
    '    if i >= 0.10:\n'
    '        return 4\n'
    '    if i <= -0.05:\n'
    '        return -6\n'
    '    if i <= 0:\n'
    '        return -4\n'
    '    if i <= 0.03:\n'
    '        return -2\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _broad_money_expansion_support_score(\n'
    '    ak: Any, al: Any, am: Any, i: Any, j: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (ak, al, am, i, j)):\n'
    '        return ""\n'
    '    if al > 1.5 and am > 1.5 and i > 0.10 and j > 0.08:\n'
    '        return 6\n'
    '    if al > 1 and am > 1 and i > 0.08:\n'
    '        return 4\n'
    '    if ak > 1.5 and al > 1 and i > 0.08:\n'
    '        return 3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _broad_money_contraction_risk_score(\n'
    '    ak: Any, al: Any, am: Any, i: Any, j: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (ak, al, am, i, j)):\n'
    '        return ""\n'
    '    if al < -1.5 and am < -1.5 and i < 0.02 and j < 0.03:\n'
    '        return -6\n'
    '    if al < -1 and am < -1 and i < 0.04:\n'
    '        return -4\n'
    '    if al > 0 and am > 0 and i > 0.05 and j > 0.05:\n'
    '        return 3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _broad_money_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _broad_money_conditions_regime(ab: Any) -> str:\n'
    '    if is_blank(ab):\n'
    '        return ""\n'
    '    if ab >= 2:\n'
    '        return "Broad money conditions improving"\n'
    '    if ab <= -2:\n'
    '        return "Broad money conditions deteriorating"\n'
    '    return "Neutral broad money conditions"\n'
    '\n'
    '\n'
    'def _broad_money_momentum_label(ab: Any, ac: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ab, ac)):\n'
    '        return ""\n'
    '    if ab >= 2 and ac >= 2:\n'
    '        return "Broad money conditions improving"\n'
    '    if ab >= 2 and ac <= -2:\n'
    '        return "Broad money supportive but deteriorating"\n'
    '    if ab >= 2 and -2 < ac < 2:\n'
    '        return "Broad money supportive / stable"\n'
    '    if ab <= -2 and ac >= 2:\n'
    '        return "Broad money weak but improving"\n'
    '    if ab <= -2 and ac <= -2:\n'
    '        return "Broad money conditions deteriorating"\n'
    '    if ab <= -2 and -2 < ac < 2:\n'
    '        return "Broad money weak / stable"\n'
    '    if -2 < ab < 2 and ac >= 2:\n'
    '        return "Broad money impulse improving"\n'
    '    if -2 < ab < 2 and ac <= -2:\n'
    '        return "Broad money impulse deteriorating"\n'
    '    return "Broad money broadly neutral"\n'
    '\n'
    '\n'
    'def _broad_money_signal_interpretation(ac: Any, ad: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ac, ad)):\n'
    '        return ""\n'
    '    if ac >= 6 and ad >= 0:\n'
    '        return "Strong positive broad-money impulse with supportive monetary-liquidity profile"\n'
    '    if ac >= 6 and ad < 0:\n'
    '        return "Strong positive broad-money impulse, but contraction-risk profile remains mixed"\n'
    '    if ac >= 2 and ad >= 0:\n'
    '        return "Moderately positive broad-money impulse signal"\n'
    '    if ac >= 2 and ad < 0:\n'
    '        return (\n'
    '            "Broad-money impulse improving, but monetary-risk profile remains fragile"\n'
    '        )\n'
    '    if ac > -2 and ad > -3 and ad < 3:\n'
    '        return "Neutral/mixed broad-money signal"\n'
    '    if ac <= -6 and ad <= 0:\n'
    '        return "Strong broad-money deterioration signal with downside confirmation"\n'
    '    if ac <= -6 and ad > 0:\n'
    '        return "Strong broad-money deterioration signal, but monetary support remains partly intact"\n'
    '    if ac <= -2 and ad <= 0:\n'
    '        return "Moderately negative broad-money impulse signal"\n'
    '    if ac <= -2 and ad > 0:\n'
    '        return "Broad-money impulse deteriorating, but monetary-liquidity support remains mixed"\n'
    '    return "Mixed/transition broad-money signal"\n'
    '\n'
    '\n'
    'def _broad_money_warning_label(ad: Any) -> str:\n'
    '    if is_blank(ad):\n'
    '        return ""\n'
    '    if ad >= 6:\n'
    '        return "High broad-money overheating / liquidity-support warning"\n'
    '    if ad >= 3:\n'
    '        return "Moderate broad-money overheating / liquidity-support warning"\n'
    '    if ad > -3:\n'
    '        return "Neutral broad-money warning profile"\n'
    '    if ad > -6:\n'
    '        return "Moderate broad-money contraction warning"\n'
    '    return "High broad-money contraction warning"\n'
    '\n'
    '\n'
    'def _broad_money_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme broad-money expansion"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong broad-money expansion"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme broad-money contraction"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong broad-money contraction"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _gov_revenue_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A34-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _gov_revenue_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    """A34-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _gov_revenue_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 40\n'
    ') -> list[Any]:\n'
    '    """Rank each A34 metric after its own numeric history reaches min_count."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _gov_revenue_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _gov_revenue_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _gov_revenue_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        "" if is_blank(rank) else _gov_revenue_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _gov_revenue_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _gov_revenue_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _gov_revenue_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A34 composites from their rounded visible score operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _gov_revenue_rounded_expanding_zscores(\n'
    '    values: list[object], counts: list[int], min_count: int = 40\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else _gov_revenue_sheets_round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, counts, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def _gov_revenue_sum_sheets(values: Sequence[Any]) -> Any:\n'
    '    clean = [\n'
    '        number\n'
    '        for number in (to_float_or_none(value) for value in values)\n'
    '        if number is not None\n'
    '    ]\n'
    '    if not clean:\n'
    '        return ""\n'
    '    total = clean[0]\n'
    '    for number in clean[1:]:\n'
    '        total += number\n'
    '    return total\n'
    '\n'
    '\n'
    'def _gov_revenue_growth(current: Any, lagged: Any, min_count_ok: bool) -> Any:\n'
    '    if not min_count_ok or is_blank(current) or is_blank(lagged) or lagged == 0:\n'
    '        return ""\n'
    '    return current / lagged - 1\n'
    '\n'
    '\n'
    'def _gov_revenue_annualized_growth(value: Any, power: int) -> Any:\n'
    '    if is_blank(value):\n'
    '        return ""\n'
    '    try:\n'
    '        return (1 + value) ** power - 1\n'
    '    except Exception:\n'
    '        return ""\n'
    '\n'
    '\n'
    'def _gov_revenue_positive_or_blank(value: Any) -> bool:\n'
    '    return value == "" or value > 0\n'
    '\n'
    '\n'
    'def _gov_revenue_growth_regime(c: Any, f: Any, g: Any, h: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if (\n'
    '        c > 0\n'
    '        and _gov_revenue_positive_or_blank(f)\n'
    '        and _gov_revenue_positive_or_blank(g)\n'
    '        and _gov_revenue_positive_or_blank(h)\n'
    '    ):\n'
    '        return "Government revenue expansion regime"\n'
    '    if not any(is_blank(x) for x in (f, g, h)) and f < 0 and g < 0 and h < 0:\n'
    '        return "Government revenue contraction regime"\n'
    '    if not any(is_blank(x) for x in (f, g, h)) and f > 0 and g > 0 and h < 0:\n'
    '        return "Early revenue recovery"\n'
    '    if (\n'
    '        not any(is_blank(x) for x in (f, g))\n'
    '        and f < 0\n'
    '        and g < 0\n'
    '        and _gov_revenue_positive_or_blank(h)\n'
    '    ):\n'
    '        return "Revenue deterioration from high base"\n'
    '    if not any(is_blank(x) for x in (f, g)) and f > 0 and g < 0:\n'
    '        return "Short-term revenue recovery / mixed impulse"\n'
    '    if not any(is_blank(x) for x in (f, g)) and f < 0 and g > 0:\n'
    '        return "Short-term revenue deterioration / mixed impulse"\n'
    '    return "Neutral/mixed revenue-growth regime"\n'
    '\n'
    '\n'
    'def _gov_revenue_asset_bias_label(ac: Any) -> str:\n'
    '    if is_blank(ac):\n'
    '        return ""\n'
    '    if ac >= 6:\n'
    '        return "Strong fiscal-revenue support / improving tax-base backdrop"\n'
    '    if ac >= 2:\n'
    '        return "Mild fiscal-revenue support / improving receipts"\n'
    '    if ac <= -6:\n'
    '        return "Strong fiscal-revenue deterioration / tax-receipt stress"\n'
    '    if ac <= -2:\n'
    '        return "Mild fiscal-revenue deterioration"\n'
    '    return "Neutral fiscal-revenue bias"\n'
    '\n'
    '\n'
    'def _gov_revenue_shock_label(i: Any) -> str:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 0.15:\n'
    '        return "Large revenue-growth surge"\n'
    '    if i >= 0.08:\n'
    '        return "Moderate revenue-growth acceleration"\n'
    '    if i <= -0.15:\n'
    '        return "Large revenue-growth contraction"\n'
    '    if i <= -0.08:\n'
    '        return "Moderate revenue-growth slowdown"\n'
    '    return "No major revenue-growth shock"\n'
    '\n'
    '\n'
    'def _gov_revenue_recovery_score(f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (f, g)):\n'
    '        return ""\n'
    '    if f > 0 and g > 0:\n'
    '        return 5\n'
    '    if f < 0 and g < 0:\n'
    '        return -5\n'
    '    if f > 0 and g < 0:\n'
    '        return 3\n'
    '    if f < 0 and g > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _gov_revenue_shock_score(i: Any) -> Any:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 0.15:\n'
    '        return 6\n'
    '    if i >= 0.08:\n'
    '        return 4\n'
    '    if i <= -0.15:\n'
    '        return -6\n'
    '    if i <= -0.08:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _gov_revenue_deterioration_risk(al: Any, am: Any, an: Any, ao: Any) -> Any:\n'
    '    """A34 Revenue Weakness Risk Score, including positive windfall extremes."""\n'
    '    zs = (al, am, an, ao)\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6.0\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4.0\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6.0\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4.0\n'
    '    return 0.0\n'
    '\n'
    '\n'
    'def _gov_revenue_windfall_score(al: Any, am: Any, ao: Any, i: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (al, am, ao, i)):\n'
    '        return ""\n'
    '    if al > 1.5 and am > 1.5 and i > 0.15:\n'
    '        return 5\n'
    '    if al > 1 and am > 1 and i > 0.10:\n'
    '        return 3\n'
    '    if al < -1 and am < -1 and i < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _gov_revenue_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6.0\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4.0\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6.0\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4.0\n'
    '    return 0.0\n'
    '\n'
    '\n'
    'def _gov_revenue_warning_score(w: Any, x: Any, y: Any, z: Any, aa: Any, ab: Any) -> Any:\n'
    '    values = (w, x, y, z, aa, ab)\n'
    '    if any(is_blank(value) for value in values):\n'
    '        return ""\n'
    '    return _gov_revenue_sheets_round(\n'
    '        clamp(\n'
    '            _gov_revenue_weighted_sum(\n'
    '                (0.20, w),\n'
    '                (0.20, x),\n'
    '                (0.20, y),\n'
    '                (0.15, z),\n'
    '                (0.10, aa),\n'
    '                (0.15, ab),\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _gov_revenue_reliability(raw_source: Any, sample_count: int) -> str:\n'
    '    if is_blank(raw_source):\n'
    '        return ""\n'
    '    if sample_count < 40:\n'
    '        return "Insufficient history"\n'
    '    if sample_count < 80:\n'
    '        return "Medium reliability"\n'
    '    return "High reliability"\n'
    '\n'
    '\n'
    'def _gov_revenue_conditions_regime(ac: Any) -> str:\n'
    '    if is_blank(ac):\n'
    '        return ""\n'
    '    if ac >= 2:\n'
    '        return "Government revenue growth improving"\n'
    '    if ac <= -2:\n'
    '        return "Government revenue growth deteriorating"\n'
    '    return "Neutral government revenue growth"\n'
    '\n'
    '\n'
    'def _gov_revenue_momentum_label(ac: Any, ad: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ac, ad)):\n'
    '        return ""\n'
    '    if ac >= 2 and ad >= 2:\n'
    '        return "Government revenue growth improving"\n'
    '    if ac >= 2 and ad <= -2:\n'
    '        return "Revenue growth strong but fading"\n'
    '    if ac >= 2 and -2 < ad < 2:\n'
    '        return "Revenue growth supportive / stable"\n'
    '    if ac <= -2 and ad >= 2:\n'
    '        return "Revenue weakness easing"\n'
    '    if ac <= -2 and ad <= -2:\n'
    '        return "Government revenue growth deteriorating"\n'
    '    if ac <= -2 and -2 < ad < 2:\n'
    '        return "Revenue growth weak / stable"\n'
    '    if -2 < ac < 2 and ad >= 2:\n'
    '        return "Revenue impulse improving"\n'
    '    if -2 < ac < 2 and ad <= -2:\n'
    '        return "Revenue impulse deteriorating"\n'
    '    return "Government revenue growth broadly neutral"\n'
    '\n'
    '\n'
    'def _gov_revenue_signal_interpretation(ad: Any, ae: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ad, ae)):\n'
    '        return ""\n'
    '    if ad >= 6 and ae >= 0:\n'
    '        return "Strong positive revenue-growth impulse with supportive fiscal-receipts profile"\n'
    '    if ad >= 6 and ae < 0:\n'
    '        return "Strong positive revenue-growth impulse, but revenue-risk profile remains mixed"\n'
    '    if ad >= 2 and ae >= 0:\n'
    '        return "Moderately positive revenue-growth impulse signal"\n'
    '    if ad >= 2 and ae < 0:\n'
    '        return "Revenue impulse improving, but revenue-risk profile remains fragile"\n'
    '    if ad > -2 and ae > -3 and ae < 3:\n'
    '        return "Neutral/mixed revenue-growth signal"\n'
    '    if ad <= -6 and ae <= 0:\n'
    '        return "Strong revenue deterioration signal with downside confirmation"\n'
    '    if ad <= -6 and ae > 0:\n'
    '        return "Strong revenue deterioration signal, but fiscal-receipts support remains partly intact"\n'
    '    if ad <= -2 and ae <= 0:\n'
    '        return "Moderately negative revenue-growth impulse signal"\n'
    '    if ad <= -2 and ae > 0:\n'
    '        return (\n'
    '            "Revenue impulse deteriorating, but fiscal-receipts support remains mixed"\n'
    '        )\n'
    '    return "Mixed/transition revenue-growth signal"\n'
    '\n'
    '\n'
    'def _gov_revenue_warning_label(ae: Any) -> str:\n'
    '    if is_blank(ae):\n'
    '        return ""\n'
    '    if ae >= 6:\n'
    '        return "High revenue-growth strength / windfall warning"\n'
    '    if ae >= 3:\n'
    '        return "Moderate revenue-growth strength warning"\n'
    '    if ae > -3:\n'
    '        return "Neutral revenue-growth warning profile"\n'
    '    if ae > -6:\n'
    '        return "Moderate revenue-weakness warning"\n'
    '    return "High revenue-weakness warning"\n'
    '\n'
    '\n'
    'def _gov_revenue_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme government-revenue growth"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong government-revenue growth"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme government-revenue contraction"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong government-revenue contraction"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_government_revenue_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    source_values = [\n'
    '        r.get("Raw Value", r.get("B", r.get("Value", ""))) for r in records\n'
    '    ]\n'
    '    trailing_totals: list[Any] = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 12:\n'
    '            trailing_totals.append("")\n'
    '        else:\n'
    '            window = [\n'
    '                v for v in raw_values[max(0, idx - 11) : idx + 1] if v is not None\n'
    '            ]\n'
    '            trailing_totals.append(_gov_revenue_sum_sheets(window))\n'
    '\n'
    '    observation_counts = _metric_counts(trailing_totals)\n'
    '    one_growth: list[Any] = []\n'
    '    three_growth: list[Any] = []\n'
    '    six_growth: list[Any] = []\n'
    '    twelve_growth: list[Any] = []\n'
    '    three_annual: list[Any] = []\n'
    '    six_annual: list[Any] = []\n'
    '    for idx, c in enumerate(trailing_totals):\n'
    '        d = observation_counts[idx]\n'
    '        c1 = trailing_totals[idx - 1] if idx >= 1 else ""\n'
    '        c3 = trailing_totals[idx - 3] if idx >= 3 else ""\n'
    '        c6 = trailing_totals[idx - 6] if idx >= 6 else ""\n'
    '        c12 = trailing_totals[idx - 12] if idx >= 12 else ""\n'
    '        e = _gov_revenue_growth(c, c1, d >= 2)\n'
    '        f = _gov_revenue_growth(c, c3, d >= 4)\n'
    '        g = _gov_revenue_growth(c, c6, d >= 7)\n'
    '        h = _gov_revenue_growth(c, c12, d >= 13)\n'
    '        one_growth.append(e)\n'
    '        three_growth.append(f)\n'
    '        six_growth.append(g)\n'
    '        twelve_growth.append(h)\n'
    '        three_annual.append(_gov_revenue_annualized_growth(f, 4))\n'
    '        six_annual.append(_gov_revenue_annualized_growth(g, 2))\n'
    '\n'
    '    f_counts = _metric_counts(three_growth)\n'
    '    g_counts = _metric_counts(six_growth)\n'
    '    h_counts = _metric_counts(twelve_growth)\n'
    '    i_counts = _metric_counts(three_annual)\n'
    '    f_ranks = _gov_revenue_metric_ranks_sheets(three_growth)\n'
    '    g_ranks = _gov_revenue_metric_ranks_sheets(six_growth)\n'
    '    h_ranks = _gov_revenue_metric_ranks_sheets(twelve_growth)\n'
    '    i_ranks = _gov_revenue_metric_ranks_sheets(three_annual)\n'
    '    f_z = _gov_revenue_rounded_expanding_zscores(three_growth, f_counts)\n'
    '    g_z = _gov_revenue_rounded_expanding_zscores(six_growth, g_counts)\n'
    '    h_z = _gov_revenue_rounded_expanding_zscores(twelve_growth, h_counts)\n'
    '    i_z = _gov_revenue_rounded_expanding_zscores(three_annual, i_counts)\n'
    '\n'
    '    momentum_scores: list[Any] = []\n'
    '    rows = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = trailing_totals[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_growth[idx]\n'
    '        f = three_growth[idx]\n'
    '        g = six_growth[idx]\n'
    '        h = twelve_growth[idx]\n'
    '        i = three_annual[idx]\n'
    '        j = six_annual[idx]\n'
    '        k = f_ranks[idx]\n'
    '        l = _gov_revenue_score_percentile(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _gov_revenue_score_percentile(m)\n'
    '        o = h_ranks[idx]\n'
    '        p_score = _gov_revenue_score_percentile(o)\n'
    '        q = i_ranks[idx]\n'
    '        r = _gov_revenue_score_percentile(q)\n'
    '        v = (\n'
    '            _gov_revenue_sheets_round(\n'
    '                clamp(\n'
    '                    _gov_revenue_weighted_sum(\n'
    '                        (0.30, l), (0.35, n), (0.20, p_score), (0.15, r)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(value) for value in (l, n, p_score, r))\n'
    '            else ""\n'
    '        )\n'
    '        momentum_scores.append(v)\n'
    '        prev_v = momentum_scores[idx - 1] if idx >= 1 else ""\n'
    '        w = (\n'
    '            _gov_revenue_sheets_round(\n'
    '                clamp(_gov_revenue_weighted_sum((2.0, v), (-2.0, prev_v)), -10, 10),\n'
    '                2,\n'
    '            )\n'
    '            if not is_blank(v) and not is_blank(prev_v)\n'
    '            else ""\n'
    '        )\n'
    '        x = _gov_revenue_recovery_score(f, g)\n'
    '        y = _gov_revenue_shock_score(i)\n'
    '        al = f_z[idx]\n'
    '        am = g_z[idx]\n'
    '        an = h_z[idx]\n'
    '        ao = i_z[idx]\n'
    '        z = _gov_revenue_deterioration_risk(al, am, an, ao)\n'
    '        aa = _gov_revenue_windfall_score(al, am, ao, i)\n'
    '        ab = _gov_revenue_outlier_score([al, am, an, ao])\n'
    '        ac = v\n'
    '        ad = (\n'
    '            _gov_revenue_sheets_round(\n'
    '                clamp(\n'
    '                    _gov_revenue_weighted_sum((0.50, v), (0.30, w), (0.20, x)), -10, 10\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(value) for value in (v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        ae = _gov_revenue_warning_score(w, x, y, z, aa, ab)\n'
    '        reliability = _gov_revenue_reliability(raw_source, d)\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 40\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(value) for value in (ac, ad, ae))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "12M rolling Government Revenue": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Revenue Growth": _gov_revenue_percent_display(e),\n'
    '                "3M Revenue Growth": _gov_revenue_percent_display(f),\n'
    '                "6M Revenue Growth": _gov_revenue_percent_display(g),\n'
    '                "12M YoY Revenue Growth": _gov_revenue_percent_display(h),\n'
    '                "3M Annualised Revenue Growth": _gov_revenue_percent_display(i),\n'
    '                "6M Annualised Revneue Growth": _gov_revenue_percent_display(j),\n'
    '                "3M Growth Percentile": _gov_revenue_percent_display(k),\n'
    '                "3M Growth Score": l,\n'
    '                "6M Growth Percentile": _gov_revenue_percent_display(m),\n'
    '                "6M Growth Score": n,\n'
    '                "12M YoY Growth Percentile": _gov_revenue_percent_display(o),\n'
    '                "12M YoY Growth Score": p_score,\n'
    '                "3M Annualised Growth Percentile": _gov_revenue_percent_display(q),\n'
    '                "3M Annualised Growth Score": r,\n'
    '                "Revenue Growth Regime": _gov_revenue_growth_regime(c, f, g, h),\n'
    '                "Fiscal Revenue Bias": _gov_revenue_asset_bias_label(ac),\n'
    '                "Government Revenue Shock Flag": _gov_revenue_shock_label(i),\n'
    '                "Government Revenue Impulse Score": v,\n'
    '                "Government Revenue Acceleration Score": w,\n'
    '                "Government Revenue Cycle Turning Point Score": x,\n'
    '                "Government Revenue Shock Score": y,\n'
    '                "Revenue Weakness Risk Score": z,\n'
    '                "Revenue Overheating Score": aa,\n'
    '                "Government Revenue Extreme Score": ab,\n'
    '                "Coincident Score": ac,\n'
    '                "Leading Score": ad,\n'
    '                "Warning Score": ae,\n'
    '                "State": _gov_revenue_conditions_regime(ac),\n'
    '                "Regime": _gov_revenue_momentum_label(ac, ad),\n'
    '                "Signal Comment": _gov_revenue_signal_interpretation(ad, ae),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _gov_revenue_warning_label(ae),\n'
    '                "3M Growth Z-Score": al,\n'
    '                "6M Growth Z-Score": am,\n'
    '                "12M Growth Z-Score": an,\n'
    '                "3M Annualised Growth Z-Score": ao,\n'
    '                "Extreme Flag": _gov_revenue_outlier_label([al, am, an, ao]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, GOVERNMENT_REVENUE_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _score_percent_rank_inverted(rank: Any) -> Any:\n'
    '    return "" if is_blank(rank) else round(10 - rank * 20, 2)\n'
    '\n'
    '\n'
    'def _public_debt_regime(c: Any, g: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if not is_blank(g) and c < 40 and g <= 0:\n'
    '        return "Low debt and improving"\n'
    '    if not is_blank(g) and c < 40 and g > 0:\n'
    '        return "Low debt but rising"\n'
    '    if not is_blank(g) and 40 <= c < 70 and g <= 0:\n'
    '        return "Moderate debt and improving"\n'
    '    if not is_blank(g) and 40 <= c < 70 and g > 0:\n'
    '        return "Moderate debt and rising"\n'
    '    if not is_blank(g) and 70 <= c < 100 and g <= 0:\n'
    '        return "High debt but improving"\n'
    '    if not is_blank(g) and 70 <= c < 100 and g > 0:\n'
    '        return "High debt and rising"\n'
    '    if not is_blank(g) and c >= 100 and g <= 0:\n'
    '        return "Very high debt but improving"\n'
    '    if not is_blank(g) and c >= 100 and g > 0:\n'
    '        return "Very high debt and rising"\n'
    '    return "Mixed public-debt regime"\n'
    '\n'
    '\n'
    'def _public_debt_bias_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 6:\n'
    '        return "Strong fiscal-sustainability support"\n'
    '    if aa >= 2:\n'
    '        return "Mild fiscal-sustainability support"\n'
    '    if aa <= -6:\n'
    '        return "Strong fiscal-sustainability headwind"\n'
    '    if aa <= -2:\n'
    '        return "Mild fiscal-sustainability headwind"\n'
    '    return "Neutral fiscal-sustainability bias"\n'
    '\n'
    '\n'
    'def _public_debt_shock_label(h: Any) -> str:\n'
    '    if is_blank(h):\n'
    '        return ""\n'
    '    if h >= 8:\n'
    '        return "Large debt-ratio deterioration shock"\n'
    '    if h >= 4:\n'
    '        return "Moderate debt-ratio deterioration"\n'
    '    if h <= -8:\n'
    '        return "Large debt-ratio improvement shock"\n'
    '    if h <= -4:\n'
    '        return "Moderate debt-ratio improvement"\n'
    '    return "No major debt-ratio shock"\n'
    '\n'
    '\n'
    'def _public_debt_vulnerability_score(c: Any, aj: Any, ak: Any, al: Any, am: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, aj, ak, al, am)):\n'
    '        return ""\n'
    '    if c >= 100 and ak > 0 and al > 0:\n'
    '        return -7\n'
    '    if c >= 70 and ak > 0 and al > 0:\n'
    '        return -5\n'
    '    if aj > 1.5 and ak > 0 and al > 0:\n'
    '        return -4\n'
    '    if c < 40 and ak < 0 and al < 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _public_debt_recovery_score(e: Any, f: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (e, f)):\n'
    '        return ""\n'
    '    if e < 0 and f < 0:\n'
    '        return 5\n'
    '    if e > 0 and f > 0:\n'
    '        return -5\n'
    '    if e < 0 and f > 0:\n'
    '        return 3\n'
    '    if e > 0 and f < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _public_debt_shock_score(h: Any) -> Any:\n'
    '    if is_blank(h):\n'
    '        return ""\n'
    '    if h >= 8:\n'
    '        return -6\n'
    '    if h >= 4:\n'
    '        return -4\n'
    '    if h <= -8:\n'
    '        return 6\n'
    '    if h <= -4:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _public_debt_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return -6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return -4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return 6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _public_debt_conditions_regime(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 2:\n'
    '        return "Public debt conditions improving"\n'
    '    if aa <= -2:\n'
    '        return "Public debt pressure building"\n'
    '    return "Neutral public debt conditions"\n'
    '\n'
    '\n'
    'def _public_debt_momentum_label(aa: Any, ab: Any) -> str:\n'
    '    if any(is_blank(x) for x in (aa, ab)):\n'
    '        return ""\n'
    '    if aa >= 2 and ab >= 2:\n'
    '        return "Public debt conditions improving"\n'
    '    if aa >= 2 and ab <= -2:\n'
    '        return "Debt conditions good but deteriorating"\n'
    '    if aa >= 2 and -2 < ab < 2:\n'
    '        return "Debt conditions supportive / stable"\n'
    '    if aa <= -2 and ab >= 2:\n'
    '        return "Debt pressure easing"\n'
    '    if aa <= -2 and ab <= -2:\n'
    '        return "Public debt pressure building"\n'
    '    if aa <= -2 and -2 < ab < 2:\n'
    '        return "Debt pressure elevated / stable"\n'
    '    if -2 < aa < 2 and ab >= 2:\n'
    '        return "Debt dynamics improving"\n'
    '    if -2 < aa < 2 and ab <= -2:\n'
    '        return "Debt dynamics deteriorating"\n'
    '    return "Public debt broadly neutral"\n'
    '\n'
    '\n'
    'def _public_debt_signal_interpretation(ab: Any, ac: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ab, ac)):\n'
    '        return ""\n'
    '    if ab >= 6 and ac >= 0:\n'
    '        return "Strong public-debt sustainability improvement with supportive fiscal-space profile"\n'
    '    if ab >= 6 and ac < 0:\n'
    '        return "Strong debt-dynamics improvement, but vulnerability risk remains mixed"\n'
    '    if ab >= 2 and ac >= 0:\n'
    '        return "Moderately positive public-debt sustainability signal"\n'
    '    if ab >= 2 and ac < 0:\n'
    '        return "Debt dynamics improving, but high-debt vulnerability remains"\n'
    '    if ab > -2 and ac > -3 and ac < 3:\n'
    '        return "Neutral/mixed public-debt signal"\n'
    '    if ab <= -6 and ac <= 0:\n'
    '        return "Severe public-debt pressure building with downside confirmation"\n'
    '    if ab <= -6 and ac > 0:\n'
    '        return "Severe debt-pressure signal, but fiscal-space support remains partly intact"\n'
    '    if ab <= -2 and ac <= 0:\n'
    '        return "Moderate public-debt pressure building"\n'
    '    if ab <= -2 and ac > 0:\n'
    '        return "Debt pressure building, but vulnerability profile remains mixed"\n'
    '    return "Mixed/transition public-debt signal"\n'
    '\n'
    '\n'
    'def _public_debt_warning_label(ac: Any, u: Any, w: Any, y: Any, z: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ac, u, w, y, z)):\n'
    '        return ""\n'
    '    if z <= -6:\n'
    '        return "High public-debt vulnerability warning"\n'
    '    if z <= -4:\n'
    '        return "Moderate public-debt vulnerability warning"\n'
    '    if y <= -6:\n'
    '        return "High debt-ratio shock warning"\n'
    '    if y <= -4:\n'
    '        return "Moderate debt-ratio shock warning"\n'
    '    if w <= -6:\n'
    '        return "High debt vulnerability warning"\n'
    '    if w <= -4:\n'
    '        return "Moderate high-debt vulnerability warning"\n'
    '    if u <= -8:\n'
    '        return "High public-debt pressure warning"\n'
    '    if u <= -6:\n'
    '        return "Moderate public-debt pressure warning"\n'
    '    if ac >= 6:\n'
    '        return "High debt-sustainability improvement signal"\n'
    '    if ac >= 3:\n'
    '        return "Moderate debt-sustainability improvement signal"\n'
    '    if ac > -3:\n'
    '        return "Neutral public-debt warning profile"\n'
    '    if ac > -6:\n'
    '        return "Moderate public-debt vulnerability warning"\n'
    '    return "High public-debt vulnerability warning"\n'
    '\n'
    '\n'
    'def _public_debt_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme public-debt stress"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong public-debt stress"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme debt-sustainability improvement"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong debt-sustainability improvement"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_public_debt_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    source_values = [\n'
    '        r.get("Raw Value", r.get("B", r.get("Value", ""))) for r in records\n'
    '    ]\n'
    '    ma_values = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 2:\n'
    '            ma_values.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 1) : idx + 1] if v is not None]\n'
    '            ma_values.append(sum(window) / len(window) if window else "")\n'
    '    observation_counts = _metric_counts(ma_values)\n'
    '    one_changes = []\n'
    '    two_changes = []\n'
    '    four_changes = []\n'
    '    one_annual = []\n'
    '    two_annual = []\n'
    '    for idx, c in enumerate(ma_values):\n'
    '        d = observation_counts[idx]\n'
    '        c1 = ma_values[idx - 1] if idx >= 1 else ""\n'
    '        c2 = ma_values[idx - 2] if idx >= 2 else ""\n'
    '        c4 = ma_values[idx - 4] if idx >= 4 else ""\n'
    '        e = (c - c1) if d >= 2 and not is_blank(c) and not is_blank(c1) else ""\n'
    '        f = (c - c2) if d >= 3 and not is_blank(c) and not is_blank(c2) else ""\n'
    '        g = (c - c4) if d >= 5 and not is_blank(c) and not is_blank(c4) else ""\n'
    '        one_changes.append(e)\n'
    '        two_changes.append(f)\n'
    '        four_changes.append(g)\n'
    '        one_annual.append(e * 4 if not is_blank(e) else "")\n'
    '        two_annual.append(f * 2 if not is_blank(f) else "")\n'
    '    c_counts = _metric_counts(ma_values)\n'
    '    e_counts = _metric_counts(one_changes)\n'
    '    f_counts = _metric_counts(two_changes)\n'
    '    g_counts = _metric_counts(four_changes)\n'
    '    c_ranks = _expectations_metric_ranks(ma_values, 40)\n'
    '    e_ranks = _expectations_metric_ranks(one_changes, 40)\n'
    '    f_ranks = _expectations_metric_ranks(two_changes, 40)\n'
    '    g_ranks = _expectations_metric_ranks(four_changes, 40)\n'
    '    c_z = _rounded_expanding_zscores(ma_values, c_counts, 40)\n'
    '    e_z = _rounded_expanding_zscores(one_changes, e_counts, 40)\n'
    '    f_z = _rounded_expanding_zscores(two_changes, f_counts, 40)\n'
    '    g_z = _rounded_expanding_zscores(four_changes, g_counts, 40)\n'
    '    momentum_scores = []\n'
    '    rows = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = ma_values[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_changes[idx]\n'
    '        f = two_changes[idx]\n'
    '        g = four_changes[idx]\n'
    '        h = one_annual[idx]\n'
    '        i = two_annual[idx]\n'
    '        j = c_ranks[idx]\n'
    '        k = _score_percent_rank_inverted(j)\n'
    '        l = e_ranks[idx]\n'
    '        m = _score_percent_rank_inverted(l)\n'
    '        n = f_ranks[idx]\n'
    '        o = _score_percent_rank_inverted(n)\n'
    '        p = g_ranks[idx]\n'
    '        q = _score_percent_rank_inverted(p)\n'
    '        u = (\n'
    '            round(clamp(0.35 * k + 0.25 * m + 0.25 * o + 0.15 * q, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (k, m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        momentum_scores.append(u)\n'
    '        prev_u = momentum_scores[idx - 1] if idx >= 1 else ""\n'
    '        v = (\n'
    '            round(clamp((u - prev_u) * 2, -10, 10), 2)\n'
    '            if not is_blank(u) and not is_blank(prev_u)\n'
    '            else ""\n'
    '        )\n'
    '        aj = c_z[idx]\n'
    '        ak = e_z[idx]\n'
    '        al = f_z[idx]\n'
    '        am = g_z[idx]\n'
    '        w = _public_debt_vulnerability_score(c, aj, ak, al, am)\n'
    '        x = _public_debt_recovery_score(e, f)\n'
    '        y = _public_debt_shock_score(h)\n'
    '        z = _public_debt_outlier_score([aj, ak, al, am])\n'
    '        aa = (\n'
    '            round(clamp(0.40 * k + 0.25 * m + 0.20 * o + 0.15 * q, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (k, m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        ab = (\n'
    '            round(clamp(0.40 * u + 0.25 * v + 0.25 * w + 0.10 * x, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        ac = (\n'
    '            round(\n'
    '                clamp(0.20 * v + 0.35 * w + 0.15 * x + 0.20 * y + 0.10 * z, -10, 10), 2\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (v, w, x, y, z))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 40\n'
    '                else ("Medium reliability" if d < 80 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 40\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (aa, ab, ac))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "2-Period Moving Average Public Debt": c,\n'
    '                "Public Debt Observation Count": d,\n'
    '                "1-Period Public Debt Change": e,\n'
    '                "2-Period Public Debt Change": f,\n'
    '                "4-Period Public Debt Change": g,\n'
    '                "1-Period Annualised Public Debt Change": h,\n'
    '                "2-Period Annualised Public Debt Change": i,\n'
    '                "Public Debt Level PercentRank": j,\n'
    '                "Public Debt Level Score": k,\n'
    '                "1-Period Public Debt Change PercentRank": l,\n'
    '                "1-Period Public Debt Change Score": m,\n'
    '                "2-Period Public Debt Change PercentRank": n,\n'
    '                "2-Period Public Debt Change Score": o,\n'
    '                "4-Period Public Debt Change PercentRank": p,\n'
    '                "4-Period Public Debt Change Score": q,\n'
    '                "Public Debt Regime": _public_debt_regime(c, g),\n'
    '                "Public Debt Fiscal-Sustainability Bias Label": _public_debt_bias_label(\n'
    '                    aa\n'
    '                ),\n'
    '                "Public Debt Shock Label": _public_debt_shock_label(h),\n'
    '                "Public Debt Momentum Composite Score": u,\n'
    '                "Public Debt Momentum Turn Score": v,\n'
    '                "Public Debt Level / Vulnerability Score": w,\n'
    '                "Public Debt Recovery / Deterioration Score": x,\n'
    '                "Public Debt Shock Score": y,\n'
    '                "Public Debt Outlier Score": z,\n'
    '                "Public Debt Leading Score": aa,\n'
    '                "Public Debt Warning Composite Score": ab,\n'
    '                "Public Debt Final Warning Score": ac,\n'
    '                "Public Debt Conditions Regime": _public_debt_conditions_regime(aa),\n'
    '                "Public Debt Momentum Label": _public_debt_momentum_label(aa, ab),\n'
    '                "Public Debt Signal Interpretation": _public_debt_signal_interpretation(\n'
    '                    ab, ac\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Public Debt Warning Risk Label": _public_debt_warning_label(\n'
    '                    ac, u, w, y, z\n'
    '                ),\n'
    '                "Public Debt Level Z-Score": aj,\n'
    '                "1-Period Public Debt Change Z-Score": ak,\n'
    '                "2-Period Public Debt Change Z-Score": al,\n'
    '                "4-Period Public Debt Change Z-Score": am,\n'
    '                "Public Debt Outlier Label": _public_debt_outlier_label(\n'
    '                    [aj, ak, al, am]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, PUBLIC_DEBT_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _interest_coverage_regime(e: Any, h: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if not is_blank(h) and e >= 8 and h > 0:\n'
    '        return "Very strong coverage and improving"\n'
    '    if not is_blank(h) and e >= 8 and h <= 0:\n'
    '        return "Very strong coverage but weakening"\n'
    '    if not is_blank(h) and 4 <= e < 8 and h > 0:\n'
    '        return "Healthy coverage and improving"\n'
    '    if not is_blank(h) and 4 <= e < 8 and h <= 0:\n'
    '        return "Healthy coverage but weakening"\n'
    '    if not is_blank(h) and 2 <= e < 4 and h > 0:\n'
    '        return "Thin coverage but improving"\n'
    '    if not is_blank(h) and 2 <= e < 4 and h <= 0:\n'
    '        return "Thin coverage and weakening"\n'
    '    if not is_blank(h) and e < 2 and h > 0:\n'
    '        return "Severe interest burden but improving"\n'
    '    if not is_blank(h) and e < 2 and h <= 0:\n'
    '        return "Severe interest burden and worsening"\n'
    '    return "Mixed interest-coverage regime"\n'
    '\n'
    '\n'
    'def _interest_coverage_bias_label(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 6:\n'
    '        return "Strong fiscal-stress relief / strong debt-servicing capacity"\n'
    '    if aa >= 2:\n'
    '        return "Mild fiscal-stress relief"\n'
    '    if aa <= -6:\n'
    '        return "Strong fiscal-stress pressure from interest burden"\n'
    '    if aa <= -2:\n'
    '        return "Mild fiscal-stress pressure from interest burden"\n'
    '    return "Neutral fiscal-stress bias"\n'
    '\n'
    '\n'
    'def _interest_coverage_shock_label(g: Any) -> str:\n'
    '    if is_blank(g):\n'
    '        return ""\n'
    '    if g >= 2:\n'
    '        return "Large improvement in interest coverage"\n'
    '    if g >= 1:\n'
    '        return "Moderate improvement in interest coverage"\n'
    '    if g <= -2:\n'
    '        return "Large deterioration in interest coverage"\n'
    '    if g <= -1:\n'
    '        return "Moderate deterioration in interest coverage"\n'
    '    return "No major interest-coverage shock"\n'
    '\n'
    '\n'
    'def _interest_coverage_vulnerability_score(\n'
    '    e: Any, aj: Any, ak: Any, al: Any, am: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (e, aj, ak, al, am)):\n'
    '        return ""\n'
    '    if e < 2 and ak < 0 and al < 0:\n'
    '        return -7\n'
    '    if e < 4 and ak < 0 and al < 0:\n'
    '        return -5\n'
    '    if aj < -1.5 and ak < 0 and al < 0:\n'
    '        return -4\n'
    '    if e >= 8 and ak > 0 and al > 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interest_coverage_recovery_score(g: Any, h: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (g, h)):\n'
    '        return ""\n'
    '    if g > 0 and h > 0:\n'
    '        return 5\n'
    '    if g < 0 and h < 0:\n'
    '        return -5\n'
    '    if g > 0 and h < 0:\n'
    '        return 3\n'
    '    if g < 0 and h > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interest_coverage_shock_score(g: Any) -> Any:\n'
    '    if is_blank(g):\n'
    '        return ""\n'
    '    if g >= 2:\n'
    '        return 6\n'
    '    if g >= 1:\n'
    '        return 4\n'
    '    if g <= -2:\n'
    '        return -6\n'
    '    if g <= -1:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interest_coverage_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _interest_coverage_conditions_regime(aa: Any) -> str:\n'
    '    if is_blank(aa):\n'
    '        return ""\n'
    '    if aa >= 2:\n'
    '        return "Interest coverage improving"\n'
    '    if aa <= -2:\n'
    '        return "Interest coverage stress building"\n'
    '    return "Neutral interest coverage conditions"\n'
    '\n'
    '\n'
    'def _interest_coverage_momentum_label(aa: Any, ab: Any) -> str:\n'
    '    if any(is_blank(x) for x in (aa, ab)):\n'
    '        return ""\n'
    '    if aa >= 2 and ab >= 2:\n'
    '        return "Interest coverage improving"\n'
    '    if aa >= 2 and ab <= -2:\n'
    '        return "Coverage conditions good but deteriorating"\n'
    '    if aa >= 2 and -2 < ab < 2:\n'
    '        return "Coverage conditions supportive / stable"\n'
    '    if aa <= -2 and ab >= 2:\n'
    '        return "Interest-coverage stress easing"\n'
    '    if aa <= -2 and ab <= -2:\n'
    '        return "Interest-coverage stress building"\n'
    '    if aa <= -2 and -2 < ab < 2:\n'
    '        return "Coverage stress elevated / stable"\n'
    '    if -2 < aa < 2 and ab >= 2:\n'
    '        return "Coverage dynamics improving"\n'
    '    if -2 < aa < 2 and ab <= -2:\n'
    '        return "Coverage dynamics deteriorating"\n'
    '    return "Interest coverage broadly neutral"\n'
    '\n'
    '\n'
    'def _interest_coverage_signal_interpretation(ab: Any, ac: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ab, ac)):\n'
    '        return ""\n'
    '    if ab >= 6 and ac >= 0:\n'
    '        return "Strong debt-servicing capacity improvement with supportive fiscal-stress profile"\n'
    '    if ab >= 6 and ac < 0:\n'
    '        return (\n'
    '            "Strong coverage improvement, but low-coverage vulnerability remains mixed"\n'
    '        )\n'
    '    if ab >= 2 and ac >= 0:\n'
    '        return "Moderately positive interest-coverage sustainability signal"\n'
    '    if ab >= 2 and ac < 0:\n'
    '        return "Coverage dynamics improving, but vulnerability remains"\n'
    '    if ab > -2 and ac > -3 and ac < 3:\n'
    '        return "Neutral/mixed interest-coverage signal"\n'
    '    if ab <= -6 and ac <= 0:\n'
    '        return "Severe interest-coverage stress building with downside confirmation"\n'
    '    if ab <= -6 and ac > 0:\n'
    '        return "Severe coverage-stress signal, but fiscal-stress relief remains partly intact"\n'
    '    if ab <= -2 and ac <= 0:\n'
    '        return "Moderate interest-coverage stress building"\n'
    '    if ab <= -2 and ac > 0:\n'
    '        return "Coverage stress building, but vulnerability profile remains mixed"\n'
    '    return "Mixed/transition interest-coverage signal"\n'
    '\n'
    '\n'
    'def _interest_coverage_warning_label(ac: Any, u: Any, w: Any, y: Any, z: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ac, u, w, y, z)):\n'
    '        return ""\n'
    '    if z <= -6:\n'
    '        return "High interest-coverage stress warning"\n'
    '    if z <= -4:\n'
    '        return "Moderate interest-coverage stress warning"\n'
    '    if y <= -6:\n'
    '        return "High interest-coverage deterioration shock"\n'
    '    if y <= -4:\n'
    '        return "Moderate interest-coverage deterioration shock"\n'
    '    if w <= -6:\n'
    '        return "High low-coverage vulnerability warning"\n'
    '    if w <= -4:\n'
    '        return "Moderate low-coverage vulnerability warning"\n'
    '    if u <= -8:\n'
    '        return "High interest-coverage pressure warning"\n'
    '    if u <= -6:\n'
    '        return "Moderate interest-coverage pressure warning"\n'
    '    if ac >= 6:\n'
    '        return "High interest-coverage improvement signal"\n'
    '    if ac >= 3:\n'
    '        return "Moderate interest-coverage improvement signal"\n'
    '    if ac > -3:\n'
    '        return "Neutral interest-coverage warning profile"\n'
    '    if ac > -6:\n'
    '        return "Moderate interest-coverage vulnerability warning"\n'
    '    return "High interest-coverage vulnerability warning"\n'
    '\n'
    '\n'
    'def _interest_coverage_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme interest-coverage strength"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong interest-coverage strength"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme interest-coverage stress"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong interest-coverage stress"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_interest_coverage_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    b_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Revenue", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    c_values = [\n'
    '        to_float_or_none(\n'
    '            r.get(\n'
    '                "Source C", r.get("C", r.get("Interest Expense", r.get("Interest", "")))\n'
    '            )\n'
    '        )\n'
    '        for r in records\n'
    '    ]\n'
    '    source_b = [r.get("Raw Value", r.get("B", r.get("Revenue", ""))) for r in records]\n'
    '    ratios = []\n'
    '    for b, c in zip(b_values, c_values):\n'
    '        ratios.append("" if c is None or c == 0 or b is None else b / c)\n'
    '    smoothed = []\n'
    '    ratio_count = 0\n'
    '    for idx, d in enumerate(ratios):\n'
    '        if to_float_or_none(d) is not None:\n'
    '            ratio_count += 1\n'
    '        if ratio_count < 3:\n'
    '            smoothed.append("")\n'
    '        else:\n'
    '            window = [to_float_or_none(v) for v in ratios[max(0, idx - 2) : idx + 1]]\n'
    '            nums = [v for v in window if v is not None]\n'
    '            smoothed.append(sum(nums) / len(nums) if nums else "")\n'
    '    observation_counts = _metric_counts(smoothed)\n'
    '    one_changes = []\n'
    '    three_changes = []\n'
    '    five_changes = []\n'
    '    for idx, e in enumerate(smoothed):\n'
    '        f_count = observation_counts[idx]\n'
    '        e1 = smoothed[idx - 1] if idx >= 1 else ""\n'
    '        e3 = smoothed[idx - 3] if idx >= 3 else ""\n'
    '        e5 = smoothed[idx - 5] if idx >= 5 else ""\n'
    '        g = (e - e1) if f_count >= 2 and not is_blank(e) and not is_blank(e1) else ""\n'
    '        h = (e - e3) if f_count >= 4 and not is_blank(e) and not is_blank(e3) else ""\n'
    '        i = (e - e5) if f_count >= 6 and not is_blank(e) and not is_blank(e5) else ""\n'
    '        one_changes.append(g)\n'
    '        three_changes.append(h)\n'
    '        five_changes.append(i)\n'
    '    e_counts = _metric_counts(smoothed)\n'
    '    g_counts = _metric_counts(one_changes)\n'
    '    h_counts = _metric_counts(three_changes)\n'
    '    i_counts = _metric_counts(five_changes)\n'
    '    e_ranks = _expectations_metric_ranks(smoothed, 20)\n'
    '    g_ranks = _expectations_metric_ranks(one_changes, 20)\n'
    '    h_ranks = _expectations_metric_ranks(three_changes, 20)\n'
    '    i_ranks = _expectations_metric_ranks(five_changes, 20)\n'
    '    e_z = _rounded_expanding_zscores(smoothed, e_counts, 20)\n'
    '    g_z = _rounded_expanding_zscores(one_changes, g_counts, 20)\n'
    '    h_z = _rounded_expanding_zscores(three_changes, h_counts, 20)\n'
    '    i_z = _rounded_expanding_zscores(five_changes, i_counts, 20)\n'
    '    momentum_scores = []\n'
    '    rows = []\n'
    '    for idx, raw_b in enumerate(source_b):\n'
    '        d = ratios[idx]\n'
    '        e = smoothed[idx]\n'
    '        f_count = observation_counts[idx]\n'
    '        g = one_changes[idx]\n'
    '        h = three_changes[idx]\n'
    '        i = five_changes[idx]\n'
    '        j = e_ranks[idx]\n'
    '        k = _score_percent_rank(j)\n'
    '        l = g_ranks[idx]\n'
    '        m = _score_percent_rank(l)\n'
    '        n = h_ranks[idx]\n'
    '        o = _score_percent_rank(n)\n'
    '        p = i_ranks[idx]\n'
    '        q = _score_percent_rank(p)\n'
    '        u = (\n'
    '            round(clamp(0.35 * k + 0.25 * m + 0.25 * o + 0.15 * q, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (k, m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        momentum_scores.append(u)\n'
    '        prev_u = momentum_scores[idx - 1] if idx >= 1 else ""\n'
    '        v = (\n'
    '            round(clamp((u - prev_u) * 2, -10, 10), 2)\n'
    '            if not is_blank(u) and not is_blank(prev_u)\n'
    '            else ""\n'
    '        )\n'
    '        aj = e_z[idx]\n'
    '        ak = g_z[idx]\n'
    '        al = h_z[idx]\n'
    '        am = i_z[idx]\n'
    '        w = _interest_coverage_vulnerability_score(e, aj, ak, al, am)\n'
    '        x = _interest_coverage_recovery_score(g, h)\n'
    '        y = _interest_coverage_shock_score(g)\n'
    '        z = _interest_coverage_outlier_score([aj, ak, al, am])\n'
    '        aa = (\n'
    '            round(clamp(0.40 * k + 0.25 * m + 0.20 * o + 0.15 * q, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (k, m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        ab = (\n'
    '            round(clamp(0.40 * u + 0.25 * v + 0.25 * w + 0.10 * x, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        ac = (\n'
    '            round(\n'
    '                clamp(0.20 * v + 0.35 * w + 0.15 * x + 0.20 * y + 0.10 * z, -10, 10), 2\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (v, w, x, y, z))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_b)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if f_count < 20\n'
    '                else ("Medium reliability" if f_count < 40 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_b)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if f_count < 20\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (aa, ab, ac))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Raw Interest Coverage Ratio": d,\n'
    '                "3-Period Moving Average Interest Coverage": e,\n'
    '                "Interest Coverage Observation Count": f_count,\n'
    '                "1-Period Interest Coverage Change": g,\n'
    '                "3-Period Interest Coverage Change": h,\n'
    '                "5-Period Interest Coverage Change": i,\n'
    '                "Interest Coverage Level PercentRank": j,\n'
    '                "Interest Coverage Level Score": k,\n'
    '                "1-Period Interest Coverage Change PercentRank": l,\n'
    '                "1-Period Interest Coverage Change Score": m,\n'
    '                "3-Period Interest Coverage Change PercentRank": n,\n'
    '                "3-Period Interest Coverage Change Score": o,\n'
    '                "5-Period Interest Coverage Change PercentRank": p,\n'
    '                "5-Period Interest Coverage Change Score": q,\n'
    '                "Interest Coverage Regime": _interest_coverage_regime(e, h),\n'
    '                "Interest Coverage Fiscal-Stress Bias Label": _interest_coverage_bias_label(\n'
    '                    aa\n'
    '                ),\n'
    '                "Interest Coverage Shock Label": _interest_coverage_shock_label(g),\n'
    '                "Interest Coverage Momentum Composite Score": u,\n'
    '                "Interest Coverage Momentum Turn Score": v,\n'
    '                "Interest Coverage Level / Vulnerability Score": w,\n'
    '                "Interest Coverage Recovery / Deterioration Score": x,\n'
    '                "Interest Coverage Shock Score": y,\n'
    '                "Interest Coverage Outlier Score": z,\n'
    '                "Interest Coverage Leading Score": aa,\n'
    '                "Interest Coverage Warning Composite Score": ab,\n'
    '                "Interest Coverage Final Warning Score": ac,\n'
    '                "Interest Coverage Conditions Regime": _interest_coverage_conditions_regime(\n'
    '                    aa\n'
    '                ),\n'
    '                "Interest Coverage Momentum Label": _interest_coverage_momentum_label(\n'
    '                    aa, ab\n'
    '                ),\n'
    '                "Interest Coverage Signal Interpretation": _interest_coverage_signal_interpretation(\n'
    '                    ab, ac\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Interest Coverage Warning Risk Label": _interest_coverage_warning_label(\n'
    '                    ac, u, w, y, z\n'
    '                ),\n'
    '                "Interest Coverage Level Z-Score": aj,\n'
    '                "1-Period Interest Coverage Change Z-Score": ak,\n'
    '                "3-Period Interest Coverage Change Z-Score": al,\n'
    '                "5-Period Interest Coverage Change Z-Score": am,\n'
    '                "Interest Coverage Outlier Label": _interest_coverage_outlier_label(\n'
    '                    [aj, ak, al, am]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, INTEREST_COVERAGE_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _financial_conditions_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A37-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _financial_conditions_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A37-only spreadsheet-order average over numeric cells."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _financial_conditions_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A37-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _financial_conditions_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    """Rank an A37 metric once its own numeric history reaches min_count."""\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _financial_conditions_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _financial_conditions_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _financial_conditions_score_percentile(rank: Any) -> Any:\n'
    '    if is_blank(rank):\n'
    '        return ""\n'
    '    return _financial_conditions_sheets_round(10 - (float(rank) * 20), 2)\n'
    '\n'
    '\n'
    'def _financial_conditions_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _financial_conditions_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _financial_conditions_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A37 composites from their rounded visible score operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _financial_conditions_rounded_expanding_zscores(\n'
    '    values: list[object], counts: list[int], min_count: int = 60\n'
    ') -> list[object]:\n'
    '    return [\n'
    '        "" if is_blank(z) else _financial_conditions_sheets_round(float(z), 2)\n'
    '        for z in expanding_zscore_sample(values, counts, min_count)\n'
    '    ]\n'
    '\n'
    '\n'
    'def _financial_conditions_to_date_or_none(value: Any) -> date | None:\n'
    '    """Parse A37 dates, including its live sheet\'s day-first display format."""\n'
    '    text = str(value).strip()\n'
    '    if "/" in text:\n'
    '        for fmt in ("%d/%m/%Y", "%d/%m/%y"):\n'
    '            try:\n'
    '                return datetime.strptime(text, fmt).date()\n'
    '            except ValueError:\n'
    '                pass\n'
    '    return to_date_or_none(value)\n'
    '\n'
    '\n'
    'def _financial_conditions_month_to_date_averages(\n'
    '    records: Sequence[dict[str, Any]],\n'
    ') -> list[Any]:\n'
    '    parsed_dates = [\n'
    '        _financial_conditions_to_date_or_none(r.get("Date", "")) for r in records\n'
    '    ]\n'
    '    raw_values = [to_float_or_none(r.get("Raw Value", r.get("B", ""))) for r in records]\n'
    '    by_month: dict[tuple[int, int], list[tuple[date, float]]] = {}\n'
    '    for current_date, value in zip(parsed_dates, raw_values):\n'
    '        if current_date is None or value is None:\n'
    '            continue\n'
    '        by_month.setdefault((current_date.year, current_date.month), []).append(\n'
    '            (current_date, value)\n'
    '        )\n'
    '    month_dates: dict[tuple[int, int], list[date]] = {}\n'
    '    month_prefix: dict[tuple[int, int], list[float]] = {}\n'
    '    for key, pairs in by_month.items():\n'
    '        pairs.sort(key=lambda item: item[0])\n'
    '        dates = [item[0] for item in pairs]\n'
    '        prefix = [0.0]\n'
    '        for _, value in pairs:\n'
    '            prefix.append(prefix[-1] + value)\n'
    '        month_dates[key] = dates\n'
    '        month_prefix[key] = prefix\n'
    '    averages: list[Any] = []\n'
    '    for current_date in parsed_dates:\n'
    '        if current_date is None:\n'
    '            averages.append("")\n'
    '            continue\n'
    '        key = (current_date.year, current_date.month)\n'
    '        dates = month_dates.get(key, [])\n'
    '        count = bisect.bisect_right(dates, current_date)\n'
    '        averages.append(month_prefix[key][count] / count if count else "")\n'
    '    return averages\n'
    '\n'
    '\n'
    'def _financial_conditions_regime(e: Any, g: Any) -> str:\n'
    '    if is_blank(e):\n'
    '        return ""\n'
    '    if not is_blank(g) and e < 0 and g < 0:\n'
    '        return "Easy and easing financial conditions"\n'
    '    if not is_blank(g) and e < 0 and g >= 0:\n'
    '        return "Easy but tightening financial conditions"\n'
    '    if not is_blank(g) and e >= 0 and g < 0:\n'
    '        return "Tight but easing financial conditions"\n'
    '    if not is_blank(g) and e >= 0 and g >= 0:\n'
    '        return "Tight and tightening financial conditions"\n'
    '    return "Mixed financial conditions regime"\n'
    '\n'
    '\n'
    'def _financial_conditions_asset_bias_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "Strongly supportive for risk assets / credit / equities"\n'
    '    if z >= 2:\n'
    '        return "Mildly supportive for risk assets"\n'
    '    if z <= -6:\n'
    '        return "Strong headwind for risk assets / credit / equities"\n'
    '    if z <= -2:\n'
    '        return "Mild headwind for risk assets"\n'
    '    return "Neutral risk-asset bias"\n'
    '\n'
    '\n'
    'def _financial_conditions_shock_label(f: Any) -> str:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if f >= 0.75:\n'
    '        return "Large financial-conditions tightening shock"\n'
    '    if f >= 0.40:\n'
    '        return "Moderate financial-conditions tightening shock"\n'
    '    if f <= -0.75:\n'
    '        return "Large financial-conditions easing shock"\n'
    '    if f <= -0.40:\n'
    '        return "Moderate financial-conditions easing shock"\n'
    '    return "No major financial-conditions shock"\n'
    '\n'
    '\n'
    'def _financial_conditions_stress_easing_score(\n'
    '    ai: Any, aj: Any, ak: Any, e: Any, g: Any, h: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (ai, aj, ak, e, g, h)):\n'
    '        return ""\n'
    '    if ai > 1.5 and aj > 0 and ak > 0 and g > 0 and h > 0:\n'
    '        return -7\n'
    '    if ai > 1 and aj > 0 and g > 0:\n'
    '        return -5\n'
    '    if ai < -1 and aj < 0 and ak < 0 and g < 0 and h < 0:\n'
    '        return 5\n'
    '    if aj < -1 and ak < -1 and g < 0 and h < 0:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _financial_conditions_shock_score(f: Any) -> Any:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if f >= 0.75:\n'
    '        return -6\n'
    '    if f >= 0.40:\n'
    '        return -4\n'
    '    if f <= -0.75:\n'
    '        return 6\n'
    '    if f <= -0.40:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _financial_conditions_recovery_score(g: Any, h: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (g, h)):\n'
    '        return ""\n'
    '    if g < 0 and h < 0:\n'
    '        return 5\n'
    '    if g > 0 and h > 0:\n'
    '        return -5\n'
    '    if g < 0 and h > 0:\n'
    '        return 3\n'
    '    if g > 0 and h < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _financial_conditions_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return -6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return -4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return 6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _financial_conditions_regime_signal(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 2:\n'
    '        return "Financial conditions easing / supportive"\n'
    '    if z <= -2:\n'
    '        return "Financial conditions tightening / restrictive"\n'
    '    return "Neutral financial conditions"\n'
    '\n'
    '\n'
    'def _financial_conditions_momentum_label(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(x) for x in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 2 and aa >= 2:\n'
    '        return "Financial conditions easing / supportive"\n'
    '    if z >= 2 and aa <= -2:\n'
    '        return "Financial conditions easy but tightening"\n'
    '    if z >= 2 and -2 < aa < 2:\n'
    '        return "Financial conditions supportive / stable"\n'
    '    if z <= -2 and aa >= 2:\n'
    '        return "Financial stress easing"\n'
    '    if z <= -2 and aa <= -2:\n'
    '        return "Financial conditions tightening / restrictive"\n'
    '    if z <= -2 and -2 < aa < 2:\n'
    '        return "Financial conditions tight / stable"\n'
    '    if -2 < z < 2 and aa >= 2:\n'
    '        return "Financial conditions impulse easing"\n'
    '    if -2 < z < 2 and aa <= -2:\n'
    '        return "Financial conditions impulse tightening"\n'
    '    return "Financial conditions broadly neutral"\n'
    '\n'
    '\n'
    'def _financial_conditions_signal_interpretation(aa: Any, ab: Any) -> str:\n'
    '    if any(is_blank(x) for x in (aa, ab)):\n'
    '        return ""\n'
    '    if aa >= 6 and ab >= 0:\n'
    '        return "Strong financial-conditions easing signal with supportive risk-asset profile"\n'
    '    if aa >= 6 and ab < 0:\n'
    '        return "Strong financial-conditions easing signal, but stress-risk profile remains mixed"\n'
    '    if aa >= 2 and ab >= 0:\n'
    '        return "Moderately positive financial-conditions easing signal"\n'
    '    if aa >= 2 and ab < 0:\n'
    '        return "Financial conditions easing, but stress risk remains fragile"\n'
    '    if aa > -2 and ab > -3 and ab < 3:\n'
    '        return "Neutral/mixed financial-conditions signal"\n'
    '    if aa <= -6 and ab <= 0:\n'
    '        return (\n'
    '            "Severe financial-conditions tightening signal with downside confirmation"\n'
    '        )\n'
    '    if aa <= -6 and ab > 0:\n'
    '        return (\n'
    '            "Severe financial-stress signal, but easing support remains partly intact"\n'
    '        )\n'
    '    if aa <= -2 and ab <= 0:\n'
    '        return "Moderately negative financial-conditions tightening signal"\n'
    '    if aa <= -2 and ab > 0:\n'
    '        return "Financial conditions tightening, but stress profile remains mixed"\n'
    '    return "Mixed/transition financial-conditions signal"\n'
    '\n'
    '\n'
    'def _financial_conditions_warning_label(ab: Any, v: Any, w: Any, y: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ab, v, w, y)):\n'
    '        return ""\n'
    '    if y <= -6:\n'
    '        return "High financial-stress extreme warning"\n'
    '    if y <= -4:\n'
    '        return "Moderate financial-stress extreme warning"\n'
    '    if w <= -6:\n'
    '        return "High financial-conditions tightening shock"\n'
    '    if w <= -4:\n'
    '        return "Moderate financial-conditions tightening shock"\n'
    '    if v <= -6:\n'
    '        return "High financial-stress warning"\n'
    '    if v <= -4:\n'
    '        return "Moderate financial-stress warning"\n'
    '    if ab >= 6:\n'
    '        return "High financial-conditions easing signal"\n'
    '    if ab >= 3:\n'
    '        return "Moderate financial-conditions easing signal"\n'
    '    if ab > -3:\n'
    '        return "Neutral financial-conditions warning profile"\n'
    '    if ab > -6:\n'
    '        return "Moderate financial-conditions tightening warning"\n'
    '    return "High financial-conditions tightening warning"\n'
    '\n'
    '\n'
    'def _financial_conditions_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme financial-stress tightening"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong financial-stress tightening"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme financial-conditions easing"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong financial-conditions easing"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_financial_conditions_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    source_values = [r.get("Raw Value", r.get("B", "")) for r in records]\n'
    '    c_values = _financial_conditions_month_to_date_averages(records)\n'
    '    c_counts = _metric_counts(c_values)\n'
    '    smoothed: list[Any] = []\n'
    '    for idx, _ in enumerate(c_values):\n'
    '        if c_counts[idx] < 3:\n'
    '            smoothed.append("")\n'
    '        else:\n'
    '            smoothed.append(\n'
    '                _financial_conditions_mean_sheets(c_values[max(0, idx - 2) : idx + 1])\n'
    '            )\n'
    '    observation_counts = _metric_counts(smoothed)\n'
    '    one_changes: list[Any] = []\n'
    '    three_changes: list[Any] = []\n'
    '    six_changes: list[Any] = []\n'
    '    twelve_changes: list[Any] = []\n'
    '    for idx, e in enumerate(smoothed):\n'
    '        d = observation_counts[idx]\n'
    '        e1 = smoothed[idx - 1] if idx >= 1 else ""\n'
    '        e3 = smoothed[idx - 3] if idx >= 3 else ""\n'
    '        e6 = smoothed[idx - 6] if idx >= 6 else ""\n'
    '        e12 = smoothed[idx - 12] if idx >= 12 else ""\n'
    '        one_changes.append(\n'
    '            e - e1 if d >= 2 and not is_blank(e) and not is_blank(e1) else ""\n'
    '        )\n'
    '        three_changes.append(\n'
    '            e - e3 if d >= 4 and not is_blank(e) and not is_blank(e3) else ""\n'
    '        )\n'
    '        six_changes.append(\n'
    '            e - e6 if d >= 7 and not is_blank(e) and not is_blank(e6) else ""\n'
    '        )\n'
    '        twelve_changes.append(\n'
    '            e - e12 if d >= 13 and not is_blank(e) and not is_blank(e12) else ""\n'
    '        )\n'
    '    e_counts = _metric_counts(smoothed)\n'
    '    g_counts = _metric_counts(three_changes)\n'
    '    h_counts = _metric_counts(six_changes)\n'
    '    i_counts = _metric_counts(twelve_changes)\n'
    '    e_ranks = _financial_conditions_metric_ranks_sheets(smoothed)\n'
    '    g_ranks = _financial_conditions_metric_ranks_sheets(three_changes)\n'
    '    h_ranks = _financial_conditions_metric_ranks_sheets(six_changes)\n'
    '    i_ranks = _financial_conditions_metric_ranks_sheets(twelve_changes)\n'
    '    e_z = _financial_conditions_rounded_expanding_zscores(smoothed, e_counts)\n'
    '    g_z = _financial_conditions_rounded_expanding_zscores(three_changes, g_counts)\n'
    '    h_z = _financial_conditions_rounded_expanding_zscores(six_changes, h_counts)\n'
    '    i_z = _financial_conditions_rounded_expanding_zscores(twelve_changes, i_counts)\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = c_values[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = smoothed[idx]\n'
    '        f = one_changes[idx]\n'
    '        g = three_changes[idx]\n'
    '        h = six_changes[idx]\n'
    '        i = twelve_changes[idx]\n'
    '        j_internal = e_ranks[idx]\n'
    '        k = _financial_conditions_score_percentile(j_internal)\n'
    '        l_internal = g_ranks[idx]\n'
    '        m = _financial_conditions_score_percentile(l_internal)\n'
    '        n_internal = h_ranks[idx]\n'
    '        o = _financial_conditions_score_percentile(n_internal)\n'
    '        p_internal = i_ranks[idx]\n'
    '        q = _financial_conditions_score_percentile(p_internal)\n'
    '        u = ""\n'
    '        if all(not is_blank(value) for value in (m, o, q, k)):\n'
    '            u = _financial_conditions_sheets_round(\n'
    '                clamp(\n'
    '                    _financial_conditions_weighted_sum(\n'
    '                        (0.45, m), (0.30, o), (0.15, q), (0.10, k)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '        ai = e_z[idx]\n'
    '        aj = g_z[idx]\n'
    '        ak = h_z[idx]\n'
    '        al = i_z[idx]\n'
    '        v = _financial_conditions_stress_easing_score(ai, aj, ak, e, g, h)\n'
    '        w = _financial_conditions_shock_score(f)\n'
    '        x = _financial_conditions_recovery_score(g, h)\n'
    '        y = _financial_conditions_outlier_score([ai, aj, ak, al])\n'
    '        z = ""\n'
    '        if all(not is_blank(value) for value in (k, m, o, q)):\n'
    '            z = _financial_conditions_sheets_round(\n'
    '                clamp(\n'
    '                    _financial_conditions_weighted_sum(\n'
    '                        (0.40, k), (0.30, m), (0.20, o), (0.10, q)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '        aa = ""\n'
    '        if all(not is_blank(value) for value in (u, v, x)):\n'
    '            aa = _financial_conditions_sheets_round(\n'
    '                clamp(\n'
    '                    _financial_conditions_weighted_sum((0.50, u), (0.30, v), (0.20, x)),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '        ab = ""\n'
    '        if all(not is_blank(value) for value in (v, w, x, y)):\n'
    '            ab = _financial_conditions_sheets_round(\n'
    '                clamp(\n'
    '                    _financial_conditions_weighted_sum(\n'
    '                        (0.35, v), (0.25, w), (0.20, x), (0.20, y)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(value) for value in (z, aa, ab))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Monthly Average Financial Conditions": c,\n'
    '                "Sample Count": d,\n'
    '                "3M MA Financial Conditions": e,\n'
    '                "1M Financial Conditions Change": f,\n'
    '                "3M Financial Conditions Change": g,\n'
    '                "6M Financial Conditions Change": h,\n'
    '                "12M Financial Conditions Change": i,\n'
    '                "Financial Conditions Level Percentile": _financial_conditions_percent_display(\n'
    '                    j_internal\n'
    '                ),\n'
    '                "Financial Conditions Level Stress Score": k,\n'
    '                "3M Change Percentile": _financial_conditions_percent_display(\n'
    '                    l_internal\n'
    '                ),\n'
    '                "3M Change Stress Score": m,\n'
    '                "6M Change Percentile": _financial_conditions_percent_display(\n'
    '                    n_internal\n'
    '                ),\n'
    '                "6M Change Stress Score": o,\n'
    '                "12M Change Percentile": _financial_conditions_percent_display(\n'
    '                    p_internal\n'
    '                ),\n'
    '                "12M Change Stress Score": q,\n'
    '                "Financial Conditions Regime": _financial_conditions_regime(e, g),\n'
    '                "Risk Asset Bias": _financial_conditions_asset_bias_label(z),\n'
    '                "Financial Conditions Shock Flag": _financial_conditions_shock_label(f),\n'
    '                "Financial Conditions Impulse Score": u,\n'
    '                "Financial Stress Risk Score": v,\n'
    '                "Financial Conditions Shock Score": w,\n'
    '                "Financial Conditions Cycle Turning Point Score": x,\n'
    '                "Financial Conditions Extreme Score": y,\n'
    '                "Coincident Score": z,\n'
    '                "Leading Score": aa,\n'
    '                "Warning Score": ab,\n'
    '                "State": _financial_conditions_regime_signal(z),\n'
    '                "Regime": _financial_conditions_momentum_label(z, aa),\n'
    '                "Signal Comment": _financial_conditions_signal_interpretation(aa, ab),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _financial_conditions_warning_label(ab, v, w, y),\n'
    '                "Level Z-Score": ai,\n'
    '                "3M Change Z-Score": aj,\n'
    '                "6M Change Z-Score": ak,\n'
    '                "12M Change Z-Score": al,\n'
    '                "Extreme Flag": _financial_conditions_outlier_label([ai, aj, ak, al]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, FINANCIAL_CONDITIONS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _bank_credit_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A38-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _bank_credit_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A38-only spreadsheet-order average over numeric cells."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _bank_credit_percentrank_inc_sheets(values: Sequence[float], current: Any) -> Any:\n'
    '    """A38-only Google Sheets PERCENTRANK.INC semantics."""\n'
    '    return _real_yield_percentrank_inc_sheets(values, current)\n'
    '\n'
    '\n'
    'def _bank_credit_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _bank_credit_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _bank_credit_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _bank_credit_score_percentile(rank: Any) -> Any:\n'
    '    return (\n'
    '        "" if is_blank(rank) else _bank_credit_sheets_round((float(rank) * 20) - 10, 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _bank_credit_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _bank_credit_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _bank_credit_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    """Evaluate A38 composites from their rounded visible operands."""\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _bank_credit_impulse_score(l: Any, n: Any, p: Any, r: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (l, n, p, r)):\n'
    '        return ""\n'
    '    return _bank_credit_sheets_round(\n'
    '        clamp(\n'
    '            _bank_credit_weighted_sum((0.05, l), (0.50, n), (0.30, p), (0.15, r)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _bank_credit_acceleration_score(v: Any, previous_v: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (v, previous_v)):\n'
    '        return ""\n'
    '    return _bank_credit_sheets_round(\n'
    '        clamp(_bank_credit_weighted_sum((2.0, v), (-2.0, previous_v)), -10, 10), 2\n'
    '    )\n'
    '\n'
    '\n'
    'def _bank_credit_coincident_score(l: Any, n: Any, p: Any, r: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (l, n, p, r)):\n'
    '        return ""\n'
    '    return _bank_credit_sheets_round(\n'
    '        clamp(\n'
    '            _bank_credit_weighted_sum((0.20, l), (0.45, n), (0.25, p), (0.10, r)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _bank_credit_leading_score(v: Any, w: Any, x: Any, z: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (v, w, x, z)):\n'
    '        return ""\n'
    '    return _bank_credit_sheets_round(\n'
    '        clamp(\n'
    '            _bank_credit_weighted_sum((0.45, v), (0.25, w), (0.20, x), (0.10, z)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _bank_credit_warning_score(w: Any, x: Any, y: Any, z: Any, aa: Any, ab: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (w, x, y, z, aa, ab)):\n'
    '        return ""\n'
    '    return _bank_credit_sheets_round(\n'
    '        clamp(\n'
    '            _bank_credit_weighted_sum(\n'
    '                (0.15, w), (0.15, x), (0.20, y), (0.30, z), (0.10, aa), (0.10, ab)\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '\n'
    '\n'
    'def _bank_credit_positive_or_blank(value: Any) -> bool:\n'
    '    return value == "" or value > 0\n'
    '\n'
    '\n'
    'def _bank_credit_growth_regime(c: Any, f: Any, g: Any, h: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if (\n'
    '        _bank_credit_positive_or_blank(h)\n'
    '        and _bank_credit_positive_or_blank(f)\n'
    '        and _bank_credit_positive_or_blank(g)\n'
    '    ):\n'
    '        return "Bank-credit expansion regime"\n'
    '    if not any(is_blank(x) for x in (h, f, g)) and h > 0 and f < 0 and g < 0:\n'
    '        return "Bank-credit deterioration from high base"\n'
    '    if not any(is_blank(x) for x in (h, f, g)) and h <= 0 and f > 0 and g > 0:\n'
    '        return "Early bank-credit recovery"\n'
    '    if not any(is_blank(x) for x in (h, f, g)) and h <= 0 and f < 0 and g < 0:\n'
    '        return "Bank-credit contraction regime"\n'
    '    if not any(is_blank(x) for x in (f, g)) and f > 0 and g < 0:\n'
    '        return "Short-term bank-credit recovery / mixed impulse"\n'
    '    if not any(is_blank(x) for x in (f, g)) and f < 0 and g > 0:\n'
    '        return "Short-term bank-credit deterioration / mixed impulse"\n'
    '    return "Neutral/mixed bank-credit regime"\n'
    '\n'
    '\n'
    'def _bank_credit_asset_bias_label(ac: Any) -> str:\n'
    '    if is_blank(ac):\n'
    '        return ""\n'
    '    if ac >= 6:\n'
    '        return "Strongly supportive for credit-sensitive growth and risk assets"\n'
    '    if ac >= 2:\n'
    '        return "Mildly supportive for credit-sensitive growth and risk assets"\n'
    '    if ac <= -6:\n'
    '        return "Strong credit-growth headwind for risk assets and domestic demand"\n'
    '    if ac <= -2:\n'
    '        return "Mild credit-growth headwind for risk assets and domestic demand"\n'
    '    return "Neutral bank-credit bias"\n'
    '\n'
    '\n'
    'def _bank_credit_shock_label(i: Any) -> str:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 0.15:\n'
    '        return "Extreme 3M annualised bank-credit growth surge"\n'
    '    if i >= 0.10:\n'
    '        return "Strong 3M annualised bank-credit growth surge"\n'
    '    if i <= -0.05:\n'
    '        return "Sharp 3M annualised bank-credit contraction"\n'
    '    if i <= 0:\n'
    '        return "Negative 3M annualised bank-credit growth"\n'
    '    if i <= 0.03:\n'
    '        return "Very weak 3M annualised bank-credit growth"\n'
    '    return "No major bank-credit growth shock"\n'
    '\n'
    '\n'
    'def _bank_credit_recovery_score(f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (f, g)):\n'
    '        return ""\n'
    '    if f > 0 and g > 0:\n'
    '        return 5\n'
    '    if f < 0 and g < 0:\n'
    '        return -5\n'
    '    if f > 0 and g < 0:\n'
    '        return 3\n'
    '    if f < 0 and g > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _bank_credit_shock_score(i: Any) -> Any:\n'
    '    if is_blank(i):\n'
    '        return ""\n'
    '    if i >= 0.15:\n'
    '        return 6\n'
    '    if i >= 0.10:\n'
    '        return 4\n'
    '    if i <= -0.05:\n'
    '        return -6\n'
    '    if i <= 0:\n'
    '        return -4\n'
    '    if i <= 0.03:\n'
    '        return -2\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _bank_credit_contraction_risk(\n'
    '    am: Any, an: Any, ao: Any, f: Any, g: Any, h: Any, i: Any\n'
    ') -> Any:\n'
    '    if any(is_blank(x) for x in (am, an, ao, f, g, h, i)):\n'
    '        return ""\n'
    '    if am < -1.5 and an < -1.5 and f < 0 and g < 0:\n'
    '        return -7\n'
    '    if am < -1 and an < -1 and f < 0:\n'
    '        return -5\n'
    '    if i < 0:\n'
    '        return -4\n'
    '    if i < 0.02 and am < 0 and an < 0:\n'
    '        return -3\n'
    '    if am > 0 and an > 0 and f > 0 and g > 0:\n'
    '        return 3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _bank_credit_overheating_score(am: Any, an: Any, i: Any, j: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (am, an, i, j)):\n'
    '        return ""\n'
    '    if am > 1.5 and an > 1.5 and i > 0.15 and j > 0.10:\n'
    '        return 5\n'
    '    if am > 1 and an > 1 and i > 0.10:\n'
    '        return 3\n'
    '    if am < -1 and an < -1 and i < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _bank_credit_growth_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _bank_credit_conditions_regime(ac: Any) -> str:\n'
    '    if is_blank(ac):\n'
    '        return ""\n'
    '    if ac >= 2:\n'
    '        return "Bank credit growth improving"\n'
    '    if ac <= -2:\n'
    '        return "Bank credit growth deteriorating"\n'
    '    return "Neutral bank credit growth"\n'
    '\n'
    '\n'
    'def _bank_credit_momentum_label(ac: Any, ad: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ac, ad)):\n'
    '        return ""\n'
    '    if ac >= 2 and ad >= 2:\n'
    '        return "Bank credit growth improving"\n'
    '    if ac >= 2 and ad <= -2:\n'
    '        return "Credit growth positive but deteriorating"\n'
    '    if ac >= 2 and -2 < ad < 2:\n'
    '        return "Bank credit growth supportive / stable"\n'
    '    if ac <= -2 and ad >= 2:\n'
    '        return "Bank credit weak but improving"\n'
    '    if ac <= -2 and ad <= -2:\n'
    '        return "Bank credit growth deteriorating"\n'
    '    if ac <= -2 and -2 < ad < 2:\n'
    '        return "Bank credit growth weak / stable"\n'
    '    if -2 < ac < 2 and ad >= 2:\n'
    '        return "Bank credit impulse improving"\n'
    '    if -2 < ac < 2 and ad <= -2:\n'
    '        return "Bank credit impulse deteriorating"\n'
    '    return "Bank credit growth broadly neutral"\n'
    '\n'
    '\n'
    'def _bank_credit_signal_interpretation(ad: Any, ae: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ad, ae)):\n'
    '        return ""\n'
    '    if ad >= 6 and ae >= 0:\n'
    '        return (\n'
    '            "Strong positive bank-credit impulse with supportive private-credit profile"\n'
    '        )\n'
    '    if ad >= 6 and ae < 0:\n'
    '        return (\n'
    '            "Strong positive bank-credit impulse, but credit-risk profile remains mixed"\n'
    '        )\n'
    '    if ad >= 2 and ae >= 0:\n'
    '        return "Moderately positive bank-credit impulse signal"\n'
    '    if ad >= 2 and ae < 0:\n'
    '        return "Bank-credit impulse improving, but credit-risk profile remains fragile"\n'
    '    if ad > -2 and ae > -3 and ae < 3:\n'
    '        return "Neutral/mixed bank-credit signal"\n'
    '    if ad <= -6 and ae <= 0:\n'
    '        return "Strong bank-credit deterioration signal with downside confirmation"\n'
    '    if ad <= -6 and ae > 0:\n'
    '        return "Strong bank-credit deterioration signal, but credit support remains partly intact"\n'
    '    if ad <= -2 and ae <= 0:\n'
    '        return "Moderately negative bank-credit impulse signal"\n'
    '    if ad <= -2 and ae > 0:\n'
    '        return "Bank-credit impulse deteriorating, but credit conditions remain mixed"\n'
    '    return "Mixed/transition bank-credit signal"\n'
    '\n'
    '\n'
    'def _bank_credit_warning_label(ae: Any, y: Any, z: Any, aa: Any, ab: Any) -> str:\n'
    '    if any(is_blank(x) for x in (ae, y, z, aa, ab)):\n'
    '        return ""\n'
    '    if z <= -6:\n'
    '        return "High bank-credit contraction warning"\n'
    '    if z <= -4:\n'
    '        return "Moderate bank-credit contraction warning"\n'
    '    if y <= -6:\n'
    '        return "High bank-credit contraction shock warning"\n'
    '    if y <= -4:\n'
    '        return "Moderate bank-credit contraction shock warning"\n'
    '    if ab <= -6:\n'
    '        return "Extreme bank-credit contraction warning"\n'
    '    if ab <= -4:\n'
    '        return "Strong bank-credit contraction warning"\n'
    '    if aa >= 5:\n'
    '        return "High credit-overheating / credit-boom warning"\n'
    '    if aa >= 3:\n'
    '        return "Moderate credit-overheating warning"\n'
    '    if ae >= 6:\n'
    '        return "High bank-credit expansion signal"\n'
    '    if ae >= 3:\n'
    '        return "Moderate bank-credit expansion signal"\n'
    '    if ae > -3:\n'
    '        return "Neutral bank-credit warning profile"\n'
    '    if ae > -6:\n'
    '        return "Moderate bank-credit deterioration warning"\n'
    '    return "High bank-credit deterioration warning"\n'
    '\n'
    '\n'
    'def _bank_credit_growth_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme bank-credit expansion"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong bank-credit expansion"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme bank-credit contraction"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong bank-credit contraction"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def _bank_credit_safe_growth(current: Any, prior: Any) -> Any:\n'
    '    if is_blank(current) or is_blank(prior) or prior == 0:\n'
    '        return ""\n'
    '    return current / prior - 1\n'
    '\n'
    '\n'
    'def calculate_bank_credit_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    source_values = [r.get("Raw Value", r.get("B", "")) for r in records]\n'
    '    raw_values = [to_float_or_none(v) for v in source_values]\n'
    '    ma_values = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 3:\n'
    '            ma_values.append("")\n'
    '        else:\n'
    '            ma_values.append(\n'
    '                _bank_credit_mean_sheets(raw_values[max(0, idx - 2) : idx + 1])\n'
    '            )\n'
    '    c_counts = _metric_counts(ma_values)\n'
    '    observation_counts = [max(0, count - 2) for count in c_counts]\n'
    '    one_growth = []\n'
    '    three_growth = []\n'
    '    six_growth = []\n'
    '    twelve_growth = []\n'
    '    three_annual = []\n'
    '    six_annual = []\n'
    '    for idx, c in enumerate(ma_values):\n'
    '        d = observation_counts[idx]\n'
    '        c1 = ma_values[idx - 1] if idx >= 1 else ""\n'
    '        c3 = ma_values[idx - 3] if idx >= 3 else ""\n'
    '        c6 = ma_values[idx - 6] if idx >= 6 else ""\n'
    '        c12 = ma_values[idx - 12] if idx >= 12 else ""\n'
    '        e = _bank_credit_safe_growth(c, c1) if d >= 2 else ""\n'
    '        f = _bank_credit_safe_growth(c, c3) if d >= 4 else ""\n'
    '        g = _bank_credit_safe_growth(c, c6) if d >= 7 else ""\n'
    '        h = _bank_credit_safe_growth(c, c12) if d >= 13 else ""\n'
    '        one_growth.append(e)\n'
    '        three_growth.append(f)\n'
    '        six_growth.append(g)\n'
    '        twelve_growth.append(h)\n'
    '        try:\n'
    '            three_annual.append((1 + f) ** 4 - 1 if not is_blank(f) else "")\n'
    '        except Exception:\n'
    '            three_annual.append("")\n'
    '        try:\n'
    '            six_annual.append((1 + g) ** 2 - 1 if not is_blank(g) else "")\n'
    '        except Exception:\n'
    '            six_annual.append("")\n'
    '    f_counts = _metric_counts(three_growth)\n'
    '    g_counts = _metric_counts(six_growth)\n'
    '    h_counts = _metric_counts(twelve_growth)\n'
    '    c_ranks = _bank_credit_metric_ranks_sheets(ma_values, 60)\n'
    '    f_ranks = _bank_credit_metric_ranks_sheets(three_growth, 60)\n'
    '    g_ranks = _bank_credit_metric_ranks_sheets(six_growth, 60)\n'
    '    h_ranks = _bank_credit_metric_ranks_sheets(twelve_growth, 60)\n'
    '    c_z = _rounded_expanding_zscores(ma_values, c_counts, 60)\n'
    '    f_z = _rounded_expanding_zscores(three_growth, f_counts, 60)\n'
    '    g_z = _rounded_expanding_zscores(six_growth, g_counts, 60)\n'
    '    h_z = _rounded_expanding_zscores(twelve_growth, h_counts, 60)\n'
    '    momentum_scores = []\n'
    '    rows = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = ma_values[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_growth[idx]\n'
    '        f = three_growth[idx]\n'
    '        g = six_growth[idx]\n'
    '        h = twelve_growth[idx]\n'
    '        i = three_annual[idx]\n'
    '        j = six_annual[idx]\n'
    '        k = c_ranks[idx]\n'
    '        l = _bank_credit_score_percentile(k)\n'
    '        m = f_ranks[idx]\n'
    '        n = _bank_credit_score_percentile(m)\n'
    '        o = g_ranks[idx]\n'
    '        p = _bank_credit_score_percentile(o)\n'
    '        q = h_ranks[idx]\n'
    '        r = _bank_credit_score_percentile(q)\n'
    '        v = _bank_credit_impulse_score(l, n, p, r)\n'
    '        momentum_scores.append(v)\n'
    '        prev_v = momentum_scores[idx - 1] if idx >= 1 else ""\n'
    '        w = _bank_credit_acceleration_score(v, prev_v)\n'
    '        al = c_z[idx]\n'
    '        am = f_z[idx]\n'
    '        an = g_z[idx]\n'
    '        ao = h_z[idx]\n'
    '        x = _bank_credit_recovery_score(f, g)\n'
    '        y = _bank_credit_shock_score(i)\n'
    '        z = _bank_credit_contraction_risk(am, an, ao, f, g, h, i)\n'
    '        aa = _bank_credit_overheating_score(am, an, i, j)\n'
    '        ab = _bank_credit_growth_outlier_score([am, an, ao])\n'
    '        ac = _bank_credit_coincident_score(l, n, p, r)\n'
    '        ad = _bank_credit_leading_score(v, w, x, z)\n'
    '        ae = _bank_credit_warning_score(w, x, y, z, aa, ab)\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (ac, ad, ae))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Bank Credit": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Bank Credit Growth": e,\n'
    '                "3M Bank Credit Growth": f,\n'
    '                "6M Bank Credit Growth": g,\n'
    '                "12M Bank Credit Growth": h,\n'
    '                "3M Annualised Bank Credit Growth": i,\n'
    '                "6M Annualised Bank Credit Growth": j,\n'
    '                "Bank Credit Level Percentile": _bank_credit_percent_display(k),\n'
    '                "Bank Credit Level Score": l,\n'
    '                "3M Growth Percentile": _bank_credit_percent_display(m),\n'
    '                "3M Growth Score": n,\n'
    '                "6M Growth Percentile": _bank_credit_percent_display(o),\n'
    '                "6M Growth Score": p,\n'
    '                "12M Growth Percentile": _bank_credit_percent_display(q),\n'
    '                "12M Growth Score": r,\n'
    '                "Bank Credit Regime": _bank_credit_growth_regime(c, f, g, h),\n'
    '                "Risk Asset Bias": _bank_credit_asset_bias_label(ac),\n'
    '                "Bank Credit Growth Shock Flag": _bank_credit_shock_label(i),\n'
    '                "Bank Credit Impulse Score": v,\n'
    '                "Bank Credit Acceleration Score": w,\n'
    '                "Bank Credit Cycle Turning Point Score": x,\n'
    '                "Bank Credit Shock Score": y,\n'
    '                "Credit Contraction Risk Score": z,\n'
    '                "Credit Overheating Risk Score": aa,\n'
    '                "Bank Credit Extreme Score": ab,\n'
    '                "Coincident Score": ac,\n'
    '                "Leading Score": ad,\n'
    '                "Warning Score": ae,\n'
    '                "State": _bank_credit_conditions_regime(ac),\n'
    '                "Regime": _bank_credit_momentum_label(ac, ad),\n'
    '                "Signal Comment": _bank_credit_signal_interpretation(ad, ae),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _bank_credit_warning_label(ae, y, z, aa, ab),\n'
    '                "Level Z-Score": al,\n'
    '                "3M Growth Z-Score": am,\n'
    '                "6M Growth Z-Score": an,\n'
    '                "12M Growth Z-Score": ao,\n'
    '                "Extreme Flag": _bank_credit_growth_outlier_label([am, an, ao]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, BANK_CREDIT_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _corporate_spread_sheets_round(value: Any, places: int = 2) -> Any:\n'
    '    """A39-only Google Sheets ROUND semantics."""\n'
    '    return _real_yield_sheets_round(value, places)\n'
    '\n'
    '\n'
    'def _corporate_spread_mean_sheets(values: Sequence[Any]) -> Any:\n'
    '    """A39-only spreadsheet-order average that preserves last-bit rank ordering."""\n'
    '    return _real_yield_mean_sheets(values)\n'
    '\n'
    '\n'
    'def _corporate_spread_percentrank_inc_sheets(\n'
    '    values: Sequence[float], current: Any\n'
    ') -> Any:\n'
    '    """A39-only Google Sheets PERCENTRANK.INC over full-precision metrics."""\n'
    '    current_float = to_float_or_none(current)\n'
    '    vals = sorted(\n'
    '        number\n'
    '        for number in (to_float_or_none(value) for value in values)\n'
    '        if number is not None\n'
    '    )\n'
    '    n = len(vals)\n'
    '    if n == 0 or current_float is None:\n'
    '        return ""\n'
    '    if n == 1:\n'
    '        return 0.0 if vals[0] == current_float else ""\n'
    '    if current_float < vals[0] or current_float > vals[-1]:\n'
    '        return ""\n'
    '    lo = bisect.bisect_left(vals, current_float)\n'
    '    hi = bisect.bisect_right(vals, current_float)\n'
    '    if lo < hi:\n'
    '        return lo / (n - 1)\n'
    '    lower_index = lo - 1\n'
    '    upper_index = lo\n'
    '    lower_value = vals[lower_index]\n'
    '    upper_value = vals[upper_index]\n'
    '    if upper_value == lower_value:\n'
    '        return lower_index / (n - 1)\n'
    '    fraction = (current_float - lower_value) / (upper_value - lower_value)\n'
    '    return (lower_index + fraction) / (n - 1)\n'
    '\n'
    '\n'
    'def _corporate_spread_metric_ranks_sheets(\n'
    '    values: Sequence[Any], min_count: int = 60\n'
    ') -> list[Any]:\n'
    '    history: list[float] = []\n'
    '    ranks: list[Any] = []\n'
    '    for value in values:\n'
    '        numeric = to_float_or_none(value)\n'
    '        if numeric is not None:\n'
    '            bisect.insort(history, numeric)\n'
    '        if numeric is None or len(history) < min_count:\n'
    '            ranks.append("")\n'
    '            continue\n'
    '        rank = _corporate_spread_percentrank_inc_sheets(history, numeric)\n'
    '        ranks.append("" if rank == "" else _corporate_spread_sheets_round(rank, 3))\n'
    '    return ranks\n'
    '\n'
    '\n'
    'def _corporate_spread_inverted_score(rank: Any) -> Any:\n'
    '    return (\n'
    '        ""\n'
    '        if is_blank(rank)\n'
    '        else _corporate_spread_sheets_round(10 - (float(rank) * 20), 2)\n'
    '    )\n'
    '\n'
    '\n'
    'def _corporate_spread_percent_display(value: Any, decimals: int = 2) -> Any:\n'
    '    numeric = to_float_or_none(value)\n'
    '    if numeric is None:\n'
    '        return ""\n'
    '    displayed = _corporate_spread_sheets_round(numeric * 100, decimals)\n'
    '    if displayed == 0:\n'
    '        displayed = 0.0\n'
    '    return f"{displayed:.{decimals}f}%"\n'
    '\n'
    '\n'
    'def _corporate_spread_weighted_sum(*terms: tuple[float, Any]) -> float:\n'
    '    total = Decimal("0")\n'
    '    for weight, value in terms:\n'
    '        total += Decimal(str(weight)) * Decimal(str(value))\n'
    '    return float(total)\n'
    '\n'
    '\n'
    'def _corporate_spread_credit_stress_score_raw(j: Any, l: Any, n: Any, p: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (j, l, n, p)):\n'
    '        return ""\n'
    '    return clamp(\n'
    '        _corporate_spread_weighted_sum((0.40, j), (0.30, l), (0.20, n), (0.10, p)),\n'
    '        -10,\n'
    '        10,\n'
    '    )\n'
    '\n'
    '\n'
    'def _corporate_spread_impulse_score_raw(j: Any, l: Any, n: Any, p: Any) -> Any:\n'
    '    if any(is_blank(value) for value in (j, l, n, p)):\n'
    '        return ""\n'
    '    return clamp(\n'
    '        _corporate_spread_weighted_sum((0.50, l), (0.30, n), (0.15, p), (0.05, j)),\n'
    '        -10,\n'
    '        10,\n'
    '    )\n'
    '\n'
    '\n'
    'def _corporate_spread_leading_score_raw(\n'
    '    credit_stress_score_raw: Any,\n'
    '    credit_spread_impulse_score_raw: Any,\n'
    '    turning_point_score: Any,\n'
    ') -> Any:\n'
    '    if any(\n'
    '        is_blank(value)\n'
    '        for value in (\n'
    '            credit_stress_score_raw,\n'
    '            credit_spread_impulse_score_raw,\n'
    '            turning_point_score,\n'
    '        )\n'
    '    ):\n'
    '        return ""\n'
    '    return clamp(\n'
    '        _corporate_spread_weighted_sum(\n'
    '            (0.45, credit_stress_score_raw),\n'
    '            (0.35, credit_spread_impulse_score_raw),\n'
    '            (0.20, turning_point_score),\n'
    '        ),\n'
    '        -10,\n'
    '        10,\n'
    '    )\n'
    '\n'
    '\n'
    'def _corporate_spread_warning_score_raw(\n'
    '    credit_stress_score_raw: Any,\n'
    '    turning_point_score: Any,\n'
    '    shock_score: Any,\n'
    '    extreme_score: Any,\n'
    ') -> Any:\n'
    '    if any(\n'
    '        is_blank(value)\n'
    '        for value in (\n'
    '            credit_stress_score_raw,\n'
    '            turning_point_score,\n'
    '            shock_score,\n'
    '            extreme_score,\n'
    '        )\n'
    '    ):\n'
    '        return ""\n'
    '    return clamp(\n'
    '        _corporate_spread_weighted_sum(\n'
    '            (0.35, credit_stress_score_raw),\n'
    '            (0.20, turning_point_score),\n'
    '            (0.25, shock_score),\n'
    '            (0.20, extreme_score),\n'
    '        ),\n'
    '        -10,\n'
    '        10,\n'
    '    )\n'
    '\n'
    '\n'
    'def _corporate_spread_at_or_above_sheets_boundary(\n'
    '    value: Any, boundary: float, places: int = 2\n'
    ') -> bool:\n'
    '    if is_blank(value):\n'
    '        return False\n'
    '    return _corporate_spread_sheets_round(value, places) >= boundary\n'
    '\n'
    '\n'
    'CORPORATE_SPREAD_EPS = 1e-9\n'
    '\n'
    '\n'
    'def _corporate_spread_ge(value: Any, threshold: float) -> bool:\n'
    '    return not is_blank(value) and float(value) >= threshold - CORPORATE_SPREAD_EPS\n'
    '\n'
    '\n'
    'def _corporate_spread_le(value: Any, threshold: float) -> bool:\n'
    '    return not is_blank(value) and float(value) <= threshold + CORPORATE_SPREAD_EPS\n'
    '\n'
    '\n'
    'def _corporate_spread_gt(value: Any, threshold: float) -> bool:\n'
    '    return not is_blank(value) and float(value) > threshold + CORPORATE_SPREAD_EPS\n'
    '\n'
    '\n'
    'def _corporate_spread_lt(value: Any, threshold: float) -> bool:\n'
    '    return not is_blank(value) and float(value) < threshold - CORPORATE_SPREAD_EPS\n'
    '\n'
    '\n'
    'def _corporate_spread_regime(c: Any, e: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    direction = e if is_blank(f) else f\n'
    '    if c >= 4.0 and (is_blank(direction) or direction >= 0):\n'
    '        return "Stress-level corporate spreads and widening"\n'
    '    if c >= 4.0 and direction < 0:\n'
    '        return "Stress-level corporate spreads but narrowing"\n'
    '    if c >= 2.5 and (is_blank(direction) or direction >= 0):\n'
    '        return "Elevated corporate spreads and widening"\n'
    '    if c >= 2.5 and direction < 0:\n'
    '        return "Elevated corporate spreads but narrowing"\n'
    '    if c < 1.5 and (is_blank(direction) or direction < 0):\n'
    '        return "Very tight corporate spreads and still narrowing"\n'
    '    if c < 1.5 and direction >= 0:\n'
    '        return "Very tight corporate spreads but widening"\n'
    '    if is_blank(direction) or direction >= 0:\n'
    '        return "Normal corporate spreads but widening"\n'
    '    return "Normal corporate spreads and narrowing"\n'
    '\n'
    '\n'
    'def _corporate_spread_asset_bias_label(credit_stress_score_raw: Any) -> str:\n'
    '    if is_blank(credit_stress_score_raw):\n'
    '        return ""\n'
    '    if _corporate_spread_ge(credit_stress_score_raw, 6):\n'
    '        return "Strongly supportive for risk assets / credit / equities"\n'
    '    if _corporate_spread_ge(credit_stress_score_raw, 2):\n'
    '        return "Mildly supportive for risk assets"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -6):\n'
    '        return "Strong headwind for risk assets / credit / equities"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -2):\n'
    '        return "Mild headwind for risk assets"\n'
    '    return "Neutral corporate credit-spread risk bias"\n'
    '\n'
    '\n'
    'def _corporate_spread_shock_label(f: Any) -> str:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if f >= 0.75:\n'
    '        return "Large corporate spread widening shock"\n'
    '    if f >= 0.40:\n'
    '        return "Moderate corporate spread widening"\n'
    '    if f <= -0.75:\n'
    '        return "Large corporate spread narrowing"\n'
    '    if f <= -0.40:\n'
    '        return "Moderate corporate spread narrowing"\n'
    '    return "No major corporate spread shock"\n'
    '\n'
    '\n'
    'def _corporate_spread_recovery_score(f: Any, g: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (f, g)):\n'
    '        return ""\n'
    '    if f < 0 and g < 0:\n'
    '        return 5\n'
    '    if f > 0 and g > 0:\n'
    '        return -5\n'
    '    if f < 0 and g > 0:\n'
    '        return 3\n'
    '    if f > 0 and g < 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _corporate_spread_shock_score(f: Any) -> Any:\n'
    '    if is_blank(f):\n'
    '        return ""\n'
    '    if f >= 0.75:\n'
    '        return -6\n'
    '    if f >= 0.40:\n'
    '        return -4\n'
    '    if f <= -0.75:\n'
    '        return 6\n'
    '    if f <= -0.40:\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _corporate_spread_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return -6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return -4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return 6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _corporate_spread_conditions_regime(credit_stress_score_raw: Any) -> str:\n'
    '    if is_blank(credit_stress_score_raw):\n'
    '        return ""\n'
    '    if _corporate_spread_ge(credit_stress_score_raw, 2):\n'
    '        return "Credit conditions easing / supportive"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -2):\n'
    '        return "Credit conditions tightening / stressful"\n'
    '    return "Neutral credit conditions"\n'
    '\n'
    '\n'
    'def _corporate_spread_momentum_label(\n'
    '    credit_stress_score_raw: Any, leading_score_raw: Any\n'
    ') -> str:\n'
    '    if any(is_blank(x) for x in (credit_stress_score_raw, leading_score_raw)):\n'
    '        return ""\n'
    '    if _corporate_spread_ge(credit_stress_score_raw, 2) and _corporate_spread_ge(\n'
    '        leading_score_raw, 2\n'
    '    ):\n'
    '        return "Corporate credit conditions easing"\n'
    '    if _corporate_spread_ge(credit_stress_score_raw, 2) and _corporate_spread_le(\n'
    '        leading_score_raw, -2\n'
    '    ):\n'
    '        return "Credit spreads tight but widening risk"\n'
    '    if (\n'
    '        _corporate_spread_ge(credit_stress_score_raw, 2)\n'
    '        and _corporate_spread_gt(leading_score_raw, -2)\n'
    '        and _corporate_spread_lt(leading_score_raw, 2)\n'
    '    ):\n'
    '        return "Credit conditions supportive / stable"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -2) and _corporate_spread_ge(\n'
    '        leading_score_raw, 2\n'
    '    ):\n'
    '        return "Credit stress easing"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -2) and _corporate_spread_le(\n'
    '        leading_score_raw, -2\n'
    '    ):\n'
    '        return "Corporate credit stress building"\n'
    '    if (\n'
    '        _corporate_spread_le(credit_stress_score_raw, -2)\n'
    '        and _corporate_spread_gt(leading_score_raw, -2)\n'
    '        and _corporate_spread_lt(leading_score_raw, 2)\n'
    '    ):\n'
    '        return "Credit stress elevated / stable"\n'
    '    if (\n'
    '        _corporate_spread_gt(credit_stress_score_raw, -2)\n'
    '        and _corporate_spread_lt(credit_stress_score_raw, 2)\n'
    '        and _corporate_spread_ge(leading_score_raw, 2)\n'
    '    ):\n'
    '        return "Credit-spread impulse easing"\n'
    '    if (\n'
    '        _corporate_spread_gt(credit_stress_score_raw, -2)\n'
    '        and _corporate_spread_lt(credit_stress_score_raw, 2)\n'
    '        and _corporate_spread_le(leading_score_raw, -2)\n'
    '    ):\n'
    '        return "Credit-spread impulse tightening"\n'
    '    return "Corporate credit spreads broadly neutral"\n'
    '\n'
    '\n'
    'def _corporate_spread_signal_interpretation(leading_score_raw: Any, aa: Any) -> str:\n'
    '    if any(is_blank(x) for x in (leading_score_raw, aa)):\n'
    '        return ""\n'
    '    strong_easing = _corporate_spread_ge(\n'
    '        leading_score_raw, 6\n'
    '    ) or _corporate_spread_at_or_above_sheets_boundary(leading_score_raw, 6, 2)\n'
    '    if strong_easing and _corporate_spread_ge(aa, 0):\n'
    '        return (\n'
    '            "Strong corporate-credit easing signal with supportive risk-asset profile"\n'
    '        )\n'
    '    if strong_easing and _corporate_spread_lt(aa, 0):\n'
    '        return "Strong corporate-credit easing signal, but stress-risk profile remains mixed"\n'
    '    if _corporate_spread_ge(leading_score_raw, 2) and _corporate_spread_ge(aa, 0):\n'
    '        return "Moderately positive corporate-credit easing signal"\n'
    '    if _corporate_spread_ge(leading_score_raw, 2) and _corporate_spread_lt(aa, 0):\n'
    '        return "Corporate credit conditions easing, but stress risk remains fragile"\n'
    '    if (\n'
    '        _corporate_spread_gt(leading_score_raw, -2)\n'
    '        and _corporate_spread_gt(aa, -3)\n'
    '        and _corporate_spread_lt(aa, 3)\n'
    '    ):\n'
    '        return "Neutral/mixed corporate credit-spread signal"\n'
    '    if _corporate_spread_le(leading_score_raw, -6) and _corporate_spread_le(aa, 0):\n'
    '        return "Severe corporate-credit stress signal with downside confirmation"\n'
    '    if _corporate_spread_le(leading_score_raw, -6) and _corporate_spread_gt(aa, 0):\n'
    '        return "Severe corporate-credit stress signal, but easing support remains partly intact"\n'
    '    if _corporate_spread_le(leading_score_raw, -2) and _corporate_spread_le(aa, 0):\n'
    '        return "Moderately negative corporate-credit stress signal"\n'
    '    if _corporate_spread_le(leading_score_raw, -2) and _corporate_spread_gt(aa, 0):\n'
    '        return "Corporate-credit stress building, but risk-asset support remains mixed"\n'
    '    return "Mixed/transition corporate credit-spread signal"\n'
    '\n'
    '\n'
    'def _corporate_spread_warning_label(\n'
    '    aa: Any, credit_stress_score_raw: Any, w: Any, x: Any\n'
    ') -> str:\n'
    '    if any(is_blank(v) for v in (aa, credit_stress_score_raw, w, x)):\n'
    '        return ""\n'
    '    if _corporate_spread_le(x, -6):\n'
    '        return "High corporate-credit extreme stress warning"\n'
    '    if _corporate_spread_le(x, -4):\n'
    '        return "Moderate corporate-credit extreme stress warning"\n'
    '    if _corporate_spread_le(w, -6):\n'
    '        return "High corporate-spread widening shock"\n'
    '    if _corporate_spread_le(w, -4):\n'
    '        return "Moderate corporate-spread widening shock"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -6):\n'
    '        return "High corporate-credit stress warning"\n'
    '    if _corporate_spread_le(credit_stress_score_raw, -4):\n'
    '        return "Moderate corporate-credit stress warning"\n'
    '    if _corporate_spread_ge(aa, 6):\n'
    '        return "High corporate-credit easing signal"\n'
    '    if _corporate_spread_ge(aa, 3):\n'
    '        return "Moderate corporate-credit easing signal"\n'
    '    return "Neutral corporate-credit warning profile"\n'
    '\n'
    '\n'
    'def _corporate_spread_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme corporate credit stress"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong corporate credit stress"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme corporate spread easing"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong corporate spread easing"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_corporate_credit_spreads_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    source_values = [r.get("Raw Value", r.get("B", "")) for r in records]\n'
    '    raw_values = [to_float_or_none(v) for v in source_values]\n'
    '    ma_values = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 3:\n'
    '            ma_values.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            ma_values.append(_corporate_spread_mean_sheets(window) if window else "")\n'
    '    c_counts = _metric_counts(ma_values)\n'
    '    observation_counts = c_counts\n'
    '    one_change = []\n'
    '    three_change = []\n'
    '    six_change = []\n'
    '    twelve_change = []\n'
    '    for idx, c in enumerate(ma_values):\n'
    '        d = observation_counts[idx]\n'
    '        c1 = ma_values[idx - 1] if idx >= 1 else ""\n'
    '        c3 = ma_values[idx - 3] if idx >= 3 else ""\n'
    '        c6 = ma_values[idx - 6] if idx >= 6 else ""\n'
    '        c12 = ma_values[idx - 12] if idx >= 12 else ""\n'
    '        one_change.append(\n'
    '            c - c1 if d >= 2 and not is_blank(c) and not is_blank(c1) else ""\n'
    '        )\n'
    '        three_change.append(\n'
    '            c - c3 if d >= 4 and not is_blank(c) and not is_blank(c3) else ""\n'
    '        )\n'
    '        six_change.append(\n'
    '            c - c6 if d >= 7 and not is_blank(c) and not is_blank(c6) else ""\n'
    '        )\n'
    '        twelve_change.append(\n'
    '            c - c12 if d >= 13 and not is_blank(c) and not is_blank(c12) else ""\n'
    '        )\n'
    '    f_counts = _metric_counts(three_change)\n'
    '    g_counts = _metric_counts(six_change)\n'
    '    h_counts = _metric_counts(twelve_change)\n'
    '    c_ranks = _corporate_spread_metric_ranks_sheets(ma_values, 60)\n'
    '    f_ranks = _corporate_spread_metric_ranks_sheets(three_change, 60)\n'
    '    g_ranks = _corporate_spread_metric_ranks_sheets(six_change, 60)\n'
    '    h_ranks = _corporate_spread_metric_ranks_sheets(twelve_change, 60)\n'
    '    c_z = _rounded_expanding_zscores(ma_values, c_counts, 60)\n'
    '    f_z = _rounded_expanding_zscores(three_change, f_counts, 60)\n'
    '    g_z = _rounded_expanding_zscores(six_change, g_counts, 60)\n'
    '    h_z = _rounded_expanding_zscores(twelve_change, h_counts, 60)\n'
    '    rows = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = ma_values[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_change[idx]\n'
    '        f = three_change[idx]\n'
    '        g = six_change[idx]\n'
    '        h = twelve_change[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _corporate_spread_inverted_score(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _corporate_spread_inverted_score(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _corporate_spread_inverted_score(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _corporate_spread_inverted_score(o)\n'
    '        credit_stress_score_raw = _corporate_spread_credit_stress_score_raw(j, l, n, p)\n'
    '        t = (\n'
    '            ""\n'
    '            if is_blank(credit_stress_score_raw)\n'
    '            else _corporate_spread_sheets_round(credit_stress_score_raw, 2)\n'
    '        )\n'
    '        credit_spread_impulse_score_raw = _corporate_spread_impulse_score_raw(\n'
    '            j, l, n, p\n'
    '        )\n'
    '        u = (\n'
    '            ""\n'
    '            if is_blank(credit_spread_impulse_score_raw)\n'
    '            else _corporate_spread_sheets_round(credit_spread_impulse_score_raw, 2)\n'
    '        )\n'
    '        v = _corporate_spread_recovery_score(f, g)\n'
    '        w = _corporate_spread_shock_score(f)\n'
    '        ah = c_z[idx]\n'
    '        ai = f_z[idx]\n'
    '        aj = g_z[idx]\n'
    '        ak = h_z[idx]\n'
    '        x = _corporate_spread_outlier_score([ah, ai, aj, ak])\n'
    '        y = t\n'
    '        leading_score_raw = _corporate_spread_leading_score_raw(\n'
    '            credit_stress_score_raw, credit_spread_impulse_score_raw, v\n'
    '        )\n'
    '        leading_score_output_raw = (\n'
    '            clamp(0.45 * t + 0.35 * u + 0.20 * v, -10, 10)\n'
    '            if all(not is_blank(value) for value in (t, u, v))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            ""\n'
    '            if is_blank(leading_score_output_raw)\n'
    '            else _corporate_spread_sheets_round(leading_score_output_raw, 2)\n'
    '        )\n'
    '        warning_score_raw = _corporate_spread_warning_score_raw(\n'
    '            credit_stress_score_raw, v, w, x\n'
    '        )\n'
    '        warning_score_output_raw = _corporate_spread_warning_score_raw(t, v, w, x)\n'
    '        aa = (\n'
    '            ""\n'
    '            if is_blank(warning_score_output_raw)\n'
    '            else _corporate_spread_sheets_round(warning_score_output_raw, 2)\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3-Period Moving Average Corporate Credit Spread": c,\n'
    '                "Corporate Credit Spread Observation Count": d,\n'
    '                "1-Period Corporate Credit Spread Change": e,\n'
    '                "3-Period Corporate Credit Spread Change": f,\n'
    '                "6-Period Corporate Credit Spread Change": g,\n'
    '                "12-Period Corporate Credit Spread Change": h,\n'
    '                "Corporate Credit Spread Level PercentRank": _corporate_spread_percent_display(\n'
    '                    i\n'
    '                ),\n'
    '                "Corporate Credit Spread Level Score": j,\n'
    '                "3-Period Corporate Credit Spread Change PercentRank": _corporate_spread_percent_display(\n'
    '                    k\n'
    '                ),\n'
    '                "3-Period Corporate Credit Spread Change Score": l,\n'
    '                "6-Period Corporate Credit Spread Change PercentRank": _corporate_spread_percent_display(\n'
    '                    m\n'
    '                ),\n'
    '                "6-Period Corporate Credit Spread Change Score": n,\n'
    '                "12-Period Corporate Credit Spread Change PercentRank": _corporate_spread_percent_display(\n'
    '                    o\n'
    '                ),\n'
    '                "12-Period Corporate Credit Spread Change Score": p,\n'
    '                "Corporate Credit Spread Regime": _corporate_spread_regime(c, e, f),\n'
    '                "Corporate Credit Spread Asset Bias Label": _corporate_spread_asset_bias_label(\n'
    '                    credit_stress_score_raw\n'
    '                ),\n'
    '                "Corporate Credit Spread Shock Label": _corporate_spread_shock_label(f),\n'
    '                "Corporate Credit Spread Leading Composite Score": t,\n'
    '                "Corporate Credit Spread Momentum Composite Score": u,\n'
    '                "Corporate Credit Spread Recovery / Deterioration Score": v,\n'
    '                "Corporate Credit Spread Shock Score": w,\n'
    '                "Corporate Credit Spread Outlier Score": x,\n'
    '                "Corporate Credit Spread Leading Score": y,\n'
    '                "Corporate Credit Spread Warning Composite Score": z,\n'
    '                "Corporate Credit Spread Final Warning Score": aa,\n'
    '                "Corporate Credit Spread Conditions Regime": _corporate_spread_conditions_regime(\n'
    '                    credit_stress_score_raw\n'
    '                ),\n'
    '                "Corporate Credit Spread Momentum Label": _corporate_spread_momentum_label(\n'
    '                    credit_stress_score_raw, leading_score_raw\n'
    '                ),\n'
    '                "Corporate Credit Spread Signal Interpretation": _corporate_spread_signal_interpretation(\n'
    '                    leading_score_raw, warning_score_raw\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Corporate Credit Spread Warning Risk Label": _corporate_spread_warning_label(\n'
    '                    warning_score_raw, credit_stress_score_raw, w, x\n'
    '                ),\n'
    '                "Corporate Credit Spread Level Z-Score": ah,\n'
    '                "3-Period Corporate Credit Spread Change Z-Score": ai,\n'
    '                "6-Period Corporate Credit Spread Change Z-Score": aj,\n'
    '                "12-Period Corporate Credit Spread Change Z-Score": ak,\n'
    '                "Corporate Credit Spread Outlier Label": _corporate_spread_outlier_label(\n'
    '                    [ah, ai, aj, ak]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _real_gov_spending_regime(c: Any, e: Any, f: Any, g: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if not any(is_blank(x) for x in (e, f, g)) and e > 0 and f > 0 and g > 0:\n'
    '        return "Real government spending expansion regime"\n'
    '    if not any(is_blank(x) for x in (e, f, g)) and e < 0 and f < 0 and g < 0:\n'
    '        return "Real government spending contraction regime"\n'
    '    if not any(is_blank(x) for x in (e, f, g)) and e > 0 and f > 0 and g < 0:\n'
    '        return "Early government spending recovery"\n'
    '    if not any(is_blank(x) for x in (e, f, g)) and e < 0 and f < 0 and g > 0:\n'
    '        return "Government spending deterioration from high base"\n'
    '    if not any(is_blank(x) for x in (e, f)) and e > 0 and f < 0:\n'
    '        return "Short-term government spending recovery / mixed impulse"\n'
    '    if not any(is_blank(x) for x in (e, f)) and e < 0 and f > 0:\n'
    '        return "Short-term government spending deterioration / mixed impulse"\n'
    '    return "Neutral/mixed government spending regime"\n'
    '\n'
    '\n'
    'def _real_gov_asset_bias_label(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 6:\n'
    '        return "Strong fiscal-demand support for GDP growth"\n'
    '    if z >= 2:\n'
    '        return "Mild fiscal-demand support for GDP growth"\n'
    '    if z <= -6:\n'
    '        return "Strong fiscal-demand drag on GDP growth"\n'
    '    if z <= -2:\n'
    '        return "Mild fiscal-demand drag on GDP growth"\n'
    '    return "Neutral fiscal-demand bias"\n'
    '\n'
    '\n'
    'def _real_gov_shock_label(h: Any) -> str:\n'
    '    if is_blank(h):\n'
    '        return ""\n'
    '    if h >= 0.10:\n'
    '        return "Large quarterly government-spending surge"\n'
    '    if h >= 0.05:\n'
    '        return "Moderate quarterly government-spending acceleration"\n'
    '    if h <= -0.10:\n'
    '        return "Large quarterly government-spending contraction"\n'
    '    if h <= -0.05:\n'
    '        return "Moderate quarterly government-spending slowdown"\n'
    '    return "No major government-spending shock"\n'
    '\n'
    '\n'
    'def _real_gov_recovery_score(e: Any, f: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (e, f)):\n'
    '        return ""\n'
    '    if e > 0 and f > 0:\n'
    '        return 5\n'
    '    if e < 0 and f < 0:\n'
    '        return -5\n'
    '    if e > 0 and f < 0:\n'
    '        return 3\n'
    '    if e < 0 and f > 0:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_gov_shock_score(h: Any) -> Any:\n'
    '    if is_blank(h):\n'
    '        return ""\n'
    '    if h >= 0.10:\n'
    '        return 6\n'
    '    if h >= 0.05:\n'
    '        return 4\n'
    '    if h <= -0.10:\n'
    '        return -6\n'
    '    if h <= -0.05:\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_gov_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _real_gov_conditions_regime(z: Any) -> str:\n'
    '    if is_blank(z):\n'
    '        return ""\n'
    '    if z >= 2:\n'
    '        return "Real government spending growth improving"\n'
    '    if z <= -2:\n'
    '        return "Real government spending growth deteriorating"\n'
    '    return "Neutral real government spending growth"\n'
    '\n'
    '\n'
    'def _real_gov_momentum_label(z: Any, aa: Any) -> str:\n'
    '    if any(is_blank(x) for x in (z, aa)):\n'
    '        return ""\n'
    '    if z >= 2 and aa >= 2:\n'
    '        return "Real government spending growth improving"\n'
    '    if z >= 2 and aa <= -2:\n'
    '        return "Government spending growth supportive but fading"\n'
    '    if z >= 2 and -2 < aa < 2:\n'
    '        return "Government spending growth supportive / stable"\n'
    '    if z <= -2 and aa >= 2:\n'
    '        return "Government spending drag easing"\n'
    '    if z <= -2 and aa <= -2:\n'
    '        return "Real government spending contraction drag"\n'
    '    if z <= -2 and -2 < aa < 2:\n'
    '        return "Government spending drag stable"\n'
    '    if -2 < z < 2 and aa >= 2:\n'
    '        return "Government spending impulse improving"\n'
    '    if -2 < z < 2 and aa <= -2:\n'
    '        return "Government spending impulse deteriorating"\n'
    '    return "Real government spending broadly neutral"\n'
    '\n'
    '\n'
    'def _real_gov_signal_interpretation(aa: Any, ab: Any) -> str:\n'
    '    if any(is_blank(x) for x in (aa, ab)):\n'
    '        return ""\n'
    '    if aa >= 6 and ab >= 0:\n'
    '        return "Strong positive government-spending impulse with supportive fiscal-demand profile"\n'
    '    if aa >= 6 and ab < 0:\n'
    '        return "Strong positive government-spending impulse, but shock/turn profile is mixed"\n'
    '    if aa >= 2 and ab >= 0:\n'
    '        return "Moderately positive government-spending impulse signal"\n'
    '    if aa >= 2 and ab < 0:\n'
    '        return "Government-spending impulse improving, but near-term risk profile is fragile"\n'
    '    if aa > -2 and ab > -3 and ab < 3:\n'
    '        return "Neutral/mixed government-spending signal"\n'
    '    if aa <= -6 and ab <= 0:\n'
    '        return (\n'
    '            "Strong government-spending contraction signal with downside confirmation"\n'
    '        )\n'
    '    if aa <= -6 and ab > 0:\n'
    '        return "Strong government-spending contraction signal, but fiscal-demand support remains partly intact"\n'
    '    if aa <= -2 and ab <= 0:\n'
    '        return "Moderately negative government-spending impulse signal"\n'
    '    if aa <= -2 and ab > 0:\n'
    '        return "Government-spending impulse deteriorating, but support profile remains mixed"\n'
    '    return "Mixed/transition government-spending signal"\n'
    '\n'
    '\n'
    'def _real_gov_warning_label(ab: Any) -> str:\n'
    '    if is_blank(ab):\n'
    '        return ""\n'
    '    if ab >= 6:\n'
    '        return "High government-spending expansion/shock warning"\n'
    '    if ab >= 3:\n'
    '        return "Moderate government-spending expansion/shock warning"\n'
    '    if ab > -3:\n'
    '        return "Neutral government-spending warning profile"\n'
    '    if ab > -6:\n'
    '        return "Moderate government-spending contraction warning"\n'
    '    return "High government-spending contraction warning"\n'
    '\n'
    '\n'
    'def _real_gov_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme government-spending expansion"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong government-spending expansion"\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme government-spending contraction"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong government-spending contraction"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_real_government_spending_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    source_values = [\n'
    '        r.get("Raw Value", r.get("B", r.get("Value", ""))) for r in records\n'
    '    ]\n'
    '    moving_averages: list[Any] = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 2:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 1) : idx + 1] if v is not None]\n'
    '            moving_averages.append(sum(window) / len(window) if window else "")\n'
    '    observation_counts = _metric_counts(moving_averages)\n'
    '    one_growth: list[Any] = []\n'
    '    two_growth: list[Any] = []\n'
    '    four_growth: list[Any] = []\n'
    '    one_annualized: list[Any] = []\n'
    '    two_annualized: list[Any] = []\n'
    '    for idx, c in enumerate(moving_averages):\n'
    '        d = observation_counts[idx]\n'
    '        c1 = moving_averages[idx - 1] if idx >= 1 else ""\n'
    '        c2 = moving_averages[idx - 2] if idx >= 2 else ""\n'
    '        c4 = moving_averages[idx - 4] if idx >= 4 else ""\n'
    '        e = _broad_money_growth(c, c1, d >= 2)\n'
    '        f = _broad_money_growth(c, c2, d >= 3)\n'
    '        g = _broad_money_growth(c, c4, d >= 5)\n'
    '        one_growth.append(e)\n'
    '        two_growth.append(f)\n'
    '        four_growth.append(g)\n'
    '        one_annualized.append(_broad_money_annualized_growth(e, 4))\n'
    '        two_annualized.append(_broad_money_annualized_growth(f, 2))\n'
    '    e_counts = _metric_counts(one_growth)\n'
    '    f_counts = _metric_counts(two_growth)\n'
    '    g_counts = _metric_counts(four_growth)\n'
    '    h_counts = _metric_counts(one_annualized)\n'
    '    e_ranks = _expectations_metric_ranks(one_growth, 40)\n'
    '    f_ranks = _expectations_metric_ranks(two_growth, 40)\n'
    '    g_ranks = _expectations_metric_ranks(four_growth, 40)\n'
    '    h_ranks = _expectations_metric_ranks(one_annualized, 40)\n'
    '    e_zscores = _rounded_expanding_zscores(one_growth, e_counts, 40)\n'
    '    f_zscores = _rounded_expanding_zscores(two_growth, f_counts, 40)\n'
    '    g_zscores = _rounded_expanding_zscores(four_growth, g_counts, 40)\n'
    '    h_zscores = _rounded_expanding_zscores(one_annualized, h_counts, 40)\n'
    '    momentum_scores: list[Any] = []\n'
    '    provisional: list[dict[str, Any]] = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_growth[idx]\n'
    '        f = two_growth[idx]\n'
    '        g = four_growth[idx]\n'
    '        h = one_annualized[idx]\n'
    '        i = two_annualized[idx]\n'
    '        j = e_ranks[idx]\n'
    '        k = _score_percent_rank(j)\n'
    '        l = f_ranks[idx]\n'
    '        m = _score_percent_rank(l)\n'
    '        n = g_ranks[idx]\n'
    '        o = _score_percent_rank(n)\n'
    '        p = h_ranks[idx]\n'
    '        q = _score_percent_rank(p)\n'
    '        u = (\n'
    '            round(clamp(0.25 * k + 0.35 * m + 0.25 * o + 0.15 * q, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (k, m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        momentum_scores.append(u)\n'
    '        ai = e_zscores[idx]\n'
    '        aj = f_zscores[idx]\n'
    '        ak = g_zscores[idx]\n'
    '        al = h_zscores[idx]\n'
    '        y = _real_gov_outlier_score([ai, aj, ak, al])\n'
    '        z = (\n'
    '            round(clamp(0.25 * k + 0.35 * m + 0.25 * o + 0.15 * q, -10, 10), 2)\n'
    '            if all(not is_blank(x) for x in (k, m, o, q))\n'
    '            else ""\n'
    '        )\n'
    '        prev_u = momentum_scores[idx - 1] if idx >= 1 else ""\n'
    '        v = (\n'
    '            round(clamp((u - prev_u) * 2, -10, 10), 2)\n'
    '            if not is_blank(u) and not is_blank(prev_u)\n'
    '            else ""\n'
    '        )\n'
    '        w = _real_gov_recovery_score(e, f)\n'
    '        x = _real_gov_shock_score(h)\n'
    '        aa = (\n'
    '            round(clamp(0.50 * u + 0.30 * v + 0.20 * w, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (u, v, w))\n'
    '            else ""\n'
    '        )\n'
    '        ab = (\n'
    '            round(clamp(0.25 * v + 0.25 * w + 0.30 * x + 0.20 * y, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (v, w, x, y))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 40\n'
    '                else ("Medium reliability" if d < 80 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 40\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (z, aa, ab))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        provisional.append(\n'
    '            {\n'
    '                "2-Period Moving Average Real Government Spending": c,\n'
    '                "Real Government Spending Observation Count": d,\n'
    '                "1-Period Real Government Spending Growth": e,\n'
    '                "2-Period Real Government Spending Growth": f,\n'
    '                "4-Period Real Government Spending Growth": g,\n'
    '                "1-Period Annualised Real Government Spending Growth": h,\n'
    '                "2-Period Annualised Real Government Spending Growth": i,\n'
    '                "1-Period Real Government Spending Growth PercentRank": j,\n'
    '                "1-Period Real Government Spending Growth Score": k,\n'
    '                "2-Period Real Government Spending Growth PercentRank": l,\n'
    '                "2-Period Real Government Spending Growth Score": m,\n'
    '                "4-Period Real Government Spending Growth PercentRank": n,\n'
    '                "4-Period Real Government Spending Growth Score": o,\n'
    '                "Annualised Real Government Spending Growth PercentRank": p,\n'
    '                "Annualised Real Government Spending Growth Score": q,\n'
    '                "Real Government Spending Regime": _real_gov_spending_regime(\n'
    '                    c, e, f, g\n'
    '                ),\n'
    '                "Real Government Spending Asset Bias Label": _real_gov_asset_bias_label(\n'
    '                    z\n'
    '                ),\n'
    '                "Real Government Spending Shock Label": _real_gov_shock_label(h),\n'
    '                "Real Government Spending Momentum Composite Score": u,\n'
    '                "Real Government Spending Momentum Turn Score": v,\n'
    '                "Real Government Spending Recovery / Deterioration Score": w,\n'
    '                "Real Government Spending Shock Score": x,\n'
    '                "Real Government Spending Outlier Score": y,\n'
    '                "Real Government Spending Leading Score": z,\n'
    '                "Real Government Spending Warning Composite Score": aa,\n'
    '                "Real Government Spending Final Warning Score": ab,\n'
    '                "Real Government Spending Conditions Regime": _real_gov_conditions_regime(\n'
    '                    z\n'
    '                ),\n'
    '                "Real Government Spending Momentum Label": _real_gov_momentum_label(\n'
    '                    z, aa\n'
    '                ),\n'
    '                "Real Government Spending Signal Interpretation": _real_gov_signal_interpretation(\n'
    '                    aa, ab\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Real Government Spending Warning Risk Label": _real_gov_warning_label(\n'
    '                    ab\n'
    '                ),\n'
    '                "1-Period Real Government Spending Growth Z-Score": ai,\n'
    '                "2-Period Real Government Spending Growth Z-Score": aj,\n'
    '                "4-Period Real Government Spending Growth Z-Score": ak,\n'
    '                "Annualised Real Government Spending Growth Z-Score": al,\n'
    '                "Real Government Spending Outlier Label": _real_gov_outlier_label(\n'
    '                    [ai, aj, ak, al]\n'
    '                ),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(provisional, REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def _fiscal_balance_regime(c: Any, f: Any) -> str:\n'
    '    if is_blank(c):\n'
    '        return ""\n'
    '    if is_blank(f):\n'
    '        return "Mixed fiscal balance regime"\n'
    '    if c >= 0 and f > 0:\n'
    '        return "Surplus and improving"\n'
    '    if c >= 0 and f <= 0:\n'
    '        return "Surplus but weakening"\n'
    '    if c < 0 and c >= -3 and f > 0:\n'
    '        return "Moderate deficit but improving"\n'
    '    if c < 0 and c >= -3 and f <= 0:\n'
    '        return "Moderate deficit and deteriorating"\n'
    '    if c < -3 and c >= -6 and f > 0:\n'
    '        return "Large deficit but improving"\n'
    '    if c < -3 and c >= -6 and f <= 0:\n'
    '        return "Large deficit and deteriorating"\n'
    '    if c < -6 and f > 0:\n'
    '        return "Severe deficit but improving"\n'
    '    if c < -6 and f <= 0:\n'
    '        return "Severe deficit and deteriorating"\n'
    '    return "Mixed fiscal balance regime"\n'
    '\n'
    '\n'
    'def _fiscal_impulse_bias_label(q: Any) -> str:\n'
    '    if is_blank(q):\n'
    '        return ""\n'
    '    if q >= 6:\n'
    '        return "Strong fiscal expansion impulse"\n'
    '    if q >= 2:\n'
    '        return "Mild fiscal expansion impulse"\n'
    '    if q <= -6:\n'
    '        return "Strong fiscal consolidation impulse"\n'
    '    if q <= -2:\n'
    '        return "Mild fiscal consolidation impulse"\n'
    '    return "Neutral fiscal impulse"\n'
    '\n'
    '\n'
    'def _fiscal_interaction_score(af: Any, q: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (af, q)):\n'
    '        return ""\n'
    '    if af < -1 and q >= 5:\n'
    '        return 5\n'
    '    if af < -0.5 and q >= 3:\n'
    '        return 3\n'
    '    if af > 1 and q <= -5:\n'
    '        return -3\n'
    '    if af > 0.5 and q <= -3:\n'
    '        return -2\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _fiscal_sustainability_risk_score(c: Any, af: Any, q: Any) -> Any:\n'
    '    if any(is_blank(x) for x in (c, af, q)):\n'
    '        return ""\n'
    '    if c < -8 and q >= 6:\n'
    '        return -7\n'
    '    if c < -6 and q >= 5:\n'
    '        return -5\n'
    '    if c < -4 and q >= 4:\n'
    '        return -3\n'
    '    if af < -1.5 and q >= 5:\n'
    '        return -3\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _fiscal_outlier_score(zs: Sequence[Any]) -> Any:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return -6\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return -4\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return 6\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return 4\n'
    '    return 0\n'
    '\n'
    '\n'
    'def _fiscal_impulse_regime(w: Any) -> str:\n'
    '    if is_blank(w):\n'
    '        return ""\n'
    '    if w >= 2:\n'
    '        return "Fiscal impulse expansionary"\n'
    '    if w <= -2:\n'
    '        return "Fiscal impulse contractionary"\n'
    '    return "Neutral fiscal impulse"\n'
    '\n'
    '\n'
    'def _fiscal_momentum_label(w: Any, x: Any) -> str:\n'
    '    if any(is_blank(v) for v in (w, x)):\n'
    '        return ""\n'
    '    if w >= 2 and x >= 2:\n'
    '        return "Fiscal impulse expansionary"\n'
    '    if w >= 2 and x <= -2:\n'
    '        return "Fiscal impulse supportive but fading"\n'
    '    if w >= 2 and -2 < x < 2:\n'
    '        return "Fiscal impulse supportive / stable"\n'
    '    if w <= -2 and x >= 2:\n'
    '        return "Fiscal drag easing"\n'
    '    if w <= -2 and x <= -2:\n'
    '        return "Fiscal consolidation drag"\n'
    '    if w <= -2 and -2 < x < 2:\n'
    '        return "Fiscal drag stable"\n'
    '    if -2 < w < 2 and x >= 2:\n'
    '        return "Fiscal impulse improving"\n'
    '    if -2 < w < 2 and x <= -2:\n'
    '        return "Fiscal impulse deteriorating"\n'
    '    return "Fiscal impulse broadly neutral"\n'
    '\n'
    '\n'
    'def _fiscal_signal_interpretation(x: Any, y: Any, u: Any) -> str:\n'
    '    if any(is_blank(v) for v in (x, y, u)):\n'
    '        return ""\n'
    '    if x >= 6 and u <= -5:\n'
    '        return "Strong expansionary fiscal impulse, but severe deficit/sustainability risk is rising"\n'
    '    if x >= 6 and y >= 0:\n'
    '        return "Strong expansionary fiscal impulse with supportive growth profile"\n'
    '    if x >= 2 and u <= -3:\n'
    '        return "Fiscal impulse improving, but deficit/sustainability risk remains"\n'
    '    if x >= 2 and y >= 0:\n'
    '        return "Moderately positive fiscal impulse signal"\n'
    '    if x > -2 and y > -3 and y < 3:\n'
    '        return "Neutral/mixed fiscal impulse signal"\n'
    '    if x <= -6 and y <= 0:\n'
    '        return "Strong fiscal consolidation drag with downside confirmation"\n'
    '    if x <= -6 and y > 0:\n'
    '        return "Strong consolidation signal, but fiscal support remains partly intact"\n'
    '    if x <= -2 and y <= 0:\n'
    '        return "Moderately negative fiscal impulse / consolidation signal"\n'
    '    if x <= -2 and y > 0:\n'
    '        return "Fiscal consolidation signal, but growth support remains mixed"\n'
    '    return "Mixed/transition fiscal impulse signal"\n'
    '\n'
    '\n'
    'def _fiscal_warning_label(y: Any, u: Any) -> str:\n'
    '    if any(is_blank(v) for v in (y, u)):\n'
    '        return ""\n'
    '    if u <= -6:\n'
    '        return "High deficit/sustainability warning"\n'
    '    if u <= -3:\n'
    '        return "Moderate deficit/sustainability warning"\n'
    '    if y >= 6:\n'
    '        return "High expansionary fiscal-impulse warning"\n'
    '    if y >= 3:\n'
    '        return "Moderate expansionary fiscal-impulse warning"\n'
    '    if y > -3:\n'
    '        return "Neutral fiscal warning profile"\n'
    '    if y > -6:\n'
    '        return "Moderate fiscal-drag warning"\n'
    '    return "High fiscal-drag warning"\n'
    '\n'
    '\n'
    'def _fiscal_outlier_label(zs: Sequence[Any]) -> str:\n'
    '    if any(is_blank(z) for z in zs):\n'
    '        return ""\n'
    '    if any(z <= -3 for z in zs):\n'
    '        return "Extreme fiscal-deficit stress"\n'
    '    if any(z <= -2 for z in zs):\n'
    '        return "Strong fiscal-deficit stress"\n'
    '    if any(z >= 3 for z in zs):\n'
    '        return "Extreme fiscal-balance strength"\n'
    '    if any(z >= 2 for z in zs):\n'
    '        return "Strong fiscal-balance strength"\n'
    '    return "Normal range"\n'
    '\n'
    '\n'
    'def calculate_fiscal_balance_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    source_values = [\n'
    '        r.get("Raw Value", r.get("B", r.get("Value", ""))) for r in records\n'
    '    ]\n'
    '\n'
    '    moving_averages: list[Any] = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [\n'
    '                value\n'
    '                for value in raw_values[max(0, idx - 2) : idx + 1]\n'
    '                if value is not None\n'
    '            ]\n'
    '            moving_averages.append(_broad_money_mean_sheets(window))\n'
    '\n'
    '    observation_counts = _metric_counts(moving_averages)\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    five_period_changes: list[Any] = []\n'
    '    for idx, c in enumerate(moving_averages):\n'
    '        d = observation_counts[idx]\n'
    '        c1 = moving_averages[idx - 1] if idx >= 1 else ""\n'
    '        c3 = moving_averages[idx - 3] if idx >= 3 else ""\n'
    '        c5 = moving_averages[idx - 5] if idx >= 5 else ""\n'
    '        one_period_changes.append(\n'
    '            c - c1 if d >= 2 and not is_blank(c) and not is_blank(c1) else ""\n'
    '        )\n'
    '        three_period_changes.append(\n'
    '            c - c3 if d >= 4 and not is_blank(c) and not is_blank(c3) else ""\n'
    '        )\n'
    '        five_period_changes.append(\n'
    '            c - c5 if d >= 6 and not is_blank(c) and not is_blank(c5) else ""\n'
    '        )\n'
    '\n'
    '    c_counts = _metric_counts(moving_averages)\n'
    '    e_counts = _metric_counts(one_period_changes)\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(five_period_changes)\n'
    '    c_ranks = _expectations_metric_ranks(moving_averages, 20)\n'
    '    e_ranks = _expectations_metric_ranks(one_period_changes, 20)\n'
    '    f_ranks = _expectations_metric_ranks(three_period_changes, 20)\n'
    '    g_ranks = _expectations_metric_ranks(five_period_changes, 20)\n'
    '    c_zscores = _rounded_expanding_zscores(moving_averages, c_counts, 20)\n'
    '    e_zscores = _rounded_expanding_zscores(one_period_changes, e_counts, 20)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 20)\n'
    '    g_zscores = _rounded_expanding_zscores(five_period_changes, g_counts, 20)\n'
    '\n'
    '    rows = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = five_period_changes[idx]\n'
    '        h = c_ranks[idx]\n'
    '        i = _score_percent_rank(h)\n'
    '        j = e_ranks[idx]\n'
    '        k = _score_percent_rank(j)\n'
    '        l = f_ranks[idx]\n'
    '        m = _score_percent_rank(l)\n'
    '        n = g_ranks[idx]\n'
    '        o = _score_percent_rank(n)\n'
    '        q = (\n'
    '            round(clamp(-0.50 * k - 0.35 * m - 0.15 * o, -10, 10), 2)\n'
    '            if all(not is_blank(v) for v in (k, m, o))\n'
    '            else ""\n'
    '        )\n'
    '        s = (\n'
    '            round(clamp(-0.50 * k - 0.30 * m - 0.20 * o, -10, 10), 2)\n'
    '            if all(not is_blank(v) for v in (k, m, o))\n'
    '            else ""\n'
    '        )\n'
    '        af = c_zscores[idx]\n'
    '        ag = e_zscores[idx]\n'
    '        ah = f_zscores[idx]\n'
    '        ai = g_zscores[idx]\n'
    '        t = _fiscal_interaction_score(af, q)\n'
    '        u = _fiscal_sustainability_risk_score(c, af, q)\n'
    '        v = _fiscal_outlier_score([af, ag, ah, ai])\n'
    '        w = (\n'
    '            round(clamp(0.60 * q + 0.25 * s - 0.15 * i, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (i, q, s))\n'
    '            else ""\n'
    '        )\n'
    '        x = (\n'
    '            round(clamp(0.50 * q + 0.25 * s + 0.15 * t + 0.10 * u, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (q, s, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        y = (\n'
    '            round(clamp(0.25 * s + 0.20 * t + 0.35 * u + 0.20 * v, -10, 10), 2)\n'
    '            if all(not is_blank(val) for val in (s, t, u, v))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 20\n'
    '                else ("Medium reliability" if d < 40 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 20\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (w, x, y))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3-Period Moving Average Fiscal Balance": c,\n'
    '                "Fiscal Balance Observation Count": d,\n'
    '                "1-Period Fiscal Balance Change": e,\n'
    '                "3-Period Fiscal Balance Change": f,\n'
    '                "5-Period Fiscal Balance Change": g,\n'
    '                "Fiscal Balance Level PercentRank": h,\n'
    '                "Fiscal Balance Level Score": i,\n'
    '                "1-Period Fiscal Balance Change PercentRank": j,\n'
    '                "1-Period Fiscal Balance Change Score": k,\n'
    '                "3-Period Fiscal Balance Change PercentRank": l,\n'
    '                "3-Period Fiscal Balance Change Score": m,\n'
    '                "5-Period Fiscal Balance Change PercentRank": n,\n'
    '                "5-Period Fiscal Balance Change Score": o,\n'
    '                "Fiscal Balance Regime": _fiscal_balance_regime(c, f),\n'
    '                "Fiscal Expansion Impulse Composite Score": q,\n'
    '                "Fiscal Impulse Bias Label": _fiscal_impulse_bias_label(q),\n'
    '                "Fiscal Impulse Momentum Score": s,\n'
    '                "Fiscal Balance Interaction Score": t,\n'
    '                "Fiscal Deficit Sustainability Risk Score": u,\n'
    '                "Fiscal Balance Outlier Score": v,\n'
    '                "Fiscal Impulse Leading Score": w,\n'
    '                "Fiscal Impulse Warning Composite Score": x,\n'
    '                "Fiscal Impulse Final Warning Score": y,\n'
    '                "Fiscal Impulse Regime": _fiscal_impulse_regime(w),\n'
    '                "Fiscal Impulse Momentum Label": _fiscal_momentum_label(w, x),\n'
    '                "Fiscal Impulse Signal Interpretation": _fiscal_signal_interpretation(\n'
    '                    x, y, u\n'
    '                ),\n'
    '                "Reliability": reliability,\n'
    '                "Data Status": data_status,\n'
    '                "Fiscal Warning Risk Label": _fiscal_warning_label(y, u),\n'
    '                "Fiscal Balance Level Z-Score": af,\n'
    '                "1-Period Fiscal Balance Change Z-Score": ag,\n'
    '                "3-Period Fiscal Balance Change Z-Score": ah,\n'
    '                "5-Period Fiscal Balance Change Z-Score": ai,\n'
    '                "Fiscal Balance Outlier Label": _fiscal_outlier_label([af, ag, ah, ai]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, FISCAL_BALANCE_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def calculate_broad_money_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    source_values = [\n'
    '        r.get("Raw Value", r.get("B", r.get("Value", ""))) for r in records\n'
    '    ]\n'
    '\n'
    '    moving_averages: list[Any] = []\n'
    '    raw_count = 0\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if current is not None:\n'
    '            raw_count += 1\n'
    '        if raw_count < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [\n'
    '                value\n'
    '                for value in raw_values[max(0, idx - 2) : idx + 1]\n'
    '                if value is not None\n'
    '            ]\n'
    '            moving_averages.append(sum(window) / len(window) if window else "")\n'
    '\n'
    '    observation_counts: list[int] = []\n'
    '    c_count = 0\n'
    '    for c in moving_averages:\n'
    '        if to_float_or_none(c) is not None:\n'
    '            c_count += 1\n'
    '        observation_counts.append(max(0, c_count - 2))\n'
    '\n'
    '    one_period_growth: list[Any] = []\n'
    '    three_period_growth: list[Any] = []\n'
    '    six_period_growth: list[Any] = []\n'
    '    twelve_period_growth: list[Any] = []\n'
    '    annualized_three: list[Any] = []\n'
    '    annualized_six: list[Any] = []\n'
    '    for idx, c in enumerate(moving_averages):\n'
    '        d = observation_counts[idx]\n'
    '\n'
    '        def lag(offset: int) -> Any:\n'
    '            return moving_averages[idx - offset] if idx >= offset else ""\n'
    '\n'
    '        c1, c3, c6, c12 = lag(1), lag(3), lag(6), lag(12)\n'
    '        e = _broad_money_growth(c, c1, d >= 2)\n'
    '        f = _broad_money_growth(c, c3, d >= 4)\n'
    '        g = _broad_money_growth(c, c6, d >= 7)\n'
    '        h = _broad_money_growth(c, c12, d >= 13)\n'
    '        one_period_growth.append(e)\n'
    '        three_period_growth.append(f)\n'
    '        six_period_growth.append(g)\n'
    '        twelve_period_growth.append(h)\n'
    '        annualized_three.append(_broad_money_annualized_growth(f, 4))\n'
    '        annualized_six.append(_broad_money_annualized_growth(g, 2))\n'
    '\n'
    '    c_counts = _metric_counts(moving_averages)\n'
    '    f_counts = _metric_counts(three_period_growth)\n'
    '    g_counts = _metric_counts(six_period_growth)\n'
    '    h_counts = _metric_counts(twelve_period_growth)\n'
    '    c_ranks = _broad_money_metric_ranks_sheets(moving_averages)\n'
    '    f_ranks = _broad_money_metric_ranks_sheets(three_period_growth)\n'
    '    g_ranks = _broad_money_metric_ranks_sheets(six_period_growth)\n'
    '    h_ranks = _broad_money_metric_ranks_sheets(twelve_period_growth)\n'
    '    c_zscores = _broad_money_rounded_expanding_zscores(moving_averages, c_counts)\n'
    '    f_zscores = _broad_money_rounded_expanding_zscores(three_period_growth, f_counts)\n'
    '    g_zscores = _broad_money_rounded_expanding_zscores(six_period_growth, g_counts)\n'
    '    h_zscores = _broad_money_rounded_expanding_zscores(twelve_period_growth, h_counts)\n'
    '\n'
    '    rows = []\n'
    '    for idx, raw_source in enumerate(source_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = one_period_growth[idx]\n'
    '        f = three_period_growth[idx]\n'
    '        g = six_period_growth[idx]\n'
    '        h = twelve_period_growth[idx]\n'
    '        i = annualized_three[idx]\n'
    '        j = annualized_six[idx]\n'
    '        k = c_ranks[idx]\n'
    '        l = _broad_money_score_percentile(k)\n'
    '        m = f_ranks[idx]\n'
    '        n = _broad_money_score_percentile(m)\n'
    '        o = g_ranks[idx]\n'
    '        p_score = _broad_money_score_percentile(o)\n'
    '        q = h_ranks[idx]\n'
    '        r = _broad_money_score_percentile(q)\n'
    '        v = (\n'
    '            _broad_money_sheets_round(\n'
    '                clamp(\n'
    '                    _broad_money_weighted_sum(\n'
    '                        (0.50, n), (0.30, p_score), (0.15, r), (0.05, l)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(value) for value in (n, p_score, r, l))\n'
    '            else ""\n'
    '        )\n'
    '        w = _broad_money_recovery_score(f, g)\n'
    '        x = _broad_money_shock_score(i)\n'
    '        ak = c_zscores[idx]\n'
    '        al = f_zscores[idx]\n'
    '        am = g_zscores[idx]\n'
    '        an = h_zscores[idx]\n'
    '        y = _broad_money_expansion_support_score(ak, al, am, i, j)\n'
    '        z = _broad_money_contraction_risk_score(ak, al, am, i, j)\n'
    '        aa = _broad_money_outlier_score([ak, al, am, an])\n'
    '        ab = (\n'
    '            _broad_money_sheets_round(\n'
    '                clamp(\n'
    '                    _broad_money_weighted_sum(\n'
    '                        (0.25, l), (0.40, n), (0.25, p_score), (0.10, r)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(value) for value in (l, n, p_score, r))\n'
    '            else ""\n'
    '        )\n'
    '        ac = (\n'
    '            _broad_money_sheets_round(\n'
    '                clamp(\n'
    '                    _broad_money_weighted_sum((0.60, v), (0.25, w), (0.15, l)), -10, 10\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(value) for value in (l, v, w))\n'
    '            else ""\n'
    '        )\n'
    '        ad = (\n'
    '            _broad_money_sheets_round(\n'
    '                clamp(\n'
    '                    _broad_money_weighted_sum(\n'
    '                        (0.20, w), (0.20, x), (0.25, y), (0.25, z), (0.10, aa)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(value) for value in (w, x, y, z, aa))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (ab, ac, ad))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Broad Money": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Broad Money Growth": e,\n'
    '                "3M Broad Money Growth": f,\n'
    '                "6M Broad Money Growth": g,\n'
    '                "12M Broad Money Growth": h,\n'
    '                "3M Annualised Broad Money Growth": i,\n'
    '                "6M Annualised Broad Money Growth": j,\n'
    '                "Broad Money Level Percentile": _broad_money_percent_display(k),\n'
    '                "Broad Money Level Score": l,\n'
    '                "3M Growth Percentile": _broad_money_percent_display(m),\n'
    '                "3M Growth Score": n,\n'
    '                "6M Growth Percentile": _broad_money_percent_display(o),\n'
    '                "6M Growth Score": p_score,\n'
    '                "12M Growth Percentile": _broad_money_percent_display(q),\n'
    '                "12M Growth Score": r,\n'
    '                "Broad Money Regime": _broad_money_regime(c, f, g, h),\n'
    '                "Risk Asset Liquidity Bias": _broad_money_asset_bias_label(ab),\n'
    '                "Money Growth Shock Flag": _broad_money_growth_shock_label(i),\n'
    '                "Broad Money Impulse Score": v,\n'
    '                "Broad Money Cycle Turning Point Score": w,\n'
    '                "Money Growth Shock Score": x,\n'
    '                "Monetary Overheating Risk Score": y,\n'
    '                "Monetary Contraction Risk Score": z,\n'
    '                "Broad Money Extreme Score": aa,\n'
    '                "Coincident Score": ab,\n'
    '                "Leading Score": ac,\n'
    '                "Warning Score": ad,\n'
    '                "State": _broad_money_conditions_regime(ab),\n'
    '                "Regime": _broad_money_momentum_label(ab, ac),\n'
    '                "Signal Comment": _broad_money_signal_interpretation(ac, ad),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _broad_money_warning_label(ad),\n'
    '                "Level Z-Score": ak,\n'
    '                "3M Growth Z-Score": al,\n'
    '                "6M Growth Z-Score": am,\n'
    '                "12M Growth Z-Score": an,\n'
    '                "Extreme Flag": _broad_money_outlier_label([ak, al, am, an]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, BROAD_MONEY_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def calculate_banking_reserves_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    source_dates = [r.get("Date", r.get("A", "")) for r in records]\n'
    '    parsed_dates = [to_date_or_none(value) for value in source_dates]\n'
    '    b_sources = [\n'
    '        r.get(\n'
    '            "Banking System Reserves",\n'
    '            r.get("Raw Value", r.get("B", r.get("Value", ""))),\n'
    '        )\n'
    '        for r in records\n'
    '    ]\n'
    '\n'
    '    mtd_reserves = _mtd_average_liquidity(parsed_dates, b_sources)\n'
    '\n'
    '    observation_counts: list[int] = []\n'
    '    c_count = 0\n'
    '    moving_averages: list[Any] = []\n'
    '    for idx, c in enumerate(mtd_reserves):\n'
    '        if to_float_or_none(c) is not None:\n'
    '            c_count += 1\n'
    '        d = max(0, c_count - 2)\n'
    '        observation_counts.append(d)\n'
    '        if c_count < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [\n'
    '                to_float_or_none(v) for v in mtd_reserves[max(0, idx - 2) : idx + 1]\n'
    '            ]\n'
    '            nums = [v for v in window if v is not None]\n'
    '            moving_averages.append(_banking_reserves_mean_sheets(nums))\n'
    '\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '    three_period_growth: list[Any] = []\n'
    '    six_period_growth: list[Any] = []\n'
    '    twelve_period_growth: list[Any] = []\n'
    '    for idx, e in enumerate(moving_averages):\n'
    '        d = observation_counts[idx]\n'
    '\n'
    '        def lag(offset: int) -> Any:\n'
    '            return moving_averages[idx - offset] if idx >= offset else ""\n'
    '\n'
    '        e1, e3, e6, e12 = lag(1), lag(3), lag(6), lag(12)\n'
    '        one_period_changes.append(\n'
    '            e - e1 if d >= 2 and not is_blank(e) and not is_blank(e1) else ""\n'
    '        )\n'
    '        three_period_changes.append(\n'
    '            e - e3 if d >= 4 and not is_blank(e) and not is_blank(e3) else ""\n'
    '        )\n'
    '        six_period_changes.append(\n'
    '            e - e6 if d >= 7 and not is_blank(e) and not is_blank(e6) else ""\n'
    '        )\n'
    '        twelve_period_changes.append(\n'
    '            e - e12 if d >= 13 and not is_blank(e) and not is_blank(e12) else ""\n'
    '        )\n'
    '        three_period_growth.append(_liquidity_growth(e, e3, d >= 4))\n'
    '        six_period_growth.append(_liquidity_growth(e, e6, d >= 7))\n'
    '        twelve_period_growth.append(_liquidity_growth(e, e12, d >= 13))\n'
    '\n'
    '    e_counts = _metric_counts(moving_averages)\n'
    '    g_counts = _metric_counts(three_period_changes)\n'
    '    h_counts = _metric_counts(six_period_changes)\n'
    '    i_counts = _metric_counts(twelve_period_changes)\n'
    '    e_ranks = _banking_reserves_metric_ranks_sheets(moving_averages)\n'
    '    g_ranks = _banking_reserves_metric_ranks_sheets(three_period_changes)\n'
    '    h_ranks = _banking_reserves_metric_ranks_sheets(six_period_changes)\n'
    '    i_ranks = _banking_reserves_metric_ranks_sheets(\n'
    '        twelve_period_changes, gate_counts=e_counts\n'
    '    )\n'
    '    e_zscores = _banking_reserves_rounded_expanding_zscores(moving_averages, e_counts)\n'
    '    g_zscores = _banking_reserves_rounded_expanding_zscores(\n'
    '        three_period_changes, g_counts\n'
    '    )\n'
    '    h_zscores = _banking_reserves_rounded_expanding_zscores(\n'
    '        six_period_changes, h_counts\n'
    '    )\n'
    '    i_zscores = _banking_reserves_rounded_expanding_zscores(\n'
    '        twelve_period_changes, i_counts\n'
    '    )\n'
    '\n'
    '    rows = []\n'
    '    for idx, raw_b in enumerate(b_sources):\n'
    '        c = mtd_reserves[idx]\n'
    '        d = observation_counts[idx]\n'
    '        e = moving_averages[idx]\n'
    '        f = one_period_changes[idx]\n'
    '        g = three_period_changes[idx]\n'
    '        h = six_period_changes[idx]\n'
    '        i = twelve_period_changes[idx]\n'
    '        j = three_period_growth[idx]\n'
    '        k = six_period_growth[idx]\n'
    '        l = twelve_period_growth[idx]\n'
    '        m = e_ranks[idx]\n'
    '        n = _banking_reserves_score_percentile(m)\n'
    '        o = g_ranks[idx]\n'
    '        p = _banking_reserves_score_percentile(o)\n'
    '        q = h_ranks[idx]\n'
    '        r = _banking_reserves_score_percentile(q)\n'
    '        s = i_ranks[idx]\n'
    '        t = _banking_reserves_score_percentile(s)\n'
    '        x = (\n'
    '            _banking_reserves_sheets_round(\n'
    '                clamp(\n'
    '                    _banking_reserves_weighted_sum(\n'
    '                        (0.05, n), (0.50, p), (0.30, r), (0.15, t)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(v) for v in (n, p, r, t))\n'
    '            else ""\n'
    '        )\n'
    '        y = _banking_reserves_recovery_score(g, h)\n'
    '        z = _banking_reserves_shock_score(f)\n'
    '        am = e_zscores[idx]\n'
    '        an = g_zscores[idx]\n'
    '        ao = h_zscores[idx]\n'
    '        ap = i_zscores[idx]\n'
    '        aa = _banking_reserves_withdrawal_recovery_score(am, an, ao, g, h)\n'
    '        ab = _banking_reserves_scarcity_stress_score(am, an, ao, g, h)\n'
    '        ac = _banking_reserves_outlier_score([am, an, ao, ap])\n'
    '        ad = (\n'
    '            _banking_reserves_sheets_round(\n'
    '                clamp(\n'
    '                    _banking_reserves_weighted_sum(\n'
    '                        (0.35, n), (0.35, p), (0.20, r), (0.10, t)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(v) for v in (n, p, r, t))\n'
    '            else ""\n'
    '        )\n'
    '        ae = (\n'
    '            _banking_reserves_sheets_round(\n'
    '                clamp(\n'
    '                    _banking_reserves_weighted_sum(\n'
    '                        (0.15, n), (0.25, p), (0.25, r), (0.20, t), (0.10, x), (0.05, y)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(v) for v in (n, p, r, t, x, y))\n'
    '            else ""\n'
    '        )\n'
    '        af = (\n'
    '            _banking_reserves_sheets_round(\n'
    '                clamp(\n'
    '                    _banking_reserves_weighted_sum(\n'
    '                        (0.25, y), (0.20, z), (0.25, aa), (0.20, ab), (0.10, ac)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(v) for v in (y, z, aa, ab, ac))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(raw_b)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(raw_b)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(v) for v in (ad, ae, af))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Monthly Average Reserves": c,\n'
    '                "Sample Count": d,\n'
    '                "3M MA Reserves": e,\n'
    '                "1M Reserve Change": f,\n'
    '                "3M Reserve Change": g,\n'
    '                "6M Reserve Change": h,\n'
    '                "12M Reserve Change": i,\n'
    '                "3M Reserve Growth": j,\n'
    '                "6M Reserve Growth": k,\n'
    '                "12M Reserve Growth": l,\n'
    '                "Reserve Level Percentile": _banking_reserves_percent_display(m),\n'
    '                "Reserve Level Score": n,\n'
    '                "3M Change Percentile": _banking_reserves_percent_display(o),\n'
    '                "3M Change Score": p,\n'
    '                "6M Change Percentile": _banking_reserves_percent_display(q),\n'
    '                "6M Change Score": r,\n'
    '                "12M Change Percentile": _banking_reserves_percent_display(s),\n'
    '                "12M Change Score": t,\n'
    '                "Reserve Liquidity Regime": _banking_reserves_regime(e, g, h),\n'
    '                "Risk Asset Liquidity Bias": _banking_reserves_asset_bias_label(ad),\n'
    '                "Reserve Shock Flag": _banking_reserves_shock_label(f),\n'
    '                "Reserve Impulse Score": x,\n'
    '                "Reserve Cycle Turning Point Score": y,\n'
    '                "Reserve Shock Score": z,\n'
    '                "Reserve Drain Risk Score": aa,\n'
    '                "Reserve Scarcity Stress Score": ab,\n'
    '                "Reserve Extreme Score": ac,\n'
    '                "Coincident Score": ad,\n'
    '                "Leading Score": ae,\n'
    '                "Warning Score": af,\n'
    '                "State": _banking_reserves_conditions_regime(ad),\n'
    '                "Regime": _banking_reserves_momentum_label(ad, ae),\n'
    '                "Signal Comment": _banking_reserves_signal_interpretation(ae, af),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _banking_reserves_warning_label(af),\n'
    '                "Level Z-Score": am,\n'
    '                "3M Change Z-Score": an,\n'
    '                "6M Change Z-Score": ao,\n'
    '                "12M Change Z-Score": ap,\n'
    '                "Extreme Flag": _banking_reserves_outlier_label([am, an, ao, ap]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, BANKING_RESERVES_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def calculate_central_bank_liquidity_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    source_dates = [r.get("Date", r.get("A", "")) for r in records]\n'
    '    parsed_dates = [to_date_or_none(value) for value in source_dates]\n'
    '    b_sources = [\n'
    '        r.get(\n'
    '            "Central Bank Total Assets",\n'
    '            r.get("Raw Value", r.get("B", r.get("Value", ""))),\n'
    '        )\n'
    '        for r in records\n'
    '    ]\n'
    '    c_sources = [\n'
    '        r.get("Cental Bank General Account", r.get("Source C", r.get("C", "")))\n'
    '        for r in records\n'
    '    ]\n'
    '    d_sources = [\n'
    '        r.get("Central Bank RRP", r.get("Source D", r.get("D", ""))) for r in records\n'
    '    ]\n'
    '\n'
    '    net_liquidity: list[Any] = []\n'
    '    for b, c, d in zip(b_sources, c_sources, d_sources):\n'
    '        b_num, b_ok = _numeric_or_zero_for_liquidity(b)\n'
    '        c_num, c_ok = _numeric_or_zero_for_liquidity(c)\n'
    '        d_num, d_ok = _numeric_or_zero_for_liquidity(d)\n'
    '        net_liquidity.append(b_num - c_num - d_num if b_ok and c_ok and d_ok else "")\n'
    '\n'
    '    mtd_average = _mtd_average_liquidity(parsed_dates, net_liquidity)\n'
    '    f_counts = _metric_counts(mtd_average)\n'
    '    liquidity_counts = [max(0, count - 2) for count in f_counts]\n'
    '    moving_average_liquidity: list[Any] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '    three_period_growth: list[Any] = []\n'
    '    six_period_growth: list[Any] = []\n'
    '    twelve_period_growth: list[Any] = []\n'
    '\n'
    '    for idx, f_value in enumerate(mtd_average):\n'
    '        if f_counts[idx] < 3:\n'
    '            h = ""\n'
    '        else:\n'
    '            window = [\n'
    '                value\n'
    '                for value in mtd_average[max(0, idx - 2) : idx + 1]\n'
    '                if to_float_or_none(value) is not None\n'
    '            ]\n'
    '            h = _central_bank_liquidity_mean_sheets(window)\n'
    '        moving_average_liquidity.append(h)\n'
    '        g = liquidity_counts[idx]\n'
    '        lag_specs = (\n'
    '            (1, 2, one_period_changes),\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        )\n'
    '        for lag, min_g, target in lag_specs:\n'
    '            if (\n'
    '                g < min_g\n'
    '                or is_blank(h)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_average_liquidity[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(h - moving_average_liquidity[idx - lag])\n'
    '        three_period_growth.append(\n'
    '            _liquidity_growth(\n'
    '                h, moving_average_liquidity[idx - 3] if idx >= 3 else "", g >= 4\n'
    '            )\n'
    '        )\n'
    '        six_period_growth.append(\n'
    '            _liquidity_growth(\n'
    '                h, moving_average_liquidity[idx - 6] if idx >= 6 else "", g >= 7\n'
    '            )\n'
    '        )\n'
    '        twelve_period_growth.append(\n'
    '            _liquidity_growth(\n'
    '                h, moving_average_liquidity[idx - 12] if idx >= 12 else "", g >= 13\n'
    '            )\n'
    '        )\n'
    '\n'
    '    h_counts = _metric_counts(moving_average_liquidity)\n'
    '    j_counts = _metric_counts(three_period_changes)\n'
    '    k_counts = _metric_counts(six_period_changes)\n'
    '    l_counts = _metric_counts(twelve_period_changes)\n'
    '    h_ranks = _central_bank_liquidity_metric_ranks_sheets(moving_average_liquidity)\n'
    '    j_ranks = _central_bank_liquidity_metric_ranks_sheets(three_period_changes)\n'
    '    k_ranks = _central_bank_liquidity_metric_ranks_sheets(six_period_changes)\n'
    '    l_ranks = _central_bank_liquidity_metric_ranks_sheets(twelve_period_changes)\n'
    '    h_zscores = _central_bank_liquidity_rounded_expanding_zscores(\n'
    '        moving_average_liquidity, h_counts\n'
    '    )\n'
    '    j_zscores = _central_bank_liquidity_rounded_expanding_zscores(\n'
    '        three_period_changes, j_counts\n'
    '    )\n'
    '    k_zscores = _central_bank_liquidity_rounded_expanding_zscores(\n'
    '        six_period_changes, k_counts\n'
    '    )\n'
    '    l_zscores = _central_bank_liquidity_rounded_expanding_zscores(\n'
    '        twelve_period_changes, l_counts\n'
    '    )\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, b_source in enumerate(b_sources):\n'
    '        e = net_liquidity[idx]\n'
    '        f = mtd_average[idx]\n'
    '        g = liquidity_counts[idx]\n'
    '        h = moving_average_liquidity[idx]\n'
    '        i = one_period_changes[idx]\n'
    '        j = three_period_changes[idx]\n'
    '        k = six_period_changes[idx]\n'
    '        l = twelve_period_changes[idx]\n'
    '        m = three_period_growth[idx]\n'
    '        n = six_period_growth[idx]\n'
    '        o = twelve_period_growth[idx]\n'
    '        p = h_ranks[idx]\n'
    '        q = _central_bank_liquidity_score_percentile(p)\n'
    '        r = j_ranks[idx]\n'
    '        s_score = _central_bank_liquidity_score_percentile(r)\n'
    '        t = k_ranks[idx]\n'
    '        u = _central_bank_liquidity_score_percentile(t)\n'
    '        v = l_ranks[idx]\n'
    '        w = _central_bank_liquidity_score_percentile(v)\n'
    '        aa = (\n'
    '            _central_bank_liquidity_sheets_round(\n'
    '                clamp(\n'
    '                    _central_bank_liquidity_weighted_sum(\n'
    '                        (0.45, s_score), (0.35, u), (0.15, w), (0.05, q)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (s_score, u, w, q))\n'
    '            else ""\n'
    '        )\n'
    '        ab = _central_bank_liquidity_recovery_score(j, k)\n'
    '        ac = _central_bank_liquidity_shock_score(i)\n'
    '        ao = h_zscores[idx]\n'
    '        ap = j_zscores[idx]\n'
    '        aq = k_zscores[idx]\n'
    '        ar = l_zscores[idx]\n'
    '        ad = _central_bank_liquidity_withdrawal_risk_score(ao, ap, aq, j, k)\n'
    '        ae = _central_bank_liquidity_outlier_score([ao, ap, aq, ar])\n'
    '        af = (\n'
    '            _central_bank_liquidity_sheets_round(\n'
    '                clamp(\n'
    '                    _central_bank_liquidity_weighted_sum(\n'
    '                        (0.40, q), (0.30, s_score), (0.20, u), (0.10, w)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (q, s_score, u, w))\n'
    '            else ""\n'
    '        )\n'
    '        ag = (\n'
    '            _central_bank_liquidity_sheets_round(\n'
    '                clamp(\n'
    '                    _central_bank_liquidity_weighted_sum(\n'
    '                        (0.15, q),\n'
    '                        (0.25, s_score),\n'
    '                        (0.25, u),\n'
    '                        (0.20, w),\n'
    '                        (0.10, aa),\n'
    '                        (0.05, ab),\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (q, s_score, u, w, aa, ab))\n'
    '            else ""\n'
    '        )\n'
    '        ah = (\n'
    '            _central_bank_liquidity_sheets_round(\n'
    '                clamp(\n'
    '                    _central_bank_liquidity_weighted_sum(\n'
    '                        (0.30, ab), (0.25, ac), (0.30, ad), (0.15, ae)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (ab, ac, ad, ae))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if is_blank(b_source)\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if g < 60\n'
    '                else ("Medium reliability" if g < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if is_blank(b_source)\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if g < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(x) for x in (af, ag, ah))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "Central Bank Net Liquidity": e,\n'
    '                "Monthly Net Liquidity": f,\n'
    '                "Sample Count": g,\n'
    '                "3M MA Net Liquidity": h,\n'
    '                "1M Net Liquidity Change": i,\n'
    '                "3M Net Liquidity Change": j,\n'
    '                "6M Net Liquidity Change": k,\n'
    '                "12M Net Liquidity Change": l,\n'
    '                "3M Net Liquidity Growth": m,\n'
    '                "6M Net Liquidity Growth": n,\n'
    '                "12M Net Liquidity Growth": o,\n'
    '                "Liquidity Level Percentile": _central_bank_liquidity_percent_display(\n'
    '                    p\n'
    '                ),\n'
    '                "Liquidity Level Score": q,\n'
    '                "3M Change Percentile": _central_bank_liquidity_percent_display(r),\n'
    '                "3M Change Score": s_score,\n'
    '                "6M Change Percentile": _central_bank_liquidity_percent_display(t),\n'
    '                "6M Change Score": u,\n'
    '                "12M Change Percentile": _central_bank_liquidity_percent_display(v),\n'
    '                "12M Change Score": w,\n'
    '                "Liquidity Regime": _central_bank_liquidity_regime(h, j, k),\n'
    '                "Risk Asset Liquidity Bias": _central_bank_liquidity_asset_bias_label(\n'
    '                    af\n'
    '                ),\n'
    '                "Liquidity Shock Flag": _central_bank_liquidity_shock_label(i),\n'
    '                "Liquidity Impulse Score": aa,\n'
    '                "Liquidity Cycle Turning Point Score": ab,\n'
    '                "Liquidity Shock Score": ac,\n'
    '                "Liquidity Withdrawal Risk Score": ad,\n'
    '                "Liquidity Extreme Score": ae,\n'
    '                "Coincident Score": af,\n'
    '                "Leading Score": ag,\n'
    '                "Warning Score": ah,\n'
    '                "State": _central_bank_liquidity_conditions_regime(af),\n'
    '                "Regime": _central_bank_liquidity_momentum_label(af, ag),\n'
    '                "Signal Comment": _central_bank_liquidity_signal_interpretation(ag, ah),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _central_bank_liquidity_warning_label(ah),\n'
    '                "Level Z-Score": ao,\n'
    '                "3M Change Z-Score": ap,\n'
    '                "6M Change Z-Score": aq,\n'
    '                "12M Change Z-Score": ar,\n'
    '                "Extreme Flag": _central_bank_liquidity_outlier_label([ao, ap, aq, ar]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'def calculate_interbank_rate_outputs(raw_df: Any) -> Any:\n'
    '    records = df_records(raw_df)\n'
    '    raw_values = [\n'
    '        to_float_or_none(r.get("Raw Value", r.get("B", r.get("Value", ""))))\n'
    '        for r in records\n'
    '    ]\n'
    '    raw_counts = _metric_counts(raw_values)\n'
    '    moving_averages: list[Any] = []\n'
    '    smoothed_count = 0\n'
    '    smoothed_counts_so_far: list[int] = []\n'
    '    one_period_changes: list[Any] = []\n'
    '    three_period_changes: list[Any] = []\n'
    '    six_period_changes: list[Any] = []\n'
    '    twelve_period_changes: list[Any] = []\n'
    '\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        if raw_counts[idx] < 3:\n'
    '            moving_averages.append("")\n'
    '        else:\n'
    '            window = [v for v in raw_values[max(0, idx - 2) : idx + 1] if v is not None]\n'
    '            moving_averages.append(\n'
    '                "" if not window else _interbank_rate_mean_sheets(window)\n'
    '            )\n'
    '\n'
    '        c = moving_averages[idx]\n'
    '        if not is_blank(c):\n'
    '            smoothed_count += 1\n'
    '        smoothed_counts_so_far.append(smoothed_count)\n'
    '        if (\n'
    '            smoothed_count < 2\n'
    '            or is_blank(c)\n'
    '            or idx < 1\n'
    '            or is_blank(moving_averages[idx - 1])\n'
    '        ):\n'
    '            one_period_changes.append("")\n'
    '        else:\n'
    '            one_period_changes.append(c - moving_averages[idx - 1])\n'
    '        for lag, min_count, target in (\n'
    '            (3, 4, three_period_changes),\n'
    '            (6, 7, six_period_changes),\n'
    '            (12, 13, twelve_period_changes),\n'
    '        ):\n'
    '            if (\n'
    '                smoothed_count < min_count\n'
    '                or is_blank(c)\n'
    '                or idx < lag\n'
    '                or is_blank(moving_averages[idx - lag])\n'
    '            ):\n'
    '                target.append("")\n'
    '            else:\n'
    '                target.append(c - moving_averages[idx - lag])\n'
    '\n'
    '    smoothed_counts = smoothed_counts_so_far\n'
    '    f_counts = _metric_counts(three_period_changes)\n'
    '    g_counts = _metric_counts(six_period_changes)\n'
    '    h_counts = _metric_counts(twelve_period_changes)\n'
    '    c_ranks = _interbank_rate_metric_ranks_sheets(moving_averages)\n'
    '    f_ranks = _interbank_rate_metric_ranks_sheets(three_period_changes)\n'
    '    g_ranks = _interbank_rate_metric_ranks_sheets(six_period_changes)\n'
    '    h_ranks = _interbank_rate_metric_ranks_sheets(twelve_period_changes)\n'
    '    c_zscores = _rounded_expanding_zscores(moving_averages, smoothed_counts, 60)\n'
    '    f_zscores = _rounded_expanding_zscores(three_period_changes, f_counts, 60)\n'
    '    g_zscores = _rounded_expanding_zscores(six_period_changes, g_counts, 60)\n'
    '    h_zscores = _rounded_expanding_zscores(twelve_period_changes, h_counts, 60)\n'
    '\n'
    '    rows: list[dict[str, Any]] = []\n'
    '    for idx, current in enumerate(raw_values):\n'
    '        c = moving_averages[idx]\n'
    '        d = smoothed_counts[idx]\n'
    '        e = one_period_changes[idx]\n'
    '        f = three_period_changes[idx]\n'
    '        g = six_period_changes[idx]\n'
    '        h = twelve_period_changes[idx]\n'
    '        i = c_ranks[idx]\n'
    '        j = _interbank_rate_score_percentile(i)\n'
    '        k = f_ranks[idx]\n'
    '        l = _interbank_rate_score_percentile(k)\n'
    '        m = g_ranks[idx]\n'
    '        n = _interbank_rate_score_percentile(m)\n'
    '        o = h_ranks[idx]\n'
    '        p = _interbank_rate_score_percentile(o)\n'
    '        t = (\n'
    '            _interbank_rate_sheets_round(\n'
    '                clamp(\n'
    '                    _interbank_rate_weighted_sum(\n'
    '                        (0.05, j), (0.45, l), (0.35, n), (0.15, p)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(x) for x in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        u = _interbank_rate_recovery_tightening_score(c, f, g)\n'
    '        v = _interbank_rate_shock_score(e)\n'
    '        w = _interbank_rate_restriction_score(c, f, g)\n'
    '        ah = c_zscores[idx]\n'
    '        ai = f_zscores[idx]\n'
    '        aj = g_zscores[idx]\n'
    '        ak = h_zscores[idx]\n'
    '        x = _interbank_rate_outlier_score([ah, ai, aj, ak])\n'
    '        y = (\n'
    '            _interbank_rate_sheets_round(\n'
    '                clamp(\n'
    '                    _interbank_rate_weighted_sum(\n'
    '                        (0.45, j), (0.30, l), (0.15, n), (0.10, p)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p))\n'
    '            else ""\n'
    '        )\n'
    '        z = (\n'
    '            _interbank_rate_sheets_round(\n'
    '                clamp(\n'
    '                    _interbank_rate_weighted_sum(\n'
    '                        (0.10, j), (0.35, l), (0.25, n), (0.15, p), (0.10, t), (0.05, u)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (j, l, n, p, t, u))\n'
    '            else ""\n'
    '        )\n'
    '        aa = (\n'
    '            _interbank_rate_sheets_round(\n'
    '                clamp(\n'
    '                    _interbank_rate_weighted_sum(\n'
    '                        (0.30, u), (0.25, v), (0.30, w), (0.15, x)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            if all(not is_blank(val) for val in (u, v, w, x))\n'
    '            else ""\n'
    '        )\n'
    '        reliability = (\n'
    '            ""\n'
    '            if current is None\n'
    '            else (\n'
    '                "Insufficient history"\n'
    '                if d < 60\n'
    '                else ("Medium reliability" if d < 120 else "High reliability")\n'
    '            )\n'
    '        )\n'
    '        data_status = (\n'
    '            "Missing Data"\n'
    '            if current is None\n'
    '            else (\n'
    '                "Low Sample"\n'
    '                if d < 60\n'
    '                else (\n'
    '                    "Incomplete Score"\n'
    '                    if any(is_blank(val) for val in (y, z, aa))\n'
    '                    else "Valid"\n'
    '                )\n'
    '            )\n'
    '        )\n'
    '        rows.append(\n'
    '            {\n'
    '                "3M MA Rate": c,\n'
    '                "Sample Count": d,\n'
    '                "1M Rate Change": e,\n'
    '                "3M Rate Change": f,\n'
    '                "6M Rate Change": g,\n'
    '                "12M Rate Change": h,\n'
    '                "Rate Level Percentile": _interbank_rate_percent_display(i),\n'
    '                "Rate Level Score": j,\n'
    '                "3M Change Percentile": _interbank_rate_percent_display(k),\n'
    '                "3M Change Score": l,\n'
    '                "6M Change Percentile": _interbank_rate_percent_display(m),\n'
    '                "6M Change Score": n,\n'
    '                "12M Change Percentile": _interbank_rate_percent_display(o),\n'
    '                "12M Change Score": p,\n'
    '                "Interbank Rate Regime": _interbank_rate_level_regime(c, f),\n'
    '                "Interbank Rate Policy Bias": _interbank_rate_policy_bias_label(y),\n'
    '                "1M Interbank Rate Shock Flag": _interbank_rate_shock_label(e),\n'
    '                "Interbank Rate Impulse Score": t,\n'
    '                "Interbank Rate Cycle Turning Point Score": u,\n'
    '                "Interbank Rate Shock Score": v,\n'
    '                "Interbank Rate Pressure Score": w,\n'
    '                "Interbank Rate Extreme Score": x,\n'
    '                "Coincident Score": y,\n'
    '                "Leading Score": z,\n'
    '                "Warning Score": aa,\n'
    '                "State": _interbank_rate_pressure_regime(y),\n'
    '                "Regime": _interbank_rate_momentum_label(y, z),\n'
    '                "Signal Comment": _interbank_rate_signal_interpretation(z, aa),\n'
    '                "Reliability": reliability,\n'
    '                "Signal Validity": data_status,\n'
    '                "Warning Flag": _interbank_rate_warning_label(aa),\n'
    '                "Level Z-Score": ah,\n'
    '                "3M Change Z-Score": ai,\n'
    '                "6M Change Z-Score": aj,\n'
    '                "12M Change Z-Score": ak,\n'
    '                "Extreme Flag": _interbank_rate_outlier_label([ah, ai, aj, ak]),\n'
    '            }\n'
    '        )\n'
    '    return make_dataframe(rows, INTERBANK_RATE_OUTPUT_COLUMNS)\n'
    '\n'
    '\n'
    'INDICATOR_ENGINES: dict[str, Callable[[Any], Any]] = {\n'
    '    "DEFAULT_PMI_ENGINE": calculate_manufacturing_pmi_outputs,\n'
    '    "MANUFACTURING_PMI_ENGINE": calculate_manufacturing_pmi_outputs,\n'
    '    "MANUFACTURING_NEW_ORDERS_ENGINE": calculate_manufacturing_new_orders_outputs,\n'
    '    "SERVICES_PMI_ENGINE": calculate_services_pmi_outputs,\n'
    '    "SERVICES_NEW_ORDERS_ENGINE": calculate_services_new_orders_outputs,\n'
    '    "SERVICES_BUSINESS_ACTIVITY_ENGINE": calculate_services_business_activity_outputs,\n'
    '    "INDUSTRIAL_PRODUCTION_ENGINE": calculate_industrial_production_outputs,\n'
    '    "CONSUMER_SPENDING_ENGINE": calculate_consumer_spending_outputs,\n'
    '    "CONSUMER_SENTIMENT_ENGINE": calculate_consumer_sentiment_outputs,\n'
    '    "CONSUMER_EXPECTATIONS_ENGINE": calculate_consumer_expectations_outputs,\n'
    '    "INITIAL_CLAIMS_ENGINE": calculate_initial_claims_outputs,\n'
    '    "EMPLOYMENT_GROWTH_ENGINE": calculate_employment_growth_outputs,\n'
    '    "AVERAGE_HOURS_ENGINE": calculate_average_hours_outputs,\n'
    '    "BUILDING_PERMITS_ENGINE": calculate_building_permits_outputs,\n'
    '    "CORE_INFLATION_ENGINE": calculate_core_inflation_outputs,\n'
    '    "CORE_CPI_INFLATION_ENGINE": calculate_core_cpi_inflation_outputs,\n'
    '    "HEADLINE_CPI_INFLATION_ENGINE": calculate_headline_cpi_inflation_outputs,\n'
    '    "CORE_PPI_ENGINE": calculate_core_ppi_outputs,\n'
    '    "HEADLINE_PPI_ENGINE": calculate_headline_ppi_outputs,\n'
    '    "WAGE_GROWTH_ENGINE": calculate_wage_growth_outputs,\n'
    '    "SHORT_TERM_INFLATION_EXPECTATIONS_ENGINE": calculate_short_term_inflation_expectations_outputs,\n'
    '    "REAL_YIELD_ENGINE": calculate_real_yield_outputs,\n'
    '    "TWO_YEAR_YIELD_ENGINE": calculate_two_year_yield_outputs,\n'
    '    "TEN_YEAR_YIELD_ENGINE": calculate_ten_year_yield_outputs,\n'
    '    "YIELD_CURVE_ENGINE": calculate_yield_curve_outputs,\n'
    '    "TERM_PREMIUM_ENGINE": calculate_term_premium_outputs,\n'
    '    "INTERBANK_RATE_ENGINE": calculate_interbank_rate_outputs,\n'
    '    "CENTRAL_BANK_LIQUIDITY_ENGINE": calculate_central_bank_liquidity_outputs,\n'
    '    "BANKING_RESERVES_ENGINE": calculate_banking_reserves_outputs,\n'
    '    "BROAD_MONEY_ENGINE": calculate_broad_money_outputs,\n'
    '    "FISCAL_BALANCE_ENGINE": calculate_fiscal_balance_outputs,\n'
    '    "REAL_GOVERNMENT_SPENDING_ENGINE": calculate_real_government_spending_outputs,\n'
    '    "GOVERNMENT_REVENUE_ENGINE": calculate_government_revenue_outputs,\n'
    '    "PUBLIC_DEBT_ENGINE": calculate_public_debt_outputs,\n'
    '    "INTEREST_COVERAGE_ENGINE": calculate_interest_coverage_outputs,\n'
    '    "FINANCIAL_CONDITIONS_ENGINE": calculate_financial_conditions_outputs,\n'
    '    "BANK_CREDIT_ENGINE": calculate_bank_credit_outputs,\n'
    '    "CORPORATE_CREDIT_SPREADS_ENGINE": calculate_corporate_credit_spreads_outputs,\n'
    '}\n'
    '\n'
    '\n'
    'def open_workbook(config: EngineConfig) -> Any:\n'
    '    import gspread\n'
    '\n'
    '    return call_with_backoff(\n'
    '        lambda: gspread.service_account(\n'
    '            filename=config.service_account_path\n'
    '        ).open_by_key(config.spreadsheet_id),\n'
    '        operation_name="open_workbook",\n'
    '    )\n'
    '\n'
    '\n'
    'def read_control_panel_indicator_name(\n'
    '    workbook: Any, control_cell: str = CONTROL_PANEL_FIRST_INDICATOR_CELL\n'
    ') -> str:\n'
    '    name = workbook.worksheet(CONTROL_PANEL_SHEET_NAME).acell(control_cell).value\n'
    '    if is_blank(name):\n'
    '        raise ValueError(f"{CONTROL_PANEL_SHEET_NAME}!{control_cell} is blank")\n'
    '    return str(name).strip()\n'
    '\n'
    '\n'
    'def select_indicator_engine(sheet_name: str, control_cell: str) -> Callable[[Any], Any]:\n'
    '    if sheet_name in INDICATOR_ENGINES:\n'
    '        return INDICATOR_ENGINES[sheet_name]\n'
    '    if control_cell.upper() == CONTROL_PANEL_SECOND_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["MANUFACTURING_NEW_ORDERS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRD_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["SERVICES_PMI_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_FOURTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["SERVICES_NEW_ORDERS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_FIFTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["SERVICES_BUSINESS_ACTIVITY_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_SIXTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["INDUSTRIAL_PRODUCTION_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_SEVENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CONSUMER_SPENDING_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_EIGHTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CONSUMER_SENTIMENT_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_NINTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CONSUMER_EXPECTATIONS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["INITIAL_CLAIMS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_ELEVENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["EMPLOYMENT_GROWTH_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWELFTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["AVERAGE_HOURS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["BUILDING_PERMITS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_FOURTEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CORE_INFLATION_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_FIFTEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CORE_CPI_INFLATION_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_SIXTEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["HEADLINE_CPI_INFLATION_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_SEVENTEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CORE_PPI_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_EIGHTEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["HEADLINE_PPI_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_NINETEENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["WAGE_GROWTH_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTIETH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["SHORT_TERM_INFLATION_EXPECTATIONS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_FIRST_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["REAL_YIELD_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_SECOND_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["TWO_YEAR_YIELD_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_THIRD_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["TEN_YEAR_YIELD_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_FOURTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["YIELD_CURVE_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_FIFTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["TERM_PREMIUM_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_SIXTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["INTERBANK_RATE_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CENTRAL_BANK_LIQUIDITY_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["BANKING_RESERVES_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["BROAD_MONEY_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTIETH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["FISCAL_BALANCE_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["REAL_GOVERNMENT_SPENDING_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_SECOND_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["GOVERNMENT_REVENUE_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_THIRD_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["PUBLIC_DEBT_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_FOURTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["INTEREST_COVERAGE_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_FIFTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["FINANCIAL_CONDITIONS_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_SIXTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["BANK_CREDIT_ENGINE"]\n'
    '    if control_cell.upper() == CONTROL_PANEL_THIRTY_SEVENTH_INDICATOR_CELL:\n'
    '        return INDICATOR_ENGINES["CORPORATE_CREDIT_SPREADS_ENGINE"]\n'
    '    return INDICATOR_ENGINES["DEFAULT_PMI_ENGINE"]\n'
    '\n'
    '\n'
    'def output_columns_for_engine(engine: Callable[[Any], Any]) -> Sequence[str]:\n'
    '    if engine is calculate_manufacturing_new_orders_outputs:\n'
    '        return MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_industrial_production_outputs:\n'
    '        return INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS\n'
    '    if engine is calculate_consumer_spending_outputs:\n'
    '        return CONSUMER_SPENDING_OUTPUT_COLUMNS\n'
    '    if engine is calculate_consumer_sentiment_outputs:\n'
    '        return CONSUMER_SENTIMENT_OUTPUT_COLUMNS\n'
    '    if engine is calculate_consumer_expectations_outputs:\n'
    '        return CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_initial_claims_outputs:\n'
    '        return INITIAL_CLAIMS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_employment_growth_outputs:\n'
    '        return EMPLOYMENT_GROWTH_OUTPUT_COLUMNS\n'
    '    if engine is calculate_average_hours_outputs:\n'
    '        return AVERAGE_HOURS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_building_permits_outputs:\n'
    '        return BUILDING_PERMITS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_core_inflation_outputs:\n'
    '        return CORE_INFLATION_OUTPUT_COLUMNS\n'
    '    if engine is calculate_core_cpi_inflation_outputs:\n'
    '        return CORE_CPI_INFLATION_OUTPUT_COLUMNS\n'
    '    if engine is calculate_headline_cpi_inflation_outputs:\n'
    '        return HEADLINE_CPI_INFLATION_OUTPUT_COLUMNS\n'
    '    if engine is calculate_core_ppi_outputs:\n'
    '        return CORE_PPI_OUTPUT_COLUMNS\n'
    '    if engine is calculate_headline_ppi_outputs:\n'
    '        return HEADLINE_PPI_OUTPUT_COLUMNS\n'
    '    if engine is calculate_wage_growth_outputs:\n'
    '        return WAGE_GROWTH_OUTPUT_COLUMNS\n'
    '    if engine is calculate_short_term_inflation_expectations_outputs:\n'
    '        return SHORT_TERM_INFLATION_EXPECTATIONS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_real_yield_outputs:\n'
    '        return REAL_YIELD_OUTPUT_COLUMNS\n'
    '    if engine is calculate_two_year_yield_outputs:\n'
    '        return TWO_YEAR_YIELD_OUTPUT_COLUMNS\n'
    '    if engine is calculate_ten_year_yield_outputs:\n'
    '        return TEN_YEAR_YIELD_OUTPUT_COLUMNS\n'
    '    if engine is calculate_yield_curve_outputs:\n'
    '        return YIELD_CURVE_OUTPUT_COLUMNS\n'
    '    if engine is calculate_term_premium_outputs:\n'
    '        return TERM_PREMIUM_OUTPUT_COLUMNS\n'
    '    if engine is calculate_interbank_rate_outputs:\n'
    '        return INTERBANK_RATE_OUTPUT_COLUMNS\n'
    '    if engine is calculate_central_bank_liquidity_outputs:\n'
    '        return CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS\n'
    '    if engine is calculate_banking_reserves_outputs:\n'
    '        return BANKING_RESERVES_OUTPUT_COLUMNS\n'
    '    if engine is calculate_broad_money_outputs:\n'
    '        return BROAD_MONEY_OUTPUT_COLUMNS\n'
    '    if engine is calculate_fiscal_balance_outputs:\n'
    '        return FISCAL_BALANCE_OUTPUT_COLUMNS\n'
    '    if engine is calculate_real_government_spending_outputs:\n'
    '        return REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS\n'
    '    if engine is calculate_government_revenue_outputs:\n'
    '        return GOVERNMENT_REVENUE_OUTPUT_COLUMNS\n'
    '    if engine is calculate_public_debt_outputs:\n'
    '        return PUBLIC_DEBT_OUTPUT_COLUMNS\n'
    '    if engine is calculate_interest_coverage_outputs:\n'
    '        return INTEREST_COVERAGE_OUTPUT_COLUMNS\n'
    '    if engine is calculate_financial_conditions_outputs:\n'
    '        return FINANCIAL_CONDITIONS_OUTPUT_COLUMNS\n'
    '    if engine is calculate_bank_credit_outputs:\n'
    '        return BANK_CREDIT_OUTPUT_COLUMNS\n'
    '    if engine is calculate_corporate_credit_spreads_outputs:\n'
    '        return CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS\n'
    '    return OUTPUT_COLUMNS\n'
    '\n'
    '\n'
    'def output_end_col_for_columns(columns: Sequence[str]) -> str:\n'
    '    cols = list(columns)\n'
    '    if cols == list(MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS):\n'
    '        return "AB"\n'
    '    if cols == list(CONSUMER_SENTIMENT_OUTPUT_COLUMNS) or cols == list(\n'
    '        CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS\n'
    '    ):\n'
    '        return "AE"\n'
    '    if cols == list(CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS):\n'
    '        return "AS"\n'
    '    if cols == list(BANKING_RESERVES_OUTPUT_COLUMNS):\n'
    '        return "AQ"\n'
    '    if cols == list(BROAD_MONEY_OUTPUT_COLUMNS):\n'
    '        return "AO"\n'
    '    if cols == list(FISCAL_BALANCE_OUTPUT_COLUMNS):\n'
    '        return "AJ"\n'
    '    if cols == list(REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS):\n'
    '        return "AM"\n'
    '    if cols == list(GOVERNMENT_REVENUE_OUTPUT_COLUMNS):\n'
    '        return "AP"\n'
    '    if cols == list(PUBLIC_DEBT_OUTPUT_COLUMNS):\n'
    '        return "AN"\n'
    '    if cols == list(INTEREST_COVERAGE_OUTPUT_COLUMNS):\n'
    '        return "AN"\n'
    '    if cols == list(FINANCIAL_CONDITIONS_OUTPUT_COLUMNS):\n'
    '        return "AM"\n'
    '    if cols == list(BANK_CREDIT_OUTPUT_COLUMNS):\n'
    '        return "AP"\n'
    '    if cols == list(CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS):\n'
    '        return "AL"\n'
    '    if (\n'
    '        cols == list(BUILDING_PERMITS_OUTPUT_COLUMNS)\n'
    '        or cols == list(CORE_INFLATION_OUTPUT_COLUMNS)\n'
    '        or cols == list(CORE_CPI_INFLATION_OUTPUT_COLUMNS)\n'
    '        or cols == list(HEADLINE_CPI_INFLATION_OUTPUT_COLUMNS)\n'
    '        or cols == list(CORE_PPI_OUTPUT_COLUMNS)\n'
    '        or cols == list(HEADLINE_PPI_OUTPUT_COLUMNS)\n'
    '        or cols == list(REAL_YIELD_OUTPUT_COLUMNS)\n'
    '        or cols == list(TWO_YEAR_YIELD_OUTPUT_COLUMNS)\n'
    '        or cols == list(TEN_YEAR_YIELD_OUTPUT_COLUMNS)\n'
    '        or cols == list(YIELD_CURVE_OUTPUT_COLUMNS)\n'
    '        or cols == list(TERM_PREMIUM_OUTPUT_COLUMNS)\n'
    '        or cols == list(INTERBANK_RATE_OUTPUT_COLUMNS)\n'
    '    ):\n'
    '        return "AL"\n'
    '    if (\n'
    '        cols == list(INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS)\n'
    '        or cols == list(CONSUMER_SPENDING_OUTPUT_COLUMNS)\n'
    '        or cols == list(INITIAL_CLAIMS_OUTPUT_COLUMNS)\n'
    '        or cols == list(EMPLOYMENT_GROWTH_OUTPUT_COLUMNS)\n'
    '        or cols == list(AVERAGE_HOURS_OUTPUT_COLUMNS)\n'
    '        or cols == list(WAGE_GROWTH_OUTPUT_COLUMNS)\n'
    '        or cols == list(SHORT_TERM_INFLATION_EXPECTATIONS_OUTPUT_COLUMNS)\n'
    '    ):\n'
    '        return "AJ"\n'
    '    if cols == list(OUTPUT_COLUMNS):\n'
    '        return "AB"\n'
    '    raise ValueError(f"Unsupported output column count: {len(columns)}")\n'
    '\n'
    '\n'
    'def engine_spec_for_engine(engine: Callable[[Any], Any]) -> IndicatorEngineSpec:\n'
    '    columns = output_columns_for_engine(engine)\n'
    '    output_end = output_end_col_for_columns(columns)\n'
    '    if engine is calculate_manufacturing_new_orders_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "MANUFACTURING_NEW_ORDERS_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AB",\n'
    '            output_columns=MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_industrial_production_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "INDUSTRIAL_PRODUCTION_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AJ",\n'
    '            output_columns=INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS,\n'
    '            summary_level_col="YoY Growth Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_average_hours_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "AVERAGE_HOURS_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AJ",\n'
    '            output_columns=AVERAGE_HOURS_OUTPUT_COLUMNS,\n'
    '            summary_level_col="YoY Growth Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_headline_cpi_inflation_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "HEADLINE_CPI_INFLATION_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=HEADLINE_CPI_OUTPUT_COLUMNS,\n'
    '            summary_level_col="YoY Inflation Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_real_yield_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "REAL_YIELD_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=REAL_YIELD_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Real Yield Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_two_year_yield_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "TWO_YEAR_YIELD_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=TWO_YEAR_YIELD_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Yield Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_ten_year_yield_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "TEN_YEAR_YIELD_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=TEN_YEAR_YIELD_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Yield Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_yield_curve_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "YIELD_CURVE_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=YIELD_CURVE_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Curve Stress Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_interbank_rate_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "INTERBANK_RATE_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=INTERBANK_RATE_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Rate Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '            input_value_render_option="UNFORMATTED_VALUE",\n'
    '        )\n'
    '    if engine is calculate_consumer_sentiment_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "CONSUMER_SENTIMENT_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AE",\n'
    '            output_columns=CONSUMER_SENTIMENT_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_consumer_expectations_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "CONSUMER_EXPECTATIONS_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AE",\n'
    '            output_columns=CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_central_bank_liquidity_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "CENTRAL_BANK_LIQUIDITY_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="D",\n'
    '            output_start_col="E",\n'
    '            output_end_col="AS",\n'
    '            output_columns=CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Liquidity Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '            input_value_render_option="UNFORMATTED_VALUE",\n'
    '        )\n'
    '    if engine is calculate_banking_reserves_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "BANKING_RESERVES_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AQ",\n'
    '            output_columns=BANKING_RESERVES_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Reserve Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '            input_value_render_option="UNFORMATTED_VALUE",\n'
    '        )\n'
    '    if engine is calculate_broad_money_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "BROAD_MONEY_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AO",\n'
    '            output_columns=BROAD_MONEY_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Broad Money Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_fiscal_balance_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "FISCAL_BALANCE_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AJ",\n'
    '            output_columns=FISCAL_BALANCE_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Fiscal Balance Level Score",\n'
    '            summary_leading_col="Fiscal Impulse Leading Score",\n'
    '            summary_warning_col="Fiscal Impulse Final Warning Score",\n'
    '        )\n'
    '    if engine is calculate_real_government_spending_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "REAL_GOVERNMENT_SPENDING_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AM",\n'
    '            output_columns=REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS,\n'
    '            summary_level_col="1-Period Real Government Spending Growth Score",\n'
    '            summary_leading_col="Real Government Spending Leading Score",\n'
    '            summary_warning_col="Real Government Spending Final Warning Score",\n'
    '        )\n'
    '    if engine is calculate_government_revenue_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "GOVERNMENT_REVENUE_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AP",\n'
    '            output_columns=GOVERNMENT_REVENUE_OUTPUT_COLUMNS,\n'
    '            summary_level_col=None,\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_public_debt_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "PUBLIC_DEBT_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AN",\n'
    '            output_columns=PUBLIC_DEBT_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Public Debt Level Score",\n'
    '            summary_leading_col="Public Debt Leading Score",\n'
    '            summary_warning_col="Public Debt Final Warning Score",\n'
    '        )\n'
    '    if engine is calculate_interest_coverage_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "INTEREST_COVERAGE_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="C",\n'
    '            output_start_col="D",\n'
    '            output_end_col="AN",\n'
    '            output_columns=INTEREST_COVERAGE_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Interest Coverage Level Score",\n'
    '            summary_leading_col="Interest Coverage Leading Score",\n'
    '            summary_warning_col="Interest Coverage Final Warning Score",\n'
    '        )\n'
    '    if engine is calculate_financial_conditions_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "FINANCIAL_CONDITIONS_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AM",\n'
    '            output_columns=FINANCIAL_CONDITIONS_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Financial Conditions Level Stress Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_bank_credit_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "BANK_CREDIT_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AP",\n'
    '            output_columns=BANK_CREDIT_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Bank Credit Level Score",\n'
    '            summary_leading_col="Leading Score",\n'
    '            summary_warning_col="Warning Score",\n'
    '            summary_status_col="Signal Validity",\n'
    '        )\n'
    '    if engine is calculate_corporate_credit_spreads_outputs:\n'
    '        return IndicatorEngineSpec(\n'
    '            "CORPORATE_CREDIT_SPREADS_ENGINE",\n'
    '            engine,\n'
    '            input_start_col="A",\n'
    '            input_end_col="B",\n'
    '            output_start_col="C",\n'
    '            output_end_col="AL",\n'
    '            output_columns=CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS,\n'
    '            summary_level_col="Corporate Credit Spread Level Score",\n'
    '            summary_leading_col="Corporate Credit Spread Leading Score",\n'
    '            summary_warning_col="Corporate Credit Spread Final Warning Score",\n'
    '        )\n'
    '    return IndicatorEngineSpec(\n'
    '        getattr(engine, "__name__", "indicator_engine"),\n'
    '        engine,\n'
    '        input_start_col="A",\n'
    '        input_end_col="B",\n'
    '        output_start_col="C",\n'
    '        output_end_col=output_end,\n'
    '        output_columns=columns,\n'
    '    )\n'
    '\n'
    '\n'
    'def select_indicator_engine_spec(\n'
    '    sheet_name: str, control_cell: str\n'
    ') -> IndicatorEngineSpec:\n'
    '    return engine_spec_for_engine(select_indicator_engine(sheet_name, control_cell))\n'
    '\n'
    '\n'
    'def _parse_a1_single_column_range(range_name: str) -> tuple[str, int, int | None]:\n'
    '    if ":" in range_name:\n'
    '        start, end = range_name.split(":", 1)\n'
    '    else:\n'
    '        start, end = range_name, range_name\n'
    '    start_col = "".join(ch for ch in start if ch.isalpha()).upper()\n'
    '    end_col = "".join(ch for ch in end if ch.isalpha()).upper()\n'
    '    if start_col != end_col:\n'
    '        raise ValueError(\n'
    '            f"Only single-column control ranges are supported: {range_name}"\n'
    '        )\n'
    '    start_row_text = "".join(ch for ch in start if ch.isdigit())\n'
    '    end_row_text = "".join(ch for ch in end if ch.isdigit())\n'
    '    if not start_row_text:\n'
    '        raise ValueError(f"Control range must include a starting row: {range_name}")\n'
    '    return start_col, int(start_row_text), int(end_row_text) if end_row_text else None\n'
    '\n'
    '\n'
    'def _cell_ref(column: str, row: int) -> str:\n'
    '    return f"{column.upper()}{row}"\n'
    '\n'
    '\n'
    'def _control_values_to_specs(\n'
    '    values: list[list[Any]], column: str, start_row: int, stop_at_first_blank: bool\n'
    ') -> list[BatchIndicatorSpec]:\n'
    '    specs: list[BatchIndicatorSpec] = []\n'
    '    for offset, row in enumerate(values):\n'
    '        value = row[0] if row else ""\n'
    '        if is_blank(value):\n'
    '            if stop_at_first_blank:\n'
    '                break\n'
    '            continue\n'
    '        specs.append(\n'
    '            BatchIndicatorSpec(\n'
    '                str(value).strip(), _cell_ref(column, start_row + offset)\n'
    '            )\n'
    '        )\n'
    '    return specs\n'
    '\n'
    '\n'
    'def read_control_panel_indicator_specs(\n'
    '    workbook: Any, control_range: str, stop_at_first_blank: bool\n'
    ') -> list[BatchIndicatorSpec]:\n'
    '    column, start_row, _ = _parse_a1_single_column_range(control_range)\n'
    '    values = workbook.worksheet(CONTROL_PANEL_SHEET_NAME).get(control_range)\n'
    '    return _control_values_to_specs(values, column, start_row, stop_at_first_blank)\n'
    '\n'
    '\n'
    'def build_indicator_specs(\n'
    '    workbook: Any, args: argparse.Namespace\n'
    ') -> list[BatchIndicatorSpec]:\n'
    '    if args.indicator_sheet:\n'
    '        return [BatchIndicatorSpec(args.indicator_sheet, args.control_cell)]\n'
    '    if args.all:\n'
    '        return read_control_panel_indicator_specs(\n'
    '            workbook, "A3:A", stop_at_first_blank=True\n'
    '        )\n'
    '    if args.control_range:\n'
    '        return read_control_panel_indicator_specs(\n'
    '            workbook, args.control_range, stop_at_first_blank=False\n'
    '        )\n'
    '    return [\n'
    '        BatchIndicatorSpec(\n'
    '            read_control_panel_indicator_name(workbook, args.control_cell),\n'
    '            args.control_cell,\n'
    '        )\n'
    '    ]\n'
    '\n'
    '\n'
    'def filter_indicator_specs(\n'
    '    specs: list[BatchIndicatorSpec],\n'
    '    only: Sequence[str] | None,\n'
    '    exclude: Sequence[str] | None,\n'
    ') -> list[BatchIndicatorSpec]:\n'
    '    only_set = set(only or [])\n'
    '    exclude_set = set(exclude or [])\n'
    '    filtered = []\n'
    '    for spec in specs:\n'
    '        if only_set and spec.sheet_name not in only_set:\n'
    '            continue\n'
    '        if spec.sheet_name in exclude_set:\n'
    '            continue\n'
    '        filtered.append(spec)\n'
    '    return filtered\n'
    '\n'
    '\n'
    'def process_indicator(\n'
    '    workbook: Any, spec: BatchIndicatorSpec, write: bool, verify: bool\n'
    ') -> BatchIndicatorResult:\n'
    '    try:\n'
    '        LOGGER.info(\n'
    '            "Selected indicator sheet name: %s (control cell %s)",\n'
    '            spec.sheet_name,\n'
    '            spec.control_cell,\n'
    '        )\n'
    '        worksheet = workbook.worksheet(spec.sheet_name)\n'
    '        engine_spec = select_indicator_engine_spec(spec.sheet_name, spec.control_cell)\n'
    '        input_range = f"{engine_spec.input_start_col}2:{engine_spec.input_end_col}"\n'
    '        if engine_spec.input_value_render_option:\n'
    '            input_values = worksheet.get(\n'
    '                input_range, value_render_option=engine_spec.input_value_render_option\n'
    '            )\n'
    '        else:\n'
    '            input_values = worksheet.get(input_range)\n'
    '        raw_df = read_indicator_values(input_values, engine_spec.input_end_col)\n'
    '        output_df = engine_spec.calculate(raw_df)\n'
    '        output_columns = engine_spec.output_columns\n'
    '        summary = build_summary(spec.sheet_name, raw_df, output_df)\n'
    '        summary = IndicatorRunSummary(\n'
    '            summary.indicator_sheet_name,\n'
    '            summary.rows_read,\n'
    '            summary.rows_calculated,\n'
    '            f"{engine_spec.output_start_col}2:{engine_spec.output_end_col}{summary.rows_calculated + 1}",\n'
    '            summary.first_date,\n'
    '            summary.last_date,\n'
    '            summary.nonblank_raw_values,\n'
    '        )\n'
    '        print_summary(summary, output_df)\n'
    '        if write:\n'
    '            write_indicator_outputs(\n'
    '                worksheet,\n'
    '                output_df,\n'
    '                summary.rows_calculated + 1,\n'
    '                output_columns,\n'
    '                engine_spec.output_end_col,\n'
    '                engine_spec.output_start_col,\n'
    '            )\n'
    '            if verify:\n'
    '                verify_written_outputs(\n'
    '                    worksheet,\n'
    '                    output_df,\n'
    '                    summary.rows_calculated + 1,\n'
    '                    output_columns,\n'
    '                    engine_spec.output_end_col,\n'
    '                    engine_spec.output_start_col,\n'
    '                )\n'
    '                LOGGER.info(\n'
    '                    "Write and verification completed successfully for %s.",\n'
    '                    spec.sheet_name,\n'
    '                )\n'
    '            else:\n'
    '                LOGGER.info(\n'
    '                    "Write completed for %s; verification skipped by --no-verify.",\n'
    '                    spec.sheet_name,\n'
    '                )\n'
    '        else:\n'
    '            LOGGER.info("Dry-run only for %s; no values written.", spec.sheet_name)\n'
    '        return BatchIndicatorResult(spec.sheet_name, True, summary.output_range)\n'
    '    except Exception as exc:\n'
    '        LOGGER.exception("Indicator failed: %s", spec.sheet_name)\n'
    '        return BatchIndicatorResult(spec.sheet_name, False, error=str(exc))\n'
    '\n'
    '\n'
    'def print_batch_summary(results: Sequence[BatchIndicatorResult]) -> None:\n'
    '    ok = [r for r in results if r.ok]\n'
    '    failed = [r for r in results if not r.ok]\n'
    '    if os.getenv("US_PIPELINE_COMPACT_OUTPUT") == "1":\n'
    '        print(\n'
    '            f"Analysis sheets: {len(ok)}/{len(results)} succeeded, "\n'
    '            f"{len(failed)} failed."\n'
    '        )\n'
    '        for result in failed:\n'
    '            print(f"  FAILED {result.sheet_name}: {result.error}")\n'
    '        return\n'
    '    LOGGER.info(\n'
    '        "Batch summary: total=%s succeeded=%s failed=%s",\n'
    '        len(results),\n'
    '        len(ok),\n'
    '        len(failed),\n'
    '    )\n'
    '    for result in ok:\n'
    '        LOGGER.info("Succeeded: %s %s", result.sheet_name, result.output_range)\n'
    '    for result in failed:\n'
    '        LOGGER.error("Failed: %s error=%s", result.sheet_name, result.error)\n'
    '\n'
    '\n'
    'def quote_sheet_name(sheet_name: str) -> str:\n'
    '    escaped = sheet_name.replace("\'", "\'\'")\n'
    '    return f"\'{escaped}\'"\n'
    '\n'
    '\n'
    'def indicator_input_range(\n'
    '    sheet_name: str, input_end_col: str = "B", input_start_col: str = "A"\n'
    ') -> str:\n'
    '    return f"{quote_sheet_name(sheet_name)}!{input_start_col}2:{input_end_col}"\n'
    '\n'
    '\n'
    'def indicator_output_range(\n'
    '    sheet_name: str,\n'
    '    last_data_row: int,\n'
    '    output_end_col: str = "AB",\n'
    '    output_start_col: str = "C",\n'
    ') -> str:\n'
    '    assert_safe_output_range(\n'
    '        f"{output_start_col}2:{output_end_col}{last_data_row}",\n'
    '        output_end_col,\n'
    '        output_start_col,\n'
    '    )\n'
    '    return f"{quote_sheet_name(sheet_name)}!{output_start_col}2:{output_end_col}{last_data_row}"\n'
    '\n'
    '\n'
    'def chunked(items: Sequence[Any], chunk_size: int) -> list[list[Any]]:\n'
    '    if chunk_size < 1:\n'
    '        raise ValueError("chunk_size must be >= 1")\n'
    '    return [list(items[i : i + chunk_size]) for i in range(0, len(items), chunk_size)]\n'
    '\n'
    '\n'
    'def is_quota_error(exc: BaseException) -> bool:\n'
    '    text = str(exc).lower()\n'
    '    status = getattr(getattr(exc, "response", None), "status_code", None)\n'
    '    return status == 429 or "429" in text or "quota" in text\n'
    '\n'
    '\n'
    'def call_with_backoff(\n'
    '    fn: Callable[[], Any],\n'
    '    *,\n'
    '    operation_name: str,\n'
    '    max_attempts: int = 7,\n'
    '    base_sleep_seconds: float = 1.0,\n'
    ') -> Any:\n'
    '    for attempt in range(1, max_attempts + 1):\n'
    '        try:\n'
    '            return fn()\n'
    '        except Exception as exc:\n'
    '            if not is_quota_error(exc) or attempt == max_attempts:\n'
    '                raise\n'
    '            sleep_for = base_sleep_seconds * (2 ** (attempt - 1)) + random.uniform(\n'
    '                0, 0.5\n'
    '            )\n'
    '            LOGGER.warning(\n'
    '                "%s hit quota; retrying in %.1fs attempt %s/%s",\n'
    '                operation_name,\n'
    '                sleep_for,\n'
    '                attempt,\n'
    '                max_attempts,\n'
    '            )\n'
    '            time.sleep(sleep_for)\n'
    '    raise RuntimeError(f"{operation_name} failed unexpectedly")\n'
    '\n'
    '\n'
    'def batch_get_values(\n'
    '    workbook: Any,\n'
    '    ranges: list[str],\n'
    '    *,\n'
    '    value_render_option: str = "UNFORMATTED_VALUE",\n'
    '    chunk_size: int = 20,\n'
    '    sleep_between_batches: float = 0.0,\n'
    '    max_retries: int = 7,\n'
    ') -> dict[str, list[list[object]]]:\n'
    '    results: dict[str, list[list[object]]] = {}\n'
    '    calls = 0\n'
    '    for chunk in chunked(ranges, chunk_size):\n'
    '\n'
    '        def do_get() -> Any:\n'
    '            return workbook.values_batch_get(\n'
    '                ranges=chunk, params={"valueRenderOption": value_render_option}\n'
    '            )\n'
    '\n'
    '        response = call_with_backoff(\n'
    '            do_get,\n'
    '            operation_name=f"values_batch_get[{len(chunk)}]",\n'
    '            max_attempts=max_retries,\n'
    '        )\n'
    '        calls += 1\n'
    '        value_ranges = (\n'
    '            response.get("valueRanges", []) if isinstance(response, dict) else []\n'
    '        )\n'
    '        for requested_range, value_range in zip(chunk, value_ranges):\n'
    '            results[requested_range] = value_range.get("values", [])\n'
    '        for requested_range in chunk:\n'
    '            results.setdefault(requested_range, [])\n'
    '        if sleep_between_batches > 0 and calls * chunk_size < len(ranges):\n'
    '            time.sleep(sleep_between_batches)\n'
    '    LOGGER.info("Batch read %s ranges using %s API calls", len(ranges), calls)\n'
    '    return results\n'
    '\n'
    '\n'
    'def _parse_quoted_sheet_range(a1_range: str) -> tuple[str, str]:\n'
    '    if "!" not in a1_range:\n'
    '        raise ValueError(f"Range must include sheet name: {a1_range}")\n'
    '    sheet_part, cell_range = a1_range.split("!", 1)\n'
    '    if sheet_part.startswith("\'") and sheet_part.endswith("\'"):\n'
    '        sheet_part = sheet_part[1:-1].replace("\'\'", "\'")\n'
    '    return sheet_part, cell_range\n'
    '\n'
    '\n'
    'def batch_update_values(\n'
    '    workbook: Any,\n'
    '    updates: list[dict[str, object]],\n'
    '    *,\n'
    '    value_input_option: str = "RAW",\n'
    '    chunk_size: int = 20,\n'
    '    sleep_between_batches: float = 0.0,\n'
    '    max_retries: int = 7,\n'
    ') -> int:\n'
    '    calls = 0\n'
    '    for chunk in chunked(updates, chunk_size):\n'
    '        from notifications import changes\n'
    '\n'
    '        comparisons = None\n'
    '        if changes.enabled():\n'
    '            try:\n'
    '                previous = batch_get_values(\n'
    '                    workbook,\n'
    '                    [str(item["range"]) for item in chunk],\n'
    '                    chunk_size=chunk_size,\n'
    '                )\n'
    '                comparisons = [\n'
    '                    changes.compare(previous[str(item["range"])], item["values"])\n'
    '                    for item in chunk\n'
    '                ]\n'
    '            except Exception:\n'
    '                comparisons = [{"unknown": True} for item in chunk]\n'
    '\n'
    '        if hasattr(workbook, "values_batch_update"):\n'
    '\n'
    '            def do_update() -> Any:\n'
    '                return workbook.values_batch_update(\n'
    '                    {"valueInputOption": value_input_option, "data": chunk}\n'
    '                )\n'
    '\n'
    '        else:\n'
    '\n'
    '            def do_update() -> None:\n'
    '                for update in chunk:\n'
    '                    sheet_name, cell_range = _parse_quoted_sheet_range(\n'
    '                        str(update["range"])\n'
    '                    )\n'
    '                    workbook.worksheet(sheet_name).update(\n'
    '                        range_name=cell_range,\n'
    '                        values=update["values"],\n'
    '                        value_input_option=value_input_option,\n'
    '                    )\n'
    '\n'
    '        call_with_backoff(\n'
    '            do_update,\n'
    '            operation_name=f"values_batch_update[{len(chunk)}]",\n'
    '            max_attempts=max_retries,\n'
    '        )\n'
    '        if comparisons is not None:\n'
    '            for item, counts in zip(chunk, comparisons):\n'
    '                changes.record(\n'
    '                    "Indicator analysis",\n'
    '                    _parse_quoted_sheet_range(str(item["range"]))[0],\n'
    '                    **counts,\n'
    '                )\n'
    '        calls += 1\n'
    '        if sleep_between_batches > 0 and calls * chunk_size < len(updates):\n'
    '            time.sleep(sleep_between_batches)\n'
    '    LOGGER.info("Batch wrote %s ranges using %s API calls", len(updates), calls)\n'
    '    return calls\n'
    '\n'
    '\n'
    'def read_indicator_values(values: list[list[Any]], input_end_col: str = "B") -> Any:\n'
    '    rows = []\n'
    '    end_col = input_end_col.upper()\n'
    '    has_c = end_col >= "C"\n'
    '    wide = end_col >= "D"\n'
    '    for row in values:\n'
    '        a = row[0] if len(row) > 0 else ""\n'
    '        b = row[1] if len(row) > 1 else ""\n'
    '        c = row[2] if len(row) > 2 else ""\n'
    '        d = row[3] if len(row) > 3 else ""\n'
    '        if (\n'
    '            is_blank(a)\n'
    '            and is_blank(b)\n'
    '            and (not has_c or is_blank(c))\n'
    '            and (not wide or is_blank(d))\n'
    '        ):\n'
    '            break\n'
    '        if wide:\n'
    '            rows.append({"Date": a, "Raw Value": b, "Source C": c, "Source D": d})\n'
    '        elif has_c:\n'
    '            rows.append({"Date": a, "Raw Value": b, "Source C": c})\n'
    '        else:\n'
    '            rows.append({"Date": a, "Raw Value": b})\n'
    '    columns = (\n'
    '        ["Date", "Raw Value", "Source C", "Source D"]\n'
    '        if wide\n'
    '        else (["Date", "Raw Value", "Source C"] if has_c else ["Date", "Raw Value"])\n'
    '    )\n'
    '    return make_dataframe(rows, columns)\n'
    '\n'
    '\n'
    'def read_indicator_ab_values(values: list[list[Any]]) -> Any:\n'
    '    return read_indicator_values(values, "B")\n'
    '\n'
    '\n'
    'def read_indicator_ab(worksheet: Any) -> Any:\n'
    '    return read_indicator_ab_values(worksheet.get("A2:B"))\n'
    '\n'
    '\n'
    'def output_rows(\n'
    '    output_df: Any, output_columns: Sequence[str] | None = None\n'
    ') -> list[list[Any]]:\n'
    '    columns = list(output_columns or getattr(output_df, "columns", OUTPUT_COLUMNS))\n'
    '    return (\n'
    '        output_df[columns].values.tolist()\n'
    '        if hasattr(output_df, "values")\n'
    '        else [[r.get(c, "") for c in columns] for r in df_records(output_df)]\n'
    '    )\n'
    '\n'
    '\n'
    'def dataframe_to_sheet_rows(\n'
    '    output_df: Any, output_columns: Sequence[str] | None = None\n'
    ') -> list[list[Any]]:\n'
    '    return output_rows(output_df, output_columns)\n'
    '\n'
    '\n'
    'def assert_safe_output_range(\n'
    '    output_range: str, output_end_col: str = "AB", output_start_col: str = "C"\n'
    ') -> None:\n'
    '    expected_prefix = f"{output_start_col}2:{output_end_col}"\n'
    '    if output_start_col not in ("C", "D", "E"):\n'
    '        raise AssertionError(f"Unsafe output start column {output_start_col!r}")\n'
    '    if output_start_col == "C" and output_end_col == "AS":\n'
    '        raise AssertionError("C-starting ranges must not use AS; A29 must start at E")\n'
    '    if output_start_col == "D" and output_end_col != "AN":\n'
    '        raise AssertionError(\n'
    '            "D-starting output ranges are reserved for D2:AN interest coverage outputs"\n'
    '        )\n'
    '    if output_start_col == "E" and output_end_col != "AS":\n'
    '        raise AssertionError(\n'
    '            "E-starting output ranges are reserved for E2:AS central bank liquidity outputs"\n'
    '        )\n'
    '    if not output_range.startswith(expected_prefix):\n'
    '        raise AssertionError(\n'
    '            f"Unsafe output range {output_range!r}; expected {expected_prefix}n"\n'
    '        )\n'
    '    try:\n'
    '        last_row = int(output_range.split(f":{output_end_col}", 1)[1])\n'
    '    except Exception as exc:\n'
    '        raise AssertionError(f"Invalid output range {output_range!r}") from exc\n'
    '    if last_row < 2:\n'
    '        raise AssertionError(\n'
    '            f"Unsafe output range {output_range!r}; last row must be >= 2"\n'
    '        )\n'
    '\n'
    '\n'
    'def normalize_grid(\n'
    '    rows: list[list[object]], row_count: int, col_count: int\n'
    ') -> list[list[object]]:\n'
    '    out: list[list[object]] = []\n'
    '    for r in range(row_count):\n'
    '        source = rows[r] if r < len(rows) else []\n'
    '        padded = list(source[:col_count]) + [""] * max(0, col_count - len(source))\n'
    '        out.append(padded)\n'
    '    return out\n'
    '\n'
    '\n'
    'def values_equivalent(expected: object, actual: object, tol: float = 1e-9) -> bool:\n'
    '    if is_blank(expected) and is_blank(actual):\n'
    '        return True\n'
    '    if is_blank(expected) or is_blank(actual):\n'
    '        return False\n'
    '    expected_num = to_float_or_none(expected)\n'
    '    actual_num = to_float_or_none(actual)\n'
    '    if expected_num is not None and actual_num is not None:\n'
    '        return abs(expected_num - actual_num) <= tol\n'
    '    return str(expected).strip() == str(actual).strip()\n'
    '\n'
    '\n'
    'def build_summary(\n'
    '    indicator_sheet_name: str, raw_df: Any, output_df: Any\n'
    ') -> IndicatorRunSummary:\n'
    '    recs = df_records(raw_df)\n'
    '    dates = [str(r.get("Date", "")) for r in recs if not is_blank(r.get("Date", ""))]\n'
    '    last_data_row = len(recs) + 1\n'
    '    return IndicatorRunSummary(\n'
    '        indicator_sheet_name,\n'
    '        len(recs),\n'
    '        len(output_df),\n'
    '        f"C2:AB{last_data_row}",\n'
    '        dates[0] if dates else "",\n'
    '        dates[-1] if dates else "",\n'
    '        sum(to_float_or_none(r.get("Raw Value")) is not None for r in recs),\n'
    '    )\n'
    '\n'
    '\n'
    'def validate_write_range(\n'
    '    output_df: Any,\n'
    '    last_data_row: int,\n'
    '    start_col: str = "C",\n'
    '    start_row: int = 2,\n'
    '    output_columns: Sequence[str] | None = None,\n'
    '    output_end_col: str | None = None,\n'
    ') -> str:\n'
    '    columns = list(output_columns or getattr(output_df, "columns", OUTPUT_COLUMNS))\n'
    '    end_col = output_end_col or output_end_col_for_columns(columns)\n'
    '    if start_row != 2 or start_col not in ("C", "D", "E"):\n'
    '        raise AssertionError(\n'
    '            "Write range must avoid protected row 1 and protected source columns"\n'
    '        )\n'
    '    if start_col == "D" and end_col != "AN":\n'
    '        raise AssertionError("Only interest coverage outputs may start at D")\n'
    '    if start_col == "E" and end_col != "AS":\n'
    '        raise AssertionError("Only central bank liquidity outputs may start at E")\n'
    '    if start_col == "C" and end_col == "AS":\n'
    '        raise AssertionError("Central bank liquidity outputs must start at E")\n'
    '    if len(output_df) != last_data_row - 1:\n'
    '        raise AssertionError("Output row count does not match last_data_row")\n'
    '    if getattr(output_df, "shape")[1] != len(columns):\n'
    '        raise AssertionError(f"Output must have exactly {len(columns)} columns")\n'
    '    output_range = f"{start_col}2:{end_col}{last_data_row}"\n'
    '    assert_safe_output_range(output_range, end_col, start_col)\n'
    '    return output_range\n'
    '\n'
    '\n'
    'def write_indicator_outputs(\n'
    '    worksheet: Any,\n'
    '    output_df: Any,\n'
    '    last_data_row: int,\n'
    '    output_columns: Sequence[str] | None = None,\n'
    '    output_end_col: str | None = None,\n'
    '    output_start_col: str = "C",\n'
    ') -> str:\n'
    '    columns = list(output_columns or getattr(output_df, "columns", OUTPUT_COLUMNS))\n'
    '    end_col = output_end_col or output_end_col_for_columns(columns)\n'
    '    rows = dataframe_to_sheet_rows(output_df, columns)\n'
    '    if output_df.shape[1] != len(columns):\n'
    '        raise AssertionError(\n'
    '            f"Expected {len(columns)} output columns, got {output_df.shape[1]}"\n'
    '        )\n'
    '    if len(rows) != last_data_row - 1:\n'
    '        raise AssertionError(\n'
    '            f"Output row count mismatch: rows={len(rows)}, last_data_row={last_data_row}"\n'
    '        )\n'
    '    output_range = f"{output_start_col}2:{end_col}{last_data_row}"\n'
    '    assert_safe_output_range(output_range, end_col, output_start_col)\n'
    '    from notifications import changes\n'
    '\n'
    '    counts = changes.capture(\n'
    '        lambda: worksheet.get(output_range, value_render_option="UNFORMATTED_VALUE"),\n'
    '        rows,\n'
    '    )\n'
    '    worksheet.update(range_name=output_range, values=rows, value_input_option="RAW")\n'
    '    if counts is not None:\n'
    '        changes.record(\n'
    '            "Indicator analysis", getattr(worksheet, "title", output_range), **counts\n'
    '        )\n'
    '\n'
    '    return output_range\n'
    '\n'
    '\n'
    'def verify_written_outputs(\n'
    '    worksheet: Any,\n'
    '    output_df: Any,\n'
    '    last_data_row: int,\n'
    '    output_columns: Sequence[str] | None = None,\n'
    '    output_end_col: str | None = None,\n'
    '    output_start_col: str = "C",\n'
    ') -> None:\n'
    '    columns = list(output_columns or getattr(output_df, "columns", OUTPUT_COLUMNS))\n'
    '    end_col = output_end_col or output_end_col_for_columns(columns)\n'
    '    expected_rows = dataframe_to_sheet_rows(output_df, columns)\n'
    '    row_count = len(expected_rows)\n'
    '    col_count = output_df.shape[1]\n'
    '    if col_count != len(columns):\n'
    '        raise AssertionError(f"Expected {len(columns)} output columns, got {col_count}")\n'
    '    output_range = f"{output_start_col}2:{end_col}{last_data_row}"\n'
    '    assert_safe_output_range(output_range, end_col, output_start_col)\n'
    '    actual_rows = worksheet.get(output_range, value_render_option="UNFORMATTED_VALUE")\n'
    '    actual_norm = normalize_grid(actual_rows, row_count, col_count)\n'
    '    for r in range(row_count):\n'
    '        for c in range(col_count):\n'
    '            expected = expected_rows[r][c]\n'
    '            actual = actual_norm[r][c]\n'
    '            if not values_equivalent(expected, actual):\n'
    '                sheet_row = r + 2\n'
    '                sheet_col = c + 3\n'
    '                raise AssertionError(\n'
    '                    f"Value mismatch at output row={r}, output col={c}, "\n'
    '                    f"sheet row={sheet_row}, sheet col={sheet_col}: "\n'
    '                    f"actual={actual!r} expected={expected!r}"\n'
    '                )\n'
    '\n'
    '\n'
    'def first_existing_column(df: Any, candidates: Sequence[str]) -> str | None:\n'
    '    columns = set(getattr(df, "columns", []))\n'
    '    for col in candidates:\n'
    '        if col in columns:\n'
    '            return col\n'
    '    return None\n'
    '\n'
    '\n'
    'def count_nonblank_column(df: Any, column_name: str | None) -> int:\n'
    '    if not column_name or column_name not in set(getattr(df, "columns", [])):\n'
    '        return 0\n'
    '    return sum(not is_blank(r.get(column_name, "")) for r in df_records(df))\n'
    '\n'
    '\n'
    'def count_matching_column(df: Any, column_name: str | None, expected: object) -> int:\n'
    '    if not column_name or column_name not in set(getattr(df, "columns", [])):\n'
    '        return 0\n'
    '    return sum(r.get(column_name) == expected for r in df_records(df))\n'
    '\n'
    '\n'
    'def print_summary(summary: IndicatorRunSummary, output_df: Any) -> None:\n'
    '    LOGGER.info("Indicator sheet: %s", summary.indicator_sheet_name)\n'
    '    LOGGER.info(\n'
    '        "Rows read/calculated: %s/%s; date range: %s -> %s",\n'
    '        summary.rows_read,\n'
    '        summary.rows_calculated,\n'
    '        summary.first_date,\n'
    '        summary.last_date,\n'
    '    )\n'
    '    LOGGER.info(\n'
    '        "Output range: %s; valid raw values: %s",\n'
    '        summary.output_range,\n'
    '        summary.nonblank_raw_values,\n'
    '    )\n'
    '    level_col = first_existing_column(output_df, LEVEL_SCORE_COLUMNS)\n'
    '    leading_col = first_existing_column(output_df, LEADING_SCORE_COLUMNS)\n'
    '    warning_col = first_existing_column(output_df, WARNING_SCORE_COLUMNS)\n'
    '    status_col = first_existing_column(output_df, STATUS_COLUMNS)\n'
    '    for label, col in (\n'
    '        ("level", level_col),\n'
    '        ("leading", leading_col),\n'
    '        ("warning", warning_col),\n'
    '        ("status", status_col),\n'
    '    ):\n'
    '        if col is None:\n'
    '            LOGGER.warning(\n'
    '                "Summary column not found for %s: %s",\n'
    '                summary.indicator_sheet_name,\n'
    '                label,\n'
    '            )\n'
    '    level_count = count_nonblank_column(output_df, level_col)\n'
    '    leading_count = count_nonblank_column(output_df, leading_col)\n'
    '    warning_count = count_nonblank_column(output_df, warning_col)\n'
    '    valid_rows = count_matching_column(output_df, status_col, "Valid")\n'
    '    LOGGER.info(\n'
    '        "Valid score counts: level=%s (%s), leading=%s (%s), warning=%s (%s), valid_rows=%s",\n'
    '        level_count,\n'
    '        level_col or "not present",\n'
    '        leading_count,\n'
    '        leading_col or "not present",\n'
    '        warning_count,\n'
    '        warning_col or "not present",\n'
    '        valid_rows,\n'
    '    )\n'
    '\n'
    '\n'
    'def run_self_tests() -> None:\n'
    '    assert "gspread" not in sys.modules\n'
    '    assert percentrank_inc([1, 2, 3], 1) == 0\n'
    '    assert percentrank_inc([1, 2, 3], 3) == 1\n'
    '    assert percentrank_inc([1, 3], 2) == 0.5\n'
    '    assert percentrank_inc([1, 1, 2], 1) == 0\n'
    '    assert percentrank_inc([1], 1) is None\n'
    '    raw = make_dataframe(\n'
    '        [{"Date": "d0", "Raw Value": ""}]\n'
    '        + [\n'
    '            {"Date": f"d{i}", "Raw Value": 50 + ((i % 20) - 10) / 2}\n'
    '            for i in range(1, 130)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    out = calculate_manufacturing_pmi_outputs(raw)\n'
    '    new_orders_out = calculate_manufacturing_new_orders_outputs(raw)\n'
    '    services_out = calculate_services_pmi_outputs(raw)\n'
    '    rows = df_records(out)\n'
    '    new_orders_rows = df_records(new_orders_out)\n'
    '    services_rows = df_records(services_out)\n'
    '    assert out.shape[1] == 26\n'
    '    assert new_orders_out.shape[1] == 26\n'
    '    assert services_out.shape[1] == 26\n'
    '    for pmi_row, services_row in zip(rows, services_rows):\n'
    '        assert [pmi_row[col] for col in OUTPUT_COLUMNS[:19]] == [\n'
    '            services_row[col] for col in OUTPUT_COLUMNS[:19]\n'
    '        ]\n'
    '    assert (\n'
    '        list(getattr(new_orders_out, "columns", []))\n'
    '        == MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS\n'
    '    )\n'
    '    assert (\n'
    '        rows[0]["Observation Count"] == "" and rows[0]["Data Status"] == "Missing Data"\n'
    '    )\n'
    '    assert (\n'
    '        rows[1]["Observation Count"] == 1\n'
    '        and rows[1]["Distance From 50"] == -4.5\n'
    '        and rows[1]["Level Score Raw"] == -9\n'
    '    )\n'
    '    assert rows[1]["1M Change"] == rows[3]["3M Change"] == rows[6]["6M Change"] == ""\n'
    '    assert rows[58]["Level PercentRank"] == ""\n'
    '    assert (\n'
    '        0 <= rows[60]["Level PercentRank"] <= 1 and -10 <= rows[60]["Level Score"] <= 10\n'
    '    )\n'
    '    later = next(r for r in rows[70:] if r["Leading Score"] != "")\n'
    '    assert (\n'
    '        -10 <= later["3M Momentum Score"] <= 10\n'
    '        and -10 <= later["6M Momentum Score"] <= 10\n'
    '    )\n'
    '    assert (\n'
    '        _early(49, 3, 1) == 8\n'
    '        and _threshold(50, 49, 1, 1) == 6\n'
    '        and _overheat(56, -1, -1) == -5\n'
    '        and _warning_raw(58, -3, -3) == -8\n'
    '    )\n'
    '    assert (\n'
    '        later["Level Composite Score"] != ""\n'
    '        and later["Leading Score"] != ""\n'
    '        and later["Warning Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        later["Regime"] in ("Expansion", "Contraction")\n'
    '        and later["Reliability"] in ("Medium reliability", "High reliability")\n'
    '        and later["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    )\n'
    '    assert (\n'
    '        later["Warning Risk Label"] != ""\n'
    '        and later["Manufacturing Signal Label"] != ""\n'
    '        and later["Signal Interpretation"] != ""\n'
    '        and later["Regime Momentum Label"] != ""\n'
    '    )\n'
    '    new_later = next(r for r in new_orders_rows[70:] if r["Leading Score"] != "")\n'
    '    assert new_later["Regime"].startswith("New orders ")\n'
    '    assert "new orders" in new_later["Signal Comment"]\n'
    '    assert "new orders" in new_later["Warning Flag"]\n'
    '    assert "new orders" in new_later["Slowdown Risk Flag"]\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 3, NEW_ORDERS_LABELS)\n'
    '        == "New orders expansion accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, -3, NEW_ORDERS_LABELS)\n'
    '        == "New orders expansion slowing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 0, NEW_ORDERS_LABELS)\n'
    '        == "New orders expansion stable"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 3, NEW_ORDERS_LABELS)\n'
    '        == "New orders contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, -3, NEW_ORDERS_LABELS)\n'
    '        == "New orders contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 0, NEW_ORDERS_LABELS)\n'
    '        == "New orders contraction stable"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, 0, NEW_ORDERS_LABELS)\n'
    '        == "Strong positive new orders leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, -1, NEW_ORDERS_LABELS)\n'
    '        == "Strong positive new orders leading signal, but warning risk is negative"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, 0, NEW_ORDERS_LABELS)\n'
    '        == "Moderately positive new orders leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, -1, NEW_ORDERS_LABELS)\n'
    '        == "Positive new orders leading signal, but fragile"\n'
    '    )\n'
    '    assert _signal(0, 0, NEW_ORDERS_LABELS) == "Neutral/mixed new orders signal"\n'
    '    assert (\n'
    '        _signal(-6, 0, NEW_ORDERS_LABELS)\n'
    '        == "Strong negative new orders leading signal with downside warning confirmation"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-6, 1, NEW_ORDERS_LABELS)\n'
    '        == "Strong negative new orders leading signal, but warning profile is improving"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 0, NEW_ORDERS_LABELS)\n'
    '        == "Moderately negative new orders leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 1, NEW_ORDERS_LABELS)\n'
    '        == "Negative new orders leading signal, but stabilisation risk is present"\n'
    '    )\n'
    '    assert _warning_label(-6, NEW_ORDERS_LABELS) == "High new orders warning risk"\n'
    '    assert _warning_label(-3, NEW_ORDERS_LABELS) == "Moderate new orders warning risk"\n'
    '    assert _warning_label(0, NEW_ORDERS_LABELS) == "Neutral new orders warning profile"\n'
    '    assert (\n'
    '        _warning_label(3, NEW_ORDERS_LABELS)\n'
    '        == "Constructive new orders warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(6, NEW_ORDERS_LABELS)\n'
    '        == "Strong positive new orders turn / warning cleared"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(47, -1, -1, NEW_ORDERS_LABELS)\n'
    '        == "High new orders contraction risk"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, -1, 1, NEW_ORDERS_LABELS)\n'
    '        == "New orders contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, 1, 1, NEW_ORDERS_LABELS)\n'
    '        == "New orders contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, -1, 1, NEW_ORDERS_LABELS)\n'
    '        == "New orders expansion losing momentum"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, 1, 1, NEW_ORDERS_LABELS)\n'
    '        == "New orders expansion strengthening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(50, 0, 0, NEW_ORDERS_LABELS)\n'
    '        == "Mixed new orders signal"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped manufacturing new orders sheet",\n'
    '            CONTROL_PANEL_SECOND_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_manufacturing_new_orders_outputs\n'
    '    )\n'
    '    a4_spec = select_indicator_engine_spec(\n'
    '        "Unmapped manufacturing new orders sheet", CONTROL_PANEL_SECOND_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        a4_spec.input_start_col == "A"\n'
    '        and a4_spec.input_end_col == "B"\n'
    '        and a4_spec.output_start_col == "C"\n'
    '        and a4_spec.output_end_col == "AB"\n'
    '    )\n'
    '    assert (\n'
    '        output_columns_for_engine(calculate_manufacturing_new_orders_outputs)\n'
    '        == MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS\n'
    '    )\n'
    '    assert output_end_col_for_columns(MANUFACTURING_NEW_ORDERS_OUTPUT_COLUMNS) == "AB"\n'
    '    a4_values = (\n'
    '        [100] + [101 + i for i in range(60)] + [101] + [130 + i for i in range(58)]\n'
    '    )\n'
    '    a4_values += [56.5, 47.4, 56.0, 55.0, 55.8, 56.2, 54.1, ""]\n'
    '    a4_raw = make_dataframe(\n'
    '        [{"Date": f"a4-{i}", "Raw Value": value} for i, value in enumerate(a4_values)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    a4_out = calculate_manufacturing_new_orders_outputs(a4_raw)\n'
    '    a4_rows = df_records(a4_out)\n'
    '    assert a4_out.shape[1] == 26\n'
    '    assert a4_rows[61]["Level Percentile"] == 0.016\n'
    '    assert a4_rows[61]["Level Score"] == -9.68\n'
    '    assert a4_rows[61]["Coincident Score"] == round(\n'
    '        clamp(\n'
    '            0.7 * a4_rows[61]["Level Score"]\n'
    '            + 0.3 * a4_rows[61]["Distance from 50 Score"]\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    blank_a4 = a4_rows[-1]\n'
    '    assert values_equivalent(blank_a4["1M Change"], -54.1)\n'
    '    assert values_equivalent(blank_a4["3M Change"], -55.8)\n'
    '    assert values_equivalent(blank_a4["6M Change"], -47.4)\n'
    '    assert (\n'
    '        blank_a4["Sample Count"]\n'
    '        == blank_a4["Distance from 50"]\n'
    '        == blank_a4["Distance from 50 Score"]\n'
    '        == ""\n'
    '    )\n'
    '    assert (\n'
    '        blank_a4["Level Percentile"]\n'
    '        == blank_a4["Level Score"]\n'
    '        == blank_a4["State"]\n'
    '        == blank_a4["Reliability"]\n'
    '        == ""\n'
    '    )\n'
    '    assert blank_a4["3M Momentum Percentile"] != "" and blank_a4[\n'
    '        "3M Momentum Score"\n'
    '    ] == round(blank_a4["3M Momentum Percentile"] * 20 - 10, 2)\n'
    '    assert blank_a4["6M Momentum Percentile"] != "" and blank_a4[\n'
    '        "6M Momentum Score"\n'
    '    ] == round(blank_a4["6M Momentum Percentile"] * 20 - 10, 2)\n'
    '    assert blank_a4["Signal Validity"] == "Missing Data"\n'
    '    assert all(\n'
    '        row["Signal Comment"] != "Moderately negative leading signal" for row in a4_rows\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 0, NEW_ORDERS_LABELS)\n'
    '        == "Moderately negative new orders leading signal"\n'
    '    )\n'
    '    services_later = next(r for r in services_rows[70:] if r["Leading Score"] != "")\n'
    '    assert services_later["Regime Momentum Label"].startswith("Services ")\n'
    '    assert "services" in services_later["Signal Interpretation"]\n'
    '    assert "services" in services_later["Warning Risk Label"]\n'
    '    assert "services" in services_later["Manufacturing Signal Label"]\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 3, SERVICES_PMI_LABELS)\n'
    '        == "Services expansion accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, -3, SERVICES_PMI_LABELS)\n'
    '        == "Services expansion slowing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 0, SERVICES_PMI_LABELS)\n'
    '        == "Services expansion stable"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 3, SERVICES_PMI_LABELS)\n'
    '        == "Services contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, -3, SERVICES_PMI_LABELS)\n'
    '        == "Services contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 0, SERVICES_PMI_LABELS)\n'
    '        == "Services contraction stable"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, 0, SERVICES_PMI_LABELS)\n'
    '        == "Strong positive services leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, -1, SERVICES_PMI_LABELS)\n'
    '        == "Strong positive services leading signal, but warning risk is negative"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, 0, SERVICES_PMI_LABELS)\n'
    '        == "Moderately positive services leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, -1, SERVICES_PMI_LABELS)\n'
    '        == "Positive services leading signal, but fragile"\n'
    '    )\n'
    '    assert _signal(0, 0, SERVICES_PMI_LABELS) == "Neutral/mixed services signal"\n'
    '    assert (\n'
    '        _signal(-6, 0, SERVICES_PMI_LABELS)\n'
    '        == "Strong negative services leading signal with downside warning confirmation"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-6, 1, SERVICES_PMI_LABELS)\n'
    '        == "Strong negative services leading signal, but warning profile is improving"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 0, SERVICES_PMI_LABELS)\n'
    '        == "Moderately negative services leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 1, SERVICES_PMI_LABELS)\n'
    '        == "Negative services leading signal, but stabilisation risk is present"\n'
    '    )\n'
    '    assert _warning_label(-6, SERVICES_PMI_LABELS) == "High services warning risk"\n'
    '    assert _warning_label(-3, SERVICES_PMI_LABELS) == "Moderate services warning risk"\n'
    '    assert _warning_label(0, SERVICES_PMI_LABELS) == "Neutral services warning profile"\n'
    '    assert (\n'
    '        _warning_label(3, SERVICES_PMI_LABELS)\n'
    '        == "Constructive services warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(6, SERVICES_PMI_LABELS)\n'
    '        == "Strong positive services turn / warning cleared"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(47, -1, -1, SERVICES_PMI_LABELS)\n'
    '        == "High services contraction risk"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, -1, 1, SERVICES_PMI_LABELS)\n'
    '        == "Services contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, 1, 1, SERVICES_PMI_LABELS)\n'
    '        == "Services contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, -1, 1, SERVICES_PMI_LABELS)\n'
    '        == "Services expansion losing momentum"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, 1, 1, SERVICES_PMI_LABELS)\n'
    '        == "Services expansion strengthening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(50, 0, 0, SERVICES_PMI_LABELS)\n'
    '        == "Mixed services signal"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped services sheet", CONTROL_PANEL_THIRD_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_services_pmi_outputs\n'
    '    )\n'
    '    services_new_orders_out = calculate_services_new_orders_outputs(raw)\n'
    '    services_new_orders_rows = df_records(services_new_orders_out)\n'
    '    assert services_new_orders_out.shape[1] == 26\n'
    '    for pmi_row, sno_row in zip(rows, services_new_orders_rows):\n'
    '        assert [pmi_row[col] for col in OUTPUT_COLUMNS[:19]] == [\n'
    '            sno_row[col] for col in OUTPUT_COLUMNS[:19]\n'
    '        ]\n'
    '    services_new_orders_later = next(\n'
    '        r for r in services_new_orders_rows[70:] if r["Leading Score"] != ""\n'
    '    )\n'
    '    assert services_new_orders_later["Regime Momentum Label"].startswith(\n'
    '        "Services new orders "\n'
    '    )\n'
    '    assert "services new orders" in services_new_orders_later["Signal Interpretation"]\n'
    '    assert "services new orders" in services_new_orders_later["Warning Risk Label"]\n'
    '    assert (\n'
    '        "services new orders" in services_new_orders_later["Manufacturing Signal Label"]\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 3, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders expansion accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, -3, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders expansion slowing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders expansion stable"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 3, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, -3, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders contraction stable"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Strong positive services new orders leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, -1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Strong positive services new orders leading signal, but warning risk is negative"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Moderately positive services new orders leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, -1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Positive services new orders leading signal, but fragile"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(0, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Neutral/mixed services new orders signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-6, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Strong negative services new orders leading signal with downside warning confirmation"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-6, 1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Strong negative services new orders leading signal, but warning profile is improving"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Moderately negative services new orders leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Negative services new orders leading signal, but stabilisation risk is present"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(-6, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "High services new orders warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(-3, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Moderate services new orders warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Neutral services new orders warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(3, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Constructive services new orders warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(6, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Strong positive services new orders turn / warning cleared"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(47, -1, -1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "High services new orders contraction risk"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, -1, 1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, 1, 1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, -1, 1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders expansion losing momentum"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, 1, 1, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Services new orders expansion strengthening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(50, 0, 0, SERVICES_NEW_ORDERS_LABELS)\n'
    '        == "Mixed services new orders signal"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped services new orders sheet", CONTROL_PANEL_FOURTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_services_new_orders_outputs\n'
    '    )\n'
    '    services_business_activity_out = calculate_services_business_activity_outputs(raw)\n'
    '    services_business_activity_rows = df_records(services_business_activity_out)\n'
    '    assert services_business_activity_out.shape[1] == 26\n'
    '    for pmi_row, sba_row in zip(rows, services_business_activity_rows):\n'
    '        assert [pmi_row[col] for col in OUTPUT_COLUMNS[:19]] == [\n'
    '            sba_row[col] for col in OUTPUT_COLUMNS[:19]\n'
    '        ]\n'
    '    services_business_activity_later = next(\n'
    '        r for r in services_business_activity_rows[70:] if r["Leading Score"] != ""\n'
    '    )\n'
    '    assert services_business_activity_later["Regime Momentum Label"].startswith(\n'
    '        "Services business activity "\n'
    '    )\n'
    '    assert (\n'
    '        "services business activity"\n'
    '        in services_business_activity_later["Signal Interpretation"]\n'
    '    )\n'
    '    assert (\n'
    '        "services business activity"\n'
    '        in services_business_activity_later["Warning Risk Label"]\n'
    '    )\n'
    '    assert (\n'
    '        "services business activity"\n'
    '        in services_business_activity_later["Manufacturing Signal Label"]\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 3, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity expansion accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, -3, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity expansion slowing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(51, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity expansion stable"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 3, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, -3, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _regime_momentum_label(49, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity contraction stable"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Strong positive services business activity leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(6, -1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Strong positive services business activity leading signal, but warning risk is negative"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Moderately positive services business activity leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(2, -1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Positive services business activity leading signal, but fragile"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(0, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Neutral/mixed services business activity signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-6, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Strong negative services business activity leading signal with downside warning confirmation"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-6, 1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Strong negative services business activity leading signal, but warning profile is improving"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Moderately negative services business activity leading signal"\n'
    '    )\n'
    '    assert (\n'
    '        _signal(-2, 1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Negative services business activity leading signal, but stabilisation risk is present"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(-6, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "High services business activity warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(-3, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Moderate services business activity warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Neutral services business activity warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(3, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Constructive services business activity warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _warning_label(6, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Strong positive services business activity turn / warning cleared"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(47, -1, -1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "High services business activity contraction risk"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, -1, 1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity contraction worsening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(49, 1, 1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity contraction easing"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, -1, 1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity expansion losing momentum"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(51, 1, 1, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Services business activity expansion strengthening"\n'
    '    )\n'
    '    assert (\n'
    '        _manufacturing_signal_label(50, 0, 0, SERVICES_BUSINESS_ACTIVITY_LABELS)\n'
    '        == "Mixed services business activity signal"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped services business activity sheet",\n'
    '            CONTROL_PANEL_FIFTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_services_business_activity_outputs\n'
    '    )\n'
    '    all_specs_with_a7 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["Services Business Activity"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Services Business Activity", "A7") in all_specs_with_a7\n'
    '    industrial_raw = make_dataframe(\n'
    '        [{"Date": f"m{i}", "Raw Value": 100 + i} for i in range(140)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    industrial_out = calculate_industrial_production_outputs(industrial_raw)\n'
    '    industrial_rows = df_records(industrial_out)\n'
    '    assert industrial_out.shape[1] == 34\n'
    '    assert output_end_col_for_columns(INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert (\n'
    '        indicator_output_range("Industrial Production", 678, "AJ")\n'
    '        == "\'Industrial Production\'!C2:AJ678"\n'
    '    )\n'
    '    assert industrial_rows[0]["3M MA"] == "" and industrial_rows[1]["3M MA"] == ""\n'
    '    assert industrial_rows[2]["3M MA"] == 101\n'
    '    assert (\n'
    '        industrial_rows[2]["Sample Count"] == 1\n'
    '        and industrial_rows[3]["Sample Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(industrial_rows[3]["1M % Change"], 102 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        industrial_rows[5]["3M Annualised Growth"], (104 / 101) ** 4 - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        industrial_rows[8]["6M Annualised Growth"], (107 / 101) ** 2 - 1\n'
    '    )\n'
    '    assert values_equivalent(industrial_rows[14]["YoY Growth"], 113 / 101 - 1)\n'
    '    assert industrial_rows[17][\n'
    '        "3M Change in YoY Growth"\n'
    '    ] == _industrial_percent_display(\n'
    '        industrial_rows[17]["YoY Growth"] - industrial_rows[14]["YoY Growth"]\n'
    '    )\n'
    '    assert industrial_rows[20][\n'
    '        "6M Change in YoY Growth"\n'
    '    ] == _industrial_percent_display(\n'
    '        industrial_rows[20]["YoY Growth"] - industrial_rows[14]["YoY Growth"]\n'
    '    )\n'
    '    assert industrial_rows[100]["YoY Growth Percentile"] == ""\n'
    '    assert industrial_rows[100]["YoY Growth Z-Score"] == ""\n'
    '    a8_spec = select_indicator_engine_spec(\n'
    '        "Unmapped industrial sheet", CONTROL_PANEL_SIXTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert a8_spec.name == "INDUSTRIAL_PRODUCTION_ENGINE"\n'
    '    assert (\n'
    '        a8_spec.input_start_col == "A"\n'
    '        and a8_spec.input_end_col == "B"\n'
    '        and a8_spec.output_start_col == "C"\n'
    '        and a8_spec.output_end_col == "AJ"\n'
    '    )\n'
    '    assert list(industrial_out.columns) == INDUSTRIAL_PRODUCTION_OUTPUT_COLUMNS\n'
    '    assert _industrial_percent_display(-0.06420729343) == "-6.42%"\n'
    '    assert _industrial_percent_display(-0.1951981579) == "-19.52%"\n'
    '    assert _industrial_percent_display(0.03575131074) == "3.58%"\n'
    '    assert (\n'
    '        _industrial_percent_rank_rounded_3([float(x) for x in range(59)], 43.0, 119)\n'
    '        == ""\n'
    '    )\n'
    '    assert (\n'
    '        _industrial_percent_rank_rounded_3([float(x) for x in range(59)], 43.0, 120)\n'
    '        == 0.741\n'
    '    )\n'
    '    assert _score_percent_rank(0.741) == 4.82\n'
    '    assert (\n'
    '        _industrial_percent_rank_rounded_3([float(x) for x in range(114)], 89.0, 120)\n'
    '        == 0.788\n'
    '    )\n'
    '    assert _score_percent_rank(0.788) == 5.76\n'
    '    assert (\n'
    '        _industrial_percent_rank_rounded_3([float(x) for x in range(58)], 40.0, 120)\n'
    '        == 0.702\n'
    '    )\n'
    '    assert _score_percent_rank(0.702) == 4.04\n'
    '    assert _industrial_downside_warning_score("", 4.82, 5.76, 4.04) == ""\n'
    '    assert _industrial_downside_warning_score(5.14, 4.82, 5.76, 4.04) == 5.02\n'
    '    assert _industrial_downside_warning_score(3.86, -0.08, 1.66, -2.44) == 1.49\n'
    '    assert round(clamp(0.30 * 4 + 0.50 * 5.02 + 0.20 * 0), 2) == 3.71\n'
    '    assert round(clamp(0.30 * -5 + 0.50 * 1.49 + 0.20 * 0), 2) == -0.76\n'
    '    formula_row = next(\n'
    '        r for r in industrial_rows if r["Industrial Downside Warning Score"] != ""\n'
    '    )\n'
    '    assert formula_row["Industrial Downside Warning Score"] == round(\n'
    '        clamp(\n'
    '            0.40 * formula_row["YoY Growth Score"]\n'
    '            + 0.25 * formula_row["3M Growth Score"]\n'
    '            + 0.20 * formula_row["6M Growth Score"]\n'
    '            + 0.15 * formula_row["Momentum Score"]\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert formula_row["Coincident Score"] == round(\n'
    '        clamp(\n'
    '            0.45 * formula_row["YoY Growth Score"]\n'
    '            + 0.25 * formula_row["3M Growth Score"]\n'
    '            + 0.20 * formula_row["6M Growth Score"]\n'
    '            + 0.10 * formula_row["Momentum Score"]\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert formula_row["Leading Score"] == round(\n'
    '        clamp(\n'
    '            0.10 * formula_row["YoY Growth Score"]\n'
    '            + 0.25 * formula_row["3M Growth Score"]\n'
    '            + 0.15 * formula_row["6M Growth Score"]\n'
    '            + 0.45 * formula_row["Momentum Score"]\n'
    '            + 0.05 * formula_row["Industrial Cycle Turning Point Score"]\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert formula_row["Warning Score"] == round(\n'
    '        clamp(\n'
    '            0.30 * formula_row["Industrial Cycle Turning Point Score"]\n'
    '            + 0.50 * formula_row["Industrial Downside Warning Score"]\n'
    '            + 0.20 * formula_row["Industrial Extreme Score"]\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (\n'
    '        _industrial_signal_interpretation(0, 0)\n'
    '        == "Neutral/mixed industrial production signal"\n'
    '    )\n'
    '    assert (\n'
    '        _industrial_warning_label(3.71)\n'
    '        == "Constructive industrial production warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _industrial_warning_label(-0.76)\n'
    '        == "Neutral industrial production warning profile"\n'
    '    )\n'
    '    sample_z = expanding_zscore_sample([1, 2, 3], [120, 120, 120], 120)\n'
    '    assert values_equivalent(sample_z[2], 1.0)\n'
    '    assert _industrial_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert _industrial_outlier_label([3.1, -3.5, 0, 0]) == "Extreme positive outlier"\n'
    '    later_industrial = industrial_rows[-1]\n'
    '    assert later_industrial["Coincident Score"] != ""\n'
    '    assert later_industrial["Leading Score"] != ""\n'
    '    assert later_industrial["Warning Score"] != ""\n'
    '    assert later_industrial["State"] in ("Expansion", "Contraction")\n'
    '    assert later_industrial["Regime"] != ""\n'
    '    assert later_industrial["Signal Comment"] != ""\n'
    '    assert later_industrial["Reliability"] == "High reliability"\n'
    '    assert later_industrial["Signal Validity"] in ("Valid", "Incomplete Score")\n'
    '    assert later_industrial["Warning Flag"] != ""\n'
    '    assert later_industrial["Slowdown Risk Flag"] != ""\n'
    '    assert later_industrial["Extreme Flag"] != ""\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped industrial sheet", CONTROL_PANEL_SIXTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_industrial_production_outputs\n'
    '    )\n'
    '    consumer_out = calculate_consumer_spending_outputs(industrial_raw)\n'
    '    consumer_rows = df_records(consumer_out)\n'
    '    assert consumer_out.shape[1] == 34\n'
    '    assert output_end_col_for_columns(CONSUMER_SPENDING_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert (\n'
    '        consumer_rows[0]["3M Moving Average"] == ""\n'
    '        and consumer_rows[1]["3M Moving Average"] == ""\n'
    '    )\n'
    '    assert consumer_rows[2]["3M Moving Average"] == 101\n'
    '    assert (\n'
    '        consumer_rows[2]["Smoothed Observation Count"] == 1\n'
    '        and consumer_rows[3]["Smoothed Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(consumer_rows[3]["1M Smoothed Growth"], 102 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        consumer_rows[5]["3M Annualised Smoothed Growth"], (104 / 101) ** 4 - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        consumer_rows[8]["6M Annualised Smoothed Growth"], (107 / 101) ** 2 - 1\n'
    '    )\n'
    '    assert values_equivalent(consumer_rows[14]["12M Smoothed Growth"], 113 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        consumer_rows[17]["3M Change in 12M Growth"],\n'
    '        consumer_rows[17]["12M Smoothed Growth"]\n'
    '        - consumer_rows[14]["12M Smoothed Growth"],\n'
    '    )\n'
    '    assert consumer_rows[73]["12M Growth PercentRank"] != ""\n'
    '    assert consumer_rows[73]["12M Growth Z-Score"] != ""\n'
    '    assert consumer_rows[73]["Data Status"] != "Low Sample"\n'
    '    assert _industrial_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert _industrial_outlier_label([3.1, -3.5, 0, 0]) == "Extreme positive outlier"\n'
    '    later_consumer = consumer_rows[-1]\n'
    '    assert later_consumer["Consumer Spending Leading Score"] != ""\n'
    '    assert later_consumer["Consumer Spending Warning Composite Score"] != ""\n'
    '    assert later_consumer["Consumer Spending Final Warning Score"] != ""\n'
    '    assert later_consumer["Consumer Spending Regime"] in ("Expansion", "Contraction")\n'
    '    assert later_consumer["Consumer Spending Regime Momentum Label"] != ""\n'
    '    assert later_consumer["Consumer Spending Signal Interpretation"] != ""\n'
    '    assert later_consumer["Reliability"] == "High reliability"\n'
    '    assert later_consumer["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert later_consumer["Consumer Spending Warning Risk Label"] != ""\n'
    '    assert later_consumer["Consumer Spending Signal Label"] != ""\n'
    '    assert later_consumer["Consumer Spending Outlier Label"] != ""\n'
    '    assert (\n'
    '        _consumer_regime_momentum_label(1, 3)\n'
    '        == "Consumer spending expansion accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        _consumer_signal_interpretation(6, 0)\n'
    '        == "Strong positive consumer spending leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _consumer_warning_label(-6) == "High consumer spending warning risk"\n'
    '    assert _consumer_signal_label(-1, 3, -7) == "High consumer slowdown/recession risk"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped consumer sheet", CONTROL_PANEL_SEVENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_consumer_spending_outputs\n'
    '    )\n'
    '    sentiment_values = [100 + (0.4 * i) + (8 * math.sin(i / 5)) for i in range(140)]\n'
    '    sentiment_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"s{i}", "Raw Value": value}\n'
    '            for i, value in enumerate(sentiment_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    sentiment_out = calculate_consumer_sentiment_outputs(sentiment_raw)\n'
    '    sentiment_rows = df_records(sentiment_out)\n'
    '    assert sentiment_out.shape[1] == 29\n'
    '    assert output_end_col_for_columns(CONSUMER_SENTIMENT_OUTPUT_COLUMNS) == "AE"\n'
    '    assert (\n'
    '        indicator_output_range("Consumer Sentiment", 678, "AE")\n'
    '        == "\'Consumer Sentiment\'!C2:AE678"\n'
    '    )\n'
    '    assert (\n'
    '        sentiment_rows[0]["Sample Count"] == 1\n'
    '        and sentiment_rows[59]["Sample Count"] == 60\n'
    '    )\n'
    '    assert sentiment_rows[58]["Distance from historical median"] == ""\n'
    '    assert sentiment_rows[59]["Distance from historical median"] != ""\n'
    '    assert values_equivalent(\n'
    '        sentiment_rows[3]["1M Change"], sentiment_values[3] - sentiment_values[2]\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        sentiment_rows[6]["3M Change"], sentiment_values[6] - sentiment_values[3]\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        sentiment_rows[9]["6M Change"], sentiment_values[9] - sentiment_values[3]\n'
    '    )\n'
    '    assert (\n'
    '        sentiment_rows[58]["Level Percentile"] == ""\n'
    '        and sentiment_rows[59]["Level Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        sentiment_rows[61]["3M Momentum Percentile"] == ""\n'
    '        and sentiment_rows[62]["3M Momentum Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        sentiment_rows[64]["6M Momentum Percentile"] == ""\n'
    '        and sentiment_rows[65]["6M Momentum Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        sentiment_rows[59]["Coincident Score"] != ""\n'
    '        and sentiment_rows[59]["Leading Score"] == ""\n'
    '    )\n'
    '    a10_spec = select_indicator_engine_spec(\n'
    '        "Unmapped sentiment sheet", CONTROL_PANEL_EIGHTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert a10_spec.name == "CONSUMER_SENTIMENT_ENGINE"\n'
    '    assert (\n'
    '        a10_spec.input_start_col == "A"\n'
    '        and a10_spec.input_end_col == "B"\n'
    '        and a10_spec.output_start_col == "C"\n'
    '        and a10_spec.output_end_col == "AE"\n'
    '    )\n'
    '    assert list(sentiment_out.columns) == CONSUMER_SENTIMENT_OUTPUT_COLUMNS\n'
    '    assert _sentiment_sheets_round(0.0875, 3) == 0.088\n'
    '    assert _sentiment_sheets_round(0.0075, 3) == 0.008\n'
    '    assert _sentiment_sheets_round(0.0625, 3) == 0.063\n'
    '    assert _sentiment_sheets_round(-8.245, 2) == -8.25\n'
    '    assert _sentiment_score_percent_rank(_sentiment_sheets_round(0.0875, 3)) == -8.24\n'
    '    assert (\n'
    '        _sentiment_percent_rank_rounded_3([float(x) for x in range(60)], 35.0) == 0.593\n'
    '    )\n'
    '    assert _sentiment_percent_display(0.593) == "59.30%"\n'
    '    assert _sentiment_score_percent_rank(0.593) == 1.86\n'
    '    assert (\n'
    '        _sentiment_percent_rank_rounded_3([float(x) for x in range(60)], 53.0) == 0.898\n'
    '    )\n'
    '    assert _sentiment_score_percent_rank(0.898) == 7.96\n'
    '    assert (\n'
    '        _sentiment_percent_rank_rounded_3([float(x) for x in range(118)], 41.0) == 0.35\n'
    '    )\n'
    '    assert _sentiment_score_percent_rank(0.35) == -3.0\n'
    '    assert _sentiment_regime(-3.0) == "Weak sentiment"\n'
    '    assert _sentiment_momentum_label(7.06, -2.0) == "Strong sentiment, weakening"\n'
    '    assert _sentiment_turn_score(100, 0.2, 1, 1) == 5\n'
    '    assert _sentiment_warning_raw(-1, -1, -1, -3.0, -9.02) == -6\n'
    '    assert _sentiment_warning_raw(1, -1, 1, 7.06, -2.0) == -3\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 1.86 + 0.05 * 3.65), 2) == 1.58\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 1.00 + 0.05 * 2.00), 2) == 0.85\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 5.08 + 0.05 * 5.75), 2) == 4.10\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 8.38 + 0.05 * 11.50), 2) == 6.86\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 9.70 + 0.05 * 21.85), 2) == 8.37\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 9.10 + 0.05 * 20.20), 2) == 7.84\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 8.38 + 0.05 * 13.90), 2) == 6.98\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 7.74 + 0.05 * 6.50), 2) == 6.13\n'
    '    assert (\n'
    '        _sentiment_sheets_round(clamp(0.25 * -6 + 0.25 * 0 + 0.35 * -6 + 0.15 * 0), 2)\n'
    '        == -3.6\n'
    '    )\n'
    '    assert (\n'
    '        _sentiment_sheets_round(clamp(0.25 * 0 + 0.25 * 0 + 0.35 * -3 + 0.15 * 0), 2)\n'
    '        == -1.05\n'
    '    )\n'
    '    assert (\n'
    '        _sentiment_sheets_round(clamp(0.25 * 6 + 0.25 * 5 + 0.35 * 0 + 0.15 * 0), 2)\n'
    '        == 2.75\n'
    '    )\n'
    '    assert (\n'
    '        _sentiment_sheets_round(\n'
    '            clamp(0.20 * 6 + 0.40 * 6 + 0.25 * 6 + 0.05 * 6 + 0.10 * 6), 2\n'
    '        )\n'
    '        == 6.0\n'
    '    )\n'
    '    assert (\n'
    '        _sentiment_signal_interpretation(6.0, 0)\n'
    '        == "Strong positive consumer sentiment leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _sentiment_warning_label(-3.6) == "Moderate consumer sentiment warning risk"\n'
    '    assert _sentiment_signal_label(-3.0, -9.02) == "Consumer confidence deteriorating"\n'
    '    assert _sentiment_signal_label(7.06, -2.0) == "Strong sentiment losing momentum"\n'
    '    assert (\n'
    '        sentiment_rows[58]["Level Z-Score"] == ""\n'
    '        and sentiment_rows[59]["Level Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        sentiment_rows[59]["3M Change Z-Score"] != ""\n'
    '        and sentiment_rows[59]["6M Change Z-Score"] != ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expanding_zscore_sample([1, 2, 3], [60, 60, 60], 60)[2], 1.0\n'
    '    )\n'
    '    assert _sentiment_outlier_score([-3.1, 3.5, 0]) == -6\n'
    '    assert _sentiment_outlier_label([3.1, -3.5, 0]) == "Extreme positive outlier"\n'
    '    later_sentiment = sentiment_rows[80]\n'
    '    assert later_sentiment["Coincident Score"] != ""\n'
    '    assert later_sentiment["Leading Score"] != ""\n'
    '    assert later_sentiment["Warning Score"] != ""\n'
    '    assert later_sentiment["State"] in (\n'
    '        "Strong sentiment",\n'
    '        "Weak sentiment",\n'
    '        "Neutral sentiment",\n'
    '    )\n'
    '    assert later_sentiment["Regime"] != ""\n'
    '    assert later_sentiment["Signal Comment"] != ""\n'
    '    assert later_sentiment["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert later_sentiment["Signal Validity"] in ("Valid", "Incomplete Score")\n'
    '    assert later_sentiment["Warning Flag"] != ""\n'
    '    assert later_sentiment["Slowdown Risk Flag"] != ""\n'
    '    assert later_sentiment["Extreme Flag"] != ""\n'
    '    assert _sentiment_recovery_score(-1, 1, 1) == 6\n'
    '    assert _sentiment_turn_score(100, 0.15, 1, 1) == 5\n'
    '    assert _sentiment_warning_raw(-1, -1, -1, -7, -2) == -8\n'
    '    assert _sentiment_regime(3) == "Strong sentiment"\n'
    '    assert _sentiment_momentum_label(-3, -2) == "Weak sentiment, deteriorating"\n'
    '    assert (\n'
    '        _sentiment_signal_interpretation(6, 0)\n'
    '        == "Strong positive consumer sentiment leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _sentiment_warning_label(-6) == "High consumer sentiment warning risk"\n'
    '    assert _sentiment_signal_label(-7, -2) == "Severe consumer confidence stress"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped sentiment sheet", CONTROL_PANEL_EIGHTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_consumer_sentiment_outputs\n'
    '    )\n'
    '    all_specs_with_a10 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["Consumer Sentiment"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Consumer Sentiment", "A10") in all_specs_with_a10\n'
    '    expectations_out = calculate_consumer_expectations_outputs(sentiment_raw)\n'
    '    expectations_rows = df_records(expectations_out)\n'
    '    assert expectations_out.shape[1] == 29\n'
    '    assert output_end_col_for_columns(CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS) == "AE"\n'
    '    assert (\n'
    '        indicator_output_range("Consumer Expectations", 678, "AE")\n'
    '        == "\'Consumer Expectations\'!C2:AE678"\n'
    '    )\n'
    '    assert list(expectations_out.columns) == CONSUMER_EXPECTATIONS_OUTPUT_COLUMNS\n'
    '    a11_spec = select_indicator_engine_spec(\n'
    '        "Unmapped expectations sheet", CONTROL_PANEL_NINTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert a11_spec.name == "CONSUMER_EXPECTATIONS_ENGINE"\n'
    '    assert (\n'
    '        a11_spec.input_start_col == "A"\n'
    '        and a11_spec.input_end_col == "B"\n'
    '        and a11_spec.output_start_col == "C"\n'
    '        and a11_spec.output_end_col == "AE"\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[0]["Sample Count"] == 1\n'
    '        and expectations_rows[59]["Sample Count"] == 60\n'
    '    )\n'
    '    assert expectations_rows[58]["Distance from historical median"] == ""\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[3]["1M Change"], sentiment_values[3] - sentiment_values[2]\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[6]["3M Change"], sentiment_values[6] - sentiment_values[3]\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[9]["6M Change"], sentiment_values[9] - sentiment_values[3]\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[58]["Level Percentile"] == ""\n'
    '        and expectations_rows[59]["Level Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[61]["3M Momentum Percentile"] == ""\n'
    '        and expectations_rows[62]["3M Momentum Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[64]["6M Momentum Percentile"] == ""\n'
    '        and expectations_rows[65]["6M Momentum Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[59]["Coincident Score"] != ""\n'
    '        and expectations_rows[59]["Leading Score"] == ""\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[58]["Level Z-Score"] == ""\n'
    '        and expectations_rows[59]["Level Z-Score"] != ""\n'
    '    )\n'
    '    assert _sentiment_sheets_round(0.8689, 3) == 0.869\n'
    '    assert _sentiment_percent_display(0.869) == "86.90%"\n'
    '    assert _sentiment_sheets_round(0.9322, 3) == 0.932\n'
    '    assert _sentiment_percent_display(0.932) == "93.20%"\n'
    '    assert _sentiment_score_percent_rank(0.985) == 9.7\n'
    '    assert _sentiment_score_percent_rank(0.15) == -7.0\n'
    '    assert _sentiment_score_percent_rank(0.4) == -2.0\n'
    '    assert _sentiment_score_percent_rank(0.058) == -8.84\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 3.22 + 0.05 * 5.85), 2) == 2.71\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 7.38 + 0.05 * 9.10), 2) == 5.99\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * 9.70 + 0.05 * 27.00), 2) == 8.63\n'
    '    assert _sentiment_sheets_round(clamp(0.75 * -7.00 + 0.05 * -17.20), 2) == -6.11\n'
    '    assert (\n'
    '        _sentiment_sheets_round(\n'
    '            clamp(0.20 * -7.0 + 0.40 * -2.0 + 0.25 * -8.84 + 0.05 * -6 + 0.10 * 0), 2\n'
    '        )\n'
    '        == -4.71\n'
    '    )\n'
    '    assert _sentiment_warning_raw(-17.2, -1, -1, -7.0, -2.0) == -8\n'
    '    assert (\n'
    '        _sentiment_sheets_round(clamp(0.25 * -6 + 0.25 * 0 + 0.35 * -8 + 0.15 * 0), 2)\n'
    '        == -4.3\n'
    '    )\n'
    '    assert _expectations_regime(3) == "Strong expectations"\n'
    '    assert _expectations_momentum_label(-3, -2) == "Weak expectations, deteriorating"\n'
    '    assert (\n'
    '        _expectations_momentum_label(-7.0, -2.0) == "Weak expectations, deteriorating"\n'
    '    )\n'
    '    assert (\n'
    '        _expectations_signal_interpretation(6, 0)\n'
    '        == "Strong positive consumer expectations leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _expectations_warning_label(-4.3)\n'
    '        == "Moderate consumer expectations warning risk"\n'
    '    )\n'
    '    assert _expectations_signal_label(-7, -2) == "Severe consumer expectations stress"\n'
    '    later_expectations = expectations_rows[80]\n'
    '    assert later_expectations["Coincident Score"] != ""\n'
    '    assert later_expectations["Leading Score"] != ""\n'
    '    assert later_expectations["Warning Score"] != ""\n'
    '    assert later_expectations["State"] in (\n'
    '        "Strong expectations",\n'
    '        "Weak expectations",\n'
    '        "Neutral expectations",\n'
    '    )\n'
    '    assert later_expectations["Regime"] != ""\n'
    '    assert later_expectations["Signal Comment"] != ""\n'
    '    assert later_expectations["Reliability"] in (\n'
    '        "Medium reliability",\n'
    '        "High reliability",\n'
    '    )\n'
    '    assert later_expectations["Signal Validity"] in ("Valid", "Incomplete Score")\n'
    '    assert later_expectations["Warning Flag"] != ""\n'
    '    assert later_expectations["Slowdown Risk Flag"] != ""\n'
    '    assert later_expectations["Extreme Flag"] != ""\n'
    '    assert _sentiment_outlier_score([-3.1, 3.5, 0]) == -6\n'
    '    assert _sentiment_outlier_label([3.1, -3.5, 0]) == "Extreme positive outlier"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped expectations sheet", CONTROL_PANEL_NINTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_consumer_expectations_outputs\n'
    '    )\n'
    '    all_specs_with_a11 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["Consumer Expectations"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Consumer Expectations", "A11") in all_specs_with_a11\n'
    '    claims_raw = make_dataframe(\n'
    '        [{"Date": f"c{i}", "Raw Value": 100 + i} for i in range(140)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    claims_out = calculate_initial_claims_outputs(claims_raw)\n'
    '    claims_rows = df_records(claims_out)\n'
    '    assert claims_out.shape[1] == 34\n'
    '    assert output_end_col_for_columns(INITIAL_CLAIMS_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert (\n'
    '        indicator_output_range("Initial Claims", 678, "AJ")\n'
    '        == "\'Initial Claims\'!C2:AJ678"\n'
    '    )\n'
    '    assert (\n'
    '        claims_rows[0]["3M Moving Average"] == ""\n'
    '        and claims_rows[1]["3M Moving Average"] == ""\n'
    '    )\n'
    '    assert claims_rows[2]["3M Moving Average"] == 101\n'
    '    assert (\n'
    '        claims_rows[2]["Smoothed Observation Count"] == 1\n'
    '        and claims_rows[3]["Smoothed Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(claims_rows[3]["1M Smoothed Growth"], 102 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        claims_rows[5]["3M Annualised Smoothed Growth"], (104 / 101) ** 4 - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        claims_rows[8]["6M Annualised Smoothed Growth"], (107 / 101) ** 2 - 1\n'
    '    )\n'
    '    assert values_equivalent(claims_rows[14]["12M Smoothed Growth"], 113 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        claims_rows[17]["3M Change in 12M Growth"],\n'
    '        claims_rows[17]["12M Smoothed Growth"] - claims_rows[14]["12M Smoothed Growth"],\n'
    '    )\n'
    '    assert (\n'
    '        claims_rows[72]["12M Claims Growth PercentRank"] == ""\n'
    '        and claims_rows[73]["12M Claims Growth PercentRank"] != ""\n'
    '    )\n'
    '    assert claims_rows[73]["12M Claims Growth Score"] == round(\n'
    '        10 - claims_rows[73]["12M Claims Growth PercentRank"] * 20, 2\n'
    '    )\n'
    '    assert (\n'
    '        claims_rows[72]["12M Claims Growth Inverted Z-Score"] == ""\n'
    '        and claims_rows[73]["12M Claims Growth Inverted Z-Score"] != ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expanding_inverted_zscore_sample([1, 2, 3], [60, 60, 60], 60)[2], -1.0\n'
    '    )\n'
    '    assert _industrial_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert (\n'
    '        _claims_outlier_label([-2.6, 3.0, 0, 0])\n'
    '        == "Extreme labour deterioration outlier"\n'
    '    )\n'
    '    assert claims_rows[80]["Initial Claims Leading Score"] != ""\n'
    '    assert claims_rows[80]["Initial Claims Warning Composite Score"] != ""\n'
    '    assert claims_rows[80]["Initial Claims Final Warning Score"] != ""\n'
    '    assert claims_rows[80]["Initial Claims Labour Market Regime"] in (\n'
    '        "Labour market improving",\n'
    '        "Labour market weakening",\n'
    '        "Neutral labour market",\n'
    '    )\n'
    '    assert claims_rows[80]["Initial Claims Momentum Label"] != ""\n'
    '    assert claims_rows[80]["Initial Claims Signal Interpretation"] != ""\n'
    '    assert claims_rows[80]["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert claims_rows[80]["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert claims_rows[80]["Initial Claims Warning Risk Label"] != ""\n'
    '    assert claims_rows[80]["Initial Claims Signal Label"] != ""\n'
    '    assert claims_rows[80]["Initial Claims Outlier Label"] != ""\n'
    '    assert _claims_recovery_score(1, -1, -1) == 6\n'
    '    assert _claims_surge_warning(0.06, 0.21, 0) == -6\n'
    '    assert _claims_regime(2) == "Labour market improving"\n'
    '    assert (\n'
    '        _claims_momentum_label(2, 2) == "Claims falling / labour market strengthening"\n'
    '    )\n'
    '    assert (\n'
    '        _claims_signal_interpretation(6, 0)\n'
    '        == "Strong positive initial claims leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _claims_warning_label(-6) == "High initial claims warning risk"\n'
    '    assert _claims_signal_label(2, -2, -6) == "Severe labour market deterioration risk"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped claims sheet", CONTROL_PANEL_TENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_initial_claims_outputs\n'
    '    )\n'
    '    employment_raw = make_dataframe(\n'
    '        [{"Date": f"e{i}", "Raw Value": 100 + i} for i in range(140)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    employment_out = calculate_employment_growth_outputs(employment_raw)\n'
    '    employment_rows = df_records(employment_out)\n'
    '    assert employment_out.shape[1] == 34\n'
    '    assert output_end_col_for_columns(EMPLOYMENT_GROWTH_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert (\n'
    '        indicator_output_range("Employment Growth", 678, "AJ")\n'
    '        == "\'Employment Growth\'!C2:AJ678"\n'
    '    )\n'
    '    assert (\n'
    '        employment_rows[0]["3M Moving Average"] == ""\n'
    '        and employment_rows[1]["3M Moving Average"] == ""\n'
    '    )\n'
    '    assert employment_rows[2]["3M Moving Average"] == 101\n'
    '    assert (\n'
    '        employment_rows[2]["Smoothed Observation Count"] == 1\n'
    '        and employment_rows[3]["Smoothed Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(employment_rows[3]["1M Smoothed Growth"], 102 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        employment_rows[5]["3M Annualised Smoothed Growth"], (104 / 101) ** 4 - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        employment_rows[8]["6M Annualised Smoothed Growth"], (107 / 101) ** 2 - 1\n'
    '    )\n'
    '    assert values_equivalent(employment_rows[14]["12M Smoothed Growth"], 113 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        employment_rows[17]["3M Change in 12M Growth"],\n'
    '        employment_rows[17]["12M Smoothed Growth"]\n'
    '        - employment_rows[14]["12M Smoothed Growth"],\n'
    '    )\n'
    '    assert (\n'
    '        employment_rows[72]["12M Employment Growth PercentRank"] == ""\n'
    '        and employment_rows[73]["12M Employment Growth PercentRank"] != ""\n'
    '    )\n'
    '    assert employment_rows[73]["12M Employment Growth Score"] == round(\n'
    '        employment_rows[73]["12M Employment Growth PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert (\n'
    '        employment_rows[72]["12M Employment Growth Z-Score"] == ""\n'
    '        and employment_rows[73]["12M Employment Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expanding_zscore_sample([1, 2, 3], [60, 60, 60], 60)[2], 1.0\n'
    '    )\n'
    '    assert _industrial_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert (\n'
    '        _employment_outlier_label([-3.1, 3.5, 0, 0])\n'
    '        == "Extreme employment deterioration signal"\n'
    '    )\n'
    '    assert employment_rows[80]["Employment Leading Score"] != ""\n'
    '    assert employment_rows[80]["Employment Warning Composite Score"] != ""\n'
    '    assert employment_rows[80]["Employment Final Warning Score"] != ""\n'
    '    assert employment_rows[80]["Employment Labour Market Regime"] in (\n'
    '        "Labour market expanding",\n'
    '        "Labour market contracting",\n'
    '        "Neutral labour market",\n'
    '    )\n'
    '    assert employment_rows[80]["Employment Momentum Label"] != ""\n'
    '    assert employment_rows[80]["Employment Signal Interpretation"] != ""\n'
    '    assert employment_rows[80]["Reliability"] in (\n'
    '        "Medium reliability",\n'
    '        "High reliability",\n'
    '    )\n'
    '    assert employment_rows[80]["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert employment_rows[80]["Employment Warning Risk Label"] != ""\n'
    '    assert employment_rows[80]["Employment Signal Label"] != ""\n'
    '    assert employment_rows[80]["Employment Outlier Label"] != ""\n'
    '    assert _industrial_recovery_score(-1, 1, 1) == 6\n'
    '    assert _employment_short_term_warning(-0.006, -0.031, 0) == -6\n'
    '    assert _employment_short_term_warning(0.006, 0.031, 0.021) == 6\n'
    '    assert _employment_regime(2) == "Labour market expanding"\n'
    '    assert _employment_momentum_label(2, 2) == "Employment growth strengthening"\n'
    '    assert (\n'
    '        _employment_signal_interpretation(6, 0)\n'
    '        == "Strong positive employment leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _employment_warning_label(-6) == "High employment warning risk"\n'
    '    assert _employment_signal_label(2, -2, -6) == "Severe employment deterioration risk"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped employment sheet", CONTROL_PANEL_ELEVENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_employment_growth_outputs\n'
    '    )\n'
    '    all_specs_with_a13 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["Employment Growth"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Employment Growth", "A13") in all_specs_with_a13\n'
    '    avg_hours_raw = make_dataframe(\n'
    '        [{"Date": f"h{i}", "Raw Value": 100 + i} for i in range(140)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    avg_hours_out = calculate_average_hours_outputs(avg_hours_raw)\n'
    '    avg_hours_rows = df_records(avg_hours_out)\n'
    '    assert list(getattr(avg_hours_out, "columns", [])) == AVERAGE_HOURS_OUTPUT_COLUMNS\n'
    '    assert avg_hours_out.shape[1] == 34\n'
    '    assert output_end_col_for_columns(AVERAGE_HOURS_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert indicator_input_range("Average Hours", "B", "A") == "\'Average Hours\'!A2:B"\n'
    '    assert (\n'
    '        indicator_output_range("Average Hours", 678, "AJ", "C")\n'
    '        == "\'Average Hours\'!C2:AJ678"\n'
    '    )\n'
    '    assert avg_hours_rows[0]["3M MA"] == "" and avg_hours_rows[1]["3M MA"] == ""\n'
    '    assert avg_hours_rows[2]["3M MA"] == 101\n'
    '    assert (\n'
    '        avg_hours_rows[2]["Sample Count"] == 1\n'
    '        and avg_hours_rows[3]["Sample Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(avg_hours_rows[3]["1M % Change"], 102 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        avg_hours_rows[5]["3M Annualised Growth"], (104 / 101) ** 4 - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        avg_hours_rows[8]["6M Annualised Growth"], (107 / 101) ** 2 - 1\n'
    '    )\n'
    '    assert values_equivalent(avg_hours_rows[14]["YoY Growth"], 113 / 101 - 1)\n'
    '    avg_hours_i_internal = (\n'
    '        avg_hours_rows[17]["YoY Growth"] - avg_hours_rows[14]["YoY Growth"]\n'
    '    )\n'
    '    avg_hours_j_internal = (\n'
    '        avg_hours_rows[20]["YoY Growth"] - avg_hours_rows[14]["YoY Growth"]\n'
    '    )\n'
    '    assert isinstance(avg_hours_i_internal, float) and isinstance(\n'
    '        avg_hours_j_internal, float\n'
    '    )\n'
    '    assert avg_hours_rows[17][\n'
    '        "3M Change in YoY Growth"\n'
    '    ] == _average_hours_percent_display(avg_hours_i_internal)\n'
    '    assert avg_hours_rows[20][\n'
    '        "6M Change in YoY Growth"\n'
    '    ] == _average_hours_percent_display(avg_hours_j_internal)\n'
    '    assert (\n'
    '        avg_hours_rows[72]["YoY Growth Percentile"] == ""\n'
    '        and avg_hours_rows[73]["YoY Growth Percentile"] != ""\n'
    '    )\n'
    '    assert avg_hours_rows[73]["YoY Growth Score"] == _average_hours_score_percent_rank(\n'
    '        avg_hours_rows[73]["YoY Growth Percentile"]\n'
    '    )\n'
    '    assert (\n'
    '        avg_hours_rows[72]["YoY Growth Z-Score"] == ""\n'
    '        and avg_hours_rows[73]["YoY Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expanding_zscore_sample([1, 2, 3], [60, 60, 60], 60)[2], 1.0\n'
    '    )\n'
    '    assert _industrial_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert (\n'
    '        _average_hours_outlier_label([-3.1, 3.5, 0, 0])\n'
    '        == "Extreme hours-worked deterioration signal"\n'
    '    )\n'
    '    assert avg_hours_rows[80]["Coincident Score"] != ""\n'
    '    assert avg_hours_rows[80]["Leading Score"] != ""\n'
    '    assert avg_hours_rows[80]["Warning Score"] != ""\n'
    '    assert avg_hours_rows[80]["State"] in (\n'
    '        "Labour demand strengthening",\n'
    '        "Labour demand weakening",\n'
    '        "Neutral labour demand",\n'
    '    )\n'
    '    assert avg_hours_rows[80]["Regime"] != ""\n'
    '    assert avg_hours_rows[80]["Signal Comment"] != ""\n'
    '    assert avg_hours_rows[80]["Reliability"] in (\n'
    '        "Medium reliability",\n'
    '        "High reliability",\n'
    '    )\n'
    '    assert avg_hours_rows[80]["Signal Validity"] in ("Valid", "Incomplete Score")\n'
    '    assert avg_hours_rows[80]["Warning Flag"] != ""\n'
    '    assert avg_hours_rows[80]["Slowdown Risk Flag"] != ""\n'
    '    assert avg_hours_rows[80]["Extreme Flag"] != ""\n'
    '    assert _average_hours_sheets_round(0.0875, 3) == 0.088\n'
    '    assert _average_hours_sheets_round(-8.245, 2) == -8.25\n'
    '    assert _average_hours_percent_display(-0.0009784292244) == "-0.10%"\n'
    '    assert _average_hours_percent_display(-0.002920176797) == "-0.29%"\n'
    '    assert _average_hours_percent_display(0.01068513008) == "1.07%"\n'
    '    assert _average_hours_percent_display(0) == "0.00%"\n'
    '    assert _average_hours_percentrank_inc_sheets([1, 1, 2], 1) == 0\n'
    '    assert _average_hours_percentrank_inc_sheets([1, 3], 2) == 0.5\n'
    '    assert (\n'
    '        _average_hours_percent_rank_rounded_3([float(x) for x in range(59)], 35.0, 59)\n'
    '        == ""\n'
    '    )\n'
    '    assert (\n'
    '        _average_hours_percent_rank_rounded_3([float(x) for x in range(60)], 35.0, 60)\n'
    '        == 0.593\n'
    '    )\n'
    '    assert _average_hours_score_percent_rank(0.593) == 1.86\n'
    '    assert _average_hours_score_percent_rank(0.153) == -6.94\n'
    '\n'
    '    # Fixed local A14 fixture copied from the source series. It deliberately\n'
    '    # contains the repeated 0.00%-growth patterns that exercise Sheets tie\n'
    '    # ordering, and never reads Google Sheets during --self-test.\n'
    '    avg_hours_regression_values = [\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.5,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.1,\n'
    '        33.9,\n'
    '        34.0,\n'
    '        34.0,\n'
    '        33.8,\n'
    '        33.8,\n'
    '        33.8,\n'
    '        33.7,\n'
    '        33.8,\n'
    '        33.8,\n'
    '        33.9,\n'
    '        33.8,\n'
    '        33.9,\n'
    '        33.9,\n'
    '        34.0,\n'
    '        33.8,\n'
    '        34.0,\n'
    '        34.1,\n'
    '        34.2,\n'
    '        34.1,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.6,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.1,\n'
    '        34.2,\n'
    '        34.7,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.7,\n'
    '        34.8,\n'
    '        34.8,\n'
    '        34.8,\n'
    '        34.8,\n'
    '        35.0,\n'
    '        34.6,\n'
    '        34.9,\n'
    '        34.9,\n'
    '        34.9,\n'
    '        34.8,\n'
    '        34.8,\n'
    '        34.8,\n'
    '        34.8,\n'
    '        34.7,\n'
    '        34.8,\n'
    '        34.9,\n'
    '        34.5,\n'
    '        34.7,\n'
    '        34.7,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.6,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.6,\n'
    '        34.5,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.3,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.2,\n'
    '        34.4,\n'
    '        34.4,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.1,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '        34.3,\n'
    '        34.2,\n'
    '        34.3,\n'
    '    ]\n'
    '    avg_hours_regression_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"fixture-{idx}", "Raw Value": value}\n'
    '            for idx, value in enumerate(avg_hours_regression_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    avg_hours_regression_rows = df_records(\n'
    '        calculate_average_hours_outputs(avg_hours_regression_raw)\n'
    '    )\n'
    '    apr_2012 = avg_hours_regression_rows[73]\n'
    '    jul_2012 = avg_hours_regression_rows[76]\n'
    '    dec_2012 = avg_hours_regression_rows[81]\n'
    '    jan_2013 = avg_hours_regression_rows[82]\n'
    '    sep_2013 = avg_hours_regression_rows[90]\n'
    '    jan_2015 = avg_hours_regression_rows[106]\n'
    '    jan_2016 = avg_hours_regression_rows[118]\n'
    '    nov_2018 = avg_hours_regression_rows[152]\n'
    '    nov_2021 = avg_hours_regression_rows[188]\n'
    '    oct_2024 = avg_hours_regression_rows[223]\n'
    '    aug_2025 = avg_hours_regression_rows[233]\n'
    '    assert [\n'
    '        apr_2012[col]\n'
    '        for col in (\n'
    '            "YoY Growth Percentile",\n'
    '            "3M Growth Percentile",\n'
    '            "6M Growth Percentile",\n'
    '        )\n'
    '    ] == [0.593, 0.471, 0.477]\n'
    '    assert [\n'
    '        apr_2012[col]\n'
    '        for col in ("YoY Growth Score", "3M Growth Score", "6M Growth Score")\n'
    '    ] == [1.86, -0.58, -0.46]\n'
    '    assert [\n'
    '        jul_2012[col]\n'
    '        for col in (\n'
    '            "YoY Growth Percentile",\n'
    '            "3M Growth Percentile",\n'
    '            "6M Growth Percentile",\n'
    '            "Momentum Percentile",\n'
    '        )\n'
    '    ] == [0.339, 0.169, 0.235, 0.153]\n'
    '    assert [\n'
    '        jul_2012[col]\n'
    '        for col in (\n'
    '            "YoY Growth Score",\n'
    '            "3M Growth Score",\n'
    '            "6M Growth Score",\n'
    '            "Momentum Score",\n'
    '        )\n'
    '    ] == [-3.22, -6.62, -5.30, -6.94]\n'
    '    assert [\n'
    '        jul_2012[col] for col in ("Coincident Score", "Leading Score", "Warning Score")\n'
    '    ] == [-5.04, -5.78, 0.0]\n'
    '    assert [\n'
    '        dec_2012[col]\n'
    '        for col in (\n'
    '            "Hours Cycle Turning Point Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '        )\n'
    '    ] == [-7, -1.24, -0.56, -3.15]\n'
    '    assert [\n'
    '        jan_2013[col]\n'
    '        for col in (\n'
    '            "Hours Cycle Turning Point Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '        )\n'
    '    ] == [5, 0.21, 1.38, 2.25]\n'
    '    assert [\n'
    '        sep_2013[col]\n'
    '        for col in ("State", "Regime", "Signal Comment", "Slowdown Risk Flag")\n'
    '    ] == [\n'
    '        "Neutral labour demand",\n'
    '        "Average hours broadly neutral",\n'
    '        "Neutral/mixed average-hours signal",\n'
    '        "Neutral average-hours signal",\n'
    '    ]\n'
    '    assert [\n'
    '        jan_2015[col]\n'
    '        for col in (\n'
    '            "YoY Growth Percentile",\n'
    '            "3M Growth Percentile",\n'
    '            "6M Growth Percentile",\n'
    '            "Momentum Percentile",\n'
    '        )\n'
    '    ] == [0.761, 0.446, 0.520, 0.708]\n'
    '    assert [\n'
    '        jan_2015[col]\n'
    '        for col in (\n'
    '            "YoY Growth Score",\n'
    '            "3M Growth Score",\n'
    '            "6M Growth Score",\n'
    '            "Momentum Score",\n'
    '        )\n'
    '    ] == [5.22, -1.08, 0.40, 4.16]\n'
    '    assert (\n'
    '        jan_2015["State"] == "Labour demand strengthening"\n'
    '        and jan_2015["Regime"] == "Average hours positive / stable"\n'
    '    )\n'
    '    assert jan_2016["Hours Cycle Turning Point Score"] == -7\n'
    '    assert [\n'
    '        jan_2016[col]\n'
    '        for col in ("Regime", "Signal Comment", "Warning Flag", "Slowdown Risk Flag")\n'
    '    ] == [\n'
    '        "Average hours deteriorating",\n'
    '        "Moderately negative average-hours leading signal",\n'
    '        "Moderate average-hours warning risk",\n'
    '        "Average hours deterioration warning",\n'
    '    ]\n'
    '    assert nov_2018["Hours Cycle Turning Point Score"] == -6\n'
    '    assert [\n'
    '        nov_2021[col]\n'
    '        for col in (\n'
    '            "3M Growth Percentile",\n'
    '            "6M Growth Percentile",\n'
    '            "Momentum Percentile",\n'
    '        )\n'
    '    ] == [0.350, 0.083, 0.058]\n'
    '    assert [\n'
    '        nov_2021[col]\n'
    '        for col in (\n'
    '            "3M Growth Score",\n'
    '            "6M Growth Score",\n'
    '            "Momentum Score",\n'
    '            "Leading Score",\n'
    '        )\n'
    '    ] == [-3.00, -8.34, -8.84, -6.00]\n'
    '    assert (\n'
    '        nov_2021["Signal Comment"]\n'
    '        == "Strong negative average-hours leading signal with downside warning confirmation"\n'
    '    )\n'
    '    assert [\n'
    '        oct_2024[col] for col in ("Regime", "Signal Comment", "Slowdown Risk Flag")\n'
    '    ] == [\n'
    '        "Average hours weak / stable",\n'
    '        "Neutral/mixed average-hours signal",\n'
    '        "Neutral average-hours signal",\n'
    '    ]\n'
    '    assert (\n'
    '        aug_2025["State"] == "Labour demand weakening"\n'
    '        and aug_2025["Regime"] == "Average hours weak / stable"\n'
    '    )\n'
    '    expected_nov_2021_zscores = [\n'
    '        -0.30253350521640326,\n'
    '        -0.4749463446028696,\n'
    '        -1.0659567416630773,\n'
    '        -1.4862208830414478,\n'
    '    ]\n'
    '    for column, expected_zscore in zip(\n'
    '        (\n'
    '            "YoY Growth Z-Score",\n'
    '            "3M Growth Z-Score",\n'
    '            "6M Growth Z-Score",\n'
    '            "Momentum Z-Score",\n'
    '        ),\n'
    '        expected_nov_2021_zscores,\n'
    '    ):\n'
    '        assert values_equivalent(nov_2021[column], expected_zscore)\n'
    '    assert nov_2021["Extreme Flag"] == "Normal range"\n'
    '    assert _average_hours_recovery_score(-1, 1, 1) == 7\n'
    '    assert _average_hours_short_term_warning(-0.003, -0.021, 0) == -6\n'
    '    assert _average_hours_short_term_warning(0.003, 0.021, 0.011) == 6\n'
    '    assert _average_hours_regime(2) == "Labour demand strengthening"\n'
    '    assert _average_hours_momentum_label(2, 2) == "Average hours strengthening"\n'
    '    assert (\n'
    '        _average_hours_signal_interpretation(6, 0)\n'
    '        == "Strong positive average-hours leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _average_hours_warning_label(-6) == "High average-hours warning risk"\n'
    '    assert (\n'
    '        _average_hours_signal_label(2, -2, -6)\n'
    '        == "Severe average-hours deterioration risk"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped average hours sheet", CONTROL_PANEL_TWELFTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_average_hours_outputs\n'
    '    )\n'
    '    avg_hours_engine_spec = select_indicator_engine_spec(\n'
    '        "Unmapped average hours sheet", CONTROL_PANEL_TWELFTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        avg_hours_engine_spec.name,\n'
    '        avg_hours_engine_spec.input_start_col,\n'
    '        avg_hours_engine_spec.input_end_col,\n'
    '    ) == ("AVERAGE_HOURS_ENGINE", "A", "B")\n'
    '    assert (\n'
    '        avg_hours_engine_spec.output_start_col,\n'
    '        avg_hours_engine_spec.output_end_col,\n'
    '        len(avg_hours_engine_spec.output_columns),\n'
    '    ) == ("C", "AJ", 34)\n'
    '    assert all(\n'
    '        select_indicator_engine("__unmapped__", f"A{row}")\n'
    '        is not calculate_average_hours_outputs\n'
    '        for row in range(3, 40)\n'
    '        if row != 14\n'
    '    )\n'
    '    all_specs_with_a14 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["Average Hours"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Average Hours", "A14") in all_specs_with_a14\n'
    '    building_raw = make_dataframe(\n'
    '        [{"Date": f"bp{i}", "Raw Value": 100 + i} for i in range(140)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    building_out = calculate_building_permits_outputs(building_raw)\n'
    '    building_rows = df_records(building_out)\n'
    '    assert building_out.shape[1] == 36\n'
    '    assert output_end_col_for_columns(BUILDING_PERMITS_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        indicator_output_range("Building Permits", 678, "AL")\n'
    '        == "\'Building Permits\'!C2:AL678"\n'
    '    )\n'
    '    assert (\n'
    '        building_rows[0]["3M Moving Average"] == ""\n'
    '        and building_rows[1]["3M Moving Average"] == ""\n'
    '    )\n'
    '    assert building_rows[2]["3M Moving Average"] == 101\n'
    '    assert (\n'
    '        building_rows[2]["Smoothed Observation Count"] == 1\n'
    '        and building_rows[3]["Smoothed Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(building_rows[3]["1M Smoothed Growth"], 102 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        building_rows[5]["3M Annualised Smoothed Growth"], (104 / 101) ** 4 - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        building_rows[8]["6M Annualised Smoothed Growth"], (107 / 101) ** 2 - 1\n'
    '    )\n'
    '    assert values_equivalent(building_rows[14]["12M Smoothed Growth"], 113 / 101 - 1)\n'
    '    assert values_equivalent(\n'
    '        building_rows[17]["3M Change in 12M Growth"],\n'
    '        building_rows[17]["12M Smoothed Growth"]\n'
    '        - building_rows[14]["12M Smoothed Growth"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        building_rows[20]["6M Change in 12M Growth"],\n'
    '        building_rows[20]["12M Smoothed Growth"]\n'
    '        - building_rows[14]["12M Smoothed Growth"],\n'
    '    )\n'
    '    assert (\n'
    '        building_rows[72]["12M Building Permits Growth PercentRank"] == ""\n'
    '        and building_rows[73]["12M Building Permits Growth PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        building_rows[60]["Smoothed Level PercentRank"] == ""\n'
    '        and building_rows[61]["Smoothed Level PercentRank"] != ""\n'
    '    )\n'
    '    assert building_rows[73]["12M Building Permits Growth Score"] == round(\n'
    '        building_rows[73]["12M Building Permits Growth PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert building_rows[61]["Smoothed Level Score"] == round(\n'
    '        building_rows[61]["Smoothed Level PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert (\n'
    '        building_rows[72]["12M Building Permits Growth Z-Score"] == ""\n'
    '        and building_rows[73]["12M Building Permits Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert _industrial_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert (\n'
    '        _building_permits_outlier_label([-3.1, 3.5, 0, 0])\n'
    '        == "Extreme building-permits deterioration signal"\n'
    '    )\n'
    '    assert building_rows[80]["Building Permits Recovery / Momentum Score"] != ""\n'
    '    assert building_rows[80]["Building Permits Short-Term Warning Score Raw"] != ""\n'
    '    assert building_rows[80]["Building Permits Outlier Score"] != ""\n'
    '    assert building_rows[80]["Building Permits Leading Score"] != ""\n'
    '    assert building_rows[80]["Building Permits Warning Composite Score"] != ""\n'
    '    assert building_rows[80]["Building Permits Final Warning Score"] != ""\n'
    '    assert building_rows[80]["Building Permits Housing Activity Regime"] in (\n'
    '        "Housing activity strengthening",\n'
    '        "Housing activity weakening",\n'
    '        "Neutral housing activity",\n'
    '    )\n'
    '    assert building_rows[80]["Building Permits Momentum Label"] != ""\n'
    '    assert building_rows[80]["Building Permits Signal Interpretation"] != ""\n'
    '    assert building_rows[80]["Reliability"] in (\n'
    '        "Medium reliability",\n'
    '        "High reliability",\n'
    '    )\n'
    '    assert building_rows[80]["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert building_rows[80]["Building Permits Warning Risk Label"] != ""\n'
    '    assert building_rows[80]["Building Permits Signal Label"] != ""\n'
    '    assert building_rows[80]["Building Permits Outlier Label"] != ""\n'
    '    assert _building_permits_recovery_score(-1, 1, 1) == 7\n'
    '    assert _building_permits_short_term_warning(-0.06, -0.26, 0) == -6\n'
    '    assert _building_permits_short_term_warning(0.06, 0.26, 0.11) == 6\n'
    '    assert _building_permits_regime(2) == "Housing activity strengthening"\n'
    '    assert _building_permits_momentum_label(2, 2) == "Building permits strengthening"\n'
    '    assert (\n'
    '        _building_permits_signal_interpretation(6, 0)\n'
    '        == "Strong positive building-permits leading signal with supportive warning profile"\n'
    '    )\n'
    '    assert _building_permits_warning_label(-6) == "High building-permits warning risk"\n'
    '    assert (\n'
    '        _building_permits_signal_label(2, -2, -6)\n'
    '        == "Severe building-permits deterioration risk"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped building permits sheet", CONTROL_PANEL_THIRTEENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_building_permits_outputs\n'
    '    )\n'
    '    all_specs_with_a15 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["Building Permits"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Building Permits", "A15") in all_specs_with_a15\n'
    '    core_raw = make_dataframe(\n'
    '        [{"Date": f"ci{i}", "Raw Value": 100 * (1.002**i)} for i in range(140)],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    core_out = calculate_core_inflation_outputs(core_raw)\n'
    '    core_rows = df_records(core_out)\n'
    '    assert core_out.shape[1] == 36\n'
    '    assert output_end_col_for_columns(CORE_INFLATION_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        indicator_output_range("Core Inflation", 678, "AL")\n'
    '        == "\'Core Inflation\'!C2:AL678"\n'
    '    )\n'
    '    assert (\n'
    '        core_rows[0]["Observation Count"] == 1\n'
    '        and core_rows[1]["Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[1]["1M Annualised Inflation"],\n'
    '        ((100 * (1.002**1)) / (100 * (1.002**0))) ** 12 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[3]["3M Annualised Inflation"],\n'
    '        ((100 * (1.002**3)) / (100 * (1.002**0))) ** 4 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[6]["6M Annualised Inflation"],\n'
    '        ((100 * (1.002**6)) / (100 * (1.002**0))) ** 2 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[12]["12M Inflation"], (100 * (1.002**12)) / (100 * (1.002**0)) - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[15]["3M Change in 12M Inflation"],\n'
    '        core_rows[15]["12M Inflation"] - core_rows[12]["12M Inflation"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[18]["6M Change in 12M Inflation"],\n'
    '        core_rows[18]["12M Inflation"] - core_rows[12]["12M Inflation"],\n'
    '    )\n'
    '    assert (\n'
    '        core_rows[70]["12M Core Inflation PercentRank"] == ""\n'
    '        and core_rows[71]["12M Core Inflation PercentRank"] != ""\n'
    '    )\n'
    '    assert core_rows[71]["12M Core Inflation Score"] == round(\n'
    '        core_rows[71]["12M Core Inflation PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert core_rows[71]["3M Annualised Core Inflation Score"] == round(\n'
    '        core_rows[71]["3M Annualised Core Inflation PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_rows[12]["Distance From 2 Percent Target"],\n'
    '        core_rows[12]["12M Inflation"] - 0.02,\n'
    '    )\n'
    '    assert (\n'
    '        _core_inflation_target_status(0.031, 0.031)\n'
    '        == "Clearly above target and persistent"\n'
    '    )\n'
    '    assert _core_inflation_target_status(0.031, 0.024) == "Above target but cooling"\n'
    '    assert (\n'
    '        _core_inflation_target_status(0.025, 0.031)\n'
    '        == "Near/below target but re-accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        core_rows[58]["12M Core Inflation Z-Score"] == ""\n'
    '        and core_rows[59]["12M Core Inflation Z-Score"] != ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expanding_zscore_sample([1, 2, 3], [60, 60, 60], 60)[2], 1.0\n'
    '    )\n'
    '    assert _core_inflation_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _core_inflation_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside core inflation pressure"\n'
    '    )\n'
    '    core_row = core_rows[80]\n'
    '    assert core_row["Core Inflation Momentum Composite Score"] != ""\n'
    '    assert core_row["Core Inflation Recovery / Cooling Score"] != ""\n'
    '    assert core_row["Core Inflation Target Pressure Score"] != ""\n'
    '    assert core_row["Core Inflation Outlier Score"] != ""\n'
    '    assert core_row["Core Inflation Leading Score"] != ""\n'
    '    assert core_row["Core Inflation Warning Composite Score"] != ""\n'
    '    assert core_row["Core Inflation Final Warning Score"] != ""\n'
    '    assert core_row["Core Inflation Pressure Regime"] in (\n'
    '        "Inflation pressure rising",\n'
    '        "Inflation pressure easing",\n'
    '        "Neutral inflation pressure",\n'
    '    )\n'
    '    assert core_row["Core Inflation Momentum Label"] != ""\n'
    '    assert core_row["Core Inflation Signal Interpretation"] != ""\n'
    '    assert core_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert core_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert core_row["Core Inflation Warning Risk Label"] != ""\n'
    '    assert core_row["Core Inflation Policy Signal Label"] != ""\n'
    '    assert core_row["Core Inflation Outlier Label"] != ""\n'
    '    k = core_row["12M Core Inflation Score"]\n'
    '    m = core_row["3M Annualised Core Inflation Score"]\n'
    '    o = core_row["6M Annualised Core Inflation Score"]\n'
    '    q = core_row["3M Change in 12M Core Inflation Score"]\n'
    '    r = core_row["Distance From 2 Percent Target"]\n'
    '    u = core_row["Core Inflation Recovery / Cooling Score"]\n'
    '    v = core_row["Core Inflation Target Pressure Score"]\n'
    '    w = core_row["Core Inflation Outlier Score"]\n'
    '    assert values_equivalent(\n'
    '        core_row["Core Inflation Momentum Composite Score"],\n'
    '        round(clamp(0.45 * m + 0.35 * o + 0.2 * q), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_row["Core Inflation Leading Score"],\n'
    '        round(\n'
    '            clamp(\n'
    '                0.35 * k + 0.25 * m + 0.2 * o + 0.1 * q + 0.1 * clamp(r * 500, -10, 10)\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_row["Core Inflation Warning Composite Score"],\n'
    '        round(clamp(0.15 * k + 0.35 * m + 0.25 * o + 0.2 * q + 0.05 * u), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_row["Core Inflation Final Warning Score"],\n'
    '        round(clamp(0.35 * u + 0.45 * v + 0.2 * w), 2),\n'
    '    )\n'
    '    assert _core_inflation_recovery_cooling_score(0.01, 1, 1, -0.01) == 6\n'
    '    assert _core_inflation_recovery_cooling_score(0.03, -1, -1, 0.01) == -5\n'
    '    assert _core_inflation_target_pressure_score(0.011, 0.03, 0.031, 0.031) == 6\n'
    '    assert _core_inflation_target_pressure_score(-0.011, 0.0, 0.009, 0.009) == -4\n'
    '    assert _core_inflation_regime(2) == "Inflation pressure rising"\n'
    '    assert _core_inflation_momentum_label(2, 2) == "Core inflation pressure rising"\n'
    '    assert (\n'
    '        _core_inflation_signal_interpretation(6, 0)\n'
    '        == "Strong upside core inflation leading signal with supportive policy-pressure profile"\n'
    '    )\n'
    '    assert _core_inflation_warning_label(6) == "High upside core inflation warning risk"\n'
    '    assert (\n'
    '        _core_inflation_policy_signal_label(6, 0.031, 0.031)\n'
    '        == "Persistent above-target core inflation pressure"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped core inflation sheet", CONTROL_PANEL_FOURTEENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_core_inflation_outputs\n'
    '    )\n'
    '    all_specs_with_a16 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["Core Inflation"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Core Inflation", "A16") in all_specs_with_a16\n'
    '    core_cpi_out = calculate_core_cpi_inflation_outputs(core_raw)\n'
    '    core_cpi_rows = df_records(core_cpi_out)\n'
    '    assert core_cpi_out.shape[1] == 36\n'
    '    assert output_end_col_for_columns(CORE_CPI_INFLATION_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        indicator_output_range("Core CPI Inflation", 678, "AL")\n'
    '        == "\'Core CPI Inflation\'!C2:AL678"\n'
    '    )\n'
    '    assert (\n'
    '        core_cpi_rows[0]["Observation Count"] == 1\n'
    '        and core_cpi_rows[1]["Observation Count"] == 2\n'
    '    )\n'
    '    numeric_pairs = [\n'
    '        ("Observation Count", "Observation Count"),\n'
    '        ("1M Annualised Inflation", "1M Annualised Inflation"),\n'
    '        ("3M Annualised Inflation", "3M Annualised Inflation"),\n'
    '        ("6M Annualised Inflation", "6M Annualised Inflation"),\n'
    '        ("12M Inflation", "12M Inflation"),\n'
    '        ("3M Change in 12M Inflation", "3M Change in 12M Inflation"),\n'
    '        ("6M Change in 12M Inflation", "6M Change in 12M Inflation"),\n'
    '        ("12M Core Inflation PercentRank", "12M Core CPI Inflation PercentRank"),\n'
    '        ("12M Core Inflation Score", "12M Core CPI Inflation Score"),\n'
    '        (\n'
    '            "3M Annualised Core Inflation PercentRank",\n'
    '            "3M Annualised Core CPI Inflation PercentRank",\n'
    '        ),\n'
    '        (\n'
    '            "3M Annualised Core Inflation Score",\n'
    '            "3M Annualised Core CPI Inflation Score",\n'
    '        ),\n'
    '        (\n'
    '            "6M Annualised Core Inflation PercentRank",\n'
    '            "6M Annualised Core CPI Inflation PercentRank",\n'
    '        ),\n'
    '        (\n'
    '            "6M Annualised Core Inflation Score",\n'
    '            "6M Annualised Core CPI Inflation Score",\n'
    '        ),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation PercentRank",\n'
    '            "3M Change in 12M Core CPI Inflation PercentRank",\n'
    '        ),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation Score",\n'
    '            "3M Change in 12M Core CPI Inflation Score",\n'
    '        ),\n'
    '        ("Distance From 2 Percent Target", "Distance From 2 Percent Target"),\n'
    '        (\n'
    '            "Core Inflation Momentum Composite Score",\n'
    '            "Core CPI Inflation Momentum Composite Score",\n'
    '        ),\n'
    '        (\n'
    '            "Core Inflation Recovery / Cooling Score",\n'
    '            "Core CPI Inflation Recovery / Cooling Score",\n'
    '        ),\n'
    '        (\n'
    '            "Core Inflation Target Pressure Score",\n'
    '            "Core CPI Inflation Target Pressure Score",\n'
    '        ),\n'
    '        ("Core Inflation Outlier Score", "Core CPI Inflation Outlier Score"),\n'
    '        ("Core Inflation Leading Score", "Core CPI Inflation Leading Score"),\n'
    '        (\n'
    '            "Core Inflation Warning Composite Score",\n'
    '            "Core CPI Inflation Warning Composite Score",\n'
    '        ),\n'
    '        (\n'
    '            "Core Inflation Final Warning Score",\n'
    '            "Core CPI Inflation Final Warning Score",\n'
    '        ),\n'
    '        ("12M Core Inflation Z-Score", "12M Core CPI Inflation Z-Score"),\n'
    '        (\n'
    '            "3M Annualised Core Inflation Z-Score",\n'
    '            "3M Annualised Core CPI Inflation Z-Score",\n'
    '        ),\n'
    '        (\n'
    '            "6M Annualised Core Inflation Z-Score",\n'
    '            "6M Annualised Core CPI Inflation Z-Score",\n'
    '        ),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation Z-Score",\n'
    '            "3M Change in 12M Core CPI Inflation Z-Score",\n'
    '        ),\n'
    '    ]\n'
    '    for core_col, cpi_col in numeric_pairs:\n'
    '        assert values_equivalent(core_rows[80][core_col], core_cpi_rows[80][cpi_col])\n'
    '    assert (\n'
    '        core_cpi_rows[70]["12M Core CPI Inflation PercentRank"] == ""\n'
    '        and core_cpi_rows[71]["12M Core CPI Inflation PercentRank"] != ""\n'
    '    )\n'
    '    assert core_cpi_rows[71]["12M Core CPI Inflation Score"] == round(\n'
    '        core_cpi_rows[71]["12M Core CPI Inflation PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_cpi_rows[12]["Distance From 2 Percent Target"],\n'
    '        core_cpi_rows[12]["12M Inflation"] - 0.02,\n'
    '    )\n'
    '    assert (\n'
    '        core_cpi_rows[58]["12M Core CPI Inflation Z-Score"] == ""\n'
    '        and core_cpi_rows[59]["12M Core CPI Inflation Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        core_cpi_rows[80]["Core CPI Inflation Target Status"]\n'
    '        == core_rows[80]["Core Inflation Target Status"]\n'
    '    )\n'
    '    assert _core_inflation_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _core_inflation_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside core inflation pressure"\n'
    '    )\n'
    '    cpi_row = core_cpi_rows[80]\n'
    '    assert cpi_row["Core CPI Inflation Pressure Regime"] in (\n'
    '        "Core CPI inflation pressure rising",\n'
    '        "Core CPI inflation pressure easing",\n'
    '        "Neutral Core CPI inflation pressure",\n'
    '    )\n'
    '    assert cpi_row["Core CPI Inflation Momentum Label"] != ""\n'
    '    assert cpi_row["Core CPI Inflation Signal Interpretation"] != ""\n'
    '    assert cpi_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert cpi_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert cpi_row["Core CPI Inflation Warning Risk Label"] != ""\n'
    '    assert cpi_row["Core CPI Inflation Policy Signal Label"] != ""\n'
    '    assert (\n'
    '        cpi_row["Core CPI Inflation Outlier Label"]\n'
    '        == core_rows[80]["Core Inflation Outlier Label"]\n'
    '    )\n'
    '    assert _core_cpi_inflation_regime(2) == "Core CPI inflation pressure rising"\n'
    '    assert (\n'
    '        _core_cpi_inflation_momentum_label(2, 2) == "Core CPI inflation pressure rising"\n'
    '    )\n'
    '    assert (\n'
    '        _core_cpi_inflation_signal_interpretation(6, 0)\n'
    '        == "Strong upside Core CPI inflation leading signal with supportive policy-pressure profile"\n'
    '    )\n'
    '    assert (\n'
    '        _core_cpi_inflation_warning_label(6)\n'
    '        == "High upside Core CPI inflation warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _core_cpi_inflation_policy_signal_label(6, 0.031, 0.031)\n'
    '        == "Persistent above-target Core CPI inflation pressure"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped core CPI inflation sheet", CONTROL_PANEL_FIFTEENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_core_cpi_inflation_outputs\n'
    '    )\n'
    '    all_specs_with_a17 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["A16 Sheet"],\n'
    '            ["Core CPI Inflation"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Core CPI Inflation", "A17") in all_specs_with_a17\n'
    '    expected_headline_cpi_columns = [\n'
    '        "Sample Count",\n'
    '        "1M Annualised Inflation",\n'
    '        "3M Annualised Inflation",\n'
    '        "6M Annualised Inflation",\n'
    '        "YoY Inflation",\n'
    '        "3M Change in YoY Inflation",\n'
    '        "6M Change in YoY Inflation",\n'
    '        "YoY Inflation Percentile",\n'
    '        "YoY Inflation Score",\n'
    '        "3M Inflation Percentile",\n'
    '        "3M Inflation Score",\n'
    '        "6M Inflation Percentile",\n'
    '        "6M Inflation Score",\n'
    '        "Momentum Percentile",\n'
    '        "Momentum Score",\n'
    '        "YoY Inflation Target Gap",\n'
    '        "Target Regime",\n'
    '        "Inflation Impulse Score",\n'
    '        "Inflation Cycle Turning Point Score",\n'
    '        "Policy Pressure Score",\n'
    '        "Inflation Extreme Score",\n'
    '        "Coincident Score",\n'
    '        "Leading Score",\n'
    '        "Warning Score",\n'
    '        "State",\n'
    '        "Regime",\n'
    '        "Signal Comment",\n'
    '        "Reliability",\n'
    '        "Signal Validity",\n'
    '        "Warning Flag",\n'
    '        "Policy flag",\n'
    '        "YoY Inflation Z-Score",\n'
    '        "3M Inflation Z-Score",\n'
    '        "6M Inflation Z-Score",\n'
    '        "Momentum Z-Score",\n'
    '        "Extreme Flag",\n'
    '    ]\n'
    '    assert HEADLINE_CPI_OUTPUT_COLUMNS == expected_headline_cpi_columns\n'
    '    assert HEADLINE_CPI_INFLATION_OUTPUT_COLUMNS is HEADLINE_CPI_OUTPUT_COLUMNS\n'
    '    headline_spec = select_indicator_engine_spec(\n'
    '        "Unmapped headline CPI inflation sheet", CONTROL_PANEL_SIXTEENTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert headline_spec.calculate is calculate_headline_cpi_inflation_outputs\n'
    '    assert (headline_spec.input_start_col, headline_spec.input_end_col) == ("A", "B")\n'
    '    assert (headline_spec.output_start_col, headline_spec.output_end_col) == ("C", "AL")\n'
    '    assert list(headline_spec.output_columns) == expected_headline_cpi_columns\n'
    '    assert (\n'
    '        indicator_input_range("Headline CPI inflation", "B", "A")\n'
    '        == "\'Headline CPI inflation\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Headline CPI inflation", 953, "AL", "C")\n'
    '        == "\'Headline CPI inflation\'!C2:AL953"\n'
    '    )\n'
    '    assert output_end_col_for_columns(HEADLINE_CPI_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        select_indicator_engine("A17 remains A17", "A17")\n'
    '        is calculate_core_cpi_inflation_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A19 remains A19", "A19") is calculate_core_ppi_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A39 remains A39", "A39")\n'
    '        is calculate_corporate_credit_spreads_outputs\n'
    '    )\n'
    '\n'
    '    assert _headline_cpi_sheets_round(2.675, 2) == 2.68 and round(2.675, 2) != 2.68\n'
    '    assert _headline_cpi_percentrank_inc_sheets([1, 2, 2, 4], 2) == 1 / 3\n'
    '    assert values_equivalent(\n'
    '        _headline_cpi_percentrank_inc_sheets([1, 2, 2, 4], 3), 2.5 / 3\n'
    '    )\n'
    '    assert _headline_cpi_percent_display(-0.01969358388) == "-1.97%"\n'
    '    assert _headline_cpi_percent_display(-0.004805022153) == "-0.48%"\n'
    '    assert _headline_cpi_percent_display(0.08242085661) == "8.24%"\n'
    '    assert _headline_cpi_percent_display(-0.01090564608) == "-1.09%"\n'
    '\n'
    '    headline_cpi_values = [\n'
    '        21.48,\n'
    '        21.62,\n'
    '        22.0,\n'
    '        22.0,\n'
    '        21.95,\n'
    '        22.08,\n'
    '        22.23,\n'
    '        22.4,\n'
    '        22.84,\n'
    '        22.91,\n'
    '        23.06,\n'
    '        23.41,\n'
    '        23.68,\n'
    '        23.67,\n'
    '        23.5,\n'
    '        23.82,\n'
    '        24.01,\n'
    '        24.15,\n'
    '        24.4,\n'
    '        24.43,\n'
    '        24.36,\n'
    '        24.31,\n'
    '        24.16,\n'
    '        24.05,\n'
    '        24.01,\n'
    '        23.91,\n'
    '        23.91,\n'
    '        23.92,\n'
    '        23.91,\n'
    '        23.92,\n'
    '        23.7,\n'
    '        23.7,\n'
    '        23.75,\n'
    '        23.67,\n'
    '        23.7,\n'
    '        23.61,\n'
    '        23.51,\n'
    '        23.61,\n'
    '        23.64,\n'
    '        23.65,\n'
    '        23.77,\n'
    '        23.88,\n'
    '        24.07,\n'
    '        24.2,\n'
    '        24.34,\n'
    '        24.5,\n'
    '        24.6,\n'
    '        24.98,\n'
    '        25.38,\n'
    '        25.83,\n'
    '        25.88,\n'
    '        25.92,\n'
    '        25.99,\n'
    '        25.93,\n'
    '        25.91,\n'
    '        25.86,\n'
    '        26.03,\n'
    '        26.16,\n'
    '        26.32,\n'
    '        26.47,\n'
    '        26.45,\n'
    '        26.41,\n'
    '        26.39,\n'
    '        26.46,\n'
    '        26.47,\n'
    '        26.53,\n'
    '        26.68,\n'
    '        26.69,\n'
    '        26.63,\n'
    '        26.69,\n'
    '        26.69,\n'
    '        26.71,\n'
    '        26.64,\n'
    '        26.59,\n'
    '        26.63,\n'
    '        26.69,\n'
    '    ]\n'
    '    headline_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"headline{i}", "Raw Value": value}\n'
    '            for i, value in enumerate(headline_cpi_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    headline_out = calculate_headline_cpi_inflation_outputs(headline_raw)\n'
    '    headline_rows = df_records(headline_out)\n'
    '    assert (\n'
    '        headline_out.shape[1] == 36\n'
    '        and list(headline_out.columns) == expected_headline_cpi_columns\n'
    '    )\n'
    '    assert (\n'
    '        headline_rows[0]["Sample Count"] == 1 and headline_rows[1]["Sample Count"] == 2\n'
    '    )\n'
    '    assert (\n'
    '        headline_rows[61]["3M Inflation Percentile"] == ""\n'
    '        and headline_rows[62]["3M Inflation Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        headline_rows[64]["6M Inflation Percentile"] == ""\n'
    '        and headline_rows[65]["6M Inflation Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        headline_rows[70]["YoY Inflation Percentile"] == ""\n'
    '        and headline_rows[71]["YoY Inflation Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        headline_rows[73]["Momentum Percentile"] == ""\n'
    '        and headline_rows[74]["Momentum Percentile"] != ""\n'
    '    )\n'
    '\n'
    '    march_1953 = headline_rows[74]\n'
    '    april_1953 = headline_rows[75]\n'
    '    assert [\n'
    '        march_1953[c]\n'
    '        for c in (\n'
    '            "YoY Inflation Percentile",\n'
    '            "3M Inflation Percentile",\n'
    '            "6M Inflation Percentile",\n'
    '            "Momentum Percentile",\n'
    '        )\n'
    '    ] == [0.290, 0.225, 0.279, 0.576]\n'
    '    assert [\n'
    '        march_1953[c]\n'
    '        for c in (\n'
    '            "YoY Inflation Score",\n'
    '            "3M Inflation Score",\n'
    '            "6M Inflation Score",\n'
    '            "Momentum Score",\n'
    '        )\n'
    '    ] == [-4.20, -5.50, -4.42, 1.52]\n'
    '    assert [\n'
    '        april_1953[c]\n'
    '        for c in (\n'
    '            "YoY Inflation Percentile",\n'
    '            "3M Inflation Percentile",\n'
    '            "6M Inflation Percentile",\n'
    '            "Momentum Percentile",\n'
    '        )\n'
    '    ] == [0.270, 0.375, 0.275, 0.600]\n'
    '    assert [\n'
    '        april_1953[c]\n'
    '        for c in (\n'
    '            "YoY Inflation Score",\n'
    '            "3M Inflation Score",\n'
    '            "6M Inflation Score",\n'
    '            "Momentum Score",\n'
    '        )\n'
    '    ] == [-4.60, -2.50, -4.50, 2.00]\n'
    '    assert (\n'
    '        march_1953["Inflation Impulse Score"] == -3.77\n'
    '        and march_1953["Coincident Score"] == -4.19\n'
    '    )\n'
    '    assert (\n'
    '        april_1953["Inflation Impulse Score"] == -2.20\n'
    '        and april_1953["Coincident Score"] == -3.40\n'
    '    )\n'
    '\n'
    '    for row_index, metric_col, rank_col in (\n'
    '        (74, "YoY Inflation", "YoY Inflation Percentile"),\n'
    '        (74, "3M Annualised Inflation", "3M Inflation Percentile"),\n'
    '        (74, "6M Annualised Inflation", "6M Inflation Percentile"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[metric_col]\n'
    '            for row in headline_rows[: row_index + 1]\n'
    '            if not is_blank(row[metric_col])\n'
    '        ]\n'
    '        expected_rank = _headline_cpi_sheets_round(\n'
    '            _headline_cpi_percentrank_inc_sheets(\n'
    '                history, headline_rows[row_index][metric_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert headline_rows[row_index][rank_col] == expected_rank\n'
    '    momentum_internal = [\n'
    '        (\n'
    '            ""\n'
    '            if idx < 3\n'
    '            or is_blank(row["YoY Inflation"])\n'
    '            or is_blank(headline_rows[idx - 3]["YoY Inflation"])\n'
    '            else row["YoY Inflation"] - headline_rows[idx - 3]["YoY Inflation"]\n'
    '        )\n'
    '        for idx, row in enumerate(headline_rows)\n'
    '    ]\n'
    '    momentum_history = [\n'
    '        value for value in momentum_internal[:75] if not is_blank(value)\n'
    '    ]\n'
    '    assert march_1953["Momentum Percentile"] == _headline_cpi_sheets_round(\n'
    '        _headline_cpi_percentrank_inc_sheets(momentum_history, momentum_internal[74]), 3\n'
    '    )\n'
    '    for row in headline_rows:\n'
    '        for percentile_col, score_col in (\n'
    '            ("YoY Inflation Percentile", "YoY Inflation Score"),\n'
    '            ("3M Inflation Percentile", "3M Inflation Score"),\n'
    '            ("6M Inflation Percentile", "6M Inflation Score"),\n'
    '            ("Momentum Percentile", "Momentum Score"),\n'
    '        ):\n'
    '            percentile = row[percentile_col]\n'
    '            if not is_blank(percentile):\n'
    '                assert percentile == _headline_cpi_sheets_round(percentile, 3)\n'
    '                assert row[score_col] == _headline_cpi_sheets_round(\n'
    '                    percentile * 20 - 10, 2\n'
    '                )\n'
    '\n'
    '    march_h_internal = march_1953["YoY Inflation"] - headline_rows[71]["YoY Inflation"]\n'
    '    march_i_internal = march_1953["YoY Inflation"] - headline_rows[68]["YoY Inflation"]\n'
    '    march_r_internal = march_1953["YoY Inflation"] - 0.02\n'
    '    assert march_1953["3M Change in YoY Inflation"] == _headline_cpi_percent_display(\n'
    '        march_h_internal\n'
    '    )\n'
    '    assert march_1953["6M Change in YoY Inflation"] == _headline_cpi_percent_display(\n'
    '        march_i_internal\n'
    '    )\n'
    '    assert (\n'
    '        march_1953["YoY Inflation Target Gap"]\n'
    '        == _headline_cpi_percent_display(march_r_internal)\n'
    '        == "-1.09%"\n'
    '    )\n'
    '    assert _headline_cpi_impulse_score(-5.50, -4.42, 1.52) == -3.77\n'
    '    assert (\n'
    '        _headline_cpi_coincident_score(-4.20, -5.50, -4.42, 1.52, march_r_internal)\n'
    '        == -4.19\n'
    '    )\n'
    '\n'
    '    # Actual threshold-sensitive 2019 score inputs, kept as a local formula fixture.\n'
    '    assert _headline_cpi_impulse_score(-8.62, -6.82, -8.02) == -7.96\n'
    '    feb_2019_t = _headline_cpi_impulse_score(-5.88, -5.90, -6.52)\n'
    '    assert feb_2019_t == -6.01\n'
    '    feb_2019_x = _headline_cpi_coincident_score(\n'
    '        -5.48, -5.88, -5.90, -6.52, -0.0048113846486781235\n'
    '    )\n'
    '    feb_2019_y = _headline_cpi_leading_score(-5.48, -5.88, -5.90, -6.52, feb_2019_t, -6)\n'
    '    feb_2019_z = _headline_cpi_warning_score(-6, 0, 0)\n'
    '    assert feb_2019_x == -5.48 and feb_2019_y == -6.00 and feb_2019_z == -1.80\n'
    '    assert (\n'
    '        _headline_cpi_inflation_signal_interpretation(feb_2019_y, feb_2019_z)\n'
    '        == "Strong disinflationary headline CPI leading signal with downside confirmation"\n'
    '    )\n'
    '    assert (\n'
    '        _headline_cpi_inflation_warning_label(feb_2019_z)\n'
    '        == "Neutral headline CPI warning profile"\n'
    '    )\n'
    '    assert (\n'
    '        _headline_cpi_inflation_warning_label(-3.40)\n'
    '        == "Moderate headline CPI disinflation warning risk"\n'
    '    )\n'
    '\n'
    '    # Displayed z-scores and Extreme Flag retain the pre-existing formulas and gates.\n'
    '    expected_yoy_z = expanding_zscore_sample(\n'
    '        [row["YoY Inflation"] for row in headline_rows],\n'
    '        [row["Sample Count"] for row in headline_rows],\n'
    '        60,\n'
    '    )\n'
    '    expected_3m_z = expanding_zscore_sample(\n'
    '        [row["3M Annualised Inflation"] for row in headline_rows],\n'
    '        [row["Sample Count"] for row in headline_rows],\n'
    '        60,\n'
    '    )\n'
    '    expected_6m_z = expanding_zscore_sample(\n'
    '        [row["6M Annualised Inflation"] for row in headline_rows],\n'
    '        [row["Sample Count"] for row in headline_rows],\n'
    '        60,\n'
    '    )\n'
    '    expected_momentum_z = expanding_zscore_sample(\n'
    '        momentum_internal, [row["Sample Count"] for row in headline_rows], 60\n'
    '    )\n'
    '    for idx in (59, 74, 75):\n'
    '        for col, expected in zip(\n'
    '            (\n'
    '                "YoY Inflation Z-Score",\n'
    '                "3M Inflation Z-Score",\n'
    '                "6M Inflation Z-Score",\n'
    '                "Momentum Z-Score",\n'
    '            ),\n'
    '            (\n'
    '                expected_yoy_z[idx],\n'
    '                expected_3m_z[idx],\n'
    '                expected_6m_z[idx],\n'
    '                expected_momentum_z[idx],\n'
    '            ),\n'
    '        ):\n'
    '            assert values_equivalent(headline_rows[idx][col], expected)\n'
    '    assert march_1953["Extreme Flag"] == "Normal range"\n'
    '    assert (\n'
    '        _headline_cpi_inflation_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside headline CPI inflation pressure"\n'
    '    )\n'
    '\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped headline CPI inflation sheet",\n'
    '            CONTROL_PANEL_SIXTEENTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_headline_cpi_inflation_outputs\n'
    '    )\n'
    '    all_specs_with_a18 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 18)] + [["Headline CPI Inflation"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Headline CPI Inflation", "A18") in all_specs_with_a18\n'
    '    core_ppi_out = calculate_core_ppi_outputs(core_raw)\n'
    '    core_ppi_rows = df_records(core_ppi_out)\n'
    '    assert core_ppi_out.shape[1] == 36\n'
    '    assert output_end_col_for_columns(CORE_PPI_OUTPUT_COLUMNS) == "AL"\n'
    '    assert indicator_output_range("Core PPI", 678, "AL") == "\'Core PPI\'!C2:AL678"\n'
    '    assert (\n'
    '        core_ppi_rows[0]["Observation Count"] == 1\n'
    '        and core_ppi_rows[1]["Observation Count"] == 2\n'
    '    )\n'
    '    for core_col, ppi_col in [\n'
    '        ("Observation Count", "Observation Count"),\n'
    '        ("1M Annualised Inflation", "1M Annualised Inflation"),\n'
    '        ("3M Annualised Inflation", "3M Annualised Inflation"),\n'
    '        ("6M Annualised Inflation", "6M Annualised Inflation"),\n'
    '        ("12M Inflation", "12M Inflation"),\n'
    '        ("3M Change in 12M Inflation", "3M Change in 12M Inflation"),\n'
    '        ("6M Change in 12M Inflation", "6M Change in 12M Inflation"),\n'
    '        ("12M Core Inflation PercentRank", "12M Core PPI PercentRank"),\n'
    '        ("12M Core Inflation Score", "12M Core PPI Score"),\n'
    '        (\n'
    '            "3M Annualised Core Inflation PercentRank",\n'
    '            "3M Annualised Core PPI PercentRank",\n'
    '        ),\n'
    '        ("3M Annualised Core Inflation Score", "3M Annualised Core PPI Score"),\n'
    '        (\n'
    '            "6M Annualised Core Inflation PercentRank",\n'
    '            "6M Annualised Core PPI PercentRank",\n'
    '        ),\n'
    '        ("6M Annualised Core Inflation Score", "6M Annualised Core PPI Score"),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation PercentRank",\n'
    '            "3M Change in 12M Core PPI PercentRank",\n'
    '        ),\n'
    '        ("3M Change in 12M Core Inflation Score", "3M Change in 12M Core PPI Score"),\n'
    '        ("Distance From 2 Percent Target", "Distance From 2 Percent Target"),\n'
    '        (\n'
    '            "Core Inflation Momentum Composite Score",\n'
    '            "Core PPI Momentum Composite Score",\n'
    '        ),\n'
    '        (\n'
    '            "Core Inflation Recovery / Cooling Score",\n'
    '            "Core PPI Recovery / Cooling Score",\n'
    '        ),\n'
    '        ("Core Inflation Target Pressure Score", "Core PPI Target Pressure Score"),\n'
    '        ("Core Inflation Outlier Score", "Core PPI Outlier Score"),\n'
    '        ("Core Inflation Leading Score", "Core PPI Leading Score"),\n'
    '        ("Core Inflation Final Warning Score", "Core PPI Final Warning Score"),\n'
    '        ("12M Core Inflation Z-Score", "12M Core PPI Z-Score"),\n'
    '        ("3M Annualised Core Inflation Z-Score", "3M Annualised Core PPI Z-Score"),\n'
    '        ("6M Annualised Core Inflation Z-Score", "6M Annualised Core PPI Z-Score"),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation Z-Score",\n'
    '            "3M Change in 12M Core PPI Z-Score",\n'
    '        ),\n'
    '    ]:\n'
    '        assert values_equivalent(core_rows[80][core_col], core_ppi_rows[80][ppi_col])\n'
    '    assert (\n'
    '        core_ppi_rows[70]["12M Core PPI PercentRank"] == ""\n'
    '        and core_ppi_rows[71]["12M Core PPI PercentRank"] != ""\n'
    '    )\n'
    '    assert core_ppi_rows[71]["12M Core PPI Score"] == round(\n'
    '        core_ppi_rows[71]["12M Core PPI PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        core_ppi_rows[12]["Distance From 2 Percent Target"],\n'
    '        core_ppi_rows[12]["12M Inflation"] - 0.02,\n'
    '    )\n'
    '    assert (\n'
    '        core_ppi_rows[80]["Core PPI Target Status"]\n'
    '        == core_rows[80]["Core Inflation Target Status"]\n'
    '    )\n'
    '    assert (\n'
    '        core_ppi_rows[58]["12M Core PPI Z-Score"] == ""\n'
    '        and core_ppi_rows[59]["12M Core PPI Z-Score"] != ""\n'
    '    )\n'
    '    assert _core_inflation_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _core_ppi_outlier_label([3.1, -3.5, 0, 0]) == "Extreme upside Core PPI pressure"\n'
    '    )\n'
    '    ppi_row = core_ppi_rows[80]\n'
    '    pk = ppi_row["12M Core PPI Score"]\n'
    '    pm = ppi_row["3M Annualised Core PPI Score"]\n'
    '    po = ppi_row["6M Annualised Core PPI Score"]\n'
    '    pq = ppi_row["3M Change in 12M Core PPI Score"]\n'
    '    pr = ppi_row["Distance From 2 Percent Target"]\n'
    '    pt = ppi_row["Core PPI Momentum Composite Score"]\n'
    '    pu = ppi_row["Core PPI Recovery / Cooling Score"]\n'
    '    pv = ppi_row["Core PPI Target Pressure Score"]\n'
    '    pw = ppi_row["Core PPI Outlier Score"]\n'
    '    assert values_equivalent(pt, round(clamp(0.45 * pm + 0.35 * po + 0.2 * pq), 2))\n'
    '    assert values_equivalent(\n'
    '        ppi_row["Core PPI Leading Score"],\n'
    '        round(\n'
    '            clamp(\n'
    '                0.35 * pk\n'
    '                + 0.25 * pm\n'
    '                + 0.2 * po\n'
    '                + 0.1 * pq\n'
    '                + 0.1 * clamp(pr * 500, -10, 10)\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        ppi_row["Core PPI Warning Composite Score"],\n'
    '        round(\n'
    '            clamp(0.1 * pk + 0.35 * pm + 0.25 * po + 0.2 * pq + 0.05 * pt + 0.05 * pu),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert round(\n'
    '        clamp(0.1 * 1 + 0.35 * 2 + 0.25 * 3 + 0.2 * 4 + 0.05 * 5 + 0.05 * 6), 2\n'
    '    ) != round(clamp(0.15 * 1 + 0.35 * 2 + 0.25 * 3 + 0.2 * 4 + 0.05 * 6), 2)\n'
    '    assert values_equivalent(\n'
    '        ppi_row["Core PPI Final Warning Score"],\n'
    '        round(clamp(0.35 * pu + 0.45 * pv + 0.2 * pw), 2),\n'
    '    )\n'
    '    assert ppi_row["Core PPI Pressure Regime"] in (\n'
    '        "Core PPI pressure rising",\n'
    '        "Core PPI pressure easing",\n'
    '        "Neutral Core PPI pressure",\n'
    '    )\n'
    '    assert ppi_row["Core PPI Momentum Label"] != ""\n'
    '    assert ppi_row["Core PPI Signal Interpretation"] != ""\n'
    '    assert ppi_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert ppi_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert ppi_row["Core PPI Warning Risk Label"] != ""\n'
    '    assert ppi_row["Core PPI Policy Signal Label"] != ""\n'
    '    assert ppi_row["Core PPI Outlier Label"] != ""\n'
    '    assert _core_ppi_regime(2) == "Core PPI pressure rising"\n'
    '    assert _core_ppi_momentum_label(2, 2) == "Core PPI pressure rising"\n'
    '    assert (\n'
    '        _core_ppi_signal_interpretation(6, 0)\n'
    '        == "Strong upside Core PPI leading signal with supportive pipeline-pressure profile"\n'
    '    )\n'
    '    assert _core_ppi_warning_label(6) == "High upside Core PPI warning risk"\n'
    '    assert (\n'
    '        _core_ppi_policy_signal_label(6, 0.031, 0.031)\n'
    '        == "Persistent upside Core PPI pipeline pressure"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped core PPI sheet", CONTROL_PANEL_SEVENTEENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_core_ppi_outputs\n'
    '    )\n'
    '    all_specs_with_a19 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["A16 Sheet"],\n'
    '            ["A17 Sheet"],\n'
    '            ["A18 Sheet"],\n'
    '            ["Core PPI"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Core PPI", "A19") in all_specs_with_a19\n'
    '    headline_ppi_out = calculate_headline_ppi_outputs(core_raw)\n'
    '    headline_ppi_rows = df_records(headline_ppi_out)\n'
    '    assert headline_ppi_out.shape[1] == 36\n'
    '    assert output_end_col_for_columns(HEADLINE_PPI_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        indicator_output_range("Headline PPI", 678, "AL") == "\'Headline PPI\'!C2:AL678"\n'
    '    )\n'
    '    assert (\n'
    '        headline_ppi_rows[0]["Observation Count"] == 1\n'
    '        and headline_ppi_rows[1]["Observation Count"] == 2\n'
    '    )\n'
    '    for core_col, ppi_col in [\n'
    '        ("Observation Count", "Observation Count"),\n'
    '        ("1M Annualised Inflation", "1M Annualised Inflation"),\n'
    '        ("3M Annualised Inflation", "3M Annualised Inflation"),\n'
    '        ("6M Annualised Inflation", "6M Annualised Inflation"),\n'
    '        ("12M Inflation", "12M Inflation"),\n'
    '        ("3M Change in 12M Inflation", "3M Change in 12M Inflation"),\n'
    '        ("6M Change in 12M Inflation", "6M Change in 12M Inflation"),\n'
    '        ("12M Core Inflation PercentRank", "12M Headline PPI PercentRank"),\n'
    '        ("12M Core Inflation Score", "12M Headline PPI Score"),\n'
    '        (\n'
    '            "3M Annualised Core Inflation PercentRank",\n'
    '            "3M Annualised Headline PPI PercentRank",\n'
    '        ),\n'
    '        ("3M Annualised Core Inflation Score", "3M Annualised Headline PPI Score"),\n'
    '        (\n'
    '            "6M Annualised Core Inflation PercentRank",\n'
    '            "6M Annualised Headline PPI PercentRank",\n'
    '        ),\n'
    '        ("6M Annualised Core Inflation Score", "6M Annualised Headline PPI Score"),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation PercentRank",\n'
    '            "3M Change in 12M Headline PPI PercentRank",\n'
    '        ),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation Score",\n'
    '            "3M Change in 12M Headline PPI Score",\n'
    '        ),\n'
    '        ("Distance From 2 Percent Target", "Distance From 2 Percent Target"),\n'
    '        (\n'
    '            "Core Inflation Recovery / Cooling Score",\n'
    '            "Headline PPI Recovery / Cooling Score",\n'
    '        ),\n'
    '        ("Core Inflation Target Pressure Score", "Headline PPI Target Pressure Score"),\n'
    '        ("Core Inflation Outlier Score", "Headline PPI Outlier Score"),\n'
    '        ("12M Core Inflation Z-Score", "12M Headline PPI Z-Score"),\n'
    '        ("3M Annualised Core Inflation Z-Score", "3M Annualised Headline PPI Z-Score"),\n'
    '        ("6M Annualised Core Inflation Z-Score", "6M Annualised Headline PPI Z-Score"),\n'
    '        (\n'
    '            "3M Change in 12M Core Inflation Z-Score",\n'
    '            "3M Change in 12M Headline PPI Z-Score",\n'
    '        ),\n'
    '    ]:\n'
    '        assert values_equivalent(\n'
    '            core_rows[80][core_col], headline_ppi_rows[80][ppi_col]\n'
    '        )\n'
    '    assert (\n'
    '        headline_ppi_rows[70]["12M Headline PPI PercentRank"] == ""\n'
    '        and headline_ppi_rows[71]["12M Headline PPI PercentRank"] != ""\n'
    '    )\n'
    '    assert headline_ppi_rows[71]["12M Headline PPI Score"] == round(\n'
    '        headline_ppi_rows[71]["12M Headline PPI PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        headline_ppi_rows[12]["Distance From 2 Percent Target"],\n'
    '        headline_ppi_rows[12]["12M Inflation"] - 0.02,\n'
    '    )\n'
    '    assert (\n'
    '        headline_ppi_rows[80]["Headline PPI Target Status"]\n'
    '        == core_rows[80]["Core Inflation Target Status"]\n'
    '    )\n'
    '    assert (\n'
    '        headline_ppi_rows[58]["12M Headline PPI Z-Score"] == ""\n'
    '        and headline_ppi_rows[59]["12M Headline PPI Z-Score"] != ""\n'
    '    )\n'
    '    assert _core_inflation_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _headline_ppi_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside Headline PPI pressure"\n'
    '    )\n'
    '    headline_ppi_row = headline_ppi_rows[80]\n'
    '    hpk = headline_ppi_row["12M Headline PPI Score"]\n'
    '    hpm = headline_ppi_row["3M Annualised Headline PPI Score"]\n'
    '    hpo = headline_ppi_row["6M Annualised Headline PPI Score"]\n'
    '    hpq = headline_ppi_row["3M Change in 12M Headline PPI Score"]\n'
    '    hpr = headline_ppi_row["Distance From 2 Percent Target"]\n'
    '    hpt = headline_ppi_row["Headline PPI Momentum Composite Score"]\n'
    '    hpu = headline_ppi_row["Headline PPI Recovery / Cooling Score"]\n'
    '    hpv = headline_ppi_row["Headline PPI Target Pressure Score"]\n'
    '    hpw = headline_ppi_row["Headline PPI Outlier Score"]\n'
    '    assert values_equivalent(hpt, round(clamp(0.5 * hpm + 0.3 * hpo + 0.2 * hpq), 2))\n'
    '    assert values_equivalent(\n'
    '        headline_ppi_row["Headline PPI Leading Score"],\n'
    '        round(\n'
    '            clamp(\n'
    '                0.3 * hpk\n'
    '                + 0.3 * hpm\n'
    '                + 0.2 * hpo\n'
    '                + 0.1 * hpq\n'
    '                + 0.1 * clamp(hpr * 500, -10, 10)\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        headline_ppi_row["Headline PPI Warning Composite Score"],\n'
    '        round(\n'
    '            clamp(\n'
    '                0.1 * hpk + 0.3 * hpm + 0.2 * hpo + 0.2 * hpq + 0.15 * hpt + 0.05 * hpu\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        headline_ppi_row["Headline PPI Final Warning Score"],\n'
    '        round(clamp(0.3 * hpu + 0.4 * hpv + 0.3 * hpw), 2),\n'
    '    )\n'
    '    assert round(clamp(0.5 * 2 + 0.3 * 3 + 0.2 * 4), 2) != round(\n'
    '        clamp(0.45 * 2 + 0.35 * 3 + 0.2 * 4), 2\n'
    '    )\n'
    '    assert headline_ppi_row["Headline PPI Pressure Regime"] in (\n'
    '        "Headline PPI pressure rising",\n'
    '        "Headline PPI pressure easing",\n'
    '        "Neutral Headline PPI pressure",\n'
    '    )\n'
    '    assert headline_ppi_row["Headline PPI Momentum Label"] != ""\n'
    '    assert headline_ppi_row["Headline PPI Signal Interpretation"] != ""\n'
    '    assert headline_ppi_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert headline_ppi_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert headline_ppi_row["Headline PPI Warning Risk Label"] != ""\n'
    '    assert headline_ppi_row["Headline PPI Policy Signal Label"] != ""\n'
    '    assert headline_ppi_row["Headline PPI Outlier Label"] != ""\n'
    '    assert _headline_ppi_regime(2) == "Headline PPI pressure rising"\n'
    '    assert _headline_ppi_momentum_label(2, 2) == "Headline PPI pressure rising"\n'
    '    assert (\n'
    '        _headline_ppi_signal_interpretation(6, 0)\n'
    '        == "Strong upside Headline PPI leading signal with supportive pipeline-pressure profile"\n'
    '    )\n'
    '    assert _headline_ppi_warning_label(6) == "High upside Headline PPI warning risk"\n'
    '    assert (\n'
    '        _headline_ppi_policy_signal_label(6, 0.031, 0.031)\n'
    '        == "Persistent upside Headline PPI pipeline pressure"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped headline PPI sheet", CONTROL_PANEL_EIGHTEENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_headline_ppi_outputs\n'
    '    )\n'
    '    all_specs_with_a20 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["A16 Sheet"],\n'
    '            ["A17 Sheet"],\n'
    '            ["A18 Sheet"],\n'
    '            ["A19 Sheet"],\n'
    '            ["Headline PPI"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Headline PPI", "A20") in all_specs_with_a20\n'
    '    wage_values = [100 + i + (0.05 * i * i) for i in range(90)]\n'
    '    wage_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"wage{i}", "Raw Value": wage_values[i]}\n'
    '            for i in range(len(wage_values))\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    wage_out = calculate_wage_growth_outputs(wage_raw)\n'
    '    wage_rows = df_records(wage_out)\n'
    '    assert wage_out.shape[1] == 34\n'
    '    assert output_end_col_for_columns(WAGE_GROWTH_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert indicator_output_range("Wage Growth", 678, "AJ") == "\'Wage Growth\'!C2:AJ678"\n'
    '    assert (\n'
    '        wage_rows[0]["Observation Count"] == 1\n'
    '        and wage_rows[1]["Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_rows[1]["1-Period Annualised Wage Growth"],\n'
    '        (wage_values[1] / wage_values[0]) ** 4 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_rows[2]["2-Period Annualised Wage Growth"],\n'
    '        (wage_values[2] / wage_values[0]) ** 2 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_rows[4]["4-Period Wage Growth"], wage_values[4] / wage_values[0] - 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_rows[5]["1-Period Change in 4-Period Wage Growth"],\n'
    '        wage_rows[5]["4-Period Wage Growth"] - wage_rows[4]["4-Period Wage Growth"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_rows[6]["2-Period Change in 4-Period Wage Growth"],\n'
    '        wage_rows[6]["4-Period Wage Growth"] - wage_rows[4]["4-Period Wage Growth"],\n'
    '    )\n'
    '    assert (\n'
    '        wage_rows[22]["4-Period Wage Growth PercentRank"] == ""\n'
    '        and wage_rows[23]["4-Period Wage Growth PercentRank"] != ""\n'
    '    )\n'
    '    assert wage_rows[23]["4-Period Wage Growth Score"] == round(\n'
    '        wage_rows[23]["4-Period Wage Growth PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert (\n'
    '        wage_rows[19]["1-Period Annualised Wage Growth PercentRank"] == ""\n'
    '        and wage_rows[20]["1-Period Annualised Wage Growth PercentRank"] != ""\n'
    '    )\n'
    '    assert wage_rows[20]["1-Period Annualised Wage Growth Score"] == round(\n'
    '        wage_rows[20]["1-Period Annualised Wage Growth PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert (\n'
    '        _wage_pressure_status(0.051, 0.051) == "Elevated and persistent wage pressure"\n'
    '    )\n'
    '    assert _wage_pressure_status(0.051, 0.039) == "Elevated but cooling wage pressure"\n'
    '    assert _wage_pressure_status(0.049, 0.051) == "Wage pressure re-accelerating"\n'
    '    assert (\n'
    '        wage_rows[22]["4-Period Wage Growth Z-Score"] == ""\n'
    '        and wage_rows[23]["4-Period Wage Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        wage_rows[19]["1-Period Annualised Wage Growth Z-Score"] == ""\n'
    '        and wage_rows[20]["1-Period Annualised Wage Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        _rounded_expanding_zscores([1, 2, 3], [20, 20, 20], 20)[2], 1.0\n'
    '    )\n'
    '    assert wage_rows[23]["4-Period Wage Growth Z-Score"] == round(\n'
    '        wage_rows[23]["4-Period Wage Growth Z-Score"], 2\n'
    '    )\n'
    '    assert _wage_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _wage_pressure_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside wage pressure"\n'
    '    )\n'
    '    wage_row = wage_rows[30]\n'
    '    wj = wage_row["4-Period Wage Growth Score"]\n'
    '    wl = wage_row["1-Period Annualised Wage Growth Score"]\n'
    '    wn = wage_row["2-Period Annualised Wage Growth Score"]\n'
    '    wp = wage_row["1-Period Change in 4-Period Wage Growth Score"]\n'
    '    wr = wage_row["Wage Pressure Momentum Composite Score"]\n'
    '    ws = wage_row["Wage Pressure Recovery / Cooling Score"]\n'
    '    wt = wage_row["Wage Pressure Persistence Score"]\n'
    '    wu = wage_row["Wage Pressure Outlier Score"]\n'
    '    assert values_equivalent(wr, round(clamp(0.45 * wl + 0.35 * wn + 0.2 * wp), 2))\n'
    '    assert _wage_recovery_cooling_score(0.02, 1, 1) == 6\n'
    '    assert _wage_recovery_cooling_score(0.04, -1, -1) == -5\n'
    '    assert _wage_persistence_score(0.041, 0.041, 0.041) == 6\n'
    '    assert _wage_persistence_score(0.019, 0.019, 0.019) == -6\n'
    '    assert values_equivalent(\n'
    '        wage_row["Wage Pressure Leading Score"],\n'
    '        round(clamp(0.4 * wj + 0.25 * wl + 0.25 * wn + 0.1 * wp), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_row["Wage Pressure Warning Composite Score"],\n'
    '        round(\n'
    '            clamp(0.15 * wj + 0.3 * wl + 0.2 * wn + 0.2 * wp + 0.1 * wr + 0.05 * ws), 2\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        wage_row["Wage Pressure Final Warning Score"],\n'
    '        round(clamp(0.35 * ws + 0.45 * wt + 0.2 * wu), 2),\n'
    '    )\n'
    '    assert wage_row["Wage Pressure Regime"] in (\n'
    '        "Wage pressure rising",\n'
    '        "Wage pressure easing",\n'
    '        "Neutral wage pressure",\n'
    '    )\n'
    '    assert wage_row["Wage Pressure Momentum Label"] != ""\n'
    '    assert wage_row["Wage Pressure Signal Interpretation"] != ""\n'
    '    assert wage_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert wage_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert wage_row["Wage Pressure Warning Risk Label"] != ""\n'
    '    assert wage_row["Wage Pressure Policy Signal Label"] != ""\n'
    '    assert wage_row["Wage Pressure Outlier Label"] != ""\n'
    '    assert _wage_pressure_regime(2) == "Wage pressure rising"\n'
    '    assert _wage_pressure_momentum_label(2, 2) == "Wage pressure rising"\n'
    '    assert (\n'
    '        _wage_pressure_signal_interpretation(6, 0)\n'
    '        == "Strong upside wage-pressure leading signal with supportive labour-cost pressure profile"\n'
    '    )\n'
    '    assert _wage_pressure_warning_label(6) == "High upside wage-pressure warning risk"\n'
    '    assert (\n'
    '        _wage_pressure_policy_signal_label(6, 0.051, 0.051)\n'
    '        == "Persistent wage-driven inflation pressure"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped wage growth sheet", CONTROL_PANEL_NINETEENTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_wage_growth_outputs\n'
    '    )\n'
    '    all_specs_with_a21 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["A16 Sheet"],\n'
    '            ["A17 Sheet"],\n'
    '            ["A18 Sheet"],\n'
    '            ["A19 Sheet"],\n'
    '            ["A20 Sheet"],\n'
    '            ["Wage Growth"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Wage Growth", "A21") in all_specs_with_a21\n'
    '    expectations_values = [2.2 + 0.01 * i + 0.25 * math.sin(i / 5) for i in range(140)]\n'
    '    expectations_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"expect{i}", "Raw Value": expectations_values[i]}\n'
    '            for i in range(len(expectations_values))\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    expectations_out = calculate_short_term_inflation_expectations_outputs(\n'
    '        expectations_raw\n'
    '    )\n'
    '    expectations_rows = df_records(expectations_out)\n'
    '    assert expectations_out.shape[1] == 34\n'
    '    assert (\n'
    '        output_end_col_for_columns(SHORT_TERM_INFLATION_EXPECTATIONS_OUTPUT_COLUMNS)\n'
    '        == "AJ"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Short-Term Inflation Expectations", 678, "AJ")\n'
    '        == "\'Short-Term Inflation Expectations\'!C2:AJ678"\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[0]["Observation Count"] == 1\n'
    '        and expectations_rows[1]["Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[2]["3-Period Moving Average"],\n'
    '        sum(expectations_values[:3]) / 3,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[1]["1-Period Change"],\n'
    '        expectations_values[1] - expectations_values[0],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[5]["3-Period Change in Moving Average"],\n'
    '        expectations_rows[5]["3-Period Moving Average"]\n'
    '        - expectations_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[8]["6-Period Change in Moving Average"],\n'
    '        expectations_rows[8]["3-Period Moving Average"]\n'
    '        - expectations_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_rows[14]["12-Period Change in Moving Average"],\n'
    '        expectations_rows[14]["3-Period Moving Average"]\n'
    '        - expectations_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[60]["Short-Term Expectations Level PercentRank"] == ""\n'
    '        and expectations_rows[61]["Short-Term Expectations Level PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[63]["3-Period Expectations Change PercentRank"] == ""\n'
    '        and expectations_rows[64]["3-Period Expectations Change PercentRank"] != ""\n'
    '    )\n'
    '    assert expectations_rows[61]["Short-Term Expectations Level Score"] == round(\n'
    '        expectations_rows[61]["Short-Term Expectations Level PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert expectations_rows[64]["3-Period Expectations Change Score"] == round(\n'
    '        expectations_rows[64]["3-Period Expectations Change PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert (\n'
    '        _short_term_expectations_status(4.1, 0.4)\n'
    '        == "Elevated and rising short-term expectations"\n'
    '    )\n'
    '    assert (\n'
    '        _short_term_expectations_status(4.1, -0.1)\n'
    '        == "Elevated but cooling short-term expectations"\n'
    '    )\n'
    '    assert (\n'
    '        _short_term_expectations_status(3.3, 0.4)\n'
    '        == "Above-normal expectations and re-accelerating"\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[60]["Short-Term Expectations Level Z-Score"] == ""\n'
    '        and expectations_rows[61]["Short-Term Expectations Level Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        expectations_rows[63]["3-Period Expectations Change Z-Score"] == ""\n'
    '        and expectations_rows[64]["3-Period Expectations Change Z-Score"] != ""\n'
    '    )\n'
    '    assert expectations_rows[61]["Short-Term Expectations Level Z-Score"] == round(\n'
    '        expectations_rows[61]["Short-Term Expectations Level Z-Score"], 2\n'
    '    )\n'
    '    assert _inflation_expectations_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _inflation_expectations_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside inflation-expectations pressure"\n'
    '    )\n'
    '    expectations_row = expectations_rows[80]\n'
    '    ej = expectations_row["Short-Term Expectations Level Score"]\n'
    '    el = expectations_row["3-Period Expectations Change Score"]\n'
    '    en = expectations_row["6-Period Expectations Change Score"]\n'
    '    ep = expectations_row["12-Period Expectations Change Score"]\n'
    '    er = expectations_row["Inflation Expectations Momentum Composite Score"]\n'
    '    es = expectations_row["Inflation Expectations Recovery / Cooling Score"]\n'
    '    et = expectations_row["Inflation Expectations Pressure Score"]\n'
    '    eu = expectations_row["Inflation Expectations Outlier Score"]\n'
    '    assert values_equivalent(er, round(clamp(0.45 * el + 0.35 * en + 0.2 * ep), 2))\n'
    '    assert _inflation_expectations_recovery_cooling_score(2.8, 1, 1) == 6\n'
    '    assert _inflation_expectations_recovery_cooling_score(3.2, -1, -1) == -5\n'
    '    assert _inflation_expectations_pressure_score(4.6, 1, 1) == 6\n'
    '    assert _inflation_expectations_pressure_score(2.4, -1, -1) == -6\n'
    '    assert values_equivalent(\n'
    '        expectations_row["Inflation Expectations Leading Score"],\n'
    '        round(clamp(0.5 * ej + 0.25 * el + 0.15 * en + 0.1 * ep), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_row["Inflation Expectations Warning Composite Score"],\n'
    '        round(\n'
    '            clamp(0.15 * ej + 0.3 * el + 0.25 * en + 0.15 * ep + 0.1 * er + 0.05 * es),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        expectations_row["Inflation Expectations Final Warning Score"],\n'
    '        round(clamp(0.35 * es + 0.45 * et + 0.2 * eu), 2),\n'
    '    )\n'
    '    assert expectations_row["Inflation Expectations Regime"] in (\n'
    '        "Inflation expectations rising",\n'
    '        "Inflation expectations easing",\n'
    '        "Neutral inflation-expectations pressure",\n'
    '    )\n'
    '    assert expectations_row["Inflation Expectations Momentum Label"] != ""\n'
    '    assert expectations_row["Inflation Expectations Signal Interpretation"] != ""\n'
    '    assert expectations_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert expectations_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert expectations_row["Inflation Expectations Warning Risk Label"] != ""\n'
    '    assert expectations_row["Inflation Expectations Policy Signal Label"] != ""\n'
    '    assert expectations_row["Inflation Expectations Outlier Label"] != ""\n'
    '    assert _inflation_expectations_regime(2) == "Inflation expectations rising"\n'
    '    assert (\n'
    '        _inflation_expectations_momentum_label(2, 2) == "Inflation expectations rising"\n'
    '    )\n'
    '    assert (\n'
    '        _inflation_expectations_signal_interpretation(6, 0)\n'
    '        == "Strong upside inflation-expectations leading signal with supportive expectations-pressure profile"\n'
    '    )\n'
    '    assert (\n'
    '        _inflation_expectations_warning_label(6)\n'
    '        == "High upside inflation-expectations warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _inflation_expectations_policy_signal_label(6, 4.1, 0.4)\n'
    '        == "Short-term expectations de-anchoring risk"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped short-term inflation expectations sheet",\n'
    '            CONTROL_PANEL_TWENTIETH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_short_term_inflation_expectations_outputs\n'
    '    )\n'
    '    all_specs_with_a22 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["A16 Sheet"],\n'
    '            ["A17 Sheet"],\n'
    '            ["A18 Sheet"],\n'
    '            ["A19 Sheet"],\n'
    '            ["A20 Sheet"],\n'
    '            ["A21 Sheet"],\n'
    '            ["Short-Term Inflation Expectations"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert (\n'
    '        BatchIndicatorSpec("Short-Term Inflation Expectations", "A22")\n'
    '        in all_specs_with_a22\n'
    '    )\n'
    '    # A23 is isolated from the generic yield engines: its schema, spreadsheet\n'
    '    # arithmetic, percentile gates, and boundary behavior are regression-tested here.\n'
    '    expected_real_yield_columns = [\n'
    '        "3M MA Real Yield",\n'
    '        "Sample Count",\n'
    '        "1M Real Yield Change",\n'
    '        "3M Real Yield Change",\n'
    '        "6M Real Yield Change",\n'
    '        "12M Real Yield Change",\n'
    '        "Real Yield Level Percentile",\n'
    '        "Real Yield Level Score",\n'
    '        "3M Change Percentile",\n'
    '        "3M Change Score",\n'
    '        "6M Change Percentile",\n'
    '        "6M Change Score",\n'
    '        "12M Change Percentile",\n'
    '        "12M Change Score",\n'
    '        "Real Yield Regime",\n'
    '        "Real Yield Asset Bias",\n'
    '        "1M Real Yield Shock Flag",\n'
    '        "Real Yield Impulse Score",\n'
    '        "Real Yield Cycle Turning Point Score",\n'
    '        "Real Yield Shock Score",\n'
    '        "Policy Restriction Score",\n'
    '        "Real Yield Extreme Score",\n'
    '        "Coincident Score",\n'
    '        "Leading Score",\n'
    '        "Warning Score",\n'
    '        "State",\n'
    '        "Regime",\n'
    '        "Signal Comment",\n'
    '        "Reliability",\n'
    '        "Signal Validity",\n'
    '        "Warning Flag",\n'
    '        "Level Z-Score",\n'
    '        "3M Change Z-Score",\n'
    '        "6M Change Z-Score",\n'
    '        "12M Change Z-Score",\n'
    '        "Extreme Flag",\n'
    '    ]\n'
    '    assert REAL_YIELD_OUTPUT_COLUMNS == expected_real_yield_columns\n'
    '    real_yield_spec = select_indicator_engine_spec(\n'
    '        "Unmapped real yield sheet", CONTROL_PANEL_TWENTY_FIRST_INDICATOR_CELL\n'
    '    )\n'
    '    assert real_yield_spec.calculate is calculate_real_yield_outputs\n'
    '    assert (real_yield_spec.input_start_col, real_yield_spec.input_end_col) == (\n'
    '        "A",\n'
    '        "B",\n'
    '    )\n'
    '    assert (real_yield_spec.output_start_col, real_yield_spec.output_end_col) == (\n'
    '        "C",\n'
    '        "AL",\n'
    '    )\n'
    '    assert list(real_yield_spec.output_columns) == expected_real_yield_columns\n'
    '    assert (\n'
    '        indicator_input_range(\n'
    '            "10Y real yield",\n'
    '            real_yield_spec.input_end_col,\n'
    '            real_yield_spec.input_start_col,\n'
    '        )\n'
    '        == "\'10Y real yield\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range(\n'
    '            "10Y real yield",\n'
    '            281,\n'
    '            real_yield_spec.output_end_col,\n'
    '            real_yield_spec.output_start_col,\n'
    '        )\n'
    '        == "\'10Y real yield\'!C2:AL281"\n'
    '    )\n'
    '    assert output_end_col_for_columns(REAL_YIELD_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        select_indicator_engine("A22 remains unchanged", "A22")\n'
    '        is calculate_short_term_inflation_expectations_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A24 remains unchanged", "A24")\n'
    '        is calculate_two_year_yield_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A39 remains unchanged", "A39")\n'
    '        is calculate_corporate_credit_spreads_outputs\n'
    '    )\n'
    '\n'
    '    assert _real_yield_sheets_round(2.675, 2) == 2.68 and round(2.675, 2) != 2.68\n'
    '    assert _real_yield_sheets_round(-2.675, 2) == -2.68\n'
    '    assert _real_yield_sheets_round(0.0625, 3) == 0.063\n'
    '    assert _real_yield_percentrank_inc_sheets([], 1) == ""\n'
    '    assert _real_yield_percentrank_inc_sheets([2], 2) == 0.0\n'
    '    assert _real_yield_percentrank_inc_sheets([1, 2, 2, 4], 2) == 1 / 3\n'
    '    assert values_equivalent(\n'
    '        _real_yield_percentrank_inc_sheets([1, 2, 2, 4], 3), 2.5 / 3\n'
    '    )\n'
    '\n'
    '    # Static A23 source fixture (no Google Sheets access from self-tests).\n'
    '    real_yield_values = [\n'
    '        2.29,\n'
    '        1.99,\n'
    '        1.94,\n'
    '        2.18,\n'
    '        1.91,\n'
    '        1.72,\n'
    '        2.11,\n'
    '        2.32,\n'
    '        2.19,\n'
    '        2.08,\n'
    '        1.96,\n'
    '        1.98,\n'
    '        1.89,\n'
    '        1.76,\n'
    '        1.47,\n'
    '        1.9,\n'
    '        2.09,\n'
    '        2.15,\n'
    '        2.02,\n'
    '        1.86,\n'
    '        1.8,\n'
    '        1.73,\n'
    '        1.68,\n'
    '        1.67,\n'
    '        1.72,\n'
    '        1.63,\n'
    '        1.79,\n'
    '        1.71,\n'
    '        1.65,\n'
    '        1.67,\n'
    '        1.88,\n'
    '        1.89,\n'
    '        1.7,\n'
    '        1.94,\n'
    '        2.06,\n'
    '        2.12,\n'
    '        2.01,\n'
    '        2.05,\n'
    '        2.2,\n'
    '        2.41,\n'
    '        2.45,\n'
    '        2.53,\n'
    '        2.51,\n'
    '        2.29,\n'
    '        2.32,\n'
    '        2.41,\n'
    '        2.29,\n'
    '        2.25,\n'
    '        2.44,\n'
    '        2.36,\n'
    '        2.18,\n'
    '        2.26,\n'
    '        2.37,\n'
    '        2.69,\n'
    '        2.64,\n'
    '        2.44,\n'
    '        2.26,\n'
    '        2.2,\n'
    '        1.77,\n'
    '        1.79,\n'
    '        1.47,\n'
    '        1.41,\n'
    '        1.09,\n'
    '        1.36,\n'
    '        1.46,\n'
    '        1.63,\n'
    '        1.57,\n'
    '        1.68,\n'
    '        1.85,\n'
    '        2.75,\n'
    '        2.89,\n'
    '        2.17,\n'
    '        1.91,\n'
    '        1.75,\n'
    '        1.71,\n'
    '        1.57,\n'
    '        1.72,\n'
    '        1.86,\n'
    '        1.82,\n'
    '        1.77,\n'
    '        1.64,\n'
    '        1.48,\n'
    '        1.28,\n'
    '        1.36,\n'
    '        1.37,\n'
    '        1.42,\n'
    '        1.51,\n'
    '        1.5,\n'
    '        1.31,\n'
    '        1.26,\n'
    '        1.24,\n'
    '        1.02,\n'
    '        0.91,\n'
    '        0.53,\n'
    '        0.67,\n'
    '        1.04,\n'
    '        1.06,\n'
    '        1.24,\n'
    '        0.96,\n'
    '        0.86,\n'
    '        0.78,\n'
    '        0.76,\n'
    '        0.62,\n'
    '        0.14,\n'
    '        0.08,\n'
    '        0.19,\n'
    '        0.0,\n'
    '        -0.03,\n'
    '        -0.11,\n'
    '        -0.25,\n'
    '        -0.14,\n'
    '        -0.21,\n'
    '        -0.34,\n'
    '        -0.5,\n'
    '        -0.6,\n'
    '        -0.59,\n'
    '        -0.71,\n'
    '        -0.75,\n'
    '        -0.77,\n'
    '        -0.76,\n'
    '        -0.61,\n'
    '        -0.57,\n'
    '        -0.59,\n'
    '        -0.65,\n'
    '        -0.36,\n'
    '        0.25,\n'
    '        0.46,\n'
    '        0.55,\n'
    '        0.66,\n'
    '        0.43,\n'
    '        0.55,\n'
    '        0.74,\n'
    '        0.63,\n'
    '        0.55,\n'
    '        0.56,\n'
    '        0.54,\n'
    '        0.37,\n'
    '        0.37,\n'
    '        0.28,\n'
    '        0.22,\n'
    '        0.46,\n'
    '        0.38,\n'
    '        0.45,\n'
    '        0.51,\n'
    '        0.27,\n'
    '        0.26,\n'
    '        0.28,\n'
    '        0.08,\n'
    '        0.33,\n'
    '        0.5,\n'
    '        0.5,\n'
    '        0.56,\n'
    '        0.65,\n'
    '        0.57,\n'
    '        0.69,\n'
    '        0.73,\n'
    '        0.67,\n'
    '        0.47,\n'
    '        0.34,\n'
    '        0.19,\n'
    '        0.21,\n'
    '        0.17,\n'
    '        0.04,\n'
    '        0.09,\n'
    '        0.12,\n'
    '        0.1,\n'
    '        0.32,\n'
    '        0.56,\n'
    '        0.42,\n'
    '        0.4,\n'
    '        0.49,\n'
    '        0.39,\n'
    '        0.47,\n'
    '        0.46,\n'
    '        0.55,\n'
    '        0.43,\n'
    '        0.37,\n'
    '        0.5,\n'
    '        0.5,\n'
    '        0.5,\n'
    '        0.54,\n'
    '        0.76,\n'
    '        0.75,\n'
    '        0.74,\n'
    '        0.84,\n'
    '        0.79,\n'
    '        0.77,\n'
    '        0.79,\n'
    '        0.88,\n'
    '        1.04,\n'
    '        1.11,\n'
    '        1.02,\n'
    '        0.92,\n'
    '        0.8,\n'
    '        0.66,\n'
    '        0.6,\n'
    '        0.57,\n'
    '        0.37,\n'
    '        0.31,\n'
    '        0.04,\n'
    '        0.11,\n'
    '        0.15,\n'
    '        0.17,\n'
    '        0.14,\n'
    '        0.04,\n'
    '        -0.11,\n'
    '        -0.12,\n'
    '        -0.45,\n'
    '        -0.44,\n'
    '        -0.54,\n'
    '        -0.83,\n'
    '        -1.01,\n'
    '        -0.98,\n'
    '        -0.92,\n'
    '        -0.84,\n'
    '        -0.98,\n'
    '        -1.0,\n'
    '        -0.92,\n'
    '        -0.66,\n'
    '        -0.71,\n'
    '        -0.85,\n'
    '        -0.82,\n'
    '        -1.01,\n'
    '        -1.07,\n'
    '        -0.97,\n'
    '        -0.95,\n'
    '        -1.06,\n'
    '        -0.99,\n'
    '        -0.69,\n'
    '        -0.52,\n'
    '        -0.72,\n'
    '        -0.14,\n'
    '        0.21,\n'
    '        0.53,\n'
    '        0.53,\n'
    '        0.39,\n'
    '        1.14,\n'
    '        1.59,\n'
    '        1.52,\n'
    '        1.36,\n'
    '        1.29,\n'
    '        1.41,\n'
    '        1.36,\n'
    '        1.19,\n'
    '        1.36,\n'
    '        1.55,\n'
    '        1.6,\n'
    '        1.83,\n'
    '        2.04,\n'
    '        2.41,\n'
    '        2.2,\n'
    '        1.84,\n'
    '        1.79,\n'
    '        1.93,\n'
    '        1.9,\n'
    '        2.15,\n'
    '        2.15,\n'
    '        2.05,\n'
    '        1.97,\n'
    '        1.76,\n'
    '        1.62,\n'
    '        1.81,\n'
    '        2.03,\n'
    '        2.09,\n'
    '        2.23,\n'
    '        2.03,\n'
    '        1.95,\n'
    '        2.04,\n'
    '        2.11,\n'
    '        2.09,\n'
    '        2.01,\n'
    '        1.88,\n'
    '        1.75,\n'
    '        1.76,\n'
    '        1.83,\n'
    '        1.9,\n'
    '        1.91,\n'
    '        1.82,\n'
    '        1.91,\n'
    '        1.94,\n'
    '    ]\n'
    '    real_yield_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"real{i}", "Raw Value": value}\n'
    '            for i, value in enumerate(real_yield_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    real_yield_out = calculate_real_yield_outputs(real_yield_raw)\n'
    '    real_yield_rows = df_records(real_yield_out)\n'
    '    assert (\n'
    '        real_yield_out.shape[1] == 36\n'
    '        and list(real_yield_out.columns) == expected_real_yield_columns\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_yield_rows[2]["3M MA Real Yield"], (2.29 + 1.99 + 1.94) / 3\n'
    '    )\n'
    '    assert (\n'
    '        real_yield_rows[0]["Sample Count"] == 0\n'
    '        and real_yield_rows[2]["Sample Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_yield_rows[3]["1M Real Yield Change"],\n'
    '        real_yield_rows[3]["3M MA Real Yield"] - real_yield_rows[2]["3M MA Real Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_yield_rows[5]["3M Real Yield Change"],\n'
    '        real_yield_rows[5]["3M MA Real Yield"] - real_yield_rows[2]["3M MA Real Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_yield_rows[8]["6M Real Yield Change"],\n'
    '        real_yield_rows[8]["3M MA Real Yield"] - real_yield_rows[2]["3M MA Real Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_yield_rows[14]["12M Real Yield Change"],\n'
    '        real_yield_rows[14]["3M MA Real Yield"]\n'
    '        - real_yield_rows[2]["3M MA Real Yield"],\n'
    '    )\n'
    '\n'
    '    # Each rank has its own 60-observation gate and ranks its specified metric.\n'
    '    assert (\n'
    '        real_yield_rows[60]["Real Yield Level Percentile"] == ""\n'
    '        and real_yield_rows[61]["Real Yield Level Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        real_yield_rows[63]["3M Change Percentile"] == ""\n'
    '        and real_yield_rows[64]["3M Change Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        real_yield_rows[66]["6M Change Percentile"] == ""\n'
    '        and real_yield_rows[67]["6M Change Percentile"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        real_yield_rows[72]["12M Change Percentile"] == ""\n'
    '        and real_yield_rows[73]["12M Change Percentile"] != ""\n'
    '    )\n'
    '    for row_index, metric_col, rank_col in (\n'
    '        (65, "3M MA Real Yield", "Real Yield Level Percentile"),\n'
    '        (65, "3M Real Yield Change", "3M Change Percentile"),\n'
    '        (73, "6M Real Yield Change", "6M Change Percentile"),\n'
    '        (73, "12M Real Yield Change", "12M Change Percentile"),\n'
    '    ):\n'
    '        metric_history = [\n'
    '            row[metric_col]\n'
    '            for row in real_yield_rows[: row_index + 1]\n'
    '            if not is_blank(row[metric_col])\n'
    '        ]\n'
    '        expected_rank = _real_yield_sheets_round(\n'
    '            _real_yield_percentrank_inc_sheets(\n'
    '                metric_history, real_yield_rows[row_index][metric_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert real_yield_rows[row_index][rank_col] == expected_rank\n'
    '    raw_level_rank = _real_yield_sheets_round(\n'
    '        _real_yield_percentrank_inc_sheets(\n'
    '            real_yield_values[:66], real_yield_values[65]\n'
    '        ),\n'
    '        3,\n'
    '    )\n'
    '    assert real_yield_rows[65]["Real Yield Level Percentile"] != raw_level_rank\n'
    '\n'
    '    # Supplied A23 spreadsheet regression rows.\n'
    '    assert real_yield_rows[64]["Real Yield Level Percentile"] == 0.016\n'
    '    assert real_yield_rows[64]["3M Change Percentile"] == 0.186\n'
    '    assert real_yield_rows[65]["Real Yield Level Percentile"] == 0.048\n'
    '    assert real_yield_rows[65]["Real Yield Level Score"] == -9.04\n'
    '    assert real_yield_rows[65]["3M Change Percentile"] == 0.783\n'
    '    assert real_yield_rows[65]["3M Change Score"] == 5.66\n'
    '    assert real_yield_rows[66]["Real Yield Level Percentile"] == 0.063\n'
    '    assert real_yield_rows[66]["Real Yield Level Score"] == -8.74\n'
    '    assert real_yield_rows[67]["6M Change Percentile"] == 0.576\n'
    '    assert real_yield_rows[67]["6M Change Score"] == 1.52\n'
    '    for row in real_yield_rows:\n'
    '        for percentile_col, score_col in (\n'
    '            ("Real Yield Level Percentile", "Real Yield Level Score"),\n'
    '            ("3M Change Percentile", "3M Change Score"),\n'
    '            ("6M Change Percentile", "6M Change Score"),\n'
    '            ("12M Change Percentile", "12M Change Score"),\n'
    '        ):\n'
    '            percentile = row[percentile_col]\n'
    '            if not is_blank(percentile):\n'
    '                assert percentile == _real_yield_sheets_round(percentile, 3)\n'
    '                assert row[score_col] == _real_yield_sheets_round(\n'
    '                    percentile * 20 - 10, 2\n'
    '                )\n'
    '\n'
    '    assert (\n'
    '        real_yield_rows[2]["Real Yield Regime"] == "High and rising real-yield regime"\n'
    '    )\n'
    '    assert (\n'
    '        real_yield_rows[3]["Real Yield Regime"] == "High and rising real-yield regime"\n'
    '    )\n'
    '    assert (\n'
    '        real_yield_rows[4]["Real Yield Regime"] == "High and rising real-yield regime"\n'
    '    )\n'
    '    assert _real_yield_level_regime(2.0, "") == "High and rising real-yield regime"\n'
    '    assert _real_yield_level_regime(0.0, 0.0) == "Positive real yields and rising"\n'
    '    assert real_yield_rows[203]["3M Real Yield Change"] == 0.0\n'
    '    assert (\n'
    '        real_yield_rows[203]["Real Yield Regime"] == "Positive real yields and rising"\n'
    '    )\n'
    '\n'
    '    feb_2009 = real_yield_rows[73]\n'
    '    apr_2009 = real_yield_rows[75]\n'
    '    nov_2009 = real_yield_rows[82]\n'
    '    assert [\n'
    '        feb_2009[c]\n'
    '        for c in (\n'
    '            "Real Yield Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [-1.26, -9.42, 6.3, 5.94]\n'
    '    assert [\n'
    '        apr_2009[c]\n'
    '        for c in (\n'
    '            "Real Yield Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [-7.54, -9.72, -7.32, 6.06]\n'
    '    assert [\n'
    '        nov_2009[c]\n'
    '        for c in (\n'
    '            "Real Yield Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [-9.24, -7.66, -5.4, -10.0]\n'
    '    for row, expected_t, expected_y in (\n'
    '        (feb_2009, -1.21, -1.85),\n'
    '        (apr_2009, -6.4, -6.8),\n'
    '        (nov_2009, -7.3, -8.27),\n'
    '    ):\n'
    '        j, l, n, p = (\n'
    '            row[c]\n'
    '            for c in (\n'
    '                "Real Yield Level Score",\n'
    '                "3M Change Score",\n'
    '                "6M Change Score",\n'
    '                "12M Change Score",\n'
    '            )\n'
    '        )\n'
    '        assert (\n'
    '            row["Real Yield Impulse Score"]\n'
    '            == _real_yield_sheets_round(\n'
    '                clamp(0.05 * j + 0.45 * l + 0.35 * n + 0.15 * p, -10, 10), 2\n'
    '            )\n'
    '            == expected_t\n'
    '        )\n'
    '        assert (\n'
    '            row["Coincident Score"]\n'
    '            == _real_yield_sheets_round(\n'
    '                clamp(0.45 * j + 0.30 * l + 0.15 * n + 0.10 * p, -10, 10), 2\n'
    '            )\n'
    '            == expected_y\n'
    '        )\n'
    '    assert nov_2009["Leading Score"] == -7.46\n'
    '    assert nov_2009["Warning Score"] == -1.8\n'
    '\n'
    '    aug_2021 = real_yield_rows[223]\n'
    '    sep_2021 = real_yield_rows[224]\n'
    '    assert aug_2021["6M Real Yield Change"] == 0.0\n'
    '    assert aug_2021["Real Yield Regime"] == "Negative and easing real-yield regime"\n'
    '    assert aug_2021["Real Yield Cycle Turning Point Score"] == -6\n'
    '    assert aug_2021["Policy Restriction Score"] == -6\n'
    '    assert aug_2021["Leading Score"] == -3.35\n'
    '    for row in (aug_2021, sep_2021):\n'
    '        u, v, w, x = (\n'
    '            row[c]\n'
    '            for c in (\n'
    '                "Real Yield Cycle Turning Point Score",\n'
    '                "Real Yield Shock Score",\n'
    '                "Policy Restriction Score",\n'
    '                "Real Yield Extreme Score",\n'
    '            )\n'
    '        )\n'
    '        assert (u, v, w, x) == (-6, 0, -6, 0)\n'
    '        assert (\n'
    '            row["Warning Score"]\n'
    '            == _real_yield_sheets_round(\n'
    '                clamp(0.30 * u + 0.25 * v + 0.30 * w + 0.15 * x, -10, 10), 2\n'
    '            )\n'
    '            == -3.6\n'
    '        )\n'
    '        assert row["Warning Flag"] == "Moderate real-yield easing warning risk"\n'
    '    assert _real_yield_warning_label(-6) == "High real-yield easing warning risk"\n'
    '    assert _real_yield_warning_label(-3.6) == "Moderate real-yield easing warning risk"\n'
    '    assert _real_yield_warning_label(6) == "High real-yield tightening warning risk"\n'
    '    assert (\n'
    '        _real_yield_warning_label(3.6) == "Moderate real-yield tightening warning risk"\n'
    '    )\n'
    '\n'
    '    # Z-score and extreme-label behavior remains on the preserved formulas.\n'
    '    assert [\n'
    '        nov_2009[c]\n'
    '        for c in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [-1.58, -1.07, -0.4, -2.07]\n'
    '    assert nov_2009["Extreme Flag"] == "Strong downside real-yield pressure"\n'
    '    assert [\n'
    '        aug_2021[c]\n'
    '        for c in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [-1.92, -0.68, 0.19, -0.01]\n'
    '    assert aug_2021["Extreme Flag"] == "Normal range"\n'
    '    assert _real_yield_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _real_yield_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside real-yield pressure"\n'
    '    )\n'
    '\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped real yield sheet", CONTROL_PANEL_TWENTY_FIRST_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_real_yield_outputs\n'
    '    )\n'
    '    all_specs_with_a23 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 23)] + [["Real Yield"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Real Yield", "A23") in all_specs_with_a23\n'
    '    expected_two_year_columns = [\n'
    '        "3M MA Yield",\n'
    '        "Sample Count",\n'
    '        "1M Yield Change",\n'
    '        "3M Yield Change",\n'
    '        "6M Yield Change",\n'
    '        "12M Yield Change",\n'
    '        "Yield Level Percentile",\n'
    '        "Yield Level Score",\n'
    '        "3M Change Percentile",\n'
    '        "3M Change Score",\n'
    '        "6M Change Percentile",\n'
    '        "6M Change Score",\n'
    '        "12M Change Percentile",\n'
    '        "12M Change Score",\n'
    '        "2Y Yield Regime",\n'
    '        "2Y Yield Policy Bias",\n'
    '        "1M 2Y Yield Shock Flag",\n'
    '        "2Y Yield Impulse Score",\n'
    '        "2Y Yield Cycle Turning Point Score",\n'
    '        "2Y Yield Shock Score",\n'
    '        "Policy Rate Restriction Score",\n'
    '        "2Y Yield Extreme Score",\n'
    '        "Coincident Score",\n'
    '        "Leading Score",\n'
    '        "Warning Score",\n'
    '        "State",\n'
    '        "Regime",\n'
    '        "Signal Comment",\n'
    '        "Reliability",\n'
    '        "Signal Validity",\n'
    '        "Warning Flag",\n'
    '        "Level Z-Score",\n'
    '        "3M Change Z-Score",\n'
    '        "6M Change Z-Score",\n'
    '        "12M Change Z-Score",\n'
    '        "Extreme Flag",\n'
    '    ]\n'
    '    assert TWO_YEAR_YIELD_OUTPUT_COLUMNS == expected_two_year_columns\n'
    '    two_year_spec = select_indicator_engine_spec(\n'
    '        "Unmapped 2Y yield sheet", CONTROL_PANEL_TWENTY_SECOND_INDICATOR_CELL\n'
    '    )\n'
    '    assert two_year_spec.calculate is calculate_two_year_yield_outputs\n'
    '    assert (two_year_spec.input_start_col, two_year_spec.input_end_col) == ("A", "B")\n'
    '    assert (two_year_spec.output_start_col, two_year_spec.output_end_col) == ("C", "AL")\n'
    '    assert list(two_year_spec.output_columns) == expected_two_year_columns\n'
    '    assert (\n'
    '        indicator_input_range(\n'
    '            "2Y government bond yield",\n'
    '            two_year_spec.input_end_col,\n'
    '            two_year_spec.input_start_col,\n'
    '        )\n'
    '        == "\'2Y government bond yield\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range(\n'
    '            "2Y government bond yield",\n'
    '            600,\n'
    '            two_year_spec.output_end_col,\n'
    '            two_year_spec.output_start_col,\n'
    '        )\n'
    '        == "\'2Y government bond yield\'!C2:AL600"\n'
    '    )\n'
    '    assert output_end_col_for_columns(TWO_YEAR_YIELD_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        select_indicator_engine("A23 remains A23", "A23")\n'
    '        is calculate_real_yield_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A25 remains A25", "A25")\n'
    '        is calculate_ten_year_yield_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A39 remains A39", "A39")\n'
    '        is calculate_corporate_credit_spreads_outputs\n'
    '    )\n'
    '\n'
    '    assert _two_year_yield_sheets_round(2.675, 2) == 2.68 and round(2.675, 2) != 2.68\n'
    '    assert _two_year_yield_sheets_round(-2.675, 2) == -2.68\n'
    '    assert _two_year_yield_sheets_round(0.0625, 3) == 0.063\n'
    '    assert _two_year_yield_percentrank_inc_sheets([1, 2, 2, 4], 2) == 1 / 3\n'
    '    assert values_equivalent(\n'
    '        _two_year_yield_percentrank_inc_sheets([1, 2, 2, 4], 3), 2.5 / 3\n'
    '    )\n'
    '\n'
    '    # Static A24 source fixture through the 1999 duplicate/tie regression rows.\n'
    '    two_year_values = [\n'
    '        7.06,\n'
    '        6.85,\n'
    '        6.63,\n'
    '        6.42,\n'
    '        5.98,\n'
    '        5.81,\n'
    '        5.38,\n'
    '        5.9,\n'
    '        6.09,\n'
    '        6.09,\n'
    '        5.96,\n'
    '        6.25,\n'
    '        6.13,\n'
    '        6.27,\n'
    '        6.61,\n'
    '        6.71,\n'
    '        7.11,\n'
    '        7.14,\n'
    '        7.18,\n'
    '        7.49,\n'
    '        7.57,\n'
    '        7.58,\n'
    '        7.74,\n'
    '        8.01,\n'
    '        8.24,\n'
    '        8.49,\n'
    '        8.37,\n'
    '        8.57,\n'
    '        8.85,\n'
    '        9.42,\n'
    '        9.72,\n'
    '        9.86,\n'
    '        9.72,\n'
    '        9.79,\n'
    '        9.78,\n'
    '        9.78,\n'
    '        9.22,\n'
    '        9.14,\n'
    '        9.46,\n'
    '        10.06,\n'
    '        11.49,\n'
    '        11.81,\n'
    '        11.39,\n'
    '        11.5,\n'
    '        13.42,\n'
    '        14.88,\n'
    '        12.5,\n'
    '        9.45,\n'
    '        8.73,\n'
    '        9.03,\n'
    '        10.53,\n'
    '        11.57,\n'
    '        12.09,\n'
    '        13.51,\n'
    '        14.08,\n'
    '        13.26,\n'
    '        13.92,\n'
    '        13.57,\n'
    '        14.15,\n'
    '        15.46,\n'
    '        14.51,\n'
    '        15.35,\n'
    '        16.28,\n'
    '        16.46,\n'
    '        15.54,\n'
    '        12.88,\n'
    '        13.29,\n'
    '        14.57,\n'
    '        14.82,\n'
    '        14.19,\n'
    '        14.2,\n'
    '        13.78,\n'
    '        14.47,\n'
    '        13.8,\n'
    '        12.32,\n'
    '        11.78,\n'
    '        10.19,\n'
    '        9.8,\n'
    '        9.66,\n'
    '        9.33,\n'
    '        9.64,\n'
    '        9.66,\n'
    '        9.57,\n'
    '        9.49,\n'
    '        10.18,\n'
    '        10.69,\n'
    '        11.07,\n'
    '        10.79,\n'
    '        10.57,\n'
    '        10.66,\n'
    '        10.84,\n'
    '        10.64,\n'
    '        10.79,\n'
    '        11.31,\n'
    '        11.69,\n'
    '        12.47,\n'
    '        12.91,\n'
    '        12.88,\n'
    '        12.43,\n'
    '        12.2,\n'
    '        11.6,\n'
    '        10.65,\n'
    '        10.18,\n'
    '        9.93,\n'
    '        10.17,\n'
    '        10.71,\n'
    '        10.09,\n'
    '        9.39,\n'
    '        8.69,\n'
    '        8.77,\n'
    '        8.94,\n'
    '        8.98,\n'
    '        8.86,\n'
    '        8.58,\n'
    '        8.15,\n'
    '        8.14,\n'
    '        7.97,\n'
    '        7.21,\n'
    '        6.7,\n'
    '        7.07,\n'
    '        7.18,\n'
    '        6.67,\n'
    '        6.33,\n'
    '        6.35,\n'
    '        6.28,\n'
    '        6.28,\n'
    '        6.27,\n'
    '        6.23,\n'
    '        6.4,\n'
    '        6.42,\n'
    '        7.02,\n'
    '        7.76,\n'
    '        7.57,\n'
    '        7.44,\n'
    '        7.75,\n'
    '        8.34,\n'
    '        8.4,\n'
    '        7.69,\n'
    '        7.86,\n'
    '        7.63,\n'
    '        7.18,\n'
    '        7.27,\n'
    '        7.59,\n'
    '        8.0,\n'
    '        8.03,\n'
    '        8.28,\n'
    '        8.63,\n'
    '        8.46,\n'
    '        8.35,\n'
    '        8.67,\n'
    '        9.09,\n'
    '        9.18,\n'
    '        9.37,\n'
    '        9.68,\n'
    '        9.45,\n'
    '        9.02,\n'
    '        8.41,\n'
    '        7.82,\n'
    '        8.14,\n'
    '        8.28,\n'
    '        7.98,\n'
    '        7.8,\n'
    '        7.78,\n'
    '        8.09,\n'
    '        8.37,\n'
    '        8.63,\n'
    '        8.72,\n'
    '        8.64,\n'
    '        8.35,\n'
    '        8.16,\n'
    '        8.06,\n'
    '        8.08,\n'
    '        7.88,\n'
    '        7.6,\n'
    '        7.31,\n'
    '        7.13,\n'
    '        6.87,\n'
    '        7.1,\n'
    '        6.95,\n'
    '        6.78,\n'
    '        6.96,\n'
    '        6.92,\n'
    '        6.43,\n'
    '        6.18,\n'
    '        5.91,\n'
    '        5.56,\n'
    '        5.03,\n'
    '        4.96,\n'
    '        5.21,\n'
    '        5.69,\n'
    '        5.34,\n'
    '        5.23,\n'
    '        5.05,\n'
    '        4.36,\n'
    '        4.19,\n'
    '        3.89,\n'
    '        4.08,\n'
    '        4.58,\n'
    '        4.67,\n'
    '        4.39,\n'
    '        4.1,\n'
    '        3.95,\n'
    '        3.84,\n'
    '        3.98,\n'
    '        4.16,\n'
    '        4.07,\n'
    '        4.0,\n'
    '        3.85,\n'
    '        3.87,\n'
    '        4.16,\n'
    '        4.21,\n'
    '        4.14,\n'
    '        4.47,\n'
    '        5.0,\n'
    '        5.55,\n'
    '        5.97,\n'
    '        5.93,\n'
    '        6.13,\n'
    '        6.18,\n'
    '        6.39,\n'
    '        6.73,\n'
    '        7.15,\n'
    '        7.59,\n'
    '        7.51,\n'
    '        7.11,\n'
    '        6.78,\n'
    '        6.57,\n'
    '        6.17,\n'
    '        5.72,\n'
    '        5.78,\n'
    '        5.98,\n'
    '        5.81,\n'
    '        5.7,\n'
    '        5.48,\n'
    '        5.32,\n'
    '        5.11,\n'
    '        5.03,\n'
    '        5.66,\n'
    '        5.96,\n'
    '        6.1,\n'
    '        6.3,\n'
    '        6.27,\n'
    '        6.03,\n'
    '        6.23,\n'
    '        5.91,\n'
    '        5.7,\n'
    '        5.78,\n'
    '        6.01,\n'
    '        5.9,\n'
    '        6.22,\n'
    '        6.45,\n'
    '        6.28,\n'
    '        6.09,\n'
    '        5.89,\n'
    '        5.94,\n'
    '        5.88,\n'
    '        5.77,\n'
    '        5.71,\n'
    '        5.72,\n'
    '        5.36,\n'
    '        5.42,\n'
    '        5.56,\n'
    '        5.56,\n'
    '        5.59,\n'
    '        5.52,\n'
    '        5.46,\n'
    '        5.27,\n'
    '        4.67,\n'
    '        4.09,\n'
    '        4.54,\n'
    '        4.51,\n'
    '        4.62,\n'
    '        4.88,\n'
    '        5.05,\n'
    '        4.98,\n'
    '        5.25,\n'
    '        5.62,\n'
    '    ]\n'
    '    two_year_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"2y{i}", "Raw Value": value}\n'
    '            for i, value in enumerate(two_year_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    two_year_out = calculate_two_year_yield_outputs(two_year_raw)\n'
    '    two_year_rows = df_records(two_year_out)\n'
    '    assert (\n'
    '        two_year_out.shape[1] == 36\n'
    '        and list(two_year_out.columns) == expected_two_year_columns\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        two_year_rows[2]["3M MA Yield"], sum(two_year_values[:3]) / 3\n'
    '    )\n'
    '    assert (\n'
    '        two_year_rows[0]["Sample Count"] == 0 and two_year_rows[2]["Sample Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        two_year_rows[3]["1M Yield Change"],\n'
    '        two_year_rows[3]["3M MA Yield"] - two_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        two_year_rows[5]["3M Yield Change"],\n'
    '        two_year_rows[5]["3M MA Yield"] - two_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        two_year_rows[8]["6M Yield Change"],\n'
    '        two_year_rows[8]["3M MA Yield"] - two_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        two_year_rows[14]["12M Yield Change"],\n'
    '        two_year_rows[14]["3M MA Yield"] - two_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '\n'
    '    rank_columns = (\n'
    '        "Yield Level Percentile",\n'
    '        "3M Change Percentile",\n'
    '        "6M Change Percentile",\n'
    '        "12M Change Percentile",\n'
    '    )\n'
    '    assert two_year_rows[60]["Sample Count"] == 59 and all(\n'
    '        two_year_rows[60][col] == "" for col in rank_columns\n'
    '    )\n'
    '    assert two_year_rows[61]["Sample Count"] == 60 and all(\n'
    '        two_year_rows[61][col] != "" for col in rank_columns\n'
    '    )\n'
    '    assert sum(not is_blank(row["3M Yield Change"]) for row in two_year_rows[:62]) == 57\n'
    '    assert sum(not is_blank(row["6M Yield Change"]) for row in two_year_rows[:62]) == 54\n'
    '    assert (\n'
    '        sum(not is_blank(row["12M Yield Change"]) for row in two_year_rows[:62]) == 48\n'
    '    )\n'
    '    for row_index, metric_col, rank_col in (\n'
    '        (61, "3M MA Yield", "Yield Level Percentile"),\n'
    '        (61, "3M Yield Change", "3M Change Percentile"),\n'
    '        (61, "6M Yield Change", "6M Change Percentile"),\n'
    '        (61, "12M Yield Change", "12M Change Percentile"),\n'
    '    ):\n'
    '        metric_history = [\n'
    '            row[metric_col]\n'
    '            for row in two_year_rows[: row_index + 1]\n'
    '            if not is_blank(row[metric_col])\n'
    '        ]\n'
    '        expected_rank = _two_year_yield_sheets_round(\n'
    '            _two_year_yield_percentrank_inc_sheets(\n'
    '                metric_history, two_year_rows[row_index][metric_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert values_equivalent(two_year_rows[row_index][rank_col], expected_rank)\n'
    '    raw_level_rank = _two_year_yield_sheets_round(\n'
    '        _two_year_yield_percentrank_inc_sheets(\n'
    '            two_year_values[:62], two_year_values[61]\n'
    '        ),\n'
    '        3,\n'
    '    )\n'
    '    assert not values_equivalent(\n'
    '        two_year_rows[61]["Yield Level Percentile"], raw_level_rank\n'
    '    )\n'
    '\n'
    '    jul_1981, aug_1981, sep_1981 = (\n'
    '        two_year_rows[61],\n'
    '        two_year_rows[62],\n'
    '        two_year_rows[63],\n'
    '    )\n'
    '    oct_1981, nov_1981, dec_1981 = (\n'
    '        two_year_rows[64],\n'
    '        two_year_rows[65],\n'
    '        two_year_rows[66],\n'
    '    )\n'
    '    assert [jul_1981[c] for c in rank_columns] == [\n'
    '        "100.00%",\n'
    '        "82.10%",\n'
    '        "79.20%",\n'
    '        "100.00%",\n'
    '    ]\n'
    '    assert [\n'
    '        jul_1981[c]\n'
    '        for c in (\n'
    '            "Yield Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [10.0, 6.42, 5.84, 10.0]\n'
    '    assert [\n'
    '        aug_1981[c]\n'
    '        for c in (\n'
    '            "3M Change Percentile",\n'
    '            "3M Change Score",\n'
    '            "6M Change Percentile",\n'
    '            "6M Change Score",\n'
    '            "12M Change Percentile",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == ["77.20%", 5.44, "79.60%", 5.92, "97.90%", 9.58]\n'
    '    assert [\n'
    '        nov_1981[c]\n'
    '        for c in (\n'
    '            "Yield Level Percentile",\n'
    '            "Yield Level Score",\n'
    '            "3M Change Percentile",\n'
    '            "3M Change Score",\n'
    '            "6M Change Percentile",\n'
    '            "6M Change Score",\n'
    '            "12M Change Percentile",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == ["93.70%", 8.74, "11.70%", -7.66, "31.60%", -3.68, "86.30%", 7.26]\n'
    '    assert [\n'
    '        dec_1981[c]\n'
    '        for c in (\n'
    '            "Yield Level Percentile",\n'
    '            "Yield Level Score",\n'
    '            "3M Change Percentile",\n'
    '            "3M Change Score",\n'
    '            "6M Change Percentile",\n'
    '            "6M Change Score",\n'
    '            "12M Change Percentile",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == ["89.10%", 7.82, "4.90%", -9.02, "10.30%", -7.94, "15.40%", -6.92]\n'
    '    for row in two_year_rows:\n'
    '        for percentile_col, score_col in (\n'
    '            ("Yield Level Percentile", "Yield Level Score"),\n'
    '            ("3M Change Percentile", "3M Change Score"),\n'
    '            ("6M Change Percentile", "6M Change Score"),\n'
    '            ("12M Change Percentile", "12M Change Score"),\n'
    '        ):\n'
    '            percentile = to_float_or_none(row[percentile_col])\n'
    '            if percentile is not None:\n'
    '                assert row[percentile_col] == _two_year_yield_percent_display(\n'
    '                    percentile\n'
    '                )\n'
    '                assert row[score_col] == _two_year_yield_sheets_round(\n'
    '                    percentile * 20 - 10, 2\n'
    '                )\n'
    '\n'
    '    assert two_year_rows[2]["2Y Yield Regime"] == "High and rising 2Y yield regime"\n'
    '    assert two_year_rows[3]["2Y Yield Regime"] == "High and rising 2Y yield regime"\n'
    '    assert two_year_rows[4]["2Y Yield Regime"] == "High and rising 2Y yield regime"\n'
    '    assert _two_year_yield_level_regime(5, "") == "High and rising 2Y yield regime"\n'
    '    assert _two_year_yield_level_regime(4, 0) == "Elevated and rising 2Y yield regime"\n'
    '    assert (\n'
    '        _two_year_yield_level_regime(4, -0.1) == "Elevated but cooling 2Y yield regime"\n'
    '    )\n'
    '    assert _two_year_yield_level_regime(3, 0) == "Moderate and rising 2Y yield regime"\n'
    '    assert _two_year_yield_level_regime(2.9, 0) == "Low but rising 2Y yield regime"\n'
    '    assert two_year_rows[6]["2Y Yield Regime"] == "High but cooling 2Y yield regime"\n'
    '    assert _two_year_yield_shock_label(0.30) == "No 1M 2Y yield shock"\n'
    '    assert _two_year_yield_shock_label(-0.30) == "1M 2Y yield downside shock"\n'
    '    assert _two_year_yield_shock_label(0.3001) == "1M 2Y yield upside shock"\n'
    '    assert _two_year_yield_shock_label(-0.3001) == "1M 2Y yield downside shock"\n'
    '    assert (\n'
    '        _two_year_yield_shock_score(0.30) == 4\n'
    '        and _two_year_yield_shock_score(-0.30) == -6\n'
    '    )\n'
    '    assert (\n'
    '        _two_year_yield_shock_score(0.3001) == 6\n'
    '        and _two_year_yield_shock_score(-0.3001) == -6\n'
    '    )\n'
    '    march_1982 = two_year_rows[69]\n'
    '    assert values_equivalent(march_1982["1M Yield Change"], 0.30)\n'
    '    assert march_1982["1M 2Y Yield Shock Flag"] == "No 1M 2Y yield shock"\n'
    '    assert march_1982["2Y Yield Shock Score"] == 4\n'
    '    assert march_1982["Warning Score"] == 1.03\n'
    '\n'
    '    for row, expected_t, expected_y, expected_z in (\n'
    '        (jul_1981, 6.93, 7.63, 7.20),\n'
    '        (aug_1981, 6.46, 7.24, 6.77),\n'
    '        (sep_1981, 7.47, 8.03, 7.61),\n'
    '        (dec_1981, -7.49, -4.15, -6.40),\n'
    '    ):\n'
    '        j, l, n, p = (\n'
    '            row[c]\n'
    '            for c in (\n'
    '                "Yield Level Score",\n'
    '                "3M Change Score",\n'
    '                "6M Change Score",\n'
    '                "12M Change Score",\n'
    '            )\n'
    '        )\n'
    '        t, u = row["2Y Yield Impulse Score"], row["2Y Yield Cycle Turning Point Score"]\n'
    '        assert (\n'
    '            t\n'
    '            == _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.05, j), (0.45, l), (0.35, n), (0.15, p)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            == expected_t\n'
    '        )\n'
    '        assert (\n'
    '            row["Coincident Score"]\n'
    '            == _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.25, j), (0.30, l), (0.20, n), (0.10, p), (0.15, t)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            == expected_y\n'
    '        )\n'
    '        assert (\n'
    '            row["Leading Score"]\n'
    '            == _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.10, j), (0.35, l), (0.25, n), (0.15, p), (0.10, t), (0.05, u)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            == expected_z\n'
    '        )\n'
    '\n'
    '    assert (\n'
    '        jul_1981["2Y Yield Extreme Score"] == 0.64\n'
    '        and jul_1981["3M Change Z-Score"] == ""\n'
    '    )\n'
    '    assert (\n'
    '        aug_1981["2Y Yield Extreme Score"] == 0.44\n'
    '        and aug_1981["3M Change Z-Score"] == ""\n'
    '    )\n'
    '    assert (\n'
    '        sep_1981["2Y Yield Extreme Score"] == 0.71\n'
    '        and sep_1981["3M Change Z-Score"] == ""\n'
    '    )\n'
    '    assert oct_1981["2Y Yield Extreme Score"] == oct_1981["3M Change Z-Score"] == 0.43\n'
    '    assert nov_1981["2Y Yield Extreme Score"] == nov_1981["3M Change Z-Score"] == -0.71\n'
    '    internal_f_zscores = expanding_zscore_sample(\n'
    '        [row["3M Yield Change"] for row in two_year_rows],\n'
    '        [row["Sample Count"] for row in two_year_rows],\n'
    '        60,\n'
    '    )\n'
    '    for idx in (61, 62, 63, 64, 65, 66):\n'
    '        assert two_year_rows[idx][\n'
    '            "2Y Yield Extreme Score"\n'
    '        ] == _two_year_yield_sheets_round(internal_f_zscores[idx], 2)\n'
    '\n'
    '    for row, expected_aa in ((jul_1981, 5.20), (aug_1981, 4.67), (dec_1981, -3.30)):\n'
    '        u, v, w, x = (\n'
    '            row[c]\n'
    '            for c in (\n'
    '                "2Y Yield Cycle Turning Point Score",\n'
    '                "2Y Yield Shock Score",\n'
    '                "Policy Rate Restriction Score",\n'
    '                "2Y Yield Extreme Score",\n'
    '            )\n'
    '        )\n'
    '        assert (\n'
    '            row["Warning Score"]\n'
    '            == _two_year_yield_sheets_round(\n'
    '                clamp(\n'
    '                    _two_year_yield_weighted_sum(\n'
    '                        (0.30, u), (0.25, v), (0.30, w), (0.15, x)\n'
    '                    ),\n'
    '                    -10,\n'
    '                    10,\n'
    '                ),\n'
    '                2,\n'
    '            )\n'
    '            == expected_aa\n'
    '        )\n'
    '    assert jul_1981["Signal Validity"] == "Valid"\n'
    '    assert (\n'
    '        jul_1981["Warning Flag"] == "Moderate upside 2Y yield restriction warning risk"\n'
    '    )\n'
    '    assert dec_1981["Warning Flag"] == "Moderate 2Y yield easing warning risk"\n'
    '\n'
    '    may_1986 = two_year_rows[119]\n'
    '    assert values_equivalent(may_1986["1M Yield Change"], -0.30)\n'
    '    assert may_1986["1M 2Y Yield Shock Flag"] == "1M 2Y yield downside shock"\n'
    '    assert may_1986["2Y Yield Shock Score"] == -6\n'
    '    assert may_1986["2Y Yield Extreme Score"] == -0.90\n'
    '    assert may_1986["Warning Score"] == -3.14\n'
    '    assert may_1986["Warning Flag"] == "Moderate 2Y yield easing warning risk"\n'
    '    assert (\n'
    '        _two_year_yield_warning_label(5.20)\n'
    '        == "Moderate upside 2Y yield restriction warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _two_year_yield_warning_label(-3.30) == "Moderate 2Y yield easing warning risk"\n'
    '    )\n'
    '\n'
    '    # Displayed z-score columns and the separate Extreme Flag retain their gates/formulas.\n'
    '    assert [\n'
    '        jul_1981[c]\n'
    '        for c in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [2.1, "", "", ""]\n'
    '    assert [\n'
    '        dec_1981[c]\n'
    '        for c in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [1.33, -2.02, "", ""]\n'
    '    assert [\n'
    '        may_1986[c]\n'
    '        for c in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [-1.23, -0.90, -1.06, -1.32]\n'
    '    assert jul_1981["Extreme Flag"] == "" and may_1986["Extreme Flag"] == "Normal range"\n'
    '    assert (\n'
    '        _two_year_yield_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside 2Y yield pressure"\n'
    '    )\n'
    '\n'
    '    april_1999, may_1999, june_1999 = (\n'
    '        two_year_rows[274],\n'
    '        two_year_rows[275],\n'
    '        two_year_rows[276],\n'
    '    )\n'
    '    assert april_1999["2Y Yield Regime"] == "Elevated and rising 2Y yield regime"\n'
    '    assert may_1999["2Y Yield Regime"] == "High and rising 2Y yield regime"\n'
    '    assert june_1999["2Y Yield Regime"] == "High and rising 2Y yield regime"\n'
    '    assert may_1999["3M Change Percentile"] == "71.90%"\n'
    '    assert may_1999["3M Change Score"] == 4.38\n'
    '    assert may_1999["2Y Yield Impulse Score"] == 2.63\n'
    '    assert may_1999["Coincident Score"] == 0.33\n'
    '    assert may_1999["Leading Score"] == 1.98\n'
    '    assert may_1999["Regime"] == "2Y yields broadly neutral"\n'
    '    assert may_1999["Signal Comment"] == "Mixed/transition 2Y yield signal"\n'
    '    assert all(\n'
    '        isinstance(row[col], str) and row[col].endswith("%")\n'
    '        for row in (jul_1981, aug_1981, nov_1981, may_1999)\n'
    '        for col in rank_columns\n'
    '    )\n'
    '\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped 2Y yield sheet", CONTROL_PANEL_TWENTY_SECOND_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_two_year_yield_outputs\n'
    '    )\n'
    '    all_specs_with_a24 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 24)] + [["2Y Yield"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("2Y Yield", "A24") in all_specs_with_a24\n'
    '    expected_ten_year_columns = [\n'
    '        "3M MA Yield",\n'
    '        "Sample Count",\n'
    '        "1M Yield Change",\n'
    '        "3M Yield Change",\n'
    '        "6M Yield Change",\n'
    '        "12M Yield Change",\n'
    '        "Yield Level Percentile",\n'
    '        "Yield Level Score",\n'
    '        "3M Change Percentile",\n'
    '        "3M Change Score",\n'
    '        "6M Change Percentile",\n'
    '        "6M Change Score",\n'
    '        "12M Change Percentile",\n'
    '        "12M Change Score",\n'
    '        "10Y Yield Regime",\n'
    '        "10Y Yield Asset Bias",\n'
    '        "1M 10Y Yield Shock Flag",\n'
    '        "10Y Yield Impulse Score",\n'
    '        "10Y Yield Cycle Turning Point Score",\n'
    '        "10Y Yield Shock Score",\n'
    '        "Long-Rate Restriction Score",\n'
    '        "10Y Yield Extreme Score",\n'
    '        "Coincident Score",\n'
    '        "Leading Score",\n'
    '        "Warning Score",\n'
    '        "State",\n'
    '        "Regime",\n'
    '        "Signal Comment",\n'
    '        "Reliability",\n'
    '        "Signal Validity",\n'
    '        "Warning Flag",\n'
    '        "Level Z-Score",\n'
    '        "3M Change Z-Score",\n'
    '        "6M Change Z-Score",\n'
    '        "12M Change Z-Score",\n'
    '        "Extreme Flag",\n'
    '    ]\n'
    '    assert TEN_YEAR_YIELD_OUTPUT_COLUMNS == expected_ten_year_columns\n'
    '    ten_year_spec = select_indicator_engine_spec(\n'
    '        "Unmapped 10Y yield sheet", CONTROL_PANEL_TWENTY_THIRD_INDICATOR_CELL\n'
    '    )\n'
    '    assert ten_year_spec.calculate is calculate_ten_year_yield_outputs\n'
    '    assert (ten_year_spec.input_start_col, ten_year_spec.input_end_col) == ("A", "B")\n'
    '    assert (ten_year_spec.output_start_col, ten_year_spec.output_end_col) == ("C", "AL")\n'
    '    assert list(ten_year_spec.output_columns) == expected_ten_year_columns\n'
    '    assert (\n'
    '        indicator_input_range(\n'
    '            "10Y government bond yield",\n'
    '            ten_year_spec.input_end_col,\n'
    '            ten_year_spec.input_start_col,\n'
    '        )\n'
    '        == "\'10Y government bond yield\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range(\n'
    '            "10Y government bond yield",\n'
    '            773,\n'
    '            ten_year_spec.output_end_col,\n'
    '            ten_year_spec.output_start_col,\n'
    '        )\n'
    '        == "\'10Y government bond yield\'!C2:AL773"\n'
    '    )\n'
    '    assert output_end_col_for_columns(TEN_YEAR_YIELD_OUTPUT_COLUMNS) == "AL"\n'
    '    assert _ten_year_yield_sheets_round(2.675, 2) == 2.68 and round(2.675, 2) != 2.68\n'
    '    assert _ten_year_yield_sheets_round(-2.675, 2) == -2.68\n'
    '    assert _ten_year_yield_sheets_round(0.0625, 3) == 0.063\n'
    '    assert _ten_year_yield_percentrank_inc_sheets([1, 2, 2, 4], 2) == 1 / 3\n'
    '    assert values_equivalent(\n'
    '        _ten_year_yield_percentrank_inc_sheets([1, 2, 2, 4], 3), 2.5 / 3\n'
    '    )\n'
    '\n'
    '    ten_year_values = [2.0 + 0.025 * i + 0.45 * math.sin(i / 7) for i in range(150)]\n'
    '    ten_year_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"10y{i}", "Raw Value": ten_year_values[i]}\n'
    '            for i in range(len(ten_year_values))\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    ten_year_out = calculate_ten_year_yield_outputs(ten_year_raw)\n'
    '    ten_year_rows = df_records(ten_year_out)\n'
    '    assert (\n'
    '        ten_year_out.shape[1] == 36\n'
    '        and list(ten_year_out.columns) == expected_ten_year_columns\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        ten_year_rows[2]["3M MA Yield"],\n'
    '        _ten_year_yield_mean_sheets(ten_year_values[:3]),\n'
    '    )\n'
    '    assert (\n'
    '        ten_year_rows[0]["Sample Count"] == 0 and ten_year_rows[2]["Sample Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        ten_year_rows[3]["1M Yield Change"],\n'
    '        ten_year_rows[3]["3M MA Yield"] - ten_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        ten_year_rows[5]["3M Yield Change"],\n'
    '        ten_year_rows[5]["3M MA Yield"] - ten_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        ten_year_rows[8]["6M Yield Change"],\n'
    '        ten_year_rows[8]["3M MA Yield"] - ten_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        ten_year_rows[14]["12M Yield Change"],\n'
    '        ten_year_rows[14]["3M MA Yield"] - ten_year_rows[2]["3M MA Yield"],\n'
    '    )\n'
    '\n'
    '    # A25 rank gates are metric-specific: C/F/G/H reach 60 at different rows.\n'
    '    for before, first, rank_col in (\n'
    '        (60, 61, "Yield Level Percentile"),\n'
    '        (63, 64, "3M Change Percentile"),\n'
    '        (66, 67, "6M Change Percentile"),\n'
    '        (72, 73, "12M Change Percentile"),\n'
    '    ):\n'
    '        assert ten_year_rows[before][rank_col] == ""\n'
    '        assert ten_year_rows[first][rank_col] != ""\n'
    '    for row_index, metric_col, rank_col, score_col in (\n'
    '        (70, "3M MA Yield", "Yield Level Percentile", "Yield Level Score"),\n'
    '        (70, "3M Yield Change", "3M Change Percentile", "3M Change Score"),\n'
    '        (75, "6M Yield Change", "6M Change Percentile", "6M Change Score"),\n'
    '        (75, "12M Yield Change", "12M Change Percentile", "12M Change Score"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[metric_col]\n'
    '            for row in ten_year_rows[: row_index + 1]\n'
    '            if not is_blank(row[metric_col])\n'
    '        ]\n'
    '        expected_rank = _ten_year_yield_sheets_round(\n'
    '            _ten_year_yield_percentrank_inc_sheets(\n'
    '                history, ten_year_rows[row_index][metric_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert ten_year_rows[row_index][rank_col] == _ten_year_yield_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert ten_year_rows[row_index][score_col] == _ten_year_yield_sheets_round(\n'
    '            expected_rank * 20 - 10, 2\n'
    '        )\n'
    '    assert all(\n'
    '        isinstance(ten_year_rows[90][col], str) and ten_year_rows[90][col].endswith("%")\n'
    '        for col in (\n'
    '            "Yield Level Percentile",\n'
    '            "3M Change Percentile",\n'
    '            "6M Change Percentile",\n'
    '            "12M Change Percentile",\n'
    '        )\n'
    '    )\n'
    '\n'
    '    early_ten_year_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": date_text, "Raw Value": value}\n'
    '            for date_text, value in (\n'
    '                ("31/01/1962", 4.15),\n'
    '                ("28/02/1962", 3.97),\n'
    '                ("31/03/1962", 3.93),\n'
    '                ("30/04/1962", 3.91),\n'
    '                ("31/05/1962", 3.80),\n'
    '            )\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    early_ten_year_rows = df_records(\n'
    '        calculate_ten_year_yield_outputs(early_ten_year_raw)\n'
    '    )\n'
    '    assert [row["10Y Yield Regime"] for row in early_ten_year_rows[2:5]] == [\n'
    '        "Elevated and rising 10Y yield regime",\n'
    '        "Elevated and rising 10Y yield regime",\n'
    '        "Elevated and rising 10Y yield regime",\n'
    '    ]\n'
    '    assert _ten_year_yield_level_regime(5.0, "") == "High and rising 10Y yield regime"\n'
    '    assert (\n'
    '        _ten_year_yield_level_regime(3.0, 0) == "Elevated and rising 10Y yield regime"\n'
    '    )\n'
    '    assert _ten_year_yield_level_regime(1.5, 0) == "Moderate 10Y yield and rising"\n'
    '    assert _ten_year_yield_level_regime(1.49, 0) == "Low 10Y yield but rising"\n'
    '    assert (\n'
    '        _ten_year_yield_level_regime(4.146666666666667, 0)\n'
    '        == "Elevated and rising 10Y yield regime"\n'
    '    )\n'
    '\n'
    '    assert _ten_year_yield_shock_label(0.30) == "1M 10Y yield upside shock"\n'
    '    assert _ten_year_yield_shock_label(-0.30) == "1M 10Y yield downside shock"\n'
    '    assert _ten_year_yield_shock_label(-0.20) == "No 1M 10Y yield shock"\n'
    '    assert _ten_year_yield_shock_score(0.30) == 6\n'
    '    assert _ten_year_yield_shock_score(0.20) == 4\n'
    '    assert _ten_year_yield_shock_score(-0.20) == -4\n'
    '    assert _ten_year_yield_shock_score(-0.30) == -6\n'
    '    assert _ten_year_yield_shock_score(-0.2999999999999998) == -6\n'
    '\n'
    '    # Supplied A25 spreadsheet-style component/composite regression rows.\n'
    '    ten_year_composite_cases = (\n'
    '        ((9.76, 1.90, -8.54, 7.28, 0, 0, 0, 4), (-0.55, 2.86, 0.54, 0.60)),\n'
    '        ((1.18, -9.52, -9.72, -10.0, -5, -6, 0, -6), (-9.13, -5.82, -8.31, -3.90)),\n'
    '        ((-6.04, -6.42, -6.66, -6.66, -5, -4, 0, 0), (-6.52, -6.37, -6.42, -2.50)),\n'
    '        ((-7.98, -8.00, -4.58, -4.34, -5, -6, 0, 0), (-6.25, -6.77, -6.27, -3.00)),\n'
    '    )\n'
    '    for (j, l, n, p, u, v, w, x), (\n'
    '        expected_t,\n'
    '        expected_y,\n'
    '        expected_z,\n'
    '        expected_aa,\n'
    '    ) in ten_year_composite_cases:\n'
    '        t = _ten_year_yield_sheets_round(\n'
    '            clamp(\n'
    '                _ten_year_yield_weighted_sum(\n'
    '                    (0.45, l), (0.35, n), (0.15, p), (0.05, j)\n'
    '                ),\n'
    '                -10,\n'
    '                10,\n'
    '            ),\n'
    '            2,\n'
    '        )\n'
    '        y = _ten_year_yield_sheets_round(\n'
    '            clamp(\n'
    '                _ten_year_yield_weighted_sum(\n'
    '                    (0.35, j), (0.25, l), (0.20, n), (0.10, p), (0.10, t)\n'
    '                ),\n'
    '                -10,\n'
    '                10,\n'
    '            ),\n'
    '            2,\n'
    '        )\n'
    '        z = _ten_year_yield_sheets_round(\n'
    '            clamp(\n'
    '                _ten_year_yield_weighted_sum(\n'
    '                    (0.10, j), (0.35, l), (0.25, n), (0.15, p), (0.10, t), (0.05, u)\n'
    '                ),\n'
    '                -10,\n'
    '                10,\n'
    '            ),\n'
    '            2,\n'
    '        )\n'
    '        aa = _ten_year_yield_sheets_round(\n'
    '            clamp(\n'
    '                _ten_year_yield_weighted_sum(\n'
    '                    (0.30, u), (0.25, v), (0.30, w), (0.15, x)\n'
    '                ),\n'
    '                -10,\n'
    '                10,\n'
    '            ),\n'
    '            2,\n'
    '        )\n'
    '        assert (t, y, z, aa) == (expected_t, expected_y, expected_z, expected_aa)\n'
    '    assert _ten_year_yield_warning_label(-2.50) == "Neutral 10Y yield warning profile"\n'
    '    assert (\n'
    '        _ten_year_yield_warning_label(-3.00) == "Moderate 10Y yield easing warning risk"\n'
    '    )\n'
    '    assert (\n'
    '        _ten_year_yield_asset_bias_label(6)\n'
    '        == "Strong bearish duration / higher discount-rate pressure"\n'
    '    )\n'
    '    assert _ten_year_yield_recovery_tightening_score(1.4, 1, 1) == 5\n'
    '    assert _ten_year_yield_restriction_score(5.1, 1, 1) == 6\n'
    '\n'
    '    # A25 AH:AK and AL retain the pre-existing expanding z-score/flag formulas.\n'
    '    ten_year_c = [row["3M MA Yield"] for row in ten_year_rows]\n'
    '    ten_year_f = [row["3M Yield Change"] for row in ten_year_rows]\n'
    '    ten_year_g = [row["6M Yield Change"] for row in ten_year_rows]\n'
    '    ten_year_h = [row["12M Yield Change"] for row in ten_year_rows]\n'
    '    ten_year_d = [row["Sample Count"] for row in ten_year_rows]\n'
    '    expected_zs = (\n'
    '        _rounded_expanding_zscores(ten_year_c, ten_year_d, 60)[90],\n'
    '        _rounded_expanding_zscores(ten_year_f, _metric_counts(ten_year_f), 60)[90],\n'
    '        _rounded_expanding_zscores(ten_year_g, _metric_counts(ten_year_g), 60)[90],\n'
    '        _rounded_expanding_zscores(ten_year_h, _metric_counts(ten_year_h), 60)[90],\n'
    '    )\n'
    '    actual_zs = tuple(\n'
    '        ten_year_rows[90][col]\n'
    '        for col in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    )\n'
    '    assert actual_zs == expected_zs\n'
    '    assert ten_year_rows[90]["Extreme Flag"] == _ten_year_yield_outlier_label(actual_zs)\n'
    '    assert _ten_year_yield_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _ten_year_yield_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside 10Y yield pressure"\n'
    '    )\n'
    '\n'
    '    class FakeTenYearVerifyWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '    fake_ten_year_verify = FakeTenYearVerifyWorksheet(\n'
    '        dataframe_to_sheet_rows(ten_year_out, TEN_YEAR_YIELD_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_ten_year_verify,\n'
    '        ten_year_out,\n'
    '        151,\n'
    '        TEN_YEAR_YIELD_OUTPUT_COLUMNS,\n'
    '        "AL",\n'
    '        "C",\n'
    '    )\n'
    '    assert fake_ten_year_verify.get_calls == [("C2:AL151", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            ten_year_out, 151, "C", 2, TEN_YEAR_YIELD_OUTPUT_COLUMNS, "AL"\n'
    '        )\n'
    '        == "C2:AL151"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped 10Y yield sheet", CONTROL_PANEL_TWENTY_THIRD_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_ten_year_yield_outputs\n'
    '    )\n'
    '    all_specs_with_a25 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 25)] + [["10Y Yield"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("10Y Yield", "A25") in all_specs_with_a25\n'
    '    expected_yield_curve_columns = [\n'
    '        "3M MA Curve Spread",\n'
    '        "Sample Count",\n'
    '        "1M Curve Change",\n'
    '        "3M Curve Change",\n'
    '        "6M Curve Change",\n'
    '        "12M Curve Change",\n'
    '        "Curve Level Percentile",\n'
    '        "Curve Stress Level Score",\n'
    '        "3M Change Percentile",\n'
    '        "3M Change Stress Score",\n'
    '        "6M Change Percentile",\n'
    '        "6M Change Stress Score",\n'
    '        "12M Change Percentile",\n'
    '        "12M Change Stress Score",\n'
    '        "Curve Regime",\n'
    '        "Recession Risk Bias",\n'
    '        "Curve Shock Flag",\n'
    '        "Curve Impulse Score",\n'
    '        "Curve Cycle Turning Point Score",\n'
    '        "Curve Shock Score",\n'
    '        "Recession Risk Pressure Score",\n'
    '        "Curve Extreme Score",\n'
    '        "Coincident Score",\n'
    '        "Leading Score",\n'
    '        "Warning Score",\n'
    '        "State",\n'
    '        "Regime",\n'
    '        "Signal Comment",\n'
    '        "Reliability",\n'
    '        "Signal Validity",\n'
    '        "Warning Flag",\n'
    '        "Level Z-Score",\n'
    '        "3M Change Z-Score",\n'
    '        "6M Change Z-Score",\n'
    '        "12M Change Z-Score",\n'
    '        "Extreme Flag",\n'
    '    ]\n'
    '    assert YIELD_CURVE_OUTPUT_COLUMNS == expected_yield_curve_columns\n'
    '    yield_curve_spec = select_indicator_engine_spec(\n'
    '        "Unmapped yield curve sheet", CONTROL_PANEL_TWENTY_FOURTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert yield_curve_spec.calculate is calculate_yield_curve_outputs\n'
    '    assert (yield_curve_spec.input_start_col, yield_curve_spec.input_end_col) == (\n'
    '        "A",\n'
    '        "B",\n'
    '    )\n'
    '    assert (yield_curve_spec.output_start_col, yield_curve_spec.output_end_col) == (\n'
    '        "C",\n'
    '        "AL",\n'
    '    )\n'
    '    assert list(yield_curve_spec.output_columns) == expected_yield_curve_columns\n'
    '    assert (\n'
    '        indicator_input_range(\n'
    '            "10Y-3M yield curve spread",\n'
    '            yield_curve_spec.input_end_col,\n'
    '            yield_curve_spec.input_start_col,\n'
    '        )\n'
    '        == "\'10Y-3M yield curve spread\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range(\n'
    '            "10Y-3M yield curve spread",\n'
    '            533,\n'
    '            yield_curve_spec.output_end_col,\n'
    '            yield_curve_spec.output_start_col,\n'
    '        )\n'
    '        == "\'10Y-3M yield curve spread\'!C2:AL533"\n'
    '    )\n'
    '    assert output_end_col_for_columns(YIELD_CURVE_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        select_indicator_engine("A25 remains A25", "A25")\n'
    '        is calculate_ten_year_yield_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A27 remains A27", "A27")\n'
    '        is calculate_term_premium_outputs\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("A39 remains A39", "A39")\n'
    '        is calculate_corporate_credit_spreads_outputs\n'
    '    )\n'
    '\n'
    '    assert _yield_curve_sheets_round(2.675, 2) == 2.68 and round(2.675, 2) != 2.68\n'
    '    assert _yield_curve_sheets_round(-2.675, 2) == -2.68\n'
    '    assert _yield_curve_sheets_round(0.0625, 3) == 0.063\n'
    '    assert _yield_curve_percentrank_inc_sheets([1, 2, 2, 4], 2) == 1 / 3\n'
    '    assert values_equivalent(\n'
    '        _yield_curve_percentrank_inc_sheets([1, 2, 2, 4], 3), 2.5 / 3\n'
    '    )\n'
    '\n'
    '    # Static A26 source fixture through the supplied February 1988 row.\n'
    '    yield_curve_values = [\n'
    '        1.68,\n'
    '        0.15,\n'
    '        0.55,\n'
    '        0.53,\n'
    '        0.91,\n'
    '        1.22,\n'
    '        2.08,\n'
    '        4.06,\n'
    '        4.15,\n'
    '        2.94,\n'
    '        2.2,\n'
    '        2.34,\n'
    '        2.34,\n'
    '        2.33,\n'
    '        1.85,\n'
    '        1.89,\n'
    '        1.88,\n'
    '        1.71,\n'
    '        1.93,\n'
    '        2.11,\n'
    '        2.29,\n'
    '        2.55,\n'
    '        2.58,\n'
    '        2.47,\n'
    '        2.41,\n'
    '        2.38,\n'
    '        2.43,\n'
    '        2.56,\n'
    '        3.19,\n'
    '        3.3,\n'
    '        2.83,\n'
    '        1.82,\n'
    '        1.73,\n'
    '        2.04,\n'
    '        2.65,\n'
    '        3.16,\n'
    '        3.36,\n'
    '        2.95,\n'
    '        3.03,\n'
    '        3.21,\n'
    '        3.11,\n'
    '        2.98,\n'
    '        2.99,\n'
    '        2.96,\n'
    '        3.04,\n'
    '        2.84,\n'
    '        2.31,\n'
    '        1.93,\n'
    '        1.89,\n'
    '        1.41,\n'
    '        1.01,\n'
    '        1.06,\n'
    '        1.37,\n'
    '        1.4,\n'
    '        1.3,\n'
    '        1.48,\n'
    '        2.1,\n'
    '        2.1,\n'
    '        1.75,\n'
    '        1.43,\n'
    '        1.51,\n'
    '        1.5,\n'
    '        1.48,\n'
    '        2.21,\n'
    '        2.77,\n'
    '        2.55,\n'
    '        2.57,\n'
    '        2.53,\n'
    '        2.8,\n'
    '        3.17,\n'
    '        2.97,\n'
    '        3.03,\n'
    '        2.66,\n'
    '        2.37,\n'
    '    ]\n'
    '    yield_curve_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"curve{i}", "Raw Value": value}\n'
    '            for i, value in enumerate(yield_curve_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    yield_curve_out = calculate_yield_curve_outputs(yield_curve_raw)\n'
    '    yield_curve_rows = df_records(yield_curve_out)\n'
    '    assert (\n'
    '        yield_curve_out.shape[1] == 36\n'
    '        and list(yield_curve_out.columns) == expected_yield_curve_columns\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        yield_curve_rows[2]["3M MA Curve Spread"],\n'
    '        _yield_curve_mean_sheets(yield_curve_values[:3]),\n'
    '    )\n'
    '    assert (\n'
    '        yield_curve_rows[0]["Sample Count"] == 0\n'
    '        and yield_curve_rows[2]["Sample Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        yield_curve_rows[3]["1M Curve Change"],\n'
    '        yield_curve_rows[3]["3M MA Curve Spread"]\n'
    '        - yield_curve_rows[2]["3M MA Curve Spread"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        yield_curve_rows[5]["3M Curve Change"],\n'
    '        yield_curve_rows[5]["3M MA Curve Spread"]\n'
    '        - yield_curve_rows[2]["3M MA Curve Spread"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        yield_curve_rows[8]["6M Curve Change"],\n'
    '        yield_curve_rows[8]["3M MA Curve Spread"]\n'
    '        - yield_curve_rows[2]["3M MA Curve Spread"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        yield_curve_rows[14]["12M Curve Change"],\n'
    '        yield_curve_rows[14]["3M MA Curve Spread"]\n'
    '        - yield_curve_rows[2]["3M MA Curve Spread"],\n'
    '    )\n'
    '\n'
    '    rank_columns = (\n'
    '        "Curve Level Percentile",\n'
    '        "3M Change Percentile",\n'
    '        "6M Change Percentile",\n'
    '        "12M Change Percentile",\n'
    '    )\n'
    '    assert yield_curve_rows[60]["Sample Count"] == 59 and all(\n'
    '        yield_curve_rows[60][col] == "" for col in rank_columns\n'
    '    )\n'
    '    assert yield_curve_rows[61]["Sample Count"] == 60 and all(\n'
    '        yield_curve_rows[61][col] != "" for col in rank_columns\n'
    '    )\n'
    '    assert (\n'
    '        sum(not is_blank(row["3M Curve Change"]) for row in yield_curve_rows[:62]) == 57\n'
    '    )\n'
    '    assert (\n'
    '        sum(not is_blank(row["6M Curve Change"]) for row in yield_curve_rows[:62]) == 54\n'
    '    )\n'
    '    assert (\n'
    '        sum(not is_blank(row["12M Curve Change"]) for row in yield_curve_rows[:62])\n'
    '        == 48\n'
    '    )\n'
    '    for metric_col, rank_col, score_col in (\n'
    '        ("3M MA Curve Spread", "Curve Level Percentile", "Curve Stress Level Score"),\n'
    '        ("3M Curve Change", "3M Change Percentile", "3M Change Stress Score"),\n'
    '        ("6M Curve Change", "6M Change Percentile", "6M Change Stress Score"),\n'
    '        ("12M Curve Change", "12M Change Percentile", "12M Change Stress Score"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[metric_col]\n'
    '            for row in yield_curve_rows[:62]\n'
    '            if not is_blank(row[metric_col])\n'
    '        ]\n'
    '        expected_rank = _yield_curve_sheets_round(\n'
    '            _yield_curve_percentrank_inc_sheets(\n'
    '                history, yield_curve_rows[61][metric_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert yield_curve_rows[61][rank_col] == _yield_curve_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert yield_curve_rows[61][score_col] == _yield_curve_sheets_round(\n'
    '            10 - expected_rank * 20, 2\n'
    '        )\n'
    '\n'
    '    feb_1987, mar_1987, apr_1987, may_1987, feb_1988 = (\n'
    '        yield_curve_rows[61],\n'
    '        yield_curve_rows[62],\n'
    '        yield_curve_rows[63],\n'
    '        yield_curve_rows[64],\n'
    '        yield_curve_rows[73],\n'
    '    )\n'
    '    assert [feb_1987[col] for col in rank_columns] == [\n'
    '        "18.60%",\n'
    '        "21.40%",\n'
    '        "50.90%",\n'
    '        "40.40%",\n'
    '    ]\n'
    '    assert [\n'
    '        feb_1987[col]\n'
    '        for col in (\n'
    '            "Curve Stress Level Score",\n'
    '            "3M Change Stress Score",\n'
    '            "6M Change Stress Score",\n'
    '            "12M Change Stress Score",\n'
    '        )\n'
    '    ] == [6.28, 5.72, -0.18, 1.92]\n'
    '    assert [\n'
    '        feb_1987[col]\n'
    '        for col in ("Curve Impulse Score", "Coincident Score", "Leading Score")\n'
    '    ] == [3.11, 4.71, 3.43]\n'
    '    assert feb_1987["State"] == feb_1987["Regime"] == "Yield-curve stress rising"\n'
    '    assert (\n'
    '        feb_1987["Signal Validity"] == "Incomplete Score"\n'
    '        and feb_1987["Warning Score"] == ""\n'
    '    )\n'
    '    assert mar_1987["Coincident Score"] == 4.14 and mar_1987["Leading Score"] == 3.33\n'
    '    assert (\n'
    '        apr_1987["State"] == "Neutral yield-curve stress"\n'
    '        and apr_1987["Regime"] == "Yield curve broadly neutral"\n'
    '    )\n'
    '    assert [may_1987[col] for col in rank_columns] == [\n'
    '        "48.40%",\n'
    '        "89.80%",\n'
    '        "55.40%",\n'
    '        "88.00%",\n'
    '    ]\n'
    '    assert [\n'
    '        may_1987[col]\n'
    '        for col in (\n'
    '            "Curve Stress Level Score",\n'
    '            "3M Change Stress Score",\n'
    '            "6M Change Stress Score",\n'
    '            "12M Change Stress Score",\n'
    '        )\n'
    '    ] == [0.32, -7.96, -1.08, -7.60]\n'
    '    assert [\n'
    '        may_1987[col]\n'
    '        for col in ("Curve Impulse Score", "Coincident Score", "Leading Score")\n'
    '    ] == [-5.08, -3.17, -4.11]\n'
    '    assert may_1987["State"] == may_1987["Regime"] == "Yield-curve stress easing"\n'
    '\n'
    '    assert (\n'
    '        yield_curve_rows[2]["Curve Regime"]\n'
    '        == "Moderately positive curve and steepening"\n'
    '    )\n'
    '    assert yield_curve_rows[3]["Curve Regime"] == "Near inversion but improving"\n'
    '    assert (\n'
    '        yield_curve_rows[4]["Curve Regime"]\n'
    '        == "Moderately positive curve and steepening"\n'
    '    )\n'
    '    assert _yield_curve_regime(1.5, "") == "Steep curve / easing-recovery signal"\n'
    '    assert _yield_curve_regime(0.5, 0) == "Moderately positive curve and steepening"\n'
    '    assert _yield_curve_regime(0.1, 0) == "Near inversion but improving"\n'
    '    assert _yield_curve_regime(0, 0) == "Inverted but steepening"\n'
    '    assert _yield_curve_regime(-0.1, -0.1) == "Inverted and flattening further"\n'
    '\n'
    '    assert _yield_curve_shock_label(-0.30) == "1M curve-flattening shock"\n'
    '    assert _yield_curve_shock_label(0.30) == "1M curve-steepening shock"\n'
    '    assert _yield_curve_shock_label(-0.20) == "No 1M curve shock"\n'
    '    assert _yield_curve_shock_score(-0.30) == 6\n'
    '    assert _yield_curve_shock_score(-0.20) == 4\n'
    '    assert _yield_curve_shock_score(0.20) == -4\n'
    '    assert _yield_curve_shock_score(0.30) == -6\n'
    '    assert _yield_curve_shock_score(-0.2999999999999998) == 6\n'
    '    assert feb_1988["Curve Shock Flag"] == "No 1M curve shock"\n'
    '    assert feb_1988["Curve Shock Score"] == 4 and feb_1988["Warning Score"] == 1.0\n'
    '    assert feb_1988["State"] == "Neutral yield-curve stress"\n'
    '    assert feb_1988["Regime"] == "Yield curve broadly neutral"\n'
    '    assert feb_1988["Signal Comment"] == "Neutral/mixed yield-curve signal"\n'
    '    assert feb_1988["Signal Validity"] == "Valid"\n'
    '    assert feb_1988["Warning Flag"] == "Neutral yield-curve warning profile"\n'
    '\n'
    '    # Supplied 2004/2019 component cases, evaluated from rounded visible operands.\n'
    '    j, l, n, p, u, v, w, x = (-7.48, 3.60, 1.48, -0.20, 4, 6, 0, 0)\n'
    '    t = _yield_curve_sheets_round(\n'
    '        clamp(\n'
    '            _yield_curve_weighted_sum((0.05, j), (0.45, l), (0.35, n), (0.15, p)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    y = _yield_curve_sheets_round(\n'
    '        clamp(\n'
    '            _yield_curve_weighted_sum((0.45, j), (0.30, l), (0.15, n), (0.10, p)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    z = _yield_curve_sheets_round(\n'
    '        clamp(\n'
    '            _yield_curve_weighted_sum(\n'
    '                (0.20, j), (0.30, l), (0.25, n), (0.10, p), (0.10, t), (0.05, u)\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    aa = _yield_curve_sheets_round(\n'
    '        clamp(\n'
    '            _yield_curve_weighted_sum((0.30, u), (0.25, v), (0.30, w), (0.15, x)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (t, y, z, aa) == (1.73, -2.08, 0.31, 2.70)\n'
    '    assert _yield_curve_warning_label(aa) == "Neutral yield-curve warning profile"\n'
    '    assert (\n'
    '        _yield_curve_sheets_round(\n'
    '            clamp(\n'
    '                _yield_curve_weighted_sum((0.30, 0), (0.25, -4), (0.30, 0), (0.15, 0)),\n'
    '                -10,\n'
    '                10,\n'
    '            ),\n'
    '            2,\n'
    '        )\n'
    '        == -1.0\n'
    '    )\n'
    '    assert _yield_curve_flattening_stress_score(1.2, -1, -1) == 4\n'
    '    assert _yield_curve_inversion_restriction_score(-0.1, -1, -1) == 6\n'
    '    assert (\n'
    '        _yield_curve_recession_risk_label(-0.1, -0.1, 6)\n'
    '        == "High recession-risk curve signal"\n'
    '    )\n'
    '\n'
    '    # A26 AH:AK and AL retain the existing expanding z-score/flag formulas.\n'
    '    curve_c = [row["3M MA Curve Spread"] for row in yield_curve_rows]\n'
    '    curve_f = [row["3M Curve Change"] for row in yield_curve_rows]\n'
    '    curve_g = [row["6M Curve Change"] for row in yield_curve_rows]\n'
    '    curve_h = [row["12M Curve Change"] for row in yield_curve_rows]\n'
    '    curve_d = [row["Sample Count"] for row in yield_curve_rows]\n'
    '    expected_curve_zs = (\n'
    '        _rounded_expanding_zscores(curve_c, curve_d, 60)[73],\n'
    '        _rounded_expanding_zscores(curve_f, _metric_counts(curve_f), 60)[73],\n'
    '        _rounded_expanding_zscores(curve_g, _metric_counts(curve_g), 60)[73],\n'
    '        _rounded_expanding_zscores(curve_h, _metric_counts(curve_h), 60)[73],\n'
    '    )\n'
    '    actual_curve_zs = tuple(\n'
    '        feb_1988[col]\n'
    '        for col in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    )\n'
    '    assert actual_curve_zs == expected_curve_zs\n'
    '    assert (\n'
    '        feb_1988["Extreme Flag"]\n'
    '        == _yield_curve_outlier_label(actual_curve_zs)\n'
    '        == "Normal range"\n'
    '    )\n'
    '    assert _yield_curve_outlier_score([-3.1, 3.5, 0, 0]) == 6\n'
    '\n'
    '    class FakeYieldCurveVerifyWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '    fake_curve_verify = FakeYieldCurveVerifyWorksheet(\n'
    '        dataframe_to_sheet_rows(yield_curve_out, YIELD_CURVE_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_curve_verify, yield_curve_out, 75, YIELD_CURVE_OUTPUT_COLUMNS, "AL", "C"\n'
    '    )\n'
    '    assert fake_curve_verify.get_calls == [("C2:AL75", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            yield_curve_out, 75, "C", 2, YIELD_CURVE_OUTPUT_COLUMNS, "AL"\n'
    '        )\n'
    '        == "C2:AL75"\n'
    '    )\n'
    '    blank_yield_curve = calculate_yield_curve_outputs(\n'
    '        make_dataframe([{"Date": "blank", "Raw Value": ""}], ["Date", "Raw Value"])\n'
    '    )\n'
    '    assert df_records(blank_yield_curve)[0]["Reliability"] == "Insufficient history"\n'
    '    assert df_records(blank_yield_curve)[0]["Signal Validity"] == "Missing Data"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped yield curve sheet", CONTROL_PANEL_TWENTY_FOURTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_yield_curve_outputs\n'
    '    )\n'
    '    all_specs_with_a26 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 26)] + [["Yield Curve"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Yield Curve", "A26") in all_specs_with_a26\n'
    '    term_premium_values = [-0.6 + 0.02 * i + 0.35 * math.sin(i / 6) for i in range(150)]\n'
    '    term_premium_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"tp{i}", "Raw Value": term_premium_values[i]}\n'
    '            for i in range(len(term_premium_values))\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    term_premium_out = calculate_term_premium_outputs(term_premium_raw)\n'
    '    term_premium_rows = df_records(term_premium_out)\n'
    '    assert term_premium_out.shape[1] == len(TERM_PREMIUM_OUTPUT_COLUMNS) == 36\n'
    '    assert output_end_col_for_columns(TERM_PREMIUM_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        indicator_output_range("Term Premium", 678, "AL") == "\'Term Premium\'!C2:AL678"\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_rows[2]["3-Period Moving Average"],\n'
    '        sum(term_premium_values[:3]) / 3,\n'
    '    )\n'
    '    assert (\n'
    '        term_premium_rows[0]["Smoothed Observation Count"] == 0\n'
    '        and term_premium_rows[2]["Smoothed Observation Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_rows[3]["1-Period Change"],\n'
    '        term_premium_rows[3]["3-Period Moving Average"]\n'
    '        - term_premium_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_rows[5]["3-Period Change"],\n'
    '        term_premium_rows[5]["3-Period Moving Average"]\n'
    '        - term_premium_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_rows[8]["6-Period Change"],\n'
    '        term_premium_rows[8]["3-Period Moving Average"]\n'
    '        - term_premium_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_rows[14]["12-Period Change"],\n'
    '        term_premium_rows[14]["3-Period Moving Average"]\n'
    '        - term_premium_rows[2]["3-Period Moving Average"],\n'
    '    )\n'
    '    assert (\n'
    '        term_premium_rows[60]["Term Premium Level PercentRank"] == ""\n'
    '        and term_premium_rows[61]["Term Premium Level PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        term_premium_rows[63]["3-Period Term Premium Change PercentRank"] == ""\n'
    '        and term_premium_rows[64]["3-Period Term Premium Change PercentRank"] != ""\n'
    '    )\n'
    '    assert term_premium_rows[61]["Term Premium Level Score"] == round(\n'
    '        term_premium_rows[61]["Term Premium Level PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert term_premium_rows[64]["3-Period Term Premium Change Score"] == round(\n'
    '        term_premium_rows[64]["3-Period Term Premium Change PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert _term_premium_level_regime(2.1, 0.1) == "High and rising term-premium regime"\n'
    '    assert (\n'
    '        _term_premium_level_regime(1.1, -0.1)\n'
    '        == "Elevated but easing term-premium regime"\n'
    '    )\n'
    '    assert _term_premium_level_regime(-0.1, 0.1) == "Negative term premium but rising"\n'
    '    assert (\n'
    '        _term_premium_asset_bias_label(6)\n'
    '        == "Strong bearish duration / rising term-premium risk"\n'
    '    )\n'
    '    assert _term_premium_shock_label(0.31) == "1M term-premium upside shock"\n'
    '    assert _term_premium_shock_label(-0.31) == "1M term-premium downside shock"\n'
    '    assert (\n'
    '        term_premium_rows[60]["Term Premium Level Z-Score"] == ""\n'
    '        and term_premium_rows[61]["Term Premium Level Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        term_premium_rows[63]["3-Period Term Premium Change Z-Score"] == ""\n'
    '        and term_premium_rows[64]["3-Period Term Premium Change Z-Score"] != ""\n'
    '    )\n'
    '    assert term_premium_rows[61]["Term Premium Level Z-Score"] == round(\n'
    '        term_premium_rows[61]["Term Premium Level Z-Score"], 2\n'
    '    )\n'
    '    assert _term_premium_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _term_premium_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme upside term-premium pressure"\n'
    '    )\n'
    '    term_premium_row = term_premium_rows[90]\n'
    '    tpj = term_premium_row["Term Premium Level Score"]\n'
    '    tpl = term_premium_row["3-Period Term Premium Change Score"]\n'
    '    tpn = term_premium_row["6-Period Term Premium Change Score"]\n'
    '    tpp = term_premium_row["12-Period Term Premium Change Score"]\n'
    '    tpt = term_premium_row["Term Premium Momentum Composite Score"]\n'
    '    tpu = term_premium_row["Term Premium Recovery / Tightening Score"]\n'
    '    tpv = term_premium_row["Term Premium Shock Score"]\n'
    '    tpw = term_premium_row["Term Premium Restriction Score"]\n'
    '    tpx = term_premium_row["Term Premium Outlier Score"]\n'
    '    assert values_equivalent(\n'
    '        tpt, round(clamp(0.45 * tpl + 0.35 * tpn + 0.15 * tpp + 0.05 * tpj), 2)\n'
    '    )\n'
    '    assert _term_premium_recovery_tightening_score(-0.1, 1, 1) == 5\n'
    '    assert _term_premium_recovery_tightening_score(1.1, 1, 1) == 6\n'
    '    assert _term_premium_shock_score(0.31) == 6\n'
    '    assert _term_premium_restriction_score(2.1, 1, 1) == 6\n'
    '    assert values_equivalent(\n'
    '        term_premium_row["Term Premium Leading Score"],\n'
    '        round(clamp(0.45 * tpj + 0.3 * tpl + 0.15 * tpn + 0.1 * tpp), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_row["Term Premium Warning Composite Score"],\n'
    '        round(\n'
    '            clamp(\n'
    '                0.15 * tpj\n'
    '                + 0.3 * tpl\n'
    '                + 0.25 * tpn\n'
    '                + 0.15 * tpp\n'
    '                + 0.1 * tpt\n'
    '                + 0.05 * tpu\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        term_premium_row["Term Premium Final Warning Score"],\n'
    '        round(clamp(0.3 * tpu + 0.25 * tpv + 0.3 * tpw + 0.15 * tpx), 2),\n'
    '    )\n'
    '    assert term_premium_row[\n'
    '        "Term Premium Asset Bias Label"\n'
    '    ] == _term_premium_asset_bias_label(term_premium_row["Term Premium Leading Score"])\n'
    '    assert term_premium_row[\n'
    '        "Term Premium Signal Interpretation"\n'
    '    ] == _term_premium_signal_interpretation(\n'
    '        term_premium_row["Term Premium Warning Composite Score"],\n'
    '        term_premium_row["Term Premium Final Warning Score"],\n'
    '    )\n'
    '    assert term_premium_row["Term Premium Pressure Regime"] in (\n'
    '        "Term-premium pressure rising",\n'
    '        "Term-premium pressure easing",\n'
    '        "Neutral term-premium pressure",\n'
    '    )\n'
    '    assert term_premium_row["Term Premium Momentum Label"] != ""\n'
    '    assert term_premium_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert term_premium_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert term_premium_row["Term Premium Warning Risk Label"] != ""\n'
    '    assert term_premium_row["Term Premium Outlier Label"] != ""\n'
    '    assert _term_premium_pressure_regime(2) == "Term-premium pressure rising"\n'
    '    assert _term_premium_momentum_label(2, 2) == "Term-premium pressure rising"\n'
    '    assert (\n'
    '        _term_premium_signal_interpretation(6, 0)\n'
    '        == "Strong upside term-premium signal with supportive risk-premium pressure profile"\n'
    '    )\n'
    '    assert _term_premium_warning_label(6) == "High upside term-premium risk warning"\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped term premium sheet", CONTROL_PANEL_TWENTY_FIFTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_term_premium_outputs\n'
    '    )\n'
    '    all_specs_with_a27 = _control_values_to_specs(\n'
    '        [\n'
    '            ["A3 Sheet"],\n'
    '            ["A4 Sheet"],\n'
    '            ["A5 Sheet"],\n'
    '            ["A6 Sheet"],\n'
    '            ["A7 Sheet"],\n'
    '            ["A8 Sheet"],\n'
    '            ["A9 Sheet"],\n'
    '            ["A10 Sheet"],\n'
    '            ["A11 Sheet"],\n'
    '            ["A12 Sheet"],\n'
    '            ["A13 Sheet"],\n'
    '            ["A14 Sheet"],\n'
    '            ["A15 Sheet"],\n'
    '            ["A16 Sheet"],\n'
    '            ["A17 Sheet"],\n'
    '            ["A18 Sheet"],\n'
    '            ["A19 Sheet"],\n'
    '            ["A20 Sheet"],\n'
    '            ["A21 Sheet"],\n'
    '            ["A22 Sheet"],\n'
    '            ["A23 Sheet"],\n'
    '            ["A24 Sheet"],\n'
    '            ["A25 Sheet"],\n'
    '            ["A26 Sheet"],\n'
    '            ["Term Premium"],\n'
    '            [""],\n'
    '        ],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Term Premium", "A27") in all_specs_with_a27\n'
    '    assert df_records(calculate_real_yield_outputs(real_yield_raw)) == real_yield_rows\n'
    '    assert df_records(calculate_two_year_yield_outputs(two_year_raw)) == two_year_rows\n'
    '    # A28 uses a fixed pre-1961 regression fixture so its percentile ranks,\n'
    '    # spreadsheet rounding, gates, and threshold-sensitive rows cannot drift.\n'
    '    interbank_rate_values = [\n'
    '        0.8,\n'
    '        1.22,\n'
    '        1.07,\n'
    '        0.85,\n'
    '        0.83,\n'
    '        1.28,\n'
    '        1.39,\n'
    '        1.29,\n'
    '        1.35,\n'
    '        1.43,\n'
    '        1.43,\n'
    '        1.64,\n'
    '        1.68,\n'
    '        1.96,\n'
    '        2.18,\n'
    '        2.24,\n'
    '        2.35,\n'
    '        2.48,\n'
    '        2.45,\n'
    '        2.5,\n'
    '        2.5,\n'
    '        2.62,\n'
    '        2.75,\n'
    '        2.71,\n'
    '        2.75,\n'
    '        2.73,\n'
    '        2.95,\n'
    '        2.96,\n'
    '        2.88,\n'
    '        2.94,\n'
    '        2.84,\n'
    '        3.0,\n'
    '        2.96,\n'
    '        3.0,\n'
    '        3.0,\n'
    '        3.0,\n'
    '        2.99,\n'
    '        3.24,\n'
    '        3.47,\n'
    '        3.5,\n'
    '        3.28,\n'
    '        2.98,\n'
    '        2.72,\n'
    '        1.67,\n'
    '        1.2,\n'
    '        1.26,\n'
    '        0.63,\n'
    '        0.93,\n'
    '        0.68,\n'
    '        1.53,\n'
    '        1.76,\n'
    '        1.8,\n'
    '        2.27,\n'
    '        2.42,\n'
    '        2.48,\n'
    '        2.43,\n'
    '        2.8,\n'
    '        2.96,\n'
    '        2.9,\n'
    '        3.39,\n'
    '        3.47,\n'
    '        3.5,\n'
    '        3.76,\n'
    '        3.98,\n'
    '        4.0,\n'
    '        3.99,\n'
    '        3.99,\n'
    '        3.97,\n'
    '        3.84,\n'
    '        3.92,\n'
    '        3.85,\n'
    '        3.32,\n'
    '        3.23,\n'
    '        2.98,\n'
    '    ]\n'
    '    interbank_rate_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"ir{i}", "Raw Value": value}\n'
    '            for i, value in enumerate(interbank_rate_values)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    interbank_rate_out = calculate_interbank_rate_outputs(interbank_rate_raw)\n'
    '    interbank_rate_rows = df_records(interbank_rate_out)\n'
    '    a28_spec = select_indicator_engine_spec(\n'
    '        "Unmapped interbank rate sheet", CONTROL_PANEL_TWENTY_SIXTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped interbank rate sheet", CONTROL_PANEL_TWENTY_SIXTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_interbank_rate_outputs\n'
    '    )\n'
    '    assert a28_spec.calculate is calculate_interbank_rate_outputs\n'
    '    assert a28_spec.input_start_col == "A" and a28_spec.input_end_col == "B"\n'
    '    assert a28_spec.output_start_col == "C" and a28_spec.output_end_col == "AL"\n'
    '    assert a28_spec.input_value_render_option == "UNFORMATTED_VALUE"\n'
    '    assert list(interbank_rate_out.columns) == INTERBANK_RATE_OUTPUT_COLUMNS\n'
    '    assert interbank_rate_out.shape[1] == len(INTERBANK_RATE_OUTPUT_COLUMNS) == 36\n'
    '    assert output_end_col_for_columns(INTERBANK_RATE_OUTPUT_COLUMNS) == "AL"\n'
    '    assert indicator_input_range("Interbank Rate", "B", "A") == "\'Interbank Rate\'!A2:B"\n'
    '    assert (\n'
    '        indicator_output_range("Interbank Rate", 75, "AL") == "\'Interbank Rate\'!C2:AL75"\n'
    '    )\n'
    '    for control_row in range(3, 40):\n'
    '        render_option = select_indicator_engine_spec(\n'
    '            f"Unmapped A{control_row}", f"A{control_row}"\n'
    '        ).input_value_render_option\n'
    '        assert render_option == (\n'
    '            "UNFORMATTED_VALUE" if control_row in (28, 29, 30) else None\n'
    '        )\n'
    '\n'
    '    # A28-only Google Sheets ROUND and PERCENTRANK.INC semantics.\n'
    '    assert _interbank_rate_sheets_round(1.005, 2) == 1.01\n'
    '    assert _interbank_rate_sheets_round(-1.005, 2) == -1.01\n'
    '    assert _interbank_rate_sheets_round(0.7965, 3) == 0.797\n'
    '    assert values_equivalent(\n'
    '        _interbank_rate_percentrank_inc_sheets([1, 2, 2, 4], 2), 1 / 3\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        _interbank_rate_percentrank_inc_sheets([1, 2, 2, 4], 3), 5 / 6\n'
    '    )\n'
    '    assert _interbank_rate_score_percentile(0.797) == 5.94\n'
    '    assert _interbank_rate_percent_display(0.797) == "79.70%"\n'
    '\n'
    '    # C:H retain full input precision and never round an intermediate moving average.\n'
    '    precision_values = [4.2711111111, 4.1811111111, 4.0588888889, 3.8827272727]\n'
    '    precision_out = df_records(\n'
    '        calculate_interbank_rate_outputs(\n'
    '            make_dataframe(\n'
    '                [\n'
    '                    {"Date": f"precision{i}", "Raw Value": value}\n'
    '                    for i, value in enumerate(precision_values)\n'
    '                ],\n'
    '                ["Date", "Raw Value"],\n'
    '            )\n'
    '        )\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        precision_out[2]["3M MA Rate"], sum(precision_values[:3]) / 3\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        precision_out[3]["1M Rate Change"],\n'
    '        sum(precision_values[1:4]) / 3 - sum(precision_values[:3]) / 3,\n'
    '    )\n'
    '\n'
    '    # Early regimes: blank F is zero, and F=0 belongs to the rising branch.\n'
    '    assert [interbank_rate_rows[i]["Interbank Rate Regime"] for i in (2, 3, 4, 5)] == [\n'
    '        "Moderate interbank rate and rising",\n'
    '        "Moderate interbank rate and rising",\n'
    '        "Low interbank rate but rising",\n'
    '        "Low and easing interbank-rate regime",\n'
    '    ]\n'
    '    assert (\n'
    '        _interbank_rate_level_regime(3, "")\n'
    '        == "Elevated and rising interbank-rate regime"\n'
    '    )\n'
    '    assert _interbank_rate_level_regime(1, 0) == "Moderate interbank rate and rising"\n'
    '    assert _interbank_rate_level_regime(0.999, 0) == "Low interbank rate but rising"\n'
    '\n'
    '    # Each percentile starts after 60 numeric observations of its own source metric.\n'
    '    for before, first, percentile_col in (\n'
    '        (60, 61, "Rate Level Percentile"),\n'
    '        (63, 64, "3M Change Percentile"),\n'
    '        (66, 67, "6M Change Percentile"),\n'
    '        (72, 73, "12M Change Percentile"),\n'
    '    ):\n'
    '        assert interbank_rate_rows[before][percentile_col] == ""\n'
    '        assert interbank_rate_rows[first][percentile_col] != ""\n'
    '    aug_1959 = interbank_rate_rows[61]\n'
    '    nov_1959 = interbank_rate_rows[64]\n'
    '    dec_1959 = interbank_rate_rows[65]\n'
    '    aug_1960 = interbank_rate_rows[73]\n'
    '    assert aug_1959["Sample Count"] == 60\n'
    '    assert (aug_1959["Rate Level Percentile"], aug_1959["Rate Level Score"]) == (\n'
    '        "100.00%",\n'
    '        10.0,\n'
    '    )\n'
    '    assert [\n'
    '        aug_1959[col]\n'
    '        for col in (\n'
    '            "3M Change Percentile",\n'
    '            "6M Change Percentile",\n'
    '            "12M Change Percentile",\n'
    '        )\n'
    '    ] == ["", "", ""]\n'
    '    assert aug_1959["Signal Validity"] == "Incomplete Score"\n'
    '    assert (nov_1959["3M Change Percentile"], nov_1959["3M Change Score"]) == (\n'
    '        "79.70%",\n'
    '        5.94,\n'
    '    )\n'
    '    assert (dec_1959["3M Change Percentile"], dec_1959["3M Change Score"]) == (\n'
    '        "73.30%",\n'
    '        4.66,\n'
    '    )\n'
    '    assert [\n'
    '        aug_1960[col]\n'
    '        for col in (\n'
    '            "Rate Level Percentile",\n'
    '            "3M Change Percentile",\n'
    '            "6M Change Percentile",\n'
    '            "12M Change Percentile",\n'
    '        )\n'
    '    ] == ["76.10%", "7.40%", "9.20%", "20.30%"]\n'
    '    assert [\n'
    '        aug_1960[col]\n'
    '        for col in (\n'
    '            "Rate Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [5.22, -8.52, -8.16, -5.94]\n'
    '    assert (\n'
    '        aug_1960["Interbank Rate Impulse Score"],\n'
    '        aug_1960["Coincident Score"],\n'
    '        aug_1960["Leading Score"],\n'
    '    ) == (-7.32, -2.03, -6.37)\n'
    '    for source_col, percentile_col in (\n'
    '        ("3M MA Rate", "Rate Level Percentile"),\n'
    '        ("3M Rate Change", "3M Change Percentile"),\n'
    '        ("6M Rate Change", "6M Change Percentile"),\n'
    '        ("12M Rate Change", "12M Change Percentile"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[source_col]\n'
    '            for row in interbank_rate_rows[:74]\n'
    '            if not is_blank(row[source_col])\n'
    '        ]\n'
    '        expected_rank = _interbank_rate_sheets_round(\n'
    '            _interbank_rate_percentrank_inc_sheets(history, aug_1960[source_col]), 3\n'
    '        )\n'
    '        assert aug_1960[percentile_col] == _interbank_rate_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '\n'
    '    # Inclusive shock boundaries, including floating-point noise from subtraction.\n'
    '    assert _interbank_rate_shock_label(0.30) == "1M interbank-rate upside shock"\n'
    '    assert _interbank_rate_shock_label(-0.30) == "1M interbank-rate downside shock"\n'
    '    assert [\n'
    '        _interbank_rate_shock_score(value) for value in (0.30, 0.20, -0.20, -0.30)\n'
    '    ] == [6, 4, -4, -6]\n'
    '    aug_1958 = interbank_rate_rows[49]\n'
    '    assert aug_1958["1M Interbank Rate Shock Flag"] == "1M interbank-rate upside shock"\n'
    '    assert aug_1958["Interbank Rate Shock Score"] == 6\n'
    '    assert aug_1959["1M Interbank Rate Shock Flag"] == "No 1M interbank-rate shock"\n'
    '    assert aug_1959["Interbank Rate Shock Score"] == 4\n'
    '\n'
    '    # Supplied 1976, 1977, and 2020 formula/label regressions use visible operands.\n'
    '    dec_1976_warning = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum((0.30, -5), (0.25, -4), (0.30, 0), (0.15, 0)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert _interbank_rate_shock_score(-0.20) == -4 and dec_1976_warning == -2.50\n'
    '    j, l, n, p, u = (2.24, -6.08, -5.64, -3.44, -5)\n'
    '    jan_1977_t = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum((0.05, j), (0.45, l), (0.35, n), (0.15, p)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    jan_1977_y = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum((0.45, j), (0.30, l), (0.15, n), (0.10, p)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    jan_1977_z = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum(\n'
    '                (0.10, j),\n'
    '                (0.35, l),\n'
    '                (0.25, n),\n'
    '                (0.15, p),\n'
    '                (0.10, jan_1977_t),\n'
    '                (0.05, u),\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (jan_1977_t, jan_1977_y, jan_1977_z) == (-5.11, -2.01, -4.59)\n'
    '    assert (\n'
    '        _interbank_rate_policy_bias_label(jan_1977_y)\n'
    '        == "Mild dovish policy-rate / easier money-market bias"\n'
    '    )\n'
    '    assert (\n'
    '        _interbank_rate_pressure_regime(jan_1977_y) == "Interbank-rate pressure easing"\n'
    '    )\n'
    '    assert (\n'
    '        _interbank_rate_momentum_label(jan_1977_y, jan_1977_z)\n'
    '        == "Interbank-rate pressure easing"\n'
    '    )\n'
    '    assert (\n'
    '        _interbank_rate_signal_interpretation(jan_1977_z, -1.5)\n'
    '        == "Moderately negative interbank-rate/easing signal"\n'
    '    )\n'
    '    j, l, n, p, u, v, w, x = (-6.48, -5.96, -6.64, -5.74, -4, -6, -4, 0)\n'
    '    mar_2020_t = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum((0.05, j), (0.45, l), (0.35, n), (0.15, p)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    mar_2020_y = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum((0.45, j), (0.30, l), (0.15, n), (0.10, p)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    mar_2020_z = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum(\n'
    '                (0.10, j),\n'
    '                (0.35, l),\n'
    '                (0.25, n),\n'
    '                (0.15, p),\n'
    '                (0.10, mar_2020_t),\n'
    '                (0.05, u),\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    mar_2020_aa = _interbank_rate_sheets_round(\n'
    '        clamp(\n'
    '            _interbank_rate_weighted_sum((0.30, u), (0.25, v), (0.30, w), (0.15, x)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (mar_2020_t, mar_2020_y, mar_2020_z, mar_2020_aa) == (\n'
    '        -6.19,\n'
    '        -6.27,\n'
    '        -6.07,\n'
    '        -3.90,\n'
    '    )\n'
    '    assert _interbank_rate_shock_label(-0.30) == "1M interbank-rate downside shock"\n'
    '    assert _interbank_rate_shock_score(-0.30) == -6\n'
    '    assert (\n'
    '        _interbank_rate_warning_label(mar_2020_aa)\n'
    '        == "Moderate interbank-rate easing warning"\n'
    '    )\n'
    '\n'
    '    # Existing AH:AK z-score and AL extreme-flag formulas are unchanged.\n'
    '    c_values = [row["3M MA Rate"] for row in interbank_rate_rows]\n'
    '    f_values = [row["3M Rate Change"] for row in interbank_rate_rows]\n'
    '    g_values = [row["6M Rate Change"] for row in interbank_rate_rows]\n'
    '    h_values = [row["12M Rate Change"] for row in interbank_rate_rows]\n'
    '    expected_interbank_zs = (\n'
    '        _rounded_expanding_zscores(\n'
    '            c_values, [row["Sample Count"] for row in interbank_rate_rows], 60\n'
    '        )[73],\n'
    '        _rounded_expanding_zscores(f_values, _metric_counts(f_values), 60)[73],\n'
    '        _rounded_expanding_zscores(g_values, _metric_counts(g_values), 60)[73],\n'
    '        _rounded_expanding_zscores(h_values, _metric_counts(h_values), 60)[73],\n'
    '    )\n'
    '    actual_interbank_zs = tuple(\n'
    '        aug_1960[col]\n'
    '        for col in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    )\n'
    '    assert actual_interbank_zs == expected_interbank_zs\n'
    '    assert (\n'
    '        aug_1960["Extreme Flag"]\n'
    '        == _interbank_rate_outlier_label(actual_interbank_zs)\n'
    '        == "Normal range"\n'
    '    )\n'
    '    assert _interbank_rate_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '\n'
    '    class FakeInterbankVerifyWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '    fake_interbank_verify = FakeInterbankVerifyWorksheet(\n'
    '        dataframe_to_sheet_rows(interbank_rate_out, INTERBANK_RATE_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_interbank_verify,\n'
    '        interbank_rate_out,\n'
    '        75,\n'
    '        INTERBANK_RATE_OUTPUT_COLUMNS,\n'
    '        "AL",\n'
    '        "C",\n'
    '    )\n'
    '    assert fake_interbank_verify.get_calls == [("C2:AL75", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            interbank_rate_out, 75, "C", 2, INTERBANK_RATE_OUTPUT_COLUMNS, "AL"\n'
    '        )\n'
    '        == "C2:AL75"\n'
    '    )\n'
    '\n'
    '    class FakeInterbankReadWorkbook:\n'
    '        def __init__(self, worksheet: FakeInterbankVerifyWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeInterbankVerifyWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_interbank_read = FakeInterbankVerifyWorksheet(\n'
    '        [[f"ir{i}", value] for i, value in enumerate(interbank_rate_values)]\n'
    '    )\n'
    '    fake_a28_result = process_indicator(\n'
    '        FakeInterbankReadWorkbook(fake_interbank_read),\n'
    '        BatchIndicatorSpec("Interbank Rate", "A28"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a28_result.ok and fake_a28_result.output_range == "C2:AL75"\n'
    '    assert fake_interbank_read.get_calls == [("A2:B", "UNFORMATTED_VALUE")]\n'
    '    all_specs_with_a28 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 28)] + [["Interbank Rate"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Interbank Rate", "A28") in all_specs_with_a28\n'
    '    assert df_records(calculate_real_yield_outputs(real_yield_raw)) == real_yield_rows\n'
    '    assert df_records(calculate_two_year_yield_outputs(two_year_raw)) == two_year_rows\n'
    '    assert df_records(calculate_ten_year_yield_outputs(ten_year_raw)) == ten_year_rows\n'
    '    assert (\n'
    '        df_records(calculate_yield_curve_outputs(yield_curve_raw)) == yield_curve_rows\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_term_premium_outputs(term_premium_raw))\n'
    '        == term_premium_rows\n'
    '    )\n'
    '    # A29 uses a fixed protected-source fixture covering January 2003 through\n'
    '    # February 2009, including every first-availability percentile gate.\n'
    '    central_dates = [\n'
    '        37652,\n'
    '        37680,\n'
    '        37711,\n'
    '        37741,\n'
    '        37772,\n'
    '        37802,\n'
    '        37833,\n'
    '        37864,\n'
    '        37894,\n'
    '        37925,\n'
    '        37955,\n'
    '        37986,\n'
    '        38017,\n'
    '        38046,\n'
    '        38077,\n'
    '        38107,\n'
    '        38138,\n'
    '        38168,\n'
    '        38199,\n'
    '        38230,\n'
    '        38260,\n'
    '        38291,\n'
    '        38321,\n'
    '        38352,\n'
    '        38383,\n'
    '        38411,\n'
    '        38442,\n'
    '        38472,\n'
    '        38503,\n'
    '        38533,\n'
    '        38564,\n'
    '        38595,\n'
    '        38625,\n'
    '        38656,\n'
    '        38686,\n'
    '        38717,\n'
    '        38748,\n'
    '        38776,\n'
    '        38807,\n'
    '        38837,\n'
    '        38868,\n'
    '        38898,\n'
    '        38929,\n'
    '        38960,\n'
    '        38990,\n'
    '        39021,\n'
    '        39051,\n'
    '        39082,\n'
    '        39113,\n'
    '        39141,\n'
    '        39172,\n'
    '        39202,\n'
    '        39233,\n'
    '        39263,\n'
    '        39294,\n'
    '        39325,\n'
    '        39355,\n'
    '        39386,\n'
    '        39416,\n'
    '        39447,\n'
    '        39478,\n'
    '        39507,\n'
    '        39538,\n'
    '        39568,\n'
    '        39599,\n'
    '        39629,\n'
    '        39660,\n'
    '        39691,\n'
    '        39721,\n'
    '        39752,\n'
    '        39782,\n'
    '        39813,\n'
    '        39844,\n'
    '        39872,\n'
    '    ]\n'
    '    central_assets = [\n'
    '        724718,\n'
    '        721326,\n'
    '        723651,\n'
    '        736926,\n'
    '        738358,\n'
    '        738712,\n'
    '        741656,\n'
    '        740217,\n'
    '        743207,\n'
    '        747068,\n'
    '        747801,\n'
    '        759810,\n'
    '        759938,\n'
    '        753644,\n'
    '        755538,\n'
    '        762182,\n'
    '        762914,\n'
    '        770590,\n'
    '        776095,\n'
    '        771096,\n'
    '        786291,\n'
    '        788388,\n'
    '        798768,\n'
    '        808973,\n'
    '        807095,\n'
    '        800557,\n'
    '        802472,\n'
    '        803656,\n'
    '        805624,\n'
    '        812643,\n'
    '        815370,\n'
    '        815142,\n'
    '        820697,\n'
    '        819207,\n'
    '        826202,\n'
    '        840054,\n'
    '        836539,\n'
    '        833804,\n'
    '        838656,\n'
    '        838094,\n'
    '        842451,\n'
    '        846038,\n'
    '        846874,\n'
    '        844411,\n'
    '        844411,\n'
    '        849621,\n'
    '        858303,\n'
    '        862757,\n'
    '        861259,\n'
    '        867225,\n'
    '        868371,\n'
    '        867195,\n'
    '        875321,\n'
    '        867222,\n'
    '        870706,\n'
    '        867259,\n'
    '        878776,\n'
    '        877941,\n'
    '        886427,\n'
    '        885790,\n'
    '        894808,\n'
    '        881700,\n'
    '        890096,\n'
    '        886511,\n'
    '        892986,\n'
    '        895392,\n'
    '        905993,\n'
    '        906234,\n'
    '        1009474,\n'
    '        1727615,\n'
    '        2145078,\n'
    '        2222737,\n'
    '        2033837,\n'
    '        1881629,\n'
    '    ]\n'
    '    central_account = [\n'
    '        5650,\n'
    '        4637,\n'
    '        4953,\n'
    '        8352,\n'
    '        6421,\n'
    '        6387,\n'
    '        6004,\n'
    '        6024,\n'
    '        5912,\n'
    '        5102,\n'
    '        4945,\n'
    '        5477,\n'
    '        5825,\n'
    '        5297,\n'
    '        5163,\n'
    '        5472,\n'
    '        5051,\n'
    '        5422,\n'
    '        4403,\n'
    '        4508,\n'
    '        5596,\n'
    '        5102,\n'
    '        4741,\n'
    '        4979,\n'
    '        5000,\n'
    '        4835,\n'
    '        4825,\n'
    '        4979,\n'
    '        5054,\n'
    '        5557,\n'
    '        4603,\n'
    '        5283,\n'
    '        4990,\n'
    '        4812,\n'
    '        4620,\n'
    '        4900,\n'
    '        4771,\n'
    '        4632,\n'
    '        4431,\n'
    '        4963,\n'
    '        4605,\n'
    '        5071,\n'
    '        5162,\n'
    '        4537,\n'
    '        4716,\n'
    '        5303,\n'
    '        4677,\n'
    '        4732,\n'
    '        5631,\n'
    '        5214,\n'
    '        4764,\n'
    '        6804,\n'
    '        8630,\n'
    '        5050,\n'
    '        4516,\n'
    '        4960,\n'
    '        4769,\n'
    '        4792,\n'
    '        5119,\n'
    '        4632,\n'
    '        5653,\n'
    '        4862,\n'
    '        5088,\n'
    '        4918,\n'
    '        4646,\n'
    '        4507,\n'
    '        4704,\n'
    '        4920,\n'
    '        5426,\n'
    '        21983,\n'
    '        39362,\n'
    '        88723,\n'
    '        44820,\n'
    '        28396,\n'
    '    ]\n'
    '    central_rrp = [\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        2.25,\n'
    '        "",\n'
    '        2.063,\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        3.375,\n'
    '        1.75,\n'
    '        2.375,\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        5.875,\n'
    '        3.625,\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        6.083,\n'
    '        3.75,\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        "",\n'
    '        24.333,\n'
    '        24.722,\n'
    '        24.167,\n'
    '        25,\n'
    '        "",\n'
    '        "",\n'
    '    ]\n'
    '    central_rows_input = [\n'
    '        {\n'
    '            "Date": serial,\n'
    '            "Central Bank Total Assets": b,\n'
    '            "Cental Bank General Account": c,\n'
    '            "Central Bank RRP": d,\n'
    '        }\n'
    '        for serial, b, c, d in zip(\n'
    '            central_dates, central_assets, central_account, central_rrp\n'
    '        )\n'
    '    ]\n'
    '    central_raw = make_dataframe(\n'
    '        central_rows_input,\n'
    '        [\n'
    '            "Date",\n'
    '            "Central Bank Total Assets",\n'
    '            "Cental Bank General Account",\n'
    '            "Central Bank RRP",\n'
    '        ],\n'
    '    )\n'
    '    central_out = calculate_central_bank_liquidity_outputs(central_raw)\n'
    '    central_rows = df_records(central_out)\n'
    '    central_spec = select_indicator_engine_spec(\n'
    '        "Unmapped central liquidity sheet", CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped central liquidity sheet",\n'
    '            CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_central_bank_liquidity_outputs\n'
    '    )\n'
    '    assert central_spec.calculate is calculate_central_bank_liquidity_outputs\n'
    '    assert central_spec.input_start_col == "A" and central_spec.input_end_col == "D"\n'
    '    assert central_spec.output_start_col == "E" and central_spec.output_end_col == "AS"\n'
    '    assert central_spec.input_value_render_option == "UNFORMATTED_VALUE"\n'
    '    assert list(central_out.columns) == CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS\n'
    '    assert central_out.shape[1] == len(CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS) == 41\n'
    '    assert output_end_col_for_columns(CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS) == "AS"\n'
    '    assert (\n'
    '        indicator_input_range("Central Bank Liquidity", "D")\n'
    '        == "\'Central Bank Liquidity\'!A2:D"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Central Bank Liquidity", 75, "AS", "E")\n'
    '        == "\'Central Bank Liquidity\'!E2:AS75"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec("Unmapped A28", "A28").input_value_render_option\n'
    '        == "UNFORMATTED_VALUE"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec("Unmapped A30", "A30").input_value_render_option\n'
    '        == "UNFORMATTED_VALUE"\n'
    '    )\n'
    '\n'
    '    # Blank C/D are numeric zero in B-C-D; the misspelled source header is intentional.\n'
    '    blank_component_raw = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Date": "2003-01-31",\n'
    '                "Central Bank Total Assets": 724718,\n'
    '                "Cental Bank General Account": 5650,\n'
    '                "Central Bank RRP": "",\n'
    '            },\n'
    '            {\n'
    '                "Date": "2003-04-30",\n'
    '                "Central Bank Total Assets": 736926,\n'
    '                "Cental Bank General Account": 8352,\n'
    '                "Central Bank RRP": 2.25,\n'
    '            },\n'
    '        ],\n'
    '        [\n'
    '            "Date",\n'
    '            "Central Bank Total Assets",\n'
    '            "Cental Bank General Account",\n'
    '            "Central Bank RRP",\n'
    '        ],\n'
    '    )\n'
    '    blank_component_rows = df_records(\n'
    '        calculate_central_bank_liquidity_outputs(blank_component_raw)\n'
    '    )\n'
    '    assert blank_component_rows[0]["Central Bank Net Liquidity"] == 719068\n'
    '    assert blank_component_rows[1]["Central Bank Net Liquidity"] == 728571.75\n'
    '    bad_liquidity = calculate_central_bank_liquidity_outputs(\n'
    '        make_dataframe(\n'
    '            [\n'
    '                {\n'
    '                    "Date": "2003-01-31",\n'
    '                    "Central Bank Total Assets": "bad",\n'
    '                    "Cental Bank General Account": 1,\n'
    '                    "Central Bank RRP": 1,\n'
    '                }\n'
    '            ],\n'
    '            [\n'
    '                "Date",\n'
    '                "Central Bank Total Assets",\n'
    '                "Cental Bank General Account",\n'
    '                "Central Bank RRP",\n'
    '            ],\n'
    '        )\n'
    '    )\n'
    '    assert df_records(bad_liquidity)[0]["Central Bank Net Liquidity"] == ""\n'
    '\n'
    '    # Monthly data has one observation per month, so F equals E from row 2 onward.\n'
    '    assert all(\n'
    '        values_equivalent(\n'
    '            row["Monthly Net Liquidity"], row["Central Bank Net Liquidity"]\n'
    '        )\n'
    '        for row in central_rows\n'
    '    )\n'
    '    assert [central_rows[i]["Sample Count"] for i in (0, 1, 2, 61)] == [0, 0, 1, 60]\n'
    '    assert (\n'
    '        central_rows[0]["3M MA Net Liquidity"] == ""\n'
    '        and central_rows[1]["3M MA Net Liquidity"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[2]["3M MA Net Liquidity"],\n'
    '        _central_bank_liquidity_mean_sheets(\n'
    '            [central_rows[i]["Monthly Net Liquidity"] for i in range(3)]\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[3]["1M Net Liquidity Change"],\n'
    '        central_rows[3]["3M MA Net Liquidity"] - central_rows[2]["3M MA Net Liquidity"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[5]["3M Net Liquidity Change"],\n'
    '        central_rows[5]["3M MA Net Liquidity"] - central_rows[2]["3M MA Net Liquidity"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[8]["6M Net Liquidity Change"],\n'
    '        central_rows[8]["3M MA Net Liquidity"] - central_rows[2]["3M MA Net Liquidity"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[14]["12M Net Liquidity Change"],\n'
    '        central_rows[14]["3M MA Net Liquidity"]\n'
    '        - central_rows[2]["3M MA Net Liquidity"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[5]["3M Net Liquidity Growth"],\n'
    '        central_rows[5]["3M MA Net Liquidity"] / central_rows[2]["3M MA Net Liquidity"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[8]["6M Net Liquidity Growth"],\n'
    '        central_rows[8]["3M MA Net Liquidity"] / central_rows[2]["3M MA Net Liquidity"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[14]["12M Net Liquidity Growth"],\n'
    '        central_rows[14]["3M MA Net Liquidity"] / central_rows[2]["3M MA Net Liquidity"]\n'
    '        - 1,\n'
    '    )\n'
    '    zero_growth_raw = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Date": (date(2024, 1, 1) + timedelta(days=31 * i)).isoformat(),\n'
    '                "Raw Value": 0,\n'
    '                "Source C": 0,\n'
    '                "Source D": 0,\n'
    '            }\n'
    '            for i in range(15)\n'
    '        ],\n'
    '        ["Date", "Raw Value", "Source C", "Source D"],\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_central_bank_liquidity_outputs(zero_growth_raw))[5][\n'
    '            "3M Net Liquidity Growth"\n'
    '        ]\n'
    '        == ""\n'
    '    )\n'
    '\n'
    '    # P/R/T/V each rank their own source metric after 60 numeric observations.\n'
    '    for before, first, percentile_col in (\n'
    '        (60, 61, "Liquidity Level Percentile"),\n'
    '        (63, 64, "3M Change Percentile"),\n'
    '        (66, 67, "6M Change Percentile"),\n'
    '        (72, 73, "12M Change Percentile"),\n'
    '    ):\n'
    '        assert central_rows[before][percentile_col] == ""\n'
    '        assert central_rows[first][percentile_col] != ""\n'
    '    feb_2008 = central_rows[61]\n'
    '    feb_2009 = central_rows[73]\n'
    '    assert (feb_2008["Sample Count"], feb_2008["3M MA Net Liquidity"]) == (\n'
    '        60,\n'
    '        882383.6666666666,\n'
    '    )\n'
    '    assert (\n'
    '        feb_2008["Liquidity Level Percentile"],\n'
    '        feb_2008["Liquidity Level Score"],\n'
    '        feb_2008["Level Z-Score"],\n'
    '    ) == ("98.30%", 9.66, 1.53)\n'
    '    assert (\n'
    '        feb_2008["Reliability"] == "Medium reliability"\n'
    '        and feb_2008["Signal Validity"] == "Incomplete Score"\n'
    '    )\n'
    '    for source_col, percentile_col in (\n'
    '        ("3M MA Net Liquidity", "Liquidity Level Percentile"),\n'
    '        ("3M Net Liquidity Change", "3M Change Percentile"),\n'
    '        ("6M Net Liquidity Change", "6M Change Percentile"),\n'
    '        ("12M Net Liquidity Change", "12M Change Percentile"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[source_col] for row in central_rows if not is_blank(row[source_col])\n'
    '        ]\n'
    '        expected_rank = _central_bank_liquidity_sheets_round(\n'
    '            _central_bank_liquidity_percentrank_inc_sheets(\n'
    '                history, feb_2009[source_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert feb_2009[percentile_col] == _central_bank_liquidity_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '    assert [\n'
    '        feb_2009[col]\n'
    '        for col in (\n'
    '            "Liquidity Level Percentile",\n'
    '            "3M Change Percentile",\n'
    '            "6M Change Percentile",\n'
    '            "12M Change Percentile",\n'
    '        )\n'
    '    ] == ["98.60%", "95.60%", "96.90%", "98.30%"]\n'
    '    assert [\n'
    '        feb_2009[col]\n'
    '        for col in (\n'
    '            "Liquidity Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [9.72, 9.12, 9.38, 9.66]\n'
    '\n'
    '    # Exact branch order, shock thresholds, components, and rounded composites.\n'
    '    assert _central_bank_liquidity_regime(1, 1, 1) == "Liquidity expansion regime"\n'
    '    assert (\n'
    '        _central_bank_liquidity_regime(1, -1, -1)\n'
    '        == "Positive liquidity but contracting"\n'
    '    )\n'
    '    assert _central_bank_liquidity_regime(-1, 1, 1) == "Low liquidity but recovering"\n'
    '    assert _central_bank_liquidity_regime(-1, -1, -1) == "Liquidity contraction regime"\n'
    '    assert (\n'
    '        _central_bank_liquidity_regime(1, 1, -1)\n'
    '        == "Early liquidity recovery / mixed impulse"\n'
    '    )\n'
    '    assert (\n'
    '        _central_bank_liquidity_regime(1, -1, 1)\n'
    '        == "Early liquidity deterioration / mixed impulse"\n'
    '    )\n'
    '    assert _central_bank_liquidity_regime(1, "", "") == "Neutral/mixed liquidity regime"\n'
    '    assert [\n'
    '        _central_bank_liquidity_shock_score(value)\n'
    '        for value in (250000, 100000, -100000, -250000)\n'
    '    ] == [6, 4, -4, -6]\n'
    '    assert (\n'
    '        _central_bank_liquidity_shock_label(250000)\n'
    '        == "1M major liquidity injection shock"\n'
    '    )\n'
    '    assert (\n'
    '        _central_bank_liquidity_shock_label(-250000) == "1M major liquidity drain shock"\n'
    '    )\n'
    '    assert (\n'
    '        _central_bank_liquidity_recovery_score(1, 1) == 5\n'
    '        and _central_bank_liquidity_recovery_score(-1, 1) == -3\n'
    '    )\n'
    '    assert _central_bank_liquidity_withdrawal_risk_score(1.2, -0.1, -0.1, -1, 1) == -4\n'
    '    assert _central_bank_liquidity_withdrawal_risk_score(-1.2, 0.1, 0.1, 1, 1) == 6\n'
    '    assert _central_bank_liquidity_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    q, s_score, u, w = (9.72, 9.12, 9.38, 9.66)\n'
    '    expected_aa = _central_bank_liquidity_sheets_round(\n'
    '        clamp(\n'
    '            _central_bank_liquidity_weighted_sum(\n'
    '                (0.45, s_score), (0.35, u), (0.15, w), (0.05, q)\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_af = _central_bank_liquidity_sheets_round(\n'
    '        clamp(\n'
    '            _central_bank_liquidity_weighted_sum(\n'
    '                (0.40, q), (0.30, s_score), (0.20, u), (0.10, w)\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_ag = _central_bank_liquidity_sheets_round(\n'
    '        clamp(\n'
    '            _central_bank_liquidity_weighted_sum(\n'
    '                (0.15, q),\n'
    '                (0.25, s_score),\n'
    '                (0.25, u),\n'
    '                (0.20, w),\n'
    '                (0.10, expected_aa),\n'
    '                (0.05, 5),\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_ah = _central_bank_liquidity_sheets_round(\n'
    '        clamp(\n'
    '            _central_bank_liquidity_weighted_sum(\n'
    '                (0.30, 5), (0.25, 0), (0.30, 4), (0.15, 6)\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (expected_aa, expected_af, expected_ag, expected_ah) == (\n'
    '        9.32,\n'
    '        9.47,\n'
    '        9.20,\n'
    '        3.60,\n'
    '    )\n'
    '    assert [\n'
    '        feb_2009[col]\n'
    '        for col in (\n'
    '            "Liquidity Impulse Score",\n'
    '            "Liquidity Cycle Turning Point Score",\n'
    '            "Liquidity Shock Score",\n'
    '            "Liquidity Withdrawal Risk Score",\n'
    '            "Liquidity Extreme Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '        )\n'
    '    ] == [9.32, 5, 0, 4, 6, 9.47, 9.20, 3.60]\n'
    '    assert (\n'
    '        feb_2009["Risk Asset Liquidity Bias"]\n'
    '        == "Strongly supportive for risk assets / liquidity-sensitive assets"\n'
    '    )\n'
    '    assert feb_2009["State"] == feb_2009["Regime"] == "Central bank liquidity improving"\n'
    '    assert (\n'
    '        feb_2009["Signal Comment"]\n'
    '        == "Strong positive liquidity impulse with supportive risk-asset liquidity profile"\n'
    '    )\n'
    '    assert (\n'
    '        feb_2009["Reliability"] == "Medium reliability"\n'
    '        and feb_2009["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert feb_2009["Warning Flag"] == "Moderate upside liquidity-support warning"\n'
    '    assert [\n'
    '        feb_2009[col]\n'
    '        for col in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [4.19, 1.76, 4.08, 3.90]\n'
    '    assert feb_2009["Extreme Flag"] == "Extreme liquidity support"\n'
    '\n'
    '    # Initial formula chain and sample-standard-deviation z-score implementation.\n'
    '    assert (\n'
    '        central_rows[0]["Central Bank Net Liquidity"]\n'
    '        == central_rows[0]["Monthly Net Liquidity"]\n'
    '        == 719068\n'
    '    )\n'
    '    assert (\n'
    '        central_rows[0]["Sample Count"] == 0\n'
    '        and central_rows[0]["Reliability"] == "Insufficient history"\n'
    '        and central_rows[0]["Signal Validity"] == "Low Sample"\n'
    '    )\n'
    '    assert (\n'
    '        central_rows[2]["Central Bank Net Liquidity"]\n'
    '        == central_rows[2]["Monthly Net Liquidity"]\n'
    '        == 718698\n'
    '    )\n'
    '    assert central_rows[2]["Sample Count"] == 1 and values_equivalent(\n'
    '        central_rows[2]["3M MA Net Liquidity"], 718151.6667, 1e-4\n'
    '    )\n'
    '    assert (\n'
    '        central_rows[3]["Central Bank Net Liquidity"]\n'
    '        == central_rows[3]["Monthly Net Liquidity"]\n'
    '        == 728571.75\n'
    '    )\n'
    '    assert central_rows[3]["Sample Count"] == 2 and values_equivalent(\n'
    '        central_rows[3]["3M MA Net Liquidity"], 721319.5833, 1e-4\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        central_rows[3]["1M Net Liquidity Change"], 3167.916667, 1e-6\n'
    '    )\n'
    '    assert (\n'
    '        central_rows[3]["Liquidity Shock Flag"] == "No major liquidity shock"\n'
    '        and central_rows[3]["Liquidity Shock Score"] == 0\n'
    '    )\n'
    '    h_values = [row["3M MA Net Liquidity"] for row in central_rows]\n'
    '    j_values = [row["3M Net Liquidity Change"] for row in central_rows]\n'
    '    k_values = [row["6M Net Liquidity Change"] for row in central_rows]\n'
    '    l_values = [row["12M Net Liquidity Change"] for row in central_rows]\n'
    '    expected_zscores = (\n'
    '        _central_bank_liquidity_rounded_expanding_zscores(\n'
    '            h_values, _metric_counts(h_values)\n'
    '        )[73],\n'
    '        _central_bank_liquidity_rounded_expanding_zscores(\n'
    '            j_values, _metric_counts(j_values)\n'
    '        )[73],\n'
    '        _central_bank_liquidity_rounded_expanding_zscores(\n'
    '            k_values, _metric_counts(k_values)\n'
    '        )[73],\n'
    '        _central_bank_liquidity_rounded_expanding_zscores(\n'
    '            l_values, _metric_counts(l_values)\n'
    '        )[73],\n'
    '    )\n'
    '    assert expected_zscores == (4.19, 1.76, 4.08, 3.90)\n'
    '\n'
    '    class FakeCentralBankWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    fake_central_write = FakeCentralBankWorksheet([])\n'
    '    assert (\n'
    '        write_indicator_outputs(\n'
    '            fake_central_write,\n'
    '            central_out,\n'
    '            75,\n'
    '            CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS,\n'
    '            "AS",\n'
    '            "E",\n'
    '        )\n'
    '        == "E2:AS75"\n'
    '    )\n'
    '    assert [(call[0], call[2]) for call in fake_central_write.update_calls] == [\n'
    '        ("E2:AS75", "RAW")\n'
    '    ]\n'
    '    fake_central_verify = FakeCentralBankWorksheet(\n'
    '        dataframe_to_sheet_rows(central_out, CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_central_verify,\n'
    '        central_out,\n'
    '        75,\n'
    '        CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS,\n'
    '        "AS",\n'
    '        "E",\n'
    '    )\n'
    '    assert fake_central_verify.get_calls == [("E2:AS75", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            central_out, 75, "E", 2, CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS, "AS"\n'
    '        )\n'
    '        == "E2:AS75"\n'
    '    )\n'
    '\n'
    '    class FakeCentralBankWorkbook:\n'
    '        def __init__(self, worksheet: FakeCentralBankWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeCentralBankWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_central_input = [\n'
    '        [a, b, c, d]\n'
    '        for a, b, c, d in zip(\n'
    '            central_dates, central_assets, central_account, central_rrp\n'
    '        )\n'
    '    ]\n'
    '    fake_central_read = FakeCentralBankWorksheet(fake_central_input)\n'
    '    fake_a29_result = process_indicator(\n'
    '        FakeCentralBankWorkbook(fake_central_read),\n'
    '        BatchIndicatorSpec("Central Bank Liquidity", "A29"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a29_result.ok and fake_a29_result.output_range == "E2:AS75"\n'
    '    assert fake_central_read.get_calls == [("A2:D", "UNFORMATTED_VALUE")]\n'
    '    all_specs_with_a29 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 29)] + [["Central Bank Liquidity"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Central Bank Liquidity", "A29") in all_specs_with_a29\n'
    '    assert (\n'
    '        df_records(calculate_interbank_rate_outputs(interbank_rate_raw))\n'
    '        == interbank_rate_rows\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_central_bank_liquidity_outputs(central_raw))\n'
    '        == central_rows\n'
    '    )\n'
    '    # A30 fixed protected-source fixture: January 2003 through February 2009.\n'
    '    reserves_dates = list(central_dates)\n'
    '    reserves_values = [\n'
    '        9522,\n'
    '        9644,\n'
    '        9700,\n'
    '        11183,\n'
    '        11446,\n'
    '        11420,\n'
    '        11804,\n'
    '        13865,\n'
    '        12128,\n'
    '        12704,\n'
    '        11484,\n'
    '        11013,\n'
    '        11036,\n'
    '        10522,\n'
    '        12623,\n'
    '        14151,\n'
    '        14116,\n'
    '        13654,\n'
    '        12187,\n'
    '        12015,\n'
    '        13839,\n'
    '        11851,\n'
    '        12084,\n'
    '        11429,\n'
    '        13198,\n'
    '        13196,\n'
    '        12796,\n'
    '        13020,\n'
    '        11892,\n'
    '        11190,\n'
    '        10595,\n'
    '        11859,\n'
    '        11834,\n'
    '        10267,\n'
    '        10062,\n'
    '        9605,\n'
    '        10085,\n'
    '        9466,\n'
    '        9830,\n'
    '        10137,\n'
    '        10572,\n'
    '        9850,\n'
    '        9116,\n'
    '        9741,\n'
    '        9212,\n'
    '        8207,\n'
    '        8224,\n'
    '        7846,\n'
    '        9374,\n'
    '        7616,\n'
    '        7396,\n'
    '        8185,\n'
    '        8818,\n'
    '        8750,\n'
    '        7913,\n'
    '        11426,\n'
    '        8492,\n'
    '        8464,\n'
    '        8263,\n'
    '        7191,\n'
    '        9537,\n'
    '        8278,\n'
    '        9839,\n'
    '        9484,\n'
    '        11060,\n'
    '        9194,\n'
    '        11377,\n'
    '        10443,\n'
    '        43106,\n'
    '        225998,\n'
    '        574333,\n'
    '        768928,\n'
    '        840840,\n'
    '        661573,\n'
    '    ]\n'
    '    reserves_rows_input = [\n'
    '        {"Date": serial, "Banking System Reserves": value}\n'
    '        for serial, value in zip(reserves_dates, reserves_values)\n'
    '    ]\n'
    '    reserves_raw = make_dataframe(\n'
    '        reserves_rows_input, ["Date", "Banking System Reserves"]\n'
    '    )\n'
    '    reserves_out = calculate_banking_reserves_outputs(reserves_raw)\n'
    '    reserves_rows = df_records(reserves_out)\n'
    '    reserves_spec = select_indicator_engine_spec(\n'
    '        "Unmapped banking reserves sheet", CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped banking reserves sheet",\n'
    '            CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_banking_reserves_outputs\n'
    '    )\n'
    '    assert reserves_spec.calculate is calculate_banking_reserves_outputs\n'
    '    assert reserves_spec.input_start_col == "A" and reserves_spec.input_end_col == "B"\n'
    '    assert (\n'
    '        reserves_spec.output_start_col == "C" and reserves_spec.output_end_col == "AQ"\n'
    '    )\n'
    '    assert reserves_spec.input_value_render_option == "UNFORMATTED_VALUE"\n'
    '    assert list(reserves_out.columns) == BANKING_RESERVES_OUTPUT_COLUMNS\n'
    '    assert reserves_out.shape[1] == len(BANKING_RESERVES_OUTPUT_COLUMNS) == 41\n'
    '    assert output_end_col_for_columns(BANKING_RESERVES_OUTPUT_COLUMNS) == "AQ"\n'
    '    assert indicator_input_range("Banking Reserves") == "\'Banking Reserves\'!A2:B"\n'
    '    assert (\n'
    '        indicator_output_range("Banking Reserves", 75, "AQ")\n'
    '        == "\'Banking Reserves\'!C2:AQ75"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec("Unmapped A29", "A29").input_value_render_option\n'
    '        == "UNFORMATTED_VALUE"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec("Unmapped A31", "A31").input_value_render_option\n'
    '        is None\n'
    '    )\n'
    '\n'
    '    # One source row per month means C equals B; D/E and changes use physical rows.\n'
    '    assert all(\n'
    '        values_equivalent(row["Monthly Average Reserves"], value)\n'
    '        for row, value in zip(reserves_rows, reserves_values)\n'
    '    )\n'
    '    assert [reserves_rows[i]["Sample Count"] for i in (0, 1, 2, 61)] == [0, 0, 1, 60]\n'
    '    assert (\n'
    '        reserves_rows[0]["3M MA Reserves"] == ""\n'
    '        and reserves_rows[1]["3M MA Reserves"] == ""\n'
    '    )\n'
    '    assert reserves_rows[2]["3M MA Reserves"] == 9622\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[3]["1M Reserve Change"],\n'
    '        reserves_rows[3]["3M MA Reserves"] - reserves_rows[2]["3M MA Reserves"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[5]["3M Reserve Change"],\n'
    '        reserves_rows[5]["3M MA Reserves"] - reserves_rows[2]["3M MA Reserves"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[8]["6M Reserve Change"],\n'
    '        reserves_rows[8]["3M MA Reserves"] - reserves_rows[2]["3M MA Reserves"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[14]["12M Reserve Change"],\n'
    '        reserves_rows[14]["3M MA Reserves"] - reserves_rows[2]["3M MA Reserves"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[5]["3M Reserve Growth"],\n'
    '        reserves_rows[5]["3M MA Reserves"] / reserves_rows[2]["3M MA Reserves"] - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[8]["6M Reserve Growth"],\n'
    '        reserves_rows[8]["3M MA Reserves"] / reserves_rows[2]["3M MA Reserves"] - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        reserves_rows[14]["12M Reserve Growth"],\n'
    '        reserves_rows[14]["3M MA Reserves"] / reserves_rows[2]["3M MA Reserves"] - 1,\n'
    '    )\n'
    '    zero_reserves_raw = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Date": (date(2024, 1, 1) + timedelta(days=31 * i)).isoformat(),\n'
    '                "Raw Value": 0,\n'
    '            }\n'
    '            for i in range(15)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_banking_reserves_outputs(zero_reserves_raw))[5][\n'
    '            "3M Reserve Growth"\n'
    '        ]\n'
    '        == ""\n'
    '    )\n'
    '\n'
    "    # M/O/Q use their own 60-value gates; A30 S uses E's 60-value gate while\n"
    '    # continuing to rank the current I value against numeric I history.\n'
    '    for before, first, percentile_col in (\n'
    '        (60, 61, "Reserve Level Percentile"),\n'
    '        (63, 64, "3M Change Percentile"),\n'
    '        (66, 67, "6M Change Percentile"),\n'
    '        (60, 61, "12M Change Percentile"),\n'
    '    ):\n'
    '        assert reserves_rows[before][percentile_col] == ""\n'
    '        assert reserves_rows[first][percentile_col] != ""\n'
    '    feb_2008_reserves = reserves_rows[61]\n'
    '    mar_2008_reserves = reserves_rows[62]\n'
    '    apr_2008_reserves = reserves_rows[63]\n'
    '    may_2008_reserves = reserves_rows[64]\n'
    '    aug_2008_reserves = reserves_rows[67]\n'
    '    sep_2008_reserves = reserves_rows[68]\n'
    '    jan_2009_reserves = reserves_rows[72]\n'
    '    feb_2009_reserves = reserves_rows[73]\n'
    '    assert feb_2008_reserves["Sample Count"] == 60 and values_equivalent(\n'
    '        feb_2008_reserves["3M MA Reserves"], 8335.333333, 1e-6\n'
    '    )\n'
    '    assert (\n'
    '        feb_2008_reserves["Reserve Level Percentile"],\n'
    '        feb_2008_reserves["Reserve Level Score"],\n'
    '    ) == ("11.90%", -7.62)\n'
    '    feb_2008_i_history = [\n'
    '        row["12M Reserve Change"]\n'
    '        for row in reserves_rows[:62]\n'
    '        if not is_blank(row["12M Reserve Change"])\n'
    '    ]\n'
    '    feb_2008_i_rank = _banking_reserves_sheets_round(\n'
    '        _banking_reserves_percentrank_inc_sheets(\n'
    '            feb_2008_i_history, feb_2008_reserves["12M Reserve Change"]\n'
    '        ),\n'
    '        3,\n'
    '    )\n'
    '    assert len(feb_2008_i_history) == 48 < 60\n'
    '    assert feb_2008_i_rank == 0.723\n'
    '    assert (\n'
    '        feb_2008_reserves["12M Change Percentile"]\n'
    '        == _banking_reserves_percent_display(feb_2008_i_rank)\n'
    '        == "72.30%"\n'
    '    )\n'
    '    assert feb_2008_reserves["12M Change Score"] == 4.46\n'
    '    assert [\n'
    '        (row["12M Change Percentile"], row["12M Change Score"])\n'
    '        for row in (\n'
    '            mar_2008_reserves,\n'
    '            apr_2008_reserves,\n'
    '            may_2008_reserves,\n'
    '            aug_2008_reserves,\n'
    '        )\n'
    '    ] == [\n'
    '        ("85.40%", 7.08),\n'
    '        ("85.70%", 7.14),\n'
    '        ("94.00%", 8.80),\n'
    '        ("75.50%", 5.10),\n'
    '    ]\n'
    '    assert (\n'
    '        aug_2008_reserves["Risk Asset Liquidity Bias"]\n'
    '        == "Mildly supportive for risk assets"\n'
    '    )\n'
    '    assert [\n'
    '        aug_2008_reserves[col]\n'
    '        for col in ("Reserve Impulse Score", "Coincident Score", "Leading Score")\n'
    '    ] == [4.55, 3.13, 4.47]\n'
    '    assert (\n'
    '        aug_2008_reserves["State"]\n'
    '        == aug_2008_reserves["Regime"]\n'
    '        == "Banking reserves improving"\n'
    '    )\n'
    '    assert [\n'
    '        sep_2008_reserves[col]\n'
    '        for col in (\n'
    '            "12M Change Score",\n'
    '            "Reserve Impulse Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '        )\n'
    '    ] == [10.0, 10.0, 10.0, 9.75]\n'
    '    assert [\n'
    '        jan_2009_reserves[col]\n'
    '        for col in (\n'
    '            "12M Change Score",\n'
    '            "Reserve Impulse Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '        )\n'
    '    ] == [10.0, 10.0, 10.0, 9.75]\n'
    '    assert (\n'
    '        feb_2008_reserves["Reserve Liquidity Regime"]\n'
    '        == "Positive reserves but draining"\n'
    '    )\n'
    '    assert (\n'
    '        feb_2008_reserves["Reliability"] == "Medium reliability"\n'
    '        and feb_2008_reserves["Signal Validity"] == "Incomplete Score"\n'
    '    )\n'
    '    assert feb_2008_reserves["Level Z-Score"] == -1.32\n'
    '    for source_col, percentile_col in (\n'
    '        ("3M MA Reserves", "Reserve Level Percentile"),\n'
    '        ("3M Reserve Change", "3M Change Percentile"),\n'
    '        ("6M Reserve Change", "6M Change Percentile"),\n'
    '        ("12M Reserve Change", "12M Change Percentile"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[source_col] for row in reserves_rows if not is_blank(row[source_col])\n'
    '        ]\n'
    '        expected_rank = _banking_reserves_sheets_round(\n'
    '            _banking_reserves_percentrank_inc_sheets(\n'
    '                history, feb_2009_reserves[source_col]\n'
    '            ),\n'
    '            3,\n'
    '        )\n'
    '        assert feb_2009_reserves[percentile_col] == _banking_reserves_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '    assert [\n'
    '        feb_2009_reserves[col]\n'
    '        for col in (\n'
    '            "Reserve Level Percentile",\n'
    '            "3M Change Percentile",\n'
    '            "6M Change Percentile",\n'
    '            "12M Change Percentile",\n'
    '        )\n'
    '    ] == ["100.00%", "97.10%", "100.00%", "100.00%"]\n'
    '    assert [\n'
    '        feb_2009_reserves[col]\n'
    '        for col in (\n'
    '            "Reserve Level Score",\n'
    '            "3M Change Score",\n'
    '            "6M Change Score",\n'
    '            "12M Change Score",\n'
    '        )\n'
    '    ] == [10.0, 9.42, 10.0, 10.0]\n'
    '\n'
    '    # Exact regime branch order, shock thresholds, component rules, and composites.\n'
    '    assert _banking_reserves_regime(1, 1, 1) == "Reserve expansion regime"\n'
    '    assert _banking_reserves_regime(1, -1, -1) == "Positive reserves but draining"\n'
    '    assert _banking_reserves_regime(-1, 1, 1) == "Low reserves but recovering"\n'
    '    assert (\n'
    '        _banking_reserves_regime(-1, -1, -1) == "Reserve contraction / scarcity regime"\n'
    '    )\n'
    '    assert (\n'
    '        _banking_reserves_regime(1, 1, -1) == "Early reserve recovery / mixed impulse"\n'
    '    )\n'
    '    assert (\n'
    '        _banking_reserves_regime(1, -1, 1)\n'
    '        == "Early reserve deterioration / mixed impulse"\n'
    '    )\n'
    '    assert _banking_reserves_regime(1, "", "") == "Reserve expansion regime"\n'
    '    assert (\n'
    '        _banking_reserves_positive_or_blank("")\n'
    '        and _banking_reserves_positive_or_blank(1)\n'
    '        and not _banking_reserves_positive_or_blank(-1)\n'
    '    )\n'
    '    assert (\n'
    '        _banking_reserves_negative_or_blank("")\n'
    '        and _banking_reserves_negative_or_blank(-1)\n'
    '        and not _banking_reserves_negative_or_blank(1)\n'
    '    )\n'
    '    assert [\n'
    '        _banking_reserves_shock_score(value)\n'
    '        for value in (250000, 50000, -50000, -250000)\n'
    '    ] == [6, 4, -4, -6]\n'
    '    assert _banking_reserves_shock_label(250000) == "1M major reserve injection shock"\n'
    '    assert _banking_reserves_shock_label(-250000) == "1M major reserve drain shock"\n'
    '    assert (\n'
    '        _banking_reserves_recovery_score(1, 1) == 5\n'
    '        and _banking_reserves_recovery_score(-1, 1) == -3\n'
    '    )\n'
    '    assert _banking_reserves_withdrawal_recovery_score(1.2, -0.1, -0.1, -1, 1) == -4\n'
    '    assert _banking_reserves_withdrawal_recovery_score(-1.2, 0.1, 0.1, 1, 1) == 6\n'
    '    assert _banking_reserves_scarcity_stress_score(-1.2, -0.1, 1, -1, 1) == -4\n'
    '    assert _banking_reserves_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    n, p, r, t = (10.0, 9.42, 10.0, 10.0)\n'
    '    expected_x = _banking_reserves_sheets_round(\n'
    '        clamp(\n'
    '            _banking_reserves_weighted_sum((0.05, n), (0.50, p), (0.30, r), (0.15, t)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_ad = _banking_reserves_sheets_round(\n'
    '        clamp(\n'
    '            _banking_reserves_weighted_sum((0.35, n), (0.35, p), (0.20, r), (0.10, t)),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_ae = _banking_reserves_sheets_round(\n'
    '        clamp(\n'
    '            _banking_reserves_weighted_sum(\n'
    '                (0.15, n),\n'
    '                (0.25, p),\n'
    '                (0.25, r),\n'
    '                (0.20, t),\n'
    '                (0.10, expected_x),\n'
    '                (0.05, 5),\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_af = _banking_reserves_sheets_round(\n'
    '        clamp(\n'
    '            _banking_reserves_weighted_sum(\n'
    '                (0.25, 5), (0.20, 0), (0.25, 4), (0.20, 0), (0.10, 6)\n'
    '            ),\n'
    '            -10,\n'
    '            10,\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (expected_x, expected_ad, expected_ae, expected_af) == (\n'
    '        9.71,\n'
    '        9.80,\n'
    '        9.58,\n'
    '        2.85,\n'
    '    )\n'
    '    assert [\n'
    '        feb_2009_reserves[col]\n'
    '        for col in (\n'
    '            "Reserve Impulse Score",\n'
    '            "Reserve Cycle Turning Point Score",\n'
    '            "Reserve Shock Score",\n'
    '            "Reserve Drain Risk Score",\n'
    '            "Reserve Scarcity Stress Score",\n'
    '            "Reserve Extreme Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '        )\n'
    '    ] == [9.71, 5, 0, 4, 0, 6, 9.80, 9.58, 2.85]\n'
    '    assert (\n'
    '        feb_2009_reserves["Risk Asset Liquidity Bias"]\n'
    '        == "Strongly supportive for risk assets / liquidity-sensitive assets"\n'
    '    )\n'
    '    assert (\n'
    '        feb_2009_reserves["State"]\n'
    '        == feb_2009_reserves["Regime"]\n'
    '        == "Banking reserves improving"\n'
    '    )\n'
    '    assert (\n'
    '        feb_2009_reserves["Signal Comment"]\n'
    '        == "Strong positive reserve-liquidity impulse with supportive banking-liquidity profile"\n'
    '    )\n'
    '    assert (\n'
    '        feb_2009_reserves["Reliability"] == "Medium reliability"\n'
    '        and feb_2009_reserves["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert (\n'
    '        feb_2009_reserves["Warning Flag"] == "Neutral reserve-liquidity warning profile"\n'
    '    )\n'
    '    assert [\n'
    '        feb_2009_reserves[col]\n'
    '        for col in (\n'
    '            "Level Z-Score",\n'
    '            "3M Change Z-Score",\n'
    '            "6M Change Z-Score",\n'
    '            "12M Change Z-Score",\n'
    '        )\n'
    '    ] == [5.20, 3.88, 4.97, 4.72]\n'
    '    assert feb_2009_reserves["Extreme Flag"] == "Extreme reserve-liquidity support"\n'
    '\n'
    '    # Initial chain and STDEV.S-equivalent expanding z-scores.\n'
    '    assert (\n'
    '        reserves_rows[0]["Monthly Average Reserves"] == 9522\n'
    '        and reserves_rows[0]["Sample Count"] == 0\n'
    '    )\n'
    '    assert (\n'
    '        reserves_rows[0]["Reliability"] == "Insufficient history"\n'
    '        and reserves_rows[0]["Signal Validity"] == "Low Sample"\n'
    '    )\n'
    '    assert (\n'
    '        reserves_rows[2]["Monthly Average Reserves"] == 9700\n'
    '        and reserves_rows[2]["Sample Count"] == 1\n'
    '        and reserves_rows[2]["3M MA Reserves"] == 9622\n'
    '    )\n'
    '    assert (\n'
    '        reserves_rows[2]["3M Reserve Change"]\n'
    '        == reserves_rows[2]["6M Reserve Change"]\n'
    '        == ""\n'
    '    )\n'
    '    assert reserves_rows[2]["Reserve Liquidity Regime"] == "Reserve expansion regime"\n'
    '    assert (\n'
    '        reserves_rows[3]["Monthly Average Reserves"] == 11183\n'
    '        and reserves_rows[3]["Sample Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(reserves_rows[3]["3M MA Reserves"], 10175.66667, 1e-5)\n'
    '    assert (\n'
    '        reserves_rows[3]["3M Reserve Change"]\n'
    '        == reserves_rows[3]["6M Reserve Change"]\n'
    '        == ""\n'
    '    )\n'
    '    assert reserves_rows[3]["Reserve Liquidity Regime"] == "Reserve expansion regime"\n'
    '    assert values_equivalent(reserves_rows[3]["1M Reserve Change"], 553.6666667, 1e-7)\n'
    '    assert (\n'
    '        reserves_rows[3]["Reserve Shock Flag"] == "No major reserve shock"\n'
    '        and reserves_rows[3]["Reserve Shock Score"] == 0\n'
    '    )\n'
    '    assert values_equivalent(reserves_rows[5]["3M MA Reserves"], 11349.66667, 1e-5)\n'
    '    assert (\n'
    '        reserves_rows[5]["3M Reserve Change"] > 0\n'
    '        and reserves_rows[5]["6M Reserve Change"] == ""\n'
    '    )\n'
    '    assert reserves_rows[5]["Reserve Liquidity Regime"] == "Reserve expansion regime"\n'
    '    assert (\n'
    '        reserves_rows[7]["3M MA Reserves"] == 12363\n'
    '        and reserves_rows[7]["3M Reserve Change"] > 0\n'
    '        and reserves_rows[7]["6M Reserve Change"] == ""\n'
    '    )\n'
    '    assert reserves_rows[7]["Reserve Liquidity Regime"] == "Reserve expansion regime"\n'
    '    e_values = [row["3M MA Reserves"] for row in reserves_rows]\n'
    '    g_values = [row["3M Reserve Change"] for row in reserves_rows]\n'
    '    h_values = [row["6M Reserve Change"] for row in reserves_rows]\n'
    '    i_values = [row["12M Reserve Change"] for row in reserves_rows]\n'
    '    expected_reserve_zscores = (\n'
    '        _banking_reserves_rounded_expanding_zscores(e_values, _metric_counts(e_values))[\n'
    '            73\n'
    '        ],\n'
    '        _banking_reserves_rounded_expanding_zscores(g_values, _metric_counts(g_values))[\n'
    '            73\n'
    '        ],\n'
    '        _banking_reserves_rounded_expanding_zscores(h_values, _metric_counts(h_values))[\n'
    '            73\n'
    '        ],\n'
    '        _banking_reserves_rounded_expanding_zscores(i_values, _metric_counts(i_values))[\n'
    '            73\n'
    '        ],\n'
    '    )\n'
    '    assert expected_reserve_zscores == (5.20, 3.88, 4.97, 4.72)\n'
    '\n'
    '    class FakeBankingReservesWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    fake_reserves_write = FakeBankingReservesWorksheet([])\n'
    '    assert (\n'
    '        write_indicator_outputs(\n'
    '            fake_reserves_write,\n'
    '            reserves_out,\n'
    '            75,\n'
    '            BANKING_RESERVES_OUTPUT_COLUMNS,\n'
    '            "AQ",\n'
    '            "C",\n'
    '        )\n'
    '        == "C2:AQ75"\n'
    '    )\n'
    '    assert [(call[0], call[2]) for call in fake_reserves_write.update_calls] == [\n'
    '        ("C2:AQ75", "RAW")\n'
    '    ]\n'
    '    fake_reserves_verify = FakeBankingReservesWorksheet(\n'
    '        dataframe_to_sheet_rows(reserves_out, BANKING_RESERVES_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_reserves_verify,\n'
    '        reserves_out,\n'
    '        75,\n'
    '        BANKING_RESERVES_OUTPUT_COLUMNS,\n'
    '        "AQ",\n'
    '        "C",\n'
    '    )\n'
    '    assert fake_reserves_verify.get_calls == [("C2:AQ75", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            reserves_out, 75, "C", 2, BANKING_RESERVES_OUTPUT_COLUMNS, "AQ"\n'
    '        )\n'
    '        == "C2:AQ75"\n'
    '    )\n'
    '\n'
    '    class FakeBankingReservesWorkbook:\n'
    '        def __init__(self, worksheet: FakeBankingReservesWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeBankingReservesWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_reserves_read = FakeBankingReservesWorksheet(\n'
    '        [[a, b] for a, b in zip(reserves_dates, reserves_values)]\n'
    '    )\n'
    '    fake_a30_result = process_indicator(\n'
    '        FakeBankingReservesWorkbook(fake_reserves_read),\n'
    '        BatchIndicatorSpec("Banking Reserves", "A30"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a30_result.ok and fake_a30_result.output_range == "C2:AQ75"\n'
    '    assert fake_reserves_read.get_calls == [("A2:B", "UNFORMATTED_VALUE")]\n'
    '    fake_reserves_no_verify = FakeBankingReservesWorksheet(\n'
    '        [[a, b] for a, b in zip(reserves_dates, reserves_values)]\n'
    '    )\n'
    '    fake_a30_no_verify_result = process_indicator(\n'
    '        FakeBankingReservesWorkbook(fake_reserves_no_verify),\n'
    '        BatchIndicatorSpec("Banking Reserves", "A30"),\n'
    '        write=True,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert (\n'
    '        fake_a30_no_verify_result.ok\n'
    '        and fake_a30_no_verify_result.output_range == "C2:AQ75"\n'
    '    )\n'
    '    assert fake_reserves_no_verify.get_calls == [("A2:B", "UNFORMATTED_VALUE")]\n'
    '    assert [(call[0], call[2]) for call in fake_reserves_no_verify.update_calls] == [\n'
    '        ("C2:AQ75", "RAW")\n'
    '    ]\n'
    '    all_specs_with_a30 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 30)] + [["Banking Reserves"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Banking Reserves", "A30") in all_specs_with_a30\n'
    '    assert (\n'
    '        df_records(calculate_central_bank_liquidity_outputs(central_raw))\n'
    '        == central_rows\n'
    '    )\n'
    '    assert df_records(calculate_banking_reserves_outputs(reserves_raw)) == reserves_rows\n'
    '    # A31 fixed protected-source fixture: January 1959 through October 2013.\n'
    '    # Packing keeps the offline regression data compact without consulting Sheets.\n'
    '    import base64\n'
    '    import zlib\n'
    '\n'
    '    broad_money_fixture = "eNotl1mSwyAMRC+UcqFd3P9i093MRyDGLNp59u2vf77zDdr7+c/v+Qyt63/qf+l/f6l21Bba0dv5Fu1qzmrO/eIXh/uwvWhdI6mR1ghnxrnYJ0xzzDlimmOt8eUqu3zrlCc8OeIFmcNHrXYI46oI7hClkWGbkiFDrcbzjUvCOpA8ytUmT6nhKXU50g5No6l79PDtHI5McO009xxpsTp9g/Isf5L7Gk+6wfWyYFxJfLl7HsNeCYs4Wtox7ailDmm0UUJzjDhmQG/MdsqecTgjuG9CV/7Xeuma6VwJXXFG0jYJ/RptcYeih7P5S4726JnWyNH6MY4MPZKjlaMzRutnOHMP50BrvN3iKkQO3iJm8P9yzzr8NWaX0cpl1KCMUVJuapPj0Cl/JZ1KOlXwpIqFBJWUoLLUXs4pWrmKlqxmxFQ395EOBemxapLjkriGkVTwE1vao5Z61rpaWru21NLHBU2w80ryHbX0ccmbJT/WpV/6MGLaqEsbc6OdcdDOEzvoh048J+O3i7t0M3t6Aj+e08uYbNiNLWfMoV8GNgu0zMpx6jGwTaHVnGQETtLmU9Rpevl/GHezjJ1B3N3fKu9Wu60ya630n2vXabkN7rZBqRf7c4S2XNl+k37YopxbWtWM2VW07FDLHcq8wzqwiIpEq7Nk0b3aH9r17x5a/SouruS5Xr8bjKGbnH0bb0bPy6y9svtVjbGj0mEH1nZ0ztJlJ6mdnTpo+mje8Bh0owlypp1LG5kpHM1wrlkwHM2KjRLd7K2xZcCYXXrJ/OBUN8ngTheaB5uUIF6JRsFoPpLG52qW/GuvmBprIaaw3BU7HcGSBllD/jUUKTRFrxnLEN80JIhhsqGTDLHqUtY0Fg7sno41Gak3KdNkae9snZvQEHok6hdEy3V1NzVTRcmQtdyqTAKUXz2pcFq1qRtsVatjmA7qtLifd9pk5w6J0Ko41goa62edXtPMy2puczBvkDvYY4Jl1UbxBl9I8EF88WkZoraHkW0MZLxDDKMJibYpF65Szbblgl2pe4+Uv8UiArcy8v2YLrbjjE0/uib8FD3hp1NToK6fDc27uh9N1nGEAOe9GHKYRN1oR7us7u668Bwv1b0FrlrnnqmumF/uShvn3cguXJ3c5wgA7hmqwx7LhPc0Zqenv6fEmywtzobYqUDxV5G9zuuM2eoss3wqTXmu9D7YppVR3qnbvVsTelmhEddGrccxxAJ02fE+8ila2KfFEKOS4KOagI7/V7aZK1lZh5adFGcNwu6bOmTlcHDIEWTY6+Biv/HIIkeYUWoYb0ARVjSQx3kA8jjkOeD+U8xlKCD9cPseYyYBSHSjnxJ0HN0YoBEXdvwziApjIA00qGsmbBgDYBKBBSJIDILEZmc0BEiF4Yq8Jq9U6k2JN1gjBC6M9XjQFaQi8oozJ+K5PoAsBJk3VEKNKM56mBKt4wknQh2xETHEiTyHTWjeUzxUfIMskq+7bxBSpC6HQPXQky6veLwRD0DQScI0e4OmBSGMI2+kOjalCSVxGI58modg8/YYyf9oInLfu307XuFY6pbG4GqX/Z/Zb6b4L3feu9bZ94mw4sS8grFUrgYhg51uMRiSURa440F0yolo8RY6Le5Hk62rCHknP7Uu1Ogrd83hlQCTKypG6BWD/AvmAbsRT84tQaHJ9BuKg01eRWBEeQo3JllRgl7G5nV55pbEvCNsZQwHeTGFjeKKPNUCx11xotEscJO4zDqFkLqD0w8dC4hcEaSSFAhJBXFZMN+TlaXJj0F8fOQYQr9slqV81s86eFMPOatcPNkMImClnghDeGrxQ7J+EDH3saSiJ0dFJaf+cTJEmNeElSZeJWwQL+cN6p6Cj7XZFeLUMZAdk5ZgGQxSdMJOe3TndgiYVB0C0xMV/0/LolWkZXZ9hJmXlitB6yuW0E8nkTfZXeJmaKjbhZqidlDmqhvR6drDyhTnbj90PCV2VKHqozsW9CiINO3SR7yA7rARFJ6hvH0Eqm26HxvXr6Dz0u/tSiN0GnRFfoc4pUNZ2IhDPqWQq1OlqfkJwUExdsO9eqdoR+gTVuWsruSN1WRoTGh9oOAmZ5CCZFmcu4UiIFq9m0rxrWAcjmfi4LJnWQTqan/iYBN56ZS+qr9z9DU0Rz6cIx8iYISzYCJ1TZOO53lMLEx2nT4h5Ji4DIjBJxQap1sm9bkwuZR5ykawfHAkUATAbMziIWJj73etDaMRT3uYCUPExfSrwgWeJjafFD2jjCy/TknPKkUbcsHywwuInPU4umlFdmzsQbXIGPKwcU0eZubyeyjYaU0VNdoGf25Lo2Vy4U2/HUaa7yjQdijROmEFBE70tsfeI4g+j7MZcotOgXVRtTH9VuB/leYRCZIkzrvgtmwHJmcC4pZlPN0tDd5Doe8tlgnguYtSz3nAhpJVeiYzk8ddldoOyq04PlWiiO77en3got8QxreuGvT51iNA9Azv6v0e04cAbCO+R5j+AcpmTU8="\n'
    '    broad_money_values = [\n'
    '        float(value)\n'
    '        for value in zlib.decompress(base64.b64decode(broad_money_fixture))\n'
    '        .decode()\n'
    '        .split(",")\n'
    '    ]\n'
    '    assert (\n'
    '        len(broad_money_values) == 658\n'
    '        and broad_money_values[:3] == [286.6, 287.7, 289.2]\n'
    '        and broad_money_values[-1] == 10986.3\n'
    '    )\n'
    '\n'
    '    def broad_money_month_end(offset: int) -> date:\n'
    '        year = 1959 + offset // 12\n'
    '        month = 1 + offset % 12\n'
    '        next_month = date(year + 1, 1, 1) if month == 12 else date(year, month + 1, 1)\n'
    '        return next_month - timedelta(days=1)\n'
    '\n'
    '    broad_money_dates = [\n'
    '        broad_money_month_end(i) for i in range(len(broad_money_values))\n'
    '    ]\n'
    '    broad_money_raw_rows = [\n'
    '        {"Date": dt.isoformat(), "Raw Value": value}\n'
    '        for dt, value in zip(broad_money_dates, broad_money_values)\n'
    '    ]\n'
    '    broad_money_raw = make_dataframe(broad_money_raw_rows, ["Date", "Raw Value"])\n'
    '    broad_money_out = calculate_broad_money_outputs(broad_money_raw)\n'
    '    broad_money_rows = df_records(broad_money_out)\n'
    '    broad_money_spec = select_indicator_engine_spec(\n'
    '        "Unmapped broad money sheet", CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped broad money sheet", CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_broad_money_outputs\n'
    '    )\n'
    '    assert broad_money_spec.calculate is calculate_broad_money_outputs\n'
    '    assert (\n'
    '        broad_money_spec.input_start_col == "A"\n'
    '        and broad_money_spec.input_end_col == "B"\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_spec.output_start_col == "C"\n'
    '        and broad_money_spec.output_end_col == "AO"\n'
    '    )\n'
    '    assert list(broad_money_out.columns) == BROAD_MONEY_OUTPUT_COLUMNS\n'
    '    assert broad_money_out.shape[1] == len(BROAD_MONEY_OUTPUT_COLUMNS) == 39\n'
    '    assert output_end_col_for_columns(BROAD_MONEY_OUTPUT_COLUMNS) == "AO"\n'
    '    assert indicator_input_range("Broad Money") == "\'Broad Money\'!A2:B"\n'
    '    assert indicator_output_range("Broad Money", 659, "AO") == "\'Broad Money\'!C2:AO659"\n'
    '\n'
    '    # Physical-row moving averages/lags and unrounded core growth calculations.\n'
    '    assert (\n'
    '        broad_money_rows[0]["3M MA Broad Money"] == ""\n'
    '        and broad_money_rows[1]["3M MA Broad Money"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[2]["3M MA Broad Money"],\n'
    '        _broad_money_mean_sheets(broad_money_values[:3]),\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_rows[0]["Sample Count"] == 0\n'
    '        and broad_money_rows[2]["Sample Count"] == 0\n'
    '        and broad_money_rows[4]["Sample Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[5]["1M Broad Money Growth"],\n'
    '        broad_money_rows[5]["3M MA Broad Money"]\n'
    '        / broad_money_rows[4]["3M MA Broad Money"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[7]["3M Broad Money Growth"],\n'
    '        broad_money_rows[7]["3M MA Broad Money"]\n'
    '        / broad_money_rows[4]["3M MA Broad Money"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[10]["6M Broad Money Growth"],\n'
    '        broad_money_rows[10]["3M MA Broad Money"]\n'
    '        / broad_money_rows[4]["3M MA Broad Money"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[16]["12M Broad Money Growth"],\n'
    '        broad_money_rows[16]["3M MA Broad Money"]\n'
    '        / broad_money_rows[4]["3M MA Broad Money"]\n'
    '        - 1,\n'
    '    )\n'
    '    blank_money_values = broad_money_values[:15]\n'
    '    blank_money_raw_rows = [\n'
    '        {"Date": f"blank-{i}", "Raw Value": ("" if i == 12 else value)}\n'
    '        for i, value in enumerate(blank_money_values)\n'
    '    ]\n'
    '    blank_money_rows = df_records(\n'
    '        calculate_broad_money_outputs(\n'
    '            make_dataframe(blank_money_raw_rows, ["Date", "Raw Value"])\n'
    '        )\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        blank_money_rows[12]["3M MA Broad Money"],\n'
    '        _broad_money_mean_sheets(blank_money_values[10:12]),\n'
    '    )\n'
    '    zero_money_raw = make_dataframe(\n'
    '        [{"Date": f"z{i}", "Raw Value": 0} for i in range(10)], ["Date", "Raw Value"]\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_broad_money_outputs(zero_money_raw))[5][\n'
    '            "3M Broad Money Growth"\n'
    '        ]\n'
    '        == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[7]["3M Annualised Broad Money Growth"],\n'
    '        (1 + broad_money_rows[7]["3M Broad Money Growth"]) ** 4 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        broad_money_rows[10]["6M Annualised Broad Money Growth"],\n'
    '        (1 + broad_money_rows[10]["6M Broad Money Growth"]) ** 2 - 1,\n'
    '    )\n'
    '\n'
    '    # K/M/O/Q each use their own 60-value history and rank C/F/G/H respectively.\n'
    '    for before, first, percentile_col in (\n'
    '        (60, 61, "Broad Money Level Percentile"),\n'
    '        (65, 66, "3M Growth Percentile"),\n'
    '        (68, 69, "6M Growth Percentile"),\n'
    '        (74, 75, "12M Growth Percentile"),\n'
    '    ):\n'
    '        assert broad_money_rows[before][percentile_col] == ""\n'
    '        assert broad_money_rows[first][percentile_col] != ""\n'
    '    oct_2013 = broad_money_rows[657]\n'
    '    for source_col, percentile_col, score_col in (\n'
    '        (\n'
    '            "3M MA Broad Money",\n'
    '            "Broad Money Level Percentile",\n'
    '            "Broad Money Level Score",\n'
    '        ),\n'
    '        ("3M Broad Money Growth", "3M Growth Percentile", "3M Growth Score"),\n'
    '        ("6M Broad Money Growth", "6M Growth Percentile", "6M Growth Score"),\n'
    '        ("12M Broad Money Growth", "12M Growth Percentile", "12M Growth Score"),\n'
    '    ):\n'
    '        history = [\n'
    '            row[source_col] for row in broad_money_rows if not is_blank(row[source_col])\n'
    '        ]\n'
    '        expected_rank = _broad_money_sheets_round(\n'
    '            _broad_money_percentrank_inc_sheets(history, oct_2013[source_col]), 3\n'
    '        )\n'
    '        assert oct_2013[percentile_col] == _broad_money_percent_display(expected_rank)\n'
    '        assert oct_2013[score_col] == _broad_money_score_percentile(expected_rank)\n'
    '        assert isinstance(oct_2013[percentile_col], str) and oct_2013[\n'
    '            percentile_col\n'
    '        ].endswith("%")\n'
    '    assert (\n'
    '        _broad_money_sheets_round(1.005, 2) == 1.01\n'
    '        and _broad_money_sheets_round(-1.005, 2) == -1.01\n'
    '    )\n'
    '\n'
    '    feb_1964 = broad_money_rows[61]\n'
    '    aug_1964 = broad_money_rows[67]\n'
    '    sep_1964 = broad_money_rows[68]\n'
    '    oct_1964 = broad_money_rows[69]\n'
    '    assert broad_money_dates[61] == date(1964, 2, 29) and (\n'
    '        feb_1964["Broad Money Level Percentile"],\n'
    '        feb_1964["Broad Money Level Score"],\n'
    '    ) == ("100.00%", 10.0)\n'
    '    assert feb_1964["Broad Money Level Percentile"] != 1\n'
    '    assert broad_money_dates[67] == date(1964, 8, 31) and (\n'
    '        aug_1964["3M Growth Percentile"],\n'
    '        aug_1964["3M Growth Score"],\n'
    '    ) == ("81.70%", 6.34)\n'
    '    assert broad_money_dates[68] == date(1964, 9, 30) and (\n'
    '        sep_1964["3M Growth Percentile"],\n'
    '        sep_1964["3M Growth Score"],\n'
    '    ) == ("98.40%", 9.68)\n'
    '    assert broad_money_dates[69] == date(1964, 10, 31) and (\n'
    '        oct_1964["6M Growth Percentile"],\n'
    '        oct_1964["6M Growth Score"],\n'
    '    ) == ("84.70%", 6.94)\n'
    '\n'
    '    # Blank-positive A31 regime handling, including the named early fixtures.\n'
    '    assert (\n'
    '        _broad_money_positive_or_blank("")\n'
    '        and _broad_money_positive_or_blank(1)\n'
    '        and not _broad_money_positive_or_blank(-1)\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_dates[2] == date(1959, 3, 31)\n'
    '        and broad_money_rows[2]["Broad Money Regime"] == "Broad-money expansion regime"\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_dates[3] == date(1959, 4, 30)\n'
    '        and broad_money_rows[3]["Broad Money Regime"] == "Broad-money expansion regime"\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_dates[7] == date(1959, 8, 31)\n'
    '        and broad_money_rows[7]["Broad Money Regime"] == "Broad-money expansion regime"\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_dates[10] == date(1959, 11, 30)\n'
    '        and broad_money_rows[10]["Broad Money Regime"] == "Broad-money expansion regime"\n'
    '    )\n'
    '    assert _broad_money_regime(1, "", "", "") == "Broad-money expansion regime"\n'
    '    assert _broad_money_regime(1, 0.1, 0.1, 0.1) == "Broad-money expansion regime"\n'
    '    assert (\n'
    '        _broad_money_regime(1, -0.1, -0.1, -0.1)\n'
    '        == "Broad-money deterioration from high base"\n'
    '    )\n'
    '    assert _broad_money_regime(-1, 0.1, 0.1, -0.1) == "Early broad-money recovery"\n'
    '    assert _broad_money_regime(-1, -0.1, -0.1, 0.1) == "Broad-money contraction regime"\n'
    '    assert (\n'
    '        _broad_money_asset_bias_label(6)\n'
    '        == "Strongly supportive for liquidity-sensitive risk assets"\n'
    '    )\n'
    '    assert (\n'
    '        _broad_money_growth_shock_label(0.15)\n'
    '        == "Extreme 3M annualised money-growth surge"\n'
    '    )\n'
    '    assert (\n'
    '        _broad_money_growth_shock_label(0.10)\n'
    '        == "Strong 3M annualised money-growth surge"\n'
    '    )\n'
    '    assert (\n'
    '        _broad_money_growth_shock_label(-0.05)\n'
    '        == "Sharp 3M annualised money contraction"\n'
    '    )\n'
    '    assert _broad_money_growth_shock_label(0) == "Negative 3M annualised money growth"\n'
    '    assert (\n'
    '        _broad_money_growth_shock_label(0.03) == "Very weak 3M annualised money growth"\n'
    '    )\n'
    '    assert (\n'
    '        _broad_money_recovery_score(0.1, 0.1) == 5\n'
    '        and _broad_money_recovery_score(-0.1, 0.1) == -3\n'
    '    )\n'
    '    assert (\n'
    '        _broad_money_shock_score(0.15) == 6\n'
    '        and _broad_money_shock_score(0.10) == 4\n'
    '        and _broad_money_shock_score(-0.05) == -6\n'
    '        and _broad_money_shock_score(0) == -4\n'
    '        and _broad_money_shock_score(0.03) == -2\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_rows[60]["Level Z-Score"] == ""\n'
    '        and broad_money_rows[61]["Level Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        broad_money_rows[65]["3M Growth Z-Score"] == ""\n'
    '        and broad_money_rows[66]["3M Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert broad_money_rows[61]["Level Z-Score"] == _broad_money_sheets_round(\n'
    '        broad_money_rows[61]["Level Z-Score"], 2\n'
    '    )\n'
    '    assert _broad_money_expansion_support_score(0, 1.6, 1.6, 0.11, 0.09) == 6\n'
    '    assert _broad_money_expansion_support_score(0, 1.1, 1.1, 0.09, 0) == 4\n'
    '    assert _broad_money_expansion_support_score(1.6, 1.1, -1, 0.09, 0) == 3\n'
    '    assert _broad_money_contraction_risk_score(0, -1.6, -1.6, 0.01, 0.02) == -6\n'
    '    assert _broad_money_contraction_risk_score(0, -1.1, -1.1, 0.03, 1) == -4\n'
    '    assert _broad_money_contraction_risk_score(0, 0.1, 0.1, 0.06, 0.06) == 3\n'
    '    assert _broad_money_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _broad_money_outlier_label([3.1, -3.5, 0, 0]) == "Extreme broad-money expansion"\n'
    '    )\n'
    '    # The October 2013 threshold case must use rounded visible L/N/P/R.\n'
    '    assert broad_money_dates[657] == date(2013, 10, 31)\n'
    '    assert (oct_2013["3M Growth Percentile"], oct_2013["3M Growth Score"]) == (\n'
    '        "54.30%",\n'
    '        0.86,\n'
    '    )\n'
    '    assert (oct_2013["6M Growth Percentile"], oct_2013["6M Growth Score"]) == (\n'
    '        "35.90%",\n'
    '        -2.82,\n'
    '    )\n'
    '    bml = oct_2013["Broad Money Level Score"]\n'
    '    bmn = oct_2013["3M Growth Score"]\n'
    '    bmp = oct_2013["6M Growth Score"]\n'
    '    bmr = oct_2013["12M Growth Score"]\n'
    '    bmv = oct_2013["Broad Money Impulse Score"]\n'
    '    bmw = oct_2013["Broad Money Cycle Turning Point Score"]\n'
    '    bmx = oct_2013["Money Growth Shock Score"]\n'
    '    bmy = oct_2013["Monetary Overheating Risk Score"]\n'
    '    bmz = oct_2013["Monetary Contraction Risk Score"]\n'
    '    bmaa = oct_2013["Broad Money Extreme Score"]\n'
    '    expected_bmv = _broad_money_sheets_round(\n'
    '        clamp(\n'
    '            _broad_money_weighted_sum(\n'
    '                (0.50, bmn), (0.30, bmp), (0.15, bmr), (0.05, bml)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_bmab = _broad_money_sheets_round(\n'
    '        clamp(\n'
    '            _broad_money_weighted_sum(\n'
    '                (0.25, bml), (0.40, bmn), (0.25, bmp), (0.10, bmr)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_bmac = _broad_money_sheets_round(\n'
    '        clamp(_broad_money_weighted_sum((0.60, bmv), (0.25, bmw), (0.15, bml))), 2\n'
    '    )\n'
    '    expected_bmad = _broad_money_sheets_round(\n'
    '        clamp(\n'
    '            _broad_money_weighted_sum(\n'
    '                (0.20, bmw), (0.20, bmx), (0.25, bmy), (0.25, bmz), (0.10, bmaa)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert oct_2013["Broad Money Impulse Score"] == expected_bmv\n'
    '    assert oct_2013["Coincident Score"] == expected_bmab == 2.0\n'
    '    assert oct_2013["Leading Score"] == expected_bmac\n'
    '    assert oct_2013["Warning Score"] == expected_bmad\n'
    '    assert (\n'
    '        oct_2013["Risk Asset Liquidity Bias"]\n'
    '        == "Mildly supportive for liquidity-sensitive risk assets"\n'
    '    )\n'
    '    assert oct_2013["State"] == oct_2013["Regime"] == "Broad money conditions improving"\n'
    '    assert oct_2013["Signal Comment"] == _broad_money_signal_interpretation(\n'
    '        oct_2013["Leading Score"], oct_2013["Warning Score"]\n'
    '    )\n'
    '    assert (\n'
    '        oct_2013["Reliability"] == "High reliability"\n'
    '        and oct_2013["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert oct_2013["Warning Flag"] != "" and oct_2013["Extreme Flag"] != ""\n'
    '    assert _broad_money_conditions_regime(2) == "Broad money conditions improving"\n'
    '    assert _broad_money_momentum_label(2, 2) == "Broad money conditions improving"\n'
    '    assert (\n'
    '        _broad_money_signal_interpretation(6, 0)\n'
    '        == "Strong positive broad-money impulse with supportive monetary-liquidity profile"\n'
    '    )\n'
    '    assert (\n'
    '        _broad_money_warning_label(6)\n'
    '        == "High broad-money overheating / liquidity-support warning"\n'
    '    )\n'
    '\n'
    '    class FakeBroadMoneyWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    fake_broad_write = FakeBroadMoneyWorksheet([])\n'
    '    assert (\n'
    '        write_indicator_outputs(\n'
    '            fake_broad_write,\n'
    '            broad_money_out,\n'
    '            659,\n'
    '            BROAD_MONEY_OUTPUT_COLUMNS,\n'
    '            "AO",\n'
    '            "C",\n'
    '        )\n'
    '        == "C2:AO659"\n'
    '    )\n'
    '    assert [(call[0], call[2]) for call in fake_broad_write.update_calls] == [\n'
    '        ("C2:AO659", "RAW")\n'
    '    ]\n'
    '    fake_broad_verify = FakeBroadMoneyWorksheet(\n'
    '        dataframe_to_sheet_rows(broad_money_out, BROAD_MONEY_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_broad_verify, broad_money_out, 659, BROAD_MONEY_OUTPUT_COLUMNS, "AO", "C"\n'
    '    )\n'
    '    assert fake_broad_verify.get_calls == [("C2:AO659", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            broad_money_out, 659, "C", 2, BROAD_MONEY_OUTPUT_COLUMNS, "AO"\n'
    '        )\n'
    '        == "C2:AO659"\n'
    '    )\n'
    '\n'
    '    class FakeBroadMoneyWorkbook:\n'
    '        def __init__(self, worksheet: FakeBroadMoneyWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeBroadMoneyWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_broad_input = [\n'
    '        [dt.isoformat(), value]\n'
    '        for dt, value in zip(broad_money_dates[:80], broad_money_values[:80])\n'
    '    ]\n'
    '    fake_broad_read = FakeBroadMoneyWorksheet(fake_broad_input)\n'
    '    fake_a31_result = process_indicator(\n'
    '        FakeBroadMoneyWorkbook(fake_broad_read),\n'
    '        BatchIndicatorSpec("Broad Money", "A31"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a31_result.ok and fake_a31_result.output_range == "C2:AO81"\n'
    '    assert fake_broad_read.get_calls == [("A2:B", None)]\n'
    '    fake_broad_no_verify = FakeBroadMoneyWorksheet(fake_broad_input)\n'
    '    fake_a31_no_verify_result = process_indicator(\n'
    '        FakeBroadMoneyWorkbook(fake_broad_no_verify),\n'
    '        BatchIndicatorSpec("Broad Money", "A31"),\n'
    '        write=True,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert (\n'
    '        fake_a31_no_verify_result.ok\n'
    '        and fake_a31_no_verify_result.output_range == "C2:AO81"\n'
    '    )\n'
    '    assert fake_broad_no_verify.get_calls == [("A2:B", None)]\n'
    '    assert [(call[0], call[2]) for call in fake_broad_no_verify.update_calls] == [\n'
    '        ("C2:AO81", "RAW")\n'
    '    ]\n'
    '    all_specs_with_a31 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 31)] + [["Broad Money"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Broad Money", "A31") in all_specs_with_a31\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped central liquidity sheet",\n'
    '            CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL,\n'
    '        ).input_end_col\n'
    '        == "D"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped banking reserves sheet",\n'
    '            CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AQ"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped broad money sheet", CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AO"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_broad_money_outputs(broad_money_raw)) == broad_money_rows\n'
    '    )\n'
    '    fiscal_values = [-2 + 0.08 * i + 0.7 * math.sin(i / 4) for i in range(100)]\n'
    '    fiscal_raw_rows = [\n'
    '        {"Date": f"f{i}", "Raw Value": fiscal_values[i]}\n'
    '        for i in range(len(fiscal_values))\n'
    '    ]\n'
    '    fiscal_raw_rows[12]["Raw Value"] = ""\n'
    '    fiscal_raw = make_dataframe(fiscal_raw_rows, ["Date", "Raw Value"])\n'
    '    fiscal_out = calculate_fiscal_balance_outputs(fiscal_raw)\n'
    '    fiscal_rows = df_records(fiscal_out)\n'
    '    fiscal_spec = select_indicator_engine_spec(\n'
    '        "Unmapped fiscal balance sheet", CONTROL_PANEL_THIRTIETH_INDICATOR_CELL\n'
    '    )\n'
    '    assert fiscal_spec.input_start_col == "A" and fiscal_spec.input_end_col == "B"\n'
    '    assert fiscal_spec.output_start_col == "C" and fiscal_spec.output_end_col == "AJ"\n'
    '    assert fiscal_out.shape[1] == len(FISCAL_BALANCE_OUTPUT_COLUMNS) == 34\n'
    '    assert output_end_col_for_columns(FISCAL_BALANCE_OUTPUT_COLUMNS) == "AJ"\n'
    '    assert indicator_input_range("Fiscal Balance") == "\'Fiscal Balance\'!A2:B"\n'
    '    assert (\n'
    '        indicator_output_range("Fiscal Balance", 678, "AJ")\n'
    '        == "\'Fiscal Balance\'!C2:AJ678"\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[0]["3-Period Moving Average Fiscal Balance"] == ""\n'
    '        and fiscal_rows[1]["3-Period Moving Average Fiscal Balance"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        fiscal_rows[2]["3-Period Moving Average Fiscal Balance"],\n'
    '        sum(fiscal_values[:3]) / 3,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        fiscal_rows[12]["3-Period Moving Average Fiscal Balance"],\n'
    '        (fiscal_values[10] + fiscal_values[11]) / 2,\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[0]["Fiscal Balance Observation Count"] == 0\n'
    '        and fiscal_rows[2]["Fiscal Balance Observation Count"] == 1\n'
    '        and fiscal_rows[4]["Fiscal Balance Observation Count"] == 3\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        fiscal_rows[3]["1-Period Fiscal Balance Change"],\n'
    '        fiscal_rows[3]["3-Period Moving Average Fiscal Balance"]\n'
    '        - fiscal_rows[2]["3-Period Moving Average Fiscal Balance"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        fiscal_rows[5]["3-Period Fiscal Balance Change"],\n'
    '        fiscal_rows[5]["3-Period Moving Average Fiscal Balance"]\n'
    '        - fiscal_rows[2]["3-Period Moving Average Fiscal Balance"],\n'
    '    )\n'
    '    assert fiscal_rows[6]["5-Period Fiscal Balance Change"] == ""\n'
    '    assert values_equivalent(\n'
    '        fiscal_rows[7]["5-Period Fiscal Balance Change"],\n'
    '        fiscal_rows[7]["3-Period Moving Average Fiscal Balance"]\n'
    '        - fiscal_rows[2]["3-Period Moving Average Fiscal Balance"],\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[20]["Fiscal Balance Level PercentRank"] == ""\n'
    '        and fiscal_rows[21]["Fiscal Balance Level PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[21]["1-Period Fiscal Balance Change PercentRank"] == ""\n'
    '        and fiscal_rows[22]["1-Period Fiscal Balance Change PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[23]["3-Period Fiscal Balance Change PercentRank"] == ""\n'
    '        and fiscal_rows[24]["3-Period Fiscal Balance Change PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[25]["5-Period Fiscal Balance Change PercentRank"] == ""\n'
    '        and fiscal_rows[26]["5-Period Fiscal Balance Change PercentRank"] != ""\n'
    '    )\n'
    '    assert fiscal_rows[21]["Fiscal Balance Level Score"] == round(\n'
    '        fiscal_rows[21]["Fiscal Balance Level PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert fiscal_rows[22]["1-Period Fiscal Balance Change Score"] == round(\n'
    '        fiscal_rows[22]["1-Period Fiscal Balance Change PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert _fiscal_balance_regime(0, 1) == "Surplus and improving"\n'
    '    assert _fiscal_balance_regime(-3, 1) == "Moderate deficit but improving"\n'
    '    assert _fiscal_balance_regime(-6, -1) == "Large deficit and deteriorating"\n'
    '    assert _fiscal_balance_regime(-6.1, 1) == "Severe deficit but improving"\n'
    '    assert _fiscal_balance_regime(1, "") == "Mixed fiscal balance regime"\n'
    '    assert _fiscal_impulse_bias_label(6) == "Strong fiscal expansion impulse"\n'
    '    assert (\n'
    '        fiscal_rows[20]["Fiscal Balance Level Z-Score"] == ""\n'
    '        and fiscal_rows[21]["Fiscal Balance Level Z-Score"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        fiscal_rows[21]["1-Period Fiscal Balance Change Z-Score"] == ""\n'
    '        and fiscal_rows[22]["1-Period Fiscal Balance Change Z-Score"] != ""\n'
    '    )\n'
    '    assert fiscal_rows[21]["Fiscal Balance Level Z-Score"] == round(\n'
    '        fiscal_rows[21]["Fiscal Balance Level Z-Score"], 2\n'
    '    )\n'
    '    assert (\n'
    '        _fiscal_interaction_score(-1.1, 5) == 5\n'
    '        and _fiscal_interaction_score(-0.6, 3) == 3\n'
    '        and _fiscal_interaction_score(1.1, -5) == -3\n'
    '    )\n'
    '    assert _fiscal_sustainability_risk_score(-8.1, 0, 6) == -7\n'
    '    assert _fiscal_sustainability_risk_score(-8, 0, 6) == -5\n'
    '    assert _fiscal_sustainability_risk_score(-6, 0, 5) == -3\n'
    '    assert _fiscal_sustainability_risk_score(-4, 0, 4) == 0\n'
    '    assert _fiscal_outlier_score([-3.1, 3.5, 0, 0]) == -6\n'
    '    assert _fiscal_outlier_label([-3.1, 3.5, 0, 0]) == "Extreme fiscal-deficit stress"\n'
    '    fiscal_row = fiscal_rows[50]\n'
    '    fbi = fiscal_row["Fiscal Balance Level Score"]\n'
    '    fbk = fiscal_row["1-Period Fiscal Balance Change Score"]\n'
    '    fbm = fiscal_row["3-Period Fiscal Balance Change Score"]\n'
    '    fbo = fiscal_row["5-Period Fiscal Balance Change Score"]\n'
    '    fbq = fiscal_row["Fiscal Expansion Impulse Composite Score"]\n'
    '    fbs = fiscal_row["Fiscal Impulse Momentum Score"]\n'
    '    fbt = fiscal_row["Fiscal Balance Interaction Score"]\n'
    '    fbu = fiscal_row["Fiscal Deficit Sustainability Risk Score"]\n'
    '    fbv = fiscal_row["Fiscal Balance Outlier Score"]\n'
    '    assert values_equivalent(fbq, round(clamp(-0.5 * fbk - 0.35 * fbm - 0.15 * fbo), 2))\n'
    '    assert values_equivalent(fbs, round(clamp(-0.5 * fbk - 0.3 * fbm - 0.2 * fbo), 2))\n'
    '    assert values_equivalent(\n'
    '        fiscal_row["Fiscal Impulse Leading Score"],\n'
    '        round(clamp(0.6 * fbq + 0.25 * fbs - 0.15 * fbi), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        fiscal_row["Fiscal Impulse Warning Composite Score"],\n'
    '        round(clamp(0.5 * fbq + 0.25 * fbs + 0.15 * fbt + 0.1 * fbu), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        fiscal_row["Fiscal Impulse Final Warning Score"],\n'
    '        round(clamp(0.25 * fbs + 0.2 * fbt + 0.35 * fbu + 0.2 * fbv), 2),\n'
    '    )\n'
    '    assert _fiscal_impulse_regime(2) == "Fiscal impulse expansionary"\n'
    '    assert _fiscal_momentum_label(2, 2) == "Fiscal impulse expansionary"\n'
    '    assert (\n'
    '        _fiscal_signal_interpretation(6, 0, -5)\n'
    '        == "Strong expansionary fiscal impulse, but severe deficit/sustainability risk is rising"\n'
    '    )\n'
    '    assert _fiscal_warning_label(6, -6) == "High deficit/sustainability warning"\n'
    '    assert fiscal_row["Fiscal Impulse Bias Label"] == _fiscal_impulse_bias_label(fbq)\n'
    '    assert fiscal_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert fiscal_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert fiscal_row["Fiscal Warning Risk Label"] != ""\n'
    '    assert fiscal_row["Fiscal Balance Outlier Label"] != ""\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped fiscal balance sheet", CONTROL_PANEL_THIRTIETH_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_fiscal_balance_outputs\n'
    '    )\n'
    '    all_specs_with_a32 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 32)] + [["Fiscal Balance"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Fiscal Balance", "A32") in all_specs_with_a32\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped central liquidity sheet",\n'
    '            CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL,\n'
    '        ).input_end_col\n'
    '        == "D"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped banking reserves sheet",\n'
    '            CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AQ"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped broad money sheet", CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AO"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped fiscal balance sheet", CONTROL_PANEL_THIRTIETH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AJ"\n'
    '    )\n'
    '    assert df_records(calculate_fiscal_balance_outputs(fiscal_raw)) == fiscal_rows\n'
    '    real_gov_values = [1000 + 12 * i + 50 * math.sin(i / 3) for i in range(120)]\n'
    '    real_gov_raw_rows = [\n'
    '        {"Date": f"g{i}", "Raw Value": real_gov_values[i]}\n'
    '        for i in range(len(real_gov_values))\n'
    '    ]\n'
    '    real_gov_raw_rows[10]["Raw Value"] = ""\n'
    '    real_gov_raw = make_dataframe(real_gov_raw_rows, ["Date", "Raw Value"])\n'
    '    real_gov_out = calculate_real_government_spending_outputs(real_gov_raw)\n'
    '    real_gov_rows = df_records(real_gov_out)\n'
    '    real_gov_spec = select_indicator_engine_spec(\n'
    '        "Unmapped real government spending sheet",\n'
    '        CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL,\n'
    '    )\n'
    '    assert real_gov_spec.input_start_col == "A" and real_gov_spec.input_end_col == "B"\n'
    '    assert (\n'
    '        real_gov_spec.output_start_col == "C" and real_gov_spec.output_end_col == "AM"\n'
    '    )\n'
    '    assert real_gov_out.shape[1] == len(REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS) == 37\n'
    '    assert output_end_col_for_columns(REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS) == "AM"\n'
    '    assert (\n'
    '        indicator_input_range("Real Government Spending")\n'
    '        == "\'Real Government Spending\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Real Government Spending", 678, "AM")\n'
    '        == "\'Real Government Spending\'!C2:AM678"\n'
    '    )\n'
    '    assert real_gov_rows[0]["2-Period Moving Average Real Government Spending"] == ""\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[1]["2-Period Moving Average Real Government Spending"],\n'
    '        sum(real_gov_values[:2]) / 2,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[10]["2-Period Moving Average Real Government Spending"],\n'
    '        real_gov_values[9],\n'
    '    )\n'
    '    assert (\n'
    '        real_gov_rows[0]["Real Government Spending Observation Count"] == 0\n'
    '        and real_gov_rows[1]["Real Government Spending Observation Count"] == 1\n'
    '        and real_gov_rows[2]["Real Government Spending Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[2]["1-Period Real Government Spending Growth"],\n'
    '        real_gov_rows[2]["2-Period Moving Average Real Government Spending"]\n'
    '        / real_gov_rows[1]["2-Period Moving Average Real Government Spending"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[3]["2-Period Real Government Spending Growth"],\n'
    '        real_gov_rows[3]["2-Period Moving Average Real Government Spending"]\n'
    '        / real_gov_rows[1]["2-Period Moving Average Real Government Spending"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[5]["4-Period Real Government Spending Growth"],\n'
    '        real_gov_rows[5]["2-Period Moving Average Real Government Spending"]\n'
    '        / real_gov_rows[1]["2-Period Moving Average Real Government Spending"]\n'
    '        - 1,\n'
    '    )\n'
    '    zero_real_gov_raw = make_dataframe(\n'
    '        [{"Date": f"zg{i}", "Raw Value": 0} for i in range(8)], ["Date", "Raw Value"]\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_real_government_spending_outputs(zero_real_gov_raw))[2][\n'
    '            "1-Period Real Government Spending Growth"\n'
    '        ]\n'
    '        == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[2]["1-Period Annualised Real Government Spending Growth"],\n'
    '        (1 + real_gov_rows[2]["1-Period Real Government Spending Growth"]) ** 4 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_rows[3]["2-Period Annualised Real Government Spending Growth"],\n'
    '        (1 + real_gov_rows[3]["2-Period Real Government Spending Growth"]) ** 2 - 1,\n'
    '    )\n'
    '    assert (\n'
    '        real_gov_rows[40]["1-Period Real Government Spending Growth PercentRank"] == ""\n'
    '        and real_gov_rows[41]["1-Period Real Government Spending Growth PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert (\n'
    '        real_gov_rows[41]["2-Period Real Government Spending Growth PercentRank"] == ""\n'
    '        and real_gov_rows[42]["2-Period Real Government Spending Growth PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert (\n'
    '        real_gov_rows[43]["4-Period Real Government Spending Growth PercentRank"] == ""\n'
    '        and real_gov_rows[44]["4-Period Real Government Spending Growth PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert (\n'
    '        real_gov_rows[40]["Annualised Real Government Spending Growth PercentRank"]\n'
    '        == ""\n'
    '        and real_gov_rows[41]["Annualised Real Government Spending Growth PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert real_gov_rows[41][\n'
    '        "Annualised Real Government Spending Growth PercentRank"\n'
    '    ] == percentrank_inc(\n'
    '        [\n'
    '            r["1-Period Annualised Real Government Spending Growth"]\n'
    '            for r in real_gov_rows[:42]\n'
    '            if to_float_or_none(\n'
    '                r["1-Period Annualised Real Government Spending Growth"]\n'
    '            )\n'
    '            is not None\n'
    '        ],\n'
    '        real_gov_rows[41]["1-Period Annualised Real Government Spending Growth"],\n'
    '    )\n'
    '    assert real_gov_rows[41]["1-Period Real Government Spending Growth Score"] == round(\n'
    '        real_gov_rows[41]["1-Period Real Government Spending Growth PercentRank"] * 20\n'
    '        - 10,\n'
    '        2,\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_spending_regime(1, 0.1, 0.1, 0.1)\n'
    '        == "Real government spending expansion regime"\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_spending_regime(1, -0.1, -0.1, -0.1)\n'
    '        == "Real government spending contraction regime"\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_spending_regime(1, 0, 0.1, 0.1)\n'
    '        == "Neutral/mixed government spending regime"\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_asset_bias_label(6) == "Strong fiscal-demand support for GDP growth"\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_shock_label(0.10) == "Large quarterly government-spending surge"\n'
    '        and _real_gov_shock_label(-0.10)\n'
    '        == "Large quarterly government-spending contraction"\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_shock_score(0.10) == 6\n'
    '        and _real_gov_shock_score(0.05) == 4\n'
    '        and _real_gov_shock_score(-0.10) == -6\n'
    '        and _real_gov_shock_score(-0.05) == -4\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_recovery_score(0.1, 0.1) == 5\n'
    '        and _real_gov_recovery_score(-0.1, 0.1) == -3\n'
    '    )\n'
    '    assert (\n'
    '        real_gov_rows[40]["1-Period Real Government Spending Growth Z-Score"] == ""\n'
    '        and real_gov_rows[41]["1-Period Real Government Spending Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert real_gov_rows[41]["Annualised Real Government Spending Growth Z-Score"] != ""\n'
    '    assert real_gov_rows[41][\n'
    '        "1-Period Real Government Spending Growth Z-Score"\n'
    '    ] == round(real_gov_rows[41]["1-Period Real Government Spending Growth Z-Score"], 2)\n'
    '    assert _real_gov_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _real_gov_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme government-spending expansion"\n'
    '    )\n'
    '    real_gov_row = real_gov_rows[60]\n'
    '    rgk = real_gov_row["1-Period Real Government Spending Growth Score"]\n'
    '    rgm = real_gov_row["2-Period Real Government Spending Growth Score"]\n'
    '    rgo = real_gov_row["4-Period Real Government Spending Growth Score"]\n'
    '    rgq = real_gov_row["Annualised Real Government Spending Growth Score"]\n'
    '    rgu = real_gov_row["Real Government Spending Momentum Composite Score"]\n'
    '    rgv = real_gov_row["Real Government Spending Momentum Turn Score"]\n'
    '    rgw = real_gov_row["Real Government Spending Recovery / Deterioration Score"]\n'
    '    rgx = real_gov_row["Real Government Spending Shock Score"]\n'
    '    rgy = real_gov_row["Real Government Spending Outlier Score"]\n'
    '    assert values_equivalent(\n'
    '        rgu, round(clamp(0.25 * rgk + 0.35 * rgm + 0.25 * rgo + 0.15 * rgq), 2)\n'
    '    )\n'
    '    assert real_gov_row["Real Government Spending Leading Score"] == rgu\n'
    '    assert values_equivalent(\n'
    '        rgv,\n'
    '        round(\n'
    '            clamp(\n'
    '                (\n'
    '                    rgu\n'
    '                    - real_gov_rows[59][\n'
    '                        "Real Government Spending Momentum Composite Score"\n'
    '                    ]\n'
    '                )\n'
    '                * 2\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_row["Real Government Spending Warning Composite Score"],\n'
    '        round(clamp(0.5 * rgu + 0.3 * rgv + 0.2 * rgw), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        real_gov_row["Real Government Spending Final Warning Score"],\n'
    '        round(clamp(0.25 * rgv + 0.25 * rgw + 0.3 * rgx + 0.2 * rgy), 2),\n'
    '    )\n'
    '    assert real_gov_row[\n'
    '        "Real Government Spending Asset Bias Label"\n'
    '    ] == _real_gov_asset_bias_label(\n'
    '        real_gov_row["Real Government Spending Leading Score"]\n'
    '    )\n'
    '    assert _real_gov_conditions_regime(2) == "Real government spending growth improving"\n'
    '    assert _real_gov_momentum_label(2, 2) == "Real government spending growth improving"\n'
    '    assert (\n'
    '        _real_gov_signal_interpretation(6, 0)\n'
    '        == "Strong positive government-spending impulse with supportive fiscal-demand profile"\n'
    '    )\n'
    '    assert (\n'
    '        _real_gov_warning_label(6) == "High government-spending expansion/shock warning"\n'
    '    )\n'
    '    assert real_gov_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert real_gov_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert real_gov_row["Real Government Spending Warning Risk Label"] != ""\n'
    '    assert real_gov_row["Real Government Spending Outlier Label"] != ""\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped real government spending sheet",\n'
    '            CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_real_government_spending_outputs\n'
    '    )\n'
    '    all_specs_with_a33 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 33)] + [["Real Government Spending"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Real Government Spending", "A33") in all_specs_with_a33\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped central liquidity sheet",\n'
    '            CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL,\n'
    '        ).input_end_col\n'
    '        == "D"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped banking reserves sheet",\n'
    '            CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AQ"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped broad money sheet", CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AO"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped fiscal balance sheet", CONTROL_PANEL_THIRTIETH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AJ"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped real government spending sheet",\n'
    '            CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AM"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_real_government_spending_outputs(real_gov_raw))\n'
    '        == real_gov_rows\n'
    '    )\n'
    '    # A34 fixed protected-source fixture: October 1980 through April 1987.\n'
    '    gov_revenue_values = [\n'
    '        38923,\n'
    '        39175,\n'
    '        48904,\n'
    '        51147,\n'
    '        38129,\n'
    '        44357,\n'
    '        74188,\n'
    '        38243,\n'
    '        70429,\n'
    '        47829,\n'
    '        47669,\n'
    '        60279,\n'
    '        45467,\n'
    '        44317,\n'
    '        57407,\n'
    '        55269,\n'
    '        43042,\n'
    '        45291,\n'
    '        75777,\n'
    '        36753,\n'
    '        66353,\n'
    '        44675,\n'
    '        44924,\n'
    '        59694,\n'
    '        40539,\n'
    '        42007,\n'
    '        54498,\n'
    '        57505,\n'
    '        38816,\n'
    '        43504,\n'
    '        66234,\n'
    '        33755,\n'
    '        66517,\n'
    '        43948,\n'
    '        49683,\n'
    '        63556,\n'
    '        45157,\n'
    '        46202,\n'
    '        58044,\n'
    '        62537,\n'
    '        47886,\n'
    '        44464,\n'
    '        80180,\n'
    '        37459,\n'
    '        69282,\n'
    '        52017,\n'
    '        55209,\n'
    '        68019,\n'
    '        52251,\n'
    '        51494,\n'
    '        62404,\n'
    '        70454,\n'
    '        54049,\n'
    '        49613,\n'
    '        94599,\n'
    '        39794,\n'
    '        72151,\n'
    '        57650,\n'
    '        55781,\n'
    '        73811,\n'
    '        57886,\n'
    '        51166,\n'
    '        68196,\n'
    '        76698,\n'
    '        53370,\n'
    '        49557,\n'
    '        91438,\n'
    '        46246,\n'
    '        77024,\n'
    '        62974,\n'
    '        56523,\n'
    '        78013,\n'
    '        59012,\n'
    '        52967,\n'
    '        78035,\n'
    '        81771,\n'
    '        55463,\n'
    '        56515,\n'
    '        122897,\n'
    '    ]\n'
    '\n'
    '    def gov_revenue_month_end(offset: int) -> date:\n'
    '        month_index = 9 + offset\n'
    '        year = 1980 + month_index // 12\n'
    '        month = 1 + month_index % 12\n'
    '        next_month = date(year + 1, 1, 1) if month == 12 else date(year, month + 1, 1)\n'
    '        return next_month - timedelta(days=1)\n'
    '\n'
    '    gov_revenue_dates = [\n'
    '        gov_revenue_month_end(i) for i in range(len(gov_revenue_values))\n'
    '    ]\n'
    '    gov_revenue_raw_rows = [\n'
    '        {"Date": dt.isoformat(), "Raw Value": value}\n'
    '        for dt, value in zip(gov_revenue_dates, gov_revenue_values)\n'
    '    ]\n'
    '    gov_revenue_raw = make_dataframe(gov_revenue_raw_rows, ["Date", "Raw Value"])\n'
    '    gov_revenue_out = calculate_government_revenue_outputs(gov_revenue_raw)\n'
    '    gov_revenue_rows = df_records(gov_revenue_out)\n'
    '    gov_revenue_spec = select_indicator_engine_spec(\n'
    '        "Unmapped government revenue sheet", CONTROL_PANEL_THIRTY_SECOND_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped government revenue sheet",\n'
    '            CONTROL_PANEL_THIRTY_SECOND_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_government_revenue_outputs\n'
    '    )\n'
    '    assert gov_revenue_spec.calculate is calculate_government_revenue_outputs\n'
    '    assert (\n'
    '        gov_revenue_spec.input_start_col == "A"\n'
    '        and gov_revenue_spec.input_end_col == "B"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_spec.output_start_col == "C"\n'
    '        and gov_revenue_spec.output_end_col == "AP"\n'
    '    )\n'
    '    assert list(gov_revenue_out.columns) == GOVERNMENT_REVENUE_OUTPUT_COLUMNS\n'
    '    assert gov_revenue_out.shape[1] == len(GOVERNMENT_REVENUE_OUTPUT_COLUMNS) == 40\n'
    '    assert GOVERNMENT_REVENUE_OUTPUT_COLUMNS[7] == "6M Annualised Revneue Growth"\n'
    '    assert output_end_col_for_columns(GOVERNMENT_REVENUE_OUTPUT_COLUMNS) == "AP"\n'
    '    assert indicator_input_range("Government Revenue") == "\'Government Revenue\'!A2:B"\n'
    '    assert (\n'
    '        indicator_output_range("Government Revenue", 80, "AP")\n'
    '        == "\'Government Revenue\'!C2:AP80"\n'
    '    )\n'
    '\n'
    '    # C/D and physical-row growth/annualisation chain.\n'
    '    assert gov_revenue_rows[10]["12M rolling Government Revenue"] == ""\n'
    '    assert (\n'
    '        gov_revenue_dates[11] == date(1981, 9, 30)\n'
    '        and gov_revenue_rows[11]["12M rolling Government Revenue"] == 599272\n'
    '    )\n'
    '    assert gov_revenue_rows[11]["Sample Count"] == 1\n'
    '    assert (\n'
    '        gov_revenue_dates[12] == date(1981, 10, 31)\n'
    '        and gov_revenue_rows[12]["12M rolling Government Revenue"] == 605816\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_rows[12]["Sample Count"] == 2\n'
    '        and gov_revenue_rows[12]["1M Revenue Growth"] == "1.09%"\n'
    '    )\n'
    '    c_values = [row["12M rolling Government Revenue"] for row in gov_revenue_rows]\n'
    '    d_values = [row["Sample Count"] for row in gov_revenue_rows]\n'
    '\n'
    '    def revenue_growth_series(lag: int, gate: int) -> list[Any]:\n'
    '        return [\n'
    '            _gov_revenue_growth(\n'
    '                current,\n'
    '                c_values[idx - lag] if idx >= lag else "",\n'
    '                d_values[idx] >= gate,\n'
    '            )\n'
    '            for idx, current in enumerate(c_values)\n'
    '        ]\n'
    '\n'
    '    e_internal = revenue_growth_series(1, 2)\n'
    '    f_internal = revenue_growth_series(3, 4)\n'
    '    g_internal = revenue_growth_series(6, 7)\n'
    '    h_internal = revenue_growth_series(12, 13)\n'
    '    i_internal = [_gov_revenue_annualized_growth(value, 4) for value in f_internal]\n'
    '    j_internal = [_gov_revenue_annualized_growth(value, 2) for value in g_internal]\n'
    '    for idx, internal, output_col in (\n'
    '        (12, e_internal, "1M Revenue Growth"),\n'
    '        (14, f_internal, "3M Revenue Growth"),\n'
    '        (17, g_internal, "6M Revenue Growth"),\n'
    '        (23, h_internal, "12M YoY Revenue Growth"),\n'
    '    ):\n'
    '        assert gov_revenue_rows[idx][output_col] == _gov_revenue_percent_display(\n'
    '            internal[idx]\n'
    '        )\n'
    '        assert isinstance(gov_revenue_rows[idx][output_col], str) and gov_revenue_rows[\n'
    '            idx\n'
    '        ][output_col].endswith("%")\n'
    '    assert (\n'
    '        gov_revenue_dates[14] == date(1981, 12, 31)\n'
    '        and gov_revenue_rows[14]["3M Revenue Growth"] == "3.37%"\n'
    '        and gov_revenue_rows[14]["3M Annualised Revenue Growth"] == "14.17%"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[17] == date(1982, 3, 31)\n'
    '        and gov_revenue_rows[17]["6M Revenue Growth"] == "5.03%"\n'
    '        and gov_revenue_rows[17]["6M Annualised Revneue Growth"] == "10.32%"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[23] == date(1982, 9, 30)\n'
    '        and gov_revenue_rows[23]["12M YoY Revenue Growth"] == "3.29%"\n'
    '    )\n'
    '    assert gov_revenue_rows[14][\n'
    '        "3M Annualised Revenue Growth"\n'
    '    ] == _gov_revenue_percent_display(i_internal[14])\n'
    '    assert gov_revenue_rows[17][\n'
    '        "6M Annualised Revneue Growth"\n'
    '    ] == _gov_revenue_percent_display(j_internal[17])\n'
    '    zero_revenue_raw = make_dataframe(\n'
    '        [{"Date": f"zr{i}", "Raw Value": 0} for i in range(30)], ["Date", "Raw Value"]\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_government_revenue_outputs(zero_revenue_raw))[12][\n'
    '            "1M Revenue Growth"\n'
    '        ]\n'
    '        == ""\n'
    '    )\n'
    '\n'
    '    # K/M/O/Q use metric-specific 40-value gates and rank F/G/H/I.\n'
    '    for before, first, percentile_col in (\n'
    '        (52, 53, "3M Growth Percentile"),\n'
    '        (55, 56, "6M Growth Percentile"),\n'
    '        (61, 62, "12M YoY Growth Percentile"),\n'
    '        (52, 53, "3M Annualised Growth Percentile"),\n'
    '    ):\n'
    '        assert gov_revenue_rows[before][percentile_col] == ""\n'
    '        assert gov_revenue_rows[first][percentile_col] != ""\n'
    '    for source_values, percentile_col, score_col in (\n'
    '        (f_internal, "3M Growth Percentile", "3M Growth Score"),\n'
    '        (g_internal, "6M Growth Percentile", "6M Growth Score"),\n'
    '        (h_internal, "12M YoY Growth Percentile", "12M YoY Growth Score"),\n'
    '        (i_internal, "3M Annualised Growth Percentile", "3M Annualised Growth Score"),\n'
    '    ):\n'
    '        history = [value for value in source_values if not is_blank(value)]\n'
    '        expected_rank = _gov_revenue_sheets_round(\n'
    '            _gov_revenue_percentrank_inc_sheets(history, history[-1]), 3\n'
    '        )\n'
    '        assert gov_revenue_rows[-1][percentile_col] == _gov_revenue_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert gov_revenue_rows[-1][score_col] == _gov_revenue_score_percentile(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert gov_revenue_rows[-1][percentile_col].endswith("%")\n'
    '    mar_1985 = gov_revenue_rows[53]\n'
    '    jun_1985 = gov_revenue_rows[56]\n'
    '    dec_1985 = gov_revenue_rows[62]\n'
    '    assert gov_revenue_dates[53] == date(1985, 3, 31) and [\n'
    '        mar_1985[col]\n'
    '        for col in (\n'
    '            "3M Growth Percentile",\n'
    '            "3M Growth Score",\n'
    '            "3M Annualised Growth Percentile",\n'
    '            "3M Annualised Growth Score",\n'
    '        )\n'
    '    ] == ["82.10%", 6.42, "82.10%", 6.42]\n'
    '    assert gov_revenue_dates[56] == date(1985, 6, 30) and [\n'
    '        jun_1985[col]\n'
    '        for col in (\n'
    '            "3M Growth Percentile",\n'
    '            "3M Growth Score",\n'
    '            "6M Growth Percentile",\n'
    '            "6M Growth Score",\n'
    '            "3M Annualised Growth Percentile",\n'
    '            "3M Annualised Growth Score",\n'
    '        )\n'
    '    ] == ["76.20%", 5.24, "82.10%", 6.42, "76.20%", 5.24]\n'
    '    assert gov_revenue_dates[62] == date(1985, 12, 31) and [\n'
    '        dec_1985[col]\n'
    '        for col in (\n'
    '            "3M Growth Percentile",\n'
    '            "3M Growth Score",\n'
    '            "6M Growth Percentile",\n'
    '            "6M Growth Score",\n'
    '            "12M YoY Growth Percentile",\n'
    '            "12M YoY Growth Score",\n'
    '            "3M Annualised Growth Percentile",\n'
    '            "3M Annualised Growth Score",\n'
    '        )\n'
    '    ] == ["41.70%", -1.66, "46.70%", -0.66, "56.40%", 1.28, "41.70%", -1.66]\n'
    '\n'
    '    # Target regime, shock, turning-point, and shock-score branches.\n'
    '    assert (\n'
    '        gov_revenue_rows[11]["Revenue Growth Regime"]\n'
    '        == "Government revenue expansion regime"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[20] == date(1982, 6, 30)\n'
    '        and gov_revenue_rows[20]["Revenue Growth Regime"]\n'
    '        == "Short-term revenue deterioration / mixed impulse"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[21] == date(1982, 7, 31)\n'
    '        and gov_revenue_rows[21]["Revenue Growth Regime"]\n'
    '        == "Revenue deterioration from high base"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[26] == date(1982, 12, 31)\n'
    '        and gov_revenue_rows[26]["Revenue Growth Regime"]\n'
    '        == "Government revenue contraction regime"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[34] == date(1983, 8, 31)\n'
    '        and gov_revenue_rows[34]["Revenue Growth Regime"]\n'
    '        == "Short-term revenue recovery / mixed impulse"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[36] == date(1983, 10, 31)\n'
    '        and gov_revenue_rows[36]["Revenue Growth Regime"] == "Early revenue recovery"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_rows[14]["Government Revenue Shock Flag"]\n'
    '        == "Moderate revenue-growth acceleration"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[30] == date(1983, 4, 30)\n'
    '        and gov_revenue_rows[30]["Government Revenue Shock Flag"]\n'
    '        == "Moderate revenue-growth slowdown"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[42] == date(1984, 4, 30)\n'
    '        and gov_revenue_rows[42]["Government Revenue Shock Flag"]\n'
    '        == "Large revenue-growth surge"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_growth_regime(1, 0.1, 0.1, 0.1)\n'
    '        == "Government revenue expansion regime"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_growth_regime(1, -0.1, -0.1, -0.1)\n'
    '        == "Government revenue contraction regime"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_growth_regime(1, 0, 0.1, 0.1)\n'
    '        == "Neutral/mixed revenue-growth regime"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_asset_bias_label(6)\n'
    '        == "Strong fiscal-revenue support / improving tax-base backdrop"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_shock_label(0.15) == "Large revenue-growth surge"\n'
    '        and _gov_revenue_shock_label(-0.15) == "Large revenue-growth contraction"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_shock_score(0.15) == 6\n'
    '        and _gov_revenue_shock_score(0.08) == 4\n'
    '        and _gov_revenue_shock_score(-0.15) == -6\n'
    '        and _gov_revenue_shock_score(-0.08) == -4\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_recovery_score(0.1, 0.1) == 5\n'
    '        and _gov_revenue_recovery_score(-0.1, 0.1) == -3\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_rows[52]["3M Growth Z-Score"] == ""\n'
    '        and gov_revenue_rows[53]["3M Growth Z-Score"] != ""\n'
    '    )\n'
    '    assert gov_revenue_rows[53]["3M Annualised Growth Z-Score"] != ""\n'
    '    assert gov_revenue_rows[53]["3M Growth Z-Score"] == _gov_revenue_sheets_round(\n'
    '        gov_revenue_rows[53]["3M Growth Z-Score"], 2\n'
    '    )\n'
    '    for source_values, zscore_col in (\n'
    '        (f_internal, "3M Growth Z-Score"),\n'
    '        (g_internal, "6M Growth Z-Score"),\n'
    '        (h_internal, "12M Growth Z-Score"),\n'
    '        (i_internal, "3M Annualised Growth Z-Score"),\n'
    '    ):\n'
    '        expected_zscores = _gov_revenue_rounded_expanding_zscores(\n'
    '            source_values, _metric_counts(source_values)\n'
    '        )\n'
    '        assert gov_revenue_rows[-1][zscore_col] == expected_zscores[-1]\n'
    '    assert _gov_revenue_deterioration_risk("", 0, 0, 0) == ""\n'
    '    assert _gov_revenue_deterioration_risk(3, 0, 0, 0) == 6.0\n'
    '    assert _gov_revenue_deterioration_risk(2, 0, 0, 0) == 4.0\n'
    '    assert _gov_revenue_deterioration_risk(-3, 0, 0, 0) == -6.0\n'
    '    assert _gov_revenue_deterioration_risk(-2, 0, 0, 0) == -4.0\n'
    '    assert _gov_revenue_deterioration_risk(-1.99, 1.99, 0, 0) == 0.0\n'
    '    assert _gov_revenue_windfall_score(1.6, 1.6, 0, 0.16) == 5\n'
    '    assert _gov_revenue_windfall_score(1.1, 1.1, "", 0.11) == ""\n'
    '    assert _gov_revenue_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _gov_revenue_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme government-revenue growth"\n'
    '    )\n'
    '    for row in gov_revenue_rows:\n'
    '        weakness = row["Revenue Weakness Risk Score"]\n'
    '        extreme = row["Government Revenue Extreme Score"]\n'
    '        if not is_blank(weakness) and not is_blank(extreme):\n'
    '            assert (\n'
    '                weakness == extreme\n'
    '                and isinstance(weakness, float)\n'
    '                and isinstance(extreme, float)\n'
    '            )\n'
    '\n'
    '    # Corrected warning-risk construction and downstream labels/comments.\n'
    '    zscore_cases = [\n'
    '        ((-1.75, -1.46, -0.83, -1.72), 0.0),\n'
    '        ((-2.78, -1.65, -0.80, -2.68), -4.0),\n'
    '        ((-0.97, -2.98, -2.32, -0.98), -4.0),\n'
    '        ((-0.70, -1.55, -3.13, -0.74), -6.0),\n'
    '        ((2.26, 1.61, 1.01, 2.40), 4.0),\n'
    '        ((2.65, -1.69, -0.81, 3.02), 6.0),\n'
    '        ((4.08, -1.75, -0.95, 4.85), 6.0),\n'
    '    ]\n'
    '    for zscores, expected in zscore_cases:\n'
    '        assert _gov_revenue_deterioration_risk(*zscores) == expected\n'
    '        assert _gov_revenue_outlier_score(zscores) == expected\n'
    '    warning_score_cases = [\n'
    '        ((-1.82, -5, 0, 0, -3, 0), -1.66),\n'
    '        ((0.84, -5, 0, 0, -3, 0), -1.13),\n'
    '        ((-2.88, -5, 0, -4, -3, -4), -3.08),\n'
    '        ((1.84, 3, 0, -4, 0, -4), -0.23),\n'
    '        ((-10, 3, 0, -6, 0, -6), -3.20),\n'
    '        ((10, 5, 6, 4, 5, 4), 5.90),\n'
    '        ((0, 3, 6, 6, 0, 6), 3.60),\n'
    '    ]\n'
    '    for operands, expected in warning_score_cases:\n'
    '        assert _gov_revenue_warning_score(*operands) == expected\n'
    '        expected_formula = _gov_revenue_sheets_round(\n'
    '            clamp(\n'
    '                _gov_revenue_weighted_sum(\n'
    '                    (0.20, operands[0]),\n'
    '                    (0.20, operands[1]),\n'
    '                    (0.20, operands[2]),\n'
    '                    (0.15, operands[3]),\n'
    '                    (0.10, operands[4]),\n'
    '                    (0.15, operands[5]),\n'
    '                )\n'
    '            ),\n'
    '            2,\n'
    '        )\n'
    '        assert _gov_revenue_warning_score(*operands) == expected_formula\n'
    '    assert _gov_revenue_warning_score("", 0, 0, 0, 0, 0) == ""\n'
    '    assert _gov_revenue_warning_label(2.54) == "Neutral revenue-growth warning profile"\n'
    '    assert _gov_revenue_warning_label(2.48) == "Neutral revenue-growth warning profile"\n'
    '    assert _gov_revenue_warning_label(-2.60) == "Neutral revenue-growth warning profile"\n'
    '    assert _gov_revenue_warning_label(-3.20) == "Moderate revenue-weakness warning"\n'
    '    assert (\n'
    '        _gov_revenue_warning_label(5.90) == "Moderate revenue-growth strength warning"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(-3.21, -0.23)\n'
    '        == "Moderately negative revenue-growth impulse signal"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(-2.60, 0.75)\n'
    '        == "Revenue impulse deteriorating, but fiscal-receipts support remains mixed"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(-3.07, 0.51)\n'
    '        == "Revenue impulse deteriorating, but fiscal-receipts support remains mixed"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(-2.26, 0.72)\n'
    '        == "Revenue impulse deteriorating, but fiscal-receipts support remains mixed"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(0.66, 3.60)\n'
    '        == "Mixed/transition revenue-growth signal"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(0.51, 3.54)\n'
    '        == "Mixed/transition revenue-growth signal"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(1.97, 3.19)\n'
    '        == "Mixed/transition revenue-growth signal"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(2, -1)\n'
    '        == "Revenue impulse improving, but revenue-risk profile remains fragile"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(-6, 1)\n'
    '        == "Strong revenue deterioration signal, but fiscal-receipts support remains partly intact"\n'
    '    )\n'
    '    assert _gov_revenue_reliability("", 80) == ""\n'
    '    assert _gov_revenue_reliability(1, 39) == "Insufficient history"\n'
    '    assert _gov_revenue_reliability(1, 40) == "Medium reliability"\n'
    '    assert _gov_revenue_reliability(1, 79) == "Medium reliability"\n'
    '    assert _gov_revenue_reliability(1, 80) == "High reliability"\n'
    '    assert _gov_revenue_reliability(1, 118) == "High reliability"\n'
    '    jan_1986 = gov_revenue_rows[63]\n'
    '    apr_1987 = gov_revenue_rows[78]\n'
    '    assert (\n'
    '        gov_revenue_dates[50] == date(1984, 12, 31)\n'
    '        and gov_revenue_rows[50]["Sample Count"] == 40\n'
    '        and gov_revenue_rows[50]["Reliability"] == "Medium reliability"\n'
    '        and gov_revenue_rows[50]["Signal Validity"] == "Incomplete Score"\n'
    '    )\n'
    '    assert (\n'
    '        gov_revenue_dates[63] == date(1986, 1, 31)\n'
    '        and jan_1986["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert [\n'
    '        jan_1986[col]\n'
    '        for col in (\n'
    '            "Government Revenue Impulse Score",\n'
    '            "Government Revenue Acceleration Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '        )\n'
    '    ] == [-0.61, 0.22, -0.61, 0.76, 1.04]\n'
    '    assert jan_1986["Extreme Flag"] == "Normal range"\n'
    '    long_revenue_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": f"long-{idx}", "Raw Value": 100000 + idx * 1000}\n'
    '            for idx in range(131)\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    long_revenue_last = df_records(\n'
    '        calculate_government_revenue_outputs(long_revenue_raw)\n'
    '    )[-1]\n'
    '    assert (\n'
    '        long_revenue_last["Sample Count"] == 120\n'
    '        and long_revenue_last["Reliability"] == "High reliability"\n'
    '    )\n'
    '    assert gov_revenue_dates[78] == date(1987, 4, 30)\n'
    '    assert [\n'
    '        apr_1987[col]\n'
    '        for col in (\n'
    '            "Government Revenue Impulse Score",\n'
    '            "Government Revenue Acceleration Score",\n'
    '            "Coincident Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '        )\n'
    '    ] == [9.06, 10.0, 9.06, 8.53, 5.90]\n'
    '    assert apr_1987["Warning Flag"] == "Moderate revenue-growth strength warning"\n'
    '    rvl = apr_1987["3M Growth Score"]\n'
    '    rvn = apr_1987["6M Growth Score"]\n'
    '    rvp = apr_1987["12M YoY Growth Score"]\n'
    '    rvr = apr_1987["3M Annualised Growth Score"]\n'
    '    expected_rvv = _gov_revenue_sheets_round(\n'
    '        clamp(\n'
    '            _gov_revenue_weighted_sum(\n'
    '                (0.30, rvl), (0.35, rvn), (0.20, rvp), (0.15, rvr)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert (\n'
    '        apr_1987["Government Revenue Impulse Score"]\n'
    '        == apr_1987["Coincident Score"]\n'
    '        == expected_rvv\n'
    '    )\n'
    '    assert apr_1987["Fiscal Revenue Bias"] == _gov_revenue_asset_bias_label(\n'
    '        apr_1987["Coincident Score"]\n'
    '    )\n'
    '    assert _gov_revenue_conditions_regime(2) == "Government revenue growth improving"\n'
    '    assert _gov_revenue_momentum_label(2, 2) == "Government revenue growth improving"\n'
    '    assert (\n'
    '        _gov_revenue_signal_interpretation(6, 0)\n'
    '        == "Strong positive revenue-growth impulse with supportive fiscal-receipts profile"\n'
    '    )\n'
    '    assert (\n'
    '        _gov_revenue_warning_label(6)\n'
    '        == "High revenue-growth strength / windfall warning"\n'
    '    )\n'
    '    assert (\n'
    '        apr_1987["Reliability"] == "Medium reliability"\n'
    '        and apr_1987["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert apr_1987["Extreme Flag"] != ""\n'
    '\n'
    '    class FakeGovernmentRevenueWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    fake_revenue_write = FakeGovernmentRevenueWorksheet([])\n'
    '    assert (\n'
    '        write_indicator_outputs(\n'
    '            fake_revenue_write,\n'
    '            gov_revenue_out,\n'
    '            80,\n'
    '            GOVERNMENT_REVENUE_OUTPUT_COLUMNS,\n'
    '            "AP",\n'
    '            "C",\n'
    '        )\n'
    '        == "C2:AP80"\n'
    '    )\n'
    '    assert [(call[0], call[2]) for call in fake_revenue_write.update_calls] == [\n'
    '        ("C2:AP80", "RAW")\n'
    '    ]\n'
    '    fake_revenue_verify = FakeGovernmentRevenueWorksheet(\n'
    '        dataframe_to_sheet_rows(gov_revenue_out, GOVERNMENT_REVENUE_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_revenue_verify,\n'
    '        gov_revenue_out,\n'
    '        80,\n'
    '        GOVERNMENT_REVENUE_OUTPUT_COLUMNS,\n'
    '        "AP",\n'
    '        "C",\n'
    '    )\n'
    '    assert fake_revenue_verify.get_calls == [("C2:AP80", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            gov_revenue_out, 80, "C", 2, GOVERNMENT_REVENUE_OUTPUT_COLUMNS, "AP"\n'
    '        )\n'
    '        == "C2:AP80"\n'
    '    )\n'
    '\n'
    '    class FakeGovernmentRevenueWorkbook:\n'
    '        def __init__(self, worksheet: FakeGovernmentRevenueWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeGovernmentRevenueWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_revenue_input = [\n'
    '        [dt.isoformat(), value]\n'
    '        for dt, value in zip(gov_revenue_dates, gov_revenue_values)\n'
    '    ]\n'
    '    fake_revenue_read = FakeGovernmentRevenueWorksheet(fake_revenue_input)\n'
    '    fake_a34_result = process_indicator(\n'
    '        FakeGovernmentRevenueWorkbook(fake_revenue_read),\n'
    '        BatchIndicatorSpec("Government Revenue", "A34"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a34_result.ok and fake_a34_result.output_range == "C2:AP80"\n'
    '    assert fake_revenue_read.get_calls == [("A2:B", None)]\n'
    '    fake_revenue_no_verify = FakeGovernmentRevenueWorksheet(fake_revenue_input)\n'
    '    fake_a34_no_verify_result = process_indicator(\n'
    '        FakeGovernmentRevenueWorkbook(fake_revenue_no_verify),\n'
    '        BatchIndicatorSpec("Government Revenue", "A34"),\n'
    '        write=True,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert (\n'
    '        fake_a34_no_verify_result.ok\n'
    '        and fake_a34_no_verify_result.output_range == "C2:AP80"\n'
    '    )\n'
    '    assert fake_revenue_no_verify.get_calls == [("A2:B", None)]\n'
    '    assert [(call[0], call[2]) for call in fake_revenue_no_verify.update_calls] == [\n'
    '        ("C2:AP80", "RAW")\n'
    '    ]\n'
    '    all_specs_with_a34 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 34)] + [["Government Revenue"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Government Revenue", "A34") in all_specs_with_a34\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped central liquidity sheet",\n'
    '            CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL,\n'
    '        ).input_end_col\n'
    '        == "D"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped banking reserves sheet",\n'
    '            CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AQ"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped broad money sheet", CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AO"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped fiscal balance sheet", CONTROL_PANEL_THIRTIETH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AJ"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped real government spending sheet",\n'
    '            CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AM"\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped government revenue sheet",\n'
    '            CONTROL_PANEL_THIRTY_SECOND_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AP"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_government_revenue_outputs(gov_revenue_raw))\n'
    '        == gov_revenue_rows\n'
    '    )\n'
    '    public_debt_values = [35 + 0.45 * i + 3 * math.sin(i / 6) for i in range(140)]\n'
    '    public_debt_raw_rows = [\n'
    '        {"Date": f"pd{i}", "Raw Value": public_debt_values[i]}\n'
    '        for i in range(len(public_debt_values))\n'
    '    ]\n'
    '    public_debt_raw_rows[9]["Raw Value"] = ""\n'
    '    public_debt_raw = make_dataframe(public_debt_raw_rows, ["Date", "Raw Value"])\n'
    '    public_debt_out = calculate_public_debt_outputs(public_debt_raw)\n'
    '    public_debt_rows = df_records(public_debt_out)\n'
    '    public_debt_spec = select_indicator_engine_spec(\n'
    '        "Unmapped public debt sheet", CONTROL_PANEL_THIRTY_THIRD_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        public_debt_spec.input_start_col == "A"\n'
    '        and public_debt_spec.input_end_col == "B"\n'
    '    )\n'
    '    assert (\n'
    '        public_debt_spec.output_start_col == "C"\n'
    '        and public_debt_spec.output_end_col == "AN"\n'
    '    )\n'
    '    assert (\n'
    '        public_debt_spec.summary_level_col == "Public Debt Level Score"\n'
    '        and public_debt_spec.summary_leading_col == "Public Debt Leading Score"\n'
    '        and public_debt_spec.summary_warning_col == "Public Debt Final Warning Score"\n'
    '    )\n'
    '    assert public_debt_out.shape[1] == len(PUBLIC_DEBT_OUTPUT_COLUMNS) == 38\n'
    '    assert output_end_col_for_columns(PUBLIC_DEBT_OUTPUT_COLUMNS) == "AN"\n'
    '    assert indicator_input_range("Public Debt") == "\'Public Debt\'!A2:B"\n'
    '    assert indicator_output_range("Public Debt", 678, "AN") == "\'Public Debt\'!C2:AN678"\n'
    '    assert public_debt_rows[0]["2-Period Moving Average Public Debt"] == ""\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[1]["2-Period Moving Average Public Debt"],\n'
    '        sum(public_debt_values[:2]) / 2,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[9]["2-Period Moving Average Public Debt"],\n'
    '        public_debt_values[8],\n'
    '    )\n'
    '    assert (\n'
    '        public_debt_rows[1]["Public Debt Observation Count"] == 1\n'
    '        and public_debt_rows[2]["Public Debt Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[2]["1-Period Public Debt Change"],\n'
    '        public_debt_rows[2]["2-Period Moving Average Public Debt"]\n'
    '        - public_debt_rows[1]["2-Period Moving Average Public Debt"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[3]["2-Period Public Debt Change"],\n'
    '        public_debt_rows[3]["2-Period Moving Average Public Debt"]\n'
    '        - public_debt_rows[1]["2-Period Moving Average Public Debt"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[5]["4-Period Public Debt Change"],\n'
    '        public_debt_rows[5]["2-Period Moving Average Public Debt"]\n'
    '        - public_debt_rows[1]["2-Period Moving Average Public Debt"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[2]["1-Period Annualised Public Debt Change"],\n'
    '        public_debt_rows[2]["1-Period Public Debt Change"] * 4,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        public_debt_rows[3]["2-Period Annualised Public Debt Change"],\n'
    '        public_debt_rows[3]["2-Period Public Debt Change"] * 2,\n'
    '    )\n'
    '    assert (\n'
    '        public_debt_rows[39]["Public Debt Level PercentRank"] == ""\n'
    '        and public_debt_rows[40]["Public Debt Level PercentRank"] != ""\n'
    '    )\n'
    '    assert public_debt_rows[40]["Public Debt Level Score"] == round(\n'
    '        10 - public_debt_rows[40]["Public Debt Level PercentRank"] * 20, 2\n'
    '    )\n'
    '    assert public_debt_rows[40]["Public Debt Level Score"] < 0\n'
    '    assert public_debt_rows[41]["1-Period Public Debt Change Score"] < 10\n'
    '    assert _public_debt_regime(39, 0) == "Low debt and improving"\n'
    '    assert _public_debt_regime(40, 0) == "Moderate debt and improving"\n'
    '    assert _public_debt_regime(70, 1) == "High debt and rising"\n'
    '    assert _public_debt_regime(100, 0) == "Very high debt but improving"\n'
    '    assert _public_debt_bias_label(6) == "Strong fiscal-sustainability support"\n'
    '    assert (\n'
    '        _public_debt_shock_label(8) == "Large debt-ratio deterioration shock"\n'
    '        and _public_debt_shock_label(-8) == "Large debt-ratio improvement shock"\n'
    '    )\n'
    '    debt_row = public_debt_rows[82]\n'
    '    pdk = debt_row["Public Debt Level Score"]\n'
    '    pdm = debt_row["1-Period Public Debt Change Score"]\n'
    '    pdo = debt_row["2-Period Public Debt Change Score"]\n'
    '    pdq = debt_row["4-Period Public Debt Change Score"]\n'
    '    pdu = debt_row["Public Debt Momentum Composite Score"]\n'
    '    pdv = debt_row["Public Debt Momentum Turn Score"]\n'
    '    pdw = debt_row["Public Debt Level / Vulnerability Score"]\n'
    '    pdx = debt_row["Public Debt Recovery / Deterioration Score"]\n'
    '    pdy = debt_row["Public Debt Shock Score"]\n'
    '    pdz = debt_row["Public Debt Outlier Score"]\n'
    '    assert values_equivalent(\n'
    '        pdu, round(clamp(0.35 * pdk + 0.25 * pdm + 0.25 * pdo + 0.15 * pdq), 2)\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        pdv,\n'
    '        round(\n'
    '            clamp(\n'
    '                (pdu - public_debt_rows[81]["Public Debt Momentum Composite Score"]) * 2\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert debt_row["Public Debt Level Z-Score"] == round(\n'
    '        debt_row["Public Debt Level Z-Score"], 2\n'
    '    )\n'
    '    assert _public_debt_vulnerability_score(100, 0, 1, 1, 0) == -7\n'
    '    assert _public_debt_vulnerability_score(70, 0, 1, 1, 0) == -5\n'
    '    assert _public_debt_vulnerability_score(50, 1.6, 1, 1, 0) == -4\n'
    '    assert _public_debt_vulnerability_score(39, 0, -1, -1, "") == ""\n'
    '    assert (\n'
    '        _public_debt_recovery_score(-1, -1) == 5\n'
    '        and _public_debt_recovery_score(1, 1) == -5\n'
    '    )\n'
    '    assert (\n'
    '        _public_debt_shock_score(8) == -6\n'
    '        and _public_debt_shock_score(4) == -4\n'
    '        and _public_debt_shock_score(-8) == 6\n'
    '        and _public_debt_shock_score(-4) == 4\n'
    '    )\n'
    '    assert _public_debt_outlier_score([3.1, -3.5, 0, 0]) == -6\n'
    '    assert _public_debt_outlier_label([3.1, -3.5, 0, 0]) == "Extreme public-debt stress"\n'
    '    assert values_equivalent(\n'
    '        debt_row["Public Debt Leading Score"],\n'
    '        round(clamp(0.40 * pdk + 0.25 * pdm + 0.20 * pdo + 0.15 * pdq), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        debt_row["Public Debt Warning Composite Score"],\n'
    '        round(clamp(0.40 * pdu + 0.25 * pdv + 0.25 * pdw + 0.10 * pdx), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        debt_row["Public Debt Final Warning Score"],\n'
    '        round(clamp(0.20 * pdv + 0.35 * pdw + 0.15 * pdx + 0.20 * pdy + 0.10 * pdz), 2),\n'
    '    )\n'
    '    assert _public_debt_conditions_regime(2) == "Public debt conditions improving"\n'
    '    assert _public_debt_momentum_label(2, 2) == "Public debt conditions improving"\n'
    '    assert (\n'
    '        _public_debt_signal_interpretation(6, 0)\n'
    '        == "Strong public-debt sustainability improvement with supportive fiscal-space profile"\n'
    '    )\n'
    '    assert (\n'
    '        _public_debt_warning_label(0, -1, -1, -1, -6)\n'
    '        == "High public-debt vulnerability warning"\n'
    '    )\n'
    '    assert debt_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert debt_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert debt_row[\n'
    '        "Public Debt Fiscal-Sustainability Bias Label"\n'
    '    ] == _public_debt_bias_label(debt_row["Public Debt Leading Score"])\n'
    '    assert (\n'
    '        debt_row["Public Debt Warning Risk Label"] != ""\n'
    '        and debt_row["Public Debt Outlier Label"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped public debt sheet", CONTROL_PANEL_THIRTY_THIRD_INDICATOR_CELL\n'
    '        )\n'
    '        is calculate_public_debt_outputs\n'
    '    )\n'
    '    all_specs_with_a35 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 35)] + [["Public Debt"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Public Debt", "A35") in all_specs_with_a35\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped public debt sheet", CONTROL_PANEL_THIRTY_THIRD_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AN"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_public_debt_outputs(public_debt_raw)) == public_debt_rows\n'
    '    )\n'
    '    coverage_rows_raw = [\n'
    '        {\n'
    '            "Date": f"ic{i}",\n'
    '            "Raw Value": 120 + 4 * i + 8 * math.sin(i / 5),\n'
    '            "Source C": 20 + 0.5 * i,\n'
    '        }\n'
    '        for i in range(90)\n'
    '    ]\n'
    '    coverage_rows_raw[5]["Source C"] = 0\n'
    '    coverage_raw = make_dataframe(coverage_rows_raw, ["Date", "Raw Value", "Source C"])\n'
    '    coverage_out = calculate_interest_coverage_outputs(coverage_raw)\n'
    '    coverage_rows = df_records(coverage_out)\n'
    '    coverage_spec = select_indicator_engine_spec(\n'
    '        "Unmapped interest coverage sheet", CONTROL_PANEL_THIRTY_FOURTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert coverage_spec.input_start_col == "A" and coverage_spec.input_end_col == "C"\n'
    '    assert (\n'
    '        coverage_spec.output_start_col == "D" and coverage_spec.output_end_col == "AN"\n'
    '    )\n'
    '    assert (\n'
    '        coverage_spec.summary_level_col == "Interest Coverage Level Score"\n'
    '        and coverage_spec.summary_leading_col == "Interest Coverage Leading Score"\n'
    '        and coverage_spec.summary_warning_col == "Interest Coverage Final Warning Score"\n'
    '    )\n'
    '    assert coverage_out.shape[1] == len(INTEREST_COVERAGE_OUTPUT_COLUMNS) == 37\n'
    '    assert output_end_col_for_columns(INTEREST_COVERAGE_OUTPUT_COLUMNS) == "AN"\n'
    '    assert indicator_input_range("Interest Coverage", "C") == "\'Interest Coverage\'!A2:C"\n'
    '    assert (\n'
    '        indicator_output_range("Interest Coverage", 678, "AN", "D")\n'
    '        == "\'Interest Coverage\'!D2:AN678"\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_rows[0]["Raw Interest Coverage Ratio"],\n'
    '        coverage_rows_raw[0]["Raw Value"] / coverage_rows_raw[0]["Source C"],\n'
    '    )\n'
    '    assert coverage_rows[5]["Raw Interest Coverage Ratio"] == ""\n'
    '    assert (\n'
    '        coverage_rows[0]["3-Period Moving Average Interest Coverage"] == ""\n'
    '        and coverage_rows[1]["3-Period Moving Average Interest Coverage"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_rows[2]["3-Period Moving Average Interest Coverage"],\n'
    '        sum(r["Raw Interest Coverage Ratio"] for r in coverage_rows[:3]) / 3,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_rows[5]["3-Period Moving Average Interest Coverage"],\n'
    '        (\n'
    '            coverage_rows[3]["Raw Interest Coverage Ratio"]\n'
    '            + coverage_rows[4]["Raw Interest Coverage Ratio"]\n'
    '        )\n'
    '        / 2,\n'
    '    )\n'
    '    assert (\n'
    '        coverage_rows[2]["Interest Coverage Observation Count"] == 1\n'
    '        and coverage_rows[3]["Interest Coverage Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_rows[3]["1-Period Interest Coverage Change"],\n'
    '        coverage_rows[3]["3-Period Moving Average Interest Coverage"]\n'
    '        - coverage_rows[2]["3-Period Moving Average Interest Coverage"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_rows[6]["3-Period Interest Coverage Change"],\n'
    '        coverage_rows[6]["3-Period Moving Average Interest Coverage"]\n'
    '        - coverage_rows[3]["3-Period Moving Average Interest Coverage"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_rows[8]["5-Period Interest Coverage Change"],\n'
    '        coverage_rows[8]["3-Period Moving Average Interest Coverage"]\n'
    '        - coverage_rows[3]["3-Period Moving Average Interest Coverage"],\n'
    '    )\n'
    '    assert (\n'
    '        coverage_rows[6]["5-Period Interest Coverage Change"] == ""\n'
    '        and coverage_rows[7]["5-Period Interest Coverage Change"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        coverage_rows[20]["Interest Coverage Level PercentRank"] == ""\n'
    '        and coverage_rows[21]["Interest Coverage Level PercentRank"] != ""\n'
    '    )\n'
    '    assert coverage_rows[21]["Interest Coverage Level Score"] == round(\n'
    '        coverage_rows[21]["Interest Coverage Level PercentRank"] * 20 - 10, 2\n'
    '    )\n'
    '    assert _interest_coverage_regime(8, 1) == "Very strong coverage and improving"\n'
    '    assert _interest_coverage_regime(4, 0) == "Healthy coverage but weakening"\n'
    '    assert _interest_coverage_regime(2, 1) == "Thin coverage but improving"\n'
    '    assert _interest_coverage_regime(1.9, 0) == "Severe interest burden and worsening"\n'
    '    assert (\n'
    '        _interest_coverage_bias_label(6)\n'
    '        == "Strong fiscal-stress relief / strong debt-servicing capacity"\n'
    '    )\n'
    '    assert (\n'
    '        _interest_coverage_shock_label(2) == "Large improvement in interest coverage"\n'
    '        and _interest_coverage_shock_label(-2)\n'
    '        == "Large deterioration in interest coverage"\n'
    '    )\n'
    '    coverage_row = coverage_rows[50]\n'
    '    ick = coverage_row["Interest Coverage Level Score"]\n'
    '    icm = coverage_row["1-Period Interest Coverage Change Score"]\n'
    '    ico = coverage_row["3-Period Interest Coverage Change Score"]\n'
    '    icq = coverage_row["5-Period Interest Coverage Change Score"]\n'
    '    icu = coverage_row["Interest Coverage Momentum Composite Score"]\n'
    '    icv = coverage_row["Interest Coverage Momentum Turn Score"]\n'
    '    icw = coverage_row["Interest Coverage Level / Vulnerability Score"]\n'
    '    icx = coverage_row["Interest Coverage Recovery / Deterioration Score"]\n'
    '    icy = coverage_row["Interest Coverage Shock Score"]\n'
    '    icz = coverage_row["Interest Coverage Outlier Score"]\n'
    '    assert values_equivalent(\n'
    '        icu, round(clamp(0.35 * ick + 0.25 * icm + 0.25 * ico + 0.15 * icq), 2)\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        icv,\n'
    '        round(\n'
    '            clamp(\n'
    '                (icu - coverage_rows[49]["Interest Coverage Momentum Composite Score"])\n'
    '                * 2\n'
    '            ),\n'
    '            2,\n'
    '        ),\n'
    '    )\n'
    '    assert coverage_row["Interest Coverage Level Z-Score"] == round(\n'
    '        coverage_row["Interest Coverage Level Z-Score"], 2\n'
    '    )\n'
    '    assert _interest_coverage_vulnerability_score(1.9, 0, -1, -1, 0) == -7\n'
    '    assert _interest_coverage_vulnerability_score(3.9, 0, -1, -1, 0) == -5\n'
    '    assert _interest_coverage_vulnerability_score(5, -1.6, -1, -1, 0) == -4\n'
    '    assert _interest_coverage_vulnerability_score(8, 0, 1, 1, 0) == 4\n'
    '    assert (\n'
    '        _interest_coverage_recovery_score(1, 1) == 5\n'
    '        and _interest_coverage_recovery_score(-1, 1) == -3\n'
    '    )\n'
    '    assert (\n'
    '        _interest_coverage_shock_score(2) == 6\n'
    '        and _interest_coverage_shock_score(1) == 4\n'
    '        and _interest_coverage_shock_score(-2) == -6\n'
    '        and _interest_coverage_shock_score(-1) == -4\n'
    '    )\n'
    '    assert _interest_coverage_outlier_score([3.1, -3.5, 0, 0]) == 6\n'
    '    assert (\n'
    '        _interest_coverage_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme interest-coverage strength"\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_row["Interest Coverage Leading Score"],\n'
    '        round(clamp(0.40 * ick + 0.25 * icm + 0.20 * ico + 0.15 * icq), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_row["Interest Coverage Warning Composite Score"],\n'
    '        round(clamp(0.40 * icu + 0.25 * icv + 0.25 * icw + 0.10 * icx), 2),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        coverage_row["Interest Coverage Final Warning Score"],\n'
    '        round(clamp(0.20 * icv + 0.35 * icw + 0.15 * icx + 0.20 * icy + 0.10 * icz), 2),\n'
    '    )\n'
    '    assert _interest_coverage_conditions_regime(2) == "Interest coverage improving"\n'
    '    assert _interest_coverage_momentum_label(2, 2) == "Interest coverage improving"\n'
    '    assert (\n'
    '        _interest_coverage_signal_interpretation(6, 0)\n'
    '        == "Strong debt-servicing capacity improvement with supportive fiscal-stress profile"\n'
    '    )\n'
    '    assert (\n'
    '        _interest_coverage_warning_label(0, -1, -1, -1, -6)\n'
    '        == "High interest-coverage stress warning"\n'
    '    )\n'
    '    assert coverage_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert coverage_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert coverage_row[\n'
    '        "Interest Coverage Fiscal-Stress Bias Label"\n'
    '    ] == _interest_coverage_bias_label(coverage_row["Interest Coverage Leading Score"])\n'
    '    assert (\n'
    '        coverage_row["Interest Coverage Warning Risk Label"] != ""\n'
    '        and coverage_row["Interest Coverage Outlier Label"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped interest coverage sheet",\n'
    '            CONTROL_PANEL_THIRTY_FOURTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_interest_coverage_outputs\n'
    '    )\n'
    '    all_specs_with_a36 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 36)] + [["Interest Coverage"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Interest Coverage", "A36") in all_specs_with_a36\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped interest coverage sheet",\n'
    '            CONTROL_PANEL_THIRTY_FOURTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AN"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_interest_coverage_outputs(coverage_raw)) == coverage_rows\n'
    '    )\n'
    '    assert df_records(read_indicator_values([["d", 10, 2]], "C"))[0]["Source C"] == 2\n'
    '    financial_rows_raw = [\n'
    '        {"Date": "2020-01-01", "Raw Value": 1.0},\n'
    '        {"Date": "2020-01-02", "Raw Value": 3.0},\n'
    '        {"Date": "2020-01-02", "Raw Value": 5.0},\n'
    '        {"Date": "2020-01-03", "Raw Value": 7.0},\n'
    '        {"Date": "not-a-date", "Raw Value": 99.0},\n'
    '        {\n'
    '            "Date": (datetime(2020, 2, 1) - GOOGLE_SHEETS_DATE_EPOCH).days,\n'
    '            "Raw Value": -1.0,\n'
    '        },\n'
    '    ]\n'
    '    financial_rows_raw.extend(\n'
    '        {\n'
    '            "Date": (date(2020, 2, 2) + timedelta(days=i)).isoformat(),\n'
    '            "Raw Value": -0.8 + i * 0.05,\n'
    '        }\n'
    '        for i in range(150)\n'
    '    )\n'
    '    financial_raw = make_dataframe(financial_rows_raw, ["Date", "Raw Value"])\n'
    '    financial_out = calculate_financial_conditions_outputs(financial_raw)\n'
    '    financial_rows = df_records(financial_out)\n'
    '    financial_spec = select_indicator_engine_spec(\n'
    '        "Unmapped financial conditions sheet", CONTROL_PANEL_THIRTY_FIFTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("Unmapped financial conditions sheet", "A37")\n'
    '        is calculate_financial_conditions_outputs\n'
    '    )\n'
    '    assert financial_spec.calculate is calculate_financial_conditions_outputs\n'
    '    assert financial_spec.input_start_col == "A" and financial_spec.input_end_col == "B"\n'
    '    assert (\n'
    '        financial_spec.output_start_col == "C" and financial_spec.output_end_col == "AM"\n'
    '    )\n'
    '    assert financial_spec.summary_level_col == "Financial Conditions Level Stress Score"\n'
    '    assert (\n'
    '        financial_spec.summary_leading_col == "Leading Score"\n'
    '        and financial_spec.summary_warning_col == "Warning Score"\n'
    '    )\n'
    '    assert financial_spec.summary_status_col == "Signal Validity"\n'
    '    assert list(financial_out.columns) == FINANCIAL_CONDITIONS_OUTPUT_COLUMNS\n'
    '    assert financial_out.shape[1] == len(FINANCIAL_CONDITIONS_OUTPUT_COLUMNS) == 37\n'
    '    assert _financial_conditions_to_date_or_none("31/01/1971") == date(1971, 1, 31)\n'
    '    assert _financial_conditions_to_date_or_none("05/01/1971") == date(1971, 1, 5)\n'
    '    assert output_end_col_for_columns(FINANCIAL_CONDITIONS_OUTPUT_COLUMNS) == "AM"\n'
    '    assert (\n'
    '        indicator_input_range(\n'
    '            "Financial Conditions",\n'
    '            financial_spec.input_end_col,\n'
    '            financial_spec.input_start_col,\n'
    '        )\n'
    '        == "\'Financial Conditions\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range(\n'
    '            "Financial Conditions",\n'
    '            157,\n'
    '            financial_spec.output_end_col,\n'
    '            financial_spec.output_start_col,\n'
    '        )\n'
    '        == "\'Financial Conditions\'!C2:AM157"\n'
    '    )\n'
    '\n'
    '    # C/E/D and physical-row lag chain.\n'
    '    assert values_equivalent(\n'
    '        financial_rows[0]["Monthly Average Financial Conditions"], 1.0\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[1]["Monthly Average Financial Conditions"], (1.0 + 3.0 + 5.0) / 3\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[3]["Monthly Average Financial Conditions"],\n'
    '        (1.0 + 3.0 + 5.0 + 7.0) / 4,\n'
    '    )\n'
    '    assert financial_rows[4]["Monthly Average Financial Conditions"] == ""\n'
    '    assert values_equivalent(\n'
    '        financial_rows[5]["Monthly Average Financial Conditions"], -1.0\n'
    '    )\n'
    '    assert (\n'
    '        financial_rows[0]["3M MA Financial Conditions"] == ""\n'
    '        and financial_rows[1]["3M MA Financial Conditions"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[2]["3M MA Financial Conditions"],\n'
    '        _financial_conditions_mean_sheets(\n'
    '            [row["Monthly Average Financial Conditions"] for row in financial_rows[:3]]\n'
    '        ),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[4]["3M MA Financial Conditions"],\n'
    '        _financial_conditions_mean_sheets(\n'
    '            [\n'
    '                financial_rows[2]["Monthly Average Financial Conditions"],\n'
    '                financial_rows[3]["Monthly Average Financial Conditions"],\n'
    '                "",\n'
    '            ]\n'
    '        ),\n'
    '    )\n'
    '    assert (\n'
    '        financial_rows[2]["Sample Count"] == 1\n'
    '        and financial_rows[3]["Sample Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[3]["1M Financial Conditions Change"],\n'
    '        financial_rows[3]["3M MA Financial Conditions"]\n'
    '        - financial_rows[2]["3M MA Financial Conditions"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[6]["3M Financial Conditions Change"],\n'
    '        financial_rows[6]["3M MA Financial Conditions"]\n'
    '        - financial_rows[3]["3M MA Financial Conditions"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[9]["6M Financial Conditions Change"],\n'
    '        financial_rows[9]["3M MA Financial Conditions"]\n'
    '        - financial_rows[3]["3M MA Financial Conditions"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        financial_rows[15]["12M Financial Conditions Change"],\n'
    '        financial_rows[15]["3M MA Financial Conditions"]\n'
    '        - financial_rows[3]["3M MA Financial Conditions"],\n'
    '    )\n'
    '\n'
    '    # Metric-specific 60-value gates, rounded Sheets ranks, display, and inverted scores.\n'
    '    for before, first, percentile_col in (\n'
    '        (60, 61, "Financial Conditions Level Percentile"),\n'
    '        (63, 64, "3M Change Percentile"),\n'
    '        (66, 67, "6M Change Percentile"),\n'
    '        (72, 73, "12M Change Percentile"),\n'
    '    ):\n'
    '        assert financial_rows[before][percentile_col] == ""\n'
    '        assert financial_rows[first][percentile_col] != ""\n'
    '    e_internal = [row["3M MA Financial Conditions"] for row in financial_rows]\n'
    '    g_internal = [row["3M Financial Conditions Change"] for row in financial_rows]\n'
    '    h_internal = [row["6M Financial Conditions Change"] for row in financial_rows]\n'
    '    i_internal = [row["12M Financial Conditions Change"] for row in financial_rows]\n'
    '    for source_values, percentile_col, score_col in (\n'
    '        (\n'
    '            e_internal,\n'
    '            "Financial Conditions Level Percentile",\n'
    '            "Financial Conditions Level Stress Score",\n'
    '        ),\n'
    '        (g_internal, "3M Change Percentile", "3M Change Stress Score"),\n'
    '        (h_internal, "6M Change Percentile", "6M Change Stress Score"),\n'
    '        (i_internal, "12M Change Percentile", "12M Change Stress Score"),\n'
    '    ):\n'
    '        history = [value for value in source_values if not is_blank(value)]\n'
    '        expected_rank = _financial_conditions_sheets_round(\n'
    '            _financial_conditions_percentrank_inc_sheets(history, history[-1]), 3\n'
    '        )\n'
    '        assert financial_rows[-1][\n'
    '            percentile_col\n'
    '        ] == _financial_conditions_percent_display(expected_rank)\n'
    '        assert financial_rows[-1][score_col] == _financial_conditions_score_percentile(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert isinstance(financial_rows[-1][percentile_col], str) and financial_rows[\n'
    '            -1\n'
    '        ][percentile_col].endswith("%")\n'
    '    rounded_rank = _financial_conditions_sheets_round(0.1235, 3)\n'
    '    assert (\n'
    '        rounded_rank == 0.124\n'
    '        and _financial_conditions_score_percentile(rounded_rank) == 7.52\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_sheets_round(1.005, 2) == 1.01\n'
    '        and _financial_conditions_sheets_round(-1.005, 2) == -1.01\n'
    '    )\n'
    '\n'
    '    # Exact branch order and score signs.\n'
    '    assert _financial_conditions_regime("", 0) == ""\n'
    '    assert (\n'
    '        _financial_conditions_regime(-1, -0.1) == "Easy and easing financial conditions"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_regime(-1, 0)\n'
    '        == "Easy but tightening financial conditions"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_regime(1, -0.1) == "Tight but easing financial conditions"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_regime(0, 0)\n'
    '        == "Tight and tightening financial conditions"\n'
    '    )\n'
    '    assert _financial_conditions_regime(0, "") == "Mixed financial conditions regime"\n'
    '    assert (\n'
    '        _financial_conditions_asset_bias_label(6)\n'
    '        == "Strongly supportive for risk assets / credit / equities"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_shock_label(0.75)\n'
    '        == "Large financial-conditions tightening shock"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_shock_label(0.40)\n'
    '        == "Moderate financial-conditions tightening shock"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_shock_label(-0.75)\n'
    '        == "Large financial-conditions easing shock"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_shock_label(-0.40)\n'
    '        == "Moderate financial-conditions easing shock"\n'
    '    )\n'
    '    assert _financial_conditions_stress_easing_score(1.6, 1, 1, 0, 0.1, 0.1) == -7\n'
    '    assert _financial_conditions_stress_easing_score(1.1, 1, -1, 0, 0.1, -0.1) == -5\n'
    '    assert _financial_conditions_stress_easing_score(-1.1, -1, -1, 0, -0.1, -0.1) == 5\n'
    '    assert _financial_conditions_stress_easing_score(0, -1.1, -1.1, 0, -0.1, -0.1) == 4\n'
    '    assert (\n'
    '        _financial_conditions_shock_score(0.75) == -6\n'
    '        and _financial_conditions_shock_score(0.40) == -4\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_shock_score(-0.75) == 6\n'
    '        and _financial_conditions_shock_score(-0.40) == 4\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_recovery_score(-0.1, -0.1) == 5\n'
    '        and _financial_conditions_recovery_score(0.1, 0.1) == -5\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_recovery_score(-0.1, 0.1) == 3\n'
    '        and _financial_conditions_recovery_score(0.1, -0.1) == -3\n'
    '    )\n'
    '    assert _financial_conditions_outlier_score([3.1, -3.5, 0, 0]) == -6\n'
    '    assert _financial_conditions_outlier_score([-3.1, 0, 0, 0]) == 6\n'
    '    assert (\n'
    '        _financial_conditions_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme financial-stress tightening"\n'
    '    )\n'
    '\n'
    '    fc_row = financial_rows[90]\n'
    '    fck = fc_row["Financial Conditions Level Stress Score"]\n'
    '    fcm = fc_row["3M Change Stress Score"]\n'
    '    fco = fc_row["6M Change Stress Score"]\n'
    '    fcq = fc_row["12M Change Stress Score"]\n'
    '    fcu = fc_row["Financial Conditions Impulse Score"]\n'
    '    fcv = fc_row["Financial Stress Risk Score"]\n'
    '    fcw = fc_row["Financial Conditions Shock Score"]\n'
    '    fcx = fc_row["Financial Conditions Cycle Turning Point Score"]\n'
    '    fcy = fc_row["Financial Conditions Extreme Score"]\n'
    '    expected_u = _financial_conditions_sheets_round(\n'
    '        clamp(\n'
    '            _financial_conditions_weighted_sum(\n'
    '                (0.45, fcm), (0.30, fco), (0.15, fcq), (0.10, fck)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_z = _financial_conditions_sheets_round(\n'
    '        clamp(\n'
    '            _financial_conditions_weighted_sum(\n'
    '                (0.40, fck), (0.30, fcm), (0.20, fco), (0.10, fcq)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_aa = _financial_conditions_sheets_round(\n'
    '        clamp(\n'
    '            _financial_conditions_weighted_sum((0.50, fcu), (0.30, fcv), (0.20, fcx))\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    expected_ab = _financial_conditions_sheets_round(\n'
    '        clamp(\n'
    '            _financial_conditions_weighted_sum(\n'
    '                (0.35, fcv), (0.25, fcw), (0.20, fcx), (0.20, fcy)\n'
    '            )\n'
    '        ),\n'
    '        2,\n'
    '    )\n'
    '    assert fc_row["Financial Conditions Impulse Score"] == expected_u\n'
    '    assert fc_row["Coincident Score"] == expected_z\n'
    '    assert fc_row["Leading Score"] == expected_aa\n'
    '    assert fc_row["Warning Score"] == expected_ab\n'
    '    assert fc_row["Risk Asset Bias"] == _financial_conditions_asset_bias_label(\n'
    '        fc_row["Coincident Score"]\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_regime_signal(2)\n'
    '        == "Financial conditions easing / supportive"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_momentum_label(2, 2)\n'
    '        == "Financial conditions easing / supportive"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_signal_interpretation(6, 0)\n'
    '        == "Strong financial-conditions easing signal with supportive risk-asset profile"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_warning_label(0, 0, 0, -6)\n'
    '        == "High financial-stress extreme warning"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_warning_label(0, 0, -6, 0)\n'
    '        == "High financial-conditions tightening shock"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_warning_label(0, -6, 0, 0)\n'
    '        == "High financial-stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _financial_conditions_warning_label(6, 0, 0, 0)\n'
    '        == "High financial-conditions easing signal"\n'
    '    )\n'
    '    assert (\n'
    '        fc_row["Reliability"] == "Medium reliability"\n'
    '        and fc_row["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert (\n'
    '        financial_rows[-1]["Reliability"] == "High reliability"\n'
    '        and financial_rows[-1]["Signal Validity"] == "Valid"\n'
    '    )\n'
    '    assert financial_rows[61]["Signal Validity"] == "Incomplete Score"\n'
    '    missing_financial_raw = make_dataframe(\n'
    '        [\n'
    '            {"Date": "2021-01-31", "Raw Value": 1},\n'
    '            {"Date": "2021-02-28", "Raw Value": 2},\n'
    '            {"Date": "2021-03-31", "Raw Value": ""},\n'
    '        ],\n'
    '        ["Date", "Raw Value"],\n'
    '    )\n'
    '    missing_financial_last = df_records(\n'
    '        calculate_financial_conditions_outputs(missing_financial_raw)\n'
    '    )[-1]\n'
    '    assert (\n'
    '        missing_financial_last["Reliability"] == ""\n'
    '        and missing_financial_last["Signal Validity"] == "Missing Data"\n'
    '    )\n'
    '    for source_values, zscore_col in (\n'
    '        (e_internal, "Level Z-Score"),\n'
    '        (g_internal, "3M Change Z-Score"),\n'
    '        (h_internal, "6M Change Z-Score"),\n'
    '        (i_internal, "12M Change Z-Score"),\n'
    '    ):\n'
    '        expected_zscores = _financial_conditions_rounded_expanding_zscores(\n'
    '            source_values, _metric_counts(source_values)\n'
    '        )\n'
    '        assert financial_rows[-1][zscore_col] == expected_zscores[-1]\n'
    '    sample_z = _financial_conditions_rounded_expanding_zscores([1, 2, 3], [1, 2, 3], 3)\n'
    '    assert sample_z == ["", "", 1.0]\n'
    '\n'
    '    class FakeFinancialConditionsWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    fake_financial_write = FakeFinancialConditionsWorksheet([])\n'
    '    assert (\n'
    '        write_indicator_outputs(\n'
    '            fake_financial_write,\n'
    '            financial_out,\n'
    '            157,\n'
    '            FINANCIAL_CONDITIONS_OUTPUT_COLUMNS,\n'
    '            "AM",\n'
    '            "C",\n'
    '        )\n'
    '        == "C2:AM157"\n'
    '    )\n'
    '    assert [(call[0], call[2]) for call in fake_financial_write.update_calls] == [\n'
    '        ("C2:AM157", "RAW")\n'
    '    ]\n'
    '    fake_financial_verify = FakeFinancialConditionsWorksheet(\n'
    '        dataframe_to_sheet_rows(financial_out, FINANCIAL_CONDITIONS_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_financial_verify,\n'
    '        financial_out,\n'
    '        157,\n'
    '        FINANCIAL_CONDITIONS_OUTPUT_COLUMNS,\n'
    '        "AM",\n'
    '        "C",\n'
    '    )\n'
    '    assert fake_financial_verify.get_calls == [("C2:AM157", "UNFORMATTED_VALUE")]\n'
    '    assert (\n'
    '        validate_write_range(\n'
    '            financial_out, 157, "C", 2, FINANCIAL_CONDITIONS_OUTPUT_COLUMNS, "AM"\n'
    '        )\n'
    '        == "C2:AM157"\n'
    '    )\n'
    '    for unsafe_start_col, unsafe_start_row in (("B", 2), ("C", 1)):\n'
    '        try:\n'
    '            validate_write_range(\n'
    '                financial_out,\n'
    '                157,\n'
    '                unsafe_start_col,\n'
    '                unsafe_start_row,\n'
    '                FINANCIAL_CONDITIONS_OUTPUT_COLUMNS,\n'
    '                "AM",\n'
    '            )\n'
    '            raise AssertionError("A37 write guard allowed a protected source cell")\n'
    '        except AssertionError:\n'
    '            pass\n'
    '\n'
    '    class FakeFinancialConditionsWorkbook:\n'
    '        def __init__(self, worksheet: FakeFinancialConditionsWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeFinancialConditionsWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_financial_input = [\n'
    '        [row["Date"], row["Raw Value"]] for row in financial_rows_raw\n'
    '    ]\n'
    '    fake_financial_read = FakeFinancialConditionsWorksheet(fake_financial_input)\n'
    '    fake_a37_result = process_indicator(\n'
    '        FakeFinancialConditionsWorkbook(fake_financial_read),\n'
    '        BatchIndicatorSpec("Financial Conditions", "A37"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a37_result.ok and fake_a37_result.output_range == "C2:AM157"\n'
    '    assert (\n'
    '        fake_financial_read.get_calls == [("A2:B", None)]\n'
    '        and fake_financial_read.update_calls == []\n'
    '    )\n'
    '    fake_financial_no_verify = FakeFinancialConditionsWorksheet(fake_financial_input)\n'
    '    fake_a37_no_verify_result = process_indicator(\n'
    '        FakeFinancialConditionsWorkbook(fake_financial_no_verify),\n'
    '        BatchIndicatorSpec("Financial Conditions", "A37"),\n'
    '        write=True,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert (\n'
    '        fake_a37_no_verify_result.ok\n'
    '        and fake_a37_no_verify_result.output_range == "C2:AM157"\n'
    '    )\n'
    '    assert fake_financial_no_verify.get_calls == [("A2:B", None)]\n'
    '    assert [(call[0], call[2]) for call in fake_financial_no_verify.update_calls] == [\n'
    '        ("C2:AM157", "RAW")\n'
    '    ]\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped financial conditions sheet",\n'
    '            CONTROL_PANEL_THIRTY_FIFTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_financial_conditions_outputs\n'
    '    )\n'
    '    all_specs_with_a37 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 37)] + [["Financial Conditions"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Financial Conditions", "A37") in all_specs_with_a37\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped financial conditions sheet",\n'
    '            CONTROL_PANEL_THIRTY_FIFTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AM"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_financial_conditions_outputs(financial_raw))\n'
    '        == financial_rows\n'
    '    )\n'
    '\n'
    '    bank_credit_values = [1000 * (1.004**i) + 15 * math.sin(i / 4) for i in range(150)]\n'
    '    bank_credit_rows_raw = [\n'
    '        {"Date": f"bc{i}", "Raw Value": bank_credit_values[i]}\n'
    '        for i in range(len(bank_credit_values))\n'
    '    ]\n'
    '    bank_credit_rows_raw[20]["Raw Value"] = ""\n'
    '    bank_credit_raw = make_dataframe(bank_credit_rows_raw, ["Date", "Raw Value"])\n'
    '    bank_credit_out = calculate_bank_credit_outputs(bank_credit_raw)\n'
    '    bank_credit_rows = df_records(bank_credit_out)\n'
    '    bank_credit_spec = select_indicator_engine_spec(\n'
    '        "Unmapped bank credit sheet", CONTROL_PANEL_THIRTY_SIXTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert (\n'
    '        select_indicator_engine("Unmapped bank credit sheet", "A38")\n'
    '        is calculate_bank_credit_outputs\n'
    '    )\n'
    '    assert bank_credit_spec.calculate is calculate_bank_credit_outputs\n'
    '    assert (\n'
    '        bank_credit_spec.input_start_col == "A"\n'
    '        and bank_credit_spec.input_end_col == "B"\n'
    '    )\n'
    '    assert (\n'
    '        bank_credit_spec.output_start_col == "C"\n'
    '        and bank_credit_spec.output_end_col == "AP"\n'
    '    )\n'
    '    assert bank_credit_spec.summary_level_col == "Bank Credit Level Score"\n'
    '    assert (\n'
    '        bank_credit_spec.summary_leading_col == "Leading Score"\n'
    '        and bank_credit_spec.summary_warning_col == "Warning Score"\n'
    '    )\n'
    '    assert bank_credit_spec.summary_status_col == "Signal Validity"\n'
    '    assert list(bank_credit_out.columns) == BANK_CREDIT_OUTPUT_COLUMNS\n'
    '    assert bank_credit_out.shape[1] == len(BANK_CREDIT_OUTPUT_COLUMNS) == 40\n'
    '    assert output_end_col_for_columns(BANK_CREDIT_OUTPUT_COLUMNS) == "AP"\n'
    '    assert (\n'
    '        indicator_input_range(\n'
    '            "Bank Credit",\n'
    '            bank_credit_spec.input_end_col,\n'
    '            bank_credit_spec.input_start_col,\n'
    '        )\n'
    '        == "\'Bank Credit\'!A2:B"\n'
    '    )\n'
    '    assert indicator_output_range("Bank Credit", 151, "AP") == "\'Bank Credit\'!C2:AP151"\n'
    '\n'
    '    # Physical-row moving average, count, growth, and annualisation chain.\n'
    '    assert (\n'
    '        bank_credit_rows[0]["3M MA Bank Credit"] == ""\n'
    '        and bank_credit_rows[1]["3M MA Bank Credit"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[2]["3M MA Bank Credit"],\n'
    '        _bank_credit_mean_sheets(bank_credit_values[:3]),\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[20]["3M MA Bank Credit"],\n'
    '        _bank_credit_mean_sheets([bank_credit_values[18], bank_credit_values[19], ""]),\n'
    '    )\n'
    '    assert (\n'
    '        bank_credit_rows[2]["Sample Count"] == 0\n'
    '        and bank_credit_rows[3]["Sample Count"] == 0\n'
    '        and bank_credit_rows[4]["Sample Count"] == 1\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[5]["1M Bank Credit Growth"],\n'
    '        bank_credit_rows[5]["3M MA Bank Credit"]\n'
    '        / bank_credit_rows[4]["3M MA Bank Credit"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[7]["3M Bank Credit Growth"],\n'
    '        bank_credit_rows[7]["3M MA Bank Credit"]\n'
    '        / bank_credit_rows[4]["3M MA Bank Credit"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[10]["6M Bank Credit Growth"],\n'
    '        bank_credit_rows[10]["3M MA Bank Credit"]\n'
    '        / bank_credit_rows[4]["3M MA Bank Credit"]\n'
    '        - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[16]["12M Bank Credit Growth"],\n'
    '        bank_credit_rows[16]["3M MA Bank Credit"]\n'
    '        / bank_credit_rows[4]["3M MA Bank Credit"]\n'
    '        - 1,\n'
    '    )\n'
    '    zero_bank_raw = make_dataframe(\n'
    '        [{"Date": f"zbc{i}", "Raw Value": 0} for i in range(30)], ["Date", "Raw Value"]\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_bank_credit_outputs(zero_bank_raw))[5][\n'
    '            "1M Bank Credit Growth"\n'
    '        ]\n'
    '        == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[7]["3M Annualised Bank Credit Growth"],\n'
    '        (1 + bank_credit_rows[7]["3M Bank Credit Growth"]) ** 4 - 1,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        bank_credit_rows[10]["6M Annualised Bank Credit Growth"],\n'
    '        (1 + bank_credit_rows[10]["6M Bank Credit Growth"]) ** 2 - 1,\n'
    '    )\n'
    '\n'
    '    # Metric-specific gates and rank sources.\n'
    '    for before, first, percentile_col in (\n'
    '        (60, 61, "Bank Credit Level Percentile"),\n'
    '        (65, 66, "3M Growth Percentile"),\n'
    '        (68, 69, "6M Growth Percentile"),\n'
    '        (74, 75, "12M Growth Percentile"),\n'
    '    ):\n'
    '        assert bank_credit_rows[before][percentile_col] == ""\n'
    '        assert bank_credit_rows[first][percentile_col] != ""\n'
    '    c_internal = [row["3M MA Bank Credit"] for row in bank_credit_rows]\n'
    '    f_internal = [row["3M Bank Credit Growth"] for row in bank_credit_rows]\n'
    '    g_internal = [row["6M Bank Credit Growth"] for row in bank_credit_rows]\n'
    '    h_internal = [row["12M Bank Credit Growth"] for row in bank_credit_rows]\n'
    '    for source_values, percentile_col, score_col in (\n'
    '        (c_internal, "Bank Credit Level Percentile", "Bank Credit Level Score"),\n'
    '        (f_internal, "3M Growth Percentile", "3M Growth Score"),\n'
    '        (g_internal, "6M Growth Percentile", "6M Growth Score"),\n'
    '        (h_internal, "12M Growth Percentile", "12M Growth Score"),\n'
    '    ):\n'
    '        history = [value for value in source_values if not is_blank(value)]\n'
    '        expected_rank = _bank_credit_sheets_round(\n'
    '            _bank_credit_percentrank_inc_sheets(history, history[-1]), 3\n'
    '        )\n'
    '        assert bank_credit_rows[-1][percentile_col] == _bank_credit_percent_display(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert bank_credit_rows[-1][score_col] == _bank_credit_score_percentile(\n'
    '            expected_rank\n'
    '        )\n'
    '        assert isinstance(\n'
    '            bank_credit_rows[-1][percentile_col], str\n'
    '        ) and bank_credit_rows[-1][percentile_col].endswith("%")\n'
    '    assert (\n'
    '        _bank_credit_percent_display(1.0) == "100.00%"\n'
    '        and _bank_credit_score_percentile(1.0) == 10.0\n'
    '    )\n'
    '    for rank, percent_text, score in (\n'
    '        (0.966, "96.60%", 9.32),\n'
    '        (0.933, "93.30%", 8.66),\n'
    '        (0.839, "83.90%", 6.78),\n'
    '        (0.983, "98.30%", 9.66),\n'
    '        (0.857, "85.70%", 7.14),\n'
    '        (0.672, "67.20%", 3.44),\n'
    '        (0.885, "88.50%", 7.70),\n'
    '    ):\n'
    '        assert _bank_credit_percent_display(rank) == percent_text\n'
    '        assert _bank_credit_score_percentile(rank) == score\n'
    '\n'
    '    # Exact early and mature bank-credit regime branches.\n'
    '    assert _bank_credit_growth_regime(1, "", "", "") == "Bank-credit expansion regime"\n'
    '    assert _bank_credit_growth_regime(1, 0.1, "", "") == "Bank-credit expansion regime"\n'
    '    assert (\n'
    '        _bank_credit_growth_regime(1, -0.1, 0.1, 0.1)\n'
    '        == "Short-term bank-credit deterioration / mixed impulse"\n'
    '    )\n'
    '    assert (\n'
    '        _bank_credit_growth_regime(1, -0.1, -0.1, 0.1)\n'
    '        == "Bank-credit deterioration from high base"\n'
    '    )\n'
    '    assert (\n'
    '        _bank_credit_growth_regime(1, -0.1, -0.1, -0.1)\n'
    '        == "Bank-credit contraction regime"\n'
    '    )\n'
    '    assert _bank_credit_growth_regime(1, 0.1, 0.1, 0) == "Early bank-credit recovery"\n'
    '\n'
    '    # Rounded-visible composite and acceleration examples.\n'
    '    jun_v = _bank_credit_impulse_score(10, 7.14, 3.44, 7.70)\n'
    '    jun_ac = _bank_credit_coincident_score(10, 7.14, 3.44, 7.70)\n'
    '    jun_w = _bank_credit_acceleration_score(jun_v, 4.13)\n'
    '    assert (jun_v, jun_w, jun_ac) == (6.26, 4.26, 6.84)\n'
    '    assert _bank_credit_leading_score(jun_v, jun_w, 5, 3) == 5.18\n'
    '    assert _bank_credit_warning_score(jun_w, 5, 6, 3, 0, 0) == 3.49\n'
    '    jul_v = _bank_credit_impulse_score(10, 8.60, 6.18, 7.74)\n'
    '    jul_ac = _bank_credit_coincident_score(10, 8.60, 6.18, 7.74)\n'
    '    jul_w = _bank_credit_acceleration_score(jul_v, jun_v)\n'
    '    assert (jul_v, jul_w, jul_ac) == (7.82, 3.12, 8.19)\n'
    '    assert _bank_credit_leading_score(jul_v, jul_w, 5, 3) == 5.60\n'
    '    assert _bank_credit_warning_score(jul_w, 5, 6, 3, 0, 0) == 3.32\n'
    '    aug_v = _bank_credit_impulse_score(10, 8.88, 7.10, 8.10)\n'
    '    assert aug_v == 8.29 and _bank_credit_acceleration_score(aug_v, jul_v) == 0.94\n'
    '    oct_v = _bank_credit_impulse_score(10, 0.92, -1.38, -2.36)\n'
    '    oct_ac = _bank_credit_coincident_score(10, 0.92, -1.38, -2.36)\n'
    '    oct_w = _bank_credit_acceleration_score(oct_v, -1.61)\n'
    '    assert (oct_v, oct_w, oct_ac) == (0.19, 3.60, 1.83)\n'
    '    assert _bank_credit_leading_score(oct_v, oct_w, 5, 0) == 1.99\n'
    '    assert _bank_credit_warning_score(oct_w, 5, 0, 0, 0, 0) == 1.29\n'
    '    assert _bank_credit_acceleration_score(2.18, 0.19) == 3.98\n'
    '    assert _bank_credit_coincident_score(10, -5.64, -4.70, -2.82) == -2.00\n'
    '    assert (\n'
    '        _bank_credit_asset_bias_label(-2.00)\n'
    '        == "Mild credit-growth headwind for risk assets and domestic demand"\n'
    '    )\n'
    '    assert _bank_credit_conditions_regime(-2.00) == "Bank credit growth deteriorating"\n'
    '    assert (\n'
    '        _bank_credit_momentum_label(-2.00, -1.15) == "Bank credit growth weak / stable"\n'
    '    )\n'
    '    assert _bank_credit_conditions_regime(1.83) == "Neutral bank credit growth"\n'
    '    assert (\n'
    '        _bank_credit_momentum_label(1.83, 1.99) == "Bank credit growth broadly neutral"\n'
    '    )\n'
    '    assert (\n'
    '        _bank_credit_signal_interpretation(1.99, 1.29)\n'
    '        == "Neutral/mixed bank-credit signal"\n'
    '    )\n'
    '    assert (\n'
    '        _bank_credit_signal_interpretation(-2, 1)\n'
    '        == "Bank-credit impulse deteriorating, but credit conditions remain mixed"\n'
    '    )\n'
    '    assert (\n'
    '        _bank_credit_signal_interpretation(6, 0)\n'
    '        == "Strong positive bank-credit impulse with supportive private-credit profile"\n'
    '    )\n'
    '    for ad_value, ae_value in (\n'
    '        (-2.35, 0.58),\n'
    '        (-2.03, 0.76),\n'
    '        (-2.27, 0.68),\n'
    '        (-2.50, 0.59),\n'
    '        (-2.52, 0.26),\n'
    '    ):\n'
    '        assert (\n'
    '            _bank_credit_signal_interpretation(ad_value, ae_value)\n'
    '            == "Bank-credit impulse deteriorating, but credit conditions remain mixed"\n'
    '        )\n'
    '    for ad_value, ae_value in ((6.59, 4.35), (6.52, 4.52)):\n'
    '        assert (\n'
    '            _bank_credit_signal_interpretation(ad_value, ae_value)\n'
    '            == "Strong positive bank-credit impulse with supportive private-credit profile"\n'
    '        )\n'
    '\n'
    '    # Preserved risk, shock, reliability, validity, warning, and z-score behavior.\n'
    '    assert (\n'
    '        _bank_credit_shock_label(0.15)\n'
    '        == "Extreme 3M annualised bank-credit growth surge"\n'
    '    )\n'
    '    assert _bank_credit_shock_label(0.0) == "Negative 3M annualised bank-credit growth"\n'
    '    assert (\n'
    '        _bank_credit_shock_score(0.15) == 6\n'
    '        and _bank_credit_shock_score(0.10) == 4\n'
    '        and _bank_credit_shock_score(-0.05) == -6\n'
    '    )\n'
    '    assert _bank_credit_contraction_risk(-1.6, -1.6, 0, -0.1, -0.1, 0.1, 0.03) == -7\n'
    '    assert _bank_credit_contraction_risk(-1.1, -1.1, 0, -0.1, 0.1, 0.1, 0.03) == -5\n'
    '    assert _bank_credit_overheating_score(1.6, 1.6, 0.16, 0.11) == 5\n'
    '    assert _bank_credit_growth_outlier_score([3.1, -3.5, 0]) == 6\n'
    '    assert (\n'
    '        _bank_credit_growth_outlier_label([3.1, -3.5, 0])\n'
    '        == "Extreme bank-credit expansion"\n'
    '    )\n'
    '    assert (\n'
    '        bank_credit_rows[60]["Level Z-Score"] == ""\n'
    '        and bank_credit_rows[61]["Level Z-Score"] != ""\n'
    '    )\n'
    '    assert bank_credit_rows[66]["3M Growth Z-Score"] == round(\n'
    '        bank_credit_rows[66]["3M Growth Z-Score"], 2\n'
    '    )\n'
    '    assert (\n'
    '        _bank_credit_warning_label(0, 0, -6, 0, 0)\n'
    '        == "High bank-credit contraction warning"\n'
    '    )\n'
    '    bank_credit_row = bank_credit_rows[90]\n'
    '    assert bank_credit_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert bank_credit_row["Signal Validity"] in ("Valid", "Incomplete Score")\n'
    '    assert bank_credit_row["Risk Asset Bias"] == _bank_credit_asset_bias_label(\n'
    '        bank_credit_row["Coincident Score"]\n'
    '    )\n'
    '    assert (\n'
    '        bank_credit_row["Warning Flag"] != "" and bank_credit_row["Extreme Flag"] != ""\n'
    '    )\n'
    '\n'
    '    class FakeBankCreditWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    fake_bank_write = FakeBankCreditWorksheet([])\n'
    '    assert (\n'
    '        write_indicator_outputs(\n'
    '            fake_bank_write, bank_credit_out, 151, BANK_CREDIT_OUTPUT_COLUMNS, "AP", "C"\n'
    '        )\n'
    '        == "C2:AP151"\n'
    '    )\n'
    '    assert [(call[0], call[2]) for call in fake_bank_write.update_calls] == [\n'
    '        ("C2:AP151", "RAW")\n'
    '    ]\n'
    '    fake_bank_verify = FakeBankCreditWorksheet(\n'
    '        dataframe_to_sheet_rows(bank_credit_out, BANK_CREDIT_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fake_bank_verify, bank_credit_out, 151, BANK_CREDIT_OUTPUT_COLUMNS, "AP", "C"\n'
    '    )\n'
    '    assert fake_bank_verify.get_calls == [("C2:AP151", "UNFORMATTED_VALUE")]\n'
    '    for unsafe_col, unsafe_row in (("B", 2), ("C", 1)):\n'
    '        try:\n'
    '            validate_write_range(\n'
    '                bank_credit_out,\n'
    '                151,\n'
    '                unsafe_col,\n'
    '                unsafe_row,\n'
    '                BANK_CREDIT_OUTPUT_COLUMNS,\n'
    '                "AP",\n'
    '            )\n'
    '            raise AssertionError("A38 write guard allowed a protected source cell")\n'
    '        except AssertionError:\n'
    '            pass\n'
    '\n'
    '    class FakeBankCreditWorkbook:\n'
    '        def __init__(self, worksheet: FakeBankCreditWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeBankCreditWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_bank_input = [[row["Date"], row["Raw Value"]] for row in bank_credit_rows_raw]\n'
    '    fake_bank_read = FakeBankCreditWorksheet(fake_bank_input)\n'
    '    fake_a38_result = process_indicator(\n'
    '        FakeBankCreditWorkbook(fake_bank_read),\n'
    '        BatchIndicatorSpec("Bank Credit", "A38"),\n'
    '        write=False,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert fake_a38_result.ok and fake_a38_result.output_range == "C2:AP151"\n'
    '    assert (\n'
    '        fake_bank_read.get_calls == [("A2:B", None)]\n'
    '        and fake_bank_read.update_calls == []\n'
    '    )\n'
    '    fake_bank_no_verify = FakeBankCreditWorksheet(fake_bank_input)\n'
    '    fake_a38_no_verify_result = process_indicator(\n'
    '        FakeBankCreditWorkbook(fake_bank_no_verify),\n'
    '        BatchIndicatorSpec("Bank Credit", "A38"),\n'
    '        write=True,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert (\n'
    '        fake_a38_no_verify_result.ok\n'
    '        and fake_a38_no_verify_result.output_range == "C2:AP151"\n'
    '    )\n'
    '    assert fake_bank_no_verify.get_calls == [("A2:B", None)]\n'
    '    assert [(call[0], call[2]) for call in fake_bank_no_verify.update_calls] == [\n'
    '        ("C2:AP151", "RAW")\n'
    '    ]\n'
    '    all_specs_with_a38 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 38)] + [["Bank Credit"], [""]], "A", 3, True\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Bank Credit", "A38") in all_specs_with_a38\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped bank credit sheet", CONTROL_PANEL_THIRTY_SIXTH_INDICATOR_CELL\n'
    '        ).output_end_col\n'
    '        == "AP"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_bank_credit_outputs(bank_credit_raw)) == bank_credit_rows\n'
    '    )\n'
    '\n'
    '    corporate_values = [2.2 + 0.35 * math.sin(i / 5) + 0.003 * i for i in range(150)]\n'
    '    corporate_rows_raw = [\n'
    '        {"Date": f"corp{i}", "Raw Value": corporate_values[i]}\n'
    '        for i in range(len(corporate_values))\n'
    '    ]\n'
    '    corporate_rows_raw[20]["Raw Value"] = ""\n'
    '    corporate_raw = make_dataframe(corporate_rows_raw, ["Date", "Raw Value"])\n'
    '    corporate_out = calculate_corporate_credit_spreads_outputs(corporate_raw)\n'
    '    corporate_rows = df_records(corporate_out)\n'
    '    corporate_spec = select_indicator_engine_spec(\n'
    '        "Unmapped corporate spread sheet", CONTROL_PANEL_THIRTY_SEVENTH_INDICATOR_CELL\n'
    '    )\n'
    '    assert corporate_spec.input_start_col == "A" and corporate_spec.input_end_col == "B"\n'
    '    assert (\n'
    '        corporate_spec.output_start_col == "C" and corporate_spec.output_end_col == "AL"\n'
    '    )\n'
    '    assert (\n'
    '        corporate_spec.summary_level_col == "Corporate Credit Spread Level Score"\n'
    '        and corporate_spec.summary_leading_col\n'
    '        == "Corporate Credit Spread Leading Score"\n'
    '        and corporate_spec.summary_warning_col\n'
    '        == "Corporate Credit Spread Final Warning Score"\n'
    '    )\n'
    '    assert corporate_out.shape[1] == len(CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS) == 36\n'
    '    assert output_end_col_for_columns(CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS) == "AL"\n'
    '    assert (\n'
    '        indicator_input_range("Corporate Credit Spreads")\n'
    '        == "\'Corporate Credit Spreads\'!A2:B"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Corporate Credit Spreads", 678, "AL")\n'
    '        == "\'Corporate Credit Spreads\'!C2:AL678"\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[0]["3-Period Moving Average Corporate Credit Spread"] == ""\n'
    '        and corporate_rows[1]["3-Period Moving Average Corporate Credit Spread"] == ""\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_rows[2]["3-Period Moving Average Corporate Credit Spread"],\n'
    '        sum(corporate_values[:3]) / 3,\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_rows[20]["3-Period Moving Average Corporate Credit Spread"],\n'
    '        (corporate_values[18] + corporate_values[19]) / 2,\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[2]["Corporate Credit Spread Observation Count"] == 1\n'
    '        and corporate_rows[3]["Corporate Credit Spread Observation Count"] == 2\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_rows[3]["1-Period Corporate Credit Spread Change"],\n'
    '        corporate_rows[3]["3-Period Moving Average Corporate Credit Spread"]\n'
    '        - corporate_rows[2]["3-Period Moving Average Corporate Credit Spread"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_rows[5]["3-Period Corporate Credit Spread Change"],\n'
    '        corporate_rows[5]["3-Period Moving Average Corporate Credit Spread"]\n'
    '        - corporate_rows[2]["3-Period Moving Average Corporate Credit Spread"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_rows[8]["6-Period Corporate Credit Spread Change"],\n'
    '        corporate_rows[8]["3-Period Moving Average Corporate Credit Spread"]\n'
    '        - corporate_rows[2]["3-Period Moving Average Corporate Credit Spread"],\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_rows[14]["12-Period Corporate Credit Spread Change"],\n'
    '        corporate_rows[14]["3-Period Moving Average Corporate Credit Spread"]\n'
    '        - corporate_rows[2]["3-Period Moving Average Corporate Credit Spread"],\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[60]["Corporate Credit Spread Level PercentRank"] == ""\n'
    '        and corporate_rows[61]["Corporate Credit Spread Level PercentRank"] != ""\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[63]["3-Period Corporate Credit Spread Change PercentRank"] == ""\n'
    '        and corporate_rows[64]["3-Period Corporate Credit Spread Change PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[66]["6-Period Corporate Credit Spread Change PercentRank"] == ""\n'
    '        and corporate_rows[67]["6-Period Corporate Credit Spread Change PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[72]["12-Period Corporate Credit Spread Change PercentRank"] == ""\n'
    '        and corporate_rows[73]["12-Period Corporate Credit Spread Change PercentRank"]\n'
    '        != ""\n'
    '    )\n'
    '    assert isinstance(\n'
    '        corporate_rows[61]["Corporate Credit Spread Level PercentRank"], str\n'
    '    ) and corporate_rows[61]["Corporate Credit Spread Level PercentRank"].endswith("%")\n'
    '    assert (\n'
    '        _corporate_spread_inverted_score(1.0) == -10\n'
    '        and _corporate_spread_inverted_score(0.0) == 10\n'
    '    )\n'
    '    # A39 ranks full-precision C/F/G/H histories, uses the first exact duplicate\n'
    '    # rank, then applies Sheets rounding to the percentile and inverted score.\n'
    '    assert values_equivalent(\n'
    '        _corporate_spread_percentrank_inc_sheets([1, 2, 2, 4], 2), 1 / 3\n'
    '    )\n'
    '    close_history = [0.10004, 0.10005, 0.10006]\n'
    '    assert _corporate_spread_percentrank_inc_sheets(close_history, 0.10005) == 0.5\n'
    '    assert (\n'
    '        _corporate_spread_percentrank_inc_sheets(\n'
    '            [round(value, 3) for value in close_history], round(0.10005, 3)\n'
    '        )\n'
    '        == 0.0\n'
    '    )\n'
    '    rounded_a39_rank = _corporate_spread_sheets_round(0.7176, 3)\n'
    '    assert (\n'
    '        rounded_a39_rank == 0.718\n'
    '        and _corporate_spread_inverted_score(rounded_a39_rank) == -4.36\n'
    '    )\n'
    '    assert _corporate_spread_inverted_score(0.943) == -8.86\n'
    '    assert _corporate_spread_mean_sheets([0.1, 0.2, 0.3]) == ((0.1 + 0.2) + 0.3) / 3\n'
    '    assert (\n'
    '        _corporate_spread_regime(1.5, "", 0) == "Normal corporate spreads but widening"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_regime(2.5, "", 0)\n'
    '        == "Elevated corporate spreads and widening"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_regime(4, "", 0)\n'
    '        == "Stress-level corporate spreads and widening"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_regime(4, "", -0.01)\n'
    '        == "Stress-level corporate spreads but narrowing"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_regime(3.99, "", 0)\n'
    '        == "Elevated corporate spreads and widening"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_asset_bias_label(6)\n'
    '        == "Strongly supportive for risk assets / credit / equities"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_shock_label(0.75) == "Large corporate spread widening shock"\n'
    '        and _corporate_spread_shock_label(-0.75) == "Large corporate spread narrowing"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_shock_score(0.75) == -6\n'
    '        and _corporate_spread_shock_score(0.40) == -4\n'
    '        and _corporate_spread_shock_score(-0.75) == 6\n'
    '        and _corporate_spread_shock_score(-0.40) == 4\n'
    '    )\n'
    '    assert (\n'
    '        corporate_rows[60]["Corporate Credit Spread Level Z-Score"] == ""\n'
    '        and corporate_rows[61]["Corporate Credit Spread Level Z-Score"] != ""\n'
    '    )\n'
    '    assert corporate_rows[65][\n'
    '        "3-Period Corporate Credit Spread Change Z-Score"\n'
    '    ] == round(corporate_rows[65]["3-Period Corporate Credit Spread Change Z-Score"], 2)\n'
    '    assert _corporate_spread_outlier_score([3.1, -3.5, 0, 0]) == -6\n'
    '    assert (\n'
    '        _corporate_spread_outlier_label([3.1, -3.5, 0, 0])\n'
    '        == "Extreme corporate credit stress"\n'
    '    )\n'
    '    corporate_row = corporate_rows[90]\n'
    '    csj = corporate_row["Corporate Credit Spread Level Score"]\n'
    '    csl = corporate_row["3-Period Corporate Credit Spread Change Score"]\n'
    '    csn = corporate_row["6-Period Corporate Credit Spread Change Score"]\n'
    '    csp = corporate_row["12-Period Corporate Credit Spread Change Score"]\n'
    '    cst = corporate_row["Corporate Credit Spread Leading Composite Score"]\n'
    '    csu = corporate_row["Corporate Credit Spread Momentum Composite Score"]\n'
    '    csv = corporate_row["Corporate Credit Spread Recovery / Deterioration Score"]\n'
    '    csw = corporate_row["Corporate Credit Spread Shock Score"]\n'
    '    csx = corporate_row["Corporate Credit Spread Outlier Score"]\n'
    '    corporate_stress_raw = _corporate_spread_credit_stress_score_raw(csj, csl, csn, csp)\n'
    '    corporate_impulse_raw = _corporate_spread_impulse_score_raw(csj, csl, csn, csp)\n'
    '    assert values_equivalent(\n'
    '        cst, _corporate_spread_sheets_round(corporate_stress_raw, 2)\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        csu, _corporate_spread_sheets_round(corporate_impulse_raw, 2)\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_recovery_score(-0.1, -0.1) == 5\n'
    '        and _corporate_spread_recovery_score(0.1, -0.1) == -3\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_recovery_score(0, 0.1) == 0\n'
    '        and _corporate_spread_recovery_score(-0.1, 0) == 0\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_row["Corporate Credit Spread Leading Score"], cst\n'
    '    )\n'
    '    corporate_leading_raw = _corporate_spread_leading_score_raw(\n'
    '        corporate_stress_raw, corporate_impulse_raw, csv\n'
    '    )\n'
    '    corporate_warning_raw = _corporate_spread_warning_score_raw(\n'
    '        corporate_stress_raw, csv, csw, csx\n'
    '    )\n'
    '    corporate_leading_output_raw = clamp(0.45 * cst + 0.35 * csu + 0.20 * csv, -10, 10)\n'
    '    assert values_equivalent(\n'
    '        corporate_row["Corporate Credit Spread Warning Composite Score"],\n'
    '        _corporate_spread_sheets_round(corporate_leading_output_raw, 2),\n'
    '    )\n'
    '    corporate_warning_output_raw = _corporate_spread_warning_score_raw(\n'
    '        cst, csv, csw, csx\n'
    '    )\n'
    '    assert values_equivalent(\n'
    '        corporate_row["Corporate Credit Spread Final Warning Score"],\n'
    '        _corporate_spread_sheets_round(corporate_warning_output_raw, 2),\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_conditions_regime(2)\n'
    '        == "Credit conditions easing / supportive"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_momentum_label(2, 2) == "Corporate credit conditions easing"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(6, 0)\n'
    '        == "Strong corporate-credit easing signal with supportive risk-asset profile"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(0, -6, 0, 0)\n'
    '        == "High corporate-credit stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(0, 0, 0, -6)\n'
    '        == "High corporate-credit extreme stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(0, 0, -6, 0)\n'
    '        == "High corporate-spread widening shock"\n'
    '    )\n'
    '    assert corporate_row["Reliability"] in ("Medium reliability", "High reliability")\n'
    '    assert corporate_row["Data Status"] in ("Valid", "Incomplete Score")\n'
    '    assert corporate_row[\n'
    '        "Corporate Credit Spread Asset Bias Label"\n'
    '    ] == _corporate_spread_asset_bias_label(corporate_stress_raw)\n'
    '    assert (\n'
    '        corporate_row["Corporate Credit Spread Warning Risk Label"] != ""\n'
    '        and corporate_row["Corporate Credit Spread Outlier Label"] != ""\n'
    '    )\n'
    '\n'
    '    # April-June 2012-style percentile and aggregate regressions.\n'
    '    apr_2012_ranks = (0.895, 0.352, 0.423, 0.874)\n'
    '    apr_2012_scores = tuple(\n'
    '        _corporate_spread_inverted_score(rank) for rank in apr_2012_ranks\n'
    '    )\n'
    '    assert tuple(\n'
    '        _corporate_spread_percent_display(rank) for rank in apr_2012_ranks\n'
    '    ) == ("89.50%", "35.20%", "42.30%", "87.40%")\n'
    '    assert apr_2012_scores == (-7.90, 2.96, 1.54, -7.48)\n'
    '    apr_2012_stress_raw = _corporate_spread_credit_stress_score_raw(*apr_2012_scores)\n'
    '    apr_2012_impulse_raw = _corporate_spread_impulse_score_raw(*apr_2012_scores)\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(apr_2012_stress_raw, 2),\n'
    '        _corporate_spread_sheets_round(apr_2012_impulse_raw, 2),\n'
    '    ) == (-2.71, 0.43)\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_leading_score_raw(-2.71, 0.43, 5), 2\n'
    '        )\n'
    '        == -0.07\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_warning_score_raw(apr_2012_stress_raw, 5, 0, 0), 2\n'
    '        )\n'
    '        == 0.05\n'
    '    )\n'
    '\n'
    '    may_2012_ranks = (0.901, 0.389, 0.435, 0.887)\n'
    '    may_2012_scores = tuple(\n'
    '        _corporate_spread_inverted_score(rank) for rank in may_2012_ranks\n'
    '    )\n'
    '    assert tuple(\n'
    '        _corporate_spread_percent_display(rank) for rank in may_2012_ranks\n'
    '    ) == ("90.10%", "38.90%", "43.50%", "88.70%")\n'
    '    assert may_2012_scores == (-8.02, 2.22, 1.30, -7.74)\n'
    '    may_2012_stress_raw = _corporate_spread_credit_stress_score_raw(*may_2012_scores)\n'
    '    may_2012_impulse_raw = _corporate_spread_impulse_score_raw(*may_2012_scores)\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(may_2012_stress_raw, 2),\n'
    '        _corporate_spread_sheets_round(may_2012_impulse_raw, 2),\n'
    '    ) == (-3.06, -0.06)\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_leading_score_raw(-3.06, -0.06, 5), 2\n'
    '        )\n'
    '        == -0.40\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_warning_score_raw(may_2012_stress_raw, 5, 0, 0), 2\n'
    '        )\n'
    '        == -0.07\n'
    '    )\n'
    '\n'
    '    a39_reference_ranks = (0.943, 0.718, 0.592, 0.894)\n'
    '    a39_reference_percentiles = tuple(\n'
    '        _corporate_spread_percent_display(rank) for rank in a39_reference_ranks\n'
    '    )\n'
    '    a39_reference_scores = tuple(\n'
    '        _corporate_spread_inverted_score(rank) for rank in a39_reference_ranks\n'
    '    )\n'
    '    assert a39_reference_percentiles == ("94.30%", "71.80%", "59.20%", "89.40%")\n'
    '    assert a39_reference_scores == (-8.86, -4.36, -1.84, -7.88)\n'
    '    a39_reference_stress_raw = _corporate_spread_credit_stress_score_raw(\n'
    '        *a39_reference_scores\n'
    '    )\n'
    '    a39_reference_impulse_raw = _corporate_spread_impulse_score_raw(\n'
    '        *a39_reference_scores\n'
    '    )\n'
    '    a39_reference_stress_display = _corporate_spread_sheets_round(\n'
    '        a39_reference_stress_raw, 2\n'
    '    )\n'
    '    assert (\n'
    '        a39_reference_stress_raw < -6\n'
    '        and a39_reference_stress_raw != a39_reference_stress_display\n'
    '    )\n'
    '    assert a39_reference_stress_display == -6.01\n'
    '    assert _corporate_spread_sheets_round(a39_reference_impulse_raw, 2) == -4.36\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_leading_score_raw(-6.01, -4.36, -5), 2\n'
    '        )\n'
    '        == -5.23\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_warning_score_raw(a39_reference_stress_raw, -5, 0, 0), 2\n'
    '        )\n'
    '        == -3.10\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_asset_bias_label(a39_reference_stress_raw)\n'
    '        == "Strong headwind for risk assets / credit / equities"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-3.10, a39_reference_stress_raw, 0, 0)\n'
    '        == "High corporate-credit stress warning"\n'
    '    )\n'
    '\n'
    '    # Raw threshold and reference hierarchy examples around +/-2 and +/-6.\n'
    '    jul_2012_stress_raw = _corporate_spread_credit_stress_score_raw(\n'
    '        -8.92, -6.48, -2.78, -7.90\n'
    '    )\n'
    '    jul_2012_impulse_raw = _corporate_spread_impulse_score_raw(\n'
    '        -8.92, -6.48, -2.78, -7.90\n'
    '    )\n'
    '    jul_2012_warning_raw = _corporate_spread_warning_score_raw(\n'
    '        jul_2012_stress_raw, -5, 0, 0\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(jul_2012_stress_raw, 2),\n'
    '        _corporate_spread_sheets_round(jul_2012_impulse_raw, 2),\n'
    '    ) == (-6.86, -5.71)\n'
    '    assert (\n'
    '        _corporate_spread_sheets_round(\n'
    '            _corporate_spread_leading_score_raw(-6.86, -5.71, -5), 2\n'
    '        )\n'
    '        == -6.09\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(\n'
    '            _corporate_spread_leading_score_raw(\n'
    '                jul_2012_stress_raw, jul_2012_impulse_raw, -5\n'
    '            ),\n'
    '            jul_2012_warning_raw,\n'
    '        )\n'
    '        == "Severe corporate-credit stress signal with downside confirmation"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-3.21, -6.31, 0, 0)\n'
    '        == "High corporate-credit stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_asset_bias_label(2.01) == "Mildly supportive for risk assets"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_asset_bias_label(6.02)\n'
    '        == "Strongly supportive for risk assets / credit / equities"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_conditions_regime(2.0000000001)\n'
    '        == "Credit conditions easing / supportive"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_momentum_label(0.03, -1.97895)\n'
    '        == "Corporate credit spreads broadly neutral"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_momentum_label(-1.63, -1.971)\n'
    '        == "Corporate credit spreads broadly neutral"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_momentum_label(2.194, 2.00165)\n'
    '        == "Corporate credit conditions easing"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(-1.97895, -0.99)\n'
    '        == "Neutral/mixed corporate credit-spread signal"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(-1.971, -1.17)\n'
    '        == "Neutral/mixed corporate credit-spread signal"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(2.00165, 1.769)\n'
    '        == "Moderately positive corporate-credit easing signal"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-5.88, -8.80, -4, -4)\n'
    '        == "Moderate corporate-credit extreme stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-7.00, -9.42, -6, -6)\n'
    '        == "High corporate-credit extreme stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-4.24, -6.39, -4, 0)\n'
    '        == "Moderate corporate-spread widening shock"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-0.89, -4.26, 0, 0)\n'
    '        == "Moderate corporate-credit stress warning"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_warning_label(-0.07, may_2012_stress_raw, 0, 0)\n'
    '        == "Neutral corporate-credit warning profile"\n'
    '    )\n'
    '\n'
    '    # Early Q uses C plus available E direction until F exists.\n'
    '    assert (\n'
    '        _corporate_spread_regime(2.456666666666667, "", "")\n'
    '        == "Normal corporate spreads but widening"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_regime(2.673333333333334, 0.21666666666666679, "")\n'
    '        == "Elevated corporate spreads and widening"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_regime(2.7300000000000004, 0.05666666666666664, "")\n'
    '        == "Elevated corporate spreads and widening"\n'
    '    )\n'
    '\n'
    '    # 2008-2009 stress-level Q rows and the August transition back to Elevated.\n'
    '    for c_value, f_value in (\n'
    '        (4.793333333333333, 1.6666666666666665),\n'
    '        (5.586666666666666, 2.2399999999999998),\n'
    '        (5.776666666666666, 1.7933333333333334),\n'
    '        (5.616666666666667, 0.8233333333333341),\n'
    '    ):\n'
    '        assert (\n'
    '            _corporate_spread_regime(c_value, "", f_value)\n'
    '            == "Stress-level corporate spreads and widening"\n'
    '        )\n'
    '    for c_value, f_value in (\n'
    '        (5.4799999999999995, -0.10666666666666647),\n'
    '        (5.423333333333333, -0.3533333333333335),\n'
    '        (5.276666666666666, -0.34000000000000075),\n'
    '        (4.666666666666667, -0.8133333333333326),\n'
    '        (4.0233333333333325, -1.4000000000000004),\n'
    '    ):\n'
    '        assert (\n'
    '            _corporate_spread_regime(c_value, "", f_value)\n'
    '            == "Stress-level corporate spreads but narrowing"\n'
    '        )\n'
    '    assert (\n'
    '        _corporate_spread_regime(3.4299999999999997, "", -1.8466666666666667)\n'
    '        == "Elevated corporate spreads but narrowing"\n'
    '    )\n'
    '\n'
    '    # Strict-sign V and its downstream Z/AA recalculations.\n'
    '    for f_value, g_value in ((0, 0.05), (-0.1633333333, 0), (0, 0.14), (-0.01, 0)):\n'
    '        assert _corporate_spread_recovery_score(f_value, g_value) == 0\n'
    '    for t_value, u_value, expected_z, expected_aa in (\n'
    '        (1.58, -1.21, 0.29, 0.55),\n'
    '        (-1.55, 1.73, -0.09, -0.54),\n'
    '        (0.64, -0.98, -0.05, 0.22),\n'
    '        (1.60, -0.83, 0.43, 0.56),\n'
    '    ):\n'
    '        leading_output_raw = clamp(0.45 * t_value + 0.35 * u_value, -10, 10)\n'
    '        warning_output_raw = _corporate_spread_warning_score_raw(t_value, 0, 0, 0)\n'
    '        assert _corporate_spread_sheets_round(leading_output_raw, 2) == expected_z\n'
    '        assert _corporate_spread_sheets_round(warning_output_raw, 2) == expected_aa\n'
    '\n'
    '    # Decimal-backed weighted inputs plus Sheets ROUND prevent half-cent drift.\n'
    '    for t_value, v_value, w_value, x_value, expected_aa in (\n'
    '        (2.13, 5, 0, 0, 1.75),\n'
    '        (5.10, 5, 0, 0, 2.79),\n'
    '        (-9.13, -5, -4, -6, -6.40),\n'
    '        (0.93, 5, 6, 6, 4.03),\n'
    '    ):\n'
    '        warning_output_raw = _corporate_spread_warning_score_raw(\n'
    '            t_value, v_value, w_value, x_value\n'
    '        )\n'
    '        assert _corporate_spread_sheets_round(warning_output_raw, 2) == expected_aa\n'
    '\n'
    '    # 28/02/2005-style raw leading score rounds to +6.00 and therefore enters\n'
    '    # the strong-easing branch; the following month remains in that branch.\n'
    '    feb_2005_stress_raw = _corporate_spread_credit_stress_score_raw(\n'
    '        3.22, 8.22, 6.20, 6.18\n'
    '    )\n'
    '    feb_2005_impulse_raw = _corporate_spread_impulse_score_raw(3.22, 8.22, 6.20, 6.18)\n'
    '    feb_2005_leading_raw = _corporate_spread_leading_score_raw(\n'
    '        feb_2005_stress_raw, feb_2005_impulse_raw, 5\n'
    '    )\n'
    '    assert feb_2005_leading_raw != _corporate_spread_sheets_round(\n'
    '        feb_2005_leading_raw, 2\n'
    '    )\n'
    '    assert _corporate_spread_sheets_round(feb_2005_leading_raw, 2) == 6\n'
    '    assert _corporate_spread_at_or_above_sheets_boundary(feb_2005_leading_raw, 6, 2)\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(feb_2005_leading_raw, 2.96)\n'
    '        == "Strong corporate-credit easing signal with supportive risk-asset profile"\n'
    '    )\n'
    '    assert (\n'
    '        _corporate_spread_signal_interpretation(7.34, 3.57)\n'
    '        == "Strong corporate-credit easing signal with supportive risk-asset profile"\n'
    '    )\n'
    '\n'
    '    class FakeCorporateSpreadWorksheet:\n'
    '        def __init__(self, rows: list[list[Any]]) -> None:\n'
    '            self.rows = rows\n'
    '            self.get_calls: list[tuple[str, str | None]] = []\n'
    '            self.update_calls: list[tuple[str, list[list[Any]], str]] = []\n'
    '\n'
    '        def get(\n'
    '            self, range_name: str, value_render_option: str | None = None\n'
    '        ) -> list[list[Any]]:\n'
    '            self.get_calls.append((range_name, value_render_option))\n'
    '            return self.rows\n'
    '\n'
    '        def update(\n'
    '            self, range_name: str, values: list[list[Any]], value_input_option: str\n'
    '        ) -> None:\n'
    '            self.update_calls.append((range_name, values, value_input_option))\n'
    '\n'
    '    class FakeCorporateSpreadWorkbook:\n'
    '        def __init__(self, worksheet: FakeCorporateSpreadWorksheet) -> None:\n'
    '            self.fake_worksheet = worksheet\n'
    '\n'
    '        def worksheet(self, _sheet_name: str) -> FakeCorporateSpreadWorksheet:\n'
    '            return self.fake_worksheet\n'
    '\n'
    '    fake_corporate_input = [\n'
    '        [row["Date"], row["Raw Value"]] for row in corporate_rows_raw\n'
    '    ]\n'
    '    fake_corporate_no_verify = FakeCorporateSpreadWorksheet(fake_corporate_input)\n'
    '    fake_a39_no_verify_result = process_indicator(\n'
    '        FakeCorporateSpreadWorkbook(fake_corporate_no_verify),\n'
    '        BatchIndicatorSpec("Corporate Credit Spreads", "A39"),\n'
    '        write=True,\n'
    '        verify=False,\n'
    '    )\n'
    '    assert (\n'
    '        fake_a39_no_verify_result.ok\n'
    '        and fake_a39_no_verify_result.output_range == "C2:AL151"\n'
    '    )\n'
    '    assert fake_corporate_no_verify.get_calls == [("A2:B", None)]\n'
    '    assert [(call[0], call[2]) for call in fake_corporate_no_verify.update_calls] == [\n'
    '        ("C2:AL151", "RAW")\n'
    '    ]\n'
    '    assert (\n'
    '        select_indicator_engine(\n'
    '            "Unmapped corporate spread sheet",\n'
    '            CONTROL_PANEL_THIRTY_SEVENTH_INDICATOR_CELL,\n'
    '        )\n'
    '        is calculate_corporate_credit_spreads_outputs\n'
    '    )\n'
    '    all_specs_with_a39 = _control_values_to_specs(\n'
    '        [[f"A{i} Sheet"] for i in range(3, 39)] + [["Corporate Credit Spreads"], [""]],\n'
    '        "A",\n'
    '        3,\n'
    '        True,\n'
    '    )\n'
    '    assert BatchIndicatorSpec("Corporate Credit Spreads", "A39") in all_specs_with_a39\n'
    '    assert (\n'
    '        select_indicator_engine_spec(\n'
    '            "Unmapped corporate spread sheet",\n'
    '            CONTROL_PANEL_THIRTY_SEVENTH_INDICATOR_CELL,\n'
    '        ).output_end_col\n'
    '        == "AL"\n'
    '    )\n'
    '    assert (\n'
    '        df_records(calculate_corporate_credit_spreads_outputs(corporate_raw))\n'
    '        == corporate_rows\n'
    '    )\n'
    '\n'
    '    mixed_updates = [\n'
    '        {"range": indicator_output_range("PMI", 10, "AB"), "values": [[1] * 26]},\n'
    '        {"range": indicator_output_range("Industrial", 10, "AJ"), "values": [[1] * 34]},\n'
    '        {"range": indicator_output_range("Consumer", 10, "AJ"), "values": [[1] * 34]},\n'
    '        {"range": indicator_output_range("Claims", 10, "AJ"), "values": [[1] * 34]},\n'
    '        {"range": indicator_output_range("Employment", 10, "AJ"), "values": [[1] * 34]},\n'
    '        {\n'
    '            "range": indicator_output_range("Average Hours", 10, "AJ"),\n'
    '            "values": [[1] * 34],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Wage Growth", 10, "AJ"),\n'
    '            "values": [[1] * 34],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range(\n'
    '                "Short-Term Inflation Expectations", 10, "AJ"\n'
    '            ),\n'
    '            "values": [[1] * 34],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Building Permits", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Core Inflation", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Core CPI Inflation", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Headline CPI Inflation", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {"range": indicator_output_range("Core PPI", 10, "AL"), "values": [[1] * 36]},\n'
    '        {\n'
    '            "range": indicator_output_range("Headline PPI", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {"range": indicator_output_range("Real Yield", 10, "AL"), "values": [[1] * 36]},\n'
    '        {"range": indicator_output_range("2Y Yield", 10, "AL"), "values": [[1] * 36]},\n'
    '        {"range": indicator_output_range("10Y Yield", 10, "AL"), "values": [[1] * 36]},\n'
    '        {\n'
    '            "range": indicator_output_range("Yield Curve", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Term Premium", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Interbank Rate", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Central Bank Liquidity", 10, "AS", "E"),\n'
    '            "values": [[1] * 41],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Banking Reserves", 10, "AQ"),\n'
    '            "values": [[1] * 41],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Broad Money", 10, "AO"),\n'
    '            "values": [[1] * 39],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Fiscal Balance", 10, "AJ"),\n'
    '            "values": [[1] * 34],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Real Government Spending", 10, "AM"),\n'
    '            "values": [[1] * 37],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Government Revenue", 10, "AP"),\n'
    '            "values": [[1] * 40],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Public Debt", 10, "AN"),\n'
    '            "values": [[1] * 38],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Interest Coverage", 10, "AN", "D"),\n'
    '            "values": [[1] * 37],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Financial Conditions", 10, "AM"),\n'
    '            "values": [[1] * 37],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Bank Credit", 10, "AP"),\n'
    '            "values": [[1] * 40],\n'
    '        },\n'
    '        {\n'
    '            "range": indicator_output_range("Corporate Credit Spreads", 10, "AL"),\n'
    '            "values": [[1] * 36],\n'
    '        },\n'
    '        {"range": indicator_output_range("Sentiment", 10, "AE"), "values": [[1] * 29]},\n'
    '        {\n'
    '            "range": indicator_output_range("Expectations", 10, "AE"),\n'
    '            "values": [[1] * 29],\n'
    '        },\n'
    '    ]\n'
    '    assert (\n'
    '        mixed_updates[0]["range"].endswith("C2:AB10")\n'
    '        and mixed_updates[1]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[2]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[3]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[4]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[5]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[6]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[7]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[8]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[9]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[10]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[11]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[12]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[13]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[14]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[15]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[16]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[17]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[18]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[19]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[20]["range"].endswith("E2:AS10")\n'
    '        and mixed_updates[21]["range"].endswith("C2:AQ10")\n'
    '        and mixed_updates[22]["range"].endswith("C2:AO10")\n'
    '        and mixed_updates[23]["range"].endswith("C2:AJ10")\n'
    '        and mixed_updates[24]["range"].endswith("C2:AM10")\n'
    '        and mixed_updates[25]["range"].endswith("C2:AP10")\n'
    '        and mixed_updates[26]["range"].endswith("C2:AN10")\n'
    '        and mixed_updates[27]["range"].endswith("D2:AN10")\n'
    '        and mixed_updates[28]["range"].endswith("C2:AM10")\n'
    '        and mixed_updates[29]["range"].endswith("C2:AP10")\n'
    '        and mixed_updates[30]["range"].endswith("C2:AL10")\n'
    '        and mixed_updates[31]["range"].endswith("C2:AE10")\n'
    '        and mixed_updates[32]["range"].endswith("C2:AE10")\n'
    '    )\n'
    '    try:\n'
    '        validate_write_range(out, len(out) + 1, start_col="B")\n'
    '        raise AssertionError("write guard allowed protected column")\n'
    '    except AssertionError:\n'
    '        pass\n'
    '    try:\n'
    '        validate_write_range(out, len(out) + 1, start_row=1)\n'
    '        raise AssertionError("write guard allowed row 1")\n'
    '    except AssertionError:\n'
    '        pass\n'
    '    central_write_range = validate_write_range(\n'
    '        central_out,\n'
    '        len(central_out) + 1,\n'
    '        start_col="E",\n'
    '        output_columns=CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS,\n'
    '        output_end_col="AS",\n'
    '    )\n'
    '    assert central_write_range.startswith("E2:AS")\n'
    '    reserves_write_range = validate_write_range(\n'
    '        reserves_out,\n'
    '        len(reserves_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=BANKING_RESERVES_OUTPUT_COLUMNS,\n'
    '        output_end_col="AQ",\n'
    '    )\n'
    '    assert reserves_write_range.startswith("C2:AQ")\n'
    '    broad_money_write_range = validate_write_range(\n'
    '        broad_money_out,\n'
    '        len(broad_money_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=BROAD_MONEY_OUTPUT_COLUMNS,\n'
    '        output_end_col="AO",\n'
    '    )\n'
    '    assert broad_money_write_range.startswith("C2:AO")\n'
    '    fiscal_write_range = validate_write_range(\n'
    '        fiscal_out,\n'
    '        len(fiscal_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=FISCAL_BALANCE_OUTPUT_COLUMNS,\n'
    '        output_end_col="AJ",\n'
    '    )\n'
    '    assert fiscal_write_range.startswith("C2:AJ")\n'
    '    real_gov_write_range = validate_write_range(\n'
    '        real_gov_out,\n'
    '        len(real_gov_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS,\n'
    '        output_end_col="AM",\n'
    '    )\n'
    '    assert real_gov_write_range.startswith("C2:AM")\n'
    '    gov_revenue_write_range = validate_write_range(\n'
    '        gov_revenue_out,\n'
    '        len(gov_revenue_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=GOVERNMENT_REVENUE_OUTPUT_COLUMNS,\n'
    '        output_end_col="AP",\n'
    '    )\n'
    '    assert gov_revenue_write_range.startswith("C2:AP")\n'
    '    public_debt_write_range = validate_write_range(\n'
    '        public_debt_out,\n'
    '        len(public_debt_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=PUBLIC_DEBT_OUTPUT_COLUMNS,\n'
    '        output_end_col="AN",\n'
    '    )\n'
    '    assert public_debt_write_range.startswith("C2:AN")\n'
    '    coverage_write_range = validate_write_range(\n'
    '        coverage_out,\n'
    '        len(coverage_out) + 1,\n'
    '        start_col="D",\n'
    '        output_columns=INTEREST_COVERAGE_OUTPUT_COLUMNS,\n'
    '        output_end_col="AN",\n'
    '    )\n'
    '    assert coverage_write_range.startswith("D2:AN")\n'
    '    financial_write_range = validate_write_range(\n'
    '        financial_out,\n'
    '        len(financial_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=FINANCIAL_CONDITIONS_OUTPUT_COLUMNS,\n'
    '        output_end_col="AM",\n'
    '    )\n'
    '    assert financial_write_range.startswith("C2:AM")\n'
    '    bank_credit_write_range = validate_write_range(\n'
    '        bank_credit_out,\n'
    '        len(bank_credit_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=BANK_CREDIT_OUTPUT_COLUMNS,\n'
    '        output_end_col="AP",\n'
    '    )\n'
    '    assert bank_credit_write_range.startswith("C2:AP")\n'
    '    headline_write_range = validate_write_range(\n'
    '        headline_out,\n'
    '        len(headline_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=HEADLINE_CPI_OUTPUT_COLUMNS,\n'
    '        output_end_col="AL",\n'
    '    )\n'
    '    assert headline_write_range == f"C2:AL{len(headline_out)+1}"\n'
    '    real_yield_write_range = validate_write_range(\n'
    '        real_yield_out,\n'
    '        len(real_yield_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=REAL_YIELD_OUTPUT_COLUMNS,\n'
    '        output_end_col="AL",\n'
    '    )\n'
    '    assert real_yield_write_range == f"C2:AL{len(real_yield_out)+1}"\n'
    '    two_year_write_range = validate_write_range(\n'
    '        two_year_out,\n'
    '        len(two_year_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=TWO_YEAR_YIELD_OUTPUT_COLUMNS,\n'
    '        output_end_col="AL",\n'
    '    )\n'
    '    assert two_year_write_range == f"C2:AL{len(two_year_out)+1}"\n'
    '    corporate_write_range = validate_write_range(\n'
    '        corporate_out,\n'
    '        len(corporate_out) + 1,\n'
    '        start_col="C",\n'
    '        output_columns=CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS,\n'
    '        output_end_col="AL",\n'
    '    )\n'
    '    assert corporate_write_range.startswith("C2:AL")\n'
    '    assert values_equivalent(0.0, "0.00%")\n'
    '    assert values_equivalent(0.0401, "4.01%")\n'
    '    assert values_equivalent("", "")\n'
    '    assert not values_equivalent("", 0.0)\n'
    '    actual = normalize_grid([["0.00%"]], 1, 1)\n'
    '    assert values_equivalent([[0.0]][0][0], actual[0][0])\n'
    '    fake_output = make_dataframe(\n'
    '        [{OUTPUT_COLUMNS[0]: 0.0, **{col: "" for col in OUTPUT_COLUMNS[1:]}}],\n'
    '        OUTPUT_COLUMNS,\n'
    '    )\n'
    '\n'
    '    class FakeWorksheet:\n'
    '        def __init__(self, rows: list[list[object]] | None = None) -> None:\n'
    '            self.ranges: list[str] = []\n'
    '            self.rows = rows or [["0.00%"]]\n'
    '\n'
    '        def get(\n'
    '            self, output_range: str, value_render_option: str | None = None\n'
    '        ) -> list[list[object]]:\n'
    '            self.ranges.append(output_range)\n'
    '            assert value_render_option == "UNFORMATTED_VALUE"\n'
    '            return self.rows\n'
    '\n'
    '    fake_ws = FakeWorksheet()\n'
    '    verify_written_outputs(fake_ws, fake_output, 2)\n'
    '    assert fake_ws.ranges == ["C2:AB2"]\n'
    '    avg_hours_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(avg_hours_out, AVERAGE_HOURS_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        avg_hours_verify_ws,\n'
    '        avg_hours_out,\n'
    '        len(avg_hours_out) + 1,\n'
    '        AVERAGE_HOURS_OUTPUT_COLUMNS,\n'
    '        "AJ",\n'
    '        "C",\n'
    '    )\n'
    '    assert avg_hours_verify_ws.ranges == [f"C2:AJ{len(avg_hours_out)+1}"]\n'
    '    central_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(central_out, CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        central_verify_ws,\n'
    '        central_out,\n'
    '        len(central_out) + 1,\n'
    '        CENTRAL_BANK_LIQUIDITY_OUTPUT_COLUMNS,\n'
    '        "AS",\n'
    '        "E",\n'
    '    )\n'
    '    assert central_verify_ws.ranges == [f"E2:AS{len(central_out)+1}"]\n'
    '    reserves_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(reserves_out, BANKING_RESERVES_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        reserves_verify_ws,\n'
    '        reserves_out,\n'
    '        len(reserves_out) + 1,\n'
    '        BANKING_RESERVES_OUTPUT_COLUMNS,\n'
    '        "AQ",\n'
    '        "C",\n'
    '    )\n'
    '    assert reserves_verify_ws.ranges == [f"C2:AQ{len(reserves_out)+1}"]\n'
    '    broad_money_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(broad_money_out, BROAD_MONEY_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        broad_money_verify_ws,\n'
    '        broad_money_out,\n'
    '        len(broad_money_out) + 1,\n'
    '        BROAD_MONEY_OUTPUT_COLUMNS,\n'
    '        "AO",\n'
    '        "C",\n'
    '    )\n'
    '    assert broad_money_verify_ws.ranges == [f"C2:AO{len(broad_money_out)+1}"]\n'
    '    fiscal_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(fiscal_out, FISCAL_BALANCE_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        fiscal_verify_ws,\n'
    '        fiscal_out,\n'
    '        len(fiscal_out) + 1,\n'
    '        FISCAL_BALANCE_OUTPUT_COLUMNS,\n'
    '        "AJ",\n'
    '        "C",\n'
    '    )\n'
    '    assert fiscal_verify_ws.ranges == [f"C2:AJ{len(fiscal_out)+1}"]\n'
    '    real_gov_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(real_gov_out, REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        real_gov_verify_ws,\n'
    '        real_gov_out,\n'
    '        len(real_gov_out) + 1,\n'
    '        REAL_GOVERNMENT_SPENDING_OUTPUT_COLUMNS,\n'
    '        "AM",\n'
    '        "C",\n'
    '    )\n'
    '    assert real_gov_verify_ws.ranges == [f"C2:AM{len(real_gov_out)+1}"]\n'
    '    gov_revenue_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(gov_revenue_out, GOVERNMENT_REVENUE_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        gov_revenue_verify_ws,\n'
    '        gov_revenue_out,\n'
    '        len(gov_revenue_out) + 1,\n'
    '        GOVERNMENT_REVENUE_OUTPUT_COLUMNS,\n'
    '        "AP",\n'
    '        "C",\n'
    '    )\n'
    '    assert gov_revenue_verify_ws.ranges == [f"C2:AP{len(gov_revenue_out)+1}"]\n'
    '    public_debt_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(public_debt_out, PUBLIC_DEBT_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        public_debt_verify_ws,\n'
    '        public_debt_out,\n'
    '        len(public_debt_out) + 1,\n'
    '        PUBLIC_DEBT_OUTPUT_COLUMNS,\n'
    '        "AN",\n'
    '        "C",\n'
    '    )\n'
    '    assert public_debt_verify_ws.ranges == [f"C2:AN{len(public_debt_out)+1}"]\n'
    '    coverage_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(coverage_out, INTEREST_COVERAGE_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        coverage_verify_ws,\n'
    '        coverage_out,\n'
    '        len(coverage_out) + 1,\n'
    '        INTEREST_COVERAGE_OUTPUT_COLUMNS,\n'
    '        "AN",\n'
    '        "D",\n'
    '    )\n'
    '    assert coverage_verify_ws.ranges == [f"D2:AN{len(coverage_out)+1}"]\n'
    '    financial_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(financial_out, FINANCIAL_CONDITIONS_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        financial_verify_ws,\n'
    '        financial_out,\n'
    '        len(financial_out) + 1,\n'
    '        FINANCIAL_CONDITIONS_OUTPUT_COLUMNS,\n'
    '        "AM",\n'
    '        "C",\n'
    '    )\n'
    '    assert financial_verify_ws.ranges == [f"C2:AM{len(financial_out)+1}"]\n'
    '    bank_credit_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(bank_credit_out, BANK_CREDIT_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        bank_credit_verify_ws,\n'
    '        bank_credit_out,\n'
    '        len(bank_credit_out) + 1,\n'
    '        BANK_CREDIT_OUTPUT_COLUMNS,\n'
    '        "AP",\n'
    '        "C",\n'
    '    )\n'
    '    assert bank_credit_verify_ws.ranges == [f"C2:AP{len(bank_credit_out)+1}"]\n'
    '    headline_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(headline_out, HEADLINE_CPI_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        headline_verify_ws,\n'
    '        headline_out,\n'
    '        len(headline_out) + 1,\n'
    '        HEADLINE_CPI_OUTPUT_COLUMNS,\n'
    '        "AL",\n'
    '        "C",\n'
    '    )\n'
    '    assert headline_verify_ws.ranges == [f"C2:AL{len(headline_out)+1}"]\n'
    '    real_yield_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(real_yield_out, REAL_YIELD_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        real_yield_verify_ws,\n'
    '        real_yield_out,\n'
    '        len(real_yield_out) + 1,\n'
    '        REAL_YIELD_OUTPUT_COLUMNS,\n'
    '        "AL",\n'
    '        "C",\n'
    '    )\n'
    '    assert real_yield_verify_ws.ranges == [f"C2:AL{len(real_yield_out)+1}"]\n'
    '    two_year_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(two_year_out, TWO_YEAR_YIELD_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        two_year_verify_ws,\n'
    '        two_year_out,\n'
    '        len(two_year_out) + 1,\n'
    '        TWO_YEAR_YIELD_OUTPUT_COLUMNS,\n'
    '        "AL",\n'
    '        "C",\n'
    '    )\n'
    '    assert two_year_verify_ws.ranges == [f"C2:AL{len(two_year_out)+1}"]\n'
    '    corporate_verify_ws = FakeWorksheet(\n'
    '        dataframe_to_sheet_rows(corporate_out, CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS)\n'
    '    )\n'
    '    verify_written_outputs(\n'
    '        corporate_verify_ws,\n'
    '        corporate_out,\n'
    '        len(corporate_out) + 1,\n'
    '        CORPORATE_CREDIT_SPREADS_OUTPUT_COLUMNS,\n'
    '        "AL",\n'
    '        "C",\n'
    '    )\n'
    '    assert corporate_verify_ws.ranges == [f"C2:AL{len(corporate_out)+1}"]\n'
    '    pmi_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["Level Score", "Leading Score", "Warning Score", "Data Status"],\n'
    '    )\n'
    '    industrial_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Growth Score": 1,\n'
    '                "Industrial Leading Score": 2,\n'
    '                "Industrial Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Growth Score",\n'
    '            "Industrial Leading Score",\n'
    '            "Industrial Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    consumer_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Growth Score": 1,\n'
    '                "Consumer Spending Leading Score": 2,\n'
    '                "Consumer Spending Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Growth Score",\n'
    '            "Consumer Spending Leading Score",\n'
    '            "Consumer Spending Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    sentiment_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Sentiment Level Score": 1,\n'
    '                "Consumer Sentiment Leading Score": 2,\n'
    '                "Consumer Sentiment Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Sentiment Level Score",\n'
    '            "Consumer Sentiment Leading Score",\n'
    '            "Consumer Sentiment Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    expectations_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Expectations Level Score": 1,\n'
    '                "Consumer Expectations Leading Score": 2,\n'
    '                "Consumer Expectations Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Expectations Level Score",\n'
    '            "Consumer Expectations Leading Score",\n'
    '            "Consumer Expectations Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    claims_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Claims Growth Score": 1,\n'
    '                "Initial Claims Leading Score": 2,\n'
    '                "Initial Claims Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Claims Growth Score",\n'
    '            "Initial Claims Leading Score",\n'
    '            "Initial Claims Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    employment_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Employment Growth Score": 1,\n'
    '                "Employment Leading Score": 2,\n'
    '                "Employment Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Employment Growth Score",\n'
    '            "Employment Leading Score",\n'
    '            "Employment Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    avg_hours_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "YoY Growth Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["YoY Growth Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    building_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Building Permits Growth Score": 1,\n'
    '                "Building Permits Leading Score": 2,\n'
    '                "Building Permits Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Building Permits Growth Score",\n'
    '            "Building Permits Leading Score",\n'
    '            "Building Permits Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    core_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Core Inflation Score": 1,\n'
    '                "Core Inflation Leading Score": 2,\n'
    '                "Core Inflation Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Core Inflation Score",\n'
    '            "Core Inflation Leading Score",\n'
    '            "Core Inflation Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    core_cpi_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Core CPI Inflation Score": 1,\n'
    '                "Core CPI Inflation Leading Score": 2,\n'
    '                "Core CPI Inflation Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Core CPI Inflation Score",\n'
    '            "Core CPI Inflation Leading Score",\n'
    '            "Core CPI Inflation Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    headline_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "YoY Inflation Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["YoY Inflation Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    core_ppi_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Core PPI Score": 1,\n'
    '                "Core PPI Leading Score": 2,\n'
    '                "Core PPI Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Core PPI Score",\n'
    '            "Core PPI Leading Score",\n'
    '            "Core PPI Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    headline_ppi_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "12M Headline PPI Score": 1,\n'
    '                "Headline PPI Leading Score": 2,\n'
    '                "Headline PPI Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "12M Headline PPI Score",\n'
    '            "Headline PPI Leading Score",\n'
    '            "Headline PPI Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    wage_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "4-Period Wage Growth Score": 1,\n'
    '                "Wage Pressure Leading Score": 2,\n'
    '                "Wage Pressure Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "4-Period Wage Growth Score",\n'
    '            "Wage Pressure Leading Score",\n'
    '            "Wage Pressure Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    short_expectations_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Short-Term Expectations Level Score": 1,\n'
    '                "Inflation Expectations Leading Score": 2,\n'
    '                "Inflation Expectations Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Short-Term Expectations Level Score",\n'
    '            "Inflation Expectations Leading Score",\n'
    '            "Inflation Expectations Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    real_yield_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Real Yield Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["Real Yield Level Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    two_year_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Yield Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["Yield Level Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    ten_year_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "10Y Yield Level Score": 1,\n'
    '                "10Y Yield Leading Score": 2,\n'
    '                "10Y Yield Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "10Y Yield Level Score",\n'
    '            "10Y Yield Leading Score",\n'
    '            "10Y Yield Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    yield_curve_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Curve Stress Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Curve Stress Level Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '            "Signal Validity",\n'
    '        ],\n'
    '    )\n'
    '    term_premium_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Term Premium Level Score": 1,\n'
    '                "Term Premium Leading Score": 2,\n'
    '                "Term Premium Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Term Premium Level Score",\n'
    '            "Term Premium Leading Score",\n'
    '            "Term Premium Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    interbank_rate_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Rate Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["Rate Level Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    central_liquidity_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Liquidity Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["Liquidity Level Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    banking_reserves_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Reserve Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        ["Reserve Level Score", "Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    broad_money_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Broad Money Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Broad Money Level Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '            "Signal Validity",\n'
    '        ],\n'
    '    )\n'
    '    fiscal_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Fiscal Balance Level Score": 1,\n'
    '                "Fiscal Impulse Leading Score": 2,\n'
    '                "Fiscal Impulse Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Fiscal Balance Level Score",\n'
    '            "Fiscal Impulse Leading Score",\n'
    '            "Fiscal Impulse Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    real_gov_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "1-Period Real Government Spending Growth Score": 1,\n'
    '                "Real Government Spending Leading Score": 2,\n'
    '                "Real Government Spending Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "1-Period Real Government Spending Growth Score",\n'
    '            "Real Government Spending Leading Score",\n'
    '            "Real Government Spending Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    gov_revenue_summary_df = make_dataframe(\n'
    '        [{"Leading Score": 2, "Warning Score": 3, "Signal Validity": "Valid"}],\n'
    '        ["Leading Score", "Warning Score", "Signal Validity"],\n'
    '    )\n'
    '    public_debt_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Public Debt Level Score": 1,\n'
    '                "Public Debt Leading Score": 2,\n'
    '                "Public Debt Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Public Debt Level Score",\n'
    '            "Public Debt Leading Score",\n'
    '            "Public Debt Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    coverage_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Interest Coverage Level Score": 1,\n'
    '                "Interest Coverage Leading Score": 2,\n'
    '                "Interest Coverage Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Interest Coverage Level Score",\n'
    '            "Interest Coverage Leading Score",\n'
    '            "Interest Coverage Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    financial_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Financial Conditions Level Stress Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Financial Conditions Level Stress Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '            "Signal Validity",\n'
    '        ],\n'
    '    )\n'
    '    bank_credit_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Bank Credit Level Score": 1,\n'
    '                "Leading Score": 2,\n'
    '                "Warning Score": 3,\n'
    '                "Signal Validity": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Bank Credit Level Score",\n'
    '            "Leading Score",\n'
    '            "Warning Score",\n'
    '            "Signal Validity",\n'
    '        ],\n'
    '    )\n'
    '    corporate_summary_df = make_dataframe(\n'
    '        [\n'
    '            {\n'
    '                "Corporate Credit Spread Level Score": 1,\n'
    '                "Corporate Credit Spread Leading Score": 2,\n'
    '                "Corporate Credit Spread Final Warning Score": 3,\n'
    '                "Data Status": "Valid",\n'
    '            }\n'
    '        ],\n'
    '        [\n'
    '            "Corporate Credit Spread Level Score",\n'
    '            "Corporate Credit Spread Leading Score",\n'
    '            "Corporate Credit Spread Final Warning Score",\n'
    '            "Data Status",\n'
    '        ],\n'
    '    )\n'
    '    missing_summary_df = make_dataframe([{"Other": 1}], ["Other"])\n'
    '    dummy_summary = IndicatorRunSummary("Summary Test", 1, 1, "C2:AB2", "d1", "d1", 1)\n'
    '    print_summary(dummy_summary, pmi_summary_df)\n'
    '    print_summary(dummy_summary, industrial_summary_df)\n'
    '    print_summary(dummy_summary, consumer_summary_df)\n'
    '    print_summary(dummy_summary, sentiment_summary_df)\n'
    '    print_summary(dummy_summary, expectations_summary_df)\n'
    '    print_summary(dummy_summary, claims_summary_df)\n'
    '    print_summary(dummy_summary, employment_summary_df)\n'
    '    print_summary(dummy_summary, avg_hours_summary_df)\n'
    '    print_summary(dummy_summary, building_summary_df)\n'
    '    print_summary(dummy_summary, core_summary_df)\n'
    '    print_summary(dummy_summary, core_cpi_summary_df)\n'
    '    print_summary(dummy_summary, headline_summary_df)\n'
    '    print_summary(dummy_summary, core_ppi_summary_df)\n'
    '    print_summary(dummy_summary, headline_ppi_summary_df)\n'
    '    print_summary(dummy_summary, wage_summary_df)\n'
    '    print_summary(dummy_summary, short_expectations_summary_df)\n'
    '    print_summary(dummy_summary, real_yield_summary_df)\n'
    '    print_summary(dummy_summary, two_year_summary_df)\n'
    '    print_summary(dummy_summary, ten_year_summary_df)\n'
    '    print_summary(dummy_summary, yield_curve_summary_df)\n'
    '    print_summary(dummy_summary, term_premium_summary_df)\n'
    '    print_summary(dummy_summary, interbank_rate_summary_df)\n'
    '    print_summary(dummy_summary, central_liquidity_summary_df)\n'
    '    print_summary(dummy_summary, banking_reserves_summary_df)\n'
    '    print_summary(dummy_summary, broad_money_summary_df)\n'
    '    print_summary(dummy_summary, fiscal_summary_df)\n'
    '    print_summary(dummy_summary, real_gov_summary_df)\n'
    '    print_summary(dummy_summary, gov_revenue_summary_df)\n'
    '    print_summary(dummy_summary, public_debt_summary_df)\n'
    '    print_summary(dummy_summary, coverage_summary_df)\n'
    '    print_summary(dummy_summary, financial_summary_df)\n'
    '    print_summary(dummy_summary, bank_credit_summary_df)\n'
    '    print_summary(dummy_summary, corporate_summary_df)\n'
    '    print_summary(dummy_summary, missing_summary_df)\n'
    '    assert (\n'
    '        count_nonblank_column(\n'
    '            missing_summary_df,\n'
    '            first_existing_column(missing_summary_df, LEVEL_SCORE_COLUMNS),\n'
    '        )\n'
    '        == 0\n'
    '    )\n'
    '    assert _control_values_to_specs([["Sheet A"], ["Sheet B"], [""]], "A", 3, True) == [\n'
    '        BatchIndicatorSpec("Sheet A", "A3"),\n'
    '        BatchIndicatorSpec("Sheet B", "A4"),\n'
    '    ]\n'
    '    assert _control_values_to_specs(\n'
    '        [["Sheet A"], [""], ["Sheet C"]], "A", 3, False\n'
    '    ) == [BatchIndicatorSpec("Sheet A", "A3"), BatchIndicatorSpec("Sheet C", "A5")]\n'
    '    assert filter_indicator_specs(\n'
    '        [BatchIndicatorSpec("Sheet A", "A3"), BatchIndicatorSpec("Sheet B", "A4")],\n'
    '        ["Sheet B"],\n'
    '        None,\n'
    '    ) == [BatchIndicatorSpec("Sheet B", "A4")]\n'
    '    assert filter_indicator_specs(\n'
    '        [BatchIndicatorSpec("Sheet A", "A3"), BatchIndicatorSpec("Sheet B", "A4")],\n'
    '        None,\n'
    '        ["Sheet A"],\n'
    '    ) == [BatchIndicatorSpec("Sheet B", "A4")]\n'
    '    assert quote_sheet_name("Services New Orders") == "\'Services New Orders\'"\n'
    '    assert quote_sheet_name("Owner\'s PMI") == "\'Owner\'\'s PMI\'"\n'
    '    assert indicator_input_range("Services New Orders") == "\'Services New Orders\'!A2:B"\n'
    '    assert (\n'
    '        indicator_input_range("Central Bank Liquidity", "D")\n'
    '        == "\'Central Bank Liquidity\'!A2:D"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Services New Orders", 678)\n'
    '        == "\'Services New Orders\'!C2:AB678"\n'
    '    )\n'
    '    assert (\n'
    '        indicator_output_range("Central Bank Liquidity", 678, "AS", "E")\n'
    '        == "\'Central Bank Liquidity\'!E2:AS678"\n'
    '    )\n'
    '    assert chunked([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]\n'
    '\n'
    '    class FakeBatchWorkbook:\n'
    '        def __init__(self) -> None:\n'
    '            self.get_calls: list[list[str]] = []\n'
    '            self.update_payloads: list[dict[str, object]] = []\n'
    '\n'
    '        def values_batch_get(\n'
    '            self, ranges: list[str], params: dict[str, object]\n'
    '        ) -> dict[str, object]:\n'
    '            self.get_calls.append(list(ranges))\n'
    '            return {"valueRanges": [{"range": r, "values": [[r]]} for r in ranges]}\n'
    '\n'
    '        def values_batch_update(self, payload: dict[str, object]) -> dict[str, object]:\n'
    '            self.update_payloads.append(payload)\n'
    '            return {}\n'
    '\n'
    '    fake_batch = FakeBatchWorkbook()\n'
    '    got = batch_get_values(\n'
    '        fake_batch,\n'
    '        ["r1", "r2", "r3"],\n'
    '        chunk_size=2,\n'
    '        sleep_between_batches=0,\n'
    '        max_retries=1,\n'
    '    )\n'
    '    assert fake_batch.get_calls == [["r1", "r2"], ["r3"]]\n'
    '    assert got["r1"] == [["r1"]]\n'
    '    updates = [\n'
    '        {"range": "\'Services New Orders\'!C2:AB678", "values": [[1] * 26]},\n'
    '        {"range": "\'Other\'!C2:AB2", "values": [[2] * 26]},\n'
    '    ]\n'
    '    batch_update_values(\n'
    '        fake_batch, updates, chunk_size=20, sleep_between_batches=0, max_retries=1\n'
    '    )\n'
    '    assert len(fake_batch.update_payloads) == 1\n'
    '    assert len(fake_batch.update_payloads[0]["data"]) == 2\n'
    '    attempts = {"count": 0}\n'
    '\n'
    '    class FakeQuotaError(Exception):\n'
    '        pass\n'
    '\n'
    '    old_sleep = time.sleep\n'
    '    old_uniform = random.uniform\n'
    '    try:\n'
    '        time.sleep = lambda seconds: None  # type: ignore[assignment]\n'
    '        random.uniform = lambda a, b: 0.0  # type: ignore[assignment]\n'
    '\n'
    '        def flaky() -> str:\n'
    '            attempts["count"] += 1\n'
    '            if attempts["count"] < 3:\n'
    '                raise FakeQuotaError("429 quota exceeded")\n'
    '            return "ok"\n'
    '\n'
    '        assert (\n'
    '            call_with_backoff(\n'
    '                flaky, operation_name="fake", max_attempts=3, base_sleep_seconds=0.0\n'
    '            )\n'
    '            == "ok"\n'
    '        )\n'
    '        assert attempts["count"] == 3\n'
    '    finally:\n'
    '        time.sleep = old_sleep  # type: ignore[assignment]\n'
    '        random.uniform = old_uniform  # type: ignore[assignment]\n'
    '    assert not should_verify_write(parse_args(["--all", "--write"]))\n'
    '    assert should_verify_write(parse_args(["--all", "--write", "--verify"]))\n'
    '    assert not should_verify_write(parse_args(["--all", "--write", "--no-verify"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A14", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A23", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A23", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A24", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A24", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A25", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A25", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A26", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A26", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A28", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A28", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A29", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A29", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A30", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A30", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A31", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A31", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A34", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A34", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A37", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A37", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A38", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A38", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A39", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A39", "--write"]))\n'
    '    assert not should_verify_write(\n'
    '        parse_args(["--control-cell", "A18", "--write", "--no-verify"])\n'
    '    )\n'
    '    assert should_verify_write(parse_args(["--control-cell", "A18", "--write"]))\n'
    '    try:\n'
    '        main(["--verify", "--no-verify"])\n'
    '        raise AssertionError("verify/no-verify conflict was not rejected")\n'
    '    except ValueError:\n'
    '        pass\n'
    '    assert "gspread" not in sys.modules\n'
    '    print("Self-tests passed.")\n'
    '\n'
    '\n'
    'def verify_output_rows(\n'
    '    actual_rows: list[list[object]], output_df: Any, output_range: str\n'
    ') -> None:\n'
    '    expected_rows = dataframe_to_sheet_rows(output_df)\n'
    '    col_count = output_df.shape[1]\n'
    '    actual_norm = normalize_grid(actual_rows, len(expected_rows), col_count)\n'
    '    for r in range(len(expected_rows)):\n'
    '        for c in range(col_count):\n'
    '            if not values_equivalent(expected_rows[r][c], actual_norm[r][c]):\n'
    '                raise AssertionError(\n'
    '                    f"Value mismatch in {output_range} at output row={r}, output col={c}: actual={actual_norm[r][c]!r} expected={expected_rows[r][c]!r}"\n'
    '                )\n'
    '\n'
    '\n'
    'def process_indicators_batch(\n'
    '    workbook: Any, specs: list[BatchIndicatorSpec], args: argparse.Namespace\n'
    ') -> tuple[list[BatchIndicatorResult], int]:\n'
    '    engine_specs = [\n'
    '        select_indicator_engine_spec(spec.sheet_name, spec.control_cell)\n'
    '        for spec in specs\n'
    '    ]\n'
    '    input_ranges = [\n'
    '        indicator_input_range(\n'
    '            spec.sheet_name, engine_spec.input_end_col, engine_spec.input_start_col\n'
    '        )\n'
    '        for spec, engine_spec in zip(specs, engine_specs)\n'
    '    ]\n'
    '    raw_by_range = batch_get_values(\n'
    '        workbook,\n'
    '        input_ranges,\n'
    '        chunk_size=args.batch_size,\n'
    '        sleep_between_batches=args.sleep_between_batches,\n'
    '        max_retries=args.max_retries,\n'
    '    )\n'
    '    results: list[BatchIndicatorResult] = []\n'
    '    calculated: list[tuple[BatchIndicatorSpec, IndicatorRunSummary, Any, str]] = []\n'
    '    total_rows = 0\n'
    '    total_cells = 0\n'
    '    for spec, engine_spec, input_range in zip(specs, engine_specs, input_ranges):\n'
    '        try:\n'
    '            raw_df = read_indicator_values(\n'
    '                raw_by_range.get(input_range, []), engine_spec.input_end_col\n'
    '            )\n'
    '            output_df = engine_spec.calculate(raw_df)\n'
    '            output_columns = engine_spec.output_columns\n'
    '            summary = build_summary(spec.sheet_name, raw_df, output_df)\n'
    '            summary = IndicatorRunSummary(\n'
    '                summary.indicator_sheet_name,\n'
    '                summary.rows_read,\n'
    '                summary.rows_calculated,\n'
    '                f"{engine_spec.output_start_col}2:{engine_spec.output_end_col}{summary.rows_calculated + 1}",\n'
    '                summary.first_date,\n'
    '                summary.last_date,\n'
    '                summary.nonblank_raw_values,\n'
    '            )\n'
    '            print_summary(summary, output_df)\n'
    '            full_output_range = indicator_output_range(\n'
    '                spec.sheet_name,\n'
    '                summary.rows_calculated + 1,\n'
    '                engine_spec.output_end_col,\n'
    '                engine_spec.output_start_col,\n'
    '            )\n'
    '            calculated.append((spec, summary, output_df, full_output_range))\n'
    '            total_rows += summary.rows_calculated\n'
    '            total_cells += summary.rows_calculated * len(output_columns)\n'
    '            if not args.write:\n'
    '                results.append(\n'
    '                    BatchIndicatorResult(spec.sheet_name, True, summary.output_range)\n'
    '                )\n'
    '        except Exception as exc:\n'
    '            LOGGER.exception("Indicator failed during calculation: %s", spec.sheet_name)\n'
    '            results.append(BatchIndicatorResult(spec.sheet_name, False, error=str(exc)))\n'
    '    write_calls = 0\n'
    '    if args.write and calculated:\n'
    '        updates = [\n'
    '            {"range": full_range, "values": dataframe_to_sheet_rows(output_df)}\n'
    '            for _, _, output_df, full_range in calculated\n'
    '        ]\n'
    '        try:\n'
    '            write_calls = batch_update_values(\n'
    '                workbook,\n'
    '                updates,\n'
    '                chunk_size=args.batch_size,\n'
    '                sleep_between_batches=args.sleep_between_batches,\n'
    '                max_retries=args.max_retries,\n'
    '            )\n'
    '        except Exception as exc:\n'
    '            LOGGER.exception("Batch write failed")\n'
    '            results.extend(\n'
    '                BatchIndicatorResult(\n'
    '                    spec.sheet_name, False, summary.output_range, str(exc)\n'
    '                )\n'
    '                for spec, summary, _, _ in calculated\n'
    '            )\n'
    '            LOGGER.info(\n'
    '                "Batch totals: requested=%s succeeded=%s failed=%s skipped=%s write_batch_calls=%s total_calculated_rows=%s total_written_cells=%s",\n'
    '                len(specs),\n'
    '                sum(r.ok for r in results),\n'
    '                sum(not r.ok for r in results),\n'
    '                max(0, len(specs) - len(results)),\n'
    '                write_calls,\n'
    '                total_rows,\n'
    '                0,\n'
    '            )\n'
    '            return results, write_calls\n'
    '        should_verify = bool(args.verify and not args.no_verify)\n'
    '        if should_verify:\n'
    '            verify_ranges = [full_range for _, _, _, full_range in calculated]\n'
    '            try:\n'
    '                actual_by_range = batch_get_values(\n'
    '                    workbook,\n'
    '                    verify_ranges,\n'
    '                    value_render_option="UNFORMATTED_VALUE",\n'
    '                    chunk_size=args.batch_size,\n'
    '                    sleep_between_batches=args.sleep_between_batches,\n'
    '                    max_retries=args.max_retries,\n'
    '                )\n'
    '            except Exception as exc:\n'
    '                LOGGER.exception("Batch verification read failed")\n'
    '                results.extend(\n'
    '                    BatchIndicatorResult(\n'
    '                        spec.sheet_name, False, summary.output_range, str(exc)\n'
    '                    )\n'
    '                    for spec, summary, _, _ in calculated\n'
    '                )\n'
    '                LOGGER.info(\n'
    '                    "Batch totals: requested=%s succeeded=%s failed=%s skipped=%s write_batch_calls=%s total_calculated_rows=%s total_written_cells=%s",\n'
    '                    len(specs),\n'
    '                    sum(r.ok for r in results),\n'
    '                    sum(not r.ok for r in results),\n'
    '                    max(0, len(specs) - len(results)),\n'
    '                    write_calls,\n'
    '                    total_rows,\n'
    '                    total_cells,\n'
    '                )\n'
    '                return results, write_calls\n'
    '            for spec, summary, output_df, full_range in calculated:\n'
    '                try:\n'
    '                    verify_output_rows(\n'
    '                        actual_by_range.get(full_range, []), output_df, full_range\n'
    '                    )\n'
    '                    results.append(\n'
    '                        BatchIndicatorResult(\n'
    '                            spec.sheet_name, True, summary.output_range\n'
    '                        )\n'
    '                    )\n'
    '                except Exception as exc:\n'
    '                    LOGGER.exception(\n'
    '                        "Indicator failed during batch verification: %s",\n'
    '                        spec.sheet_name,\n'
    '                    )\n'
    '                    results.append(\n'
    '                        BatchIndicatorResult(\n'
    '                            spec.sheet_name, False, summary.output_range, str(exc)\n'
    '                        )\n'
    '                    )\n'
    '        else:\n'
    '            LOGGER.info(\n'
    '                "Batch mode verification skipped by default for speed/quota safety. Use --verify to enable batch verification."\n'
    '            )\n'
    '            results.extend(\n'
    '                BatchIndicatorResult(spec.sheet_name, True, summary.output_range)\n'
    '                for spec, summary, _, _ in calculated\n'
    '            )\n'
    '    LOGGER.info(\n'
    '        "Batch totals: requested=%s succeeded=%s failed=%s skipped=%s write_batch_calls=%s total_calculated_rows=%s total_written_cells=%s",\n'
    '        len(specs),\n'
    '        sum(r.ok for r in results),\n'
    '        sum(not r.ok for r in results),\n'
    '        max(0, len(specs) - len(results)),\n'
    '        write_calls,\n'
    '        total_rows,\n'
    '        total_cells if args.write else 0,\n'
    '    )\n'
    '    return results, write_calls\n'
    '\n'
    '\n'
    'def should_use_batch_mode(args: argparse.Namespace) -> bool:\n'
    '    return bool(args.all or args.control_range)\n'
    '\n'
    '\n'
    'def should_verify_write(args: argparse.Namespace) -> bool:\n'
    '    if not args.write or args.no_verify:\n'
    '        return False\n'
    '    if should_use_batch_mode(args):\n'
    '        return bool(args.verify)\n'
    '    return True\n'
    '\n'
    '\n'
    'def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:\n'
    '    parser = argparse.ArgumentParser(description=__doc__)\n'
    '    parser.add_argument(\n'
    '        "--write",\n'
    '        action="store_true",\n'
    '        default=True,\n'
    '        help="Write and verify calculated values (default).",\n'
    '    )\n'
    '    source = parser.add_mutually_exclusive_group()\n'
    '    source.add_argument(\n'
    '        "--indicator-sheet",\n'
    '        help="Process one indicator sheet directly by name, bypassing Control Panel lookup.",\n'
    '    )\n'
    '    source.add_argument(\n'
    '        "--control-range",\n'
    '        help="Read indicator sheet names from this Control Panel range, e.g. A3:A20.",\n'
    '    )\n'
    '    source.add_argument(\n'
    '        "--all",\n'
    '        action="store_true",\n'
    '        help="Read Control Panel!A3:A once until the first blank and process all listed indicators.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--self-test",\n'
    '        action="store_true",\n'
    '        help="Run local calculation self-tests without Google Sheets.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--verify", action="store_true", help="Read back and verify written output."\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--no-verify",\n'
    '        action="store_true",\n'
    '        help="Skip read-back verification after writing.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--batch-size",\n'
    '        type=int,\n'
    '        default=20,\n'
    '        help="Ranges per batch_get/batch_update chunk.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--sleep-between-batches",\n'
    '        type=float,\n'
    '        default=1.0,\n'
    '        help="Seconds to sleep between batch API chunks.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--max-retries",\n'
    '        type=int,\n'
    '        default=7,\n'
    '        help="Maximum quota-backoff retry attempts.",\n'
    '    )\n'
    '    parser.add_argument("--service-account-path", default=DEFAULT_SERVICE_ACCOUNT_PATH)\n'
    '    parser.add_argument("--spreadsheet-id", default=SPREADSHEET_ID)\n'
    '    parser.add_argument(\n'
    '        "--control-cell",\n'
    '        default=CONTROL_PANEL_FIRST_INDICATOR_CELL,\n'
    '        choices=[\n'
    '            CONTROL_PANEL_FIRST_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_SECOND_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRD_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_FOURTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_FIFTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_SIXTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_SEVENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_EIGHTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_NINTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_ELEVENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWELFTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_FOURTEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_FIFTEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_SIXTEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_SEVENTEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_EIGHTEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_NINETEENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTIETH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_FIRST_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_SECOND_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_THIRD_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_FOURTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_FIFTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_SIXTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_SEVENTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_EIGHTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_TWENTY_NINTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTIETH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_FIRST_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_SECOND_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_THIRD_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_FOURTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_FIFTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_SIXTH_INDICATOR_CELL,\n'
    '            CONTROL_PANEL_THIRTY_SEVENTH_INDICATOR_CELL,\n'
    '        ],\n'
    '        help="Control Panel cell for single-indicator lookup; default A3.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--only",\n'
    '        nargs="*",\n'
    '        help="When used with --all or --control-range, process only these sheet names.",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--exclude",\n'
    '        nargs="*",\n'
    '        help="When used with --all or --control-range, skip these sheet names.",\n'
    '    )\n'
    '    parser.add_argument("--log-level", default="INFO")\n'
    '    return parser.parse_args(argv)\n'
    '\n'
    '\n'
    'from notifications.lifecycle import notify_entrypoint, record_failure\n'
    '\n'
    '\n'
    '@notify_entrypoint("Indicator analysis")\n'
    'def main(argv: Sequence[str] | None = None) -> int:\n'
    '    args = parse_args(argv)\n'
    '    logging.basicConfig(\n'
    '        level=(\n'
    '            logging.WARNING\n'
    '            if os.getenv("US_PIPELINE_COMPACT_OUTPUT") == "1"\n'
    '            else getattr(logging, args.log_level.upper(), logging.INFO)\n'
    '        ),\n'
    '        format="%(levelname)s:%(name)s:%(message)s",\n'
    '    )\n'
    '    if args.self_test:\n'
    '        run_self_tests()\n'
    '        return 0\n'
    '    config = EngineConfig(args.spreadsheet_id, args.service_account_path)\n'
    '    if args.verify and args.no_verify:\n'
    '        raise ValueError("Use either --verify or --no-verify, not both.")\n'
    '    if (args.only or args.exclude) and not (args.all or args.control_range):\n'
    '        raise ValueError(\n'
    '            "--only/--exclude are only valid with --all or --control-range"\n'
    '        )\n'
    '    workbook = open_workbook(config)\n'
    '    specs = build_indicator_specs(workbook, args)\n'
    '    specs = filter_indicator_specs(specs, args.only, args.exclude)\n'
    '    if not specs:\n'
    '        LOGGER.warning("No indicators selected.")\n'
    '        return 0\n'
    '    if should_use_batch_mode(args):\n'
    '        results, _ = process_indicators_batch(workbook, specs, args)\n'
    '    else:\n'
    '        results = [\n'
    '            process_indicator(\n'
    '                workbook, spec, write=args.write, verify=should_verify_write(args)\n'
    '            )\n'
    '            for spec in specs\n'
    '        ]\n'
    '    print_batch_summary(results)\n'
    '    if any(not result.ok for result in results):\n'
    '        record_failure(\n'
    '            "; ".join(\n'
    '                f"{result.sheet_name}: {result.error}"\n'
    '                for result in results\n'
    '                if not result.ok\n'
    '            )\n'
    '        )\n'
    '    return 1 if any(not result.ok for result in results) else 0\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    sys.exit(main())\n'
)

# Embedded source: run_relationships.py
SOURCES['run_relationships.py'] = (
    '#!/usr/bin/env python3\n'
    '"""Rebuild coincident and leading level and momentum histories in the configured Google Sheets."""\n'
    '\n'
    'import argparse\n'
    'from datetime import datetime, timezone\n'
    'from run_lock import acquire_lock\n'
    'import json\n'
    'import logging\n'
    'import sys\n'
    'from config import ROOT, WORKBOOKS, credentials_path, select_workbooks\n'
    'from google_sheets import Sheets\n'
    'import momentum_pipeline\n'
    'import calculation_helper\n'
    'from pipeline import load, validate_before_publish, verify\n'
    'from notifications.lifecycle import notify_entrypoint, record_failure, relationship_name\n'
    '\n'
    '\n'
    '@notify_entrypoint(relationship_name)\n'
    'def main(argv=None):\n'
    '    parser = argparse.ArgumentParser(description=__doc__)\n'
    '    selection = parser.add_mutually_exclusive_group()\n'
    '    selection.add_argument(\n'
    '        "--correlation", action="store_true", help="Run only rolling correlation"\n'
    '    )\n'
    '    selection.add_argument(\n'
    '        "--spread", action="store_true", help="Run only spread z-scores"\n'
    '    )\n'
    '    selection.add_argument("--spread-momentum", action="store_true")\n'
    '    selection.add_argument("--correlation-momentum", action="store_true")\n'
    '    selection.add_argument(\n'
    '        "--momentum",\n'
    '        action="store_true",\n'
    '        help="Only momentum workbooks; original workbooks remain read-only",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--dataset",\n'
    '        choices=("all", "coincident", "leading"),\n'
    '        default="all",\n'
    '        help="Keep each dataset in its own four workbooks (default: both datasets)",\n'
    '    )\n'
    '    parser.add_argument(\n'
    '        "--credentials-file",\n'
    '        help="Existing Google credential file; otherwise environment/parent discovery",\n'
    '    )\n'
    '    args = parser.parse_args(argv)\n'
    '    logging.basicConfig(\n'
    '        level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s"\n'
    '    )\n'
    '    run_dir = ROOT / "work" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")\n'
    '    run_dir.mkdir(parents=True)\n'
    '    reports = []\n'
    '    sheets = None\n'
    '    with (ROOT / "work" / "pipeline.lock").open("w") as lock:\n'
    '        try:\n'
    '            acquire_lock(lock)\n'
    '        except BlockingIOError:\n'
    '            record_failure("Another pipeline run is active.")\n'
    '            print("Another pipeline run is active.", file=sys.stderr)\n'
    '            return 1\n'
    '        try:\n'
    '            choice = next(\n'
    '                (\n'
    '                    k\n'
    '                    for k in (\n'
    '                        "correlation",\n'
    '                        "spread",\n'
    '                        "spread_momentum",\n'
    '                        "correlation_momentum",\n'
    '                    )\n'
    '                    if getattr(args, k)\n'
    '                ),\n'
    '                None,\n'
    '            )\n'
    '            selected = select_workbooks(args.dataset, choice, args.momentum)\n'
    '            logging.info(\n'
    '                "Selected %s: %s", args.dataset, ", ".join(b.key for b in selected)\n'
    '            )\n'
    '            sheets = Sheets(\n'
    '                credentials_path(args.credentials_file),\n'
    '                write_ids={b.spreadsheet_id for b in selected},\n'
    '            )\n'
    '            from source_freshness import validate_macro_summary\n'
    '\n'
    '            checked_datasets = set()\n'
    '            for book in selected:\n'
    '                if book.dataset not in checked_datasets:\n'
    '                    validate_macro_summary(sheets, book)\n'
    '                    checked_datasets.add(book.dataset)\n'
    '            # Publish base results first; momentum then requires refreshed imported values.\n'
    '            for momentum_phase in (False, True):\n'
    '                states = []\n'
    '                for book in selected:\n'
    '                    if book.kind.endswith("_momentum") != momentum_phase:\n'
    '                        continue\n'
    '                    logging.info("Reading %s (%s)", book.title, book.output_tab)\n'
    '                    from source_freshness import prepare_dataset_links\n'
    '\n'
    '                    prepare_dataset_links(sheets, book, run_dir)\n'
    '                    loader = momentum_pipeline.load if momentum_phase else load\n'
    '                    state = loader(sheets, book)\n'
    '                    state["helper"] = calculation_helper.prepare(sheets, state)\n'
    '                    state["report"]["helper_preview"] = {\n'
    '                        "pair": state["helper"]["selection"],\n'
    '                        "dates": len(state["helper"]["table"]),\n'
    '                        "numeric_checks": state["helper"]["matched"],\n'
    '                    }\n'
    '                    states.append(state)\n'
    '                    reports.append(state["report"])\n'
    '                    (run_dir / (book.key + "_report.json")).write_text(\n'
    '                        json.dumps(state["report"], indent=2)\n'
    '                    )\n'
    '                for state in states:\n'
    '                    if momentum_phase:\n'
    '                        momentum_pipeline.assert_fresh(\n'
    '                            sheets,\n'
    '                            state["book"],\n'
    '                            state["source"],\n'
    '                            state["output"],\n'
    '                            state.get("pair_source"),\n'
    '                            state.get("pair_controls"),\n'
    '                        )\n'
    '                    state["formulas"] = validate_before_publish(sheets, state)\n'
    '                for state in states:\n'
    '                    b = state["book"]\n'
    '                    if momentum_phase:\n'
    '                        momentum_pipeline.assert_fresh(\n'
    '                            sheets,\n'
    '                            b,\n'
    '                            state["source"],\n'
    '                            state["output"],\n'
    '                            state.get("pair_source"),\n'
    '                            state.get("pair_controls"),\n'
    '                        )\n'
    '                    state["report"]["status"] = "write_started"\n'
    '                    snapshot = {\n'
    '                        k: state[k]\n'
    '                        for k in (\n'
    '                            "meta",\n'
    '                            "source",\n'
    '                            "output",\n'
    '                            "controls",\n'
    '                            "formulas",\n'
    '                            "report",\n'
    '                        )\n'
    '                    }\n'
    '                    snapshot.update(\n'
    '                        {\n'
    '                            k: state[k]\n'
    '                            for k in ("pair_source", "pair_controls")\n'
    '                            if k in state\n'
    '                        }\n'
    '                    )\n'
    '                    snapshot["spreadsheet_id"] = b.spreadsheet_id\n'
    '                    backup = run_dir / (b.key + "_before.json.gz")\n'
    '                    state["report"]["backup"] = str(backup)\n'
    '                    state["report"]["cells_written"] = sheets.publish(\n'
    '                        b, state["properties"], state["values"], backup, snapshot\n'
    '                    )\n'
    '                    verify(sheets, state)\n'
    '                    calculation_helper.publish(sheets, state, state["helper"], run_dir)\n'
    '                    if momentum_phase:\n'
    '                        momentum_pipeline.assert_fresh(\n'
    '                            sheets,\n'
    '                            b,\n'
    '                            state["source"],\n'
    '                            state["output"],\n'
    '                            state.get("pair_source"),\n'
    '                            state.get("pair_controls"),\n'
    '                        )\n'
    '                    state["report"]["status"] = "published_and_verified"\n'
    '                    from notifications.changes import record\n'
    '\n'
    '                    difference = state["report"]["comparison"]\n'
    '                    record(\n'
    '                        "Relationship scores",\n'
    '                        b.key.replace("_", " "),\n'
    '                        added=difference["added_numeric"],\n'
    '                        updated=difference["numeric_mismatches"],\n'
    '                        removed=difference["removed_numeric"],\n'
    '                    )\n'
    '\n'
    '                    (run_dir / (b.key + "_report.json")).write_text(\n'
    '                        json.dumps(state["report"], indent=2)\n'
    '                    )\n'
    '            (run_dir / "summary.json").write_text(json.dumps(reports, indent=2))\n'
    '            for report in reports:\n'
    '                print(\n'
    '                    f"{report[\'key\'].upper()}: {report[\'status\']} | window={report[\'window\']} | source dates={report[\'source_dates\']} | indicators={report[\'indicators\']} | pairs={report[\'pairs\']}"\n'
    '                )\n'
    '                print(\n'
    '                    f"  Calculated: {report[\'first_calculated_date\']} to {report[\'latest_calculated_date\']} | numeric={report[\'numeric_cells\']:,} | blank/insufficient/undefined={report[\'blank_observations\']:,}"\n'
    '                )\n'
    '                print(\n'
    '                    f"  Invalid mappings={report[\'invalid_indicator_mappings\']} | score cells written={report[\'cells_written\']:,}"\n'
    '                )\n'
    '                if "momentum_lookback" in report:\n'
    '                    print(\n'
    '                        f"  Momentum lookback={report[\'momentum_lookback\']} paired updates; normalisation uses prior changes only"\n'
    '                    )\n'
    '                if "helper" in report:\n'
    '                    print(\n'
    '                        f"  Helper: {report[\'helper\'][\'pair\']} | verified dates={report[\'helper\'][\'dates\']}"\n'
    '                    )\n'
    '                comparison = report["comparison"]\n'
    '                print(\n'
    '                    f"  Compared={comparison[\'compared_numeric\']:,} | numeric differences={comparison[\'numeric_mismatches\']} | historical blank differences={comparison[\'historical_blank_mismatches\']} | recovered tail={comparison[\'new_tail_numeric\']}"\n'
    '                )\n'
    '            print(f"Audit directory: {run_dir}")\n'
    '            return 0\n'
    '        except KeyboardInterrupt:\n'
    '            possible_writes = bool(getattr(sheets, "writes_attempted", False))\n'
    '            message = (\n'
    '                "Stopped by Ctrl+C. Writes may be incomplete; rerun to rebuild and verify all selected outputs."\n'
    '                if possible_writes\n'
    '                else "Stopped by Ctrl+C before any Google Sheets writes. Existing outputs were not changed."\n'
    '            )\n'
    '            (run_dir / "interrupted.json").write_text(\n'
    '                json.dumps(\n'
    '                    {\n'
    '                        "status": "interrupted",\n'
    '                        "writes_attempted": possible_writes,\n'
    '                        "reports": reports,\n'
    '                    },\n'
    '                    indent=2,\n'
    '                )\n'
    '            )\n'
    '            logging.warning(message)\n'
    '            record_failure(message)\n'
    '            print(f"Audit directory: {run_dir}", file=sys.stderr)\n'
    '            return 130\n'
    '        except Exception as exc:\n'
    '            record_failure(f"{type(exc).__name__}: {exc}")\n'
    '            failure = {\n'
    '                "error": str(exc),\n'
    '                "reports": reports,\n'
    '                "status": "failed; inspect reports/backups before retrying",\n'
    '            }\n'
    '            (run_dir / "failure.json").write_text(json.dumps(failure, indent=2))\n'
    '            logging.error("%s: %s", type(exc).__name__, exc)\n'
    '            print(f"Audit directory: {run_dir}", file=sys.stderr)\n'
    '            return 1\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    sys.exit(main())\n'
)

# Embedded source: config.py
SOURCES['config.py'] = (
    '"""Workbook identities, configuration parsing, and shared credential discovery."""\n'
    '\n'
    'from dataclasses import dataclass\n'
    'from pathlib import Path\n'
    'import os\n'
    '\n'
    'ROOT = Path(__file__).resolve().parent\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class Workbook:\n'
    '    kind: str\n'
    '    spreadsheet_id: str\n'
    '    title: str\n'
    '    output_tab: str\n'
    '    output_id: int\n'
    '    control_key: str\n'
    '    source_tab: str = "Final scores data"\n'
    '    control_tab: str = "Control Panel"\n'
    '    dataset: str = "coincident"\n'
    '\n'
    '    @property\n'
    '    def key(self):\n'
    '        return f"{self.dataset}_{self.kind}"\n'
    '\n'
    '    @property\n'
    '    def macro_tab(self):\n'
    '        return {"coincident": "Coincident Score", "leading": "Leading Score"}[\n'
    '            self.dataset\n'
    '        ]\n'
    '\n'
    '\n'
    'WORKBOOKS = (\n'
    '    Workbook(\n'
    '        "correlation",\n'
    '        "1h-70prpKWYT7Cstg8CR6VRJdfJXaVaxeJL2xX5cXK_k",\n'
    '        "Correl Score - COIN",\n'
    '        "Correlation Score",\n'
    '        1400973068,\n'
    '        "Correlation Window",\n'
    '    ),\n'
    '    Workbook(\n'
    '        "spread",\n'
    '        "1TpE-sMW3mxdeXfWavdSfK2EIkz_WzbQbpc0AyjAJXPo",\n'
    '        "Spread Score - COIN",\n'
    '        "Spread Score",\n'
    '        0,\n'
    '        "Spread Z-Score Window",\n'
    '    ),\n'
    ')\n'
    '\n'
    '\n'
    'def credentials_path(explicit=None):\n'
    '    configured = explicit or os.getenv("GOOGLE_APPLICATION_CREDENTIALS")\n'
    '    if configured:\n'
    '        path = Path(configured).expanduser()\n'
    '        if not path.is_file():\n'
    '            raise ValueError(f"Credential file does not exist: {path}")\n'
    '        return path\n'
    '    for parent in (ROOT, *ROOT.parents):\n'
    '        path = parent / "google_credentials.json"\n'
    '        if path.is_file():\n'
    '            return path\n'
    '    raise ValueError(\n'
    '        "Set GOOGLE_APPLICATION_CREDENTIALS or --credentials-file to the existing Google credential file."\n'
    '    )\n'
    '\n'
    '\n'
    'def window_value(rows, key):\n'
    '    found = [r[1] for r in rows if r and r[0] == key and len(r) > 1]\n'
    '    if len(found) != 1:\n'
    '        raise ValueError(\n'
    '            f"Control Panel must contain exactly one {key!r} setting in columns A:B"\n'
    '        )\n'
    '    v = found[0]\n'
    '    if isinstance(v, bool):\n'
    '        raise ValueError(f"{key} must be an integer >= 2")\n'
    '    try:\n'
    '        n = float(v)\n'
    '        if not n.is_integer() or n < 2:\n'
    '            raise ValueError()\n'
    '    except (ValueError, TypeError, OverflowError):\n'
    '        raise ValueError(f"{key} must be an integer >= 2; got {v!r}") from None\n'
    '    return int(n)\n'
    '\n'
    '\n'
    'BASE_WORKBOOKS = WORKBOOKS\n'
    'MOMENTUM_WORKBOOKS = (\n'
    '    Workbook(\n'
    '        "spread_momentum",\n'
    '        "1PPVvOClJ3Ttk5We81pDIQyrYKtywuWuxhFITOulfXZA",\n'
    '        "Spread Momentum Score",\n'
    '        "Spread Momentum Score",\n'
    '        0,\n'
    '        "Momentum Z-Score Window",\n'
    '        "Imported Spread Data",\n'
    '    ),\n'
    '    Workbook(\n'
    '        "correlation_momentum",\n'
    '        "1ZoApVQIQmm0UUuG4A1s1CJ-b7ITbql5eWwZYvrsb8zc",\n'
    '        "Correlation Momentum Score",\n'
    '        "Correlation Momentum Score",\n'
    '        1400973068,\n'
    '        "Momentum Z-Score Window",\n'
    '        "Imported Correlation Data",\n'
    '    ),\n'
    ')\n'
    'COINCIDENT_WORKBOOKS = BASE_WORKBOOKS + MOMENTUM_WORKBOOKS\n'
    'LEADING_BASE_WORKBOOKS = (\n'
    '    Workbook(\n'
    '        "correlation",\n'
    '        "1Ma0pirU2pmMSFgZXKB4v5y1gw9W8vzBI8xt6tPxiz90",\n'
    '        "Correl Score - LEAD",\n'
    '        "Correlation Score",\n'
    '        1400973068,\n'
    '        "Correlation Window",\n'
    '        dataset="leading",\n'
    '    ),\n'
    '    Workbook(\n'
    '        "spread",\n'
    '        "1sOkwKFe0d42NC7phUkGGOrhhwhDoff78rlQCL-1GUt0",\n'
    '        "Spread Score - LEAD",\n'
    '        "Spread Score",\n'
    '        0,\n'
    '        "Spread Z-Score Window",\n'
    '        dataset="leading",\n'
    '    ),\n'
    ')\n'
    'LEADING_MOMENTUM_WORKBOOKS = (\n'
    '    Workbook(\n'
    '        "spread_momentum",\n'
    '        "18uNC1bykf_a-F9tHTpgCgPk2e_Bw3TPMsJ6BQuZiL4g",\n'
    '        "Spread Momentum Score - LEAD",\n'
    '        "Spread Momentum Score",\n'
    '        0,\n'
    '        "Momentum Z-Score Window",\n'
    '        "Imported Spread Data",\n'
    '        dataset="leading",\n'
    '    ),\n'
    '    Workbook(\n'
    '        "correlation_momentum",\n'
    '        "1b-TWGIzqmzfG76ssiMAwROdCQQh7fhXEh2u4NTPRCvA",\n'
    '        "Correlation Momentum Score - LEAD",\n'
    '        "Correlation Momentum Score",\n'
    '        1400973068,\n'
    '        "Momentum Z-Score Window",\n'
    '        "Imported Correlation Data",\n'
    '        dataset="leading",\n'
    '    ),\n'
    ')\n'
    'LEADING_WORKBOOKS = LEADING_BASE_WORKBOOKS + LEADING_MOMENTUM_WORKBOOKS\n'
    'BASE_WORKBOOKS += LEADING_BASE_WORKBOOKS\n'
    'MOMENTUM_WORKBOOKS += LEADING_MOMENTUM_WORKBOOKS\n'
    'WORKBOOKS = COINCIDENT_WORKBOOKS + LEADING_WORKBOOKS\n'
    'UPSTREAM = {(b.dataset, b.kind + "_momentum"): b for b in BASE_WORKBOOKS}\n'
    '\n'
    '\n'
    'def upstream_book(book):\n'
    '    return UPSTREAM[(book.dataset, book.kind)]\n'
    '\n'
    '\n'
    'def select_workbooks(dataset="all", kind=None, momentum_only=False):\n'
    '    if dataset not in ("all", "coincident", "leading"):\n'
    '        raise ValueError(f"Unknown dataset: {dataset}")\n'
    '    if kind is not None and kind not in {b.kind for b in WORKBOOKS}:\n'
    '        raise ValueError(f"Unknown calculation: {kind}")\n'
    '    return tuple(\n'
    '        b\n'
    '        for b in WORKBOOKS\n'
    '        if (dataset == "all" or b.dataset == dataset)\n'
    '        and (kind is None or b.kind == kind)\n'
    '        and (not momentum_only or b.kind.endswith("_momentum"))\n'
    '    )\n'
    '\n'
    '\n'
    'def momentum_lookback(rows):\n'
    '    # Reuse integer validation with a shifted lower bound; no hard-coded setting.\n'
    '    values = [r[1] for r in rows if r and r[0] == "Momentum Lookback" and len(r) > 1]\n'
    '    if len(values) != 1 or isinstance(values[0], bool):\n'
    '        raise ValueError("Momentum Lookback must occur once and be an integer >= 1")\n'
    '    try:\n'
    '        n = float(values[0])\n'
    '        if not n.is_integer() or n < 1:\n'
    '            raise ValueError()\n'
    '        return int(n)\n'
    '    except (ValueError, TypeError, OverflowError):\n'
    '        raise ValueError("Momentum Lookback must be an integer >= 1") from None\n'
)

# Embedded source: google_sheets.py
SOURCES['google_sheets.py'] = (
    '"""Batched Sheets REST I/O using the existing external Google credentials."""\n'
    '\n'
    'import gzip\n'
    'import json\n'
    'import logging\n'
    'import time\n'
    'import threading\n'
    'from contextlib import contextmanager\n'
    'from urllib.parse import quote\n'
    'from google.auth import load_credentials_from_file\n'
    'from google.auth.transport.requests import AuthorizedSession\n'
    'import requests\n'
    'from config import WORKBOOKS\n'
    '\n'
    'LOG = logging.getLogger(__name__)\n'
    '\n'
    '\n'
    '@contextmanager\n'
    'def request_progress(label, interval=15):\n'
    '    """Report waits without moving network I/O off the interruptible main thread."""\n'
    '    finished = threading.Event()\n'
    '    started = time.monotonic()\n'
    '\n'
    '    def report():\n'
    '        while not finished.wait(interval):\n'
    '            LOG.info(\n'
    '                "%s: still waiting for Google Sheets (%.0fs elapsed)",\n'
    '                label,\n'
    '                time.monotonic() - started,\n'
    '            )\n'
    '\n'
    '    worker = threading.Thread(target=report, name="sheets-progress", daemon=True)\n'
    '    worker.start()\n'
    '    try:\n'
    '        yield\n'
    '    finally:\n'
    '        finished.set()\n'
    '        worker.join()\n'
    '\n'
    '\n'
    'def a1(tab, cells=""):\n'
    '    return "\'" + tab.replace("\'", "\'\'") + "\'" + ("!" + cells if cells else "")\n'
    '\n'
    '\n'
    'def column_name(n):\n'
    '    result = ""\n'
    '    while n:\n'
    '        n, r = divmod(n - 1, 26)\n'
    '        result = chr(65 + r) + result\n'
    '    return result\n'
    '\n'
    '\n'
    'class Sheets:\n'
    '    def __init__(self, credential_file, write_ids=None):\n'
    '        self.write_ids = (\n'
    '            {b.spreadsheet_id for b in WORKBOOKS}\n'
    '            if write_ids is None\n'
    '            else set(write_ids)\n'
    '        )\n'
    '        creds, _ = load_credentials_from_file(\n'
    '            str(credential_file),\n'
    '            scopes=["https://www.googleapis.com/auth/spreadsheets"],\n'
    '        )\n'
    '        self.session = AuthorizedSession(creds)\n'
    '        self.writes_attempted = False\n'
    '\n'
    '    def request(self, method, book_id, suffix="", **kwargs):\n'
    '        if method != "GET" and book_id not in getattr(self, "write_ids", set()):\n'
    '            raise ValueError("Writes to upstream/source workbooks are prohibited")\n'
    '        if method != "GET":\n'
    '            self.writes_attempted = True\n'
    '        label = next(\n'
    '            (b.key for b in WORKBOOKS if b.spreadsheet_id == book_id), "source workbook"\n'
    '        )\n'
    '        url = "https://sheets.googleapis.com/v4/spreadsheets/" + book_id + suffix\n'
    '        for attempt in range(6):\n'
    '            try:\n'
    '                with request_progress(\n'
    '                    f\'{label} {method} {suffix or "metadata"} (attempt {attempt+1}/6)\'\n'
    '                ):\n'
    '                    response = self.session.request(\n'
    '                        method, url, timeout=(15, 180), **kwargs\n'
    '                    )\n'
    '                    if response.status_code not in (429, 500, 502, 503, 504):\n'
    '                        response.raise_for_status()\n'
    '                        return response.json()\n'
    '                    error = f"Google Sheets HTTP {response.status_code}"\n'
    '            except (requests.Timeout, requests.ConnectionError) as exc:\n'
    '                error = type(exc).__name__\n'
    '            if attempt == 5:\n'
    '                raise RuntimeError(\n'
    '                    f"{book_id}: {method} {suffix}: {error}; retries exhausted"\n'
    '                )\n'
    '            delay = min(2**attempt, 32)\n'
    '            LOG.warning("%s; retrying in %ss", error, delay)\n'
    '            time.sleep(delay)\n'
    '\n'
    '    def metadata(self, book_id):\n'
    '        return self.request(\n'
    '            "GET",\n'
    '            book_id,\n'
    '            params={\n'
    '                "fields": "spreadsheetId,properties(title,locale),sheets(properties,protectedRanges,merges)"\n'
    '            },\n'
    '        )\n'
    '\n'
    '    def values(self, book_id, ranges, formulas=False):\n'
    '        started = time.monotonic()\n'
    '        LOG.info(\n'
    '            "Downloading %s: %s",\n'
    '            "formulas" if formulas else "values",\n'
    '            ", ".join(ranges),\n'
    '        )\n'
    '        obj = self.request(\n'
    '            "GET",\n'
    '            book_id,\n'
    '            "/values:batchGet",\n'
    '            params={\n'
    '                "ranges": ranges,\n'
    '                "valueRenderOption": "FORMULA" if formulas else "UNFORMATTED_VALUE",\n'
    '                "dateTimeRenderOption": "SERIAL_NUMBER",\n'
    '            },\n'
    '        )\n'
    '        result = [r.get("values", []) for r in obj["valueRanges"]]\n'
    '        LOG.info(\n'
    '            "Download complete in %.1fs; rows per range: %s",\n'
    '            time.monotonic() - started,\n'
    '            [len(r) for r in result],\n'
    '        )\n'
    '        return result\n'
    '\n'
    '    def batch(self, book_id, requests_):\n'
    '        return self.request(\n'
    '            "POST", book_id, ":batchUpdate", json={"requests": requests_}\n'
    '        )\n'
    '\n'
    '    def write_values(self, book_id, ranges):\n'
    '        return self.request(\n'
    '            "POST",\n'
    '            book_id,\n'
    '            "/values:batchUpdate",\n'
    '            json={"valueInputOption": "RAW", "data": ranges},\n'
    '        )\n'
    '\n'
    '    def backup(self, path, payload):\n'
    '        path.parent.mkdir(parents=True, exist_ok=True)\n'
    '        with gzip.open(path, "wt", encoding="utf-8") as handle:\n'
    '            json.dump(payload, handle, allow_nan=False)\n'
    '\n'
    '    def publish(self, book, properties, matrix, backup_path, snapshot):\n'
    '        self.backup(backup_path, snapshot)\n'
    '        grid = properties["gridProperties"]\n'
    '        last_col = column_name(grid["columnCount"])\n'
    '        clear = a1(book.output_tab, f\'B3:{last_col}{grid["rowCount"]}\')\n'
    '        self.request(\n'
    '            "POST", book.spreadsheet_id, "/values:batchClear", json={"ranges": [clear]}\n'
    '        )\n'
    '        written = 0\n'
    '        # At most ~50k cells per request, independent of total history length.\n'
    '        block_rows = max(1, 50000 // max(1, len(matrix[0])))\n'
    '        for start in range(0, len(matrix), block_rows):\n'
    '            block = matrix[start : start + block_rows]\n'
    '            last = column_name(len(block[0]) + 1)\n'
    '            target = a1(book.output_tab, f"B{start+3}:{last}{start+2+len(block)}")\n'
    '            self.write_values(book.spreadsheet_id, [{"range": target, "values": block}])\n'
    '            written += sum(map(len, block))\n'
    '            LOG.info(\n'
    '                "%s: wrote rows %s:%s (%s/%s matrix cells)",\n'
    '                book.key,\n'
    '                start + 3,\n'
    '                start + 2 + len(block),\n'
    '                written,\n'
    '                len(matrix) * len(matrix[0]),\n'
    '            )\n'
    '        return written\n'
)

# Embedded source: pipeline.py
SOURCES['pipeline.py'] = (
    '"""Read, calculate, compare, and verify complete output histories."""\n'
    '\n'
    'import hashlib\n'
    'import json\n'
    'import logging\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'from config import window_value\n'
    'from validation import (\n'
    '    source_frame,\n'
    '    pair_definitions,\n'
    '    output_dates,\n'
    '    numeric,\n'
    '    ValidationError,\n'
    ')\n'
    'from correlation import rolling_correlation\n'
    'from spread import spread_zscore\n'
    'from google_sheets import a1\n'
    '\n'
    'LOG = logging.getLogger(__name__)\n'
    '\n'
    '\n'
    'def digest(value):\n'
    '    return hashlib.sha256(\n'
    '        json.dumps(value, sort_keys=True, allow_nan=False).encode()\n'
    '    ).hexdigest()\n'
    '\n'
    '\n'
    'def as_matrix(rows, shape):\n'
    '    result = np.full(shape, np.nan)\n'
    '    for i, row in enumerate(rows[: shape[0]]):\n'
    '        for j, value in enumerate(row[: shape[1]]):\n'
    '            result[i, j] = numeric(value)\n'
    '    return result\n'
    '\n'
    '\n'
    'def comparison(old, new):\n'
    '    common = np.isfinite(old) & np.isfinite(new)\n'
    '    bad = common & ~np.isclose(old, new, rtol=1e-9, atol=1e-10)\n'
    '    blank = np.isfinite(old) ^ np.isfinite(new)\n'
    '    diff = np.abs(old[common] - new[common])\n'
    '    last_old = max(np.flatnonzero(np.isfinite(old).any(axis=1)), default=-1)\n'
    '    new_tail = (\n'
    '        (~np.isfinite(old))\n'
    '        & np.isfinite(new)\n'
    '        & (np.arange(len(new))[:, None] > last_old)\n'
    '    )\n'
    '    return {\n'
    '        "existing_numeric": int(np.isfinite(old).sum()),\n'
    '        "python_numeric": int(np.isfinite(new).sum()),\n'
    '        "compared_numeric": int(common.sum()),\n'
    '        "numeric_mismatches": int(bad.sum()),\n'
    '        "blank_mismatches": int(blank.sum()),\n'
    '        "added_numeric": int((~np.isfinite(old) & np.isfinite(new)).sum()),\n'
    '        "removed_numeric": int((np.isfinite(old) & ~np.isfinite(new)).sum()),\n'
    '        "new_tail_numeric": int(new_tail.sum()),\n'
    '        "historical_blank_mismatches": int((blank & ~new_tail).sum()),\n'
    '        "max_absolute_difference": float(diff.max()) if len(diff) else None,\n'
    '        "mismatch_coordinates": [\n'
    '            [int(i) + 3, int(j) + 2] for i, j in np.argwhere(bad | blank)[:30]\n'
    '        ],\n'
    '    }\n'
    '\n'
    '\n'
    'def calculate(book, source, output, controls):\n'
    '    frame, source_issues = source_frame(source)\n'
    '    window = window_value(controls, book.control_key)\n'
    '    pairs, pair_issues = pair_definitions(output, frame)\n'
    '    dates = output_dates(output, frame.index)\n'
    '    if not len(dates):\n'
    '        raise ValidationError("No writable output dates")\n'
    '    result = np.full((len(frame), len(pairs)), np.nan)\n'
    '    fn = rolling_correlation if book.kind == "correlation" else spread_zscore\n'
    '    for j, (x, y, error) in enumerate(pairs):\n'
    '        if error:\n'
    '            continue\n'
    '        try:\n'
    '            result[:, j] = fn(frame[x].to_numpy(), frame[y].to_numpy(), window)\n'
    '        except Exception as exc:\n'
    '            raise ValidationError(\n'
    '                f"{book.kind} column {j+2}, pair {x!r}, {y!r}: {exc}"\n'
    '            ) from exc\n'
    '    aligned = pd.DataFrame(result, index=frame.index).reindex(dates).to_numpy()\n'
    '    old = as_matrix([row[1:] for row in output[2:]], aligned.shape)\n'
    '    finite = np.isfinite(aligned)\n'
    '    active = dates[finite.any(axis=1)]\n'
    '    values = np.where(finite, aligned.astype(object), "").tolist()\n'
    '    issues = source_issues + pair_issues\n'
    '    for issue in issues:\n'
    '        LOG.warning("%s: %s", book.kind, issue)\n'
    '    report = {\n'
    '        "kind": book.kind,\n'
    '        "dataset": book.dataset,\n'
    '        "key": book.key,\n'
    '        "spreadsheet_id": book.spreadsheet_id,\n'
    '        "output_tab": book.output_tab,\n'
    '        "window": window,\n'
    '        "window_unit": "valid_paired_observations",\n'
    '        "history_cutoff": "strictly_before_target_date",\n'
    '        "source_dates": len(frame),\n'
    '        "indicators": len(frame.columns),\n'
    '        "pairs": len(pairs),\n'
    '        "output_dates": len(dates),\n'
    '        "source_first_date": str(frame.index.min().date()),\n'
    '        "source_latest_date": str(frame.index.max().date()),\n'
    '        "first_calculated_date": str(active.min().date()) if len(active) else None,\n'
    '        "latest_calculated_date": str(active.max().date()) if len(active) else None,\n'
    '        "numeric_cells": int(finite.sum()),\n'
    '        "blank_observations": int((~finite).sum()),\n'
    '        "invalid_indicator_mappings": len(pair_issues),\n'
    '        "issues": issues,\n'
    '        "cells_written": 0,\n'
    '        "comparison": comparison(old, aligned),\n'
    '    }\n'
    '    return values, report\n'
    '\n'
    '\n'
    'def load(sheets, book):\n'
    '    meta = sheets.metadata(book.spreadsheet_id)\n'
    '    if meta["properties"]["title"] != book.title:\n'
    '        LOG.info(\n'
    '            "Workbook display title is %r; using the configured workbook ID",\n'
    '            meta["properties"]["title"],\n'
    '        )\n'
    '    tabs = {s["properties"]["title"]: s["properties"] for s in meta["sheets"]}\n'
    '    for name in (book.source_tab, book.output_tab, book.control_tab):\n'
    '        if name not in tabs:\n'
    '            raise ValidationError(f"{book.title}: missing required tab {name!r}")\n'
    '    if tabs[book.output_tab]["sheetId"] != book.output_id:\n'
    '        raise ValidationError("Output sheet identity changed")\n'
    '    from source_freshness import wait_for_macro\n'
    '\n'
    '    wait_for_macro(sheets, book)\n'
    '    source, output, controls = sheets.values(\n'
    '        book.spreadsheet_id,\n'
    '        [a1(book.source_tab), a1(book.output_tab), a1(book.control_tab, "A:B")],\n'
    '    )\n'
    '    # Input tabs are read-only; reserve capacity only in the output tab.\n'
    '    output_props = tabs[book.output_tab]\n'
    '    source_capacity = tabs[book.source_tab]["gridProperties"]["rowCount"]\n'
    '    requests = []\n'
    '    output_capacity = max(len(source) + 2, source_capacity + 2)\n'
    '    if output_capacity > output_props["gridProperties"]["rowCount"]:\n'
    '        requests.append(\n'
    '            {\n'
    '                "updateSheetProperties": {\n'
    '                    "properties": {\n'
    '                        "sheetId": output_props["sheetId"],\n'
    '                        "gridProperties": {"rowCount": output_capacity},\n'
    '                    },\n'
    '                    "fields": "gridProperties.rowCount",\n'
    '                }\n'
    '            }\n'
    '        )\n'
    '        output_props["gridProperties"]["rowCount"] = output_capacity\n'
    '    if requests:\n'
    '        sheets.batch(book.spreadsheet_id, requests)\n'
    '        output = sheets.values(book.spreadsheet_id, [a1(book.output_tab)])[0]\n'
    '    values, report = calculate(book, source, output, controls)\n'
    '    return {\n'
    '        "book": book,\n'
    '        "meta": meta,\n'
    '        "properties": tabs[book.output_tab],\n'
    '        "source": source,\n'
    '        "output": output,\n'
    '        "controls": controls,\n'
    '        "values": values,\n'
    '        "report": report,\n'
    '    }\n'
    '\n'
    '\n'
    'def validate_before_publish(sheets, state):\n'
    '    b = state["book"]\n'
    '    if not b.kind.endswith("_momentum"):\n'
    '        from source_freshness import wait_for_macro\n'
    '\n'
    '        wait_for_macro(sheets, b)\n'
    '    # Detect concurrent source, header/date or control edits before replacing anything.\n'
    '    source, structure, dates, controls = sheets.values(\n'
    '        b.spreadsheet_id,\n'
    '        [\n'
    '            a1(b.source_tab),\n'
    '            a1(b.output_tab, "1:2"),\n'
    '            a1(b.output_tab, "A:A"),\n'
    '            a1(b.control_tab, "A:B"),\n'
    '        ],\n'
    '    )\n'
    '    expected_dates = [[r[0]] if r and r[0] != "" else [] for r in state["output"]]\n'
    '    while expected_dates and not expected_dates[-1]:\n'
    '        expected_dates.pop()\n'
    '    expected_source = state["source"]\n'
    '    if (\n'
    '        source != expected_source\n'
    '        or structure != state["output"][:2]\n'
    '        or dates != expected_dates\n'
    '        or controls != state["controls"]\n'
    '    ):\n'
    '        raise ValidationError(\n'
    '            f"{b.kind}: source, dates, pair definitions or controls changed during calculation; rerun"\n'
    '        )\n'
    '    formulas = sheets.values(b.spreadsheet_id, [a1(b.output_tab)], formulas=True)[0]\n'
    '    has_formulas = any(\n'
    '        isinstance(v, str) and v.startswith("=") for r in formulas[2:] for v in r[1:]\n'
    '    )\n'
    '    if has_formulas:\n'
    '        c = state["report"]["comparison"]\n'
    '        # Clearing scores on dates without a current pair is an expected correction.\n'
    '        if (\n'
    '            c["compared_numeric"] < min(20, state["report"]["numeric_cells"])\n'
    '            or not c["compared_numeric"]\n'
    '            or c["numeric_mismatches"]\n'
    '            or c["historical_blank_mismatches"] > c["removed_numeric"]\n'
    '        ):\n'
    '            raise ValidationError(\n'
    '                f"{b.kind}: existing formula comparison failed; formulas preserved. See audit report: {c}"\n'
    '            )\n'
    '    return formulas\n'
    '\n'
    '\n'
    'def verify(sheets, state):\n'
    '    b = state["book"]\n'
    '    rows = sheets.values(b.spreadsheet_id, [a1(b.output_tab)], formulas=True)[0]\n'
    '    expected = state["values"]\n'
    '    # Formula-rendered values expose any surviving calculated formula.\n'
    '    old_structure = state["formulas"]\n'
    '    header = lambda data: [data[i] if i < len(data) else [] for i in range(2)]\n'
    '    if header(rows) != header(old_structure):\n'
    '        raise ValidationError("Readback: pair/header rows changed")\n'
    '    n = max(len(rows), len(old_structure), len(expected) + 2)\n'
    '    for i in range(n):\n'
    '        actual = rows[i] if i < len(rows) else []\n'
    '        prior = old_structure[i] if i < len(old_structure) else []\n'
    '        if (actual[0] if actual else "") != (prior[0] if prior else ""):\n'
    '            raise ValidationError(f"Readback: column A changed at row {i+1}")\n'
    '        if i < 2:\n'
    '            continue\n'
    '        want = expected[i - 2] if i - 2 < len(expected) else []\n'
    '        for j in range(1, max(len(actual), len(want) + 1)):\n'
    '            a = actual[j] if j < len(actual) else ""\n'
    '            e = want[j - 1] if j <= len(want) else ""\n'
    '            if isinstance(a, str) and a.startswith("="):\n'
    '                raise ValidationError(\n'
    '                    f"Readback: formula remains at row {i+1}, column {j+1}"\n'
    '                )\n'
    '            if e == "":\n'
    '                if a != "":\n'
    '                    raise ValidationError(\n'
    '                        f"Readback: stale/nonblank cell at row {i+1}, column {j+1}"\n'
    '                    )\n'
    '            elif not np.isfinite(numeric(a)) or not np.isclose(\n'
    '                a, e, rtol=1e-12, atol=1e-12\n'
    '            ):\n'
    '                raise ValidationError(\n'
    '                    f"Readback: numeric mismatch row {i+1}, column {j+1}"\n'
    '                )\n'
    '    if b.kind.endswith("_momentum"):\n'
    '        effective = sheets.values(\n'
    '            b.spreadsheet_id, [a1(b.output_tab, "1:2"), a1(b.output_tab, "A:A")]\n'
    '        )\n'
    '        expected_dates = [[r[0]] if r and r[0] != "" else [] for r in state["output"]]\n'
    '        while expected_dates and not expected_dates[-1]:\n'
    '            expected_dates.pop()\n'
    '        if effective[0] != state["output"][:2] or effective[1] != expected_dates:\n'
    '            raise ValidationError("Readback: effective imported headers/dates changed")\n'
    '    return True\n'
)

# Embedded source: momentum_pipeline.py
SOURCES['momentum_pipeline.py'] = (
    '"""Momentum inputs are IMPORTRANGE results; upstream books are read-only."""\n'
    '\n'
    'from collections import Counter\n'
    'import logging\n'
    'import time\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'from config import upstream_book, window_value, momentum_lookback\n'
    'from google_sheets import a1, column_name\n'
    'from validation import (\n'
    '    source_frame,\n'
    '    pair_definitions,\n'
    '    output_dates,\n'
    '    parse_date,\n'
    '    numeric,\n'
    '    ValidationError,\n'
    ')\n'
    'from momentum import momentum_zscore\n'
    'from spread import spread_series\n'
    'from correlation import rolling_correlation\n'
    'from pipeline import as_matrix, comparison\n'
    '\n'
    'LOG = logging.getLogger(__name__)\n'
    '\n'
    '\n'
    'def canonical(rows):\n'
    '    rows = [list(r) for r in rows]\n'
    '    for r in rows:\n'
    '        while r and r[-1] == "":\n'
    '            r.pop()\n'
    '    while rows and not rows[-1]:\n'
    '        rows.pop()\n'
    '    return rows\n'
    '\n'
    '\n'
    'def fisher_frame(rows):\n'
    '    if len(rows) < 3 or len(rows[1]) < 1 or rows[1][0] != "Date":\n'
    '        raise ValidationError(\n'
    '            "Imported Fisher history must have Date in A2 and resolved pair headers"\n'
    '        )\n'
    '    width = max(len(rows[0]), len(rows[1]))\n'
    '    keys = [\n'
    '        (rows[0][j] if j < len(rows[0]) else "", rows[1][j] if j < len(rows[1]) else "")\n'
    '        for j in range(1, width)\n'
    '    ]\n'
    '    counts = Counter(keys)\n'
    '    issues = []\n'
    '    columns = {}\n'
    '    dates = []\n'
    '    body = []\n'
    '    for i, row in enumerate(rows[2:], 3):\n'
    '        if not row or row[0] == "":\n'
    '            if any(v != "" for v in row[1:]):\n'
    '                raise ValidationError(\n'
    '                    f"Imported Fisher row {i} has values without date"\n'
    '                )\n'
    '            continue\n'
    '        dates.append(parse_date(row[0]))\n'
    '        body.append(row)\n'
    '    if not dates or len(set(dates)) != len(dates):\n'
    '        raise ValidationError("Empty or duplicate imported Fisher dates")\n'
    '    for j, key in enumerate(keys, 1):\n'
    '        if (\n'
    '            not all(isinstance(v, str) and v and not v.startswith("#") for v in key)\n'
    '            or counts[key] != 1\n'
    '        ):\n'
    '            issues.append(\n'
    '                f"Imported Fisher column {j+1}, pair {key}: missing or duplicate definition"\n'
    '            )\n'
    '            continue\n'
    '        columns[key] = [numeric(r[j]) if j < len(r) else np.nan for r in body]\n'
    '        for d, r, v in zip(dates, body, columns[key]):\n'
    '            if j < len(r) and r[j] != "" and not np.isfinite(v):\n'
    '                issues.append(\n'
    '                    f"Imported Fisher {d.date()}, pair {key}: nonnumeric observation {r[j]!r}"\n'
    '                )\n'
    '    frame = pd.DataFrame(columns, index=pd.DatetimeIndex(dates)).sort_index()\n'
    '    if not frame.notna().any().any():\n'
    '        raise ValidationError("No finite Fisher values; import may be unavailable")\n'
    '    return frame, issues\n'
    '\n'
    '\n'
    'def correlation_events(book, frame, pair_source, pair_controls):\n'
    '    """Verify Fisher lineage and identify new strictly-prior paired observations.\n'
    '\n'
    '    Dates, never changes in values, define updates. Reindex on the union so a\n'
    '    missing display row cannot shift the cutoff or create a stale observation.\n'
    '    """\n'
    '    if pair_source is None or pair_controls is None:\n'
    '        raise ValidationError(\n'
    '            "Correlation momentum requires dated paired source data and upstream controls"\n'
    '        )\n'
    '    raw, issues = source_frame(pair_source)\n'
    '    if issues:\n'
    '        raise ValidationError("Invalid paired source data: " + str(issues[:5]))\n'
    '    window = window_value(pair_controls, upstream_book(book).control_key)\n'
    '    union = raw.index.union(frame.index).sort_values()\n'
    '    expanded = raw.reindex(union)\n'
    '    masks = {}\n'
    '    latest = {}\n'
    '    for x, y in frame.columns:\n'
    '        if x not in raw or y not in raw:\n'
    '            raise ValidationError(f"Missing paired source indicator: {x!r}, {y!r}")\n'
    '        paired = np.isfinite(raw[x]) & np.isfinite(raw[y])\n'
    '        paired_dates = raw.index[paired]\n'
    '        counts = paired_dates.searchsorted(frame.index, side="right")\n'
    '        masks[(x, y)] = counts > np.r_[0, counts[:-1]]\n'
    '        latest[(x, y)] = [paired_dates[n - 1] if n else pd.NaT for n in counts]\n'
    '        expected = (\n'
    '            pd.Series(\n'
    '                rolling_correlation(expanded[x], expanded[y], window), index=union\n'
    '            )\n'
    '            .reindex(frame.index)\n'
    '            .to_numpy()\n'
    '        )\n'
    '        actual = frame[(x, y)].to_numpy()\n'
    '        if not np.all(\n'
    '            np.isclose(actual, expected, rtol=1e-9, atol=1e-10, equal_nan=True)\n'
    '        ):\n'
    '            raise ValidationError(\n'
    '                f"Upstream Fisher history does not match dated paired inputs and controls for {x!r}, {y!r}; rerun correlation before momentum"\n'
    '            )\n'
    '    return masks, latest\n'
    '\n'
    '\n'
    'def calculate(book, source, output, controls, pair_source=None, pair_controls=None):\n'
    '    m = momentum_lookback(controls)\n'
    '    w = window_value(controls, book.control_key)\n'
    '    if book.kind == "spread_momentum":\n'
    '        frame, issues = source_frame(source)\n'
    '        pairs, pair_issues = pair_definitions(output, frame)\n'
    '        series = [\n'
    '            None if err else spread_series(frame[x], frame[y]) for x, y, err in pairs\n'
    '        ]\n'
    '        indicators = len(frame.columns)\n'
    '    else:\n'
    '        frame, issues = fisher_frame(source)\n'
    '        if issues:\n'
    '            raise ValidationError("Invalid Fisher input: " + str(issues[:5]))\n'
    '        masks, latest = correlation_events(book, frame, pair_source, pair_controls)\n'
    '        width = max(map(len, output[:2]), default=0)\n'
    '        if width < 2:\n'
    '            raise ValidationError("Missing output pair definitions")\n'
    '        pairs = []\n'
    '        series = []\n'
    '        pair_issues = []\n'
    '        for j in range(1, width):\n'
    '            x = output[0][j] if j < len(output[0]) else ""\n'
    '            y = output[1][j] if j < len(output[1]) else ""\n'
    '            err = (\n'
    '                None\n'
    '                if (x, y) in frame.columns\n'
    '                else "missing or duplicate imported pair"\n'
    '            )\n'
    '            pairs.append((x, y, err))\n'
    '            series.append(\n'
    '                None\n'
    '                if err\n'
    '                else np.where(masks[(x, y)], frame[(x, y)].to_numpy(), np.nan)\n'
    '            )\n'
    '            if err:\n'
    '                pair_issues.append(f"Column {j+1}, pair {x!r}, {y!r}: {err}")\n'
    '        indicators = len({v for x, y, _ in pairs for v in (x, y) if v})\n'
    '    dates = output_dates(output, frame.index)\n'
    '    result = np.full((len(frame), len(pairs)), np.nan)\n'
    '    for j, values in enumerate(series):\n'
    '        if values is None:\n'
    '            continue\n'
    '        try:\n'
    '            result[:, j] = momentum_zscore(values, m, w)\n'
    '        except Exception as e:\n'
    '            raise ValidationError(f"{book.kind}, pair {pairs[j][:2]}: {e}") from e\n'
    '    aligned = pd.DataFrame(result, index=frame.index).reindex(dates).to_numpy()\n'
    '    finite = np.isfinite(aligned)\n'
    '    active = dates[finite.any(axis=1)]\n'
    '    issues += pair_issues\n'
    '    for issue in issues:\n'
    '        LOG.warning("%s: %s", book.kind, issue)\n'
    '    report = {\n'
    '        "kind": book.kind,\n'
    '        "dataset": book.dataset,\n'
    '        "key": book.key,\n'
    '        "spreadsheet_id": book.spreadsheet_id,\n'
    '        "output_tab": book.output_tab,\n'
    '        "window": w,\n'
    '        "momentum_lookback": m,\n'
    '        "window_unit": "prior_changes_between_paired_updates",\n'
    '        "history_cutoff": "strictly_before_target_date",\n'
    '        "source_dates": len(frame),\n'
    '        "indicators": indicators,\n'
    '        "pairs": len(pairs),\n'
    '        "output_dates": len(dates),\n'
    '        "first_calculated_date": str(active.min().date()) if len(active) else None,\n'
    '        "latest_calculated_date": str(active.max().date()) if len(active) else None,\n'
    '        "numeric_cells": int(finite.sum()),\n'
    '        "blank_observations": int((~finite).sum()),\n'
    '        "invalid_indicator_mappings": len(pair_issues),\n'
    '        "issues": issues,\n'
    '        "cells_written": 0,\n'
    '        "comparison": comparison(\n'
    '            as_matrix([r[1:] for r in output[2:]], aligned.shape), aligned\n'
    '        ),\n'
    '    }\n'
    '    report["momentum_clock"] = (\n'
    '        "paired_input_updates"\n'
    '        if book.kind == "correlation_momentum"\n'
    '        else "valid_current_paired_observations"\n'
    '    )\n'
    '    if book.kind == "correlation_momentum":\n'
    '        report["upstream_fisher_lineage_verified"] = True\n'
    '        report["method_version"] = "paired_updates_v1"\n'
    '        report["upstream_correlation_window"] = window_value(\n'
    '            pair_controls, upstream_book(book).control_key\n'
    '        )\n'
    '        report["suppressed_repeated_fisher_cells"] = int(\n'
    '            sum(np.sum(np.isfinite(frame[key]) & ~mask) for key, mask in masks.items())\n'
    '        )\n'
    '        report["pair_status"] = []\n'
    '        for j, (x, y, err) in enumerate(pairs):\n'
    '            if err:\n'
    '                continue\n'
    '            event_dates = frame.index[np.isfinite(series[j])]\n'
    '            score_dates = dates[np.isfinite(aligned[:, j])]\n'
    '            last_input = latest[(x, y)][-1]\n'
    '            lag_date = event_dates[-m - 1] if len(event_dates) > m else pd.NaT\n'
    '            report["pair_status"].append(\n'
    '                {\n'
    '                    "x": x,\n'
    '                    "y": y,\n'
    '                    "valid_update_count": len(event_dates),\n'
    '                    "last_paired_input_date": (\n'
    '                        str(last_input.date()) if pd.notna(last_input) else None\n'
    '                    ),\n'
    '                    "last_update_date": (\n'
    '                        str(event_dates[-1].date()) if len(event_dates) else None\n'
    '                    ),\n'
    '                    "latest_lag_date": (\n'
    '                        str(lag_date.date()) if pd.notna(lag_date) else None\n'
    '                    ),\n'
    '                    "last_momentum_date": (\n'
    '                        str(score_dates[-1].date()) if len(score_dates) else None\n'
    '                    ),\n'
    '                    "status": (\n'
    '                        "No finite momentum history"\n'
    '                        if not len(score_dates)\n'
    '                        else (\n'
    '                            "OK"\n'
    '                            if score_dates[-1] == dates[-1]\n'
    '                            else "No current momentum score"\n'
    '                        )\n'
    '                    ),\n'
    '                }\n'
    '            )\n'
    '    return np.where(finite, aligned.astype(object), "").tolist(), report\n'
    '\n'
    '\n'
    'def desired_formulas(book, source_width, pair_width):\n'
    '    up = upstream_book(book)\n'
    '    bid = up.spreadsheet_id\n'
    '    last = column_name(pair_width)\n'
    '    if book.kind == "spread_momentum":\n'
    '        return [\n'
    '            (\n'
    '                book.source_tab,\n'
    '                "A1",\n'
    '                f\'=IMPORTRANGE("{bid}","\\\'Final scores data\\\'!A:{column_name(source_width)}")\',\n'
    '            ),\n'
    '            (\n'
    '                "Imported Pair Definitions",\n'
    '                "A1",\n'
    '                f\'=IMPORTRANGE("{bid}","\\\'Spread Score\\\'!A1:{last}2")\',\n'
    '            ),\n'
    '            (book.output_tab, "A1", f"=ARRAYFORMULA(\'{book.source_tab}\'!A1:A)"),\n'
    '            (\n'
    '                book.output_tab,\n'
    '                "B1",\n'
    '                f"=ARRAYFORMULA(\'Imported Pair Definitions\'!B1:{last}2)",\n'
    '            ),\n'
    '        ]\n'
    '    formulas = [\n'
    '        (book.source_tab, "A1", f\'=IMPORTRANGE("{bid}","\\\'Correlation Score\\\'!A:A")\'),\n'
    '        (book.output_tab, "A1", "=\'Control Panel\'!B3"),\n'
    '        (book.output_tab, "A2", f"=ARRAYFORMULA(\'{book.source_tab}\'!A2:A)"),\n'
    '        (book.output_tab, "B1", f"=ARRAYFORMULA(\'{book.source_tab}\'!B1:{last}2)"),\n'
    '    ]\n'
    '    for start in range(2, pair_width + 1, 64):\n'
    '        end = min(start + 63, pair_width)\n'
    '        cell = column_name(start) + "1"\n'
    '        segment = f"\'Correlation Score\'!{cell}:{column_name(end)}"\n'
    '        formulas.append(\n'
    '            (\n'
    '                book.source_tab,\n'
    '                cell,\n'
    '                f\'=IMPORTRANGE("{bid}","{segment}"&MAX(FILTER(ROW($A:$A),$A:$A<>"")))\',\n'
    '            )\n'
    '        )\n'
    '    return formulas\n'
    '\n'
    '\n'
    'def prepare(sheets, book):\n'
    '    """Refresh only new-workbook import geometry. Never source values or controls."""\n'
    '    up = upstream_book(book)\n'
    '    tab = up.source_tab if book.kind == "spread_momentum" else up.output_tab\n'
    '    upstream, pairs = sheets.values(\n'
    '        up.spreadsheet_id, [a1(tab), a1(up.output_tab, "1:2")]\n'
    '    )\n'
    '    if not upstream or len(pairs) < 2:\n'
    '        raise ValidationError("Upstream source or pair definitions unavailable")\n'
    '    sw = max(map(len, upstream[:2]))\n'
    '    pw = max(map(len, pairs))\n'
    '    meta = sheets.metadata(book.spreadsheet_id)\n'
    '    if meta["properties"]["title"] != book.title:\n'
    '        LOG.info(\n'
    '            "Workbook display title is %r; using the configured workbook ID",\n'
    '            meta["properties"]["title"],\n'
    '        )\n'
    '    tabs = {x["properties"]["title"]: x["properties"] for x in meta["sheets"]}\n'
    '    if tabs[book.output_tab]["sheetId"] != book.output_id:\n'
    '        raise ValidationError("Momentum output identity changed")\n'
    '    formulas = desired_formulas(book, sw, pw)\n'
    '    current = sheets.values(\n'
    '        book.spreadsheet_id, [a1(t, c) for t, c, _ in formulas], formulas=True\n'
    '    )\n'
    '    changes = [\n'
    '        {"range": a1(t, c), "values": [[f]]}\n'
    '        for (t, c, f), v in zip(formulas, current)\n'
    '        if v != [[f]]\n'
    '    ]\n'
    '    existing_anchors = sheets.values(\n'
    '        book.spreadsheet_id, [a1(book.source_tab, "1:1")], formulas=True\n'
    '    )[0]\n'
    '    desired_cells = {c for t, c, _ in formulas if t == book.source_tab}\n'
    '    obsolete = any(\n'
    '        isinstance(v, str)\n'
    '        and v.startswith("=")\n'
    '        and column_name(j + 1) + "1" not in desired_cells\n'
    '        for j, v in enumerate(existing_anchors[0] if existing_anchors else [])\n'
    '    )\n'
    '    upstream_meta = sheets.metadata(up.spreadsheet_id)\n'
    '    capacity = next(\n'
    '        x["properties"]["gridProperties"]["rowCount"]\n'
    '        for x in upstream_meta["sheets"]\n'
    '        if x["properties"]["title"] == tab\n'
    '    )\n'
    '    req = []\n'
    '    for name, cols, rows in [\n'
    '        (book.source_tab, sw, capacity),\n'
    '        (book.output_tab, pw, capacity + 2),\n'
    '    ] + (\n'
    '        [("Imported Pair Definitions", pw, 2)] if book.kind == "spread_momentum" else []\n'
    '    ):\n'
    '        p = tabs[name]\n'
    '        grid = p["gridProperties"]\n'
    '        target = {\n'
    '            "rowCount": max(rows, grid["rowCount"]),\n'
    '            "columnCount": max(cols, grid["columnCount"]),\n'
    '        }\n'
    '        if any(target[k] != grid[k] for k in target):\n'
    '            req.append(\n'
    '                {\n'
    '                    "updateSheetProperties": {\n'
    '                        "properties": {\n'
    '                            "sheetId": p["sheetId"],\n'
    '                            "gridProperties": target,\n'
    '                        },\n'
    '                        "fields": "gridProperties.rowCount,gridProperties.columnCount",\n'
    '                    }\n'
    '                }\n'
    '            )\n'
    '    if req:\n'
    '        sheets.batch(book.spreadsheet_id, req)\n'
    '    if changes or obsolete:\n'
    '        # Clear only the new imported source when its block layout changes, including obsolete blocks.\n'
    '        source_changed = any(\n'
    '            v["range"].startswith(a1(book.source_tab) + "!") for v in changes\n'
    '        )\n'
    '        if source_changed or obsolete:\n'
    '            sheets.request(\n'
    '                "POST",\n'
    '                book.spreadsheet_id,\n'
    '                "/values:batchClear",\n'
    '                json={"ranges": [a1(book.source_tab)]},\n'
    '            )\n'
    '            changes = [{"range": a1(t, c), "values": [[f]]} for t, c, f in formulas]\n'
    '        sheets.request(\n'
    '            "POST",\n'
    '            book.spreadsheet_id,\n'
    '            "/values:batchUpdate",\n'
    '            json={"valueInputOption": "USER_ENTERED", "data": changes},\n'
    '        )\n'
    '    return upstream, pairs\n'
    '\n'
    '\n'
    'def assert_fresh(sheets, book, source, output, pair_source=None, pair_controls=None):\n'
    '    up = upstream_book(book)\n'
    '    tab = up.source_tab if book.kind == "spread_momentum" else up.output_tab\n'
    '    upstream, pairs = sheets.values(\n'
    '        up.spreadsheet_id, [a1(tab), a1(up.output_tab, "1:2")]\n'
    '    )\n'
    '    if canonical(source) != canonical(upstream):\n'
    '        raise ValidationError(\n'
    '            f"{book.kind}: IMPORTRANGE is stale/loading or differs from upstream; existing scores preserved. Wait for imports and rerun."\n'
    '        )\n'
    '    if [r[1:] for r in output[:2]] != [r[1:] for r in pairs]:\n'
    '        raise ValidationError("Imported pair order differs from source")\n'
    '    if pair_source is not None:\n'
    '        current, controls = sheets.values(\n'
    '            up.spreadsheet_id, [a1(up.source_tab), a1(up.control_tab, "A:B")]\n'
    '        )\n'
    '        if canonical(current) != canonical(pair_source) or canonical(\n'
    '            controls\n'
    '        ) != canonical(pair_controls):\n'
    '            raise ValidationError(\n'
    '                "Paired source data or upstream controls changed during calculation; rerun"\n'
    '            )\n'
    '        from source_freshness import wait_for_macro\n'
    '\n'
    '        macro = wait_for_macro(sheets, up)\n'
    '        if canonical(macro) != canonical(pair_source):\n'
    '            raise ValidationError(\n'
    '                "Paired source no longer matches the macro summary; rerun correlation"\n'
    '            )\n'
    '\n'
    '\n'
    'def load(sheets, book):\n'
    '    prepare(sheets, book)\n'
    '    for attempt in range(6):\n'
    '        source, output, controls = sheets.values(\n'
    '            book.spreadsheet_id,\n'
    '            [a1(book.source_tab), a1(book.output_tab), a1(book.control_tab, "A:B")],\n'
    '        )\n'
    '        try:\n'
    '            assert_fresh(sheets, book, source, output)\n'
    '            break\n'
    '        except ValidationError:\n'
    '            if attempt == 5:\n'
    '                raise\n'
    '            LOG.info("%s: waiting 10 seconds for IMPORTRANGE freshness", book.kind)\n'
    '            time.sleep(10)\n'
    '    meta = sheets.metadata(book.spreadsheet_id)\n'
    '    props = next(\n'
    '        x["properties"]\n'
    '        for x in meta["sheets"]\n'
    '        if x["properties"]["title"] == book.output_tab\n'
    '    )\n'
    '    pair_source = pair_controls = None\n'
    '    if book.kind == "correlation_momentum":\n'
    '        up = upstream_book(book)\n'
    '        from source_freshness import wait_for_macro\n'
    '\n'
    '        wait_for_macro(sheets, up)\n'
    '        pair_source, pair_controls = sheets.values(\n'
    '            up.spreadsheet_id, [a1(up.source_tab), a1(up.control_tab, "A:B")]\n'
    '        )\n'
    '    values, report = calculate(\n'
    '        book, source, output, controls, pair_source, pair_controls\n'
    '    )\n'
    '    if book.kind == "correlation_momentum":\n'
    '        LOG.info(\n'
    '            "%s: Fisher lineage verified; excluded %s repeated-window cells; %s pairs have no finite momentum history",\n'
    '            book.key,\n'
    '            report["suppressed_repeated_fisher_cells"],\n'
    '            sum(r["last_momentum_date"] is None for r in report["pair_status"]),\n'
    '        )\n'
    '    return {\n'
    '        "book": book,\n'
    '        "meta": meta,\n'
    '        "properties": props,\n'
    '        "source": source,\n'
    '        "output": output,\n'
    '        "controls": controls,\n'
    '        "values": values,\n'
    '        "report": report,\n'
    '        "pair_source": pair_source,\n'
    '        "pair_controls": pair_controls,\n'
    '    }\n'
)

# Embedded source: calculation_helper.py
SOURCES['calculation_helper.py'] = (
    '"""Readable, independently reconstructed histories for any selected output pair."""\n'
    '\n'
    'import math\n'
    'import statistics as stats\n'
    'from datetime import datetime, timezone\n'
    'import numpy as np\n'
    'from validation import source_frame, output_dates, ValidationError\n'
    'from momentum_pipeline import fisher_frame, canonical\n'
    'from google_sheets import a1, column_name\n'
    '\n'
    'TAB = "Calculation Helper"\n'
    'HELPER_ID = 2026091401\n'
    '\n'
    '\n'
    'def pair_labels(output):\n'
    '    width = max(map(len, output[:2]), default=0)\n'
    '    return [\n'
    '        f\'{column_name(j+1)} | {output[0][j] if j<len(output[0]) else ""} | {output[1][j] if j<len(output[1]) else ""}\'\n'
    '        for j in range(1, width)\n'
    '    ]\n'
    '\n'
    '\n'
    'def clean(v):\n'
    '    if isinstance(v, (float, np.floating)) and not math.isfinite(v):\n'
    '        return ""\n'
    '    return v\n'
    '\n'
    '\n'
    'def trace(state, pair_index):\n'
    '    """Independent scalar oracle, checked against the production matrix by date."""\n'
    '    book = state["book"]\n'
    '    kind = book.kind\n'
    '    output = state["output"]\n'
    '    w = state["report"]["window"]\n'
    '    m = state["report"].get("momentum_lookback")\n'
    '    labels = pair_labels(output)\n'
    '    if not 0 <= pair_index < len(labels):\n'
    '        raise ValidationError("Helper pair selection outside output matrix")\n'
    '    j = pair_index + 1\n'
    '    x = output[0][j] if j < len(output[0]) else ""\n'
    '    y = output[1][j] if j < len(output[1]) else ""\n'
    '    if kind == "correlation_momentum":\n'
    '        frame, _ = fisher_frame(state["source"])\n'
    '        valid_mapping = (x, y) in frame.columns\n'
    '        values = (\n'
    '            frame[(x, y)].to_numpy() if valid_mapping else np.full(len(frame), np.nan)\n'
    '        )\n'
    '        xv = yv = np.full(len(frame), np.nan)\n'
    '        if state.get("pair_source") is None:\n'
    '            raise ValidationError("Helper requires dated paired inputs")\n'
    '        raw, _ = source_frame(state["pair_source"])\n'
    '        paired_dates = (\n'
    '            [\n'
    '                d\n'
    '                for d, a, b in zip(raw.index, raw[x], raw[y])\n'
    '                if math.isfinite(a) and math.isfinite(b)\n'
    '            ]\n'
    '            if x in raw and y in raw\n'
    '            else []\n'
    '        )\n'
    '        input_count = 0\n'
    '        previous_count = 0\n'
    '    else:\n'
    '        frame, _ = source_frame(state["source"])\n'
    '        valid_mapping = x in frame and y in frame\n'
    '        xv = frame[x].to_numpy() if x in frame else np.full(len(frame), np.nan)\n'
    '        yv = frame[y].to_numpy() if y in frame else np.full(len(frame), np.nan)\n'
    '        values = xv - yv\n'
    '    date_index = output_dates(output, frame.index)\n'
    '    production = {d: state["values"][i][pair_index] for i, d in enumerate(date_index)}\n'
    '    common = ["Date", "Current X", "Current Y"]\n'
    '    if kind == "correlation":\n'
    '        headers = common + [\n'
    '            "Prior pair count",\n'
    '            "Window first date",\n'
    '            "Window last date",\n'
    '            "Mean X",\n'
    '            "Mean Y",\n'
    '            "Sample std X",\n'
    '            "Sample std Y",\n'
    '            "Sample covariance",\n'
    '            "Pearson r",\n'
    '            "Fisher score",\n'
    '            "Output score",\n'
    '            "Status",\n'
    '        ]\n'
    '    elif kind == "spread":\n'
    '        headers = common + [\n'
    '            "Signed spread",\n'
    '            "Prior spread count",\n'
    '            "Window first date",\n'
    '            "Window last date",\n'
    '            "Prior mean",\n'
    '            "Prior sample std",\n'
    '            "Numerator",\n'
    '            "Calculated z-score",\n'
    '            "Output score",\n'
    '            "Status",\n'
    '        ]\n'
    '    else:\n'
    '        headers = (\n'
    '            common + ["Signed spread"]\n'
    '            if kind == "spread_momentum"\n'
    '            else ["Date", "Fisher value"]\n'
    '        ) + [\n'
    '            "Lag observation date",\n'
    '            "Lag value",\n'
    '            "Signed change",\n'
    '            "Prior change count",\n'
    '            "Window first date",\n'
    '            "Window last date",\n'
    '            "Prior mean change",\n'
    '            "Prior sample std",\n'
    '            "Numerator",\n'
    '            "Calculated z-score",\n'
    '            "Output score",\n'
    '            "Status",\n'
    '        ]\n'
    '    if kind == "correlation_momentum":\n'
    '        headers += [\n'
    '            "Last paired input date",\n'
    '            "Input age days",\n'
    '            "New paired observation",\n'
    '        ]\n'
    '    history = []\n'
    '    seen = []\n'
    '    table = []\n'
    '    matched = 0\n'
    '    for i, date in enumerate(frame.index):\n'
    '        fresh = True\n'
    '        last_input = ""\n'
    '        input_age = ""\n'
    '        if kind == "correlation_momentum":\n'
    '            while input_count < len(paired_dates) and paired_dates[input_count] <= date:\n'
    '                input_count += 1\n'
    '            fresh = input_count > previous_count\n'
    '            previous_count = input_count\n'
    '            if input_count:\n'
    '                last_input = str(paired_dates[input_count - 1].date())\n'
    '                input_age = int((date - paired_dates[input_count - 1]).days)\n'
    '        d = str(date.date())\n'
    '        score = math.nan\n'
    '        status = "Insufficient history"\n'
    '        h = history[-w:]\n'
    '        count = len(h)\n'
    '        first = h[0][0] if h else ""\n'
    '        last = h[-1][0] if h else ""\n'
    '        if kind == "correlation":\n'
    '            mx = my = sx = sy = cov = r = math.nan\n'
    '            if count == w and math.isfinite(xv[i]) and math.isfinite(yv[i]):\n'
    '                a = [v[1] for v in h]\n'
    '                b = [v[2] for v in h]\n'
    '                mx = stats.mean(a)\n'
    '                my = stats.mean(b)\n'
    '                sx = stats.stdev(a)\n'
    '                sy = stats.stdev(b)\n'
    '                cov = sum((aa - mx) * (bb - my) for aa, bb in zip(a, b)) / (w - 1)\n'
    '                if sx > 0 and sy > 0:\n'
    '                    r = cov / (sx * sy)\n'
    '                    if abs(r) < 1 - 1e-14:\n'
    '                        score = math.atanh(r)\n'
    '                        status = "OK"\n'
    '                    else:\n'
    '                        status = "Perfect/near-perfect correlation"\n'
    '                else:\n'
    '                    status = "Zero input variance"\n'
    '            row = [d, xv[i], yv[i], count, first, last, mx, my, sx, sy, cov, r, score]\n'
    '            if math.isfinite(xv[i]) and math.isfinite(yv[i]):\n'
    '                history.append((d, float(xv[i]), float(yv[i])))\n'
    '            else:\n'
    '                status = "Missing current pair"\n'
    '        else:\n'
    '            display_val = float(values[i])\n'
    '            val = display_val if fresh else math.nan\n'
    '            change = math.nan\n'
    '            lagdate = ""\n'
    '            lagval = math.nan\n'
    '            if m is not None and math.isfinite(val) and len(seen) >= m:\n'
    '                lagdate, lagval = seen[-m]\n'
    '                change = val - lagval\n'
    '            target = val if m is None else change\n'
    '            mean = sd = numerator = math.nan\n'
    '            if count == w:\n'
    '                mean = stats.mean(v[1] for v in h)\n'
    '                sd = stats.stdev(v[1] for v in h)\n'
    '                if math.isfinite(target):\n'
    '                    numerator = target - mean\n'
    '                    if sd > 0:\n'
    '                        score = numerator / sd\n'
    '                        status = "OK"\n'
    '                    else:\n'
    '                        status = "Zero historical standard deviation"\n'
    '            if not math.isfinite(val):\n'
    '                status = "Missing current observation"\n'
    '            elif m is not None and not math.isfinite(change):\n'
    '                status = "Insufficient lag history"\n'
    '            if m is None:\n'
    '                row = [\n'
    '                    d,\n'
    '                    xv[i],\n'
    '                    yv[i],\n'
    '                    val,\n'
    '                    count,\n'
    '                    first,\n'
    '                    last,\n'
    '                    mean,\n'
    '                    sd,\n'
    '                    numerator,\n'
    '                    score,\n'
    '                ]\n'
    '            else:\n'
    '                row = (\n'
    '                    [d, xv[i], yv[i], val]\n'
    '                    if kind == "spread_momentum"\n'
    '                    else [d, display_val]\n'
    '                ) + [\n'
    '                    lagdate,\n'
    '                    lagval,\n'
    '                    change,\n'
    '                    count,\n'
    '                    first,\n'
    '                    last,\n'
    '                    mean,\n'
    '                    sd,\n'
    '                    numerator,\n'
    '                    score,\n'
    '                ]\n'
    '            if math.isfinite(target):\n'
    '                history.append((d, target))\n'
    '            if math.isfinite(val):\n'
    '                seen.append((d, val))\n'
    '        if not valid_mapping:\n'
    '            status = "Invalid pair mapping"\n'
    '            score = math.nan\n'
    '        published = production.get(date, "")\n'
    '        if date in production:\n'
    '            if published == "":\n'
    '                if math.isfinite(score):\n'
    '                    raise ValidationError(\n'
    '                        f"Helper disagrees with blank output: {labels[pair_index]}, {d}"\n'
    '                    )\n'
    '            elif not math.isfinite(score) or not math.isclose(\n'
    '                score, published, rel_tol=1e-9, abs_tol=1e-10\n'
    '            ):\n'
    '                raise ValidationError(\n'
    '                    f"Helper differs from output: {labels[pair_index]}, {d}: {score} vs {published}"\n'
    '                )\n'
    '            else:\n'
    '                matched += 1\n'
    '        if kind == "correlation_momentum" and not fresh:\n'
    '            status = "No new paired observation"\n'
    '        row.extend([published, status])\n'
    '        if kind == "correlation_momentum":\n'
    '            row.extend([last_input, input_age, "Yes" if fresh else "No"])\n'
    '        table.append([clean(v) for v in row])\n'
    '    return headers, table, matched\n'
    '\n'
    '\n'
    'def prepare(sheets, state):\n'
    '    labels = pair_labels(state["output"])\n'
    '    if not labels:\n'
    '        raise ValidationError("No helper pairs")\n'
    '    tabs = {x["properties"]["title"]: x["properties"] for x in state["meta"]["sheets"]}\n'
    '    selected = ""\n'
    '    if TAB in tabs:\n'
    '        row = sheets.values(state["book"].spreadsheet_id, [a1(TAB, "B2")])[0]\n'
    '        selected = row[0][0] if row and row[0] else ""\n'
    '    # Fail clearly when the selected pair was removed/reordered, rather than show another pair.\n'
    '    if selected and selected not in labels:\n'
    '        raise ValidationError(\n'
    '            "Helper pair no longer exists; choose a valid pair or clear Calculation Helper!B2"\n'
    '        )\n'
    '    selected = selected or labels[0]\n'
    '    idx = labels.index(selected)\n'
    '    headers, table, matched = trace(state, idx)\n'
    '    return {\n'
    '        "selection": selected,\n'
    '        "labels": labels,\n'
    '        "headers": headers,\n'
    '        "table": table,\n'
    '        "matched": matched,\n'
    '        "old_selection": selected if TAB in tabs else None,\n'
    '    }\n'
    '\n'
    '\n'
    'def publish(sheets, state, helper, run_dir):\n'
    '    book = state["book"]\n'
    '    bid = book.spreadsheet_id\n'
    '    meta = sheets.metadata(bid)\n'
    '    props = next(\n'
    '        (x["properties"] for x in meta["sheets"] if x["properties"]["title"] == TAB),\n'
    '        None,\n'
    '    )\n'
    '    required = max(len(helper["table"]) + 6, len(helper["labels"]) + 1)\n'
    '    req = []\n'
    '    if props is None:\n'
    '        req.append(\n'
    '            {\n'
    '                "addSheet": {\n'
    '                    "properties": {\n'
    '                        "sheetId": HELPER_ID,\n'
    '                        "title": TAB,\n'
    '                        "gridProperties": {\n'
    '                            "rowCount": required,\n'
    '                            "columnCount": 22,\n'
    '                            "frozenRowCount": 5,\n'
    '                            "frozenColumnCount": 1,\n'
    '                        },\n'
    '                    }\n'
    '                }\n'
    '            }\n'
    '        )\n'
    '        sid = HELPER_ID\n'
    '    else:\n'
    '        sid = props["sheetId"]\n'
    '        old = sheets.values(bid, [a1(TAB)])[0]\n'
    '        current = old[1][1] if len(old) > 1 and len(old[1]) > 1 else ""\n'
    '        if current and current != helper["selection"]:\n'
    '            raise ValidationError("Helper selection changed during run; rerun")\n'
    '        sheets.backup(\n'
    '            run_dir / (book.key + "_helper_before.json.gz"),\n'
    '            {"spreadsheet_id": bid, "sheet": TAB, "values": old},\n'
    '        )\n'
    '        grid = props["gridProperties"]\n'
    '        req.append(\n'
    '            {\n'
    '                "updateSheetProperties": {\n'
    '                    "properties": {\n'
    '                        "sheetId": sid,\n'
    '                        "gridProperties": {\n'
    '                            "rowCount": max(required, grid["rowCount"]),\n'
    '                            "columnCount": max(22, grid["columnCount"]),\n'
    '                        },\n'
    '                    },\n'
    '                    "fields": "gridProperties.rowCount,gridProperties.columnCount",\n'
    '                }\n'
    '            }\n'
    '        )\n'
    '    sheets.batch(bid, req)\n'
    '    n = len(helper["headers"])\n'
    '    end = column_name(n)\n'
    '    rows = [\n'
    '        ["Calculation audit", book.title],\n'
    '        ["Selected pair", helper["selection"]],\n'
    '        ["Calculated pair", helper["selection"]],\n'
    '        [\n'
    '            "Window",\n'
    '            state["report"]["window"],\n'
    '            "Lookback",\n'
    '            state["report"].get("momentum_lookback", "N/A"),\n'
    '            "Refreshed UTC",\n'
    '            datetime.now(timezone.utc).isoformat(timespec="seconds"),\n'
    '            "",\n'
    '            "",\n'
    '            "Select B2, then rerun Python. Momentum windows count paired updates, not calendar months. Lag and window dates show actual horizons.",\n'
    '        ],\n'
    '        helper["headers"],\n'
    '    ] + helper["table"]\n'
    '    sheets.request(\n'
    '        "POST",\n'
    '        bid,\n'
    '        "/values:batchClear",\n'
    '        json={"ranges": [a1(TAB, "A1:V1"), a1(TAB, "A3:V"), a1(TAB, "U2:V2")]},\n'
    '    )\n'
    '    sheets.write_values(\n'
    '        bid,\n'
    '        [\n'
    '            {"range": a1(TAB, "A1"), "values": rows},\n'
    '            {\n'
    '                "range": a1(TAB, "U1"),\n'
    '                "values": [["Available pairs"]]\n'
    '                + [[label] for label in helper["labels"]],\n'
    '            },\n'
    '        ],\n'
    '    )\n'
    '    format_req = [\n'
    '        {\n'
    '            "setDataValidation": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "startRowIndex": 1,\n'
    '                    "endRowIndex": 2,\n'
    '                    "startColumnIndex": 1,\n'
    '                    "endColumnIndex": 2,\n'
    '                },\n'
    '                "rule": {\n'
    '                    "condition": {\n'
    '                        "type": "ONE_OF_RANGE",\n'
    '                        "values": [\n'
    '                            {\n'
    '                                "userEnteredValue": f"=\'{TAB}\'!$U$2:$U${len(helper[\'labels\'])+1}"\n'
    '                            }\n'
    '                        ],\n'
    '                    },\n'
    '                    "strict": True,\n'
    '                    "showCustomUi": True,\n'
    '                },\n'
    '            }\n'
    '        },\n'
    '        {\n'
    '            "repeatCell": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "startRowIndex": 0,\n'
    '                    "endRowIndex": len(rows),\n'
    '                    "startColumnIndex": 0,\n'
    '                    "endColumnIndex": n,\n'
    '                },\n'
    '                "cell": {\n'
    '                    "userEnteredFormat": {\n'
    '                        "textFormat": {"fontFamily": "Arial", "fontSize": 10},\n'
    '                        "verticalAlignment": "MIDDLE",\n'
    '                    }\n'
    '                },\n'
    '                "fields": "userEnteredFormat.textFormat,userEnteredFormat.verticalAlignment",\n'
    '            }\n'
    '        },\n'
    '        {\n'
    '            "repeatCell": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "startRowIndex": 4,\n'
    '                    "endRowIndex": 5,\n'
    '                    "startColumnIndex": 0,\n'
    '                    "endColumnIndex": n,\n'
    '                },\n'
    '                "cell": {\n'
    '                    "userEnteredFormat": {\n'
    '                        "backgroundColor": {"red": 0.85, "green": 0.85, "blue": 0.85},\n'
    '                        "textFormat": {"bold": True},\n'
    '                        "wrapStrategy": "WRAP",\n'
    '                    }\n'
    '                },\n'
    '                "fields": "userEnteredFormat.backgroundColor,userEnteredFormat.textFormat.bold,userEnteredFormat.wrapStrategy",\n'
    '            }\n'
    '        },\n'
    '        {\n'
    '            "repeatCell": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "startRowIndex": 1,\n'
    '                    "endRowIndex": 2,\n'
    '                    "startColumnIndex": 1,\n'
    '                    "endColumnIndex": 2,\n'
    '                },\n'
    '                "cell": {\n'
    '                    "userEnteredFormat": {\n'
    '                        "backgroundColor": {"red": 1, "green": 0.9, "blue": 0.8}\n'
    '                    }\n'
    '                },\n'
    '                "fields": "userEnteredFormat.backgroundColor",\n'
    '            }\n'
    '        },\n'
    '        {\n'
    '            "repeatCell": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "startRowIndex": 5,\n'
    '                    "endRowIndex": len(rows),\n'
    '                    "startColumnIndex": 1,\n'
    '                    "endColumnIndex": n - 1,\n'
    '                },\n'
    '                "cell": {\n'
    '                    "userEnteredFormat": {\n'
    '                        "numberFormat": {"type": "NUMBER", "pattern": "0.000000"}\n'
    '                    }\n'
    '                },\n'
    '                "fields": "userEnteredFormat.numberFormat",\n'
    '            }\n'
    '        },\n'
    '        {\n'
    '            "updateDimensionProperties": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "dimension": "COLUMNS",\n'
    '                    "startIndex": 0,\n'
    '                    "endIndex": n,\n'
    '                },\n'
    '                "properties": {"pixelSize": 145},\n'
    '                "fields": "pixelSize",\n'
    '            }\n'
    '        },\n'
    '        {\n'
    '            "updateDimensionProperties": {\n'
    '                "range": {\n'
    '                    "sheetId": sid,\n'
    '                    "dimension": "ROWS",\n'
    '                    "startIndex": 4,\n'
    '                    "endIndex": 5,\n'
    '                },\n'
    '                "properties": {"pixelSize": 52},\n'
    '                "fields": "pixelSize",\n'
    '            }\n'
    '        },\n'
    '    ]\n'
    '    # Keep merges wholly to the right of the frozen date column. Repair partial setup safely.\n'
    '    existing_merges = next(\n'
    '        (\n'
    '            x.get("merges", [])\n'
    '            for x in meta["sheets"]\n'
    '            if x["properties"]["title"] == TAB\n'
    '        ),\n'
    '        [],\n'
    '    )\n'
    '    for row in (0, 1, 2):\n'
    '        r = {\n'
    '            "sheetId": sid,\n'
    '            "startRowIndex": row,\n'
    '            "endRowIndex": row + 1,\n'
    '            "startColumnIndex": 1,\n'
    '            "endColumnIndex": 5,\n'
    '        }\n'
    '        if r not in existing_merges:\n'
    '            format_req.append({"mergeCells": {"range": r, "mergeType": "MERGE_ALL"}})\n'
    '    r = {\n'
    '        "sheetId": sid,\n'
    '        "startRowIndex": 3,\n'
    '        "endRowIndex": 4,\n'
    '        "startColumnIndex": 5,\n'
    '        "endColumnIndex": 8,\n'
    '    }\n'
    '    if r not in existing_merges:\n'
    '        format_req.append({"mergeCells": {"range": r, "mergeType": "MERGE_ALL"}})\n'
    '    for j, label in enumerate(helper["headers"]):\n'
    '        if label.endswith("count"):\n'
    '            format_req.append(\n'
    '                {\n'
    '                    "repeatCell": {\n'
    '                        "range": {\n'
    '                            "sheetId": sid,\n'
    '                            "startRowIndex": 5,\n'
    '                            "endRowIndex": len(rows),\n'
    '                            "startColumnIndex": j,\n'
    '                            "endColumnIndex": j + 1,\n'
    '                        },\n'
    '                        "cell": {\n'
    '                            "userEnteredFormat": {\n'
    '                                "numberFormat": {"type": "NUMBER", "pattern": "0"}\n'
    '                            }\n'
    '                        },\n'
    '                        "fields": "userEnteredFormat.numberFormat",\n'
    '                    }\n'
    '                }\n'
    '            )\n'
    '    sheets.batch(bid, format_req)\n'
    '    actual = sheets.values(bid, [a1(TAB, f"A5:{end}{len(rows)}")])[0]\n'
    '    if canonical(actual) != canonical([helper["headers"]] + helper["table"]):\n'
    '        raise ValidationError("Helper readback mismatch")\n'
    '    state["report"]["helper"] = {\n'
    '        "tab": TAB,\n'
    '        "pair": helper["selection"],\n'
    '        "dates": len(helper["table"]),\n'
    '        "numeric_checks": helper["matched"],\n'
    '        "verified": True,\n'
    '    }\n'
)

# Embedded source: validation.py
SOURCES['validation.py'] = (
    '"""Strict date parsing, exact mappings, and auditable source validation."""\n'
    '\n'
    'from collections import Counter\n'
    'from datetime import datetime\n'
    'from numbers import Real\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    '\n'
    '\n'
    'class ValidationError(ValueError):\n'
    '    pass\n'
    '\n'
    '\n'
    'def parse_date(value):\n'
    '    if value is None or value == "":\n'
    '        return pd.NaT\n'
    '    if isinstance(value, bool):\n'
    '        raise ValidationError(f"Malformed date: {value!r}")\n'
    '    try:\n'
    '        if isinstance(value, Real):\n'
    '            if not np.isfinite(value) or float(value) != int(value):\n'
    '                raise ValueError()\n'
    '            result = pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(value))\n'
    '        elif isinstance(value, (datetime, pd.Timestamp)):\n'
    '            result = pd.Timestamp(value)\n'
    '        else:\n'
    '            result = None\n'
    '            for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%Y-%m-%dT%H:%M:%S"):\n'
    '                try:\n'
    '                    result = pd.Timestamp(datetime.strptime(str(value), fmt))\n'
    '                    break\n'
    '                except ValueError:\n'
    '                    pass\n'
    '            if result is None:\n'
    '                raise ValueError()\n'
    '        if result.tzinfo is not None or result != result.normalize():\n'
    '            raise ValueError()\n'
    '        return result\n'
    '    except (ValueError, TypeError, OverflowError):\n'
    '        raise ValidationError(f"Malformed date: {value!r}") from None\n'
    '\n'
    '\n'
    'def numeric(value):\n'
    '    # Sheets ISNUMBER semantics: numeric-looking text and booleans are not observations.\n'
    '    return (\n'
    '        float(value)\n'
    '        if isinstance(value, Real)\n'
    '        and not isinstance(value, bool)\n'
    '        and np.isfinite(value)\n'
    '        else np.nan\n'
    '    )\n'
    '\n'
    '\n'
    'def source_frame(rows):\n'
    '    if not rows or not rows[0] or rows[0][0] != "Date":\n'
    '        raise ValidationError(\n'
    '            "Source must have Date in A1 and resolved indicator headers; an import may be loading or broken"\n'
    '        )\n'
    '    headers = rows[0][1:]\n'
    '    counts = Counter(headers)\n'
    '    invalid = {h for h in headers if not isinstance(h, str) or not h or counts[h] != 1}\n'
    '    issues = [f"Duplicate/invalid source header: {h!r}" for h in invalid]\n'
    '    indices = [j for j, h in enumerate(headers, 1) if h not in invalid]\n'
    '    dates, values = [], []\n'
    '    for rownum, row in enumerate(rows[1:], 2):\n'
    '        if not row or row[0] == "":\n'
    '            if any(x != "" for x in row[1:]):\n'
    '                raise ValidationError(\n'
    '                    f"Source row {rownum} has observations without a date"\n'
    '                )\n'
    '            continue\n'
    '        try:\n'
    '            date = parse_date(row[0])\n'
    '        except ValidationError as e:\n'
    '            raise ValidationError(f"Source row {rownum}: {e}") from e\n'
    '        dates.append(date)\n'
    '        values.append([numeric(row[j]) if j < len(row) else np.nan for j in indices])\n'
    '        for j in indices:\n'
    '            if j < len(row) and row[j] != "" and not np.isfinite(numeric(row[j])):\n'
    '                issues.append(\n'
    '                    f"Source row {rownum}, indicator {headers[j-1]!r}: nonnumeric observation {row[j]!r}"\n'
    '                )\n'
    '    if not dates or len(set(dates)) != len(dates):\n'
    '        raise ValidationError("Source dates are empty or duplicated")\n'
    '    frame = pd.DataFrame(\n'
    '        values, index=pd.DatetimeIndex(dates), columns=[headers[j - 1] for j in indices]\n'
    '    ).sort_index()\n'
    '    if not frame.notna().any().any():\n'
    '        raise ValidationError("Source contains no finite indicator observations")\n'
    '    return frame, issues\n'
    '\n'
    '\n'
    'def pair_definitions(rows, frame):\n'
    '    if len(rows) < 2:\n'
    '        raise ValidationError("Output is missing the two pair-definition rows")\n'
    '    width = max(len(rows[0]), len(rows[1]))\n'
    '    if width < 2:\n'
    '        raise ValidationError("No pair definitions found; refusing to clear output")\n'
    '    pairs, issues = [], []\n'
    '    for j in range(1, width):\n'
    '        x = rows[0][j] if j < len(rows[0]) else ""\n'
    '        y = rows[1][j] if j < len(rows[1]) else ""\n'
    '        error = None\n'
    '        if not x and not y:\n'
    '            error = "completely empty pair column"\n'
    '        elif not x or not y:\n'
    '            error = "missing pair definition"\n'
    '        elif x not in frame.columns or y not in frame.columns:\n'
    '            error = "missing or duplicated indicator header"\n'
    '        if error:\n'
    '            issues.append(f"Column {j+1}, pair {x!r} minus/vs {y!r}: {error}")\n'
    '        pairs.append((x, y, error))\n'
    '    return pairs, issues\n'
    '\n'
    '\n'
    'def output_dates(rows, source_dates):\n'
    "    # Rows 1:2 and column A are immutable, including Spread Score's date in A2.\n"
    '    dates = []\n'
    '    last = max((i for i, r in enumerate(rows[2:], 2) if r and r[0] != ""), default=1)\n'
    '    for i in range(2, last + 1):\n'
    '        try:\n'
    '            dates.append(parse_date(rows[i][0] if rows[i] else ""))\n'
    '        except ValidationError as e:\n'
    '            raise ValidationError(f"Output A{i+1}: {e}") from e\n'
    '    valid = [d for d in dates if not pd.isna(d)]\n'
    '    if valid != sorted(set(valid)):\n'
    '        raise ValidationError(\n'
    '            "Output dates must be unique and chronological; column A is preserved"\n'
    '        )\n'
    '    # A source date held in a header row cannot receive a result, but is still used as history.\n'
    '    header_dates = set()\n'
    '    for r in rows[:2]:\n'
    '        try:\n'
    '            d = parse_date(r[0] if r else "")\n'
    '            if not pd.isna(d):\n'
    '                header_dates.add(d)\n'
    '        except ValidationError:\n'
    '            pass\n'
    '    missing = set(source_dates) - set(valid) - header_dates\n'
    '    if missing:\n'
    '        raise ValidationError(\n'
    '            f"{len(missing)} source dates missing from preserved output column A (latest {max(missing).date()}); extend its date structure before publishing"\n'
    '        )\n'
    '    return pd.DatetimeIndex(dates)\n'
)

# Embedded source: spread.py
SOURCES['spread.py'] = (
    '"""Directional spreads and sample statistics over prior valid observations."""\n'
    '\n'
    'import numpy as np\n'
    'from correlation import prior_windows\n'
    '\n'
    '\n'
    'def spread_series(x, y):\n'
    '    return np.asarray(x, dtype=float) - np.asarray(y, dtype=float)\n'
    '\n'
    '\n'
    'def rolling_statistics(values, window):\n'
    '    history = prior_windows(values, window)\n'
    '    valid = np.isfinite(history).all(axis=1)\n'
    '    mean = np.full(len(history), np.nan)\n'
    '    std = np.full(len(history), np.nan)\n'
    '    mean[valid] = history[valid].mean(axis=1)\n'
    '    std[valid] = history[valid].std(axis=1, ddof=1)\n'
    '    return mean, std\n'
    '\n'
    '\n'
    'def spread_zscore(x, y, window):\n'
    '    spread = spread_series(x, y)\n'
    '    mean, std = rolling_statistics(spread, window)\n'
    '    result = np.full(len(spread), np.nan)\n'
    '    valid = np.isfinite(spread) & np.isfinite(mean) & np.isfinite(std) & (std > 0)\n'
    '    result[valid] = (spread[valid] - mean[valid]) / std[valid]\n'
    '    result[~np.isfinite(result)] = np.nan\n'
    '    return result\n'
)

# Embedded source: correlation.py
SOURCES['correlation.py'] = (
    '"""Pearson/Fisher scores over prior valid paired observations."""\n'
    '\n'
    'import numpy as np\n'
    'from numpy.lib.stride_tricks import sliding_window_view\n'
    '\n'
    '\n'
    'def prior_windows(values, window):\n'
    '    """At each row, return the last w finite observations strictly before it."""\n'
    '    if window < 2:\n'
    '        raise ValueError("Window must be >= 2")\n'
    '    values = np.asarray(values, dtype=float)\n'
    '    if not len(values):\n'
    '        return np.empty((0, window))\n'
    '    finite = np.isfinite(values)\n'
    '    # Subtract the current-row flag so t never enters its own historical window.\n'
    '    prior_counts = np.cumsum(finite) - finite\n'
    '    compact = values[finite]\n'
    '    windows = sliding_window_view(np.r_[np.full(window, np.nan), compact], window)\n'
    '    return windows[prior_counts]\n'
    '\n'
    '\n'
    'def fisher_transform(r):\n'
    '    r = np.asarray(r, dtype=float)\n'
    '    out = np.full(r.shape, np.nan)\n'
    '    # Perfect correlations have an undefined finite Fisher score; never clip to artificial values.\n'
    '    valid = np.isfinite(r) & (np.abs(r) < 1 - 1e-14)\n'
    '    out[valid] = np.arctanh(r[valid])\n'
    '    return out\n'
    '\n'
    '\n'
    'def rolling_correlation(x, y, window):\n'
    '    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)\n'
    '    if x.shape != y.shape:\n'
    '        raise ValueError("Series lengths differ")\n'
    '    paired = np.isfinite(x) & np.isfinite(y)\n'
    '    # Apply one shared mask: never join X and Y from different observation dates.\n'
    '    xw = prior_windows(np.where(paired, x, np.nan), window)\n'
    '    yw = prior_windows(np.where(paired, y, np.nan), window)\n'
    '    if xw.shape != yw.shape:\n'
    '        raise ValueError("Series lengths differ")\n'
    '    # A prior window alone must not carry a score into a date without a pair.\n'
    '    valid = paired & np.isfinite(xw).all(axis=1) & np.isfinite(yw).all(axis=1)\n'
    '    result = np.full(len(xw), np.nan)\n'
    '    a, b = xw[valid], yw[valid]\n'
    '    a = a - a.mean(axis=1, keepdims=True)\n'
    '    b = b - b.mean(axis=1, keepdims=True)\n'
    '    den = np.sqrt(np.sum(a * a, axis=1) * np.sum(b * b, axis=1))\n'
    '    corr = np.divide(\n'
    '        np.sum(a * b, axis=1), den, out=np.full(len(a), np.nan), where=den > 0\n'
    '    )\n'
    '    result[valid] = fisher_transform(corr)\n'
    '    return result\n'
)

# Embedded source: momentum.py
SOURCES['momentum.py'] = (
    '"""Signed changes over valid observations, standardised against prior changes."""\n'
    '\n'
    'import numpy as np\n'
    'from spread import rolling_statistics\n'
    '\n'
    '\n'
    'def observation_change(values, lookback):\n'
    '    if (\n'
    '        isinstance(lookback, bool)\n'
    '        or not isinstance(lookback, (int, np.integer))\n'
    '        or lookback < 1\n'
    '    ):\n'
    '        raise ValueError("Momentum Lookback must be an integer >= 1")\n'
    '    values = np.asarray(values, dtype=float)\n'
    '    result = np.full(values.shape, np.nan)\n'
    '    positions = np.flatnonzero(np.isfinite(values))\n'
    '    if len(positions) > lookback:\n'
    '        with np.errstate(over="ignore", invalid="ignore"):\n'
    '            result[positions[lookback:]] = (\n'
    '                values[positions[lookback:]] - values[positions[:-lookback]]\n'
    '            )\n'
    '    result[~np.isfinite(result)] = np.nan\n'
    '    return result\n'
    '\n'
    '\n'
    'def momentum_zscore(values, lookback, window):\n'
    '    change = observation_change(values, lookback)\n'
    '    mean, std = rolling_statistics(change, window)\n'
    '    result = np.full(len(change), np.nan)\n'
    '    valid = np.isfinite(change) & np.isfinite(mean) & np.isfinite(std) & (std > 0)\n'
    '    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n'
    '        result[valid] = (change[valid] - mean[valid]) / std[valid]\n'
    '    result[~np.isfinite(result)] = np.nan\n'
    '    return result\n'
)

# Embedded source: source_freshness.py
SOURCES['source_freshness.py'] = (
    '"""Require evaluated macro imports to match the configured upstream workbook."""\n'
    '\n'
    'import logging\n'
    'import os\n'
    'import re\n'
    'import time\n'
    'from google_sheets import a1, column_name\n'
    'from validation import ValidationError\n'
    '\n'
    '\n'
    'def canonical(rows):\n'
    '    result = [list(row) for row in rows]\n'
    '    for row in result:\n'
    '        while row and row[-1] == "":\n'
    '            row.pop()\n'
    '    while result and not result[-1]:\n'
    '        result.pop()\n'
    '    return result\n'
    '\n'
    '\n'
    'def check_summary_selectors(headers, helpers, expected_score):\n'
    '    """Audit the existing 37-pair macro summary layout before publication."""\n'
    '    if len(headers) != 38 or headers[0] != "Date" or len(set(headers[1:])) != 37:\n'
    '        raise ValidationError(\n'
    '            "Macro summary must contain Date and 37 unique indicator headers"\n'
    '        )\n'
    '    if len(helpers) < 2:\n'
    '        raise ValidationError("Macro summary helper formulas are missing")\n'
    '    for index, indicator in enumerate(headers[1:]):\n'
    '        offset = index * 2\n'
    '        cell = column_name(40 + offset)\n'
    '        heading = helpers[0][offset] if offset < len(helpers[0]) else ""\n'
    '        formula = helpers[1][offset] if offset < len(helpers[1]) else ""\n'
    '        compact = re.sub(r"\\s+", "", formula) if isinstance(formula, str) else ""\n'
    '        selector = (\n'
    '            re.findall(r\'scoreCol\\s*,\\s*MATCH\\(\\s*"([^"]+)"\', formula, re.I)\n'
    '            if isinstance(formula, str)\n'
    '            else []\n'
    '        )\n'
    '        if (\n'
    '            heading != "=" + column_name(index + 2) + "1"\n'
    '            or f"sheet,{cell}1," not in compact\n'
    '            or selector != [expected_score]\n'
    '        ):\n'
    '            raise ValidationError(\n'
    '                f"{expected_score}!{cell}2 ({indicator}): expected {expected_score}; "\n'
    '                f\'found {selector or "unrecognised mapping"}. Correct the macro summary before publishing.\'\n'
    '            )\n'
    '\n'
    '\n'
    'def validate_macro_summary(sheets, book):\n'
    '    source_id = os.getenv(\n'
    '        "GOOGLE_SPREADSHEET_ID", "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c"\n'
    '    )\n'
    '    headers = sheets.values(source_id, [a1(book.macro_tab, "A1:AL1")])[0]\n'
    '    helpers = sheets.values(source_id, [a1(book.macro_tab, "AN1:DI2")], formulas=True)[\n'
    '        0\n'
    '    ]\n'
    '    check_summary_selectors(headers[0] if headers else [], helpers, book.macro_tab)\n'
    '    logging.info("%s: verified all 37 macro summary score selections", book.macro_tab)\n'
    '\n'
    '\n'
    'def wait_for_macro(sheets, book, attempts=12, delay=10):\n'
    '    anchor = sheets.values(\n'
    '        book.spreadsheet_id, [a1(book.source_tab, "A1")], formulas=True\n'
    '    )[0]\n'
    '    formula = anchor[0][0] if anchor and anchor[0] else ""\n'
    '    match = re.fullmatch(\n'
    '        r\'\\s*=IMPORTRANGE\\("([^"]+)"\\s*[,;]\\s*"([^"]+)"\\)\\s*\', formula, re.I\n'
    '    )\n'
    '    if not match:\n'
    '        raise ValidationError(\n'
    '            f"{book.key}: expected a single IMPORTRANGE source at A1; inspect source mapping"\n'
    '        )\n'
    '    origin, source_range = match.groups()\n'
    '    source_id = origin.split("/d/")[1].split("/")[0] if "/d/" in origin else origin\n'
    '    expected = os.getenv(\n'
    '        "GOOGLE_SPREADSHEET_ID", "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c"\n'
    '    )\n'
    '    if source_id != expected:\n'
    '        raise ValidationError(\n'
    '            f"{book.key}: source import points to a different macro workbook"\n'
    '        )\n'
    '    imported_tab = source_range.rsplit("!", 1)[0].strip("\'")\n'
    '    if imported_tab != book.macro_tab:\n'
    '        raise ValidationError(\n'
    '            f"{book.key}: expected {book.macro_tab}, found {imported_tab}; wrong score dataset"\n'
    '        )\n'
    '    for attempt in range(attempts):\n'
    '        upstream = sheets.values(source_id, [source_range])[0]\n'
    '        imported = sheets.values(book.spreadsheet_id, [a1(book.source_tab)])[0]\n'
    '        if upstream and canonical(upstream) == canonical(imported):\n'
    '            return imported\n'
    '        if attempt + 1 < attempts:\n'
    '            logging.info(\n'
    '                "%s: waiting for refreshed macro scores (%s/%s)",\n'
    '                book.key,\n'
    '                attempt + 1,\n'
    '                attempts,\n'
    '            )\n'
    '            time.sleep(delay)\n'
    '    raise ValidationError(\n'
    '        f"{book.key}: macro import is stale or loading; downstream publication stopped"\n'
    '    )\n'
    '\n'
    '\n'
    'def prepare_dataset_links(sheets, book, run_dir):\n'
    '    """Correct copied LEAD date/source metadata only inside the new workbooks.\n'
    '\n'
    '    Momentum import anchors are generated by momentum_pipeline.prepare using the\n'
    '    dataset-qualified upstream map. Save their original state before it runs.\n'
    '    Numerical controls, pair definitions and the macro workbook are untouched.\n'
    '    """\n'
    '    if book.dataset != "leading":\n'
    '        return\n'
    '    from config import upstream_book, window_value, momentum_lookback\n'
    '\n'
    '    meta = sheets.metadata(book.spreadsheet_id)\n'
    '    tabs = {s["properties"]["title"]: s["properties"] for s in meta["sheets"]}\n'
    '    if tabs.get(book.output_tab, {}).get("sheetId") != book.output_id:\n'
    '        raise ValidationError(f"{book.key}: output sheet identity changed")\n'
    '    controls = sheets.values(book.spreadsheet_id, [a1(book.control_tab, "A1:B20")])[0]\n'
    '    window_value(controls, book.control_key)\n'
    '    changes = []\n'
    '    if book.kind.endswith("_momentum"):\n'
    '        momentum_lookback(controls)\n'
    '        up = upstream_book(book)\n'
    '        wait_for_macro(sheets, up)\n'
    '        expected = {\n'
    '            "Source Spreadsheet ID": up.spreadsheet_id,\n'
    '            "Source Tab": (\n'
    '                up.source_tab if book.kind == "spread_momentum" else up.output_tab\n'
    '            ),\n'
    '        }\n'
    '        for key, value in expected.items():\n'
    '            matches = [\n'
    '                (i, row) for i, row in enumerate(controls, 1) if row and row[0] == key\n'
    '            ]\n'
    '            if len(matches) != 1:\n'
    '                raise ValidationError(f"{book.key}: expected one {key} control")\n'
    '            index, row = matches[0]\n'
    '            if len(row) < 2 or row[1] != value:\n'
    '                changes.append(\n'
    '                    {"range": a1(book.control_tab, f"B{index}"), "values": [[value]]}\n'
    '                )\n'
    '        ranges = [a1(book.source_tab, "1:1"), a1(book.output_tab, "1:2")]\n'
    '        if book.kind == "spread_momentum":\n'
    '            ranges.append(a1("Imported Pair Definitions", "1:2"))\n'
    '    else:\n'
    '        wait_for_macro(sheets, book)\n'
    '        ranges = [a1(book.output_tab, "A1:A2")]\n'
    '        if book.kind == "spread":\n'
    '            current = sheets.values(\n'
    '                book.spreadsheet_id, [a1(book.output_tab, "A1")], formulas=True\n'
    '            )[0]\n'
    '            formula = current[0][0] if current and current[0] else ""\n'
    '            # Repair the known copied coincident date formula. Existing leading\n'
    '            # and local date formulas are not replaced merely for stylistic reasons.\n'
    '            if "Coincident Score" in formula:\n'
    '                changes.append(\n'
    '                    {\n'
    '                        "range": a1(book.output_tab, "A1"),\n'
    '                        "values": [[f"=ARRAYFORMULA(\'{book.source_tab}\'!A1:A)"]],\n'
    '                    }\n'
    '                )\n'
    '    original = sheets.values(book.spreadsheet_id, ranges, formulas=True)\n'
    '    sheets.backup(\n'
    '        run_dir / (book.key + "_links_before.json.gz"),\n'
    '        {\n'
    '            "spreadsheet_id": book.spreadsheet_id,\n'
    '            "metadata": meta,\n'
    '            "ranges": ranges,\n'
    '            "formulas": original,\n'
    '            "controls": controls,\n'
    '            "changes": changes,\n'
    '        },\n'
    '    )\n'
    '    if changes:\n'
    '        logging.info(\n'
    '            "%s: correcting copied source/date links in %s",\n'
    '            book.key,\n'
    '            [c["range"] for c in changes],\n'
    '        )\n'
    '        sheets.request(\n'
    '            "POST",\n'
    '            book.spreadsheet_id,\n'
    '            "/values:batchUpdate",\n'
    '            json={"valueInputOption": "USER_ENTERED", "data": changes},\n'
    '        )\n'
    '        actual = sheets.values(\n'
    '            book.spreadsheet_id, [c["range"] for c in changes], formulas=True\n'
    '        )\n'
    '        if actual != [c["values"] for c in changes]:\n'
    '            raise ValidationError(f"{book.key}: source/date link readback failed")\n'
)

# Embedded source: run_lock.py
SOURCES['run_lock.py'] = (
    '"""Process locks on macOS, Linux and Windows; released when the process exits."""\n'
    '\n'
    'import os\n'
    '\n'
    '\n'
    'def acquire_lock(handle):\n'
    '    if os.name == "nt":\n'
    '        import msvcrt\n'
    '\n'
    '        handle.seek(0)\n'
    '        handle.write("0")\n'
    '        handle.flush()\n'
    '        handle.seek(0)\n'
    '        try:\n'
    '            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)\n'
    '        except OSError as exc:\n'
    '            raise BlockingIOError("Another run is active") from exc\n'
    '    else:\n'
    '        import fcntl\n'
    '\n'
    '        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n'
)

# Embedded source: requirements.txt
SOURCES['requirements.txt'] = (
    'pandas>=2.0,<4\n'
    'numpy>=1.26,<3\n'
    'requests>=2.31,<3\n'
    'beautifulsoup4>=4.12,<5\n'
    'urllib3>=2.0,<3\n'
    'xlrd>=2.0.1,<3\n'
    'openpyxl>=3.1,<4\n'
    'google-auth>=2.30,<3\n'
    'google-api-python-client>=2.140,<3\n'
    'gspread>=6,<7\n'
)

# Embedded source: notifications/__init__.py
SOURCES['notifications/__init__.py'] = (
    '"""Shared, best-effort pipeline notifications."""\n'
)

# Embedded source: notifications/telegram.py
SOURCES['notifications/telegram.py'] = (
    '"""Telegram alerts using environment credentials, with no extra dependencies."""\n'
    '\n'
    'import json\n'
    'import os\n'
    'import sys\n'
    'from urllib.parse import urlencode\n'
    'from urllib.request import Request, urlopen\n'
    '\n'
    '\n'
    'def credential(name):\n'
    '    """Read process settings, then current Windows user\'s persistent settings."""\n'
    '    value = os.getenv(name, "").strip()\n'
    '    if not value and sys.platform == "win32":\n'
    '        try:\n'
    '            import winreg\n'
    '\n'
    '            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as key:\n'
    '                value = str(winreg.QueryValueEx(key, name)[0]).strip()\n'
    '        except OSError:\n'
    '            pass\n'
    '    return value\n'
    '\n'
    '\n'
    'def send_telegram(message, title="US pipeline", *, secrets=()):\n'
    '    """Return delivery success; missing credentials/network errors never stop a run."""\n'
    '    try:\n'
    '        token = credential("TELEGRAM_BOT_TOKEN")\n'
    '        chat_id = credential("TELEGRAM_CHAT_ID")\n'
    '        if not token or not chat_id:\n'
    '            print(\n'
    '                "NOTIFY | Skipped: set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID",\n'
    '                file=sys.stderr,\n'
    '            )\n'
    '            return False\n'
    '\n'
    '        def clean(value):\n'
    '            value = str(value)\n'
    '            for secret in (token, chat_id, credential("FRED_API_KEY"), *secrets):\n'
    '                if secret:\n'
    '                    value = value.replace(str(secret), "[redacted]")\n'
    '            return value\n'
    '\n'
    '        text = clean(f"{title}\\n{message}")\n'
    '        # Preserve both the context and the final error when an alert is too long.\n'
    '        if len(text) > 4096:\n'
    '            marker = "\\n[... shortened; see full local log ...]\\n"\n'
    '            text = text[:2000] + marker + text[-(4096 - 2000 - len(marker)) :]\n'
    '        payload = urlencode(\n'
    '            {\n'
    '                "chat_id": chat_id,\n'
    '                "text": text,\n'
    '            }\n'
    '        ).encode()\n'
    '        request = Request(\n'
    '            f"https://api.telegram.org/bot{token}/sendMessage",\n'
    '            data=payload,\n'
    '            headers={"Content-Type": "application/x-www-form-urlencoded"},\n'
    '        )\n'
    '        with urlopen(request, timeout=10) as response:\n'
    '            if json.load(response).get("ok") is not True:\n'
    '                raise ValueError("Telegram rejected the notification")\n'
    '        return True\n'
    '    except Exception:\n'
    '        # Do not print transport errors: they can contain request credentials.\n'
    '        print(\n'
    '            "NOTIFY | Delivery failed; pipeline behavior is unchanged", file=sys.stderr\n'
    '        )\n'
    '        return False\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    raise SystemExit(\n'
    '        0\n'
    '        if send_telegram(\n'
    '            "Telegram test from the US pipeline.", "Pipeline notification test"\n'
    '        )\n'
    '        else 1\n'
    '    )\n'
)

# Embedded source: notifications/lifecycle.py
SOURCES['notifications/lifecycle.py'] = (
    '"""Tailored alerts around executable entry points, on Windows, macOS and Colab."""\n'
    '\n'
    'from contextvars import ContextVar\n'
    'from datetime import datetime, timezone\n'
    'from uuid import uuid4\n'
    'from functools import wraps\n'
    'import os\n'
    'import platform\n'
    'import sys\n'
    'import time\n'
    '\n'
    'from notifications.telegram import send_telegram\n'
    'from notifications import changes\n'
    '\n'
    'STAGE_NAMES = {\n'
    '    "all": "Pipeline",\n'
    '    "importer": "Indicators",\n'
    '    "analysis": "Indicator analysis",\n'
    '    "relationships": "Relationships",\n'
    '    "spread": "Spread",\n'
    '    "correlation": "Correlation",\n'
    '    "momentum": "Momentum",\n'
    '    "spread-momentum": "Spread momentum",\n'
    '    "correlation-momentum": "Correlation momentum",\n'
    '}\n'
    '_active = ContextVar("notification_run", default=None)\n'
    '\n'
    '\n'
    'def record_failure(error, *, secrets=()):\n'
    '    """Retain errors from entry points that deliberately return an exit code."""\n'
    '    state = _active.get()\n'
    '    if state is not None:\n'
    '        state["error"] = str(error)\n'
    '        state["secrets"] = state.get("secrets", ()) + tuple(secrets)\n'
    '\n'
    '\n'
    'def option(argv, name, default=None):\n'
    '    for index, value in enumerate(argv):\n'
    '        if value.startswith(name + "="):\n'
    '            return value.split("=", 1)[1]\n'
    '        if value == name and index + 1 < len(argv):\n'
    '            return argv[index + 1]\n'
    '    return default\n'
    '\n'
    '\n'
    'def relationship_name(argv):\n'
    '    return next(\n'
    '        (\n'
    '            STAGE_NAMES[key]\n'
    '            for key in (\n'
    '                "correlation",\n'
    '                "spread",\n'
    '                "spread-momentum",\n'
    '                "correlation-momentum",\n'
    '                "momentum",\n'
    '            )\n'
    '            if "--" + key in argv\n'
    '        ),\n'
    '        "Relationships",\n'
    '    )\n'
    '\n'
    '\n'
    'def set_run_details(*, log_dir=None, stages=(), secrets=()):\n'
    '    """Attach verified run metadata without emitting an extra alert."""\n'
    '    state = _active.get()\n'
    '    if state is not None:\n'
    '        state["log_dir"] = str(log_dir) if log_dir else None\n'
    '        state["stages"] = list(stages)\n'
    '        state["journal"] = str(log_dir) + "/changes.jsonl" if log_dir else None\n'
    '        state["secrets"] += tuple(secrets)\n'
    '\n'
    '\n'
    'def duration(seconds):\n'
    '    seconds = max(0, int(round(seconds)))\n'
    '    hours, rest = divmod(seconds, 3600)\n'
    '    minutes, seconds = divmod(rest, 60)\n'
    '    return (\n'
    '        f"{hours}h {minutes:02d}m {seconds:02d}s"\n'
    '        if hours\n'
    '        else f"{minutes}m {seconds:02d}s"\n'
    '    )\n'
    '\n'
    '\n'
    'def emit_event(title, detail=""):\n'
    '    state = _active.get()\n'
    '    if state is None:\n'
    '        return\n'
    '    starting = title.endswith(" started")\n'
    '    scope = (\n'
    '        "Entire US pipeline"\n'
    '        if state["name"] == "Pipeline"\n'
    '        else {\n'
    '            "Indicators": "Source data import",\n'
    '            "Relationships": "Relationship calculations",\n'
    '        }.get(state["name"], state["name"])\n'
    '        + " only"\n'
    '    )\n'
    '    lines = [f"Scope: {scope}"]\n'
    '    if state["dataset"] != "default":\n'
    '        lines.append("Dataset: " + state["dataset"].capitalize())\n'
    '    if starting:\n'
    '        lines.append("Started: " + state["started_at"])\n'
    '        lines.append("You will receive one completion report when this run ends.")\n'
    '    else:\n'
    '        lines.append("Duration: " + duration(time.monotonic() - state["started"]))\n'
    '        lines.append(\n'
    '            "Finished: " + datetime.now(timezone.utc).strftime("%d %b %Y, %H:%M UTC")\n'
    '        )\n'
    '        if state.get("stages"):\n'
    '            lines.append(\n'
    '                f"Stages completed: {len(state[\'completed\'])}/{len(state[\'stages\'])}"\n'
    '            )\n'
    '        lines.append("")\n'
    '        try:\n'
    '            lines.extend(changes.describe(changes.events(state.get("journal"))))\n'
    '        except Exception:\n'
    '            lines.append("Change summary unavailable; see the run log.")\n'
    '        if state["completed"]:\n'
    '            lines.extend(["", "Completed:", *state["completed"]])\n'
    '        if detail and not title.endswith(" finished"):\n'
    '            lines.extend(\n'
    '                [\n'
    '                    "",\n'
    '                    "Issue: " + str(detail)[-700:],\n'
    '                    "Some writes may be incomplete. Check /status or /logs before retrying.",\n'
    '                ]\n'
    '            )\n'
    '        lines.extend(["", "Details: /status or /logs"])\n'
    '    device = "Mac" if platform.system() == "Darwin" else platform.system()\n'
    '    lines.extend(["", f"{device} ({platform.node()}) | Run {state[\'run_id\']}"])\n'
    '    send_telegram("\\n".join(lines), title, secrets=state["secrets"])\n'
    '\n'
    '\n'
    'def stage_event(index, total, name, outcome, *, seconds=None):\n'
    '    state = _active.get()\n'
    '    if state is None:\n'
    '        return\n'
    '    state["current_stage"] = f"{index}/{total} {name}"\n'
    '    detail = name\n'
    '    if seconds is not None:\n'
    '        detail += f" — {duration(seconds)}"\n'
    '    if outcome == "finished":\n'
    '        state["completed"].append(detail)\n'
    '\n'
    '\n'
    'def run_notified(name, action, *, argv=(), enabled=True):\n'
    '    if (\n'
    '        not enabled\n'
    '        or os.getenv("US_PIPELINE_NOTIFICATIONS_MANAGED") == "1"\n'
    '        or _active.get() is not None\n'
    '    ):\n'
    '        return action()\n'
    '    state = {\n'
    '        "name": name,\n'
    '        "run_id": uuid4().hex[:12],\n'
    '        "started": time.monotonic(),\n'
    '        "started_at": datetime.now(timezone.utc).strftime("%d %b %Y, %H:%M UTC"),\n'
    '        "dataset": option(argv, "--dataset", "default"),\n'
    '        "secrets": (option(argv, "--api-key"),),\n'
    '        "completed": [],\n'
    '    }\n'
    '    change_token = changes.begin()\n'
    '    token = _active.set(state)\n'
    '    started = state["started"]\n'
    '\n'
    '    def notify(outcome, detail=""):\n'
    '        emit_event(f"{name} {outcome}", detail)\n'
    '\n'
    '    try:\n'
    '        notify("started")\n'
    '        try:\n'
    '            result = action()\n'
    '        except SystemExit as exc:\n'
    '            if exc.code in (None, 0):\n'
    '                notify("finished", f"Completed in {time.monotonic()-started:.1f}s.")\n'
    '            else:\n'
    '                notify("failed", f"SystemExit: {exc.code}")\n'
    '            raise\n'
    '        except (Exception, KeyboardInterrupt) as exc:\n'
    '            notify(\n'
    '                "stopped" if isinstance(exc, KeyboardInterrupt) else "failed",\n'
    '                f\'{type(exc).__name__}: {exc or "Interrupted"}\',\n'
    '            )\n'
    '            raise\n'
    '        if result in (None, 0):\n'
    '            notify("finished", f"Completed in {time.monotonic()-started:.1f}s.")\n'
    '        else:\n'
    '            notify(\n'
    '                "stopped" if result == 130 else "failed",\n'
    '                state.get(\n'
    '                    "error", f"Exited with status {result}; inspect the terminal/logs."\n'
    '                ),\n'
    '            )\n'
    '        return result\n'
    '    finally:\n'
    '        _active.reset(token)\n'
    '        changes.end(change_token)\n'
    '\n'
    '\n'
    'def notify_entrypoint(name):\n'
    '    """Wrap a CLI main while preserving its return value and exceptions."""\n'
    '\n'
    '    def decorate(function):\n'
    '        @wraps(function)\n'
    '        def wrapped(*args, **kwargs):\n'
    '            argv = kwargs.get("argv", args[0] if args else None)\n'
    '            argv = list(sys.argv[1:] if argv is None else argv)\n'
    '            enabled = not any(flag in argv for flag in ("--self-test", "--help", "-h"))\n'
    '            label = name(argv) if callable(name) else name\n'
    '            return run_notified(\n'
    '                label, lambda: function(*args, **kwargs), argv=argv, enabled=enabled\n'
    '            )\n'
    '\n'
    '        return wrapped\n'
    '\n'
    '    return decorate\n'
)

# Embedded source: notifications/changes.py
SOURCES['notifications/changes.py'] = (
    '"""Small, best-effort value-change journal shared by notified subprocesses."""\n'
    '\n'
    'from contextvars import ContextVar\n'
    'import json\n'
    'import math\n'
    'import os\n'
    'from pathlib import Path\n'
    '\n'
    '_events = ContextVar("notification_changes", default=None)\n'
    '\n'
    '\n'
    'def begin():\n'
    '    return _events.set([])\n'
    '\n'
    '\n'
    'def end(token):\n'
    '    _events.reset(token)\n'
    '\n'
    '\n'
    'def events(path=None):\n'
    '    result = list(_events.get() or [])\n'
    '    if path and Path(path).exists():\n'
    '        try:\n'
    '            lines = Path(path).read_text(encoding="utf-8").splitlines()\n'
    '        except OSError:\n'
    '            return result + [\n'
    '                {"category": "Run", "name": "change journal", "unknown": True}\n'
    '            ]\n'
    '        for line in lines:\n'
    '            try:\n'
    '                item = json.loads(line)\n'
    '                if isinstance(item, dict):\n'
    '                    result.append(item)\n'
    '            except ValueError:\n'
    '                continue\n'
    '    return result\n'
    '\n'
    '\n'
    'def enabled():\n'
    '    return _events.get() is not None or bool(os.getenv("US_PIPELINE_CHANGE_REPORT"))\n'
    '\n'
    '\n'
    'def record(category, name, *, added=0, updated=0, removed=0, unknown=False, note=""):\n'
    '    item = dict(\n'
    '        category=category,\n'
    '        name=str(name),\n'
    '        added=int(added),\n'
    '        updated=int(updated),\n'
    '        removed=int(removed),\n'
    '        unknown=bool(unknown),\n'
    '        note=note,\n'
    '    )\n'
    '    try:\n'
    '        path = os.getenv("US_PIPELINE_CHANGE_REPORT")\n'
    '        if path:\n'
    '            with Path(path).open("a", encoding="utf-8") as journal:\n'
    '                journal.write(json.dumps(item) + "\\n")\n'
    '        elif _events.get() is not None:\n'
    '            _events.get().append(item)\n'
    '    except OSError:\n'
    '        # Notification instrumentation must not interrupt spreadsheet work.\n'
    '        pass\n'
    '\n'
    '\n'
    'def compare(old, new):\n'
    '    counts = {"added": 0, "updated": 0, "removed": 0}\n'
    '    for i in range(max(len(old), len(new))):\n'
    '        before = old[i] if i < len(old) else []\n'
    '        after = new[i] if i < len(new) else []\n'
    '        for j in range(max(len(before), len(after))):\n'
    '            a = before[j] if j < len(before) else ""\n'
    '            b = after[j] if j < len(after) else ""\n'
    '            blank_a, blank_b = a in (None, ""), b in (None, "")\n'
    '            if blank_a and blank_b:\n'
    '                continue\n'
    '            if blank_a:\n'
    '                counts["added"] += 1\n'
    '            elif blank_b:\n'
    '                counts["removed"] += 1\n'
    '            else:\n'
    '                try:\n'
    '                    same = math.isclose(float(a), float(b), rel_tol=1e-9, abs_tol=1e-9)\n'
    '                except (TypeError, ValueError):\n'
    '                    same = str(a).strip() == str(b).strip()\n'
    '                if not same:\n'
    '                    counts["updated"] += 1\n'
    '    return counts\n'
    '\n'
    '\n'
    'def capture(reader, new):\n'
    '    if not enabled():\n'
    '        return None\n'
    '    try:\n'
    '        return compare(reader(), new)\n'
    '    except Exception:\n'
    '        return {"unknown": True}\n'
    '\n'
    '\n'
    'def describe(items):\n'
    '    if not items:\n'
    '        return ["Changes: not measured for this run."]\n'
    '    lines = []\n'
    '    for category in dict.fromkeys(item["category"] for item in items):\n'
    '        group = [item for item in items if item["category"] == category]\n'
    '        totals = {\n'
    '            key: sum(item.get(key, 0) for item in group)\n'
    '            for key in ("added", "updated", "removed")\n'
    '        }\n'
    '        unknown = sum(bool(item.get("unknown")) for item in group)\n'
    '        changes = sum(totals.values())\n'
    '        if changes:\n'
    '            details = ", ".join(\n'
    '                f"{value:,} {key}" for key, value in totals.items() if value\n'
    '            )\n'
    '            lines.append(f"{category}: {details} values.")\n'
    '            affected = list(\n'
    '                dict.fromkeys(\n'
    '                    item["name"]\n'
    '                    for item in group\n'
    '                    if any(item.get(key) for key in totals)\n'
    '                )\n'
    '            )\n'
    '            names = ", ".join(affected[:4])\n'
    '            if len(affected) > 4:\n'
    '                names += f" and {len(affected) - 4} more"\n'
    '            lines.append("  Affected: " + names)\n'
    '        elif not unknown:\n'
    '            lines.append(\n'
    '                f"{category}: no value changes detected in tracked ranges ({len(group)} outputs checked)."\n'
    '            )\n'
    '        if unknown:\n'
    '            lines.append(\n'
    '                f"{category}: {unknown} outputs without a reliable change comparison."\n'
    '            )\n'
    '    return lines\n'
)

# Embedded source: notifications/control.py
SOURCES['notifications/control.py'] = (
    '"""Authorised Telegram commands for the existing Windows scheduled pipeline."""\n'
    '\n'
    'import json\n'
    'import os\n'
    'import re\n'
    'from pathlib import Path\n'
    'import subprocess\n'
    'import sys\n'
    'import time\n'
    'from urllib.request import Request, urlopen\n'
    '\n'
    'from notifications.telegram import credential, send_telegram\n'
    'from run_lock import acquire_lock\n'
    '\n'
    'ROOT = Path(__file__).resolve().parents[1]\n'
    'WORK = ROOT / "work"\n'
    'SERVER_ROOT = ROOT.parents[1]\n'
    'TASK = "US complete pipeline"\n'
    'LAUNCHER = SERVER_ROOT / "Scripts" / "run_us_pipeline.bat"\n'
    '_child = None\n'
    'BACKUP_TASK = "Fund Server GitHub Backup"\n'
    'COT_TASK = "US COT report"\n'
    'COT_ROOT = SERVER_ROOT / "Code" / "us_cot_import"\n'
    'COT_WORK = COT_ROOT / "work"\n'
    'COT_LAUNCHER = SERVER_ROOT / "Scripts" / "run_us_COT.cmd"\n'
    'COT_LOG_PATH = SERVER_ROOT / "Logs" / "us_COT.log"\n'
    '_cot_child = None\n'
    'SINGLE_STOCK_ROOT = SERVER_ROOT / "Code" / "US_singlestock_complete_pipeline"\n'
    'SINGLE_STOCK_WORK = SINGLE_STOCK_ROOT / "work"\n'
    'SINGLE_STOCK_TASK = "US single-stock pipeline"\n'
    'SINGLE_STOCK_LAUNCHER = SERVER_ROOT / "Scripts" / "run_us_single_stock_pipeline.bat"\n'
    'SINGLE_STOCK_LOG_PATH = SERVER_ROOT / "Logs" / "us_single_stock_pipeline.log"\n'
    '_single_stock_child = None\n'
    'SINGLE_STOCK_COMMANDS = {\n'
    '    "us-stock-run": "all",\n'
    '    "us-stock-primary": "primary",\n'
    '    "us-stock-secondary": "secondary",\n'
    '    "us-stock-summary": "summary",\n'
    '}\n'
    'STAGE_COMMANDS = {\n'
    '    "run": "all",\n'
    '    "indicators": "importer",\n'
    '    "analysis": "analysis",\n'
    '    "relationships": "relationships",\n'
    '    "spread": "spread",\n'
    '    "correlation": "correlation",\n'
    '    "momentum": "momentum",\n'
    '    "spread-momentum": "spread-momentum",\n'
    '    "correlation-momentum": "correlation-momentum",\n'
    '}\n'
    'LOG_PATH = SERVER_ROOT / "Logs" / "us_complete_pipeline.log"\n'
    'HELP = (\n'
    '    "Commands (hyphens or underscores, with or without /):\\n"\n'
    '    + "\\n".join(\n'
    '        "us-macroanalysis-" + name\n'
    '        for name in (*STAGE_COMMANDS, "status", "logs", "stop")\n'
    '    )\n'
    '    + "\\npythonstopall — stop ALL Python processes (including this bot; requires administrator listener)."\n'
    '    + "\\nShortcuts: /run /status /logs /stop /help."\n'
    '    + "\\nCOT: /us_cot_import (or /cot) runs the COT import; /us_cot_status (or /cot_status) checks it."\n'
    '    + "\\nSingle stock: /us_stock_run, /us_stock_primary, /us_stock_secondary, /us_stock_summary; status/logs/stop use the same us_stock prefix."\n'
    '    + "\\n/backup starts the safe Fund Server GitHub backup; this chat receives only its completion or failure notice."\n'
    '    + "\\n/logs toggles a live log view every 30 seconds with no time limit."\n'
    '    + "\\nRuns update Google Sheets. Stops do not roll back partial writes. Daily schedule stays enabled."\n'
    ')\n'
    'MENU = [\n'
    '    {\n'
    '        "command": (\n'
    '            "us_macro_" if name == "correlation-momentum" else "us_macroanalysis_"\n'
    '        )\n'
    '        + name.replace("-", "_"),\n'
    '        "description": description,\n'
    '    }\n'
    '    for name, description in [\n'
    '        ("run", "Run the entire pipeline"),\n'
    '        ("status", "Show pipeline status"),\n'
    '        ("logs", "Watch live logs; repeat to stop watching"),\n'
    '        ("stop", "Stop this pipeline only"),\n'
    '        ("indicators", "Run importer only"),\n'
    '        ("analysis", "Run indicator analysis only"),\n'
    '        ("relationships", "Run all relationships"),\n'
    '        ("spread", "Run spread only"),\n'
    '        ("correlation", "Run correlation only"),\n'
    '        ("momentum", "Run momentum only"),\n'
    '        ("spread-momentum", "Run spread momentum only"),\n'
    '        ("correlation-momentum", "Run correlation momentum only"),\n'
    '    ]\n'
    '] + [\n'
    '    {\n'
    '        "command": command.replace("-", "_"),\n'
    '        "description": description,\n'
    '    }\n'
    '    for command, description in [\n'
    '        ("us-stock-run", "Run the full single-stock pipeline"),\n'
    '        ("us-stock-status", "Show single-stock pipeline status"),\n'
    '        ("us-stock-logs", "Watch the single-stock log"),\n'
    '        ("us-stock-stop", "Stop the single-stock pipeline"),\n'
    '        ("us-stock-primary", "Run Primary screen only"),\n'
    '        ("us-stock-secondary", "Run Secondary screen only"),\n'
    '        ("us-stock-summary", "Run evidence summary only"),\n'
    '    ]\n'
    '] + [\n'
    '    {\n'
    '        "command": "us_cot_import",\n'
    '        "description": "Run the CFTC COT import",\n'
    '    },\n'
    '    {\n'
    '        "command": "us_cot_status",\n'
    '        "description": "Show COT import status",\n'
    '    },\n'
    '    {\n'
    '        "command": "backup",\n'
    '        "description": "Back up Fund Server safely to GitHub",\n'
    '    },\n'
    '    {\n'
    '        "command": "pythonstopall",\n'
    '        "description": "Stop ALL Python processes; includes this bot",\n'
    '    },\n'
    '    {"command": "help", "description": "List commands and shortcuts"},\n'
    ']\n'
    '\n'
    '\n'
    'def api(method, **payload):\n'
    '    token = credential("TELEGRAM_BOT_TOKEN")\n'
    '    request = Request(\n'
    '        f"https://api.telegram.org/bot{token}/{method}",\n'
    '        data=json.dumps(payload).encode(),\n'
    '        headers={"Content-Type": "application/json"},\n'
    '    )\n'
    '    with urlopen(request, timeout=40) as response:\n'
    '        result = json.load(response)\n'
    '    if result.get("ok") is not True:\n'
    '        raise RuntimeError("Telegram request failed")\n'
    '    return result["result"]\n'
    '\n'
    '\n'
    'def save_json(path, value):\n'
    '    path.parent.mkdir(parents=True, exist_ok=True)\n'
    '    temporary = path.with_suffix(".tmp")\n'
    '    temporary.write_text(json.dumps(value), encoding="utf-8")\n'
    '    temporary.replace(path)\n'
    '\n'
    '\n'
    'def powershell(script):\n'
    '    # Scripts are fixed by this module; Telegram text never becomes shell code.\n'
    '    result = subprocess.run(\n'
    '        [\n'
    '            "powershell.exe",\n'
    '            "-NoProfile",\n'
    '            "-NonInteractive",\n'
    '            "-Command",\n'
    '            "$ErrorActionPreference=\'Stop\'; " + script,\n'
    '        ],\n'
    '        capture_output=True,\n'
    '        text=True,\n'
    '        timeout=20,\n'
    '        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),\n'
    '    )\n'
    '    if result.returncode:\n'
    '        raise RuntimeError("Could not access the Windows pipeline task")\n'
    '    return result.stdout.strip()\n'
    '\n'
    '\n'
    'def task_state():\n'
    '    return powershell(f"(Get-ScheduledTask -TaskName \'{TASK}\').State.ToString()")\n'
    '\n'
    '\n'
    'def cot_task_state():\n'
    '    return powershell(f"(Get-ScheduledTask -TaskName \'{COT_TASK}\').State.ToString()")\n'
    '\n'
    '\n'
    'def single_stock_task_state():\n'
    '    try:\n'
    '        return powershell(\n'
    '            f"(Get-ScheduledTask -TaskName \'{SINGLE_STOCK_TASK}\').State.ToString()"\n'
    '        )\n'
    '    except RuntimeError:\n'
    '        return "Not installed"\n'
    '\n'
    '\n'
    'def backup_task_state():\n'
    '    return powershell(\n'
    '        f"(Get-ScheduledTask -TaskName \'{BACKUP_TASK}\').State.ToString()"\n'
    '    )\n'
    '\n'
    '\n'
    'def backup_busy():\n'
    '    return backup_task_state() in ("Running", "Queued")\n'
    '\n'
    '\n'
    'def start_backup():\n'
    '    """Start the same scheduled task used for the daily safe GitHub backup."""\n'
    '    powershell(f"Start-ScheduledTask -TaskName \'{BACKUP_TASK}\'")\n'
    '\n'
    '\n'
    'def pipeline_locked():\n'
    '    WORK.mkdir(parents=True, exist_ok=True)\n'
    '    with (WORK / "complete_pipeline.lock").open("a+") as handle:\n'
    '        try:\n'
    '            acquire_lock(handle)\n'
    '        except OSError:\n'
    '            return True\n'
    '    return False\n'
    '\n'
    '\n'
    'def busy():\n'
    '    return (\n'
    '        (_child is not None and _child.poll() is None)\n'
    '        or task_state() in ("Running", "Queued")\n'
    '        or pipeline_locked()\n'
    '    )\n'
    '\n'
    '\n'
    'def cot_locked():\n'
    '    COT_WORK.mkdir(parents=True, exist_ok=True)\n'
    '    with (COT_WORK / "cot_import.lock").open("a+") as handle:\n'
    '        try:\n'
    '            acquire_lock(handle)\n'
    '        except OSError:\n'
    '            return True\n'
    '    return False\n'
    '\n'
    '\n'
    'def cot_busy():\n'
    '    return (\n'
    '        (_cot_child is not None and _cot_child.poll() is None)\n'
    '        or cot_task_state() in ("Running", "Queued")\n'
    '        or cot_locked()\n'
    '    )\n'
    '\n'
    '\n'
    'def single_stock_locked():\n'
    '    SINGLE_STOCK_WORK.mkdir(parents=True, exist_ok=True)\n'
    '    with (SINGLE_STOCK_WORK / "single_stock_pipeline.lock").open("a+") as handle:\n'
    '        try:\n'
    '            acquire_lock(handle)\n'
    '        except OSError:\n'
    '            return True\n'
    '    return False\n'
    '\n'
    '\n'
    'def single_stock_busy():\n'
    '    return (\n'
    '        (_single_stock_child is not None and _single_stock_child.poll() is None)\n'
    '        or single_stock_task_state() in ("Running", "Queued")\n'
    '        or single_stock_locked()\n'
    '    )\n'
    '\n'
    '\n'
    'def start_pipeline(stage="all"):\n'
    '    global _child\n'
    '    if stage not in STAGE_COMMANDS.values():\n'
    '        raise ValueError("Unsupported pipeline stage")\n'
    '    if not LAUNCHER.is_file():\n'
    '        raise RuntimeError("The server pipeline launcher is missing")\n'
    '    environment = dict(os.environ)\n'
    '    for key in (\n'
    '        "FRED_API_KEY",\n'
    '        "GOOGLE_APPLICATION_CREDENTIALS",\n'
    '        "TELEGRAM_BOT_TOKEN",\n'
    '        "TELEGRAM_CHAT_ID",\n'
    '    ):\n'
    '        if not environment.get(key):\n'
    '            value = credential(key)\n'
    '            if value:\n'
    '                environment[key] = value\n'
    '    _child = subprocess.Popen(\n'
    '        ["cmd.exe", "/d", "/c", str(LAUNCHER), "--stage", stage],\n'
    '        cwd=ROOT,\n'
    '        env=environment,\n'
    '        stdin=subprocess.DEVNULL,\n'
    '        stdout=subprocess.DEVNULL,\n'
    '        stderr=subprocess.DEVNULL,\n'
    '        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),\n'
    '    )\n'
    '\n'
    '\n'
    'def start_cot_import():\n'
    '    """Launch the same runner used by the Saturday task."""\n'
    '    global _cot_child\n'
    '    if not COT_LAUNCHER.is_file():\n'
    '        raise RuntimeError("The server COT launcher is missing")\n'
    '    environment = dict(os.environ)\n'
    '    for key in ("TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID"):\n'
    '        if not environment.get(key):\n'
    '            value = credential(key)\n'
    '            if value:\n'
    '                environment[key] = value\n'
    '    _cot_child = subprocess.Popen(\n'
    '        ["cmd.exe", "/d", "/c", str(COT_LAUNCHER)],\n'
    '        cwd=COT_ROOT,\n'
    '        env=environment,\n'
    '        stdin=subprocess.DEVNULL,\n'
    '        stdout=subprocess.DEVNULL,\n'
    '        stderr=subprocess.DEVNULL,\n'
    '        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),\n'
    '    )\n'
    '\n'
    '\n'
    'def start_single_stock_pipeline(stage="all"):\n'
    '    """Start a fixed, allow-listed single-stock stage without shell interpolation."""\n'
    '    global _single_stock_child\n'
    '    if stage not in ("all", "primary", "secondary", "summary"):\n'
    '        raise ValueError("Unsupported single-stock stage")\n'
    '    if not SINGLE_STOCK_LAUNCHER.is_file():\n'
    '        raise RuntimeError("The single-stock pipeline launcher is missing")\n'
    '    environment = dict(os.environ)\n'
    '    for key in ("TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID"):\n'
    '        if not environment.get(key):\n'
    '            value = credential(key)\n'
    '            if value:\n'
    '                environment[key] = value\n'
    '    _single_stock_child = subprocess.Popen(\n'
    '        ["cmd.exe", "/d", "/c", str(SINGLE_STOCK_LAUNCHER), "--stage", stage],\n'
    '        cwd=SINGLE_STOCK_ROOT,\n'
    '        env=environment,\n'
    '        stdin=subprocess.DEVNULL,\n'
    '        stdout=subprocess.DEVNULL,\n'
    '        stderr=subprocess.DEVNULL,\n'
    '        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),\n'
    '    )\n'
    '\n'
    '\n'
    'def cot_status():\n'
    '    state = cot_task_state()\n'
    '    active = (\n'
    '        (_cot_child is not None and _cot_child.poll() is None)\n'
    '        or state in ("Running", "Queued")\n'
    '        or cot_locked()\n'
    '    )\n'
    '    lines = [f"COT import: {\'running\' if active else \'idle\'} (task: {state})."]\n'
    '    try:\n'
    '        with COT_LOG_PATH.open("rb") as log:\n'
    '            log.seek(0, 2)\n'
    '            log.seek(max(0, log.tell() - 5000))\n'
    '            recent = safe_log_text(log.read().decode("utf-8", errors="replace"))\n'
    '        markers = [line for line in recent.splitlines() if line.startswith("====")]\n'
    '        if markers:\n'
    '            lines.append("Latest run: " + markers[-1].strip("= "))\n'
    '        else:\n'
    '            lines.append("No completed COT run is recorded in the log yet.")\n'
    '    except OSError:\n'
    '        lines.append("The COT log is not available yet.")\n'
    '    lines.append("A running import will send its own completion report.")\n'
    '    return "\\n".join(lines)\n'
    '\n'
    '\n'
    'def single_stock_status():\n'
    '    state = single_stock_task_state()\n'
    '    active = (\n'
    '        (_single_stock_child is not None and _single_stock_child.poll() is None)\n'
    '        or state in ("Running", "Queued")\n'
    '        or single_stock_locked()\n'
    '    )\n'
    '    lines = [f"Single-stock pipeline: {\'running\' if active else \'idle\'} (task: {state})."]\n'
    '    candidates = sorted(SINGLE_STOCK_ROOT.glob("outputs/*/summary.json"), reverse=True)\n'
    '    if candidates:\n'
    '        try:\n'
    '            report = json.loads(candidates[0].read_text(encoding="utf-8"))\n'
    '            lines.extend(\n'
    '                [\n'
    '                    f"Latest recorded run: {candidates[0].parent.name}",\n'
    '                    f"Recorded status: {report.get(\'status\', \'unknown\')}",\n'
    '                    f"Scope: {report.get(\'stage\', \'unknown\')}",\n'
    '                    f"Current/last stage: {report.get(\'current_stage\', \'not recorded\')}",\n'
    '                    f"Completed stages: {len(report.get(\'stages\', []))}",\n'
    '                ]\n'
    '            )\n'
    '            if "seconds" in report:\n'
    '                lines.append(f"Recorded elapsed: {report[\'seconds\']}s")\n'
    '            if report.get("error"):\n'
    '                lines.append("Error: " + str(report["error"])[-700:])\n'
    '        except (OSError, ValueError, TypeError):\n'
    '            lines.append("Latest summary is unavailable; inspect the local log.")\n'
    '    else:\n'
    '        lines.append("No single-stock run summary recorded yet.")\n'
    '    lines.append(f"Log: {SINGLE_STOCK_LOG_PATH}")\n'
    '    return "\\n".join(lines)\n'
    '\n'
    '\n'
    'def status():\n'
    '    state = task_state()\n'
    '    active = (\n'
    '        (_child is not None and _child.poll() is None)\n'
    '        or state in ("Running", "Queued")\n'
    '        or pipeline_locked()\n'
    '    )\n'
    '    lines = [f"Pipeline: {\'running\' if active else \'idle\'} (task: {state})."]\n'
    '    summaries = []\n'
    '    for candidate in sorted(WORK.glob("[0-9]*T*/summary.json"), reverse=True):\n'
    '        try:\n'
    '            candidate_report = json.loads(candidate.read_text(encoding="utf-8"))\n'
    '            if (\n'
    '                isinstance(candidate_report, dict)\n'
    '                and "status" in candidate_report\n'
    '                and "stages" in candidate_report\n'
    '            ):\n'
    '                summaries.append(candidate)\n'
    '                break\n'
    '        except (OSError, ValueError):\n'
    '            continue\n'
    '    if summaries:\n'
    '        try:\n'
    '            report = json.loads(summaries[0].read_text(encoding="utf-8"))\n'
    '            if not isinstance(report, dict):\n'
    '                raise ValueError("Unrecognised summary")\n'
    '            lines += [\n'
    '                f"Latest recorded run: {summaries[0].parent.name}",\n'
    '                f"Recorded status: {report.get(\'status\', \'unknown\')}",\n'
    '                f"Dataset: {report.get(\'dataset\', \'unknown\')}",\n'
    '                f"Current/last stage: {report.get(\'current_stage\', \'not recorded\')}",\n'
    '                f"Completed stages: {len(report.get(\'stages\', []))}",\n'
    '            ]\n'
    '            for stage in report.get("stages", []):\n'
    '                lines.append(f"{stage[\'stage\']}: {stage.get(\'seconds\', \'?\')}s")\n'
    '            if "seconds" in report:\n'
    '                lines.append(f"Recorded elapsed: {report[\'seconds\']}s")\n'
    '            if report.get("error"):\n'
    '                lines.append("Error: " + str(report["error"]))\n'
    '            lines.append(f"Log: {summaries[0].parent / \'pipeline.log\'}")\n'
    '            if not active and report.get("status") == "running":\n'
    '                lines.append(\n'
    '                    "No active run remains; the last run may have been interrupted."\n'
    '                )\n'
    '        except (OSError, ValueError, KeyError, TypeError):\n'
    '            lines.append("Latest summary is unavailable; inspect the local log.")\n'
    '    else:\n'
    '        lines.append("No run summary recorded yet.")\n'
    '    return "\\n".join(lines)\n'
    '\n'
    '\n'
    'def stop_pipeline():\n'
    '    if not pipeline_locked():\n'
    '        return "No active pipeline holds the run lock. If a run is starting, retry shortly."\n'
    '    try:\n'
    '        active = json.loads((WORK / "active-pipeline.json").read_text(encoding="utf-8"))\n'
    '        run_id = active["run_id"]\n'
    '        if not re.fullmatch(r"[0-9a-f]{32}", run_id):\n'
    '            raise ValueError("Invalid run ID")\n'
    '    except (OSError, ValueError, KeyError, TypeError):\n'
    '        return "Active run predates remote stopping or cannot be identified. No processes were stopped."\n'
    '    (WORK / f"stop-{run_id}.request").touch()\n'
    '    return "Stop requested for this pipeline only. Its runner will terminate the current stage and child processes; setup may need to finish first. Partial writes are not rolled back. The daily schedule stays enabled."\n'
    '\n'
    '\n'
    'def stop_single_stock_pipeline():\n'
    '    if not single_stock_locked():\n'
    '        return "No active single-stock pipeline holds the run lock. If a run is starting, retry shortly."\n'
    '    try:\n'
    '        active = json.loads(\n'
    '            (SINGLE_STOCK_WORK / "active-single-stock-pipeline.json").read_text(\n'
    '                encoding="utf-8"\n'
    '            )\n'
    '        )\n'
    '        run_id = active["run_id"]\n'
    '        if not re.fullmatch(r"[0-9a-f]{32}", run_id):\n'
    '            raise ValueError("Invalid run ID")\n'
    '    except (OSError, ValueError, KeyError, TypeError):\n'
    '        return "Active single-stock run cannot be identified. No processes were stopped."\n'
    '    (SINGLE_STOCK_WORK / f"stop-{run_id}.request").touch()\n'
    '    return "Stop requested for the single-stock pipeline only. Its current child process will terminate; partial workbook writes are not rolled back. The weekly schedule stays enabled."\n'
    '\n'
    '\n'
    'def is_admin():\n'
    '    if sys.platform != "win32":\n'
    '        return False\n'
    '    import ctypes\n'
    '\n'
    '    return bool(ctypes.windll.shell32.IsUserAnAdmin())\n'
    '\n'
    '\n'
    'def stop_all_python():\n'
    '    # Separate PowerShell process survives the Python processes it terminates.\n'
    '    # Snapshot the targets once; the supervisor can restart the listener afterward.\n'
    '    script = (\n'
    '        "Start-Sleep -Seconds 2; "\n'
    '        "$targets = Get-Process | Where-Object { $_.ProcessName -match \'^python(?:w|[0-9]+(?:[.][0-9]+)*)?$|^py$\' }; "\n'
    '        "$targets | Stop-Process -Force -ErrorAction Continue"\n'
    '    )\n'
    '    subprocess.Popen(\n'
    '        ["powershell.exe", "-NoProfile", "-NonInteractive", "-Command", script],\n'
    '        stdin=subprocess.DEVNULL,\n'
    '        stdout=subprocess.DEVNULL,\n'
    '        stderr=subprocess.DEVNULL,\n'
    '        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),\n'
    '    )\n'
    '\n'
    '\n'
    'def safe_log_text(text):\n'
    '    for key in ("TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID", "FRED_API_KEY"):\n'
    '        value = credential(key)\n'
    '        if value:\n'
    '            text = text.replace(value, "[redacted]")\n'
    '    text = re.sub(\n'
    '        r"(?i)(api[_-]?key|token|authorization)([=:]\\s*)[^\\s&]+",\n'
    '        r"\\1\\2[redacted]",\n'
    '        text,\n'
    '    )\n'
    '    return text\n'
    '\n'
    '\n'
    'def log_tail():\n'
    '    try:\n'
    '        with LOG_PATH.open("rb") as log:\n'
    '            log.seek(0, 2)\n'
    '            log.seek(max(0, log.tell() - 6000))\n'
    '            text = log.read().decode("utf-8", errors="replace")\n'
    '        return safe_log_text(text)[-3000:] or "No log output yet."\n'
    '    except OSError:\n'
    '        return "The server log is not available yet."\n'
    '\n'
    '\n'
    'def single_stock_log_tail():\n'
    '    try:\n'
    '        with SINGLE_STOCK_LOG_PATH.open("rb") as log:\n'
    '            log.seek(0, 2)\n'
    '            log.seek(max(0, log.tell() - 6000))\n'
    '            text = log.read().decode("utf-8", errors="replace")\n'
    '        return safe_log_text(text)[-3000:] or "No log output yet."\n'
    '    except OSError:\n'
    '        return "The single-stock server log is not available yet."\n'
    '\n'
    '\n'
    'def parse_allowed_user_ids(raw_users):\n'
    '    """Require explicit, positive Telegram user IDs for group control."""\n'
    '    users = [value.strip() for value in raw_users.split(",")]\n'
    '    if not users or any(\n'
    '        not re.fullmatch(r"[0-9]+", value)\n'
    '        or len(value.lstrip("0")) > 19\n'
    '        or not 0 < int(value) <= 2**63 - 1\n'
    '        for value in users\n'
    '    ):\n'
    '        raise ValueError("TELEGRAM_ALLOWED_USER_IDS must contain positive numeric IDs separated by commas")\n'
    '    return frozenset(str(int(value)) for value in users)\n'
    '\n'
    '\n'
    'class Controller:\n'
    '    def __init__(self, chat_id, user_ids, username, offset_path=None):\n'
    '        self.chat_id = str(chat_id)\n'
    '        if not re.fullmatch(r"-[0-9]+", self.chat_id) or int(self.chat_id) >= 0:\n'
    '            raise ValueError("TELEGRAM_CHAT_ID must be a negative group ID")\n'
    '        self.user_ids = parse_allowed_user_ids(\n'
    '            user_ids if isinstance(user_ids, str) else ",".join(str(user) for user in user_ids)\n'
    '        )\n'
    '        self.username = username.lower()\n'
    '        self.offset_path = offset_path or WORK / "telegram-offset.json"\n'
    '        self.offset = None\n'
    '        if self.offset_path.exists():\n'
    '            self.offset = int(json.loads(self.offset_path.read_text())["offset"])\n'
    '        self.last_launch = 0.0\n'
    '        self.last_cot_launch = 0.0\n'
    '        self.last_single_stock_launch = 0.0\n'
    '        self.last_backup_launch = 0.0\n'
    '        self.log_watch = None\n'
    '\n'
    '    def checkpoint(self, offset):\n'
    '        save_json(self.offset_path, {"offset": offset})\n'
    '        self.offset = offset\n'
    '\n'
    '    def handle(self, update):\n'
    '        update_id = int(update["update_id"])\n'
    '        if self.offset is not None and update_id < self.offset:\n'
    '            return\n'
    '        # Persist before executing, so a reconnect/restart cannot replay /run.\n'
    '        self.checkpoint(update_id + 1)\n'
    '        message = update.get("message", {})\n'
    '        if (\n'
    '            str(message.get("chat", {}).get("id")) != self.chat_id\n'
    '            or message.get("chat", {}).get("type") not in ("group", "supergroup")\n'
    '            or str(message.get("from", {}).get("id")) not in self.user_ids\n'
    '            or message.get("from", {}).get("is_bot")\n'
    '            or message.get("forward_origin")\n'
    '        ):\n'
    '            return\n'
    '        text = message.get("text", "").strip()\n'
    '        parts = text.split()\n'
    '        if not parts:\n'
    '            return\n'
    '        command, _, mention = parts[0].partition("@")\n'
    '        if mention and mention.lower() != self.username:\n'
    '            return\n'
    '        command = command.lstrip("/").replace("_", "-").lower()\n'
    '        if command.startswith("us-macroanalysis-"):\n'
    '            command = command[len("us-macroanalysis-") :]\n'
    '        elif command.startswith("us-macro-"):\n'
    '            command = command[len("us-macro-") :]\n'
    '        elif command.startswith("us-single-stock-"):\n'
    '            command = "us-stock-" + command[len("us-single-stock-") :]\n'
    '        if not text.startswith("/") and command not in (\n'
    '            *STAGE_COMMANDS,\n'
    '            "cot",\n'
    '            "cot-import",\n'
    '            "cot-status",\n'
    '            "us-cot-import",\n'
    '            "us-cot-status",\n'
    '            "status",\n'
    '            "logs",\n'
    '            "stop",\n'
    '            "help",\n'
    '            "pythonstopall",\n'
    '            "backup",\n'
    '            "fund-backup",\n'
    '            "fund-server-backup",\n'
    '            *SINGLE_STOCK_COMMANDS,\n'
    '            "us-stock-status",\n'
    '            "us-stock-logs",\n'
    '            "us-stock-stop",\n'
    '        ):\n'
    '            return\n'
    '        if time.time() - message.get("date", 0) > 120:\n'
    '            send_telegram(\n'
    '                "This command expired while the listener was offline. Send it again.",\n'
    '                "Telegram control",\n'
    '            )\n'
    '            return\n'
    '        if len(parts) != 1:\n'
    '            send_telegram("Commands take no arguments.\\n" + HELP, "Telegram control")\n'
    '            return\n'
    '        try:\n'
    '            if command in ("help", "start"):\n'
    '                reply = HELP\n'
    '            elif command == "status":\n'
    '                reply = status()\n'
    '            elif command in STAGE_COMMANDS:\n'
    '                if time.monotonic() - self.last_launch < 30 or busy():\n'
    '                    reply = "Already running or recently requested.\\n" + status()\n'
    '                else:\n'
    '                    start_pipeline(STAGE_COMMANDS[command])\n'
    '                    self.last_launch = time.monotonic()\n'
    '                    return  # The runner sends the single start notification.\n'
    '            elif command in SINGLE_STOCK_COMMANDS:\n'
    '                if time.monotonic() - self.last_single_stock_launch < 30 or single_stock_busy():\n'
    '                    reply = "Already running or recently requested.\\n" + single_stock_status()\n'
    '                else:\n'
    '                    start_single_stock_pipeline(SINGLE_STOCK_COMMANDS[command])\n'
    '                    self.last_single_stock_launch = time.monotonic()\n'
    '                    return  # The runner sends the start and completion reports.\n'
    '            elif command == "us-stock-status":\n'
    '                reply = single_stock_status()\n'
    '            elif command == "us-stock-stop":\n'
    '                reply = stop_single_stock_pipeline()\n'
    '            elif command == "us-stock-logs":\n'
    '                self.toggle_logs("single-stock")\n'
    '                return\n'
    '            elif command in ("cot", "cot-import", "us-cot-import"):\n'
    '                if time.monotonic() - self.last_cot_launch < 30 or cot_busy():\n'
    '                    reply = "Already running or recently requested.\\n" + cot_status()\n'
    '                else:\n'
    '                    start_cot_import()\n'
    '                    self.last_cot_launch = time.monotonic()\n'
    '                    return  # The COT runner sends the start and completion reports.\n'
    '            elif command in ("cot-status", "us-cot-status"):\n'
    '                reply = cot_status()\n'
    '            elif command in ("backup", "fund-backup", "fund-server-backup"):\n'
    '                if time.monotonic() - self.last_backup_launch < 30 or backup_busy():\n'
    '                    reply = "Backup is already running or was requested recently. Completion or failure will be reported here."\n'
    '                else:\n'
    '                    start_backup()\n'
    '                    self.last_backup_launch = time.monotonic()\n'
    '                    return  # The backup script sends the completion/failure notice.\n'
    '            elif command == "stop":\n'
    '                reply = stop_pipeline()\n'
    '            elif command == "logs":\n'
    '                self.toggle_logs()\n'
    '                return\n'
    '            elif command == "pythonstopall":\n'
    '                if not is_admin():\n'
    '                    reply = "Stopping every Python process requires the administrator listener setup. Run C:\\\\Server\\\\Scripts\\\\install_telegram_startup.ps1 in administrator PowerShell, then restart the listener task. No processes were stopped."\n'
    '                else:\n'
    '                    if send_telegram(\n'
    '                        "Stopping ALL Python processes on this server, including this bot. Partial writes are not rolled back. The supervisor will restart the bot; pipeline runs are not restarted.",\n'
    '                        "Python stop all",\n'
    '                    ):\n'
    '                        stop_all_python()\n'
    '                    return\n'
    '            else:\n'
    '                reply = "Unknown command.\\n" + HELP\n'
    '        except Exception:\n'
    '            reply = "Could not complete this command. Check the server listener and Task Scheduler. No extra run was queued; use /status before retrying."\n'
    '        send_telegram(reply, "Telegram control")\n'
    '\n'
    '    def toggle_logs(self, source="macro"):\n'
    '        if self.log_watch is not None:\n'
    '            self.log_watch = None\n'
    '            send_telegram("Live log watching stopped.", "Pipeline logs")\n'
    '            return\n'
    '        label = "single-stock" if source == "single-stock" else "pipeline"\n'
    '        tail = single_stock_log_tail if source == "single-stock" else log_tail\n'
    '        text = f"Live {label} log — updates every 30 seconds with no time limit. Send this logs command again to stop.\\n\\n" + tail()\n'
    '        sent = api("sendMessage", chat_id=self.chat_id, text=text)\n'
    '        self.log_watch = {\n'
    '            "message_id": sent["message_id"],\n'
    '            "next": time.monotonic() + 30,\n'
    '            "text": text,\n'
    '            "source": source,\n'
    '        }\n'
    '\n'
    '    def update_logs(self):\n'
    '        watch = self.log_watch\n'
    '        if watch is None or time.monotonic() < watch["next"]:\n'
    '            return\n'
    '        watch["next"] = time.monotonic() + 30\n'
    '        single_stock = watch.get("source") == "single-stock"\n'
    '        label = "single-stock" if single_stock else "pipeline"\n'
    '        tail = single_stock_log_tail if single_stock else log_tail\n'
    '        header = f"Live {label} log — updates every 30 seconds with no time limit. Send this logs command again to stop."\n'
    '        text = header + "\\n\\n" + tail()\n'
    '        if text != watch["text"]:\n'
    '            api(\n'
    '                "editMessageText",\n'
    '                chat_id=self.chat_id,\n'
    '                message_id=watch["message_id"],\n'
    '                text=text,\n'
    '            )\n'
    '            watch["text"] = text\n'
    '\n'
    '\n'
    'def main():\n'
    '    if sys.platform != "win32":\n'
    '        raise RuntimeError("This controller requires Windows Task Scheduler")\n'
    '    chat = credential("TELEGRAM_CHAT_ID")\n'
    '    raw_users = credential("TELEGRAM_ALLOWED_USER_IDS")\n'
    '    if not credential("TELEGRAM_BOT_TOKEN") or not chat or not raw_users:\n'
    '        raise RuntimeError("Configure TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID and TELEGRAM_ALLOWED_USER_IDS")\n'
    '    users = parse_allowed_user_ids(raw_users)\n'
    '    WORK.mkdir(parents=True, exist_ok=True)\n'
    '    with (WORK / "telegram-listener.lock").open("a+") as lock:\n'
    '        acquire_lock(lock)\n'
    '        me = api("getMe")\n'
    '        if api("getWebhookInfo").get("url"):\n'
    '            raise RuntimeError(\n'
    '                "This bot has a webhook; use a dedicated bot for the pipeline"\n'
    '            )\n'
    '        task_state()  # Fail before accepting commands if this account cannot read the task.\n'
    '        backup_task_state()\n'
    '        controller = Controller(chat, users, me["username"])\n'
    '        if controller.offset is None:\n'
    '            updates = api(\n'
    '                "getUpdates", offset=-1, timeout=0, allowed_updates=["message"]\n'
    '            )\n'
    '            controller.checkpoint(updates[-1]["update_id"] + 1 if updates else 0)\n'
    '        api(\n'
    '            "setMyCommands",\n'
    '            commands=MENU,\n'
    '            scope={"type": "chat", "chat_id": chat},\n'
    '        )\n'
    '        print("Telegram controller ready; restricted commands enabled", flush=True)\n'
    '        while True:\n'
    '            try:\n'
    '                updates = api(\n'
    '                    "getUpdates",\n'
    '                    offset=controller.offset,\n'
    '                    timeout=5 if controller.log_watch else 25,\n'
    '                    allowed_updates=["message"],\n'
    '                )\n'
    '                for update in updates:\n'
    '                    controller.handle(update)\n'
    '                controller.update_logs()\n'
    '                save_json(\n'
    '                    WORK / "telegram-health.json",\n'
    '                    {"last_poll": time.time(), "pid": os.getpid()},\n'
    '                )\n'
    '            except Exception:\n'
    '                # Never log raw HTTP exceptions: the request URL contains the token.\n'
    '                print("Telegram poll failed; retrying in 10 seconds", flush=True)\n'
    '                time.sleep(10)\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    try:\n'
    '        main()\n'
    '    except Exception:\n'
    '        print(\n'
    '            "Telegram controller stopped; check configuration, task access and logs",\n'
    '            file=sys.stderr,\n'
    '        )\n'
    '        raise SystemExit(1)\n'
)

# Embedded source: tests/test_calculations.py
SOURCES['tests/test_calculations.py'] = (
    'import sys\n'
    'from pathlib import Path\n'
    '\n'
    'sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import pytest\n'
    'from config import WORKBOOKS, window_value\n'
    'from validation import (\n'
    '    source_frame,\n'
    '    pair_definitions,\n'
    '    parse_date,\n'
    '    output_dates,\n'
    '    ValidationError,\n'
    ')\n'
    'from correlation import rolling_correlation, fisher_transform\n'
    'from spread import spread_series, rolling_statistics, spread_zscore\n'
    'from pipeline import calculate, comparison\n'
    '\n'
    '\n'
    'def test_exact_headers_and_missing_mapping():\n'
    '    frame, _ = source_frame([["Date", "X", "Y"], ["2020-01-01", 1, 2]])\n'
    '    pairs, issues = pair_definitions(\n'
    '        [["Date", "X", "x", "X", ""], ["", "Y", "Y", "", ""]], frame\n'
    '    )\n'
    '    assert pairs[0][2] is None\n'
    '    assert len(issues) == 3\n'
    '\n'
    '\n'
    'def test_duplicate_header_invalidates_mapping():\n'
    '    frame, issues = source_frame([["Date", "X", "X", "Y"], ["2020-01-01", 1, 2, 3]])\n'
    '    assert list(frame.columns) == ["Y"] and issues\n'
    '    assert pair_definitions([["", "X"], ["", "Y"]], frame)[0][0][2]\n'
    '\n'
    '\n'
    'def test_dates_are_actual_dates_and_source_sorted():\n'
    '    frame, _ = source_frame(\n'
    '        [\n'
    '            ["Date", "X", "Y"],\n'
    '            ["31/03/2020", 3, 1],\n'
    '            ["2020-01-31", 1, 3],\n'
    '            ["29/02/2020", 2, 2],\n'
    '        ]\n'
    '    )\n'
    '    assert list(frame["X"]) == [1, 2, 3]\n'
    '    assert parse_date(43861) == pd.Timestamp("2020-01-31")\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("value", ["01/02/03", "invalid", "2020-02-30", True, 43861.5])\n'
    'def test_bad_dates_fail(value):\n'
    '    with pytest.raises(ValidationError):\n'
    '        parse_date(value)\n'
    '\n'
    '\n'
    'def test_duplicate_dates_fail():\n'
    '    with pytest.raises(ValidationError):\n'
    '        source_frame([["Date", "X"], ["2020-01-01", 1], ["2020-01-01", 2]])\n'
    '\n'
    '\n'
    'def test_pearson_and_fisher_manual():\n'
    '    # Prior x=(1,2,3), y=(1,3,2): r=1/2.\n'
    '    result = rolling_correlation([1, 2, 3, 100], [1, 3, 2, -100], 3)\n'
    '    assert np.isnan(result[:3]).all()\n'
    '    assert result[3] == pytest.approx(0.5 * np.log(3))\n'
    '    assert fisher_transform([0, 0.5, -0.5])[1] == pytest.approx(np.arctanh(0.5))\n'
    '\n'
    '\n'
    'def test_perfect_and_constant_correlation_blank():\n'
    '    assert np.isnan(rolling_correlation([1, 2, 3, 4], [2, 4, 6, 8], 3)).all()\n'
    '    assert np.isnan(rolling_correlation([1, 1, 1, 1], [2, 4, 6, 8], 3)).all()\n'
    '    assert np.isnan(fisher_transform([1, -1, 1.1, np.inf])).all()\n'
    '\n'
    '\n'
    'def test_directional_spread_and_aligned_series():\n'
    '    np.testing.assert_equal(spread_series([1, 2, np.nan], [3, 1, 5]), [-2, 1, np.nan])\n'
    '\n'
    '\n'
    'def test_sample_mean_std_and_zscore_manual():\n'
    '    spread = [1, 2, 3, 5]\n'
    '    mean, std = rolling_statistics(spread, 3)\n'
    '    assert mean[-1] == 2 and std[-1] == 1\n'
    '    assert spread_zscore(spread, [0, 0, 0, 0], 3)[-1] == 3\n'
    '\n'
    '\n'
    'def test_missing_values_are_skipped_in_observation_windows():\n'
    '    x = [1, 2, np.nan, 4, 5, 6, 7]\n'
    '    result = spread_zscore(x, np.zeros(7), 3)\n'
    '    assert np.isnan(result[:4]).all()\n'
    '    assert result[4] == pytest.approx(\n'
    '        (5 - np.mean([1, 2, 4])) / np.std([1, 2, 4], ddof=1)\n'
    '    )\n'
    '    assert result[-1] == pytest.approx(2)\n'
    '    c = rolling_correlation(x, [1, 3, 2, 6, 4, 8, 5], 3)\n'
    '    assert np.isnan(c[:4]).all()\n'
    '    assert c[4] == pytest.approx(np.arctanh(np.corrcoef([1, 2, 4], [1, 3, 6])[0, 1]))\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("kind", ["correlation", "spread"])\n'
    'def test_missing_current_input_blanks_only_affected_relationship(kind):\n'
    '    dates = pd.date_range("2026-01-31", periods=8, freq="ME")\n'
    '    x = [1, 2, 4, 3, 5, 7, 6, 8]\n'
    '    y = [2, 5, 1, 4, "", 3, 6, 2]\n'
    '    z = [3, 1, 5, 2, 6, 4, 7, 8]\n'
    '    source = [["Date", "X", "Y", "Z"]] + [\n'
    '        [str(d.date()), a, b, c] for d, a, b, c in zip(dates, x, y, z)\n'
    '    ]\n'
    '    output = [["", "X", "X"], ["Date", "Y", "Z"]] + [\n'
    '        [str(d.date())] for d in dates\n'
    '    ]\n'
    '    book = next(b for b in WORKBOOKS if b.kind == kind)\n'
    '    values, _ = calculate(book, source, output, [[book.control_key, 3]])\n'
    '    assert values[4][0] == ""\n'
    '    assert values[4][1] != ""\n'
    '    assert values[5][0] != ""\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("kind", ["correlation", "spread"])\n'
    'def test_blank_is_missing_but_numeric_zero_is_a_valid_pair_value(kind):\n'
    '    source = [\n'
    '        ["Date", "X", "Y"],\n'
    '        ["2026-01-31", 1, 2],\n'
    '        ["2026-02-28", 2, 5],\n'
    '        ["2026-03-31", 4, 1],\n'
    '        ["2026-04-30", 3, ""],\n'
    '    ]\n'
    '    output = [["", "X"], ["Date", "Y"]] + [[r[0]] for r in source[1:]]\n'
    '    book = next(b for b in WORKBOOKS if b.kind == kind)\n'
    '    frame, _ = source_frame(source)\n'
    '    assert np.isnan(frame.loc["2026-04-30", "Y"])\n'
    '    blank, _ = calculate(book, source, output, [[book.control_key, 3]])\n'
    '    assert blank[-1] == [""]\n'
    '    source[-1][2] = 0\n'
    '    frame, _ = source_frame(source)\n'
    '    assert frame.loc["2026-04-30", "Y"] == 0\n'
    '    zero, _ = calculate(book, source, output, [[book.control_key, 3]])\n'
    '    assert zero[-1][0] != ""\n'
    '\n'
    '\n'
    'def test_numeric_strings_bools_and_errors_are_missing():\n'
    '    frame, issues = source_frame(\n'
    '        [\n'
    '            ["Date", "X"],\n'
    '            ["2020-01-01", 0],\n'
    '            ["2020-02-01", "2"],\n'
    '            ["2020-03-01", True],\n'
    '            ["2020-04-01", "#N/A"],\n'
    '        ]\n'
    '    )\n'
    '    assert frame.iloc[0, 0] == 0 and frame.iloc[1:, 0].isna().all() and len(issues) == 3\n'
    '\n'
    '\n'
    'def test_insufficient_history_and_zero_std():\n'
    '    assert np.isnan(spread_zscore([1, 2], [0, 0], 3)).all()\n'
    '    assert np.isnan(spread_zscore([1, 1, 1, 5], [0, 0, 0, 0], 3)).all()\n'
    '\n'
    '\n'
    'def test_no_lookahead_correlation_and_spread():\n'
    '    x = np.array([1.0, 2, 3, 5, 4, 9])\n'
    '    y = np.array([1.0, 3, 2, 1, 5, 8])\n'
    '    initial = rolling_correlation(x, y, 3)\n'
    '    changed = x.copy()\n'
    '    changed[3:] = [999, 1000, -999]\n'
    '    assert rolling_correlation(changed, y, 3)[3] == pytest.approx(initial[3])\n'
    '    mu, sd = rolling_statistics(x - y, 3)\n'
    '    mu2, sd2 = rolling_statistics(changed - y, 3)\n'
    '    assert mu[3] == mu2[3] and sd[3] == sd2[3]\n'
    '    # Current spread affects only numerator at t.\n'
    '    assert spread_zscore(changed, y, 3)[3] == pytest.approx(\n'
    '        (changed[3] - y[3] - mu[3]) / sd[3]\n'
    '    )\n'
    '\n'
    '\n'
    'def fixture(n):\n'
    '    dates = (\n'
    '        pd.date_range("2020-01-01", periods=n, freq="MS").strftime("%Y-%m-%d").tolist()\n'
    '    )\n'
    '    source = [["Date", "X", "Y"]] + [\n'
    '        [d, float(i * i), float(i % 3)] for i, d in enumerate(dates)\n'
    '    ]\n'
    '    output = [["", "X"], ["Date", "Y"]] + [[d] for d in dates]\n'
    '    return source, output\n'
    '\n'
    '\n'
    'def test_append_data_automatically_extends_history():\n'
    '    book = WORKBOOKS[1]\n'
    '    source, output = fixture(6)\n'
    '    first, _ = calculate(book, source, output, [[book.control_key, 3]])\n'
    '    source2, output2 = fixture(7)\n'
    '    extended, _ = calculate(book, source2, output2, [[book.control_key, 3]])\n'
    '    assert len(extended) == 7 and first == extended[:6] and extended[-1][0] != ""\n'
    '\n'
    '\n'
    'def test_output_is_joined_by_date_not_position():\n'
    '    book = WORKBOOKS[1]\n'
    '    source, output = fixture(6)\n'
    '    complete, _ = calculate(book, source, output, [[book.control_key, 3]])\n'
    '    # A2 contains earliest date (actual Spread Score layout).\n'
    '    shifted = [output[0], [source[1][0], "Y"]] + output[3:]\n'
    '    actual, _ = calculate(book, source, shifted, [[book.control_key, 3]])\n'
    '    assert actual == complete[1:]\n'
    '\n'
    '\n'
    'def test_output_missing_unsorted_or_duplicate_dates_fail():\n'
    '    source, output = fixture(5)\n'
    '    frame, _ = source_frame(source)\n'
    '    with pytest.raises(ValidationError):\n'
    '        output_dates(output[:-1], frame.index)\n'
    '    with pytest.raises(ValidationError):\n'
    '        output_dates(output[:2] + list(reversed(output[2:])), frame.index)\n'
    '    with pytest.raises(ValidationError):\n'
    '        output_dates(output + [output[-1]], frame.index)\n'
    '\n'
    '\n'
    'def test_control_window_changes_calculations():\n'
    '    source, output = fixture(6)\n'
    '    book = WORKBOOKS[1]\n'
    '    a, _ = calculate(book, source, output, [[book.control_key, 3]])\n'
    '    b, _ = calculate(book, source, output, [[book.control_key, 4]])\n'
    '    assert a[3][0] != "" and b[3][0] == ""\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("v", [1, 0, -1, 2.5, True, "oops", float("inf")])\n'
    'def test_bad_window(v):\n'
    '    with pytest.raises(ValueError):\n'
    '        window_value([["Window", v]], "Window")\n'
    '\n'
    '\n'
    'def test_duplicate_setting_rejected():\n'
    '    with pytest.raises(ValueError):\n'
    '        window_value([["Window", 3], ["Window", 3]], "Window")\n'
    '\n'
    '\n'
    'def test_missing_mapping_blanks_only_affected_pair():\n'
    '    source, output = fixture(6)\n'
    '    output[0].append("missing")\n'
    '    output[1].append("Y")\n'
    '    result, report = calculate(\n'
    '        WORKBOOKS[1], source, output, [[WORKBOOKS[1].control_key, 3]]\n'
    '    )\n'
    '    assert all(row[1] == "" for row in result)\n'
    '    assert result[-1][0] != "" and report["invalid_indicator_mappings"] == 1\n'
    '\n'
    '\n'
    'def test_upstream_write_prohibited_before_network():\n'
    '    from google_sheets import Sheets\n'
    '\n'
    '    api = Sheets.__new__(Sheets)\n'
    '    with pytest.raises(ValueError, match="prohibited"):\n'
    '        api.request(\n'
    '            "POST",\n'
    '            "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c",\n'
    '            ":batchUpdate",\n'
    '            json={},\n'
    '        )\n'
    '\n'
    '\n'
    'def test_new_tail_and_historical_blank_mismatch_distinguished():\n'
    '    old = np.array([[np.nan], [1.0], [np.nan]])\n'
    '    new = np.array([[2.0], [1.0], [3.0]])\n'
    '    c = comparison(old, new)\n'
    '    assert c["new_tail_numeric"] == 1 and c["historical_blank_mismatches"] == 1\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("window", [2, 3, 5, 24])\n'
    'def test_every_rolling_row_against_independent_prior_pair_oracle(window):\n'
    '    import statistics\n'
    '\n'
    '    rng = np.random.default_rng(42)\n'
    '    x = rng.normal(size=60)\n'
    '    y = 0.3 * x + rng.normal(size=60)\n'
    '    x[[1, 4, 6, 20, 39]] = np.nan\n'
    '    y[[2, 5, 16, 29, 45]] = np.nan\n'
    '    c = rolling_correlation(x, y, window)\n'
    '    mu, sd = rolling_statistics(x - y, window)\n'
    '    z = spread_zscore(x, y, window)\n'
    '    for t in range(len(x)):\n'
    '        pairs = [\n'
    '            (x[i], y[i]) for i in range(t) if np.isfinite(x[i]) and np.isfinite(y[i])\n'
    '        ][-window:]\n'
    '        if len(pairs) < window:\n'
    '            assert np.isnan([c[t], mu[t], sd[t], z[t]]).all()\n'
    '            continue\n'
    '        xx, yy = zip(*pairs)\n'
    '        h = [a - b for a, b in pairs]\n'
    '        if window > 2 and np.isfinite(x[t]) and np.isfinite(y[t]):\n'
    '            assert c[t] == pytest.approx(\n'
    '                np.arctanh(statistics.correlation(xx, yy))\n'
    '            )\n'
    '        else:\n'
    '            assert np.isnan(c[t])\n'
    '        assert mu[t] == pytest.approx(statistics.mean(h))\n'
    '        assert sd[t] == pytest.approx(statistics.stdev(h))\n'
    '        if np.isfinite(x[t]) and np.isfinite(y[t]):\n'
    '            assert z[t] == pytest.approx(\n'
    '                (x[t] - y[t] - statistics.mean(h)) / statistics.stdev(h)\n'
    '            )\n'
    '        else:\n'
    '            assert np.isnan(z[t])\n'
    '        # Changing current and future data must not change ANY historical statistic at t.\n'
    '        changed_x = x.copy()\n'
    '        changed_y = y.copy()\n'
    '        changed_x[t + 1 :] = 1000\n'
    '        changed_y[t + 1 :] = -1000\n'
    '        np.testing.assert_allclose(\n'
    '            rolling_correlation(changed_x, changed_y, window)[: t + 1],\n'
    '            c[: t + 1],\n'
    '            equal_nan=True,\n'
    '        )\n'
    '        mm, ss = rolling_statistics(changed_x - changed_y, window)\n'
    '        np.testing.assert_allclose(mm[: t + 1], mu[: t + 1], equal_nan=True)\n'
    '        np.testing.assert_allclose(ss[: t + 1], sd[: t + 1], equal_nan=True)\n'
    '\n'
    '\n'
    'def test_quarterly_observations_in_monthly_grid_and_gaps():\n'
    '    dates = pd.date_range("2010-01-31", periods=85, freq="ME")\n'
    '    rows = [["Date", "X", "Y"]]\n'
    '    k = 0\n'
    '    for i, d in enumerate(dates):\n'
    '        if i % 3 == 2 and i != 20:\n'
    '            k += 1\n'
    '            rows.append([str(d.date()), k * k, k % 4])\n'
    '        else:\n'
    '            rows.append([str(d.date()), "", ""])\n'
    '    output = [["", "X"], ["Date", "Y"]] + [[r[0]] for r in rows[1:]]\n'
    '    result, report = calculate(\n'
    '        WORKBOOKS[1], rows, output, [[WORKBOOKS[1].control_key, 24]]\n'
    '    )\n'
    '    valid = [r for r in rows[1:] if r[1] != ""]\n'
    '    assert len(valid) > 24\n'
    '    first_date = valid[24][0]\n'
    '    assert report["first_calculated_date"] == first_date\n'
    '    h = [r[1] - r[2] for r in valid[:24]]\n'
    '    pos = next(i for i, r in enumerate(rows[1:]) if r[0] == first_date)\n'
    '    assert result[pos][0] == pytest.approx(\n'
    '        (valid[24][1] - valid[24][2] - np.mean(h)) / np.std(h, ddof=1)\n'
    '    )\n'
    '    # Removing blank months entirely gives the same quarterly-date scores.\n'
    '    compact = [rows[0]] + valid\n'
    '    compact_output = output[:2] + [[r[0]] for r in valid]\n'
    '    expected, _ = calculate(\n'
    '        WORKBOOKS[1], compact, compact_output, [[WORKBOOKS[1].control_key, 24]]\n'
    '    )\n'
    '    actual = [result[i] for i, r in enumerate(rows[1:]) if r[1] != ""]\n'
    '    assert actual == expected\n'
)

# Embedded source: tests/test_change_notifications.py
SOURCES['tests/test_change_notifications.py'] = (
    'import json\n'
    'import os\n'
    'import subprocess\n'
    'import sys\n'
    '\n'
    'import numpy as np\n'
    'import pytest\n'
    '\n'
    'from notifications import changes, lifecycle\n'
    '\n'
    '\n'
    '@pytest.fixture\n'
    'def collecting(monkeypatch):\n'
    '    monkeypatch.delenv("US_PIPELINE_CHANGE_REPORT", raising=False)\n'
    '    token = changes.begin()\n'
    '    try:\n'
    '        yield\n'
    '    finally:\n'
    '        changes.end(token)\n'
    '\n'
    '\n'
    'def test_compare_distinguishes_new_revised_removed_and_unchanged():\n'
    '    assert changes.compare([[1, "", 5, 4]], [["1", 3, "", 6]]) == {\n'
    '        "added": 1,\n'
    '        "updated": 1,\n'
    '        "removed": 1,\n'
    '    }\n'
    '    assert changes.compare([[1.0, None]], [[1.00000000001, ""]]) == {\n'
    '        "added": 0,\n'
    '        "updated": 0,\n'
    '        "removed": 0,\n'
    '    }\n'
    '\n'
    '\n'
    'def test_subprocess_journal_is_read_by_parent(tmp_path, collecting):\n'
    '    journal = tmp_path / "changes.jsonl"\n'
    '    environment = dict(os.environ, US_PIPELINE_CHANGE_REPORT=str(journal))\n'
    '    subprocess.run(\n'
    '        [\n'
    '            sys.executable,\n'
    '            "-c",\n'
    '            "from notifications.changes import record; record(\'Source data\', \'GDP\', added=2)",\n'
    '        ],\n'
    '        env=environment,\n'
    '        check=True,\n'
    '    )\n'
    '    assert changes.events(journal)[0]["added"] == 2\n'
    '    assert changes.events() == []\n'
    '\n'
    '\n'
    'def test_unknown_is_not_reported_as_unchanged_or_refreshed():\n'
    '    text = "\\n".join(\n'
    '        changes.describe([{"category": "Analysis", "name": "GDP", "unknown": True}])\n'
    '    )\n'
    '    assert "without a reliable change comparison" in text\n'
    '    assert "no value changes" not in text\n'
    '    assert "refreshed" not in text\n'
    '\n'
    '\n'
    'def test_only_start_and_end_with_real_metrics(monkeypatch, tmp_path):\n'
    '    monkeypatch.delenv("US_PIPELINE_NOTIFICATIONS_MANAGED", raising=False)\n'
    '    monkeypatch.delenv("US_PIPELINE_CHANGE_REPORT", raising=False)\n'
    '    calls = []\n'
    '    now = [0]\n'
    '    monkeypatch.setattr(lifecycle.time, "monotonic", lambda: now[0])\n'
    '    monkeypatch.setattr(\n'
    '        lifecycle,\n'
    '        "send_telegram",\n'
    '        lambda text, title, **kw: calls.append((title, text)),\n'
    '    )\n'
    '\n'
    '    def action():\n'
    '        lifecycle.set_run_details(log_dir=tmp_path, stages=["Import", "Analysis"])\n'
    '        lifecycle.stage_event(1, 2, "Import", "started")\n'
    '        changes.record("Source data", "GDP", added=3, updated=1)\n'
    '        lifecycle.stage_event(1, 2, "Import", "finished", seconds=100)\n'
    '        lifecycle.stage_event(2, 2, "Analysis", "started")\n'
    '        changes.record("Indicator analysis", "GDP")\n'
    '        lifecycle.stage_event(2, 2, "Analysis", "finished", seconds=95)\n'
    '        now[0] = 195\n'
    '        return 0\n'
    '\n'
    '    assert lifecycle.run_notified("Pipeline", action, argv=["--dataset", "all"]) == 0\n'
    '    assert len(calls) == 2\n'
    '    assert "Entire US pipeline" in calls[0][1]\n'
    '    assert "Duration: 3m 15s" in calls[1][1]\n'
    '    assert "3 added, 1 updated values" in calls[1][1]\n'
    '    assert "Affected: GDP" in calls[1][1]\n'
    '    assert "no value changes detected in tracked ranges" in calls[1][1]\n'
    '    assert "Stages completed: 2/2" in calls[1][1]\n'
    '\n'
    '\n'
    'def test_analysis_tracks_changed_values_after_successful_write(collecting):\n'
    '    import us_indicator_analysis as analysis\n'
    '\n'
    '    class Workbook:\n'
    '        def values_batch_get(self, ranges, params):\n'
    '            return {"valueRanges": [{"values": [[1, "", 8]]} for _ in ranges]}\n'
    '\n'
    '        def values_batch_update(self, payload):\n'
    '            return {}\n'
    '\n'
    '    assert (\n'
    '        analysis.batch_update_values(\n'
    '            Workbook(), [{"range": "\'GDP\'!C2:E2", "values": [[1, 4, 9]]}]\n'
    '        )\n'
    '        == 1\n'
    '    )\n'
    '    result = changes.events()[0]\n'
    '    assert result["added"] == 1 and result["updated"] == 1\n'
    '\n'
    '\n'
    'def test_comparison_read_failure_does_not_prevent_write(collecting):\n'
    '    import us_indicator_analysis as analysis\n'
    '\n'
    '    calls = []\n'
    '\n'
    '    class Workbook:\n'
    '        def values_batch_get(self, ranges, params):\n'
    '            raise ValueError("Comparison unavailable")\n'
    '\n'
    '        def values_batch_update(self, payload):\n'
    '            calls.append(payload)\n'
    '\n'
    '    analysis.batch_update_values(Workbook(), [{"range": "\'GDP\'!C2", "values": [[1]]}])\n'
    '    assert len(calls) == 1\n'
    '    assert changes.events()[0]["unknown"] is True\n'
    '\n'
    '\n'
    'def test_failed_write_does_not_claim_successful_changes(collecting, monkeypatch):\n'
    '    import us_indicator_analysis as analysis\n'
    '\n'
    '    monkeypatch.setattr(analysis, "call_with_backoff", lambda fn, **kw: fn())\n'
    '\n'
    '    class Workbook:\n'
    '        def values_batch_get(self, ranges, params):\n'
    '            return {"valueRanges": [{"values": [[1]]}]}\n'
    '\n'
    '        def values_batch_update(self, payload):\n'
    '            raise ValueError("Write failed")\n'
    '\n'
    '    with pytest.raises(ValueError):\n'
    '        analysis.batch_update_values(\n'
    '            Workbook(), [{"range": "\'GDP\'!C2", "values": [[2]]}]\n'
    '        )\n'
    '    assert changes.events() == []\n'
    '\n'
    '\n'
    'def test_relationship_diff_counts_actual_additions_and_removals():\n'
    '    from pipeline import comparison\n'
    '\n'
    '    diff = comparison(np.array([[1.0, np.nan, 3.0]]), np.array([[2.0, 4.0, np.nan]]))\n'
    '    assert diff["numeric_mismatches"] == 1\n'
    '    assert diff["added_numeric"] == 1\n'
    '    assert diff["removed_numeric"] == 1\n'
    '\n'
    '\n'
    'def test_import_preview_does_not_claim_changes(collecting):\n'
    '    from monthly_indicators import print_sync_summary, sync_result_row\n'
    '\n'
    '    result = sync_result_row("GDP", "GDP", None, 1, 2, None, "DRY RUN: would append")\n'
    '    print_sync_summary([result], validate_only=True)\n'
    '    assert changes.events() == []\n'
    '\n'
    '\n'
    'def test_import_backfill_is_reported_as_an_added_value(collecting):\n'
    '    from monthly_indicators import print_sync_summary, sync_result_row\n'
    '\n'
    '    result = sync_result_row(\n'
    '        "Central bank liquidity growth",\n'
    '        "Central bank liquidity growth",\n'
    '        None,\n'
    '        2,\n'
    '        0,\n'
    '        None,\n'
    '        source_change_counts={"added": 2, "updated": 0, "removed": 0},\n'
    '    )\n'
    '    print_sync_summary([result])\n'
    '    assert changes.events() == [\n'
    '        {\n'
    '            "category": "Source data",\n'
    '            "name": "Central bank liquidity growth",\n'
    '            "added": 2,\n'
    '            "updated": 0,\n'
    '            "removed": 0,\n'
    '            "unknown": False,\n'
    '            "note": "",\n'
    '        }\n'
    '    ]\n'
    '\n'
    '\n'
    'def test_unwritable_journal_does_not_block_calculations(tmp_path, monkeypatch):\n'
    '    monkeypatch.setenv(\n'
    '        "US_PIPELINE_CHANGE_REPORT", str(tmp_path / "missing" / "changes.jsonl")\n'
    '    )\n'
    '    changes.record("Source data", "GDP", added=1)\n'
)

# Embedded source: tests/test_complete_pipeline.py
SOURCES['tests/test_complete_pipeline.py'] = (
    'import json\n'
    'import subprocess\n'
    'import sys\n'
    'from pathlib import Path\n'
    'import pytest\n'
    'import run_us_pipeline as runner\n'
    'from config import BASE_WORKBOOKS\n'
    'from source_freshness import wait_for_macro\n'
    'from validation import ValidationError\n'
    '\n'
    '\n'
    'def test_run_order_and_isolated_selectors():\n'
    '    args = runner.parse_args([])\n'
    '    commands = runner.build_commands(args)\n'
    '    assert [Path(cmd[1]).name for _, cmd in commands] == [\n'
    '        "monthly_indicators.py",\n'
    '        "us_indicator_analysis.py",\n'
    '        "run_relationships.py",\n'
    '    ]\n'
    '    assert all(\n'
    '        cmd[0] == sys.executable and Path(cmd[1]).parent == runner.ROOT\n'
    '        for _, cmd in commands\n'
    '    )\n'
    '    for stage in runner.STAGES:\n'
    '        commands = runner.build_commands(runner.parse_args(["--stage", stage]))\n'
    '        assert len(commands) == 1\n'
    '        if stage not in ("importer", "analysis", "relationships"):\n'
    '            assert commands[0][1][-1] == "--" + stage\n'
    '\n'
    '\n'
    'def test_failure_stops_later_stages(tmp_path, monkeypatch):\n'
    '    monkeypatch.setattr("notifications.lifecycle.send_telegram", lambda *a, **kw: True)\n'
    '    credential = tmp_path / "credentials.json"\n'
    '    credential.write_text("{}")\n'
    '    monkeypatch.setattr(runner, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(runner, "dependencies", lambda: None)\n'
    '    seen = []\n'
    '\n'
    '    def fail(index, total, name, *args):\n'
    '        seen.append(name)\n'
    '        raise RuntimeError("test importer failure")\n'
    '\n'
    '    monkeypatch.setattr(runner, "run_stage", fail)\n'
    '    assert runner.main(["--credentials", str(credential), "--api-key", "test"]) == 1\n'
    '    assert seen == ["Macro importer"]\n'
    '    report = json.loads(next((tmp_path / "work").glob("*/summary.json")).read_text())\n'
    '    assert report["status"] == "failed"\n'
    '    assert "test" not in json.dumps(report.get("credentials", {}))\n'
    '\n'
    '\n'
    'class Imports:\n'
    '    def __init__(self, upstream, imported, origin=None):\n'
    '        self.upstream = upstream\n'
    '        self.imported = imported\n'
    '        self.reads = 0\n'
    '        self.origin = origin or "19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c"\n'
    '\n'
    '    def values(self, id, ranges, formulas=False):\n'
    '        if formulas:\n'
    '            return [[[f\'=IMPORTRANGE("{self.origin}","\\\'Coincident Score\\\'!A1:AL")\']]]\n'
    '        self.reads += 1\n'
    '        return [\n'
    '            self.imported if id == BASE_WORKBOOKS[0].spreadsheet_id else self.upstream\n'
    '        ]\n'
    '\n'
    '\n'
    'def test_freshness_checks_values_not_just_last_date():\n'
    '    upstream = [["Date", "X"], [45000, 1.2]]\n'
    '    api = Imports(upstream, [["Date", "X"], [45000, 0.8]])\n'
    '    with pytest.raises(ValidationError, match="stale"):\n'
    '        wait_for_macro(api, BASE_WORKBOOKS[0], attempts=1)\n'
    '    api.imported = [["Date", "X", ""], [45000, 1.2], []]\n'
    '    assert wait_for_macro(api, BASE_WORKBOOKS[0], attempts=1) == api.imported\n'
    '\n'
    '\n'
    'def test_wrong_macro_workbook_is_rejected():\n'
    '    with pytest.raises(ValidationError, match="different macro"):\n'
    '        wait_for_macro(Imports([[1]], [[1]], "wrong-id"), BASE_WORKBOOKS[0], attempts=1)\n'
    '\n'
    '\n'
    'def test_removed_preview_flag_is_rejected():\n'
    '    with pytest.raises(SystemExit):\n'
    '        runner.parse_args(["--dry-run"])\n'
    '\n'
    '\n'
    'def test_lock_rejects_second_process(tmp_path):\n'
    '    from run_lock import acquire_lock\n'
    '\n'
    '    with (tmp_path / "lock").open("a+") as handle:\n'
    '        acquire_lock(handle)\n'
    '        code = (\n'
    '            "from run_lock import acquire_lock; from pathlib import Path; f=Path("\n'
    '            + repr(str(tmp_path / "lock"))\n'
    '            + \').open("a+"); acquire_lock(f)\'\n'
    '        )\n'
    '        result = subprocess.run(\n'
    '            [sys.executable, "-c", code], cwd=runner.ROOT, capture_output=True\n'
    '        )\n'
    '        assert result.returncode != 0\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "stage",\n'
    '    [\n'
    '        "all",\n'
    '        "relationships",\n'
    '        "spread",\n'
    '        "correlation",\n'
    '        "momentum",\n'
    '        "spread-momentum",\n'
    '        "correlation-momentum",\n'
    '    ],\n'
    ')\n'
    'def test_dataset_forwarded_to_relationship_stage(stage):\n'
    '    commands = runner.build_commands(\n'
    '        runner.parse_args(["--stage", stage, "--dataset", "leading"])\n'
    '    )\n'
    '    command = commands[-1][1]\n'
    '    assert command[command.index("--dataset") + 1] == "leading"\n'
    '    assert command[1].endswith("run_relationships.py")\n'
)

# Embedded source: tests/test_datasets.py
SOURCES['tests/test_datasets.py'] = (
    'from dataclasses import replace\n'
    'from pathlib import Path\n'
    'import pytest\n'
    'from config import (\n'
    '    WORKBOOKS,\n'
    '    COINCIDENT_WORKBOOKS,\n'
    '    LEADING_WORKBOOKS,\n'
    '    LEADING_BASE_WORKBOOKS,\n'
    '    LEADING_MOMENTUM_WORKBOOKS,\n'
    '    select_workbooks,\n'
    '    upstream_book,\n'
    ')\n'
    'from google_sheets import Sheets, a1\n'
    'from momentum_pipeline import desired_formulas, assert_fresh\n'
    'from source_freshness import wait_for_macro, prepare_dataset_links\n'
    'from validation import ValidationError\n'
    '\n'
    'LEAD_IDS = {\n'
    '    "1sOkwKFe0d42NC7phUkGGOrhhwhDoff78rlQCL-1GUt0",\n'
    '    "1Ma0pirU2pmMSFgZXKB4v5y1gw9W8vzBI8xt6tPxiz90",\n'
    '    "1b-TWGIzqmzfG76ssiMAwROdCQQh7fhXEh2u4NTPRCvA",\n'
    '    "18uNC1bykf_a-F9tHTpgCgPk2e_Bw3TPMsJ6BQuZiL4g",\n'
    '}\n'
    '\n'
    '\n'
    'def test_exact_separate_user_supplied_workbooks_and_audit_keys():\n'
    '    assert {b.spreadsheet_id for b in LEADING_WORKBOOKS} == LEAD_IDS\n'
    '    assert len({b.spreadsheet_id for b in WORKBOOKS}) == 8\n'
    '    assert len({b.key for b in WORKBOOKS}) == 8\n'
    '    assert not LEAD_IDS & {b.spreadsheet_id for b in COINCIDENT_WORKBOOKS}\n'
    '    assert {b.macro_tab for b in LEADING_WORKBOOKS} == {"Leading Score"}\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "dataset,count", [("all", 8), ("coincident", 4), ("leading", 4)]\n'
    ')\n'
    'def test_selectors(dataset, count):\n'
    '    selected = select_workbooks(dataset)\n'
    '    assert len(selected) == count\n'
    '    assert all(dataset == "all" or b.dataset == dataset for b in selected)\n'
    '    for kind in ("spread", "correlation", "spread_momentum", "correlation_momentum"):\n'
    '        subset = select_workbooks(dataset, kind)\n'
    '        assert len(subset) == count // 4\n'
    '        assert all(b.kind == kind for b in subset)\n'
    '    assert len(select_workbooks(dataset, momentum_only=True)) == count // 2\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", LEADING_MOMENTUM_WORKBOOKS, ids=lambda b: b.key)\n'
    'def test_leading_momentum_never_imports_coincident_sources(book):\n'
    '    up = upstream_book(book)\n'
    '    assert up.dataset == book.dataset == "leading"\n'
    '    formulas = desired_formulas(book, 38, 667)\n'
    '    imports = [f for _, _, f in formulas if "IMPORTRANGE" in f]\n'
    '    assert imports and all(up.spreadsheet_id in f for f in imports)\n'
    '    assert not any(\n'
    '        old.spreadsheet_id in f for old in COINCIDENT_WORKBOOKS for f in imports\n'
    '    )\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", COINCIDENT_WORKBOOKS, ids=lambda b: b.key)\n'
    'def test_leading_only_write_client_rejects_every_coincident_book(book):\n'
    '    api = Sheets.__new__(Sheets)\n'
    '    api.write_ids = LEAD_IDS\n'
    '    with pytest.raises(ValueError, match="prohibited"):\n'
    '        api.request("POST", book.spreadsheet_id, ":batchUpdate", json={})\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", LEADING_BASE_WORKBOOKS, ids=lambda b: b.key)\n'
    'def test_correct_macro_id_but_wrong_dataset_is_rejected(book):\n'
    '    class Fake:\n'
    '        def values(self, *a, **kw):\n'
    '            return [\n'
    '                [\n'
    '                    [\n'
    '                        \'=IMPORTRANGE("19E_Za0DOHMY_9AFSPatK8Cp2vCQypI81QnB3Hz4ve9c","\\\'Coincident Score\\\'!A1:AL")\'\n'
    '                    ]\n'
    '                ]\n'
    '            ]\n'
    '\n'
    '    with pytest.raises(ValidationError, match="wrong score dataset"):\n'
    '        wait_for_macro(Fake(), book, attempts=1)\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", LEADING_MOMENTUM_WORKBOOKS, ids=lambda b: b.key)\n'
    'def test_momentum_freshness_reads_matching_leading_upstream(book):\n'
    '    up = upstream_book(book)\n'
    '    source = [["Date", "X"], [45000, 1.0]]\n'
    '    pairs = [["Date", "X"], [45000, "Y"]]\n'
    '\n'
    '    class Fake:\n'
    '        def values(self, id, ranges):\n'
    '            assert id == up.spreadsheet_id\n'
    '            return [source, pairs]\n'
    '\n'
    '    assert_fresh(Fake(), book, source, pairs)\n'
    '    with pytest.raises(ValidationError, match="stale"):\n'
    '        assert_fresh(Fake(), book, [["Date", "X"], [45000, 99.0]], pairs)\n'
    '\n'
    '\n'
    'class LinkSheets:\n'
    '    def __init__(self, book):\n'
    '        self.book = book\n'
    '        self.posts = []\n'
    '        self.backups = []\n'
    '        old = next(b for b in COINCIDENT_WORKBOOKS if b.kind == book.kind)\n'
    '        self.controls = [["Setting", "Value"], [book.control_key, 24]]\n'
    '        if book.kind.endswith("_momentum"):\n'
    '            self.controls += [\n'
    '                ["Momentum Lookback", 3],\n'
    '                ["Source Spreadsheet ID", upstream_book(old).spreadsheet_id],\n'
    '                ["Source Tab", upstream_book(old).output_tab],\n'
    '            ]\n'
    '        self.formulas = {\n'
    '            a1(book.output_tab, "A1"): [\n'
    '                [\'=IMPORTRANGE("old","\\\'Coincident Score\\\'!A1:A")\']\n'
    '            ]\n'
    '        }\n'
    '\n'
    '    def metadata(self, id):\n'
    '        assert id == self.book.spreadsheet_id\n'
    '        return {\n'
    '            "sheets": [\n'
    '                {\n'
    '                    "properties": {\n'
    '                        "title": self.book.output_tab,\n'
    '                        "sheetId": self.book.output_id,\n'
    '                    }\n'
    '                }\n'
    '            ]\n'
    '        }\n'
    '\n'
    '    def values(self, id, ranges, formulas=False):\n'
    '        assert id == self.book.spreadsheet_id\n'
    '        if ranges == [a1(self.book.control_tab, "A1:B20")]:\n'
    '            return [self.controls]\n'
    '        return [self.formulas.get(r, []) for r in ranges]\n'
    '\n'
    '    def backup(self, path, data):\n'
    '        self.backups.append((path, data))\n'
    '\n'
    '    def request(self, method, id, suffix, json):\n'
    '        assert method == "POST" and id in LEAD_IDS\n'
    '        self.posts.append(json)\n'
    '        for change in json["data"]:\n'
    '            self.formulas[change["range"]] = change["values"]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", LEADING_WORKBOOKS, ids=lambda b: b.key)\n'
    'def test_leading_link_repairs_are_scoped_and_backed_up(book, tmp_path, monkeypatch):\n'
    '    monkeypatch.setattr("source_freshness.wait_for_macro", lambda *a: None)\n'
    '    api = LinkSheets(book)\n'
    '    original = [row[:] for row in api.controls]\n'
    '    prepare_dataset_links(api, book, tmp_path)\n'
    '    assert api.backups and api.backups[0][0].name == book.key + "_links_before.json.gz"\n'
    '    assert api.controls == original  # numerical settings are never rewritten\n'
    '    writes = [r for batch in api.posts for r in batch["data"]]\n'
    '    if book.kind == "spread":\n'
    '        assert writes == [\n'
    '            {\n'
    '                "range": a1(book.output_tab, "A1"),\n'
    '                "values": [["=ARRAYFORMULA(\'Final scores data\'!A1:A)"]],\n'
    '            }\n'
    '        ]\n'
    '    elif book.kind.endswith("_momentum"):\n'
    '        assert any(\n'
    '            r["values"] == [[upstream_book(book).spreadsheet_id]] for r in writes\n'
    '        )\n'
    '        assert all(r["range"].startswith("\'Control Panel\'!B") for r in writes)\n'
    '    else:\n'
    '        assert not writes\n'
    '\n'
    '\n'
    'def test_existing_coincident_links_are_not_repaired(tmp_path):\n'
    '    class NoAccess:\n'
    '        def __getattr__(self, name):\n'
    '            raise AssertionError("Coincident setup should be unchanged")\n'
    '\n'
    '    for book in COINCIDENT_WORKBOOKS:\n'
    '        prepare_dataset_links(NoAccess(), book, tmp_path)\n'
    '\n'
    '\n'
    'def test_leading_cli_client_cannot_publish_to_original_books(tmp_path, monkeypatch):\n'
    '    monkeypatch.setattr("notifications.lifecycle.send_telegram", lambda *a, **kw: True)\n'
    '    import run_relationships as main\n'
    '\n'
    '    seen = {}\n'
    '\n'
    '    class Fake:\n'
    '        writes_attempted = False\n'
    '\n'
    '        def __init__(self, credentials, write_ids):\n'
    '            seen["ids"] = write_ids\n'
    '\n'
    '    def interrupt(sheets, book):\n'
    '        assert book.dataset == "leading"\n'
    '        raise KeyboardInterrupt\n'
    '\n'
    '    monkeypatch.setattr(main, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(main, "credentials_path", lambda *a: "unused")\n'
    '    monkeypatch.setattr("source_freshness.validate_macro_summary", lambda *a: None)\n'
    '    monkeypatch.setattr(main, "Sheets", Fake)\n'
    '    monkeypatch.setattr("source_freshness.prepare_dataset_links", lambda *a: None)\n'
    '    monkeypatch.setattr(main, "load", interrupt)\n'
    '    assert main.main(["--dataset", "leading"]) == 130\n'
    '    assert seen["ids"] == LEAD_IDS\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("expected", ["Coincident Score", "Leading Score"])\n'
    'def test_summary_selectors_reject_warning_mixup_and_wrong_indicator(expected):\n'
    '    from source_freshness import check_summary_selectors\n'
    '    from google_sheets import column_name\n'
    '\n'
    '    headers = ["Date"] + [f"Indicator {i}" for i in range(37)]\n'
    '    helpers = [[], []]\n'
    '    for i in range(37):\n'
    '        cell = column_name(40 + 2 * i)\n'
    '        helpers[0].extend(["=" + column_name(i + 2) + "1", ""])\n'
    '        helpers[1].extend(\n'
    '            [\n'
    '                f\'=LET(sheet,{cell}1,scoreCol,MATCH("{expected}",INDIRECT("header"),0),scoreCol)\',\n'
    '                "",\n'
    '            ]\n'
    '        )\n'
    '    check_summary_selectors(headers, helpers, expected)\n'
    '    good = helpers[1][68]\n'
    '    helpers[1][68] = good.replace(expected, "Warning Score")\n'
    '    with pytest.raises(ValidationError, match="Warning Score"):\n'
    '        check_summary_selectors(headers, helpers, expected)\n'
    '    helpers[1][68] = good\n'
    '    helpers[0][68] = "=B1"\n'
    '    with pytest.raises(ValidationError, match="unrecognised|expected"):\n'
    '        check_summary_selectors(headers, helpers, expected)\n'
)

# Embedded source: tests/test_helpers_growth.py
SOURCES['tests/test_helpers_growth.py'] = (
    'import copy\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import pytest\n'
    'from config import WORKBOOKS, BASE_WORKBOOKS, MOMENTUM_WORKBOOKS\n'
    'from pipeline import calculate, load\n'
    'from momentum_pipeline import (\n'
    '    calculate as momentum_calculate,\n'
    '    prepare as prepare_imports,\n'
    '    desired_formulas,\n'
    ')\n'
    'from calculation_helper import trace, pair_labels\n'
    'from google_sheets import a1\n'
    '\n'
    '\n'
    'def fixture(book, n=50):\n'
    '    dates = pd.date_range("2000-03-31", periods=n, freq="QE")\n'
    '    source = [["Date", "X", "Y"]] + [\n'
    '        [str(d.date()), float(i * i + 2), float(i % 5)] for i, d in enumerate(dates)\n'
    '    ]\n'
    '    output = [["", "X"], ["Date", "Y"]] + [[str(d.date())] for d in dates]\n'
    '    controls = [[book.control_key, 4]]\n'
    '    if book.kind.endswith("_momentum"):\n'
    '        controls.append(["Momentum Lookback", 3])\n'
    '    if book.kind == "correlation_momentum":\n'
    '        from correlation import rolling_correlation\n'
    '\n'
    '        f = rolling_correlation(\n'
    '            [r[1] for r in source[1:]], [r[2] for r in source[1:]], 4\n'
    '        )\n'
    '        source = [[4, "X"], ["Date", "Y"]] + [\n'
    '            [str(d.date()), float(v) if np.isfinite(v) else ""]\n'
    '            for d, v in zip(dates, f)\n'
    '        ]\n'
    '    return source, output, controls\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", WORKBOOKS, ids=lambda b: b.kind)\n'
    'def test_helper_independent_scores_and_cutoff(book):\n'
    '    source, output, controls = fixture(book)\n'
    '    fn = momentum_calculate if book.kind.endswith("_momentum") else calculate\n'
    '    extra = {}\n'
    '    if book.kind == "correlation_momentum":\n'
    '        from config import upstream_book\n'
    '\n'
    '        up = upstream_book(book)\n'
    '        raw, _, _ = fixture(up)\n'
    '        extra = {"pair_source": raw, "pair_controls": [[up.control_key, 4]]}\n'
    '    values, report = fn(book, source, output, controls, **extra)\n'
    '    state = {\n'
    '        "book": book,\n'
    '        "source": source,\n'
    '        "output": output,\n'
    '        "values": values,\n'
    '        "report": report,\n'
    '        **extra,\n'
    '    }\n'
    '    headers, table, matched = trace(state, 0)\n'
    '    assert len(table) == 50 and matched > 20\n'
    '    endcol = headers.index("Window last date")\n'
    '    for row in table:\n'
    '        if row[endcol]:\n'
    '            assert row[endcol] < row[0]\n'
    '    assert pair_labels(output) == ["B | X | Y"]\n'
    '    wrong = copy.deepcopy(state)\n'
    '    idx = next(i for i, r in enumerate(values) if r[0] != "")\n'
    '    wrong["values"][idx][0] += 1\n'
    '    with pytest.raises(ValueError, match="Helper differs"):\n'
    '        trace(wrong, 0)\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "book",\n'
    '    [b for b in WORKBOOKS if b.kind != "correlation_momentum"],\n'
    '    ids=lambda b: b.kind,\n'
    ')\n'
    'def test_append_quarterly_data_and_gap_all_scores(book):\n'
    '    source, out, ctrl = fixture(book, 50)\n'
    '    fn = momentum_calculate if book.kind.endswith("_momentum") else calculate\n'
    '    old, oldreport = fn(book, source, out, ctrl)\n'
    '    extended, extendedout, _ = fixture(book, 52)\n'
    '    # New dated row can contain a gap without zero imputation.\n'
    '    extended[-2][1] = ""\n'
    '    new, report = fn(book, extended, extendedout, ctrl)\n'
    '    assert len(new) == len(old) + 2\n'
    '    assert new[: len(old)] == old\n'
    '    assert report["source_dates"] == oldreport["source_dates"] + 2\n'
    '    if book.kind != "correlation":\n'
    '        assert new[-2][0] == ""\n'
    '    assert new[-1][0] != ""\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", BASE_WORKBOOKS, ids=lambda b: b.kind)\n'
    'def test_base_grid_edge_expands_before_date_validation(book, monkeypatch):\n'
    '    source, out, ctrl = fixture(book, 50)\n'
    '    monkeypatch.setattr("source_freshness.wait_for_macro", lambda *args: source)\n'
    '\n'
    '    class Fake:\n'
    '        expanded = False\n'
    '\n'
    '        def metadata(self, id):\n'
    '            return {\n'
    '                "properties": {"title": book.title + " renamed"},\n'
    '                "sheets": [\n'
    '                    {\n'
    '                        "properties": {\n'
    '                            "title": name,\n'
    '                            "sheetId": sid,\n'
    '                            "gridProperties": {\n'
    '                                "rowCount": 52 if name == book.source_tab else 10,\n'
    '                                "columnCount": 3,\n'
    '                            },\n'
    '                        }\n'
    '                    }\n'
    '                    for name, sid in [\n'
    '                        (book.source_tab, 123),\n'
    '                        (book.output_tab, book.output_id),\n'
    '                        (book.control_tab, 456),\n'
    '                    ]\n'
    '                ],\n'
    '            }\n'
    '\n'
    '        def values(self, id, ranges, **kw):\n'
    '            if len(ranges) == 1:\n'
    '                return [out if self.expanded else out[:10]]\n'
    '            return [source, out[:10], ctrl]\n'
    '\n'
    '        def batch(self, id, requests):\n'
    '            assert len(requests) == 1\n'
    '            p = requests[0]["updateSheetProperties"]["properties"]\n'
    '            assert p["sheetId"] == book.output_id and p["gridProperties"][\n'
    '                "rowCount"\n'
    '            ] >= len(out)\n'
    '            self.expanded = True\n'
    '\n'
    '    api = Fake()\n'
    '    state = load(api, book)\n'
    '    assert api.expanded and len(state["values"]) == 50\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", MOMENTUM_WORKBOOKS, ids=lambda b: b.kind)\n'
    'def test_momentum_import_and_output_capacity_grow(book):\n'
    '    from config import upstream_book\n'
    '\n'
    '    up = upstream_book(book)\n'
    '    is_spread = book.kind == "spread_momentum"\n'
    '    source, out, ctrl = fixture(book, 50)\n'
    '    pairs = [["", "X"], ["Date", "Y"]]\n'
    '    sw = max(map(len, source[:2]))\n'
    '    pw = 2\n'
    '    formulas = desired_formulas(book, sw, pw)\n'
    '    names = [book.source_tab, book.output_tab] + (\n'
    '        ["Imported Pair Definitions"] if is_spread else []\n'
    '    )\n'
    '\n'
    '    class Fake:\n'
    '        def __init__(self):\n'
    '            self.writes = []\n'
    '\n'
    '        def metadata(self, id):\n'
    '            if id == up.spreadsheet_id:\n'
    '                return {\n'
    '                    "sheets": [\n'
    '                        {\n'
    '                            "properties": {\n'
    '                                "title": up.source_tab if is_spread else up.output_tab,\n'
    '                                "gridProperties": {"rowCount": 100},\n'
    '                            }\n'
    '                        }\n'
    '                    ]\n'
    '                }\n'
    '            return {\n'
    '                "properties": {"title": book.title + " renamed"},\n'
    '                "sheets": [\n'
    '                    {\n'
    '                        "properties": {\n'
    '                            "title": name,\n'
    '                            "sheetId": (\n'
    '                                book.output_id if name == book.output_tab else 123 + i\n'
    '                            ),\n'
    '                            "gridProperties": {\n'
    '                                "rowCount": 10,\n'
    '                                "columnCount": sw if name == book.source_tab else pw,\n'
    '                            },\n'
    '                        }\n'
    '                    }\n'
    '                    for i, name in enumerate(names)\n'
    '                ],\n'
    '            }\n'
    '\n'
    '        def values(self, id, ranges, formulas=False):\n'
    '            if id == up.spreadsheet_id:\n'
    '                return [source, pairs]\n'
    '            if len(ranges) == 1:\n'
    '                return [\n'
    '                    [\n'
    '                        [\n'
    '                            v\n'
    '                            for t, c, v in desired_formulas(book, sw, pw)\n'
    '                            if t == book.source_tab\n'
    '                        ][0]\n'
    '                    ]\n'
    '                ]\n'
    '            return [[[f]] for _, _, f in desired_formulas(book, sw, pw)]\n'
    '\n'
    '        def batch(self, id, req):\n'
    '            assert id == book.spreadsheet_id\n'
    '            self.writes += req\n'
    '\n'
    '    fake = Fake()\n'
    '    prepare_imports(fake, book)\n'
    '    assert len(fake.writes) == 2\n'
    '    assert all(\n'
    '        v["updateSheetProperties"]["properties"]["gridProperties"]["rowCount"] >= 100\n'
    '        for v in fake.writes\n'
    '    )\n'
    '\n'
    '\n'
    'def test_helper_publication_roundtrip_and_frozen_boundary(tmp_path):\n'
    '    from calculation_helper import publish, trace, pair_labels\n'
    '\n'
    '    book = WORKBOOKS[0]\n'
    '    source, out, ctrl = fixture(book)\n'
    '    values, report = calculate(book, source, out, ctrl)\n'
    '    state = {\n'
    '        "book": book,\n'
    '        "source": source,\n'
    '        "output": out,\n'
    '        "values": values,\n'
    '        "report": report,\n'
    '    }\n'
    '    headers, table, matched = trace(state, 0)\n'
    '    helper = {\n'
    '        "selection": pair_labels(out)[0],\n'
    '        "labels": pair_labels(out),\n'
    '        "headers": headers,\n'
    '        "table": table,\n'
    '        "matched": matched,\n'
    '    }\n'
    '\n'
    '    class Fake:\n'
    '        def metadata(self, id):\n'
    '            return {"sheets": []}\n'
    '\n'
    '        def batch(self, id, requests):\n'
    '            for req in requests:\n'
    '                if "mergeCells" in req:\n'
    '                    r = req["mergeCells"]["range"]\n'
    '                    assert (\n'
    '                        r["startColumnIndex"] >= 1\n'
    '                    )  # Never merge across frozen date column A.\n'
    '\n'
    '        def request(self, *args, **kwargs):\n'
    '            pass\n'
    '\n'
    '        def write_values(self, id, data):\n'
    '            self.rows = data[0]["values"]\n'
    '\n'
    '        def values(self, id, ranges):\n'
    '            return [self.rows[4:]]\n'
    '\n'
    '    publish(Fake(), state, helper, tmp_path)\n'
    '    assert state["report"]["helper"]["verified"]\n'
)

# Embedded source: tests/test_incremental_import.py
SOURCES['tests/test_incremental_import.py'] = (
    'from datetime import date\n'
    '\n'
    'import pytest\n'
    '\n'
    'import monthly_indicators as m\n'
    '\n'
    '\n'
    'def serial(value: date) -> int:\n'
    '    return (value - date(1899, 12, 30)).days\n'
    '\n'
    '\n'
    'def test_reference_period_dates_remain_the_published_baseline(monkeypatch):\n'
    '    monkeypatch.setattr(m, "FULL_HISTORY_MODE", True)\n'
    '    monthly = {\n'
    '        "observation_date": date(2026, 1, 1),\n'
    '        "release_date": date(2026, 2, 3),\n'
    '        "selection_date": date(2026, 1, 1),\n'
    '        "period_aligned": True,\n'
    '        "value": 51.2,\n'
    '    }\n'
    '    quarterly = {\n'
    '        "observation_date": date(2026, 1, 1),\n'
    '        "release_date": date(2026, 4, 25),\n'
    '        "selection_date": date(2026, 3, 1),\n'
    '        "period_aligned": True,\n'
    '        "native_frequency": "Quarterly",\n'
    '        "value": 123.4,\n'
    '    }\n'
    '    annual = {\n'
    '        "observation_date": date(2025, 1, 1),\n'
    '        "release_date": date(2026, 2, 1),\n'
    '        "selection_date": date(2025, 12, 1),\n'
    '        "period_aligned": True,\n'
    '        "native_frequency": "Annual",\n'
    '        "value": 10.5,\n'
    '    }\n'
    '\n'
    '    assert m.output_month_records_for_indicator([monthly]) == {\n'
    '        date(2026, 1, 1): 51.2\n'
    '    }\n'
    '    assert m.output_month_records_for_indicator([quarterly]) == {\n'
    '        date(2026, 3, 1): 123.4\n'
    '    }\n'
    '    assert m.output_month_records_for_indicator([annual]) == {\n'
    '        date(2025, 12, 1): 10.5\n'
    '    }\n'
    '\n'
    '\n'
    'def test_incremental_table_fills_blanks_appends_and_preserves_populated_values():\n'
    '    existing = [\n'
    '        ["Date", "Indicator A", "Indicator B"],\n'
    '        [serial(date(2026, 1, 31)), 1.0, ""],\n'
    '        [serial(date(2026, 2, 28)), "", 2.0],\n'
    '    ]\n'
    '    incoming = [\n'
    '        ["Date", "Indicator A", "Indicator B"],\n'
    '        [serial(date(2026, 1, 31)), 99.0, 11.0],\n'
    '        [serial(date(2026, 2, 28)), 22.0, 20.0],\n'
    '        [serial(date(2026, 3, 31)), 3.0, 4.0],\n'
    '    ]\n'
    '\n'
    '    fills, appends, merged = m.prepare_incremental_table_changes(\n'
    '        existing, incoming, "test wide table"\n'
    '    )\n'
    '\n'
    '    assert fills == [(2, 2, 11.0), (3, 1, 22.0)]\n'
    '    assert appends == [[serial(date(2026, 3, 31)), 3.0, 4.0]]\n'
    '    assert merged == [\n'
    '        existing[0],\n'
    '        [serial(date(2026, 1, 31)), 1.0, 11.0],\n'
    '        [serial(date(2026, 2, 28)), 22.0, 2.0],\n'
    '        appends[0],\n'
    '    ]\n'
    '\n'
    '    rerun_fills, rerun_appends, rerun = m.prepare_incremental_table_changes(\n'
    '        merged, incoming, "test wide table"\n'
    '    )\n'
    '    assert rerun_fills == []\n'
    '    assert rerun_appends == []\n'
    '    assert rerun == merged\n'
    '\n'
    '\n'
    'def test_incremental_table_rejects_missing_historical_dates():\n'
    '    existing = [\n'
    '        ["Date", "Indicator"],\n'
    '        [serial(date(2026, 1, 31)), 1.0],\n'
    '        [serial(date(2026, 3, 31)), 3.0],\n'
    '    ]\n'
    '    incoming = [\n'
    '        ["Date", "Indicator"],\n'
    '        [serial(date(2026, 2, 28)), 2.0],\n'
    '        [serial(date(2026, 3, 31)), 4.0],\n'
    '    ]\n'
    '    with pytest.raises(RuntimeError, match="missing historical date"):\n'
    '        m.prepare_incremental_table_changes(existing, incoming, "test table")\n'
    '\n'
    '\n'
    'def test_indicator_sync_only_updates_blank_values_by_default(monkeypatch):\n'
    '    monkeypatch.setattr(m, "FULL_HISTORY_MODE", False)\n'
    '    existing = {\n'
    '        date(2026, 1, 1): {"row": 2, "value": 1.0},\n'
    '        date(2026, 2, 1): {"row": 3, "value": None},\n'
    '    }\n'
    '    incoming = {\n'
    '        date(2026, 1, 1): 9.0,\n'
    '        date(2026, 2, 1): 2.0,\n'
    '        date(2026, 3, 1): 3.0,\n'
    '    }\n'
    '\n'
    '    fills, appends = m.prepare_sheet_changes(existing, incoming)\n'
    '\n'
    '    assert fills == [(3, date(2026, 2, 1), 2.0)]\n'
    '    assert appends == [(date(2026, 3, 1), 3.0)]\n'
    '\n'
    '\n'
    'def test_incremental_import_ignores_revisions_and_blank_replacements(monkeypatch):\n'
    '    monkeypatch.setattr(m, "FULL_HISTORY_MODE", False)\n'
    '    existing = {date(2026, 1, 1): {"row": 2, "value": 1.0, "raw_value": 1.0}}\n'
    '    m.validate_overlapping_indicator_values(\n'
    '        existing,\n'
    '        {date(2026, 1, 1): 99.0},\n'
    '        "test indicator",\n'
    '        allow_value_revisions=False,\n'
    '    )\n'
    '    assert m.prepare_sheet_changes(existing, {date(2026, 1, 1): None}) == ([], [])\n'
    '\n'
    '\n'
    'def test_explicit_full_history_mode_can_revise_values(monkeypatch):\n'
    '    monkeypatch.setattr(m, "FULL_HISTORY_MODE", True)\n'
    '    existing = {date(2026, 1, 1): {"row": 2, "value": 1.0}}\n'
    '    fills, appends = m.prepare_sheet_changes(existing, {date(2026, 1, 1): 9.0})\n'
    '    assert fills == [(2, date(2026, 1, 1), 9.0)]\n'
    '    assert appends == []\n'
    '\n'
    '\n'
    'def test_incremental_value_ranges_are_bounded_and_do_not_clear():\n'
    '    ranges = m.incremental_table_value_ranges(\n'
    '        "All data",\n'
    '        [(2, 1, 10), (2, 2, 20), (4, 40, 30)],\n'
    '        [[serial(date(2026, 4, 30)), 1, 2]],\n'
    '        existing_row_count=4,\n'
    '    )\n'
    '    assert ranges == [\n'
    '        {"range": "\'All data\'!B2:C2", "values": [[10, 20]]},\n'
    '        {"range": "\'All data\'!AO4:AO4", "values": [[30]]},\n'
    '        {\n'
    '            "range": "\'All data\'!A5:C5",\n'
    '            "values": [[serial(date(2026, 4, 30)), 1, 2]],\n'
    '        },\n'
    '    ]\n'
)

# Embedded source: tests/test_io.py
SOURCES['tests/test_io.py'] = (
    'import sys\n'
    'from pathlib import Path\n'
    '\n'
    'sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n'
    'import numpy as np\n'
    'import pytest\n'
    'from config import WORKBOOKS\n'
    'from google_sheets import Sheets, column_name\n'
    'from pipeline import verify, validate_before_publish\n'
    'from validation import ValidationError\n'
    '\n'
    '\n'
    'def test_batch_write_preserves_headers_and_dates_and_clears_stale(tmp_path):\n'
    '    api = Sheets.__new__(Sheets)\n'
    '    calls = []\n'
    '    api.request = (\n'
    '        lambda method, id, suffix="", **kw: calls.append((method, suffix, kw)) or {}\n'
    '    )\n'
    '    matrix = [[float(i)] * 3 for i in range(5)]\n'
    '    count = api.publish(\n'
    '        WORKBOOKS[0],\n'
    '        {"gridProperties": {"rowCount": 20, "columnCount": 8}},\n'
    '        matrix,\n'
    '        tmp_path / "before.gz",\n'
    '        {},\n'
    '    )\n'
    '    assert count == 15\n'
    '    assert calls[0][1] == "/values:batchClear"\n'
    '    assert calls[0][2]["json"]["ranges"] == ["\'Correlation Score\'!B3:H20"]\n'
    '    assert calls[1][2]["json"]["data"][0]["range"] == "\'Correlation Score\'!B3:D7"\n'
    '    assert (tmp_path / "before.gz").exists()\n'
    '\n'
    '\n'
    'def test_large_matrix_is_batched(tmp_path):\n'
    '    api = Sheets.__new__(Sheets)\n'
    '    calls = []\n'
    '    api.request = lambda method, id, suffix="", **kw: calls.append((suffix, kw)) or {}\n'
    '    matrix = [[""] * 666 for _ in range(200)]\n'
    '    api.publish(\n'
    '        WORKBOOKS[1],\n'
    '        {"gridProperties": {"rowCount": 250, "columnCount": 667}},\n'
    '        matrix,\n'
    '        tmp_path / "before.gz",\n'
    '        {},\n'
    '    )\n'
    '    writes = [c for c in calls if c[0] == "/values:batchUpdate"]\n'
    '    assert len(writes) == 3\n'
    '    assert all(\n'
    '        sum(len(r) for r in w[1]["json"]["data"][0]["values"]) <= 50000 for w in writes\n'
    '    )\n'
    '\n'
    '\n'
    'def test_verification_detects_stale_output_and_preserves_structure():\n'
    '    class Fake:\n'
    '        def values(self, *args, **kwargs):\n'
    '            return [self.rows]\n'
    '\n'
    '    api = Fake()\n'
    '    state = {\n'
    '        "book": WORKBOOKS[1],\n'
    '        "values": [[""], [2.0]],\n'
    '        "formulas": [["Date", "X"], [43831, "Y"], [43862, "=FORMULA()"], [43891]],\n'
    '    }\n'
    '    api.rows = [["Date", "X"], [43831, "Y"], [43862], [43891, 2.0]]\n'
    '    assert verify(api, state)\n'
    '    api.rows.append(["", 123])\n'
    '    with pytest.raises(ValidationError, match="stale"):\n'
    '        verify(api, state)\n'
    '    api.rows = [["Date", "wrong"], [43831, "Y"], [43862], [43891, 2.0]]\n'
    '    with pytest.raises(ValidationError, match="header"):\n'
    '        verify(api, state)\n'
    '\n'
    '\n'
    'def test_source_change_blocks_write(monkeypatch):\n'
    '    monkeypatch.setattr("source_freshness.wait_for_macro", lambda *args: None)\n'
    '\n'
    '    class Fake:\n'
    '        def values(self, *args, **kwargs):\n'
    '            return [\n'
    '                [["changed"]],\n'
    '                [["", "X"], ["Date", "Y"]],\n'
    '                [[], ["Date"], [43831]],\n'
    '                [["Correlation Window", 3]],\n'
    '            ]\n'
    '\n'
    '    state = {\n'
    '        "book": WORKBOOKS[0],\n'
    '        "source": [["Date", "X", "Y"]],\n'
    '        "output": [["", "X"], ["Date", "Y"], [43831]],\n'
    '        "controls": [["Correlation Window", 3]],\n'
    '    }\n'
    '    with pytest.raises(ValidationError, match="changed"):\n'
    '        validate_before_publish(Fake(), state)\n'
    '\n'
    '\n'
    'def test_column_names_have_no_fixed_pair_limit():\n'
    '    assert column_name(667) == "YQ"\n'
    '    assert column_name(703) == "AAA"\n'
    '\n'
    '\n'
    'def test_spread_preserves_source_and_resizes_only_output(monkeypatch):\n'
    '    from pipeline import load\n'
    '\n'
    '    book = WORKBOOKS[1]\n'
    '    source = [\n'
    '        ["Date", "X", "Y"],\n'
    '        ["2020-01-01", 1, 0],\n'
    '        ["2020-02-01", 2, 0],\n'
    '        ["2020-03-01", 4, 0],\n'
    '        ["2020-04-01", 7, 0],\n'
    '    ]\n'
    '    output = [["", "X"], ["Date", "Y"]] + [[r[0]] for r in source[1:]]\n'
    '\n'
    '    class Fake:\n'
    '        def __init__(self):\n'
    '            self.writes = []\n'
    '            self.reads = []\n'
    '\n'
    '        def metadata(self, id):\n'
    '            assert id == book.spreadsheet_id\n'
    '            return {\n'
    '                "properties": {"title": book.title},\n'
    '                "sheets": [\n'
    '                    {\n'
    '                        "properties": {\n'
    '                            "title": name,\n'
    '                            "sheetId": sid,\n'
    '                            "gridProperties": {"rowCount": 5, "columnCount": 3},\n'
    '                        }\n'
    '                    }\n'
    '                    for name, sid in [\n'
    '                        (book.source_tab, 1545232009),\n'
    '                        (book.output_tab, book.output_id),\n'
    '                        (book.control_tab, 2026091101),\n'
    '                    ]\n'
    '                ],\n'
    '            }\n'
    '\n'
    '        def values(self, id, ranges, **kwargs):\n'
    '            assert id == book.spreadsheet_id\n'
    '            self.reads.extend(ranges)\n'
    '            if len(ranges) == 1:\n'
    '                return [output]\n'
    '            return [source, output, [[book.control_key, 3]]]\n'
    '\n'
    '        def batch(self, id, requests):\n'
    '            assert id == book.spreadsheet_id\n'
    '            for request in requests:\n'
    '                assert (\n'
    '                    request["updateSheetProperties"]["properties"]["sheetId"]\n'
    '                    == book.output_id\n'
    '                )\n'
    '            self.writes.extend(requests)\n'
    '\n'
    '    monkeypatch.setattr("source_freshness.wait_for_macro", lambda *args: source)\n'
    '    api = Fake()\n'
    '    state = load(api, book)\n'
    '    assert state["source"] == source\n'
    '    assert state["values"][-1][0] == pytest.approx(\n'
    '        (7 - 7 / 3) / np.std([1, 2, 4], ddof=1)\n'
    '    )\n'
    '    assert "source_refresh" not in state\n'
    '    assert all("Coincident" not in r for r in api.reads)\n'
    '    assert api.writes\n'
    '\n'
    '\n'
    'def test_formula_only_header_snapshot_omits_blank_second_row():\n'
    '    # Sheets FORMULA rendering omits spill values; trailing blank rows may be absent.\n'
    '    class Fake:\n'
    '        def values(self, *args, **kwargs):\n'
    '            return [[["=DATES()", "=PAIRS()"], [], ["", 1.0]]]\n'
    '\n'
    '    state = {\n'
    '        "book": WORKBOOKS[0],\n'
    '        "values": [[1.0]],\n'
    '        "formulas": [["=DATES()", "=PAIRS()"]],\n'
    '    }\n'
    '    assert verify(Fake(), state)\n'
)

# Embedded source: tests/test_ism_updates.py
SOURCES['tests/test_ism_updates.py'] = (
    'from datetime import date\n'
    'import pandas as pd\n'
    'import pytest\n'
    'import monthly_indicators as m\n'
    '\n'
    '\n'
    'def frame():\n'
    '    current = date.today().replace(day=1)\n'
    '    releases = [m.add_months(current, -2), current]\n'
    '    return pd.DataFrame(\n'
    '        {\n'
    '            "date": [d.isoformat() for d in releases],\n'
    '            "report_month": [m.add_months(d, -1).isoformat() for d in releases],\n'
    '            **{column: [51.0, 52.0] for column in m.ISM_VALUE_COLUMNS},\n'
    '        }\n'
    '    )\n'
    '\n'
    '\n'
    'def test_missing_months_and_components_preserve_existing_history(monkeypatch):\n'
    '    data = frame()\n'
    '    data.loc[1, "services_pmi"] = float("nan")\n'
    '    monkeypatch.setattr(m, "VALIDATION_MESSAGES", [])\n'
    '    m.validate_ism_frame(data, expected_months=3)\n'
    '    assert any(\n'
    '        "Unavailable release month" in x["message"] for x in m.VALIDATION_MESSAGES\n'
    '    )\n'
    '    monkeypatch.setattr(m, "_ISM_CACHE", {3: data})\n'
    '    monkeypatch.setattr(m, "_EXPLICIT_BLANK_OUTPUT_MONTHS", {})\n'
    '    indicator = next(i for i in m.INDICATORS if i.source_column == "services_pmi")\n'
    '    records = m.get_ism_indicator_observations(indicator, 3)\n'
    '    incoming = m.sheet_rows_for_indicator(indicator.name, records)\n'
    '    current = date.today().replace(day=1)\n'
    '    assert current not in incoming\n'
    '    # Exercise the real default history merge and write/readback path with a fake API.\n'
    '    old = [\n'
    '        ["Date", "Value"],\n'
    '        *[\n'
    '            [m.format_output_month(m.add_months(current, offset)), 49.0]\n'
    '            for offset in [-3, -2, -1, 0]\n'
    '        ],\n'
    '    ]\n'
    '    state = {"values": old}\n'
    '\n'
    '    def request(method, url, token, body=None):\n'
    '        if method == "PUT":\n'
    '            state["values"] = body["values"]\n'
    '        return {"values": state["values"]}\n'
    '\n'
    '    monkeypatch.setattr(m, "google_sheets_request", request)\n'
    '    monkeypatch.setattr(m, "FULL_HISTORY_MODE", True)\n'
    '    m.sync_indicator_sheet_direct(\n'
    '        "https://example.test", "fake", indicator.name, indicator.name, records\n'
    '    )\n'
    '    values = dict(state["values"][1:])\n'
    '    assert values[m.format_output_month(current)] == 49.0\n'
    '    assert values[m.format_output_month(m.add_months(current, -1))] == 49.0\n'
    '    assert values[m.format_output_month(m.add_months(current, -2))] == 51.0\n'
    '    assert len(values) == 4\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("bad", [101, -1, "invalid"])\n'
    'def test_known_invalid_values_still_fail(bad):\n'
    '    data = frame().astype(object)\n'
    '    data.loc[1, "manufacturing_pmi"] = bad\n'
    '    with pytest.raises(RuntimeError, match="implausible|nonnumeric"):\n'
    '        m.validate_ism_frame(data, expected_months=3)\n'
    '\n'
    '\n'
    'def test_stale_reports_still_fail():\n'
    '    data = frame()\n'
    '    data["date"] = ["2020-01-01", "2020-02-01"]\n'
    '    data["report_month"] = ["2019-12-01", "2020-01-01"]\n'
    '    with pytest.raises(RuntimeError, match="stale"):\n'
    '        m.validate_ism_frame(data, expected_months=3)\n'
    '\n'
    '\n'
    'def test_missing_entire_component_is_no_update(monkeypatch):\n'
    '    data = frame().drop(columns="services_pmi")\n'
    '    m.validate_ism_frame(data, expected_months=3)\n'
    '    monkeypatch.setattr(m, "_ISM_CACHE", {3: data})\n'
    '    indicator = next(i for i in m.INDICATORS if i.source_column == "services_pmi")\n'
    '    assert m.get_indicator_observations_safely("", indicator, 3) == []\n'
    '\n'
    '\n'
    'def test_press_release_allowlist_is_scoped():\n'
    '    for url in m.ISM_PRESS_RELEASES.values():\n'
    '        m.validate_official_url(url, "ISM")\n'
    '    with pytest.raises(m.OfficialSourceRedirectError):\n'
    '        m.validate_official_url("https://www.prnewswire.com/unverified.html", "ISM")\n'
)

# Embedded source: tests/test_momentum.py
SOURCES['tests/test_momentum.py'] = (
    'import math\n'
    'import statistics\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import pytest\n'
    'from momentum import observation_change, momentum_zscore\n'
    'from momentum_pipeline import (\n'
    '    calculate,\n'
    '    fisher_frame,\n'
    '    desired_formulas,\n'
    '    canonical,\n'
    '    assert_fresh,\n'
    ')\n'
    'from config import MOMENTUM_WORKBOOKS, momentum_lookback\n'
    'from validation import ValidationError\n'
    '\n'
    '\n'
    'def oracle(values, m, w):\n'
    '    seen = []\n'
    '    changes = []\n'
    '    out = []\n'
    '    for value in values:\n'
    '        score = math.nan\n'
    '        if math.isfinite(value):\n'
    '            if len(seen) >= m:\n'
    '                change = value - seen[-m]\n'
    '                if len(changes) >= w:\n'
    '                    hist = changes[-w:]\n'
    '                    sd = statistics.stdev(hist)\n'
    '                    if sd > 0:\n'
    '                        score = (change - statistics.mean(hist)) / sd\n'
    '                changes.append(change)\n'
    '            seen.append(value)\n'
    '        out.append(score)\n'
    '    return out\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("m,w", [(1, 2), (3, 4), (3, 24), (5, 10)])\n'
    '@pytest.mark.parametrize(\n'
    '    "values",\n'
    '    [\n'
    '        np.arange(80.0) ** 2,\n'
    '        -np.arange(80.0) ** 2,\n'
    '        np.ones(80),\n'
    '        np.arange(80.0),\n'
    '        np.sin(np.arange(80.0)),\n'
    '        np.where(np.arange(80) % 7 == 0, np.nan, np.arange(80.0) ** 2),\n'
    '    ],\n'
    ')\n'
    'def test_independent_oracle(values, m, w):\n'
    '    np.testing.assert_allclose(\n'
    '        momentum_zscore(values, m, w), oracle(values, m, w), atol=1e-12, equal_nan=True\n'
    '    )\n'
    '\n'
    '\n'
    'def test_manual_m3_sample_std():\n'
    '    # Squared series: m=3 changes are 9,15,21,27. Prior mean=15, sample sd=6.\n'
    '    s = np.arange(7.0) ** 2\n'
    '    np.testing.assert_allclose(observation_change(s, 3)[3:], [9, 15, 21, 27])\n'
    '    assert momentum_zscore(s, 3, 3)[6] == 2\n'
    '    assert np.isnan(momentum_zscore(s, 3, 3)[:6]).all()\n'
    '    assert momentum_zscore(-s, 3, 3)[6] == -2\n'
    '\n'
    '\n'
    'def test_no_current_or_future_in_normalisation():\n'
    '    values = np.arange(12.0) ** 2\n'
    '    original = momentum_zscore(values, 3, 3)\n'
    '    changed = values.copy()\n'
    '    changed[6] += 60\n'
    '    changed[7:] = 999\n'
    '    result = momentum_zscore(changed, 3, 3)\n'
    '    np.testing.assert_allclose(result[:6], original[:6], equal_nan=True)\n'
    '    assert (\n'
    '        result[6] == 12\n'
    '    )  # (87 - 15)/6; current change does not alter historical stats.\n'
    '\n'
    '\n'
    'def test_gaps_are_valid_observations_and_blank_current():\n'
    '    x = np.array([0, np.nan, 1, 4, np.nan, 9, 16, 25, 36.0])\n'
    '    ch = observation_change(x, 3)\n'
    '    np.testing.assert_allclose(ch[[5, 6, 7, 8]], [9, 15, 21, 27])\n'
    '    assert momentum_zscore(x, 3, 3)[8] == 2\n'
    '    assert np.isnan(ch[[1, 4]]).all()\n'
    '\n'
    '\n'
    'def test_fisher_values_used_directly_and_reordered_by_pair():\n'
    '    from correlation import rolling_correlation\n'
    '    from config import upstream_book\n'
    '\n'
    '    dates = pd.date_range("2000-01-01", periods=40, freq="QS")\n'
    '    x = np.sin(np.arange(40.0))\n'
    '    y = np.cos(np.arange(40.0) / 3)\n'
    '    z = -x\n'
    '    a = rolling_correlation(x, y, 4)\n'
    '    b = rolling_correlation(y, z, 4)\n'
    '    cell = lambda v: float(v) if np.isfinite(v) else ""\n'
    '    source = [[4, "X", "Y"], ["Date", "Y", "Z"]] + [\n'
    '        [str(d.date()), cell(u), cell(v)] for d, u, v in zip(dates, a, b)\n'
    '    ]\n'
    '    raw = [["Date", "X", "Y", "Z"]] + [\n'
    '        [str(d.date()), u, v, t] for d, u, v, t in zip(dates, x, y, z)\n'
    '    ]\n'
    '    output = [[4, "Y", "X"], ["Date", "Z", "Y"]] + [[str(d.date())] for d in dates]\n'
    '    book = MOMENTUM_WORKBOOKS[1]\n'
    '    values, report = calculate(\n'
    '        book,\n'
    '        source,\n'
    '        output,\n'
    '        [["Momentum Lookback", 3], ["Momentum Z-Score Window", 3]],\n'
    '        raw,\n'
    '        [[upstream_book(book).control_key, 4]],\n'
    '    )\n'
    '    actual = np.array([[np.nan if v == "" else v for v in r] for r in values])\n'
    '    np.testing.assert_allclose(actual[:, 0], oracle(b, 3, 3), equal_nan=True)\n'
    '    np.testing.assert_allclose(actual[:, 1], oracle(a, 3, 3), equal_nan=True)\n'
    '    assert report["pairs"] == 2 and report["source_dates"] == 40\n'
    '\n'
    '\n'
    'def test_spread_direction_header_matching_and_appended_data():\n'
    '    dates = pd.date_range("2020-01-01", periods=8, freq="QS")\n'
    '    source = [["Date", "Y", "X"]] + [\n'
    '        [str(d.date()), 10, 10 + i * i] for i, d in enumerate(dates)\n'
    '    ]\n'
    '    output = [["Date", "X"], [str(dates[0].date()), "Y"]] + [\n'
    '        [str(d.date())] for d in dates[1:]\n'
    '    ]\n'
    '    ctrl = [["Momentum Lookback", 3], ["Momentum Z-Score Window", 3]]\n'
    '    v, r = calculate(MOMENTUM_WORKBOOKS[0], source, output, ctrl)\n'
    '    assert v[5] == [2]\n'
    '    v0, _ = calculate(MOMENTUM_WORKBOOKS[0], source[:-1], output[:-1], ctrl)\n'
    '    assert len(v) == len(v0) + 1\n'
    '    assert v[:-1] == v0\n'
    '    for c in (\n'
    '        [["Momentum Lookback", 1], ["Momentum Z-Score Window", 3]],\n'
    '        [["Momentum Lookback", 3], ["Momentum Z-Score Window", 2]],\n'
    '    ):\n'
    '        changed, _ = calculate(MOMENTUM_WORKBOOKS[0], source, output, c)\n'
    '        assert changed != v\n'
    '\n'
    '\n'
    'def test_duplicate_fisher_pair_rejected_not_substituted():\n'
    '    with pytest.raises(ValidationError):\n'
    '        fisher_frame([[24, "X", "X"], ["Date", "Y", "Y"], ["2020-01-01", 1, 2]])\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("value", [0, -1, 1.5, True, "bad"])\n'
    'def test_invalid_lookback(value):\n'
    '    with pytest.raises(ValueError):\n'
    '        momentum_lookback([["Momentum Lookback", value]])\n'
    '\n'
    '\n'
    'def test_imports_dynamic_and_only_existing_score_sources():\n'
    '    formulas = desired_formulas(MOMENTUM_WORKBOOKS[1], 700, 700)\n'
    '    assert len(formulas) < 20\n'
    '    assert any(\n'
    '        "ZX" in f for _, _, f in formulas\n'
    '    )  # 700 columns, beyond original pair width.\n'
    '    assert all("9881" not in f and "3522" not in f for _, _, f in formulas)\n'
    '    assert "MAX(FILTER" in formulas[-1][2]\n'
    '\n'
    '\n'
    'def test_stale_import_rejected():\n'
    '    class Fake:\n'
    '        def values(self, *args):\n'
    '            return [\n'
    '                [["Date", "X"], ["2020-01-01", 2]],\n'
    '                [["Date", "X"], ["2020-01-01", "Y"]],\n'
    '            ]\n'
    '\n'
    '    with pytest.raises(ValidationError, match="stale"):\n'
    '        assert_fresh(\n'
    '            Fake(),\n'
    '            MOMENTUM_WORKBOOKS[0],\n'
    '            [["Date", "X"], ["2020-01-01", 1]],\n'
    '            [["Date", "X"], ["2020-01-01", "Y"]],\n'
    '        )\n'
    '\n'
    '\n'
    'def test_momentum_only_client_blocks_both_original_workbooks():\n'
    '    from google_sheets import Sheets\n'
    '    from config import BASE_WORKBOOKS\n'
    '\n'
    '    api = Sheets.__new__(Sheets)\n'
    '    api.write_ids = {b.spreadsheet_id for b in MOMENTUM_WORKBOOKS}\n'
    '    for book in BASE_WORKBOOKS:\n'
    '        with pytest.raises(ValueError, match="prohibited"):\n'
    '            api.request("POST", book.spreadsheet_id, ":batchUpdate", json={})\n'
    '\n'
    '\n'
    'def test_invalid_mapping_produces_blank_with_pair_log():\n'
    '    book = MOMENTUM_WORKBOOKS[0]\n'
    '    source = [["Date", "X", "Y"], ["2020-01-01", 1, 2], ["2020-04-01", 2, 3]]\n'
    '    output = [["Date", "MISSING", ""], ["2020-01-01", "Y", ""], ["2020-04-01"]]\n'
    '    values, report = calculate(\n'
    '        book, source, output, [["Momentum Lookback", 1], ["Momentum Z-Score Window", 2]]\n'
    '    )\n'
    '    assert values == [["", ""]]\n'
    '    assert report["invalid_indicator_mappings"] == 2\n'
    '    assert "MISSING" in str(report["issues"])\n'
)

# Embedded source: tests/test_momentum_updates.py
SOURCES['tests/test_momentum_updates.py'] = (
    '"""Mixed-frequency regression tests with independent scalar calculation checks."""\n'
    '\n'
    'import copy\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import pytest\n'
    'from config import MOMENTUM_WORKBOOKS, upstream_book\n'
    'from correlation import rolling_correlation\n'
    'from momentum import momentum_zscore\n'
    'from momentum_pipeline import calculate, correlation_events, fisher_frame, assert_fresh\n'
    'from calculation_helper import trace\n'
    'from validation import ValidationError\n'
    '\n'
    'BOOKS = [b for b in MOMENTUM_WORKBOOKS if b.kind == "correlation_momentum"]\n'
    '\n'
    '\n'
    'def fixture(book, step=12, n=840, gap=None):\n'
    '    dates = pd.date_range("1950-01-31", periods=n, freq="ME")\n'
    '    x = np.where(np.arange(n) % step == 0, np.sin(np.arange(n) / step * 0.31), np.nan)\n'
    '    y = np.cos(np.arange(n) / step * 0.71)\n'
    '    if gap is not None:\n'
    '        y[gap] = np.nan\n'
    '    fisher = rolling_correlation(x, y, 24)\n'
    '    cell = lambda v: float(v) if np.isfinite(v) else ""\n'
    '    raw = [["Date", "X", "Y"]] + [\n'
    '        [str(d.date()), cell(a), cell(b)] for d, a, b in zip(dates, x, y)\n'
    '    ]\n'
    '    source = [[24, "X"], ["Date", "Y"]] + [\n'
    '        [str(d.date()), cell(v)] for d, v in zip(dates, fisher)\n'
    '    ]\n'
    '    output = [[24, "X"], ["Date", "Y"]] + [[str(d.date())] for d in dates]\n'
    '    controls = [["Momentum Lookback", 3], ["Momentum Z-Score Window", 24]]\n'
    '    upstream_controls = [[upstream_book(book).control_key, 24]]\n'
    '    return source, output, controls, raw, upstream_controls\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", BOOKS, ids=lambda b: b.dataset)\n'
    '@pytest.mark.parametrize("step", [1, 3, 12])\n'
    'def test_updates_scalar_oracle_causality_and_missing_data(book, step):\n'
    '    args = fixture(book, step, gap=step * 35)\n'
    '    values, report = calculate(book, *args)\n'
    '    state = {\n'
    '        "book": book,\n'
    '        "source": args[0],\n'
    '        "output": args[1],\n'
    '        "values": values,\n'
    '        "report": report,\n'
    '        "pair_source": args[3],\n'
    '    }\n'
    '    headers, table, matched = trace(state, 0)\n'
    '    assert matched > 0\n'
    '    for i, row in enumerate(table):\n'
    '        fresh = i % step == 0 and i != step * 35\n'
    '        assert row[headers.index("New paired observation")] == (\n'
    '            "Yes" if fresh else "No"\n'
    '        )\n'
    '        if not fresh:\n'
    '            assert values[i] == [""]\n'
    '    # A shorter history has exactly the same previously calculated scores.\n'
    '    short = fixture(book, step, n=720, gap=step * 35)\n'
    '    previous, _ = calculate(book, *short)\n'
    '    assert previous == values[:720]\n'
    '    if step == 1:\n'
    "        # Dense histories use each valid current pair's Fisher value.\n"
    '        frame, _ = fisher_frame(args[0])\n'
    '        f = frame.iloc[:, 0].to_numpy().copy()\n'
    '        expected = momentum_zscore(f, 3, 24)\n'
    '        actual = np.array([r[0] if r[0] != "" else np.nan for r in values])\n'
    '        np.testing.assert_allclose(actual, expected, equal_nan=True)\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("book", BOOKS, ids=lambda b: b.dataset)\n'
    'def test_annual_warmup_requires_real_observations(book):\n'
    '    args = fixture(book, n=600)\n'
    '    values, report = calculate(book, *args)\n'
    '    assert all(\n'
    '        r == [""] for r in values\n'
    '    )  # Fewer than 24+3+24 actual paired observations.\n'
    '    assert report["suppressed_repeated_fisher_cells"] == 0\n'
    '    assert report["pair_status"][0]["status"] == "No finite momentum history"\n'
    '\n'
    '\n'
    'def test_unchanged_correlation_on_new_input_still_counts():\n'
    '    book = BOOKS[0]\n'
    '    dates = pd.date_range("2000-01-31", periods=20, freq="ME")\n'
    '    x = np.resize([1.0, 2.0, 4.0, 3.0], 20)\n'
    '    y = np.resize([2.0, 1.0, 3.0, 4.0], 20)\n'
    '    f = rolling_correlation(x, y, 4)\n'
    '    raw = [["Date", "X", "Y"]] + [[str(d.date()), a, b] for d, a, b in zip(dates, x, y)]\n'
    '    frame = pd.DataFrame({("X", "Y"): f}, index=dates)\n'
    '    masks, _ = correlation_events(\n'
    '        book, frame, raw, [[upstream_book(book).control_key, 4]]\n'
    '    )\n'
    '    assert masks[("X", "Y")][1:].all()\n'
    '    assert len(np.unique(f[4:])) == 1  # Not deduplicated by value.\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "tamper",\n'
    '    ["missing_source", "value", "blank", "control", "duplicate_date", "invalid_number"],\n'
    ')\n'
    'def test_lineage_fails_closed(tamper):\n'
    '    book = BOOKS[0]\n'
    '    args = list(fixture(book, step=1, n=80))\n'
    '    if tamper == "missing_source":\n'
    '        args[3] = None\n'
    '    elif tamper == "value":\n'
    '        args[0][-1][1] += 1\n'
    '    elif tamper == "blank":\n'
    '        args[0][-1][1] = ""\n'
    '    elif tamper == "control":\n'
    '        args[4][0][1] = 25\n'
    '    elif tamper == "duplicate_date":\n'
    '        args[3][-1][0] = args[3][-2][0]\n'
    '    elif tamper == "invalid_number":\n'
    '        args[3][-1][1] = "123"\n'
    '    with pytest.raises(ValidationError):\n'
    '        calculate(book, *args)\n'
    '\n'
    '\n'
    'def test_missing_display_dates_preserve_strict_cutoff():\n'
    '    book = BOOKS[0]\n'
    '    args = list(fixture(book, step=3, n=240))\n'
    "    # Removing an unpaired display row does not move its preceding pair's update.\n"
    '    del args[0][102]\n'
    '    del args[1][102]\n'
    '    values, report = calculate(book, *args)\n'
    '    headers, table, _ = trace(\n'
    '        {\n'
    '            "book": book,\n'
    '            "source": args[0],\n'
    '            "output": args[1],\n'
    '            "values": values,\n'
    '            "report": report,\n'
    '            "pair_source": args[3],\n'
    '        },\n'
    '        0,\n'
    '    )\n'
    '    assert table[99][headers.index("New paired observation")] == "Yes"\n'
    '    assert table[100][headers.index("New paired observation")] == "No"\n'
    '\n'
    '\n'
    'def test_concurrent_pair_source_or_controls_abort():\n'
    '    book = BOOKS[0]\n'
    '    source, output, ctrl, raw, upctrl = fixture(book, step=1, n=80)\n'
    '\n'
    '    class Fake:\n'
    '        calls = 0\n'
    '\n'
    '        def values(self, *args):\n'
    '            self.calls += 1\n'
    '            if self.calls == 1:\n'
    '                return [source, output[:2]]\n'
    '            altered = copy.deepcopy(raw)\n'
    '            altered[-1][1] += 1\n'
    '            return [altered, upctrl]\n'
    '\n'
    '    with pytest.raises(ValidationError, match="changed during calculation"):\n'
    '        assert_fresh(Fake(), book, source, output, raw, upctrl)\n'
    '\n'
    '\n'
    'def test_undefined_fisher_update_does_not_reappear_as_stale_momentum():\n'
    '    book = BOOKS[0]\n'
    '    args = list(fixture(book, step=3, n=240))\n'
    '    # A long constant X segment makes several correlation windows undefined;\n'
    '    # later varying inputs recover. Empty values must never count as updates.\n'
    '    for i in range(80, 165):\n'
    '        if args[3][i + 1][1] != "":\n'
    '            args[3][i + 1][1] = 1.0\n'
    '    from validation import source_frame\n'
    '\n'
    '    raw, _ = source_frame(args[3])\n'
    '    f = rolling_correlation(raw.X, raw.Y, 24)\n'
    '    for row, v in zip(args[0][2:], f):\n'
    '        row[1] = float(v) if np.isfinite(v) else ""\n'
    '    values, report = calculate(book, *args)\n'
    '    headers, table, _ = trace(\n'
    '        {\n'
    '            "book": book,\n'
    '            "source": args[0],\n'
    '            "output": args[1],\n'
    '            "values": values,\n'
    '            "report": report,\n'
    '            "pair_source": args[3],\n'
    '        },\n'
    '        0,\n'
    '    )\n'
    '    assert np.isnan(f[155:165]).any()\n'
    '    for i, v in enumerate(f):\n'
    '        if not np.isfinite(v):\n'
    '            assert values[i] == [""]\n'
    '    assert report["numeric_cells"] > 0\n'
)

# Embedded source: tests/test_request_progress.py
SOURCES['tests/test_request_progress.py'] = (
    'import logging\n'
    'import threading\n'
    'import pytest\n'
    'from google_sheets import request_progress\n'
    'import run_relationships as main\n'
    '\n'
    '\n'
    'def test_progress_stops_after_interrupt(caplog):\n'
    '    emitted = threading.Event()\n'
    '\n'
    '    class Signal(logging.Handler):\n'
    '        def emit(self, record):\n'
    '            emitted.set()\n'
    '\n'
    '    logger = logging.getLogger("google_sheets")\n'
    '    handler = Signal()\n'
    '    logger.addHandler(handler)\n'
    '    try:\n'
    '        with caplog.at_level(logging.INFO):\n'
    '            with pytest.raises(KeyboardInterrupt):\n'
    '                with request_progress("test", interval=0.01):\n'
    '                    assert emitted.wait(2)\n'
    '                    raise KeyboardInterrupt\n'
    '        assert "still waiting for Google Sheets" in caplog.text\n'
    '        assert not any(t.name == "sheets-progress" for t in threading.enumerate())\n'
    '    finally:\n'
    '        logger.removeHandler(handler)\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("writes", [False, True])\n'
    'def test_clean_interrupt_records_possible_writes(tmp_path, monkeypatch, capsys, writes):\n'
    '    monkeypatch.setattr("notifications.lifecycle.send_telegram", lambda *a, **kw: True)\n'
    '\n'
    '    class FakeSheets:\n'
    '        def __init__(self, *a, **kw):\n'
    '            self.writes_attempted = writes\n'
    '\n'
    '    def interrupt(*a, **kw):\n'
    '        raise KeyboardInterrupt\n'
    '\n'
    '    monkeypatch.setattr(main, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(main, "credentials_path", lambda *a: "unused")\n'
    '    monkeypatch.setattr("source_freshness.validate_macro_summary", lambda *a: None)\n'
    '    monkeypatch.setattr(main, "Sheets", FakeSheets)\n'
    '    monkeypatch.setattr(main, "load", interrupt)\n'
    '    assert main.main(["--correlation"]) == 130\n'
    '    import json\n'
    '\n'
    '    report = json.loads(\n'
    '        next((tmp_path / "work").glob("*/interrupted.json")).read_text()\n'
    '    )\n'
    '    assert report["writes_attempted"] == writes\n'
    '    assert "Traceback" not in capsys.readouterr().err\n'
)

# Embedded source: tests/test_telegram.py
SOURCES['tests/test_telegram.py'] = (
    'import io\n'
    'import json\n'
    'import os\n'
    'import runpy\n'
    'import sys\n'
    'import types\n'
    'from pathlib import Path\n'
    'from urllib.parse import parse_qs\n'
    '\n'
    'import pytest\n'
    'import run_us_pipeline as runner\n'
    'from notifications import telegram\n'
    'from notifications import lifecycle\n'
    '\n'
    '\n'
    '@pytest.fixture\n'
    'def alerts(monkeypatch):\n'
    '    calls = []\n'
    '    monkeypatch.setattr(\n'
    '        lifecycle, "send_telegram", lambda *a, **kw: calls.append((a, kw))\n'
    '    )\n'
    '    monkeypatch.delenv("US_PIPELINE_NOTIFICATIONS_MANAGED", raising=False)\n'
    '    return calls\n'
    '\n'
    '\n'
    'def test_success(alerts, monkeypatch):\n'
    '    monkeypatch.setattr(runner, "_run", lambda args: 0)\n'
    '    assert runner.main(["--stage", "analysis"]) == 0\n'
    '    assert [a[1] for a, kw in alerts] == [\n'
    '        "Indicator analysis started",\n'
    '        "Indicator analysis finished",\n'
    '    ]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("error", [ValueError("setup failed"), KeyboardInterrupt()])\n'
    'def test_setup_failure_and_interrupt(alerts, monkeypatch, error):\n'
    '    def fail(args):\n'
    '        raise error\n'
    '\n'
    '    monkeypatch.setattr(runner, "_run", fail)\n'
    '    with pytest.raises(type(error)):\n'
    '        runner.main([])\n'
    '    assert [a[1] for a, kw in alerts] == [\n'
    '        "Pipeline started",\n'
    '        (\n'
    '            "Pipeline stopped"\n'
    '            if isinstance(error, KeyboardInterrupt)\n'
    '            else "Pipeline failed"\n'
    '        ),\n'
    '    ]\n'
    '    assert type(error).__name__ in alerts[-1][0][0]\n'
    '\n'
    '\n'
    'def test_stage_failure_alert_once(alerts, monkeypatch, tmp_path):\n'
    '    credential = tmp_path / "credentials.json"\n'
    '    credential.write_text("{}")\n'
    '    monkeypatch.setattr(runner, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(runner, "dependencies", lambda: None)\n'
    '\n'
    '    def fail(*args):\n'
    '        raise RuntimeError("actual stage exception")\n'
    '\n'
    '    monkeypatch.setattr(runner, "run_stage", fail)\n'
    '    assert runner.main(["--credentials", str(credential), "--api-key", "fake"]) == 1\n'
    '    assert [a[1] for a, kw in alerts if "— stage" not in a[1]] == [\n'
    '        "Pipeline started",\n'
    '        "Pipeline failed",\n'
    '    ]\n'
    '    assert "actual stage exception" in alerts[-1][0][0]\n'
    '\n'
    '\n'
    'def test_self_test_does_not_notify(alerts, monkeypatch):\n'
    '    monkeypatch.setattr(runner, "_run", lambda args: 0)\n'
    '    assert runner.main(["--self-test"]) == 0\n'
    '    assert alerts == []\n'
    '\n'
    '\n'
    'def test_child_exception_included(monkeypatch, tmp_path):\n'
    '    monkeypatch.setattr(runner, "ROOT", tmp_path)\n'
    '    with pytest.raises(RuntimeError, match="child failure detail"):\n'
    '        runner.run_stage(\n'
    '            1,\n'
    '            1,\n'
    '            "Test stage",\n'
    '            [sys.executable, "-c", \'raise ValueError("child failure detail")\'],\n'
    '            dict(os.environ),\n'
    '            io.StringIO(),\n'
    '        )\n'
    '\n'
    '\n'
    'def test_delivery_redaction_limits(monkeypatch):\n'
    '    monkeypatch.setattr(telegram.sys, "platform", "linux")\n'
    '    monkeypatch.setenv("TELEGRAM_BOT_TOKEN", "fake-token")\n'
    '    monkeypatch.setenv("TELEGRAM_CHAT_ID", "fake-user")\n'
    '\n'
    '    def deliver(request, timeout):\n'
    '        payload = parse_qs(request.data.decode())\n'
    '        assert timeout == 10\n'
    '        assert request.full_url == "https://api.telegram.org/botfake-token/sendMessage"\n'
    '        assert payload["chat_id"] == ["fake-user"]\n'
    '        assert request.get_method() == "POST"\n'
    '        assert len(payload["text"][0]) == 4096\n'
    '        assert "fake-token" not in payload["text"][0]\n'
    '        assert "fake-user" not in payload["text"][0]\n'
    '        assert payload["text"][0].startswith("[redacted] <plain>\\n")\n'
    '        assert "fred-secret" not in payload["text"][0]\n'
    '        return io.BytesIO(b\'{"ok":true}\')\n'
    '\n'
    '    monkeypatch.setattr(telegram, "urlopen", deliver)\n'
    '    assert telegram.send_telegram(\n'
    '        "fake-token fred-secret " + "x" * 5000,\n'
    '        title="fake-user <plain>",\n'
    '        secrets=("fred-secret",),\n'
    '    )\n'
    '\n'
    '\n'
    'def test_group_notifications_use_group_chat_id(monkeypatch):\n'
    '    monkeypatch.setattr(telegram.sys, "platform", "linux")\n'
    '    monkeypatch.setenv("TELEGRAM_BOT_TOKEN", "fake-token")\n'
    '    monkeypatch.setenv("TELEGRAM_CHAT_ID", "-1001234567890")\n'
    '    captured = []\n'
    '\n'
    '    def deliver(request, timeout):\n'
    '        captured.append(parse_qs(request.data.decode()))\n'
    '        return io.BytesIO(b\'{"ok":true}\')\n'
    '\n'
    '    monkeypatch.setattr(telegram, "urlopen", deliver)\n'
    '    assert telegram.send_telegram("Pipeline started")\n'
    '    assert captured[0]["chat_id"] == ["-1001234567890"]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "mode", ["missing", "missing_token", "timeout", "rejected", "invalid_json"]\n'
    ')\n'
    'def test_delivery_problems_nonfatal(monkeypatch, mode):\n'
    '    monkeypatch.setattr(telegram.sys, "platform", "linux")\n'
    '    monkeypatch.setenv("TELEGRAM_BOT_TOKEN", "fake-token")\n'
    '    monkeypatch.setenv("TELEGRAM_CHAT_ID", "fake-user")\n'
    '    if mode == "missing":\n'
    '        monkeypatch.delenv("TELEGRAM_CHAT_ID")\n'
    '\n'
    '    if mode == "missing_token":\n'
    '        monkeypatch.delenv("TELEGRAM_BOT_TOKEN")\n'
    '\n'
    '    def deliver(*a, **kw):\n'
    '        if mode in ("missing", "missing_token"):\n'
    '            pytest.fail("Missing credentials must not make a request")\n'
    '        if mode == "timeout":\n'
    '            raise TimeoutError("network error")\n'
    '        if mode == "invalid_json":\n'
    '            return io.BytesIO(b"invalid JSON")\n'
    '        return io.BytesIO(b\'{"ok":false}\')\n'
    '\n'
    '    monkeypatch.setattr(telegram, "urlopen", deliver)\n'
    '    assert not telegram.send_telegram("test")\n'
    '\n'
    '\n'
    'def test_portable_sources_and_notebook_match(tmp_path):\n'
    '    root = Path(__file__).resolve().parents[1]\n'
    '    script = root / "portable" / "US_PIPELINE_COLAB.py"\n'
    '    if not script.exists():\n'
    '        pytest.skip("Portable export verification runs in the source project")\n'
    '    embedded = runpy.run_path(str(script), run_name="portable_test")\n'
    '    embedded["materialize"](tmp_path)\n'
    '    for name in embedded["SOURCES"]:\n'
    '        assert "\\\\" not in name, "Embedded paths must also work on macOS and Linux"\n'
    '        assert (tmp_path / name).read_text(encoding="utf-8") == (root / name).read_text(\n'
    '            encoding="utf-8"\n'
    '        )\n'
    '    notebook = json.loads(script.with_suffix(".ipynb").read_text(encoding="utf-8"))\n'
    '    assert "".join(notebook["cells"][0]["source"]) == script.read_text(encoding="utf-8")\n'
    '    assert \'"TELEGRAM_CHAT_ID", "TELEGRAM_BOT_TOKEN"\' in script.read_text(\n'
    '        encoding="utf-8"\n'
    '    )\n'
    '\n'
    '\n'
    'def test_colab_secrets_forwarded_without_duplicate_alerts(monkeypatch, tmp_path):\n'
    '    import subprocess\n'
    '    import types\n'
    '\n'
    '    root = Path(__file__).resolve().parents[1]\n'
    '    script = root / "portable" / "US_PIPELINE_COLAB.py"\n'
    '    if not script.exists():\n'
    '        pytest.skip("Portable export verification runs in the source project")\n'
    '    embedded = runpy.run_path(str(script), run_name="portable_test")\n'
    '    credential = tmp_path / "credential.json"\n'
    '    credential.write_text("{}")\n'
    '    secrets = {\n'
    '        "TELEGRAM_CHAT_ID": "colab-user",\n'
    '        "TELEGRAM_BOT_TOKEN": "colab-token",\n'
    '        "GOOGLE_APPLICATION_CREDENTIALS": str(credential),\n'
    '        "FRED_API_KEY": "fake-fred",\n'
    '    }\n'
    '    for key in secrets:\n'
    '        monkeypatch.delenv(key, raising=False)\n'
    '    colab = types.ModuleType("google.colab")\n'
    '    colab.userdata = types.SimpleNamespace(get=secrets.get)\n'
    '    monkeypatch.setitem(sys.modules, "google.colab", colab)\n'
    '    monkeypatch.chdir(tmp_path)\n'
    '    captured = []\n'
    '\n'
    '    def launch(command, env):\n'
    '        captured.append(env)\n'
    '        return types.SimpleNamespace(wait=lambda: 0)\n'
    '\n'
    '    monkeypatch.setattr(subprocess, "Popen", launch)\n'
    '    embedded["run_embedded"](notebook=True)\n'
    '    assert len(captured) == 1\n'
    '    assert captured[0]["TELEGRAM_CHAT_ID"] == "colab-user"\n'
    '    assert captured[0]["TELEGRAM_BOT_TOKEN"] == "colab-token"\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("stage,name", list(lifecycle.STAGE_NAMES.items()))\n'
    'def test_each_launcher_stage_has_tailored_title(alerts, monkeypatch, stage, name):\n'
    '    monkeypatch.setattr(runner, "_run", lambda args: 0)\n'
    '    assert runner.main(["--stage", stage]) == 0\n'
    '    assert [a[1] for a, kw in alerts] == [name + " started", name + " finished"]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("failed", [False, True])\n'
    'def test_direct_indicators_on_mac(alerts, monkeypatch, tmp_path, failed):\n'
    '    import types\n'
    '    import monthly_indicators as indicators\n'
    '\n'
    '    monkeypatch.setattr(sys, "argv", ["monthly_indicators.py"])\n'
    '    monkeypatch.setattr(lifecycle.platform, "system", lambda: "Darwin")\n'
    '    monkeypatch.setattr(lifecycle.platform, "node", lambda: "Test-Mac")\n'
    '    monkeypatch.setattr(indicators, "ERROR_LOG_PATH", tmp_path / "errors.log")\n'
    '    monkeypatch.setattr(indicators.logging, "basicConfig", lambda **kw: None)\n'
    '    monkeypatch.setattr(\n'
    '        indicators,\n'
    '        "parse_args",\n'
    '        lambda: types.SimpleNamespace(audit_google_sheet=False),\n'
    '    )\n'
    '\n'
    '    def run(args):\n'
    '        if failed:\n'
    '            raise ValueError("Indicator download failed")\n'
    '        return 0\n'
    '\n'
    '    monkeypatch.setattr(indicators, "run", run)\n'
    '    assert indicators.main() == (1 if failed else 0)\n'
    '    assert [a[1] for a, kw in alerts] == [\n'
    '        "Indicators started",\n'
    '        "Indicators failed" if failed else "Indicators finished",\n'
    '    ]\n'
    '    assert "Mac (Test-Mac)" in alerts[0][0][0]\n'
    '    if failed:\n'
    '        assert "Indicator download failed" in alerts[-1][0][0]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "flag,name",\n'
    '    [\n'
    '        ("--correlation", "Correlation"),\n'
    '        ("--spread", "Spread"),\n'
    '        ("--momentum", "Momentum"),\n'
    '        ("--spread-momentum", "Spread momentum"),\n'
    '        ("--correlation-momentum", "Correlation momentum"),\n'
    '    ],\n'
    ')\n'
    '@pytest.mark.parametrize("failed", [False, True])\n'
    'def test_direct_relationship_entrypoint(\n'
    '    alerts, monkeypatch, tmp_path, flag, name, failed\n'
    '):\n'
    '    import run_relationships as relationships\n'
    '\n'
    '    monkeypatch.setattr(relationships, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(relationships, "select_workbooks", lambda *a: [])\n'
    '    monkeypatch.setattr(relationships, "credentials_path", lambda *a: "unused")\n'
    '\n'
    '    def sheets(*a, **kw):\n'
    '        if failed:\n'
    '            raise ValueError("Cannot open workbook")\n'
    '        return object()\n'
    '\n'
    '    monkeypatch.setattr(relationships, "Sheets", sheets)\n'
    '    assert relationships.main([flag, "--dataset", "leading"]) == (1 if failed else 0)\n'
    '    assert [a[1] for a, kw in alerts] == [\n'
    '        name + " started",\n'
    '        name + (" failed" if failed else " finished"),\n'
    '    ]\n'
    '    assert "Dataset: Leading" in alerts[0][0][0]\n'
    '    if failed:\n'
    '        assert "Cannot open workbook" in alerts[-1][0][0]\n'
    '\n'
    '\n'
    'def test_direct_analysis_reports_batch_error(alerts, monkeypatch):\n'
    '    import us_indicator_analysis as analysis\n'
    '\n'
    '    monkeypatch.setattr(analysis, "open_workbook", lambda *a: object())\n'
    '    monkeypatch.setattr(analysis, "build_indicator_specs", lambda *a: [object()])\n'
    '    monkeypatch.setattr(analysis, "filter_indicator_specs", lambda specs, *a: specs)\n'
    '    monkeypatch.setattr(\n'
    '        analysis,\n'
    '        "process_indicators_batch",\n'
    '        lambda *a: (\n'
    '            [analysis.BatchIndicatorResult("GDP", False, error="Missing data")],\n'
    '            None,\n'
    '        ),\n'
    '    )\n'
    '    monkeypatch.setattr(analysis, "print_batch_summary", lambda *a: None)\n'
    '    assert analysis.main(["--all"]) == 1\n'
    '    assert [a[1] for a, kw in alerts] == [\n'
    '        "Indicator analysis started",\n'
    '        "Indicator analysis failed",\n'
    '    ]\n'
    '    assert "GDP: Missing data" in alerts[-1][0][0]\n'
    '\n'
    '\n'
    'def test_open_workbook_uses_quota_backoff(monkeypatch):\n'
    '    import us_indicator_analysis as analysis\n'
    '\n'
    '    opened = object()\n'
    '    calls = []\n'
    '\n'
    '    class Client:\n'
    '        def open_by_key(self, spreadsheet_id):\n'
    '            calls.append(("open", spreadsheet_id))\n'
    '            return opened\n'
    '\n'
    '    fake_gspread = types.SimpleNamespace(\n'
    '        service_account=lambda *, filename: calls.append(("credentials", filename))\n'
    '        or Client()\n'
    '    )\n'
    '    monkeypatch.setitem(sys.modules, "gspread", fake_gspread)\n'
    '    backoff_calls = []\n'
    '\n'
    '    def backoff(fn, **kwargs):\n'
    '        backoff_calls.append(kwargs)\n'
    '        return fn()\n'
    '\n'
    '    monkeypatch.setattr(analysis, "call_with_backoff", backoff)\n'
    '\n'
    '    result = analysis.open_workbook(analysis.EngineConfig("sheet-id", "credentials.json"))\n'
    '\n'
    '    assert result is opened\n'
    '    assert calls == [("credentials", "credentials.json"), ("open", "sheet-id")]\n'
    '    assert backoff_calls == [{"operation_name": "open_workbook"}]\n'
    '\n'
    '\n'
    'def test_parent_suppresses_child_notifications(alerts, monkeypatch, tmp_path):\n'
    '    credential = tmp_path / "credentials.json"\n'
    '    credential.write_text("{}")\n'
    '    monkeypatch.setattr(runner, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(runner, "dependencies", lambda: None)\n'
    '\n'
    '    def stage(index, total, name, command, env, log):\n'
    '        assert env["US_PIPELINE_NOTIFICATIONS_MANAGED"] == "1"\n'
    '        return {"stage": name}\n'
    '\n'
    '    monkeypatch.setattr(runner, "run_stage", stage)\n'
    '    assert runner.main(["--credentials", str(credential), "--api-key", "fake"]) == 0\n'
    '    assert [a[1] for a, kw in alerts if "— stage" not in a[1]] == [\n'
    '        "Pipeline started",\n'
    '        "Pipeline finished",\n'
    '    ]\n'
    '    assert len(alerts) == 2\n'
    '    assert "Stages completed: 3/3" in alerts[-1][0][0]\n'
    '    assert "Details: /status or /logs" in alerts[-1][0][0]\n'
    '    alerts.clear()\n'
    '    monkeypatch.setenv("US_PIPELINE_NOTIFICATIONS_MANAGED", "1")\n'
    '    assert lifecycle.run_notified("Indicators", lambda: 0) == 0\n'
    '    assert alerts == []\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("flag", ["--help", "--self-test"])\n'
    'def test_direct_entrypoint_help_and_tests_are_quiet(alerts, monkeypatch, flag):\n'
    '    @lifecycle.notify_entrypoint("Indicators")\n'
    '    def fake(argv):\n'
    '        return 0\n'
    '\n'
    '    assert fake([flag]) == 0\n'
    '    assert alerts == []\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("exit_code", [0, 2, "Credentials missing"])\n'
    'def test_system_exit_preserved_and_reported(alerts, exit_code):\n'
    '    def action():\n'
    '        raise SystemExit(exit_code)\n'
    '\n'
    '    with pytest.raises(SystemExit) as result:\n'
    '        lifecycle.run_notified("Indicators", action)\n'
    '    assert result.value.code == exit_code\n'
    '    assert alerts[-1][0][1] == (\n'
    '        "Indicators finished" if exit_code == 0 else "Indicators failed"\n'
    '    )\n'
    '\n'
    '\n'
    'def test_run_metadata_and_context_reset(alerts):\n'
    '    lifecycle.run_notified("Indicators", lambda: 0, argv=["--dataset", "leading"])\n'
    '    start, finish = [args[0] for args, _ in alerts]\n'
    '    for label in ("Scope:", "Dataset:", "Run "):\n'
    '        assert label in start and label in finish\n'
    '    assert "Started:" in start\n'
    '    assert "Duration:" in finish and "Finished:" in finish\n'
    '    assert start.splitlines()[-1] == finish.splitlines()[-1]\n'
    '    assert lifecycle._active.get() is None\n'
    '\n'
    '\n'
    'def test_long_error_retains_end_and_redacts(monkeypatch):\n'
    '    monkeypatch.setattr(\n'
    '        telegram,\n'
    '        "credential",\n'
    '        lambda name: "token-secret" if name.endswith("TOKEN") else "chat-secret",\n'
    '    )\n'
    '    payloads = []\n'
    '\n'
    '    def deliver(request, timeout):\n'
    '        payloads.append(parse_qs(request.data.decode())["text"][0])\n'
    '        return io.BytesIO(b\'{"ok":true}\')\n'
    '\n'
    '    monkeypatch.setattr(telegram, "urlopen", deliver)\n'
    '    assert telegram.send_telegram(\n'
    '        "x" * 6000 + " final error token-secret", "Run details"\n'
    '    )\n'
    '    assert len(payloads[0]) == 4096\n'
    '    assert payloads[0].startswith("Run details")\n'
    '    assert payloads[0].endswith("final error [redacted]")\n'
    '\n'
    '\n'
    'def test_windows_persistent_credentials(monkeypatch):\n'
    '    import types\n'
    '\n'
    '    monkeypatch.delenv("TELEGRAM_BOT_TOKEN", raising=False)\n'
    '    monkeypatch.setattr(telegram.sys, "platform", "win32")\n'
    '\n'
    '    class Key:\n'
    '        def __enter__(self):\n'
    '            return self\n'
    '\n'
    '        def __exit__(self, *args):\n'
    '            pass\n'
    '\n'
    '    monkeypatch.setitem(\n'
    '        sys.modules,\n'
    '        "winreg",\n'
    '        types.SimpleNamespace(\n'
    '            HKEY_CURRENT_USER=1,\n'
    '            OpenKey=lambda *args: Key(),\n'
    '            QueryValueEx=lambda key, name: ("saved-token", 1),\n'
    '        ),\n'
    '    )\n'
    '    assert telegram.credential("TELEGRAM_BOT_TOKEN") == "saved-token"\n'
    '    monkeypatch.setenv("TELEGRAM_BOT_TOKEN", "override")\n'
    '    assert telegram.credential("TELEGRAM_BOT_TOKEN") == "override"\n'
)

# Embedded source: tests/test_telegram_control.py
SOURCES['tests/test_telegram_control.py'] = (
    'import json\n'
    'import time\n'
    '\n'
    'import pytest\n'
    '\n'
    'from notifications import control\n'
    'from run_lock import acquire_lock\n'
    '\n'
    '\n'
    '@pytest.fixture\n'
    'def controller(tmp_path, monkeypatch):\n'
    '    replies = []\n'
    '    launches = []\n'
    '    monkeypatch.setattr(control, "WORK", tmp_path)\n'
    '    monkeypatch.setattr(\n'
    '        control, "send_telegram", lambda text, title: replies.append(text)\n'
    '    )\n'
    '    monkeypatch.setattr(control, "task_state", lambda: "Ready")\n'
    '    monkeypatch.setattr(\n'
    '        control, "start_pipeline", lambda stage="all": launches.append(stage)\n'
    '    )\n'
    '    return control.Controller("-100123", "123, 456", "test_bot"), replies, launches\n'
    '\n'
    '\n'
    'def update(text="/run", uid=1, user=123, chat=-100123, age=0):\n'
    '    return {\n'
    '        "update_id": uid,\n'
    '        "message": {\n'
    '            "chat": {"id": chat, "type": "supergroup" if chat < 0 else "private"},\n'
    '            "from": {"id": user, "is_bot": False},\n'
    '            "date": time.time() - age,\n'
    '            "text": text,\n'
    '        },\n'
    '    }\n'
    '\n'
    '\n'
    'def test_run_and_replay_protection(controller):\n'
    '    handler, replies, launches = controller\n'
    '    handler.handle(update())\n'
    '    handler.handle(update())\n'
    '    restarted = control.Controller("-100123", "123,456", "test_bot")\n'
    '    restarted.handle(update())\n'
    '    assert len(launches) == 1\n'
    '    assert replies == []  # Only the pipeline itself sends the start alert.\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "message",\n'
    '    [\n'
    '        update(user=999),\n'
    '        update(chat=999),\n'
    '        update("/run@other_bot"),\n'
    '        update("/run", age=180),\n'
    '        update("/run ; calc.exe"),\n'
    '        {"update_id": 1, "edited_message": update()["message"]},\n'
    '    ],\n'
    ')\n'
    'def test_untrusted_and_stale_commands_cannot_launch(controller, message):\n'
    '    handler, _, launches = controller\n'
    '    handler.handle(message)\n'
    '    assert not launches\n'
    '\n'
    '\n'
    'def test_running_task_blocks_launch(controller, monkeypatch):\n'
    '    handler, replies, launches = controller\n'
    '    monkeypatch.setattr(control, "task_state", lambda: "Running")\n'
    '    handler.handle(update())\n'
    '    assert not launches\n'
    '    assert "Already running" in replies[0]\n'
    '\n'
    '\n'
    'def test_manual_run_lock_blocks_launch(controller):\n'
    '    handler, replies, launches = controller\n'
    '    with (control.WORK / "complete_pipeline.lock").open("a+") as lock:\n'
    '        acquire_lock(lock)\n'
    '        handler.handle(update())\n'
    '    assert not launches\n'
    '    assert "Already running" in replies[0]\n'
    '\n'
    '\n'
    'def test_rapid_commands_do_not_launch_twice(controller):\n'
    '    handler, _, launches = controller\n'
    '    handler.handle(update())\n'
    '    handler.handle(update(uid=2))\n'
    '    assert len(launches) == 1\n'
    '\n'
    '\n'
    'def test_status_and_help_do_not_launch(controller):\n'
    '    handler, replies, launches = controller\n'
    '    handler.handle(update("/status"))\n'
    '    handler.handle(update("/help@test_bot", uid=2))\n'
    '    assert "idle" in replies[0]\n'
    '    assert "/run" in replies[1]\n'
    '    assert not launches\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("command", ["/us_cot_import", "/cot", "us-cot-import"])\n'
    'def test_cot_commands_launch_scheduled_runner(controller, monkeypatch, command):\n'
    '    handler, replies, launches = controller\n'
    '    cot_launches = []\n'
    '    monkeypatch.setattr(control, "cot_busy", lambda: False)\n'
    '    monkeypatch.setattr(control, "start_cot_import", lambda: cot_launches.append(True))\n'
    '    handler.handle(update(command))\n'
    '    assert cot_launches == [True]\n'
    '    assert not replies and not launches\n'
    '\n'
    '\n'
    'def test_cot_status_command_reports_without_launch(controller, monkeypatch):\n'
    '    handler, replies, launches = controller\n'
    '    monkeypatch.setattr(control, "cot_status", lambda: "COT import: idle (task: Ready).")\n'
    '    handler.handle(update("/us_cot_status"))\n'
    '    assert replies == ["COT import: idle (task: Ready)."]\n'
    '    assert not launches\n'
    '\n'
    '\n'
    'def test_cot_command_rejects_duplicate_run(controller, monkeypatch):\n'
    '    handler, replies, launches = controller\n'
    '    monkeypatch.setattr(control, "cot_busy", lambda: True)\n'
    '    monkeypatch.setattr(control, "cot_status", lambda: "COT import: running.")\n'
    '    monkeypatch.setattr(\n'
    '        control, "start_cot_import", lambda: pytest.fail("duplicate COT run launched")\n'
    '    )\n'
    '    handler.handle(update("/cot"))\n'
    '    assert "Already running" in replies[0]\n'
    '    assert not launches\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    ("command", "stage"),\n'
    '    [\n'
    '        ("/us_stock_run", "all"),\n'
    '        ("/us-stock-primary", "primary"),\n'
    '        ("/us_single_stock_secondary", "secondary"),\n'
    '        ("us-stock-summary", "summary"),\n'
    '    ],\n'
    ')\n'
    'def test_single_stock_commands_launch_only_fixed_stages(controller, monkeypatch, command, stage):\n'
    '    handler, replies, launches = controller\n'
    '    single_stock_launches = []\n'
    '    monkeypatch.setattr(control, "single_stock_busy", lambda: False)\n'
    '    monkeypatch.setattr(\n'
    '        control, "start_single_stock_pipeline",\n'
    '        lambda requested: single_stock_launches.append(requested),\n'
    '    )\n'
    '    handler.handle(update(command))\n'
    '    assert single_stock_launches == [stage]\n'
    '    assert not replies and not launches\n'
    '\n'
    '\n'
    'def test_single_stock_status_and_stop_do_not_launch(controller, monkeypatch):\n'
    '    handler, replies, launches = controller\n'
    '    monkeypatch.setattr(control, "single_stock_status", lambda: "Single-stock pipeline: idle.")\n'
    '    monkeypatch.setattr(control, "stop_single_stock_pipeline", lambda: "Stop requested.")\n'
    '    handler.handle(update("/us_stock_status"))\n'
    '    handler.handle(update("/us_stock_stop", uid=2))\n'
    '    assert replies == ["Single-stock pipeline: idle.", "Stop requested."]\n'
    '    assert not launches\n'
    '\n'
    '\n'
    'def test_single_stock_dispatch_uses_only_fixed_launcher(monkeypatch):\n'
    '    import types\n'
    '\n'
    '    calls = []\n'
    '    monkeypatch.setattr(control, "credential", lambda key: "configured")\n'
    '    monkeypatch.setattr(\n'
    '        control.subprocess,\n'
    '        "Popen",\n'
    '        lambda command, **kw: calls.append((command, kw))\n'
    '        or types.SimpleNamespace(poll=lambda: 0),\n'
    '    )\n'
    '    monkeypatch.setattr(control.Path, "is_file", lambda path: True)\n'
    '    control.start_single_stock_pipeline("summary")\n'
    '    assert calls[0][0] == [\n'
    '        "cmd.exe", "/d", "/c", str(control.SINGLE_STOCK_LAUNCHER), "--stage", "summary",\n'
    '    ]\n'
    '    with pytest.raises(ValueError):\n'
    '        control.start_single_stock_pipeline("summary & calc.exe")\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("command", ["/backup", "/fund_backup", "fund-server-backup"])\n'
    'def test_backup_commands_start_the_existing_backup_task(controller, monkeypatch, command):\n'
    '    handler, replies, launches = controller\n'
    '    backup_launches = []\n'
    '    monkeypatch.setattr(control, "backup_busy", lambda: False)\n'
    '    monkeypatch.setattr(control, "start_backup", lambda: backup_launches.append(True))\n'
    '    handler.handle(update(command))\n'
    '    assert backup_launches == [True]\n'
    '    assert not replies and not launches\n'
    '\n'
    '\n'
    'def test_backup_command_rejects_duplicate_run(controller, monkeypatch):\n'
    '    handler, replies, launches = controller\n'
    '    monkeypatch.setattr(control, "backup_busy", lambda: True)\n'
    '    monkeypatch.setattr(\n'
    '        control, "start_backup", lambda: pytest.fail("duplicate backup run launched")\n'
    '    )\n'
    '    handler.handle(update("/backup"))\n'
    '    assert "already running" in replies[0].lower()\n'
    '    assert not launches\n'
    '\n'
    '\n'
    'def test_start_backup_uses_the_existing_scheduled_task(monkeypatch):\n'
    '    calls = []\n'
    '    monkeypatch.setattr(control, "powershell", lambda script: calls.append(script))\n'
    '    control.start_backup()\n'
    '    assert calls == [\n'
    '        "Start-ScheduledTask -TaskName \'Fund Server GitHub Backup\'"\n'
    '    ]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("user", [123, 456])\n'
    'def test_both_group_users_can_request_status(controller, user):\n'
    '    handler, replies, launches = controller\n'
    '    handler.handle(update("/status", user=user))\n'
    '    assert len(replies) == 1 and "idle" in replies[0]\n'
    '    assert not launches\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("user", [123, 456])\n'
    'def test_both_group_users_can_use_stage_commands(controller, user):\n'
    '    handler, replies, launches = controller\n'
    '    handler.handle(update("/analysis", user=user))\n'
    '    assert launches == ["analysis"]\n'
    '    assert not replies\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("user,chat", [(999, -100123), (123, 123), (456, 456), (123, -100999), (456, -100999)])\n'
    'def test_group_only_identity_checks(controller, user, chat):\n'
    '    handler, replies, launches = controller\n'
    '    handler.handle(update("/status", user=user, chat=chat))\n'
    '    assert not replies and not launches\n'
    '\n'
    '\n'
    'def test_bot_forward_and_wrong_chat_type_are_ignored(controller):\n'
    '    handler, replies, launches = controller\n'
    '    bot = update("/status")\n'
    '    bot["message"]["from"]["is_bot"] = True\n'
    '    handler.handle(bot)\n'
    '    forwarded = update("/status", uid=2)\n'
    '    forwarded["message"]["forward_origin"] = {"type": "user"}\n'
    '    handler.handle(forwarded)\n'
    '    wrong_type = update("/status", uid=3)\n'
    '    wrong_type["message"]["chat"]["type"] = "private"\n'
    '    handler.handle(wrong_type)\n'
    '    assert not replies and not launches\n'
    '\n'
    '\n'
    'def test_allowed_user_ids_validation():\n'
    '    assert control.parse_allowed_user_ids(" 123 , 456,123, 00123 ") == frozenset({"123", "456"})\n'
    '    for raw in ("", "0", "-1", "123,", ",123", "123,abc", "1.0", "1 2"):\n'
    '        with pytest.raises(ValueError, match="TELEGRAM_ALLOWED_USER_IDS"):\n'
    '            control.parse_allowed_user_ids(raw)\n'
    '\n'
    '\n'
    'def test_chat_id_never_authorises_user(tmp_path):\n'
    '    handler = control.Controller("-100123", "456", "test_bot", tmp_path / "offset.json")\n'
    '    assert "100123" not in handler.user_ids\n'
    '    assert "456" in handler.user_ids\n'
    '    with pytest.raises(ValueError, match="negative group ID"):\n'
    '        control.Controller("123", "123", "test_bot", tmp_path / "other.json")\n'
    '\n'
    '\n'
    'def test_legacy_single_user_setting_is_not_used(monkeypatch):\n'
    '    monkeypatch.setattr(control.sys, "platform", "win32")\n'
    '    settings = {"TELEGRAM_BOT_TOKEN": "token", "TELEGRAM_CHAT_ID": "-100123", "TELEGRAM_ALLOWED_USER_ID": "123"}\n'
    '    monkeypatch.setattr(control, "credential", lambda key: settings.get(key, ""))\n'
    '    with pytest.raises(RuntimeError, match="TELEGRAM_ALLOWED_USER_IDS"):\n'
    '        control.main()\n'
    '\n'
    '\n'
    'def test_status_reports_progress_and_stale_run(controller):\n'
    '    run = control.WORK / "20260919T123000Z"\n'
    '    run.mkdir()\n'
    '    (run / "summary.json").write_text(\n'
    '        json.dumps(\n'
    '            {\n'
    '                "status": "running",\n'
    '                "dataset": "all",\n'
    '                "current_stage": "2/3 Analysis",\n'
    '                "stages": [{"stage": "Import", "seconds": 12}],\n'
    '            }\n'
    '        )\n'
    '    )\n'
    '    message = control.status()\n'
    '    assert "2/3 Analysis" in message\n'
    '    assert "Import: 12s" in message\n'
    '    assert "may have been interrupted" in message\n'
    '\n'
    '\n'
    'def test_task_failure_is_reported_without_raw_error(controller, monkeypatch):\n'
    '    handler, replies, _ = controller\n'
    '\n'
    '    def fail(stage="all"):\n'
    '        raise RuntimeError("secret-token")\n'
    '\n'
    '    monkeypatch.setattr(control, "start_pipeline", fail)\n'
    '    handler.handle(update())\n'
    '    assert "Could not complete" in replies[0]\n'
    '    assert "secret-token" not in replies[0]\n'
    '\n'
    '\n'
    'def test_dispatch_uses_only_fixed_launcher(monkeypatch):\n'
    '    import types\n'
    '\n'
    '    calls = []\n'
    '    monkeypatch.setattr(control, "credential", lambda key: "configured")\n'
    '    monkeypatch.setattr(\n'
    '        control.subprocess,\n'
    '        "Popen",\n'
    '        lambda command, **kw: calls.append((command, kw))\n'
    '        or types.SimpleNamespace(poll=lambda: 0),\n'
    '    )\n'
    '    monkeypatch.setattr(control.Path, "is_file", lambda path: True)\n'
    '    control.start_pipeline()\n'
    '    assert calls[0][0] == [\n'
    '        "cmd.exe",\n'
    '        "/d",\n'
    '        "/c",\n'
    '        str(control.LAUNCHER),\n'
    '        "--stage",\n'
    '        "all",\n'
    '    ]\n'
    '    assert calls[0][1]["stdin"] == control.subprocess.DEVNULL\n'
    '\n'
    '\n'
    '@pytest.mark.skipif(control.sys.platform != "win32", reason="Windows launcher")\n'
    'def test_real_windows_launcher_with_harmless_job(tmp_path, monkeypatch):\n'
    '    launcher = tmp_path / "test_job.bat"\n'
    '    marker = tmp_path / "launched.txt"\n'
    '    launcher.write_text(\'@echo off\\necho launched>"\' + str(marker) + \'"\\n\')\n'
    '    monkeypatch.setattr(control, "LAUNCHER", launcher)\n'
    '    monkeypatch.setattr(control, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(control, "credential", lambda key: "")\n'
    '    monkeypatch.setattr(control, "_child", None)\n'
    '    control.start_pipeline()\n'
    '    assert control._child.wait(timeout=10) == 0\n'
    '    assert marker.read_text().strip() == "launched"\n'
    '\n'
    '\n'
    'def test_runner_lock_skips_before_calculations(tmp_path, monkeypatch):\n'
    '    import run_us_pipeline as runner\n'
    '    from notifications import telegram\n'
    '\n'
    '    replies = []\n'
    '    monkeypatch.setattr(runner, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(\n'
    '        runner, "_run", lambda args: pytest.fail("Must not enter calculations")\n'
    '    )\n'
    '    monkeypatch.setattr(telegram, "send_telegram", lambda *args: replies.append(args))\n'
    '    (tmp_path / "work").mkdir()\n'
    '    with (tmp_path / "work" / "complete_pipeline.lock").open("a+") as lock:\n'
    '        acquire_lock(lock)\n'
    '        assert runner.main([]) == 0\n'
    '    assert "already running" in replies[0][1]\n'
    '\n'
    '\n'
    'def test_status_skips_other_summary_formats(controller):\n'
    '    for name, report in [\n'
    '        ("20260919T120000Z", {"status": "completed", "stages": []}),\n'
    '        ("20260919T130000Z", [{"other": "analysis output"}]),\n'
    '    ]:\n'
    '        folder = control.WORK / name\n'
    '        folder.mkdir()\n'
    '        (folder / "summary.json").write_text(json.dumps(report))\n'
    '    assert "Recorded status: completed" in control.status()\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize("name,stage", list(control.STAGE_COMMANDS.items()))\n'
    '@pytest.mark.parametrize("style", ["hyphen", "slash", "underscore"])\n'
    'def test_every_requested_stage_alias(controller, name, stage, style):\n'
    '    handler, _, launches = controller\n'
    '    command = "us-macroanalysis-" + name\n'
    '    if style == "slash":\n'
    '        command = "/" + command\n'
    '    elif style == "underscore":\n'
    '        command = "/" + command.replace("-", "_")\n'
    '    handler.handle(update(command))\n'
    '    assert launches == [stage]\n'
    '\n'
    '\n'
    'def test_menu_names_meet_telegram_limits(controller):\n'
    '    import re\n'
    '\n'
    '    for item in control.MENU:\n'
    '        assert re.fullmatch("[a-z0-9_]{1,32}", item["command"])\n'
    '    handler, _, launches = controller\n'
    '    handler.handle(update("/us_macro_correlation_momentum"))\n'
    '    assert launches == ["correlation-momentum"]\n'
    '\n'
    '\n'
    '@pytest.mark.parametrize(\n'
    '    "name", [\n'
    '        "pythonstopall", "us-macroanalysis-stop", "us-macroanalysis-logs",\n'
    '        "us-cot-import", "us-cot-status",\n'
    '    ]\n'
    ')\n'
    'def test_new_commands_reject_other_users(controller, monkeypatch, name):\n'
    '    def forbidden(*args, **kwargs):\n'
    '        pytest.fail("Unauthorised command caused a side effect")\n'
    '\n'
    '    monkeypatch.setattr(control, "stop_all_python", forbidden)\n'
    '    monkeypatch.setattr(control, "stop_pipeline", forbidden)\n'
    '    monkeypatch.setattr(control, "api", forbidden)\n'
    '    handler, replies, launches = controller\n'
    '    handler.handle(update(name, user=999))\n'
    '    assert not replies and not launches\n'
    '\n'
    '\n'
    'def test_stop_is_bound_to_the_active_run(controller):\n'
    '    run_id = "a" * 32\n'
    '    (control.WORK / "active-pipeline.json").write_text(json.dumps({"run_id": run_id}))\n'
    '    with (control.WORK / "complete_pipeline.lock").open("a+") as lock:\n'
    '        acquire_lock(lock)\n'
    '        reply = control.stop_pipeline()\n'
    '    assert "Stop requested" in reply\n'
    '    assert (control.WORK / f"stop-{run_id}.request").exists()\n'
    '    assert "No active" in control.stop_pipeline()\n'
    '\n'
    '\n'
    'def test_stop_rejects_invalid_run_identity(controller):\n'
    '    (control.WORK / "active-pipeline.json").write_text(\n'
    '        json.dumps({"run_id": "../../bad"})\n'
    '    )\n'
    '    with (control.WORK / "complete_pipeline.lock").open("a+") as lock:\n'
    '        acquire_lock(lock)\n'
    '        assert "No processes were stopped" in control.stop_pipeline()\n'
    '\n'
    '\n'
    'def test_kill_all_requires_elevation_and_acknowledgement(controller, monkeypatch):\n'
    '    handler, replies, _ = controller\n'
    '    kills = []\n'
    '    monkeypatch.setattr(control, "is_admin", lambda: False)\n'
    '    monkeypatch.setattr(control, "stop_all_python", lambda: kills.append(True))\n'
    '    handler.handle(update("pythonstopall"))\n'
    '    assert not kills\n'
    '    assert "administrator" in replies[-1]\n'
    '    monkeypatch.setattr(control, "is_admin", lambda: True)\n'
    '    monkeypatch.setattr(control, "send_telegram", lambda *args: False)\n'
    '    handler.handle(update("pythonstopall", uid=2))\n'
    '    assert not kills\n'
    '    monkeypatch.setattr(control, "send_telegram", lambda *args: True)\n'
    '    handler.handle(update("pythonstopall", uid=3))\n'
    '    assert kills == [True]\n'
    '    handler.handle(update("pythonstopall", uid=3))\n'
    '    assert kills == [True]\n'
    '\n'
    '\n'
    'def test_log_watch_edits_one_message_and_can_stop(controller, monkeypatch, tmp_path):\n'
    '    handler, _, _ = controller\n'
    '    calls = []\n'
    '    path = tmp_path / "pipeline.log"\n'
    '    path.write_text("first output secret-key")\n'
    '    monkeypatch.setattr(control, "LOG_PATH", path)\n'
    '    monkeypatch.setattr(control, "credential", lambda key: "secret-key")\n'
    '    monkeypatch.setattr(\n'
    '        control,\n'
    '        "api",\n'
    '        lambda method, **kw: calls.append((method, kw)) or {"message_id": 42},\n'
    '    )\n'
    '    handler.handle(update("us-macroanalysis-logs"))\n'
    '    assert calls[0][0] == "sendMessage"\n'
    '    assert "secret-key" not in calls[0][1]["text"]\n'
    '    path.write_text("new output")\n'
    '    handler.log_watch["next"] = 0\n'
    '    handler.update_logs()\n'
    '    assert calls[-1][0] == "editMessageText"\n'
    '    assert calls[-1][1]["message_id"] == 42\n'
    '    assert "new output" in calls[-1][1]["text"]\n'
    '    handler.handle(update("/logs", uid=2))\n'
    '    assert handler.log_watch is None\n'
    '\n'
    '\n'
    'def test_log_watch_continues_indefinitely_and_checks_every_30_seconds(\n'
    '    controller, monkeypatch\n'
    '):\n'
    '    handler, _, _ = controller\n'
    '    calls = []\n'
    '    now = [100.0]\n'
    '    monkeypatch.setattr(control.time, "monotonic", lambda: now[0])\n'
    '    monkeypatch.setattr(\n'
    '        control,\n'
    '        "api",\n'
    '        lambda *args, **kw: calls.append((args, kw)) or {"message_id": 42},\n'
    '    )\n'
    '    monkeypatch.setattr(control, "log_tail", lambda: str(now[0]))\n'
    '    handler.toggle_logs()\n'
    '    assert handler.log_watch["next"] == 130\n'
    '    now[0] = 129\n'
    '    handler.update_logs()\n'
    '    assert len(calls) == 1\n'
    '    now[0] = 130\n'
    '    handler.update_logs()\n'
    '    assert len(calls) == 2\n'
    '    assert handler.log_watch["next"] == 160\n'
    '    now[0] = 1000000\n'
    '    handler.update_logs()\n'
    '    assert handler.log_watch is not None\n'
    '    assert handler.log_watch["next"] == 1000030\n'
    '\n'
    '\n'
    '@pytest.mark.skipif(control.sys.platform != "win32", reason="Windows process tree")\n'
    'def test_pipeline_stop_terminates_only_its_test_stage(tmp_path, monkeypatch):\n'
    '    import io\n'
    '    import os\n'
    '    import sys\n'
    '    import threading\n'
    '    import run_us_pipeline as runner\n'
    '\n'
    '    request = tmp_path / "stop.request"\n'
    '    monkeypatch.setattr(runner, "ROOT", tmp_path)\n'
    '    monkeypatch.setattr(runner, "STOP_REQUEST", request)\n'
    '    trigger = threading.Timer(1.5, request.touch)\n'
    '    trigger.start()\n'
    '    try:\n'
    '        with pytest.raises(runner.PipelineStopped):\n'
    '            runner.run_stage(\n'
    '                1,\n'
    '                1,\n'
    '                "Harmless test sleeper",\n'
    '                [\n'
    '                    sys.executable,\n'
    '                    "-u",\n'
    '                    "-c",\n'
    '                    "import time; print(\'ready\'); time.sleep(10)",\n'
    '                ],\n'
    '                dict(os.environ),\n'
    '                io.StringIO(),\n'
    '            )\n'
    '    finally:\n'
    '        trigger.cancel()\n'
    '        trigger.join()\n'
    '\n'
    '\n'
    'def test_stage_allowlist_rejects_shell_text(monkeypatch):\n'
    '    monkeypatch.setattr(\n'
    '        control.subprocess,\n'
    '        "Popen",\n'
    '        lambda *args, **kw: pytest.fail("Launched invalid stage"),\n'
    '    )\n'
    '    with pytest.raises(ValueError):\n'
    '        control.start_pipeline("all & calc.exe")\n'
)

# Embedded source: tests/test_telegram_group_setup.py
SOURCES['tests/test_telegram_group_setup.py'] = (
    '"""The migration consumes updates without running any pipeline command."""\n'
    '\n'
    'import json\n'
    '\n'
    'from tools import configure_telegram_group as setup\n'
    '\n'
    '\n'
    'def test_discovery_checkpoints_all_consumed_updates(monkeypatch, tmp_path):\n'
    '    path = tmp_path / "offset.json"\n'
    '    monkeypatch.setattr(setup, "OFFSET_PATH", path)\n'
    '    monkeypatch.setattr(setup, "credential", lambda name: "existing-token")\n'
    '    calls = []\n'
    '\n'
    '    def api(method, **payload):\n'
    '        calls.append((method, payload))\n'
    '        if method == "getWebhookInfo":\n'
    '            return {"url": ""}\n'
    '        if payload.get("offset") == -1:\n'
    '            return [{"update_id": 100, "message": {"text": "/run"}}]\n'
    '        if payload.get("offset") == 101:\n'
    '            return [\n'
    '                {"update_id": 101, "message": {"chat": {"id": 100, "type": "private"}, "from": {"id": 1}}},\n'
    '                {"update_id": 102, "message": {"chat": {"id": -100123, "title": "Fund", "type": "supergroup"}, "from": {"id": 1, "first_name": "Theo", "is_bot": False}}},\n'
    '                {"update_id": 103, "message": {"chat": {"id": -100123, "title": "Fund", "type": "supergroup"}, "from": {"id": 9, "is_bot": True}}},\n'
    '                {"update_id": 104, "message": {"chat": {"id": -100123, "title": "Fund", "type": "supergroup"}, "from": {"id": 2, "first_name": "Cofounder", "is_bot": False}}},\n'
    '            ]\n'
    '        return []\n'
    '\n'
    '    monkeypatch.setattr(setup, "api", api)\n'
    '    groups, offset = setup.discover(timeout_seconds=30)\n'
    '    assert offset == 105\n'
    '    assert json.loads(path.read_text()) == {"offset": 105}\n'
    '    assert set(groups) == {-100123}\n'
    '    assert set(groups[-100123]["users"]) == {1, 2}\n'
    '    assert calls[1][1]["offset"] == -1\n'
)

SOURCE_HASHES = {'run_us_pipeline.py': '228ab7547af33f27bc6b5b298608677c95ce1ede4857088b55fd8e4b556b7f18', 'monthly_indicators.py': '666502452ed9e857e6a8020693bc11a81e4ebd20ce6056c70220baa6a2c5a2de', 'us_indicator_analysis.py': 'a317c8b3988a96cbba2fe0358fddc80139671276512df3ee7071c7ed5f35ba41', 'run_relationships.py': '573b4465c6501a3312b2569d714e4a76b4fe5434e5ecfa6573024ab0caf4c765', 'config.py': '23b9e659231c77c413c967263fd5f3c22934fd3b3b92f2305cef656066361997', 'google_sheets.py': 'c686cc5e14ab8fb09a97b1d266035bc71e55be4e2f218709c0419d2020e7defe', 'pipeline.py': 'fbf88fe486bfb91c85b63750503340e59c71207f7501b8dca4cee85a03f7835c', 'momentum_pipeline.py': '1bbcb6c183edb97c46ed668460c2492a8b38177fddcdec837affff924ea1c2a2', 'calculation_helper.py': 'f5d69e49b9e1c4e2ed9fc1410f7f1f696cc24bdb1d06c5028f0c771a891e47ab', 'validation.py': 'db8d2c397ee4d5e8687e9175e509218a2b7bbb499d6990febe1dea00d7f90dad', 'spread.py': '1543056ea968d40f346a72e5823b5066aae87bba4cc35ceced53c39a69e8e1e0', 'correlation.py': 'eb63e7c18343826e939a308ce0947517f1708f85f92356a4dd25743f2676064b', 'momentum.py': '6c29fd410b897b6c4a07c794614465388a7e8e8b17648f72e68edd7611818f93', 'source_freshness.py': '2837ababe0bce57d14d57e020fb3c0f816e2488b4edbc843768da517d389b596', 'run_lock.py': 'ffaf9e38d9e026b113b5d23353764b085ce944df5d18cf986ca1cf2b2e0b47ed', 'requirements.txt': 'a723cc70515e5aab11a71705299cf0d6042915a84136e2505fdba96bbbebfd95', 'notifications/__init__.py': 'fde0fa24901b678294e1c05bb15cabb385f34c6f6473232977b94550ba163e05', 'notifications/telegram.py': 'bf48bfec288e3aa5e381fa860f9fb4967846bbefe05034f292a28db1efb85f67', 'notifications/lifecycle.py': '8f99705e2ec66221c01393e080fbb8556d42d7754f7fb2c2857695d9da527767', 'notifications/changes.py': '4816ce8a6b51221b6a2a949a22f8b948bfadc888f8d573a93cbf93d3e804e129', 'notifications/control.py': '859bfd510473fdd4c0ad202e37628cc2a9c4394b91724301955f2402099d85ea', 'tests/test_calculations.py': 'a0f020cac7954f09cc379370cdc144f1bde8d0a20dbac0594d1c17a04278399a', 'tests/test_change_notifications.py': 'ffefed189c1dbfc2cf5ebc709cf7abbe7c1f4a9a44c0b4be9945dbdad3ffac63', 'tests/test_complete_pipeline.py': '96460f96c85a1a36ab1d7545c0c9c7dc5dcbd14a28010dc26750913954012bfe', 'tests/test_datasets.py': 'd6c4cd77d2bed354af3e6626c045225f061b7e1fcf14e959919a534bd3be9293', 'tests/test_helpers_growth.py': 'f940a394d8fb8a364861bac26de35cb7392b2450d237a761e99587fccb50003a', 'tests/test_incremental_import.py': '6c4e12df52fdfde8b8377ba6fd90b21d80ae9a766bfa6b5e43764feb45d375ec', 'tests/test_io.py': 'd825bfbf426dbe338d172008e2a1e538138b8188bb26eefca7ff0e9e21243247', 'tests/test_ism_updates.py': 'c7433f1647e0f2c1f2b3bd4d5740fec40124ced7b586dae49eb65c948371812d', 'tests/test_momentum.py': '3df167c961555868a48df0beafb8df289c178d7138faf9ff8527a2c936a05302', 'tests/test_momentum_updates.py': '6a4578666bb9fb140f975be67c469e339171ab494acb17777e2f68a2add310ac', 'tests/test_request_progress.py': '400b066b8370a388c0f2e0cf60cbb85d2a2e79f19cd6bfa9d5340c172b68f7fa', 'tests/test_telegram.py': '361232bfda42344c38e91d65bd5dd2def4a934fec112aa312f04f531741f7265', 'tests/test_telegram_control.py': 'a94c6738d97e9674c5a6fd60ec11846efdd12056f54ec643006e5695f81c5273', 'tests/test_telegram_group_setup.py': '3dfd9bae62584a652e6abc4fb54e1a45f209acc45b6e3df1ea1e3e286f8c65f7'}

def materialize(destination):
    """Extract the exact embedded sources; does not execute or contact services."""
    import hashlib
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for name, source in SOURCES.items():
        if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
            raise ValueError("Embedded source integrity check failed: " + name)
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(source, encoding="utf-8")
    return destination


def run_embedded(notebook=False):
    import getpass
    import subprocess
    import tempfile
    if notebook:
        destination = Path.cwd() / "us_complete_pipeline_runtime"
    else:
        artifact_folder = Path(__file__).resolve().parent
        if artifact_folder.name == "portable" and (artifact_folder.parent / "run_us_pipeline.py").is_file():
            artifact_folder = artifact_folder.parent
        destination = artifact_folder / "work" / "standalone_runtime"
    materialize(destination)
    environment = dict(os.environ)
    if notebook:
        try:
            from google.colab import userdata
            for key in ("FRED_API_KEY", "GOOGLE_APPLICATION_CREDENTIALS", "TELEGRAM_CHAT_ID", "TELEGRAM_BOT_TOKEN"):
                if not environment.get(key):
                    try:
                        value = userdata.get(key)
                        if value: environment[key] = value
                    except Exception: pass
        except ImportError: pass
        credential = environment.get("GOOGLE_APPLICATION_CREDENTIALS")
        if not credential or not Path(credential).expanduser().is_file():
            from google.colab import files
            print("Upload the Google service-account JSON used for these workbooks.")
            secret_dir = Path(tempfile.mkdtemp(prefix="us_google_credentials_"))
            original = Path.cwd()
            try:
                os.chdir(secret_dir)
                uploaded = files.upload()
            finally:
                os.chdir(original)
            candidates = [secret_dir / Path(name).name for name in uploaded if name.lower().endswith(".json")]
            if len(candidates) != 1:
                raise ValueError("Upload exactly one Google credential JSON file")
            credential = str(candidates[0])
            Path(credential).chmod(0o600)
        environment["GOOGLE_APPLICATION_CREDENTIALS"] = str(Path(credential).expanduser().resolve())
        if STAGE in ("all", "importer") and not environment.get("FRED_API_KEY"):
            environment["FRED_API_KEY"] = getpass.getpass("FRED API key: ").strip()
        if STAGE in ("all", "importer") and not environment.get("FRED_API_KEY"):
            raise ValueError("A FRED API key is required")
        arguments = ["--stage", STAGE, "--dataset", DATASET]
    else:
        arguments = sys.argv[1:] or ["--stage", STAGE, "--dataset", DATASET]
    print("Extracted complete pipeline to", destination, flush=True)
    child = subprocess.Popen([sys.executable, "-u", str(destination / "run_us_pipeline.py"), *arguments], env=environment)
    try:
        code = child.wait()
    except KeyboardInterrupt:
        child.terminate()
        try: child.wait(timeout=10)
        except subprocess.TimeoutExpired: child.kill(); child.wait()
        raise
    if code:
        raise RuntimeError(f"Pipeline stopped with status {code}. Inspect the stage output above and {destination / 'work'}.")


if __name__ == "__main__":
    run_embedded(notebook="get_ipython" in globals())
